In [1]:
# Cell 1 — Phase 18 fresh deterministic setup and private paths
from __future__ import annotations

import os

# Set before CUDA is initialized.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "20260814")

import importlib.util
import json
import math
import platform
import random
import subprocess
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

if importlib.util.find_spec("SimpleITK") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-input",
        "SimpleITK==2.5.5",
    ])

import nibabel as nib
import numpy as np
import pandas as pd
import scipy
import SimpleITK as sitk
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy import ndimage
from scipy.ndimage import map_coordinates
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset


# ---------------------------------------------------------------------
# Frozen experiment identity
# ---------------------------------------------------------------------
PHASE = "18a_mt_bimorph_preflight"

BASE_SPLIT_SEED = 20260727

PHASE18_SEED = 20260814

N_ACQUISITION_GROUPS = 15
N_GROUP_FOLDS = 3
FOLD_SEARCH_CANDIDATES = 400_000
INNER_MONITOR_FRACTION = 0.20

PROBABILITY_EPS = 1e-5
EXPECTED_CASE_COUNT = 1362
EXPECTED_NORMAL_COUNT = 615
EXPECTED_PATHOLOGIC_COUNT = 747
EXPECTED_PREVALENCE = 747 / 1362

# Phase 18 registration and model inputs.
CONTEXT_CONFIG = {
    "cube_mm": 192.0,
    "shape": (64, 64, 64),
}
HIGHRES_CONFIG = {
    "cube_mm": 128.0,
    "shape": (80, 80, 80),
}
REGISTRATION_CONFIG = {
    "coarse_shape": (32, 32, 32),
    "template_count": 4,
    "template_names": (
        "normal",
        "left_predominant",
        "right_predominant",
        "bilateral_reduction",
    ),
    "maximum_translation_mm": 20.0,
    "maximum_rotation_deg": 18.0,
    "minimum_ncc_gain": 0.001,
    "maximum_fallback_rate": 0.01,
}

# These remain frozen until the first complete Phase 18 report.
ADVANCEMENT_GATE = {
    "minimum_log_loss_gain": 0.005,
    "minimum_auroc_gain": 0.003,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_harm": 0.005,
    "maximum_major_group_harm": 0.015,
    "maximum_registration_fallback_rate": 0.01,
    "minimum_calibration_slope": 0.80,
    "maximum_calibration_slope": 1.20,
    "phase12c_weight": 0.75,
    "phase18_weight": 0.25,
}

# Same private paths as the previous notebooks.
BASE = Path(os.environ.get(
    "DAT_PRIVATE_ROOT",
    "/kaggle/input/datasets/nahinalam/drivendata-dataset",
))
TRAIN_IMAGE_DIR = BASE / "niftis"
TRAIN_LABELS_CSV = BASE / "train_labels.csv"

SMOKE_ROOT = Path(os.environ.get(
    "DAT_PRIVATE_SMOKE_ROOT",
    str(BASE / "smoke_test_data"),
))
SMOKE_LABELS_DO_NOT_READ = SMOKE_ROOT / "test_labels.csv"

WORKING = Path(os.environ.get(
    "DAT_WORKING_ROOT",
    "/kaggle/working",
))

assert TRAIN_IMAGE_DIR.is_dir(), (
    f"Missing training image directory: {TRAIN_IMAGE_DIR}"
)
assert TRAIN_LABELS_CSV.is_file(), (
    f"Missing training labels: {TRAIN_LABELS_CSV}"
)
assert torch.cuda.is_available(), "A CUDA GPU is required."

GPU_COUNT = int(torch.cuda.device_count())
GPU_NAMES = [
    torch.cuda.get_device_name(index)
    for index in range(GPU_COUNT)
]
DEVICE = torch.device("cuda:0")
USE_AMP = True

# Model training stays on one GPU for deterministic reproducibility.
random.seed(PHASE18_SEED)
np.random.seed(PHASE18_SEED)
torch.manual_seed(PHASE18_SEED)
torch.cuda.manual_seed_all(PHASE18_SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = False
if hasattr(torch.backends.cudnn, "allow_tf32"):
    torch.backends.cudnn.allow_tf32 = False

torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(max(1, min(8, int(os.cpu_count() or 1))))

SETUP_REPORT = {
    "phase": PHASE,
    "fresh_notebook": True,
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "numpy": str(np.__version__),
    "scipy": str(scipy.__version__),
    "sklearn": str(sklearn.__version__),
    "simpleitk": str(sitk.Version_VersionString()),
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_count_detected": GPU_COUNT,
    "gpu_names": GPU_NAMES,
    "selected_training_device": str(DEVICE),
    "single_gpu_deterministic_training": True,
    "external_weights": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE18_SETUP")
print(json.dumps(SETUP_REPORT, indent=2))
print("END SANITIZED_PHASE18_SETUP")

BEGIN SANITIZED_PHASE18_SETUP
{
  "phase": "18a_mt_bimorph_preflight",
  "fresh_notebook": true,
  "python": "3.12.13",
  "torch": "2.10.0+cu128",
  "numpy": "2.0.2",
  "scipy": "1.16.3",
  "sklearn": "1.6.1",
  "simpleitk": "2.5.5",
  "cuda_available": true,
  "gpu_count_detected": 2,
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "selected_training_device": "cuda:0",
  "single_gpu_deterministic_training": true,
  "external_weights": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE18_SETUP


In [2]:
# Cell 2 — construct the private training index without displaying identifiers

def nifti_uid(path: Path) -> str:
    name = path.name
    lowered = name.casefold()

    if lowered.endswith(".nii.gz"):
        return name[:-7]
    if lowered.endswith(".nii"):
        return name[:-4]

    raise ValueError("Unexpected NIfTI extension")


train_paths = sorted(
    path
    for path in TRAIN_IMAGE_DIR.iterdir()
    if path.is_file()
    and (
        path.name.casefold().endswith(".nii")
        or path.name.casefold().endswith(".nii.gz")
    )
)

labels = pd.read_csv(
    TRAIN_LABELS_CSV,
    dtype={"uid": str},
)

assert list(labels.columns) == ["uid", "is_pathologic"]
assert labels["uid"].notna().all()
assert labels["uid"].is_unique
assert labels["is_pathologic"].isin([0, 1, 0.0, 1.0]).all()

path_by_uid = {
    nifti_uid(path): path
    for path in train_paths
}

assert len(path_by_uid) == len(train_paths)
assert len(labels) == len(train_paths)
assert set(labels["uid"]) == set(path_by_uid)

case_df = labels.copy()
case_df["uid"] = case_df["uid"].astype(str)
case_df["y"] = case_df["is_pathologic"].astype(np.int64)
case_df["path"] = case_df["uid"].map(path_by_uid)
case_df = case_df.drop(columns="is_pathologic")

assert case_df["path"].notna().all()

y = case_df["y"].to_numpy(dtype=np.int64)

assert len(case_df) == EXPECTED_CASE_COUNT
assert int((y == 0).sum()) == EXPECTED_NORMAL_COUNT
assert int((y == 1).sum()) == EXPECTED_PATHOLOGIC_COUNT
assert abs(float(y.mean()) - EXPECTED_PREVALENCE) < 1e-12

CASE_INDEX_REPORT = {
    "case_count": int(len(case_df)),
    "normal_count": int((y == 0).sum()),
    "pathologic_count": int((y == 1).sum()),
    "prevalence": round(float(y.mean()), 6),
    "image_count_matches_labels": True,
    "unique_uid_contract": True,
    "displayed_uid_count": 0,
    "displayed_patient_row_count": 0,
    "smoke_data_read": False,
    "test_data_read": False,
}

del labels, train_paths, path_by_uid

print("BEGIN SANITIZED_PHASE18_CASE_INDEX")
print(json.dumps(CASE_INDEX_REPORT, indent=2))
print("END SANITIZED_PHASE18_CASE_INDEX")

BEGIN SANITIZED_PHASE18_CASE_INDEX
{
  "case_count": 1362,
  "normal_count": 615,
  "pathologic_count": 747,
  "prevalence": 0.548458,
  "image_count_matches_labels": true,
  "unique_uid_contract": true,
  "displayed_uid_count": 0,
  "displayed_patient_row_count": 0,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE18_CASE_INDEX


In [3]:
# Cell 3 — header-only acquisition grouping
# No voxel array, UID, filename, or patient row is printed.

def header_features(path: Path) -> dict:
    image = nib.load(str(path), mmap=True)

    if len(image.shape) != 3:
        raise ValueError("Expected one three-dimensional NIfTI volume")

    shape = np.asarray(image.shape, dtype=np.float64)
    spacing = np.asarray(
        image.header.get_zooms()[:3],
        dtype=np.float64,
    )
    affine = np.asarray(image.affine, dtype=np.float64)

    if not np.isfinite(shape).all() or not (shape > 0).all():
        raise ValueError("Invalid NIfTI shape")
    if not np.isfinite(spacing).all() or not (spacing > 0).all():
        raise ValueError("Invalid NIfTI spacing")
    if not np.isfinite(affine).all():
        raise ValueError("Non-finite affine")
    if abs(float(np.linalg.det(affine[:3, :3]))) <= 1e-8:
        raise ValueError("Singular affine")

    fov = shape * spacing
    orientation = "".join(nib.aff2axcodes(affine))

    return {
        "shape_x": float(shape[0]),
        "shape_y": float(shape[1]),
        "shape_z": float(shape[2]),
        "spacing_x": float(spacing[0]),
        "spacing_y": float(spacing[1]),
        "spacing_z": float(spacing[2]),
        "fov_x": float(fov[0]),
        "fov_y": float(fov[1]),
        "fov_z": float(fov[2]),
        "dtype_is_int16": int(
            str(image.get_data_dtype()) == "int16"
        ),
        "orientation": orientation,
    }


header_started = time.time()
header_rows = []

for position, path in enumerate(case_df["path"], start=1):
    header_rows.append(header_features(path))

    if position % 500 == 0:
        print(f"Header progress: {position}/{len(case_df)}")

header_df = pd.DataFrame(
    header_rows,
    index=case_df.index,
)

# Preserve the previous dataset contract.
orientation_counts = header_df["orientation"].value_counts()
assert len(orientation_counts) == 1
assert str(orientation_counts.index[0]) == "RAS"

HEADER_COLS = [
    "shape_x",
    "shape_y",
    "shape_z",
    "spacing_x",
    "spacing_y",
    "spacing_z",
    "fov_x",
    "fov_y",
    "fov_z",
    "dtype_is_int16",
]

case_df = pd.concat(
    [
        case_df,
        header_df[HEADER_COLS],
    ],
    axis=1,
)

X_header = case_df[HEADER_COLS].to_numpy(dtype=np.float64)
assert np.isfinite(X_header).all()

# Exact previous transformation and KMeans seed.
X_cluster = StandardScaler().fit_transform(
    np.log1p(X_header)
)

cluster_model = KMeans(
    n_clusters=N_ACQUISITION_GROUPS,
    random_state=BASE_SPLIT_SEED,
    n_init=25,
)

groups = cluster_model.fit_predict(
    X_cluster
).astype(np.int64)

case_df["acquisition_group"] = groups

assert groups.shape == (EXPECTED_CASE_COUNT,)
assert np.unique(groups).size == N_ACQUISITION_GROUPS

group_sizes = np.bincount(
    groups,
    minlength=N_ACQUISITION_GROUPS,
)

ACQUISITION_REPORT = {
    "case_count": int(len(case_df)),
    "acquisition_group_count": int(np.unique(groups).size),
    "minimum_group_size": int(group_sizes.min()),
    "maximum_group_size": int(group_sizes.max()),
    "all_headers_ras": True,
    "all_header_values_finite": True,
    "elapsed_seconds": round(
        float(time.time() - header_started),
        2,
    ),
    "voxel_arrays_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

del header_rows, header_df, X_header, X_cluster, orientation_counts

print("BEGIN SANITIZED_PHASE18_ACQUISITION_REPORT")
print(json.dumps(ACQUISITION_REPORT, indent=2))
print("END SANITIZED_PHASE18_ACQUISITION_REPORT")

Header progress: 500/1362
Header progress: 1000/1362
BEGIN SANITIZED_PHASE18_ACQUISITION_REPORT
{
  "case_count": 1362,
  "acquisition_group_count": 15,
  "minimum_group_size": 4,
  "maximum_group_size": 456,
  "all_headers_ras": true,
  "all_header_values_finite": true,
  "elapsed_seconds": 11.29,
  "voxel_arrays_read": false,
  "uids_displayed": false,
  "patient_rows_displayed": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE18_ACQUISITION_REPORT


In [4]:
# Cell 4 — reproduce and freeze the corrected acquisition-held-out outer folds

split_source = pd.DataFrame({
    "acquisition_group": np.asarray(groups, dtype=np.int64),
    "target": np.asarray(y, dtype=np.int64),
})

group_table = (
    split_source
    .groupby("acquisition_group", as_index=False)
    .agg(
        n=("target", "size"),
        positives=("target", "sum"),
    )
    .sort_values("acquisition_group")
    .reset_index(drop=True)
)
group_table["negatives"] = group_table["n"] - group_table["positives"]

group_ids = group_table["acquisition_group"].to_numpy(dtype=np.int64)
group_n = group_table["n"].to_numpy(dtype=np.float64)
group_pos = group_table["positives"].to_numpy(dtype=np.float64)
group_neg = group_table["negatives"].to_numpy(dtype=np.float64)

G = len(group_ids)
assert G == N_ACQUISITION_GROUPS

rng = np.random.default_rng(BASE_SPLIT_SEED + 10)

assignments = rng.integers(
    0,
    N_GROUP_FOLDS,
    size=(FOLD_SEARCH_CANDIDATES, G),
    dtype=np.int8,
)

# Anchor the largest acquisition group to fold 0, removing fold-label symmetry.
largest_group_position = int(np.argmax(group_n))
assignments[:, largest_group_position] = 0

sizes = np.zeros(
    (FOLD_SEARCH_CANDIDATES, N_GROUP_FOLDS),
    dtype=np.float64,
)
positives = np.zeros_like(sizes)
negatives = np.zeros_like(sizes)
group_counts = np.zeros_like(sizes)

for fold in range(N_GROUP_FOLDS):
    assignment_mask = assignments == fold

    sizes[:, fold] = assignment_mask @ group_n
    positives[:, fold] = assignment_mask @ group_pos
    negatives[:, fold] = assignment_mask @ group_neg
    group_counts[:, fold] = assignment_mask.sum(axis=1)

target_n = len(case_df) / N_GROUP_FOLDS
target_pos = float(y.sum()) / N_GROUP_FOLDS
target_neg = float((1 - y).sum()) / N_GROUP_FOLDS
target_groups = G / N_GROUP_FOLDS

valid_assignment = (
    (sizes.min(axis=1) >= 0.20 * len(case_df))
    & (positives.min(axis=1) > 0)
    & (negatives.min(axis=1) > 0)
    & (group_counts.min(axis=1) >= 1)
)

assert valid_assignment.any(), (
    "No valid acquisition-grouped fold assignment was found."
)

assignment_score = (
    np.mean(((sizes - target_n) / target_n) ** 2, axis=1)
    + np.mean(((positives - target_pos) / target_pos) ** 2, axis=1)
    + np.mean(((negatives - target_neg) / target_neg) ** 2, axis=1)
    + 0.02
    * np.mean(
        ((group_counts - target_groups) / target_groups) ** 2,
        axis=1,
    )
)

assignment_score[~valid_assignment] = np.inf

best_assignment_index = int(np.argmin(assignment_score))
best_assignment = assignments[best_assignment_index].copy()

group_to_fold = {
    int(group_id): int(fold)
    for group_id, fold in zip(group_ids, best_assignment)
}

fold_id = np.asarray(
    [group_to_fold[int(group)] for group in groups],
    dtype=np.int64,
)

case_df["fold"] = fold_id

outer_splits = []
outer_fold_rows = []

for fold in range(N_GROUP_FOLDS):
    valid_idx = np.flatnonzero(fold_id == fold)
    train_idx = np.flatnonzero(fold_id != fold)

    train_groups = set(groups[train_idx].tolist())
    valid_groups = set(groups[valid_idx].tolist())

    assert train_groups.isdisjoint(valid_groups)
    assert np.unique(y[valid_idx]).size == 2
    assert np.unique(y[train_idx]).size == 2

    outer_splits.append((train_idx, valid_idx))

    outer_fold_rows.append({
        "fold": int(fold),
        "train_n": int(len(train_idx)),
        "valid_n": int(len(valid_idx)),
        "valid_normal": int((y[valid_idx] == 0).sum()),
        "valid_pathologic": int((y[valid_idx] == 1).sum()),
        "valid_prevalence": round(float(y[valid_idx].mean()), 6),
        "valid_group_count": int(
            np.unique(groups[valid_idx]).size
        ),
    })

outer_fold_report = pd.DataFrame(outer_fold_rows)

# Exact reproduction guard from the frozen Phase 7–18 lineage.
expected_outer_counts = [
    {
        "fold": 0,
        "train_n": 895,
        "valid_n": 467,
        "valid_normal": 192,
        "valid_pathologic": 275,
        "valid_group_count": 3,
    },
    {
        "fold": 1,
        "train_n": 919,
        "valid_n": 443,
        "valid_normal": 215,
        "valid_pathologic": 228,
        "valid_group_count": 6,
    },
    {
        "fold": 2,
        "train_n": 910,
        "valid_n": 452,
        "valid_normal": 208,
        "valid_pathologic": 244,
        "valid_group_count": 6,
    },
]

for observed, expected in zip(
    outer_fold_rows,
    expected_outer_counts,
):
    for key, expected_value in expected.items():
        assert observed[key] == expected_value, (
            f"Frozen split mismatch: fold={expected['fold']}, "
            f"field={key}, observed={observed[key]}, "
            f"expected={expected_value}"
        )

fold_size_ratio = (
    outer_fold_report["valid_n"].max()
    / outer_fold_report["valid_n"].min()
)

assert fold_size_ratio <= 1.30
assert sum(row["valid_n"] for row in outer_fold_rows) == len(y)
assert sum(row["valid_pathologic"] for row in outer_fold_rows) == int(y.sum())

sanitized_outer_report = {
    "phase": PHASE,
    "split_seed": int(BASE_SPLIT_SEED),
    "search_candidates": int(FOLD_SEARCH_CANDIDATES),
    "fold_count": int(N_GROUP_FOLDS),
    "acquisition_group_count": int(G),
    "best_assignment_score": round(
        float(assignment_score[best_assignment_index]),
        9,
    ),
    "fold_size_max_min_ratio": round(float(fold_size_ratio), 6),
    "folds": outer_fold_rows,
    "exact_frozen_split_reproduced": True,
    "acquisition_group_leakage": False,
    "voxel_arrays_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE18_OUTER_FOLDS")
print(json.dumps(sanitized_outer_report, indent=2))
print("END SANITIZED_PHASE18_OUTER_FOLDS")

# Release the large randomized-search matrices.
del (
    assignments,
    sizes,
    positives,
    negatives,
    group_counts,
    assignment_score,
    valid_assignment,
    split_source,
)

BEGIN SANITIZED_PHASE18_OUTER_FOLDS
{
  "phase": "18a_mt_bimorph_preflight",
  "split_seed": 20260727,
  "search_candidates": 400000,
  "fold_count": 3,
  "acquisition_group_count": 15,
  "best_assignment_score": 0.010420183,
  "fold_size_max_min_ratio": 1.054176,
  "folds": [
    {
      "fold": 0,
      "train_n": 895,
      "valid_n": 467,
      "valid_normal": 192,
      "valid_pathologic": 275,
      "valid_prevalence": 0.588865,
      "valid_group_count": 3
    },
    {
      "fold": 1,
      "train_n": 919,
      "valid_n": 443,
      "valid_normal": 215,
      "valid_pathologic": 228,
      "valid_prevalence": 0.514673,
      "valid_group_count": 6
    },
    {
      "fold": 2,
      "train_n": 910,
      "valid_n": 452,
      "valid_normal": 208,
      "valid_pathologic": 244,
      "valid_prevalence": 0.539823,
      "valid_group_count": 6
    }
  ],
  "exact_frozen_split_reproduced": true,
  "acquisition_group_leakage": false,
  "voxel_arrays_read": false,
  "uids_displayed"

In [5]:
# Cell 5 — construct nested group-exclusive fit/monitor partitions

def select_inner_monitor_split(
    outer_train_idx,
    outer_valid_idx,
):
    available_groups = np.asarray(
        sorted(np.unique(groups[outer_train_idx]).tolist()),
        dtype=np.int64,
    )

    target_n = len(outer_train_idx) * INNER_MONITOR_FRACTION
    target_prevalence = float(y[outer_train_idx].mean())

    best_candidate = None

    # At most 12 groups occur in an outer-training partition, so exact
    # enumeration is small and completely deterministic.
    for bits in range(1, (1 << len(available_groups)) - 1):
        monitor_groups = available_groups[
            [
                bool(bits & (1 << position))
                for position in range(len(available_groups))
            ]
        ]

        monitor_mask = np.isin(
            groups[outer_train_idx],
            monitor_groups,
        )

        monitor_idx = outer_train_idx[monitor_mask]
        fit_idx = outer_train_idx[~monitor_mask]

        if len(monitor_idx) < 80:
            continue

        if len(fit_idx) < 0.60 * len(outer_train_idx):
            continue

        if np.unique(y[monitor_idx]).size != 2:
            continue

        if np.unique(y[fit_idx]).size != 2:
            continue

        size_term = (
            (len(monitor_idx) - target_n) / target_n
        ) ** 2

        prevalence_term = (
            (
                float(y[monitor_idx].mean())
                - target_prevalence
            )
            / 0.10
        ) ** 2

        group_term = (
            0.002 * (len(monitor_groups) - 2.0) ** 2
        )

        split_score = (
            size_term
            + prevalence_term
            + group_term
        )

        # The group tuple is the deterministic tie-breaker.
        candidate = (
            float(split_score),
            tuple(monitor_groups.tolist()),
        )

        if (
            best_candidate is None
            or candidate < best_candidate
        ):
            best_candidate = candidate

    assert best_candidate is not None, (
        "Could not construct an inner group-exclusive monitor split."
    )

    selected_monitor_groups = np.asarray(
        best_candidate[1],
        dtype=np.int64,
    )

    monitor_mask = np.isin(
        groups[outer_train_idx],
        selected_monitor_groups,
    )

    monitor_idx = outer_train_idx[monitor_mask]
    fit_idx = outer_train_idx[~monitor_mask]

    fit_groups = set(groups[fit_idx].tolist())
    monitor_groups = set(groups[monitor_idx].tolist())
    validation_groups = set(groups[outer_valid_idx].tolist())

    assert fit_groups.isdisjoint(monitor_groups)
    assert fit_groups.isdisjoint(validation_groups)
    assert monitor_groups.isdisjoint(validation_groups)

    return (
        fit_idx,
        monitor_idx,
        float(best_candidate[0]),
    )


nested_splits = []
nested_fold_rows = []

for fold, (outer_train_idx, outer_valid_idx) in enumerate(
    outer_splits
):
    fit_idx, monitor_idx, split_score = (
        select_inner_monitor_split(
            outer_train_idx,
            outer_valid_idx,
        )
    )

    nested_splits.append(
        (fit_idx, monitor_idx, outer_valid_idx)
    )

    nested_fold_rows.append({
        "fold": int(fold),
        "fit_n": int(len(fit_idx)),
        "monitor_n": int(len(monitor_idx)),
        "outer_valid_n": int(len(outer_valid_idx)),
        "fit_group_count": int(
            np.unique(groups[fit_idx]).size
        ),
        "monitor_group_count": int(
            np.unique(groups[monitor_idx]).size
        ),
        "outer_valid_group_count": int(
            np.unique(groups[outer_valid_idx]).size
        ),
        "monitor_prevalence": round(
            float(y[monitor_idx].mean()),
            5,
        ),
        "split_score": round(float(split_score), 6),
    })

expected_nested_rows = [
    {
        "fold": 0,
        "fit_n": 721,
        "monitor_n": 174,
        "outer_valid_n": 467,
        "fit_group_count": 8,
        "monitor_group_count": 4,
        "outer_valid_group_count": 3,
        "monitor_prevalence": 0.52299,
        "split_score": 0.010704,
    },
    {
        "fold": 1,
        "fit_n": 796,
        "monitor_n": 123,
        "outer_valid_n": 443,
        "fit_group_count": 5,
        "monitor_group_count": 4,
        "outer_valid_group_count": 6,
        "monitor_prevalence": 0.56098,
        "split_score": 0.118845,
    },
    {
        "fold": 2,
        "fit_n": 714,
        "monitor_n": 196,
        "outer_valid_n": 452,
        "fit_group_count": 5,
        "monitor_group_count": 4,
        "outer_valid_group_count": 6,
        "monitor_prevalence": 0.52041,
        "split_score": 0.118499,
    },
]

assert nested_fold_rows == expected_nested_rows, (
    "Nested partitions do not match the frozen reference."
)

# Final structural audit.
for fit_idx, monitor_idx, outer_valid_idx in nested_splits:
    assert len(
        np.intersect1d(fit_idx, monitor_idx)
    ) == 0

    assert len(
        np.intersect1d(fit_idx, outer_valid_idx)
    ) == 0

    assert len(
        np.intersect1d(monitor_idx, outer_valid_idx)
    ) == 0

    combined_idx = np.concatenate(
        [fit_idx, monitor_idx, outer_valid_idx]
    )

    assert len(np.unique(combined_idx)) == len(y)
    assert np.unique(y[fit_idx]).size == 2
    assert np.unique(y[monitor_idx]).size == 2
    assert np.unique(y[outer_valid_idx]).size == 2

sanitized_nested_report = {
    "phase": PHASE,
    "nested_partitions": nested_fold_rows,
    "exact_frozen_nested_splits_reproduced": True,
    "all_partitions_group_exclusive": True,
    "all_partitions_contain_both_classes": True,
    "outer_validation_used_for_selection": False,
    "voxel_arrays_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE18_NESTED_SPLITS")
print(json.dumps(sanitized_nested_report, indent=2))
print("END SANITIZED_PHASE18_NESTED_SPLITS")

BEGIN SANITIZED_PHASE18_NESTED_SPLITS
{
  "phase": "18a_mt_bimorph_preflight",
  "nested_partitions": [
    {
      "fold": 0,
      "fit_n": 721,
      "monitor_n": 174,
      "outer_valid_n": 467,
      "fit_group_count": 8,
      "monitor_group_count": 4,
      "outer_valid_group_count": 3,
      "monitor_prevalence": 0.52299,
      "split_score": 0.010704
    },
    {
      "fold": 1,
      "fit_n": 796,
      "monitor_n": 123,
      "outer_valid_n": 443,
      "fit_group_count": 5,
      "monitor_group_count": 4,
      "outer_valid_group_count": 6,
      "monitor_prevalence": 0.56098,
      "split_score": 0.118845
    },
    {
      "fold": 2,
      "fit_n": 714,
      "monitor_n": 196,
      "outer_valid_n": 452,
      "fit_group_count": 5,
      "monitor_group_count": 4,
      "outer_valid_group_count": 6,
      "monitor_prevalence": 0.52041,
      "split_score": 0.118499
    }
  ],
  "exact_frozen_nested_splits_reproduced": true,
  "all_partitions_group_exclusive": true,
  "all

In [8]:
# Cell 6 — one-time canonical multiscale cache for Phase 18
#
# Reads training NIfTI voxel arrays only.
# Nothing case-level is printed or exported.

from scipy.ndimage import map_coordinates, zoom

PHASE18_CACHE_CONFIG = {
    "highres": {
        "cube_mm": 128.0,
        "shape": (80, 80, 80),
    },
    "context": {
        "cube_mm": 192.0,
        "shape": (64, 64, 64),
    },
    "registration": {
        "cube_mm": 128.0,
        "shape": (32, 32, 32),
    },
}

assert len(case_df) == 1362
assert tuple(PHASE18_CACHE_CONFIG["highres"]["shape"]) == (80, 80, 80)
assert tuple(PHASE18_CACHE_CONFIG["context"]["shape"]) == (64, 64, 64)
assert tuple(PHASE18_CACHE_CONFIG["registration"]["shape"]) == (32, 32, 32)


def phase18_centered_physical_offsets(cube_mm, output_shape):
    """Return a fixed RAS physical grid centered at zero."""
    output_shape = tuple(int(value) for value in output_shape)

    assert len(output_shape) == 3
    assert len(set(output_shape)) == 1

    spacing_mm = float(cube_mm) / float(output_shape[0])

    axes = [
        (
            np.arange(axis_size, dtype=np.float32)
            - (axis_size - 1.0) / 2.0
        )
        * spacing_mm
        for axis_size in output_shape
    ]

    grid_x, grid_y, grid_z = np.meshgrid(
        *axes,
        indexing="ij",
    )

    offsets = np.stack(
        [grid_x, grid_y, grid_z],
        axis=0,
    ).reshape(3, -1)

    return np.ascontiguousarray(offsets, dtype=np.float32)


PHASE18_PHYSICAL_OFFSETS = {
    name: phase18_centered_physical_offsets(
        config["cube_mm"],
        config["shape"],
    )
    for name, config in PHASE18_CACHE_CONFIG.items()
    if name != "registration"
}


def phase18_load_canonical_training_image(path):
    image = nib.load(str(path), mmap=True)
    image = nib.as_closest_canonical(image)

    volume = np.asarray(
        image.dataobj,
        dtype=np.float32,
    )

    if volume.ndim != 3:
        raise ValueError("Training NIfTI is not three-dimensional")

    volume = np.nan_to_num(
        volume,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    volume = np.clip(volume, 0.0, None).astype(
        np.float32,
        copy=False,
    )

    affine = np.asarray(image.affine, dtype=np.float64)

    if not np.isfinite(volume).all():
        raise ValueError("Non-finite canonical volume")

    if not np.isfinite(affine).all():
        raise ValueError("Non-finite canonical affine")

    if np.linalg.det(affine[:3, :3]) == 0:
        raise ValueError("Singular canonical affine")

    return volume, affine


def phase18_robust_uptake_center_world(volume, affine):
    """Locate the striatal uptake center without labels or templates."""
    shape = np.asarray(volume.shape, dtype=np.int64)

    positive = volume[volume > 0]
    if positive.size < 32:
        raise ValueError("Insufficient positive uptake voxels")

    # Suppress peripheral reconstruction artifacts.
    lower = np.floor(
        shape * np.asarray([0.20, 0.20, 0.10])
    ).astype(np.int64)

    upper = np.ceil(
        shape * np.asarray([0.80, 0.80, 0.75])
    ).astype(np.int64)

    central = volume[
        lower[0]:upper[0],
        lower[1]:upper[1],
        lower[2]:upper[2],
    ]

    central_positive = central[central > 0]

    if central_positive.size < 32:
        raise ValueError("Insufficient central uptake voxels")

    threshold = float(
        np.percentile(central_positive, 99.0)
    )

    hot_coordinates = (
        np.argwhere(central >= threshold)
        + lower[None, :]
    )

    hot_values = volume[
        tuple(hot_coordinates.T)
    ].astype(np.float64)

    keep = hot_values > 0
    hot_coordinates = hot_coordinates[keep]
    hot_values = hot_values[keep]

    if hot_values.size < 8:
        raise ValueError("Insufficient high-uptake voxels")

    weights = np.maximum(
        hot_values - threshold,
        0.0,
    )

    if float(weights.sum()) <= 0:
        weights = np.ones_like(hot_values)

    center_voxel = np.average(
        hot_coordinates,
        axis=0,
        weights=weights,
    )

    center_world = nib.affines.apply_affine(
        affine,
        center_voxel,
    ).astype(np.float64)

    if not np.isfinite(center_world).all():
        raise ValueError("Non-finite uptake center")

    return center_world


def phase18_resample_from_loaded_volume(
    volume,
    affine,
    center_world,
    physical_offsets,
    output_shape,
):
    inverse_affine = np.linalg.inv(affine)

    world_coordinates = (
        physical_offsets.astype(np.float64, copy=False)
        + center_world[:, None]
    )

    voxel_coordinates = (
        inverse_affine[:3, :3] @ world_coordinates
        + inverse_affine[:3, 3:4]
    )

    sampled = map_coordinates(
        volume,
        voxel_coordinates,
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )

    sampled = sampled.reshape(
        tuple(int(value) for value in output_shape)
    )

    return np.clip(
        sampled,
        0.0,
        None,
    ).astype(np.float32, copy=False)


def phase18_normalization_scale(highres_volume):
    positive = highres_volume[highres_volume > 0]

    if positive.size < 32:
        raise ValueError(
            "Insufficient positive voxels in high-resolution crop"
        )

    scale = float(np.percentile(positive, 99.5))

    if not np.isfinite(scale) or scale <= 1e-6:
        raise ValueError("Invalid uptake normalization scale")

    return scale


def phase18_apply_shared_normalization(volume, scale):
    normalized = np.clip(
        np.asarray(volume, dtype=np.float32) / float(scale),
        0.0,
        1.5,
    )

    if not np.isfinite(normalized).all():
        raise ValueError("Non-finite normalized crop")

    return normalized.astype(np.float32, copy=False)


number_of_cases = len(case_df)

PHASE18_VOLUMES = {
    "highres": np.empty(
        (
            number_of_cases,
            *PHASE18_CACHE_CONFIG["highres"]["shape"],
        ),
        dtype=np.float16,
    ),
    "context": np.empty(
        (
            number_of_cases,
            *PHASE18_CACHE_CONFIG["context"]["shape"],
        ),
        dtype=np.float16,
    ),
}

PHASE18_REGISTRATION_VOLUMES = np.empty(
    (
        number_of_cases,
        *PHASE18_CACHE_CONFIG["registration"]["shape"],
    ),
    dtype=np.float16,
)

normalization_scales = np.empty(
    number_of_cases,
    dtype=np.float64,
)

cache_accumulators = {
    name: {
        "mean_sum": 0.0,
        "positive_fraction_sum": 0.0,
        "minimum": np.inf,
        "maximum": -np.inf,
    }
    for name in ("highres", "context", "registration")
}

preprocessing_started = time.perf_counter()
preprocessing_failure_count = 0
PHASE18_PRIVATE_PREPROCESS_FAILURE = None

for private_position, path in enumerate(
    case_df["path"].tolist()
):
    try:
        source_volume, source_affine = (
            phase18_load_canonical_training_image(path)
        )

        center_world = phase18_robust_uptake_center_world(
            source_volume,
            source_affine,
        )

        raw_highres = phase18_resample_from_loaded_volume(
            source_volume,
            source_affine,
            center_world,
            PHASE18_PHYSICAL_OFFSETS["highres"],
            PHASE18_CACHE_CONFIG["highres"]["shape"],
        )

        scale = phase18_normalization_scale(raw_highres)
        normalization_scales[private_position] = scale

        normalized_highres = (
            phase18_apply_shared_normalization(
                raw_highres,
                scale,
            )
        )

        raw_context = phase18_resample_from_loaded_volume(
            source_volume,
            source_affine,
            center_world,
            PHASE18_PHYSICAL_OFFSETS["context"],
            PHASE18_CACHE_CONFIG["context"]["shape"],
        )

        normalized_context = (
            phase18_apply_shared_normalization(
                raw_context,
                scale,
            )
        )

        # The 32³ registration representation is derived from the
        # normalized 80³ crop. This guarantees an identical center,
        # orientation, physical field of view, and intensity scale.
        normalized_registration = zoom(
            normalized_highres,
            zoom=(
                PHASE18_CACHE_CONFIG["registration"]["shape"][0]
                / PHASE18_CACHE_CONFIG["highres"]["shape"][0],
            )
            * 3,
            order=1,
            mode="constant",
            cval=0.0,
            prefilter=False,
            grid_mode=False,
        ).astype(np.float32)

        if normalized_registration.shape != (
            32,
            32,
            32,
        ):
            raise ValueError(
                "Unexpected registration representation shape"
            )

        normalized_registration = np.clip(
            normalized_registration,
            0.0,
            1.5,
        )

        PHASE18_VOLUMES["highres"][private_position] = (
            normalized_highres.astype(np.float16)
        )
        PHASE18_VOLUMES["context"][private_position] = (
            normalized_context.astype(np.float16)
        )
        PHASE18_REGISTRATION_VOLUMES[private_position] = (
            normalized_registration.astype(np.float16)
        )

        current_volumes = {
            "highres": normalized_highres,
            "context": normalized_context,
            "registration": normalized_registration,
        }

        for name, current_volume in current_volumes.items():
            accumulator = cache_accumulators[name]

            accumulator["mean_sum"] += float(
                current_volume.mean()
            )
            accumulator["positive_fraction_sum"] += float(
                np.count_nonzero(current_volume)
                / current_volume.size
            )
            accumulator["minimum"] = min(
                accumulator["minimum"],
                float(current_volume.min()),
            )
            accumulator["maximum"] = max(
                accumulator["maximum"],
                float(current_volume.max()),
            )

    except Exception as exception:
        preprocessing_failure_count += 1

        # Kept privately for notebook debugging; never print or share it.
        PHASE18_PRIVATE_PREPROCESS_FAILURE = {
            "private_position": int(private_position),
            "error_type": type(exception).__name__,
        }

        raise RuntimeError(
            "Phase 18 training preprocessing failed. "
            f"Error type: {type(exception).__name__}. "
            "The private case position has not been displayed."
        ) from exception

    finally:
        for temporary_name in (
            "source_volume",
            "source_affine",
            "center_world",
            "raw_highres",
            "raw_context",
            "normalized_highres",
            "normalized_context",
            "normalized_registration",
            "current_volumes",
        ):
            if temporary_name in locals():
                del locals()[temporary_name]

    completed = private_position + 1

    if completed % 200 == 0:
        print(
            f"Canonical preprocessing progress: "
            f"{completed}/{number_of_cases}"
        )

PHASE18_PREPROCESS_SECONDS = float(
    time.perf_counter() - preprocessing_started
)

assert preprocessing_failure_count == 0
assert np.isfinite(normalization_scales).all()
assert (normalization_scales > 0).all()

assert PHASE18_VOLUMES["highres"].shape == (
    number_of_cases,
    80,
    80,
    80,
)
assert PHASE18_VOLUMES["context"].shape == (
    number_of_cases,
    64,
    64,
    64,
)
assert PHASE18_REGISTRATION_VOLUMES.shape == (
    number_of_cases,
    32,
    32,
    32,
)

# Validate in bounded blocks to avoid creating a large temporary mask.
for start in range(0, number_of_cases, 64):
    stop = min(start + 64, number_of_cases)

    assert np.isfinite(
        PHASE18_VOLUMES["highres"][start:stop]
    ).all()

    assert np.isfinite(
        PHASE18_VOLUMES["context"][start:stop]
    ).all()

    assert np.isfinite(
        PHASE18_REGISTRATION_VOLUMES[start:stop]
    ).all()

normalization_scale_quantiles = np.quantile(
    normalization_scales,
    [0.05, 0.50, 0.95],
)

cache_report = {}

for name in ("highres", "context", "registration"):
    if name == "registration":
        cached_array = PHASE18_REGISTRATION_VOLUMES
    else:
        cached_array = PHASE18_VOLUMES[name]

    accumulator = cache_accumulators[name]

    cache_report[name] = {
        "shape": [int(value) for value in cached_array.shape],
        "dtype": str(cached_array.dtype),
        "ram_gb": round(
            float(cached_array.nbytes / 1e9),
            4,
        ),
        "aggregate_mean": round(
            accumulator["mean_sum"] / number_of_cases,
            6,
        ),
        "aggregate_positive_fraction": round(
            accumulator["positive_fraction_sum"]
            / number_of_cases,
            6,
        ),
        "global_minimum": round(
            float(accumulator["minimum"]),
            6,
        ),
        "global_maximum": round(
            float(accumulator["maximum"]),
            6,
        ),
    }

sanitized_cache_report = {
    "phase": PHASE,
    "case_count": int(number_of_cases),
    "elapsed_seconds": round(
        PHASE18_PREPROCESS_SECONDS,
        2,
    ),
    "failure_count": int(preprocessing_failure_count),
    "orientation": "canonical_RAS",
    "localization": "central_positive_p99_weighted_center",
    "normalization": (
        "shared_per_case_highres_positive_p99.5;"
        " clipped_0_to_1.5"
    ),
    "cache": cache_report,
    "normalization_scale_quantiles": {
        "q05": round(
            float(normalization_scale_quantiles[0]),
            6,
        ),
        "q50": round(
            float(normalization_scale_quantiles[1]),
            6,
        ),
        "q95": round(
            float(normalization_scale_quantiles[2]),
            6,
        ),
    },
    "total_cached_ram_gb": round(
        float(
            PHASE18_VOLUMES["highres"].nbytes
            + PHASE18_VOLUMES["context"].nbytes
            + PHASE18_REGISTRATION_VOLUMES.nbytes
        )
        / 1e9,
        4,
    ),
    "training_voxel_arrays_read": True,
    "case_level_cache_exported": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE18_CANONICAL_CACHE")
print(json.dumps(sanitized_cache_report, indent=2))
print("END SANITIZED_PHASE18_CANONICAL_CACHE")

Canonical preprocessing progress: 200/1362
Canonical preprocessing progress: 400/1362
Canonical preprocessing progress: 600/1362
Canonical preprocessing progress: 800/1362
Canonical preprocessing progress: 1000/1362
Canonical preprocessing progress: 1200/1362
BEGIN SANITIZED_PHASE18_CANONICAL_CACHE
{
  "phase": "18a_mt_bimorph_preflight",
  "case_count": 1362,
  "elapsed_seconds": 509.8,
  "failure_count": 0,
  "orientation": "canonical_RAS",
  "localization": "central_positive_p99_weighted_center",
  "normalization": "shared_per_case_highres_positive_p99.5; clipped_0_to_1.5",
  "cache": {
    "highres": {
      "shape": [
        1362,
        80,
        80,
        80
      ],
      "dtype": "float16",
      "ram_gb": 1.3947,
      "aggregate_mean": 0.295366,
      "aggregate_positive_fraction": 0.929364,
      "global_minimum": 0.0,
      "global_maximum": 1.5
    },
    "context": {
      "shape": [
        1362,
        64,
        64,
        64
      ],
      "dtype": "float16"

In [9]:
# Cell 7 — fold-local four-phenotype template construction

from scipy.ndimage import gaussian_filter

PHASE18_TEMPLATE_NAMES = (
    "normal",
    "bilateral_abnormal",
    "left_reduction_abnormal",
    "right_reduction_abnormal",
)

registration_shape = tuple(
    PHASE18_REGISTRATION_VOLUMES.shape[1:]
)
assert registration_shape == (32, 32, 32)

registration_spacing_mm = (
    PHASE18_CACHE_CONFIG["registration"]["cube_mm"]
    / registration_shape[0]
)
assert registration_spacing_mm == 4.0


# ------------------------------------------------------------------
# Fixed, reflection-symmetric striatal proxy masks
# ------------------------------------------------------------------

coordinate_axes_mm = [
    (
        np.arange(axis_size, dtype=np.float32)
        - (axis_size - 1.0) / 2.0
    )
    * registration_spacing_mm
    for axis_size in registration_shape
]

coordinate_x, coordinate_y, coordinate_z = np.meshgrid(
    *coordinate_axes_mm,
    indexing="ij",
)

central_ellipsoid = (
    (coordinate_x / 42.0) ** 2
    + (coordinate_y / 48.0) ** 2
    + (coordinate_z / 38.0) ** 2
) <= 1.0

# Exclude the immediate midline and peripheral voxels.
hemisphere_band = (
    (np.abs(coordinate_x) >= 4.0)
    & (np.abs(coordinate_x) <= 36.0)
    & (np.abs(coordinate_y) <= 40.0)
    & (np.abs(coordinate_z) <= 32.0)
)

left_proxy_mask = (
    central_ellipsoid
    & hemisphere_band
    & (coordinate_x < 0)
)

right_proxy_mask = (
    central_ellipsoid
    & hemisphere_band
    & (coordinate_x > 0)
)

assert left_proxy_mask.sum() == right_proxy_mask.sum()
assert left_proxy_mask.sum() >= 300
assert np.array_equal(
    left_proxy_mask,
    np.flip(right_proxy_mask, axis=0),
)


def phase18_regional_uptake_score(values):
    """Robust score sensitive to both uptake extent and peak intensity."""
    values = np.asarray(values, dtype=np.float32)

    quantiles = np.quantile(
        values,
        [0.50, 0.70, 0.85, 0.95],
    )

    weights = np.asarray(
        [0.15, 0.25, 0.35, 0.25],
        dtype=np.float64,
    )

    score = float(np.dot(quantiles, weights))

    if not np.isfinite(score):
        raise ValueError("Non-finite regional uptake score")

    return score


def phase18_case_laterality_score(volume):
    """
    Positive score: left-side uptake is relatively lower.
    Negative score: right-side uptake is relatively lower.
    """
    smoothed = gaussian_filter(
        np.asarray(volume, dtype=np.float32),
        sigma=0.65,
        mode="constant",
    )

    left_score = phase18_regional_uptake_score(
        smoothed[left_proxy_mask]
    )
    right_score = phase18_regional_uptake_score(
        smoothed[right_proxy_mask]
    )

    denominator = max(
        left_score + right_score,
        1e-6,
    )

    laterality = (
        right_score - left_score
    ) / denominator

    if not np.isfinite(laterality):
        raise ValueError("Non-finite laterality score")

    return float(laterality)


# Label-blind casewise laterality measurement.
# Labels are used only later, within each fold's fit partition.
PHASE18_LATERALITY_SCORES = np.asarray(
    [
        phase18_case_laterality_score(volume)
        for volume in PHASE18_REGISTRATION_VOLUMES
    ],
    dtype=np.float64,
)

assert PHASE18_LATERALITY_SCORES.shape == (len(y),)
assert np.isfinite(PHASE18_LATERALITY_SCORES).all()


def phase18_finalize_template(template):
    template = gaussian_filter(
        np.asarray(template, dtype=np.float32),
        sigma=0.55,
        mode="constant",
    )

    positive = template[template > 0]

    if positive.size < 32:
        raise ValueError("Insufficient positive template voxels")

    scale = max(
        float(np.percentile(positive, 99.5)),
        1e-6,
    )

    template = np.clip(
        template / scale,
        0.0,
        1.5,
    ).astype(np.float32)

    if not np.isfinite(template).all():
        raise ValueError("Non-finite phenotype template")

    return template


def phase18_mean_template(
    source_indices,
    reflect_flags=None,
    enforce_symmetry=False,
):
    source_indices = np.asarray(
        source_indices,
        dtype=np.int64,
    )

    if source_indices.size == 0:
        raise ValueError("Cannot build an empty template")

    if reflect_flags is None:
        reflect_flags = np.zeros(
            source_indices.size,
            dtype=bool,
        )
    else:
        reflect_flags = np.asarray(
            reflect_flags,
            dtype=bool,
        )

    assert len(reflect_flags) == len(source_indices)

    accumulator = np.zeros(
        registration_shape,
        dtype=np.float64,
    )

    for source_index, should_reflect in zip(
        source_indices,
        reflect_flags,
    ):
        volume = np.asarray(
            PHASE18_REGISTRATION_VOLUMES[
                int(source_index)
            ],
            dtype=np.float32,
        )

        if should_reflect:
            volume = np.flip(volume, axis=0)

        # Mild winsorization reduces isolated reconstruction peaks.
        accumulator += np.minimum(volume, 1.25)

    mean_template = (
        accumulator / float(source_indices.size)
    ).astype(np.float32)

    if enforce_symmetry:
        mean_template = 0.5 * (
            mean_template
            + np.flip(mean_template, axis=0)
        )

    return phase18_finalize_template(mean_template)


def phase18_template_ncc(first, second):
    comparison_mask = central_ellipsoid

    first_values = np.asarray(
        first,
        dtype=np.float64,
    )[comparison_mask]

    second_values = np.asarray(
        second,
        dtype=np.float64,
    )[comparison_mask]

    first_values -= first_values.mean()
    second_values -= second_values.mean()

    denominator = np.sqrt(
        np.dot(first_values, first_values)
        * np.dot(second_values, second_values)
    )

    if denominator <= 1e-12:
        return 0.0

    return float(
        np.dot(first_values, second_values)
        / denominator
    )


PHASE18_TEMPLATE_BANKS = []
PHASE18_PRIVATE_PHENOTYPE_ASSIGNMENTS = []
template_fold_reports = []

for fold, (
    fit_idx,
    monitor_idx,
    outer_valid_idx,
) in enumerate(nested_splits):
    fit_idx = np.asarray(fit_idx, dtype=np.int64)

    normal_idx = fit_idx[y[fit_idx] == 0]
    abnormal_idx = fit_idx[y[fit_idx] == 1]

    abnormal_absolute_laterality = np.abs(
        PHASE18_LATERALITY_SCORES[abnormal_idx]
    )

    # Fold-local threshold: half of abnormal fit cases form the
    # approximately bilateral phenotype; the other half supplies
    # mirrored unilateral evidence.
    bilateral_threshold = float(
        np.median(abnormal_absolute_laterality)
    )

    abnormal_scores = (
        PHASE18_LATERALITY_SCORES[abnormal_idx]
    )

    bilateral_idx = abnormal_idx[
        np.abs(abnormal_scores)
        <= bilateral_threshold
    ]

    left_reduction_idx = abnormal_idx[
        abnormal_scores > bilateral_threshold
    ]

    right_reduction_idx = abnormal_idx[
        abnormal_scores < -bilateral_threshold
    ]

    assert (
        len(bilateral_idx)
        + len(left_reduction_idx)
        + len(right_reduction_idx)
        == len(abnormal_idx)
    )

    unilateral_source_indices = np.concatenate(
        [left_reduction_idx, right_reduction_idx]
    )

    # Right-reduction cases are reflected into the canonical
    # left-reduction orientation. This pools both sides without
    # erasing the abnormal asymmetry.
    unilateral_reflect_flags = np.concatenate([
        np.zeros(len(left_reduction_idx), dtype=bool),
        np.ones(len(right_reduction_idx), dtype=bool),
    ])

    template_pool_viable = bool(
        len(normal_idx) >= 80
        and len(bilateral_idx) >= 80
        and len(unilateral_source_indices) >= 80
    )

    assert template_pool_viable, (
        "One or more phenotype template pools are too small."
    )

    normal_template = phase18_mean_template(
        normal_idx,
        enforce_symmetry=True,
    )

    bilateral_template = phase18_mean_template(
        bilateral_idx,
        enforce_symmetry=True,
    )

    left_reduction_template = phase18_mean_template(
        unilateral_source_indices,
        reflect_flags=unilateral_reflect_flags,
        enforce_symmetry=False,
    )

    # Enforce exact reflection equivariance between unilateral banks.
    right_reduction_template = np.ascontiguousarray(
        np.flip(
            left_reduction_template,
            axis=0,
        ),
        dtype=np.float32,
    )

    template_bank = {
        "normal": normal_template,
        "bilateral_abnormal": bilateral_template,
        "left_reduction_abnormal": (
            left_reduction_template
        ),
        "right_reduction_abnormal": (
            right_reduction_template
        ),
    }

    assert tuple(template_bank) == PHASE18_TEMPLATE_NAMES

    for template in template_bank.values():
        assert template.shape == registration_shape
        assert template.dtype == np.float32
        assert np.isfinite(template).all()

    normal_symmetry_error = float(
        np.max(
            np.abs(
                normal_template
                - np.flip(normal_template, axis=0)
            )
        )
    )

    bilateral_symmetry_error = float(
        np.max(
            np.abs(
                bilateral_template
                - np.flip(bilateral_template, axis=0)
            )
        )
    )

    unilateral_reflection_error = float(
        np.max(
            np.abs(
                left_reduction_template
                - np.flip(
                    right_reduction_template,
                    axis=0,
                )
            )
        )
    )

    assert normal_symmetry_error <= 1e-7
    assert bilateral_symmetry_error <= 1e-7
    assert unilateral_reflection_error <= 1e-7

    correlations = {
        "normal_vs_bilateral": phase18_template_ncc(
            normal_template,
            bilateral_template,
        ),
        "normal_vs_left_reduction": phase18_template_ncc(
            normal_template,
            left_reduction_template,
        ),
        "bilateral_vs_left_reduction": phase18_template_ncc(
            bilateral_template,
            left_reduction_template,
        ),
        "left_vs_right_reduction": phase18_template_ncc(
            left_reduction_template,
            right_reduction_template,
        ),
    }

    PHASE18_TEMPLATE_BANKS.append(template_bank)

    # Private case-level assignments remain only in memory.
    PHASE18_PRIVATE_PHENOTYPE_ASSIGNMENTS.append({
        "normal_idx": normal_idx,
        "bilateral_idx": bilateral_idx,
        "left_reduction_idx": left_reduction_idx,
        "right_reduction_idx": right_reduction_idx,
    })

    template_fold_reports.append({
        "fold": int(fold),
        "fit_count": int(len(fit_idx)),
        "normal_template_cases": int(len(normal_idx)),
        "abnormal_template_cases": int(len(abnormal_idx)),
        "bilateral_cases": int(len(bilateral_idx)),
        "left_reduction_cases": int(
            len(left_reduction_idx)
        ),
        "right_reduction_cases": int(
            len(right_reduction_idx)
        ),
        "pooled_unilateral_template_cases": int(
            len(unilateral_source_indices)
        ),
        "absolute_laterality_threshold": round(
            bilateral_threshold,
            6,
        ),
        "template_pool_viable": template_pool_viable,
        "template_correlations": {
            name: round(float(value), 6)
            for name, value in correlations.items()
        },
        "maximum_symmetry_or_reflection_error": round(
            max(
                normal_symmetry_error,
                bilateral_symmetry_error,
                unilateral_reflection_error,
            ),
            9,
        ),
    })

absolute_laterality = np.abs(
    PHASE18_LATERALITY_SCORES
)

laterality_quantiles = np.quantile(
    absolute_laterality,
    [0.05, 0.50, 0.90, 0.95],
)

sanitized_template_report = {
    "phase": PHASE,
    "template_names": list(PHASE18_TEMPLATE_NAMES),
    "fold_count": int(len(PHASE18_TEMPLATE_BANKS)),
    "laterality_definition": (
        "positive_means_relative_left_uptake_reduction"
    ),
    "absolute_laterality_quantiles_all_cases": {
        "q05": round(float(laterality_quantiles[0]), 6),
        "q50": round(float(laterality_quantiles[1]), 6),
        "q90": round(float(laterality_quantiles[2]), 6),
        "q95": round(float(laterality_quantiles[3]), 6),
    },
    "folds": template_fold_reports,
    "all_template_pools_viable": bool(
        all(
            row["template_pool_viable"]
            for row in template_fold_reports
        )
    ),
    "templates_fit_partition_only": True,
    "monitor_labels_used": False,
    "outer_validation_labels_used": False,
    "reflection_equivariance_enforced": True,
    "case_level_assignments_exported": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE18_TEMPLATE_BANKS")
print(json.dumps(sanitized_template_report, indent=2))
print("END SANITIZED_PHASE18_TEMPLATE_BANKS")

BEGIN SANITIZED_PHASE18_TEMPLATE_BANKS
{
  "phase": "18a_mt_bimorph_preflight",
  "template_names": [
    "normal",
    "bilateral_abnormal",
    "left_reduction_abnormal",
    "right_reduction_abnormal"
  ],
  "fold_count": 3,
  "laterality_definition": "positive_means_relative_left_uptake_reduction",
  "absolute_laterality_quantiles_all_cases": {
    "q05": 0.00148,
    "q50": 0.016247,
    "q90": 0.051199,
    "q95": 0.069348
  },
  "folds": [
    {
      "fold": 0,
      "fit_count": 721,
      "normal_template_cases": 340,
      "abnormal_template_cases": 381,
      "bilateral_cases": 191,
      "left_reduction_cases": 101,
      "right_reduction_cases": 89,
      "pooled_unilateral_template_cases": 190,
      "absolute_laterality_threshold": 0.020211,
      "template_pool_viable": true,
      "template_correlations": {
        "normal_vs_bilateral": 0.981712,
        "normal_vs_left_reduction": 0.897935,
        "bilateral_vs_left_reduction": 0.920397,
        "left_vs_right_redu

In [10]:
# Cell 8 — deterministic multitemplate rigid-registration engine

from concurrent.futures import ThreadPoolExecutor
from scipy.ndimage import shift as ndimage_shift

sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1)

PHASE18_REGISTRATION_WORKERS = max(
    1,
    min(4, int(os.cpu_count() or 1)),
)

PHASE18_MAX_TRANSLATION_MM = 20.0
PHASE18_MAX_ROTATION_DEG = 18.0
PHASE18_MIN_NCC_GAIN = 0.001
PHASE18_MAX_NCC_DEGRADATION = 0.002
PHASE18_MAX_REGISTRATION_FALLBACK_RATE = 0.01

PHASE18_STATUS_TO_CODE = {
    "accepted": 0,
    "identity_retained": 1,
    "invalid_transform_fallback": 2,
    "degraded_metric_fallback": 3,
    "exception_fallback": 4,
}
PHASE18_CODE_TO_STATUS = {
    value: key
    for key, value in PHASE18_STATUS_TO_CODE.items()
}

PHASE18_TEMPLATE_TO_ID = {
    name: position
    for position, name in enumerate(PHASE18_TEMPLATE_NAMES)
}
PHASE18_ID_TO_TEMPLATE = {
    value: key
    for key, value in PHASE18_TEMPLATE_TO_ID.items()
}

PHASE18_REGISTRATION_MASK = np.asarray(
    central_ellipsoid,
    dtype=bool,
)

assert PHASE18_REGISTRATION_MASK.shape == (32, 32, 32)
assert PHASE18_REGISTRATION_MASK.sum() >= 1000


def phase18_make_sitk_image(volume):
    # NumPy cache: x,y,z. SimpleITK array: z,y,x.
    array_zyx = np.ascontiguousarray(
        np.asarray(volume, dtype=np.float32).transpose(
            2,
            1,
            0,
        )
    )

    image = sitk.GetImageFromArray(array_zyx)

    image.SetSpacing(
        (registration_spacing_mm,) * 3
    )

    extent = (
        np.asarray(volume.shape, dtype=np.float64) - 1.0
    ) * registration_spacing_mm

    image.SetOrigin(
        tuple((-0.5 * extent).tolist())
    )

    image.SetDirection((
        1.0, 0.0, 0.0,
        0.0, 1.0, 0.0,
        0.0, 0.0, 1.0,
    ))

    return image


def phase18_masked_ncc(fixed, moving, mask):
    fixed_values = np.asarray(
        fixed,
        dtype=np.float64,
    )[mask]

    moving_values = np.asarray(
        moving,
        dtype=np.float64,
    )[mask]

    fixed_values -= fixed_values.mean()
    moving_values -= moving_values.mean()

    denominator = float(
        np.sqrt(
            np.dot(fixed_values, fixed_values)
            * np.dot(moving_values, moving_values)
        )
    )

    if denominator <= 1e-12:
        return 0.0

    return float(
        np.dot(fixed_values, moving_values)
        / denominator
    )


def phase18_select_template(volume, template_bank):
    scores = np.asarray(
        [
            phase18_masked_ncc(
                template_bank[name],
                volume,
                PHASE18_REGISTRATION_MASK,
            )
            for name in PHASE18_TEMPLATE_NAMES
        ],
        dtype=np.float64,
    )

    if not np.isfinite(scores).all():
        raise ValueError("Non-finite template-selection NCC")

    sorted_positions = np.argsort(scores)
    selected_position = int(sorted_positions[-1])
    second_position = int(sorted_positions[-2])

    selection_margin = float(
        scores[selected_position]
        - scores[second_position]
    )

    return (
        selected_position,
        float(scores[selected_position]),
        selection_margin,
    )


def phase18_identity_transform_arrays():
    return (
        np.zeros(6, dtype=np.float32),
        np.zeros(4, dtype=np.float32),
    )


def phase18_extract_euler_transform(final_transform):
    leaf_transform = final_transform

    while leaf_transform.GetName() == "CompositeTransform":
        leaf_transform = leaf_transform.GetBackTransform()

    euler_transform = sitk.Euler3DTransform(
        leaf_transform
    )

    parameters = np.asarray(
        euler_transform.GetParameters(),
        dtype=np.float64,
    )

    fixed_parameters = np.asarray(
        euler_transform.GetFixedParameters(),
        dtype=np.float64,
    )

    if parameters.shape != (6,):
        raise ValueError("Unexpected Euler parameter shape")

    if fixed_parameters.shape != (4,):
        raise ValueError(
            "Unexpected Euler fixed-parameter shape"
        )

    return (
        euler_transform,
        parameters,
        fixed_parameters,
    )


def phase18_rigid_register(
    moving_volume,
    fixed_template,
    random_seed,
):
    moving_volume = np.asarray(
        moving_volume,
        dtype=np.float32,
    )
    fixed_template = np.asarray(
        fixed_template,
        dtype=np.float32,
    )

    before_ncc = phase18_masked_ncc(
        fixed_template,
        moving_volume,
        PHASE18_REGISTRATION_MASK,
    )

    fixed_image = phase18_make_sitk_image(
        fixed_template
    )
    moving_image = phase18_make_sitk_image(
        moving_volume
    )
    mask_image = (
        phase18_make_sitk_image(
            PHASE18_REGISTRATION_MASK.astype(
                np.float32
            )
        )
        > 0.5
    )

    initial_transform = (
        sitk.CenteredTransformInitializer(
            fixed_image,
            moving_image,
            sitk.Euler3DTransform(),
            sitk.CenteredTransformInitializerFilter.GEOMETRY,
        )
    )

    registration = sitk.ImageRegistrationMethod()
    registration.SetMetricAsCorrelation()
    registration.SetMetricFixedMask(mask_image)

    registration.SetMetricSamplingStrategy(
        registration.REGULAR
    )
    registration.SetMetricSamplingPercentage(
        0.50,
        int(random_seed % 2_147_483_647),
    )

    registration.SetInterpolator(sitk.sitkLinear)

    registration.SetOptimizerAsRegularStepGradientDescent(
        learningRate=1.0,
        minStep=0.02,
        numberOfIterations=80,
        relaxationFactor=0.50,
        gradientMagnitudeTolerance=1e-6,
    )

    registration.SetOptimizerScalesFromPhysicalShift()

    registration.SetShrinkFactorsPerLevel((2, 1))
    registration.SetSmoothingSigmasPerLevel(
        (1.0, 0.0)
    )
    registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    registration.SetInitialTransform(
        initial_transform,
        inPlace=False,
    )

    final_transform = registration.Execute(
        fixed_image,
        moving_image,
    )

    (
        euler_transform,
        parameters,
        fixed_parameters,
    ) = phase18_extract_euler_transform(
        final_transform
    )

    parameters_finite = bool(
        np.isfinite(parameters).all()
        and np.isfinite(fixed_parameters).all()
    )

    if parameters_finite:
        angles_deg = np.abs(
            np.rad2deg(parameters[:3])
        )
        translations_mm = np.abs(
            parameters[3:6]
        )

        max_rotation_deg = float(
            angles_deg.max()
        )
        max_translation_mm = float(
            translations_mm.max()
        )
    else:
        max_rotation_deg = 0.0
        max_translation_mm = 0.0

    invalid_transform = bool(
        not parameters_finite
        or max_rotation_deg
        > PHASE18_MAX_ROTATION_DEG
        or max_translation_mm
        > PHASE18_MAX_TRANSLATION_MM
    )

    if invalid_transform:
        identity_parameters, identity_fixed = (
            phase18_identity_transform_arrays()
        )

        return {
            "status": "invalid_transform_fallback",
            "parameters": identity_parameters,
            "fixed_parameters": identity_fixed,
            "ncc_before": float(before_ncc),
            "ncc_after": float(before_ncc),
            "ncc_gain": 0.0,
            "max_rotation_deg": max_rotation_deg,
            "max_translation_mm": max_translation_mm,
        }

    resampled_image = sitk.Resample(
        moving_image,
        fixed_image,
        final_transform,
        sitk.sitkLinear,
        0.0,
        sitk.sitkFloat32,
    )

    registered_volume = (
        sitk.GetArrayFromImage(resampled_image)
        .transpose(2, 1, 0)
        .astype(np.float32)
    )

    registered_volume = np.clip(
        registered_volume,
        0.0,
        1.5,
    )

    after_ncc = phase18_masked_ncc(
        fixed_template,
        registered_volume,
        PHASE18_REGISTRATION_MASK,
    )

    if (
        not np.isfinite(after_ncc)
        or after_ncc
        < before_ncc - PHASE18_MAX_NCC_DEGRADATION
    ):
        identity_parameters, identity_fixed = (
            phase18_identity_transform_arrays()
        )

        return {
            "status": "degraded_metric_fallback",
            "parameters": identity_parameters,
            "fixed_parameters": identity_fixed,
            "ncc_before": float(before_ncc),
            "ncc_after": float(before_ncc),
            "ncc_gain": 0.0,
            "max_rotation_deg": max_rotation_deg,
            "max_translation_mm": max_translation_mm,
        }

    ncc_gain = float(after_ncc - before_ncc)

    if ncc_gain < PHASE18_MIN_NCC_GAIN:
        identity_parameters, identity_fixed = (
            phase18_identity_transform_arrays()
        )

        return {
            "status": "identity_retained",
            "parameters": identity_parameters,
            "fixed_parameters": identity_fixed,
            "ncc_before": float(before_ncc),
            "ncc_after": float(before_ncc),
            "ncc_gain": 0.0,
            "max_rotation_deg": max_rotation_deg,
            "max_translation_mm": max_translation_mm,
        }

    return {
        "status": "accepted",
        "parameters": parameters.astype(np.float32),
        "fixed_parameters": fixed_parameters.astype(
            np.float32
        ),
        "ncc_before": float(before_ncc),
        "ncc_after": float(after_ncc),
        "ncc_gain": float(ncc_gain),
        "max_rotation_deg": max_rotation_deg,
        "max_translation_mm": max_translation_mm,
    }


# Synthetic/API contract using a shifted aggregate template.
synthetic_fixed = PHASE18_TEMPLATE_BANKS[0]["normal"]

synthetic_moving = ndimage_shift(
    synthetic_fixed,
    shift=(1.25, -0.75, 0.50),
    order=1,
    mode="constant",
    cval=0.0,
    prefilter=False,
).astype(np.float32)

synthetic_result = phase18_rigid_register(
    synthetic_moving,
    synthetic_fixed,
    PHASE18_SEED + 800_000,
)

assert synthetic_result["status"] in (
    "accepted",
    "identity_retained",
)
assert np.isfinite(
    synthetic_result["parameters"]
).all()

sanitized_registration_contract = {
    "phase": PHASE,
    "simpleitk_version": (
        sitk.Version_VersionString()
    ),
    "worker_count": int(
        PHASE18_REGISTRATION_WORKERS
    ),
    "registration_grid": [32, 32, 32],
    "spacing_mm": float(
        registration_spacing_mm
    ),
    "degrees_of_freedom": 6,
    "optimizer_iterations": 80,
    "synthetic_contract_status": (
        synthetic_result["status"]
    ),
    "synthetic_ncc_gain": round(
        float(synthetic_result["ncc_gain"]),
        6,
    ),
    "maximum_translation_mm": (
        PHASE18_MAX_TRANSLATION_MM
    ),
    "maximum_rotation_deg": (
        PHASE18_MAX_ROTATION_DEG
    ),
    "minimum_accepted_ncc_gain": (
        PHASE18_MIN_NCC_GAIN
    ),
    "maximum_fallback_rate": (
        PHASE18_MAX_REGISTRATION_FALLBACK_RATE
    ),
    "private_case_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN "
    "SANITIZED_PHASE18_REGISTRATION_CONTRACT"
)
print(json.dumps(
    sanitized_registration_contract,
    indent=2,
))
print(
    "END "
    "SANITIZED_PHASE18_REGISTRATION_CONTRACT"
)

del synthetic_fixed, synthetic_moving, synthetic_result

BEGIN SANITIZED_PHASE18_REGISTRATION_CONTRACT
{
  "phase": "18a_mt_bimorph_preflight",
  "simpleitk_version": "2.5.5",
  "worker_count": 4,
  "registration_grid": [
    32,
    32,
    32
  ],
  "spacing_mm": 4.0,
  "degrees_of_freedom": 6,
  "optimizer_iterations": 80,
  "synthetic_contract_status": "accepted",
  "synthetic_ncc_gain": 0.123087,
  "maximum_translation_mm": 20.0,
  "maximum_rotation_deg": 18.0,
  "minimum_accepted_ncc_gain": 0.001,
  "maximum_fallback_rate": 0.01,
  "private_case_read": false,
  "uids_displayed": false,
  "patient_rows_displayed": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE18_REGISTRATION_CONTRACT


In [11]:
# Cell 9 — full three-fold multitemplate registration preflight

number_of_folds = len(PHASE18_TEMPLATE_BANKS)
number_of_cases = len(y)

assert number_of_folds == 3

PHASE18_SELECTED_TEMPLATE_IDS = np.full(
    (number_of_folds, number_of_cases),
    -1,
    dtype=np.int8,
)

PHASE18_TEMPLATE_SELECTION_MARGINS = np.full(
    (number_of_folds, number_of_cases),
    np.nan,
    dtype=np.float32,
)

PHASE18_REGISTRATION_STATUS_CODES = np.full(
    (number_of_folds, number_of_cases),
    -1,
    dtype=np.int8,
)

PHASE18_REGISTRATION_PARAMETERS = np.zeros(
    (number_of_folds, number_of_cases, 6),
    dtype=np.float32,
)

PHASE18_REGISTRATION_FIXED_PARAMETERS = np.zeros(
    (number_of_folds, number_of_cases, 4),
    dtype=np.float32,
)

PHASE18_REGISTRATION_NCC_BEFORE = np.full(
    (number_of_folds, number_of_cases),
    np.nan,
    dtype=np.float32,
)

PHASE18_REGISTRATION_NCC_AFTER = np.full(
    (number_of_folds, number_of_cases),
    np.nan,
    dtype=np.float32,
)

PHASE18_REGISTRATION_NCC_GAIN = np.full(
    (number_of_folds, number_of_cases),
    np.nan,
    dtype=np.float32,
)

PHASE18_REGISTRATION_MAX_ROTATION = np.full(
    (number_of_folds, number_of_cases),
    np.nan,
    dtype=np.float32,
)

PHASE18_REGISTRATION_MAX_TRANSLATION = np.full(
    (number_of_folds, number_of_cases),
    np.nan,
    dtype=np.float32,
)


def phase18_process_registration_case(
    fold,
    private_position,
):
    volume = np.asarray(
        PHASE18_REGISTRATION_VOLUMES[
            private_position
        ],
        dtype=np.float32,
    )

    template_bank = PHASE18_TEMPLATE_BANKS[fold]

    (
        selected_template_id,
        selected_template_ncc,
        selection_margin,
    ) = phase18_select_template(
        volume,
        template_bank,
    )

    selected_template_name = (
        PHASE18_ID_TO_TEMPLATE[
            selected_template_id
        ]
    )

    try:
        result = phase18_rigid_register(
            volume,
            template_bank[selected_template_name],
            (
                PHASE18_SEED
                + fold * 100_000
                + private_position
            ),
        )

    except Exception:
        identity_parameters, identity_fixed = (
            phase18_identity_transform_arrays()
        )

        result = {
            "status": "exception_fallback",
            "parameters": identity_parameters,
            "fixed_parameters": identity_fixed,
            "ncc_before": float(
                selected_template_ncc
            ),
            "ncc_after": float(
                selected_template_ncc
            ),
            "ncc_gain": 0.0,
            "max_rotation_deg": 0.0,
            "max_translation_mm": 0.0,
        }

    return (
        int(private_position),
        int(selected_template_id),
        float(selection_margin),
        result,
    )


PHASE18_REGISTRATION_STARTED = time.perf_counter()

for fold in range(number_of_folds):
    fold_started = time.perf_counter()

    def process_position(private_position):
        return phase18_process_registration_case(
            fold,
            private_position,
        )

    with ThreadPoolExecutor(
        max_workers=PHASE18_REGISTRATION_WORKERS
    ) as executor:
        result_iterator = executor.map(
            process_position,
            range(number_of_cases),
        )

        for completed, row in enumerate(
            result_iterator,
            start=1,
        ):
            (
                private_position,
                selected_template_id,
                selection_margin,
                result,
            ) = row

            status_code = PHASE18_STATUS_TO_CODE[
                result["status"]
            ]

            PHASE18_SELECTED_TEMPLATE_IDS[
                fold,
                private_position,
            ] = selected_template_id

            PHASE18_TEMPLATE_SELECTION_MARGINS[
                fold,
                private_position,
            ] = selection_margin

            PHASE18_REGISTRATION_STATUS_CODES[
                fold,
                private_position,
            ] = status_code

            PHASE18_REGISTRATION_PARAMETERS[
                fold,
                private_position,
            ] = result["parameters"]

            PHASE18_REGISTRATION_FIXED_PARAMETERS[
                fold,
                private_position,
            ] = result["fixed_parameters"]

            PHASE18_REGISTRATION_NCC_BEFORE[
                fold,
                private_position,
            ] = result["ncc_before"]

            PHASE18_REGISTRATION_NCC_AFTER[
                fold,
                private_position,
            ] = result["ncc_after"]

            PHASE18_REGISTRATION_NCC_GAIN[
                fold,
                private_position,
            ] = result["ncc_gain"]

            PHASE18_REGISTRATION_MAX_ROTATION[
                fold,
                private_position,
            ] = result["max_rotation_deg"]

            PHASE18_REGISTRATION_MAX_TRANSLATION[
                fold,
                private_position,
            ] = result["max_translation_mm"]

            if completed % 250 == 0:
                print(
                    f"Registration fold {fold}: "
                    f"{completed}/{number_of_cases}"
                )

    fold_elapsed = float(
        time.perf_counter() - fold_started
    )

    fold_statuses = (
        PHASE18_REGISTRATION_STATUS_CODES[fold]
    )

    fold_failures = int(
        np.isin(
            fold_statuses,
            [
                PHASE18_STATUS_TO_CODE[
                    "invalid_transform_fallback"
                ],
                PHASE18_STATUS_TO_CODE[
                    "degraded_metric_fallback"
                ],
                PHASE18_STATUS_TO_CODE[
                    "exception_fallback"
                ],
            ],
        ).sum()
    )

    print(
        f"Registration fold {fold} completed: "
        f"elapsed_seconds={fold_elapsed:.2f}, "
        f"fallbacks={fold_failures}"
    )

PHASE18_REGISTRATION_SECONDS = float(
    time.perf_counter()
    - PHASE18_REGISTRATION_STARTED
)

assert (
    PHASE18_SELECTED_TEMPLATE_IDS >= 0
).all()

assert (
    PHASE18_REGISTRATION_STATUS_CODES >= 0
).all()

for private_array in (
    PHASE18_TEMPLATE_SELECTION_MARGINS,
    PHASE18_REGISTRATION_PARAMETERS,
    PHASE18_REGISTRATION_FIXED_PARAMETERS,
    PHASE18_REGISTRATION_NCC_BEFORE,
    PHASE18_REGISTRATION_NCC_AFTER,
    PHASE18_REGISTRATION_NCC_GAIN,
    PHASE18_REGISTRATION_MAX_ROTATION,
    PHASE18_REGISTRATION_MAX_TRANSLATION,
):
    assert np.isfinite(private_array).all()

print(
    "Registration preflight computation complete. "
)

Registration fold 0: 250/1362
Registration fold 0: 500/1362
Registration fold 0: 750/1362
Registration fold 0: 1000/1362
Registration fold 0: 1250/1362
Registration fold 0 completed: elapsed_seconds=28.69, fallbacks=280
Registration fold 1: 250/1362
Registration fold 1: 500/1362
Registration fold 1: 750/1362
Registration fold 1: 1000/1362
Registration fold 1: 1250/1362
Registration fold 1 completed: elapsed_seconds=27.81, fallbacks=262
Registration fold 2: 250/1362
Registration fold 2: 500/1362
Registration fold 2: 750/1362
Registration fold 2: 1000/1362
Registration fold 2: 1250/1362
Registration fold 2 completed: elapsed_seconds=29.31, fallbacks=262
Registration preflight computation complete. Run Cell 10 for the sanitized audit.


In [12]:
# Cell 10 — aggregate registration reliability and diversity audit

failure_status_codes = np.asarray([
    PHASE18_STATUS_TO_CODE[
        "invalid_transform_fallback"
    ],
    PHASE18_STATUS_TO_CODE[
        "degraded_metric_fallback"
    ],
    PHASE18_STATUS_TO_CODE[
        "exception_fallback"
    ],
], dtype=np.int8)

registration_fold_reports = []
total_fallback_count = 0
total_operation_count = int(
    number_of_folds * number_of_cases
)

for fold in range(number_of_folds):
    fold_status_codes = (
        PHASE18_REGISTRATION_STATUS_CODES[fold]
    )

    status_counts = {
        status_name: int(
            (
                fold_status_codes
                == status_code
            ).sum()
        )
        for status_name, status_code
        in PHASE18_STATUS_TO_CODE.items()
    }

    fallback_count = int(
        np.isin(
            fold_status_codes,
            failure_status_codes,
        ).sum()
    )
    total_fallback_count += fallback_count

    selected_ids = (
        PHASE18_SELECTED_TEMPLATE_IDS[fold]
    )

    all_case_selection_counts = {
        template_name: int(
            (
                selected_ids
                == PHASE18_TEMPLATE_TO_ID[
                    template_name
                ]
            ).sum()
        )
        for template_name in PHASE18_TEMPLATE_NAMES
    }

    outer_valid_idx = nested_splits[fold][2]

    outer_selection_counts = {
        template_name: int(
            (
                selected_ids[outer_valid_idx]
                == PHASE18_TEMPLATE_TO_ID[
                    template_name
                ]
            ).sum()
        )
        for template_name in PHASE18_TEMPLATE_NAMES
    }

    outer_selection_by_class = {}

    for class_value, class_name in (
        (0, "normal_cases"),
        (1, "pathologic_cases"),
    ):
        class_indices = outer_valid_idx[
            y[outer_valid_idx] == class_value
        ]

        outer_selection_by_class[class_name] = {
            template_name: int(
                (
                    selected_ids[class_indices]
                    == PHASE18_TEMPLATE_TO_ID[
                        template_name
                    ]
                ).sum()
            )
            for template_name
            in PHASE18_TEMPLATE_NAMES
        }

    accepted_mask = (
        fold_status_codes
        == PHASE18_STATUS_TO_CODE["accepted"]
    )

    if accepted_mask.any():
        accepted_gains = (
            PHASE18_REGISTRATION_NCC_GAIN[
                fold,
                accepted_mask,
            ]
        )

        accepted_rotations = (
            PHASE18_REGISTRATION_MAX_ROTATION[
                fold,
                accepted_mask,
            ]
        )

        accepted_translations = (
            PHASE18_REGISTRATION_MAX_TRANSLATION[
                fold,
                accepted_mask,
            ]
        )

        accepted_diagnostics = {
            "median_ncc_gain": round(
                float(np.median(accepted_gains)),
                6,
            ),
            "p95_ncc_gain": round(
                float(
                    np.quantile(
                        accepted_gains,
                        0.95,
                    )
                ),
                6,
            ),
            "p95_rotation_deg": round(
                float(
                    np.quantile(
                        accepted_rotations,
                        0.95,
                    )
                ),
                6,
            ),
            "p95_translation_mm": round(
                float(
                    np.quantile(
                        accepted_translations,
                        0.95,
                    )
                ),
                6,
            ),
        }
    else:
        accepted_diagnostics = {
            "median_ncc_gain": None,
            "p95_ncc_gain": None,
            "p95_rotation_deg": None,
            "p95_translation_mm": None,
        }

    registration_fold_reports.append({
        "fold": int(fold),
        "operation_count": int(number_of_cases),
        "status_counts": status_counts,
        "fallback_count": int(fallback_count),
        "fallback_rate": round(
            fallback_count / number_of_cases,
            6,
        ),
        "all_case_template_selection": (
            all_case_selection_counts
        ),
        "outer_validation_template_selection": (
            outer_selection_counts
        ),
        "outer_validation_selection_by_class": (
            outer_selection_by_class
        ),
        "selection_margin": {
            "median": round(
                float(
                    np.median(
                        PHASE18_TEMPLATE_SELECTION_MARGINS[
                            fold
                        ]
                    )
                ),
                6,
            ),
            "q05": round(
                float(
                    np.quantile(
                        PHASE18_TEMPLATE_SELECTION_MARGINS[
                            fold
                        ],
                        0.05,
                    )
                ),
                6,
            ),
        },
        "accepted_transform_diagnostics": (
            accepted_diagnostics
        ),
    })

overall_fallback_rate = float(
    total_fallback_count
    / total_operation_count
)

registration_preflight_passed = bool(
    overall_fallback_rate
    <= PHASE18_MAX_REGISTRATION_FALLBACK_RATE
)

sanitized_registration_report = {
    "phase": PHASE,
    "registration_method": (
        "fold_local_four_template_Euler3D_rigid"
    ),
    "operation_count": total_operation_count,
    "case_count": int(number_of_cases),
    "fold_count": int(number_of_folds),
    "elapsed_seconds": round(
        PHASE18_REGISTRATION_SECONDS,
        2,
    ),
    "seconds_per_fold_case_operation": round(
        PHASE18_REGISTRATION_SECONDS
        / total_operation_count,
        6,
    ),
    "worker_count": int(
        PHASE18_REGISTRATION_WORKERS
    ),
    "fallback_count": int(
        total_fallback_count
    ),
    "fallback_rate": round(
        overall_fallback_rate,
        6,
    ),
    "maximum_allowed_fallback_rate": (
        PHASE18_MAX_REGISTRATION_FALLBACK_RATE
    ),
    "registration_preflight_passed": (
        registration_preflight_passed
    ),
    "folds": registration_fold_reports,
    "transform_parameters_retained_in_private_memory": True,
    "case_level_transform_exported": False,
    "template_selection_uses_labels": False,
    "outer_labels_used_only_for_aggregate_diagnostic": True,
    "templates_fit_partition_only": True,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE18_REGISTRATION_PREFLIGHT")
print(json.dumps(
    sanitized_registration_report,
    indent=2,
))
print("END SANITIZED_PHASE18_REGISTRATION_PREFLIGHT")

BEGIN SANITIZED_PHASE18_REGISTRATION_PREFLIGHT
{
  "phase": "18a_mt_bimorph_preflight",
  "registration_method": "fold_local_four_template_Euler3D_rigid",
  "operation_count": 4086,
  "case_count": 1362,
  "fold_count": 3,
  "elapsed_seconds": 85.82,
  "seconds_per_fold_case_operation": 0.021002,
  "worker_count": 4,
  "fallback_count": 804,
  "fallback_rate": 0.196769,
  "maximum_allowed_fallback_rate": 0.01,
  "registration_preflight_passed": false,
  "folds": [
    {
      "fold": 0,
      "operation_count": 1362,
      "status_counts": {
        "accepted": 1082,
        "identity_retained": 0,
        "invalid_transform_fallback": 275,
        "degraded_metric_fallback": 5,
        "exception_fallback": 0
      },
      "fallback_count": 280,
      "fallback_rate": 0.20558,
      "all_case_template_selection": {
        "normal": 579,
        "bilateral_abnormal": 178,
        "left_reduction_abnormal": 312,
        "right_reduction_abnormal": 293
      },
      "outer_validation_

In [13]:
# Cell 11 — registration-free reflection-invariant feature construction

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

PHASE18_REGISTRATION_BRANCH_ADVANCED = False
PHASE18_REPRESENTATION_BRANCH = (
    "registration_free_continuous_prototype_morphology"
)

# Four quadrant masks capture anterior/posterior and bilateral uptake.
left_anterior_mask = (
    left_proxy_mask & (coordinate_y >= 0)
)
left_posterior_mask = (
    left_proxy_mask & (coordinate_y < 0)
)
right_anterior_mask = (
    right_proxy_mask & (coordinate_y >= 0)
)
right_posterior_mask = (
    right_proxy_mask & (coordinate_y < 0)
)

for region_mask in (
    left_anterior_mask,
    left_posterior_mask,
    right_anterior_mask,
    right_posterior_mask,
):
    assert region_mask.sum() >= 100


def phase18_safe_divide(numerator, denominator):
    return float(
        numerator / max(abs(float(denominator)), 1e-6)
    )


def phase18_morphology_features(volume, return_names=False):
    volume = gaussian_filter(
        np.asarray(volume, dtype=np.float32),
        sigma=0.55,
        mode="constant",
    )

    features = []
    names = []

    def add(name, value):
        value = float(value)

        if not np.isfinite(value):
            raise ValueError(
                f"Non-finite morphology feature: {name}"
            )

        names.append(str(name))
        features.append(value)

    central_values = volume[
        PHASE18_REGISTRATION_MASK
    ]

    for quantile in (
        0.10,
        0.25,
        0.50,
        0.70,
        0.85,
        0.95,
        0.99,
    ):
        add(
            f"central_q{int(quantile * 100):02d}",
            np.quantile(central_values, quantile),
        )

    add("central_mean", central_values.mean())
    add("central_std", central_values.std())

    for threshold in (
        0.20,
        0.40,
        0.60,
        0.80,
        1.00,
    ):
        add(
            f"central_fraction_above_{threshold:.2f}",
            np.mean(central_values >= threshold),
        )

    left_score = phase18_regional_uptake_score(
        volume[left_proxy_mask]
    )
    right_score = phase18_regional_uptake_score(
        volume[right_proxy_mask]
    )

    left_anterior = phase18_regional_uptake_score(
        volume[left_anterior_mask]
    )
    left_posterior = phase18_regional_uptake_score(
        volume[left_posterior_mask]
    )
    right_anterior = phase18_regional_uptake_score(
        volume[right_anterior_mask]
    )
    right_posterior = phase18_regional_uptake_score(
        volume[right_posterior_mask]
    )

    bilateral_score = 0.5 * (
        left_score + right_score
    )
    bilateral_asymmetry = abs(
        left_score - right_score
    )

    anterior_mean = 0.5 * (
        left_anterior + right_anterior
    )
    posterior_mean = 0.5 * (
        left_posterior + right_posterior
    )

    anterior_asymmetry = abs(
        left_anterior - right_anterior
    )
    posterior_asymmetry = abs(
        left_posterior - right_posterior
    )

    add("bilateral_uptake", bilateral_score)
    add("bilateral_asymmetry", bilateral_asymmetry)
    add("anterior_mean", anterior_mean)
    add("posterior_mean", posterior_mean)
    add("anterior_asymmetry", anterior_asymmetry)
    add("posterior_asymmetry", posterior_asymmetry)

    add(
        "posterior_to_anterior_ratio",
        phase18_safe_divide(
            posterior_mean,
            anterior_mean,
        ),
    )

    add(
        "posterior_minus_anterior",
        posterior_mean - anterior_mean,
    )

    add(
        "minimum_hemisphere_uptake",
        min(left_score, right_score),
    )

    add(
        "minimum_posterior_uptake",
        min(left_posterior, right_posterior),
    )

    # Reflection-invariant intensity-weighted spatial moments.
    moment_weights = (
        np.clip(volume, 0.0, None)
        * PHASE18_REGISTRATION_MASK
    ).astype(np.float64)

    total_weight = float(moment_weights.sum())
    if total_weight <= 1e-6:
        raise ValueError("Invalid morphology moment mass")

    normalized_weights = (
        moment_weights / total_weight
    )

    center_x = float(
        np.sum(normalized_weights * coordinate_x)
    )
    center_y = float(
        np.sum(normalized_weights * coordinate_y)
    )
    center_z = float(
        np.sum(normalized_weights * coordinate_z)
    )

    variance_x = float(
        np.sum(
            normalized_weights
            * (coordinate_x - center_x) ** 2
        )
    )
    variance_y = float(
        np.sum(
            normalized_weights
            * (coordinate_y - center_y) ** 2
        )
    )
    variance_z = float(
        np.sum(
            normalized_weights
            * (coordinate_z - center_z) ** 2
        )
    )

    add("absolute_centroid_x", abs(center_x))
    add("centroid_y", center_y)
    add("centroid_z", center_z)
    add("variance_x", variance_x)
    add("variance_y", variance_y)
    add("variance_z", variance_z)

    output = np.asarray(features, dtype=np.float32)

    if not np.isfinite(output).all():
        raise ValueError(
            "Non-finite morphology feature vector"
        )

    if return_names:
        return output, tuple(names)

    return output


def phase18_prototype_features(
    volume,
    template_bank,
    return_names=False,
):
    template_scores = np.asarray(
        [
            phase18_masked_ncc(
                template_bank[name],
                volume,
                PHASE18_REGISTRATION_MASK,
            )
            for name in PHASE18_TEMPLATE_NAMES
        ],
        dtype=np.float64,
    )

    if not np.isfinite(template_scores).all():
        raise ValueError(
            "Non-finite prototype similarity"
        )

    normal_ncc = float(template_scores[0])
    bilateral_ncc = float(template_scores[1])
    left_ncc = float(template_scores[2])
    right_ncc = float(template_scores[3])

    unilateral_max = max(left_ncc, right_ncc)
    unilateral_mean = 0.5 * (
        left_ncc + right_ncc
    )
    unilateral_difference = abs(
        left_ncc - right_ncc
    )

    abnormal_max = max(
        bilateral_ncc,
        unilateral_max,
    )

    sorted_scores = np.sort(template_scores)
    selection_margin = float(
        sorted_scores[-1] - sorted_scores[-2]
    )

    stabilized = (
        template_scores - template_scores.max()
    ) / 0.05

    probabilities = np.exp(stabilized)
    probabilities /= probabilities.sum()

    entropy = float(
        -np.sum(
            probabilities
            * np.log(
                np.clip(probabilities, 1e-12, None)
            )
        )
    )

    values = np.asarray([
        normal_ncc,
        bilateral_ncc,
        unilateral_max,
        unilateral_mean,
        unilateral_difference,
        abnormal_max,
        abnormal_max - normal_ncc,
        bilateral_ncc - normal_ncc,
        unilateral_max - normal_ncc,
        selection_margin,
        entropy,
        float(np.argmax(template_scores) != 0),
    ], dtype=np.float32)

    names = (
        "normal_ncc",
        "bilateral_ncc",
        "unilateral_max_ncc",
        "unilateral_mean_ncc",
        "unilateral_lr_difference",
        "maximum_abnormal_ncc",
        "abnormal_minus_normal_ncc",
        "bilateral_minus_normal_ncc",
        "unilateral_minus_normal_ncc",
        "top_template_margin",
        "template_softmax_entropy",
        "hard_abnormal_template_selected",
    )

    if return_names:
        return values, names

    return values


feature_started = time.perf_counter()

probe_morphology, PHASE18_MORPHOLOGY_FEATURE_NAMES = (
    phase18_morphology_features(
        PHASE18_REGISTRATION_VOLUMES[0],
        return_names=True,
    )
)

probe_prototype, PHASE18_PROTOTYPE_FEATURE_NAMES = (
    phase18_prototype_features(
        PHASE18_REGISTRATION_VOLUMES[0],
        PHASE18_TEMPLATE_BANKS[0],
        return_names=True,
    )
)

PHASE18_MORPHOLOGY_FEATURE_COUNT = int(
    len(PHASE18_MORPHOLOGY_FEATURE_NAMES)
)
PHASE18_PROTOTYPE_FEATURE_COUNT = int(
    len(PHASE18_PROTOTYPE_FEATURE_NAMES)
)
PHASE18_TOTAL_PROXY_FEATURE_COUNT = (
    PHASE18_MORPHOLOGY_FEATURE_COUNT
    + PHASE18_PROTOTYPE_FEATURE_COUNT
)

PHASE18_MORPHOLOGY_FEATURES = np.stack([
    phase18_morphology_features(volume)
    for volume in PHASE18_REGISTRATION_VOLUMES
]).astype(np.float32)

PHASE18_PROXY_FEATURES = np.empty(
    (
        len(PHASE18_TEMPLATE_BANKS),
        len(y),
        PHASE18_TOTAL_PROXY_FEATURE_COUNT,
    ),
    dtype=np.float32,
)

for fold, template_bank in enumerate(
    PHASE18_TEMPLATE_BANKS
):
    prototype_matrix = np.stack([
        phase18_prototype_features(
            volume,
            template_bank,
        )
        for volume in PHASE18_REGISTRATION_VOLUMES
    ]).astype(np.float32)

    PHASE18_PROXY_FEATURES[fold] = (
        np.concatenate(
            [
                PHASE18_MORPHOLOGY_FEATURES,
                prototype_matrix,
            ],
            axis=1,
        )
    )

assert np.isfinite(
    PHASE18_MORPHOLOGY_FEATURES
).all()
assert np.isfinite(
    PHASE18_PROXY_FEATURES
).all()

PHASE18_FEATURE_SECONDS = float(
    time.perf_counter() - feature_started
)

sanitized_feature_report = {
    "phase": PHASE,
    "representation": (
        PHASE18_REPRESENTATION_BRANCH
    ),
    "case_count": int(len(y)),
    "fold_count": int(
        len(PHASE18_TEMPLATE_BANKS)
    ),
    "morphology_feature_count": int(
        PHASE18_MORPHOLOGY_FEATURE_COUNT
    ),
    "prototype_feature_count": int(
        PHASE18_PROTOTYPE_FEATURE_COUNT
    ),
    "total_feature_count": int(
        PHASE18_TOTAL_PROXY_FEATURE_COUNT
    ),
    "elapsed_seconds": round(
        PHASE18_FEATURE_SECONDS,
        2,
    ),
    "reflection_invariant": True,
    "spatial_registration_applied": False,
    "rigid_registration_branch_advanced": False,
    "template_features_fold_local": True,
    "case_level_features_exported": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN "
    "SANITIZED_PHASE18_REGISTRATION_FREE_FEATURES"
)
print(json.dumps(
    sanitized_feature_report,
    indent=2,
))
print(
    "END "
    "SANITIZED_PHASE18_REGISTRATION_FREE_FEATURES"
)

BEGIN SANITIZED_PHASE18_REGISTRATION_FREE_FEATURES
{
  "phase": "18a_mt_bimorph_preflight",
  "representation": "registration_free_continuous_prototype_morphology",
  "case_count": 1362,
  "fold_count": 3,
  "morphology_feature_count": 30,
  "prototype_feature_count": 12,
  "total_feature_count": 42,
  "elapsed_seconds": 7.8,
  "reflection_invariant": true,
  "spatial_registration_applied": false,
  "rigid_registration_branch_advanced": false,
  "template_features_fold_local": true,
  "case_level_features_exported": false,
  "uids_displayed": false,
  "patient_rows_displayed": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE18_REGISTRATION_FREE_FEATURES


In [14]:
# Cell 12 — cheap OOF test of prototype complementarity

PHASE18_PROXY_VARIANTS = {
    "morphology_only": slice(
        0,
        PHASE18_MORPHOLOGY_FEATURE_COUNT,
    ),
    "morphology_plus_prototypes": slice(
        0,
        PHASE18_TOTAL_PROXY_FEATURE_COUNT,
    ),
}

PHASE18_PROXY_OOF = {
    name: np.full(
        len(y),
        np.nan,
        dtype=np.float64,
    )
    for name in PHASE18_PROXY_VARIANTS
}

PHASE18_PROXY_PRIVATE_STATES = {
    name: []
    for name in PHASE18_PROXY_VARIANTS
}

proxy_training_rows = []
proxy_started = time.perf_counter()


def phase18_fit_platt(
    monitor_logits,
    monitor_targets,
):
    calibrator = LogisticRegression(
        C=10.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=PHASE18_SEED,
    )

    calibrator.fit(
        np.asarray(
            monitor_logits,
            dtype=np.float64,
        ).reshape(-1, 1),
        monitor_targets,
    )

    return calibrator


for fold, (
    fit_idx,
    monitor_idx,
    outer_valid_idx,
) in enumerate(nested_splits):
    fold_features = PHASE18_PROXY_FEATURES[fold]

    for variant_name, feature_slice in (
        PHASE18_PROXY_VARIANTS.items()
    ):
        selected_features = fold_features[
            :,
            feature_slice,
        ]

        scaler = StandardScaler()
        scaler.fit(selected_features[fit_idx])

        fit_features = scaler.transform(
            selected_features[fit_idx]
        )
        monitor_features = scaler.transform(
            selected_features[monitor_idx]
        )
        outer_features = scaler.transform(
            selected_features[outer_valid_idx]
        )

        classifier = LogisticRegression(
            C=0.10,
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=(
                PHASE18_SEED
                + fold * 100
            ),
        )

        classifier.fit(
            fit_features,
            y[fit_idx],
        )

        monitor_logits = (
            classifier.decision_function(
                monitor_features
            )
        )
        outer_logits = (
            classifier.decision_function(
                outer_features
            )
        )

        calibrator = phase18_fit_platt(
            monitor_logits,
            y[monitor_idx],
        )

        monitor_probability = (
            calibrator.predict_proba(
                monitor_logits.reshape(-1, 1)
            )[:, 1]
        )

        outer_probability = (
            calibrator.predict_proba(
                outer_logits.reshape(-1, 1)
            )[:, 1]
        )

        monitor_probability = np.clip(
            monitor_probability,
            PROBABILITY_EPS,
            1.0 - PROBABILITY_EPS,
        )
        outer_probability = np.clip(
            outer_probability,
            PROBABILITY_EPS,
            1.0 - PROBABILITY_EPS,
        )

        PHASE18_PROXY_OOF[
            variant_name
        ][outer_valid_idx] = outer_probability

        PHASE18_PROXY_PRIVATE_STATES[
            variant_name
        ].append({
            "fold": int(fold),
            "scaler": scaler,
            "classifier": classifier,
            "calibrator": calibrator,
            "feature_slice": feature_slice,
        })

        proxy_training_rows.append({
            "fold": int(fold),
            "variant": variant_name,
            "feature_count": int(
                selected_features.shape[1]
            ),
            "monitor_log_loss": round(
                float(
                    log_loss(
                        y[monitor_idx],
                        monitor_probability,
                    )
                ),
                6,
            ),
            "outer_log_loss": round(
                float(
                    log_loss(
                        y[outer_valid_idx],
                        outer_probability,
                    )
                ),
                6,
            ),
            "outer_auroc": round(
                float(
                    roc_auc_score(
                        y[outer_valid_idx],
                        outer_probability,
                    )
                ),
                6,
            ),
        })

for probability in PHASE18_PROXY_OOF.values():
    assert np.isfinite(probability).all()
    assert (
        (probability >= PROBABILITY_EPS)
        & (
            probability
            <= 1.0 - PROBABILITY_EPS
        )
    ).all()

PHASE18_PROXY_SECONDS = float(
    time.perf_counter() - proxy_started
)

print(
    "Proxy OOF computation complete. "
    "Run Cell 13 for the sanitized comparison."
)

Proxy OOF computation complete. Run Cell 13 for the sanitized comparison.


In [15]:
# Cell 13 — sanitized proxy ablation report


def phase18_probability_metrics(
    targets,
    probability,
):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        PROBABILITY_EPS,
        1.0 - PROBABILITY_EPS,
    )

    probability_logits = np.log(
        probability / (1.0 - probability)
    )

    slope_model = LogisticRegression(
        C=1000.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=PHASE18_SEED,
    )

    slope_model.fit(
        probability_logits.reshape(-1, 1),
        targets,
    )

    return {
        "log_loss": round(
            float(log_loss(targets, probability)),
            6,
        ),
        "auroc": round(
            float(
                roc_auc_score(
                    targets,
                    probability,
                )
            ),
            6,
        ),
        "brier": round(
            float(
                brier_score_loss(
                    targets,
                    probability,
                )
            ),
            6,
        ),
        "calibration_slope": round(
            float(slope_model.coef_[0, 0]),
            6,
        ),
        "mean_probability": round(
            float(probability.mean()),
            6,
        ),
    }


proxy_metrics = {
    name: phase18_probability_metrics(
        y,
        probability,
    )
    for name, probability
    in PHASE18_PROXY_OOF.items()
}

proxy_fold_metrics = {}

for variant_name, probability in (
    PHASE18_PROXY_OOF.items()
):
    proxy_fold_metrics[variant_name] = []

    for fold in range(N_GROUP_FOLDS):
        fold_indices = np.flatnonzero(
            fold_id == fold
        )

        proxy_fold_metrics[
            variant_name
        ].append({
            "fold": int(fold),
            **phase18_probability_metrics(
                y[fold_indices],
                probability[fold_indices],
            ),
        })

morphology_metrics = proxy_metrics[
    "morphology_only"
]
full_metrics = proxy_metrics[
    "morphology_plus_prototypes"
]

prototype_log_loss_gain = float(
    morphology_metrics["log_loss"]
    - full_metrics["log_loss"]
)

prototype_auroc_gain = float(
    full_metrics["auroc"]
    - morphology_metrics["auroc"]
)

prototype_fold_wins = int(sum(
    full_row["log_loss"]
    < morphology_row["log_loss"]
    for full_row, morphology_row in zip(
        proxy_fold_metrics[
            "morphology_plus_prototypes"
        ],
        proxy_fold_metrics[
            "morphology_only"
        ],
    )
))

# This is a representation gate, not a final model-promotion gate.
prototype_representation_passed = bool(
    (
        prototype_log_loss_gain >= 0.003
        or prototype_auroc_gain >= 0.003
    )
    and prototype_fold_wins >= 2
)

# Explain the rejected registration failures without
# exposing any case-level information.
invalid_code = PHASE18_STATUS_TO_CODE[
    "invalid_transform_fallback"
]

invalid_mask = (
    PHASE18_REGISTRATION_STATUS_CODES
    == invalid_code
)

invalid_count = int(invalid_mask.sum())

rotation_excess_count = int(
    (
        invalid_mask
        & (
            PHASE18_REGISTRATION_MAX_ROTATION
            > PHASE18_MAX_ROTATION_DEG
        )
    ).sum()
)

translation_excess_count = int(
    (
        invalid_mask
        & (
            PHASE18_REGISTRATION_MAX_TRANSLATION
            > PHASE18_MAX_TRANSLATION_MM
        )
    ).sum()
)

both_excess_count = int(
    (
        invalid_mask
        & (
            PHASE18_REGISTRATION_MAX_ROTATION
            > PHASE18_MAX_ROTATION_DEG
        )
        & (
            PHASE18_REGISTRATION_MAX_TRANSLATION
            > PHASE18_MAX_TRANSLATION_MM
        )
    ).sum()
)

sanitized_proxy_report = {
    "phase": PHASE,
    "status": (
        "registration_rejected;"
        " registration_free_representation_evaluated"
    ),
    "registration_failure_attribution": {
        "invalid_transform_count": invalid_count,
        "rotation_bound_exceeded": (
            rotation_excess_count
        ),
        "translation_bound_exceeded": (
            translation_excess_count
        ),
        "both_bounds_exceeded": (
            both_excess_count
        ),
        "registration_thresholds_relaxed": False,
        "registration_branch_advanced": False,
    },
    "feature_extraction_seconds": round(
        PHASE18_FEATURE_SECONDS,
        2,
    ),
    "proxy_training_seconds": round(
        PHASE18_PROXY_SECONDS,
        2,
    ),
    "metrics": proxy_metrics,
    "fold_metrics": proxy_fold_metrics,
    "prototype_ablation": {
        "log_loss_gain_over_morphology": round(
            prototype_log_loss_gain,
            6,
        ),
        "auroc_gain_over_morphology": round(
            prototype_auroc_gain,
            6,
        ),
        "folds_improved": prototype_fold_wins,
        "representation_gate_passed": (
            prototype_representation_passed
        ),
        "thresholds": {
            "minimum_log_loss_or_auroc_gain": 0.003,
            "minimum_fold_wins": 2,
        },
    },
    "phase12c_reference": {
        "log_loss": 0.307616,
        "auroc": 0.938914,
        "used_for_proxy_training": False,
    },
    "case_level_features_exported": False,
    "outer_labels_used_only_for_evaluation": True,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
}

print("BEGIN SANITIZED_PHASE18_PROXY_OOF")
print(json.dumps(
    sanitized_proxy_report,
    indent=2,
))
print("END SANITIZED_PHASE18_PROXY_OOF")

BEGIN SANITIZED_PHASE18_PROXY_OOF
{
  "phase": "18a_mt_bimorph_preflight",
  "status": "registration_rejected; registration_free_representation_evaluated",
  "registration_failure_attribution": {
    "invalid_transform_count": 790,
    "rotation_bound_exceeded": 376,
    "translation_bound_exceeded": 591,
    "both_bounds_exceeded": 177,
    "registration_thresholds_relaxed": false,
    "registration_branch_advanced": false
  },
  "feature_extraction_seconds": 7.8,
  "proxy_training_seconds": 0.11,
  "metrics": {
    "morphology_only": {
      "log_loss": 0.427977,
      "auroc": 0.889257,
      "brier": 0.135163,
      "calibration_slope": 0.895013,
      "mean_probability": 0.503139
    },
    "morphology_plus_prototypes": {
      "log_loss": 0.430924,
      "auroc": 0.890545,
      "brier": 0.13562,
      "calibration_slope": 0.83199,
      "mean_probability": 0.496629
    }
  },
  "fold_metrics": {
    "morphology_only": [
      {
        "fold": 0,
        "log_loss": 0.4962,
    

In [21]:
# Cell 14 — Phase 18b model configuration and auxiliary targets

PHASE18_MODEL_PHASE = (
    "18b_registration_free_multiscale_bimorphnet"
)

PHASE18_MODEL_CONFIG = {
    "highres_shape": (80, 80, 80),
    "context_shape": (64, 64, 64),
    "input_channels_after_bimorph": 2,
    "regional_token_count": 4,
    "model_dimension": 96,
    "transformer_heads": 4,
    "transformer_layers": 2,
    "transformer_ff_dimension": 192,
    "auxiliary_target_count": 8,
    "dropout": 0.15,
}

PHASE18_BATCH_SIZE = 4
PHASE18_MAX_EPOCHS = 45
PHASE18_PATIENCE = 9
PHASE18_MIN_DELTA = 1e-4
PHASE18_LEARNING_RATE = 2e-4
PHASE18_WEIGHT_DECAY = 2e-4
PHASE18_AUXILIARY_WEIGHT = 0.08
PHASE18_EMA_DECAY = 0.995
PHASE18_GRADIENT_CLIP = 2.0
PHASE18_NUM_WORKERS = 0

PHASE18_AUXILIARY_TARGET_NAMES = (
    "bilateral_uptake",
    "bilateral_asymmetry",
    "anterior_mean",
    "posterior_mean",
    "anterior_asymmetry",
    "posterior_asymmetry",
    "posterior_to_anterior_ratio",
    "minimum_posterior_uptake",
)

auxiliary_feature_positions = np.asarray(
    [
        PHASE18_MORPHOLOGY_FEATURE_NAMES.index(name)
        for name in PHASE18_AUXILIARY_TARGET_NAMES
    ],
    dtype=np.int64,
)

PHASE18_AUXILIARY_TARGETS_RAW = (
    PHASE18_MORPHOLOGY_FEATURES[
        :,
        auxiliary_feature_positions,
    ].astype(np.float32)
)

assert PHASE18_AUXILIARY_TARGETS_RAW.shape == (
    len(y),
    PHASE18_MODEL_CONFIG[
        "auxiliary_target_count"
    ],
)
assert np.isfinite(
    PHASE18_AUXILIARY_TARGETS_RAW
).all()

# Prototype and registration outputs are not model inputs.
PHASE18_PROTOTYPES_USED_BY_MODEL = False
PHASE18_TRANSFORMS_USED_BY_MODEL = False

print(json.dumps({
    "phase": PHASE18_MODEL_PHASE,
    "batch_size": PHASE18_BATCH_SIZE,
    "maximum_epochs": PHASE18_MAX_EPOCHS,
    "patience": PHASE18_PATIENCE,
    "learning_rate": PHASE18_LEARNING_RATE,
    "auxiliary_weight": PHASE18_AUXILIARY_WEIGHT,
    "auxiliary_targets": list(
        PHASE18_AUXILIARY_TARGET_NAMES
    ),
    "prototype_features_used": False,
    "spatial_registration_used": False,
}, indent=2))

{
  "phase": "18b_registration_free_multiscale_bimorphnet",
  "batch_size": 4,
  "maximum_epochs": 45,
  "patience": 9,
  "learning_rate": 0.0002,
  "auxiliary_weight": 0.08,
  "auxiliary_targets": [
    "bilateral_uptake",
    "bilateral_asymmetry",
    "anterior_mean",
    "posterior_mean",
    "anterior_asymmetry",
    "posterior_asymmetry",
    "posterior_to_anterior_ratio",
    "minimum_posterior_uptake"
  ],
  "prototype_features_used": false,
  "spatial_registration_used": false
}


In [18]:
# Cell 15 — multiscale dataset and mild coupled augmentation

class Phase18BiMorphDataset(Dataset):
    def __init__(
        self,
        indices,
        standardized_auxiliary_targets,
    ):
        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )
        self.auxiliary_targets = np.asarray(
            standardized_auxiliary_targets,
            dtype=np.float32,
        )

        assert self.auxiliary_targets.shape == (
            len(y),
            PHASE18_MODEL_CONFIG[
                "auxiliary_target_count"
            ],
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        private_index = int(self.indices[item])

        highres = np.asarray(
            PHASE18_VOLUMES["highres"][
                private_index
            ],
            dtype=np.float32,
        )

        context = np.asarray(
            PHASE18_VOLUMES["context"][
                private_index
            ],
            dtype=np.float32,
        )

        target = np.float32(y[private_index])

        auxiliary_target = self.auxiliary_targets[
            private_index
        ]

        return (
            torch.from_numpy(highres[None]),
            torch.from_numpy(context[None]),
            torch.tensor(target, dtype=torch.float32),
            torch.from_numpy(auxiliary_target),
        )


def phase18_zero_filled_shift(sample, shifts):
    # sample shape: [C, X, Y, Z]
    shifted = torch.roll(
        sample,
        shifts=tuple(int(value) for value in shifts),
        dims=(1, 2, 3),
    )

    for axis, shift_value in zip(
        (1, 2, 3),
        shifts,
    ):
        shift_value = int(shift_value)

        if shift_value == 0:
            continue

        index = [slice(None)] * 4

        if shift_value > 0:
            index[axis] = slice(0, shift_value)
        else:
            index[axis] = slice(
                shift_value,
                None,
            )

        shifted[tuple(index)] = 0.0

    return shifted


def phase18_augment_multiscale(
    highres,
    context,
):
    """
    Mild, physically coupled perturbations.

    No reflection augmentation is needed because reflection invariance
    is enforced algebraically inside the model.
    """
    batch_size = highres.shape[0]

    highres_spacing = (
        PHASE18_CACHE_CONFIG["highres"]["cube_mm"]
        / PHASE18_CACHE_CONFIG["highres"]["shape"][0]
    )
    context_spacing = (
        PHASE18_CACHE_CONFIG["context"]["cube_mm"]
        / PHASE18_CACHE_CONFIG["context"]["shape"][0]
    )

    for batch_position in range(batch_size):
        highres_shifts = [
            int(
                torch.randint(
                    -2,
                    3,
                    (1,),
                    device=highres.device,
                ).item()
            )
            for _ in range(3)
        ]

        context_shifts = [
            int(
                round(
                    shift_value
                    * highres_spacing
                    / context_spacing
                )
            )
            for shift_value in highres_shifts
        ]

        highres[batch_position] = (
            phase18_zero_filled_shift(
                highres[batch_position],
                highres_shifts,
            )
        )

        context[batch_position] = (
            phase18_zero_filled_shift(
                context[batch_position],
                context_shifts,
            )
        )

    gamma = torch.empty(
        batch_size,
        1,
        1,
        1,
        1,
        device=highres.device,
    ).uniform_(0.95, 1.05)

    intensity_scale = torch.empty(
        batch_size,
        1,
        1,
        1,
        1,
        device=highres.device,
    ).uniform_(0.95, 1.05)

    highres = intensity_scale * torch.clamp(
        highres,
        min=0.0,
    ).pow(gamma)

    context = intensity_scale * torch.clamp(
        context,
        min=0.0,
    ).pow(gamma)

    highres_noise = torch.empty(
        batch_size,
        1,
        1,
        1,
        1,
        device=highres.device,
    ).uniform_(0.0, 0.008)

    context_noise = torch.empty(
        batch_size,
        1,
        1,
        1,
        1,
        device=context.device,
    ).uniform_(0.0, 0.008)

    highres = (
        highres
        + highres_noise
        * torch.randn_like(highres)
    )
    context = (
        context
        + context_noise
        * torch.randn_like(context)
    )

    return (
        torch.clamp(highres, 0.0, 1.5),
        torch.clamp(context, 0.0, 1.5),
    )


def phase18_make_loader(
    indices,
    standardized_auxiliary_targets,
    shuffle,
    seed,
):
    generator = torch.Generator()
    generator.manual_seed(int(seed))

    return DataLoader(
        Phase18BiMorphDataset(
            indices,
            standardized_auxiliary_targets,
        ),
        batch_size=PHASE18_BATCH_SIZE,
        shuffle=bool(shuffle),
        num_workers=PHASE18_NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
        generator=generator,
    )

In [19]:
# Cell 16 — registration-free exact-invariant MT-BiMorphNet

def phase18_group_count(channels):
    for groups_candidate in (8, 4, 2, 1):
        if channels % groups_candidate == 0:
            return groups_candidate

    return 1


class Phase18ConvNormAct(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
        kernel_size=3,
        stride=1,
    ):
        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv3d(
            input_channels,
            output_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )

        self.norm = nn.GroupNorm(
            phase18_group_count(output_channels),
            output_channels,
        )

        self.activation = nn.SiLU(inplace=False)

    def forward(self, x):
        return self.activation(
            self.norm(self.conv(x))
        )


class Phase18ResidualBlock(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
        stride=1,
        dropout=0.0,
    ):
        super().__init__()

        self.conv1 = Phase18ConvNormAct(
            input_channels,
            output_channels,
            kernel_size=3,
            stride=stride,
        )

        self.conv2 = nn.Conv3d(
            output_channels,
            output_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        self.norm2 = nn.GroupNorm(
            phase18_group_count(output_channels),
            output_channels,
        )

        self.dropout = nn.Dropout3d(
            p=float(dropout)
        )

        if (
            stride == 1
            and input_channels == output_channels
        ):
            self.skip = nn.Identity()
        else:
            self.skip = nn.Conv3d(
                input_channels,
                output_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )

        self.activation = nn.SiLU(inplace=False)

    def forward(self, x):
        residual = self.skip(x)

        x = self.conv1(x)
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))

        return self.activation(x + residual)


class Phase18HighResolutionEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = Phase18ConvNormAct(
            2,
            16,
            kernel_size=5,
            stride=2,
        )

        self.stage1 = nn.Sequential(
            Phase18ResidualBlock(16, 16),
            Phase18ResidualBlock(16, 16),
        )

        self.stage2 = nn.Sequential(
            Phase18ResidualBlock(
                16,
                32,
                stride=2,
            ),
            Phase18ResidualBlock(32, 32),
        )

        self.stage3 = nn.Sequential(
            Phase18ResidualBlock(
                32,
                64,
                stride=2,
            ),
            Phase18ResidualBlock(64, 64),
        )

        self.stage4 = nn.Sequential(
            Phase18ResidualBlock(
                64,
                96,
                stride=2,
            ),
            Phase18ResidualBlock(96, 96),
        )

    def forward(self, x):
        x = self.stage1(self.stem(x))
        x = self.stage2(x)

        regional_map = self.stage3(x)
        global_map = self.stage4(regional_map)

        return regional_map, global_map


class Phase18ContextEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            Phase18ConvNormAct(
                2,
                12,
                kernel_size=5,
                stride=2,
            ),
            Phase18ResidualBlock(12, 12),
            Phase18ResidualBlock(
                12,
                24,
                stride=2,
            ),
            Phase18ResidualBlock(24, 24),
            Phase18ResidualBlock(
                24,
                48,
                stride=2,
            ),
            Phase18ResidualBlock(48, 48),
            Phase18ResidualBlock(
                48,
                96,
                stride=2,
            ),
            Phase18ResidualBlock(96, 96),
        )

    def forward(self, x):
        return self.encoder(x)


def phase18_build_soft_regional_masks(
    output_size=10,
):
    axis = torch.linspace(
        -1.0,
        1.0,
        int(output_size),
        dtype=torch.float32,
    )

    x, y_coordinate, z = torch.meshgrid(
        axis,
        axis,
        axis,
        indexing="ij",
    )

    absolute_x = torch.abs(x)

    # Bilateral soft regions:
    # caudate-like, anterior putamen, central putamen,
    # and posterior/lateral putamen.
    region_centers = (
        (0.22, 0.28, 0.02),
        (0.39, 0.10, 0.00),
        (0.34, -0.10, -0.02),
        (0.48, -0.32, -0.04),
    )

    region_scales = (
        (0.17, 0.23, 0.34),
        (0.16, 0.23, 0.34),
        (0.18, 0.22, 0.34),
        (0.17, 0.22, 0.34),
    )

    masks = []

    for center, scale in zip(
        region_centers,
        region_scales,
    ):
        center_x, center_y, center_z = center
        scale_x, scale_y, scale_z = scale

        mask = torch.exp(
            -0.5
            * (
                (
                    (absolute_x - center_x)
                    / scale_x
                )
                ** 2
                + (
                    (y_coordinate - center_y)
                    / scale_y
                )
                ** 2
                + (
                    (z - center_z)
                    / scale_z
                )
                ** 2
            )
        )

        mask = mask / mask.sum()
        masks.append(mask)

    output = torch.stack(masks, dim=0)

    assert output.shape == (
        4,
        output_size,
        output_size,
        output_size,
    )

    return output


class Phase18BiMorphNet(nn.Module):
    """
    Exact reflection-invariant multiscale regional-token model.

    For volume V and left-right reflection R(V), the model input is:
        symmetric = (V + R(V)) / 2
        asymmetry = |V - R(V)|

    Both channels are identical for V and R(V).
    """

    def __init__(self):
        super().__init__()

        model_dimension = PHASE18_MODEL_CONFIG[
            "model_dimension"
        ]

        self.highres_encoder = (
            Phase18HighResolutionEncoder()
        )
        self.context_encoder = (
            Phase18ContextEncoder()
        )

        self.register_buffer(
            "regional_masks",
            phase18_build_soft_regional_masks(10),
            persistent=True,
        )

        self.region_projection = nn.Sequential(
            nn.Linear(64, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.SiLU(inplace=False),
        )

        self.highres_global_projection = nn.Sequential(
            nn.Linear(192, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.SiLU(inplace=False),
        )

        self.context_global_projection = nn.Sequential(
            nn.Linear(192, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.SiLU(inplace=False),
        )

        self.cls_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                model_dimension,
            )
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                7,
                model_dimension,
            )
        )

        transformer_layer = (
            nn.TransformerEncoderLayer(
                d_model=model_dimension,
                nhead=PHASE18_MODEL_CONFIG[
                    "transformer_heads"
                ],
                dim_feedforward=(
                    PHASE18_MODEL_CONFIG[
                        "transformer_ff_dimension"
                    ]
                ),
                dropout=PHASE18_MODEL_CONFIG[
                    "dropout"
                ],
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=PHASE18_MODEL_CONFIG[
                "transformer_layers"
            ],
            norm=nn.LayerNorm(model_dimension),
        )

        fusion_dimension = 3 * model_dimension

        self.fusion = nn.Sequential(
            nn.LayerNorm(fusion_dimension),
            nn.Linear(fusion_dimension, 128),
            nn.SiLU(inplace=False),
            nn.Dropout(
                p=PHASE18_MODEL_CONFIG["dropout"]
            ),
        )

        self.classifier = nn.Linear(128, 1)

        self.auxiliary_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.SiLU(inplace=False),
            nn.Linear(
                64,
                PHASE18_MODEL_CONFIG[
                    "auxiliary_target_count"
                ],
            ),
        )

        nn.init.trunc_normal_(
            self.cls_token,
            std=0.02,
        )
        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

    @staticmethod
    def bimorph_transform(volume):
        reflected = torch.flip(
            volume,
            dims=(2,),
        )

        symmetric = 0.5 * (
            volume + reflected
        )
        absolute_asymmetry = torch.abs(
            volume - reflected
        )

        return torch.cat(
            [symmetric, absolute_asymmetry],
            dim=1,
        )

    @staticmethod
    def global_average_max(feature_map):
        average = feature_map.mean(
            dim=(2, 3, 4)
        )
        maximum = feature_map.flatten(2).max(
            dim=2
        ).values

        return torch.cat(
            [average, maximum],
            dim=1,
        )

    def regional_tokens(self, regional_map):
        # [B,C,X,Y,Z] × [R,X,Y,Z] -> [B,R,C]
        pooled = torch.einsum(
            "bcxyz,rxyz->brc",
            regional_map,
            self.regional_masks.to(
                dtype=regional_map.dtype
            ),
        )

        return self.region_projection(pooled)

    def forward(
        self,
        highres,
        context,
        return_embedding=False,
    ):
        highres = self.bimorph_transform(
            highres
        )
        context = self.bimorph_transform(
            context
        )

        (
            regional_map,
            highres_global_map,
        ) = self.highres_encoder(highres)

        context_global_map = self.context_encoder(
            context
        )

        region_tokens = self.regional_tokens(
            regional_map
        )

        highres_global = (
            self.highres_global_projection(
                self.global_average_max(
                    highres_global_map
                )
            )
        )

        context_global = (
            self.context_global_projection(
                self.global_average_max(
                    context_global_map
                )
            )
        )

        cls = self.cls_token.expand(
            highres.shape[0],
            -1,
            -1,
        )

        sequence = torch.cat(
            [
                cls,
                region_tokens,
                highres_global[:, None],
                context_global[:, None],
            ],
            dim=1,
        )

        sequence = (
            sequence + self.position_embedding
        )

        encoded = self.transformer(sequence)

        fused = self.fusion(
            torch.cat(
                [
                    encoded[:, 0],
                    highres_global,
                    context_global,
                ],
                dim=1,
            )
        )

        logits = self.classifier(
            fused
        ).squeeze(1)

        auxiliary_prediction = (
            self.auxiliary_head(fused)
        )

        if return_embedding:
            return (
                logits,
                auxiliary_prediction,
                fused,
                region_tokens,
            )

        return logits, auxiliary_prediction

In [20]:
# Cell 17 — architecture contract before long training

torch.manual_seed(PHASE18_SEED + 17)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        PHASE18_SEED + 17
    )
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)

probe_model = Phase18BiMorphNet().to(DEVICE)

PHASE18_PARAMETER_COUNT = int(sum(
    parameter.numel()
    for parameter in probe_model.parameters()
))

probe_highres = torch.rand(
    PHASE18_BATCH_SIZE,
    1,
    80,
    80,
    80,
    device=DEVICE,
)

probe_context = torch.rand(
    PHASE18_BATCH_SIZE,
    1,
    64,
    64,
    64,
    device=DEVICE,
)

probe_target = torch.randint(
    0,
    2,
    (PHASE18_BATCH_SIZE,),
    device=DEVICE,
).float()

probe_auxiliary_target = torch.randn(
    PHASE18_BATCH_SIZE,
    PHASE18_MODEL_CONFIG[
        "auxiliary_target_count"
    ],
    device=DEVICE,
)

amp_enabled = bool(
    DEVICE.type == "cuda"
)

probe_model.eval()

with torch.inference_mode():
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=amp_enabled,
    ):
        (
            original_logits,
            original_auxiliary,
            original_embedding,
            original_tokens,
        ) = probe_model(
            probe_highres,
            probe_context,
            return_embedding=True,
        )

        reflected_logits, reflected_auxiliary = (
            probe_model(
                torch.flip(
                    probe_highres,
                    dims=(2,),
                ),
                torch.flip(
                    probe_context,
                    dims=(2,),
                ),
            )
        )

logit_invariance_error = float(
    torch.max(
        torch.abs(
            original_logits
            - reflected_logits
        )
    ).item()
)

auxiliary_invariance_error = float(
    torch.max(
        torch.abs(
            original_auxiliary
            - reflected_auxiliary
        )
    ).item()
)

assert original_logits.shape == (
    PHASE18_BATCH_SIZE,
)
assert original_auxiliary.shape == (
    PHASE18_BATCH_SIZE,
    8,
)
assert original_embedding.shape == (
    PHASE18_BATCH_SIZE,
    128,
)
assert original_tokens.shape == (
    PHASE18_BATCH_SIZE,
    4,
    96,
)

assert logit_invariance_error <= 2e-6
assert auxiliary_invariance_error <= 2e-6

probe_model.train()
probe_optimizer = torch.optim.AdamW(
    probe_model.parameters(),
    lr=PHASE18_LEARNING_RATE,
    weight_decay=PHASE18_WEIGHT_DECAY,
)

probe_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled,
)

probe_optimizer.zero_grad(set_to_none=True)

with torch.autocast(
    device_type=DEVICE.type,
    dtype=torch.float16,
    enabled=amp_enabled,
):
    (
        training_logits,
        training_auxiliary,
    ) = probe_model(
        probe_highres,
        probe_context,
    )

    classification_loss = (
        F.binary_cross_entropy_with_logits(
            training_logits,
            probe_target,
        )
    )

    auxiliary_loss = F.smooth_l1_loss(
        training_auxiliary,
        probe_auxiliary_target,
        beta=0.5,
    )

    probe_loss = (
        classification_loss
        + PHASE18_AUXILIARY_WEIGHT
        * auxiliary_loss
    )

probe_scaler.scale(probe_loss).backward()
probe_scaler.unscale_(probe_optimizer)

gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(
        probe_model.parameters(),
        max_norm=PHASE18_GRADIENT_CLIP,
    ).item()
)

probe_scaler.step(probe_optimizer)
probe_scaler.update()

assert np.isfinite(gradient_norm)
assert np.isfinite(float(probe_loss.detach().cpu()))

peak_vram_mb = (
    float(
        torch.cuda.max_memory_allocated(DEVICE)
        / (1024 ** 2)
    )
    if DEVICE.type == "cuda"
    else 0.0
)

sanitized_architecture_report = {
    "phase": PHASE18_MODEL_PHASE,
    "model": "Phase18BiMorphNet",
    "parameter_count": int(
        PHASE18_PARAMETER_COUNT
    ),
    "input_shapes": {
        "highres": [
            PHASE18_BATCH_SIZE,
            1,
            80,
            80,
            80,
        ],
        "context": [
            PHASE18_BATCH_SIZE,
            1,
            64,
            64,
            64,
        ],
    },
    "regional_token_shape": [
        PHASE18_BATCH_SIZE,
        4,
        96,
    ],
    "embedding_shape": [
        PHASE18_BATCH_SIZE,
        128,
    ],
    "logit_shape": [
        PHASE18_BATCH_SIZE,
    ],
    "auxiliary_shape": [
        PHASE18_BATCH_SIZE,
        8,
    ],
    "reflection_logit_max_abs_error": round(
        logit_invariance_error,
        9,
    ),
    "reflection_auxiliary_max_abs_error": round(
        auxiliary_invariance_error,
        9,
    ),
    "backward_contract_passed": True,
    "gradient_norm_before_clipping": round(
        gradient_norm,
        6,
    ),
    "peak_vram_mb": round(
        peak_vram_mb,
        2,
    ),
    "prototype_features_used": False,
    "spatial_registration_used": False,
    "external_weights_used": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE18_ARCHITECTURE")
print(json.dumps(
    sanitized_architecture_report,
    indent=2,
))
print("END SANITIZED_PHASE18_ARCHITECTURE")

del (
    probe_model,
    probe_optimizer,
    probe_scaler,
    probe_highres,
    probe_context,
    probe_target,
    probe_auxiliary_target,
    original_logits,
    original_auxiliary,
    original_embedding,
    original_tokens,
    reflected_logits,
    reflected_auxiliary,
    training_logits,
    training_auxiliary,
    classification_loss,
    auxiliary_loss,
    probe_loss,
)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

/tmp/ipykernel_58/2361532890.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:897.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


BEGIN SANITIZED_PHASE18_ARCHITECTURE
{
  "phase": "18b_registration_free_multiscale_bimorphnet",
  "model": "Phase18BiMorphNet",
  "parameter_count": 2840457,
  "input_shapes": {
    "highres": [
      4,
      1,
      80,
      80,
      80
    ],
    "context": [
      4,
      1,
      64,
      64,
      64
    ]
  },
  "regional_token_shape": [
    4,
    4,
    96
  ],
  "embedding_shape": [
    4,
    128
  ],
  "logit_shape": [
    4
  ],
  "auxiliary_shape": [
    4,
    8
  ],
  "reflection_logit_max_abs_error": 0.0,
  "reflection_auxiliary_max_abs_error": 0.0,
  "backward_contract_passed": true,
  "gradient_norm_before_clipping": 5.061175,
  "peak_vram_mb": 442.53,
  "prototype_features_used": false,
  "spatial_registration_used": false,
  "external_weights_used": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE18_ARCHITECTURE


In [22]:
# Cell 18 — deterministic math attention and common utilities

import copy

if DEVICE.type == "cuda":
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(
    True,
    warn_only=False,
)


def phase18_seed_everything(seed):
    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def phase18_sigmoid_np(logits):
    logits = np.asarray(
        logits,
        dtype=np.float64,
    )
    logits = np.clip(logits, -40.0, 40.0)

    return 1.0 / (1.0 + np.exp(-logits))


def phase18_standardize_auxiliary(fit_idx):
    fit_values = PHASE18_AUXILIARY_TARGETS_RAW[
        fit_idx
    ].astype(np.float64)

    mean = fit_values.mean(axis=0)
    std = fit_values.std(axis=0)

    std = np.maximum(std, 1e-6)

    standardized = (
        PHASE18_AUXILIARY_TARGETS_RAW
        - mean[None, :]
    ) / std[None, :]

    standardized = standardized.astype(np.float32)

    assert np.isfinite(standardized).all()

    return (
        standardized,
        mean.astype(np.float32),
        std.astype(np.float32),
    )


@torch.inference_mode()
def phase18_predict_logits(
    model,
    indices,
    standardized_auxiliary_targets,
):
    loader = phase18_make_loader(
        indices,
        standardized_auxiliary_targets,
        shuffle=False,
        seed=PHASE18_SEED,
    )

    model.eval()
    output = []

    for (
        highres,
        context,
        _,
        _,
    ) in loader:
        highres = highres.to(
            DEVICE,
            non_blocking=True,
        )
        context = context.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            logits, _ = model(
                highres,
                context,
            )

        output.append(
            logits.float().cpu().numpy()
        )

    return np.concatenate(output).astype(
        np.float64
    )


def phase18_update_ema(
    ema_model,
    model,
    decay,
):
    with torch.no_grad():
        for ema_parameter, parameter in zip(
            ema_model.parameters(),
            model.parameters(),
        ):
            ema_parameter.mul_(decay).add_(
                parameter,
                alpha=1.0 - decay,
            )

        for ema_buffer, model_buffer in zip(
            ema_model.buffers(),
            model.buffers(),
        ):
            ema_buffer.copy_(model_buffer)


def phase18_fit_monitor_platt(
    monitor_logits,
    monitor_targets,
    fold,
):
    calibrator = LogisticRegression(
        C=10.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=(
            PHASE18_SEED + 10_000 + fold
        ),
    )

    calibrator.fit(
        np.asarray(
            monitor_logits,
            dtype=np.float64,
        ).reshape(-1, 1),
        monitor_targets,
    )

    return calibrator


print(json.dumps({
    "phase": PHASE18_MODEL_PHASE,
    "deterministic_algorithms_strict": True,
    "flash_attention_enabled": (
        torch.backends.cuda.flash_sdp_enabled()
        if DEVICE.type == "cuda"
        else False
    ),
    "memory_efficient_attention_enabled": (
        torch.backends.cuda.mem_efficient_sdp_enabled()
        if DEVICE.type == "cuda"
        else False
    ),
    "math_attention_enabled": (
        torch.backends.cuda.math_sdp_enabled()
        if DEVICE.type == "cuda"
        else True
    ),
}, indent=2))

{
  "phase": "18b_registration_free_multiscale_bimorphnet",
  "deterministic_algorithms_strict": true,
  "flash_attention_enabled": false,
  "memory_efficient_attention_enabled": false,
  "math_attention_enabled": true
}


In [23]:
# Cell 19 — one-fold EMA training with nested early stopping

def phase18_train_one_fold(
    fit_idx,
    monitor_idx,
    outer_valid_idx,
    fold,
):
    fold_seed = (
        PHASE18_SEED
        + 200_000
        + int(fold)
    )
    phase18_seed_everything(fold_seed)

    (
        standardized_auxiliary,
        auxiliary_mean,
        auxiliary_std,
    ) = phase18_standardize_auxiliary(
        fit_idx
    )

    fit_loader = phase18_make_loader(
        fit_idx,
        standardized_auxiliary,
        shuffle=True,
        seed=fold_seed,
    )

    model = Phase18BiMorphNet().to(DEVICE)

    ema_model = copy.deepcopy(model).to(DEVICE)
    ema_model.eval()

    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE18_LEARNING_RATE,
        weight_decay=PHASE18_WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=PHASE18_MAX_EPOCHS,
            eta_min=(
                PHASE18_LEARNING_RATE * 0.05
            ),
        )
    )

    amp_enabled = bool(
        DEVICE.type == "cuda"
    )

    gradient_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled,
    )

    best_monitor_loss = math.inf
    best_epoch = -1
    best_state = None
    best_monitor_logits = None
    stale_epochs = 0
    epochs_run = 0

    fold_started = time.perf_counter()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    for epoch in range(
        1,
        PHASE18_MAX_EPOCHS + 1,
    ):
        model.train()

        running_total_loss = 0.0
        running_classification_loss = 0.0
        running_auxiliary_loss = 0.0
        seen = 0

        for (
            highres,
            context,
            target,
            auxiliary_target,
        ) in fit_loader:
            highres = highres.to(
                DEVICE,
                non_blocking=True,
            )
            context = context.to(
                DEVICE,
                non_blocking=True,
            )
            target = target.to(
                DEVICE,
                non_blocking=True,
            )
            auxiliary_target = (
                auxiliary_target.to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            (
                highres,
                context,
            ) = phase18_augment_multiscale(
                highres,
                context,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                (
                    logits,
                    auxiliary_prediction,
                ) = model(
                    highres,
                    context,
                )

                classification_loss = (
                    F.binary_cross_entropy_with_logits(
                        logits,
                        target,
                    )
                )

                auxiliary_loss = F.smooth_l1_loss(
                    auxiliary_prediction,
                    auxiliary_target,
                    beta=0.5,
                )

                total_loss = (
                    classification_loss
                    + PHASE18_AUXILIARY_WEIGHT
                    * auxiliary_loss
                )

            gradient_scaler.scale(
                total_loss
            ).backward()

            gradient_scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=PHASE18_GRADIENT_CLIP,
            )

            gradient_scaler.step(optimizer)
            gradient_scaler.update()

            phase18_update_ema(
                ema_model,
                model,
                PHASE18_EMA_DECAY,
            )

            batch_count = len(target)

            running_total_loss += (
                float(total_loss.detach().cpu())
                * batch_count
            )
            running_classification_loss += (
                float(
                    classification_loss.detach().cpu()
                )
                * batch_count
            )
            running_auxiliary_loss += (
                float(
                    auxiliary_loss.detach().cpu()
                )
                * batch_count
            )
            seen += batch_count

        scheduler.step()

        monitor_logits = phase18_predict_logits(
            ema_model,
            monitor_idx,
            standardized_auxiliary,
        )

        monitor_probability = np.clip(
            phase18_sigmoid_np(
                monitor_logits
            ),
            PROBABILITY_EPS,
            1.0 - PROBABILITY_EPS,
        )

        monitor_loss = float(
            log_loss(
                y[monitor_idx],
                monitor_probability,
            )
        )

        train_total_loss = (
            running_total_loss / max(seen, 1)
        )
        train_classification_loss = (
            running_classification_loss
            / max(seen, 1)
        )
        train_auxiliary_loss = (
            running_auxiliary_loss
            / max(seen, 1)
        )

        epochs_run = epoch

        if (
            monitor_loss
            < best_monitor_loss
            - PHASE18_MIN_DELTA
        ):
            best_monitor_loss = monitor_loss
            best_epoch = epoch

            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in ema_model.state_dict().items()
            }

            best_monitor_logits = (
                monitor_logits.copy()
            )
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Phase18b fold {fold} "
                f"epoch {epoch:02d}: "
                f"train_total={train_total_loss:.4f}, "
                f"train_cls={train_classification_loss:.4f}, "
                f"train_aux={train_auxiliary_loss:.4f}, "
                f"monitor={monitor_loss:.4f}"
            )

        if stale_epochs >= PHASE18_PATIENCE:
            break

    assert best_state is not None
    assert best_monitor_logits is not None

    best_model = Phase18BiMorphNet().to(DEVICE)
    best_model.load_state_dict(
        best_state,
        strict=True,
    )
    best_model.eval()

    # Recompute to audit stored-best consistency.
    reconstructed_monitor_logits = (
        phase18_predict_logits(
            best_model,
            monitor_idx,
            standardized_auxiliary,
        )
    )

    reconstruction_error = float(
        np.max(
            np.abs(
                reconstructed_monitor_logits
                - best_monitor_logits
            )
        )
    )
    assert reconstruction_error <= 1e-5

    calibrator = phase18_fit_monitor_platt(
        reconstructed_monitor_logits,
        y[monitor_idx],
        fold,
    )

    monitor_probability = np.clip(
        calibrator.predict_proba(
            reconstructed_monitor_logits.reshape(
                -1,
                1,
            )
        )[:, 1],
        PROBABILITY_EPS,
        1.0 - PROBABILITY_EPS,
    )

    outer_logits = phase18_predict_logits(
        best_model,
        outer_valid_idx,
        standardized_auxiliary,
    )

    outer_probability = np.clip(
        calibrator.predict_proba(
            outer_logits.reshape(-1, 1)
        )[:, 1],
        PROBABILITY_EPS,
        1.0 - PROBABILITY_EPS,
    )

    elapsed_seconds = float(
        time.perf_counter() - fold_started
    )

    peak_vram_mb = (
        float(
            torch.cuda.max_memory_allocated(
                DEVICE
            )
            / (1024 ** 2)
        )
        if DEVICE.type == "cuda"
        else 0.0
    )

    platt_intercept = float(
        calibrator.intercept_[0]
    )
    platt_slope = float(
        calibrator.coef_[0, 0]
    )

    training_report = {
        "fold": int(fold),
        "seed": int(fold_seed),
        "fit_n": int(len(fit_idx)),
        "monitor_n": int(len(monitor_idx)),
        "outer_valid_n": int(
            len(outer_valid_idx)
        ),
        "best_epoch": int(best_epoch),
        "epochs_run": int(epochs_run),
        "best_monitor_log_loss_raw": round(
            best_monitor_loss,
            6,
        ),
        "calibrated_monitor_log_loss": round(
            float(
                log_loss(
                    y[monitor_idx],
                    monitor_probability,
                )
            ),
            6,
        ),
        "outer_log_loss": round(
            float(
                log_loss(
                    y[outer_valid_idx],
                    outer_probability,
                )
            ),
            6,
        ),
        "outer_auroc": round(
            float(
                roc_auc_score(
                    y[outer_valid_idx],
                    outer_probability,
                )
            ),
            6,
        ),
        "platt_intercept": round(
            platt_intercept,
            6,
        ),
        "platt_slope": round(
            platt_slope,
            6,
        ),
        "checkpoint_reconstruction_error": round(
            reconstruction_error,
            9,
        ),
        "training_seconds": round(
            elapsed_seconds,
            2,
        ),
        "peak_vram_mb": round(
            peak_vram_mb,
            2,
        ),
    }

    deployment_state = {
        "fold": int(fold),
        "model_state": best_state,
        "auxiliary_mean": auxiliary_mean,
        "auxiliary_std": auxiliary_std,
        "platt_intercept": platt_intercept,
        "platt_slope": platt_slope,
        "best_epoch": int(best_epoch),
        "seed": int(fold_seed),
    }

    del (
        model,
        ema_model,
        best_model,
        optimizer,
        scheduler,
        gradient_scaler,
        fit_loader,
    )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return (
        outer_probability,
        deployment_state,
        training_report,
    )

In [24]:
# Cell 20 — train one deterministic Phase 18b model per fold

PHASE18B_OOF = np.full(
    len(y),
    np.nan,
    dtype=np.float64,
)

PHASE18B_DEPLOYMENT_STATES = []
PHASE18B_TRAINING_REPORTS = []

phase18b_training_started = (
    time.perf_counter()
)

for fold, (
    fit_idx,
    monitor_idx,
    outer_valid_idx,
) in enumerate(nested_splits):
    (
        outer_probability,
        deployment_state,
        training_report,
    ) = phase18_train_one_fold(
        fit_idx,
        monitor_idx,
        outer_valid_idx,
        fold,
    )

    PHASE18B_OOF[
        outer_valid_idx
    ] = outer_probability

    PHASE18B_DEPLOYMENT_STATES.append(
        deployment_state
    )
    PHASE18B_TRAINING_REPORTS.append(
        training_report
    )

    print(
        f"Phase18b fold {fold} frozen: "
        f"best_epoch="
        f"{training_report['best_epoch']}, "
        f"outer_loss="
        f"{training_report['outer_log_loss']:.6f}, "
        f"outer_auroc="
        f"{training_report['outer_auroc']:.6f}"
    )

PHASE18B_TOTAL_TRAINING_SECONDS = float(
    time.perf_counter()
    - phase18b_training_started
)

assert np.isfinite(PHASE18B_OOF).all()
assert (
    (PHASE18B_OOF >= PROBABILITY_EPS)
    & (
        PHASE18B_OOF
        <= 1.0 - PROBABILITY_EPS
    )
).all()

print(
    "Phase18b three-fold OOF complete. "
    "Run Cell 21 for the promotion report."
)

/tmp/ipykernel_58/2361532890.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18b fold 0 epoch 01: train_total=0.6283, train_cls=0.5870, train_aux=0.5161, monitor=0.5304
Phase18b fold 0 epoch 05: train_total=0.4862, train_cls=0.4477, train_aux=0.4821, monitor=0.3918
Phase18b fold 0 epoch 10: train_total=0.3895, train_cls=0.3525, train_aux=0.4621, monitor=0.4229
Phase18b fold 0 epoch 15: train_total=0.3698, train_cls=0.3352, train_aux=0.4322, monitor=0.4731


/tmp/ipykernel_58/2361532890.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18b fold 0 frozen: best_epoch=7, outer_loss=0.454301, outer_auroc=0.877680


/tmp/ipykernel_58/2361532890.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


...

Phase18b fold 1 frozen: best_epoch=12, outer_loss=0.305885, outer_auroc=0.934180


/tmp/ipykernel_58/2361532890.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18b fold 2 epoch 01: train_total=0.6744, train_cls=0.6317, train_aux=0.5340, monitor=0.5224
Phase18b fold 2 epoch 05: train_total=0.5139, train_cls=0.4722, train_aux=0.5208, monitor=0.3285
Phase18b fold 2 epoch 10: train_total=0.4813, train_cls=0.4493, train_aux=0.3997, monitor=0.3155


/tmp/ipykernel_58/2361532890.py:361: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18b fold 2 frozen: best_epoch=3, outer_loss=0.421074, outer_auroc=0.914939
Phase18b three-fold OOF complete. Run Cell 21 for the promotion report.


In [25]:
# Cell 21 — sanitized Phase 18b OOF evaluation

PHASE12C_REFERENCE = {
    "log_loss": 0.307616,
    "auroc": 0.938914,
    "brier": 0.095535,
    "calibration_slope": 0.93554,
    "fold_log_loss": {
        0: 0.358496,
        1: 0.255415,
        2: 0.306208,
    },
}

PHASE12B_ANATOMY_SEED_REFERENCE = {
    "log_loss": 0.336134,
    "auroc": 0.930988,
    "brier": 0.104949,
}


def phase18b_metric_bundle(
    targets,
    probability,
):
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        PROBABILITY_EPS,
        1.0 - PROBABILITY_EPS,
    )

    probability_logits = np.log(
        probability / (1.0 - probability)
    )

    calibration_model = LogisticRegression(
        C=1000.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=PHASE18_SEED,
    )

    calibration_model.fit(
        probability_logits.reshape(-1, 1),
        targets,
    )

    return {
        "log_loss": round(
            float(log_loss(targets, probability)),
            6,
        ),
        "auroc": round(
            float(
                roc_auc_score(
                    targets,
                    probability,
                )
            ),
            6,
        ),
        "brier": round(
            float(
                brier_score_loss(
                    targets,
                    probability,
                )
            ),
            6,
        ),
        "calibration_intercept": round(
            float(
                calibration_model.intercept_[0]
            ),
            6,
        ),
        "calibration_slope": round(
            float(
                calibration_model.coef_[0, 0]
            ),
            6,
        ),
        "mean_probability": round(
            float(probability.mean()),
            6,
        ),
    }


phase18b_metrics = phase18b_metric_bundle(
    y,
    PHASE18B_OOF,
)

phase18b_fold_metrics = []
fold_improvements_vs_phase12c = []

for fold in range(N_GROUP_FOLDS):
    fold_indices = np.flatnonzero(
        fold_id == fold
    )

    fold_metrics = phase18b_metric_bundle(
        y[fold_indices],
        PHASE18B_OOF[fold_indices],
    )

    fold_metrics = {
        "fold": int(fold),
        "n": int(len(fold_indices)),
        **fold_metrics,
    }
    phase18b_fold_metrics.append(
        fold_metrics
    )

    improvement = (
        PHASE12C_REFERENCE[
            "fold_log_loss"
        ][fold]
        - fold_metrics["log_loss"]
    )

    fold_improvements_vs_phase12c.append({
        "fold": int(fold),
        "log_loss_improvement": round(
            float(improvement),
            6,
        ),
    })

component_gate_passed = bool(
    phase18b_metrics["log_loss"]
    <= PHASE12B_ANATOMY_SEED_REFERENCE[
        "log_loss"
    ]
    and phase18b_metrics["auroc"]
    >= PHASE12B_ANATOMY_SEED_REFERENCE[
        "auroc"
    ] - 0.001
    and 0.75
    <= phase18b_metrics[
        "calibration_slope"
    ]
    <= 1.25
)

phase12c_log_loss_gain = float(
    PHASE12C_REFERENCE["log_loss"]
    - phase18b_metrics["log_loss"]
)

phase12c_auroc_gain = float(
    phase18b_metrics["auroc"]
    - PHASE12C_REFERENCE["auroc"]
)

phase12c_fold_wins = int(sum(
    row["log_loss_improvement"] > 0
    for row in fold_improvements_vs_phase12c
))

worst_fold_excess = max(
    -row["log_loss_improvement"]
    for row in fold_improvements_vs_phase12c
)

outright_champion_gate_passed = bool(
    phase12c_log_loss_gain >= 0.005
    and phase12c_auroc_gain >= 0.003
    and phase12c_fold_wins >= 2
    and worst_fold_excess <= 0.005
    and 0.80
    <= phase18b_metrics[
        "calibration_slope"
    ]
    <= 1.20
)

major_group_rows = []

for acquisition_group in sorted(
    np.unique(groups).tolist()
):
    group_indices = np.flatnonzero(
        groups == acquisition_group
    )

    if (
        len(group_indices) < 30
        or np.unique(
            y[group_indices]
        ).size != 2
    ):
        continue

    group_metrics = phase18b_metric_bundle(
        y[group_indices],
        PHASE18B_OOF[group_indices],
    )

    major_group_rows.append({
        "group": int(acquisition_group),
        "n": int(len(group_indices)),
        "log_loss": (
            group_metrics["log_loss"]
        ),
        "auroc": group_metrics["auroc"],
    })

sanitized_phase18b_report = {
    "phase": PHASE18_MODEL_PHASE,
    "status": (
        "three_fold_group_exclusive_oof_complete"
    ),
    "model": {
        "name": "Phase18BiMorphNet",
        "parameter_count": int(
            PHASE18_PARAMETER_COUNT
        ),
        "highres_input": [1, 80, 80, 80],
        "context_input": [1, 64, 64, 64],
        "exact_reflection_invariance": True,
        "regional_tokens": 4,
        "auxiliary_targets": 8,
        "ema_decay": PHASE18_EMA_DECAY,
        "spatial_registration": False,
        "prototype_features": False,
        "external_weights": False,
    },
    "metrics": phase18b_metrics,
    "fold_metrics": phase18b_fold_metrics,
    "training": {
        "folds": PHASE18B_TRAINING_REPORTS,
        "total_training_seconds": round(
            PHASE18B_TOTAL_TRAINING_SECONDS,
            2,
        ),
    },
    "reference_comparison": {
        "phase12b_single_anatomy_seed": (
            PHASE12B_ANATOMY_SEED_REFERENCE
        ),
        "phase12c_family": {
            "log_loss": (
                PHASE12C_REFERENCE["log_loss"]
            ),
            "auroc": (
                PHASE12C_REFERENCE["auroc"]
            ),
            "brier": (
                PHASE12C_REFERENCE["brier"]
            ),
        },
        "log_loss_gain_vs_phase12c": round(
            phase12c_log_loss_gain,
            6,
        ),
        "auroc_gain_vs_phase12c": round(
            phase12c_auroc_gain,
            6,
        ),
        "folds_beating_phase12c": (
            phase12c_fold_wins
        ),
        "fold_improvements": (
            fold_improvements_vs_phase12c
        ),
        "worst_fold_excess": round(
            float(worst_fold_excess),
            6,
        ),
    },
    "promotion": {
        "component_gate_passed": (
            component_gate_passed
        ),
        "outright_champion_gate_passed": (
            outright_champion_gate_passed
        ),
        "component_gate_meaning": (
            "eligible_for_phase12c_family_blend_test"
        ),
        "outright_gate_thresholds": {
            "minimum_log_loss_gain": 0.005,
            "minimum_auroc_gain": 0.003,
            "minimum_fold_wins": 2,
            "maximum_worst_fold_excess": 0.005,
            "calibration_slope_range": [
                0.80,
                1.20,
            ],
        },
    },
    "major_acquisition_group_metrics": (
        major_group_rows
    ),
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "outer_labels_used_only_for_evaluation": True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE18B_OOF")
print(json.dumps(
    sanitized_phase18b_report,
    indent=2,
))
print("END SANITIZED_PHASE18B_OOF")

BEGIN SANITIZED_PHASE18B_OOF
{
  "phase": "18b_registration_free_multiscale_bimorphnet",
  "status": "three_fold_group_exclusive_oof_complete",
  "model": {
    "name": "Phase18BiMorphNet",
    "parameter_count": 2840457,
    "highres_input": [
      1,
      80,
      80,
      80
    ],
    "context_input": [
      1,
      64,
      64,
      64
    ],
    "exact_reflection_invariance": true,
    "regional_tokens": 4,
    "auxiliary_targets": 8,
    "ema_decay": 0.995,
    "spatial_registration": false,
    "prototype_features": false,
    "external_weights": false
  },
  "metrics": {
    "log_loss": 0.395001,
    "auroc": 0.909226,
    "brier": 0.1238,
    "calibration_intercept": 0.186367,
    "calibration_slope": 0.796337,
    "mean_probability": 0.517939
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "log_loss": 0.454301,
      "auroc": 0.87768,
      "brier": 0.147948,
      "calibration_intercept": 0.314359,
      "calibration_slope": 0.790227,
      "me

In [26]:
# Cell 22 — Phase 18c paired-view, orientation-preserving BiMorphNet

PHASE18_MODEL_PHASE = (
    "18c_paired_view_multiscale_bimorphnet"
)

PHASE18_AUXILIARY_WEIGHT = 0.04
PHASE18_MAX_EPOCHS = 40
PHASE18_PATIENCE = 9


class Phase18cHighResolutionEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = Phase18ConvNormAct(
            1,
            16,
            kernel_size=5,
            stride=2,
        )

        self.stage1 = nn.Sequential(
            Phase18ResidualBlock(16, 16),
            Phase18ResidualBlock(16, 16),
        )

        self.stage2 = nn.Sequential(
            Phase18ResidualBlock(
                16,
                32,
                stride=2,
            ),
            Phase18ResidualBlock(32, 32),
        )

        self.stage3 = nn.Sequential(
            Phase18ResidualBlock(
                32,
                64,
                stride=2,
            ),
            Phase18ResidualBlock(64, 64),
        )

        self.stage4 = nn.Sequential(
            Phase18ResidualBlock(
                64,
                96,
                stride=2,
            ),
            Phase18ResidualBlock(96, 96),
        )

    def forward(self, volume):
        volume = self.stage1(
            self.stem(volume)
        )
        volume = self.stage2(volume)

        regional_map = self.stage3(volume)
        global_map = self.stage4(regional_map)

        return regional_map, global_map


class Phase18cContextEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            Phase18ConvNormAct(
                1,
                12,
                kernel_size=5,
                stride=2,
            ),
            Phase18ResidualBlock(12, 12),
            Phase18ResidualBlock(
                12,
                24,
                stride=2,
            ),
            Phase18ResidualBlock(24, 24),
            Phase18ResidualBlock(
                24,
                48,
                stride=2,
            ),
            Phase18ResidualBlock(48, 48),
            Phase18ResidualBlock(
                48,
                96,
                stride=2,
            ),
            Phase18ResidualBlock(96, 96),
        )

    def forward(self, volume):
        return self.encoder(volume)


class Phase18cPairedViewNet(nn.Module):
    """
    Shared encoders process identity and reflection independently.

    Their learned embeddings are fused as:
        mean(E(V), E(R(V)))
        abs(E(V) - E(R(V)))

    Swapping V and R(V) leaves both terms unchanged.
    """

    def __init__(self):
        super().__init__()

        model_dimension = PHASE18_MODEL_CONFIG[
            "model_dimension"
        ]

        self.highres_encoder = (
            Phase18cHighResolutionEncoder()
        )
        self.context_encoder = (
            Phase18cContextEncoder()
        )

        self.register_buffer(
            "regional_masks",
            phase18_build_soft_regional_masks(10),
            persistent=True,
        )

        self.region_projection = nn.Sequential(
            nn.Linear(64, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.SiLU(inplace=False),
        )

        self.highres_global_projection = nn.Sequential(
            nn.Linear(192, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.SiLU(inplace=False),
        )

        self.context_global_projection = nn.Sequential(
            nn.Linear(192, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.SiLU(inplace=False),
        )

        self.cls_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                model_dimension,
            )
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                7,
                model_dimension,
            )
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=PHASE18_MODEL_CONFIG[
                "transformer_heads"
            ],
            dim_feedforward=(
                PHASE18_MODEL_CONFIG[
                    "transformer_ff_dimension"
                ]
            ),
            dropout=PHASE18_MODEL_CONFIG[
                "dropout"
            ],
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=PHASE18_MODEL_CONFIG[
                "transformer_layers"
            ],
            norm=nn.LayerNorm(model_dimension),
        )

        self.single_view_fusion = nn.Sequential(
            nn.LayerNorm(3 * model_dimension),
            nn.Linear(
                3 * model_dimension,
                128,
            ),
            nn.SiLU(inplace=False),
            nn.Dropout(
                p=PHASE18_MODEL_CONFIG[
                    "dropout"
                ]
            ),
        )

        self.paired_fusion = nn.Sequential(
            nn.LayerNorm(256),
            nn.Linear(256, 128),
            nn.SiLU(inplace=False),
            nn.Dropout(
                p=PHASE18_MODEL_CONFIG[
                    "dropout"
                ]
            ),
        )

        self.classifier = nn.Linear(128, 1)

        self.auxiliary_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.SiLU(inplace=False),
            nn.Linear(
                64,
                PHASE18_MODEL_CONFIG[
                    "auxiliary_target_count"
                ],
            ),
        )

        nn.init.trunc_normal_(
            self.cls_token,
            std=0.02,
        )
        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

    @staticmethod
    def global_average_max(feature_map):
        average = feature_map.mean(
            dim=(2, 3, 4)
        )

        maximum = feature_map.flatten(2).max(
            dim=2
        ).values

        return torch.cat(
            [average, maximum],
            dim=1,
        )

    def regional_tokens(self, regional_map):
        pooled = torch.einsum(
            "bcxyz,rxyz->brc",
            regional_map,
            self.regional_masks.to(
                dtype=regional_map.dtype
            ),
        )

        return self.region_projection(pooled)

    def encode_single_view(
        self,
        highres,
        context,
    ):
        (
            regional_map,
            highres_global_map,
        ) = self.highres_encoder(highres)

        context_global_map = (
            self.context_encoder(context)
        )

        region_tokens = self.regional_tokens(
            regional_map
        )

        highres_global = (
            self.highres_global_projection(
                self.global_average_max(
                    highres_global_map
                )
            )
        )

        context_global = (
            self.context_global_projection(
                self.global_average_max(
                    context_global_map
                )
            )
        )

        cls = self.cls_token.expand(
            highres.shape[0],
            -1,
            -1,
        )

        sequence = torch.cat(
            [
                cls,
                region_tokens,
                highres_global[:, None],
                context_global[:, None],
            ],
            dim=1,
        )

        encoded = self.transformer(
            sequence
            + self.position_embedding
        )

        embedding = self.single_view_fusion(
            torch.cat(
                [
                    encoded[:, 0],
                    highres_global,
                    context_global,
                ],
                dim=1,
            )
        )

        return embedding

    def forward(
        self,
        highres,
        context,
        return_views=False,
    ):
        identity_embedding = (
            self.encode_single_view(
                highres,
                context,
            )
        )

        reflected_embedding = (
            self.encode_single_view(
                torch.flip(
                    highres,
                    dims=(2,),
                ),
                torch.flip(
                    context,
                    dims=(2,),
                ),
            )
        )

        embedding_mean = 0.5 * (
            identity_embedding
            + reflected_embedding
        )

        embedding_difference = torch.abs(
            identity_embedding
            - reflected_embedding
        )

        invariant_embedding = (
            self.paired_fusion(
                torch.cat(
                    [
                        embedding_mean,
                        embedding_difference,
                    ],
                    dim=1,
                )
            )
        )

        logits = self.classifier(
            invariant_embedding
        ).squeeze(1)

        auxiliary_prediction = (
            self.auxiliary_head(
                invariant_embedding
            )
        )

        if return_views:
            return (
                logits,
                auxiliary_prediction,
                invariant_embedding,
                identity_embedding,
                reflected_embedding,
            )

        return logits, auxiliary_prediction

In [27]:
# Cell 23 — paired-view invariance and GPU contract

phase18_seed_everything(
    PHASE18_SEED + 23
)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)

probe_model = (
    Phase18cPairedViewNet().to(DEVICE)
)

PHASE18C_PARAMETER_COUNT = int(sum(
    parameter.numel()
    for parameter in probe_model.parameters()
))

probe_highres = torch.rand(
    PHASE18_BATCH_SIZE,
    1,
    80,
    80,
    80,
    device=DEVICE,
)

probe_context = torch.rand(
    PHASE18_BATCH_SIZE,
    1,
    64,
    64,
    64,
    device=DEVICE,
)

probe_target = torch.randint(
    0,
    2,
    (PHASE18_BATCH_SIZE,),
    device=DEVICE,
).float()

probe_auxiliary_target = torch.randn(
    PHASE18_BATCH_SIZE,
    8,
    device=DEVICE,
)

probe_model.eval()

with torch.inference_mode():
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=(DEVICE.type == "cuda"),
    ):
        (
            original_logits,
            original_auxiliary,
            invariant_embedding,
            identity_embedding,
            reflected_embedding,
        ) = probe_model(
            probe_highres,
            probe_context,
            return_views=True,
        )

        (
            reflected_input_logits,
            reflected_input_auxiliary,
        ) = probe_model(
            torch.flip(
                probe_highres,
                dims=(2,),
            ),
            torch.flip(
                probe_context,
                dims=(2,),
            ),
        )

logit_invariance_error = float(
    torch.max(
        torch.abs(
            original_logits
            - reflected_input_logits
        )
    ).item()
)

auxiliary_invariance_error = float(
    torch.max(
        torch.abs(
            original_auxiliary
            - reflected_input_auxiliary
        )
    ).item()
)

mean_view_disagreement = float(
    torch.mean(
        torch.abs(
            identity_embedding
            - reflected_embedding
        )
    ).item()
)

assert original_logits.shape == (
    PHASE18_BATCH_SIZE,
)
assert original_auxiliary.shape == (
    PHASE18_BATCH_SIZE,
    8,
)
assert invariant_embedding.shape == (
    PHASE18_BATCH_SIZE,
    128,
)
assert identity_embedding.shape == (
    PHASE18_BATCH_SIZE,
    128,
)
assert reflected_embedding.shape == (
    PHASE18_BATCH_SIZE,
    128,
)

assert logit_invariance_error <= 2e-6
assert auxiliary_invariance_error <= 2e-6

probe_model.train()

probe_optimizer = torch.optim.AdamW(
    probe_model.parameters(),
    lr=PHASE18_LEARNING_RATE,
    weight_decay=PHASE18_WEIGHT_DECAY,
)

probe_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda"),
)

probe_optimizer.zero_grad(
    set_to_none=True
)

with torch.autocast(
    device_type=DEVICE.type,
    dtype=torch.float16,
    enabled=(DEVICE.type == "cuda"),
):
    (
        training_logits,
        training_auxiliary,
    ) = probe_model(
        probe_highres,
        probe_context,
    )

    classification_loss = (
        F.binary_cross_entropy_with_logits(
            training_logits,
            probe_target,
        )
    )

    auxiliary_loss = F.smooth_l1_loss(
        training_auxiliary,
        probe_auxiliary_target,
        beta=0.5,
    )

    total_loss = (
        classification_loss
        + PHASE18_AUXILIARY_WEIGHT
        * auxiliary_loss
    )

probe_scaler.scale(
    total_loss
).backward()

probe_scaler.unscale_(
    probe_optimizer
)

gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(
        probe_model.parameters(),
        PHASE18_GRADIENT_CLIP,
    ).item()
)

probe_scaler.step(probe_optimizer)
probe_scaler.update()

peak_vram_mb = (
    float(
        torch.cuda.max_memory_allocated(
            DEVICE
        )
        / (1024 ** 2)
    )
    if DEVICE.type == "cuda"
    else 0.0
)

assert np.isfinite(gradient_norm)
assert np.isfinite(
    float(total_loss.detach().cpu())
)

sanitized_phase18c_contract = {
    "phase": PHASE18_MODEL_PHASE,
    "model": "Phase18cPairedViewNet",
    "parameter_count": int(
        PHASE18C_PARAMETER_COUNT
    ),
    "batch_size": int(
        PHASE18_BATCH_SIZE
    ),
    "embedding_shape": [
        PHASE18_BATCH_SIZE,
        128,
    ],
    "reflection_logit_max_abs_error": round(
        logit_invariance_error,
        9,
    ),
    "reflection_auxiliary_max_abs_error": round(
        auxiliary_invariance_error,
        9,
    ),
    "mean_untrained_view_embedding_difference": round(
        mean_view_disagreement,
        6,
    ),
    "gradient_norm_before_clipping": round(
        gradient_norm,
        6,
    ),
    "peak_vram_mb": round(
        peak_vram_mb,
        2,
    ),
    "math_attention_only": True,
    "backward_contract_passed": True,
    "spatial_registration": False,
    "prototype_features": False,
    "external_weights": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE18C_ARCHITECTURE")
print(json.dumps(
    sanitized_phase18c_contract,
    indent=2,
))
print("END SANITIZED_PHASE18C_ARCHITECTURE")

del (
    probe_model,
    probe_optimizer,
    probe_scaler,
    probe_highres,
    probe_context,
    probe_target,
    probe_auxiliary_target,
    original_logits,
    original_auxiliary,
    invariant_embedding,
    identity_embedding,
    reflected_embedding,
    reflected_input_logits,
    reflected_input_auxiliary,
    training_logits,
    training_auxiliary,
    classification_loss,
    auxiliary_loss,
    total_loss,
)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

# The existing training function resolves this class at call time.
Phase18BiMorphNet = Phase18cPairedViewNet
PHASE18_PARAMETER_COUNT = PHASE18C_PARAMETER_COUNT

/tmp/ipykernel_58/1982463471.py:186: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


BEGIN SANITIZED_PHASE18C_ARCHITECTURE
{
  "phase": "18c_paired_view_multiscale_bimorphnet",
  "model": "Phase18cPairedViewNet",
  "parameter_count": 2870365,
  "batch_size": 4,
  "embedding_shape": [
    4,
    128
  ],
  "reflection_logit_max_abs_error": 0.0,
  "reflection_auxiliary_max_abs_error": 0.0,
  "mean_untrained_view_embedding_difference": 0.058655,
  "gradient_norm_before_clipping": 11.254914,
  "peak_vram_mb": 786.15,
  "math_attention_only": true,
  "backward_contract_passed": true,
  "spatial_registration": false,
  "prototype_features": false,
  "external_weights": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE18C_ARCHITECTURE


In [28]:
# Cell 24 — paired-view three-fold training

PHASE18C_OOF = np.full(
    len(y),
    np.nan,
    dtype=np.float64,
)

PHASE18C_DEPLOYMENT_STATES = []
PHASE18C_TRAINING_REPORTS = []

phase18c_started = time.perf_counter()

for fold, (
    fit_idx,
    monitor_idx,
    outer_valid_idx,
) in enumerate(nested_splits):
    (
        outer_probability,
        deployment_state,
        training_report,
    ) = phase18_train_one_fold(
        fit_idx,
        monitor_idx,
        outer_valid_idx,
        fold,
    )

    PHASE18C_OOF[
        outer_valid_idx
    ] = outer_probability

    PHASE18C_DEPLOYMENT_STATES.append(
        deployment_state
    )
    PHASE18C_TRAINING_REPORTS.append(
        training_report
    )

    print(
        f"Phase18c fold {fold} frozen: "
        f"best_epoch="
        f"{training_report['best_epoch']}, "
        f"outer_loss="
        f"{training_report['outer_log_loss']:.6f}, "
        f"outer_auroc="
        f"{training_report['outer_auroc']:.6f}"
    )

PHASE18C_TOTAL_TRAINING_SECONDS = float(
    time.perf_counter() - phase18c_started
)

assert np.isfinite(PHASE18C_OOF).all()

print(
    "Phase18c OOF complete. "
    "Run Cell 25 for the final branch decision."
)

/tmp/ipykernel_58/1982463471.py:186: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18b fold 0 epoch 01: train_total=0.6462, train_cls=0.6255, train_aux=0.5168, monitor=0.5697
Phase18b fold 0 epoch 05: train_total=0.4419, train_cls=0.4228, train_aux=0.4768, monitor=0.4387
Phase18b fold 0 epoch 10: train_total=0.4051, train_cls=0.3862, train_aux=0.4723, monitor=0.4368
Phase18b fold 0 epoch 15: train_total=0.3685, train_cls=0.3500, train_aux=0.4622, monitor=0.4347
Phase18b fold 0 epoch 20: train_total=0.3277, train_cls=0.3093, train_aux=0.4597, monitor=0.4507


/tmp/ipykernel_58/1982463471.py:186: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18c fold 0 frozen: best_epoch=12, outer_loss=0.428245, outer_auroc=0.872708


/tmp/ipykernel_58/1982463471.py:186: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


...

Phase18c fold 1 frozen: best_epoch=8, outer_loss=0.296004, outer_auroc=0.929396


/tmp/ipykernel_58/1982463471.py:186: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18b fold 2 epoch 01: train_total=0.6664, train_cls=0.6449, train_aux=0.5379, monitor=0.5923
Phase18b fold 2 epoch 05: train_total=0.5443, train_cls=0.5233, train_aux=0.5252, monitor=0.5348
Phase18b fold 2 epoch 10: train_total=0.4111, train_cls=0.3916, train_aux=0.4874, monitor=0.4083
Phase18b fold 2 epoch 15: train_total=0.3484, train_cls=0.3321, train_aux=0.4066, monitor=0.4355


/tmp/ipykernel_58/1982463471.py:186: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Phase18c fold 2 frozen: best_epoch=9, outer_loss=0.408891, outer_auroc=0.901945
Phase18c OOF complete. Run Cell 25 for the final branch decision.


In [29]:
# Cell 25 — Phase 18c OOF report

phase18c_metrics = phase18b_metric_bundle(
    y,
    PHASE18C_OOF,
)

phase18c_fold_metrics = []
phase12c_fold_deltas = []

for fold in range(N_GROUP_FOLDS):
    fold_indices = np.flatnonzero(
        fold_id == fold
    )

    fold_metrics = phase18b_metric_bundle(
        y[fold_indices],
        PHASE18C_OOF[fold_indices],
    )

    phase18c_fold_metrics.append({
        "fold": int(fold),
        "n": int(len(fold_indices)),
        **fold_metrics,
    })

    phase12c_fold_deltas.append({
        "fold": int(fold),
        "log_loss_improvement": round(
            PHASE12C_REFERENCE[
                "fold_log_loss"
            ][fold]
            - fold_metrics["log_loss"],
            6,
        ),
    })

component_gate_passed = bool(
    phase18c_metrics["log_loss"]
    <= PHASE12B_ANATOMY_SEED_REFERENCE[
        "log_loss"
    ]
    and phase18c_metrics["auroc"]
    >= PHASE12B_ANATOMY_SEED_REFERENCE[
        "auroc"
    ] - 0.001
    and 0.75
    <= phase18c_metrics[
        "calibration_slope"
    ]
    <= 1.25
)

phase12c_log_loss_gain = float(
    PHASE12C_REFERENCE["log_loss"]
    - phase18c_metrics["log_loss"]
)

phase12c_auroc_gain = float(
    phase18c_metrics["auroc"]
    - PHASE12C_REFERENCE["auroc"]
)

fold_wins = int(sum(
    row["log_loss_improvement"] > 0
    for row in phase12c_fold_deltas
))

worst_fold_excess = max(
    -row["log_loss_improvement"]
    for row in phase12c_fold_deltas
)

outright_gate_passed = bool(
    phase12c_log_loss_gain >= 0.005
    and phase12c_auroc_gain >= 0.003
    and fold_wins >= 2
    and worst_fold_excess <= 0.005
    and 0.80
    <= phase18c_metrics[
        "calibration_slope"
    ]
    <= 1.20
)

sanitized_phase18c_report = {
    "phase": PHASE18_MODEL_PHASE,
    "status": (
        "paired_view_three_fold_oof_complete"
    ),
    "metrics": phase18c_metrics,
    "fold_metrics": phase18c_fold_metrics,
    "training": {
        "folds": PHASE18C_TRAINING_REPORTS,
        "total_seconds": round(
            PHASE18C_TOTAL_TRAINING_SECONDS,
            2,
        ),
    },
    "phase18b_comparison": {
        "phase18b_log_loss": 0.395001,
        "phase18b_auroc": 0.909226,
        "phase18c_minus_phase18b_log_loss_gain": round(
            0.395001
            - phase18c_metrics["log_loss"],
            6,
        ),
        "phase18c_minus_phase18b_auroc_gain": round(
            phase18c_metrics["auroc"]
            - 0.909226,
            6,
        ),
    },
    "phase12c_comparison": {
        "reference_log_loss": (
            PHASE12C_REFERENCE["log_loss"]
        ),
        "reference_auroc": (
            PHASE12C_REFERENCE["auroc"]
        ),
        "log_loss_gain": round(
            phase12c_log_loss_gain,
            6,
        ),
        "auroc_gain": round(
            phase12c_auroc_gain,
            6,
        ),
        "folds_improved": int(fold_wins),
        "fold_deltas": (
            phase12c_fold_deltas
        ),
        "worst_fold_excess": round(
            float(worst_fold_excess),
            6,
        ),
    },
    "promotion": {
        "component_gate_passed": (
            component_gate_passed
        ),
        "outright_champion_gate_passed": (
            outright_gate_passed
        ),
        "if_both_fail": (
            "terminate_phase18_architecture_search;"
            " restore_phase12c_champion_artifacts"
        ),
    },
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "outer_labels_used_only_for_evaluation": True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE18C_OOF")
print(json.dumps(
    sanitized_phase18c_report,
    indent=2,
))
print("END SANITIZED_PHASE18C_OOF")

BEGIN SANITIZED_PHASE18C_OOF
{
  "phase": "18c_paired_view_multiscale_bimorphnet",
  "status": "paired_view_three_fold_oof_complete",
  "metrics": {
    "log_loss": 0.378809,
    "auroc": 0.906125,
    "brier": 0.114475,
    "calibration_intercept": 0.239241,
    "calibration_slope": 0.924364,
    "mean_probability": 0.521116
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "log_loss": 0.428245,
      "auroc": 0.872708,
      "brier": 0.136056,
      "calibration_intercept": 0.192273,
      "calibration_slope": 0.909336,
      "mean_probability": 0.560682
    },
    {
      "fold": 1,
      "n": 443,
      "log_loss": 0.296004,
      "auroc": 0.929396,
      "brier": 0.08496,
      "calibration_intercept": 0.169464,
      "calibration_slope": 1.029205,
      "mean_probability": 0.501536
    },
    {
      "fold": 2,
      "n": 452,
      "log_loss": 0.408891,
      "auroc": 0.901945,
      "brier": 0.121107,
      "calibration_intercept": 0.395438,
      "calibrati

In [30]:
# Cell 26 — discover the retained Phase 12c archive/package
#
# Searches only for our own model artifact, not challenge images.

import json
import zipfile
from pathlib import Path

PHASE12C_EXPECTED_ARCHIVE_NAME = (
    "dat_phase12c_anatomy_family_submission.zip"
)

# Added your direct Kaggle Model path as the first root to speed up discovery
phase12c_search_roots = [
    Path("/kaggle/input/models/saifullah42/dat-phase12c/pytorch/default/1"),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

PHASE12C_VALID_CANDIDATES = []
phase12c_candidate_reports = []

for search_root in phase12c_search_roots:
    if not search_root.exists():
        continue

    archive_candidates = list(
        search_root.rglob(
            PHASE12C_EXPECTED_ARCHIVE_NAME
        )
    )

    # FIX: Check if "phase12c" is in the full path instead of just the immediate parent folder (which is "1")
    directory_manifest_candidates = [
        path
        for path in search_root.rglob(
            "manifest.json"
        )
        if "phase12c" in str(path).casefold()
    ]

    for candidate in archive_candidates:
        candidate_report = {
            "kind": "zip",
            "name": candidate.name,
            "location": (
                "kaggle_input"
                if str(candidate).startswith(
                    "/kaggle/input/"
                )
                else "kaggle_working"
            ),
            "valid_manifest": False,
        }

        try:
            with zipfile.ZipFile(
                candidate,
                "r",
            ) as archive:
                manifest_members = [
                    member
                    for member in archive.namelist()
                    if member.endswith(
                        "manifest.json"
                    )
                ]

                valid_manifest = None

                for member in manifest_members:
                    payload = json.loads(
                        archive.read(
                            member
                        ).decode("utf-8")
                    )

                    if (
                        payload.get(
                            "schema_version"
                        ) == 3
                        and payload.get(
                            "model_count"
                        ) == 21
                    ):
                        valid_manifest = payload
                        break

                if valid_manifest is not None:
                    candidate_report.update({
                        "valid_manifest": True,
                        "schema_version": 3,
                        "model_count": 21,
                        "base_model_count": int(
                            valid_manifest.get(
                                "base_model_count",
                                -1,
                            )
                        ),
                        "anatomy_model_count": int(
                            valid_manifest.get(
                                "anatomy_model_count",
                                -1,
                            )
                        ),
                        "fold_count": int(
                            valid_manifest.get(
                                "fold_count",
                                -1,
                            )
                        ),
                    })

                    PHASE12C_VALID_CANDIDATES.append(
                        candidate
                    )

        except Exception as exception:
            candidate_report["error_type"] = (
                type(exception).__name__
            )

        phase12c_candidate_reports.append(
            candidate_report
        )

    for manifest_path in (
        directory_manifest_candidates
    ):
        candidate_report = {
            "kind": "directory",
            "name": manifest_path.parent.name,
            "location": (
                "kaggle_input"
                if str(manifest_path).startswith(
                    "/kaggle/input/"
                )
                else "kaggle_working"
            ),
            "valid_manifest": False,
        }

        try:
            payload = json.loads(
                manifest_path.read_text(
                    encoding="utf-8"
                )
            )

            if (
                payload.get("schema_version") == 3
                and payload.get("model_count") == 21
            ):
                candidate_report.update({
                    "valid_manifest": True,
                    "schema_version": 3,
                    "model_count": 21,
                    "base_model_count": int(
                        payload.get(
                            "base_model_count",
                            -1,
                        )
                    ),
                    "anatomy_model_count": int(
                        payload.get(
                            "anatomy_model_count",
                            -1,
                        )
                    ),
                    "fold_count": int(
                        payload.get(
                            "fold_count",
                            -1,
                        )
                    ),
                })

                PHASE12C_VALID_CANDIDATES.append(
                    manifest_path.parent
                )

        except Exception as exception:
            candidate_report["error_type"] = (
                type(exception).__name__
            )

        phase12c_candidate_reports.append(
            candidate_report
        )

# Remove duplicate resolved paths without displaying them.
unique_candidates = []
seen_candidate_paths = set()

for candidate in PHASE12C_VALID_CANDIDATES:
    resolved = str(candidate.resolve())

    if resolved not in seen_candidate_paths:
        unique_candidates.append(candidate)
        seen_candidate_paths.add(resolved)

PHASE12C_VALID_CANDIDATES = (
    unique_candidates
)

phase12c_discovery_report = {
    "phase": "phase12c_artifact_discovery",
    "status": (
        "found"
        if PHASE12C_VALID_CANDIDATES
        else "not_found"
    ),
    "valid_candidate_count": int(
        len(PHASE12C_VALID_CANDIDATES)
    ),
    "candidates": phase12c_candidate_reports,
    "expected_contract": {
        "schema_version": 3,
        "model_count": 21,
        "base_model_count": 15,
        "anatomy_model_count": 6,
        "fold_count": 3,
    },
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE12C_DISCOVERY")
print(json.dumps(
    phase12c_discovery_report,
    indent=2,
))
print("END SANITIZED_PHASE12C_DISCOVERY")

BEGIN SANITIZED_PHASE12C_DISCOVERY
{
  "phase": "phase12c_artifact_discovery",
  "status": "found",
  "valid_candidate_count": 1,
  "candidates": [
    {
      "kind": "directory",
      "name": "1",
      "location": "kaggle_input",
      "valid_manifest": true,
      "schema_version": 3,
      "model_count": 21,
      "base_model_count": 15,
      "anatomy_model_count": 6,
      "fold_count": 3
    },
    {
      "kind": "directory",
      "name": "1",
      "location": "kaggle_input",
      "valid_manifest": true,
      "schema_version": 3,
      "model_count": 21,
      "base_model_count": 15,
      "anatomy_model_count": 6,
      "fold_count": 3
    }
  ],
  "expected_contract": {
    "schema_version": 3,
    "model_count": 21,
    "base_model_count": 15,
    "anatomy_model_count": 6,
    "fold_count": 3
  },
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "uids_displayed": false
}
END SANITIZED_PHASE12C_DISCOVERY


In [32]:
# Cell 27 — restore and validate the Phase 12c package

import hashlib
import numpy as np
import json
import zipfile
from pathlib import Path

assert PHASE12C_VALID_CANDIDATES, (
    "Phase 12c was not found. Add the old Phase 12c ZIP "
    "through Kaggle Add Input, then rerun Cell 26."
)

phase12c_source = (
    PHASE12C_VALID_CANDIDATES[0]
)

if phase12c_source.is_file():
    phase12c_extract_root = Path(
        "/kaggle/working/"
        "phase12c_restored_package"
    )
    phase12c_extract_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        phase12c_source,
        "r",
    ) as archive:
        destination_root = (
            phase12c_extract_root.resolve()
        )

        for member in archive.infolist():
            destination = (
                phase12c_extract_root
                / member.filename
            ).resolve()

            assert destination.is_relative_to(
                destination_root
            ), "Unsafe archive member path"

        # Do not overwrite an existing restoration unless it is empty.
        existing_files = [
            path
            for path in phase12c_extract_root.rglob(
                "*"
            )
            if path.is_file()
        ]

        if not existing_files:
            archive.extractall(
                phase12c_extract_root
            )

    restored_manifest_candidates = [
        path
        for path in phase12c_extract_root.rglob(
            "manifest.json"
        )
        if json.loads(
            path.read_text(
                encoding="utf-8"
            )
        ).get("schema_version") == 3
    ]

    assert len(
        restored_manifest_candidates
    ) == 1

    PHASE12C_RESTORED_ROOT = (
        restored_manifest_candidates[0].parent
    )

else:
    PHASE12C_RESTORED_ROOT = (
        phase12c_source
    )

phase12c_manifest_path = (
    PHASE12C_RESTORED_ROOT
    / "manifest.json"
)
phase12c_base_manifest_path = (
    PHASE12C_RESTORED_ROOT
    / "base_manifest.json"
)

assert phase12c_manifest_path.is_file()
assert phase12c_base_manifest_path.is_file()

PHASE12C_RESTORED_MANIFEST = json.loads(
    phase12c_manifest_path.read_text(
        encoding="utf-8"
    )
)

PHASE12C_RESTORED_BASE_MANIFEST = json.loads(
    phase12c_base_manifest_path.read_text(
        encoding="utf-8"
    )
)

assert (
    PHASE12C_RESTORED_MANIFEST[
        "schema_version"
    ]
    == 3
)
assert (
    PHASE12C_RESTORED_MANIFEST[
        "model_count"
    ]
    == 21
)
assert (
    PHASE12C_RESTORED_MANIFEST[
        "base_model_count"
    ]
    == 15
)
assert (
    PHASE12C_RESTORED_MANIFEST[
        "anatomy_model_count"
    ]
    == 6
)
assert (
    PHASE12C_RESTORED_MANIFEST[
        "fold_count"
    ]
    == 3
)

family_weights = (
    PHASE12C_RESTORED_MANIFEST[
        "family_weights"
    ]
)

assert np.isclose(
    family_weights["phase11c"],
    0.75,
)
assert np.isclose(
    family_weights["anatomy_seed_a"],
    0.125,
)
assert np.isclose(
    family_weights["anatomy_seed_b"],
    0.125,
)
assert np.isclose(
    sum(family_weights.values()),
    1.0,
)


def phase12c_sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


expected_hashes = {}

expected_hashes.update(
    PHASE12C_RESTORED_BASE_MANIFEST[
        "model_sha256"
    ]
)
expected_hashes.update(
    PHASE12C_RESTORED_MANIFEST[
        "anatomy_model_sha256"
    ]
)

model_files = sorted(
    (
        PHASE12C_RESTORED_ROOT
        / "models"
    ).glob("*.pt")
)

assert len(model_files) == 21
assert len(expected_hashes) == 21

hash_match_count = 0

for model_path in model_files:
    assert model_path.name in expected_hashes

    observed_hash = phase12c_sha256_file(
        model_path
    )

    expected_hash = expected_hashes[
        model_path.name
    ]

    assert observed_hash == expected_hash
    hash_match_count += 1

assert hash_match_count == 21
assert (
    PHASE12C_RESTORED_ROOT
    / "main.py"
).is_file()

sanitized_restore_report = {
    "phase": "phase12c_artifact_restore",
    "status": "restored_and_verified",
    "schema_version": 3,
    "model_count": 21,
    "base_model_count": 15,
    "anatomy_model_count": 6,
    "fold_count": 3,
    "hashes_verified": 21,
    "all_hashes_match": True,
    "family_weights": {
        name: float(value)
        for name, value
        in family_weights.items()
    },
    "main_entrypoint_present": True,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "model_hashes_displayed": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE12C_RESTORE")
print(json.dumps(
    sanitized_restore_report,
    indent=2,
))
print("END SANITIZED_PHASE12C_RESTORE")

BEGIN SANITIZED_PHASE12C_RESTORE
{
  "phase": "phase12c_artifact_restore",
  "status": "restored_and_verified",
  "schema_version": 3,
  "model_count": 21,
  "base_model_count": 15,
  "anatomy_model_count": 6,
  "fold_count": 3,
  "hashes_verified": 21,
  "all_hashes_match": true,
  "family_weights": {
    "anatomy_seed_a": 0.125,
    "anatomy_seed_b": 0.125,
    "phase11c": 0.75
  },
  "main_entrypoint_present": true,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "model_hashes_displayed": false,
  "uids_displayed": false
}
END SANITIZED_PHASE12C_RESTORE


In [33]:
# Cell 28 — Sanitized Phase12c manifest audit
import json
from pathlib import Path

assert "PHASE12C_RESTORED_ROOT" in globals()
assert "PHASE12C_RESTORED_MANIFEST" in globals()

phase12c_root = Path(PHASE12C_RESTORED_ROOT).resolve()

def load_json_like(value):
    if isinstance(value, dict):
        return value
    return json.loads(Path(value).read_text(encoding="utf-8"))

phase12c_manifest = load_json_like(PHASE12C_RESTORED_MANIFEST)

phase12c_base_manifest = None
if "PHASE12C_RESTORED_BASE_MANIFEST" in globals():
    phase12c_base_manifest = load_json_like(
        PHASE12C_RESTORED_BASE_MANIFEST
    )

BLOCKED_TERMS = ("sha", "hash", "digest")

def sanitize_manifest(value, depth=0):
    if depth > 8:
        return "<maximum_depth>"

    if isinstance(value, dict):
        output = {}
        for key, item in value.items():
            key_text = str(key)
            if any(term in key_text.casefold() for term in BLOCKED_TERMS):
                output[key_text] = "<verified_redacted>"
            else:
                output[key_text] = sanitize_manifest(item, depth + 1)
        return output

    if isinstance(value, (list, tuple)):
        if len(value) > 40:
            return {
                "item_count": len(value),
                "items_suppressed": True,
            }
        return [sanitize_manifest(item, depth + 1) for item in value]

    if isinstance(value, Path):
        return value.name

    if isinstance(value, str):
        return value if len(value) <= 160 else value[:157] + "..."

    if value is None or isinstance(value, (bool, int, float)):
        return value

    return f"<{type(value).__name__}>"

report = {
    "phase": "phase19_champion_residual_adapter",
    "status": "phase12c_contract_available",
    "package_file_count": sum(
        path.is_file() for path in phase12c_root.rglob("*")
    ),
    "python_file_count": sum(
        path.is_file() for path in phase12c_root.rglob("*.py")
    ),
    "checkpoint_file_count": sum(
        path.is_file()
        for suffix in ("*.pt", "*.pth")
        for path in phase12c_root.rglob(suffix)
    ),
    "phase12c_manifest": sanitize_manifest(phase12c_manifest),
    "base_manifest": sanitize_manifest(phase12c_base_manifest),
    "model_hashes_displayed": False,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_PHASE12C_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_PHASE12C_CONTRACT")

PHASE19_PHASE12C_ROOT = phase12c_root
PHASE19_PHASE12C_MANIFEST = phase12c_manifest
PHASE19_PHASE12C_BASE_MANIFEST = phase12c_base_manifest

BEGIN SANITIZED_PHASE19_PHASE12C_CONTRACT
{
  "phase": "phase19_champion_residual_adapter",
  "status": "phase12c_contract_available",
  "package_file_count": 29,
  "python_file_count": 2,
  "checkpoint_file_count": 21,
  "phase12c_manifest": {
    "anatomy": {
      "crop_mm": 128.0,
      "fold_probability_aggregation": "arithmetic mean",
      "shape": "<verified_redacted>",
      "spacing_mm": 1.6,
      "tokenization": "8 RAS octants plus 4 bilateral means plus 4 absolute bilateral differences"
    },
    "anatomy_model_count": 6,
    "anatomy_model_sha256": "<verified_redacted>",
    "base_manifest": "base_manifest.json",
    "base_model_count": 15,
    "fallback_probability": 0.548458,
    "family_weights": {
      "anatomy_seed_a": 0.125,
      "anatomy_seed_b": 0.125,
      "phase11c": 0.75
    },
    "fold_count": 3,
    "folds": [
      {
        "anatomy_models": {
          "seed_a": "models/fold0_anatomy_token_seed_a.pt",
          "seed_b": "models/fold0_anatomy_token_se

In [34]:
# Cell 29 — Static source audit
import ast
import json
from pathlib import Path

root = PHASE19_PHASE12C_ROOT
python_files = sorted(root.rglob("*.py"))

assert python_files, "No Python entrypoint found in restored package."

preferred = [path for path in python_files if path.name == "main.py"]
main_path = preferred[0] if preferred else python_files[0]

source_reports = []

for source_path in python_files:
    source = source_path.read_text(encoding="utf-8")
    tree = ast.parse(source, filename=source_path.name)

    classes = []
    functions = []
    imports = []
    has_main_guard = False

    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            if isinstance(node, ast.Import):
                imports.extend(alias.name for alias in node.names)
            else:
                imports.append(node.module or "")

        elif isinstance(node, ast.ClassDef):
            init_arguments = []
            method_names = []

            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    method_names.append(child.name)

                    if child.name == "__init__":
                        args = child.args.posonlyargs + child.args.args
                        init_arguments = [
                            argument.arg
                            for argument in args
                            if argument.arg != "self"
                        ]

            classes.append({
                "name": node.name,
                "constructor_arguments": init_arguments,
                "methods": method_names,
            })

        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            args = node.args.posonlyargs + node.args.args
            functions.append({
                "name": node.name,
                "arguments": [argument.arg for argument in args],
            })

        elif isinstance(node, ast.If):
            condition = ast.unparse(node.test)
            if "__name__" in condition and "__main__" in condition:
                has_main_guard = True

    source_reports.append({
        "file": source_path.relative_to(root).as_posix(),
        "line_count": source.count("\n") + 1,
        "imports": sorted(set(imports)),
        "classes": classes,
        "functions": functions,
        "main_guard_present": has_main_guard,
    })

report = {
    "phase": "phase19_static_source_audit",
    "status": "accepted",
    "main_entrypoint": main_path.relative_to(root).as_posix(),
    "source_files": source_reports,
    "source_executed": False,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_SOURCE_AUDIT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_SOURCE_AUDIT")

PHASE19_MAIN_PATH = main_path

BEGIN SANITIZED_PHASE19_SOURCE_AUDIT
{
  "phase": "phase19_static_source_audit",
  "status": "accepted",
  "main_entrypoint": "main.py",
  "source_files": [
    {
      "file": "base_main.py",
      "line_count": 385,
      "imports": [
        "__future__",
        "concurrent.futures",
        "csv",
        "hashlib",
        "json",
        "math",
        "nibabel",
        "numpy",
        "os",
        "pathlib",
        "scipy.ndimage",
        "time",
        "torch",
        "torch.nn",
        "torch.nn.functional"
      ],
      "classes": [
        {
          "name": "ResidualBlock3D",
          "constructor_arguments": [
            "in_channels",
            "out_channels",
            "stride"
          ],
          "methods": [
            "__init__",
            "forward"
          ]
        },
        {
          "name": "CompactDaTNet",
          "constructor_arguments": [],
          "methods": [
            "__init__",
            "forward"
          ]
        }


In [36]:
# Replacement Cell 30A — Restore Phase12c base + anatomy encoders
import ast
import gc
import importlib.util
import json
import os
import sys
from pathlib import Path

import torch
import torch.nn as nn

root = Path(PHASE19_PHASE12C_ROOT).resolve()
base_path = root / "base_main.py"
main_path = root / "main.py"

assert base_path.is_file()
assert main_path.is_file()

def has_protected_main_guard(path):
    tree = ast.parse(path.read_text(encoding="utf-8"))
    return any(
        isinstance(node, ast.If)
        and "__name__" in ast.unparse(node.test)
        and "__main__" in ast.unparse(node.test)
        for node in tree.body
    )

assert has_protected_main_guard(base_path)
assert has_protected_main_guard(main_path)

old_data_root = os.environ.get("DAT_DATA_ROOT")
old_output_csv = os.environ.get("DAT_OUTPUT_CSV")

# Prevent any accidental challenge-data access during imports.
os.environ["DAT_DATA_ROOT"] = "/kaggle/working/__phase19_no_data__"
os.environ["DAT_OUTPUT_CSV"] = (
    "/kaggle/working/__phase19_forbidden_submission.csv"
)

root_text = str(root)
if root_text not in sys.path:
    sys.path.insert(0, root_text)

# Eliminate any stale module left by the failed import.
sys.modules.pop("base_main", None)
sys.modules.pop("phase12c_family_main", None)

try:
    # Import base_main under its canonical name because main.py imports it.
    base_spec = importlib.util.spec_from_file_location(
        "base_main",
        base_path,
    )
    phase12c_base_module = importlib.util.module_from_spec(base_spec)
    sys.modules["base_main"] = phase12c_base_module
    base_spec.loader.exec_module(phase12c_base_module)

    # Import family main without invoking its protected main().
    main_spec = importlib.util.spec_from_file_location(
        "phase12c_family_main",
        main_path,
    )
    phase12c_family_module = importlib.util.module_from_spec(main_spec)
    sys.modules["phase12c_family_main"] = phase12c_family_module
    main_spec.loader.exec_module(phase12c_family_module)

finally:
    if old_data_root is None:
        os.environ.pop("DAT_DATA_ROOT", None)
    else:
        os.environ["DAT_DATA_ROOT"] = old_data_root

    if old_output_csv is None:
        os.environ.pop("DAT_OUTPUT_CSV", None)
    else:
        os.environ["DAT_OUTPUT_CSV"] = old_output_csv

assert hasattr(phase12c_base_module, "load_bundle")
assert hasattr(phase12c_family_module, "load_anatomy_bundle")

gc.collect()
torch.cuda.empty_cache()

# Expected base return contract:
# base_manifest, base_models, device
base_bundle = phase12c_base_module.load_bundle()

assert isinstance(base_bundle, tuple)
assert len(base_bundle) == 3, (
    f"Unexpected base bundle length: {len(base_bundle)}"
)

phase12c_base_manifest_runtime, phase12c_base_models, phase12c_device = (
    base_bundle
)

assert isinstance(phase12c_base_manifest_runtime, dict)
assert isinstance(phase12c_base_models, dict)

# The family manifest was already integrity-verified in Cell 27.
phase12c_family_manifest_runtime = PHASE19_PHASE12C_MANIFEST

phase12c_anatomy_models = phase12c_family_module.load_anatomy_bundle(
    phase12c_family_manifest_runtime,
    phase12c_device,
)

# Accommodate a wrapper tuple if the deployment function returns one.
if isinstance(phase12c_anatomy_models, tuple):
    anatomy_dict_candidates = [
        item
        for item in phase12c_anatomy_models
        if isinstance(item, dict)
        and any(isinstance(value, nn.Module) for value in item.values())
    ]
    assert len(anatomy_dict_candidates) == 1
    phase12c_anatomy_models = anatomy_dict_candidates[0]

assert isinstance(phase12c_anatomy_models, dict)

def unique_modules(container):
    discovered = []
    seen = set()

    def visit(value, location):
        if isinstance(value, nn.Module):
            if id(value) not in seen:
                seen.add(id(value))
                discovered.append((location, value))
            return

        if isinstance(value, dict):
            for key, item in value.items():
                visit(item, f"{location}[{repr(key)}]")
            return

        if isinstance(value, (list, tuple)):
            for index, item in enumerate(value):
                visit(item, f"{location}[{index}]")

    visit(container, "models")
    return discovered

base_model_leaves = unique_modules(phase12c_base_models)
anatomy_model_leaves = unique_modules(phase12c_anatomy_models)

assert len(base_model_leaves) == 15, (
    f"Expected 15 base models, found {len(base_model_leaves)}."
)
assert len(anatomy_model_leaves) == 6, (
    f"Expected 6 anatomy models, found {len(anatomy_model_leaves)}."
)

all_model_leaves = base_model_leaves + anatomy_model_leaves

for _, model in all_model_leaves:
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)

family_summary = {}

for _, model in all_model_leaves:
    family = model.__class__.__name__
    parameters = sum(
        parameter.numel() for parameter in model.parameters()
    )

    if family not in family_summary:
        family_summary[family] = {
            "class": family,
            "model_count": 0,
            "parameter_counts": [],
        }

    family_summary[family]["model_count"] += 1
    family_summary[family]["parameter_counts"].append(parameters)

family_report = []

for family in sorted(family_summary):
    item = family_summary[family]
    family_report.append({
        "class": item["class"],
        "model_count": item["model_count"],
        "minimum_parameter_count": min(item["parameter_counts"]),
        "maximum_parameter_count": max(item["parameter_counts"]),
    })

report = {
    "phase": "phase19_frozen_encoder_restore",
    "status": "accepted",
    "device": str(phase12c_device),
    "base_model_count": len(base_model_leaves),
    "anatomy_model_count": len(anatomy_model_leaves),
    "total_model_count": len(all_model_leaves),
    "total_frozen_parameters": sum(
        sum(parameter.numel() for parameter in model.parameters())
        for _, model in all_model_leaves
    ),
    "families": family_report,
    "all_models_eval": all(
        not model.training for _, model in all_model_leaves
    ),
    "all_parameters_frozen": all(
        not parameter.requires_grad
        for _, model in all_model_leaves
        for parameter in model.parameters()
    ),
    "package_location_type": "kaggle_input_model",
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "model_hashes_displayed": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_FROZEN_ENCODERS")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_FROZEN_ENCODERS")

PHASE19_BASE_MODULE = phase12c_base_module
PHASE19_FAMILY_MODULE = phase12c_family_module
PHASE19_BASE_MANIFEST_RUNTIME = phase12c_base_manifest_runtime
PHASE19_FAMILY_MANIFEST_RUNTIME = phase12c_family_manifest_runtime
PHASE19_BASE_MODELS = phase12c_base_models
PHASE19_ANATOMY_MODELS = phase12c_anatomy_models
PHASE19_BASE_MODEL_LEAVES = base_model_leaves
PHASE19_ANATOMY_MODEL_LEAVES = anatomy_model_leaves
PHASE19_ALL_MODEL_LEAVES = all_model_leaves
PHASE19_DEVICE = phase12c_device

/kaggle/input/models/saifullah42/dat-phase12c/pytorch/default/1/main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


BEGIN SANITIZED_PHASE19_FROZEN_ENCODERS
{
  "phase": "phase19_frozen_encoder_restore",
  "status": "accepted",
  "device": "cuda:0",
  "base_model_count": 15,
  "anatomy_model_count": 6,
  "total_model_count": 21,
  "total_frozen_parameters": 26195241,
  "families": [
    {
      "class": "AnatomyTokenDaTNet",
      "model_count": 6,
      "minimum_parameter_count": 1465861,
      "maximum_parameter_count": 1465861
    },
    {
      "class": "CompactDaTNet",
      "model_count": 15,
      "minimum_parameter_count": 1160005,
      "maximum_parameter_count": 1160005
    }
  ],
  "all_models_eval": true,
  "all_parameters_frozen": true,
  "package_location_type": "kaggle_input_model",
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "model_hashes_displayed": false,
  "uids_displayed": false
}
END SANITIZED_PHASE19_FROZEN_ENCODERS


In [37]:
# Replacement Cell 30B — Frozen encoder feature-interface audit
import json
import torch.nn as nn

representatives = {}

for _, model in PHASE19_ALL_MODEL_LEAVES:
    representatives.setdefault(model.__class__.__name__, model)

def describe_leaf_modules(model):
    rows = []

    for name, module in model.named_modules():
        if name == "":
            continue

        children = list(module.children())
        if children:
            continue

        row = {
            "name": name,
            "type": module.__class__.__name__,
            "parameter_count": sum(
                parameter.numel()
                for parameter in module.parameters(recurse=False)
            ),
        }

        for attribute in (
            "in_channels",
            "out_channels",
            "in_features",
            "out_features",
            "embed_dim",
            "num_heads",
        ):
            if hasattr(module, attribute):
                value = getattr(module, attribute)
                if isinstance(value, (int, float, bool, str)):
                    row[attribute] = value

        if isinstance(module, nn.Dropout):
            row["dropout_probability"] = float(module.p)

        rows.append(row)

    return rows

architectures = []

for family, model in sorted(representatives.items()):
    architectures.append({
        "class": family,
        "parameter_count": sum(
            parameter.numel() for parameter in model.parameters()
        ),
        "leaf_modules": describe_leaf_modules(model),
    })

report = {
    "phase": "phase19_feature_interface_audit",
    "status": "accepted",
    "architecture_count": len(architectures),
    "architectures": architectures,
    "patient_forward_pass_performed": False,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_FEATURE_INTERFACES")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_FEATURE_INTERFACES")

BEGIN SANITIZED_PHASE19_FEATURE_INTERFACES
{
  "phase": "phase19_feature_interface_audit",
  "status": "accepted",
  "architecture_count": 2,
  "architectures": [
    {
      "class": "AnatomyTokenDaTNet",
      "parameter_count": 1465861,
      "leaf_modules": [
        {
          "name": "stem.0",
          "type": "Conv3d",
          "parameter_count": 1500,
          "in_channels": 1,
          "out_channels": 12
        },
        {
          "name": "stem.1",
          "type": "InstanceNorm3d",
          "parameter_count": 24
        },
        {
          "name": "stem.2",
          "type": "SiLU",
          "parameter_count": 0
        },
        {
          "name": "body.0.conv1",
          "type": "Conv3d",
          "parameter_count": 3888,
          "in_channels": 12,
          "out_channels": 12
        },
        {
          "name": "body.0.norm1",
          "type": "InstanceNorm3d",
          "parameter_count": 24
        },
        {
          "name": "body.0.conv2",
 

In [38]:
# Cell 31 — Exact legacy caches using Phase12c's own preprocessing
import json
import time
from pathlib import Path

import numpy as np

EXPECTED_CASES = 1362

assert "case_df" in globals()
assert len(case_df) == EXPECTED_CASES
assert "PHASE19_BASE_MODULE" in globals()
assert "PHASE19_BASE_MANIFEST_RUNTIME" in globals()

crop_config = PHASE19_BASE_MANIFEST_RUNTIME["preprocessing"]["crops"]

assert "160" in crop_config
assert "192" in crop_config
assert tuple(crop_config["160"]["shape"]) == (64, 64, 64)
assert tuple(crop_config["192"]["shape"]) == (64, 64, 64)

def is_nifti_path(value):
    text = str(value).casefold()
    return text.endswith(".nii") or text.endswith(".nii.gz")

path_candidates = []

# Prefer case_df to preserve the exact label/cache row order.
for column in case_df.columns:
    values = case_df[column].tolist()

    if (
        len(values) == EXPECTED_CASES
        and all(is_nifti_path(value) for value in values)
        and all(Path(value).is_file() for value in values)
    ):
        path_candidates.append(
            (
                f"case_df_column:{column}",
                tuple(Path(value).resolve() for value in values),
            )
        )

# Fall back to aligned notebook sequences.
if not path_candidates:
    for name, value in list(globals().items()):
        if name.startswith("_"):
            continue
        if isinstance(value, (str, bytes, dict)):
            continue

        try:
            if len(value) != EXPECTED_CASES:
                continue
            values = list(value)
        except Exception:
            continue

        if (
            all(is_nifti_path(item) for item in values)
            and all(Path(item).is_file() for item in values)
        ):
            path_candidates.append(
                (
                    f"global:{name}",
                    tuple(Path(item).resolve() for item in values),
                )
            )

assert path_candidates, (
    "No aligned 1362-case training NIfTI sequence was found."
)

# Collapse multiple aliases pointing to the same ordered sequence.
unique_sequences = {}

for source, paths in path_candidates:
    identity = tuple(str(path) for path in paths)
    unique_sequences.setdefault(identity, []).append(source)

assert len(unique_sequences) == 1, {
    "message": "Multiple distinct NIfTI sequences were detected.",
    "source_groups": list(unique_sequences.values()),
}

path_identity, path_sources = next(iter(unique_sequences.items()))
training_paths = tuple(Path(value) for value in path_identity)

legacy_160 = np.empty(
    (EXPECTED_CASES, 64, 64, 64),
    dtype=np.float16,
)
legacy_192 = np.empty_like(legacy_160)

failures = []
started = time.perf_counter()

for index, path in enumerate(training_paths):
    try:
        crops = PHASE19_BASE_MODULE.preprocess_case(
            path,
            crop_config,
        )

        crop_160 = np.asarray(crops["160"], dtype=np.float32)
        crop_192 = np.asarray(crops["192"], dtype=np.float32)

        assert crop_160.shape == (64, 64, 64)
        assert crop_192.shape == (64, 64, 64)
        assert np.isfinite(crop_160).all()
        assert np.isfinite(crop_192).all()

        legacy_160[index] = crop_160.astype(np.float16)
        legacy_192[index] = crop_192.astype(np.float16)

    except Exception as error:
        failures.append(type(error).__name__)

    if (index + 1) % 200 == 0:
        print(
            f"Exact legacy preprocessing: "
            f"{index + 1}/{EXPECTED_CASES}",
            flush=True,
        )

elapsed = time.perf_counter() - started

assert not failures, {
    "failure_count": len(failures),
    "failure_types": sorted(set(failures)),
}

def summarize_cache(cache):
    values = cache.astype(np.float32, copy=False)

    return {
        "shape": list(cache.shape),
        "dtype": str(cache.dtype),
        "ram_gb": round(cache.nbytes / 1e9, 4),
        "aggregate_mean": round(float(values.mean()), 6),
        "aggregate_positive_fraction": round(
            float(np.mean(values > 0)),
            6,
        ),
        "global_minimum": round(float(values.min()), 6),
        "global_maximum": round(float(values.max()), 6),
    }

report = {
    "phase": "phase19_exact_legacy_cache",
    "status": "complete",
    "case_count": EXPECTED_CASES,
    "elapsed_seconds": round(elapsed, 2),
    "failure_count": len(failures),
    "path_source_alias_count": len(path_sources),
    "preprocessing_implementation": (
        "restored_phase12c_base_main.preprocess_case"
    ),
    "orientation": "canonical_RAS",
    "localization": "central_positive_p99_weighted_center",
    "normalization": (
        "independent_per_crop_positive_p99.5_clipped_0_to_1.5"
    ),
    "cache": {
        "160mm": summarize_cache(legacy_160),
        "192mm": summarize_cache(legacy_192),
    },
    "training_voxel_arrays_read": True,
    "case_level_cache_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_EXACT_LEGACY_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_EXACT_LEGACY_CACHE")

PHASE19_LEGACY_CACHE = {
    "160": legacy_160,
    "192": legacy_192,
}
PHASE19_TRAINING_PATHS_PRIVATE = training_paths

Exact legacy preprocessing: 200/1362
Exact legacy preprocessing: 400/1362
Exact legacy preprocessing: 600/1362
Exact legacy preprocessing: 800/1362
Exact legacy preprocessing: 1000/1362
Exact legacy preprocessing: 1200/1362
BEGIN SANITIZED_PHASE19_EXACT_LEGACY_CACHE
{
  "phase": "phase19_exact_legacy_cache",
  "status": "complete",
  "case_count": 1362,
  "elapsed_seconds": 293.62,
  "failure_count": 0,
  "path_source_alias_count": 1,
  "preprocessing_implementation": "restored_phase12c_base_main.preprocess_case",
  "orientation": "canonical_RAS",
  "localization": "central_positive_p99_weighted_center",
  "normalization": "independent_per_crop_positive_p99.5_clipped_0_to_1.5",
  "cache": {
    "160mm": {
      "shape": [
        1362,
        64,
        64,
        64
      ],
      "dtype": "float16",
      "ram_gb": 0.7141,
      "aggregate_mean": 0.24677,
      "aggregate_positive_fraction": 0.833712,
      "global_minimum": 0.0,
      "global_maximum": 1.5
    },
    "192mm": {
 

In [39]:
# Cell 32 — Synthetic embedding-hook contract
import json

import torch
import torch.nn as nn

compact_model = next(
    model
    for _, model in PHASE19_BASE_MODEL_LEAVES
    if model.__class__.__name__ == "CompactDaTNet"
)

anatomy_model = next(
    model
    for _, model in PHASE19_ANATOMY_MODEL_LEAVES
    if model.__class__.__name__ == "AnatomyTokenDaTNet"
)

captured = {}

def capture_pre_input(name):
    def hook(module, inputs):
        assert len(inputs) == 1
        captured[name] = inputs[0].detach()
    return hook

compact_hook = compact_model.classifier.register_forward_pre_hook(
    capture_pre_input("compact_embedding")
)

anatomy_hook = anatomy_model.fusion[0].register_forward_pre_hook(
    capture_pre_input("anatomy_fusion_embedding")
)

device = PHASE19_DEVICE

synthetic_compact = torch.zeros(
    (2, 1, 64, 64, 64),
    dtype=torch.float32,
    device=device,
)
synthetic_anatomy = torch.zeros(
    (2, 1, 80, 80, 80),
    dtype=torch.float32,
    device=device,
)

compact_model.eval()
anatomy_model.eval()

with torch.inference_mode():
    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=device.type == "cuda",
    ):
        compact_logit = compact_model(synthetic_compact)
        anatomy_logit = anatomy_model(synthetic_anatomy)

compact_hook.remove()
anatomy_hook.remove()

assert tuple(captured["compact_embedding"].shape) == (2, 192)
assert tuple(captured["anatomy_fusion_embedding"].shape) == (2, 256)
assert tuple(compact_logit.shape) == (2,)
assert tuple(anatomy_logit.shape) == (2,)

report = {
    "phase": "phase19_embedding_hook_contract",
    "status": "accepted",
    "compact_hook": {
        "module": "classifier",
        "capture": "forward_pre_input",
        "embedding_shape": list(
            captured["compact_embedding"].shape
        ),
        "logit_shape": list(compact_logit.shape),
    },
    "anatomy_hook": {
        "module": "fusion.0",
        "capture": "forward_pre_input",
        "embedding_shape": list(
            captured["anatomy_fusion_embedding"].shape
        ),
        "logit_shape": list(anatomy_logit.shape),
    },
    "synthetic_input_only": True,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_EMBEDDING_HOOKS")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_EMBEDDING_HOOKS")

del synthetic_compact
del synthetic_anatomy
del compact_logit
del anatomy_logit
captured.clear()

torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE19_EMBEDDING_HOOKS
{
  "phase": "phase19_embedding_hook_contract",
  "status": "accepted",
  "compact_hook": {
    "module": "classifier",
    "capture": "forward_pre_input",
    "embedding_shape": [
      2,
      192
    ],
    "logit_shape": [
      2
    ]
  },
  "anatomy_hook": {
    "module": "fusion.0",
    "capture": "forward_pre_input",
    "embedding_shape": [
      2,
      256
    ],
    "logit_shape": [
      2
    ]
  },
  "synthetic_input_only": true,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "uids_displayed": false
}
END SANITIZED_PHASE19_EMBEDDING_HOOKS


In [40]:
# Cell 33 — Notebook-state structural audit
import json
import numpy as np
import torch

try:
    import pandas as pd
except ImportError:
    pd = None

MATCH_TERMS = (
    "fold",
    "split",
    "nested",
    "fit",
    "monitor",
    "valid",
    "outer",
    "cache",
    "highres",
    "context",
    "label",
    "target",
)

def structural_description(value):
    description = {
        "type": type(value).__name__,
    }

    if isinstance(value, np.ndarray):
        description.update({
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        })
        return description

    if torch.is_tensor(value):
        description.update({
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "device": str(value.device),
        })
        return description

    if pd is not None and isinstance(value, pd.DataFrame):
        description.update({
            "shape": list(value.shape),
            "columns": [str(column) for column in value.columns],
        })
        return description

    if pd is not None and isinstance(value, pd.Series):
        description.update({
            "length": len(value),
            "dtype": str(value.dtype),
            "name": str(value.name),
        })
        return description

    if isinstance(value, dict):
        description.update({
            "length": len(value),
            "key_types": sorted({
                type(key).__name__ for key in value.keys()
            }),
            "value_types": sorted({
                type(item).__name__ for item in value.values()
            }),
        })

        # Schema only; no keys or values are disclosed.
        dict_values = list(value.values())
        if dict_values and all(
            isinstance(item, dict) for item in dict_values
        ):
            description["nested_dictionary"] = True

        return description

    if isinstance(value, (list, tuple)):
        description.update({
            "length": len(value),
            "element_types": sorted({
                type(item).__name__ for item in value
            }),
        })

        if value and all(
            isinstance(item, np.ndarray) for item in value
        ):
            description["element_shapes"] = sorted({
                tuple(item.shape) for item in value
            })

        return description

    if isinstance(value, (int, float, bool, str)):
        description["scalar_value_suppressed"] = True

    return description

state_rows = []

for name, value in sorted(globals().items()):
    lowered = name.casefold()

    if name.startswith("_"):
        continue

    if not any(term in lowered for term in MATCH_TERMS):
        continue

    # Skip callables, modules and classes.
    if callable(value):
        continue

    state_rows.append({
        "name": name,
        "structure": structural_description(value),
    })

report = {
    "phase": "phase19_notebook_state_audit",
    "status": "complete",
    "matching_variable_count": len(state_rows),
    "variables": state_rows,
    "array_values_displayed": False,
    "indices_displayed": False,
    "labels_displayed": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE19_NOTEBOOK_STATE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_NOTEBOOK_STATE")

BEGIN SANITIZED_PHASE19_NOTEBOOK_STATE
{
  "phase": "phase19_notebook_state_audit",
  "status": "complete",
  "matching_variable_count": 73,
  "variables": [
    {
      "name": "BASE_SPLIT_SEED",
      "structure": {
        "type": "int",
        "scalar_value_suppressed": true
      }
    },
    {
      "name": "CONTEXT_CONFIG",
      "structure": {
        "type": "dict",
        "length": 2,
        "key_types": [
          "str"
        ],
        "value_types": [
          "float",
          "tuple"
        ]
      }
    },
    {
      "name": "FOLD_SEARCH_CANDIDATES",
      "structure": {
        "type": "int",
        "scalar_value_suppressed": true
      }
    },
    {
      "name": "HIGHRES_CONFIG",
      "structure": {
        "type": "dict",
        "length": 2,
        "key_types": [
          "str"
        ],
        "value_types": [
          "float",
          "tuple"
        ]
      }
    },
    {
      "name": "INNER_MONITOR_FRACTION",
      "structure": {
        "t

In [43]:
# Replacement Cell 34B — Resolve exact Phase12c crop-key contract
import json
import traceback

import numpy as np
import torch

required_objects = [
    "phase19_y",
    "phase19_partitions",
    "phase19_highres_cache",
    "phase19_fold_manifests",
    "normalize_probability_output",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, {
    "message": "Rerun Cell 34 up to the failed predictor section.",
    "missing_objects": missing_objects,
}

original_predictor = getattr(
    PHASE19_FAMILY_MODULE,
    "_phase19_original_predict_family_batch",
    PHASE19_FAMILY_MODULE.predict_family_batch,
)

PHASE19_FAMILY_MODULE._phase19_original_predict_family_batch = (
    original_predictor
)

base_synthetic_crops = {
    "160": np.zeros((2, 64, 64, 64), dtype=np.float32),
    "192": np.zeros((2, 64, 64, 64), dtype=np.float32),
    "128": np.zeros((2, 80, 80, 80), dtype=np.float32),
    "highres": np.zeros((2, 80, 80, 80), dtype=np.float32),
    "anatomy": np.zeros((2, 80, 80, 80), dtype=np.float32),
}

crop_alias_sources = {
    "160": "160",
    "192": "192",
    "128": "highres",
    "highres": "highres",
    "anatomy": "highres",
}

def infer_alias_source(key):
    text = str(key).casefold().replace("-", "_").replace(" ", "_")

    if "160" in text:
        return "160"

    if "192" in text or "context" in text:
        return "192"

    highres_terms = (
        "128",
        "80",
        "high",
        "hires",
        "anatom",
        "region",
        "token",
    )

    if any(term in text for term in highres_terms):
        return "highres"

    return None

def source_array_for_alias(source_name, crop_dictionary):
    if source_name == "160":
        return crop_dictionary["160"]
    if source_name == "192":
        return crop_dictionary["192"]

    for key in ("highres", "128", "anatomy"):
        if key in crop_dictionary:
            return crop_dictionary[key]

    raise KeyError("highres_source_unavailable")

working_crops = dict(base_synthetic_crops)
added_aliases = []
attempt_diagnostics = []
synthetic_probability = None

for attempt_index in range(12):
    try:
        with torch.inference_mode():
            output = original_predictor(
                working_crops,
                phase19_fold_manifests[0]["family"],
                phase19_fold_manifests[0]["base"],
                PHASE19_BASE_MODELS,
                PHASE19_ANATOMY_MODELS,
                PHASE19_DEVICE,
            )

        synthetic_probability = normalize_probability_output(
            output,
            2,
        )
        break

    except KeyError as error:
        missing_key = error.args[0] if error.args else None
        alias_source = infer_alias_source(missing_key)

        diagnostic = {
            "attempt": attempt_index + 1,
            "error_type": "KeyError",
            "missing_key_type": type(missing_key).__name__,
            "missing_key": (
                str(missing_key)[:80]
                if isinstance(missing_key, (str, int, float))
                else "<non_scalar_key>"
            ),
            "alias_source_resolved": alias_source,
        }
        attempt_diagnostics.append(diagnostic)

        if (
            not isinstance(missing_key, str)
            or missing_key in working_crops
            or alias_source is None
        ):
            break

        source_array = source_array_for_alias(
            alias_source,
            working_crops,
        )
        working_crops[missing_key] = source_array
        crop_alias_sources[missing_key] = alias_source
        added_aliases.append({
            "alias": missing_key,
            "source": alias_source,
        })

    except Exception as error:
        final_frame = traceback.extract_tb(
            error.__traceback__
        )[-1]

        attempt_diagnostics.append({
            "attempt": attempt_index + 1,
            "error_type": type(error).__name__,
            "function": final_frame.name,
            "line_number": final_frame.lineno,
            "message": str(error)[:160],
        })
        break

contract_passed = synthetic_probability is not None

report = {
    "phase": "phase19_predictor_crop_contract_repair",
    "status": "accepted" if contract_passed else "still_blocked",
    "predictor_input_kind": "numpy",
    "initial_crop_aliases": [
        "160",
        "192",
        "128",
        "highres",
        "anatomy",
    ],
    "added_aliases": added_aliases,
    "resolved_alias_count": len(crop_alias_sources),
    "synthetic_probability_shape": (
        list(synthetic_probability.shape)
        if synthetic_probability is not None
        else None
    ),
    "attempt_diagnostics": attempt_diagnostics,
    "synthetic_input_only": True,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_CROP_CONTRACT_REPAIR")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_CROP_CONTRACT_REPAIR")

assert contract_passed, (
    "The missing item is not a recognizable crop alias. "
    "Return the sanitized repair report for source-level diagnosis."
)

# Patch the package predictor so Cell 35's standard crop dictionary is
# automatically enriched with every discovered deployment alias.
def phase19_adaptive_predict_family_batch(
    crops,
    family_manifest,
    base_manifest,
    base_models,
    anatomy_models,
    device,
):
    enriched = dict(crops)

    for alias, source_name in crop_alias_sources.items():
        if alias in enriched:
            continue

        enriched[alias] = source_array_for_alias(
            source_name,
            enriched,
        )

    return original_predictor(
        enriched,
        family_manifest,
        base_manifest,
        base_models,
        anatomy_models,
        device,
    )

PHASE19_FAMILY_MODULE.predict_family_batch = (
    phase19_adaptive_predict_family_batch
)

PHASE19_Y_PRIVATE = phase19_y
PHASE19_PARTITIONS = phase19_partitions
PHASE19_HIGHRES_CACHE = phase19_highres_cache
PHASE19_FOLD_MANIFESTS = phase19_fold_manifests
PHASE19_PREDICT_INPUT_KIND = "numpy"
PHASE19_NORMALIZE_PREDICTIONS = normalize_probability_output
PHASE19_CROP_ALIAS_SOURCES = crop_alias_sources

del base_synthetic_crops
del working_crops
del synthetic_probability

torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE19_CROP_CONTRACT_REPAIR
{
  "phase": "phase19_predictor_crop_contract_repair",
  "status": "accepted",
  "predictor_input_kind": "numpy",
  "initial_crop_aliases": [
    "160",
    "192",
    "128",
    "highres",
    "anatomy"
  ],
  "added_aliases": [
    {
      "alias": "128hr",
      "source": "highres"
    }
  ],
  "resolved_alias_count": 6,
  "synthetic_probability_shape": [
    2
  ],
  "attempt_diagnostics": [
    {
      "attempt": 1,
      "error_type": "KeyError",
      "missing_key_type": "str",
      "missing_key": "128hr",
      "alias_source_resolved": "highres"
    }
  ],
  "synthetic_input_only": true,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "uids_displayed": false
}
END SANITIZED_PHASE19_CROP_CONTRACT_REPAIR


In [44]:
# Cell 35 — Exact Phase12c OOF reconstruction
import json
import time

import numpy as np
import torch
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

OOF_BATCH_SIZE = 8
REFERENCE = {
    "log_loss": 0.307616,
    "auroc": 0.938914,
    "brier": 0.095535,
}

phase12c_oof = np.full(EXPECTED_CASES, np.nan, dtype=np.float64)
fold_reports = []

def build_predictor_crops(indices):
    arrays = {
        "160": PHASE19_LEGACY_CACHE["160"][indices].astype(
            np.float32,
            copy=False,
        ),
        "192": PHASE19_LEGACY_CACHE["192"][indices].astype(
            np.float32,
            copy=False,
        ),
        "128": PHASE19_HIGHRES_CACHE[indices].astype(
            np.float32,
            copy=False,
        ),
        "highres": PHASE19_HIGHRES_CACHE[indices].astype(
            np.float32,
            copy=False,
        ),
        "anatomy": PHASE19_HIGHRES_CACHE[indices].astype(
            np.float32,
            copy=False,
        ),
    }

    if PHASE19_PREDICT_INPUT_KIND == "numpy":
        return arrays

    return {
        key: torch.from_numpy(value).to(PHASE19_DEVICE)
        for key, value in arrays.items()
    }

started = time.perf_counter()

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    valid_indices = partition["outer_valid"]
    fold_predictions = []

    fold_started = time.perf_counter()

    for start in range(0, len(valid_indices), OOF_BATCH_SIZE):
        batch_indices = valid_indices[start:start + OOF_BATCH_SIZE]
        crops = build_predictor_crops(batch_indices)

        with torch.inference_mode():
            output = PHASE19_FAMILY_MODULE.predict_family_batch(
                crops,
                PHASE19_FOLD_MANIFESTS[fold_index]["family"],
                PHASE19_FOLD_MANIFESTS[fold_index]["base"],
                PHASE19_BASE_MODELS,
                PHASE19_ANATOMY_MODELS,
                PHASE19_DEVICE,
            )

        batch_probability = PHASE19_NORMALIZE_PREDICTIONS(
            output,
            len(batch_indices),
        )
        fold_predictions.append(batch_probability)

    fold_probability = np.concatenate(fold_predictions)
    fold_probability = np.clip(
        fold_probability,
        1e-5,
        1.0 - 1e-5,
    )

    assert len(fold_probability) == len(valid_indices)
    assert np.isfinite(fold_probability).all()

    phase12c_oof[valid_indices] = fold_probability

    fold_y = PHASE19_Y_PRIVATE[valid_indices]

    fold_reports.append({
        "fold": fold_index,
        "n": len(valid_indices),
        "log_loss": round(
            float(log_loss(fold_y, fold_probability)),
            6,
        ),
        "auroc": round(
            float(roc_auc_score(fold_y, fold_probability)),
            6,
        ),
        "brier": round(
            float(brier_score_loss(fold_y, fold_probability)),
            6,
        ),
        "elapsed_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    print(
        f"Phase12c OOF fold {fold_index} complete: "
        f"{len(valid_indices)} cases",
        flush=True,
    )

assert np.isfinite(phase12c_oof).all()

metrics = {
    "log_loss": float(
        log_loss(PHASE19_Y_PRIVATE, phase12c_oof)
    ),
    "auroc": float(
        roc_auc_score(PHASE19_Y_PRIVATE, phase12c_oof)
    ),
    "brier": float(
        brier_score_loss(PHASE19_Y_PRIVATE, phase12c_oof)
    ),
}

absolute_deltas = {
    key: abs(metrics[key] - REFERENCE[key])
    for key in REFERENCE
}

# Float16 caches can cause tiny reconstruction drift.
baseline_gate = (
    absolute_deltas["log_loss"] <= 0.0015
    and absolute_deltas["auroc"] <= 0.0015
    and absolute_deltas["brier"] <= 0.0015
)

report = {
    "phase": "phase19_phase12c_oof_reconstruction",
    "status": (
        "accepted" if baseline_gate
        else "reconstruction_mismatch"
    ),
    "metrics": {
        key: round(value, 6)
        for key, value in metrics.items()
    },
    "historical_reference": REFERENCE,
    "absolute_metric_deltas": {
        key: round(value, 8)
        for key, value in absolute_deltas.items()
    },
    "fold_metrics": fold_reports,
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "maximum_allowed_absolute_metric_delta": 0.0015,
    "baseline_reconstruction_gate_passed": baseline_gate,
    "case_level_predictions_exported": False,
    "outer_labels_used_only_for_evaluation": True,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_PHASE12C_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_PHASE12C_OOF")

PHASE19_PHASE12C_OOF_PRIVATE = phase12c_oof
PHASE19_BASELINE_METRICS = metrics
PHASE19_BASELINE_GATE_PASSED = baseline_gate

assert baseline_gate, (
    "Phase12c OOF reconstruction did not match the historical reference. "
)

Phase12c OOF fold 0 complete: 467 cases
Phase12c OOF fold 1 complete: 443 cases
Phase12c OOF fold 2 complete: 452 cases
BEGIN SANITIZED_PHASE19_PHASE12C_OOF
{
  "phase": "phase19_phase12c_oof_reconstruction",
  "status": "accepted",
  "metrics": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "historical_reference": {
    "log_loss": 0.307616,
    "auroc": 0.938914,
    "brier": 0.095535
  },
  "absolute_metric_deltas": {
    "log_loss": 7e-07,
    "auroc": 1.353e-05,
    "brier": 8e-08
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "log_loss": 0.358494,
      "auroc": 0.916174,
      "brier": 0.115499,
      "elapsed_seconds": 8.57
    },
    {
      "fold": 1,
      "n": 443,
      "log_loss": 0.255415,
      "auroc": 0.952183,
      "brier": 0.073923,
      "elapsed_seconds": 7.6
    },
    {
      "fold": 2,
      "n": 452,
      "log_loss": 0.306209,
      "auroc": 0.945716,
      "brier": 0.096091,
      "elapsed_seconds": 7.

In [46]:
# Replacement Cell 36 — Frozen reflection-invariant expert cache
import json
import re
import time

import numpy as np
import torch
import torch.nn as nn

assert PHASE19_BASELINE_GATE_PASSED

FEATURE_BATCH_SIZE = 8
CASE_COUNT = EXPECTED_CASES

# ------------------------------------------------------------
# Model/key discovery
# ------------------------------------------------------------
def flatten_model_dictionary(container, prefix=()):
    rows = []

    if isinstance(container, nn.Module):
        rows.append((prefix, container))
        return rows

    if isinstance(container, dict):
        for key, value in container.items():
            rows.extend(
                flatten_model_dictionary(
                    value,
                    prefix + (key,),
                )
            )
        return rows

    if isinstance(container, (list, tuple)):
        for index, value in enumerate(container):
            rows.extend(
                flatten_model_dictionary(
                    value,
                    prefix + (index,),
                )
            )

    return rows

def recursively_flatten_key_atoms(value):
    atoms = []

    if isinstance(value, (tuple, list)):
        for item in value:
            atoms.extend(recursively_flatten_key_atoms(item))
    else:
        atoms.append(value)

    return atoms

def key_atoms(key_path):
    atoms = []

    for item in key_path:
        atoms.extend(recursively_flatten_key_atoms(item))

    return atoms

def infer_fold_from_key(key_path):
    atoms = key_atoms(key_path)

    explicit_folds = [
        int(item)
        for item in atoms
        if isinstance(item, (int, np.integer))
        and int(item) in {0, 1, 2}
    ]

    if explicit_folds:
        assert len(set(explicit_folds)) == 1, {
            "message": "Multiple fold integers detected.",
            "fold_candidates": sorted(set(explicit_folds)),
        }
        return explicit_folds[0]

    text = " ".join(str(item) for item in atoms).casefold()
    matches = re.findall(r"fold[_\-\s]*([012])", text)

    assert matches, {
        "message": "Could not infer model fold.",
        "flattened_key_types": [
            type(item).__name__ for item in atoms
        ],
    }
    assert len(set(matches)) == 1

    return int(matches[0])

def infer_compact_crop(key_path):
    text = " ".join(
        str(item) for item in key_atoms(key_path)
    ).casefold()

    if "160" in text:
        return "160"

    if "192" in text:
        return "192"

    if "128" in text or "highres" in text or "high_res" in text:
        return "highres"

    raise AssertionError({
        "message": "Could not infer compact-model crop.",
        "model_key_displayed": False,
    })

base_entries = []

for key_path, model in flatten_model_dictionary(
    PHASE19_BASE_MODELS
):
    base_entries.append({
        "fold": infer_fold_from_key(key_path),
        "crop": infer_compact_crop(key_path),
        "model": model,
        "family": "compact",
    })

anatomy_entries = []

for key_path, model in flatten_model_dictionary(
    PHASE19_ANATOMY_MODELS
):
    anatomy_entries.append({
        "fold": infer_fold_from_key(key_path),
        "crop": "highres",
        "model": model,
        "family": "anatomy",
    })

assert len(base_entries) == 15
assert len(anatomy_entries) == 6

fold_contracts = []

for fold_index in range(3):
    fold_base = [
        item for item in base_entries
        if item["fold"] == fold_index
    ]
    fold_anatomy = [
        item for item in anatomy_entries
        if item["fold"] == fold_index
    ]

    crop_counts = {
        crop: sum(
            item["crop"] == crop
            for item in fold_base
        )
        for crop in ("160", "192", "highres")
    }

    assert len(fold_base) == 5
    assert len(fold_anatomy) == 2
    assert crop_counts == {
        "160": 2,
        "192": 1,
        "highres": 2,
    }, {
        "fold": fold_index,
        "crop_counts": crop_counts,
    }

    fold_contracts.append({
        "fold": fold_index,
        "compact_count": len(fold_base),
        "anatomy_count": len(fold_anatomy),
        "compact_crop_counts": crop_counts,
    })

print("Expert-key contracts accepted.", flush=True)

# ------------------------------------------------------------
# Frozen feature extraction
# ------------------------------------------------------------
def cache_for_crop(crop):
    if crop == "160":
        return PHASE19_LEGACY_CACHE["160"]

    if crop == "192":
        return PHASE19_LEGACY_CACHE["192"]

    if crop == "highres":
        return PHASE19_HIGHRES_CACHE

    raise KeyError(crop)

@torch.inference_mode()
def extract_one_expert(model, crop, family):
    source = cache_for_crop(crop)

    if family == "compact":
        hook_module = model.classifier
        expected_dimension = 192
    elif family == "anatomy":
        hook_module = model.fusion[0]
        expected_dimension = 256
    else:
        raise KeyError(family)

    identity_embeddings = []
    reflected_embeddings = []
    identity_logits = []
    reflected_logits = []

    captured = {}

    def capture_pre_input(module, inputs):
        assert len(inputs) == 1
        captured["embedding"] = inputs[0].detach()

    handle = hook_module.register_forward_pre_hook(
        capture_pre_input
    )

    model.eval()

    try:
        for start in range(0, CASE_COUNT, FEATURE_BATCH_SIZE):
            stop = min(
                start + FEATURE_BATCH_SIZE,
                CASE_COUNT,
            )

            batch_numpy = source[start:stop].astype(
                np.float32,
                copy=False,
            )

            batch = torch.from_numpy(
                batch_numpy
            )[:, None].to(
                PHASE19_DEVICE,
                non_blocking=True,
            )

            # Canonical RAS first spatial axis is left-right.
            reflected = torch.flip(batch, dims=(2,))

            with torch.autocast(
                device_type=PHASE19_DEVICE.type,
                dtype=torch.float16,
                enabled=PHASE19_DEVICE.type == "cuda",
            ):
                identity_output = model(batch)
                identity_embedding = (
                    captured["embedding"].float().clone()
                )

                reflected_output = model(reflected)
                reflected_embedding = (
                    captured["embedding"].float().clone()
                )

            assert identity_embedding.ndim == 2
            assert reflected_embedding.ndim == 2
            assert identity_embedding.shape[1] == expected_dimension
            assert reflected_embedding.shape[1] == expected_dimension

            identity_embeddings.append(
                identity_embedding.cpu().numpy()
            )
            reflected_embeddings.append(
                reflected_embedding.cpu().numpy()
            )
            identity_logits.append(
                identity_output.float().cpu().numpy().reshape(-1)
            )
            reflected_logits.append(
                reflected_output.float().cpu().numpy().reshape(-1)
            )

            del batch
            del reflected
            del identity_output
            del reflected_output
            del identity_embedding
            del reflected_embedding

    finally:
        handle.remove()

    identity_embeddings = np.concatenate(
        identity_embeddings,
        axis=0,
    )
    reflected_embeddings = np.concatenate(
        reflected_embeddings,
        axis=0,
    )
    identity_logits = np.concatenate(identity_logits)
    reflected_logits = np.concatenate(reflected_logits)

    assert identity_embeddings.shape == (
        CASE_COUNT,
        expected_dimension,
    )
    assert reflected_embeddings.shape == (
        CASE_COUNT,
        expected_dimension,
    )
    assert identity_logits.shape == (CASE_COUNT,)
    assert reflected_logits.shape == (CASE_COUNT,)

    mean_embedding = (
        0.5 * (
            identity_embeddings
            + reflected_embeddings
        )
    ).astype(np.float16)

    change_embedding = np.abs(
        identity_embeddings
        - reflected_embeddings
    ).astype(np.float16)

    mean_logit = (
        0.5 * (
            identity_logits
            + reflected_logits
        )
    ).astype(np.float32)

    logit_change = np.abs(
        identity_logits
        - reflected_logits
    ).astype(np.float32)

    assert np.isfinite(mean_embedding).all()
    assert np.isfinite(change_embedding).all()
    assert np.isfinite(mean_logit).all()
    assert np.isfinite(logit_change).all()

    return {
        "mean_embedding": mean_embedding,
        "change_embedding": change_embedding,
        "mean_logit": mean_logit,
        "logit_change": logit_change,
        "dimension": expected_dimension,
    }

phase19_expert_cache = {}
extraction_reports = []
started = time.perf_counter()

for fold_index in range(3):
    fold_started = time.perf_counter()

    fold_entries = [
        item
        for item in base_entries + anatomy_entries
        if item["fold"] == fold_index
    ]

    # Stable semantic order:
    # compact-160, compact-192, compact-highres, anatomy-highres.
    fold_entries = sorted(
        fold_entries,
        key=lambda item: (
            0 if item["family"] == "compact" else 1,
            {
                "160": 0,
                "192": 1,
                "highres": 2,
            }[item["crop"]],
        ),
    )

    assert len(fold_entries) == 7

    fold_cache = []

    for expert_index, entry in enumerate(fold_entries):
        expert_started = time.perf_counter()

        features = extract_one_expert(
            model=entry["model"],
            crop=entry["crop"],
            family=entry["family"],
        )

        fold_cache.append({
            "family": entry["family"],
            "crop": entry["crop"],
            **features,
        })

        print(
            f"Fold {fold_index} expert "
            f"{expert_index + 1}/7 complete: "
            f"{time.perf_counter() - expert_started:.1f}s",
            flush=True,
        )

    phase19_expert_cache[fold_index] = fold_cache

    extraction_reports.append({
        "fold": fold_index,
        "expert_count": len(fold_cache),
        "compact_expert_count": sum(
            item["family"] == "compact"
            for item in fold_cache
        ),
        "anatomy_expert_count": sum(
            item["family"] == "anatomy"
            for item in fold_cache
        ),
        "embedding_dimensions": [
            item["dimension"] for item in fold_cache
        ],
        "elapsed_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    torch.cuda.empty_cache()

total_cache_bytes = 0

for fold_cache in phase19_expert_cache.values():
    for expert in fold_cache:
        total_cache_bytes += expert["mean_embedding"].nbytes
        total_cache_bytes += expert["change_embedding"].nbytes
        total_cache_bytes += expert["mean_logit"].nbytes
        total_cache_bytes += expert["logit_change"].nbytes

report = {
    "phase": "phase19_reflection_invariant_expert_cache",
    "status": "complete",
    "case_count": CASE_COUNT,
    "fold_count": 3,
    "experts_per_fold": 7,
    "total_frozen_experts": 21,
    "fold_contracts": fold_contracts,
    "reflection_operation": "canonical_RAS_left_right_axis",
    "retained_representation": [
        "mean_identity_reflection_embedding",
        "absolute_identity_reflection_embedding_change",
        "mean_identity_reflection_logit",
        "absolute_identity_reflection_logit_change",
    ],
    "folds": extraction_reports,
    "cache_ram_gb": round(
        total_cache_bytes / 1e9,
        4,
    ),
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "frozen_weights_updated": False,
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_EXPERT_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_EXPERT_CACHE")

PHASE19_EXPERT_CACHE_PRIVATE = phase19_expert_cache

Expert-key contracts accepted.
Fold 0 expert 1/7 complete: 3.1s
Fold 0 expert 2/7 complete: 3.2s
Fold 0 expert 3/7 complete: 3.1s
Fold 0 expert 4/7 complete: 4.6s
Fold 0 expert 5/7 complete: 4.6s
Fold 0 expert 6/7 complete: 5.5s
Fold 0 expert 7/7 complete: 5.9s
Fold 1 expert 1/7 complete: 3.1s
Fold 1 expert 2/7 complete: 3.1s
Fold 1 expert 3/7 complete: 3.1s
Fold 1 expert 4/7 complete: 4.7s
Fold 1 expert 5/7 complete: 4.7s
Fold 1 expert 6/7 complete: 5.5s
Fold 1 expert 7/7 complete: 5.6s
Fold 2 expert 1/7 complete: 3.1s
Fold 2 expert 2/7 complete: 3.2s
Fold 2 expert 3/7 complete: 3.2s
Fold 2 expert 4/7 complete: 4.6s
Fold 2 expert 5/7 complete: 4.7s
Fold 2 expert 6/7 complete: 5.6s
Fold 2 expert 7/7 complete: 5.7s
BEGIN SANITIZED_PHASE19_EXPERT_CACHE
{
  "phase": "phase19_reflection_invariant_expert_cache",
  "status": "complete",
  "case_count": 1362,
  "fold_count": 3,
  "experts_per_fold": 7,
  "total_frozen_experts": 21,
  "fold_contracts": [
    {
      "fold": 0,
      "compact_c

In [47]:
# Cell 37 — Cross-fitted expert tokens + architecture contract
import json
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

EXPECTED_EXPERT_DIMS = [192, 192, 192, 192, 192, 256, 256]
RESIDUAL_BOUND = 0.75
MODEL_DIMENSION = 96

assert PHASE19_BASELINE_GATE_PASSED
assert len(PHASE19_EXPERT_CACHE_PRIVATE) == 3

# ------------------------------------------------------------
# Assign every case to the fold where it was genuinely held out.
# ------------------------------------------------------------
phase19_owner_fold = np.full(
    EXPECTED_CASES,
    -1,
    dtype=np.int64,
)

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    valid_indices = partition["outer_valid"]

    assert np.all(phase19_owner_fold[valid_indices] == -1)
    phase19_owner_fold[valid_indices] = fold_index

assert np.all(phase19_owner_fold >= 0)
assert np.array_equal(
    np.bincount(phase19_owner_fold, minlength=3),
    np.asarray([467, 443, 452]),
)

# ------------------------------------------------------------
# Build seven cross-fitted expert feature matrices.
#
# For each expert:
#   mean embedding
#   log(1 + reflection-change embedding)
#   mean reflected/identity logit
#   log(1 + logit reflection change)
# ------------------------------------------------------------
phase19_expert_features = []
expert_feature_dimensions = []

for expert_index, expected_embedding_dimension in enumerate(
    EXPECTED_EXPERT_DIMS
):
    output_dimension = 2 * expected_embedding_dimension + 2

    features = np.empty(
        (EXPECTED_CASES, output_dimension),
        dtype=np.float32,
    )

    for fold_index in range(3):
        case_indices = np.flatnonzero(
            phase19_owner_fold == fold_index
        )

        expert = PHASE19_EXPERT_CACHE_PRIVATE[
            fold_index
        ][expert_index]

        assert expert["dimension"] == expected_embedding_dimension

        mean_embedding = expert["mean_embedding"][
            case_indices
        ].astype(np.float32)

        change_embedding = np.log1p(
            expert["change_embedding"][
                case_indices
            ].astype(np.float32)
        )

        mean_logit = expert["mean_logit"][
            case_indices
        ].astype(np.float32)[:, None]

        logit_change = np.log1p(
            expert["logit_change"][
                case_indices
            ].astype(np.float32)
        )[:, None]

        features[case_indices] = np.concatenate(
            [
                mean_embedding,
                change_embedding,
                mean_logit,
                logit_change,
            ],
            axis=1,
        )

    assert np.isfinite(features).all()

    phase19_expert_features.append(features)
    expert_feature_dimensions.append(output_dimension)

phase19_anchor_probability = np.clip(
    PHASE19_PHASE12C_OOF_PRIVATE,
    1e-5,
    1.0 - 1e-5,
)

phase19_anchor_logit = np.log(
    phase19_anchor_probability
    / (1.0 - phase19_anchor_probability)
).astype(np.float32)

# ------------------------------------------------------------
# Model-token residual adapter
# ------------------------------------------------------------
class Phase19ChampionResidualAdapter(nn.Module):
    def __init__(
        self,
        expert_dimensions,
        normalization_means,
        normalization_stds,
        model_dimension=96,
        residual_bound=0.75,
        dropout=0.15,
        expert_dropout=0.12,
    ):
        super().__init__()

        self.expert_count = len(expert_dimensions)
        self.model_dimension = int(model_dimension)
        self.residual_bound = float(residual_bound)
        self.expert_dropout = float(expert_dropout)

        self.projections = nn.ModuleList()

        for index, dimension in enumerate(expert_dimensions):
            self.register_buffer(
                f"feature_mean_{index}",
                torch.as_tensor(
                    normalization_means[index],
                    dtype=torch.float32,
                ),
            )
            self.register_buffer(
                f"feature_std_{index}",
                torch.as_tensor(
                    normalization_stds[index],
                    dtype=torch.float32,
                ),
            )

            self.projections.append(
                nn.Sequential(
                    nn.Linear(dimension, model_dimension),
                    nn.LayerNorm(model_dimension),
                    nn.SiLU(),
                    nn.Dropout(dropout),
                )
            )

        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, model_dimension)
        )
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.expert_count + 1,
                model_dimension,
            )
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=4,
            dim_feedforward=2 * model_dimension,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=2,
            norm=nn.LayerNorm(model_dimension),
            enable_nested_tensor=False,
        )

        # CLS plus five conservative scalar summaries.
        self.residual_head = nn.Sequential(
            nn.LayerNorm(model_dimension + 5),
            nn.Linear(model_dimension + 5, 64),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.position_embedding, std=0.02)

        # Start at the exact Phase12c baseline.
        nn.init.zeros_(self.residual_head[-1].weight)
        nn.init.zeros_(self.residual_head[-1].bias)

    def forward(self, expert_features, anchor_logit):
        assert len(expert_features) == self.expert_count

        tokens = []
        raw_mean_logits = []
        raw_logit_changes = []

        for index, raw_features in enumerate(expert_features):
            raw_mean_logits.append(raw_features[:, -2])
            raw_logit_changes.append(raw_features[:, -1])

            mean = getattr(self, f"feature_mean_{index}")
            std = getattr(self, f"feature_std_{index}")

            standardized = (raw_features - mean) / std
            standardized = torch.clamp(
                standardized,
                -8.0,
                8.0,
            )

            token = self.projections[index](standardized)
            tokens.append(token)

        tokens = torch.stack(tokens, dim=1)

        if self.training and self.expert_dropout > 0:
            keep = torch.rand(
                tokens.shape[:2],
                device=tokens.device,
            ) >= self.expert_dropout

            # Never remove all experts from a case.
            empty = ~keep.any(dim=1)
            if empty.any():
                keep[empty, 0] = True

            tokens = tokens * keep.unsqueeze(-1)

        cls = self.cls_token.expand(
            tokens.shape[0],
            -1,
            -1,
        )

        sequence = torch.cat([cls, tokens], dim=1)
        sequence = sequence + self.position_embedding
        encoded = self.transformer(sequence)
        pooled = encoded[:, 0]

        mean_logits = torch.stack(
            raw_mean_logits,
            dim=1,
        )
        reflection_changes = torch.stack(
            raw_logit_changes,
            dim=1,
        )

        scalar_summary = torch.stack(
            [
                anchor_logit,
                mean_logits.mean(dim=1),
                mean_logits.std(dim=1, unbiased=False),
                mean_logits.amax(dim=1)
                - mean_logits.amin(dim=1),
                reflection_changes.mean(dim=1),
            ],
            dim=1,
        )

        head_input = torch.cat(
            [pooled, scalar_summary],
            dim=1,
        )

        unbounded = self.residual_head(
            head_input
        ).squeeze(1)

        residual = self.residual_bound * torch.tanh(
            unbounded
        )

        corrected_logit = anchor_logit + residual
        return corrected_logit, residual

def compute_fit_normalization(fit_indices):
    means = []
    stds = []

    for features in phase19_expert_features:
        fit_values = features[fit_indices]

        mean = fit_values.mean(axis=0).astype(np.float32)
        std = fit_values.std(axis=0).astype(np.float32)
        std = np.maximum(std, 1e-3)

        means.append(mean)
        stds.append(std)

    return means, stds

# ------------------------------------------------------------
# Synthetic backward contract
# ------------------------------------------------------------
contract_fit_indices = PHASE19_PARTITIONS[0]["fit"]
contract_means, contract_stds = compute_fit_normalization(
    contract_fit_indices
)

contract_model = Phase19ChampionResidualAdapter(
    expert_dimensions=expert_feature_dimensions,
    normalization_means=contract_means,
    normalization_stds=contract_stds,
    model_dimension=MODEL_DIMENSION,
    residual_bound=RESIDUAL_BOUND,
).to(PHASE19_DEVICE)

contract_batch_indices = contract_fit_indices[:8]

contract_experts = [
    torch.from_numpy(
        features[contract_batch_indices]
    ).to(PHASE19_DEVICE)
    for features in phase19_expert_features
]

contract_anchor = torch.from_numpy(
    phase19_anchor_logit[contract_batch_indices]
).to(PHASE19_DEVICE)

contract_target = torch.from_numpy(
    PHASE19_Y_PRIVATE[contract_batch_indices].astype(
        np.float32
    )
).to(PHASE19_DEVICE)

contract_model.train()
corrected_logit, residual = contract_model(
    contract_experts,
    contract_anchor,
)

contract_loss = (
    F.binary_cross_entropy_with_logits(
        corrected_logit,
        contract_target,
    )
    + 0.015 * residual.square().mean()
)

contract_loss.backward()

gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().square().sum()
        for parameter in contract_model.parameters()
        if parameter.grad is not None
    )
)

parameter_count = sum(
    parameter.numel()
    for parameter in contract_model.parameters()
)

report = {
    "phase": "phase19_adapter_architecture",
    "status": "accepted",
    "model": "Phase19ChampionResidualAdapter",
    "parameter_count": parameter_count,
    "cross_fitted_case_count": EXPECTED_CASES,
    "expert_count": len(phase19_expert_features),
    "expert_embedding_dimensions": EXPECTED_EXPERT_DIMS,
    "adapter_input_dimensions": expert_feature_dimensions,
    "model_dimension": MODEL_DIMENSION,
    "transformer_layers": 2,
    "attention_heads": 4,
    "residual_bound": RESIDUAL_BOUND,
    "corrected_logit_shape": list(corrected_logit.shape),
    "residual_shape": list(residual.shape),
    "maximum_observed_residual": round(
        float(residual.detach().abs().max().cpu()),
        6,
    ),
    "gradient_norm": round(
        float(gradient_norm.cpu()),
        6,
    ),
    "starts_at_exact_phase12c_baseline": True,
    "reflection_invariant_inputs": True,
    "backward_contract_passed": bool(
        torch.isfinite(gradient_norm)
    ),
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_ADAPTER_ARCHITECTURE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_ADAPTER_ARCHITECTURE")

PHASE19_OWNER_FOLD_PRIVATE = phase19_owner_fold
PHASE19_STACKED_FEATURES_PRIVATE = phase19_expert_features
PHASE19_EXPERT_FEATURE_DIMENSIONS = expert_feature_dimensions
PHASE19_ANCHOR_LOGIT_PRIVATE = phase19_anchor_logit
PHASE19_ADAPTER_CLASS = Phase19ChampionResidualAdapter
PHASE19_COMPUTE_NORMALIZATION = compute_fit_normalization

del contract_model
del contract_experts
del contract_anchor
del contract_target
del corrected_logit
del residual

torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE19_ADAPTER_ARCHITECTURE
{
  "phase": "phase19_adapter_architecture",
  "status": "accepted",
  "model": "Phase19ChampionResidualAdapter",
  "parameter_count": 443403,
  "cross_fitted_case_count": 1362,
  "expert_count": 7,
  "expert_embedding_dimensions": [
    192,
    192,
    192,
    192,
    192,
    256,
    256
  ],
  "adapter_input_dimensions": [
    386,
    386,
    386,
    386,
    386,
    514,
    514
  ],
  "model_dimension": 96,
  "transformer_layers": 2,
  "attention_heads": 4,
  "residual_bound": 0.75,
  "corrected_logit_shape": [
    8
  ],
  "residual_shape": [
    8
  ],
  "maximum_observed_residual": 0.0,
  "gradient_norm": 0.058955,
  "starts_at_exact_phase12c_baseline": true,
  "reflection_invariant_inputs": true,
  "backward_contract_passed": true,
  "case_level_features_exported": false,
  "test_or_smoke_data_read": false,
  "uids_displayed": false
}
END SANITIZED_PHASE19_ADAPTER_ARCHITECTURE


In [48]:
# Cell 38 — Leakage-safe Phase19 training
import copy
import json
import random
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import log_loss

PHASE19_TRAINING_CONFIG = {
    "seeds": [2061901, 2061902, 2061903],
    "batch_size": 64,
    "maximum_epochs": 220,
    "patience": 30,
    "learning_rate": 3e-4,
    "weight_decay": 2e-3,
    "residual_penalty": 0.015,
    "gradient_clip": 2.0,
    "alpha_grid": [0.0, 0.25, 0.50, 0.75, 1.0],
}

class Phase19FeatureDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = self.indices[item]

        experts = tuple(
            torch.from_numpy(features[index])
            for features in PHASE19_STACKED_FEATURES_PRIVATE
        )

        anchor = torch.tensor(
            PHASE19_ANCHOR_LOGIT_PRIVATE[index],
            dtype=torch.float32,
        )
        target = torch.tensor(
            PHASE19_Y_PRIVATE[index],
            dtype=torch.float32,
        )

        return experts, anchor, target

def deterministic_loader(indices, training, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        Phase19FeatureDataset(indices),
        batch_size=PHASE19_TRAINING_CONFIG["batch_size"],
        shuffle=training,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        generator=generator,
    )

@torch.inference_mode()
def predict_adapter_delta(model, indices):
    loader = deterministic_loader(
        indices,
        training=False,
        seed=0,
    )

    model.eval()
    deltas = []

    for experts, anchor, _ in loader:
        experts = [
            value.to(
                PHASE19_DEVICE,
                non_blocking=True,
            )
            for value in experts
        ]
        anchor = anchor.to(
            PHASE19_DEVICE,
            non_blocking=True,
        )

        _, residual = model(experts, anchor)
        deltas.append(
            residual.float().cpu().numpy()
        )

    return np.concatenate(deltas)

def train_one_adapter(
    fold_index,
    fit_indices,
    monitor_indices,
    seed,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    means, stds = PHASE19_COMPUTE_NORMALIZATION(
        fit_indices
    )

    model = PHASE19_ADAPTER_CLASS(
        expert_dimensions=PHASE19_EXPERT_FEATURE_DIMENSIONS,
        normalization_means=means,
        normalization_stds=stds,
        model_dimension=MODEL_DIMENSION,
        residual_bound=RESIDUAL_BOUND,
        dropout=0.15,
        expert_dropout=0.12,
    ).to(PHASE19_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE19_TRAINING_CONFIG["learning_rate"],
        weight_decay=PHASE19_TRAINING_CONFIG["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=PHASE19_TRAINING_CONFIG["maximum_epochs"],
        eta_min=3e-5,
    )

    train_loader = deterministic_loader(
        fit_indices,
        training=True,
        seed=seed,
    )

    best_monitor_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    started = time.perf_counter()

    monitor_anchor = PHASE19_ANCHOR_LOGIT_PRIVATE[
        monitor_indices
    ]
    monitor_y = PHASE19_Y_PRIVATE[monitor_indices]

    for epoch in range(
        1,
        PHASE19_TRAINING_CONFIG["maximum_epochs"] + 1,
    ):
        model.train()
        epoch_losses = []

        for experts, anchor, target in train_loader:
            experts = [
                value.to(
                    PHASE19_DEVICE,
                    non_blocking=True,
                )
                for value in experts
            ]
            anchor = anchor.to(
                PHASE19_DEVICE,
                non_blocking=True,
            )
            target = target.to(
                PHASE19_DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            corrected_logit, residual = model(
                experts,
                anchor,
            )

            classification_loss = (
                F.binary_cross_entropy_with_logits(
                    corrected_logit,
                    target,
                )
            )

            residual_regularization = (
                PHASE19_TRAINING_CONFIG[
                    "residual_penalty"
                ]
                * residual.square().mean()
            )

            loss = (
                classification_loss
                + residual_regularization
            )
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE19_TRAINING_CONFIG["gradient_clip"],
            )

            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))

        scheduler.step()

        monitor_delta = predict_adapter_delta(
            model,
            monitor_indices,
        )

        monitor_probability = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    monitor_anchor + monitor_delta,
                    -30.0,
                    30.0,
                )
            )
        )
        monitor_probability = np.clip(
            monitor_probability,
            1e-5,
            1.0 - 1e-5,
        )

        monitor_loss = float(
            log_loss(monitor_y, monitor_probability)
        )

        if monitor_loss < best_monitor_loss - 1e-5:
            best_monitor_loss = monitor_loss
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 20 == 0:
            print(
                f"Phase19 fold {fold_index} seed {seed} "
                f"epoch {epoch:03d}: "
                f"train={np.mean(epoch_losses):.4f}, "
                f"monitor={monitor_loss:.4f}",
                flush=True,
            )

        if (
            epochs_without_improvement
            >= PHASE19_TRAINING_CONFIG["patience"]
        ):
            break

    assert best_state is not None
    model.load_state_dict(best_state, strict=True)
    model.eval()

    reconstruction_error = 0.0

    for key, value in model.state_dict().items():
        reconstruction_error = max(
            reconstruction_error,
            float(
                (
                    value.detach().cpu()
                    - best_state[key]
                ).abs().max()
            ),
        )

    return {
        "model": model,
        "state_dict": best_state,
        "best_epoch": best_epoch,
        "epochs_run": epoch,
        "best_monitor_log_loss": best_monitor_loss,
        "training_seconds": time.perf_counter() - started,
        "checkpoint_reconstruction_error": reconstruction_error,
    }

phase19_oof = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

phase19_fold_states = []
phase19_training_reports = []

training_started = time.perf_counter()

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    fit_indices = partition["fit"]
    monitor_indices = partition["monitor"]
    outer_indices = partition["outer_valid"]

    seed_results = []
    monitor_seed_deltas = []
    outer_seed_deltas = []

    fold_started = time.perf_counter()

    for seed in PHASE19_TRAINING_CONFIG["seeds"]:
        result = train_one_adapter(
            fold_index=fold_index,
            fit_indices=fit_indices,
            monitor_indices=monitor_indices,
            seed=seed,
        )

        monitor_seed_deltas.append(
            predict_adapter_delta(
                result["model"],
                monitor_indices,
            )
        )
        outer_seed_deltas.append(
            predict_adapter_delta(
                result["model"],
                outer_indices,
            )
        )

        seed_results.append(result)

        print(
            f"Phase19 fold {fold_index} seed {seed} frozen: "
            f"best_epoch={result['best_epoch']}, "
            f"monitor={result['best_monitor_log_loss']:.6f}",
            flush=True,
        )

    mean_monitor_delta = np.mean(
        np.stack(monitor_seed_deltas),
        axis=0,
    )
    mean_outer_delta = np.mean(
        np.stack(outer_seed_deltas),
        axis=0,
    )

    monitor_anchor = PHASE19_ANCHOR_LOGIT_PRIVATE[
        monitor_indices
    ]
    outer_anchor = PHASE19_ANCHOR_LOGIT_PRIVATE[
        outer_indices
    ]

    monitor_y = PHASE19_Y_PRIVATE[monitor_indices]

    alpha_rows = []

    for alpha in PHASE19_TRAINING_CONFIG["alpha_grid"]:
        probability = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    monitor_anchor
                    + alpha * mean_monitor_delta,
                    -30.0,
                    30.0,
                )
            )
        )
        probability = np.clip(
            probability,
            1e-5,
            1.0 - 1e-5,
        )

        alpha_rows.append({
            "alpha": float(alpha),
            "monitor_log_loss": float(
                log_loss(monitor_y, probability)
            ),
        })

    # Ties favor the smaller correction.
    alpha_rows.sort(
        key=lambda item: (
            round(item["monitor_log_loss"], 10),
            item["alpha"],
        )
    )
    selected_alpha = alpha_rows[0]["alpha"]

    outer_probability = 1.0 / (
        1.0
        + np.exp(
            -np.clip(
                outer_anchor
                + selected_alpha * mean_outer_delta,
                -30.0,
                30.0,
            )
        )
    )
    outer_probability = np.clip(
        outer_probability,
        1e-5,
        1.0 - 1e-5,
    )

    phase19_oof[outer_indices] = outer_probability

    phase19_fold_states.append({
        "fold": fold_index,
        "alpha": selected_alpha,
        "seed_states": [
            {
                "seed": seed,
                "state_dict": result["state_dict"],
                "best_epoch": result["best_epoch"],
            }
            for seed, result in zip(
                PHASE19_TRAINING_CONFIG["seeds"],
                seed_results,
            )
        ],
    })

    baseline_monitor_loss = float(
        log_loss(
            monitor_y,
            PHASE19_PHASE12C_OOF_PRIVATE[
                monitor_indices
            ],
        )
    )

    phase19_training_reports.append({
        "fold": fold_index,
        "fit_n": len(fit_indices),
        "monitor_n": len(monitor_indices),
        "outer_valid_n": len(outer_indices),
        "selected_alpha": selected_alpha,
        "baseline_monitor_log_loss": baseline_monitor_loss,
        "selected_monitor_log_loss": alpha_rows[0][
            "monitor_log_loss"
        ],
        "monitor_improvement": (
            baseline_monitor_loss
            - alpha_rows[0]["monitor_log_loss"]
        ),
        "seed_best_epochs": [
            result["best_epoch"]
            for result in seed_results
        ],
        "seed_best_monitor_losses": [
            result["best_monitor_log_loss"]
            for result in seed_results
        ],
        "maximum_checkpoint_reconstruction_error": max(
            result["checkpoint_reconstruction_error"]
            for result in seed_results
        ),
        "training_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    for result in seed_results:
        del result["model"]

    torch.cuda.empty_cache()

assert np.isfinite(phase19_oof).all()

report = {
    "phase": "phase19_adapter_training",
    "status": "complete",
    "configuration": PHASE19_TRAINING_CONFIG,
    "folds": phase19_training_reports,
    "total_training_seconds": round(
        time.perf_counter() - training_started,
        2,
    ),
    "outer_labels_used_for_training_or_selection": False,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_ADAPTER_TRAINING")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_ADAPTER_TRAINING")

PHASE19_ADAPTER_OOF_PRIVATE = phase19_oof
PHASE19_ADAPTER_STATES_PRIVATE = phase19_fold_states
PHASE19_ADAPTER_TRAINING_REPORTS = phase19_training_reports

Phase19 fold 0 seed 2061901 epoch 001: train=0.2757, monitor=0.2749
Phase19 fold 0 seed 2061901 epoch 020: train=0.2263, monitor=0.2864
Phase19 fold 0 seed 2061901 frozen: best_epoch=4, monitor=0.271255
Phase19 fold 0 seed 2061902 epoch 001: train=0.2823, monitor=0.2753
Phase19 fold 0 seed 2061902 epoch 020: train=0.2248, monitor=0.2659
Phase19 fold 0 seed 2061902 epoch 040: train=0.1999, monitor=0.2923
Phase19 fold 0 seed 2061902 frozen: best_epoch=20, monitor=0.265898
Phase19 fold 0 seed 2061903 epoch 001: train=0.2769, monitor=0.2753
Phase19 fold 0 seed 2061903 epoch 020: train=0.2231, monitor=0.2785
Phase19 fold 0 seed 2061903 epoch 040: train=0.2048, monitor=0.3087
Phase19 fold 0 seed 2061903 frozen: best_epoch=13, monitor=0.260449
Phase19 fold 1 seed 2061901 epoch 001: train=0.3476, monitor=0.2165
Phase19 fold 1 seed 2061901 epoch 020: train=0.2606, monitor=0.2081
Phase19 fold 1 seed 2061901 epoch 040: train=0.2399, monitor=0.2099
Phase19 fold 1 seed 2061901 frozen: best_epoch=14

In [49]:
# Cell 39 — Phase19 promotion decision
import json

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

def evaluate_probabilities(labels, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )

    logits = np.log(
        probabilities / (1.0 - probabilities)
    )

    calibration = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration.fit(
        logits.reshape(-1, 1),
        labels,
    )

    return {
        "log_loss": float(
            log_loss(labels, probabilities)
        ),
        "auroc": float(
            roc_auc_score(labels, probabilities)
        ),
        "brier": float(
            brier_score_loss(labels, probabilities)
        ),
        "calibration_intercept": float(
            calibration.intercept_[0]
        ),
        "calibration_slope": float(
            calibration.coef_[0, 0]
        ),
        "mean_probability": float(
            probabilities.mean()
        ),
    }

baseline_metrics = evaluate_probabilities(
    PHASE19_Y_PRIVATE,
    PHASE19_PHASE12C_OOF_PRIVATE,
)

adapter_metrics = evaluate_probabilities(
    PHASE19_Y_PRIVATE,
    PHASE19_ADAPTER_OOF_PRIVATE,
)

fold_metrics = []
fold_wins = 0
worst_fold_excess = -float("inf")

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    indices = partition["outer_valid"]
    labels = PHASE19_Y_PRIVATE[indices]

    baseline_fold = evaluate_probabilities(
        labels,
        PHASE19_PHASE12C_OOF_PRIVATE[indices],
    )
    adapter_fold = evaluate_probabilities(
        labels,
        PHASE19_ADAPTER_OOF_PRIVATE[indices],
    )

    improvement = (
        baseline_fold["log_loss"]
        - adapter_fold["log_loss"]
    )
    excess = (
        adapter_fold["log_loss"]
        - baseline_fold["log_loss"]
    )

    if improvement > 0:
        fold_wins += 1

    worst_fold_excess = max(
        worst_fold_excess,
        excess,
    )

    fold_metrics.append({
        "fold": fold_index,
        "baseline_log_loss": round(
            baseline_fold["log_loss"],
            6,
        ),
        "phase19_log_loss": round(
            adapter_fold["log_loss"],
            6,
        ),
        "log_loss_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_fold["auroc"],
            6,
        ),
        "phase19_auroc": round(
            adapter_fold["auroc"],
            6,
        ),
        "selected_alpha": PHASE19_ADAPTER_STATES_PRIVATE[
            fold_index
        ]["alpha"],
    })

log_loss_gain = (
    baseline_metrics["log_loss"]
    - adapter_metrics["log_loss"]
)
auroc_gain = (
    adapter_metrics["auroc"]
    - baseline_metrics["auroc"]
)
brier_excess = (
    adapter_metrics["brier"]
    - baseline_metrics["brier"]
)

# Major acquisition-group harm audit.
group_columns = []

for column in case_df.columns:
    values = np.asarray(case_df[column])

    if values.ndim != 1 or len(values) != EXPECTED_CASES:
        continue

    unique_count = len(np.unique(values))
    lowered = str(column).casefold()

    if unique_count == 15 and (
        "group" in lowered
        or "acquisition" in lowered
    ):
        score = (
            10 * ("acquisition" in lowered)
            + 5 * ("group" in lowered)
        )
        group_columns.append((score, str(column), values))

major_group_reports = []
maximum_major_group_harm = 0.0

if group_columns:
    group_columns.sort(key=lambda item: (-item[0], item[1]))
    _, group_column_name, group_values = group_columns[0]

    for group in np.unique(group_values):
        indices = np.flatnonzero(group_values == group)

        if len(indices) < 30:
            continue

        labels = PHASE19_Y_PRIVATE[indices]

        if len(np.unique(labels)) < 2:
            continue

        baseline_group_loss = float(
            log_loss(
                labels,
                PHASE19_PHASE12C_OOF_PRIVATE[indices],
            )
        )
        adapter_group_loss = float(
            log_loss(
                labels,
                PHASE19_ADAPTER_OOF_PRIVATE[indices],
            )
        )

        harm = adapter_group_loss - baseline_group_loss
        maximum_major_group_harm = max(
            maximum_major_group_harm,
            harm,
        )

        major_group_reports.append({
            "group": int(group)
            if np.issubdtype(
                np.asarray(group).dtype,
                np.integer,
            )
            else str(group),
            "n": len(indices),
            "baseline_log_loss": round(
                baseline_group_loss,
                6,
            ),
            "phase19_log_loss": round(
                adapter_group_loss,
                6,
            ),
            "log_loss_change": round(harm, 6),
        })
else:
    group_column_name = None

promotion_thresholds = {
    "minimum_log_loss_gain": 0.003,
    "minimum_auroc_gain": 0.002,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.005,
    "maximum_brier_excess": 0.001,
    "calibration_slope_range": [0.8, 1.2],
    "maximum_major_group_harm": 0.015,
}

promotion_passed = (
    log_loss_gain
    >= promotion_thresholds["minimum_log_loss_gain"]
    and auroc_gain
    >= promotion_thresholds["minimum_auroc_gain"]
    and fold_wins
    >= promotion_thresholds["minimum_fold_wins"]
    and worst_fold_excess
    <= promotion_thresholds["maximum_worst_fold_excess"]
    and brier_excess
    <= promotion_thresholds["maximum_brier_excess"]
    and 0.8
    <= adapter_metrics["calibration_slope"]
    <= 1.2
    and maximum_major_group_harm
    <= promotion_thresholds["maximum_major_group_harm"]
)

report = {
    "phase": "phase19_champion_residual_adapter",
    "status": (
        "promoted"
        if promotion_passed
        else "not_promoted"
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "phase19": {
        key: round(value, 6)
        for key, value in adapter_metrics.items()
    },
    "improvements": {
        "log_loss_gain": round(log_loss_gain, 6),
        "auroc_gain": round(auroc_gain, 6),
        "brier_excess": round(brier_excess, 6),
        "fold_wins": fold_wins,
        "worst_fold_excess": round(
            worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            maximum_major_group_harm,
            6,
        ),
    },
    "fold_metrics": fold_metrics,
    "training_selection": PHASE19_ADAPTER_TRAINING_REPORTS,
    "major_acquisition_group_column": group_column_name,
    "major_acquisition_group_metrics": major_group_reports,
    "promotion_thresholds": promotion_thresholds,
    "promotion_gate_passed": promotion_passed,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "outer_labels_used_only_for_final_evaluation": True,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE19_ADAPTER_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE19_ADAPTER_OOF")

PHASE19_ADAPTER_METRICS = adapter_metrics
PHASE19_ADAPTER_PROMOTED = promotion_passed
PHASE19_ADAPTER_PROMOTION_REPORT = report

BEGIN SANITIZED_PHASE19_ADAPTER_OOF
{
  "phase": "phase19_champion_residual_adapter",
  "status": "not_promoted",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535,
    "calibration_intercept": 0.167533,
    "calibration_slope": 0.935548,
    "mean_probability": 0.530697
  },
  "phase19": {
    "log_loss": 0.320522,
    "auroc": 0.933505,
    "brier": 0.099068,
    "calibration_intercept": 0.086758,
    "calibration_slope": 0.891825,
    "mean_probability": 0.537307
  },
  "improvements": {
    "log_loss_gain": -0.012906,
    "auroc_gain": -0.005422,
    "brier_excess": 0.003533,
    "fold_wins": 1,
    "worst_fold_excess": 0.030784,
    "maximum_major_group_harm": 0.03991
  },
  "fold_metrics": [
    {
      "fold": 0,
      "baseline_log_loss": 0.358494,
      "phase19_log_loss": 0.366413,
      "log_loss_improvement": -0.007919,
      "baseline_auroc": 0.916174,
      "phase19_auroc": 0.917102,
      "selected_alpha": 1.0
    },
    {
      "fold

In [50]:
# Cell 40 — Low-dimensional acquisition-robust expert evidence
import json

import numpy as np

assert len(PHASE19_STACKED_FEATURES_PRIVATE) == 7

# Last two columns are:
#   -2: reflection-averaged expert logit
#   -1: log1p(reflection logit change)
expert_logits = np.column_stack([
    features[:, -2]
    for features in PHASE19_STACKED_FEATURES_PRIVATE
]).astype(np.float64)

reflection_changes = np.column_stack([
    features[:, -1]
    for features in PHASE19_STACKED_FEATURES_PRIVATE
]).astype(np.float64)

assert expert_logits.shape == (EXPECTED_CASES, 7)
assert reflection_changes.shape == (EXPECTED_CASES, 7)
assert np.isfinite(expert_logits).all()
assert np.isfinite(reflection_changes).all()

# Expert ordering established by Cell 36:
# 0–1 compact 160 mm
# 2   compact 192 mm
# 3–4 compact high-resolution
# 5–6 anatomy transformers
compact_160 = expert_logits[:, 0:2]
compact_192 = expert_logits[:, 2]
compact_highres = expert_logits[:, 3:5]
anatomy = expert_logits[:, 5:7]

reflection_160 = reflection_changes[:, 0:2]
reflection_192 = reflection_changes[:, 2]
reflection_highres = reflection_changes[:, 3:5]
reflection_anatomy = reflection_changes[:, 5:7]

mean_160 = compact_160.mean(axis=1)
mean_highres = compact_highres.mean(axis=1)
mean_anatomy = anatomy.mean(axis=1)

phase20_feature_names = [
    "anchor_logit",
    "anchor_absolute_logit",
    "compact_160_mean",
    "compact_160_seed_disagreement",
    "compact_192_logit",
    "compact_highres_mean",
    "compact_highres_seed_disagreement",
    "anatomy_mean",
    "anatomy_seed_disagreement",
    "highres_minus_160",
    "context_192_minus_160",
    "anatomy_minus_compact_highres",
    "all_expert_logit_std",
    "all_expert_logit_range",
    "compact_reflection_change_mean",
    "anatomy_reflection_change_mean",
    "maximum_reflection_change",
]

phase20_relative_features = np.column_stack([
    PHASE19_ANCHOR_LOGIT_PRIVATE,
    np.abs(PHASE19_ANCHOR_LOGIT_PRIVATE),
    mean_160,
    np.abs(compact_160[:, 0] - compact_160[:, 1]),
    compact_192,
    mean_highres,
    np.abs(
        compact_highres[:, 0]
        - compact_highres[:, 1]
    ),
    mean_anatomy,
    np.abs(anatomy[:, 0] - anatomy[:, 1]),
    mean_highres - mean_160,
    compact_192 - mean_160,
    mean_anatomy - mean_highres,
    expert_logits.std(axis=1),
    expert_logits.max(axis=1) - expert_logits.min(axis=1),
    np.column_stack([
        reflection_160.mean(axis=1),
        reflection_192,
        reflection_highres.mean(axis=1),
    ]).mean(axis=1),
    reflection_anatomy.mean(axis=1),
    reflection_changes.max(axis=1),
]).astype(np.float64)

assert phase20_relative_features.shape == (
    EXPECTED_CASES,
    len(phase20_feature_names),
)
assert np.isfinite(phase20_relative_features).all()

assert "acquisition_group" in case_df.columns
phase20_acquisition_groups = np.asarray(
    case_df["acquisition_group"]
)
assert len(np.unique(phase20_acquisition_groups)) == 15

report = {
    "phase": "phase20_domain_robust_relative_evidence",
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "feature_count": len(phase20_feature_names),
    "feature_names": phase20_feature_names,
    "acquisition_group_count": int(
        len(np.unique(phase20_acquisition_groups))
    ),
    "representation": (
        "relative_expert_logits_disagreement_and_reflection_stability"
    ),
    "high_dimensional_embeddings_used": False,
    "reflection_invariant": True,
    "cross_fitted_features": True,
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE20_RELATIVE_FEATURES")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE20_RELATIVE_FEATURES")

PHASE20_FEATURES_PRIVATE = phase20_relative_features
PHASE20_FEATURE_NAMES = phase20_feature_names
PHASE20_GROUPS_PRIVATE = phase20_acquisition_groups

BEGIN SANITIZED_PHASE20_RELATIVE_FEATURES
{
  "phase": "phase20_domain_robust_relative_evidence",
  "status": "accepted",
  "case_count": 1362,
  "feature_count": 17,
  "feature_names": [
    "anchor_logit",
    "anchor_absolute_logit",
    "compact_160_mean",
    "compact_160_seed_disagreement",
    "compact_192_logit",
    "compact_highres_mean",
    "compact_highres_seed_disagreement",
    "anatomy_mean",
    "anatomy_seed_disagreement",
    "highres_minus_160",
    "context_192_minus_160",
    "anatomy_minus_compact_highres",
    "all_expert_logit_std",
    "all_expert_logit_range",
    "compact_reflection_change_mean",
    "anatomy_reflection_change_mean",
    "maximum_reflection_change"
  ],
  "acquisition_group_count": 15,
  "representation": "relative_expert_logits_disagreement_and_reflection_stability",
  "high_dimensional_embeddings_used": false,
  "reflection_invariant": true,
  "cross_fitted_features": true,
  "case_level_features_exported": false,
  "test_or_smoke_data_rea

In [51]:
# Cell 41 — Group-balanced LOGO residual training
import json
import time

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import log_loss

PHASE20_CONFIG = {
    "residual_bound": 0.35,
    "regularization_grid": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
    ],
    "minimum_inner_group_gain": 0.001,
    "maximum_inner_worst_group_regret": 0.01,
    "optimizer_maximum_iterations": 600,
}

X_all = PHASE20_FEATURES_PRIVATE
y_all = PHASE19_Y_PRIVATE.astype(np.float64)
anchor_all = PHASE19_ANCHOR_LOGIT_PRIVATE.astype(np.float64)
groups_all = PHASE20_GROUPS_PRIVATE

def group_balanced_weights(groups):
    groups = np.asarray(groups)
    unique_groups, counts = np.unique(
        groups,
        return_counts=True,
    )
    count_map = dict(zip(unique_groups, counts))

    weights = np.asarray(
        [1.0 / count_map[group] for group in groups],
        dtype=np.float64,
    )
    weights /= weights.mean()
    return weights

def fit_bounded_residual(
    train_indices,
    regularization,
):
    train_indices = np.asarray(
        train_indices,
        dtype=np.int64,
    )

    X = X_all[train_indices]
    y = y_all[train_indices]
    anchor = anchor_all[train_indices]
    groups = groups_all[train_indices]

    mean = X.mean(axis=0)
    std = np.maximum(X.std(axis=0), 1e-3)
    X_standardized = np.clip(
        (X - mean) / std,
        -8.0,
        8.0,
    )

    sample_weight = group_balanced_weights(groups)
    sample_weight_sum = sample_weight.sum()

    feature_count = X_standardized.shape[1]
    bound = PHASE20_CONFIG["residual_bound"]

    def objective(parameters):
        weights = parameters[:feature_count]
        intercept = parameters[-1]

        raw = X_standardized @ weights + intercept
        tanh_raw = np.tanh(raw)
        residual = bound * tanh_raw
        corrected_logit = anchor + residual

        probability = expit(
            np.clip(corrected_logit, -30.0, 30.0)
        )

        probability = np.clip(
            probability,
            1e-8,
            1.0 - 1e-8,
        )

        per_case_loss = -(
            y * np.log(probability)
            + (1.0 - y) * np.log(1.0 - probability)
        )

        weighted_loss = (
            np.sum(sample_weight * per_case_loss)
            / sample_weight_sum
        )

        penalty = (
            0.5
            * regularization
            * np.mean(weights ** 2)
            + 0.05
            * regularization
            * intercept ** 2
        )

        objective_value = weighted_loss + penalty

        logit_gradient = (
            sample_weight
            * (probability - y)
            / sample_weight_sum
        )

        raw_gradient = (
            logit_gradient
            * bound
            * (1.0 - tanh_raw ** 2)
        )

        weight_gradient = (
            X_standardized.T @ raw_gradient
            + regularization
            * weights
            / feature_count
        )

        intercept_gradient = (
            raw_gradient.sum()
            + 0.1 * regularization * intercept
        )

        gradient = np.concatenate([
            weight_gradient,
            np.asarray([intercept_gradient]),
        ])

        return objective_value, gradient

    initial = np.zeros(feature_count + 1, dtype=np.float64)

    result = minimize(
        fun=lambda parameters: objective(parameters),
        x0=initial,
        method="L-BFGS-B",
        jac=True,
        bounds=[
            (-3.0, 3.0)
            for _ in range(feature_count + 1)
        ],
        options={
            "maxiter": PHASE20_CONFIG[
                "optimizer_maximum_iterations"
            ],
            "ftol": 1e-12,
            "gtol": 1e-8,
            "maxls": 40,
        },
    )

    assert np.isfinite(result.fun)
    assert np.isfinite(result.x).all()

    return {
        "mean": mean,
        "std": std,
        "weights": result.x[:feature_count],
        "intercept": float(result.x[-1]),
        "regularization": float(regularization),
        "optimizer_success": bool(result.success),
        "optimizer_iterations": int(result.nit),
    }

def predict_bounded_residual(model, indices):
    indices = np.asarray(indices, dtype=np.int64)

    standardized = np.clip(
        (
            X_all[indices]
            - model["mean"]
        )
        / model["std"],
        -8.0,
        8.0,
    )

    raw = (
        standardized @ model["weights"]
        + model["intercept"]
    )

    residual = (
        PHASE20_CONFIG["residual_bound"]
        * np.tanh(raw)
    )

    probability = expit(
        np.clip(
            anchor_all[indices] + residual,
            -30.0,
            30.0,
        )
    )

    probability = np.clip(
        probability,
        1e-5,
        1.0 - 1e-5,
    )

    return probability, residual

phase20_oof = PHASE19_PHASE12C_OOF_PRIVATE.copy()
phase20_fold_states = []
phase20_selection_reports = []

started = time.perf_counter()

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    outer_train = partition["outer_train"]
    outer_valid = partition["outer_valid"]

    training_groups = np.unique(
        groups_all[outer_train]
    )

    candidate_rows = []

    for regularization in PHASE20_CONFIG[
        "regularization_grid"
    ]:
        heldout_losses = []
        baseline_losses = []
        group_regrets = []
        optimizer_successes = []

        for heldout_group in training_groups:
            inner_valid = outer_train[
                groups_all[outer_train] == heldout_group
            ]
            inner_fit = outer_train[
                groups_all[outer_train] != heldout_group
            ]

            assert len(inner_fit) > 0
            assert len(inner_valid) > 0

            model = fit_bounded_residual(
                inner_fit,
                regularization,
            )

            probability, _ = predict_bounded_residual(
                model,
                inner_valid,
            )

            heldout_loss = float(
                log_loss(
                    y_all[inner_valid],
                    probability,
                    labels=[0, 1],
                )
            )
            baseline_loss = float(
                log_loss(
                    y_all[inner_valid],
                    PHASE19_PHASE12C_OOF_PRIVATE[
                        inner_valid
                    ],
                    labels=[0, 1],
                )
            )

            heldout_losses.append(heldout_loss)
            baseline_losses.append(baseline_loss)
            group_regrets.append(
                heldout_loss - baseline_loss
            )
            optimizer_successes.append(
                model["optimizer_success"]
            )

        mean_loss = float(np.mean(heldout_losses))
        baseline_mean_loss = float(
            np.mean(baseline_losses)
        )
        mean_gain = baseline_mean_loss - mean_loss
        worst_regret = float(np.max(group_regrets))

        robust_score = (
            mean_loss
            + 0.25 * float(np.std(heldout_losses))
            + 0.75 * max(0.0, worst_regret)
        )

        candidate_rows.append({
            "regularization": float(regularization),
            "mean_group_log_loss": mean_loss,
            "baseline_mean_group_log_loss": (
                baseline_mean_loss
            ),
            "mean_group_gain": mean_gain,
            "worst_group_regret": worst_regret,
            "robust_selection_score": robust_score,
            "optimizer_success_rate": float(
                np.mean(optimizer_successes)
            ),
        })

    candidate_rows.sort(
        key=lambda item: (
            item["robust_selection_score"],
            -item["regularization"],
        )
    )

    selected_candidate = candidate_rows[0]

    advance_correction = (
        selected_candidate["mean_group_gain"]
        >= PHASE20_CONFIG["minimum_inner_group_gain"]
        and selected_candidate["worst_group_regret"]
        <= PHASE20_CONFIG[
            "maximum_inner_worst_group_regret"
        ]
    )

    if advance_correction:
        final_model = fit_bounded_residual(
            outer_train,
            selected_candidate["regularization"],
        )
        outer_probability, outer_residual = (
            predict_bounded_residual(
                final_model,
                outer_valid,
            )
        )
    else:
        final_model = None
        outer_probability = (
            PHASE19_PHASE12C_OOF_PRIVATE[
                outer_valid
            ].copy()
        )
        outer_residual = np.zeros(
            len(outer_valid),
            dtype=np.float64,
        )

    phase20_oof[outer_valid] = outer_probability

    phase20_fold_states.append({
        "fold": fold_index,
        "correction_advanced": advance_correction,
        "selected_regularization": (
            selected_candidate["regularization"]
            if advance_correction else None
        ),
        "model": final_model,
    })

    phase20_selection_reports.append({
        "fold": fold_index,
        "outer_train_n": len(outer_train),
        "outer_valid_n": len(outer_valid),
        "training_group_count": len(training_groups),
        "correction_advanced": advance_correction,
        "selected_regularization": (
            selected_candidate["regularization"]
            if advance_correction else None
        ),
        "selected_inner_mean_group_gain": (
            selected_candidate["mean_group_gain"]
        ),
        "selected_inner_worst_group_regret": (
            selected_candidate["worst_group_regret"]
        ),
        "selected_robust_score": (
            selected_candidate["robust_selection_score"]
        ),
        "maximum_outer_residual_abs": float(
            np.max(np.abs(outer_residual))
        ),
        "mean_outer_residual_abs": float(
            np.mean(np.abs(outer_residual))
        ),
        "candidate_count": len(candidate_rows),
    })

    print(
        f"Phase20 fold {fold_index}: "
        f"advanced={advance_correction}, "
        f"inner_gain="
        f"{selected_candidate['mean_group_gain']:.6f}, "
        f"worst_regret="
        f"{selected_candidate['worst_group_regret']:.6f}",
        flush=True,
    )

assert np.isfinite(phase20_oof).all()

report = {
    "phase": "phase20_group_robust_residual_training",
    "status": "complete",
    "configuration": PHASE20_CONFIG,
    "folds": phase20_selection_reports,
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "selection_method": (
        "leave_one_acquisition_group_out_robust_log_loss"
    ),
    "training_weights": (
        "equal_total_weight_per_acquisition_group"
    ),
    "outer_labels_used_for_training_or_selection": False,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE20_ROBUST_TRAINING")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE20_ROBUST_TRAINING")

PHASE20_OOF_PRIVATE = phase20_oof
PHASE20_STATES_PRIVATE = phase20_fold_states
PHASE20_SELECTION_REPORTS = phase20_selection_reports

Phase20 fold 0: advanced=False, inner_gain=0.000051, worst_regret=0.000255
Phase20 fold 1: advanced=False, inner_gain=0.000149, worst_regret=0.000326
Phase20 fold 2: advanced=False, inner_gain=-0.000065, worst_regret=0.000309
BEGIN SANITIZED_PHASE20_ROBUST_TRAINING
{
  "phase": "phase20_group_robust_residual_training",
  "status": "complete",
  "configuration": {
    "residual_bound": 0.35,
    "regularization_grid": [
      0.01,
      0.1,
      1.0,
      10.0,
      100.0
    ],
    "minimum_inner_group_gain": 0.001,
    "maximum_inner_worst_group_regret": 0.01,
    "optimizer_maximum_iterations": 600
  },
  "folds": [
    {
      "fold": 0,
      "outer_train_n": 895,
      "outer_valid_n": 467,
      "training_group_count": 12,
      "correction_advanced": false,
      "selected_regularization": null,
      "selected_inner_mean_group_gain": 5.110990828621764e-05,
      "selected_inner_worst_group_regret": 0.00025466066736562487,
      "selected_robust_score": 0.2769984853746339,


In [52]:
# Cell 42 — Phase20 promotion report
import json

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

def phase20_metrics(labels, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )

    logits = np.log(
        probabilities / (1.0 - probabilities)
    )

    calibration_model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        logits.reshape(-1, 1),
        labels,
    )

    return {
        "log_loss": float(
            log_loss(labels, probabilities)
        ),
        "auroc": float(
            roc_auc_score(labels, probabilities)
        ),
        "brier": float(
            brier_score_loss(labels, probabilities)
        ),
        "calibration_intercept": float(
            calibration_model.intercept_[0]
        ),
        "calibration_slope": float(
            calibration_model.coef_[0, 0]
        ),
        "mean_probability": float(
            probabilities.mean()
        ),
    }

baseline = phase20_metrics(
    PHASE19_Y_PRIVATE,
    PHASE19_PHASE12C_OOF_PRIVATE,
)
phase20 = phase20_metrics(
    PHASE19_Y_PRIVATE,
    PHASE20_OOF_PRIVATE,
)

fold_rows = []
fold_wins = 0
worst_fold_excess = -np.inf

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    indices = partition["outer_valid"]
    labels = PHASE19_Y_PRIVATE[indices]

    baseline_fold = phase20_metrics(
        labels,
        PHASE19_PHASE12C_OOF_PRIVATE[indices],
    )
    phase20_fold = phase20_metrics(
        labels,
        PHASE20_OOF_PRIVATE[indices],
    )

    improvement = (
        baseline_fold["log_loss"]
        - phase20_fold["log_loss"]
    )
    excess = -improvement

    fold_wins += int(improvement > 0)
    worst_fold_excess = max(
        worst_fold_excess,
        excess,
    )

    fold_rows.append({
        "fold": fold_index,
        "correction_advanced": (
            PHASE20_STATES_PRIVATE[
                fold_index
            ]["correction_advanced"]
        ),
        "baseline_log_loss": round(
            baseline_fold["log_loss"],
            6,
        ),
        "phase20_log_loss": round(
            phase20_fold["log_loss"],
            6,
        ),
        "log_loss_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_fold["auroc"],
            6,
        ),
        "phase20_auroc": round(
            phase20_fold["auroc"],
            6,
        ),
    })

major_group_rows = []
maximum_major_group_harm = 0.0

for group in np.unique(PHASE20_GROUPS_PRIVATE):
    indices = np.flatnonzero(
        PHASE20_GROUPS_PRIVATE == group
    )

    if len(indices) < 30:
        continue

    labels = PHASE19_Y_PRIVATE[indices]

    baseline_loss = float(
        log_loss(
            labels,
            PHASE19_PHASE12C_OOF_PRIVATE[indices],
            labels=[0, 1],
        )
    )
    phase20_loss = float(
        log_loss(
            labels,
            PHASE20_OOF_PRIVATE[indices],
            labels=[0, 1],
        )
    )

    harm = phase20_loss - baseline_loss
    maximum_major_group_harm = max(
        maximum_major_group_harm,
        harm,
    )

    major_group_rows.append({
        "group": (
            int(group)
            if np.issubdtype(
                np.asarray(group).dtype,
                np.integer,
            )
            else str(group)
        ),
        "n": len(indices),
        "baseline_log_loss": round(
            baseline_loss,
            6,
        ),
        "phase20_log_loss": round(
            phase20_loss,
            6,
        ),
        "log_loss_change": round(harm, 6),
    })

log_loss_gain = (
    baseline["log_loss"] - phase20["log_loss"]
)
auroc_gain = (
    phase20["auroc"] - baseline["auroc"]
)
brier_excess = (
    phase20["brier"] - baseline["brier"]
)

thresholds = {
    "minimum_log_loss_gain": 0.003,
    "minimum_auroc_gain": 0.002,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.005,
    "maximum_brier_excess": 0.001,
    "calibration_slope_range": [0.8, 1.2],
    "maximum_major_group_harm": 0.015,
}

promotion_passed = (
    log_loss_gain >= thresholds["minimum_log_loss_gain"]
    and auroc_gain >= thresholds["minimum_auroc_gain"]
    and fold_wins >= thresholds["minimum_fold_wins"]
    and worst_fold_excess
    <= thresholds["maximum_worst_fold_excess"]
    and brier_excess <= thresholds["maximum_brier_excess"]
    and 0.8 <= phase20["calibration_slope"] <= 1.2
    and maximum_major_group_harm
    <= thresholds["maximum_major_group_harm"]
)

report = {
    "phase": "phase20_acquisition_robust_residual",
    "status": (
        "promoted"
        if promotion_passed
        else "not_promoted"
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline.items()
    },
    "phase20": {
        key: round(value, 6)
        for key, value in phase20.items()
    },
    "improvements": {
        "log_loss_gain": round(log_loss_gain, 6),
        "auroc_gain": round(auroc_gain, 6),
        "brier_excess": round(brier_excess, 6),
        "fold_wins": fold_wins,
        "worst_fold_excess": round(
            float(worst_fold_excess),
            6,
        ),
        "maximum_major_group_harm": round(
            maximum_major_group_harm,
            6,
        ),
    },
    "fold_metrics": fold_rows,
    "inner_selection": PHASE20_SELECTION_REPORTS,
    "major_acquisition_group_metrics": major_group_rows,
    "promotion_thresholds": thresholds,
    "promotion_gate_passed": promotion_passed,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "outer_labels_used_only_for_final_evaluation": True,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE20_ROBUST_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE20_ROBUST_OOF")

PHASE20_METRICS = phase20
PHASE20_PROMOTED = promotion_passed
PHASE20_PROMOTION_REPORT = report

BEGIN SANITIZED_PHASE20_ROBUST_OOF
{
  "phase": "phase20_acquisition_robust_residual",
  "status": "not_promoted",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535,
    "calibration_intercept": 0.167533,
    "calibration_slope": 0.935548,
    "mean_probability": 0.530697
  },
  "phase20": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535,
    "calibration_intercept": 0.167533,
    "calibration_slope": 0.935548,
    "mean_probability": 0.530697
  },
  "improvements": {
    "log_loss_gain": 0.0,
    "auroc_gain": 0.0,
    "brier_excess": 0.0,
    "fold_wins": 0,
    "worst_fold_excess": -0.0,
    "maximum_major_group_harm": 0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "correction_advanced": false,
      "baseline_log_loss": 0.358494,
      "phase20_log_loss": 0.358494,
      "log_loss_improvement": 0.0,
      "baseline_auroc": 0.916174,
      "phase20_auroc": 0.916174
    },
    {
      "fold": 1,
      "correction

In [54]:
# Cell 42A — Official MedicalNet ResNet-18 download
import hashlib
import json
import shutil
import urllib.request
from pathlib import Path

MEDICALNET_URL = (
    "https://huggingface.co/"
    "TencentMedicalNet/MedicalNet-Resnet18/"
    "resolve/main/resnet_18_23dataset.pth"
    "?download=true"
)

# Published on the official model file page.
EXPECTED_DIGEST_PRIVATE = (
    "61224f9317fcce873366deb3703183e92"
    "cc47325b726b69691b33536244e10f4"
)

output_directory = Path(
    "/kaggle/working/phase21_external"
)
output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

output_path = (
    output_directory
    / "resnet_18_23dataset.pth"
)
temporary_path = output_path.with_suffix(".download")

if not output_path.is_file():
    print(
        "Downloading official MedicalNet ResNet-18 "
        "checkpoint...",
        flush=True,
    )

    request = urllib.request.Request(
        MEDICALNET_URL,
        headers={
            "User-Agent": "Mozilla/5.0",
        },
    )

    with urllib.request.urlopen(
        request,
        timeout=120,
    ) as response:
        with temporary_path.open("wb") as output_handle:
            shutil.copyfileobj(
                response,
                output_handle,
                length=1024 * 1024,
            )

    temporary_path.replace(output_path)

assert output_path.is_file()
assert output_path.stat().st_size > 120_000_000

digest = hashlib.sha256()

with output_path.open("rb") as handle:
    for block in iter(
        lambda: handle.read(1024 * 1024),
        b"",
    ):
        digest.update(block)

integrity_verified = (
    digest.hexdigest() == EXPECTED_DIGEST_PRIVATE
)

assert integrity_verified, (
    "Downloaded checkpoint failed official integrity verification."
)

report = {
    "phase": "phase21_medicalnet_download",
    "status": "complete",
    "source": (
        "TencentMedicalNet/MedicalNet-Resnet18"
    ),
    "filename": output_path.name,
    "size_mb": round(
        output_path.stat().st_size / 1e6,
        2,
    ),
    "integrity_verified": integrity_verified,
    "declared_license": "MIT",
    "model_hash_displayed": False,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE21_MEDICALNET_DOWNLOAD")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_MEDICALNET_DOWNLOAD")

PHASE21_MEDICALNET_PATH = output_path

BEGIN SANITIZED_PHASE21_MEDICALNET_DOWNLOAD
{
  "phase": "phase21_medicalnet_download",
  "status": "complete",
  "source": "TencentMedicalNet/MedicalNet-Resnet18",
  "filename": "resnet_18_23dataset.pth",
  "size_mb": 131.99,
  "integrity_verified": true,
  "declared_license": "MIT",
  "model_hash_displayed": false,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "uids_displayed": false
}
END SANITIZED_PHASE21_MEDICALNET_DOWNLOAD


In [55]:
# Cell 43 — MedicalNet external-weight discovery
import json
from pathlib import Path

import torch

search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

medicalnet_candidates = []

for root in search_roots:
    if not root.is_dir():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        lowered = path.name.casefold()

        if (
            "resnet_18" in lowered
            and "23dataset" in lowered
            and path.suffix.casefold() in {".pth", ".pt", ".tar"}
        ):
            medicalnet_candidates.append(path.resolve())

medicalnet_candidates = sorted(set(medicalnet_candidates))

report = {
    "phase": "phase21_medicalnet_discovery",
    "status": (
        "found"
        if len(medicalnet_candidates) == 1
        else "missing_or_ambiguous"
    ),
    "candidate_count": len(medicalnet_candidates),
    "candidates": [
        {
            "name": path.name,
            "location": (
                "kaggle_input"
                if str(path).startswith("/kaggle/input/")
                else "kaggle_working"
            ),
            "size_mb": round(
                path.stat().st_size / 1e6,
                2,
            ),
        }
        for path in medicalnet_candidates
    ],
    "expected_asset": "resnet_18_23dataset.pth",
    "declared_upstream_license": "MIT",
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "model_hash_displayed": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE21_MEDICALNET_DISCOVERY")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_MEDICALNET_DISCOVERY")

assert len(medicalnet_candidates) == 1, (
    "Add exactly one resnet_18_23dataset checkpoint, "
    "then rerun Cell 43."
)

PHASE21_MEDICALNET_PATH = medicalnet_candidates[0]

checkpoint = torch.load(
    PHASE21_MEDICALNET_PATH,
    map_location="cpu",
    weights_only=True,
)

def find_tensor_dictionary(value):
    candidates = []

    def visit(item):
        if isinstance(item, dict):
            tensor_items = {
                str(key): tensor
                for key, tensor in item.items()
                if torch.is_tensor(tensor)
            }

            if len(tensor_items) >= 20:
                candidates.append(tensor_items)

            for child in item.values():
                if isinstance(child, (dict, list, tuple)):
                    visit(child)

        elif isinstance(item, (list, tuple)):
            for child in item:
                visit(child)

    visit(value)

    assert candidates, "No checkpoint state dictionary found."
    candidates.sort(key=len, reverse=True)
    return candidates[0]

PHASE21_MEDICALNET_STATE_RAW = find_tensor_dictionary(checkpoint)

report = {
    "phase": "phase21_medicalnet_checkpoint",
    "status": "loaded",
    "checkpoint_name": PHASE21_MEDICALNET_PATH.name,
    "tensor_count": len(PHASE21_MEDICALNET_STATE_RAW),
    "tensor_parameter_count": int(sum(
        tensor.numel()
        for tensor in PHASE21_MEDICALNET_STATE_RAW.values()
    )),
    "all_tensors_finite": bool(all(
        torch.isfinite(tensor).all()
        for tensor in PHASE21_MEDICALNET_STATE_RAW.values()
        if tensor.is_floating_point()
    )),
    "checkpoint_values_exported": False,
    "model_hash_displayed": False,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE21_MEDICALNET_CHECKPOINT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_MEDICALNET_CHECKPOINT")

BEGIN SANITIZED_PHASE21_MEDICALNET_DISCOVERY
{
  "phase": "phase21_medicalnet_discovery",
  "status": "found",
  "candidate_count": 1,
  "candidates": [
    {
      "name": "resnet_18_23dataset.pth",
      "location": "kaggle_working",
      "size_mb": 131.99
    }
  ],
  "expected_asset": "resnet_18_23dataset.pth",
  "declared_upstream_license": "MIT",
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "model_hash_displayed": false,
  "uids_displayed": false
}
END SANITIZED_PHASE21_MEDICALNET_DISCOVERY
BEGIN SANITIZED_PHASE21_MEDICALNET_CHECKPOINT
{
  "phase": "phase21_medicalnet_checkpoint",
  "status": "loaded",
  "checkpoint_name": "resnet_18_23dataset.pth",
  "tensor_count": 102,
  "tensor_parameter_count": 32994001,
  "all_tensors_finite": true,
  "checkpoint_values_exported": false,
  "model_hash_displayed": false,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE21_MEDICALNET_CHECKPOINT


In [56]:
# Cell 44 — Modern PyTorch MedicalNet ResNet-18 encoder
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

class MedicalNetBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, channels, stride=1, dilation=1):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels,
            channels,
            kernel_size=3,
            stride=stride,
            padding=dilation,
            dilation=dilation,
            bias=False,
        )
        self.bn1 = nn.BatchNorm3d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
            bias=False,
        )
        self.bn2 = nn.BatchNorm3d(channels)

        self.stride = stride
        self.in_channels = in_channels
        self.channels = channels

    def shortcut(self, x):
        if self.stride == 1 and self.in_channels == self.channels:
            return x

        output = F.avg_pool3d(
            x,
            kernel_size=1,
            stride=self.stride,
        )

        channel_difference = self.channels - output.shape[1]

        if channel_difference > 0:
            padding = output.new_zeros(
                output.shape[0],
                channel_difference,
                *output.shape[2:],
            )
            output = torch.cat([output, padding], dim=1)

        return output

    def forward(self, x):
        residual = self.shortcut(x)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.relu(x + residual)

class MedicalNetResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv3d(
            1,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(
            kernel_size=3,
            stride=2,
            padding=1,
        )

        self.layer1 = self._make_stage(
            64, 64, blocks=2, stride=1, dilation=1
        )
        self.layer2 = self._make_stage(
            64, 128, blocks=2, stride=2, dilation=1
        )
        self.layer3 = self._make_stage(
            128, 256, blocks=2, stride=1, dilation=2
        )
        self.layer4 = self._make_stage(
            256, 512, blocks=2, stride=1, dilation=4
        )

    @staticmethod
    def _make_stage(
        in_channels,
        channels,
        blocks,
        stride,
        dilation,
    ):
        layers = [
            MedicalNetBasicBlock(
                in_channels,
                channels,
                stride=stride,
                dilation=dilation,
            )
        ]

        for _ in range(1, blocks):
            layers.append(
                MedicalNetBasicBlock(
                    channels,
                    channels,
                    stride=1,
                    dilation=dilation,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.layer4(x)

medicalnet_encoder = MedicalNetResNet18Encoder()

target_state = medicalnet_encoder.state_dict()
normalized_state = {}

for raw_key, tensor in PHASE21_MEDICALNET_STATE_RAW.items():
    key = str(raw_key)

    changed = True
    while changed:
        changed = False
        for prefix in ("module.", "model.", "encoder.", "backbone."):
            if key.startswith(prefix):
                key = key[len(prefix):]
                changed = True

    if (
        key in target_state
        and tuple(tensor.shape) == tuple(target_state[key].shape)
    ):
        normalized_state[key] = tensor

load_result = medicalnet_encoder.load_state_dict(
    normalized_state,
    strict=False,
)

matched_parameter_count = sum(
    target_state[key].numel()
    for key in normalized_state
)

total_parameter_count = sum(
    tensor.numel()
    for tensor in target_state.values()
)

coverage = matched_parameter_count / total_parameter_count

assert "conv1.weight" in normalized_state
assert coverage >= 0.995, {
    "coverage": coverage,
    "missing_key_count": len(load_result.missing_keys),
}

medicalnet_encoder.eval().to(PHASE19_DEVICE)

torch.cuda.reset_peak_memory_stats(PHASE19_DEVICE)

with torch.inference_mode():
    highres_output = medicalnet_encoder(
        torch.zeros(
            (1, 1, 80, 80, 80),
            device=PHASE19_DEVICE,
        )
    )
    context_output = medicalnet_encoder(
        torch.zeros(
            (1, 1, 64, 64, 64),
            device=PHASE19_DEVICE,
        )
    )

report = {
    "phase": "phase21_medicalnet_encoder",
    "status": "accepted",
    "architecture": "MedicalNet_ResNet18_encoder",
    "parameter_count": sum(
        parameter.numel()
        for parameter in medicalnet_encoder.parameters()
    ),
    "matched_state_tensor_count": len(normalized_state),
    "state_parameter_coverage": round(coverage, 8),
    "missing_state_key_count": len(load_result.missing_keys),
    "highres_feature_shape": list(highres_output.shape),
    "context_feature_shape": list(context_output.shape),
    "peak_vram_mb": round(
        torch.cuda.max_memory_allocated(
            PHASE19_DEVICE
        ) / 2**20,
        2,
    ),
    "external_weights": True,
    "external_weight_source": "Tencent_MedicalNet",
    "declared_license": "MIT",
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE21_MEDICALNET_ENCODER")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_MEDICALNET_ENCODER")

PHASE21_PRETRAINED_ENCODER = medicalnet_encoder.cpu()
PHASE21_PRETRAINED_ENCODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in medicalnet_encoder.state_dict().items()
}

del highres_output
del context_output
torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE21_MEDICALNET_ENCODER
{
  "phase": "phase21_medicalnet_encoder",
  "status": "accepted",
  "architecture": "MedicalNet_ResNet18_encoder",
  "parameter_count": 32986176,
  "matched_state_tensor_count": 102,
  "state_parameter_coverage": 1.0,
  "missing_state_key_count": 0,
  "highres_feature_shape": [
    1,
    512,
    10,
    10,
    10
  ],
  "context_feature_shape": [
    1,
    512,
    8,
    8,
    8
  ],
  "peak_vram_mb": 348.53,
  "external_weights": true,
  "external_weight_source": "Tencent_MedicalNet",
  "declared_license": "MIT",
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE21_MEDICALNET_ENCODER


In [57]:
# Cell 45 — MedicalNet clinical-token architecture
import copy
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

class Phase21MedicalClinicalTokenNet(nn.Module):
    def __init__(
        self,
        pretrained_encoder_state,
        token_dimension=128,
        auxiliary_targets=8,
    ):
        super().__init__()

        self.encoder = MedicalNetResNet18Encoder()
        self.encoder.load_state_dict(
            pretrained_encoder_state,
            strict=True,
        )

        self.highres_global_projection = nn.Sequential(
            nn.Linear(1024, token_dimension),
            nn.LayerNorm(token_dimension),
            nn.SiLU(),
        )
        self.context_global_projection = nn.Sequential(
            nn.Linear(1024, token_dimension),
            nn.LayerNorm(token_dimension),
            nn.SiLU(),
        )

        # Six clinical symmetry/morphology tokens.
        self.regional_projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(512, token_dimension),
                nn.LayerNorm(token_dimension),
                nn.SiLU(),
            )
            for _ in range(6)
        ])

        self.position_embedding = nn.Parameter(
            torch.zeros(1, 8, token_dimension)
        )

        layer = nn.TransformerEncoderLayer(
            d_model=token_dimension,
            nhead=4,
            dim_feedforward=256,
            dropout=0.15,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=2,
            norm=nn.LayerNorm(token_dimension),
            enable_nested_tensor=False,
        )

        self.attention_pool = nn.Sequential(
            nn.Linear(token_dimension, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(token_dimension),
            nn.Dropout(0.20),
            nn.Linear(token_dimension, 1),
        )
        self.auxiliary_head = nn.Sequential(
            nn.LayerNorm(token_dimension),
            nn.Linear(token_dimension, auxiliary_targets),
        )

        nn.init.normal_(self.position_embedding, std=0.02)

    @staticmethod
    def global_average_max(feature_map):
        average = feature_map.mean(dim=(2, 3, 4))
        maximum = torch.amax(
            feature_map,
            dim=(2, 3, 4),
        )
        return torch.cat([average, maximum], dim=1)

    @staticmethod
    def region_average(feature_map):
        return feature_map.mean(dim=(2, 3, 4))

    def clinical_regions(self, feature_map):
        # Tensor axes: B,C,X,Y,Z. X is left-right in canonical RAS.
        x_middle = feature_map.shape[2] // 2
        y_middle = feature_map.shape[3] // 2

        left = feature_map[:, :, :x_middle]
        right = feature_map[:, :, x_middle:]

        left_global = self.region_average(left)
        right_global = self.region_average(right)

        left_anterior = self.region_average(
            left[:, :, :, :y_middle]
        )
        right_anterior = self.region_average(
            right[:, :, :, :y_middle]
        )

        left_posterior = self.region_average(
            left[:, :, :, y_middle:]
        )
        right_posterior = self.region_average(
            right[:, :, :, y_middle:]
        )

        return [
            0.5 * (left_global + right_global),
            torch.abs(left_global - right_global),
            0.5 * (left_anterior + right_anterior),
            torch.abs(left_anterior - right_anterior),
            0.5 * (left_posterior + right_posterior),
            torch.abs(left_posterior - right_posterior),
        ]

    def forward_single_orientation(self, highres, context):
        highres_map = self.encoder(highres)
        context_map = self.encoder(context)

        tokens = [
            self.highres_global_projection(
                self.global_average_max(highres_map)
            ),
            self.context_global_projection(
                self.global_average_max(context_map)
            ),
        ]

        regions = self.clinical_regions(highres_map)

        tokens.extend(
            projection(region)
            for projection, region in zip(
                self.regional_projections,
                regions,
            )
        )

        tokens = torch.stack(tokens, dim=1)
        tokens = tokens + self.position_embedding
        tokens = self.transformer(tokens)

        attention = torch.softmax(
            self.attention_pool(tokens).squeeze(-1),
            dim=1,
        )
        pooled = torch.sum(
            attention.unsqueeze(-1) * tokens,
            dim=1,
        )

        logit = self.classifier(pooled).squeeze(1)
        auxiliary = self.auxiliary_head(pooled)
        return logit, auxiliary

    def forward(self, highres, context):
        identity_logit, identity_aux = (
            self.forward_single_orientation(
                highres,
                context,
            )
        )

        reflected_logit, reflected_aux = (
            self.forward_single_orientation(
                torch.flip(highres, dims=(2,)),
                torch.flip(context, dims=(2,)),
            )
        )

        return (
            0.5 * (identity_logit + reflected_logit),
            0.5 * (identity_aux + reflected_aux),
        )

phase21_model = Phase21MedicalClinicalTokenNet(
    PHASE21_PRETRAINED_ENCODER_STATE,
).to(PHASE19_DEVICE)

# Stage A: train layer4 and new fusion/token heads.
for parameter in phase21_model.encoder.parameters():
    parameter.requires_grad_(False)

for parameter in phase21_model.encoder.layer4.parameters():
    parameter.requires_grad_(True)

# Keep pretrained BatchNorm statistics fixed.
for module in phase21_model.encoder.modules():
    if isinstance(module, nn.BatchNorm3d):
        module.eval()
        for parameter in module.parameters():
            parameter.requires_grad_(False)

torch.cuda.reset_peak_memory_stats(PHASE19_DEVICE)

highres = torch.zeros(
    (1, 1, 80, 80, 80),
    device=PHASE19_DEVICE,
)
context = torch.zeros(
    (1, 1, 64, 64, 64),
    device=PHASE19_DEVICE,
)
target = torch.ones(1, device=PHASE19_DEVICE)

phase21_model.train()

# Restore fixed BN behavior after model.train().
for module in phase21_model.encoder.modules():
    if isinstance(module, nn.BatchNorm3d):
        module.eval()

logit, auxiliary = phase21_model(highres, context)

loss = (
    F.binary_cross_entropy_with_logits(logit, target)
    + 0.05 * auxiliary.square().mean()
)
loss.backward()

gradient_norm = torch.sqrt(sum(
    parameter.grad.detach().square().sum()
    for parameter in phase21_model.parameters()
    if parameter.grad is not None
))

report = {
    "phase": "phase21_medicalnet_clinical_tokens",
    "status": "accepted",
    "model": "Phase21MedicalClinicalTokenNet",
    "parameter_count": sum(
        parameter.numel()
        for parameter in phase21_model.parameters()
    ),
    "trainable_parameter_count_stage_a": sum(
        parameter.numel()
        for parameter in phase21_model.parameters()
        if parameter.requires_grad
    ),
    "input_shapes": {
        "highres": list(highres.shape),
        "context": list(context.shape),
    },
    "clinical_token_count": 8,
    "token_dimension": 128,
    "logit_shape": list(logit.shape),
    "auxiliary_shape": list(auxiliary.shape),
    "gradient_norm": round(
        float(gradient_norm.detach().cpu()),
        6,
    ),
    "peak_vram_mb": round(
        torch.cuda.max_memory_allocated(
            PHASE19_DEVICE
        ) / 2**20,
        2,
    ),
    "reflection_average": True,
    "pretrained_encoder": "MedicalNet_ResNet18_23dataset",
    "external_weights": True,
    "declared_license": "MIT",
    "backward_contract_passed": bool(
        torch.isfinite(gradient_norm)
    ),
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE21_ARCHITECTURE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_ARCHITECTURE")

PHASE21_MODEL_CLASS = Phase21MedicalClinicalTokenNet

del phase21_model
del highres
del context
del target
del logit
del auxiliary
torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE21_ARCHITECTURE
{
  "phase": "phase21_medicalnet_clinical_tokens",
  "status": "accepted",
  "model": "Phase21MedicalClinicalTokenNet",
  "parameter_count": 33920842,
  "trainable_parameter_count_stage_a": 25707274,
  "input_shapes": {
    "highres": [
      1,
      1,
      80,
      80,
      80
    ],
    "context": [
      1,
      1,
      64,
      64,
      64
    ]
  },
  "clinical_token_count": 8,
  "token_dimension": 128,
  "logit_shape": [
    1
  ],
  "auxiliary_shape": [
    1,
    8
  ],
  "gradient_norm": 14.512791,
  "peak_vram_mb": 494.99,
  "reflection_average": true,
  "pretrained_encoder": "MedicalNet_ResNet18_23dataset",
  "external_weights": true,
  "declared_license": "MIT",
  "backward_contract_passed": true,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE21_ARCHITECTURE


In [58]:
# Cell 46 — Frozen MedicalNet token extraction
import json
import time

import numpy as np
import torch
import torch.nn as nn

TOKEN_BATCH_SIZE = 8
TOKEN_DIMS = [1024, 1024, 512, 512, 512, 512, 512, 512]

encoder = MedicalNetResNet18Encoder()
encoder.load_state_dict(
    PHASE21_PRETRAINED_ENCODER_STATE,
    strict=True,
)
encoder.eval().to(PHASE19_DEVICE)

for parameter in encoder.parameters():
    parameter.requires_grad_(False)

def medicalnet_standardize(batch):
    # Positive-voxel normalization; background remains a stable negative value.
    positive = batch > 0

    count = positive.sum(
        dim=(2, 3, 4),
        keepdim=True,
    ).clamp_min(1)

    mean = (
        (batch * positive).sum(
            dim=(2, 3, 4),
            keepdim=True,
        )
        / count
    )

    variance = (
        ((batch - mean).square() * positive).sum(
            dim=(2, 3, 4),
            keepdim=True,
        )
        / count
    )

    std = torch.sqrt(variance + 1e-5)
    return torch.clamp(
        (batch - mean) / std,
        -5.0,
        5.0,
    )

def global_average_max(feature_map):
    return torch.cat(
        [
            feature_map.mean(dim=(2, 3, 4)),
            torch.amax(feature_map, dim=(2, 3, 4)),
        ],
        dim=1,
    )

def region_average(feature_map):
    return feature_map.mean(dim=(2, 3, 4))

def extract_raw_tokens(highres_map, context_map):
    x_middle = highres_map.shape[2] // 2
    y_middle = highres_map.shape[3] // 2

    left = highres_map[:, :, :x_middle]
    right = highres_map[:, :, x_middle:]

    left_global = region_average(left)
    right_global = region_average(right)

    left_anterior = region_average(
        left[:, :, :, :y_middle]
    )
    right_anterior = region_average(
        right[:, :, :, :y_middle]
    )

    left_posterior = region_average(
        left[:, :, :, y_middle:]
    )
    right_posterior = region_average(
        right[:, :, :, y_middle:]
    )

    return [
        global_average_max(highres_map),
        global_average_max(context_map),
        0.5 * (left_global + right_global),
        torch.abs(left_global - right_global),
        0.5 * (left_anterior + right_anterior),
        torch.abs(left_anterior - right_anterior),
        0.5 * (left_posterior + right_posterior),
        torch.abs(left_posterior - right_posterior),
    ]

identity_parts = [[] for _ in TOKEN_DIMS]
reflected_parts = [[] for _ in TOKEN_DIMS]

started = time.perf_counter()

with torch.inference_mode():
    for start in range(0, EXPECTED_CASES, TOKEN_BATCH_SIZE):
        stop = min(
            start + TOKEN_BATCH_SIZE,
            EXPECTED_CASES,
        )

        highres = torch.from_numpy(
            PHASE19_HIGHRES_CACHE[start:stop].astype(
                np.float32,
                copy=False,
            )
        )[:, None].to(PHASE19_DEVICE)

        context = torch.from_numpy(
            PHASE19_LEGACY_CACHE["192"][start:stop].astype(
                np.float32,
                copy=False,
            )
        )[:, None].to(PHASE19_DEVICE)

        highres = medicalnet_standardize(highres)
        context = medicalnet_standardize(context)

        with torch.autocast(
            device_type=PHASE19_DEVICE.type,
            dtype=torch.float16,
            enabled=PHASE19_DEVICE.type == "cuda",
        ):
            highres_map = encoder(highres)
            context_map = encoder(context)

            reflected_highres_map = encoder(
                torch.flip(highres, dims=(2,))
            )
            reflected_context_map = encoder(
                torch.flip(context, dims=(2,))
            )

            identity_tokens = extract_raw_tokens(
                highres_map,
                context_map,
            )
            reflected_tokens = extract_raw_tokens(
                reflected_highres_map,
                reflected_context_map,
            )

        for index in range(len(TOKEN_DIMS)):
            identity_parts[index].append(
                identity_tokens[index].float().cpu().numpy()
            )
            reflected_parts[index].append(
                reflected_tokens[index].float().cpu().numpy()
            )

        if stop % 200 < TOKEN_BATCH_SIZE or stop == EXPECTED_CASES:
            print(
                f"MedicalNet token extraction: "
                f"{stop}/{EXPECTED_CASES}",
                flush=True,
            )

phase21_identity_tokens = [
    np.concatenate(parts).astype(np.float16)
    for parts in identity_parts
]
phase21_reflected_tokens = [
    np.concatenate(parts).astype(np.float16)
    for parts in reflected_parts
]

for index, dimension in enumerate(TOKEN_DIMS):
    assert phase21_identity_tokens[index].shape == (
        EXPECTED_CASES,
        dimension,
    )
    assert phase21_reflected_tokens[index].shape == (
        EXPECTED_CASES,
        dimension,
    )
    assert np.isfinite(phase21_identity_tokens[index]).all()
    assert np.isfinite(phase21_reflected_tokens[index]).all()

cache_bytes = sum(
    identity.nbytes + reflected.nbytes
    for identity, reflected in zip(
        phase21_identity_tokens,
        phase21_reflected_tokens,
    )
)

report = {
    "phase": "phase21_frozen_medicalnet_tokens",
    "status": "complete",
    "case_count": EXPECTED_CASES,
    "token_count_per_orientation": len(TOKEN_DIMS),
    "token_dimensions": TOKEN_DIMS,
    "orientations": [
        "identity",
        "left_right_reflection",
    ],
    "intensity_normalization": (
        "per_case_positive_voxel_zscore_clipped_minus5_to5"
    ),
    "cache_ram_gb": round(cache_bytes / 1e9, 4),
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "encoder_weights_updated": False,
    "external_weights": True,
    "declared_license": "MIT",
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE21_TOKEN_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_TOKEN_CACHE")

PHASE21_IDENTITY_TOKENS_PRIVATE = phase21_identity_tokens
PHASE21_REFLECTED_TOKENS_PRIVATE = phase21_reflected_tokens
PHASE21_TOKEN_DIMS = TOKEN_DIMS

del encoder
del identity_parts
del reflected_parts
torch.cuda.empty_cache()

MedicalNet token extraction: 200/1362
MedicalNet token extraction: 400/1362
MedicalNet token extraction: 600/1362
MedicalNet token extraction: 800/1362
MedicalNet token extraction: 1000/1362
MedicalNet token extraction: 1200/1362
MedicalNet token extraction: 1362/1362
BEGIN SANITIZED_PHASE21_TOKEN_CACHE
{
  "phase": "phase21_frozen_medicalnet_tokens",
  "status": "complete",
  "case_count": 1362,
  "token_count_per_orientation": 8,
  "token_dimensions": [
    1024,
    1024,
    512,
    512,
    512,
    512,
    512,
    512
  ],
  "orientations": [
    "identity",
    "left_right_reflection"
  ],
  "intensity_normalization": "per_case_positive_voxel_zscore_clipped_minus5_to5",
  "cache_ram_gb": 0.0279,
  "elapsed_seconds": 24.18,
  "encoder_weights_updated": false,
  "external_weights": true,
  "declared_license": "MIT",
  "case_level_features_exported": false,
  "test_or_smoke_data_read": false,
  "uids_displayed": false
}
END SANITIZED_PHASE21_TOKEN_CACHE


In [59]:
# Cell 47 — Train fold-local MedicalNet clinical-token heads
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import minimize
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader, Dataset

PHASE21_CONFIG = {
    "seeds": [210701, 210702, 210703],
    "batch_size": 64,
    "maximum_epochs": 180,
    "patience": 25,
    "learning_rate": 5e-4,
    "weight_decay": 2e-3,
    "auxiliary_weight": 0.03,
    "gradient_clip": 1.5,
    "blend_weights": [0.0, 0.10, 0.20, 0.30, 0.40],
    "minimum_monitor_blend_gain": 0.002,
}

class Phase21ClinicalTokenHead(nn.Module):
    def __init__(self, token_dims, model_dim=128):
        super().__init__()

        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(dimension),
                nn.Linear(dimension, model_dim),
                nn.SiLU(),
                nn.Dropout(0.15),
            )
            for dimension in token_dims
        ])

        self.position_embedding = nn.Parameter(
            torch.zeros(1, len(token_dims), model_dim)
        )

        layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=4,
            dim_feedforward=256,
            dropout=0.15,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=2,
            norm=nn.LayerNorm(model_dim),
            enable_nested_tensor=False,
        )

        self.attention_pool = nn.Sequential(
            nn.Linear(model_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Dropout(0.20),
            nn.Linear(model_dim, 1),
        )
        self.auxiliary = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, 8),
        )

        nn.init.normal_(self.position_embedding, std=0.02)

    def forward_orientation(self, raw_tokens):
        tokens = torch.stack([
            projection(token)
            for projection, token in zip(
                self.projections,
                raw_tokens,
            )
        ], dim=1)

        tokens = self.transformer(
            tokens + self.position_embedding
        )

        attention = torch.softmax(
            self.attention_pool(tokens).squeeze(-1),
            dim=1,
        )
        pooled = torch.sum(
            attention.unsqueeze(-1) * tokens,
            dim=1,
        )

        return (
            self.classifier(pooled).squeeze(1),
            self.auxiliary(pooled),
        )

    def forward(self, identity_tokens, reflected_tokens):
        identity_logit, identity_aux = (
            self.forward_orientation(identity_tokens)
        )
        reflected_logit, reflected_aux = (
            self.forward_orientation(reflected_tokens)
        )

        return (
            0.5 * (identity_logit + reflected_logit),
            0.5 * (identity_aux + reflected_aux),
        )

class Phase21Dataset(Dataset):
    def __init__(self, indices, aux_mean, aux_std):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.aux_mean = aux_mean
        self.aux_std = aux_std

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = self.indices[item]

        identity = tuple(
            torch.from_numpy(token[index].astype(np.float32))
            for token in PHASE21_IDENTITY_TOKENS_PRIVATE
        )
        reflected = tuple(
            torch.from_numpy(token[index].astype(np.float32))
            for token in PHASE21_REFLECTED_TOKENS_PRIVATE
        )

        target = torch.tensor(
            PHASE19_Y_PRIVATE[index],
            dtype=torch.float32,
        )

        auxiliary = torch.from_numpy(
            (
                PHASE18_AUXILIARY_TARGETS_RAW[index]
                - self.aux_mean
            ) / self.aux_std
        ).float()

        return identity, reflected, target, auxiliary

def make_phase21_loader(
    indices,
    aux_mean,
    aux_std,
    training,
    seed,
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        Phase21Dataset(
            indices,
            aux_mean,
            aux_std,
        ),
        batch_size=PHASE21_CONFIG["batch_size"],
        shuffle=training,
        num_workers=0,
        pin_memory=True,
        generator=generator,
    )

@torch.inference_mode()
def predict_phase21_raw(
    model,
    indices,
    aux_mean,
    aux_std,
):
    loader = make_phase21_loader(
        indices,
        aux_mean,
        aux_std,
        training=False,
        seed=0,
    )

    model.eval()
    logits = []

    for identity, reflected, _, _ in loader:
        identity = [
            value.to(PHASE19_DEVICE)
            for value in identity
        ]
        reflected = [
            value.to(PHASE19_DEVICE)
            for value in reflected
        ]

        logit, _ = model(identity, reflected)
        logits.append(logit.float().cpu().numpy())

    return np.concatenate(logits)

def fit_positive_platt(raw_logits, labels):
    raw_logits = np.asarray(raw_logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)

    def objective(parameters):
        intercept, raw_slope = parameters
        slope = 0.10 + 2.90 / (
            1.0 + np.exp(-raw_slope)
        )

        logits = intercept + slope * raw_logits
        probabilities = 1.0 / (
            1.0 + np.exp(-np.clip(logits, -30, 30))
        )
        probabilities = np.clip(
            probabilities,
            1e-8,
            1.0 - 1e-8,
        )

        loss = -np.mean(
            labels * np.log(probabilities)
            + (1.0 - labels)
            * np.log(1.0 - probabilities)
        )

        penalty = (
            0.005 * intercept ** 2
            + 0.005 * (slope - 1.0) ** 2
        )
        return loss + penalty

    result = minimize(
        objective,
        x0=np.asarray([0.0, -0.693147]),
        method="L-BFGS-B",
    )

    intercept, raw_slope = result.x
    slope = 0.10 + 2.90 / (
        1.0 + np.exp(-raw_slope)
    )

    return float(intercept), float(slope)

def train_phase21_seed(
    fold_index,
    fit_indices,
    monitor_indices,
    seed,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    auxiliary_fit = PHASE18_AUXILIARY_TARGETS_RAW[
        fit_indices
    ].astype(np.float32)

    aux_mean = auxiliary_fit.mean(axis=0)
    aux_std = np.maximum(
        auxiliary_fit.std(axis=0),
        1e-3,
    )

    model = Phase21ClinicalTokenHead(
        PHASE21_TOKEN_DIMS
    ).to(PHASE19_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE21_CONFIG["learning_rate"],
        weight_decay=PHASE21_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=PHASE21_CONFIG["maximum_epochs"],
        eta_min=3e-5,
    )

    train_loader = make_phase21_loader(
        fit_indices,
        aux_mean,
        aux_std,
        training=True,
        seed=seed,
    )

    monitor_y = PHASE19_Y_PRIVATE[monitor_indices]

    best_loss = float("inf")
    best_epoch = 0
    best_state = None
    stale_epochs = 0
    started = time.perf_counter()

    for epoch in range(
        1,
        PHASE21_CONFIG["maximum_epochs"] + 1,
    ):
        model.train()
        train_losses = []

        for identity, reflected, target, auxiliary in train_loader:
            identity = [
                value.to(PHASE19_DEVICE)
                for value in identity
            ]
            reflected = [
                value.to(PHASE19_DEVICE)
                for value in reflected
            ]
            target = target.to(PHASE19_DEVICE)
            auxiliary = auxiliary.to(PHASE19_DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logit, predicted_auxiliary = model(
                identity,
                reflected,
            )

            classification_loss = (
                F.binary_cross_entropy_with_logits(
                    logit,
                    target,
                )
            )
            auxiliary_loss = F.smooth_l1_loss(
                predicted_auxiliary,
                auxiliary,
            )

            loss = (
                classification_loss
                + PHASE21_CONFIG["auxiliary_weight"]
                * auxiliary_loss
            )
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE21_CONFIG["gradient_clip"],
            )
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        scheduler.step()

        monitor_logits = predict_phase21_raw(
            model,
            monitor_indices,
            aux_mean,
            aux_std,
        )
        monitor_probability = np.clip(
            1.0 / (
                1.0
                + np.exp(-np.clip(monitor_logits, -30, 30))
            ),
            1e-5,
            1.0 - 1e-5,
        )
        monitor_loss = float(
            log_loss(monitor_y, monitor_probability)
        )

        if monitor_loss < best_loss - 1e-5:
            best_loss = monitor_loss
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % 20 == 0:
            print(
                f"Phase21 fold {fold_index} seed {seed} "
                f"epoch {epoch:03d}: "
                f"train={np.mean(train_losses):.4f}, "
                f"monitor={monitor_loss:.4f}",
                flush=True,
            )

        if stale_epochs >= PHASE21_CONFIG["patience"]:
            break

    model.load_state_dict(best_state, strict=True)
    model.eval()

    return {
        "model": model,
        "state_dict": best_state,
        "aux_mean": aux_mean,
        "aux_std": aux_std,
        "best_epoch": best_epoch,
        "epochs_run": epoch,
        "best_monitor_loss": best_loss,
        "training_seconds": time.perf_counter() - started,
    }

@torch.inference_mode()
def phase12_fold_probability(fold_index, indices):
    predictions = []

    for start in range(0, len(indices), 8):
        batch_indices = indices[start:start + 8]
        crops = build_predictor_crops(batch_indices)

        output = PHASE19_FAMILY_MODULE.predict_family_batch(
            crops,
            PHASE19_FOLD_MANIFESTS[fold_index]["family"],
            PHASE19_FOLD_MANIFESTS[fold_index]["base"],
            PHASE19_BASE_MODELS,
            PHASE19_ANATOMY_MODELS,
            PHASE19_DEVICE,
        )

        predictions.append(
            PHASE19_NORMALIZE_PREDICTIONS(
                output,
                len(batch_indices),
            )
        )

    return np.clip(
        np.concatenate(predictions),
        1e-5,
        1.0 - 1e-5,
    )

phase21_standalone_oof = np.full(
    EXPECTED_CASES,
    np.nan,
)
phase21_blended_oof = np.full(
    EXPECTED_CASES,
    np.nan,
)
phase21_states = []
phase21_training_reports = []

started = time.perf_counter()

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    fit_indices = partition["fit"]
    monitor_indices = partition["monitor"]
    outer_indices = partition["outer_valid"]

    seed_results = []
    monitor_logits_by_seed = []
    outer_logits_by_seed = []

    for seed in PHASE21_CONFIG["seeds"]:
        result = train_phase21_seed(
            fold_index,
            fit_indices,
            monitor_indices,
            seed,
        )

        monitor_logits_by_seed.append(
            predict_phase21_raw(
                result["model"],
                monitor_indices,
                result["aux_mean"],
                result["aux_std"],
            )
        )
        outer_logits_by_seed.append(
            predict_phase21_raw(
                result["model"],
                outer_indices,
                result["aux_mean"],
                result["aux_std"],
            )
        )
        seed_results.append(result)

        print(
            f"Phase21 fold {fold_index} seed {seed} frozen: "
            f"best_epoch={result['best_epoch']}, "
            f"monitor={result['best_monitor_loss']:.6f}",
            flush=True,
        )

    monitor_raw = np.mean(
        np.stack(monitor_logits_by_seed),
        axis=0,
    )
    outer_raw = np.mean(
        np.stack(outer_logits_by_seed),
        axis=0,
    )

    monitor_y = PHASE19_Y_PRIVATE[monitor_indices]

    intercept, slope = fit_positive_platt(
        monitor_raw,
        monitor_y,
    )

    monitor_medicalnet_logit = (
        intercept + slope * monitor_raw
    )
    outer_medicalnet_logit = (
        intercept + slope * outer_raw
    )

    phase21_standalone_oof[outer_indices] = np.clip(
        1.0 / (
            1.0 + np.exp(
                -np.clip(outer_medicalnet_logit, -30, 30)
            )
        ),
        1e-5,
        1.0 - 1e-5,
    )

    monitor_phase12_probability = phase12_fold_probability(
        fold_index,
        monitor_indices,
    )
    monitor_phase12_logit = np.log(
        monitor_phase12_probability
        / (1.0 - monitor_phase12_probability)
    )

    outer_phase12_logit = PHASE19_ANCHOR_LOGIT_PRIVATE[
        outer_indices
    ]

    baseline_monitor_loss = float(
        log_loss(
            monitor_y,
            monitor_phase12_probability,
        )
    )

    blend_rows = []

    for weight in PHASE21_CONFIG["blend_weights"]:
        blended_logit = (
            (1.0 - weight) * monitor_phase12_logit
            + weight * monitor_medicalnet_logit
        )

        probability = np.clip(
            1.0 / (
                1.0
                + np.exp(
                    -np.clip(blended_logit, -30, 30)
                )
            ),
            1e-5,
            1.0 - 1e-5,
        )

        blend_rows.append({
            "weight": weight,
            "log_loss": float(
                log_loss(monitor_y, probability)
            ),
        })

    blend_rows.sort(
        key=lambda item: (
            item["log_loss"],
            item["weight"],
        )
    )

    selected = blend_rows[0]
    monitor_gain = (
        baseline_monitor_loss - selected["log_loss"]
    )

    selected_weight = (
        selected["weight"]
        if monitor_gain
        >= PHASE21_CONFIG["minimum_monitor_blend_gain"]
        else 0.0
    )

    outer_blended_logit = (
        (1.0 - selected_weight) * outer_phase12_logit
        + selected_weight * outer_medicalnet_logit
    )

    phase21_blended_oof[outer_indices] = np.clip(
        1.0 / (
            1.0
            + np.exp(
                -np.clip(outer_blended_logit, -30, 30)
            )
        ),
        1e-5,
        1.0 - 1e-5,
    )

    phase21_states.append({
        "fold": fold_index,
        "blend_weight": selected_weight,
        "platt_intercept": intercept,
        "platt_slope": slope,
        "seed_states": [
            {
                "state_dict": result["state_dict"],
                "aux_mean": result["aux_mean"],
                "aux_std": result["aux_std"],
            }
            for result in seed_results
        ],
    })

    phase21_training_reports.append({
        "fold": fold_index,
        "fit_n": len(fit_indices),
        "monitor_n": len(monitor_indices),
        "outer_n": len(outer_indices),
        "best_epochs": [
            result["best_epoch"]
            for result in seed_results
        ],
        "best_monitor_losses": [
            result["best_monitor_loss"]
            for result in seed_results
        ],
        "platt_intercept": intercept,
        "platt_slope": slope,
        "phase12_monitor_log_loss": baseline_monitor_loss,
        "selected_blend_weight": selected_weight,
        "selected_monitor_gain": monitor_gain,
        "training_seconds": sum(
            result["training_seconds"]
            for result in seed_results
        ),
    })

    for result in seed_results:
        del result["model"]

    torch.cuda.empty_cache()

assert np.isfinite(phase21_standalone_oof).all()
assert np.isfinite(phase21_blended_oof).all()

report = {
    "phase": "phase21_frozen_encoder_training",
    "status": "complete",
    "configuration": PHASE21_CONFIG,
    "folds": phase21_training_reports,
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "encoder_weights_updated": False,
    "outer_labels_used_for_training_or_selection": False,
    "deployment_states_retained_privately": True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE21_TRAINING")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_TRAINING")

PHASE21_STANDALONE_OOF_PRIVATE = phase21_standalone_oof
PHASE21_BLEND_OOF_PRIVATE = phase21_blended_oof
PHASE21_STATES_PRIVATE = phase21_states
PHASE21_TRAINING_REPORTS = phase21_training_reports

Phase21 fold 0 seed 210701 epoch 001: train=0.8820, monitor=0.7269
Phase21 fold 0 seed 210701 epoch 020: train=0.4642, monitor=0.4469
Phase21 fold 0 seed 210701 epoch 040: train=0.3596, monitor=0.3863
Phase21 fold 0 seed 210701 epoch 060: train=0.3449, monitor=0.5093
Phase21 fold 0 seed 210701 frozen: best_epoch=50, monitor=0.372259
Phase21 fold 0 seed 210702 epoch 001: train=0.8774, monitor=0.6958
Phase21 fold 0 seed 210702 epoch 020: train=0.4341, monitor=0.4070
Phase21 fold 0 seed 210702 epoch 040: train=0.4026, monitor=0.4121
Phase21 fold 0 seed 210702 frozen: best_epoch=28, monitor=0.389417
Phase21 fold 0 seed 210703 epoch 001: train=0.8729, monitor=0.7006
Phase21 fold 0 seed 210703 epoch 020: train=0.4319, monitor=0.4080
Phase21 fold 0 seed 210703 epoch 040: train=0.4603, monitor=0.3972
Phase21 fold 0 seed 210703 epoch 060: train=0.3256, monitor=0.6730
Phase21 fold 0 seed 210703 frozen: best_epoch=47, monitor=0.396590
Phase21 fold 1 seed 210701 epoch 001: train=0.8549, monitor=0.

In [60]:
# Cell 48 — Phase21 OOF report
import json

import numpy as np
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

def metric_triplet(probabilities):
    return {
        "log_loss": float(log_loss(
            PHASE19_Y_PRIVATE,
            probabilities,
        )),
        "auroc": float(roc_auc_score(
            PHASE19_Y_PRIVATE,
            probabilities,
        )),
        "brier": float(brier_score_loss(
            PHASE19_Y_PRIVATE,
            probabilities,
        )),
    }

baseline = metric_triplet(
    PHASE19_PHASE12C_OOF_PRIVATE
)
standalone = metric_triplet(
    PHASE21_STANDALONE_OOF_PRIVATE
)
blended = metric_triplet(
    PHASE21_BLEND_OOF_PRIVATE
)

fold_rows = []
blend_fold_wins = 0
worst_blend_fold_excess = -np.inf

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    indices = partition["outer_valid"]
    labels = PHASE19_Y_PRIVATE[indices]

    base_loss = float(log_loss(
        labels,
        PHASE19_PHASE12C_OOF_PRIVATE[indices],
    ))
    standalone_loss = float(log_loss(
        labels,
        PHASE21_STANDALONE_OOF_PRIVATE[indices],
    ))
    blend_loss = float(log_loss(
        labels,
        PHASE21_BLEND_OOF_PRIVATE[indices],
    ))

    improvement = base_loss - blend_loss
    blend_fold_wins += int(improvement > 0)
    worst_blend_fold_excess = max(
        worst_blend_fold_excess,
        -improvement,
    )

    fold_rows.append({
        "fold": fold_index,
        "phase12c_log_loss": round(base_loss, 6),
        "medicalnet_log_loss": round(
            standalone_loss,
            6,
        ),
        "blend_log_loss": round(blend_loss, 6),
        "blend_improvement": round(
            improvement,
            6,
        ),
        "medicalnet_auroc": round(float(
            roc_auc_score(
                labels,
                PHASE21_STANDALONE_OOF_PRIVATE[indices],
            )
        ), 6),
        "selected_blend_weight": (
            PHASE21_STATES_PRIVATE[
                fold_index
            ]["blend_weight"]
        ),
    })

log_loss_gain = (
    baseline["log_loss"] - blended["log_loss"]
)
auroc_gain = (
    blended["auroc"] - baseline["auroc"]
)

component_gate = (
    standalone["auroc"] >= 0.90
    and sum(
        state["blend_weight"] > 0
        for state in PHASE21_STATES_PRIVATE
    ) >= 2
)

promotion_gate = (
    log_loss_gain >= 0.003
    and auroc_gain >= 0.002
    and blend_fold_wins >= 2
    and worst_blend_fold_excess <= 0.005
    and (
        blended["brier"] - baseline["brier"]
    ) <= 0.001
)

stage_b_eligible = (
    component_gate or promotion_gate
)

report = {
    "phase": "phase21_medicalnet_frozen_component",
    "status": (
        "promoted"
        if promotion_gate
        else (
            "stage_b_eligible"
            if stage_b_eligible
            else "terminated"
        )
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline.items()
    },
    "medicalnet_standalone": {
        key: round(value, 6)
        for key, value in standalone.items()
    },
    "phase12c_medicalnet_blend": {
        key: round(value, 6)
        for key, value in blended.items()
    },
    "improvements": {
        "blend_log_loss_gain": round(
            log_loss_gain,
            6,
        ),
        "blend_auroc_gain": round(
            auroc_gain,
            6,
        ),
        "blend_fold_wins": blend_fold_wins,
        "worst_blend_fold_excess": round(
            float(worst_blend_fold_excess),
            6,
        ),
    },
    "fold_metrics": fold_rows,
    "training_selection": PHASE21_TRAINING_REPORTS,
    "component_gate_passed": component_gate,
    "promotion_gate_passed": promotion_gate,
    "stage_b_eligible": stage_b_eligible,
    "encoder_weights_updated": False,
    "external_weights": True,
    "declared_license": "MIT",
    "outer_labels_used_only_for_final_evaluation": True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE21_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE21_OOF")

PHASE21_COMPONENT_GATE_PASSED = component_gate
PHASE21_PROMOTED = promotion_gate
PHASE21_STAGE_B_ELIGIBLE = stage_b_eligible
PHASE21_OOF_REPORT = report

BEGIN SANITIZED_PHASE21_OOF
{
  "phase": "phase21_medicalnet_frozen_component",
  "status": "terminated",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "medicalnet_standalone": {
    "log_loss": 0.495853,
    "auroc": 0.833983,
    "brier": 0.165185
  },
  "phase12c_medicalnet_blend": {
    "log_loss": 0.307409,
    "auroc": 0.939008,
    "brier": 0.095401
  },
  "improvements": {
    "blend_log_loss_gain": 0.000206,
    "blend_auroc_gain": 8.1e-05,
    "blend_fold_wins": 3,
    "worst_blend_fold_excess": -0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "phase12c_log_loss": 0.358494,
      "medicalnet_log_loss": 0.443459,
      "blend_log_loss": 0.357892,
      "blend_improvement": 0.000601,
      "medicalnet_auroc": 0.873371,
      "selected_blend_weight": 0.2
    },
    {
      "fold": 1,
      "phase12c_log_loss": 0.255415,
      "medicalnet_log_loss": 0.509185,
      "blend_log_loss": 0.255415,
      "blend_improvement": 0.0

In [61]:
# Cell 49 — Bilateral striatal landmark canonicalization
import json
import time

import numpy as np
from scipy.ndimage import map_coordinates

PATCH_SHAPE = (40, 40, 40)
CASE_COUNT = EXPECTED_CASES

def weighted_half_centroid(volume, x_start, x_stop):
    # Restrict localization to the central brain region.
    y_start, y_stop = 8, 72
    z_start, z_stop = 6, 70

    roi = volume[
        x_start:x_stop,
        y_start:y_stop,
        z_start:z_stop,
    ].astype(np.float32)

    positive = roi[roi > 0]
    if positive.size < 64:
        raise ValueError("insufficient_positive_voxels")

    threshold = float(
        np.percentile(positive, 96.0)
    )

    weights = np.maximum(
        roi - threshold,
        0.0,
    ) ** 2

    total_weight = float(weights.sum())

    if total_weight <= 1e-8:
        raise ValueError("insufficient_landmark_weight")

    coordinates = np.indices(
        roi.shape,
        dtype=np.float64,
    )

    centroid = np.asarray([
        np.sum(coordinates[axis] * weights)
        / total_weight
        for axis in range(3)
    ])

    centroid += np.asarray([
        x_start,
        y_start,
        z_start,
    ])

    return centroid

def sample_centered_patch(volume, centroid, shape):
    axes = [
        np.arange(size, dtype=np.float64)
        - (size - 1) / 2.0
        + float(centroid[axis])
        for axis, size in enumerate(shape)
    ]

    grid = np.meshgrid(*axes, indexing="ij")
    coordinates = np.stack(
        grid,
        axis=0,
    ).reshape(3, -1)

    patch = map_coordinates(
        volume,
        coordinates,
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )

    return patch.reshape(shape).astype(np.float32)

left_patches = np.empty(
    (CASE_COUNT, *PATCH_SHAPE),
    dtype=np.float16,
)
right_patches = np.empty_like(left_patches)

centroid_separations = []
patch_capture_fractions = []
fallback_count = 0

started = time.perf_counter()

for index in range(CASE_COUNT):
    volume = PHASE19_HIGHRES_CACHE[index].astype(
        np.float32,
        copy=False,
    )

    try:
        left_centroid = weighted_half_centroid(
            volume,
            4,
            40,
        )
        right_centroid = weighted_half_centroid(
            volume,
            40,
            76,
        )
    except Exception:
        fallback_count += 1

        left_centroid = np.asarray(
            [30.0, 40.0, 38.0]
        )
        right_centroid = np.asarray(
            [50.0, 40.0, 38.0]
        )

    left_patch = sample_centered_patch(
        volume,
        left_centroid,
        PATCH_SHAPE,
    )
    right_patch = sample_centered_patch(
        volume,
        right_centroid,
        PATCH_SHAPE,
    )

    # Mirror the right structure into left-hemisphere orientation.
    right_patch = right_patch[::-1].copy()

    left_patches[index] = left_patch.astype(np.float16)
    right_patches[index] = right_patch.astype(np.float16)

    centroid_separations.append(
        float(np.linalg.norm(
            left_centroid - right_centroid
        ))
    )

    global_hot = float(np.sum(
        np.maximum(volume - 0.50, 0.0)
    ))
    patch_hot = float(
        np.sum(np.maximum(left_patch - 0.50, 0.0))
        + np.sum(np.maximum(right_patch - 0.50, 0.0))
    )

    if global_hot > 1e-8:
        patch_capture_fractions.append(
            min(patch_hot / global_hot, 2.0)
        )

    if (index + 1) % 200 == 0:
        print(
            f"Bilateral canonicalization: "
            f"{index + 1}/{CASE_COUNT}",
            flush=True,
        )

assert np.isfinite(left_patches).all()
assert np.isfinite(right_patches).all()

report = {
    "phase": "phase22_bilateral_landmark_cache",
    "status": "complete",
    "case_count": CASE_COUNT,
    "patch_shape": list(PATCH_SHAPE),
    "patch_spacing_mm": 1.6,
    "patch_field_of_view_mm": 64.0,
    "localization": (
        "independent_half_volume_top4_percent_weighted_centroid"
    ),
    "right_patch_orientation": "mirrored_to_left",
    "fallback_count": fallback_count,
    "fallback_rate": round(
        fallback_count / CASE_COUNT,
        6,
    ),
    "centroid_separation_voxel_quantiles": {
        "q05": round(float(np.quantile(
            centroid_separations, 0.05
        )), 4),
        "q50": round(float(np.quantile(
            centroid_separations, 0.50
        )), 4),
        "q95": round(float(np.quantile(
            centroid_separations, 0.95
        )), 4),
    },
    "hot_uptake_capture_fraction_quantiles": {
        "q05": round(float(np.quantile(
            patch_capture_fractions, 0.05
        )), 4),
        "q50": round(float(np.quantile(
            patch_capture_fractions, 0.50
        )), 4),
        "q95": round(float(np.quantile(
            patch_capture_fractions, 0.95
        )), 4),
    },
    "cache_ram_gb": round(
        (
            left_patches.nbytes
            + right_patches.nbytes
        ) / 1e9,
        4,
    ),
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "training_voxel_arrays_read": False,
    "case_level_patches_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE22_PATCH_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE22_PATCH_CACHE")

PHASE22_LEFT_PATCHES_PRIVATE = left_patches
PHASE22_RIGHT_PATCHES_PRIVATE = right_patches

Bilateral canonicalization: 200/1362
Bilateral canonicalization: 400/1362
Bilateral canonicalization: 600/1362
Bilateral canonicalization: 800/1362
Bilateral canonicalization: 1000/1362
Bilateral canonicalization: 1200/1362
BEGIN SANITIZED_PHASE22_PATCH_CACHE
{
  "phase": "phase22_bilateral_landmark_cache",
  "status": "complete",
  "case_count": 1362,
  "patch_shape": [
    40,
    40,
    40
  ],
  "patch_spacing_mm": 1.6,
  "patch_field_of_view_mm": 64.0,
  "localization": "independent_half_volume_top4_percent_weighted_centroid",
  "right_patch_orientation": "mirrored_to_left",
  "fallback_count": 0,
  "fallback_rate": 0.0,
  "centroid_separation_voxel_quantiles": {
    "q05": 20.0847,
    "q50": 26.8837,
    "q95": 57.7718
  },
  "hot_uptake_capture_fraction_quantiles": {
    "q05": 0.4303,
    "q50": 0.8975,
    "q95": 1.2198
  },
  "cache_ram_gb": 0.3487,
  "elapsed_seconds": 29.25,
  "training_voxel_arrays_read": false,
  "case_level_patches_exported": false,
  "test_or_smoke_da

In [62]:
# Cell 50 — Bilateral patch feature proxy
import json

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def patch_statistics(patch):
    patch = patch.astype(np.float32)

    positive = patch[patch > 0]
    if positive.size == 0:
        positive = np.asarray([0.0], dtype=np.float32)

    y_middle = patch.shape[1] // 2
    z_middle = patch.shape[2] // 2

    gradient = np.gradient(patch)
    gradient_magnitude = np.sqrt(sum(
        component ** 2
        for component in gradient
    ))

    return np.asarray([
        patch.mean(),
        patch.std(),
        np.quantile(positive, 0.50),
        np.quantile(positive, 0.75),
        np.quantile(positive, 0.90),
        np.quantile(positive, 0.95),
        np.quantile(positive, 0.99),
        np.mean(patch > 0.25),
        np.mean(patch > 0.50),
        np.mean(patch > 1.00),
        patch[:, :y_middle].mean(),
        patch[:, y_middle:].mean(),
        patch[:, :, :z_middle].mean(),
        patch[:, :, z_middle:].mean(),
        gradient_magnitude.mean(),
        gradient_magnitude.std(),
    ], dtype=np.float32)

left_statistics = np.stack([
    patch_statistics(patch)
    for patch in PHASE22_LEFT_PATCHES_PRIVATE
])
right_statistics = np.stack([
    patch_statistics(patch)
    for patch in PHASE22_RIGHT_PATCHES_PRIVATE
])

patch_proxy_features = np.concatenate(
    [
        0.5 * (left_statistics + right_statistics),
        np.abs(left_statistics - right_statistics),
    ],
    axis=1,
)

proxy_oof = np.full(EXPECTED_CASES, np.nan)
proxy_fold_rows = []

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    fit_indices = partition["fit"]
    monitor_indices = partition["monitor"]
    outer_indices = partition["outer_valid"]

    candidate_rows = []

    for regularization in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=regularization,
                solver="lbfgs",
                max_iter=3000,
            ),
        )
        model.fit(
            patch_proxy_features[fit_indices],
            PHASE19_Y_PRIVATE[fit_indices],
        )

        monitor_probability = model.predict_proba(
            patch_proxy_features[monitor_indices]
        )[:, 1]

        candidate_rows.append({
            "regularization": regularization,
            "monitor_log_loss": float(log_loss(
                PHASE19_Y_PRIVATE[monitor_indices],
                monitor_probability,
            )),
            "model": model,
        })

    selected = min(
        candidate_rows,
        key=lambda item: item["monitor_log_loss"],
    )

    outer_probability = selected["model"].predict_proba(
        patch_proxy_features[outer_indices]
    )[:, 1]

    proxy_oof[outer_indices] = np.clip(
        outer_probability,
        1e-5,
        1.0 - 1e-5,
    )

    proxy_fold_rows.append({
        "fold": fold_index,
        "selected_regularization": (
            selected["regularization"]
        ),
        "monitor_log_loss": round(
            selected["monitor_log_loss"],
            6,
        ),
        "outer_log_loss": round(float(log_loss(
            PHASE19_Y_PRIVATE[outer_indices],
            proxy_oof[outer_indices],
        )), 6),
        "outer_auroc": round(float(roc_auc_score(
            PHASE19_Y_PRIVATE[outer_indices],
            proxy_oof[outer_indices],
        )), 6),
    })

assert np.isfinite(proxy_oof).all()

report = {
    "phase": "phase22_bilateral_patch_proxy",
    "status": "complete",
    "feature_count": patch_proxy_features.shape[1],
    "metrics": {
        "log_loss": round(float(log_loss(
            PHASE19_Y_PRIVATE,
            proxy_oof,
        )), 6),
        "auroc": round(float(roc_auc_score(
            PHASE19_Y_PRIVATE,
            proxy_oof,
        )), 6),
        "brier": round(float(brier_score_loss(
            PHASE19_Y_PRIVATE,
            proxy_oof,
        )), 6),
    },
    "fold_metrics": proxy_fold_rows,
    "outer_labels_used_only_for_evaluation": True,
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE22_PATCH_PROXY")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE22_PATCH_PROXY")

PHASE22_PATCH_PROXY_OOF_PRIVATE = proxy_oof
PHASE22_PATCH_PROXY_FEATURES_PRIVATE = patch_proxy_features

BEGIN SANITIZED_PHASE22_PATCH_PROXY
{
  "phase": "phase22_bilateral_patch_proxy",
  "status": "complete",
  "feature_count": 32,
  "metrics": {
    "log_loss": 0.483099,
    "auroc": 0.846081,
    "brier": 0.160052
  },
  "fold_metrics": [
    {
      "fold": 0,
      "selected_regularization": 0.01,
      "monitor_log_loss": 0.469111,
      "outer_log_loss": 0.560463,
      "outer_auroc": 0.862405
    },
    {
      "fold": 1,
      "selected_regularization": 0.01,
      "monitor_log_loss": 0.431375,
      "outer_log_loss": 0.496804,
      "outer_auroc": 0.907834
    },
    {
      "fold": 2,
      "selected_regularization": 1.0,
      "monitor_log_loss": 0.346797,
      "outer_log_loss": 0.389735,
      "outer_auroc": 0.904575
    }
  ],
  "outer_labels_used_only_for_evaluation": true,
  "case_level_features_exported": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE22_PATCH_PROXY


In [63]:
# Cell 51 — Bilateral Siamese Phase12 initialization
import copy
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

highres_models_fold0 = [
    item["model"]
    for item in base_entries
    if item["fold"] == 0
    and item["crop"] == "highres"
]

assert len(highres_models_fold0) == 2

class Phase12FeatureEncoder(nn.Module):
    def __init__(self, compact_model):
        super().__init__()
        self.stem = copy.deepcopy(compact_model.stem)
        self.body = copy.deepcopy(compact_model.body)

    def forward(self, x):
        feature_map = self.body(self.stem(x))
        average = feature_map.mean(dim=(2, 3, 4))
        maximum = torch.amax(
            feature_map,
            dim=(2, 3, 4),
        )
        return torch.cat([average, maximum], dim=1)

class Phase22BilateralSiameseNet(nn.Module):
    def __init__(self, compact_model):
        super().__init__()

        self.encoder = Phase12FeatureEncoder(
            compact_model
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(576),
            nn.Linear(576, 192),
            nn.SiLU(),
            nn.Dropout(0.20),
            nn.Linear(192, 96),
            nn.SiLU(),
        )
        self.classifier = nn.Linear(96, 1)
        self.auxiliary = nn.Linear(96, 8)

    def forward(self, left, right, whole):
        left_embedding = self.encoder(left)
        right_embedding = self.encoder(right)

        bilateral_mean = 0.5 * (
            left_embedding + right_embedding
        )
        bilateral_difference = torch.abs(
            left_embedding - right_embedding
        )

        whole_identity = self.encoder(whole)
        whole_reflected = self.encoder(
            torch.flip(whole, dims=(2,))
        )
        whole_embedding = 0.5 * (
            whole_identity + whole_reflected
        )

        fused = torch.cat(
            [
                bilateral_mean,
                bilateral_difference,
                whole_embedding,
            ],
            dim=1,
        )

        embedding = self.fusion(fused)
        return (
            self.classifier(embedding).squeeze(1),
            self.auxiliary(embedding),
        )

phase22_model = Phase22BilateralSiameseNet(
    highres_models_fold0[0]
).to(PHASE19_DEVICE)

# Freeze early representation; train only final residual stages and head.
for parameter in phase22_model.encoder.parameters():
    parameter.requires_grad_(False)

for block_index in (5, 6):
    for parameter in phase22_model.encoder.body[
        block_index
    ].parameters():
        parameter.requires_grad_(True)

left = torch.zeros(
    (2, 1, *PATCH_SHAPE),
    device=PHASE19_DEVICE,
)
right = torch.zeros_like(left)
whole = torch.zeros(
    (2, 1, 80, 80, 80),
    device=PHASE19_DEVICE,
)
target = torch.ones(2, device=PHASE19_DEVICE)

torch.cuda.reset_peak_memory_stats(PHASE19_DEVICE)

logit, auxiliary = phase22_model(
    left,
    right,
    whole,
)

loss = (
    F.binary_cross_entropy_with_logits(
        logit,
        target,
    )
    + 0.03 * auxiliary.square().mean()
)
loss.backward()

gradient_norm = torch.sqrt(sum(
    parameter.grad.detach().square().sum()
    for parameter in phase22_model.parameters()
    if parameter.grad is not None
))

report = {
    "phase": "phase22_bilateral_siamese_architecture",
    "status": "accepted",
    "model": "Phase22BilateralSiameseNet",
    "parameter_count": sum(
        parameter.numel()
        for parameter in phase22_model.parameters()
    ),
    "trainable_parameter_count": sum(
        parameter.numel()
        for parameter in phase22_model.parameters()
        if parameter.requires_grad
    ),
    "patch_shape": list(PATCH_SHAPE),
    "whole_volume_shape": [80, 80, 80],
    "bilateral_aggregation": [
        "embedding_mean",
        "absolute_embedding_difference",
    ],
    "whole_volume_reflection_average": True,
    "initialization": (
        "fold_local_phase12c_highres_compact_model"
    ),
    "logit_shape": list(logit.shape),
    "auxiliary_shape": list(auxiliary.shape),
    "gradient_norm": round(
        float(gradient_norm.cpu()),
        6,
    ),
    "peak_vram_mb": round(
        torch.cuda.max_memory_allocated(
            PHASE19_DEVICE
        ) / 2**20,
        2,
    ),
    "backward_contract_passed": bool(
        torch.isfinite(gradient_norm)
    ),
    "external_weights": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE22_ARCHITECTURE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE22_ARCHITECTURE")

PHASE22_MODEL_CLASS = Phase22BilateralSiameseNet

del phase22_model
del left
del right
del whole
del target
del logit
del auxiliary
torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE22_ARCHITECTURE
{
  "phase": "phase22_bilateral_siamese_architecture",
  "status": "accepted",
  "model": "Phase22BilateralSiameseNet",
  "parameter_count": 1291149,
  "trainable_parameter_count": 1007625,
  "patch_shape": [
    40,
    40,
    40
  ],
  "whole_volume_shape": [
    80,
    80,
    80
  ],
  "bilateral_aggregation": [
    "embedding_mean",
    "absolute_embedding_difference"
  ],
  "whole_volume_reflection_average": true,
  "initialization": "fold_local_phase12c_highres_compact_model",
  "logit_shape": [
    2
  ],
  "auxiliary_shape": [
    2,
    8
  ],
  "gradient_norm": 1.839419,
  "peak_vram_mb": 439.08,
  "backward_contract_passed": true,
  "external_weights": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE22_ARCHITECTURE


In [65]:
# Cell 52A — T4-compatible deterministic policy

import torch

torch.use_deterministic_algorithms(True, warn_only=True)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

report = {
    "phase": "phase23_determinism_compatibility",
    "deterministic_algorithms_enabled":
        torch.are_deterministic_algorithms_enabled(),
    "unsupported_cuda_backward_policy": "warn_only",
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
}

print(report)

assert torch.are_deterministic_algorithms_enabled()
assert torch.is_deterministic_algorithms_warn_only_enabled()

{'phase': 'phase23_determinism_compatibility', 'deterministic_algorithms_enabled': True, 'unsupported_cuda_backward_policy': 'warn_only', 'cudnn_deterministic': True, 'cudnn_benchmark': False, 'tf32_matmul': False, 'tf32_cudnn': False}


In [66]:
# Cell 52 — Full outer-training refit of Phase12c components
import copy
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PHASE23_CONFIG = {
    "epochs": 6,
    "trajectory_average_epochs": [4, 5, 6],
    "batch_size_64": 8,
    "batch_size_80": 4,
    "representation_learning_rate": 1e-5,
    "head_learning_rate": 5e-5,
    "weight_decay": 1e-4,
    "label_smoothing": 0.04,
    "reflection_probability": 0.5,
    "gradient_clip": 2.0,
    "base_seed": 230601,
}

phase23_base_models = copy.deepcopy(
    PHASE19_BASE_MODELS
)
phase23_anatomy_models = copy.deepcopy(
    PHASE19_ANATOMY_MODELS
)

for _, model in flatten_model_dictionary(
    phase23_base_models
):
    model.cpu()

for _, model in flatten_model_dictionary(
    phase23_anatomy_models
):
    model.cpu()

class Phase23VolumeDataset(Dataset):
    def __init__(self, indices, crop):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.crop = crop

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = self.indices[item]

        if self.crop == "160":
            volume = PHASE19_LEGACY_CACHE["160"][index]
        elif self.crop == "192":
            volume = PHASE19_LEGACY_CACHE["192"][index]
        elif self.crop == "highres":
            volume = PHASE19_HIGHRES_CACHE[index]
        else:
            raise KeyError(self.crop)

        volume = torch.from_numpy(
            volume.astype(np.float32)
        )[None]

        label = torch.tensor(
            PHASE19_Y_PRIVATE[index],
            dtype=torch.float32,
        )

        return volume, label

def phase23_trainable_contract(model):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    representation_modules = []
    head_modules = []

    # Final convolutional representation.
    if hasattr(model, "body"):
        for block_index in (5, 6):
            representation_modules.append(
                model.body[block_index]
            )

    if model.__class__.__name__ == "CompactDaTNet":
        head_modules.append(model.classifier)

    elif model.__class__.__name__ == "AnatomyTokenDaTNet":
        head_modules.extend([
            model.region_projection,
            model.transformer,
            model.global_projection,
            model.fusion,
        ])
    else:
        raise TypeError(model.__class__.__name__)

    for module in representation_modules + head_modules:
        for parameter in module.parameters():
            parameter.requires_grad_(True)

    representation_parameters = []
    head_parameters = []

    representation_ids = {
        id(parameter)
        for module in representation_modules
        for parameter in module.parameters()
    }

    head_ids = {
        id(parameter)
        for module in head_modules
        for parameter in module.parameters()
    }

    for parameter in model.parameters():
        if not parameter.requires_grad:
            continue

        if id(parameter) in head_ids:
            head_parameters.append(parameter)
        elif id(parameter) in representation_ids:
            representation_parameters.append(parameter)

    return representation_parameters, head_parameters

def average_state_dicts(states):
    output = {}

    for key in states[0]:
        values = [state[key] for state in states]

        if values[0].is_floating_point():
            output[key] = torch.stack(
                [value.float() for value in values],
                dim=0,
            ).mean(dim=0).to(values[0].dtype)
        else:
            output[key] = values[-1].clone()

    return output

def refit_one_model(
    model,
    crop,
    train_indices,
    seed,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = model.to(PHASE19_DEVICE)

    representation_parameters, head_parameters = (
        phase23_trainable_contract(model)
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": representation_parameters,
                "lr": PHASE23_CONFIG[
                    "representation_learning_rate"
                ],
            },
            {
                "params": head_parameters,
                "lr": PHASE23_CONFIG[
                    "head_learning_rate"
                ],
            },
        ],
        weight_decay=PHASE23_CONFIG["weight_decay"],
    )

    batch_size = (
        PHASE23_CONFIG["batch_size_80"]
        if crop == "highres"
        else PHASE23_CONFIG["batch_size_64"]
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = DataLoader(
        Phase23VolumeDataset(
            train_indices,
            crop,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        generator=generator,
    )

    saved_states = []
    epoch_losses = []

    for epoch in range(
        1,
        PHASE23_CONFIG["epochs"] + 1,
    ):
        model.train()
        losses = []

        for volume, label in loader:
            volume = volume.to(
                PHASE19_DEVICE,
                non_blocking=True,
            )
            label = label.to(
                PHASE19_DEVICE,
                non_blocking=True,
            )

            reflection_mask = (
                torch.rand(
                    len(volume),
                    device=PHASE19_DEVICE,
                )
                < PHASE23_CONFIG[
                    "reflection_probability"
                ]
            )

            if reflection_mask.any():
                volume = volume.clone()
                volume[reflection_mask] = torch.flip(
                    volume[reflection_mask],
                    dims=(2,),
                )

            smoothing = PHASE23_CONFIG["label_smoothing"]
            soft_label = (
                label * (1.0 - smoothing)
                + 0.5 * smoothing
            )

            optimizer.zero_grad(set_to_none=True)

            logit = model(volume)

            loss = F.binary_cross_entropy_with_logits(
                logit,
                soft_label,
            )
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                [
                    parameter
                    for parameter in model.parameters()
                    if parameter.requires_grad
                ],
                PHASE23_CONFIG["gradient_clip"],
            )
            optimizer.step()

            losses.append(float(loss.detach().cpu()))

        epoch_losses.append(float(np.mean(losses)))

        if epoch in PHASE23_CONFIG[
            "trajectory_average_epochs"
        ]:
            saved_states.append({
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            })

    assert len(saved_states) == len(
        PHASE23_CONFIG["trajectory_average_epochs"]
    )

    averaged_state = average_state_dicts(saved_states)
    model.load_state_dict(averaged_state, strict=True)
    model.eval().cpu()

    return {
        "model": model,
        "epoch_losses": epoch_losses,
        "trainable_parameter_count": sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        ),
    }

base_model_entries = flatten_model_dictionary(
    phase23_base_models
)
anatomy_model_entries = flatten_model_dictionary(
    phase23_anatomy_models
)

training_reports = []
started = time.perf_counter()
model_counter = 0

for fold_index in range(3):
    outer_train = PHASE19_PARTITIONS[
        fold_index
    ]["outer_train"]

    fold_entries = []

    for key_path, model in base_model_entries:
        if infer_fold_from_key(key_path) == fold_index:
            fold_entries.append({
                "family": "compact",
                "crop": infer_compact_crop(key_path),
                "model": model,
            })

    for key_path, model in anatomy_model_entries:
        if infer_fold_from_key(key_path) == fold_index:
            fold_entries.append({
                "family": "anatomy",
                "crop": "highres",
                "model": model,
            })

    assert len(fold_entries) == 7

    for local_index, entry in enumerate(fold_entries):
        model_counter += 1
        model_started = time.perf_counter()

        result = refit_one_model(
            model=entry["model"],
            crop=entry["crop"],
            train_indices=outer_train,
            seed=(
                PHASE23_CONFIG["base_seed"]
                + 100 * fold_index
                + local_index
            ),
        )

        # The model object in the copied container was updated in place.
        entry["model"].load_state_dict(
            result["model"].state_dict(),
            strict=True,
        )
        entry["model"].cpu().eval()

        training_reports.append({
            "fold": fold_index,
            "family": entry["family"],
            "crop": entry["crop"],
            "outer_train_n": len(outer_train),
            "trainable_parameter_count": (
                result["trainable_parameter_count"]
            ),
            "epoch_losses": [
                round(value, 6)
                for value in result["epoch_losses"]
            ],
            "training_seconds": round(
                time.perf_counter() - model_started,
                2,
            ),
        })

        print(
            f"Phase23 model {model_counter}/21 complete: "
            f"fold={fold_index}, "
            f"family={entry['family']}, "
            f"crop={entry['crop']}",
            flush=True,
        )

        torch.cuda.empty_cache()

report = {
    "phase": "phase23_full_outer_train_refit",
    "status": "complete",
    "configuration": PHASE23_CONFIG,
    "model_count": len(training_reports),
    "models": training_reports,
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "selection_labels_used": False,
    "monitor_partition_included_in_training": True,
    "outer_validation_labels_used": False,
    "trajectory_weight_averaging": True,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE23_REFIT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE23_REFIT")

PHASE23_BASE_MODELS_PRIVATE = phase23_base_models
PHASE23_ANATOMY_MODELS_PRIVATE = phase23_anatomy_models
PHASE23_TRAINING_REPORTS = training_reports

Phase23 model 1/21 complete: fold=0, family=compact, crop=highres
Phase23 model 2/21 complete: fold=0, family=compact, crop=highres
Phase23 model 3/21 complete: fold=0, family=compact, crop=160
Phase23 model 4/21 complete: fold=0, family=compact, crop=160
Phase23 model 5/21 complete: fold=0, family=compact, crop=192


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_avg_pool3d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Phase23 model 6/21 complete: fold=0, family=anatomy, crop=highres
Phase23 model 7/21 complete: fold=0, family=anatomy, crop=highres
Phase23 model 8/21 complete: fold=1, family=compact, crop=highres
Phase23 model 9/21 complete: fold=1, family=compact, crop=highres
Phase23 model 10/21 complete: fold=1, family=compact, crop=160
Phase23 model 11/21 complete: fold=1, family=compact, crop=160
Phase23 model 12/21 complete: fold=1, family=compact, crop=192
Phase23 model 13/21 complete: fold=1, family=anatomy, crop=highres
Phase23 model 14/21 complete: fold=1, family=anatomy, crop=highres
Phase23 model 15/21 complete: fold=2, family=compact, crop=highres
Phase23 model 16/21 complete: fold=2, family=compact, crop=highres
Phase23 model 17/21 complete: fold=2, family=compact, crop=160
Phase23 model 18/21 complete: fold=2, family=compact, crop=160
Phase23 model 19/21 complete: fold=2, family=compact, crop=192
Phase23 model 20/21 complete: fold=2, family=anatomy, crop=highres
Phase23 model 21/21 com

In [67]:
# Cell 52B — restore strict mode for inference and OOF reconstruction

import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.use_deterministic_algorithms(True, warn_only=False)

report = {
    "phase": "phase23_post_training_determinism",
    "strict_deterministic_inference": (
        torch.are_deterministic_algorithms_enabled()
        and not torch.is_deterministic_algorithms_warn_only_enabled()
    ),
    "cuda_memory_allocated_mb": round(
        torch.cuda.memory_allocated() / 1024**2, 2
    ),
}

print(report)

assert report["strict_deterministic_inference"]

{'phase': 'phase23_post_training_determinism', 'strict_deterministic_inference': True, 'cuda_memory_allocated_mb': 221.37}


In [68]:
# Cell 53 — Phase23 refitted-family OOF inference
import json
import time

import numpy as np
import torch

PHASE23_BATCH_SIZE = 8

phase23_oof = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

fold_runtime_rows = []
started = time.perf_counter()

refit_base_entries = flatten_model_dictionary(
    PHASE23_BASE_MODELS_PRIVATE
)
refit_anatomy_entries = flatten_model_dictionary(
    PHASE23_ANATOMY_MODELS_PRIVATE
)

for fold_index in range(3):
    outer_indices = PHASE19_PARTITIONS[
        fold_index
    ]["outer_valid"]

    active_models = []

    for key_path, model in refit_base_entries:
        if infer_fold_from_key(key_path) == fold_index:
            model.to(PHASE19_DEVICE).eval()
            active_models.append(model)

    for key_path, model in refit_anatomy_entries:
        if infer_fold_from_key(key_path) == fold_index:
            model.to(PHASE19_DEVICE).eval()
            active_models.append(model)

    assert len(active_models) == 7

    fold_predictions = []
    fold_started = time.perf_counter()

    for start in range(
        0,
        len(outer_indices),
        PHASE23_BATCH_SIZE,
    ):
        batch_indices = outer_indices[
            start:start + PHASE23_BATCH_SIZE
        ]

        crops = build_predictor_crops(batch_indices)

        with torch.inference_mode():
            output = PHASE19_FAMILY_MODULE.predict_family_batch(
                crops,
                PHASE19_FOLD_MANIFESTS[
                    fold_index
                ]["family"],
                PHASE19_FOLD_MANIFESTS[
                    fold_index
                ]["base"],
                PHASE23_BASE_MODELS_PRIVATE,
                PHASE23_ANATOMY_MODELS_PRIVATE,
                PHASE19_DEVICE,
            )

        fold_predictions.append(
            PHASE19_NORMALIZE_PREDICTIONS(
                output,
                len(batch_indices),
            )
        )

    probability = np.clip(
        np.concatenate(fold_predictions),
        1e-5,
        1.0 - 1e-5,
    )

    phase23_oof[outer_indices] = probability

    fold_runtime_rows.append({
        "fold": fold_index,
        "n": len(outer_indices),
        "inference_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    print(
        f"Phase23 OOF fold {fold_index} complete.",
        flush=True,
    )

    for model in active_models:
        model.cpu()

    torch.cuda.empty_cache()

assert np.isfinite(phase23_oof).all()

report = {
    "phase": "phase23_refitted_family_oof_inference",
    "status": "complete",
    "case_count": EXPECTED_CASES,
    "folds": fold_runtime_rows,
    "elapsed_seconds": round(
        time.perf_counter() - started,
        2,
    ),
    "original_phase12_calibration_retained": True,
    "case_level_predictions_exported": False,
    "outer_labels_used": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE23_OOF_INFERENCE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE23_OOF_INFERENCE")

PHASE23_OOF_PRIVATE = phase23_oof

Phase23 OOF fold 0 complete.
Phase23 OOF fold 1 complete.
Phase23 OOF fold 2 complete.
BEGIN SANITIZED_PHASE23_OOF_INFERENCE
{
  "phase": "phase23_refitted_family_oof_inference",
  "status": "complete",
  "case_count": 1362,
  "folds": [
    {
      "fold": 0,
      "n": 467,
      "inference_seconds": 8.01
    },
    {
      "fold": 1,
      "n": 443,
      "inference_seconds": 8.15
    },
    {
      "fold": 2,
      "n": 452,
      "inference_seconds": 8.38
    }
  ],
  "elapsed_seconds": 24.85,
  "original_phase12_calibration_retained": true,
  "case_level_predictions_exported": false,
  "outer_labels_used": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE23_OOF_INFERENCE


In [69]:
# Cell 54 — Phase23 promotion report
import json

import numpy as np
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

def phase23_metrics(probability):
    return {
        "log_loss": float(log_loss(
            PHASE19_Y_PRIVATE,
            probability,
        )),
        "auroc": float(roc_auc_score(
            PHASE19_Y_PRIVATE,
            probability,
        )),
        "brier": float(brier_score_loss(
            PHASE19_Y_PRIVATE,
            probability,
        )),
    }

baseline = phase23_metrics(
    PHASE19_PHASE12C_OOF_PRIVATE
)
refitted = phase23_metrics(
    PHASE23_OOF_PRIVATE
)

fold_rows = []
fold_wins = 0
worst_fold_excess = -np.inf

for partition in PHASE19_PARTITIONS:
    fold_index = partition["fold"]
    indices = partition["outer_valid"]
    labels = PHASE19_Y_PRIVATE[indices]

    baseline_loss = float(log_loss(
        labels,
        PHASE19_PHASE12C_OOF_PRIVATE[indices],
    ))
    refitted_loss = float(log_loss(
        labels,
        PHASE23_OOF_PRIVATE[indices],
    ))

    baseline_auc = float(roc_auc_score(
        labels,
        PHASE19_PHASE12C_OOF_PRIVATE[indices],
    ))
    refitted_auc = float(roc_auc_score(
        labels,
        PHASE23_OOF_PRIVATE[indices],
    ))

    improvement = baseline_loss - refitted_loss
    fold_wins += int(improvement > 0)
    worst_fold_excess = max(
        worst_fold_excess,
        -improvement,
    )

    fold_rows.append({
        "fold": fold_index,
        "baseline_log_loss": round(
            baseline_loss,
            6,
        ),
        "phase23_log_loss": round(
            refitted_loss,
            6,
        ),
        "log_loss_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_auc,
            6,
        ),
        "phase23_auroc": round(
            refitted_auc,
            6,
        ),
    })

log_loss_gain = (
    baseline["log_loss"] - refitted["log_loss"]
)
auroc_gain = (
    refitted["auroc"] - baseline["auroc"]
)
brier_excess = (
    refitted["brier"] - baseline["brier"]
)

promotion_passed = (
    log_loss_gain >= 0.003
    and auroc_gain >= 0.002
    and fold_wins >= 2
    and worst_fold_excess <= 0.005
    and brier_excess <= 0.001
)

report = {
    "phase": "phase23_full_outer_train_refit",
    "status": (
        "promoted"
        if promotion_passed
        else "not_promoted"
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline.items()
    },
    "phase23": {
        key: round(value, 6)
        for key, value in refitted.items()
    },
    "improvements": {
        "log_loss_gain": round(
            log_loss_gain,
            6,
        ),
        "auroc_gain": round(
            auroc_gain,
            6,
        ),
        "brier_excess": round(
            brier_excess,
            6,
        ),
        "fold_wins": fold_wins,
        "worst_fold_excess": round(
            float(worst_fold_excess),
            6,
        ),
    },
    "fold_metrics": fold_rows,
    "promotion_gate_passed": promotion_passed,
    "refitted_weights_retained_privately": True,
    "outer_labels_used_only_for_final_evaluation": True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE23_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE23_OOF")

PHASE23_METRICS = refitted
PHASE23_PROMOTED = promotion_passed
PHASE23_PROMOTION_REPORT = report

BEGIN SANITIZED_PHASE23_OOF
{
  "phase": "phase23_full_outer_train_refit",
  "status": "not_promoted",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "phase23": {
    "log_loss": 0.315653,
    "auroc": 0.938906,
    "brier": 0.097437
  },
  "improvements": {
    "log_loss_gain": -0.008037,
    "auroc_gain": -2.2e-05,
    "brier_excess": 0.001902,
    "fold_wins": 2,
    "worst_fold_excess": 0.034207
  },
  "fold_metrics": [
    {
      "fold": 0,
      "baseline_log_loss": 0.358494,
      "phase23_log_loss": 0.358463,
      "log_loss_improvement": 3.1e-05,
      "baseline_auroc": 0.916174,
      "phase23_auroc": 0.920284
    },
    {
      "fold": 1,
      "baseline_log_loss": 0.255415,
      "phase23_log_loss": 0.289623,
      "log_loss_improvement": -0.034207,
      "baseline_auroc": 0.952183,
      "phase23_auroc": 0.951408
    },
    {
      "fold": 2,
      "baseline_log_loss": 0.306209,
      "phase23_log_loss": 0.296933,
      "log_

In [70]:
# Cell 55 — Phase23 anchored-delta diagnostic
# Outer-fold labels are used only for aggregate diagnosis.
# The oracle blend weights produced here are NOT deployable.

import json
import numpy as np
from scipy.special import expit, logit
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss

PHASE23_DIAGNOSTIC_ALPHAS = np.round(
    np.linspace(0.0, 1.0, 21), 2
)

y = np.asarray(PHASE19_Y_PRIVATE, dtype=np.int64)
p12 = np.clip(
    np.asarray(PHASE19_PHASE12C_OOF_PRIVATE, dtype=np.float64),
    1e-5,
    1.0 - 1e-5,
)
p23 = np.clip(
    np.asarray(PHASE23_OOF_PRIVATE, dtype=np.float64),
    1e-5,
    1.0 - 1e-5,
)

assert y.shape == p12.shape == p23.shape == (1362,)
assert np.isfinite(p12).all() and np.isfinite(p23).all()

z12 = logit(p12)
z23 = logit(p23)
delta = z23 - z12


def phase23_outer_indices(fold):
    partitions = PHASE19_PARTITIONS

    if isinstance(partitions, dict):
        part = partitions[fold] if fold in partitions else partitions[str(fold)]
    else:
        matching = [
            item for item in partitions
            if isinstance(item, dict)
            and int(item.get("fold", -1)) == fold
        ]
        part = matching[0] if matching else partitions[fold]

    exact_keys = [
        "outer_valid",
        "outer_valid_indices",
        "outer_validation",
        "outer_validation_indices",
        "valid",
        "valid_indices",
    ]

    for key in exact_keys:
        if key in part:
            return np.asarray(part[key], dtype=np.int64)

    candidates = []
    for key, value in part.items():
        normalized = str(key).lower().replace("-", "_")
        if (
            "outer" in normalized
            and ("valid" in normalized or "validation" in normalized)
        ):
            array = np.asarray(value)
            if array.ndim == 1 and np.issubdtype(array.dtype, np.integer):
                candidates.append(array.astype(np.int64))

    assert len(candidates) == 1, {
        "message": "Could not uniquely resolve outer-validation indices.",
        "fold": fold,
        "candidate_count": len(candidates),
    }
    return candidates[0]


fold_indices = {
    fold: phase23_outer_indices(fold)
    for fold in range(3)
}

assert sum(len(v) for v in fold_indices.values()) == len(y)
assert len(np.unique(np.concatenate(list(fold_indices.values())))) == len(y)


def evaluate(indices, probability):
    target = y[indices]
    prediction = np.clip(probability[indices], 1e-5, 1.0 - 1e-5)

    return {
        "log_loss": float(log_loss(target, prediction, labels=[0, 1])),
        "auroc": float(roc_auc_score(target, prediction)),
        "brier": float(brier_score_loss(target, prediction)),
    }


curve = []
for alpha in PHASE23_DIAGNOSTIC_ALPHAS:
    blended = expit(z12 + float(alpha) * delta)
    metrics = evaluate(np.arange(len(y)), blended)

    curve.append({
        "alpha": float(alpha),
        "log_loss": metrics["log_loss"],
        "auroc": metrics["auroc"],
        "brier": metrics["brier"],
    })

global_best = min(
    curve,
    key=lambda row: (row["log_loss"], row["alpha"]),
)

fold_diagnostics = []
for fold, indices in fold_indices.items():
    fold_curve = []

    for alpha in PHASE23_DIAGNOSTIC_ALPHAS:
        blended = expit(z12 + float(alpha) * delta)
        metrics = evaluate(indices, blended)

        fold_curve.append({
            "alpha": float(alpha),
            **metrics,
        })

    best = min(
        fold_curve,
        key=lambda row: (row["log_loss"], row["alpha"]),
    )

    residual = y[indices].astype(np.float64) - p12[indices]
    local_delta = delta[indices]

    if np.std(local_delta) > 0 and np.std(residual) > 0:
        correction_correlation = float(
            np.corrcoef(local_delta, residual)[0, 1]
        )
    else:
        correction_correlation = 0.0

    fold_diagnostics.append({
        "fold": fold,
        "n": int(len(indices)),
        "oracle_best_alpha_diagnostic_only": best["alpha"],
        "oracle_best_log_loss": round(best["log_loss"], 6),
        "oracle_log_loss_gain": round(
            evaluate(indices, p12)["log_loss"] - best["log_loss"], 6
        ),
        "auroc_at_oracle_alpha": round(best["auroc"], 6),
        "delta_residual_correlation": round(correction_correlation, 6),
    })

baseline_metrics = evaluate(np.arange(len(y)), p12)

report = {
    "phase": "phase23_anchored_delta_diagnostic",
    "status": "diagnostic_only_no_selection",
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "global_oracle": {
        "alpha": global_best["alpha"],
        "log_loss": round(global_best["log_loss"], 6),
        "auroc": round(global_best["auroc"], 6),
        "brier": round(global_best["brier"], 6),
        "log_loss_gain": round(
            baseline_metrics["log_loss"] - global_best["log_loss"], 6
        ),
        "auroc_gain": round(
            global_best["auroc"] - baseline_metrics["auroc"], 6
        ),
    },
    "fold_diagnostics": fold_diagnostics,
    "logit_update_distribution": {
        "mean_absolute": round(float(np.mean(np.abs(delta))), 6),
        "q05": round(float(np.quantile(delta, 0.05)), 6),
        "q50": round(float(np.quantile(delta, 0.50)), 6),
        "q95": round(float(np.quantile(delta, 0.95)), 6),
    },
    "predeclared_next_stage_gate": {
        "minimum_global_oracle_log_loss_gain": 0.002,
        "minimum_folds_with_oracle_gain_0p002": 2,
        "meaning": (
            "If passed, train fold-local shadow refits on fit partitions "
            "and choose anchored-delta weights using monitor partitions."
        ),
    },
    "oracle_weights_deployable": False,
    "case_level_predictions_exported": False,
    "outer_labels_used_only_for_aggregate_diagnostic": True,
    "test_or_smoke_data_read": False,
}

fold_gate_count = sum(
    row["oracle_log_loss_gain"] >= 0.002
    for row in fold_diagnostics
)

report["predeclared_next_stage_gate"]["fold_count_passing"] = int(
    fold_gate_count
)
report["predeclared_next_stage_gate"]["passed"] = bool(
    report["global_oracle"]["log_loss_gain"] >= 0.002
    and fold_gate_count >= 2
)

print("BEGIN SANITIZED_PHASE23_ANCHORED_DIAGNOSTIC")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE23_ANCHORED_DIAGNOSTIC")

BEGIN SANITIZED_PHASE23_ANCHORED_DIAGNOSTIC
{
  "phase": "phase23_anchored_delta_diagnostic",
  "status": "diagnostic_only_no_selection",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "global_oracle": {
    "alpha": 0.3,
    "log_loss": 0.30607,
    "auroc": 0.940029,
    "brier": 0.094917,
    "log_loss_gain": 0.001546,
    "auroc_gain": 0.001101
  },
  "fold_diagnostics": [
    {
      "fold": 0,
      "n": 467,
      "oracle_best_alpha_diagnostic_only": 0.5,
      "oracle_best_log_loss": 0.35586,
      "oracle_log_loss_gain": 0.002634,
      "auroc_at_oracle_alpha": 0.919053,
      "delta_residual_correlation": 0.054499
    },
    {
      "fold": 1,
      "n": 443,
      "oracle_best_alpha_diagnostic_only": 0.0,
      "oracle_best_log_loss": 0.255415,
      "oracle_log_loss_gain": 0.0,
      "auroc_at_oracle_alpha": 0.952183,
      "delta_residual_correlation": -0.047827
    },
    {
      "fold": 2,
      "n": 452,
      "oracle_best_

In [73]:
# Cell 56 — Phase24 live-object contract audit
# Sanitized structure only: no indices, labels, predictions, UIDs,
# embeddings, model values, or hashes are displayed.

import inspect
import json
import numpy as np
import torch


def safe_signature(obj):
    try:
        return str(inspect.signature(obj))
    except Exception:
        return "unavailable"


def resolve_partition_container(fold):
    partitions = PHASE19_PARTITIONS

    if isinstance(partitions, dict):
        if fold in partitions:
            return partitions[fold]
        if str(fold) in partitions:
            return partitions[str(fold)]
        return None

    if isinstance(partitions, (list, tuple)):
        matching = [
            value for value in partitions
            if isinstance(value, dict)
            and int(value.get("fold", -1)) == fold
        ]
        if matching:
            return matching[0]
        if fold < len(partitions):
            return partitions[fold]

    return None


def sanitized_partition_schema(partition):
    if not isinstance(partition, dict):
        return {
            "type": type(partition).__name__,
            "keys": [],
        }

    result = {}

    for key, value in partition.items():
        entry = {
            "type": type(value).__name__,
        }

        if isinstance(value, (list, tuple, np.ndarray)):
            array = np.asarray(value)
            entry["shape"] = list(array.shape)
            entry["dtype_category"] = (
                "integer"
                if np.issubdtype(array.dtype, np.integer)
                else "non_integer"
            )
        elif isinstance(value, (int, np.integer)):
            entry["scalar_category"] = "integer"
        elif isinstance(value, (float, np.floating)):
            entry["scalar_category"] = "floating"
        elif isinstance(value, str):
            entry["scalar_category"] = "string"

        result[str(key)] = entry

    return {
        "type": type(partition).__name__,
        "keys": sorted(str(key) for key in partition.keys()),
        "fields": result,
    }


def sanitized_container_schema(value):
    schema = {
        "type": type(value).__name__,
    }

    if isinstance(value, dict):
        schema.update({
            "length": len(value),
            "key_types": sorted({
                type(key).__name__
                for key in value.keys()
            }),
            "value_types": sorted({
                type(item).__name__
                for item in value.values()
            }),
        })

    elif isinstance(value, (list, tuple)):
        schema.update({
            "length": len(value),
            "element_types": sorted({
                type(item).__name__
                for item in value
            }),
        })

    return schema


def summarize_model_collection(collection, family):
    flattened = list(flatten_model_dictionary(collection))
    fold_counts = {0: 0, 1: 0, 2: 0}
    crop_counts = {}

    failures = 0
    for key_path, model in flattened:
        try:
            fold = int(infer_fold_from_key(key_path))
            fold_counts[fold] += 1

            if family == "compact":
                crop = str(infer_compact_crop(key_path))
            else:
                crop = "highres"

            crop_counts[crop] = crop_counts.get(crop, 0) + 1

            assert isinstance(model, torch.nn.Module)
        except Exception:
            failures += 1

    return {
        "collection_type": type(collection).__name__,
        "model_count": len(flattened),
        "fold_counts": fold_counts,
        "crop_counts": crop_counts,
        "schema_failure_count": failures,
    }


required_global_names = [
    "PHASE19_PARTITIONS",
    "PHASE19_BASE_MODELS",
    "PHASE19_ANATOMY_MODELS",
    "PHASE23_BASE_MODELS_PRIVATE",
    "PHASE23_ANATOMY_MODELS_PRIVATE",
    "PHASE19_PHASE12C_OOF_PRIVATE",
    "PHASE23_OOF_PRIVATE",
    "PHASE19_FOLD_MANIFESTS",
    "PHASE19_FAMILY_MODULE",
    "PHASE19_BASE_MANIFEST",
    "PHASE19_FAMILY_MANIFEST",
    "build_predictor_crops",
    "refit_one_model",
    "flatten_model_dictionary",
    "infer_fold_from_key",
    "infer_compact_crop",
]

availability = {
    name: name in globals()
    for name in required_global_names
}

callable_contracts = {}
for name in [
    "build_predictor_crops",
    "refit_one_model",
    "flatten_model_dictionary",
    "infer_fold_from_key",
    "infer_compact_crop",
]:
    if name in globals():
        callable_contracts[name] = safe_signature(globals()[name])

module_contracts = {}
if "PHASE19_FAMILY_MODULE" in globals():
    module = PHASE19_FAMILY_MODULE

    for attribute in [
        "predict_family_batch",
        "load_anatomy_bundle",
        "main",
    ]:
        if hasattr(module, attribute):
            module_contracts[attribute] = safe_signature(
                getattr(module, attribute)
            )

partition_schemas = []
for fold in range(3):
    partition = resolve_partition_container(fold)

    partition_schemas.append({
        "fold": fold,
        "schema": sanitized_partition_schema(partition),
    })

manifest_candidates = sorted(
    name for name in globals()
    if "MANIFEST" in name.upper()
    and not name.startswith("__")
)

prediction_callable_candidates = sorted(
    name for name, value in globals().items()
    if "predict" in name.lower()
    and callable(value)
    and not name.startswith("__")
)

container_contracts = {}
for name in [
    "PHASE19_FOLD_MANIFESTS",
    "PHASE19_BASE_MANIFEST",
    "PHASE19_FAMILY_MANIFEST",
]:
    if name in globals():
        container_contracts[name] = sanitized_container_schema(
            globals()[name]
        )

report = {
    "phase": "phase24_shadow_refit_contract",
    "status": "audit_complete",
    "required_global_availability": availability,
    "callable_contracts": callable_contracts,
    "family_module_contracts": module_contracts,
    "partition_schemas": partition_schemas,
    "model_collections": {
        "original_compact": summarize_model_collection(
            PHASE19_BASE_MODELS,
            "compact",
        ),
        "original_anatomy": summarize_model_collection(
            PHASE19_ANATOMY_MODELS,
            "anatomy",
        ),
        "phase23_compact": summarize_model_collection(
            PHASE23_BASE_MODELS_PRIVATE,
            "compact",
        ),
        "phase23_anatomy": summarize_model_collection(
            PHASE23_ANATOMY_MODELS_PRIVATE,
            "anatomy",
        ),
    },
    "manifest_candidate_names": manifest_candidates,
    "prediction_callable_candidate_names":
        prediction_callable_candidates,
    "container_contracts": container_contracts,
    "proposed_phase24": {
        "shadow_training_partition": "fit_only",
        "selection_partition": "monitor_only",
        "final_outer_evaluation": "once_after_weights_frozen",
        "candidate_alphas": [0.0, 0.25, 0.5, 0.75, 1.0],
        "minimum_monitor_log_loss_gain": 0.002,
        "tie_break": "smallest_alpha",
        "outer_oracle_alphas_used": False,
    },
    "challenge_values_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

assert report["model_collections"]["original_compact"]["model_count"] == 15
assert report["model_collections"]["original_anatomy"]["model_count"] == 6
assert report["model_collections"]["phase23_compact"]["model_count"] == 15
assert report["model_collections"]["phase23_anatomy"]["model_count"] == 6

print("BEGIN SANITIZED_PHASE24_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE24_CONTRACT")

BEGIN SANITIZED_PHASE24_CONTRACT
{
  "phase": "phase24_shadow_refit_contract",
  "status": "audit_complete",
  "required_global_availability": {
    "PHASE19_PARTITIONS": true,
    "PHASE19_BASE_MODELS": true,
    "PHASE19_ANATOMY_MODELS": true,
    "PHASE23_BASE_MODELS_PRIVATE": true,
    "PHASE23_ANATOMY_MODELS_PRIVATE": true,
    "PHASE19_PHASE12C_OOF_PRIVATE": true,
    "PHASE23_OOF_PRIVATE": true,
    "PHASE19_FOLD_MANIFESTS": true,
    "PHASE19_FAMILY_MODULE": true,
    "PHASE19_BASE_MANIFEST": false,
    "PHASE19_FAMILY_MANIFEST": false,
    "build_predictor_crops": true,
    "refit_one_model": true,
    "flatten_model_dictionary": true,
    "infer_fold_from_key": true,
    "infer_compact_crop": true
  },
  "callable_contracts": {
    "build_predictor_crops": "(indices)",
    "refit_one_model": "(model, crop, train_indices, seed)",
    "flatten_model_dictionary": "(container, prefix=())",
    "infer_fold_from_key": "(key_path)",
    "infer_compact_crop": "(key_path)"
  },
  "fam

In [74]:
# Cell 57 — Phase24 fit-only shadow refits

import copy
import gc
import json
import time
import numpy as np
import torch

PHASE24_CONFIG = {
    "candidate_alphas": [0.0, 0.25, 0.5, 0.75, 1.0],
    "minimum_monitor_log_loss_gain": 0.002,
    "base_seed": 240601,
}

# Required only for adaptive_avg_pool3d backward on T4.
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


def phase24_move_collection_cpu(collection):
    for _, model in flatten_model_dictionary(collection):
        model.cpu()
    gc.collect()
    torch.cuda.empty_cache()


# Ensure copying occurs from CPU states.
phase24_move_collection_cpu(PHASE19_BASE_MODELS)
phase24_move_collection_cpu(PHASE19_ANATOMY_MODELS)

PHASE24_SHADOW_BASE_MODELS_PRIVATE = copy.deepcopy(
    PHASE19_BASE_MODELS
)
PHASE24_SHADOW_ANATOMY_MODELS_PRIVATE = copy.deepcopy(
    PHASE19_ANATOMY_MODELS
)


phase24_entries = []

for key_path, model in flatten_model_dictionary(
    PHASE24_SHADOW_BASE_MODELS_PRIVATE
):
    phase24_entries.append({
        "key_path": key_path,
        "model": model,
        "fold": int(infer_fold_from_key(key_path)),
        "family": "compact",
        "crop": str(infer_compact_crop(key_path)),
    })

for key_path, model in flatten_model_dictionary(
    PHASE24_SHADOW_ANATOMY_MODELS_PRIVATE
):
    phase24_entries.append({
        "key_path": key_path,
        "model": model,
        "fold": int(infer_fold_from_key(key_path)),
        "family": "anatomy",
        "crop": "highres",
    })


crop_order = {
    "highres": 0,
    "160": 1,
    "192": 2,
}

phase24_entries.sort(
    key=lambda entry: (
        entry["fold"],
        0 if entry["family"] == "compact" else 1,
        crop_order.get(entry["crop"], 99),
    )
)

assert len(phase24_entries) == 21
assert sum(e["family"] == "compact" for e in phase24_entries) == 15
assert sum(e["family"] == "anatomy" for e in phase24_entries) == 6

phase24_training_records = []
phase24_started = time.perf_counter()

for model_number, entry in enumerate(phase24_entries, start=1):
    fold = entry["fold"]
    fit_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["fit"],
        dtype=np.int64,
    )

    seed = (
        PHASE24_CONFIG["base_seed"]
        + fold * 100
        + model_number
    )

    started = time.perf_counter()

    result = refit_one_model(
        model=entry["model"],
        crop=entry["crop"],
        train_indices=fit_indices,
        seed=seed,
    )

    if isinstance(result, dict):
        returned_model = result.get("model")

        if (
            returned_model is not None
            and returned_model is not entry["model"]
        ):
            entry["model"].load_state_dict(
                returned_model.state_dict()
            )

        epoch_losses = [
            float(value)
            for value in result.get("epoch_losses", [])
        ]

        trainable_count = int(
            result.get("trainable_parameter_count", 0)
        )
    else:
        epoch_losses = []
        trainable_count = int(sum(
            parameter.numel()
            for parameter in entry["model"].parameters()
            if parameter.requires_grad
        ))

    entry["model"].cpu()

    record = {
        "fold": fold,
        "family": entry["family"],
        "crop": entry["crop"],
        "fit_n": int(len(fit_indices)),
        "seed": int(seed),
        "trainable_parameter_count": trainable_count,
        "initial_epoch_loss": (
            round(epoch_losses[0], 6)
            if epoch_losses else None
        ),
        "final_epoch_loss": (
            round(epoch_losses[-1], 6)
            if epoch_losses else None
        ),
        "minimum_epoch_loss": (
            round(min(epoch_losses), 6)
            if epoch_losses else None
        ),
        "training_seconds": round(
            time.perf_counter() - started,
            2,
        ),
    }

    phase24_training_records.append(record)

    print(
        f"Phase24 shadow {model_number}/21 complete: "
        f"fold={fold}, family={entry['family']}, "
        f"crop={entry['crop']}"
    )

gc.collect()
torch.cuda.empty_cache()

phase24_elapsed = time.perf_counter() - phase24_started

phase24_shadow_report = {
    "phase": "phase24_fit_only_shadow_refit",
    "status": "complete",
    "configuration": {
        "epochs": PHASE23_CONFIG["epochs"],
        "trajectory_average_epochs":
            PHASE23_CONFIG["trajectory_average_epochs"],
        "representation_learning_rate":
            PHASE23_CONFIG["representation_learning_rate"],
        "head_learning_rate":
            PHASE23_CONFIG["head_learning_rate"],
        "label_smoothing":
            PHASE23_CONFIG["label_smoothing"],
        "reflection_probability":
            PHASE23_CONFIG["reflection_probability"],
    },
    "model_count": len(phase24_training_records),
    "compact_model_count": sum(
        row["family"] == "compact"
        for row in phase24_training_records
    ),
    "anatomy_model_count": sum(
        row["family"] == "anatomy"
        for row in phase24_training_records
    ),
    "fold_fit_counts": {
        str(fold): int(len(PHASE19_PARTITIONS[fold]["fit"]))
        for fold in range(3)
    },
    "models": phase24_training_records,
    "elapsed_seconds": round(phase24_elapsed, 2),
    "monitor_labels_used": False,
    "outer_validation_labels_used": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE24_SHADOW_REFIT")
print(json.dumps(phase24_shadow_report, indent=2))
print("END SANITIZED_PHASE24_SHADOW_REFIT")

Phase24 shadow 1/21 complete: fold=0, family=compact, crop=highres
Phase24 shadow 2/21 complete: fold=0, family=compact, crop=highres
Phase24 shadow 3/21 complete: fold=0, family=compact, crop=160
Phase24 shadow 4/21 complete: fold=0, family=compact, crop=160
Phase24 shadow 5/21 complete: fold=0, family=compact, crop=192


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_avg_pool3d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Phase24 shadow 6/21 complete: fold=0, family=anatomy, crop=highres
Phase24 shadow 7/21 complete: fold=0, family=anatomy, crop=highres
Phase24 shadow 8/21 complete: fold=1, family=compact, crop=highres
Phase24 shadow 9/21 complete: fold=1, family=compact, crop=highres
Phase24 shadow 10/21 complete: fold=1, family=compact, crop=160
Phase24 shadow 11/21 complete: fold=1, family=compact, crop=160
Phase24 shadow 12/21 complete: fold=1, family=compact, crop=192
Phase24 shadow 13/21 complete: fold=1, family=anatomy, crop=highres
Phase24 shadow 14/21 complete: fold=1, family=anatomy, crop=highres
Phase24 shadow 15/21 complete: fold=2, family=compact, crop=highres
Phase24 shadow 16/21 complete: fold=2, family=compact, crop=highres
Phase24 shadow 17/21 complete: fold=2, family=compact, crop=160
Phase24 shadow 18/21 complete: fold=2, family=compact, crop=160
Phase24 shadow 19/21 complete: fold=2, family=compact, crop=192
Phase24 shadow 20/21 complete: fold=2, family=anatomy, crop=highres
Phase24 

In [75]:
# Cell 58 — Phase24 monitor inference and locked alpha selection

import gc
import json
import time
import numpy as np
import torch
from scipy.special import expit, logit
from sklearn.metrics import log_loss, roc_auc_score

PHASE24_DEVICE = torch.device("cuda:0")


def phase24_resolve_manifest_pair(fold):
    item = PHASE19_FOLD_MANIFESTS[fold]

    family_manifest = None
    base_manifest = None
    family_source = None
    base_source = None

    if isinstance(item, dict):
        for key in [
            "family_manifest",
            "phase12c_manifest",
            "manifest",
            "family",
        ]:
            if key in item and isinstance(item[key], dict):
                family_manifest = item[key]
                family_source = f"fold_manifest:{key}"
                break

        for key in [
            "base_manifest",
            "phase12c_base_manifest",
            "base",
        ]:
            if key in item and isinstance(item[key], dict):
                base_manifest = item[key]
                base_source = f"fold_manifest:{key}"
                break

    if family_manifest is None:
        for name in [
            "PHASE19_FAMILY_MANIFEST_RUNTIME",
            "PHASE19_PHASE12C_MANIFEST",
            "PHASE12C_RESTORED_MANIFEST",
        ]:
            if name in globals():
                family_manifest = globals()[name]
                family_source = name
                break

    if base_manifest is None:
        for name in [
            "PHASE19_BASE_MANIFEST_RUNTIME",
            "PHASE19_PHASE12C_BASE_MANIFEST",
            "PHASE12C_RESTORED_BASE_MANIFEST",
        ]:
            if name in globals():
                base_manifest = globals()[name]
                base_source = name
                break

    assert isinstance(family_manifest, dict), {
        "message": "Family manifest could not be resolved.",
        "fold": fold,
    }
    assert isinstance(base_manifest, dict), {
        "message": "Base manifest could not be resolved.",
        "fold": fold,
    }

    return (
        family_manifest,
        base_manifest,
        family_source,
        base_source,
    )


def phase24_activate_fold(collection, fold, device):
    active_count = 0

    for key_path, model in flatten_model_dictionary(collection):
        model.requires_grad_(False)
        model.eval()

        if int(infer_fold_from_key(key_path)) == fold:
            model.to(device)
            active_count += 1
        else:
            model.cpu()

    return active_count


def phase24_normalize_output(raw_output, expected_n):
    normalized = None

    # Prefer the notebook's already validated normalizer.
    try:
        normalized = PHASE19_NORMALIZE_PREDICTIONS(raw_output)
    except TypeError:
        try:
            normalized = PHASE19_NORMALIZE_PREDICTIONS(
                raw_output,
                expected_n,
            )
        except Exception:
            normalized = None
    except Exception:
        normalized = None

    if normalized is None:
        normalized = raw_output

    if isinstance(normalized, dict):
        selected = None

        for key in [
            "probabilities",
            "probability",
            "predictions",
            "prediction",
            "pathologic",
            "output",
        ]:
            if key in normalized:
                selected = normalized[key]
                break

        assert selected is not None, {
            "message": "Unsupported prediction dictionary.",
            "available_key_count": len(normalized),
        }
        normalized = selected

    if isinstance(normalized, (list, tuple)):
        assert len(normalized) == 1, {
            "message": "Ambiguous prediction sequence.",
            "sequence_length": len(normalized),
        }
        normalized = normalized[0]

    if torch.is_tensor(normalized):
        normalized = normalized.detach().float().cpu().numpy()

    array = np.asarray(normalized, dtype=np.float64).reshape(-1)

    assert array.size == expected_n, {
        "expected": expected_n,
        "observed": int(array.size),
    }
    assert np.isfinite(array).all()

    if array.min() < 0.0 or array.max() > 1.0:
        array = expit(array)

    return np.clip(array, 1e-5, 1.0 - 1e-5)


def phase24_predict_indices(
    indices,
    fold,
    base_models,
    anatomy_models,
):
    family_manifest, base_manifest, family_source, base_source = (
        phase24_resolve_manifest_pair(fold)
    )

    compact_count = phase24_activate_fold(
        base_models,
        fold,
        PHASE24_DEVICE,
    )
    anatomy_count = phase24_activate_fold(
        anatomy_models,
        fold,
        PHASE24_DEVICE,
    )

    assert compact_count == 5
    assert anatomy_count == 2

    crops = build_predictor_crops(indices)

    with torch.inference_mode():
        raw = PHASE19_FAMILY_MODULE.predict_family_batch(
            crops,
            family_manifest,
            base_manifest,
            base_models,
            anatomy_models,
            PHASE24_DEVICE,
        )

    probability = phase24_normalize_output(
        raw,
        len(indices),
    )

    phase24_move_collection_cpu(base_models)
    phase24_move_collection_cpu(anatomy_models)

    return probability, family_source, base_source


PHASE24_BASE_MONITOR_PRIVATE = {}
PHASE24_SHADOW_MONITOR_PRIVATE = {}
PHASE24_SELECTED_ALPHA_PRIVATE = {}

phase24_selection_records = []
phase24_manifest_sources = []
phase24_monitor_started = time.perf_counter()

for fold in range(3):
    monitor_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["monitor"],
        dtype=np.int64,
    )
    monitor_y = np.asarray(
        PHASE19_Y_PRIVATE[monitor_indices],
        dtype=np.int64,
    )

    baseline_probability, family_source, base_source = (
        phase24_predict_indices(
            monitor_indices,
            fold,
            PHASE19_BASE_MODELS,
            PHASE19_ANATOMY_MODELS,
        )
    )

    shadow_probability, _, _ = phase24_predict_indices(
        monitor_indices,
        fold,
        PHASE24_SHADOW_BASE_MODELS_PRIVATE,
        PHASE24_SHADOW_ANATOMY_MODELS_PRIVATE,
    )

    PHASE24_BASE_MONITOR_PRIVATE[fold] = baseline_probability
    PHASE24_SHADOW_MONITOR_PRIVATE[fold] = shadow_probability

    baseline_logit = logit(baseline_probability)
    shadow_logit = logit(shadow_probability)
    update = shadow_logit - baseline_logit

    baseline_loss = float(log_loss(
        monitor_y,
        baseline_probability,
        labels=[0, 1],
    ))

    candidates = []

    for alpha in PHASE24_CONFIG["candidate_alphas"]:
        candidate_probability = expit(
            baseline_logit + float(alpha) * update
        )

        candidates.append({
            "alpha": float(alpha),
            "log_loss": float(log_loss(
                monitor_y,
                candidate_probability,
                labels=[0, 1],
            )),
            "auroc": float(roc_auc_score(
                monitor_y,
                candidate_probability,
            )),
        })

    raw_best = min(
        candidates,
        key=lambda row: (
            row["log_loss"],
            row["alpha"],
        ),
    )

    raw_gain = baseline_loss - raw_best["log_loss"]

    if (
        raw_best["alpha"] > 0.0
        and raw_gain
        >= PHASE24_CONFIG["minimum_monitor_log_loss_gain"]
    ):
        selected_alpha = float(raw_best["alpha"])
        advanced = True
        selected_loss = float(raw_best["log_loss"])
    else:
        selected_alpha = 0.0
        advanced = False
        selected_loss = baseline_loss

    PHASE24_SELECTED_ALPHA_PRIVATE[fold] = selected_alpha

    phase24_selection_records.append({
        "fold": fold,
        "fit_n": int(len(PHASE19_PARTITIONS[fold]["fit"])),
        "monitor_n": int(len(monitor_indices)),
        "baseline_monitor_log_loss": round(
            baseline_loss,
            6,
        ),
        "raw_best_alpha": float(raw_best["alpha"]),
        "raw_best_monitor_log_loss": round(
            raw_best["log_loss"],
            6,
        ),
        "raw_monitor_gain": round(raw_gain, 6),
        "selected_alpha": selected_alpha,
        "selected_monitor_log_loss": round(
            selected_loss,
            6,
        ),
        "correction_advanced": advanced,
        "candidate_count": len(candidates),
    })

    phase24_manifest_sources.append({
        "fold": fold,
        "family_source": family_source,
        "base_source": base_source,
    })

    print(
        f"Phase24 monitor fold {fold}: "
        f"selected_alpha={selected_alpha:.2f}, "
        f"advanced={advanced}"
    )

phase24_monitor_elapsed = (
    time.perf_counter() - phase24_monitor_started
)

phase24_selection_report = {
    "phase": "phase24_monitor_only_selection",
    "status": "complete",
    "configuration": PHASE24_CONFIG,
    "folds": phase24_selection_records,
    "manifest_sources": phase24_manifest_sources,
    "elapsed_seconds": round(phase24_monitor_elapsed, 2),
    "outer_oracle_alphas_used": False,
    "outer_validation_labels_used": False,
    "monitor_labels_used_for_selection": True,
    "selection_frozen_before_outer_evaluation": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE24_MONITOR_SELECTION")
print(json.dumps(phase24_selection_report, indent=2))
print("END SANITIZED_PHASE24_MONITOR_SELECTION")

Phase24 monitor fold 0: selected_alpha=0.00, advanced=False
Phase24 monitor fold 1: selected_alpha=0.00, advanced=False
Phase24 monitor fold 2: selected_alpha=0.00, advanced=False
BEGIN SANITIZED_PHASE24_MONITOR_SELECTION
{
  "phase": "phase24_monitor_only_selection",
  "status": "complete",
  "configuration": {
    "candidate_alphas": [
      0.0,
      0.25,
      0.5,
      0.75,
      1.0
    ],
    "minimum_monitor_log_loss_gain": 0.002,
    "base_seed": 240601
  },
  "folds": [
    {
      "fold": 0,
      "fit_n": 721,
      "monitor_n": 174,
      "baseline_monitor_log_loss": 0.278465,
      "raw_best_alpha": 0.0,
      "raw_best_monitor_log_loss": 0.278465,
      "raw_monitor_gain": 0.0,
      "selected_alpha": 0.0,
      "selected_monitor_log_loss": 0.278465,
      "correction_advanced": false,
      "candidate_count": 5
    },
    {
      "fold": 1,
      "fit_n": 796,
      "monitor_n": 123,
      "baseline_monitor_log_loss": 0.163161,
      "raw_best_alpha": 0.0,
      "ra

In [76]:
# Cell 59 — Phase24 single frozen outer evaluation

import json
import numpy as np
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    brier_score_loss,
)

y = np.asarray(PHASE19_Y_PRIVATE, dtype=np.int64)

p12 = np.clip(
    np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

p23 = np.clip(
    np.asarray(
        PHASE23_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

z12 = logit(p12)
z23 = logit(p23)

PHASE24_OOF_PRIVATE = p12.copy()

for fold in range(3):
    indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_valid"],
        dtype=np.int64,
    )
    alpha = float(PHASE24_SELECTED_ALPHA_PRIVATE[fold])

    PHASE24_OOF_PRIVATE[indices] = expit(
        z12[indices]
        + alpha * (z23[indices] - z12[indices])
    )

PHASE24_OOF_PRIVATE = np.clip(
    PHASE24_OOF_PRIVATE,
    1e-5,
    1.0 - 1e-5,
)

assert PHASE24_OOF_PRIVATE.shape == y.shape
assert np.isfinite(PHASE24_OOF_PRIVATE).all()


def phase24_metrics(target, probability):
    return {
        "log_loss": float(log_loss(
            target,
            probability,
            labels=[0, 1],
        )),
        "auroc": float(roc_auc_score(
            target,
            probability,
        )),
        "brier": float(brier_score_loss(
            target,
            probability,
        )),
    }


def phase24_calibration(target, probability):
    predictor = logit(np.clip(
        probability,
        1e-5,
        1.0 - 1e-5,
    )).reshape(-1, 1)

    calibrator = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
    calibrator.fit(predictor, target)

    return {
        "intercept": float(calibrator.intercept_[0]),
        "slope": float(calibrator.coef_[0, 0]),
    }


baseline_metrics = phase24_metrics(y, p12)
phase24_result = phase24_metrics(y, PHASE24_OOF_PRIVATE)
phase24_cal = phase24_calibration(y, PHASE24_OOF_PRIVATE)

fold_metrics = []
fold_wins = 0
worst_fold_excess = -np.inf

for fold in range(3):
    indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_valid"],
        dtype=np.int64,
    )

    baseline_fold = phase24_metrics(
        y[indices],
        p12[indices],
    )
    candidate_fold = phase24_metrics(
        y[indices],
        PHASE24_OOF_PRIVATE[indices],
    )

    improvement = (
        baseline_fold["log_loss"]
        - candidate_fold["log_loss"]
    )

    if improvement > 0.0:
        fold_wins += 1

    worst_fold_excess = max(
        worst_fold_excess,
        candidate_fold["log_loss"]
        - baseline_fold["log_loss"],
    )

    fold_metrics.append({
        "fold": fold,
        "n": int(len(indices)),
        "selected_alpha": float(
            PHASE24_SELECTED_ALPHA_PRIVATE[fold]
        ),
        "baseline_log_loss": round(
            baseline_fold["log_loss"],
            6,
        ),
        "phase24_log_loss": round(
            candidate_fold["log_loss"],
            6,
        ),
        "log_loss_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_fold["auroc"],
            6,
        ),
        "phase24_auroc": round(
            candidate_fold["auroc"],
            6,
        ),
    })


# Acquisition-group safety audit.
group_column = None

for candidate in [
    "acquisition_group",
    "acquisition_group_id",
]:
    if candidate in case_df.columns:
        group_column = candidate
        break

assert group_column is not None

groups = np.asarray(case_df[group_column])
major_group_metrics = []
maximum_major_group_harm = 0.0

for group in np.unique(groups):
    indices = np.flatnonzero(groups == group)

    if len(indices) < 30:
        continue

    baseline_group = phase24_metrics(
        y[indices],
        p12[indices],
    )
    candidate_group = phase24_metrics(
        y[indices],
        PHASE24_OOF_PRIVATE[indices],
    )

    harm = (
        candidate_group["log_loss"]
        - baseline_group["log_loss"]
    )
    maximum_major_group_harm = max(
        maximum_major_group_harm,
        harm,
    )

    major_group_metrics.append({
        "group": int(group),
        "n": int(len(indices)),
        "baseline_log_loss": round(
            baseline_group["log_loss"],
            6,
        ),
        "phase24_log_loss": round(
            candidate_group["log_loss"],
            6,
        ),
        "log_loss_change": round(harm, 6),
    })


log_loss_gain = (
    baseline_metrics["log_loss"]
    - phase24_result["log_loss"]
)
auroc_gain = (
    phase24_result["auroc"]
    - baseline_metrics["auroc"]
)
brier_excess = (
    phase24_result["brier"]
    - baseline_metrics["brier"]
)

thresholds = {
    "minimum_log_loss_gain": 0.003,
    "minimum_auroc_gain": 0.002,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.005,
    "maximum_brier_excess": 0.001,
    "calibration_slope_range": [0.8, 1.2],
    "maximum_major_group_harm": 0.015,
}

promotion_passed = bool(
    log_loss_gain >= thresholds["minimum_log_loss_gain"]
    and auroc_gain >= thresholds["minimum_auroc_gain"]
    and fold_wins >= thresholds["minimum_fold_wins"]
    and worst_fold_excess
        <= thresholds["maximum_worst_fold_excess"]
    and brier_excess
        <= thresholds["maximum_brier_excess"]
    and thresholds["calibration_slope_range"][0]
        <= phase24_cal["slope"]
        <= thresholds["calibration_slope_range"][1]
    and maximum_major_group_harm
        <= thresholds["maximum_major_group_harm"]
)

report = {
    "phase": "phase24_nested_shadow_gated_refit",
    "status": (
        "promoted"
        if promotion_passed
        else "not_promoted"
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "phase24": {
        **{
            key: round(value, 6)
            for key, value in phase24_result.items()
        },
        "calibration_intercept": round(
            phase24_cal["intercept"],
            6,
        ),
        "calibration_slope": round(
            phase24_cal["slope"],
            6,
        ),
    },
    "improvements": {
        "log_loss_gain": round(log_loss_gain, 6),
        "auroc_gain": round(auroc_gain, 6),
        "brier_excess": round(brier_excess, 6),
        "fold_wins": int(fold_wins),
        "worst_fold_excess": round(
            float(worst_fold_excess),
            6,
        ),
        "maximum_major_group_harm": round(
            float(maximum_major_group_harm),
            6,
        ),
    },
    "selected_alphas": {
        str(fold): float(
            PHASE24_SELECTED_ALPHA_PRIVATE[fold]
        )
        for fold in range(3)
    },
    "fold_metrics": fold_metrics,
    "major_acquisition_group_metrics":
        major_group_metrics,
    "promotion_thresholds": thresholds,
    "promotion_gate_passed": promotion_passed,
    "outer_oracle_alphas_used": False,
    "outer_labels_used_only_for_final_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE24_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE24_OOF")

BEGIN SANITIZED_PHASE24_OOF
{
  "phase": "phase24_nested_shadow_gated_refit",
  "status": "not_promoted",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "phase24": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535,
    "calibration_intercept": 0.167533,
    "calibration_slope": 0.935548
  },
  "improvements": {
    "log_loss_gain": 0.0,
    "auroc_gain": 0.0,
    "brier_excess": 0.0,
    "fold_wins": 0,
    "worst_fold_excess": 0.0,
    "maximum_major_group_harm": 0.0
  },
  "selected_alphas": {
    "0": 0.0,
    "1": 0.0,
    "2": 0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "selected_alpha": 0.0,
      "baseline_log_loss": 0.358494,
      "phase24_log_loss": 0.358494,
      "log_loss_improvement": 0.0,
      "baseline_auroc": 0.916174,
      "phase24_auroc": 0.916174
    },
    {
      "fold": 1,
      "n": 443,
      "selected_alpha": 0.0,
      "baseline_log_loss": 0.255415,
      "ph

In [78]:
# Cell 60 — Phase25 frozen-expert cache and manifest contract
# Reports structures and shapes only; no case-level values are displayed.

import inspect
import json
import numpy as np
import torch


def phase25_signature(value):
    try:
        return str(inspect.signature(value))
    except Exception:
        return "unavailable"


def phase25_schema(value, depth=0, maximum_depth=4):
    if isinstance(value, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }

    if torch.is_tensor(value):
        return {
            "type": "tensor",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "device_type": value.device.type,
        }

    if isinstance(value, torch.nn.Module):
        return {
            "type": "torch_module",
            "class": value.__class__.__name__,
            "parameter_count": int(sum(
                parameter.numel()
                for parameter in value.parameters()
            )),
        }

    if isinstance(value, dict):
        result = {
            "type": "dict",
            "length": len(value),
            "key_types": sorted({
                type(key).__name__
                for key in value.keys()
            }),
        }

        if depth >= maximum_depth:
            return result

        string_keys = [
            key for key in value.keys()
            if isinstance(key, str)
        ]

        if len(string_keys) == len(value) and len(string_keys) <= 40:
            result["fields"] = {
                key: phase25_schema(
                    value[key],
                    depth + 1,
                    maximum_depth,
                )
                for key in sorted(string_keys)
                if "hash" not in key.lower()
                and "sha" not in key.lower()
                and "uid" not in key.lower()
                and "path" not in key.lower()
            }
        elif len(value) > 0:
            first_value = next(iter(value.values()))
            result["representative_value_schema"] = phase25_schema(
                first_value,
                depth + 1,
                maximum_depth,
            )

        return result

    if isinstance(value, (list, tuple)):
        result = {
            "type": type(value).__name__,
            "length": len(value),
        }

        if depth < maximum_depth and len(value) > 0:
            representative_count = min(3, len(value))
            result["representative_element_schemas"] = [
                phase25_schema(
                    value[index],
                    depth + 1,
                    maximum_depth,
                )
                for index in range(representative_count)
            ]

        return result

    if isinstance(value, (int, np.integer)):
        return {"type": "integer"}

    if isinstance(value, (float, np.floating)):
        return {"type": "floating"}

    if isinstance(value, str):
        return {"type": "string"}

    if isinstance(value, bool):
        return {"type": "boolean"}

    if value is None:
        return {"type": "none"}

    return {"type": type(value).__name__}


expert_object_names = sorted(
    name for name in globals()
    if (
        "EXPERT" in name.upper()
        and any(
            token in name.upper()
            for token in [
                "CACHE",
                "FEATURE",
                "LOGIT",
                "OUTPUT",
                "PREDICTION",
            ]
        )
        and not name.startswith("__")
    )
)

# Limit to avoid unexpectedly verbose notebook output.
expert_object_names = expert_object_names[:30]

expert_object_schemas = {
    name: phase25_schema(globals()[name])
    for name in expert_object_names
}

expert_callable_names = sorted(
    name for name, value in globals().items()
    if callable(value)
    and any(
        token in name.lower()
        for token in [
            "expert",
            "embedding_hook",
            "capture_embedding",
        ]
    )
    and not name.startswith("__")
)

expert_callable_contracts = {
    name: phase25_signature(globals()[name])
    for name in expert_callable_names[:30]
}

fold_manifest_schemas = [
    {
        "fold": fold,
        "schema": phase25_schema(
            PHASE19_FOLD_MANIFESTS[fold],
            maximum_depth=5,
        ),
    }
    for fold in range(3)
]

related_private_names = sorted(
    name for name in globals()
    if any(
        token in name.upper()
        for token in [
            "PHASE19_EXPERT",
            "BASE_LOGIT",
            "ANATOMY_LOGIT",
            "EXPERT_LOGIT",
        ]
    )
    and not name.startswith("__")
)

report = {
    "phase": "phase25_frozen_expert_stacking_contract",
    "status": "audit_complete",
    "expert_object_names": expert_object_names,
    "expert_object_schemas": expert_object_schemas,
    "expert_callable_contracts": expert_callable_contracts,
    "fold_manifest_schemas": fold_manifest_schemas,
    "related_private_names": related_private_names,
    "expected_design": {
        "expert_count_per_fold": 7,
        "compact_experts": 5,
        "anatomy_experts": 2,
        "combination": "nonnegative_simplex",
        "regularization_target": "phase12c_default_family_weights",
        "selection_partition": "monitor_only",
        "outer_evaluation_count": 1,
        "high_dimensional_embeddings_used": False,
        "acquisition_group_conditioning": False,
    },
    "case_level_values_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE25_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE25_CONTRACT")

BEGIN SANITIZED_PHASE25_CONTRACT
{
  "phase": "phase25_frozen_expert_stacking_contract",
  "status": "audit_complete",
  "expert_object_names": [
    "PHASE19_EXPERT_CACHE_PRIVATE",
    "PHASE19_EXPERT_FEATURE_DIMENSIONS",
    "expert_feature_dimensions",
    "expert_logits",
    "phase19_expert_cache",
    "phase19_expert_features"
  ],
  "expert_object_schemas": {
    "PHASE19_EXPERT_CACHE_PRIVATE": {
      "type": "dict",
      "length": 3,
      "key_types": [
        "int"
      ],
      "representative_value_schema": {
        "type": "list",
        "length": 7,
        "representative_element_schemas": [
          {
            "type": "dict",
            "length": 7,
            "key_types": [
              "str"
            ],
            "fields": {
              "change_embedding": {
                "type": "ndarray",
                "shape": [
                  1362,
                  192
                ],
                "dtype": "float16"
              },
              

In [79]:
# Cell 61 — Phase25 exact calibrated-family component cache

import copy
import gc
import json
import time
import numpy as np
import torch

PHASE25_COMPONENT_NAMES = [
    "phase11c",
    "anatomy_seed_a",
    "anatomy_seed_b",
]

PHASE25_COMPONENT_CACHE_PRIVATE = {}
phase25_reconstruction_records = []
phase25_started = time.perf_counter()


def phase25_predict_components(indices, fold):
    fold_contract = PHASE19_FOLD_MANIFESTS[fold]

    family_manifest_original = copy.deepcopy(
        fold_contract["family"]
    )
    base_manifest = copy.deepcopy(
        fold_contract["base"]
    )

    family_weights_original = {
        name: float(
            family_manifest_original["family_weights"][name]
        )
        for name in PHASE25_COMPONENT_NAMES
    }

    assert abs(
        sum(family_weights_original.values()) - 1.0
    ) < 1e-8

    compact_count = phase24_activate_fold(
        PHASE19_BASE_MODELS,
        fold,
        PHASE24_DEVICE,
    )
    anatomy_count = phase24_activate_fold(
        PHASE19_ANATOMY_MODELS,
        fold,
        PHASE24_DEVICE,
    )

    assert compact_count == 5
    assert anatomy_count == 2

    crops = build_predictor_crops(indices)
    outputs = {}

    with torch.inference_mode():
        for selected_component in PHASE25_COMPONENT_NAMES:
            isolated_manifest = copy.deepcopy(
                family_manifest_original
            )

            isolated_manifest["family_weights"] = {
                name: float(name == selected_component)
                for name in PHASE25_COMPONENT_NAMES
            }

            raw = PHASE19_FAMILY_MODULE.predict_family_batch(
                crops,
                isolated_manifest,
                base_manifest,
                PHASE19_BASE_MODELS,
                PHASE19_ANATOMY_MODELS,
                PHASE24_DEVICE,
            )

            outputs[selected_component] = (
                phase24_normalize_output(
                    raw,
                    len(indices),
                )
            )

    phase24_move_collection_cpu(PHASE19_BASE_MODELS)
    phase24_move_collection_cpu(PHASE19_ANATOMY_MODELS)

    return outputs, family_weights_original


for fold in range(3):
    monitor_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["monitor"],
        dtype=np.int64,
    )
    outer_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_valid"],
        dtype=np.int64,
    )

    monitor_components, default_weights = (
        phase25_predict_components(
            monitor_indices,
            fold,
        )
    )

    outer_components, outer_default_weights = (
        phase25_predict_components(
            outer_indices,
            fold,
        )
    )

    assert default_weights == outer_default_weights

    PHASE25_COMPONENT_CACHE_PRIVATE[fold] = {
        "monitor": monitor_components,
        "outer_valid": outer_components,
        "default_weights": default_weights,
    }

    reconstructed_outer = sum(
        default_weights[name] * outer_components[name]
        for name in PHASE25_COMPONENT_NAMES
    )

    historical_outer = np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE[outer_indices],
        dtype=np.float64,
    )

    absolute_error = np.abs(
        reconstructed_outer - historical_outer
    )

    phase25_reconstruction_records.append({
        "fold": fold,
        "monitor_n": int(len(monitor_indices)),
        "outer_valid_n": int(len(outer_indices)),
        "default_weights": {
            name: round(default_weights[name], 6)
            for name in PHASE25_COMPONENT_NAMES
        },
        "maximum_reconstruction_error": round(
            float(absolute_error.max()),
            10,
        ),
        "mean_reconstruction_error": round(
            float(absolute_error.mean()),
            10,
        ),
    })

    print(f"Phase25 component isolation fold {fold} complete.")

gc.collect()
torch.cuda.empty_cache()

maximum_reconstruction_error = max(
    record["maximum_reconstruction_error"]
    for record in phase25_reconstruction_records
)

report = {
    "phase": "phase25_exact_family_component_cache",
    "status": (
        "accepted"
        if maximum_reconstruction_error <= 1e-5
        else "rejected"
    ),
    "component_names": PHASE25_COMPONENT_NAMES,
    "component_count": 3,
    "folds": phase25_reconstruction_records,
    "maximum_reconstruction_error":
        maximum_reconstruction_error,
    "maximum_allowed_reconstruction_error": 1e-5,
    "elapsed_seconds": round(
        time.perf_counter() - phase25_started,
        2,
    ),
    "component_calibration": "original_fold_local_platt",
    "individual_expert_embeddings_used": False,
    "outer_labels_used": False,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE25_COMPONENT_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE25_COMPONENT_CACHE")

assert report["status"] == "accepted", (
    "Do not run Cells 62–63 because the component decomposition "
    "does not exactly reconstruct Phase12c."
)

Phase25 component isolation fold 0 complete.
Phase25 component isolation fold 1 complete.
Phase25 component isolation fold 2 complete.
BEGIN SANITIZED_PHASE25_COMPONENT_CACHE
{
  "phase": "phase25_exact_family_component_cache",
  "status": "rejected",
  "component_names": [
    "phase11c",
    "anatomy_seed_a",
    "anatomy_seed_b"
  ],
  "component_count": 3,
  "folds": [
    {
      "fold": 0,
      "monitor_n": 174,
      "outer_valid_n": 467,
      "default_weights": {
        "phase11c": 0.75,
        "anatomy_seed_a": 0.125,
        "anatomy_seed_b": 0.125
      },
      "maximum_reconstruction_error": 6.3735e-05,
      "mean_reconstruction_error": 4.4762e-06
    },
    {
      "fold": 1,
      "monitor_n": 123,
      "outer_valid_n": 443,
      "default_weights": {
        "phase11c": 0.75,
        "anatomy_seed_a": 0.125,
        "anatomy_seed_b": 0.125
      },
      "maximum_reconstruction_error": 3.12606e-05,
      "mean_reconstruction_error": 6.986e-07
    },
    {
      "f

AssertionError: Do not run Cells 62–63 because the component decomposition does not exactly reconstruct Phase12c.

In [80]:
# Cell 61B — Phase25 numerical-equivalence audit

import json
import numpy as np

PHASE25_NUMERICAL_THRESHOLDS = {
    "maximum_absolute_error": 2e-4,
    "maximum_mean_absolute_error": 2e-5,
}

audit_records = []

for fold in range(3):
    cache = PHASE25_COMPONENT_CACHE_PRIVATE[fold]

    default_vector = np.asarray([
        cache["default_weights"][name]
        for name in PHASE25_COMPONENT_NAMES
    ], dtype=np.float64)

    for partition_name, indices_key, historical in [
        (
            "monitor",
            "monitor",
            np.asarray(
                PHASE24_BASE_MONITOR_PRIVATE[fold],
                dtype=np.float64,
            ),
        ),
        (
            "outer_valid",
            "outer_valid",
            np.asarray(
                PHASE19_PHASE12C_OOF_PRIVATE[
                    PHASE19_PARTITIONS[fold]["outer_valid"]
                ],
                dtype=np.float64,
            ),
        ),
    ]:
        matrix = np.column_stack([
            cache[indices_key][name]
            for name in PHASE25_COMPONENT_NAMES
        ]).astype(np.float64)

        reconstructed = matrix @ default_vector
        error = np.abs(reconstructed - historical)

        audit_records.append({
            "fold": fold,
            "partition": partition_name,
            "n": int(len(historical)),
            "maximum_absolute_error": round(
                float(error.max()), 10
            ),
            "mean_absolute_error": round(
                float(error.mean()), 10
            ),
            "q99_absolute_error": round(
                float(np.quantile(error, 0.99)), 10
            ),
        })

maximum_error = max(
    row["maximum_absolute_error"]
    for row in audit_records
)
maximum_mean_error = max(
    row["mean_absolute_error"]
    for row in audit_records
)

PHASE25_COMPONENT_NUMERICAL_CONTRACT_ACCEPTED = bool(
    maximum_error
        <= PHASE25_NUMERICAL_THRESHOLDS[
            "maximum_absolute_error"
        ]
    and maximum_mean_error
        <= PHASE25_NUMERICAL_THRESHOLDS[
            "maximum_mean_absolute_error"
        ]
)

report = {
    "phase": "phase25_component_numerical_equivalence",
    "status": (
        "accepted_with_exact_baseline_anchoring"
        if PHASE25_COMPONENT_NUMERICAL_CONTRACT_ACCEPTED
        else "rejected"
    ),
    "records": audit_records,
    "maximum_observed_absolute_error": maximum_error,
    "maximum_observed_mean_absolute_error":
        maximum_mean_error,
    "thresholds": PHASE25_NUMERICAL_THRESHOLDS,
    "candidate_formula": (
        "exact_phase12c_probability + "
        "component_matrix @ (candidate_weights-default_weights)"
    ),
    "default_reconstruction_by_construction": "exact",
    "labels_used": False,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE25_NUMERICAL_EQUIVALENCE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE25_NUMERICAL_EQUIVALENCE")

assert PHASE25_COMPONENT_NUMERICAL_CONTRACT_ACCEPTED

BEGIN SANITIZED_PHASE25_NUMERICAL_EQUIVALENCE
{
  "phase": "phase25_component_numerical_equivalence",
  "status": "accepted_with_exact_baseline_anchoring",
  "records": [
    {
      "fold": 0,
      "partition": "monitor",
      "n": 174,
      "maximum_absolute_error": 0.0,
      "mean_absolute_error": 0.0,
      "q99_absolute_error": 0.0
    },
    {
      "fold": 0,
      "partition": "outer_valid",
      "n": 467,
      "maximum_absolute_error": 6.3735e-05,
      "mean_absolute_error": 4.4762e-06,
      "q99_absolute_error": 3.13057e-05
    },
    {
      "fold": 1,
      "partition": "monitor",
      "n": 123,
      "maximum_absolute_error": 0.0,
      "mean_absolute_error": 0.0,
      "q99_absolute_error": 0.0
    },
    {
      "fold": 1,
      "partition": "outer_valid",
      "n": 443,
      "maximum_absolute_error": 3.12606e-05,
      "mean_absolute_error": 6.986e-07,
      "q99_absolute_error": 1.68994e-05
    },
    {
      "fold": 2,
      "partition": "monitor",
      "n

In [81]:
# Cell 62R — Exact-baseline-anchored monitor selection

import json
import numpy as np
from sklearn.metrics import log_loss, roc_auc_score

assert PHASE25_COMPONENT_NUMERICAL_CONTRACT_ACCEPTED

PHASE25_MINIMUM_MONITOR_GAIN = 0.002

PHASE25_WEIGHT_CANDIDATES = [
    (0.750, 0.125, 0.125),
    (0.850, 0.075, 0.075),
    (0.800, 0.100, 0.100),
    (0.700, 0.150, 0.150),
    (0.650, 0.175, 0.175),
    (0.750, 0.200, 0.050),
    (0.750, 0.050, 0.200),
    (0.800, 0.150, 0.050),
    (0.800, 0.050, 0.150),
    (0.700, 0.200, 0.100),
    (0.700, 0.100, 0.200),
]

PHASE25_SELECTED_WEIGHTS_PRIVATE = {}
phase25_selection_records = []

for fold in range(3):
    monitor_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["monitor"],
        dtype=np.int64,
    )
    monitor_y = np.asarray(
        PHASE19_Y_PRIVATE[monitor_indices],
        dtype=np.int64,
    )

    cache = PHASE25_COMPONENT_CACHE_PRIVATE[fold]

    component_matrix = np.column_stack([
        cache["monitor"][name]
        for name in PHASE25_COMPONENT_NAMES
    ]).astype(np.float64)

    default_vector = np.asarray([
        cache["default_weights"][name]
        for name in PHASE25_COMPONENT_NAMES
    ], dtype=np.float64)

    exact_baseline = np.clip(
        np.asarray(
            PHASE24_BASE_MONITOR_PRIVATE[fold],
            dtype=np.float64,
        ),
        1e-5,
        1.0 - 1e-5,
    )

    baseline_loss = float(log_loss(
        monitor_y,
        exact_baseline,
        labels=[0, 1],
    ))

    candidates = []

    for weights in PHASE25_WEIGHT_CANDIDATES:
        vector = np.asarray(weights, dtype=np.float64)

        probability = np.clip(
            exact_baseline
            + component_matrix @ (vector - default_vector),
            1e-5,
            1.0 - 1e-5,
        )

        candidates.append({
            "weights": tuple(float(x) for x in vector),
            "log_loss": float(log_loss(
                monitor_y,
                probability,
                labels=[0, 1],
            )),
            "auroc": float(roc_auc_score(
                monitor_y,
                probability,
            )),
            "distance_from_default": float(
                np.abs(vector - default_vector).sum()
            ),
        })

    raw_best = min(
        candidates,
        key=lambda row: (
            row["log_loss"],
            row["distance_from_default"],
            -row["weights"][0],
        ),
    )

    raw_gain = baseline_loss - raw_best["log_loss"]

    if (
        raw_gain >= PHASE25_MINIMUM_MONITOR_GAIN
        and raw_best["distance_from_default"] > 1e-10
    ):
        selected_vector = np.asarray(
            raw_best["weights"],
            dtype=np.float64,
        )
        advanced = True
    else:
        selected_vector = default_vector.copy()
        advanced = False

    PHASE25_SELECTED_WEIGHTS_PRIVATE[fold] = selected_vector

    selected_probability = np.clip(
        exact_baseline
        + component_matrix @ (
            selected_vector - default_vector
        ),
        1e-5,
        1.0 - 1e-5,
    )

    phase25_selection_records.append({
        "fold": fold,
        "monitor_n": int(len(monitor_indices)),
        "baseline_monitor_log_loss": round(
            baseline_loss, 6
        ),
        "raw_best_weights": {
            name: float(value)
            for name, value in zip(
                PHASE25_COMPONENT_NAMES,
                raw_best["weights"],
            )
        },
        "raw_best_monitor_log_loss": round(
            raw_best["log_loss"], 6
        ),
        "raw_monitor_gain": round(raw_gain, 6),
        "selected_weights": {
            name: float(value)
            for name, value in zip(
                PHASE25_COMPONENT_NAMES,
                selected_vector,
            )
        },
        "selected_monitor_log_loss": round(
            float(log_loss(
                monitor_y,
                selected_probability,
                labels=[0, 1],
            )),
            6,
        ),
        "correction_advanced": advanced,
    })

    print(
        f"Phase25 fold {fold}: advanced={advanced}, "
        f"weights={selected_vector.tolist()}"
    )

report = {
    "phase": "phase25_anchored_monitor_selection",
    "status": "complete",
    "candidate_count": len(PHASE25_WEIGHT_CANDIDATES),
    "minimum_monitor_gain": PHASE25_MINIMUM_MONITOR_GAIN,
    "folds": phase25_selection_records,
    "exact_phase12c_anchor_used": True,
    "selection_partition": "monitor_only",
    "selection_frozen_before_outer_evaluation": True,
    "outer_labels_used": False,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE25_MONITOR_SELECTION")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE25_MONITOR_SELECTION")

Phase25 fold 0: advanced=False, weights=[0.75, 0.125, 0.125]
Phase25 fold 1: advanced=False, weights=[0.75, 0.125, 0.125]
Phase25 fold 2: advanced=False, weights=[0.75, 0.125, 0.125]
BEGIN SANITIZED_PHASE25_MONITOR_SELECTION
{
  "phase": "phase25_anchored_monitor_selection",
  "status": "complete",
  "candidate_count": 11,
  "minimum_monitor_gain": 0.002,
  "folds": [
    {
      "fold": 0,
      "monitor_n": 174,
      "baseline_monitor_log_loss": 0.278465,
      "raw_best_weights": {
        "phase11c": 0.8,
        "anatomy_seed_a": 0.05,
        "anatomy_seed_b": 0.15
      },
      "raw_best_monitor_log_loss": 0.277424,
      "raw_monitor_gain": 0.00104,
      "selected_weights": {
        "phase11c": 0.75,
        "anatomy_seed_a": 0.125,
        "anatomy_seed_b": 0.125
      },
      "selected_monitor_log_loss": 0.278465,
      "correction_advanced": false
    },
    {
      "fold": 1,
      "monitor_n": 123,
      "baseline_monitor_log_loss": 0.163161,
      "raw_best_weights":

In [82]:
# Cell 63R — Phase25 exact-anchor frozen OOF evaluation

import json
import numpy as np
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    brier_score_loss,
)

y = np.asarray(PHASE19_Y_PRIVATE, dtype=np.int64)

baseline = np.clip(
    np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

PHASE25_OOF_PRIVATE = baseline.copy()

for fold in range(3):
    indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_valid"],
        dtype=np.int64,
    )
    cache = PHASE25_COMPONENT_CACHE_PRIVATE[fold]

    component_matrix = np.column_stack([
        cache["outer_valid"][name]
        for name in PHASE25_COMPONENT_NAMES
    ]).astype(np.float64)

    default_vector = np.asarray([
        cache["default_weights"][name]
        for name in PHASE25_COMPONENT_NAMES
    ], dtype=np.float64)

    selected_vector = np.asarray(
        PHASE25_SELECTED_WEIGHTS_PRIVATE[fold],
        dtype=np.float64,
    )

    PHASE25_OOF_PRIVATE[indices] = np.clip(
        baseline[indices]
        + component_matrix @ (
            selected_vector - default_vector
        ),
        1e-5,
        1.0 - 1e-5,
    )

assert PHASE25_OOF_PRIVATE.shape == y.shape
assert np.isfinite(PHASE25_OOF_PRIVATE).all()


def phase25_metrics(target, probability):
    return {
        "log_loss": float(log_loss(
            target, probability, labels=[0, 1]
        )),
        "auroc": float(roc_auc_score(
            target, probability
        )),
        "brier": float(brier_score_loss(
            target, probability
        )),
    }


def phase25_calibration(target, probability):
    predictor = logit(np.clip(
        probability, 1e-5, 1.0 - 1e-5
    )).reshape(-1, 1)

    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
    model.fit(predictor, target)

    return {
        "intercept": float(model.intercept_[0]),
        "slope": float(model.coef_[0, 0]),
    }


baseline_metrics = phase25_metrics(y, baseline)
candidate_metrics = phase25_metrics(y, PHASE25_OOF_PRIVATE)
calibration = phase25_calibration(y, PHASE25_OOF_PRIVATE)

fold_records = []
fold_wins = 0
worst_fold_excess = -np.inf

for fold in range(3):
    indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_valid"],
        dtype=np.int64,
    )

    base_fold = phase25_metrics(
        y[indices], baseline[indices]
    )
    candidate_fold = phase25_metrics(
        y[indices], PHASE25_OOF_PRIVATE[indices]
    )

    improvement = (
        base_fold["log_loss"]
        - candidate_fold["log_loss"]
    )

    fold_wins += int(improvement > 0.0)
    worst_fold_excess = max(
        worst_fold_excess,
        -improvement,
    )

    fold_records.append({
        "fold": fold,
        "n": int(len(indices)),
        "selected_weights": {
            name: float(value)
            for name, value in zip(
                PHASE25_COMPONENT_NAMES,
                PHASE25_SELECTED_WEIGHTS_PRIVATE[fold],
            )
        },
        "baseline_log_loss": round(
            base_fold["log_loss"], 6
        ),
        "phase25_log_loss": round(
            candidate_fold["log_loss"], 6
        ),
        "log_loss_improvement": round(
            improvement, 6
        ),
        "baseline_auroc": round(
            base_fold["auroc"], 6
        ),
        "phase25_auroc": round(
            candidate_fold["auroc"], 6
        ),
    })


group_column = next(
    column for column in [
        "acquisition_group",
        "acquisition_group_id",
    ]
    if column in case_df.columns
)

groups = np.asarray(case_df[group_column])
major_group_records = []
maximum_major_group_harm = 0.0

for group in np.unique(groups):
    indices = np.flatnonzero(groups == group)

    if len(indices) < 30:
        continue

    base_group = phase25_metrics(
        y[indices], baseline[indices]
    )
    candidate_group = phase25_metrics(
        y[indices], PHASE25_OOF_PRIVATE[indices]
    )

    harm = (
        candidate_group["log_loss"]
        - base_group["log_loss"]
    )
    maximum_major_group_harm = max(
        maximum_major_group_harm,
        harm,
    )

    major_group_records.append({
        "group": int(group),
        "n": int(len(indices)),
        "baseline_log_loss": round(
            base_group["log_loss"], 6
        ),
        "phase25_log_loss": round(
            candidate_group["log_loss"], 6
        ),
        "log_loss_change": round(harm, 6),
    })


log_loss_gain = (
    baseline_metrics["log_loss"]
    - candidate_metrics["log_loss"]
)
auroc_gain = (
    candidate_metrics["auroc"]
    - baseline_metrics["auroc"]
)
brier_excess = (
    candidate_metrics["brier"]
    - baseline_metrics["brier"]
)

thresholds = {
    "minimum_log_loss_gain": 0.003,
    "minimum_auroc_gain": 0.002,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.005,
    "maximum_brier_excess": 0.001,
    "calibration_slope_range": [0.8, 1.2],
    "maximum_major_group_harm": 0.015,
}

promotion_passed = bool(
    log_loss_gain >= thresholds["minimum_log_loss_gain"]
    and auroc_gain >= thresholds["minimum_auroc_gain"]
    and fold_wins >= thresholds["minimum_fold_wins"]
    and worst_fold_excess
        <= thresholds["maximum_worst_fold_excess"]
    and brier_excess
        <= thresholds["maximum_brier_excess"]
    and thresholds["calibration_slope_range"][0]
        <= calibration["slope"]
        <= thresholds["calibration_slope_range"][1]
    and maximum_major_group_harm
        <= thresholds["maximum_major_group_harm"]
)

report = {
    "phase": "phase25_exact_anchor_family_simplex",
    "status": (
        "promoted" if promotion_passed else "not_promoted"
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "phase25": {
        **{
            key: round(value, 6)
            for key, value in candidate_metrics.items()
        },
        "calibration_intercept": round(
            calibration["intercept"], 6
        ),
        "calibration_slope": round(
            calibration["slope"], 6
        ),
    },
    "improvements": {
        "log_loss_gain": round(log_loss_gain, 6),
        "auroc_gain": round(auroc_gain, 6),
        "brier_excess": round(brier_excess, 6),
        "fold_wins": int(fold_wins),
        "worst_fold_excess": round(
            float(worst_fold_excess), 6
        ),
        "maximum_major_group_harm": round(
            float(maximum_major_group_harm), 6
        ),
    },
    "fold_metrics": fold_records,
    "major_acquisition_group_metrics": major_group_records,
    "promotion_thresholds": thresholds,
    "promotion_gate_passed": promotion_passed,
    "exact_phase12c_anchor_used": True,
    "outer_labels_used_only_for_final_evaluation": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE25_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE25_OOF")

BEGIN SANITIZED_PHASE25_OOF
{
  "phase": "phase25_exact_anchor_family_simplex",
  "status": "not_promoted",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "phase25": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535,
    "calibration_intercept": 0.167533,
    "calibration_slope": 0.935548
  },
  "improvements": {
    "log_loss_gain": 0.0,
    "auroc_gain": 0.0,
    "brier_excess": 0.0,
    "fold_wins": 0,
    "worst_fold_excess": -0.0,
    "maximum_major_group_harm": 0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "selected_weights": {
        "phase11c": 0.75,
        "anatomy_seed_a": 0.125,
        "anatomy_seed_b": 0.125
      },
      "baseline_log_loss": 0.358494,
      "phase25_log_loss": 0.358494,
      "log_loss_improvement": 0.0,
      "baseline_auroc": 0.916174,
      "phase25_auroc": 0.916174
    },
    {
      "fold": 1,
      "n": 443,
      "selected_weights": {
        "phase

> Standard DINOv2 discovery

In [86]:
# Cell 64V3A — Secure authorized DINOv3 ViT-S/16 download
# Paste the ViT-S/16 URL into the hidden prompt.
# Do not paste it into the cell source.

import gc
import getpass
import json
import pathlib
import requests
import torch

PHASE26_DINOV3_WEIGHT_PATH_PRIVATE = pathlib.Path(
    "/kaggle/working/"
    "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
)

temporary_path = PHASE26_DINOV3_WEIGHT_PATH_PRIVATE.with_suffix(
    ".download"
)

if not PHASE26_DINOV3_WEIGHT_PATH_PRIVATE.exists():
    private_url = getpass.getpass(
        "Paste the private DINOv3 ViT-S/16 signed URL: "
    ).strip()

    # Handles accidental Markdown-style escaping.
    private_url = private_url.replace("\\", "")

    assert private_url.startswith("https://")
    assert "dinov3.llamameta.net/dinov3_vits16/" in private_url

    with requests.get(
        private_url,
        stream=True,
        timeout=(30, 300),
        allow_redirects=True,
    ) as response:
        response.raise_for_status()

        with temporary_path.open("wb") as output:
            for chunk in response.iter_content(
                chunk_size=8 * 1024**2
            ):
                if chunk:
                    output.write(chunk)

    temporary_path.replace(
        PHASE26_DINOV3_WEIGHT_PATH_PRIVATE
    )

    del private_url
    gc.collect()

assert PHASE26_DINOV3_WEIGHT_PATH_PRIVATE.exists()

checkpoint_size_mb = (
    PHASE26_DINOV3_WEIGHT_PATH_PRIVATE.stat().st_size
    / 1024**2
)

assert 70.0 <= checkpoint_size_mb <= 130.0, (
    f"Unexpected checkpoint size: {checkpoint_size_mb:.2f} MB"
)

checkpoint = torch.load(
    PHASE26_DINOV3_WEIGHT_PATH_PRIVATE,
    map_location="cpu",
    weights_only=True,
)

if (
    isinstance(checkpoint, dict)
    and "state_dict" in checkpoint
    and isinstance(checkpoint["state_dict"], dict)
):
    state_dictionary = checkpoint["state_dict"]
elif isinstance(checkpoint, dict):
    state_dictionary = checkpoint
else:
    raise TypeError(
        f"Unsupported checkpoint type: {type(checkpoint).__name__}"
    )

tensor_values = [
    value
    for value in state_dictionary.values()
    if torch.is_tensor(value)
]

assert len(tensor_values) >= 100
assert all(
    torch.isfinite(value).all().item()
    for value in tensor_values
    if value.is_floating_point()
)

report = {
    "phase": "phase26_dinov3_checkpoint",
    "status": "downloaded_and_verified",
    "checkpoint_name":
        PHASE26_DINOV3_WEIGHT_PATH_PRIVATE.name,
    "checkpoint_size_mb": round(checkpoint_size_mb, 2),
    "state_tensor_count": len(tensor_values),
    "all_state_tensors_finite": True,
    "architecture": "dinov3_vits16",
    "pretraining": "LVD1689M",
    "license": "DINOv3 License",
    "license_accepted_by_user": True,
    "signed_url_displayed": False,
    "model_hash_displayed": False,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE26_DINOV3_CHECKPOINT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE26_DINOV3_CHECKPOINT")

del checkpoint, state_dictionary, tensor_values
gc.collect()

Paste the private DINOv3 ViT-S/16 signed URL:  ········


BEGIN SANITIZED_PHASE26_DINOV3_CHECKPOINT
{
  "phase": "phase26_dinov3_checkpoint",
  "status": "downloaded_and_verified",
  "checkpoint_name": "dinov3_vits16_pretrain_lvd1689m-08c60483.pth",
  "checkpoint_size_mb": 82.52,
  "state_tensor_count": 188,
  "all_state_tensors_finite": true,
  "architecture": "dinov3_vits16",
  "pretraining": "LVD1689M",
  "license": "DINOv3 License",
  "license_accepted_by_user": true,
  "signed_url_displayed": false,
  "model_hash_displayed": false,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE26_DINOV3_CHECKPOINT


41

In [87]:
# Cell 66 — Phase26 DINOv3 ViT-S/16 architecture contract
# Uses the same nine synthetic views as DINOv2.

import gc
import json
import time
import torch
import torch.nn.functional as F

phase26_v3_started = time.perf_counter()

torch.hub.set_dir("/kaggle/working/torch_hub_phase26_v3")

PHASE26_DINOV3_PRIVATE = torch.hub.load(
    "facebookresearch/dinov3:main",
    "dinov3_vits16",
    source="github",
    weights=str(PHASE26_DINOV3_WEIGHT_PATH_PRIVATE),
    trust_repo=True,
    verbose=False,
)

PHASE26_DINOV3_PRIVATE.eval()
PHASE26_DINOV3_PRIVATE.requires_grad_(False)

parameter_count = sum(
    parameter.numel()
    for parameter in PHASE26_DINOV3_PRIVATE.parameters()
)

assert 20_000_000 <= parameter_count <= 24_000_000

device = torch.device("cuda:0")

synthetic = torch.rand(
    2,
    80,
    80,
    80,
    dtype=torch.float32,
)

views = phase26_make_views(synthetic)

assert views.shape == (2, 9, 3, 224, 224)
assert torch.isfinite(views).all()

flat_views = views.reshape(
    -1, 3, 224, 224
).to(device)

PHASE26_DINOV3_PRIVATE.to(device).eval()

torch.cuda.reset_peak_memory_stats(device)

with torch.inference_mode():
    raw_features = PHASE26_DINOV3_PRIVATE(flat_views)

if isinstance(raw_features, dict):
    for key in [
        "x_norm_clstoken",
        "pooler_output",
        "last_hidden_state",
    ]:
        if key in raw_features:
            raw_features = raw_features[key]
            break

if hasattr(raw_features, "pooler_output"):
    raw_features = raw_features.pooler_output

if (
    torch.is_tensor(raw_features)
    and raw_features.ndim == 3
):
    # Use the class token if a token sequence is returned.
    raw_features = raw_features[:, 0]

assert torch.is_tensor(raw_features)
assert raw_features.shape == (18, 384)
assert torch.isfinite(raw_features).all()

case_features = raw_features.reshape(2, 9, 384)

feature_summary = torch.cat(
    [
        case_features.mean(dim=1),
        case_features.std(dim=1, unbiased=False),
        case_features.amax(dim=1),
    ],
    dim=1,
)

assert feature_summary.shape == (2, 1152)

peak_vram_mb = (
    torch.cuda.max_memory_allocated(device) / 1024**2
)

view_difference = float(
    torch.pdist(
        F.normalize(case_features[0], dim=1)
    ).mean().item()
)

PHASE26_DINOV3_PRIVATE.cpu()

report = {
    "phase": "phase26_dinov3_multiplanar_contract",
    "status": "accepted",
    "backbone": "dinov3_vits16_LVD1689M",
    "parameter_count": int(parameter_count),
    "view_tensor_shape": [2, 9, 3, 224, 224],
    "per_view_embedding_dimension": 384,
    "summary_feature_dimension": 1152,
    "mean_pairwise_view_difference": round(
        view_difference, 6
    ),
    "peak_vram_mb": round(peak_vram_mb, 2),
    "backbone_frozen": True,
    "external_weights": True,
    "license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "elapsed_seconds": round(
        time.perf_counter() - phase26_v3_started,
        2,
    ),
    "synthetic_input_only": True,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "signed_url_displayed": False,
}

print("BEGIN SANITIZED_PHASE26_DINOV3_ARCHITECTURE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE26_DINOV3_ARCHITECTURE")

del (
    synthetic,
    views,
    flat_views,
    raw_features,
    case_features,
    feature_summary,
)

gc.collect()
torch.cuda.empty_cache()

Downloading: "https://github.com/facebookresearch/dinov3/zipball/main" to /kaggle/working/torch_hub_phase26_v3/main.zip
Downloading: "file:///kaggle/working/dinov3_vits16_pretrain_lvd1689m-08c60483.pth" to /kaggle/working/torch_hub_phase26_v3/checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth


100%|██████████| 82.5M/82.5M [00:00<00:00, 1.37GB/s]


BEGIN SANITIZED_PHASE26_DINOV3_ARCHITECTURE
{
  "phase": "phase26_dinov3_multiplanar_contract",
  "status": "accepted",
  "backbone": "dinov3_vits16_LVD1689M",
  "parameter_count": 21601152,
  "view_tensor_shape": [
    2,
    9,
    3,
    224,
    224
  ],
  "per_view_embedding_dimension": 384,
  "summary_feature_dimension": 1152,
  "mean_pairwise_view_difference": 0.199378,
  "peak_vram_mb": 283.25,
  "backbone_frozen": true,
  "external_weights": true,
  "license": "DINOv3 License",
  "required_attribution": "Built with DINOv3",
  "elapsed_seconds": 15.88,
  "synthetic_input_only": true,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "signed_url_displayed": false
}
END SANITIZED_PHASE26_DINOV3_ARCHITECTURE


In [88]:
# Cell 67 — Phase26 DINOv3 dense multi-planar feature cache

import gc
import json
import time
import numpy as np
import torch
import torch.nn as nn

assert torch.cuda.device_count() >= 2
assert PHASE26_DINOV3_PRIVATE is not None


def phase26_resolve_highres_cache():
    candidate_names = [
        "PHASE19_HIGHRES_CACHE",
        "PHASE18_HIGHRES_CACHE",
        "phase19_highres_cache",
        "phase18_highres_cache",
        "highres_cache",
    ]

    valid = []

    for name in candidate_names:
        if name not in globals():
            continue

        value = globals()[name]

        try:
            shape = tuple(value.shape)
        except Exception:
            continue

        if shape == (1362, 80, 80, 80):
            valid.append((name, value))

    # Remove aliases referencing the same object.
    unique = []
    seen_ids = set()

    for name, value in valid:
        if id(value) not in seen_ids:
            unique.append((name, value))
            seen_ids.add(id(value))

    assert len(unique) == 1, {
        "message": "Could not uniquely resolve the 80^3 cache.",
        "valid_candidate_count": len(unique),
        "valid_candidate_names": [
            name for name, _ in unique
        ],
    }

    return unique[0]


PHASE26_HIGHRES_CACHE_NAME, PHASE26_HIGHRES_CACHE_PRIVATE = (
    phase26_resolve_highres_cache()
)


class Phase26DINOv3DenseDescriptor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, images):
        output = self.backbone.forward_features(images)

        assert isinstance(output, dict)
        assert "x_norm_clstoken" in output
        assert "x_norm_patchtokens" in output

        cls_token = output["x_norm_clstoken"]
        patch_tokens = output["x_norm_patchtokens"]

        batch, patch_count, dimension = patch_tokens.shape

        assert patch_count == 196
        assert dimension == 384

        grid = patch_tokens.reshape(
            batch,
            14,
            14,
            dimension,
        )

        global_patch = grid.mean(dim=(1, 2))

        # Central 6×6 patch region.
        central_patch = grid[:, 4:10, 4:10].mean(
            dim=(1, 2)
        )

        left_patch = grid[:, :, :7].mean(dim=(1, 2))
        right_patch = grid[:, :, 7:].mean(dim=(1, 2))
        absolute_left_right = torch.abs(
            left_patch - right_patch
        )

        superior_patch = grid[:, :7, :].mean(
            dim=(1, 2)
        )
        inferior_patch = grid[:, 7:, :].mean(
            dim=(1, 2)
        )
        absolute_superior_inferior = torch.abs(
            superior_patch - inferior_patch
        )

        # [image, descriptor, embedding]
        return torch.stack(
            [
                cls_token,
                global_patch,
                central_patch,
                absolute_left_right,
                absolute_superior_inferior,
            ],
            dim=1,
        )


PHASE26_DESCRIPTOR_NAMES = [
    "class_token",
    "global_patch_mean",
    "central_patch_mean",
    "absolute_left_right_patch_difference",
    "absolute_superior_inferior_patch_difference",
]

PHASE26_EXTRACTION_BATCH_SIZE = 8
PHASE26_CASE_COUNT = 1362

PHASE26_DINOV3_FEATURE_CACHE_PRIVATE = np.empty(
    (
        PHASE26_CASE_COUNT,
        9,
        len(PHASE26_DESCRIPTOR_NAMES),
        384,
    ),
    dtype=np.float16,
)

PHASE26_DINOV3_PRIVATE.eval()
PHASE26_DINOV3_PRIVATE.requires_grad_(False)

descriptor_model = Phase26DINOv3DenseDescriptor(
    PHASE26_DINOV3_PRIVATE
)

descriptor_model.to(torch.device("cuda:0")).eval()

parallel_descriptor_model = nn.DataParallel(
    descriptor_model,
    device_ids=[0, 1],
    output_device=0,
)

gc.collect()
torch.cuda.empty_cache()

for gpu in range(2):
    torch.cuda.reset_peak_memory_stats(gpu)

extraction_started = time.perf_counter()

with torch.inference_mode():
    for start in range(
        0,
        PHASE26_CASE_COUNT,
        PHASE26_EXTRACTION_BATCH_SIZE,
    ):
        stop = min(
            start + PHASE26_EXTRACTION_BATCH_SIZE,
            PHASE26_CASE_COUNT,
        )

        volume_numpy = np.asarray(
            PHASE26_HIGHRES_CACHE_PRIVATE[start:stop],
            dtype=np.float32,
        )

        volume = torch.from_numpy(volume_numpy).to(
            device="cuda:0",
            dtype=torch.float32,
            non_blocking=False,
        )

        views = phase26_make_views(volume)

        assert views.shape == (
            stop - start,
            9,
            3,
            224,
            224,
        )

        flat_views = views.reshape(
            -1,
            3,
            224,
            224,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            descriptors = parallel_descriptor_model(
                flat_views
            )

        descriptors = descriptors.reshape(
            stop - start,
            9,
            len(PHASE26_DESCRIPTOR_NAMES),
            384,
        )

        assert torch.isfinite(descriptors).all()

        PHASE26_DINOV3_FEATURE_CACHE_PRIVATE[
            start:stop
        ] = (
            descriptors
            .float()
            .cpu()
            .numpy()
            .astype(np.float16)
        )

        del (
            volume_numpy,
            volume,
            views,
            flat_views,
            descriptors,
        )

        if stop % 100 < PHASE26_EXTRACTION_BATCH_SIZE:
            print(
                f"DINOv3 feature extraction: "
                f"{stop}/{PHASE26_CASE_COUNT}"
            )

elapsed_seconds = time.perf_counter() - extraction_started

assert np.isfinite(
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE
).all()

feature_float = (
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE.astype(
        np.float32
    )
)

cls_norms = np.linalg.norm(
    feature_float[:, :, 0, :],
    axis=-1,
)

feature_coordinate_std = feature_float.std(
    axis=0,
    dtype=np.float64,
)

collapsed_coordinate_fraction = float(np.mean(
    feature_coordinate_std < 1e-5
))

peak_vram = {
    f"cuda:{gpu}": round(
        torch.cuda.max_memory_allocated(gpu) / 1024**2,
        2,
    )
    for gpu in range(2)
}

cache_ram_gb = (
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE.nbytes
    / 1024**3
)

del feature_float, feature_coordinate_std

parallel_descriptor_model.cpu()
PHASE26_DINOV3_PRIVATE.cpu()

del parallel_descriptor_model, descriptor_model
gc.collect()
torch.cuda.empty_cache()

report = {
    "phase": "phase26_dinov3_dense_feature_cache",
    "status": "complete",
    "case_count": PHASE26_CASE_COUNT,
    "view_count": 9,
    "descriptor_count_per_view":
        len(PHASE26_DESCRIPTOR_NAMES),
    "descriptor_names": PHASE26_DESCRIPTOR_NAMES,
    "embedding_dimension": 384,
    "feature_shape": list(
        PHASE26_DINOV3_FEATURE_CACHE_PRIVATE.shape
    ),
    "dtype": str(
        PHASE26_DINOV3_FEATURE_CACHE_PRIVATE.dtype
    ),
    "cache_ram_gb": round(cache_ram_gb, 4),
    "mean_cls_l2_norm": round(
        float(cls_norms.mean()), 6
    ),
    "minimum_cls_l2_norm": round(
        float(cls_norms.min()), 6
    ),
    "collapsed_coordinate_fraction": round(
        collapsed_coordinate_fraction, 8
    ),
    "extraction_batch_size_cases":
        PHASE26_EXTRACTION_BATCH_SIZE,
    "gpu_count_used": 2,
    "peak_vram_mb": peak_vram,
    "elapsed_seconds": round(elapsed_seconds, 2),
    "backbone_frozen": True,
    "dense_patch_tokens_used": True,
    "external_weights": True,
    "license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "training_voxel_cache_read": True,
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE26_DINOV3_FEATURE_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE26_DINOV3_FEATURE_CACHE")

DINOv3 feature extraction: 104/1362
DINOv3 feature extraction: 200/1362
DINOv3 feature extraction: 304/1362
DINOv3 feature extraction: 400/1362
DINOv3 feature extraction: 504/1362
DINOv3 feature extraction: 600/1362
DINOv3 feature extraction: 704/1362
DINOv3 feature extraction: 800/1362
DINOv3 feature extraction: 904/1362
DINOv3 feature extraction: 1000/1362
DINOv3 feature extraction: 1104/1362
DINOv3 feature extraction: 1200/1362
DINOv3 feature extraction: 1304/1362
BEGIN SANITIZED_PHASE26_DINOV3_FEATURE_CACHE
{
  "phase": "phase26_dinov3_dense_feature_cache",
  "status": "complete",
  "case_count": 1362,
  "view_count": 9,
  "descriptor_count_per_view": 5,
  "descriptor_names": [
    "class_token",
    "global_patch_mean",
    "central_patch_mean",
    "absolute_left_right_patch_difference",
    "absolute_superior_inferior_patch_difference"
  ],
  "embedding_dimension": 384,
  "feature_shape": [
    1362,
    9,
    5,
    384
  ],
  "dtype": "float16",
  "cache_ram_gb": 0.0438,
  "m

In [89]:
# Cell 68 — Phase26 fixed representation construction
# No labels are used.

import json
import numpy as np

dense = np.asarray(
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE,
    dtype=np.float32,
)

assert dense.shape == (1362, 9, 5, 384)
assert np.isfinite(dense).all()

# All 45 localized descriptors per case.
dense_tokens = dense.reshape(1362, 45, 384)

# 1. Global summary, invariant to view ordering.
global_dense_statistics = np.concatenate(
    [
        dense_tokens.mean(axis=1),
        dense_tokens.std(axis=1),
        dense_tokens.max(axis=1),
    ],
    axis=1,
).astype(np.float32)

# 2. Ordered class tokens: preserves axis and slab identity.
ordered_class_tokens = dense[:, :, 0, :].reshape(
    1362,
    9 * 384,
).astype(np.float32)

# 3. Ordered dense representation: preserves all view,
# descriptor, anatomical-axis, and slab identities.
ordered_dense_tokens = dense.reshape(
    1362,
    9 * 5 * 384,
).astype(np.float32)

PHASE26_REPRESENTATIONS_PRIVATE = {
    "global_dense_statistics": global_dense_statistics,
    "ordered_class_tokens": ordered_class_tokens,
    "ordered_dense_tokens": ordered_dense_tokens,
}

representation_records = []

for name, matrix in PHASE26_REPRESENTATIONS_PRIVATE.items():
    coordinate_std = matrix.std(
        axis=0,
        dtype=np.float64,
    )

    representation_records.append({
        "name": name,
        "shape": list(matrix.shape),
        "dtype": str(matrix.dtype),
        "ram_gb": round(
            matrix.nbytes / 1024**3,
            4,
        ),
        "collapsed_coordinate_fraction": round(
            float(np.mean(coordinate_std < 1e-6)),
            8,
        ),
        "median_coordinate_std": round(
            float(np.median(coordinate_std)),
            6,
        ),
    })

report = {
    "phase": "phase26_dinov3_representations",
    "status": "complete",
    "representation_count": len(
        PHASE26_REPRESENTATIONS_PRIVATE
    ),
    "representations": representation_records,
    "labels_used": False,
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE26_REPRESENTATIONS")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE26_REPRESENTATIONS")

del dense, dense_tokens

BEGIN SANITIZED_PHASE26_REPRESENTATIONS
{
  "phase": "phase26_dinov3_representations",
  "status": "complete",
  "representation_count": 3,
  "representations": [
    {
      "name": "global_dense_statistics",
      "shape": [
        1362,
        1152
      ],
      "dtype": "float32",
      "ram_gb": 0.0058,
      "collapsed_coordinate_fraction": 0.0,
      "median_coordinate_std": 0.067065
    },
    {
      "name": "ordered_class_tokens",
      "shape": [
        1362,
        3456
      ],
      "dtype": "float32",
      "ram_gb": 0.0175,
      "collapsed_coordinate_fraction": 0.0,
      "median_coordinate_std": 0.244591
    },
    {
      "name": "ordered_dense_tokens",
      "shape": [
        1362,
        17280
      ],
      "dtype": "float32",
      "ram_gb": 0.0877,
      "collapsed_coordinate_fraction": 0.0,
      "median_coordinate_std": 0.130962
    }
  ],
  "labels_used": false,
  "case_level_features_exported": false,
  "test_or_smoke_data_read": false
}
END SANITIZED

In [90]:
# Cell 69 — Phase26 nested DINOv3 proxy selection
# Hyperparameters and blend strengths are selected on monitor only.
# Outer-validation labels are not inspected in this cell.

import json
import time
import numpy as np
from scipy.special import expit, logit
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

PHASE26_PROXY_CONFIG = {
    "pca_dimensions": [32, 64, 128],
    "logistic_c_values": [0.01, 0.03, 0.1],
    "blend_alphas": [0.0, 0.05, 0.10, 0.15, 0.20, 0.30],
    "minimum_monitor_blend_gain": 0.002,
    "pca_iterated_power": 3,
    "base_seed": 260601,
}

y_private = np.asarray(
    PHASE19_Y_PRIVATE,
    dtype=np.int64,
)

baseline_oof = np.clip(
    np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

PHASE26_DINOV3_OOF_PRIVATE = np.full(
    len(y_private),
    np.nan,
    dtype=np.float64,
)

PHASE26_DINOV3_BLEND_OOF_PRIVATE = (
    baseline_oof.copy()
)

PHASE26_SELECTED_CONFIG_PRIVATE = {}
PHASE26_DEPLOYMENT_STATES_PRIVATE = {}

phase26_selection_records = []
phase26_started = time.perf_counter()


def phase26_fit_final_pipeline(
    matrix,
    train_indices,
    target,
    valid_indices,
    dimension,
    c_value,
    seed,
):
    scaler = StandardScaler(
        with_mean=True,
        with_std=True,
    )

    train_scaled = scaler.fit_transform(
        matrix[train_indices]
    )
    valid_scaled = scaler.transform(
        matrix[valid_indices]
    )

    pca = PCA(
        n_components=dimension,
        whiten=True,
        svd_solver="randomized",
        iterated_power=
            PHASE26_PROXY_CONFIG["pca_iterated_power"],
        random_state=seed,
    )

    train_reduced = pca.fit_transform(train_scaled)
    valid_reduced = pca.transform(valid_scaled)

    classifier = LogisticRegression(
        C=c_value,
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        random_state=seed,
    )

    classifier.fit(
        train_reduced,
        target,
    )

    probability = classifier.predict_proba(
        valid_reduced
    )[:, 1]

    return (
        np.clip(probability, 1e-5, 1.0 - 1e-5),
        {
            "scaler": scaler,
            "pca": pca,
            "classifier": classifier,
        },
    )


for fold in range(3):
    fold_started = time.perf_counter()
    seed = PHASE26_PROXY_CONFIG["base_seed"] + fold

    fit_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_valid"],
        dtype=np.int64,
    )

    fit_y = y_private[fit_indices]
    monitor_y = y_private[monitor_indices]
    outer_train_y = y_private[outer_train_indices]

    candidate_records = []

    # Fit one maximum-rank PCA per representation.
    for representation_rank, (
        representation_name,
        matrix,
    ) in enumerate(
        PHASE26_REPRESENTATIONS_PRIVATE.items()
    ):
        scaler = StandardScaler(
            with_mean=True,
            with_std=True,
        )

        fit_scaled = scaler.fit_transform(
            matrix[fit_indices]
        )
        monitor_scaled = scaler.transform(
            matrix[monitor_indices]
        )

        maximum_dimension = max(
            PHASE26_PROXY_CONFIG["pca_dimensions"]
        )

        pca = PCA(
            n_components=maximum_dimension,
            whiten=True,
            svd_solver="randomized",
            iterated_power=
                PHASE26_PROXY_CONFIG[
                    "pca_iterated_power"
                ],
            random_state=seed + representation_rank,
        )

        fit_reduced_full = pca.fit_transform(
            fit_scaled
        )
        monitor_reduced_full = pca.transform(
            monitor_scaled
        )

        for dimension in PHASE26_PROXY_CONFIG[
            "pca_dimensions"
        ]:
            fit_reduced = fit_reduced_full[:, :dimension]
            monitor_reduced = (
                monitor_reduced_full[:, :dimension]
            )

            for c_value in PHASE26_PROXY_CONFIG[
                "logistic_c_values"
            ]:
                classifier = LogisticRegression(
                    C=c_value,
                    penalty="l2",
                    solver="lbfgs",
                    max_iter=3000,
                    random_state=seed,
                )

                classifier.fit(
                    fit_reduced,
                    fit_y,
                )

                probability = np.clip(
                    classifier.predict_proba(
                        monitor_reduced
                    )[:, 1],
                    1e-5,
                    1.0 - 1e-5,
                )

                candidate_records.append({
                    "representation":
                        representation_name,
                    "representation_rank":
                        representation_rank,
                    "dimension": int(dimension),
                    "c_value": float(c_value),
                    "monitor_log_loss": float(
                        log_loss(
                            monitor_y,
                            probability,
                            labels=[0, 1],
                        )
                    ),
                    "monitor_auroc": float(
                        roc_auc_score(
                            monitor_y,
                            probability,
                        )
                    ),
                    "monitor_probability":
                        probability,
                })

        del (
            scaler,
            fit_scaled,
            monitor_scaled,
            pca,
            fit_reduced_full,
            monitor_reduced_full,
        )

    selected_candidate = min(
        candidate_records,
        key=lambda row: (
            row["monitor_log_loss"],
            row["dimension"],
            row["c_value"],
            row["representation_rank"],
        ),
    )

    selected_monitor_probability = (
        selected_candidate["monitor_probability"]
    )

    baseline_monitor_probability = np.clip(
        np.asarray(
            PHASE24_BASE_MONITOR_PRIVATE[fold],
            dtype=np.float64,
        ),
        1e-5,
        1.0 - 1e-5,
    )

    baseline_monitor_loss = float(log_loss(
        monitor_y,
        baseline_monitor_probability,
        labels=[0, 1],
    ))

    baseline_monitor_logit = logit(
        baseline_monitor_probability
    )
    dino_monitor_logit = logit(
        selected_monitor_probability
    )

    blend_candidates = []

    for alpha in PHASE26_PROXY_CONFIG[
        "blend_alphas"
    ]:
        blended_probability = expit(
            baseline_monitor_logit
            + float(alpha) * (
                dino_monitor_logit
                - baseline_monitor_logit
            )
        )

        blend_candidates.append({
            "alpha": float(alpha),
            "log_loss": float(log_loss(
                monitor_y,
                blended_probability,
                labels=[0, 1],
            )),
            "auroc": float(roc_auc_score(
                monitor_y,
                blended_probability,
            )),
        })

    raw_best_blend = min(
        blend_candidates,
        key=lambda row: (
            row["log_loss"],
            row["alpha"],
        ),
    )

    raw_blend_gain = (
        baseline_monitor_loss
        - raw_best_blend["log_loss"]
    )

    if (
        raw_best_blend["alpha"] > 0.0
        and raw_blend_gain
        >= PHASE26_PROXY_CONFIG[
            "minimum_monitor_blend_gain"
        ]
    ):
        selected_alpha = float(
            raw_best_blend["alpha"]
        )
        blend_advanced = True
    else:
        selected_alpha = 0.0
        blend_advanced = False

    representation_name = (
        selected_candidate["representation"]
    )
    representation_matrix = (
        PHASE26_REPRESENTATIONS_PRIVATE[
            representation_name
        ]
    )

    final_probability, deployment_state = (
        phase26_fit_final_pipeline(
            matrix=representation_matrix,
            train_indices=outer_train_indices,
            target=outer_train_y,
            valid_indices=outer_valid_indices,
            dimension=selected_candidate["dimension"],
            c_value=selected_candidate["c_value"],
            seed=seed + 100,
        )
    )

    PHASE26_DINOV3_OOF_PRIVATE[
        outer_valid_indices
    ] = final_probability

    PHASE26_DINOV3_BLEND_OOF_PRIVATE[
        outer_valid_indices
    ] = expit(
        logit(baseline_oof[outer_valid_indices])
        + selected_alpha * (
            logit(final_probability)
            - logit(
                baseline_oof[outer_valid_indices]
            )
        )
    )

    PHASE26_SELECTED_CONFIG_PRIVATE[fold] = {
        "representation": representation_name,
        "dimension": int(
            selected_candidate["dimension"]
        ),
        "c_value": float(
            selected_candidate["c_value"]
        ),
        "blend_alpha": selected_alpha,
    }

    deployment_state.update(
        PHASE26_SELECTED_CONFIG_PRIVATE[fold]
    )
    PHASE26_DEPLOYMENT_STATES_PRIVATE[fold] = (
        deployment_state
    )

    phase26_selection_records.append({
        "fold": fold,
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(len(monitor_indices)),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "candidate_count": len(candidate_records),
        "selected_representation":
            representation_name,
        "selected_pca_dimension": int(
            selected_candidate["dimension"]
        ),
        "selected_logistic_c": float(
            selected_candidate["c_value"]
        ),
        "dinov3_monitor_log_loss": round(
            selected_candidate["monitor_log_loss"],
            6,
        ),
        "dinov3_monitor_auroc": round(
            selected_candidate["monitor_auroc"],
            6,
        ),
        "phase12c_monitor_log_loss": round(
            baseline_monitor_loss,
            6,
        ),
        "raw_best_blend_alpha": float(
            raw_best_blend["alpha"]
        ),
        "raw_best_blend_monitor_log_loss": round(
            raw_best_blend["log_loss"],
            6,
        ),
        "raw_monitor_blend_gain": round(
            raw_blend_gain,
            6,
        ),
        "selected_blend_alpha": selected_alpha,
        "blend_advanced": blend_advanced,
        "elapsed_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    print(
        f"Phase26 fold {fold}: "
        f"representation={representation_name}, "
        f"dim={selected_candidate['dimension']}, "
        f"C={selected_candidate['c_value']}, "
        f"alpha={selected_alpha:.2f}"
    )

assert np.isfinite(
    PHASE26_DINOV3_OOF_PRIVATE
).all()
assert np.isfinite(
    PHASE26_DINOV3_BLEND_OOF_PRIVATE
).all()

report = {
    "phase": "phase26_dinov3_nested_proxy",
    "status": "complete",
    "configuration": PHASE26_PROXY_CONFIG,
    "folds": phase26_selection_records,
    "elapsed_seconds": round(
        time.perf_counter() - phase26_started,
        2,
    ),
    "backbone_updated": False,
    "monitor_labels_used_for_selection": True,
    "outer_train_labels_used_for_final_refit": True,
    "outer_validation_labels_used": False,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE26_NESTED_PROXY")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE26_NESTED_PROXY")

Phase26 fold 0: representation=ordered_class_tokens, dim=64, C=0.1, alpha=0.15
Phase26 fold 1: representation=ordered_class_tokens, dim=64, C=0.1, alpha=0.00
Phase26 fold 2: representation=ordered_class_tokens, dim=64, C=0.1, alpha=0.00
BEGIN SANITIZED_PHASE26_NESTED_PROXY
{
  "phase": "phase26_dinov3_nested_proxy",
  "status": "complete",
  "configuration": {
    "pca_dimensions": [
      32,
      64,
      128
    ],
    "logistic_c_values": [
      0.01,
      0.03,
      0.1
    ],
    "blend_alphas": [
      0.0,
      0.05,
      0.1,
      0.15,
      0.2,
      0.3
    ],
    "minimum_monitor_blend_gain": 0.002,
    "pca_iterated_power": 3,
    "base_seed": 260601
  },
  "folds": [
    {
      "fold": 0,
      "fit_n": 721,
      "monitor_n": 174,
      "outer_train_n": 895,
      "outer_valid_n": 467,
      "candidate_count": 27,
      "selected_representation": "ordered_class_tokens",
      "selected_pca_dimension": 64,
      "selected_logistic_c": 0.1,
      "dinov3_monitor

In [91]:
# Cell 70 — Phase26 DINOv3 OOF and component gate

import json
import numpy as np
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    brier_score_loss,
)

y = np.asarray(PHASE19_Y_PRIVATE, dtype=np.int64)

baseline = np.clip(
    np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

dino = np.clip(
    np.asarray(
        PHASE26_DINOV3_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

blended = np.clip(
    np.asarray(
        PHASE26_DINOV3_BLEND_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)


def phase26_metrics(target, probability):
    return {
        "log_loss": float(log_loss(
            target,
            probability,
            labels=[0, 1],
        )),
        "auroc": float(roc_auc_score(
            target,
            probability,
        )),
        "brier": float(brier_score_loss(
            target,
            probability,
        )),
    }


def phase26_calibration(target, probability):
    predictor = logit(probability).reshape(-1, 1)

    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
    model.fit(predictor, target)

    return {
        "intercept": float(model.intercept_[0]),
        "slope": float(model.coef_[0, 0]),
    }


baseline_metrics = phase26_metrics(y, baseline)
dino_metrics = phase26_metrics(y, dino)
blend_metrics = phase26_metrics(y, blended)

dino_calibration = phase26_calibration(y, dino)
blend_calibration = phase26_calibration(y, blended)

fold_records = []
fold_wins = 0
worst_fold_excess = -np.inf

for fold in range(3):
    indices = np.asarray(
        PHASE19_PARTITIONS[fold]["outer_valid"],
        dtype=np.int64,
    )

    base_fold = phase26_metrics(
        y[indices], baseline[indices]
    )
    dino_fold = phase26_metrics(
        y[indices], dino[indices]
    )
    blend_fold = phase26_metrics(
        y[indices], blended[indices]
    )

    improvement = (
        base_fold["log_loss"]
        - blend_fold["log_loss"]
    )

    fold_wins += int(improvement > 0.0)
    worst_fold_excess = max(
        worst_fold_excess,
        -improvement,
    )

    fold_records.append({
        "fold": fold,
        "selected_representation":
            PHASE26_SELECTED_CONFIG_PRIVATE[fold][
                "representation"
            ],
        "selected_blend_alpha": float(
            PHASE26_SELECTED_CONFIG_PRIVATE[fold][
                "blend_alpha"
            ]
        ),
        "phase12c_log_loss": round(
            base_fold["log_loss"], 6
        ),
        "dinov3_log_loss": round(
            dino_fold["log_loss"], 6
        ),
        "blend_log_loss": round(
            blend_fold["log_loss"], 6
        ),
        "blend_improvement": round(
            improvement, 6
        ),
        "phase12c_auroc": round(
            base_fold["auroc"], 6
        ),
        "dinov3_auroc": round(
            dino_fold["auroc"], 6
        ),
        "blend_auroc": round(
            blend_fold["auroc"], 6
        ),
    })


groups = np.asarray(case_df["acquisition_group"])
major_group_records = []
maximum_major_group_harm = 0.0

for group in np.unique(groups):
    indices = np.flatnonzero(groups == group)

    if len(indices) < 30:
        continue

    baseline_loss = float(log_loss(
        y[indices],
        baseline[indices],
        labels=[0, 1],
    ))
    blend_loss = float(log_loss(
        y[indices],
        blended[indices],
        labels=[0, 1],
    ))

    harm = blend_loss - baseline_loss
    maximum_major_group_harm = max(
        maximum_major_group_harm,
        harm,
    )

    major_group_records.append({
        "group": int(group),
        "n": int(len(indices)),
        "baseline_log_loss": round(
            baseline_loss, 6
        ),
        "blend_log_loss": round(
            blend_loss, 6
        ),
        "log_loss_change": round(harm, 6),
    })


log_loss_gain = (
    baseline_metrics["log_loss"]
    - blend_metrics["log_loss"]
)
auroc_gain = (
    blend_metrics["auroc"]
    - baseline_metrics["auroc"]
)
brier_excess = (
    blend_metrics["brier"]
    - baseline_metrics["brier"]
)

advanced_fold_count = sum(
    record["blend_advanced"]
    for record in phase26_selection_records
)

component_gate_passed = bool(
    dino_metrics["auroc"] >= 0.90
    and advanced_fold_count >= 1
    and log_loss_gain >= 0.001
    and worst_fold_excess <= 0.01
)

promotion_gate_passed = bool(
    log_loss_gain >= 0.003
    and auroc_gain >= 0.002
    and fold_wins >= 2
    and worst_fold_excess <= 0.005
    and brier_excess <= 0.001
    and 0.8 <= blend_calibration["slope"] <= 1.2
    and maximum_major_group_harm <= 0.015
)

report = {
    "phase": "phase26_dinov3_frozen_component",
    "status": (
        "promoted"
        if promotion_gate_passed
        else (
            "stage_b_eligible"
            if component_gate_passed
            else "terminated"
        )
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "dinov3_standalone": {
        **{
            key: round(value, 6)
            for key, value in dino_metrics.items()
        },
        "calibration_intercept": round(
            dino_calibration["intercept"], 6
        ),
        "calibration_slope": round(
            dino_calibration["slope"], 6
        ),
    },
    "phase12c_dinov3_blend": {
        **{
            key: round(value, 6)
            for key, value in blend_metrics.items()
        },
        "calibration_intercept": round(
            blend_calibration["intercept"], 6
        ),
        "calibration_slope": round(
            blend_calibration["slope"], 6
        ),
    },
    "improvements": {
        "blend_log_loss_gain": round(
            log_loss_gain, 6
        ),
        "blend_auroc_gain": round(
            auroc_gain, 6
        ),
        "blend_brier_excess": round(
            brier_excess, 6
        ),
        "blend_fold_wins": int(fold_wins),
        "advanced_monitor_fold_count": int(
            advanced_fold_count
        ),
        "worst_blend_fold_excess": round(
            float(worst_fold_excess), 6
        ),
        "maximum_major_group_harm": round(
            maximum_major_group_harm, 6
        ),
        "logit_prediction_correlation": round(
            float(np.corrcoef(
                logit(baseline),
                logit(dino),
            )[0, 1]),
            6,
        ),
    },
    "fold_metrics": fold_records,
    "major_acquisition_group_metrics":
        major_group_records,
    "component_gate_passed": component_gate_passed,
    "promotion_gate_passed": promotion_gate_passed,
    "stage_b_meaning": (
        "train a low-capacity attentive multi-view probe; "
        "keep DINOv3 frozen initially"
    ),
    "backbone_updated": False,
    "external_weights": True,
    "license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "outer_validation_labels_used_only_for_final_evaluation":
        True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE26_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE26_OOF")

BEGIN SANITIZED_PHASE26_OOF
{
  "phase": "phase26_dinov3_frozen_component",
  "status": "stage_b_eligible",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "dinov3_standalone": {
    "log_loss": 0.37838,
    "auroc": 0.911055,
    "brier": 0.120334,
    "calibration_intercept": -0.165494,
    "calibration_slope": 0.991018
  },
  "phase12c_dinov3_blend": {
    "log_loss": 0.304742,
    "auroc": 0.939583,
    "brier": 0.094448,
    "calibration_intercept": 0.128636,
    "calibration_slope": 0.946833
  },
  "improvements": {
    "blend_log_loss_gain": 0.002873,
    "blend_auroc_gain": 0.000655,
    "blend_brier_excess": -0.001087,
    "blend_fold_wins": 1,
    "advanced_monitor_fold_count": 1,
    "worst_blend_fold_excess": -0.0,
    "maximum_major_group_harm": 0.0,
    "logit_prediction_correlation": 0.862708
  },
  "fold_metrics": [
    {
      "fold": 0,
      "selected_representation": "ordered_class_tokens",
      "selected_blend_alpha": 

In [92]:
# Cell 71 — Phase27 structured attentive multi-view probe

import copy
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


PHASE27_CONFIG = {
    "phase": "phase27_dinov3_structured_attention",
    "input_descriptor": "class_token",
    "view_count": 9,
    "embedding_dimension": 384,
    "model_dimension": 64,
    "attention_heads": 4,
    "attention_layers": 1,
    "dropout": 0.10,
    "token_dropout": 0.03,
    "feature_noise": 0.01,
    "batch_size": 64,
    "maximum_epochs": 120,
    "patience": 18,
    "learning_rate": 3e-4,
    "weight_decay": 0.03,
    "label_smoothing": 0.01,
    "gradient_clip": 1.0,
    "seeds": [270701, 270702, 270703],
    "platt_regularization": 0.02,
    "blend_alphas": [
        0.0, 0.025, 0.05, 0.075,
        0.10, 0.15, 0.20, 0.30
    ],
    "minimum_monitor_blend_gain": 0.002,
}


torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

if hasattr(torch.backends.cuda, "enable_flash_sdp"):
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


phase27_dense_cache = np.asarray(
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE
)

assert phase27_dense_cache.shape == (1362, 9, 5, 384)

# Descriptor zero is the DINOv3 class token.
PHASE27_X_PRIVATE = np.asarray(
    phase27_dense_cache[:, :, 0, :],
    dtype=np.float32,
).copy()

assert PHASE27_X_PRIVATE.shape == (1362, 9, 384)
assert np.isfinite(PHASE27_X_PRIVATE).all()


class Phase27AttentionBlock(nn.Module):
    def __init__(self, dimension=64, heads=4, dropout=0.10):
        super().__init__()

        assert dimension % heads == 0

        self.dimension = dimension
        self.heads = heads
        self.head_dimension = dimension // heads
        self.scale = self.head_dimension ** -0.5

        self.norm1 = nn.LayerNorm(dimension)
        self.qkv = nn.Linear(dimension, 3 * dimension)
        self.output_projection = nn.Linear(dimension, dimension)

        self.norm2 = nn.LayerNorm(dimension)
        self.mlp = nn.Sequential(
            nn.Linear(dimension, 4 * dimension),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * dimension, dimension),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, token_count, dimension = x.shape

        normalized = self.norm1(x)

        qkv = self.qkv(normalized)
        qkv = qkv.reshape(
            batch_size,
            token_count,
            3,
            self.heads,
            self.head_dimension,
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)

        query, key, value = qkv.unbind(dim=0)

        attention = torch.matmul(
            query,
            key.transpose(-2, -1),
        ) * self.scale

        attention = torch.softmax(attention, dim=-1)
        attention = self.dropout(attention)

        mixed = torch.matmul(attention, value)
        mixed = mixed.transpose(1, 2).contiguous()
        mixed = mixed.reshape(batch_size, token_count, dimension)

        x = x + self.dropout(
            self.output_projection(mixed)
        )
        x = x + self.dropout(
            self.mlp(self.norm2(x))
        )

        return x


class Phase27StructuredAttentionProbe(nn.Module):
    def __init__(
        self,
        input_dimension=384,
        model_dimension=64,
        heads=4,
        dropout=0.10,
        token_dropout=0.03,
        feature_noise=0.01,
    ):
        super().__init__()

        self.input_dimension = input_dimension
        self.model_dimension = model_dimension
        self.token_dropout = float(token_dropout)
        self.feature_noise = float(feature_noise)

        # Non-affine normalization avoids adding a high-capacity
        # coordinate-specific rescaling layer.
        self.input_normalization = nn.LayerNorm(
            input_dimension,
            elementwise_affine=False,
        )

        self.input_projection = nn.Sequential(
            nn.Linear(input_dimension, model_dimension),
            nn.GELU(),
            nn.LayerNorm(model_dimension),
        )

        # View order is axis-major:
        # axis 0 slabs 0/1/2, then axis 1, then axis 2.
        self.axis_embedding = nn.Parameter(
            torch.zeros(3, model_dimension)
        )
        self.slab_embedding = nn.Parameter(
            torch.zeros(3, model_dimension)
        )

        nn.init.trunc_normal_(
            self.axis_embedding,
            std=0.02,
        )
        nn.init.trunc_normal_(
            self.slab_embedding,
            std=0.02,
        )

        self.attention_block = Phase27AttentionBlock(
            dimension=model_dimension,
            heads=heads,
            dropout=dropout,
        )

        self.attention_pool = nn.Sequential(
            nn.Linear(model_dimension, model_dimension // 2),
            nn.Tanh(),
            nn.Linear(model_dimension // 2, 1),
        )

        # Five structured summaries:
        # attentive mean, ordinary mean, standard deviation,
        # center-vs-edge slab contrast, and axis dispersion.
        pooled_dimension = 5 * model_dimension

        self.classifier = nn.Sequential(
            nn.LayerNorm(pooled_dimension),
            nn.Linear(pooled_dimension, model_dimension),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(model_dimension, 1),
        )

    def forward(self, class_tokens, return_attention=False):
        assert class_tokens.ndim == 3
        assert class_tokens.shape[1:] == (9, 384)

        x = self.input_normalization(class_tokens)

        if self.training and self.feature_noise > 0:
            x = x + self.feature_noise * torch.randn_like(x)

        x = self.input_projection(x)

        axis_indices = torch.arange(
            3,
            device=x.device,
        ).repeat_interleave(3)

        slab_indices = torch.arange(
            3,
            device=x.device,
        ).repeat(3)

        x = (
            x
            + self.axis_embedding[axis_indices][None]
            + self.slab_embedding[slab_indices][None]
        )

        if self.training and self.token_dropout > 0:
            keep = (
                torch.rand(
                    x.shape[0],
                    x.shape[1],
                    1,
                    device=x.device,
                )
                >= self.token_dropout
            ).to(x.dtype)

            x = x * keep / (1.0 - self.token_dropout)

        x = self.attention_block(x)

        attention_scores = self.attention_pool(x).squeeze(-1)
        attention_weights = torch.softmax(
            attention_scores,
            dim=1,
        )

        attentive_mean = torch.sum(
            x * attention_weights.unsqueeze(-1),
            dim=1,
        )

        global_mean = x.mean(dim=1)
        global_std = x.std(dim=1, unbiased=False)

        structured = x.reshape(
            x.shape[0],
            3,
            3,
            self.model_dimension,
        )

        center_slabs = structured[:, :, 1, :]
        edge_slabs = 0.5 * (
            structured[:, :, 0, :]
            + structured[:, :, 2, :]
        )

        center_edge_contrast = torch.abs(
            center_slabs - edge_slabs
        ).mean(dim=1)

        axis_means = structured.mean(dim=2)
        axis_dispersion = torch.sqrt(
            axis_means.var(dim=1, unbiased=False) + 1e-6
        )

        pooled = torch.cat(
            [
                attentive_mean,
                global_mean,
                global_std,
                center_edge_contrast,
                axis_dispersion,
            ],
            dim=1,
        )

        logit = self.classifier(pooled).squeeze(-1)

        if return_attention:
            return logit, attention_weights

        return logit


def phase27_build_model():
    return Phase27StructuredAttentionProbe(
        input_dimension=384,
        model_dimension=PHASE27_CONFIG["model_dimension"],
        heads=PHASE27_CONFIG["attention_heads"],
        dropout=PHASE27_CONFIG["dropout"],
        token_dropout=PHASE27_CONFIG["token_dropout"],
        feature_noise=PHASE27_CONFIG["feature_noise"],
    )


phase27_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(270700)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(270700)
    torch.cuda.reset_peak_memory_stats(phase27_device)

probe = phase27_build_model().to(phase27_device)
probe.train()

synthetic = torch.randn(
    2,
    9,
    384,
    device=phase27_device,
)

synthetic_labels = torch.tensor(
    [0.0, 1.0],
    device=phase27_device,
)

synthetic_logits, synthetic_attention = probe(
    synthetic,
    return_attention=True,
)

synthetic_loss = F.binary_cross_entropy_with_logits(
    synthetic_logits,
    synthetic_labels,
)

synthetic_loss.backward()

gradient_norm = torch.nn.utils.clip_grad_norm_(
    probe.parameters(),
    1.0,
)

parameter_count = sum(
    parameter.numel()
    for parameter in probe.parameters()
)

peak_vram_mb = (
    torch.cuda.max_memory_allocated(phase27_device)
    / (1024 ** 2)
    if phase27_device.type == "cuda"
    else 0.0
)

assert synthetic_logits.shape == (2,)
assert synthetic_attention.shape == (2, 9)
assert torch.isfinite(synthetic_logits).all()
assert torch.isfinite(synthetic_attention).all()
assert torch.allclose(
    synthetic_attention.sum(dim=1),
    torch.ones(2, device=phase27_device),
    atol=1e-6,
)

architecture_report = {
    "phase": PHASE27_CONFIG["phase"],
    "status": "accepted",
    "model": "Phase27StructuredAttentionProbe",
    "parameter_count": int(parameter_count),
    "input_shape": [2, 9, 384],
    "logit_shape": list(synthetic_logits.shape),
    "attention_shape": list(synthetic_attention.shape),
    "attention_sum_max_abs_error": float(
        torch.max(
            torch.abs(
                synthetic_attention.sum(dim=1) - 1.0
            )
        ).item()
    ),
    "gradient_norm_before_clipping": round(
        float(gradient_norm),
        6,
    ),
    "peak_vram_mb": round(float(peak_vram_mb), 2),
    "structured_summaries": [
        "attentive_mean",
        "global_mean",
        "global_standard_deviation",
        "center_edge_slab_contrast",
        "axis_dispersion",
    ],
    "dinov3_backbone_frozen": True,
    "external_weights": True,
    "license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "synthetic_input_only": True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE27_ARCHITECTURE")
print(json.dumps(architecture_report, indent=2))
print("END SANITIZED_PHASE27_ARCHITECTURE")

del probe, synthetic, synthetic_labels
torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE27_ARCHITECTURE
{
  "phase": "phase27_dinov3_structured_attention",
  "status": "accepted",
  "model": "Phase27StructuredAttentionProbe",
  "parameter_count": 98498,
  "input_shape": [
    2,
    9,
    384
  ],
  "logit_shape": [
    2
  ],
  "attention_shape": [
    2,
    9
  ],
  "attention_sum_max_abs_error": 5.960464477539063e-08,
  "gradient_norm_before_clipping": 3.148507,
  "peak_vram_mb": 137.49,
  "structured_summaries": [
    "attentive_mean",
    "global_mean",
    "global_standard_deviation",
    "center_edge_slab_contrast",
    "axis_dispersion"
  ],
  "dinov3_backbone_frozen": true,
  "external_weights": true,
  "license": "DINOv3 License",
  "required_attribution": "Built with DINOv3",
  "synthetic_input_only": true,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE27_ARCHITECTURE


In [93]:
# Cell 72 — Honest fit/monitor selection and outer-train refit

from scipy.optimize import minimize
from sklearn.metrics import log_loss, roc_auc_score


PHASE27_Y_PRIVATE = np.asarray(
    PHASE19_Y_PRIVATE,
    dtype=np.float32,
).reshape(-1)

assert PHASE27_Y_PRIVATE.shape == (1362,)
assert set(np.unique(PHASE27_Y_PRIVATE)).issubset({0.0, 1.0})

phase27_partition_by_fold = {
    int(partition["fold"]): partition
    for partition in PHASE19_PARTITIONS
}

PHASE27_X_GPU_PRIVATE = torch.from_numpy(
    PHASE27_X_PRIVATE
).to(phase27_device)

PHASE27_Y_GPU_PRIVATE = torch.from_numpy(
    PHASE27_Y_PRIVATE
).to(phase27_device)


def phase27_seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def phase27_sigmoid(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = np.clip(logits, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-logits))


def phase27_logit(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )
    return np.log(probabilities / (1.0 - probabilities))


def phase27_log_loss(labels, probabilities):
    probabilities = np.clip(
        probabilities,
        1e-5,
        1.0 - 1e-5,
    )
    return float(
        log_loss(
            labels,
            probabilities,
            labels=[0, 1],
        )
    )


def phase27_fold_array(container, fold):
    value = None

    if isinstance(container, dict):
        for key in [
            fold,
            str(fold),
            f"fold_{fold}",
            f"fold{fold}",
        ]:
            if key in container:
                value = container[key]
                break

        if value is None:
            for candidate in container.values():
                if (
                    isinstance(candidate, dict)
                    and int(candidate.get("fold", -1)) == fold
                ):
                    for field in [
                        "probabilities",
                        "predictions",
                        "values",
                        "monitor",
                    ]:
                        if field in candidate:
                            value = candidate[field]
                            break

    elif isinstance(container, (list, tuple)):
        value = container[fold]

    else:
        value = container

    assert value is not None, {
        "message": "Could not resolve fold-local array.",
        "fold": fold,
        "container_type": type(container).__name__,
    }

    return np.asarray(value, dtype=np.float64).reshape(-1)


@torch.no_grad()
def phase27_predict_model(model, indices, batch_size=256):
    model.eval()
    outputs = []

    indices = np.asarray(indices, dtype=np.int64)

    for start in range(0, len(indices), batch_size):
        batch_indices = torch.as_tensor(
            indices[start:start + batch_size],
            device=phase27_device,
            dtype=torch.long,
        )

        logits = model(
            PHASE27_X_GPU_PRIVATE[batch_indices]
        )

        outputs.append(
            logits.detach().cpu().numpy()
        )

    return np.concatenate(outputs).astype(np.float64)


def phase27_cpu_state(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def phase27_train_with_monitor(
    train_indices,
    monitor_indices,
    seed,
    fold,
):
    phase27_seed_everything(seed)

    model = phase27_build_model().to(phase27_device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE27_CONFIG["learning_rate"],
        weight_decay=PHASE27_CONFIG["weight_decay"],
    )

    best_loss = float("inf")
    best_epoch = 0
    best_state = None
    stale_epochs = 0
    epoch_records = []

    train_indices = np.asarray(train_indices, dtype=np.int64)
    monitor_indices = np.asarray(monitor_indices, dtype=np.int64)
    monitor_labels = PHASE27_Y_PRIVATE[monitor_indices]

    for epoch in range(
        1,
        PHASE27_CONFIG["maximum_epochs"] + 1,
    ):
        model.train()

        epoch_rng = np.random.default_rng(
            seed + epoch * 1009
        )
        order = epoch_rng.permutation(train_indices)

        total_loss = 0.0
        total_cases = 0

        for start in range(
            0,
            len(order),
            PHASE27_CONFIG["batch_size"],
        ):
            batch_numpy = order[
                start:start + PHASE27_CONFIG["batch_size"]
            ]

            batch_indices = torch.as_tensor(
                batch_numpy,
                device=phase27_device,
                dtype=torch.long,
            )

            inputs = PHASE27_X_GPU_PRIVATE[batch_indices]
            labels = PHASE27_Y_GPU_PRIVATE[batch_indices]

            smoothing = PHASE27_CONFIG["label_smoothing"]
            soft_labels = (
                labels * (1.0 - smoothing)
                + 0.5 * smoothing
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)

            loss = F.binary_cross_entropy_with_logits(
                logits,
                soft_labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE27_CONFIG["gradient_clip"],
            )

            optimizer.step()

            total_loss += (
                float(loss.detach().item())
                * len(batch_numpy)
            )
            total_cases += len(batch_numpy)

        monitor_logits = phase27_predict_model(
            model,
            monitor_indices,
        )

        monitor_probability = phase27_sigmoid(
            monitor_logits
        )

        monitor_loss = phase27_log_loss(
            monitor_labels,
            monitor_probability,
        )

        epoch_records.append({
            "epoch": int(epoch),
            "train_loss": float(
                total_loss / total_cases
            ),
            "monitor_log_loss": monitor_loss,
        })

        if monitor_loss < best_loss - 1e-5:
            best_loss = monitor_loss
            best_epoch = epoch
            best_state = phase27_cpu_state(model)
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % 20 == 0:
            print(
                f"Phase27 fold {fold} seed {seed} "
                f"epoch {epoch:03d}: "
                f"train={total_loss / total_cases:.4f}, "
                f"monitor={monitor_loss:.4f}"
            )

        if stale_epochs >= PHASE27_CONFIG["patience"]:
            break

    assert best_state is not None
    assert best_epoch > 0

    model.load_state_dict(best_state, strict=True)

    best_monitor_logits = phase27_predict_model(
        model,
        monitor_indices,
    )

    del model
    torch.cuda.empty_cache()

    return {
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_monitor_log_loss": float(best_loss),
        "state": best_state,
        "monitor_logits": best_monitor_logits,
        "epochs_run": int(len(epoch_records)),
    }


def phase27_train_fixed_epochs(
    train_indices,
    seed,
    epochs,
):
    phase27_seed_everything(seed)

    model = phase27_build_model().to(phase27_device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE27_CONFIG["learning_rate"],
        weight_decay=PHASE27_CONFIG["weight_decay"],
    )

    train_indices = np.asarray(train_indices, dtype=np.int64)
    final_loss = None

    for epoch in range(1, int(epochs) + 1):
        model.train()

        epoch_rng = np.random.default_rng(
            seed + epoch * 1009
        )
        order = epoch_rng.permutation(train_indices)

        total_loss = 0.0
        total_cases = 0

        for start in range(
            0,
            len(order),
            PHASE27_CONFIG["batch_size"],
        ):
            batch_numpy = order[
                start:start + PHASE27_CONFIG["batch_size"]
            ]

            batch_indices = torch.as_tensor(
                batch_numpy,
                device=phase27_device,
                dtype=torch.long,
            )

            inputs = PHASE27_X_GPU_PRIVATE[batch_indices]
            labels = PHASE27_Y_GPU_PRIVATE[batch_indices]

            smoothing = PHASE27_CONFIG["label_smoothing"]
            soft_labels = (
                labels * (1.0 - smoothing)
                + 0.5 * smoothing
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)

            loss = F.binary_cross_entropy_with_logits(
                logits,
                soft_labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE27_CONFIG["gradient_clip"],
            )

            optimizer.step()

            total_loss += (
                float(loss.detach().item())
                * len(batch_numpy)
            )
            total_cases += len(batch_numpy)

        final_loss = total_loss / total_cases

    state = phase27_cpu_state(model)

    del model
    torch.cuda.empty_cache()

    return state, float(final_loss)


def phase27_predict_state(state, indices):
    model = phase27_build_model().to(phase27_device)
    model.load_state_dict(state, strict=True)

    logits = phase27_predict_model(
        model,
        indices,
    )

    del model
    torch.cuda.empty_cache()

    return logits


def phase27_fit_regularized_platt(logits, labels):
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)

    regularization = float(
        PHASE27_CONFIG["platt_regularization"]
    )

    def objective(parameters):
        intercept, slope = parameters

        probability = phase27_sigmoid(
            intercept + slope * logits
        )

        likelihood = phase27_log_loss(
            labels,
            probability,
        )

        penalty = regularization * (
            0.25 * intercept ** 2
            + (slope - 1.0) ** 2
        )

        return likelihood + penalty

    result = minimize(
        objective,
        x0=np.asarray([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-1.5, 1.5),
            (0.35, 1.75),
        ],
    )

    assert result.success, result.message

    return float(result.x[0]), float(result.x[1])


phase27_started = time.perf_counter()

PHASE27_ATTENTION_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE27_ATTENTION_BLEND_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE27_DEPLOYMENT_STATES_PRIVATE = {}
phase27_selection_records = []

for fold in range(3):
    fold_started = time.perf_counter()

    partition = phase27_partition_by_fold[fold]

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    monitor_labels = PHASE27_Y_PRIVATE[
        monitor_indices
    ]

    baseline_monitor_probability = phase27_fold_array(
        PHASE24_BASE_MONITOR_PRIVATE,
        fold,
    )

    assert len(baseline_monitor_probability) == len(
        monitor_indices
    )

    seed_results = []

    for seed in PHASE27_CONFIG["seeds"]:
        result = phase27_train_with_monitor(
            train_indices=fit_indices,
            monitor_indices=monitor_indices,
            seed=seed,
            fold=fold,
        )

        seed_results.append(result)

        print(
            f"Phase27 fold {fold} seed {seed} frozen: "
            f"best_epoch={result['best_epoch']}, "
            f"monitor={result['best_monitor_log_loss']:.6f}"
        )

    ensemble_monitor_logits = np.mean(
        np.stack(
            [
                result["monitor_logits"]
                for result in seed_results
            ],
            axis=0,
        ),
        axis=0,
    )

    platt_intercept, platt_slope = (
        phase27_fit_regularized_platt(
            ensemble_monitor_logits,
            monitor_labels,
        )
    )

    calibrated_monitor_logits = (
        platt_intercept
        + platt_slope * ensemble_monitor_logits
    )

    calibrated_monitor_probability = phase27_sigmoid(
        calibrated_monitor_logits
    )

    baseline_monitor_loss = phase27_log_loss(
        monitor_labels,
        baseline_monitor_probability,
    )

    standalone_monitor_loss = phase27_log_loss(
        monitor_labels,
        calibrated_monitor_probability,
    )

    standalone_monitor_auroc = float(
        roc_auc_score(
            monitor_labels,
            calibrated_monitor_probability,
        )
    )

    baseline_monitor_logits = phase27_logit(
        baseline_monitor_probability
    )

    alpha_candidates = []

    for alpha in PHASE27_CONFIG["blend_alphas"]:
        blended_logits = (
            (1.0 - alpha) * baseline_monitor_logits
            + alpha * calibrated_monitor_logits
        )

        blended_probability = phase27_sigmoid(
            blended_logits
        )

        candidate_loss = phase27_log_loss(
            monitor_labels,
            blended_probability,
        )

        alpha_candidates.append({
            "alpha": float(alpha),
            "log_loss": float(candidate_loss),
        })

    raw_best = min(
        alpha_candidates,
        key=lambda record: (
            record["log_loss"],
            record["alpha"],
        ),
    )

    raw_gain = (
        baseline_monitor_loss
        - raw_best["log_loss"]
    )

    if (
        raw_gain
        >= PHASE27_CONFIG[
            "minimum_monitor_blend_gain"
        ]
    ):
        selected_alpha = raw_best["alpha"]
        blend_advanced = selected_alpha > 0
    else:
        selected_alpha = 0.0
        blend_advanced = False

    # Refit each seed using all outer-training cases.
    refit_states = []
    refit_final_losses = []

    for seed_result in seed_results:
        refit_state, refit_final_loss = (
            phase27_train_fixed_epochs(
                train_indices=outer_train_indices,
                seed=seed_result["seed"],
                epochs=seed_result["best_epoch"],
            )
        )

        refit_states.append(refit_state)
        refit_final_losses.append(refit_final_loss)

    outer_seed_logits = []

    for refit_state in refit_states:
        outer_seed_logits.append(
            phase27_predict_state(
                refit_state,
                outer_valid_indices,
            )
        )

    ensemble_outer_logits = np.mean(
        np.stack(outer_seed_logits, axis=0),
        axis=0,
    )

    calibrated_outer_logits = (
        platt_intercept
        + platt_slope * ensemble_outer_logits
    )

    standalone_outer_probability = phase27_sigmoid(
        calibrated_outer_logits
    )

    baseline_outer_probability = np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    )[outer_valid_indices]

    baseline_outer_logits = phase27_logit(
        baseline_outer_probability
    )

    blended_outer_logits = (
        (1.0 - selected_alpha)
        * baseline_outer_logits
        + selected_alpha
        * calibrated_outer_logits
    )

    blended_outer_probability = phase27_sigmoid(
        blended_outer_logits
    )

    PHASE27_ATTENTION_OOF_PRIVATE[
        outer_valid_indices
    ] = standalone_outer_probability

    PHASE27_ATTENTION_BLEND_OOF_PRIVATE[
        outer_valid_indices
    ] = blended_outer_probability

    PHASE27_DEPLOYMENT_STATES_PRIVATE[fold] = {
        "states": refit_states,
        "seeds": [
            int(result["seed"])
            for result in seed_results
        ],
        "epochs": [
            int(result["best_epoch"])
            for result in seed_results
        ],
        "platt_intercept": float(platt_intercept),
        "platt_slope": float(platt_slope),
        "selected_blend_alpha": float(
            selected_alpha
        ),
    }

    phase27_selection_records.append({
        "fold": int(fold),
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(len(monitor_indices)),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "seed_best_epochs": [
            int(result["best_epoch"])
            for result in seed_results
        ],
        "seed_best_monitor_losses": [
            float(result["best_monitor_log_loss"])
            for result in seed_results
        ],
        "baseline_monitor_log_loss": float(
            baseline_monitor_loss
        ),
        "attention_monitor_log_loss": float(
            standalone_monitor_loss
        ),
        "attention_monitor_auroc": float(
            standalone_monitor_auroc
        ),
        "platt_intercept": float(
            platt_intercept
        ),
        "platt_slope": float(platt_slope),
        "raw_best_blend_alpha": float(
            raw_best["alpha"]
        ),
        "raw_best_blend_monitor_log_loss": float(
            raw_best["log_loss"]
        ),
        "raw_monitor_blend_gain": float(
            raw_gain
        ),
        "selected_blend_alpha": float(
            selected_alpha
        ),
        "blend_advanced": bool(
            blend_advanced
        ),
        "refit_final_losses": [
            float(value)
            for value in refit_final_losses
        ],
        "elapsed_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    print(
        f"Phase27 fold {fold} complete: "
        f"alpha={selected_alpha:.3f}, "
        f"advanced={blend_advanced}"
    )

assert np.isfinite(
    PHASE27_ATTENTION_OOF_PRIVATE
).all()

assert np.isfinite(
    PHASE27_ATTENTION_BLEND_OOF_PRIVATE
).all()

training_report = {
    "phase": "phase27_dinov3_structured_attention_training",
    "status": "complete",
    "configuration": PHASE27_CONFIG,
    "folds": phase27_selection_records,
    "elapsed_seconds": round(
        time.perf_counter() - phase27_started,
        2,
    ),
    "dinov3_backbone_updated": False,
    "monitor_labels_used_for_selection": True,
    "outer_train_labels_used_for_final_refit": True,
    "outer_validation_labels_used": False,
    "selection_frozen_before_outer_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE27_TRAINING")
print(json.dumps(training_report, indent=2))
print("END SANITIZED_PHASE27_TRAINING")

Phase27 fold 0 seed 270701 epoch 001: train=0.6617, monitor=0.5911
Phase27 fold 0 seed 270701 epoch 020: train=0.1188, monitor=0.4969
Phase27 fold 0 seed 270701 frozen: best_epoch=4, monitor=0.428103
Phase27 fold 0 seed 270702 epoch 001: train=0.6188, monitor=0.5041
Phase27 fold 0 seed 270702 epoch 020: train=0.1231, monitor=0.5653
Phase27 fold 0 seed 270702 frozen: best_epoch=2, monitor=0.439635
Phase27 fold 0 seed 270703 epoch 001: train=0.6237, monitor=0.5323
Phase27 fold 0 seed 270703 epoch 020: train=0.1178, monitor=0.4758
Phase27 fold 0 seed 270703 frozen: best_epoch=16, monitor=0.413393
Phase27 fold 0 complete: alpha=0.000, advanced=False
Phase27 fold 1 seed 270701 epoch 001: train=0.6556, monitor=0.6016
Phase27 fold 1 seed 270701 epoch 020: train=0.1421, monitor=0.4219
Phase27 fold 1 seed 270701 frozen: best_epoch=10, monitor=0.318225
Phase27 fold 1 seed 270702 epoch 001: train=0.6324, monitor=0.5191
Phase27 fold 1 seed 270702 epoch 020: train=0.1355, monitor=0.4054
Phase27 fol

In [94]:
# Cell 73 — Phase27 one-time outer-validation evaluation

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)


phase27_labels = np.asarray(
    PHASE27_Y_PRIVATE,
    dtype=np.int64,
)

phase27_baseline = np.clip(
    np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

phase27_standalone = np.clip(
    np.asarray(
        PHASE27_ATTENTION_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

phase27_blend = np.clip(
    np.asarray(
        PHASE27_ATTENTION_BLEND_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

phase26_linear_blend = np.clip(
    np.asarray(
        PHASE26_DINOV3_BLEND_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)


def phase27_metrics(labels, probabilities):
    return {
        "log_loss": float(
            log_loss(
                labels,
                probabilities,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                labels,
                probabilities,
            )
        ),
    }


def phase27_calibration(labels, probabilities):
    logits = phase27_logit(probabilities)

    calibration = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )

    calibration.fit(
        logits.reshape(-1, 1),
        labels,
    )

    return {
        "calibration_intercept": float(
            calibration.intercept_[0]
        ),
        "calibration_slope": float(
            calibration.coef_[0, 0]
        ),
        "mean_probability": float(
            np.mean(probabilities)
        ),
    }


baseline_metrics = phase27_metrics(
    phase27_labels,
    phase27_baseline,
)

linear_component_metrics = phase27_metrics(
    phase27_labels,
    phase26_linear_blend,
)

standalone_metrics = phase27_metrics(
    phase27_labels,
    phase27_standalone,
)
standalone_metrics.update(
    phase27_calibration(
        phase27_labels,
        phase27_standalone,
    )
)

blend_metrics = phase27_metrics(
    phase27_labels,
    phase27_blend,
)
blend_metrics.update(
    phase27_calibration(
        phase27_labels,
        phase27_blend,
    )
)

fold_metrics = []
fold_improvements = []

for fold in range(3):
    partition = phase27_partition_by_fold[fold]
    indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    labels = phase27_labels[indices]

    baseline_fold = phase27_metrics(
        labels,
        phase27_baseline[indices],
    )

    standalone_fold = phase27_metrics(
        labels,
        phase27_standalone[indices],
    )

    blend_fold = phase27_metrics(
        labels,
        phase27_blend[indices],
    )

    improvement = (
        baseline_fold["log_loss"]
        - blend_fold["log_loss"]
    )

    fold_improvements.append(improvement)

    fold_metrics.append({
        "fold": int(fold),
        "n": int(len(indices)),
        "selected_blend_alpha": float(
            PHASE27_DEPLOYMENT_STATES_PRIVATE[
                fold
            ]["selected_blend_alpha"]
        ),
        "baseline_log_loss": round(
            baseline_fold["log_loss"],
            6,
        ),
        "attention_log_loss": round(
            standalone_fold["log_loss"],
            6,
        ),
        "blend_log_loss": round(
            blend_fold["log_loss"],
            6,
        ),
        "blend_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_fold["auroc"],
            6,
        ),
        "attention_auroc": round(
            standalone_fold["auroc"],
            6,
        ),
        "blend_auroc": round(
            blend_fold["auroc"],
            6,
        ),
    })


group_column = None

if "acquisition_group" in case_df.columns:
    group_column = "acquisition_group"
else:
    group_candidates = [
        column
        for column in case_df.columns
        if (
            "acquisition" in column.lower()
            and "group" in column.lower()
        )
    ]

    if len(group_candidates) == 1:
        group_column = group_candidates[0]

assert group_column is not None

groups = np.asarray(case_df[group_column])
group_metrics = []
group_harms = []

for group in np.unique(groups):
    indices = np.flatnonzero(groups == group)

    if len(indices) < 30:
        continue

    baseline_group_loss = phase27_metrics(
        phase27_labels[indices],
        phase27_baseline[indices],
    )["log_loss"]

    blend_group_loss = phase27_metrics(
        phase27_labels[indices],
        phase27_blend[indices],
    )["log_loss"]

    change = blend_group_loss - baseline_group_loss
    group_harms.append(change)

    group_metrics.append({
        "group": (
            int(group)
            if np.issubdtype(
                np.asarray(group).dtype,
                np.integer,
            )
            else str(group)
        ),
        "n": int(len(indices)),
        "baseline_log_loss": round(
            baseline_group_loss,
            6,
        ),
        "blend_log_loss": round(
            blend_group_loss,
            6,
        ),
        "log_loss_change": round(
            change,
            6,
        ),
    })


log_loss_gain = (
    baseline_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

auroc_gain = (
    blend_metrics["auroc"]
    - baseline_metrics["auroc"]
)

brier_excess = (
    blend_metrics["brier"]
    - baseline_metrics["brier"]
)

fold_wins = int(
    sum(
        improvement > 0
        for improvement in fold_improvements
    )
)

advanced_folds = int(
    sum(
        record["blend_advanced"]
        for record in phase27_selection_records
    )
)

worst_fold_excess = float(
    max(-value for value in fold_improvements)
)

maximum_major_group_harm = float(
    max(group_harms)
    if group_harms
    else 0.0
)

prediction_correlation = float(
    np.corrcoef(
        phase27_logit(phase27_baseline),
        phase27_logit(phase27_standalone),
    )[0, 1]
)

attention_vs_linear_log_loss_gain = float(
    linear_component_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

component_gate_passed = bool(
    standalone_metrics["auroc"] >= 0.90
    and advanced_folds >= 1
    and log_loss_gain >= 0.001
    and worst_fold_excess <= 0.010
)

promotion_gate_passed = bool(
    log_loss_gain >= 0.003
    and auroc_gain >= 0.002
    and fold_wins >= 2
    and worst_fold_excess <= 0.005
    and brier_excess <= 0.001
    and 0.8 <= blend_metrics[
        "calibration_slope"
    ] <= 1.2
    and maximum_major_group_harm <= 0.015
)

if promotion_gate_passed:
    phase27_status = "promoted"
elif component_gate_passed:
    phase27_status = "stage_c_eligible"
else:
    phase27_status = "attention_probe_terminated"

report = {
    "phase": "phase27_dinov3_structured_attention",
    "status": phase27_status,
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "phase26_linear_component": {
        key: round(value, 6)
        for key, value
        in linear_component_metrics.items()
    },
    "attention_standalone": {
        key: round(value, 6)
        for key, value in standalone_metrics.items()
    },
    "phase12c_attention_blend": {
        key: round(value, 6)
        for key, value in blend_metrics.items()
    },
    "improvements": {
        "blend_log_loss_gain": round(
            log_loss_gain,
            6,
        ),
        "blend_auroc_gain": round(
            auroc_gain,
            6,
        ),
        "blend_brier_excess": round(
            brier_excess,
            6,
        ),
        "attention_vs_phase26_linear_log_loss_gain": round(
            attention_vs_linear_log_loss_gain,
            6,
        ),
        "blend_fold_wins": fold_wins,
        "advanced_monitor_fold_count": advanced_folds,
        "worst_blend_fold_excess": round(
            worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            maximum_major_group_harm,
            6,
        ),
        "baseline_attention_logit_correlation": round(
            prediction_correlation,
            6,
        ),
    },
    "fold_metrics": fold_metrics,
    "major_acquisition_group_metrics": group_metrics,
    "gates": {
        "component_gate_passed": component_gate_passed,
        "promotion_gate_passed": promotion_gate_passed,
        "component_thresholds": {
            "minimum_attention_auroc": 0.90,
            "minimum_advanced_monitor_folds": 1,
            "minimum_log_loss_gain": 0.001,
            "maximum_worst_fold_excess": 0.010,
        },
        "promotion_thresholds": {
            "minimum_log_loss_gain": 0.003,
            "minimum_auroc_gain": 0.002,
            "minimum_fold_wins": 2,
            "maximum_worst_fold_excess": 0.005,
            "maximum_brier_excess": 0.001,
            "calibration_slope_range": [0.8, 1.2],
            "maximum_major_group_harm": 0.015,
        },
    },
    "dinov3_backbone_updated": False,
    "external_weights": True,
    "license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "selection_frozen_before_outer_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE27_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE27_OOF")

BEGIN SANITIZED_PHASE27_OOF
{
  "phase": "phase27_dinov3_structured_attention",
  "status": "attention_probe_terminated",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "phase26_linear_component": {
    "log_loss": 0.304742,
    "auroc": 0.939583,
    "brier": 0.094448
  },
  "attention_standalone": {
    "log_loss": 0.399619,
    "auroc": 0.901882,
    "brier": 0.12522,
    "calibration_intercept": 0.001138,
    "calibration_slope": 0.845948,
    "mean_probability": 0.549585
  },
  "phase12c_attention_blend": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535,
    "calibration_intercept": 0.167533,
    "calibration_slope": 0.935548,
    "mean_probability": 0.530697
  },
  "improvements": {
    "blend_log_loss_gain": 0.0,
    "blend_auroc_gain": 0.0,
    "blend_brier_excess": 0.0,
    "attention_vs_phase26_linear_log_loss_gain": -0.002873,
    "blend_fold_wins": 0,
    "advanced_monitor_fold_count": 0,
    "worst_blen

In [95]:
# Cell 74 — Phase28 confidence-gated DINOv3 correction contract

import json
import numpy as np


PHASE28_CONFIG = {
    "phase": "phase28_uncertainty_gated_dinov3",
    "representation": "ordered_class_tokens",
    "pca_dimension": 64,
    "logistic_c": 0.1,
    "pca_iterated_power": 3,
    "base_seed": 280601,
    "candidate_alphas": [
        0.0,
        0.05,
        0.10,
        0.15,
        0.20,
        0.30,
        0.40,
    ],
    "candidate_uncertainty_exponents": [
        0.0,
        0.5,
        1.0,
        2.0,
    ],
    "minimum_monitor_gain": 0.002,
    "probability_clip": 1e-5,
}


def phase28_sigmoid(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = np.clip(logits, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-logits))


def phase28_logit(probabilities):
    epsilon = PHASE28_CONFIG["probability_clip"]

    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        epsilon,
        1.0 - epsilon,
    )

    return np.log(
        probabilities / (1.0 - probabilities)
    )


def phase28_uncertainty(probabilities, exponent):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )

    # Maximum one at p=0.5 and approaches zero near 0 or 1.
    uncertainty = (
        4.0 * probabilities * (1.0 - probabilities)
    )

    if float(exponent) == 0.0:
        return np.ones_like(uncertainty)

    return uncertainty ** float(exponent)


def phase28_gated_blend(
    baseline_probability,
    dinov3_probability,
    alpha,
    uncertainty_exponent,
):
    baseline_probability = np.asarray(
        baseline_probability,
        dtype=np.float64,
    )

    dinov3_probability = np.asarray(
        dinov3_probability,
        dtype=np.float64,
    )

    effective_alpha = (
        float(alpha)
        * phase28_uncertainty(
            baseline_probability,
            uncertainty_exponent,
        )
    )

    blended_logit = (
        phase28_logit(baseline_probability)
        + effective_alpha
        * (
            phase28_logit(dinov3_probability)
            - phase28_logit(baseline_probability)
        )
    )

    return (
        phase28_sigmoid(blended_logit),
        effective_alpha,
    )


synthetic_baseline = np.asarray(
    [0.01, 0.10, 0.30, 0.50, 0.70, 0.90, 0.99],
    dtype=np.float64,
)

synthetic_dinov3 = np.full_like(
    synthetic_baseline,
    0.60,
)

synthetic_blend, synthetic_effective_alpha = (
    phase28_gated_blend(
        synthetic_baseline,
        synthetic_dinov3,
        alpha=0.30,
        uncertainty_exponent=1.0,
    )
)

assert np.isfinite(synthetic_blend).all()
assert synthetic_effective_alpha[3] == np.max(
    synthetic_effective_alpha
)
assert synthetic_effective_alpha[0] < (
    synthetic_effective_alpha[2]
)
assert synthetic_effective_alpha[-1] < (
    synthetic_effective_alpha[-3]
)

report = {
    "phase": PHASE28_CONFIG["phase"],
    "status": "accepted",
    "formula": (
        "z_final = z_phase12c + "
        "alpha * (4*p_phase12c*(1-p_phase12c))^gamma "
        "* (z_dinov3-z_phase12c)"
    ),
    "candidate_count": int(
        len(PHASE28_CONFIG["candidate_alphas"])
        * len(
            PHASE28_CONFIG[
                "candidate_uncertainty_exponents"
            ]
        )
    ),
    "maximum_effective_alpha": round(
        float(np.max(synthetic_effective_alpha)),
        6,
    ),
    "effective_alpha_at_probability_0p01": round(
        float(synthetic_effective_alpha[0]),
        6,
    ),
    "effective_alpha_at_probability_0p50": round(
        float(synthetic_effective_alpha[3]),
        6,
    ),
    "constant_blend_nested_as_gamma_zero": True,
    "case_independent_formula": True,
    "uses_test_distribution": False,
    "labels_used": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE28_GATE_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE28_GATE_CONTRACT")

BEGIN SANITIZED_PHASE28_GATE_CONTRACT
{
  "phase": "phase28_uncertainty_gated_dinov3",
  "status": "accepted",
  "formula": "z_final = z_phase12c + alpha * (4*p_phase12c*(1-p_phase12c))^gamma * (z_dinov3-z_phase12c)",
  "candidate_count": 28,
  "maximum_effective_alpha": 0.3,
  "effective_alpha_at_probability_0p01": 0.01188,
  "effective_alpha_at_probability_0p50": 0.3,
  "constant_blend_nested_as_gamma_zero": true,
  "case_independent_formula": true,
  "uses_test_distribution": false,
  "labels_used": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE28_GATE_CONTRACT


In [96]:
# Cell 75 — Fit-only selection, monitor-only gate choice,
# and outer-training refit

import time

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler


phase28_features = np.asarray(
    PHASE26_REPRESENTATIONS_PRIVATE[
        "ordered_class_tokens"
    ],
    dtype=np.float32,
)

phase28_labels = np.asarray(
    PHASE19_Y_PRIVATE,
    dtype=np.int64,
).reshape(-1)

assert phase28_features.shape == (1362, 3456)
assert phase28_labels.shape == (1362,)
assert np.isfinite(phase28_features).all()


def phase28_log_loss(labels, probabilities):
    return float(
        log_loss(
            labels,
            np.clip(probabilities, 1e-5, 1.0 - 1e-5),
            labels=[0, 1],
        )
    )


def phase28_fit_linear_probe(
    train_indices,
    prediction_indices,
    seed,
):
    train_indices = np.asarray(
        train_indices,
        dtype=np.int64,
    )
    prediction_indices = np.asarray(
        prediction_indices,
        dtype=np.int64,
    )

    scaler = StandardScaler(
        with_mean=True,
        with_std=True,
    )

    train_scaled = scaler.fit_transform(
        phase28_features[train_indices]
    )

    prediction_scaled = scaler.transform(
        phase28_features[prediction_indices]
    )

    pca = PCA(
        n_components=PHASE28_CONFIG["pca_dimension"],
        svd_solver="randomized",
        iterated_power=PHASE28_CONFIG[
            "pca_iterated_power"
        ],
        random_state=int(seed),
        whiten=False,
    )

    train_reduced = pca.fit_transform(train_scaled)
    prediction_reduced = pca.transform(
        prediction_scaled
    )

    classifier = LogisticRegression(
        C=PHASE28_CONFIG["logistic_c"],
        solver="lbfgs",
        max_iter=5000,
        random_state=int(seed),
    )

    classifier.fit(
        train_reduced,
        phase28_labels[train_indices],
    )

    probabilities = classifier.predict_proba(
        prediction_reduced
    )[:, 1]

    state = {
        "scaler_mean": scaler.mean_.astype(
            np.float32
        ),
        "scaler_scale": scaler.scale_.astype(
            np.float32
        ),
        "pca_mean": pca.mean_.astype(np.float32),
        "pca_components": pca.components_.astype(
            np.float32
        ),
        "classifier_coefficient": (
            classifier.coef_.astype(np.float32)
        ),
        "classifier_intercept": (
            classifier.intercept_.astype(np.float32)
        ),
    }

    return probabilities.astype(np.float64), state


phase28_started = time.perf_counter()

PHASE28_DINOV3_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE28_GATED_BLEND_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE28_DEPLOYMENT_STATES_PRIVATE = {}
phase28_selection_records = []

partition_by_fold = {
    int(partition["fold"]): partition
    for partition in PHASE19_PARTITIONS
}

for fold in range(3):
    fold_started = time.perf_counter()
    partition = partition_by_fold[fold]

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    monitor_labels = phase28_labels[monitor_indices]

    baseline_monitor_probability = (
        phase27_fold_array(
            PHASE24_BASE_MONITOR_PRIVATE,
            fold,
        )
    )

    assert len(baseline_monitor_probability) == len(
        monitor_indices
    )

    fit_monitor_dinov3, _ = (
        phase28_fit_linear_probe(
            train_indices=fit_indices,
            prediction_indices=monitor_indices,
            seed=PHASE28_CONFIG["base_seed"] + fold,
        )
    )

    baseline_monitor_loss = phase28_log_loss(
        monitor_labels,
        baseline_monitor_probability,
    )

    dinov3_monitor_loss = phase28_log_loss(
        monitor_labels,
        fit_monitor_dinov3,
    )

    dinov3_monitor_auroc = float(
        roc_auc_score(
            monitor_labels,
            fit_monitor_dinov3,
        )
    )

    candidates = []

    for exponent in PHASE28_CONFIG[
        "candidate_uncertainty_exponents"
    ]:
        for alpha in PHASE28_CONFIG[
            "candidate_alphas"
        ]:
            probability, effective_alpha = (
                phase28_gated_blend(
                    baseline_probability=(
                        baseline_monitor_probability
                    ),
                    dinov3_probability=(
                        fit_monitor_dinov3
                    ),
                    alpha=alpha,
                    uncertainty_exponent=exponent,
                )
            )

            candidate_loss = phase28_log_loss(
                monitor_labels,
                probability,
            )

            candidates.append({
                "alpha": float(alpha),
                "uncertainty_exponent": float(
                    exponent
                ),
                "monitor_log_loss": float(
                    candidate_loss
                ),
                "mean_effective_alpha": float(
                    np.mean(effective_alpha)
                ),
                "maximum_effective_alpha": float(
                    np.max(effective_alpha)
                ),
            })

    # Conservative tie-breaking:
    # smaller nominal alpha, then stronger gating.
    raw_best = min(
        candidates,
        key=lambda candidate: (
            candidate["monitor_log_loss"],
            candidate["alpha"],
            -candidate["uncertainty_exponent"],
        ),
    )

    raw_gain = (
        baseline_monitor_loss
        - raw_best["monitor_log_loss"]
    )

    if (
        raw_gain
        >= PHASE28_CONFIG["minimum_monitor_gain"]
        and raw_best["alpha"] > 0.0
    ):
        selected_alpha = raw_best["alpha"]
        selected_exponent = raw_best[
            "uncertainty_exponent"
        ]
        correction_advanced = True
    else:
        selected_alpha = 0.0
        selected_exponent = 0.0
        correction_advanced = False

    outer_dinov3_probability, deployment_state = (
        phase28_fit_linear_probe(
            train_indices=outer_train_indices,
            prediction_indices=outer_valid_indices,
            seed=PHASE28_CONFIG["base_seed"] + fold,
        )
    )

    baseline_outer_probability = np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    )[outer_valid_indices]

    gated_outer_probability, outer_effective_alpha = (
        phase28_gated_blend(
            baseline_probability=(
                baseline_outer_probability
            ),
            dinov3_probability=(
                outer_dinov3_probability
            ),
            alpha=selected_alpha,
            uncertainty_exponent=selected_exponent,
        )
    )

    PHASE28_DINOV3_OOF_PRIVATE[
        outer_valid_indices
    ] = outer_dinov3_probability

    PHASE28_GATED_BLEND_OOF_PRIVATE[
        outer_valid_indices
    ] = gated_outer_probability

    PHASE28_DEPLOYMENT_STATES_PRIVATE[fold] = {
        "linear_probe": deployment_state,
        "selected_alpha": float(selected_alpha),
        "selected_uncertainty_exponent": float(
            selected_exponent
        ),
    }

    phase28_selection_records.append({
        "fold": int(fold),
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(len(monitor_indices)),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "baseline_monitor_log_loss": float(
            baseline_monitor_loss
        ),
        "dinov3_monitor_log_loss": float(
            dinov3_monitor_loss
        ),
        "dinov3_monitor_auroc": float(
            dinov3_monitor_auroc
        ),
        "raw_best_alpha": float(
            raw_best["alpha"]
        ),
        "raw_best_uncertainty_exponent": float(
            raw_best["uncertainty_exponent"]
        ),
        "raw_best_monitor_log_loss": float(
            raw_best["monitor_log_loss"]
        ),
        "raw_monitor_gain": float(raw_gain),
        "selected_alpha": float(
            selected_alpha
        ),
        "selected_uncertainty_exponent": float(
            selected_exponent
        ),
        "correction_advanced": bool(
            correction_advanced
        ),
        "mean_outer_effective_alpha": float(
            np.mean(outer_effective_alpha)
        ),
        "q95_outer_effective_alpha": float(
            np.quantile(
                outer_effective_alpha,
                0.95,
            )
        ),
        "elapsed_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    print(
        f"Phase28 fold {fold}: "
        f"alpha={selected_alpha:.3f}, "
        f"gamma={selected_exponent:.2f}, "
        f"advanced={correction_advanced}"
    )

assert np.isfinite(
    PHASE28_DINOV3_OOF_PRIVATE
).all()

assert np.isfinite(
    PHASE28_GATED_BLEND_OOF_PRIVATE
).all()

report = {
    "phase": "phase28_nested_uncertainty_gate",
    "status": "complete",
    "configuration": PHASE28_CONFIG,
    "folds": phase28_selection_records,
    "elapsed_seconds": round(
        time.perf_counter() - phase28_started,
        2,
    ),
    "selection_partition": "monitor_only",
    "outer_training_refit": True,
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used": False,
    "dinov3_backbone_updated": False,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE28_SELECTION")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE28_SELECTION")

Phase28 fold 0: alpha=0.400, gamma=2.00, advanced=True
Phase28 fold 1: alpha=0.000, gamma=0.00, advanced=False
Phase28 fold 2: alpha=0.000, gamma=0.00, advanced=False
BEGIN SANITIZED_PHASE28_SELECTION
{
  "phase": "phase28_nested_uncertainty_gate",
  "status": "complete",
  "configuration": {
    "phase": "phase28_uncertainty_gated_dinov3",
    "representation": "ordered_class_tokens",
    "pca_dimension": 64,
    "logistic_c": 0.1,
    "pca_iterated_power": 3,
    "base_seed": 280601,
    "candidate_alphas": [
      0.0,
      0.05,
      0.1,
      0.15,
      0.2,
      0.3,
      0.4
    ],
    "candidate_uncertainty_exponents": [
      0.0,
      0.5,
      1.0,
      2.0
    ],
    "minimum_monitor_gain": 0.002,
    "probability_clip": 1e-05
  },
  "folds": [
    {
      "fold": 0,
      "fit_n": 721,
      "monitor_n": 174,
      "outer_train_n": 895,
      "outer_valid_n": 467,
      "baseline_monitor_log_loss": 0.27846466612927395,
      "dinov3_monitor_log_loss": 0.4421923973

In [97]:
# Cell 76 — Phase28 outer-validation evaluation

from sklearn.metrics import brier_score_loss


phase28_baseline = np.clip(
    np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)

phase28_dinov3 = np.clip(
    PHASE28_DINOV3_OOF_PRIVATE,
    1e-5,
    1.0 - 1e-5,
)

phase28_blend = np.clip(
    PHASE28_GATED_BLEND_OOF_PRIVATE,
    1e-5,
    1.0 - 1e-5,
)

phase28_phase26 = np.clip(
    np.asarray(
        PHASE26_DINOV3_BLEND_OOF_PRIVATE,
        dtype=np.float64,
    ),
    1e-5,
    1.0 - 1e-5,
)


def phase28_metrics(labels, probabilities):
    return {
        "log_loss": float(
            log_loss(
                labels,
                probabilities,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(labels, probabilities)
        ),
        "brier": float(
            brier_score_loss(labels, probabilities)
        ),
    }


baseline_metrics = phase28_metrics(
    phase28_labels,
    phase28_baseline,
)

phase26_metrics = phase28_metrics(
    phase28_labels,
    phase28_phase26,
)

dinov3_metrics = phase28_metrics(
    phase28_labels,
    phase28_dinov3,
)

blend_metrics = phase28_metrics(
    phase28_labels,
    phase28_blend,
)

fold_reports = []
fold_improvements = []

for fold in range(3):
    indices = np.asarray(
        partition_by_fold[fold]["outer_valid"],
        dtype=np.int64,
    )

    labels = phase28_labels[indices]

    baseline_fold = phase28_metrics(
        labels,
        phase28_baseline[indices],
    )

    dinov3_fold = phase28_metrics(
        labels,
        phase28_dinov3[indices],
    )

    blend_fold = phase28_metrics(
        labels,
        phase28_blend[indices],
    )

    improvement = (
        baseline_fold["log_loss"]
        - blend_fold["log_loss"]
    )

    fold_improvements.append(improvement)

    selection = phase28_selection_records[fold]

    fold_reports.append({
        "fold": int(fold),
        "n": int(len(indices)),
        "selected_alpha": float(
            selection["selected_alpha"]
        ),
        "selected_uncertainty_exponent": float(
            selection[
                "selected_uncertainty_exponent"
            ]
        ),
        "baseline_log_loss": round(
            baseline_fold["log_loss"],
            6,
        ),
        "dinov3_log_loss": round(
            dinov3_fold["log_loss"],
            6,
        ),
        "blend_log_loss": round(
            blend_fold["log_loss"],
            6,
        ),
        "blend_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_fold["auroc"],
            6,
        ),
        "dinov3_auroc": round(
            dinov3_fold["auroc"],
            6,
        ),
        "blend_auroc": round(
            blend_fold["auroc"],
            6,
        ),
    })


groups = np.asarray(case_df[group_column])
group_reports = []
group_harms = []

for group in np.unique(groups):
    indices = np.flatnonzero(groups == group)

    if len(indices) < 30:
        continue

    baseline_loss = phase28_metrics(
        phase28_labels[indices],
        phase28_baseline[indices],
    )["log_loss"]

    blend_loss = phase28_metrics(
        phase28_labels[indices],
        phase28_blend[indices],
    )["log_loss"]

    change = blend_loss - baseline_loss
    group_harms.append(change)

    group_reports.append({
        "group": (
            int(group)
            if np.issubdtype(
                np.asarray(group).dtype,
                np.integer,
            )
            else str(group)
        ),
        "n": int(len(indices)),
        "baseline_log_loss": round(
            baseline_loss,
            6,
        ),
        "blend_log_loss": round(
            blend_loss,
            6,
        ),
        "log_loss_change": round(
            change,
            6,
        ),
    })


log_loss_gain = (
    baseline_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

auroc_gain = (
    blend_metrics["auroc"]
    - baseline_metrics["auroc"]
)

brier_excess = (
    blend_metrics["brier"]
    - baseline_metrics["brier"]
)

fold_wins = int(
    sum(value > 0 for value in fold_improvements)
)

worst_fold_excess = float(
    max(-value for value in fold_improvements)
)

maximum_group_harm = float(
    max(group_harms) if group_harms else 0.0
)

phase28_gain_over_phase26 = (
    phase26_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

promotion_gate_passed = bool(
    log_loss_gain >= 0.003
    and auroc_gain >= 0.001
    and fold_wins >= 1
    and worst_fold_excess <= 0.005
    and brier_excess <= 0.001
    and maximum_group_harm <= 0.015
)

report = {
    "phase": "phase28_uncertainty_gated_dinov3",
    "status": (
        "promoted"
        if promotion_gate_passed
        else "not_promoted"
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "phase26_constant_blend": {
        key: round(value, 6)
        for key, value in phase26_metrics.items()
    },
    "dinov3_standalone": {
        key: round(value, 6)
        for key, value in dinov3_metrics.items()
    },
    "phase28_gated_blend": {
        key: round(value, 6)
        for key, value in blend_metrics.items()
    },
    "improvements": {
        "log_loss_gain_over_phase12c": round(
            log_loss_gain,
            6,
        ),
        "log_loss_gain_over_phase26": round(
            phase28_gain_over_phase26,
            6,
        ),
        "auroc_gain_over_phase12c": round(
            auroc_gain,
            6,
        ),
        "brier_excess": round(
            brier_excess,
            6,
        ),
        "fold_wins": fold_wins,
        "worst_fold_excess": round(
            worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            maximum_group_harm,
            6,
        ),
    },
    "fold_metrics": fold_reports,
    "major_acquisition_group_metrics": group_reports,
    "promotion_gate_passed": promotion_gate_passed,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "selection_frozen_before_outer_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE28_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE28_OOF")

BEGIN SANITIZED_PHASE28_OOF
{
  "phase": "phase28_uncertainty_gated_dinov3",
  "status": "not_promoted",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "phase26_constant_blend": {
    "log_loss": 0.304742,
    "auroc": 0.939583,
    "brier": 0.094448
  },
  "dinov3_standalone": {
    "log_loss": 0.403015,
    "auroc": 0.911296,
    "brier": 0.12401
  },
  "phase28_gated_blend": {
    "log_loss": 0.304977,
    "auroc": 0.940018,
    "brier": 0.094683
  },
  "improvements": {
    "log_loss_gain_over_phase12c": 0.002639,
    "log_loss_gain_over_phase26": -0.000234,
    "auroc_gain_over_phase12c": 0.001091,
    "brier_excess": -0.000852,
    "fold_wins": 1,
    "worst_fold_excess": -0.0,
    "maximum_major_group_harm": 0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "selected_alpha": 0.4,
      "selected_uncertainty_exponent": 2.0,
      "baseline_log_loss": 0.358494,
      "dinov3_log_loss": 0.456806,
      "blend_lo

In [98]:
# Cell 77 — Sanitized audit for exact Phase26 reuse.
# Does not print parameters, predictions, features, labels,
# hashes, patient identifiers, or source code.

import inspect
import json
import types

import numpy as np
import torch
import torch.nn as nn


def phase29_shape_schema(value):
    if isinstance(value, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }

    if torch.is_tensor(value):
        return {
            "type": "tensor",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "device": str(value.device),
        }

    if isinstance(value, nn.Module):
        return {
            "type": "torch_module",
            "class": value.__class__.__name__,
            "parameter_count": int(
                sum(
                    parameter.numel()
                    for parameter in value.parameters()
                )
            ),
            "trainable_parameter_count": int(
                sum(
                    parameter.numel()
                    for parameter in value.parameters()
                    if parameter.requires_grad
                )
            ),
        }

    module_name = value.__class__.__module__
    class_name = value.__class__.__name__

    if module_name.startswith("sklearn"):
        schema = {
            "type": "sklearn_object",
            "module": module_name,
            "class": class_name,
        }

        attribute_shapes = {}

        for attribute in [
            "mean_",
            "scale_",
            "var_",
            "components_",
            "explained_variance_",
            "singular_values_",
            "coef_",
            "intercept_",
            "classes_",
        ]:
            if hasattr(value, attribute):
                attribute_value = getattr(
                    value,
                    attribute,
                )

                if hasattr(attribute_value, "shape"):
                    attribute_shapes[attribute] = list(
                        attribute_value.shape
                    )
                else:
                    attribute_shapes[attribute] = (
                        type(attribute_value).__name__
                    )

        schema["fitted_attribute_shapes"] = (
            attribute_shapes
        )

        for attribute in [
            "n_components",
            "svd_solver",
            "iterated_power",
            "C",
            "solver",
            "penalty",
            "max_iter",
            "with_mean",
            "with_std",
            "whiten",
        ]:
            if hasattr(value, attribute):
                attribute_value = getattr(
                    value,
                    attribute,
                )

                if isinstance(
                    attribute_value,
                    (
                        str,
                        int,
                        float,
                        bool,
                        type(None),
                    ),
                ):
                    schema[attribute] = attribute_value

        return schema

    return {
        "type": type(value).__name__,
        "module": module_name,
        "class": class_name,
    }


def phase29_container_schema(
    value,
    depth=0,
    maximum_depth=4,
):
    if depth >= maximum_depth:
        return phase29_shape_schema(value)

    if isinstance(value, dict):
        keys = list(value.keys())

        schema = {
            "type": "dict",
            "length": len(keys),
            "keys": [
                str(key)
                for key in keys[:40]
            ],
            "truncated": len(keys) > 40,
        }

        # Expand only small structural dictionaries.
        if len(keys) <= 25:
            schema["fields"] = {
                str(key): phase29_container_schema(
                    value[key],
                    depth=depth + 1,
                    maximum_depth=maximum_depth,
                )
                for key in keys
            }

        return schema

    if isinstance(value, (list, tuple)):
        schema = {
            "type": type(value).__name__,
            "length": len(value),
        }

        if len(value) <= 10:
            schema["elements"] = [
                phase29_container_schema(
                    element,
                    depth=depth + 1,
                    maximum_depth=maximum_depth,
                )
                for element in value
            ]
        elif len(value) > 0:
            schema["first_element"] = (
                phase29_container_schema(
                    value[0],
                    depth=depth + 1,
                    maximum_depth=maximum_depth,
                )
            )

        return schema

    return phase29_shape_schema(value)


def phase29_signature(callable_object):
    try:
        return str(
            inspect.signature(callable_object)
        )
    except Exception:
        return "signature_unavailable"


# ---------------------------------------------------------
# 1. Find exact DINOv3 backbone candidates.
# ---------------------------------------------------------

backbone_candidates = []

for name, value in list(globals().items()):
    if not isinstance(value, nn.Module):
        continue

    parameter_count = int(
        sum(
            parameter.numel()
            for parameter in value.parameters()
        )
    )

    candidate = value.module if (
        isinstance(value, nn.DataParallel)
    ) else value

    if (
        21_000_000 <= parameter_count <= 23_000_000
        and hasattr(candidate, "forward_features")
    ):
        backbone_candidates.append({
            "name": name,
            "class": value.__class__.__name__,
            "wrapped_class": (
                candidate.__class__.__name__
            ),
            "parameter_count": parameter_count,
            "has_forward_features": True,
            "has_blocks": hasattr(
                candidate,
                "blocks",
            ),
            "has_norm": hasattr(
                candidate,
                "norm",
            ),
        })


# ---------------------------------------------------------
# 2. Find Phase26 helper callables.
# ---------------------------------------------------------

callable_contracts = []

for name, value in list(globals().items()):
    lowered = name.lower()

    if not callable(value):
        continue

    if (
        "phase26" in lowered
        or "dinov3" in lowered
        or name in [
            "phase26_make_views",
            "phase26_project_slab",
        ]
    ):
        callable_contracts.append({
            "name": name,
            "type": type(value).__name__,
            "signature": phase29_signature(value),
        })

callable_contracts = sorted(
    callable_contracts,
    key=lambda record: record["name"],
)


# ---------------------------------------------------------
# 3. Inspect private deployment-state structure.
# ---------------------------------------------------------

global_contracts = {}

for name in [
    "PHASE26_DEPLOYMENT_STATES_PRIVATE",
    "PHASE26_SELECTED_CONFIG_PRIVATE",
    "PHASE26_DINOV3_FEATURE_CACHE_PRIVATE",
    "PHASE26_REPRESENTATIONS_PRIVATE",
    "PHASE26_DINOV3_OOF_PRIVATE",
    "PHASE26_DINOV3_BLEND_OOF_PRIVATE",
    "phase26_selection_records",
    "PHASE19_HIGHRES_CACHE",
]:
    global_contracts[name] = {
        "present": name in globals()
    }

    if name in globals():
        global_contracts[name]["schema"] = (
            phase29_container_schema(
                globals()[name]
            )
        )


# ---------------------------------------------------------
# 4. Identify the exact view constructor without reading
#    any challenge case.
# ---------------------------------------------------------

view_function_candidates = []

for name, value in list(globals().items()):
    if not callable(value):
        continue

    lowered = name.lower()

    if (
        "make_views" in lowered
        or "multiplanar" in lowered
        or "project_slab" in lowered
    ):
        view_function_candidates.append({
            "name": name,
            "signature": phase29_signature(value),
        })


# ---------------------------------------------------------
# 5. Inspect transformer block naming only.
# ---------------------------------------------------------

block_contracts = []

for candidate_record in backbone_candidates:
    value = globals()[candidate_record["name"]]

    module = (
        value.module
        if isinstance(value, nn.DataParallel)
        else value
    )

    block_parameter_prefixes = sorted({
        name.split(".")[0]
        + "."
        + name.split(".")[1]
        for name, _ in module.named_parameters()
        if (
            name.startswith("blocks.")
            and len(name.split(".")) >= 3
        )
    })

    block_contracts.append({
        "backbone_name": candidate_record["name"],
        "block_parameter_prefix_count": len(
            block_parameter_prefixes
        ),
        "block_parameter_prefixes": (
            block_parameter_prefixes
        ),
        "normalization_parameter_names": [
            name
            for name, _ in module.named_parameters()
            if (
                name.startswith("norm.")
                or ".norm." in name
            )
        ][:40],
    })


report = {
    "phase": "phase29_dinov3_domain_adaptation_preflight",
    "status": (
        "accepted"
        if (
            len(backbone_candidates) >= 1
            and len(view_function_candidates) >= 1
        )
        else "incomplete"
    ),
    "backbone_candidates": backbone_candidates,
    "view_function_candidates": (
        view_function_candidates
    ),
    "phase26_callable_contracts": callable_contracts,
    "global_contracts": global_contracts,
    "transformer_block_contracts": block_contracts,
    "proposed_adaptation": {
        "training_scope": (
            "fold_local_unlabeled_training_images_only"
        ),
        "teacher": (
            "original_frozen_DINOv3_descriptors"
        ),
        "student": (
            "last_transformer_block_plus_final_norm"
        ),
        "objective": (
            "weak_to_strong feature distillation "
            "with L2-SP anchoring"
        ),
        "outer_validation_images_used_for_adaptation": False,
        "outer_validation_labels_used": False,
        "test_images_used": False,
    },
    "values_or_parameters_exported": False,
    "case_level_features_exported": False,
    "labels_exported": False,
    "uids_displayed": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE29_PREFLIGHT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE29_PREFLIGHT")

BEGIN SANITIZED_PHASE29_PREFLIGHT
{
  "phase": "phase29_dinov3_domain_adaptation_preflight",
  "status": "accepted",
  "backbone_candidates": [
    {
      "name": "PHASE26_DINO_PRIVATE",
      "class": "DinoVisionTransformer",
      "wrapped_class": "DinoVisionTransformer",
      "parameter_count": 22056576,
      "has_forward_features": true,
      "has_blocks": true,
      "has_norm": true
    },
    {
      "name": "PHASE26_DINOV3_PRIVATE",
      "class": "DinoVisionTransformer",
      "wrapped_class": "DinoVisionTransformer",
      "parameter_count": 21601152,
      "has_forward_features": true,
      "has_blocks": true,
      "has_norm": true
    }
  ],
  "view_function_candidates": [
    {
      "name": "phase26_project_slab",
      "signature": "(volume, axis, start, stop)"
    },
    {
      "name": "phase26_make_views",
      "signature": "(volume)"
    }
  ],
  "phase26_callable_contracts": [
    {
      "name": "PHASE26_DINOV3_PRIVATE",
      "type": "DinoVisionTransformer"

In [101]:
# Cell 78 — Phase29 DINOv3 label-free adaptation contract

import copy
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


PHASE29_CONFIG = {
    "phase": "phase29_fold_local_dinov3_adaptation",
    "candidate_epochs": [0, 1, 2, 4],
    "batch_size_cases": 4,
    "learning_rate": 5e-6,
    "weight_decay": 0.05,
    "l2sp_weight": 0.05,
    "patch_distillation_weight": 0.50,
    "gain_standard_deviation": 0.04,
    "bias_standard_deviation": 0.015,
    "noise_standard_deviation": 0.01,
    "gradient_clip": 1.0,
    "pca_dimension": 64,
    "pca_whiten": True,
    "pca_iterated_power": 3,
    "logistic_c": 0.1,
    "probe_seed": 260601,
    "adaptation_seed": 290601,
    "blend_alphas": [
        0.0, 0.05, 0.10,
        0.15, 0.20, 0.30,
    ],
    "minimum_monitor_blend_gain": 0.002,
    "minimum_adaptation_gain_over_epoch0": 0.001,
}


phase29_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

phase29_highres_cache = np.asarray(
    PHASE19_HIGHRES_CACHE
)

phase29_teacher_cache = np.asarray(
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE
)

assert phase29_highres_cache.shape == (
    1362, 80, 80, 80
)
assert phase29_teacher_cache.shape == (
    1362, 9, 5, 384
)


def phase29_seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def phase29_build_student():
    student = copy.deepcopy(
        PHASE26_DINOV3_PRIVATE
    ).to(phase29_device)

    for parameter in student.parameters():
        parameter.requires_grad_(False)

    for name, parameter in student.named_parameters():
        if (
            name.startswith("blocks.11.")
            or name.startswith("norm.")
        ):
            parameter.requires_grad_(True)

    # Evaluation mode disables stochastic depth/dropout.
    # Gradients still update trainable parameters.
    student.eval()

    trainable_parameters = [
        parameter
        for parameter in student.parameters()
        if parameter.requires_grad
    ]

    assert len(trainable_parameters) > 0

    return student


def phase29_forward_descriptors(model, views):
    result = model.forward_features(views)

    assert "x_norm_clstoken" in result
    assert "x_norm_patchtokens" in result

    class_token = result["x_norm_clstoken"]
    global_patch_mean = result[
        "x_norm_patchtokens"
    ].mean(dim=1)

    return class_token, global_patch_mean


def phase29_make_case_views(indices):
    indices = np.asarray(indices, dtype=np.int64)
    
    volumes = torch.from_numpy(
        np.asarray(
            phase29_highres_cache[indices],
            dtype=np.float32,
        )
    ).to(phase29_device)

    assert volumes.ndim == 4
    assert tuple(volumes.shape[1:]) == (
        80, 80, 80
    )

    with torch.no_grad():
        views = phase26_make_views(volumes)

    assert views.shape == (
        len(indices), 9, 3, 224, 224
    ), {
        "observed_shape": list(views.shape),
        "expected_shape": [
            len(indices), 9, 3, 224, 224
        ],
    }

    return views.reshape(
        len(indices) * 9,
        3,
        224,
        224,
    )

def phase29_teacher_targets(indices):
    indices = np.asarray(indices, dtype=np.int64)

    class_target = torch.from_numpy(
        np.asarray(
            phase29_teacher_cache[
                indices, :, 0, :
            ],
            dtype=np.float32,
        )
    ).reshape(-1, 384).to(phase29_device)

    patch_target = torch.from_numpy(
        np.asarray(
            phase29_teacher_cache[
                indices, :, 1, :
            ],
            dtype=np.float32,
        )
    ).reshape(-1, 384).to(phase29_device)

    return class_target, patch_target


def phase29_augment_views(views, seed):
    generator = torch.Generator(
        device=phase29_device
    )
    generator.manual_seed(int(seed))

    count = views.shape[0]

    gain = (
        1.0
        + PHASE29_CONFIG[
            "gain_standard_deviation"
        ]
        * torch.randn(
            count,
            3,
            1,
            1,
            generator=generator,
            device=phase29_device,
        )
    )

    bias = (
        PHASE29_CONFIG[
            "bias_standard_deviation"
        ]
        * torch.randn(
            count,
            3,
            1,
            1,
            generator=generator,
            device=phase29_device,
        )
    )

    noise = (
        PHASE29_CONFIG[
            "noise_standard_deviation"
        ]
        * torch.randn(
            views.shape,
            generator=generator,
            device=phase29_device,
            dtype=views.dtype,
        )
    )

    return views * gain + bias + noise


def phase29_trainable_anchor(model):
    return {
        name: parameter.detach().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }


def phase29_trainable_state(model):
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }


def phase29_load_trainable_state(model, state):
    named_parameters = dict(
        model.named_parameters()
    )

    assert set(state) == {
        name
        for name, parameter
        in model.named_parameters()
        if parameter.requires_grad
    }

    with torch.no_grad():
        for name, value in state.items():
            named_parameters[name].copy_(
                value.to(
                    device=named_parameters[name].device,
                    dtype=named_parameters[name].dtype,
                )
            )


def phase29_l2sp_loss(model, anchor):
    losses = []

    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            losses.append(
                torch.mean(
                    (
                        parameter
                        - anchor[name]
                    ) ** 2
                )
            )

    return torch.stack(losses).mean()


@torch.no_grad()
def phase29_extract_class_tokens(
    model,
    indices,
    batch_size_cases=4,
):
    model.eval()

    indices = np.asarray(indices, dtype=np.int64)
    outputs = []

    for start in range(
        0,
        len(indices),
        batch_size_cases,
    ):
        batch_indices = indices[
            start:start + batch_size_cases
        ]

        views = phase29_make_case_views(
            batch_indices
        )

        class_token, _ = (
            phase29_forward_descriptors(
                model,
                views,
            )
        )

        outputs.append(
            class_token.reshape(
                len(batch_indices),
                9,
                384,
            ).detach().cpu().numpy()
        )

        del views, class_token

    return np.concatenate(
        outputs,
        axis=0,
    ).astype(np.float32)


def phase29_fit_probe(
    train_features,
    train_labels,
    prediction_features,
    seed,
):
    train_features = np.asarray(
        train_features,
        dtype=np.float32,
    )
    prediction_features = np.asarray(
        prediction_features,
        dtype=np.float32,
    )

    scaler = StandardScaler(
        with_mean=True,
        with_std=True,
    )

    train_scaled = scaler.fit_transform(
        train_features
    )
    prediction_scaled = scaler.transform(
        prediction_features
    )

    pca = PCA(
        n_components=PHASE29_CONFIG[
            "pca_dimension"
        ],
        whiten=True,
        svd_solver="randomized",
        iterated_power=PHASE29_CONFIG[
            "pca_iterated_power"
        ],
        random_state=int(seed),
    )

    train_reduced = pca.fit_transform(
        train_scaled
    )
    prediction_reduced = pca.transform(
        prediction_scaled
    )

    classifier = LogisticRegression(
        C=PHASE29_CONFIG["logistic_c"],
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        random_state=int(seed),
    )

    classifier.fit(
        train_reduced,
        train_labels,
    )

    probability = classifier.predict_proba(
        prediction_reduced
    )[:, 1]

    state = {
        "scaler": scaler,
        "pca": pca,
        "classifier": classifier,
    }

    return probability.astype(np.float64), state


# Verify that regenerated clean views reproduce the cached
# DINOv3 teacher descriptors.
phase29_seed_everything(290600)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(
        phase29_device
    )

contract_model = phase29_build_student()
contract_indices = np.asarray([0, 1])

contract_views = phase29_make_case_views(
    contract_indices
)

with torch.no_grad():
    contract_cls, contract_patch = (
        phase29_forward_descriptors(
            contract_model,
            contract_views,
        )
    )

target_cls, target_patch = (
    phase29_teacher_targets(contract_indices)
)

cls_alignment_error = float(
    torch.max(
        torch.abs(
            contract_cls - target_cls
        )
    ).item()
)

patch_alignment_error = float(
    torch.max(
        torch.abs(
            contract_patch - target_patch
        )
    ).item()
)

anchor = phase29_trainable_anchor(
    contract_model
)

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter in contract_model.parameters()
        if parameter.requires_grad
    ],
    lr=PHASE29_CONFIG["learning_rate"],
    weight_decay=PHASE29_CONFIG[
        "weight_decay"
    ],
)

strong_views = phase29_augment_views(
    contract_views,
    seed=290600,
)

student_cls, student_patch = (
    phase29_forward_descriptors(
        contract_model,
        strong_views,
    )
)

distillation_loss = (
    1.0
    - F.cosine_similarity(
        student_cls,
        target_cls,
        dim=1,
    ).mean()
)

distillation_loss = (
    distillation_loss
    + PHASE29_CONFIG[
        "patch_distillation_weight"
    ]
    * (
        1.0
        - F.cosine_similarity(
            student_patch,
            target_patch,
            dim=1,
        ).mean()
    )
)

l2sp_loss = phase29_l2sp_loss(
    contract_model,
    anchor,
)

contract_loss = (
    distillation_loss
    + PHASE29_CONFIG["l2sp_weight"]
    * l2sp_loss
)

optimizer.zero_grad(set_to_none=True)
contract_loss.backward()

gradient_norm = torch.nn.utils.clip_grad_norm_(
    [
        parameter
        for parameter in contract_model.parameters()
        if parameter.requires_grad
    ],
    PHASE29_CONFIG["gradient_clip"],
)

optimizer.step()

trainable_parameter_count = int(
    sum(
        parameter.numel()
        for parameter in contract_model.parameters()
        if parameter.requires_grad
    )
)

peak_vram_mb = (
    torch.cuda.max_memory_allocated(
        phase29_device
    ) / (1024 ** 2)
    if phase29_device.type == "cuda"
    else 0.0
)

report = {
    "phase": "phase29_dinov3_adaptation_contract",
    "status": (
        "accepted"
        if (
            cls_alignment_error <= 0.01
            and patch_alignment_error <= 0.01
        )
        else "rejected_cache_misalignment"
    ),
    "backbone": "dinov3_vits16_LVD1689M",
    "total_parameter_count": int(
        sum(
            parameter.numel()
            for parameter
            in contract_model.parameters()
        )
    ),
    "trainable_parameter_count": (
        trainable_parameter_count
    ),
    "trainable_scope": [
        "blocks.11",
        "norm",
    ],
    "teacher_descriptors": [
        "class_token",
        "global_patch_mean",
    ],
    "maximum_class_cache_alignment_error": round(
        cls_alignment_error,
        8,
    ),
    "maximum_patch_cache_alignment_error": round(
        patch_alignment_error,
        8,
    ),
    "synthetic_adaptation_loss": round(
        float(contract_loss.detach().item()),
        8,
    ),
    "gradient_norm": round(
        float(gradient_norm),
        6,
    ),
    "peak_vram_mb": round(
        float(peak_vram_mb),
        2,
    ),
    "backward_contract_passed": True,
    "training_voxel_cache_read": True,
    "training_labels_used": False,
    "outer_validation_images_used": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE29_ADAPTATION_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE29_ADAPTATION_CONTRACT")

assert report["status"] == "accepted", report

del (
    contract_model,
    contract_views,
    contract_cls,
    contract_patch,
    target_cls,
    target_patch,
    strong_views,
    student_cls,
    student_patch,
    anchor,
    optimizer,
)

torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE29_ADAPTATION_CONTRACT
{
  "phase": "phase29_dinov3_adaptation_contract",
  "status": "accepted",
  "backbone": "dinov3_vits16_LVD1689M",
  "total_parameter_count": 21601152,
  "trainable_parameter_count": 1776000,
  "trainable_scope": [
    "blocks.11",
    "norm"
  ],
  "teacher_descriptors": [
    "class_token",
    "global_patch_mean"
  ],
  "maximum_class_cache_alignment_error": 0.00763679,
  "maximum_patch_cache_alignment_error": 0.0050053,
  "synthetic_adaptation_loss": 0.04352567,
  "gradient_norm": 0.16334,
  "peak_vram_mb": 442.38,
  "backward_contract_passed": true,
  "training_voxel_cache_read": true,
  "training_labels_used": false,
  "outer_validation_images_used": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE29_ADAPTATION_CONTRACT


In [102]:
# Cell 79 — Train fit-only adaptation trajectories, select
# using monitor partitions, then refit on outer training.

from sklearn.metrics import log_loss, roc_auc_score


def phase29_sigmoid(logits):
    logits = np.clip(
        np.asarray(logits, dtype=np.float64),
        -30.0,
        30.0,
    )
    return 1.0 / (1.0 + np.exp(-logits))


def phase29_logit(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )
    return np.log(
        probabilities / (1.0 - probabilities)
    )


def phase29_log_loss(labels, probability):
    return float(
        log_loss(
            labels,
            np.clip(
                probability,
                1e-5,
                1.0 - 1e-5,
            ),
            labels=[0, 1],
        )
    )


def phase29_blend(
    baseline_probability,
    component_probability,
    alpha,
):
    blended_logit = (
        (1.0 - float(alpha))
        * phase29_logit(
            baseline_probability
        )
        + float(alpha)
        * phase29_logit(
            component_probability
        )
    )

    return phase29_sigmoid(blended_logit)


def phase29_train_ssl_epoch(
    model,
    optimizer,
    anchor,
    train_indices,
    fold,
    epoch,
):
    model.eval()

    rng = np.random.default_rng(
        PHASE29_CONFIG["adaptation_seed"]
        + fold * 1000
        + epoch * 100
    )

    order = rng.permutation(
        np.asarray(train_indices, dtype=np.int64)
    )

    total_loss = 0.0
    total_distillation = 0.0
    total_cases = 0

    batch_size = PHASE29_CONFIG[
        "batch_size_cases"
    ]

    for batch_number, start in enumerate(
        range(0, len(order), batch_size)
    ):
        batch_indices = order[
            start:start + batch_size
        ]

        clean_views = phase29_make_case_views(
            batch_indices
        )

        class_target, patch_target = (
            phase29_teacher_targets(
                batch_indices
            )
        )

        strong_views = phase29_augment_views(
            clean_views,
            seed=(
                PHASE29_CONFIG[
                    "adaptation_seed"
                ]
                + fold * 100000
                + epoch * 1000
                + batch_number
            ),
        )

        student_cls, student_patch = (
            phase29_forward_descriptors(
                model,
                strong_views,
            )
        )

        class_loss = (
            1.0
            - F.cosine_similarity(
                student_cls,
                class_target,
                dim=1,
            ).mean()
        )

        patch_loss = (
            1.0
            - F.cosine_similarity(
                student_patch,
                patch_target,
                dim=1,
            ).mean()
        )

        distillation = (
            class_loss
            + PHASE29_CONFIG[
                "patch_distillation_weight"
            ]
            * patch_loss
        )

        anchor_loss = phase29_l2sp_loss(
            model,
            anchor,
        )

        loss = (
            distillation
            + PHASE29_CONFIG["l2sp_weight"]
            * anchor_loss
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter in model.parameters()
                if parameter.requires_grad
            ],
            PHASE29_CONFIG["gradient_clip"],
        )

        optimizer.step()

        total_loss += (
            float(loss.detach().item())
            * len(batch_indices)
        )

        total_distillation += (
            float(distillation.detach().item())
            * len(batch_indices)
        )

        total_cases += len(batch_indices)

        del (
            clean_views,
            strong_views,
            class_target,
            patch_target,
            student_cls,
            student_patch,
        )

    return {
        "total_loss": float(
            total_loss / total_cases
        ),
        "distillation_loss": float(
            total_distillation / total_cases
        ),
    }


def phase29_train_to_epoch(
    train_indices,
    fold,
    maximum_epoch,
):
    phase29_seed_everything(
        PHASE29_CONFIG["adaptation_seed"]
        + fold
    )

    model = phase29_build_student()
    anchor = phase29_trainable_anchor(model)

    optimizer = torch.optim.AdamW(
        [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ],
        lr=PHASE29_CONFIG["learning_rate"],
        weight_decay=PHASE29_CONFIG[
            "weight_decay"
        ],
    )

    epoch_records = []

    for epoch in range(
        1,
        int(maximum_epoch) + 1,
    ):
        epoch_record = phase29_train_ssl_epoch(
            model=model,
            optimizer=optimizer,
            anchor=anchor,
            train_indices=train_indices,
            fold=fold,
            epoch=epoch,
        )

        epoch_record["epoch"] = int(epoch)
        epoch_records.append(epoch_record)

        print(
            f"Phase29 fold {fold} SSL epoch "
            f"{epoch}: loss="
            f"{epoch_record['total_loss']:.6f}"
        )

    return model, epoch_records


phase29_labels = np.asarray(
    PHASE19_Y_PRIVATE,
    dtype=np.int64,
).reshape(-1)

phase29_original_cls = np.asarray(
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE[
        :, :, 0, :
    ],
    dtype=np.float32,
)

phase29_original_flat = (
    phase29_original_cls.reshape(1362, -1)
)

partition_by_fold = {
    int(partition["fold"]): partition
    for partition in PHASE19_PARTITIONS
}

PHASE29_DINOV3_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE29_BLEND_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE29_DEPLOYMENT_STATES_PRIVATE = {}
phase29_selection_records = []

phase29_started = time.perf_counter()

for fold in range(3):
    fold_started = time.perf_counter()
    partition = partition_by_fold[fold]

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    fit_labels = phase29_labels[fit_indices]
    monitor_labels = phase29_labels[
        monitor_indices
    ]

    baseline_monitor_probability = (
        phase27_fold_array(
            PHASE24_BASE_MONITOR_PRIVATE,
            fold,
        )
    )

    baseline_monitor_loss = phase29_log_loss(
        monitor_labels,
        baseline_monitor_probability,
    )

    epoch_candidates = []
    epoch_training_records = {}

    # Epoch zero is the exact original frozen representation.
    epoch_feature_sets = {
        0: (
            phase29_original_flat[fit_indices],
            phase29_original_flat[monitor_indices],
        )
    }

    model = None
    optimizer = None
    anchor = None

    phase29_seed_everything(
        PHASE29_CONFIG["adaptation_seed"]
        + fold
    )

    model = phase29_build_student()
    anchor = phase29_trainable_anchor(model)

    optimizer = torch.optim.AdamW(
        [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ],
        lr=PHASE29_CONFIG["learning_rate"],
        weight_decay=PHASE29_CONFIG[
            "weight_decay"
        ],
    )

    previous_epoch = 0

    for target_epoch in PHASE29_CONFIG[
        "candidate_epochs"
    ]:
        if target_epoch == 0:
            continue

        for epoch in range(
            previous_epoch + 1,
            target_epoch + 1,
        ):
            training_record = (
                phase29_train_ssl_epoch(
                    model=model,
                    optimizer=optimizer,
                    anchor=anchor,
                    train_indices=fit_indices,
                    fold=fold,
                    epoch=epoch,
                )
            )

            epoch_training_records[epoch] = (
                training_record
            )

            print(
                f"Phase29 fold {fold} SSL "
                f"epoch {epoch}: loss="
                f"{training_record['total_loss']:.6f}"
            )

        previous_epoch = target_epoch

        combined_indices = np.concatenate(
            [fit_indices, monitor_indices]
        )

        adapted_tokens = (
            phase29_extract_class_tokens(
                model,
                combined_indices,
                batch_size_cases=(
                    PHASE29_CONFIG[
                        "batch_size_cases"
                    ]
                ),
            )
        )

        fit_count = len(fit_indices)

        epoch_feature_sets[target_epoch] = (
            adapted_tokens[:fit_count].reshape(
                fit_count,
                -1,
            ),
            adapted_tokens[fit_count:].reshape(
                len(monitor_indices),
                -1,
            ),
        )

        del adapted_tokens
        torch.cuda.empty_cache()

    # Evaluate each epoch using the exact Phase26
    # StandardScaler -> whitened PCA -> logistic pipeline.
    for candidate_epoch in PHASE29_CONFIG[
        "candidate_epochs"
    ]:
        fit_features, monitor_features = (
            epoch_feature_sets[candidate_epoch]
        )

        component_probability, _ = (
            phase29_fit_probe(
                train_features=fit_features,
                train_labels=fit_labels,
                prediction_features=monitor_features,
                seed=(
                    PHASE29_CONFIG["probe_seed"]
                    + fold
                ),
            )
        )

        component_loss = phase29_log_loss(
            monitor_labels,
            component_probability,
        )

        component_auroc = float(
            roc_auc_score(
                monitor_labels,
                component_probability,
            )
        )

        for alpha in PHASE29_CONFIG[
            "blend_alphas"
        ]:
            blended_probability = phase29_blend(
                baseline_monitor_probability,
                component_probability,
                alpha,
            )

            blend_loss = phase29_log_loss(
                monitor_labels,
                blended_probability,
            )

            epoch_candidates.append({
                "epoch": int(candidate_epoch),
                "alpha": float(alpha),
                "component_monitor_log_loss": (
                    float(component_loss)
                ),
                "component_monitor_auroc": (
                    float(component_auroc)
                ),
                "blend_monitor_log_loss": float(
                    blend_loss
                ),
            })

    epoch0_best = min(
        [
            candidate
            for candidate in epoch_candidates
            if candidate["epoch"] == 0
        ],
        key=lambda candidate: (
            candidate["blend_monitor_log_loss"],
            candidate["alpha"],
        ),
    )

    raw_best = min(
        epoch_candidates,
        key=lambda candidate: (
            candidate["blend_monitor_log_loss"],
            candidate["epoch"],
            candidate["alpha"],
        ),
    )

    gain_over_baseline = (
        baseline_monitor_loss
        - raw_best["blend_monitor_log_loss"]
    )

    adaptation_gain_over_epoch0 = (
        epoch0_best["blend_monitor_log_loss"]
        - raw_best["blend_monitor_log_loss"]
    )

    adaptation_advanced = bool(
        raw_best["epoch"] > 0
        and gain_over_baseline
        >= PHASE29_CONFIG[
            "minimum_monitor_blend_gain"
        ]
        and adaptation_gain_over_epoch0
        >= PHASE29_CONFIG[
            "minimum_adaptation_gain_over_epoch0"
        ]
    )

    if adaptation_advanced:
        selected_epoch = int(raw_best["epoch"])
        selected_alpha = float(raw_best["alpha"])
    else:
        selected_epoch = 0

        epoch0_gain = (
            baseline_monitor_loss
            - epoch0_best[
                "blend_monitor_log_loss"
            ]
        )

        selected_alpha = (
            float(epoch0_best["alpha"])
            if epoch0_gain
            >= PHASE29_CONFIG[
                "minimum_monitor_blend_gain"
            ]
            else 0.0
        )

    del model, optimizer, anchor
    torch.cuda.empty_cache()

    if selected_epoch == 0:
        # Preserve the exact Phase26 component and its fitted
        # deployment pipeline.
        outer_component_probability = np.asarray(
            PHASE26_DINOV3_OOF_PRIVATE,
            dtype=np.float64,
        )[outer_valid_indices]

        deployment_state = {
            "adaptation_epoch": 0,
            "trainable_dinov3_state": None,
            "probe_state": (
                PHASE26_DEPLOYMENT_STATES_PRIVATE[
                    fold
                ]
            ),
        }

    else:
        outer_model, outer_training_records = (
            phase29_train_to_epoch(
                train_indices=outer_train_indices,
                fold=fold,
                maximum_epoch=selected_epoch,
            )
        )

        combined_outer_indices = np.concatenate(
            [
                outer_train_indices,
                outer_valid_indices,
            ]
        )

        adapted_outer_tokens = (
            phase29_extract_class_tokens(
                outer_model,
                combined_outer_indices,
                batch_size_cases=(
                    PHASE29_CONFIG[
                        "batch_size_cases"
                    ]
                ),
            )
        )

        outer_train_count = len(
            outer_train_indices
        )

        outer_train_features = (
            adapted_outer_tokens[
                :outer_train_count
            ].reshape(
                outer_train_count,
                -1,
            )
        )

        outer_valid_features = (
            adapted_outer_tokens[
                outer_train_count:
            ].reshape(
                len(outer_valid_indices),
                -1,
            )
        )

        outer_component_probability, probe_state = (
            phase29_fit_probe(
                train_features=outer_train_features,
                train_labels=phase29_labels[
                    outer_train_indices
                ],
                prediction_features=outer_valid_features,
                seed=(
                    PHASE29_CONFIG["probe_seed"]
                    + fold
                ),
            )
        )

        deployment_state = {
            "adaptation_epoch": int(
                selected_epoch
            ),
            "trainable_dinov3_state": (
                phase29_trainable_state(
                    outer_model
                )
            ),
            "probe_state": probe_state,
        }

        del (
            outer_model,
            adapted_outer_tokens,
            outer_train_features,
            outer_valid_features,
        )
        torch.cuda.empty_cache()

    baseline_outer_probability = np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    )[outer_valid_indices]

    blended_outer_probability = phase29_blend(
        baseline_outer_probability,
        outer_component_probability,
        selected_alpha,
    )

    PHASE29_DINOV3_OOF_PRIVATE[
        outer_valid_indices
    ] = outer_component_probability

    PHASE29_BLEND_OOF_PRIVATE[
        outer_valid_indices
    ] = blended_outer_probability

    deployment_state["blend_alpha"] = float(
        selected_alpha
    )

    PHASE29_DEPLOYMENT_STATES_PRIVATE[
        fold
    ] = deployment_state

    selected_candidate = min(
        [
            candidate
            for candidate in epoch_candidates
            if (
                candidate["epoch"]
                == selected_epoch
                and candidate["alpha"]
                == selected_alpha
            )
        ],
        key=lambda candidate: (
            candidate["blend_monitor_log_loss"]
        ),
    )

    phase29_selection_records.append({
        "fold": int(fold),
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(
            len(monitor_indices)
        ),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "baseline_monitor_log_loss": float(
            baseline_monitor_loss
        ),
        "epoch0_best_alpha": float(
            epoch0_best["alpha"]
        ),
        "epoch0_best_monitor_log_loss": float(
            epoch0_best[
                "blend_monitor_log_loss"
            ]
        ),
        "raw_best_epoch": int(
            raw_best["epoch"]
        ),
        "raw_best_alpha": float(
            raw_best["alpha"]
        ),
        "raw_best_monitor_log_loss": float(
            raw_best[
                "blend_monitor_log_loss"
            ]
        ),
        "raw_gain_over_baseline": float(
            gain_over_baseline
        ),
        "raw_adaptation_gain_over_epoch0": float(
            adaptation_gain_over_epoch0
        ),
        "selected_epoch": int(
            selected_epoch
        ),
        "selected_alpha": float(
            selected_alpha
        ),
        "selected_component_monitor_log_loss": float(
            selected_candidate[
                "component_monitor_log_loss"
            ]
        ),
        "selected_component_monitor_auroc": float(
            selected_candidate[
                "component_monitor_auroc"
            ]
        ),
        "selected_blend_monitor_log_loss": float(
            selected_candidate[
                "blend_monitor_log_loss"
            ]
        ),
        "adaptation_advanced": bool(
            adaptation_advanced
        ),
        "ssl_epoch_losses": {
            str(epoch): round(
                record["total_loss"],
                8,
            )
            for epoch, record
            in epoch_training_records.items()
        },
        "elapsed_seconds": round(
            time.perf_counter()
            - fold_started,
            2,
        ),
    })

    print(
        f"Phase29 fold {fold} selected: "
        f"epoch={selected_epoch}, "
        f"alpha={selected_alpha:.2f}, "
        f"adapted={adaptation_advanced}"
    )

assert np.isfinite(
    PHASE29_DINOV3_OOF_PRIVATE
).all()

assert np.isfinite(
    PHASE29_BLEND_OOF_PRIVATE
).all()

report = {
    "phase": "phase29_fold_local_ssl_selection",
    "status": "complete",
    "configuration": PHASE29_CONFIG,
    "folds": phase29_selection_records,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase29_started,
        2,
    ),
    "adapted_fold_count": int(
        sum(
            record["adaptation_advanced"]
            for record in phase29_selection_records
        )
    ),
    "adaptation_images": (
        "fit_only_during_selection; "
        "outer_train_only_after_selection"
    ),
    "adaptation_labels_used": False,
    "monitor_labels_used_for_selection": True,
    "outer_validation_images_used_for_adaptation": False,
    "outer_validation_labels_used": False,
    "test_or_smoke_data_read": False,
    "deployment_states_retained_privately": True,
    "case_level_features_exported": False,
}

print("BEGIN SANITIZED_PHASE29_SELECTION")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE29_SELECTION")

Phase29 fold 0 SSL epoch 1: loss=0.026039
Phase29 fold 0 SSL epoch 2: loss=0.023336
Phase29 fold 0 SSL epoch 3: loss=0.022124
Phase29 fold 0 SSL epoch 4: loss=0.021650
Phase29 fold 0 selected: epoch=0, alpha=0.15, adapted=False
Phase29 fold 1 SSL epoch 1: loss=0.030202
Phase29 fold 1 SSL epoch 2: loss=0.026859
Phase29 fold 1 SSL epoch 3: loss=0.025409
Phase29 fold 1 SSL epoch 4: loss=0.024490
Phase29 fold 1 selected: epoch=0, alpha=0.00, adapted=False
Phase29 fold 2 SSL epoch 1: loss=0.025899
Phase29 fold 2 SSL epoch 2: loss=0.022674
Phase29 fold 2 SSL epoch 3: loss=0.021729
Phase29 fold 2 SSL epoch 4: loss=0.020975
Phase29 fold 2 selected: epoch=0, alpha=0.00, adapted=False
BEGIN SANITIZED_PHASE29_SELECTION
{
  "phase": "phase29_fold_local_ssl_selection",
  "status": "complete",
  "configuration": {
    "phase": "phase29_fold_local_dinov3_adaptation",
    "candidate_epochs": [
      0,
      1,
      2,
      4
    ],
    "batch_size_cases": 4,
    "learning_rate": 5e-06,
    "weight_

In [103]:
# Cell 80 — One-time Phase29 outer evaluation

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)


def phase29_metrics(labels, probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )

    return {
        "log_loss": float(
            log_loss(
                labels,
                probability,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                labels,
                probability,
            )
        ),
        "brier": float(
            brier_score_loss(
                labels,
                probability,
            )
        ),
    }


phase29_baseline = np.asarray(
    PHASE19_PHASE12C_OOF_PRIVATE,
    dtype=np.float64,
)

phase29_phase26 = np.asarray(
    PHASE26_DINOV3_BLEND_OOF_PRIVATE,
    dtype=np.float64,
)

phase29_component = np.asarray(
    PHASE29_DINOV3_OOF_PRIVATE,
    dtype=np.float64,
)

phase29_blend = np.asarray(
    PHASE29_BLEND_OOF_PRIVATE,
    dtype=np.float64,
)

baseline_metrics = phase29_metrics(
    phase29_labels,
    phase29_baseline,
)

phase26_metrics = phase29_metrics(
    phase29_labels,
    phase29_phase26,
)

component_metrics = phase29_metrics(
    phase29_labels,
    phase29_component,
)

blend_metrics = phase29_metrics(
    phase29_labels,
    phase29_blend,
)

fold_reports = []
fold_improvements = []

for fold in range(3):
    indices = np.asarray(
        partition_by_fold[fold][
            "outer_valid"
        ],
        dtype=np.int64,
    )

    labels = phase29_labels[indices]

    baseline_fold = phase29_metrics(
        labels,
        phase29_baseline[indices],
    )

    component_fold = phase29_metrics(
        labels,
        phase29_component[indices],
    )

    blend_fold = phase29_metrics(
        labels,
        phase29_blend[indices],
    )

    improvement = (
        baseline_fold["log_loss"]
        - blend_fold["log_loss"]
    )

    fold_improvements.append(improvement)

    selection = phase29_selection_records[fold]

    fold_reports.append({
        "fold": int(fold),
        "n": int(len(indices)),
        "selected_adaptation_epoch": int(
            selection["selected_epoch"]
        ),
        "selected_blend_alpha": float(
            selection["selected_alpha"]
        ),
        "adaptation_advanced": bool(
            selection["adaptation_advanced"]
        ),
        "baseline_log_loss": round(
            baseline_fold["log_loss"],
            6,
        ),
        "component_log_loss": round(
            component_fold["log_loss"],
            6,
        ),
        "blend_log_loss": round(
            blend_fold["log_loss"],
            6,
        ),
        "blend_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_fold["auroc"],
            6,
        ),
        "component_auroc": round(
            component_fold["auroc"],
            6,
        ),
        "blend_auroc": round(
            blend_fold["auroc"],
            6,
        ),
    })


groups = np.asarray(case_df[group_column])
group_reports = []
group_harms = []

for group in np.unique(groups):
    indices = np.flatnonzero(groups == group)

    if len(indices) < 30:
        continue

    baseline_loss = phase29_metrics(
        phase29_labels[indices],
        phase29_baseline[indices],
    )["log_loss"]

    blend_loss = phase29_metrics(
        phase29_labels[indices],
        phase29_blend[indices],
    )["log_loss"]

    change = blend_loss - baseline_loss
    group_harms.append(change)

    group_reports.append({
        "group": (
            int(group)
            if np.issubdtype(
                np.asarray(group).dtype,
                np.integer,
            )
            else str(group)
        ),
        "n": int(len(indices)),
        "baseline_log_loss": round(
            baseline_loss,
            6,
        ),
        "blend_log_loss": round(
            blend_loss,
            6,
        ),
        "log_loss_change": round(
            change,
            6,
        ),
    })


log_loss_gain = (
    baseline_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

gain_over_phase26 = (
    phase26_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

auroc_gain = (
    blend_metrics["auroc"]
    - baseline_metrics["auroc"]
)

brier_excess = (
    blend_metrics["brier"]
    - baseline_metrics["brier"]
)

fold_wins = int(
    sum(value > 0 for value in fold_improvements)
)

worst_fold_excess = float(
    max(-value for value in fold_improvements)
)

maximum_group_harm = float(
    max(group_harms) if group_harms else 0.0
)

adapted_fold_count = int(
    sum(
        record["adaptation_advanced"]
        for record in phase29_selection_records
    )
)

adaptation_gate_passed = bool(
    adapted_fold_count >= 1
    and gain_over_phase26 >= 0.001
    and worst_fold_excess <= 0.005
)

promotion_gate_passed = bool(
    log_loss_gain >= 0.004
    and gain_over_phase26 >= 0.001
    and auroc_gain >= 0.001
    and fold_wins >= 1
    and worst_fold_excess <= 0.005
    and brier_excess <= 0.001
    and maximum_group_harm <= 0.015
)

if promotion_gate_passed:
    status = "promoted"
elif adaptation_gate_passed:
    status = "component_retained"
else:
    status = "adaptation_terminated_retain_phase26"

report = {
    "phase": "phase29_fold_local_dinov3_adaptation",
    "status": status,
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "phase26_champion": {
        key: round(value, 6)
        for key, value in phase26_metrics.items()
    },
    "adapted_dinov3_component": {
        key: round(value, 6)
        for key, value in component_metrics.items()
    },
    "phase29_blend": {
        key: round(value, 6)
        for key, value in blend_metrics.items()
    },
    "improvements": {
        "log_loss_gain_over_phase12c": round(
            log_loss_gain,
            6,
        ),
        "log_loss_gain_over_phase26": round(
            gain_over_phase26,
            6,
        ),
        "auroc_gain_over_phase12c": round(
            auroc_gain,
            6,
        ),
        "brier_excess": round(
            brier_excess,
            6,
        ),
        "fold_wins": fold_wins,
        "adapted_fold_count": adapted_fold_count,
        "worst_fold_excess": round(
            worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            maximum_group_harm,
            6,
        ),
    },
    "fold_metrics": fold_reports,
    "major_acquisition_group_metrics": group_reports,
    "gates": {
        "adaptation_gate_passed": (
            adaptation_gate_passed
        ),
        "promotion_gate_passed": (
            promotion_gate_passed
        ),
    },
    "dinov3_license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "outer_validation_images_used_for_adaptation": False,
    "adaptation_labels_used": False,
    "test_or_smoke_data_read": False,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
}

print("BEGIN SANITIZED_PHASE29_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE29_OOF")

BEGIN SANITIZED_PHASE29_OOF
{
  "phase": "phase29_fold_local_dinov3_adaptation",
  "status": "adaptation_terminated_retain_phase26",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "phase26_champion": {
    "log_loss": 0.304742,
    "auroc": 0.939583,
    "brier": 0.094448
  },
  "adapted_dinov3_component": {
    "log_loss": 0.37838,
    "auroc": 0.911055,
    "brier": 0.120334
  },
  "phase29_blend": {
    "log_loss": 0.304742,
    "auroc": 0.939583,
    "brier": 0.094448
  },
  "improvements": {
    "log_loss_gain_over_phase12c": 0.002873,
    "log_loss_gain_over_phase26": 0.0,
    "auroc_gain_over_phase12c": 0.000655,
    "brier_excess": -0.001087,
    "fold_wins": 1,
    "adapted_fold_count": 0,
    "worst_fold_excess": -0.0,
    "maximum_major_group_harm": 0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "selected_adaptation_epoch": 0,
      "selected_blend_alpha": 0.15,
      "adaptation_advanced": false,
    

In [104]:
# Cell 81 — Phase30 thin-slab and bilateral-comparison views

import copy
import json
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


PHASE30_CONFIG = {
    "phase": "phase30_clinical_view_dinov3",
    "thin_slab_centers": [24, 32, 40, 48, 56],
    "thin_slab_thickness": 24,
    "bilateral_slab_ranges": [
        [0, 40],
        [20, 60],
        [40, 80],
    ],
    "projection_axes": [0, 1, 2],
    "bilateral_projection_axes": [1, 2],
    "thin_view_count": 15,
    "bilateral_view_count": 6,
    "total_view_count": 21,
    "image_size": 224,
    "extraction_batch_size_cases": 4,
    "pca_dimensions": [32, 64, 128],
    "logistic_c_values": [0.03, 0.1],
    "blend_alphas": [
        0.0, 0.05, 0.10,
        0.15, 0.20, 0.30,
    ],
    "minimum_monitor_blend_gain": 0.002,
    "base_seed": 300601,
}


PHASE30_IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(1, 3, 1, 1)

PHASE30_IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(1, 3, 1, 1)


def phase30_robust_scale(image):
    assert image.ndim == 3

    batch_size = image.shape[0]
    flattened = image.reshape(batch_size, -1)

    lower = torch.quantile(
        flattened,
        0.01,
        dim=1,
        keepdim=True,
    )

    upper = torch.quantile(
        flattened,
        0.995,
        dim=1,
        keepdim=True,
    )

    denominator = torch.clamp(
        upper - lower,
        min=1e-5,
    )

    scaled = (
        flattened - lower
    ) / denominator

    return torch.clamp(
        scaled,
        0.0,
        1.0,
    ).reshape_as(image)


def phase30_top_tail_mean(
    slab,
    reduction_dimension,
):
    threshold = torch.quantile(
        slab,
        0.80,
        dim=reduction_dimension,
        keepdim=True,
    )

    mask = slab >= threshold

    numerator = torch.sum(
        slab * mask,
        dim=reduction_dimension,
    )

    denominator = torch.clamp(
        torch.sum(
            mask,
            dim=reduction_dimension,
        ),
        min=1,
    )

    return numerator / denominator


def phase30_projection_triplet(
    volume,
    axis,
    start,
    stop,
):
    assert volume.ndim == 4
    assert tuple(volume.shape[1:]) == (
        80, 80, 80
    )

    reduction_dimension = axis + 1

    slab = volume.narrow(
        dim=reduction_dimension,
        start=int(start),
        length=int(stop - start),
    )

    maximum = torch.amax(
        slab,
        dim=reduction_dimension,
    )

    hot_mean = phase30_top_tail_mean(
        slab,
        reduction_dimension,
    )

    ordinary_mean = torch.mean(
        slab,
        dim=reduction_dimension,
    )

    channels = torch.stack(
        [
            phase30_robust_scale(maximum),
            phase30_robust_scale(hot_mean),
            phase30_robust_scale(
                ordinary_mean
            ),
        ],
        dim=1,
    )

    channels = F.interpolate(
        channels,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
    )

    mean = PHASE30_IMAGENET_MEAN.to(
        channels.device
    )
    std = PHASE30_IMAGENET_STD.to(
        channels.device
    )

    return (channels - mean) / std


def phase30_bilateral_comparison(
    volume,
    axis,
    start,
    stop,
):
    assert axis in (1, 2)

    reduction_dimension = axis + 1

    slab = volume.narrow(
        dim=reduction_dimension,
        start=int(start),
        length=int(stop - start),
    )

    hot_projection = phase30_top_tail_mean(
        slab,
        reduction_dimension,
    )

    # After projecting axis 1 or 2, the first remaining
    # spatial dimension is canonical left-right.
    assert hot_projection.shape[1] == 80

    first_half = hot_projection[:, :40, :]
    second_half = hot_projection[:, 40:, :]

    # Mirror the second half into the first-half frame.
    second_half = torch.flip(
        second_half,
        dims=[1],
    )

    combined = torch.cat(
        [first_half, second_half],
        dim=1,
    )

    batch_size = combined.shape[0]
    flattened = combined.reshape(
        batch_size,
        -1,
    )

    lower = torch.quantile(
        flattened,
        0.01,
        dim=1,
        keepdim=True,
    )

    upper = torch.quantile(
        flattened,
        0.995,
        dim=1,
        keepdim=True,
    )

    denominator = torch.clamp(
        upper - lower,
        min=1e-5,
    )

    lower = lower.view(batch_size, 1, 1)
    denominator = denominator.view(
        batch_size, 1, 1
    )

    first_scaled = torch.clamp(
        (first_half - lower) / denominator,
        0.0,
        1.0,
    )

    second_scaled = torch.clamp(
        (second_half - lower) / denominator,
        0.0,
        1.0,
    )

    # Swap-invariant channels.
    bilateral_mean = 0.5 * (
        first_scaled + second_scaled
    )
    bilateral_difference = torch.abs(
        first_scaled - second_scaled
    )
    bilateral_minimum = torch.minimum(
        first_scaled,
        second_scaled,
    )

    comparison = torch.stack(
        [
            bilateral_mean,
            bilateral_difference,
            bilateral_minimum,
        ],
        dim=1,
    )

    comparison = F.interpolate(
        comparison,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
    )

    mean = PHASE30_IMAGENET_MEAN.to(
        comparison.device
    )
    std = PHASE30_IMAGENET_STD.to(
        comparison.device
    )

    return (comparison - mean) / std


def phase30_make_clinical_views(volume):
    assert volume.ndim == 4
    assert tuple(volume.shape[1:]) == (
        80, 80, 80
    )

    views = []

    half_thickness = (
        PHASE30_CONFIG[
            "thin_slab_thickness"
        ] // 2
    )

    # 3 axes × 5 localized thin slabs.
    for axis in PHASE30_CONFIG[
        "projection_axes"
    ]:
        for center in PHASE30_CONFIG[
            "thin_slab_centers"
        ]:
            start = center - half_thickness
            stop = center + half_thickness

            views.append(
                phase30_projection_triplet(
                    volume,
                    axis,
                    start,
                    stop,
                )
            )

    # 2 axes × 3 reflection-invariant bilateral views.
    for axis in PHASE30_CONFIG[
        "bilateral_projection_axes"
    ]:
        for start, stop in PHASE30_CONFIG[
            "bilateral_slab_ranges"
        ]:
            views.append(
                phase30_bilateral_comparison(
                    volume,
                    axis,
                    start,
                    stop,
                )
            )

    result = torch.stack(views, dim=1)

    assert result.shape == (
        volume.shape[0],
        21,
        3,
        224,
        224,
    )

    return result


class Phase30DINOCLS(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, images):
        result = self.backbone.forward_features(
            images
        )
        return result["x_norm_clstoken"]


phase30_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

synthetic_volume = torch.rand(
    2,
    80,
    80,
    80,
    device=phase30_device,
)

synthetic_views = phase30_make_clinical_views(
    synthetic_volume
)

contract_backbone = copy.deepcopy(
    PHASE26_DINOV3_PRIVATE
).to(phase30_device)

contract_backbone.eval()

for parameter in contract_backbone.parameters():
    parameter.requires_grad_(False)

contract_wrapper = Phase30DINOCLS(
    contract_backbone
).to(phase30_device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(
        phase30_device
    )

with torch.no_grad():
    synthetic_features = contract_wrapper(
        synthetic_views.reshape(
            -1, 3, 224, 224
        )
    ).reshape(2, 21, 384)

thin_features = synthetic_features[:, :15]
bilateral_features = synthetic_features[:, 15:]

assert synthetic_features.shape == (
    2, 21, 384
)
assert torch.isfinite(
    synthetic_features
).all()

# Bilateral channels are invariant to swapping halves
# by construction.
example = synthetic_volume[:1]
bilateral_example = (
    phase30_bilateral_comparison(
        example,
        axis=1,
        start=20,
        stop=60,
    )
)

peak_vram_mb = (
    torch.cuda.max_memory_allocated(
        phase30_device
    ) / (1024 ** 2)
    if phase30_device.type == "cuda"
    else 0.0
)

report = {
    "phase": "phase30_clinical_view_contract",
    "status": "accepted",
    "view_count": 21,
    "thin_slab_view_count": 15,
    "bilateral_comparison_view_count": 6,
    "thin_slab_axes": 3,
    "thin_slabs_per_axis": 5,
    "bilateral_projection_axes": 2,
    "bilateral_slabs_per_axis": 3,
    "bilateral_channels": [
        "hemisphere_mean",
        "absolute_hemisphere_difference",
        "hemisphere_minimum",
    ],
    "view_tensor_shape": list(
        synthetic_views.shape
    ),
    "feature_tensor_shape": list(
        synthetic_features.shape
    ),
    "mean_thin_feature_std": round(
        float(
            thin_features.std(
                dim=1
            ).mean().item()
        ),
        6,
    ),
    "mean_bilateral_feature_std": round(
        float(
            bilateral_features.std(
                dim=1
            ).mean().item()
        ),
        6,
    ),
    "peak_vram_mb": round(
        float(peak_vram_mb),
        2,
    ),
    "bilateral_representation_reflection_invariant": True,
    "dinov3_backbone_frozen": True,
    "external_weights": True,
    "license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "synthetic_input_only": True,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_VIEW_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_VIEW_CONTRACT")

del (
    synthetic_volume,
    synthetic_views,
    synthetic_features,
    thin_features,
    bilateral_features,
    bilateral_example,
    contract_wrapper,
    contract_backbone,
)

torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE30_VIEW_CONTRACT
{
  "phase": "phase30_clinical_view_contract",
  "status": "accepted",
  "view_count": 21,
  "thin_slab_view_count": 15,
  "bilateral_comparison_view_count": 6,
  "thin_slab_axes": 3,
  "thin_slabs_per_axis": 5,
  "bilateral_projection_axes": 2,
  "bilateral_slabs_per_axis": 3,
  "bilateral_channels": [
    "hemisphere_mean",
    "absolute_hemisphere_difference",
    "hemisphere_minimum"
  ],
  "view_tensor_shape": [
    2,
    21,
    3,
    224,
    224
  ],
  "feature_tensor_shape": [
    2,
    21,
    384
  ],
  "mean_thin_feature_std": 0.074852,
  "mean_bilateral_feature_std": 0.063837,
  "peak_vram_mb": 486.79,
  "bilateral_representation_reflection_invariant": true,
  "dinov3_backbone_frozen": true,
  "external_weights": true,
  "license": "DINOv3 License",
  "required_attribution": "Built with DINOv3",
  "synthetic_input_only": true,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE30_VIEW_CONTRACT


In [105]:
# Cell 82 — Phase30 two-GPU clinical-view extraction

phase30_started = time.perf_counter()

extraction_backbone = copy.deepcopy(
    PHASE26_DINOV3_PRIVATE
).to(phase30_device)

extraction_backbone.eval()

for parameter in extraction_backbone.parameters():
    parameter.requires_grad_(False)

extraction_model = Phase30DINOCLS(
    extraction_backbone
).to(phase30_device)

phase30_gpu_count = min(
    2,
    torch.cuda.device_count(),
)

if phase30_gpu_count == 2:
    extraction_model = nn.DataParallel(
        extraction_model,
        device_ids=[0, 1],
        output_device=0,
    )

extraction_model.eval()

if torch.cuda.is_available():
    for device_index in range(
        phase30_gpu_count
    ):
        torch.cuda.reset_peak_memory_stats(
            device_index
        )

PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE = (
    np.empty(
        (1362, 21, 384),
        dtype=np.float16,
    )
)

batch_size_cases = PHASE30_CONFIG[
    "extraction_batch_size_cases"
]

with torch.inference_mode():
    for start in range(
        0,
        1362,
        batch_size_cases,
    ):
        stop = min(
            start + batch_size_cases,
            1362,
        )

        volume = torch.from_numpy(
            np.asarray(
                PHASE19_HIGHRES_CACHE[
                    start:stop
                ],
                dtype=np.float32,
            )
        ).to(phase30_device)

        views = phase30_make_clinical_views(
            volume
        )

        features = extraction_model(
            views.reshape(
                -1,
                3,
                224,
                224,
            )
        )

        features = features.reshape(
            stop - start,
            21,
            384,
        )

        PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE[
            start:stop
        ] = (
            features.detach()
            .cpu()
            .numpy()
            .astype(np.float16)
        )

        if (
            stop % 100 < batch_size_cases
            or stop == 1362
        ):
            print(
                f"Phase30 feature extraction: "
                f"{stop}/1362"
            )

        del volume, views, features

assert np.isfinite(
    PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE
).all()

phase30_feature_float = np.asarray(
    PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE,
    dtype=np.float32,
)

thin_std = float(
    np.mean(
        np.std(
            phase30_feature_float[:, :15],
            axis=1,
        )
    )
)

bilateral_std = float(
    np.mean(
        np.std(
            phase30_feature_float[:, 15:],
            axis=1,
        )
    )
)

collapsed_fraction = float(
    np.mean(
        np.std(
            phase30_feature_float.reshape(
                1362, -1
            ),
            axis=0,
        ) < 1e-6
    )
)

peak_vram = {}

if torch.cuda.is_available():
    for device_index in range(
        phase30_gpu_count
    ):
        peak_vram[f"cuda:{device_index}"] = round(
            float(
                torch.cuda.max_memory_allocated(
                    device_index
                ) / (1024 ** 2)
            ),
            2,
        )

report = {
    "phase": "phase30_clinical_feature_cache",
    "status": "complete",
    "case_count": 1362,
    "view_count": 21,
    "embedding_dimension": 384,
    "feature_shape": list(
        PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE.shape
    ),
    "dtype": str(
        PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE.dtype
    ),
    "cache_ram_gb": round(
        float(
            PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE.nbytes
            / (1024 ** 3)
        ),
        4,
    ),
    "mean_thin_view_feature_std": round(
        thin_std,
        6,
    ),
    "mean_bilateral_view_feature_std": round(
        bilateral_std,
        6,
    ),
    "collapsed_coordinate_fraction": round(
        collapsed_fraction,
        8,
    ),
    "gpu_count_used": phase30_gpu_count,
    "peak_vram_mb": peak_vram,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase30_started,
        2,
    ),
    "dinov3_backbone_frozen": True,
    "training_voxel_cache_read": True,
    "labels_used": False,
    "test_or_smoke_data_read": False,
    "case_level_features_exported": False,
}

print("BEGIN SANITIZED_PHASE30_FEATURE_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_FEATURE_CACHE")

del (
    extraction_model,
    extraction_backbone,
    phase30_feature_float,
)

torch.cuda.empty_cache()

Phase30 feature extraction: 100/1362
Phase30 feature extraction: 200/1362
Phase30 feature extraction: 300/1362
Phase30 feature extraction: 400/1362
Phase30 feature extraction: 500/1362
Phase30 feature extraction: 600/1362
Phase30 feature extraction: 700/1362
Phase30 feature extraction: 800/1362
Phase30 feature extraction: 900/1362
Phase30 feature extraction: 1000/1362
Phase30 feature extraction: 1100/1362
Phase30 feature extraction: 1200/1362
Phase30 feature extraction: 1300/1362
Phase30 feature extraction: 1362/1362
BEGIN SANITIZED_PHASE30_FEATURE_CACHE
{
  "phase": "phase30_clinical_feature_cache",
  "status": "complete",
  "case_count": 1362,
  "view_count": 21,
  "embedding_dimension": 384,
  "feature_shape": [
    1362,
    21,
    384
  ],
  "dtype": "float16",
  "cache_ram_gb": 0.0205,
  "mean_thin_view_feature_std": 0.196742,
  "mean_bilateral_view_feature_std": 0.142586,
  "collapsed_coordinate_fraction": 0.0,
  "gpu_count_used": 2,
  "peak_vram_mb": {
    "cuda:0": 518.15,
  

In [106]:
# Cell 83 — Phase30 nested linear-probe experiment

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler


phase30_clinical = np.asarray(
    PHASE30_CLINICAL_FEATURE_CACHE_PRIVATE,
    dtype=np.float32,
)

phase30_original = np.asarray(
    PHASE26_DINOV3_FEATURE_CACHE_PRIVATE[
        :, :, 0, :
    ],
    dtype=np.float32,
)

PHASE30_REPRESENTATIONS_PRIVATE = {
    "thin_slabs": (
        phase30_clinical[:, :15].reshape(
            1362, -1
        )
    ),
    "bilateral_comparisons": (
        phase30_clinical[:, 15:].reshape(
            1362, -1
        )
    ),
    "original_plus_bilateral": (
        np.concatenate(
            [
                phase30_original,
                phase30_clinical[:, 15:],
            ],
            axis=1,
        ).reshape(1362, -1)
    ),
    "clinical_all": (
        phase30_clinical.reshape(
            1362, -1
        )
    ),
    "original_plus_clinical": (
        np.concatenate(
            [
                phase30_original,
                phase30_clinical,
            ],
            axis=1,
        ).reshape(1362, -1)
    ),
}

phase30_labels = np.asarray(
    PHASE19_Y_PRIVATE,
    dtype=np.int64,
).reshape(-1)


def phase30_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )
    return np.log(
        probability / (1.0 - probability)
    )


def phase30_sigmoid(logit):
    logit = np.clip(
        np.asarray(logit, dtype=np.float64),
        -30.0,
        30.0,
    )
    return 1.0 / (1.0 + np.exp(-logit))


def phase30_loss(labels, probability):
    return float(
        log_loss(
            labels,
            np.clip(
                probability,
                1e-5,
                1.0 - 1e-5,
            ),
            labels=[0, 1],
        )
    )


def phase30_fit_pipeline(
    matrix,
    train_indices,
    prediction_indices,
    dimension,
    c_value,
    seed,
):
    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        matrix[train_indices]
    )

    prediction_scaled = scaler.transform(
        matrix[prediction_indices]
    )

    pca = PCA(
        n_components=int(dimension),
        whiten=True,
        svd_solver="randomized",
        iterated_power=3,
        random_state=int(seed),
    )

    train_reduced = pca.fit_transform(
        train_scaled
    )

    prediction_reduced = pca.transform(
        prediction_scaled
    )

    classifier = LogisticRegression(
        C=float(c_value),
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        random_state=int(seed),
    )

    classifier.fit(
        train_reduced,
        phase30_labels[train_indices],
    )

    probability = classifier.predict_proba(
        prediction_reduced
    )[:, 1]

    return probability.astype(np.float64), {
        "scaler": scaler,
        "pca": pca,
        "classifier": classifier,
    }


partition_by_fold = {
    int(partition["fold"]): partition
    for partition in PHASE19_PARTITIONS
}

PHASE30_COMPONENT_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE30_BLEND_OOF_PRIVATE = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

PHASE30_DEPLOYMENT_STATES_PRIVATE = {}
phase30_selection_records = []

phase30_training_started = time.perf_counter()

for fold in range(3):
    fold_started = time.perf_counter()
    partition = partition_by_fold[fold]

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    monitor_labels = phase30_labels[
        monitor_indices
    ]

    candidate_records = []

    for representation_name, matrix in (
        PHASE30_REPRESENTATIONS_PRIVATE.items()
    ):
        for dimension in PHASE30_CONFIG[
            "pca_dimensions"
        ]:
            for c_value in PHASE30_CONFIG[
                "logistic_c_values"
            ]:
                probability, _ = (
                    phase30_fit_pipeline(
                        matrix=matrix,
                        train_indices=fit_indices,
                        prediction_indices=(
                            monitor_indices
                        ),
                        dimension=dimension,
                        c_value=c_value,
                        seed=(
                            PHASE30_CONFIG[
                                "base_seed"
                            ] + fold
                        ),
                    )
                )

                candidate_records.append({
                    "representation": (
                        representation_name
                    ),
                    "dimension": int(dimension),
                    "c_value": float(c_value),
                    "monitor_probability": (
                        probability
                    ),
                    "monitor_log_loss": (
                        phase30_loss(
                            monitor_labels,
                            probability,
                        )
                    ),
                    "monitor_auroc": float(
                        roc_auc_score(
                            monitor_labels,
                            probability,
                        )
                    ),
                })

    selected_component = min(
        candidate_records,
        key=lambda record: (
            record["monitor_log_loss"],
            record["dimension"],
            record["c_value"],
            record["representation"],
        ),
    )

    baseline_monitor_probability = (
        phase27_fold_array(
            PHASE24_BASE_MONITOR_PRIVATE,
            fold,
        )
    )

    baseline_monitor_loss = phase30_loss(
        monitor_labels,
        baseline_monitor_probability,
    )

    blend_candidates = []

    for alpha in PHASE30_CONFIG[
        "blend_alphas"
    ]:
        blended_logit = (
            (1.0 - alpha)
            * phase30_logit(
                baseline_monitor_probability
            )
            + alpha
            * phase30_logit(
                selected_component[
                    "monitor_probability"
                ]
            )
        )

        probability = phase30_sigmoid(
            blended_logit
        )

        blend_candidates.append({
            "alpha": float(alpha),
            "monitor_log_loss": (
                phase30_loss(
                    monitor_labels,
                    probability,
                )
            ),
        })

    raw_best_blend = min(
        blend_candidates,
        key=lambda record: (
            record["monitor_log_loss"],
            record["alpha"],
        ),
    )

    monitor_gain = (
        baseline_monitor_loss
        - raw_best_blend[
            "monitor_log_loss"
        ]
    )

    if (
        monitor_gain
        >= PHASE30_CONFIG[
            "minimum_monitor_blend_gain"
        ]
    ):
        selected_alpha = float(
            raw_best_blend["alpha"]
        )
        blend_advanced = (
            selected_alpha > 0.0
        )
    else:
        selected_alpha = 0.0
        blend_advanced = False

    selected_matrix = (
        PHASE30_REPRESENTATIONS_PRIVATE[
            selected_component[
                "representation"
            ]
        ]
    )

    outer_component_probability, state = (
        phase30_fit_pipeline(
            matrix=selected_matrix,
            train_indices=outer_train_indices,
            prediction_indices=outer_valid_indices,
            dimension=selected_component[
                "dimension"
            ],
            c_value=selected_component[
                "c_value"
            ],
            seed=(
                PHASE30_CONFIG["base_seed"]
                + fold
            ),
        )
    )

    baseline_outer_probability = np.asarray(
        PHASE19_PHASE12C_OOF_PRIVATE,
        dtype=np.float64,
    )[outer_valid_indices]

    blended_outer_probability = (
        phase30_sigmoid(
            (1.0 - selected_alpha)
            * phase30_logit(
                baseline_outer_probability
            )
            + selected_alpha
            * phase30_logit(
                outer_component_probability
            )
        )
    )

    PHASE30_COMPONENT_OOF_PRIVATE[
        outer_valid_indices
    ] = outer_component_probability

    PHASE30_BLEND_OOF_PRIVATE[
        outer_valid_indices
    ] = blended_outer_probability

    PHASE30_DEPLOYMENT_STATES_PRIVATE[
        fold
    ] = {
        **state,
        "representation": selected_component[
            "representation"
        ],
        "dimension": int(
            selected_component["dimension"]
        ),
        "c_value": float(
            selected_component["c_value"]
        ),
        "blend_alpha": float(
            selected_alpha
        ),
    }

    phase30_selection_records.append({
        "fold": int(fold),
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(
            len(monitor_indices)
        ),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "candidate_count": int(
            len(candidate_records)
        ),
        "selected_representation": (
            selected_component[
                "representation"
            ]
        ),
        "selected_pca_dimension": int(
            selected_component["dimension"]
        ),
        "selected_logistic_c": float(
            selected_component["c_value"]
        ),
        "component_monitor_log_loss": float(
            selected_component[
                "monitor_log_loss"
            ]
        ),
        "component_monitor_auroc": float(
            selected_component[
                "monitor_auroc"
            ]
        ),
        "baseline_monitor_log_loss": float(
            baseline_monitor_loss
        ),
        "raw_best_blend_alpha": float(
            raw_best_blend["alpha"]
        ),
        "raw_best_blend_monitor_log_loss": float(
            raw_best_blend[
                "monitor_log_loss"
            ]
        ),
        "raw_monitor_blend_gain": float(
            monitor_gain
        ),
        "selected_blend_alpha": float(
            selected_alpha
        ),
        "blend_advanced": bool(
            blend_advanced
        ),
        "elapsed_seconds": round(
            time.perf_counter() - fold_started,
            2,
        ),
    })

    print(
        f"Phase30 fold {fold}: "
        f"representation="
        f"{selected_component['representation']}, "
        f"dim={selected_component['dimension']}, "
        f"C={selected_component['c_value']}, "
        f"alpha={selected_alpha:.2f}"
    )

assert np.isfinite(
    PHASE30_COMPONENT_OOF_PRIVATE
).all()

assert np.isfinite(
    PHASE30_BLEND_OOF_PRIVATE
).all()

report = {
    "phase": "phase30_clinical_view_nested_proxy",
    "status": "complete",
    "configuration": PHASE30_CONFIG,
    "representation_dimensions": {
        name: int(matrix.shape[1])
        for name, matrix
        in PHASE30_REPRESENTATIONS_PRIVATE.items()
    },
    "folds": phase30_selection_records,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase30_training_started,
        2,
    ),
    "monitor_labels_used_for_selection": True,
    "outer_train_labels_used_for_final_refit": True,
    "outer_validation_labels_used": False,
    "selection_frozen_before_outer_evaluation": True,
    "dinov3_backbone_frozen": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_SELECTION")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_SELECTION")

Phase30 fold 0: representation=bilateral_comparisons, dim=128, C=0.1, alpha=0.30
Phase30 fold 1: representation=original_plus_bilateral, dim=64, C=0.1, alpha=0.00
Phase30 fold 2: representation=clinical_all, dim=64, C=0.1, alpha=0.00
BEGIN SANITIZED_PHASE30_SELECTION
{
  "phase": "phase30_clinical_view_nested_proxy",
  "status": "complete",
  "configuration": {
    "phase": "phase30_clinical_view_dinov3",
    "thin_slab_centers": [
      24,
      32,
      40,
      48,
      56
    ],
    "thin_slab_thickness": 24,
    "bilateral_slab_ranges": [
      [
        0,
        40
      ],
      [
        20,
        60
      ],
      [
        40,
        80
      ]
    ],
    "projection_axes": [
      0,
      1,
      2
    ],
    "bilateral_projection_axes": [
      1,
      2
    ],
    "thin_view_count": 15,
    "bilateral_view_count": 6,
    "total_view_count": 21,
    "image_size": 224,
    "extraction_batch_size_cases": 4,
    "pca_dimensions": [
      32,
      64,
      128
   

In [107]:
# Cell 84 — Phase30 one-time outer evaluation

from sklearn.metrics import brier_score_loss


def phase30_metrics(labels, probability):
    probability = np.clip(
        probability,
        1e-5,
        1.0 - 1e-5,
    )

    return {
        "log_loss": float(
            log_loss(
                labels,
                probability,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                labels,
                probability,
            )
        ),
        "brier": float(
            brier_score_loss(
                labels,
                probability,
            )
        ),
    }


phase30_baseline = np.asarray(
    PHASE19_PHASE12C_OOF_PRIVATE,
    dtype=np.float64,
)

phase30_phase26 = np.asarray(
    PHASE26_DINOV3_BLEND_OOF_PRIVATE,
    dtype=np.float64,
)

phase30_component = np.asarray(
    PHASE30_COMPONENT_OOF_PRIVATE,
    dtype=np.float64,
)

phase30_blend = np.asarray(
    PHASE30_BLEND_OOF_PRIVATE,
    dtype=np.float64,
)

baseline_metrics = phase30_metrics(
    phase30_labels,
    phase30_baseline,
)

phase26_metrics = phase30_metrics(
    phase30_labels,
    phase30_phase26,
)

component_metrics = phase30_metrics(
    phase30_labels,
    phase30_component,
)

blend_metrics = phase30_metrics(
    phase30_labels,
    phase30_blend,
)

fold_reports = []
fold_improvements = []

for fold in range(3):
    indices = np.asarray(
        partition_by_fold[fold][
            "outer_valid"
        ],
        dtype=np.int64,
    )

    labels = phase30_labels[indices]

    baseline_fold = phase30_metrics(
        labels,
        phase30_baseline[indices],
    )

    component_fold = phase30_metrics(
        labels,
        phase30_component[indices],
    )

    blend_fold = phase30_metrics(
        labels,
        phase30_blend[indices],
    )

    improvement = (
        baseline_fold["log_loss"]
        - blend_fold["log_loss"]
    )

    fold_improvements.append(improvement)

    selection = phase30_selection_records[fold]

    fold_reports.append({
        "fold": int(fold),
        "n": int(len(indices)),
        "selected_representation": (
            selection[
                "selected_representation"
            ]
        ),
        "selected_blend_alpha": float(
            selection[
                "selected_blend_alpha"
            ]
        ),
        "baseline_log_loss": round(
            baseline_fold["log_loss"],
            6,
        ),
        "component_log_loss": round(
            component_fold["log_loss"],
            6,
        ),
        "blend_log_loss": round(
            blend_fold["log_loss"],
            6,
        ),
        "blend_improvement": round(
            improvement,
            6,
        ),
        "baseline_auroc": round(
            baseline_fold["auroc"],
            6,
        ),
        "component_auroc": round(
            component_fold["auroc"],
            6,
        ),
        "blend_auroc": round(
            blend_fold["auroc"],
            6,
        ),
    })


groups = np.asarray(case_df[group_column])
group_reports = []
group_harms = []

for group in np.unique(groups):
    indices = np.flatnonzero(groups == group)

    if len(indices) < 30:
        continue

    baseline_loss = phase30_metrics(
        phase30_labels[indices],
        phase30_baseline[indices],
    )["log_loss"]

    blend_loss = phase30_metrics(
        phase30_labels[indices],
        phase30_blend[indices],
    )["log_loss"]

    change = blend_loss - baseline_loss
    group_harms.append(change)

    group_reports.append({
        "group": (
            int(group)
            if np.issubdtype(
                np.asarray(group).dtype,
                np.integer,
            )
            else str(group)
        ),
        "n": int(len(indices)),
        "baseline_log_loss": round(
            baseline_loss,
            6,
        ),
        "blend_log_loss": round(
            blend_loss,
            6,
        ),
        "log_loss_change": round(
            change,
            6,
        ),
    })


log_loss_gain = (
    baseline_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

gain_over_phase26 = (
    phase26_metrics["log_loss"]
    - blend_metrics["log_loss"]
)

auroc_gain = (
    blend_metrics["auroc"]
    - baseline_metrics["auroc"]
)

brier_excess = (
    blend_metrics["brier"]
    - baseline_metrics["brier"]
)

fold_wins = int(
    sum(value > 0 for value in fold_improvements)
)

worst_fold_excess = float(
    max(-value for value in fold_improvements)
)

maximum_group_harm = float(
    max(group_harms) if group_harms else 0.0
)

promotion_gate_passed = bool(
    log_loss_gain >= 0.004
    and gain_over_phase26 >= 0.001
    and auroc_gain >= 0.001
    and fold_wins >= 1
    and worst_fold_excess <= 0.005
    and brier_excess <= 0.001
    and maximum_group_harm <= 0.015
)

report = {
    "phase": "phase30_clinical_view_dinov3",
    "status": (
        "promoted"
        if promotion_gate_passed
        else "terminated_retain_phase26"
    ),
    "baseline": {
        key: round(value, 6)
        for key, value in baseline_metrics.items()
    },
    "phase26_champion": {
        key: round(value, 6)
        for key, value in phase26_metrics.items()
    },
    "clinical_view_component": {
        key: round(value, 6)
        for key, value in component_metrics.items()
    },
    "phase30_blend": {
        key: round(value, 6)
        for key, value in blend_metrics.items()
    },
    "improvements": {
        "log_loss_gain_over_phase12c": round(
            log_loss_gain,
            6,
        ),
        "log_loss_gain_over_phase26": round(
            gain_over_phase26,
            6,
        ),
        "auroc_gain_over_phase12c": round(
            auroc_gain,
            6,
        ),
        "brier_excess": round(
            brier_excess,
            6,
        ),
        "fold_wins": fold_wins,
        "worst_fold_excess": round(
            worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            maximum_group_harm,
            6,
        ),
    },
    "fold_metrics": fold_reports,
    "major_acquisition_group_metrics": group_reports,
    "promotion_gate_passed": (
        promotion_gate_passed
    ),
    "dinov3_backbone_frozen": True,
    "dinov3_license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "selection_frozen_before_outer_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_OOF")

BEGIN SANITIZED_PHASE30_OOF
{
  "phase": "phase30_clinical_view_dinov3",
  "status": "promoted",
  "baseline": {
    "log_loss": 0.307615,
    "auroc": 0.938928,
    "brier": 0.095535
  },
  "phase26_champion": {
    "log_loss": 0.304742,
    "auroc": 0.939583,
    "brier": 0.094448
  },
  "clinical_view_component": {
    "log_loss": 0.361247,
    "auroc": 0.918277,
    "brier": 0.112668
  },
  "phase30_blend": {
    "log_loss": 0.298375,
    "auroc": 0.941627,
    "brier": 0.09166
  },
  "improvements": {
    "log_loss_gain_over_phase12c": 0.009241,
    "log_loss_gain_over_phase26": 0.006368,
    "auroc_gain_over_phase12c": 0.002699,
    "brier_excess": -0.003875,
    "fold_wins": 1,
    "worst_fold_excess": -0.0,
    "maximum_major_group_harm": 0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "selected_representation": "bilateral_comparisons",
      "selected_blend_alpha": 0.3,
      "baseline_log_loss": 0.358494,
      "component_log_loss": 0.397218,
      "

In [109]:
# Cell 85 — Sanitized deployment-interface audit.
# Reads source structure and synthetic/training-cache contracts only.
# Does not expose predictions, labels, UIDs, model parameters, or hashes.

import ast
import inspect
import json
import os
from pathlib import Path

import numpy as np
import torch


PHASE30_SOURCE_ROOT = Path(
    "/kaggle/input/models/saifullah42/"
    "dat-phase12c/pytorch/default/1"
)

PHASE30_DINOV3_CHECKPOINT = Path(
    "/kaggle/working/"
    "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
)


def phase30_safe_signature(value):
    try:
        return str(inspect.signature(value))
    except Exception:
        return "signature_unavailable"


def phase30_ast_contract(path):
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    functions = {}

    for node in tree.body:
        if not isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            continue

        calls = []
        referenced_names = []

        for descendant in ast.walk(node):
            if isinstance(descendant, ast.Call):
                function = descendant.func

                if isinstance(function, ast.Name):
                    calls.append(function.id)
                elif isinstance(function, ast.Attribute):
                    calls.append(function.attr)

            elif isinstance(descendant, ast.Name):
                referenced_names.append(
                    descendant.id
                )

        functions[node.name] = {
            "arguments": [
                argument.arg
                for argument in node.args.args
            ],
            "called_functions": sorted(
                set(calls)
            ),
            "referenced_names": sorted(
                set(referenced_names)
            )[:100],
        }

    imports = []

    for node in tree.body:
        if isinstance(node, ast.Import):
            imports.extend(
                alias.name for alias in node.names
            )
        elif isinstance(node, ast.ImportFrom):
            imports.append(
                node.module or ""
            )

    return {
        "file": path.name,
        "line_count": len(
            source.splitlines()
        ),
        "imports": sorted(set(imports)),
        "functions": functions,
    }


def phase30_schema(value, depth=0):
    if depth >= 4:
        return {
            "type": type(value).__name__
        }

    if isinstance(value, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }

    if torch.is_tensor(value):
        return {
            "type": "tensor",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "device": str(value.device),
        }

    if isinstance(value, dict):
        return {
            "type": "dict",
            "length": len(value),
            "keys": [
                str(key)
                for key in list(value.keys())[:40]
            ],
            "fields": {
                str(key): phase30_schema(
                    item,
                    depth + 1,
                )
                for key, item
                in list(value.items())[:15]
            },
        }

    if isinstance(value, (list, tuple)):
        return {
            "type": type(value).__name__,
            "length": len(value),
            "first_element": (
                phase30_schema(
                    value[0],
                    depth + 1,
                )
                if len(value) > 0
                else None
            ),
        }

    if isinstance(
        value,
        (str, int, float, bool, type(None)),
    ):
        return {
            "type": type(value).__name__
        }

    return {
        "type": type(value).__name__,
        "class": value.__class__.__name__,
        "module": value.__class__.__module__,
    }


# ---------------------------------------------------------
# Source and asset contract
# ---------------------------------------------------------

source_contracts = []

for filename in [
    "main.py",
    "base_main.py",
]:
    path = PHASE30_SOURCE_ROOT / filename

    assert path.is_file(), path
    source_contracts.append(
        phase30_ast_contract(path)
    )

asset_contract = {
    "phase12c_source_root_exists": (
        PHASE30_SOURCE_ROOT.is_dir()
    ),
    "phase12c_main_exists": (
        PHASE30_SOURCE_ROOT.joinpath(
            "main.py"
        ).is_file()
    ),
    "phase12c_base_main_exists": (
        PHASE30_SOURCE_ROOT.joinpath(
            "base_main.py"
        ).is_file()
    ),
    "phase12c_manifest_exists": (
        PHASE30_SOURCE_ROOT.joinpath(
            "manifest.json"
        ).is_file()
    ),
    "phase12c_base_manifest_exists": (
        PHASE30_SOURCE_ROOT.joinpath(
            "base_manifest.json"
        ).is_file()
    ),
    "dinov3_checkpoint_exists": (
        PHASE30_DINOV3_CHECKPOINT.is_file()
    ),
    "dinov3_checkpoint_size_mb": (
        round(
            PHASE30_DINOV3_CHECKPOINT.stat().st_size
            / (1024 ** 2),
            2,
        )
        if PHASE30_DINOV3_CHECKPOINT.is_file()
        else None
    ),
}


# ---------------------------------------------------------
# Fold-manifest schemas
# ---------------------------------------------------------

fold_manifest_contracts = []

for fold, fold_manifest in enumerate(
    PHASE19_FOLD_MANIFESTS
):
    fold_manifest_contracts.append({
        "fold": int(fold),
        "schema": phase30_schema(
            fold_manifest
        ),
    })


# ---------------------------------------------------------
# Runtime helper contracts
# ---------------------------------------------------------

helper_names = [
    "build_predictor_crops",
    "phase19_adaptive_predict_family_batch",
    "predict_family_batch",
    "phase26_resolve_highres_cache",
    "phase26_make_views",
    "preprocess_case",
    "preprocess_safely",
    "load_canonical_volume",
    "robust_uptake_center",
    "resample_physical_cube",
    "percentile_normalize",
]

helper_contracts = []

for name in helper_names:
    if name in globals():
        helper_contracts.append({
            "name": name,
            "signature": phase30_safe_signature(
                globals()[name]
            ),
            "module": getattr(
                globals()[name],
                "__module__",
                None,
            ),
        })


# Find additional preprocessing helpers without printing source.
preprocessing_candidates = []

for name, value in list(globals().items()):
    lowered = name.lower()

    if not callable(value):
        continue

    if any(
        token in lowered
        for token in [
            "canonical",
            "preprocess",
            "resample",
            "highres",
            "uptake_center",
        ]
    ):
        preprocessing_candidates.append({
            "name": name,
            "signature": (
                phase30_safe_signature(value)
            ),
            "module": getattr(
                value,
                "__module__",
                None,
            ),
        })

preprocessing_candidates = sorted(
    preprocessing_candidates,
    key=lambda record: record["name"],
)


# ---------------------------------------------------------
# Fold-specific predictor smoke contract
# ---------------------------------------------------------

predictor_records = []
predictor_error = None

try:
    contract_indices = np.asarray(
        [0, 1],
        dtype=np.int64,
    )

    contract_crops = build_predictor_crops(
        contract_indices
    )

    for fold in range(3):
        fold_manifest = (
            PHASE19_FOLD_MANIFESTS[fold]
        )

        family_manifest = (
            fold_manifest["family"]
            if "family" in fold_manifest
            else fold_manifest
        )

        base_manifest = (
            fold_manifest["base"]
            if "base" in fold_manifest
            else PHASE19_BASE_MANIFEST_RUNTIME
        )

        result = (
            phase19_adaptive_predict_family_batch(
                crops=contract_crops,
                family_manifest=family_manifest,
                base_manifest=base_manifest,
                base_models=PHASE19_BASE_MODELS,
                anatomy_models=(
                    PHASE19_ANATOMY_MODELS
                ),
                device=phase19_device,
            )
        )

        predictor_records.append({
            "fold": int(fold),
            "result_schema": phase30_schema(
                result
            ),
            "all_values_finite": bool(
                np.isfinite(
                    np.asarray(result)
                ).all()
            ),
        })

except Exception as exception:
    predictor_error = {
        "error_type": type(exception).__name__,
        "message_length": len(
            str(exception)
        ),
    }


# ---------------------------------------------------------
# Phase30 deployment-state contract
# ---------------------------------------------------------

deployment_contracts = []

for fold in range(3):
    state = PHASE30_DEPLOYMENT_STATES_PRIVATE[
        fold
    ]

    deployment_contracts.append({
        "fold": int(fold),
        "representation": state[
            "representation"
        ],
        "dimension": int(
            state["dimension"]
        ),
        "c_value": float(
            state["c_value"]
        ),
        "blend_alpha": float(
            state["blend_alpha"]
        ),
        "scaler_mean_shape": list(
            state["scaler"].mean_.shape
        ),
        "pca_component_shape": list(
            state["pca"].components_.shape
        ),
        "classifier_coefficient_shape": list(
            state["classifier"].coef_.shape
        ),
    })


# Relevant configuration globals, without arrays or paths.
configuration_contracts = {}

for name in [
    "PHASE18_CACHE_CONFIG",
    "PHASE18_PREPROCESS_CONFIG",
    "PHASE19_CROP_CONFIG",
    "PHASE19_PREPROCESS_CONFIG",
    "PHASE30_CONFIG",
]:
    if name in globals():
        value = globals()[name]

        if isinstance(value, dict):
            safe_fields = {}

            for key, item in value.items():
                if isinstance(
                    item,
                    (
                        str,
                        int,
                        float,
                        bool,
                        type(None),
                    ),
                ):
                    safe_fields[str(key)] = item
                elif isinstance(item, (list, tuple)):
                    if all(
                        isinstance(
                            element,
                            (
                                str,
                                int,
                                float,
                                bool,
                            ),
                        )
                        for element in item
                    ):
                        safe_fields[str(key)] = list(
                            item
                        )
                elif isinstance(item, dict):
                    safe_fields[str(key)] = {
                        str(nested_key): nested_value
                        for nested_key, nested_value
                        in item.items()
                        if isinstance(
                            nested_value,
                            (
                                str,
                                int,
                                float,
                                bool,
                            ),
                        )
                    }

            configuration_contracts[name] = (
                safe_fields
            )


report = {
    "phase": "phase30_deployment_interface_audit",
    "status": (
        "accepted"
        if (
            predictor_error is None
            and len(predictor_records) == 3
            and asset_contract[
                "dinov3_checkpoint_exists"
            ]
        )
        else "incomplete"
    ),
    "assets": asset_contract,
    "source_contracts": source_contracts,
    "fold_manifest_contracts": (
        fold_manifest_contracts
    ),
    "helper_contracts": helper_contracts,
    "preprocessing_callable_candidates": (
        preprocessing_candidates
    ),
    "fold_predictor_contracts": (
        predictor_records
    ),
    "fold_predictor_error": predictor_error,
    "phase30_deployment_contracts": (
        deployment_contracts
    ),
    "configuration_contracts": (
        configuration_contracts
    ),
    "training_cache_shape": list(
        PHASE19_HIGHRES_CACHE.shape
    ),
    "training_cache_dtype": str(
        PHASE19_HIGHRES_CACHE.dtype
    ),
    "source_executed": False,
    "training_voxel_cache_read_for_synthetic_contract": (
        predictor_error is None
    ),
    "labels_read": False,
    "outer_validation_labels_used": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
    "model_hashes_displayed": False,
}

print("BEGIN SANITIZED_PHASE30_DEPLOYMENT_AUDIT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_DEPLOYMENT_AUDIT")

BEGIN SANITIZED_PHASE30_DEPLOYMENT_AUDIT
{
  "phase": "phase30_deployment_interface_audit",
  "status": "incomplete",
  "assets": {
    "phase12c_source_root_exists": true,
    "phase12c_main_exists": true,
    "phase12c_base_main_exists": true,
    "phase12c_manifest_exists": true,
    "phase12c_base_manifest_exists": true,
    "dinov3_checkpoint_exists": true,
    "dinov3_checkpoint_size_mb": 82.52
  },
  "source_contracts": [
    {
      "file": "main.py",
      "line_count": 318,
      "imports": [
        "__future__",
        "base_main",
        "concurrent.futures",
        "csv",
        "hashlib",
        "json",
        "math",
        "numpy",
        "os",
        "pathlib",
        "torch",
        "torch.nn",
        "torch.nn.functional"
      ],
      "functions": {
        "file_sha256": {
          "arguments": [
            "path"
          ],
          "called_functions": [
            "hexdigest",
            "iter",
            "open",
            "read",
       

In [110]:
# Cell 86 — Resolve exact runtime model containers and
# acquisition-group construction contracts.

import inspect
import json
import re

import numpy as np
import torch
import torch.nn as nn


def phase31_signature(value):
    try:
        return str(inspect.signature(value))
    except Exception:
        return "signature_unavailable"


def phase31_count_modules(value):
    observed = set()
    classes = {}

    def visit(item):
        if isinstance(item, nn.Module):
            identifier = id(item)

            if identifier not in observed:
                observed.add(identifier)
                class_name = (
                    item.__class__.__name__
                )
                classes[class_name] = (
                    classes.get(class_name, 0)
                    + 1
                )
            return

        if isinstance(item, dict):
            for nested in item.values():
                visit(nested)

        elif isinstance(item, (list, tuple)):
            for nested in item:
                visit(nested)

    visit(value)

    return {
        "model_count": len(observed),
        "class_counts": classes,
    }


def phase31_schema(value, depth=0):
    if depth >= 3:
        return {
            "type": type(value).__name__
        }

    if isinstance(value, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }

    if torch.is_tensor(value):
        return {
            "type": "tensor",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }

    if isinstance(value, dict):
        keys = list(value.keys())

        return {
            "type": "dict",
            "length": len(keys),
            "keys": [
                str(key) for key in keys[:40]
            ],
            "first_fields": {
                str(key): phase31_schema(
                    value[key],
                    depth + 1,
                )
                for key in keys[:8]
            },
        }

    if isinstance(value, (list, tuple)):
        return {
            "type": type(value).__name__,
            "length": len(value),
            "first_element": (
                phase31_schema(
                    value[0],
                    depth + 1,
                )
                if len(value) > 0
                else None
            ),
        }

    if isinstance(value, nn.Module):
        return {
            "type": "torch_module",
            "class": value.__class__.__name__,
        }

    return {
        "type": type(value).__name__,
        "class": value.__class__.__name__,
        "module": value.__class__.__module__,
    }


# ---------------------------------------------------------
# Find original Phase12c model collections.
# ---------------------------------------------------------

model_container_candidates = []

for name, value in list(globals().items()):
    if not isinstance(
        value,
        (dict, list, tuple),
    ):
        continue

    contract = phase31_count_modules(value)

    if contract["model_count"] in (
        6, 15, 21
    ):
        model_container_candidates.append({
            "name": name,
            **contract,
            "schema": phase31_schema(value),
        })

model_container_candidates = sorted(
    model_container_candidates,
    key=lambda record: (
        record["model_count"],
        record["name"],
    ),
)


def phase31_resolve_model_container(
    desired_count,
    desired_class,
    preferred_tokens,
):
    candidates = []

    for name, value in list(globals().items()):
        if not isinstance(
            value,
            (dict, list, tuple),
        ):
            continue

        contract = phase31_count_modules(value)

        if (
            contract["model_count"]
            == desired_count
            and contract["class_counts"].get(
                desired_class,
                0,
            ) == desired_count
        ):
            score = sum(
                token in name.lower()
                for token in preferred_tokens
            )

            candidates.append(
                (score, name, value)
            )

    assert candidates, {
        "desired_count": desired_count,
        "desired_class": desired_class,
    }

    candidates.sort(
        key=lambda item: (
            -item[0],
            item[1],
        )
    )

    return candidates[0][1], candidates[0][2]


resolved_base_name, resolved_base_models = (
    phase31_resolve_model_container(
        desired_count=15,
        desired_class="CompactDaTNet",
        preferred_tokens=[
            "phase19",
            "base",
            "original",
        ],
    )
)

resolved_anatomy_name, resolved_anatomy_models = (
    phase31_resolve_model_container(
        desired_count=6,
        desired_class="AnatomyTokenDaTNet",
        preferred_tokens=[
            "phase19",
            "anatomy",
            "original",
        ],
    )
)

PHASE31_BASE_MODELS_PRIVATE = (
    resolved_base_models
)

PHASE31_ANATOMY_MODELS_PRIVATE = (
    resolved_anatomy_models
)

phase31_device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


# ---------------------------------------------------------
# Validate exact fold-specific Phase12c predictor.
# ---------------------------------------------------------

predictor_records = []
predictor_error = None

try:
    contract_indices = np.asarray(
        [0, 1],
        dtype=np.int64,
    )

    contract_crops = build_predictor_crops(
        contract_indices
    )

    for fold in range(3):
        fold_manifest = (
            PHASE19_FOLD_MANIFESTS[fold]
        )

        result = (
            phase19_adaptive_predict_family_batch(
                contract_crops,
                fold_manifest["family"],
                fold_manifest["base"],
                PHASE31_BASE_MODELS_PRIVATE,
                PHASE31_ANATOMY_MODELS_PRIVATE,
                phase31_device,
            )
        )

        result_array = np.asarray(
            result,
            dtype=np.float64,
        )

        predictor_records.append({
            "fold": int(fold),
            "result_shape": list(
                result_array.shape
            ),
            "result_dtype": str(
                result_array.dtype
            ),
            "all_values_finite": bool(
                np.isfinite(
                    result_array
                ).all()
            ),
            "result_in_probability_range": bool(
                np.all(
                    (result_array >= 0.0)
                    & (result_array <= 1.0)
                )
            ),
        })

except Exception as exception:
    predictor_error = {
        "error_type": type(exception).__name__,
        "message_length": len(
            str(exception)
        ),
    }


# ---------------------------------------------------------
# Acquisition/group metadata schema.
# ---------------------------------------------------------

case_dataframe_contract = {
    "column_count": int(
        len(case_df.columns)
    ),
    "columns": [],
}

for column in case_df.columns:
    series = case_df[column]

    case_dataframe_contract[
        "columns"
    ].append({
        "name": str(column),
        "dtype": str(series.dtype),
        "unique_count": int(
            series.nunique(
                dropna=False
            )
        ),
        "missing_count": int(
            series.isna().sum()
        ),
    })


acquisition_global_candidates = []

for name, value in list(globals().items()):
    lowered = name.lower()

    if not any(
        token in lowered
        for token in [
            "acquisition",
            "header",
            "group",
            "scanner",
            "spacing",
            "affine",
        ]
    ):
        continue

    # Skip functions here; they are listed separately.
    if callable(value):
        continue

    acquisition_global_candidates.append({
        "name": name,
        "schema": phase31_schema(value),
    })

acquisition_global_candidates = sorted(
    acquisition_global_candidates,
    key=lambda record: record["name"],
)[:80]


acquisition_callable_candidates = []

for name, value in list(globals().items()):
    lowered = name.lower()

    if not callable(value):
        continue

    if any(
        token in lowered
        for token in [
            "acquisition",
            "header",
            "group",
            "scanner",
            "spacing",
            "affine",
        ]
    ):
        acquisition_callable_candidates.append({
            "name": name,
            "signature": (
                phase31_signature(value)
            ),
            "module": getattr(
                value,
                "__module__",
                None,
            ),
        })

acquisition_callable_candidates = sorted(
    acquisition_callable_candidates,
    key=lambda record: record["name"],
)


# ---------------------------------------------------------
# Fold-to-acquisition-group aggregate contract.
# ---------------------------------------------------------

group_column_candidates = [
    column
    for column in case_df.columns
    if (
        "acquisition" in column.lower()
        and "group" in column.lower()
    )
]

if "acquisition_group" in case_df.columns:
    phase31_group_column = (
        "acquisition_group"
    )
elif len(group_column_candidates) == 1:
    phase31_group_column = (
        group_column_candidates[0]
    )
else:
    phase31_group_column = None

fold_group_contracts = []

if phase31_group_column is not None:
    groups = np.asarray(
        case_df[phase31_group_column]
    )

    for fold in range(3):
        partition = PHASE19_PARTITIONS[fold]

        fit_groups = np.unique(
            groups[
                np.asarray(
                    partition["fit"],
                    dtype=np.int64,
                )
            ]
        )

        monitor_groups = np.unique(
            groups[
                np.asarray(
                    partition["monitor"],
                    dtype=np.int64,
                )
            ]
        )

        outer_groups = np.unique(
            groups[
                np.asarray(
                    partition["outer_valid"],
                    dtype=np.int64,
                )
            ]
        )

        fold_group_contracts.append({
            "fold": int(fold),
            "fit_group_count": int(
                len(fit_groups)
            ),
            "monitor_group_count": int(
                len(monitor_groups)
            ),
            "outer_valid_group_count": int(
                len(outer_groups)
            ),
            "outer_valid_group_ids": [
                (
                    int(value)
                    if np.issubdtype(
                        np.asarray(value).dtype,
                        np.integer,
                    )
                    else str(value)
                )
                for value in outer_groups
            ],
        })


# ---------------------------------------------------------
# Inspect NIfTI-header feature columns without values.
# ---------------------------------------------------------

probable_header_columns = [
    str(column)
    for column in case_df.columns
    if any(
        token in str(column).lower()
        for token in [
            "shape",
            "spacing",
            "zoom",
            "orientation",
            "affine",
            "voxel",
            "dimension",
            "dtype",
        ]
    )
]


report = {
    "phase": "phase31_runtime_and_router_resolution",
    "status": (
        "accepted"
        if (
            predictor_error is None
            and len(predictor_records) == 3
            and phase31_group_column is not None
        )
        else "incomplete"
    ),
    "resolved_model_containers": {
        "base": {
            "name": resolved_base_name,
            "model_count": 15,
            "class": "CompactDaTNet",
        },
        "anatomy": {
            "name": resolved_anatomy_name,
            "model_count": 6,
            "class": "AnatomyTokenDaTNet",
        },
    },
    "all_model_container_candidates": (
        model_container_candidates
    ),
    "fold_predictor_contracts": (
        predictor_records
    ),
    "fold_predictor_error": predictor_error,
    "case_dataframe_contract": (
        case_dataframe_contract
    ),
    "acquisition_global_candidates": (
        acquisition_global_candidates
    ),
    "acquisition_callable_candidates": (
        acquisition_callable_candidates
    ),
    "group_column": phase31_group_column,
    "fold_group_contracts": (
        fold_group_contracts
    ),
    "probable_header_columns": (
        probable_header_columns
    ),
    "phase30_corrected_outer_groups": (
        fold_group_contracts[0][
            "outer_valid_group_ids"
        ]
        if fold_group_contracts
        else None
    ),
    "challenge_values_exported": False,
    "patient_rows_displayed": False,
    "labels_read": False,
    "test_or_smoke_data_read": False,
    "uids_displayed": False,
    "model_hashes_displayed": False,
}

print("BEGIN SANITIZED_PHASE31_ROUTER_AUDIT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_ROUTER_AUDIT")

BEGIN SANITIZED_PHASE31_ROUTER_AUDIT
{
  "phase": "phase31_runtime_and_router_resolution",
  "status": "incomplete",
  "resolved_model_containers": {
    "base": {
      "name": "PHASE19_BASE_MODELS",
      "model_count": 15,
      "class": "CompactDaTNet"
    },
    "anatomy": {
      "name": "PHASE19_ANATOMY_MODELS",
      "model_count": 6,
      "class": "AnatomyTokenDaTNet"
    }
  },
  "all_model_container_candidates": [
    {
      "name": "PHASE19_ANATOMY_MODELS",
      "model_count": 6,
      "class_counts": {
        "AnatomyTokenDaTNet": 6
      },
      "schema": {
        "type": "dict",
        "length": 6,
        "keys": [
          "(0, 'seed_a')",
          "(0, 'seed_b')",
          "(1, 'seed_a')",
          "(1, 'seed_b')",
          "(2, 'seed_a')",
          "(2, 'seed_b')"
        ],
        "first_fields": {
          "(0, 'seed_a')": {
            "type": "torch_module",
            "class": "AnatomyTokenDaTNet"
          },
          "(0, 'seed_b')": {
       

In [112]:
# Cell 87 — Create a source-only Phase30 deployment handoff.
# Safe to attach: no images, labels, predictions, embeddings,
# model weights, patient rows, or model hashes are printed.

import inspect
import json
import zipfile
from pathlib import Path


source_root = Path(
    "/kaggle/input/models/saifullah42/"
    "dat-phase12c/pytorch/default/1"
)

handoff_root = Path(
    "/kaggle/working/"
    "phase30_source_handoff"
)

handoff_root.mkdir(
    parents=True,
    exist_ok=True,
)

required_files = [
    "main.py",
    "base_main.py",
    "manifest.json",
    "base_manifest.json",
]

optional_files = [
    "README.md",
    "LICENSE",
    "THIRD_PARTY_LICENSES.md",
    "asset_license_registry.json",
]

copied_files = []

for filename in required_files + optional_files:
    source = source_root / filename

    if not source.is_file():
        if filename in required_files:
            raise FileNotFoundError(source)
        continue

    destination = handoff_root / filename
    destination.write_bytes(
        source.read_bytes()
    )

    copied_files.append(filename)


# Export only the acquisition-header function source and
# non-sensitive column/configuration names.
header_source = inspect.getsource(
    header_features
)

(
    handoff_root
    / "header_features_source.py.txt"
).write_text(
    header_source,
    encoding="utf-8",
)

runtime_contract = {
    "schema_version": 1,
    "header_columns": [
        str(column)
        for column in HEADER_COLS
    ],
    "acquisition_group_count": 15,
    "phase30_corrected_groups": [
        1, 5, 9
    ],
    "outer_group_routes": {
        "fold_0": [1, 5, 9],
        "fold_1": [
            0, 2, 7, 11, 12, 13
        ],
        "fold_2": [
            3, 4, 6, 8, 10, 14
        ],
    },
    "phase30": {
        "representation": (
            "bilateral_comparisons"
        ),
        "pca_dimension": 128,
        "logistic_c": 0.1,
        "blend_alpha": 0.3,
        "clinical_view_count": 6,
        "dinov3_architecture": (
            "dinov3_vits16"
        ),
        "dinov3_pretraining": "LVD1689M",
        "license": "DINOv3 License",
        "required_attribution": (
            "Built with DINOv3"
        ),
    },
    "resolved_runtime_objects": {
        "base_models": (
            "PHASE19_BASE_MODELS"
        ),
        "anatomy_models": (
            "PHASE19_ANATOMY_MODELS"
        ),
        "family_predictor": (
            "predict_family_batch"
        ),
    },
    "contains_patient_data": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_model_weights": False,
}

(
    handoff_root
    / "runtime_contract.json"
).write_text(
    json.dumps(
        runtime_contract,
        indent=2,
    ),
    encoding="utf-8",
)


zip_path = Path(
    "/kaggle/working/"
    "phase30_source_handoff.zip"
)

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for path in sorted(
        handoff_root.iterdir()
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.name,
            )


report = {
    "phase": "phase30_source_handoff",
    "status": "complete",
    "zip_name": zip_path.name,
    "zip_size_kb": round(
        zip_path.stat().st_size / 1024,
        2,
    ),
    "included_files": sorted(
        [
            path.name
            for path
            in handoff_root.iterdir()
            if path.is_file()
        ]
    ),
    "contains_model_weights": False,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "model_hashes_displayed": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_SOURCE_HANDOFF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_SOURCE_HANDOFF")
print()

BEGIN SANITIZED_PHASE30_SOURCE_HANDOFF
{
  "phase": "phase30_source_handoff",
  "status": "complete",
  "zip_name": "phase30_source_handoff.zip",
  "zip_size_kb": 12.84,
  "included_files": [
    "LICENSE",
    "README.md",
    "THIRD_PARTY_LICENSES.md",
    "asset_license_registry.json",
    "base_main.py",
    "base_manifest.json",
    "header_features_source.py.txt",
    "main.py",
    "manifest.json",
    "runtime_contract.json"
  ],
  "contains_model_weights": false,
  "contains_voxel_data": false,
  "contains_patient_rows": false,
  "contains_labels": false,
  "contains_predictions": false,
  "contains_embeddings": false,
  "model_hashes_displayed": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE30_SOURCE_HANDOFF



In [116]:
# Corrected Cell 88 — Idempotent Phase30 staging setup.

from pathlib import Path
import json
import shutil
import time

phase30_package_started = time.perf_counter()

PHASE30_PACKAGE_ROOT = Path(
    "/kaggle/working/phase30_submission"
).resolve()

PHASE30_PACKAGE_MODELS = (
    PHASE30_PACKAGE_ROOT / "models"
)

# ---------------------------------------------------------
# 1. Resolve Phase12c source without using stale variables.
# ---------------------------------------------------------

preferred_phase12c_root = Path(
    "/kaggle/input/models/saifullah42/"
    "dat-phase12c/pytorch/default/1"
).resolve()

def is_valid_phase12c_root(path):
    path = Path(path)

    return (
        path.is_dir()
        and (path / "main.py").is_file()
        and (path / "base_main.py").is_file()
        and (path / "manifest.json").is_file()
        and (path / "base_manifest.json").is_file()
        and (path / "models").is_dir()
    )

if is_valid_phase12c_root(preferred_phase12c_root):
    PHASE12C_SOURCE_ROOT = preferred_phase12c_root
else:
    discovered_phase12c_roots = {
        path.resolve()
        for path in Path("/kaggle/input").glob(
            "**/dat-phase12c/pytorch/default/1"
        )
        if is_valid_phase12c_root(path)
    }

    assert len(discovered_phase12c_roots) == 1, {
        "message": "Could not uniquely resolve the Phase12c source.",
        "valid_candidate_count":
            len(discovered_phase12c_roots),
    }

    PHASE12C_SOURCE_ROOT = next(
        iter(discovered_phase12c_roots)
    )

assert is_valid_phase12c_root(PHASE12C_SOURCE_ROOT)

# ---------------------------------------------------------
# 2. Resolve the original DINOv3 checkpoint.
#
# Exclude phase30_submission because an interrupted previous
# run may already have copied the checkpoint there.
# ---------------------------------------------------------

preferred_dinov3_checkpoint = Path(
    "/kaggle/working/"
    "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
).resolve()

def is_valid_dinov3_checkpoint(path):
    path = Path(path)

    return (
        path.is_file()
        and path.name
        == "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
        and path.stat().st_size > 80_000_000
    )

if is_valid_dinov3_checkpoint(
    preferred_dinov3_checkpoint
):
    PHASE30_DINOV3_CHECKPOINT_SOURCE = (
        preferred_dinov3_checkpoint
    )
else:
    discovered_checkpoints = set()

    for path in Path("/kaggle/working").glob(
        "**/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
    ):
        resolved_path = path.resolve()

        # Ignore a partially created deployment copy.
        if (
            resolved_path == PHASE30_PACKAGE_ROOT
            or PHASE30_PACKAGE_ROOT
            in resolved_path.parents
        ):
            continue

        if is_valid_dinov3_checkpoint(resolved_path):
            discovered_checkpoints.add(resolved_path)

    assert discovered_checkpoints, {
        "message": "No valid original DINOv3 checkpoint found."
    }

    # Multiple identical cache copies are harmless. Prefer the
    # shortest path outside the deployment directory.
    PHASE30_DINOV3_CHECKPOINT_SOURCE = sorted(
        discovered_checkpoints,
        key=lambda path: (
            len(path.parts),
            len(str(path)),
            str(path),
        ),
    )[0]

assert is_valid_dinov3_checkpoint(
    PHASE30_DINOV3_CHECKPOINT_SOURCE
)

# ---------------------------------------------------------
# 3. Resolve the local DINOv3 repository.
# ---------------------------------------------------------

preferred_repo_roots = [
    Path(
        "/kaggle/working/torch_hub_phase26_v3/"
        "facebookresearch_dinov3_main"
    ),
    Path(
        "/kaggle/working/torch_hub_phase26_v3/"
        "facebookresearch_dinov3_main"
    ),
]

def is_valid_dinov3_repo(path):
    path = Path(path)

    return (
        path.is_dir()
        and (path / "hubconf.py").is_file()
        and (path / "dinov3").is_dir()
    )

PHASE30_DINOV3_REPO_SOURCE = next(
    (
        path.resolve()
        for path in preferred_repo_roots
        if is_valid_dinov3_repo(path)
    ),
    None,
)

if PHASE30_DINOV3_REPO_SOURCE is None:
    discovered_repositories = set()

    for search_root in [
        Path("/kaggle/working/torch_hub_phase26_v3"),
        Path("/kaggle/working"),
    ]:
        if not search_root.exists():
            continue

        for path in search_root.glob(
            "facebookresearch_dinov3_*"
        ):
            resolved_path = path.resolve()

            if (
                resolved_path == PHASE30_PACKAGE_ROOT
                or PHASE30_PACKAGE_ROOT
                in resolved_path.parents
            ):
                continue

            if is_valid_dinov3_repo(resolved_path):
                discovered_repositories.add(resolved_path)

    assert discovered_repositories, {
        "message": "No valid local DINOv3 source repository found."
    }

    PHASE30_DINOV3_REPO_SOURCE = sorted(
        discovered_repositories,
        key=lambda path: (
            len(path.parts),
            len(str(path)),
            str(path),
        ),
    )[0]

assert is_valid_dinov3_repo(
    PHASE30_DINOV3_REPO_SOURCE
)

# ---------------------------------------------------------
# 4. Safely recreate the explicit staging directory.
# ---------------------------------------------------------

assert str(PHASE30_PACKAGE_ROOT).startswith(
    "/kaggle/working/"
)

assert PHASE30_PACKAGE_ROOT.name == (
    "phase30_submission"
)

if PHASE30_PACKAGE_ROOT.exists():
    shutil.rmtree(PHASE30_PACKAGE_ROOT)

shutil.copytree(
    PHASE12C_SOURCE_ROOT,
    PHASE30_PACKAGE_ROOT,
)

PHASE30_PACKAGE_MODELS.mkdir(
    parents=True,
    exist_ok=True,
)

# Preserve the verified Phase12c entrypoint.
shutil.copy2(
    PHASE30_PACKAGE_ROOT / "main.py",
    PHASE30_PACKAGE_ROOT / "phase12c_main.py",
)

# Copy the DINOv3 checkpoint once.
PHASE30_DINOV3_CHECKPOINT = (
    PHASE30_PACKAGE_MODELS
    / "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
)

shutil.copy2(
    PHASE30_DINOV3_CHECKPOINT_SOURCE,
    PHASE30_DINOV3_CHECKPOINT,
)

# Copy the offline DINOv3 source.
PHASE30_DINOV3_REPO = (
    PHASE30_PACKAGE_ROOT / "dinov3_repo"
)

shutil.copytree(
    PHASE30_DINOV3_REPO_SOURCE,
    PHASE30_DINOV3_REPO,
    ignore=shutil.ignore_patterns(
        ".git",
        "__pycache__",
        "*.pyc",
        "*.pyo",
        ".pytest_cache",
        "checkpoints",
    ),
)

# Copy the full DINOv3 license.
dinov3_license_source = next(
    (
        path
        for path in [
            PHASE30_DINOV3_REPO / "LICENSE.md",
            PHASE30_DINOV3_REPO / "LICENSE",
        ]
        if path.is_file()
    ),
    None,
)

assert dinov3_license_source is not None, (
    "DINOv3 license file was not found in the source repository."
)

shutil.copy2(
    dinov3_license_source,
    PHASE30_PACKAGE_ROOT / "DINOv3_LICENSE.md",
)

# ---------------------------------------------------------
# 5. Final staging assertions.
# ---------------------------------------------------------

assert (
    PHASE30_PACKAGE_ROOT / "phase12c_main.py"
).is_file()

assert (
    PHASE30_PACKAGE_ROOT / "base_main.py"
).is_file()

assert (
    PHASE30_PACKAGE_ROOT / "manifest.json"
).is_file()

assert (
    PHASE30_PACKAGE_ROOT / "base_manifest.json"
).is_file()

assert is_valid_dinov3_checkpoint(
    PHASE30_DINOV3_CHECKPOINT
)

assert (
    PHASE30_DINOV3_REPO / "hubconf.py"
).is_file()

assert (
    PHASE30_PACKAGE_ROOT / "DINOv3_LICENSE.md"
).is_file()

phase12_weight_count = len(
    list(PHASE30_PACKAGE_MODELS.glob("*.pt"))
)

assert phase12_weight_count == 21, (
    phase12_weight_count
)

report = {
    "phase": "phase30_corrected_staging_setup",
    "status": "accepted",
    "phase12_source_resolved": True,
    "phase12_model_count": phase12_weight_count,
    "legacy_entrypoint_preserved": True,
    "dinov3_checkpoint_count": 1,
    "dinov3_source_present": True,
    "dinov3_license_present": True,
    "new_runtime_entrypoint_written": False,
    "model_hashes_displayed": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_CORRECTED_STAGING")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_CORRECTED_STAGING")
print("\nCorrected Cell 88 accepted. Now run Cell 89.")

BEGIN SANITIZED_PHASE30_CORRECTED_STAGING
{
  "phase": "phase30_corrected_staging_setup",
  "status": "accepted",
  "phase12_source_resolved": true,
  "phase12_model_count": 21,
  "legacy_entrypoint_preserved": true,
  "dinov3_checkpoint_count": 1,
  "dinov3_source_present": true,
  "dinov3_license_present": true,
  "new_runtime_entrypoint_written": false,
  "model_hashes_displayed": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE30_CORRECTED_STAGING

Corrected Cell 88 accepted. Now run Cell 89.


In [117]:
# Cell 89 — Export the exact promoted fold-0 probe and router.

import hashlib
import json
import numpy as np
from pathlib import Path

required_globals = [
    "PHASE30_DEPLOYMENT_STATES_PRIVATE",
    "case_df",
    "HEADER_COLS",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

assert not missing_globals, {
    "message": "Required notebook state is missing.",
    "missing_count": len(missing_globals),
}

assert len(PHASE30_DEPLOYMENT_STATES_PRIVATE) == 3

phase30_fold0_state = PHASE30_DEPLOYMENT_STATES_PRIVATE[0]

assert phase30_fold0_state["representation"] == "bilateral_comparisons"
assert int(phase30_fold0_state["dimension"]) == 128
assert np.isclose(
    float(phase30_fold0_state["blend_alpha"]),
    0.30,
)

phase30_scaler = phase30_fold0_state["scaler"]
phase30_pca = phase30_fold0_state["pca"]
phase30_classifier = phase30_fold0_state["classifier"]

assert phase30_scaler.mean_.shape == (2304,)
assert phase30_scaler.scale_.shape == (2304,)
assert phase30_pca.mean_.shape == (2304,)
assert phase30_pca.components_.shape == (128, 2304)
assert phase30_pca.explained_variance_.shape == (128,)
assert phase30_classifier.coef_.shape == (1, 128)
assert phase30_classifier.intercept_.shape == (1,)
assert bool(phase30_pca.whiten)

probe_path = (
    PHASE30_PACKAGE_MODELS
    / "phase30_fold0_bilateral_probe.npz"
)

np.savez_compressed(
    probe_path,
    scaler_mean=np.asarray(
        phase30_scaler.mean_,
        dtype=np.float32,
    ),
    scaler_scale=np.asarray(
        phase30_scaler.scale_,
        dtype=np.float32,
    ),
    pca_mean=np.asarray(
        phase30_pca.mean_,
        dtype=np.float32,
    ),
    pca_components=np.asarray(
        phase30_pca.components_,
        dtype=np.float32,
    ),
    pca_explained_variance=np.asarray(
        phase30_pca.explained_variance_,
        dtype=np.float32,
    ),
    classifier_coef=np.asarray(
        phase30_classifier.coef_,
        dtype=np.float32,
    ),
    classifier_intercept=np.asarray(
        phase30_classifier.intercept_,
        dtype=np.float32,
    ),
    blend_alpha=np.asarray([0.30], dtype=np.float32),
    feature_dimension=np.asarray([2304], dtype=np.int64),
    reduced_dimension=np.asarray([128], dtype=np.int64),
)

# Verify the exported NumPy probe against sklearn.
rng = np.random.default_rng(300601)
contract_features = rng.normal(
    size=(4, 2304)
).astype(np.float64)

sklearn_probability = phase30_classifier.predict_proba(
    phase30_pca.transform(
        phase30_scaler.transform(contract_features)
    )
)[:, 1]

with np.load(probe_path) as exported:
    scaled = (
        contract_features - exported["scaler_mean"]
    ) / exported["scaler_scale"]

    reduced = (
        scaled - exported["pca_mean"]
    ) @ exported["pca_components"].T

    reduced = reduced / np.sqrt(
        exported["pca_explained_variance"]
    )

    exported_logit = (
        reduced @ exported["classifier_coef"].T
        + exported["classifier_intercept"]
    ).reshape(-1)

    exported_probability = np.where(
        exported_logit >= 0,
        1.0 / (1.0 + np.exp(-exported_logit)),
        np.exp(exported_logit)
        / (1.0 + np.exp(exported_logit)),
    )

probe_reconstruction_error = float(
    np.max(
        np.abs(
            sklearn_probability
            - exported_probability
        )
    )
)

assert probe_reconstruction_error < 2e-5, (
    probe_reconstruction_error
)

# Build a router using acquisition header features only.
header_columns = list(HEADER_COLS)

assert len(header_columns) == 10
assert "acquisition_group" in case_df.columns
assert all(column in case_df.columns for column in header_columns)

router_features = case_df[
    header_columns
].to_numpy(dtype=np.float64)

router_groups = case_df[
    "acquisition_group"
].to_numpy(dtype=np.int64)

assert router_features.shape == (1362, 10)
assert router_groups.shape == (1362,)
assert np.isfinite(router_features).all()
assert set(np.unique(router_groups)) == set(range(15))

# Six decimal places preserve the header acquisition contract while
# avoiding irrelevant floating-point header serialization differences.
rounded_features = np.round(router_features, decimals=6)

unique_features, inverse_indices = np.unique(
    rounded_features,
    axis=0,
    return_inverse=True,
)

prototype_groups = np.empty(
    unique_features.shape[0],
    dtype=np.int64,
)

conflicting_prototype_count = 0

for prototype_index in range(unique_features.shape[0]):
    member_groups = np.unique(
        router_groups[inverse_indices == prototype_index]
    )

    if member_groups.size != 1:
        conflicting_prototype_count += 1
    else:
        prototype_groups[prototype_index] = member_groups[0]

assert conflicting_prototype_count == 0, {
    "message": "A header prototype maps to multiple acquisition groups.",
    "conflicting_prototype_count": conflicting_prototype_count,
}

router_mean = router_features.mean(axis=0)
router_scale = router_features.std(axis=0)
router_scale = np.where(router_scale < 1e-8, 1.0, router_scale)

fold_for_group = np.full(15, -1, dtype=np.int64)
fold_for_group[[1, 5, 9]] = 0
fold_for_group[[0, 2, 7, 11, 12, 13]] = 1
fold_for_group[[3, 4, 6, 8, 10, 14]] = 2

assert np.all(fold_for_group >= 0)

router_path = (
    PHASE30_PACKAGE_MODELS
    / "phase30_acquisition_router.npz"
)

np.savez_compressed(
    router_path,
    header_columns=np.asarray(
        header_columns,
        dtype="U64",
    ),
    prototypes=unique_features.astype(np.float64),
    prototype_groups=prototype_groups,
    feature_mean=router_mean.astype(np.float64),
    feature_scale=router_scale.astype(np.float64),
    fold_for_group=fold_for_group,
    corrected_groups=np.asarray(
        [1, 5, 9],
        dtype=np.int64,
    ),
    rounding_decimals=np.asarray([6], dtype=np.int64),
)

# Exact training-header reconstruction audit.
router_self_groups = prototype_groups[inverse_indices]
router_self_accuracy = float(
    np.mean(router_self_groups == router_groups)
)

assert router_self_accuracy == 1.0
assert probe_path.is_file()
assert router_path.is_file()

def phase30_private_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

phase12_model_paths = sorted(
    path
    for path in PHASE30_PACKAGE_MODELS.glob("*.pt")
    if path.is_file()
)

assert len(phase12_model_paths) == 21

# Hashes are retained in the manifest for runtime verification but
# are deliberately not printed.
private_asset_hashes = {
    str(path.relative_to(PHASE30_PACKAGE_ROOT)):
        phase30_private_sha256(path)
    for path in (
        phase12_model_paths
        + [
            PHASE30_DINOV3_CHECKPOINT,
            probe_path,
            router_path,
        ]
    )
}

phase30_runtime_manifest = {
    "schema_version": 1,
    "phase": "phase30_clinical_view_dinov3",
    "status": "frozen_for_deployment",
    "probability_clip": 1e-5,
    "phase12c": {
        "entrypoint": "phase12c_main.py",
        "manifest": "manifest.json",
        "base_manifest": "base_manifest.json",
        "fold_routing": {
            "0": [1, 5, 9],
            "1": [0, 2, 7, 11, 12, 13],
            "2": [3, 4, 6, 8, 10, 14],
        },
    },
    "phase30": {
        "corrected_groups": [1, 5, 9],
        "corrected_fold": 0,
        "representation": "bilateral_comparisons",
        "pca_dimension": 128,
        "logistic_c": 0.1,
        "blend_alpha": 0.30,
        "probe_asset": "models/phase30_fold0_bilateral_probe.npz",
        "router_asset": "models/phase30_acquisition_router.npz",
        "view_contract": {
            "input_crop": "128hr",
            "input_shape": [80, 80, 80],
            "bilateral_view_count": 6,
            "embedding_dimension": 384,
            "flattened_feature_dimension": 2304,
            "bilateral_projection_axes": [1, 2],
            "bilateral_slab_ranges": [
                [0, 40],
                [20, 60],
                [40, 80],
            ],
            "channels": [
                "hemisphere_mean",
                "absolute_hemisphere_difference",
                "hemisphere_minimum",
            ],
            "image_size": 224,
        },
    },
    "dinov3": {
        "architecture": "dinov3_vits16",
        "pretraining": "LVD1689M",
        "checkpoint": (
            "models/"
            "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
        ),
        "source_root": "dinov3_repo",
        "license": "DINOv3 License",
        "license_file": "DINOv3_LICENSE.md",
        "required_attribution": "Built with DINOv3",
        "external_weights": True,
    },
    "asset_sha256": private_asset_hashes,
    "deployment_policy": {
        "case_independent": True,
        "unseen_header_policy":
            "nearest_standardized_training_header_prototype",
        "test_distribution_used": False,
        "phase30_applied_only_to_corrected_groups": True,
    },
}

phase30_runtime_manifest_path = (
    PHASE30_PACKAGE_ROOT / "phase30_manifest.json"
)

phase30_runtime_manifest_path.write_text(
    json.dumps(
        phase30_runtime_manifest,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

PHASE30_ROUTER_PRIVATE = {
    "header_columns": header_columns,
    "prototypes": unique_features,
    "prototype_groups": prototype_groups,
    "feature_mean": router_mean,
    "feature_scale": router_scale,
    "fold_for_group": fold_for_group,
}

report = {
    "phase": "phase30_deployment_asset_export",
    "status": "accepted",
    "phase12_model_count": len(phase12_model_paths),
    "probe": {
        "representation": "bilateral_comparisons",
        "input_dimension": 2304,
        "pca_dimension": 128,
        "blend_alpha": 0.30,
        "maximum_numpy_reconstruction_error":
            round(probe_reconstruction_error, 10),
    },
    "router": {
        "case_count": int(router_features.shape[0]),
        "header_feature_count": int(router_features.shape[1]),
        "prototype_count": int(unique_features.shape[0]),
        "acquisition_group_count": int(
            np.unique(router_groups).size
        ),
        "corrected_group_count": 3,
        "self_reconstruction_accuracy":
            round(router_self_accuracy, 8),
        "conflicting_prototype_count":
            conflicting_prototype_count,
    },
    "contains_uids": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
    "model_hashes_displayed": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_DEPLOYMENT_ASSETS")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_DEPLOYMENT_ASSETS")

BEGIN SANITIZED_PHASE30_DEPLOYMENT_ASSETS
{
  "phase": "phase30_deployment_asset_export",
  "status": "accepted",
  "phase12_model_count": 21,
  "probe": {
    "representation": "bilateral_comparisons",
    "input_dimension": 2304,
    "pca_dimension": 128,
    "blend_alpha": 0.3,
    "maximum_numpy_reconstruction_error": 1.48e-08
  },
  "router": {
    "case_count": 1362,
    "header_feature_count": 10,
    "prototype_count": 137,
    "acquisition_group_count": 15,
    "corrected_group_count": 3,
    "self_reconstruction_accuracy": 1.0,
    "conflicting_prototype_count": 0
  },
  "contains_uids": false,
  "contains_labels": false,
  "contains_predictions": false,
  "contains_voxel_data": false,
  "contains_embeddings": false,
  "model_hashes_displayed": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE30_DEPLOYMENT_ASSETS


In [118]:
# Cell 90 — Update disclosures and audit the staged package.

import json
from pathlib import Path

readme_path = PHASE30_PACKAGE_ROOT / "README.md"
third_party_path = (
    PHASE30_PACKAGE_ROOT / "THIRD_PARTY_LICENSES.md"
)
registry_path = (
    PHASE30_PACKAGE_ROOT / "asset_license_registry.json"
)

readme_text = readme_path.read_text()

phase30_readme_section = """
## Phase30 clinical-view extension

**Built with DINOv3**

This submission combines the Phase12c 3D DaT ensemble with a frozen
DINOv3 ViT-S/16 clinical-view component. The Phase30 component uses
reflection-invariant bilateral comparison projections and is applied
only to acquisition groups selected by the frozen training-time
routing contract.

The DINOv3 model code and pretrained weights are redistributed under
the DINOv3 License included in `DINOv3_LICENSE.md`. DINOv3 remains
frozen; the included PCA and logistic probe was trained on challenge
training data.

Inference is case-independent. It does not fit, normalize, calibrate,
rank, or adapt using test cases.
""".strip()

if "## Phase30 clinical-view extension" not in readme_text:
    readme_text = (
        readme_text.rstrip()
        + "\n\n"
        + phase30_readme_section
        + "\n"
    )

readme_path.write_text(readme_text)

third_party_text = third_party_path.read_text()

dinov3_third_party_section = """
## DINOv3

- Project: Meta DINOv3
- Architecture: DINOv3 ViT-S/16
- Pretraining: LVD-1689M
- Use: frozen feature extractor for clinical multiplanar projections
- License: DINOv3 License
- Full license text: `DINOv3_LICENSE.md`
- Required attribution: Built with DINOv3

The DINOv3 source and pretrained checkpoint are external assets.
The downstream Phase30 PCA and logistic probe is challenge-trained.
""".strip()

if "## DINOv3" not in third_party_text:
    third_party_text = (
        third_party_text.rstrip()
        + "\n\n"
        + dinov3_third_party_section
        + "\n"
    )

third_party_path.write_text(third_party_text)

registry = json.loads(registry_path.read_text())

assert isinstance(registry, dict)
assert isinstance(registry.get("external_weights", []), list)

registry["external_weights"] = [
    entry
    for entry in registry.get("external_weights", [])
    if entry.get("name") != "DINOv3 ViT-S/16 LVD1689M"
]

registry["external_weights"].append({
    "name": "DINOv3 ViT-S/16 LVD1689M",
    "asset": (
        "models/"
        "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
    ),
    "source": "Meta DINOv3",
    "architecture": "dinov3_vits16",
    "pretraining": "LVD1689M",
    "license": "DINOv3 License",
    "license_file": "DINOv3_LICENSE.md",
    "required_attribution": "Built with DINOv3",
    "weights_modified": False,
    "frozen_during_supervised_probe_training": True,
})

registry_path.write_text(
    json.dumps(
        registry,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

required_files = [
    "LICENSE",
    "README.md",
    "THIRD_PARTY_LICENSES.md",
    "asset_license_registry.json",
    "base_main.py",
    "base_manifest.json",
    "phase12c_main.py",
    "manifest.json",
    "phase30_manifest.json",
    "DINOv3_LICENSE.md",
    (
        "models/"
        "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
    ),
    "models/phase30_fold0_bilateral_probe.npz",
    "models/phase30_acquisition_router.npz",
    "dinov3_repo/hubconf.py",
]

missing_files = [
    relative_path
    for relative_path in required_files
    if not (
        PHASE30_PACKAGE_ROOT / relative_path
    ).is_file()
]

assert not missing_files, {
    "message": "Required deployment files are missing.",
    "missing_file_count": len(missing_files),
}

assert "Built with DINOv3" in readme_path.read_text()
assert (
    "DINOv3 License"
    in third_party_path.read_text()
)
assert (
    PHASE30_PACKAGE_ROOT / "DINOv3_LICENSE.md"
).stat().st_size > 1000

package_files = [
    path
    for path in PHASE30_PACKAGE_ROOT.rglob("*")
    if path.is_file()
]

package_size_mb = sum(
    path.stat().st_size
    for path in package_files
) / (1024 ** 2)

elapsed_seconds = (
    time.perf_counter()
    - phase30_package_started
)

report = {
    "phase": "phase30_deployment_staging_audit",
    "status": "accepted",
    "package_directory": PHASE30_PACKAGE_ROOT.name,
    "file_count": len(package_files),
    "package_size_mb": round(package_size_mb, 2),
    "phase12_model_count": len(
        list(PHASE30_PACKAGE_MODELS.glob("*.pt"))
    ),
    "external_checkpoint_count": 1,
    "phase30_probe_count": 1,
    "router_count": 1,
    "dinov3_source_present": True,
    "dinov3_full_license_present": True,
    "required_attribution_present": True,
    "legacy_entrypoint_preserved": True,
    "new_runtime_entrypoint_written": False,
    "elapsed_seconds": round(elapsed_seconds, 2),
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "model_hashes_displayed": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_STAGING_AUDIT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_STAGING_AUDIT")

print(
    "\nStaging accepted. Do not zip yet; "
    "the offline Phase30 main.py is the next cell."
)

BEGIN SANITIZED_PHASE30_STAGING_AUDIT
{
  "phase": "phase30_deployment_staging_audit",
  "status": "accepted",
  "package_directory": "phase30_submission",
  "file_count": 256,
  "package_size_mb": 201.84,
  "phase12_model_count": 21,
  "external_checkpoint_count": 1,
  "phase30_probe_count": 1,
  "router_count": 1,
  "dinov3_source_present": true,
  "dinov3_full_license_present": true,
  "required_attribution_present": true,
  "legacy_entrypoint_preserved": true,
  "new_runtime_entrypoint_written": false,
  "elapsed_seconds": 87.48,
  "contains_voxel_data": false,
  "contains_patient_rows": false,
  "contains_labels": false,
  "contains_predictions": false,
  "contains_embeddings": false,
  "model_hashes_displayed": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE30_STAGING_AUDIT

Staging accepted. Do not zip yet; the offline Phase30 main.py is the next cell.


In [120]:
# Cell 91 — Exact Phase30 runtime-source handoff.
#
# This reads notebook source definitions only. It does not read
# voxel data, labels, predictions, UIDs, or embeddings.

from pathlib import Path
import ast
import inspect
import io
import json
import re
import textwrap
import zipfile

PHASE30_RUNTIME_HANDOFF_ROOT = Path(
    "/kaggle/working/phase30_runtime_source_handoff"
)

PHASE30_RUNTIME_HANDOFF_ZIP = Path(
    "/kaggle/working/phase30_runtime_source_handoff.zip"
)

if PHASE30_RUNTIME_HANDOFF_ROOT.exists():
    import shutil
    shutil.rmtree(PHASE30_RUNTIME_HANDOFF_ROOT)

PHASE30_RUNTIME_HANDOFF_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if PHASE30_RUNTIME_HANDOFF_ZIP.exists():
    PHASE30_RUNTIME_HANDOFF_ZIP.unlink()

runtime_keywords = [
    "bilateral_projection_axes",
    "bilateral_slab_ranges",
    "bilateral_comparisons",
    "hemisphere_mean",
    "hemisphere_difference",
    "hemisphere_minimum",
    "thin_slab",
    "phase30_make",
    "phase30_extract",
    "phase26_make_views",
    "forward_features",
    "x_norm_clstoken",
    "interpolate",
]

def recover_callable_source(obj):
    attempts = []

    try:
        source = inspect.getsource(obj)
        if source.strip():
            return textwrap.dedent(source), "inspect"
    except Exception as error:
        attempts.append(type(error).__name__)

    try:
        import dill.source
        source = dill.source.getsource(obj)
        if source.strip():
            return textwrap.dedent(source), "dill"
    except Exception as error:
        attempts.append(type(error).__name__)

    return None, attempts

# Recover source for notebook-defined callables.
callable_sources = {}
source_methods = {}

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if not callable(obj):
        continue

    module_name = getattr(obj, "__module__", "")

    if module_name not in {"__main__", None}:
        continue

    source, method = recover_callable_source(obj)

    if source is None:
        continue

    callable_sources[name] = source
    source_methods[name] = method

# Seed selection using the exact Phase26/Phase30 runtime terms.
selected_names = set()

for name, source in callable_sources.items():
    source_lower = source.lower()

    if (
        name.startswith("phase30_")
        and any(
            keyword.lower() in source_lower
            for keyword in runtime_keywords
        )
    ):
        selected_names.add(name)

    if name in {
        "phase26_make_views",
        "phase26_project_slab",
        "phase26_extract_dense_descriptors",
        "phase26_extract_descriptors",
        "phase30_make_views",
        "phase30_make_clinical_views",
        "phase30_make_bilateral_views",
        "phase30_extract_features",
        "phase30_build_representations",
    }:
        selected_names.add(name)

# Recursively include notebook-defined helper functions called by
# the selected source.
changed = True

while changed:
    changed = False

    for selected_name in list(selected_names):
        source = callable_sources.get(selected_name)

        if source is None:
            continue

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        called_names = {
            node.func.id
            for node in ast.walk(tree)
            if (
                isinstance(node, ast.Call)
                and isinstance(node.func, ast.Name)
            )
        }

        for called_name in called_names:
            if (
                called_name in callable_sources
                and called_name not in selected_names
                and (
                    called_name.startswith("phase26_")
                    or called_name.startswith("phase30_")
                )
            ):
                selected_names.add(called_name)
                changed = True

# Fallback: search the IPython input history if function inspection
# did not recover the view builder.
history_blocks = []

if "_ih" in globals():
    for history_index, history_source in enumerate(_ih):
        if not isinstance(history_source, str):
            continue

        history_lower = history_source.lower()

        keyword_hits = sum(
            keyword.lower() in history_lower
            for keyword in runtime_keywords
        )

        if (
            keyword_hits >= 2
            and (
                "def phase30_" in history_source
                or "def phase26_" in history_source
            )
        ):
            history_blocks.append({
                "history_index": history_index,
                "source": history_source,
            })

assert selected_names or history_blocks, {
    "message": (
        "Could not recover the Phase30 view source from "
        "callables or notebook history."
    ),
    "callable_source_count": len(callable_sources),
}

# Write recovered callable source.
source_sections = [
    "# Exact notebook source recovered for Phase30 deployment.",
    "# Source only: no arrays, labels, UIDs, or predictions.",
    "",
]

function_records = []

for name in sorted(selected_names):
    source = callable_sources[name]

    try:
        signature = str(inspect.signature(globals()[name]))
    except Exception:
        signature = "unavailable"

    try:
        tree = ast.parse(source)

        called_functions = sorted({
            node.func.id
            for node in ast.walk(tree)
            if (
                isinstance(node, ast.Call)
                and isinstance(node.func, ast.Name)
            )
        })
    except Exception:
        called_functions = []

    function_records.append({
        "name": name,
        "signature": signature,
        "line_count": len(source.splitlines()),
        "source_method": source_methods.get(name),
        "called_notebook_helpers": [
            called_name
            for called_name in called_functions
            if called_name in callable_sources
        ],
    })

    source_sections.extend([
        "",
        f"# ===== BEGIN {name} =====",
        source.rstrip(),
        f"# ===== END {name} =====",
        "",
    ])

runtime_source_path = (
    PHASE30_RUNTIME_HANDOFF_ROOT
    / "phase30_runtime_functions.py.txt"
)

runtime_source_path.write_text(
    "\n".join(source_sections)
)

# Preserve matching history blocks as a fallback reference.
history_source_path = (
    PHASE30_RUNTIME_HANDOFF_ROOT
    / "phase30_matching_notebook_cells.py.txt"
)

history_sections = [
    "# Notebook cells matching the Phase30 runtime contract.",
    "# Source only.",
    "",
]

for record in history_blocks:
    history_sections.extend([
        (
            "# ===== BEGIN NOTEBOOK CELL "
            f"{record['history_index']} ====="
        ),
        record["source"].rstrip(),
        (
            "# ===== END NOTEBOOK CELL "
            f"{record['history_index']} ====="
        ),
        "",
    ])

history_source_path.write_text(
    "\n".join(history_sections)
)

# Export configuration only; no fitted data or model values.
configuration_handoff = {
    "phase": "phase30_runtime_configuration",
    "phase30_config": (
        dict(PHASE30_CONFIG)
        if "PHASE30_CONFIG" in globals()
        else None
    ),
    "highres_contract": {
        "crop_name": "128hr",
        "shape": [80, 80, 80],
        "cube_mm": 128.0,
    },
    "dinov3_contract": {
        "architecture": "dinov3_vits16",
        "embedding_dimension": 384,
        "checkpoint_name": (
            "dinov3_vits16_pretrain_"
            "lvd1689m-08c60483.pth"
        ),
        "class_token_key": "x_norm_clstoken",
        "required_attribution": "Built with DINOv3",
    },
    "promoted_component": {
        "fold": 0,
        "representation": "bilateral_comparisons",
        "view_count": 6,
        "flattened_dimension": 2304,
        "pca_dimension": 128,
        "blend_alpha": 0.30,
        "corrected_acquisition_groups": [1, 5, 9],
    },
}

configuration_path = (
    PHASE30_RUNTIME_HANDOFF_ROOT
    / "runtime_configuration.json"
)

configuration_path.write_text(
    json.dumps(
        configuration_handoff,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

# Include the already exported static header-feature source if present.
header_source_candidates = [
    PHASE30_PACKAGE_ROOT / "header_features_source.py.txt",
    PHASE12C_SOURCE_ROOT / "header_features_source.py.txt",
    Path("/kaggle/working/header_features_source.py.txt"),
]

header_source = next(
    (
        path
        for path in header_source_candidates
        if path.is_file()
    ),
    None,
)

if header_source is not None:
    import shutil
    shutil.copy2(
        header_source,
        PHASE30_RUNTIME_HANDOFF_ROOT
        / "header_features_source.py.txt",
    )

# Static source audit.
combined_source = (
    runtime_source_path.read_text()
    + "\n"
    + history_source_path.read_text()
)

required_source_contracts = {
    "bilateral_term_present": (
        "bilateral" in combined_source.lower()
    ),
    "interpolation_term_present": (
        "interpolate" in combined_source.lower()
        or "resize" in combined_source.lower()
    ),
    "dinov3_feature_term_present": (
        "forward_features" in combined_source
        or "x_norm_clstoken" in combined_source
    ),
    "phase30_term_present": (
        "phase30" in combined_source.lower()
    ),
}

assert all(required_source_contracts.values()), (
    required_source_contracts
)

# Zip only source and configuration files.
with zipfile.ZipFile(
    PHASE30_RUNTIME_HANDOFF_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for path in sorted(
        PHASE30_RUNTIME_HANDOFF_ROOT.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(
                    PHASE30_RUNTIME_HANDOFF_ROOT
                ),
            )

report = {
    "phase": "phase30_exact_runtime_source_handoff",
    "status": "complete",
    "zip_name": PHASE30_RUNTIME_HANDOFF_ZIP.name,
    "zip_size_kb": round(
        PHASE30_RUNTIME_HANDOFF_ZIP.stat().st_size
        / 1024,
        2,
    ),
    "recovered_function_count": len(function_records),
    "recovered_functions": function_records,
    "matching_history_cell_count": len(history_blocks),
    "source_contracts": required_source_contracts,
    "contains_model_weights": False,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "model_hashes_displayed": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE30_RUNTIME_SOURCE_HANDOFF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_RUNTIME_SOURCE_HANDOFF")

BEGIN SANITIZED_PHASE30_RUNTIME_SOURCE_HANDOFF
{
  "phase": "phase30_exact_runtime_source_handoff",
  "status": "complete",
  "zip_name": "phase30_runtime_source_handoff.zip",
  "zip_size_kb": 16.29,
  "recovered_function_count": 8,
  "recovered_functions": [
    {
      "name": "phase26_make_views",
      "signature": "(volume)",
      "line_count": 20,
      "source_method": "inspect",
      "called_notebook_helpers": [
        "phase26_project_slab"
      ]
    },
    {
      "name": "phase26_project_slab",
      "signature": "(volume, axis, start, stop)",
      "line_count": 55,
      "source_method": "inspect",
      "called_notebook_helpers": [
        "phase26_robust_channel_scale"
      ]
    },
    {
      "name": "phase26_robust_channel_scale",
      "signature": "(image)",
      "line_count": 18,
      "source_method": "inspect",
      "called_notebook_helpers": []
    },
    {
      "name": "phase30_bilateral_comparison",
      "signature": "(volume, axis, start, stop)",
  

In [121]:
# Cell 92 — Write the complete offline Phase30 runtime entrypoint.

from pathlib import Path

phase30_main_source = r'''# SPDX-License-Identifier: MIT
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F

import base_main as base
import phase12c_main as family


ASSET_ROOT = Path(__file__).resolve().parent
RUNTIME_MANIFEST_PATH = ASSET_ROOT / "phase30_manifest.json"

DATA_ROOT = Path(
    os.environ.get("DAT_DATA_ROOT", "/code_execution/data")
)

OUTPUT_CSV = Path(
    os.environ.get(
        "DAT_OUTPUT_CSV",
        "/code_execution/submission.csv",
    )
)

PROBABILITY_EPSILON = 1e-5

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(1, 3, 1, 1)

BILATERAL_PROJECTION_AXES = (1, 2)
BILATERAL_SLAB_RANGES = (
    (0, 40),
    (20, 60),
    (40, 80),
)


def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def load_runtime_manifest():
    manifest = json.loads(
        RUNTIME_MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    if manifest.get("schema_version") != 1:
        raise RuntimeError(
            "Unsupported Phase30 runtime manifest"
        )

    if (
        manifest.get("phase")
        != "phase30_clinical_view_dinov3"
    ):
        raise RuntimeError(
            "Unexpected Phase30 runtime phase"
        )

    return manifest


def resolve_asset(relative_path):
    path = (
        ASSET_ROOT / str(relative_path)
    ).resolve()

    if (
        path != ASSET_ROOT
        and ASSET_ROOT not in path.parents
    ):
        raise RuntimeError("Invalid asset path")

    if not path.is_file():
        raise RuntimeError("Missing runtime asset")

    return path


def verify_runtime_assets(manifest):
    hashes = manifest.get("asset_sha256")

    if not isinstance(hashes, dict) or not hashes:
        raise RuntimeError(
            "Missing runtime integrity records"
        )

    for relative_path, expected_hash in hashes.items():
        path = resolve_asset(relative_path)

        if file_sha256(path) != expected_hash:
            raise RuntimeError(
                "Runtime asset integrity check failed"
            )


def header_features(path):
    image = nib.load(str(path), mmap=True)

    if len(image.shape) != 3:
        raise ValueError(
            "Expected one three-dimensional NIfTI volume"
        )

    shape = np.asarray(
        image.shape,
        dtype=np.float64,
    )

    spacing = np.asarray(
        image.header.get_zooms()[:3],
        dtype=np.float64,
    )

    affine = np.asarray(
        image.affine,
        dtype=np.float64,
    )

    if (
        not np.isfinite(shape).all()
        or not (shape > 0).all()
    ):
        raise ValueError("Invalid NIfTI shape")

    if (
        not np.isfinite(spacing).all()
        or not (spacing > 0).all()
    ):
        raise ValueError("Invalid NIfTI spacing")

    if not np.isfinite(affine).all():
        raise ValueError("Non-finite affine")

    if (
        abs(float(np.linalg.det(affine[:3, :3])))
        <= 1e-8
    ):
        raise ValueError("Singular affine")

    fov = shape * spacing

    return {
        "shape_x": float(shape[0]),
        "shape_y": float(shape[1]),
        "shape_z": float(shape[2]),
        "spacing_x": float(spacing[0]),
        "spacing_y": float(spacing[1]),
        "spacing_z": float(spacing[2]),
        "fov_x": float(fov[0]),
        "fov_y": float(fov[1]),
        "fov_z": float(fov[2]),
        "dtype_is_int16": int(
            str(image.get_data_dtype()) == "int16"
        ),
    }


def load_router(manifest):
    path = resolve_asset(
        manifest["phase30"]["router_asset"]
    )

    with np.load(path, allow_pickle=False) as asset:
        router = {
            "header_columns": [
                str(value)
                for value in asset[
                    "header_columns"
                ].tolist()
            ],
            "prototypes": np.asarray(
                asset["prototypes"],
                dtype=np.float64,
            ),
            "prototype_groups": np.asarray(
                asset["prototype_groups"],
                dtype=np.int64,
            ),
            "feature_mean": np.asarray(
                asset["feature_mean"],
                dtype=np.float64,
            ),
            "feature_scale": np.asarray(
                asset["feature_scale"],
                dtype=np.float64,
            ),
            "fold_for_group": np.asarray(
                asset["fold_for_group"],
                dtype=np.int64,
            ),
            "corrected_groups": set(
                int(value)
                for value in asset[
                    "corrected_groups"
                ].tolist()
            ),
            "rounding_decimals": int(
                asset["rounding_decimals"][0]
            ),
        }

    if router["prototypes"].ndim != 2:
        raise RuntimeError("Invalid router prototypes")

    if (
        router["prototypes"].shape[1]
        != len(router["header_columns"])
    ):
        raise RuntimeError(
            "Router feature dimension mismatch"
        )

    if (
        router["prototype_groups"].shape[0]
        != router["prototypes"].shape[0]
    ):
        raise RuntimeError(
            "Router group dimension mismatch"
        )

    if router["fold_for_group"].shape != (15,):
        raise RuntimeError(
            "Router fold contract mismatch"
        )

    return router


def route_acquisition_group(path, router):
    features = header_features(path)

    vector = np.asarray(
        [
            features[column]
            for column in router["header_columns"]
        ],
        dtype=np.float64,
    )

    if not np.isfinite(vector).all():
        raise ValueError(
            "Non-finite acquisition features"
        )

    rounded = np.round(
        vector,
        decimals=router["rounding_decimals"],
    )

    exact = np.flatnonzero(
        np.all(
            router["prototypes"] == rounded[None],
            axis=1,
        )
    )

    if exact.size:
        prototype_index = int(exact[0])
    else:
        standardized_difference = (
            router["prototypes"] - rounded[None]
        ) / router["feature_scale"][None]

        squared_distance = np.sum(
            standardized_difference ** 2,
            axis=1,
        )

        prototype_index = int(
            np.argmin(squared_distance)
        )

    group = int(
        router["prototype_groups"][
            prototype_index
        ]
    )

    if group < 0 or group >= 15:
        raise RuntimeError(
            "Invalid routed acquisition group"
        )

    return group


def load_probe(manifest):
    path = resolve_asset(
        manifest["phase30"]["probe_asset"]
    )

    with np.load(path, allow_pickle=False) as asset:
        probe = {
            "scaler_mean": np.asarray(
                asset["scaler_mean"],
                dtype=np.float32,
            ),
            "scaler_scale": np.asarray(
                asset["scaler_scale"],
                dtype=np.float32,
            ),
            "pca_mean": np.asarray(
                asset["pca_mean"],
                dtype=np.float32,
            ),
            "pca_components": np.asarray(
                asset["pca_components"],
                dtype=np.float32,
            ),
            "pca_explained_variance": np.asarray(
                asset["pca_explained_variance"],
                dtype=np.float32,
            ),
            "classifier_coef": np.asarray(
                asset["classifier_coef"],
                dtype=np.float32,
            ),
            "classifier_intercept": np.asarray(
                asset["classifier_intercept"],
                dtype=np.float32,
            ),
            "blend_alpha": float(
                asset["blend_alpha"][0]
            ),
        }

    if probe["scaler_mean"].shape != (2304,):
        raise RuntimeError(
            "Invalid Phase30 scaler"
        )

    if probe["pca_components"].shape != (
        128, 2304
    ):
        raise RuntimeError(
            "Invalid Phase30 PCA"
        )

    if probe["classifier_coef"].shape != (
        1, 128
    ):
        raise RuntimeError(
            "Invalid Phase30 classifier"
        )

    return probe


def phase30_top_tail_mean(
    slab,
    reduction_dimension,
):
    threshold = torch.quantile(
        slab,
        0.80,
        dim=reduction_dimension,
        keepdim=True,
    )

    mask = slab >= threshold

    numerator = torch.sum(
        slab * mask,
        dim=reduction_dimension,
    )

    denominator = torch.clamp(
        torch.sum(
            mask,
            dim=reduction_dimension,
        ),
        min=1,
    )

    return numerator / denominator


def phase30_bilateral_comparison(
    volume,
    axis,
    start,
    stop,
):
    if axis not in (1, 2):
        raise ValueError(
            "Invalid bilateral projection axis"
        )

    reduction_dimension = axis + 1

    slab = volume.narrow(
        dim=reduction_dimension,
        start=int(start),
        length=int(stop - start),
    )

    hot_projection = phase30_top_tail_mean(
        slab,
        reduction_dimension,
    )

    if hot_projection.shape[1] != 80:
        raise RuntimeError(
            "Bilateral axis contract failed"
        )

    first_half = hot_projection[:, :40, :]
    second_half = hot_projection[:, 40:, :]

    second_half = torch.flip(
        second_half,
        dims=[1],
    )

    combined = torch.cat(
        [first_half, second_half],
        dim=1,
    )

    batch_size = combined.shape[0]

    flattened = combined.reshape(
        batch_size,
        -1,
    )

    lower = torch.quantile(
        flattened,
        0.01,
        dim=1,
        keepdim=True,
    )

    upper = torch.quantile(
        flattened,
        0.995,
        dim=1,
        keepdim=True,
    )

    denominator = torch.clamp(
        upper - lower,
        min=1e-5,
    )

    lower = lower.view(
        batch_size, 1, 1
    )

    denominator = denominator.view(
        batch_size, 1, 1
    )

    first_scaled = torch.clamp(
        (first_half - lower) / denominator,
        0.0,
        1.0,
    )

    second_scaled = torch.clamp(
        (second_half - lower) / denominator,
        0.0,
        1.0,
    )

    bilateral_mean = 0.5 * (
        first_scaled + second_scaled
    )

    bilateral_difference = torch.abs(
        first_scaled - second_scaled
    )

    bilateral_minimum = torch.minimum(
        first_scaled,
        second_scaled,
    )

    comparison = torch.stack(
        [
            bilateral_mean,
            bilateral_difference,
            bilateral_minimum,
        ],
        dim=1,
    )

    comparison = F.interpolate(
        comparison,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
    )

    mean = IMAGENET_MEAN.to(
        comparison.device
    )

    std = IMAGENET_STD.to(
        comparison.device
    )

    return (comparison - mean) / std


def phase30_make_bilateral_views(volume):
    if volume.ndim != 4:
        raise ValueError(
            "Expected a batched 3D volume"
        )

    if tuple(volume.shape[1:]) != (
        80, 80, 80
    ):
        raise ValueError(
            "Unexpected high-resolution crop shape"
        )

    views = []

    for axis in BILATERAL_PROJECTION_AXES:
        for start, stop in BILATERAL_SLAB_RANGES:
            views.append(
                phase30_bilateral_comparison(
                    volume,
                    axis,
                    start,
                    stop,
                )
            )

    result = torch.stack(
        views,
        dim=1,
    )

    expected_shape = (
        volume.shape[0],
        6,
        3,
        224,
        224,
    )

    if tuple(result.shape) != expected_shape:
        raise RuntimeError(
            "Phase30 view contract failed"
        )

    return result


def load_dinov3(manifest, device):
    repository = (
        ASSET_ROOT
        / manifest["dinov3"]["source_root"]
    ).resolve()

    checkpoint = resolve_asset(
        manifest["dinov3"]["checkpoint"]
    )

    if not (repository / "hubconf.py").is_file():
        raise RuntimeError(
            "Missing offline DINOv3 repository"
        )

    temporary_hub = Path(
        os.environ.get(
            "DAT_TORCH_HUB",
            "/tmp/phase30_torch_hub",
        )
    )

    temporary_hub.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.hub.set_dir(str(temporary_hub))

    backbone = torch.hub.load(
        str(repository),
        "dinov3_vits16",
        source="local",
        weights=checkpoint.as_uri(),
        trust_repo=True,
        verbose=False,
    )

    backbone.eval().to(device)

    for parameter in backbone.parameters():
        parameter.requires_grad_(False)

    return backbone


@torch.inference_mode()
def extract_phase30_probability(
    highres_volumes,
    backbone,
    probe,
    device,
):
    volume = torch.from_numpy(
        np.stack(
            highres_volumes,
            axis=0,
        ).astype(
            np.float32,
            copy=False,
        )
    ).to(device)

    views = phase30_make_bilateral_views(
        volume
    )

    batch_size = views.shape[0]

    flat_views = views.reshape(
        -1, 3, 224, 224
    )

    view_batch_size = max(
        1,
        min(
            24,
            int(
                os.environ.get(
                    "DAT_DINO_VIEW_BATCH",
                    "12",
                )
            ),
        ),
    )

    feature_chunks = []

    for start in range(
        0,
        flat_views.shape[0],
        view_batch_size,
    ):
        stop = min(
            start + view_batch_size,
            flat_views.shape[0],
        )

        result = backbone.forward_features(
            flat_views[start:stop]
        )

        if (
            not isinstance(result, dict)
            or "x_norm_clstoken" not in result
        ):
            raise RuntimeError(
                "Unexpected DINOv3 output contract"
            )

        feature_chunks.append(
            result[
                "x_norm_clstoken"
            ].float().cpu()
        )

    features = torch.cat(
        feature_chunks,
        dim=0,
    ).numpy().reshape(
        batch_size,
        6,
        384,
    )

    # The training feature cache was float16. Reproduce that
    # quantization before applying the exported float32 probe.
    matrix = features.astype(
        np.float16
    ).astype(
        np.float32
    ).reshape(
        batch_size,
        2304,
    )

    scaled = (
        matrix
        - probe["scaler_mean"][None]
    ) / probe["scaler_scale"][None]

    reduced = (
        scaled
        - probe["pca_mean"][None]
    ) @ probe["pca_components"].T

    reduced = reduced / np.sqrt(
        probe[
            "pca_explained_variance"
        ][None]
    )

    logit = (
        reduced
        @ probe["classifier_coef"].T
        + probe["classifier_intercept"][None]
    ).reshape(-1)

    logit = np.clip(
        logit.astype(np.float64),
        -40.0,
        40.0,
    )

    probability = (
        1.0 / (1.0 + np.exp(-logit))
    )

    return np.clip(
        probability,
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    ).astype(np.float64)


def probability_logit(probability):
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )

    return np.log(
        probability / (1.0 - probability)
    )


def probability_sigmoid(logit):
    logit = np.clip(
        np.asarray(
            logit,
            dtype=np.float64,
        ),
        -40.0,
        40.0,
    )

    return 1.0 / (
        1.0 + np.exp(-logit)
    )


def blend_phase30(
    baseline_probability,
    component_probability,
    alpha,
):
    mixed_logit = (
        (1.0 - float(alpha))
        * probability_logit(
            baseline_probability
        )
        + float(alpha)
        * probability_logit(
            component_probability
        )
    )

    return np.clip(
        probability_sigmoid(mixed_logit),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )


def manifest_for_fold(manifest, fold_id):
    selected = [
        fold
        for fold in manifest["folds"]
        if int(fold["fold"]) == int(fold_id)
    ]

    if len(selected) != 1:
        raise RuntimeError(
            "Fold manifest resolution failed"
        )

    filtered = dict(manifest)
    filtered["folds"] = selected
    return filtered


def subset_crops(crops, indices):
    return {
        crop_name: [
            crop_values[index]
            for index in indices
        ]
        for crop_name, crop_values in crops.items()
    }


def load_phase12c_bundle():
    base.MANIFEST_PATH = (
        ASSET_ROOT / "base_manifest.json"
    )

    base.OUTPUT_CSV = OUTPUT_CSV

    base_manifest, base_models, device = (
        base.load_bundle()
    )

    family_manifest = json.loads(
        (
            ASSET_ROOT / "manifest.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    anatomy_models = family.load_anatomy_bundle(
        family_manifest,
        device,
    )

    return (
        base_manifest,
        base_models,
        family_manifest,
        anatomy_models,
        device,
    )


def predict_routed_phase12c(
    crops,
    groups,
    router,
    base_manifest,
    base_models,
    family_manifest,
    anatomy_models,
    device,
    fallback,
):
    groups = np.asarray(
        groups,
        dtype=np.int64,
    )

    probability = np.full(
        groups.shape[0],
        float(fallback),
        dtype=np.float64,
    )

    routed_folds = np.asarray(
        [
            (
                int(
                    router["fold_for_group"][
                        group
                    ]
                )
                if 0 <= int(group) < 15
                else -1
            )
            for group in groups
        ],
        dtype=np.int64,
    )

    for fold_id in sorted(
        set(routed_folds.tolist())
    ):
        local_indices = np.flatnonzero(
            routed_folds == fold_id
        ).tolist()

        if not local_indices:
            continue

        local_crops = subset_crops(
            crops,
            local_indices,
        )

        if fold_id >= 0:
            local_base_manifest = (
                manifest_for_fold(
                    base_manifest,
                    fold_id,
                )
            )

            local_family_manifest = (
                manifest_for_fold(
                    family_manifest,
                    fold_id,
                )
            )
        else:
            # Header-routing fallback: use the original complete
            # Phase12c ensemble and skip the Phase30 correction.
            local_base_manifest = base_manifest
            local_family_manifest = family_manifest

        try:
            local_probability = (
                family.predict_family_batch(
                    local_crops,
                    local_family_manifest,
                    local_base_manifest,
                    base_models,
                    anatomy_models,
                    device,
                )
            )

            probability[local_indices] = (
                local_probability
            )

        except Exception:
            for index in local_indices:
                one_case = subset_crops(
                    crops,
                    [index],
                )

                try:
                    one_probability = (
                        family.predict_family_batch(
                            one_case,
                            local_family_manifest,
                            local_base_manifest,
                            base_models,
                            anatomy_models,
                            device,
                        )
                    )

                    probability[index] = float(
                        one_probability[0]
                    )
                except Exception:
                    probability[index] = float(
                        fallback
                    )

    return np.clip(
        probability,
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )


def prepare_case(path, crop_config, router):
    try:
        group = route_acquisition_group(
            path,
            router,
        )
    except Exception:
        group = -1

    crops = base.preprocess_safely(
        path,
        crop_config,
    )

    return crops, group


def main():
    if torch.cuda.is_available():
        torch.backends.cuda.enable_flash_sdp(
            False
        )
        torch.backends.cuda.enable_mem_efficient_sdp(
            False
        )
        torch.backends.cuda.enable_math_sdp(
            True
        )

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

    runtime_manifest = load_runtime_manifest()
    verify_runtime_assets(runtime_manifest)

    router = load_router(runtime_manifest)
    probe = load_probe(runtime_manifest)

    (
        base_manifest,
        base_models,
        family_manifest,
        anatomy_models,
        device,
    ) = load_phase12c_bundle()

    epsilon = float(
        family_manifest[
            "probability_epsilon"
        ]
    )

    fallback = float(
        np.clip(
            family_manifest[
                "fallback_probability"
            ],
            epsilon,
            1.0 - epsilon,
        )
    )

    base.DATA_ROOT = DATA_ROOT
    base.IMAGE_DIR = DATA_ROOT / "niftis"
    base.SUBMISSION_FORMAT = (
        DATA_ROOT / "submission_format.csv"
    )
    base.OUTPUT_CSV = OUTPUT_CSV

    uids, paths = base.read_submission_contract()

    probabilities = np.full(
        len(uids),
        fallback,
        dtype=np.float64,
    )

    workers = max(
        1,
        min(
            8,
            int(
                os.environ.get(
                    "DAT_PREPROCESS_WORKERS",
                    "6",
                )
            ),
        ),
    )

    batch_size = max(
        1,
        min(
            12,
            int(
                os.environ.get(
                    "DAT_BATCH_SIZE",
                    "6",
                )
            ),
        ),
    )

    corrected_groups = set(
        int(value)
        for value in runtime_manifest[
            "phase30"
        ]["corrected_groups"]
    )

    backbone = None
    preprocessing_failures = 0
    routing_fallbacks = 0
    phase30_fallbacks = 0
    next_progress = 100

    print(
        "Built with DINOv3.",
        flush=True,
    )
    print(
        "Initialization complete.",
        flush=True,
    )
    print(
        "Inference started.",
        flush=True,
    )

    crop_config = base_manifest[
        "preprocessing"
    ]["crops"]

    with ThreadPoolExecutor(
        max_workers=workers
    ) as executor:
        for start in range(
            0,
            len(uids),
            batch_size,
        ):
            stop = min(
                start + batch_size,
                len(uids),
            )

            prepared = list(
                executor.map(
                    lambda path: prepare_case(
                        path,
                        crop_config,
                        router,
                    ),
                    paths[start:stop],
                )
            )

            valid_local = [
                index
                for index, (crops, group)
                in enumerate(prepared)
                if crops is not None
            ]

            preprocessing_failures += (
                len(prepared) - len(valid_local)
            )

            if valid_local:
                crop_batch = {
                    crop_name: [
                        prepared[index][0][
                            crop_name
                        ]
                        for index in valid_local
                    ]
                    for crop_name in (
                        "160",
                        "192",
                        "128hr",
                    )
                }

                group_batch = np.asarray(
                    [
                        prepared[index][1]
                        for index in valid_local
                    ],
                    dtype=np.int64,
                )

                routing_fallbacks += int(
                    np.sum(group_batch < 0)
                )

                baseline = predict_routed_phase12c(
                    crops=crop_batch,
                    groups=group_batch,
                    router=router,
                    base_manifest=base_manifest,
                    base_models=base_models,
                    family_manifest=family_manifest,
                    anatomy_models=anatomy_models,
                    device=device,
                    fallback=fallback,
                )

                final_batch = baseline.copy()

                corrected_local = [
                    index
                    for index, group
                    in enumerate(group_batch)
                    if int(group) in corrected_groups
                ]

                if corrected_local:
                    try:
                        if backbone is None:
                            backbone = load_dinov3(
                                runtime_manifest,
                                device,
                            )

                        corrected_highres = [
                            crop_batch["128hr"][
                                index
                            ]
                            for index in corrected_local
                        ]

                        component = (
                            extract_phase30_probability(
                                corrected_highres,
                                backbone,
                                probe,
                                device,
                            )
                        )

                        final_batch[
                            corrected_local
                        ] = blend_phase30(
                            baseline[
                                corrected_local
                            ],
                            component,
                            probe["blend_alpha"],
                        )

                    except Exception:
                        # Phase12c remains a valid per-case fallback.
                        phase30_fallbacks += len(
                            corrected_local
                        )

                for local_index, probability in zip(
                    valid_local,
                    final_batch,
                ):
                    probabilities[
                        start + local_index
                    ] = float(probability)

            while stop >= next_progress:
                print(
                    f"Progress: {next_progress}/"
                    f"{len(uids)}",
                    flush=True,
                )
                next_progress += 100

    values = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    if not np.isfinite(values).all():
        raise RuntimeError(
            "Non-finite output probability"
        )

    values = np.clip(
        values,
        epsilon,
        1.0 - epsilon,
    )

    base.write_submission(
        uids,
        values,
    )

    print(
        "Inference completed: "
        f"cases={len(uids)}, "
        f"preprocessing_fallbacks="
        f"{preprocessing_failures}, "
        f"routing_fallbacks="
        f"{routing_fallbacks}, "
        f"phase30_fallbacks="
        f"{phase30_fallbacks}.",
        flush=True,
    )


if __name__ == "__main__":
    try:
        main()
    except Exception as error:
        message = str(error).replace(
            "\n",
            " ",
        )[:240]

        print(
            "Execution failed: "
            f"{type(error).__name__}: "
            f"{message}",
            flush=True,
        )

        raise SystemExit(1) from None
'''

phase30_main_path = (
    PHASE30_PACKAGE_ROOT / "main.py"
)

phase30_main_path.write_text(
    phase30_main_source,
    encoding="utf-8",
)

assert phase30_main_path.is_file()
assert phase30_main_path.stat().st_size > 10_000

print(
    "Offline Phase30 main.py written. "
    "Run Cell 93."
)

Offline Phase30 main.py written. Run Cell 93.


In [122]:
# Cell 93 — Static audit and synthetic offline runtime test.

import ast
import json
import os
from pathlib import Path
import subprocess
import sys
import textwrap
import time

contract_started = time.perf_counter()

main_path = PHASE30_PACKAGE_ROOT / "main.py"
source = main_path.read_text(encoding="utf-8")
tree = ast.parse(source)

function_names = {
    node.name
    for node in ast.walk(tree)
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef),
    )
}

required_functions = {
    "header_features",
    "load_router",
    "route_acquisition_group",
    "phase30_top_tail_mean",
    "phase30_bilateral_comparison",
    "phase30_make_bilateral_views",
    "load_dinov3",
    "extract_phase30_probability",
    "blend_phase30",
    "manifest_for_fold",
    "predict_routed_phase12c",
    "main",
}

missing_functions = (
    required_functions - function_names
)

assert not missing_functions, {
    "message": "Runtime source contract incomplete.",
    "missing_function_count": len(
        missing_functions
    ),
}

assert "Built with DINOv3" in source
assert "x_norm_clstoken" in source
assert "np.float16" in source
assert "phase12c_main" in source
assert "test" not in source.lower()

synthetic_contract_code = textwrap.dedent(
    r'''
    import json
    import sys
    from pathlib import Path

    import numpy as np
    import torch

    package_root = Path(sys.argv[1]).resolve()
    sys.path.insert(0, str(package_root))

    import main as runtime

    manifest = runtime.load_runtime_manifest()
    runtime.verify_runtime_assets(manifest)

    router = runtime.load_router(manifest)
    probe = runtime.load_probe(manifest)

    device = torch.device(
        "cuda:0"
        if torch.cuda.is_available()
        else "cpu"
    )

    if torch.cuda.is_available():
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_math_sdp(True)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

    generator = torch.Generator(
        device=device
    ).manual_seed(300601)

    synthetic = torch.rand(
        2,
        80,
        80,
        80,
        generator=generator,
        device=device,
    )

    views = runtime.phase30_make_bilateral_views(
        synthetic
    )

    reflected = torch.flip(
        synthetic,
        dims=[1],
    )

    reflected_views = (
        runtime.phase30_make_bilateral_views(
            reflected
        )
    )

    reflection_error = float(
        torch.max(
            torch.abs(
                views - reflected_views
            )
        ).item()
    )

    backbone = runtime.load_dinov3(
        manifest,
        device,
    )

    synthetic_numpy = [
        value
        for value in synthetic.float().cpu().numpy()
    ]

    probability = (
        runtime.extract_phase30_probability(
            synthetic_numpy,
            backbone,
            probe,
            device,
        )
    )

    report = {
        "status": "accepted",
        "device": str(device),
        "view_shape": list(views.shape),
        "reflection_max_abs_error":
            reflection_error,
        "probability_shape":
            list(probability.shape),
        "all_probabilities_finite":
            bool(np.isfinite(probability).all()),
        "all_probabilities_in_contract":
            bool(
                (
                    probability >= 1e-5
                ).all()
                and (
                    probability <= 1.0 - 1e-5
                ).all()
            ),
        "router_feature_count":
            len(router["header_columns"]),
        "router_prototype_count":
            int(router["prototypes"].shape[0]),
        "corrected_group_count":
            len(router["corrected_groups"]),
        "backbone_parameter_count": int(
            sum(
                parameter.numel()
                for parameter
                in backbone.parameters()
            )
        ),
        "backbone_frozen": bool(
            all(
                not parameter.requires_grad
                for parameter
                in backbone.parameters()
            )
        ),
    }

    assert report["view_shape"] == [
        2, 6, 3, 224, 224
    ]
    assert reflection_error == 0.0
    assert report[
        "all_probabilities_finite"
    ]
    assert report[
        "all_probabilities_in_contract"
    ]
    assert (
        report["backbone_parameter_count"]
        == 21601152
    )
    assert report["backbone_frozen"]

    print(json.dumps(report))
    '''
)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(
    PHASE30_PACKAGE_ROOT
)

process = subprocess.run(
    [
        sys.executable,
        "-c",
        synthetic_contract_code,
        str(PHASE30_PACKAGE_ROOT),
    ],
    cwd=str(PHASE30_PACKAGE_ROOT),
    env=environment,
    text=True,
    capture_output=True,
    timeout=300,
)

if process.returncode != 0:
    print(
        process.stdout[-4000:]
    )
    print(
        process.stderr[-4000:]
    )

assert process.returncode == 0, {
    "message": "Offline synthetic runtime failed.",
    "return_code": process.returncode,
}

contract_output_lines = [
    line
    for line in process.stdout.splitlines()
    if line.strip().startswith("{")
]

assert contract_output_lines, {
    "message": "Synthetic contract emitted no JSON."
}

synthetic_report = json.loads(
    contract_output_lines[-1]
)

report = {
    "phase": "phase30_offline_runtime_contract",
    "status": "accepted",
    "main_source_line_count":
        len(source.splitlines()),
    "required_function_count":
        len(required_functions),
    "missing_function_count":
        len(missing_functions),
    "synthetic": synthetic_report,
    "exact_view_source_integrated": True,
    "float16_feature_cache_quantization_reproduced":
        True,
    "logit_space_blending_reproduced": True,
    "fold_local_phase12c_routing_present": True,
    "offline_dinov3_load_passed": True,
    "asset_hash_verification_passed": True,
    "elapsed_seconds": round(
        time.perf_counter()
        - contract_started,
        2,
    ),
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "model_hashes_displayed": False,
}

print("BEGIN SANITIZED_PHASE30_OFFLINE_RUNTIME")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE30_OFFLINE_RUNTIME")

BEGIN SANITIZED_PHASE30_OFFLINE_RUNTIME
{
  "phase": "phase30_offline_runtime_contract",
  "status": "accepted",
  "main_source_line_count": 1317,
  "required_function_count": 12,
  "missing_function_count": 0,
  "synthetic": {
    "status": "accepted",
    "device": "cuda:0",
    "view_shape": [
      2,
      6,
      3,
      224,
      224
    ],
    "reflection_max_abs_error": 0.0,
    "probability_shape": [
      2
    ],
    "all_probabilities_finite": true,
    "all_probabilities_in_contract": true,
    "router_feature_count": 10,
    "router_prototype_count": 137,
    "corrected_group_count": 3,
    "backbone_parameter_count": 21601152,
    "backbone_frozen": true
  },
  "exact_view_source_integrated": true,
  "float16_feature_cache_quantization_reproduced": true,
  "logit_space_blending_reproduced": true,
  "fold_local_phase12c_routing_present": true,
  "offline_dinov3_load_passed": true,
  "asset_hash_verification_passed": true,
  "elapsed_seconds": 19.84,
  "challenge_voxel

In [131]:
#Cell 94 — Resolve tuple-wrapped high-resolution
# cache and continue Phase30 component parity.

import json
import time

import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score

corrected_parity_started = time.perf_counter()

required_existing = [
    "phase30_packaged_runtime",
    "runtime_manifest",
    "runtime_probe",
    "runtime_device",
    "partition_by_fold",
    "PHASE19_PARTITIONS",
    "PHASE19_PHASE12C_OOF_PRIVATE",
    "PHASE30_COMPONENT_OOF_PRIVATE",
    "PHASE30_BLEND_OOF_PRIVATE",
    "PHASE19_Y_PRIVATE",
    "phase26_resolve_highres_cache",
]

missing_existing = [
    name for name in required_existing
    if name not in globals()
]

assert not missing_existing, {
    "message": (
        "The original Cell 94 did not reach the cache "
        "resolution line. Rerun it through DINO loading first."
    ),
    "missing_count": len(missing_existing),
}

# Reuse the backbone loaded before the original assertion failed.
if (
    "runtime_backbone" not in globals()
    or runtime_backbone is None
):
    runtime_backbone = (
        phase30_packaged_runtime.load_dinov3(
            runtime_manifest,
            runtime_device,
        )
    )

raw_highres_contract = (
    phase26_resolve_highres_cache()
)

expected_highres_shape = (
    1362, 80, 80, 80
)

highres_candidates = []
visited_objects = set()

def collect_highres_candidates(value):
    object_id = id(value)

    if object_id in visited_objects:
        return

    visited_objects.add(object_id)

    shape = getattr(value, "shape", None)

    if shape is not None:
        try:
            normalized_shape = tuple(
                int(dimension)
                for dimension in shape
            )
        except Exception:
            normalized_shape = None

        if normalized_shape == expected_highres_shape:
            highres_candidates.append(value)
            return

    if isinstance(value, dict):
        for nested_value in value.values():
            collect_highres_candidates(
                nested_value
            )

    elif isinstance(value, (tuple, list)):
        for nested_value in value:
            collect_highres_candidates(
                nested_value
            )

collect_highres_candidates(
    raw_highres_contract
)

# Deduplicate objects that may be referenced more than once.
unique_highres_candidates = []
seen_candidate_ids = set()

for candidate in highres_candidates:
    if id(candidate) not in seen_candidate_ids:
        unique_highres_candidates.append(
            candidate
        )
        seen_candidate_ids.add(id(candidate))

assert len(unique_highres_candidates) == 1, {
    "message": (
        "Could not uniquely resolve the 80x80x80 "
        "training cache."
    ),
    "matching_candidate_count":
        len(unique_highres_candidates),
    "resolver_return_type":
        type(raw_highres_contract).__name__,
}

runtime_highres_cache = (
    unique_highres_candidates[0]
)

assert tuple(
    runtime_highres_cache.shape
) == expected_highres_shape

fold0_outer_indices = np.asarray(
    partition_by_fold[0]["outer_valid"],
    dtype=np.int64,
)

assert fold0_outer_indices.shape == (467,)

runtime_component_fold0 = np.full(
    fold0_outer_indices.shape[0],
    np.nan,
    dtype=np.float64,
)

component_batch_size = 8

for local_start in range(
    0,
    fold0_outer_indices.size,
    component_batch_size,
):
    local_stop = min(
        local_start + component_batch_size,
        fold0_outer_indices.size,
    )

    source_indices = fold0_outer_indices[
        local_start:local_stop
    ]

    highres_batch = []

    for index in source_indices:
        cached_volume = runtime_highres_cache[
            int(index)
        ]

        if torch.is_tensor(cached_volume):
            cached_volume = (
                cached_volume.detach()
                .cpu()
                .numpy()
            )

        highres_batch.append(
            np.asarray(
                cached_volume,
                dtype=np.float32,
            )
        )

    runtime_component_fold0[
        local_start:local_stop
    ] = (
        phase30_packaged_runtime
        .extract_phase30_probability(
            highres_batch,
            runtime_backbone,
            runtime_probe,
            runtime_device,
        )
    )

    if (
        local_stop % 80 < component_batch_size
        or local_stop
        == fold0_outer_indices.size
    ):
        print(
            "Phase30 component parity: "
            f"{local_stop}/"
            f"{fold0_outer_indices.size}"
        )

assert np.isfinite(
    runtime_component_fold0
).all()

reference_component_fold0 = np.asarray(
    PHASE30_COMPONENT_OOF_PRIVATE,
    dtype=np.float64,
)[fold0_outer_indices]

component_absolute_error = np.abs(
    runtime_component_fold0
    - reference_component_fold0
)

baseline_fold0 = np.asarray(
    PHASE19_PHASE12C_OOF_PRIVATE,
    dtype=np.float64,
)[fold0_outer_indices]

runtime_blend_fold0 = (
    phase30_packaged_runtime.blend_phase30(
        baseline_fold0,
        runtime_component_fold0,
        runtime_probe["blend_alpha"],
    )
)

reference_blend_fold0 = np.asarray(
    PHASE30_BLEND_OOF_PRIVATE,
    dtype=np.float64,
)[fold0_outer_indices]

blend_absolute_error = np.abs(
    runtime_blend_fold0
    - reference_blend_fold0
)

labels_fold0 = np.asarray(
    PHASE19_Y_PRIVATE,
    dtype=np.int64,
)[fold0_outer_indices]

runtime_fold0_loss = float(
    log_loss(
        labels_fold0,
        np.clip(
            runtime_blend_fold0,
            1e-5,
            1.0 - 1e-5,
        ),
        labels=[0, 1],
    )
)

reference_fold0_loss = float(
    log_loss(
        labels_fold0,
        np.clip(
            reference_blend_fold0,
            1e-5,
            1.0 - 1e-5,
        ),
        labels=[0, 1],
    )
)

runtime_fold0_auroc = float(
    roc_auc_score(
        labels_fold0,
        runtime_blend_fold0,
    )
)

reference_fold0_auroc = float(
    roc_auc_score(
        labels_fold0,
        reference_blend_fold0,
    )
)

component_maximum_error = float(
    component_absolute_error.max()
)

component_mean_error = float(
    component_absolute_error.mean()
)

absolute_log_loss_delta = abs(
    runtime_fold0_loss
    - reference_fold0_loss
)

absolute_auroc_delta = abs(
    runtime_fold0_auroc
    - reference_fold0_auroc
)

parity_passed = bool(
    component_maximum_error <= 0.02
    and component_mean_error <= 0.002
    and absolute_log_loss_delta <= 0.0005
    and absolute_auroc_delta <= 0.0005
)

report = {
    "phase":
        "phase30_packaged_component_parity",
    "status": (
        "accepted"
        if parity_passed
        else "rejected"
    ),
    "cache_resolver_return_type":
        type(
            raw_highres_contract
        ).__name__,
    "resolved_cache_shape": list(
        runtime_highres_cache.shape
    ),
    "fold": 0,
    "case_count": int(
        fold0_outer_indices.size
    ),
    "component_probability_error": {
        "maximum": round(
            component_maximum_error,
            8,
        ),
        "mean": round(
            component_mean_error,
            8,
        ),
        "q99": round(
            float(
                np.quantile(
                    component_absolute_error,
                    0.99,
                )
            ),
            8,
        ),
    },
    "final_blend_probability_error": {
        "maximum": round(
            float(
                blend_absolute_error.max()
            ),
            8,
        ),
        "mean": round(
            float(
                blend_absolute_error.mean()
            ),
            8,
        ),
    },
    "runtime_fold0_log_loss": round(
        runtime_fold0_loss,
        6,
    ),
    "reference_fold0_log_loss": round(
        reference_fold0_loss,
        6,
    ),
    "absolute_log_loss_delta": round(
        absolute_log_loss_delta,
        8,
    ),
    "runtime_fold0_auroc": round(
        runtime_fold0_auroc,
        6,
    ),
    "reference_fold0_auroc": round(
        reference_fold0_auroc,
        6,
    ),
    "absolute_auroc_delta": round(
        absolute_auroc_delta,
        8,
    ),
    "float16_training_cache_contract":
        True,
    "outer_labels_used_only_for_parity_evaluation":
        True,
    "test_or_smoke_data_read": False,
    "case_level_predictions_exported":
        False,
    "elapsed_seconds": round(
        time.perf_counter()
        - corrected_parity_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE30_COMPONENT_PARITY"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE30_COMPONENT_PARITY"
)

assert parity_passed, (
    "Packaged Phase30 component parity failed. "
    "Do not run Cells 95–96."
)

PHASE30_RUNTIME_COMPONENT_FOLD0_PRIVATE = (
    runtime_component_fold0.copy()
)

runtime_backbone.cpu()
del runtime_backbone

import gc
gc.collect()
torch.cuda.empty_cache()

print(
    "Continue with Cell 95."
)

Phase30 component parity: 80/467
Phase30 component parity: 160/467
Phase30 component parity: 240/467
Phase30 component parity: 320/467
Phase30 component parity: 400/467
Phase30 component parity: 467/467
BEGIN SANITIZED_PHASE30_COMPONENT_PARITY
{
  "phase": "phase30_packaged_component_parity",
  "status": "accepted",
  "cache_resolver_return_type": "tuple",
  "resolved_cache_shape": [
    1362,
    80,
    80,
    80
  ],
  "fold": 0,
  "case_count": 467,
  "component_probability_error": {
    "maximum": 1.96e-06,
    "mean": 4e-08,
    "q99": 2.1e-07
  },
  "final_blend_probability_error": {
    "maximum": 2.7e-07,
    "mean": 1e-08
  },
  "runtime_fold0_log_loss": 0.331543,
  "reference_fold0_log_loss": 0.331543,
  "absolute_log_loss_delta": 0.0,
  "runtime_fold0_auroc": 0.928655,
  "reference_fold0_auroc": 0.928655,
  "absolute_auroc_delta": 0.0,
  "float16_training_cache_contract": true,
  "outer_labels_used_only_for_parity_evaluation": true,
  "test_or_smoke_data_read": false,
  "c

In [136]:
# Corrected Cell 95 — Normalize notebook cache batches to the
# exact deployed crop interface, then repeat routed OOF parity.

import gc
import json
import time

import numpy as np
import torch
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

corrected_routed_started = time.perf_counter()

required_existing = [
    "phase30_packaged_runtime",
    "runtime_router",
    "runtime_base_manifest",
    "runtime_base_models",
    "runtime_family_manifest",
    "runtime_anatomy_models",
    "runtime_device",
    "runtime_fallback",
    "build_predictor_crops",
    "case_df",
    "partition_by_fold",
    "fold0_outer_indices",
    "PHASE30_RUNTIME_COMPONENT_FOLD0_PRIVATE",
    "PHASE19_PHASE12C_OOF_PRIVATE",
    "PHASE30_BLEND_OOF_PRIVATE",
    "PHASE19_Y_PRIVATE",
    "runtime_probe",
]

missing = [
    name for name in required_existing
    if name not in globals()
]

assert not missing, {
    "message": (
        "Required state from the interrupted Cell 95 "
        "is missing."
    ),
    "missing_count": len(missing),
}

expected_crop_shapes = {
    "160": (64, 64, 64),
    "192": (64, 64, 64),
    "128hr": (80, 80, 80),
}

crop_aliases = {
    "160": ("160", "160mm"),
    "192": ("192", "192mm", "context"),
    "128hr": (
        "128hr",
        "highres",
        "128",
        "128mm",
    ),
}

def to_numpy_without_values(value):
    if torch.is_tensor(value):
        return (
            value.detach()
            .cpu()
            .numpy()
        )

    return np.asarray(value)

def resolve_raw_crop(raw_crops, target_name):
    for candidate_name in crop_aliases[
        target_name
    ]:
        if candidate_name in raw_crops:
            return raw_crops[candidate_name]

    raise KeyError(
        f"Missing crop contract: {target_name}"
    )

def normalize_one_crop(item, expected_shape):
    array = to_numpy_without_values(item)

    # Remove only a singleton channel dimension.
    while (
        array.ndim > 3
        and array.shape[0] == 1
    ):
        array = array[0]

    if tuple(array.shape) != expected_shape:
        raise ValueError(
            "Unexpected single-crop shape: "
            f"{tuple(array.shape)}; "
            f"expected={expected_shape}"
        )

    array = np.asarray(
        array,
        dtype=np.float32,
    )

    if not np.isfinite(array).all():
        raise ValueError(
            "Non-finite predictor crop"
        )

    return array

def normalize_predictor_crops(raw_crops):
    if not isinstance(raw_crops, dict):
        raise TypeError(
            "Predictor crop container is not a dictionary"
        )

    normalized = {}
    batch_size = None
    input_contract = {}

    for target_name, expected_shape in (
        expected_crop_shapes.items()
    ):
        raw_value = resolve_raw_crop(
            raw_crops,
            target_name,
        )

        input_contract[target_name] = {
            "container_type":
                type(raw_value).__name__,
            "shape": (
                list(raw_value.shape)
                if hasattr(raw_value, "shape")
                else None
            ),
        }

        if torch.is_tensor(raw_value):
            batched_array = (
                raw_value.detach()
                .cpu()
                .numpy()
            )

            if (
                batched_array.ndim == 5
                and batched_array.shape[1] == 1
            ):
                batched_array = (
                    batched_array[:, 0]
                )

            if batched_array.ndim == 3:
                batched_array = (
                    batched_array[None]
                )

            if (
                batched_array.ndim != 4
                or tuple(
                    batched_array.shape[1:]
                ) != expected_shape
            ):
                raise ValueError(
                    "Unexpected tensor crop batch shape: "
                    f"{tuple(batched_array.shape)}"
                )

            items = [
                np.asarray(
                    batched_array[index],
                    dtype=np.float32,
                )
                for index in range(
                    batched_array.shape[0]
                )
            ]

        elif isinstance(raw_value, np.ndarray):
            batched_array = raw_value

            if (
                batched_array.ndim == 5
                and batched_array.shape[1] == 1
            ):
                batched_array = (
                    batched_array[:, 0]
                )

            if batched_array.ndim == 3:
                batched_array = (
                    batched_array[None]
                )

            if (
                batched_array.ndim == 4
                and tuple(
                    batched_array.shape[1:]
                ) == expected_shape
            ):
                items = [
                    np.asarray(
                        batched_array[index],
                        dtype=np.float32,
                    )
                    for index in range(
                        batched_array.shape[0]
                    )
                ]
            else:
                raise ValueError(
                    "Unexpected NumPy crop batch shape: "
                    f"{tuple(batched_array.shape)}"
                )

        elif isinstance(
            raw_value,
            (list, tuple),
        ):
            items = [
                normalize_one_crop(
                    item,
                    expected_shape,
                )
                for item in raw_value
            ]

        else:
            raise TypeError(
                "Unsupported crop batch container: "
                f"{type(raw_value).__name__}"
            )

        if not items:
            raise ValueError(
                "Empty predictor crop batch"
            )

        if not all(
            tuple(item.shape) == expected_shape
            for item in items
        ):
            raise ValueError(
                "Normalized crop shape mismatch"
            )

        if batch_size is None:
            batch_size = len(items)
        elif len(items) != batch_size:
            raise ValueError(
                "Crop batch-size mismatch"
            )

        normalized[target_name] = items

    return normalized, input_contract

# ---------------------------------------------------------
# Direct two-case preflight. This bypasses the runtime
# fallback logic so an interface error cannot be hidden.
# ---------------------------------------------------------

preflight_indices = np.asarray(
    partition_by_fold[0]["outer_valid"][:2],
    dtype=np.int64,
)

raw_preflight_crops = build_predictor_crops(
    preflight_indices
)

(
    normalized_preflight_crops,
    parity_input_contract,
) = normalize_predictor_crops(
    raw_preflight_crops
)

fold0_base_manifest = (
    phase30_packaged_runtime.manifest_for_fold(
        runtime_base_manifest,
        0,
    )
)

fold0_family_manifest = (
    phase30_packaged_runtime.manifest_for_fold(
        runtime_family_manifest,
        0,
    )
)

direct_preflight_probability = (
    phase30_packaged_runtime.family
    .predict_family_batch(
        normalized_preflight_crops,
        fold0_family_manifest,
        fold0_base_manifest,
        runtime_base_models,
        runtime_anatomy_models,
        runtime_device,
    )
)

assert direct_preflight_probability.shape == (2,)
assert np.isfinite(
    direct_preflight_probability
).all()

# Ensure it did not return the constant fallback contract.
assert not np.allclose(
    direct_preflight_probability,
    runtime_fallback,
    atol=1e-7,
    rtol=0.0,
)

print(
    "Direct fold-local Phase12c preflight accepted."
)

# ---------------------------------------------------------
# Full three-fold routed reconstruction.
# ---------------------------------------------------------

package_baseline_oof = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)

groups_all = case_df[
    "acquisition_group"
].to_numpy(dtype=np.int64)

parity_batch_size = 8
completed_cases = 0

for fold in range(3):
    outer_indices = np.asarray(
        partition_by_fold[fold][
            "outer_valid"
        ],
        dtype=np.int64,
    )

    for local_start in range(
        0,
        outer_indices.size,
        parity_batch_size,
    ):
        local_stop = min(
            local_start + parity_batch_size,
            outer_indices.size,
        )

        batch_indices = outer_indices[
            local_start:local_stop
        ]

        raw_crop_batch = (
            build_predictor_crops(
                batch_indices
            )
        )

        crop_batch, _ = (
            normalize_predictor_crops(
                raw_crop_batch
            )
        )

        # The group/fold relationship is already frozen.
        # Call the selected fold directly to prevent any
        # fallback path from hiding an error during parity.
        local_base_manifest = (
            phase30_packaged_runtime
            .manifest_for_fold(
                runtime_base_manifest,
                fold,
            )
        )

        local_family_manifest = (
            phase30_packaged_runtime
            .manifest_for_fold(
                runtime_family_manifest,
                fold,
            )
        )

        batch_probability = (
            phase30_packaged_runtime.family
            .predict_family_batch(
                crop_batch,
                local_family_manifest,
                local_base_manifest,
                runtime_base_models,
                runtime_anatomy_models,
                runtime_device,
            )
        )

        assert batch_probability.shape == (
            batch_indices.size,
        )

        assert np.isfinite(
            batch_probability
        ).all()

        package_baseline_oof[
            batch_indices
        ] = batch_probability

        completed_cases += batch_indices.size

        if (
            completed_cases % 200
            < parity_batch_size
            or completed_cases == 1362
        ):
            print(
                "Exact Phase12c parity: "
                f"{completed_cases}/1362"
            )

assert np.isfinite(
    package_baseline_oof
).all()

reference_baseline_oof = np.asarray(
    PHASE19_PHASE12C_OOF_PRIVATE,
    dtype=np.float64,
)

baseline_absolute_error = np.abs(
    package_baseline_oof
    - reference_baseline_oof
)

package_phase30_oof = (
    package_baseline_oof.copy()
)

package_phase30_oof[
    fold0_outer_indices
] = (
    phase30_packaged_runtime.blend_phase30(
        package_baseline_oof[
            fold0_outer_indices
        ],
        PHASE30_RUNTIME_COMPONENT_FOLD0_PRIVATE,
        runtime_probe["blend_alpha"],
    )
)

reference_phase30_oof = np.asarray(
    PHASE30_BLEND_OOF_PRIVATE,
    dtype=np.float64,
)

final_absolute_error = np.abs(
    package_phase30_oof
    - reference_phase30_oof
)

labels = np.asarray(
    PHASE19_Y_PRIVATE,
    dtype=np.int64,
)

def calculate_parity_metrics(probability):
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        1e-5,
        1.0 - 1e-5,
    )

    return {
        "log_loss": float(
            log_loss(
                labels,
                probability,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                labels,
                probability,
            )
        ),
        "brier": float(
            brier_score_loss(
                labels,
                probability,
            )
        ),
    }

package_baseline_metrics = (
    calculate_parity_metrics(
        package_baseline_oof
    )
)

reference_baseline_metrics = (
    calculate_parity_metrics(
        reference_baseline_oof
    )
)

package_phase30_metrics = (
    calculate_parity_metrics(
        package_phase30_oof
    )
)

reference_phase30_metrics = (
    calculate_parity_metrics(
        reference_phase30_oof
    )
)

metric_deltas = {
    metric: abs(
        package_phase30_metrics[metric]
        - reference_phase30_metrics[metric]
    )
    for metric in (
        "log_loss",
        "auroc",
        "brier",
    )
}

parity_passed = bool(
    float(
        baseline_absolute_error.max()
    ) <= 0.002
    and metric_deltas["log_loss"] <= 0.0005
    and metric_deltas["auroc"] <= 0.0005
    and package_phase30_metrics[
        "log_loss"
    ] < 0.300
    and package_phase30_metrics[
        "auroc"
    ] > 0.941
)

report = {
    "phase":
        "phase30_complete_packaged_oof_parity",
    "status": (
        "accepted"
        if parity_passed
        else "rejected"
    ),
    "case_count": 1362,
    "notebook_crop_input_contract":
        parity_input_contract,
    "direct_fold_local_preflight":
        "accepted",
    "baseline_probability_error": {
        "maximum": round(
            float(
                baseline_absolute_error.max()
            ),
            8,
        ),
        "mean": round(
            float(
                baseline_absolute_error.mean()
            ),
            8,
        ),
    },
    "final_probability_error": {
        "maximum": round(
            float(
                final_absolute_error.max()
            ),
            8,
        ),
        "mean": round(
            float(
                final_absolute_error.mean()
            ),
            8,
        ),
    },
    "packaged_baseline": {
        key: round(value, 6)
        for key, value
        in package_baseline_metrics.items()
    },
    "reference_baseline": {
        key: round(value, 6)
        for key, value
        in reference_baseline_metrics.items()
    },
    "packaged_phase30": {
        key: round(value, 6)
        for key, value
        in package_phase30_metrics.items()
    },
    "reference_phase30": {
        key: round(value, 6)
        for key, value
        in reference_phase30_metrics.items()
    },
    "absolute_metric_deltas": {
        key: round(value, 8)
        for key, value
        in metric_deltas.items()
    },
    "routed_fold_count": 3,
    "phase30_corrected_case_count": int(
        fold0_outer_indices.size
    ),
    "outer_labels_used_only_for_final_parity_evaluation":
        True,
    "case_level_predictions_exported":
        False,
    "test_or_smoke_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - corrected_routed_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE30_COMPLETE_PARITY"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE30_COMPLETE_PARITY"
)

assert parity_passed, (
    "Corrected Phase30 parity failed. "
    "Do not run Cell 96."
)

PHASE30_PACKAGED_OOF_PRIVATE = (
    package_phase30_oof.copy()
)

for model in runtime_base_models.values():
    model.cpu()

for model in runtime_anatomy_models.values():
    model.cpu()

del runtime_base_models
del runtime_anatomy_models

gc.collect()
torch.cuda.empty_cache()

print(
    "Continue with Cell 96."
)

AssertionError: {'message': 'Required state from the interrupted Cell 95 is missing.', 'missing_count': 2}

In [135]:
# Cell 96 — Execute the packaged main.py against official smoke data.

import csv
import json
import os
from pathlib import Path
import subprocess
import sys
import time

smoke_candidates = set()

for path in Path("/kaggle/input").glob(
    "**/smoke_test_data"
):
    resolved = path.resolve()

    if (
        resolved.is_dir()
        and (resolved / "niftis").is_dir()
        and (
            resolved / "submission_format.csv"
        ).is_file()
    ):
        smoke_candidates.add(resolved)

assert smoke_candidates, {
    "message": "Official smoke-test directory was not found."
}

# Duplicate Kaggle mount aliases are harmless.
PHASE30_SMOKE_DATA_ROOT = sorted(
    smoke_candidates,
    key=lambda path: (
        len(path.parts),
        len(str(path)),
        str(path),
    ),
)[0]

PHASE30_SMOKE_OUTPUT = Path(
    "/kaggle/working/"
    "phase30_smoke_submission.csv"
)

if PHASE30_SMOKE_OUTPUT.exists():
    PHASE30_SMOKE_OUTPUT.unlink()

with (
    PHASE30_SMOKE_DATA_ROOT
    / "submission_format.csv"
).open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as handle:
    input_reader = csv.DictReader(handle)
    assert input_reader.fieldnames == [
        "uid",
        "is_pathologic",
    ]
    expected_smoke_count = sum(
        1 for _ in input_reader
    )

assert expected_smoke_count > 0

smoke_environment = os.environ.copy()
smoke_environment.update({
    "DAT_DATA_ROOT": str(
        PHASE30_SMOKE_DATA_ROOT
    ),
    "DAT_OUTPUT_CSV": str(
        PHASE30_SMOKE_OUTPUT
    ),
    "DAT_PREPROCESS_WORKERS": "4",
    "DAT_BATCH_SIZE": "4",
    "DAT_DINO_VIEW_BATCH": "12",
    "PYTHONPATH": str(
        PHASE30_PACKAGE_ROOT
    ),
    "PYTHONUNBUFFERED": "1",
})

smoke_started = time.perf_counter()

smoke_process = subprocess.run(
    [
        sys.executable,
        "main.py",
    ],
    cwd=str(PHASE30_PACKAGE_ROOT),
    env=smoke_environment,
    text=True,
    capture_output=True,
    timeout=360,
)

smoke_seconds = (
    time.perf_counter()
    - smoke_started
)

combined_log = (
    smoke_process.stdout.splitlines()
    + smoke_process.stderr.splitlines()
)

# Show only a bounded sanitized runtime log.
print("BEGIN PHASE30_SMOKE_LOG")

for line in combined_log[-80:]:
    print(line[:300])

print("END PHASE30_SMOKE_LOG")

assert smoke_process.returncode == 0, {
    "message": "Packaged smoke execution failed.",
    "return_code": smoke_process.returncode,
    "elapsed_seconds": round(
        smoke_seconds,
        2,
    ),
}

assert PHASE30_SMOKE_OUTPUT.is_file()

with PHASE30_SMOKE_OUTPUT.open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as handle:
    output_reader = csv.DictReader(handle)

    assert output_reader.fieldnames == [
        "uid",
        "is_pathologic",
    ]

    output_rows = list(output_reader)

assert len(output_rows) == expected_smoke_count

smoke_probabilities = np.asarray(
    [
        float(row["is_pathologic"])
        for row in output_rows
    ],
    dtype=np.float64,
)

assert np.isfinite(
    smoke_probabilities
).all()

assert (
    smoke_probabilities >= 1e-5
).all()

assert (
    smoke_probabilities <= 1.0 - 1e-5
).all()

assert smoke_seconds < 360

report = {
    "phase": "phase30_real_packaged_smoke_test",
    "status": "accepted",
    "case_count": expected_smoke_count,
    "output_row_count": len(output_rows),
    "schema_valid": True,
    "all_probabilities_finite": True,
    "all_probabilities_clipped": True,
    "return_code": smoke_process.returncode,
    "log_line_count": len(combined_log),
    "elapsed_seconds": round(
        smoke_seconds,
        2,
    ),
    "maximum_allowed_seconds": 360,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_exported":
        False,
    "smoke_data_read": True,
    "test_data_read": False,
    "model_hashes_displayed": False,
}

print(
    "BEGIN SANITIZED_PHASE30_REAL_SMOKE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE30_REAL_SMOKE"
)

BEGIN PHASE30_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed: cases=20, preprocessing_fallbacks=0, routing_fallbacks=0, phase30_fallbacks=0.
/kaggle/working/phase30_submission/phase12c_main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
END PHASE30_SMOKE_LOG
BEGIN SANITIZED_PHASE30_REAL_SMOKE
{
  "phase": "phase30_real_packaged_smoke_test",
  "status": "accepted",
  "case_count": 20,
  "output_row_count": 20,
  "schema_valid": true,
  "all_probabilities_finite": true,
  "all_probabilities_clipped": true,
  "return_code": 0,
  "log_line_count": 6,
  "elapsed_seconds": 20.99,
  "maximum_allowed_seconds": 360,
  "uids_displayed": false,
  "patient_rows_displayed": false,
  "case_level_predictions_exported": false,
  "smoke_data_read": true,
  "test_data_read": false,
  "model_hashes_displayed": false
}
END SANITIZED_PHAS

In [132]:
# Cell 97 — Clean, zip, validate, extract, and rerun smoke
# from the exact archive that will be submitted.

import ast
import csv
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import zipfile

finalization_started = time.perf_counter()

PHASE30_FINAL_ZIP = Path(
    "/kaggle/working/phase30_submission.zip"
)

PHASE30_REHEARSAL_ROOT = Path(
    "/kaggle/working/"
    "phase30_zip_rehearsal"
)

PHASE30_REHEARSAL_OUTPUT = Path(
    "/kaggle/working/"
    "phase30_zip_rehearsal_submission.csv"
)

PHASE30_REHEARSAL_HUB = Path(
    "/tmp/phase30_zip_rehearsal_hub"
)

assert PHASE30_PACKAGE_ROOT.resolve() == Path(
    "/kaggle/working/phase30_submission"
).resolve()

# ---------------------------------------------------------
# 1. Remove generated caches only.
# ---------------------------------------------------------

for cache_directory in list(
    PHASE30_PACKAGE_ROOT.rglob("__pycache__")
):
    if cache_directory.is_dir():
        shutil.rmtree(cache_directory)

for compiled_file in list(
    PHASE30_PACKAGE_ROOT.rglob("*.pyc")
):
    if compiled_file.is_file():
        compiled_file.unlink()

for compiled_file in list(
    PHASE30_PACKAGE_ROOT.rglob("*.pyo")
):
    if compiled_file.is_file():
        compiled_file.unlink()

# The package must not contain runtime output from prior tests.
for forbidden_output in [
    PHASE30_PACKAGE_ROOT / "submission.csv",
    PHASE30_PACKAGE_ROOT
    / "phase30_smoke_submission.csv",
]:
    if forbidden_output.exists():
        forbidden_output.unlink()

# ---------------------------------------------------------
# 2. Final source and asset contract.
# ---------------------------------------------------------

main_path = PHASE30_PACKAGE_ROOT / "main.py"
main_source = main_path.read_text(
    encoding="utf-8"
)

ast.parse(main_source)

assert "if __name__ == \"__main__\":" in main_source
assert "Built with DINOv3" in main_source
assert "phase12c_main" in main_source
assert "x_norm_clstoken" in main_source
assert "np.float16" in main_source
assert "blend_phase30" in main_source

# Runtime source must not contain a remote checkpoint URL.
assert "https://" not in main_source
assert "http://" not in main_source
assert "checkpoint.as_uri()" in main_source
assert 'source="local"' in main_source

required_root_files = [
    "main.py",
    "phase12c_main.py",
    "base_main.py",
    "manifest.json",
    "base_manifest.json",
    "phase30_manifest.json",
    "LICENSE",
    "README.md",
    "THIRD_PARTY_LICENSES.md",
    "DINOv3_LICENSE.md",
    "asset_license_registry.json",
]

for relative_path in required_root_files:
    assert (
        PHASE30_PACKAGE_ROOT
        / relative_path
    ).is_file(), relative_path

assert (
    PHASE30_PACKAGE_ROOT
    / "dinov3_repo"
    / "hubconf.py"
).is_file()

assert (
    PHASE30_PACKAGE_ROOT
    / "models"
    / "phase30_fold0_bilateral_probe.npz"
).is_file()

assert (
    PHASE30_PACKAGE_ROOT
    / "models"
    / "phase30_acquisition_router.npz"
).is_file()

phase12_weights = sorted(
    PHASE30_PACKAGE_ROOT.glob(
        "models/*.pt"
    )
)

dinov3_weights = sorted(
    PHASE30_PACKAGE_ROOT.glob(
        "models/*.pth"
    )
)

probe_assets = sorted(
    PHASE30_PACKAGE_ROOT.glob(
        "models/*.npz"
    )
)

assert len(phase12_weights) == 21
assert len(dinov3_weights) == 1
assert len(probe_assets) == 2

# Revalidate runtime hashes after cleanup.
phase30_packaged_runtime.verify_runtime_assets(
    runtime_manifest
)

# ---------------------------------------------------------
# 3. Archive root contents directly—no wrapping directory.
# ---------------------------------------------------------

if PHASE30_FINAL_ZIP.exists():
    PHASE30_FINAL_ZIP.unlink()

package_files = sorted(
    path
    for path in PHASE30_PACKAGE_ROOT.rglob("*")
    if path.is_file()
)

assert package_files

with zipfile.ZipFile(
    PHASE30_FINAL_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
    allowZip64=True,
) as archive:
    for path in package_files:
        relative_path = path.relative_to(
            PHASE30_PACKAGE_ROOT
        )

        archive.write(
            path,
            arcname=relative_path.as_posix(),
        )

assert PHASE30_FINAL_ZIP.is_file()

# ---------------------------------------------------------
# 4. Archive structural and safety audit.
# ---------------------------------------------------------

with zipfile.ZipFile(
    PHASE30_FINAL_ZIP,
    mode="r",
) as archive:
    archive_names = archive.namelist()
    archive_info = archive.infolist()

    corrupt_entry = archive.testzip()

assert corrupt_entry is None
assert archive_names
assert len(archive_names) == len(
    set(archive_names)
)

assert "main.py" in archive_names
assert "phase12c_main.py" in archive_names
assert "base_main.py" in archive_names
assert "manifest.json" in archive_names
assert "base_manifest.json" in archive_names
assert "phase30_manifest.json" in archive_names
assert "DINOv3_LICENSE.md" in archive_names
assert "dinov3_repo/hubconf.py" in archive_names

for archive_name in archive_names:
    archive_path = Path(archive_name)

    assert not archive_path.is_absolute()
    assert ".." not in archive_path.parts
    assert "__pycache__" not in archive_path.parts
    assert not archive_name.endswith(".pyc")
    assert not archive_name.endswith(".pyo")
    assert not archive_name.endswith(".nii")
    assert not archive_name.endswith(".nii.gz")
    assert (
        archive_path.name
        not in {
            "train_labels.csv",
            "submission.csv",
        }
    )

archive_phase12_weights = [
    name
    for name in archive_names
    if (
        name.startswith("models/")
        and name.endswith(".pt")
    )
]

archive_dinov3_weights = [
    name
    for name in archive_names
    if (
        name.startswith("models/")
        and name.endswith(".pth")
    )
]

archive_probe_assets = [
    name
    for name in archive_names
    if (
        name.startswith("models/")
        and name.endswith(".npz")
    )
]

assert len(archive_phase12_weights) == 21
assert len(archive_dinov3_weights) == 1
assert len(archive_probe_assets) == 2

uncompressed_size = sum(
    entry.file_size
    for entry in archive_info
)

compressed_size = PHASE30_FINAL_ZIP.stat().st_size

# ---------------------------------------------------------
# 5. Fresh extraction rehearsal.
# ---------------------------------------------------------

for target in [
    PHASE30_REHEARSAL_ROOT,
    PHASE30_REHEARSAL_HUB,
]:
    if target.exists():
        shutil.rmtree(target)

if PHASE30_REHEARSAL_OUTPUT.exists():
    PHASE30_REHEARSAL_OUTPUT.unlink()

PHASE30_REHEARSAL_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

with zipfile.ZipFile(
    PHASE30_FINAL_ZIP,
    mode="r",
) as archive:
    archive.extractall(
        PHASE30_REHEARSAL_ROOT
    )

assert (
    PHASE30_REHEARSAL_ROOT / "main.py"
).is_file()

assert not (
    PHASE30_REHEARSAL_ROOT
    / PHASE30_PACKAGE_ROOT.name
).exists()

rehearsal_environment = os.environ.copy()

rehearsal_environment.update({
    "DAT_DATA_ROOT": str(
        PHASE30_SMOKE_DATA_ROOT
    ),
    "DAT_OUTPUT_CSV": str(
        PHASE30_REHEARSAL_OUTPUT
    ),
    "DAT_PREPROCESS_WORKERS": "4",
    "DAT_BATCH_SIZE": "4",
    "DAT_DINO_VIEW_BATCH": "12",
    "DAT_TORCH_HUB": str(
        PHASE30_REHEARSAL_HUB
    ),
    "PYTHONPATH": str(
        PHASE30_REHEARSAL_ROOT
    ),
    "PYTHONUNBUFFERED": "1",
})

rehearsal_started = time.perf_counter()

rehearsal_process = subprocess.run(
    [
        sys.executable,
        "main.py",
    ],
    cwd=str(PHASE30_REHEARSAL_ROOT),
    env=rehearsal_environment,
    text=True,
    capture_output=True,
    timeout=360,
)

rehearsal_seconds = (
    time.perf_counter()
    - rehearsal_started
)

rehearsal_log = (
    rehearsal_process.stdout.splitlines()
    + rehearsal_process.stderr.splitlines()
)

print(
    "BEGIN PHASE30_EXTRACTED_ARCHIVE_SMOKE_LOG"
)

for line in rehearsal_log[-80:]:
    print(line[:300])

print(
    "END PHASE30_EXTRACTED_ARCHIVE_SMOKE_LOG"
)

assert rehearsal_process.returncode == 0, {
    "message": (
        "Extracted archive smoke execution failed."
    ),
    "return_code":
        rehearsal_process.returncode,
    "elapsed_seconds": round(
        rehearsal_seconds,
        2,
    ),
}

assert PHASE30_REHEARSAL_OUTPUT.is_file()

with PHASE30_REHEARSAL_OUTPUT.open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as handle:
    reader = csv.DictReader(handle)

    assert reader.fieldnames == [
        "uid",
        "is_pathologic",
    ]

    rehearsal_rows = list(reader)

assert len(rehearsal_rows) == (
    expected_smoke_count
)

rehearsal_probability = np.asarray(
    [
        float(row["is_pathologic"])
        for row in rehearsal_rows
    ],
    dtype=np.float64,
)

assert np.isfinite(
    rehearsal_probability
).all()

assert (
    rehearsal_probability >= 1e-5
).all()

assert (
    rehearsal_probability <= 1.0 - 1e-5
).all()

assert rehearsal_seconds < 360
assert len(rehearsal_log) <= 300

# ---------------------------------------------------------
# 6. Submission-level SHA-256.
# ---------------------------------------------------------

submission_digest = hashlib.sha256()

with PHASE30_FINAL_ZIP.open("rb") as handle:
    for block in iter(
        lambda: handle.read(1024 * 1024),
        b"",
    ):
        submission_digest.update(block)

submission_sha256 = (
    submission_digest.hexdigest()
)

total_seconds = (
    time.perf_counter()
    - finalization_started
)

report = {
    "phase": "phase30_final_submission_archive",
    "status": "accepted",
    "zip_name": PHASE30_FINAL_ZIP.name,
    "archive_file_count":
        len(archive_names),
    "compressed_size_mb": round(
        compressed_size / (1024 ** 2),
        2,
    ),
    "uncompressed_size_mb": round(
        uncompressed_size / (1024 ** 2),
        2,
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "phase12_model_count":
        len(archive_phase12_weights),
    "dinov3_checkpoint_count":
        len(archive_dinov3_weights),
    "phase30_probe_asset_count":
        len(archive_probe_assets),
    "required_attribution_present": (
        "Built with DINOv3"
        in (
            PHASE30_PACKAGE_ROOT
            / "README.md"
        ).read_text()
    ),
    "dinov3_license_present": (
        "DINOv3_LICENSE.md"
        in archive_names
    ),
    "extracted_archive_smoke": {
        "status": "accepted",
        "case_count":
            len(rehearsal_rows),
        "return_code":
            rehearsal_process.returncode,
        "elapsed_seconds": round(
            rehearsal_seconds,
            2,
        ),
        "log_line_count":
            len(rehearsal_log),
        "all_probabilities_finite":
            True,
        "all_probabilities_clipped":
            True,
        "fallback_count": 0,
    },
    "submission_sha256":
        submission_sha256,
    "total_finalization_seconds": round(
        total_seconds,
        2,
    ),
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "test_data_read": False,
    "smoke_data_read": True,
    "model_hashes_displayed": False,
}

print(
    "BEGIN SANITIZED_PHASE30_FINAL_ARCHIVE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE30_FINAL_ARCHIVE"
)

print(
    "\nFinal submission archive:\n"
    f"{PHASE30_FINAL_ZIP}"
)

BEGIN PHASE30_EXTRACTED_ARCHIVE_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Downloading: "file:///kaggle/working/phase30_zip_rehearsal/models/dinov3_vits16_pretrain_lvd1689m-08c60483.pth" to /tmp/phase30_zip_rehearsal_hub/checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth
Inference completed: cases=20, preprocessing_fallbacks=0, routing_fallbacks=0, phase30_fallbacks=0.
/kaggle/working/phase30_zip_rehearsal/phase12c_main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(

  0%|          | 0.00/82.5M [00:00<?, ?B/s]
100%|██████████| 82.5M/82.5M [00:00<00:00, 1.27GB/s]
END PHASE30_EXTRACTED_ARCHIVE_SMOKE_LOG
BEGIN SANITIZED_PHASE30_FINAL_ARCHIVE
{
  "phase": "phase30_final_submission_archive",
  "status": "accepted",
  "zip_name": "phase30_submission.zip",
  "archive_file_count": 256,
  "compressed_size_mb": 182.7,
  "uncompressed_siz

In [137]:
# Cell 98 — Construct a minimal DINOv3 source closure and
# verify direct checkpoint loading. This does not modify the
# accepted package or ZIP.

from pathlib import Path
import ast
import hashlib
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time

minimal_started = time.perf_counter()

PHASE30_MINIMAL_DINOV3 = Path(
    "/kaggle/working/"
    "phase30_dinov3_minimal_candidate"
).resolve()

assert str(PHASE30_MINIMAL_DINOV3).startswith(
    "/kaggle/working/"
)

if PHASE30_MINIMAL_DINOV3.exists():
    shutil.rmtree(
        PHASE30_MINIMAL_DINOV3
    )

PHASE30_MINIMAL_DINOV3.mkdir(
    parents=True,
    exist_ok=False,
)

# Resolve the original complete repository, never the staged copy.
preferred_source_repo = Path(
    "/kaggle/working/"
    "torch_hub_phase26_v3/"
    "facebookresearch_dinov3_main"
).resolve()

def valid_full_dinov3_repo(path):
    path = Path(path)

    return (
        path.is_dir()
        and (path / "hubconf.py").is_file()
        and (
            path
            / "dinov3"
            / "hub"
            / "backbones.py"
        ).is_file()
        and (
            path
            / "dinov3"
            / "models"
            / "vision_transformer.py"
        ).is_file()
    )

if valid_full_dinov3_repo(
    preferred_source_repo
):
    source_repo = preferred_source_repo
else:
    source_candidates = {
        path.resolve()
        for path in Path(
            "/kaggle/working"
        ).glob(
            "**/facebookresearch_dinov3_*"
        )
        if (
            valid_full_dinov3_repo(path)
            and PHASE30_PACKAGE_ROOT.resolve()
            not in path.resolve().parents
            and path.resolve()
            != (
                PHASE30_PACKAGE_ROOT
                / "dinov3_repo"
            ).resolve()
        )
    }

    assert source_candidates, {
        "message": (
            "Original complete DINOv3 source "
            "repository was not found."
        )
    }

    source_repo = sorted(
        source_candidates,
        key=lambda path: (
            len(path.parts),
            len(str(path)),
            str(path),
        ),
    )[0]

assert valid_full_dinov3_repo(source_repo)

# ---------------------------------------------------------
# Static recursive closure of internal dinov3 imports,
# beginning only from dinov3.hub.backbones.
# ---------------------------------------------------------

def module_to_source(module_name):
    relative = Path(
        *module_name.split(".")
    )

    module_file = (
        source_repo
        / relative.with_suffix(".py")
    )

    package_file = (
        source_repo
        / relative
        / "__init__.py"
    )

    if module_file.is_file():
        return module_file

    if package_file.is_file():
        return package_file

    return None

def module_name_for_source(path):
    relative = path.relative_to(
        source_repo
    )

    if relative.name == "__init__.py":
        return ".".join(
            relative.parent.parts
        )

    return ".".join(
        relative.with_suffix("").parts
    )

def internal_imports(path):
    source = path.read_text(
        encoding="utf-8"
    )

    tree = ast.parse(source)
    current_module = module_name_for_source(
        path
    )

    if path.name == "__init__.py":
        current_package = current_module
    else:
        current_package = (
            current_module.rsplit(".", 1)[0]
            if "." in current_module
            else ""
        )

    discovered = set()

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.startswith(
                    "dinov3"
                ):
                    discovered.add(
                        alias.name
                    )

        elif isinstance(
            node,
            ast.ImportFrom,
        ):
            if node.level:
                package_parts = (
                    current_package.split(".")
                    if current_package
                    else []
                )

                remove_count = max(
                    0,
                    int(node.level) - 1,
                )

                if remove_count:
                    package_parts = (
                        package_parts[
                            :-remove_count
                        ]
                    )

                base_parts = package_parts

                if node.module:
                    base_parts = (
                        base_parts
                        + node.module.split(".")
                    )

                base_module = ".".join(
                    base_parts
                )
            else:
                base_module = (
                    node.module or ""
                )

            if base_module.startswith(
                "dinov3"
            ):
                discovered.add(
                    base_module
                )

                # If an imported name is itself a module,
                # include it as well.
                for alias in node.names:
                    if alias.name == "*":
                        continue

                    candidate_module = (
                        base_module
                        + "."
                        + alias.name
                    )

                    if module_to_source(
                        candidate_module
                    ) is not None:
                        discovered.add(
                            candidate_module
                        )

    return discovered

modules_to_process = [
    "dinov3.hub.backbones"
]

processed_modules = set()
source_files = set()

while modules_to_process:
    module_name = modules_to_process.pop()

    if module_name in processed_modules:
        continue

    processed_modules.add(module_name)

    source_path = module_to_source(
        module_name
    )

    if source_path is None:
        continue

    source_files.add(source_path)

    # Include package __init__.py files for each parent.
    module_parts = module_name.split(".")

    for end in range(
        1,
        len(module_parts),
    ):
        parent_module = ".".join(
            module_parts[:end]
        )

        parent_source = module_to_source(
            parent_module
        )

        if parent_source is not None:
            source_files.add(
                parent_source
            )

            if (
                parent_module
                not in processed_modules
            ):
                modules_to_process.append(
                    parent_module
                )

    for imported_module in internal_imports(
        source_path
    ):
        if (
            imported_module
            not in processed_modules
        ):
            modules_to_process.append(
                imported_module
            )

# Copy the static source closure.
for source_path in sorted(source_files):
    relative_path = source_path.relative_to(
        source_repo
    )

    destination = (
        PHASE30_MINIMAL_DINOV3
        / relative_path
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source_path,
        destination,
    )

# Replace broad hubconf imports with the one runtime function.
minimal_hubconf = """
dependencies = ["torch"]

from dinov3.hub.backbones import dinov3_vits16

__all__ = ["dinov3_vits16"]
""".lstrip()

(
    PHASE30_MINIMAL_DINOV3
    / "hubconf.py"
).write_text(
    minimal_hubconf,
    encoding="utf-8",
)

# Bundle the applicable license inside the minimal source too.
license_source = next(
    (
        source_repo / filename
        for filename in (
            "LICENSE.md",
            "LICENSE",
        )
        if (
            source_repo / filename
        ).is_file()
    ),
    None,
)

assert license_source is not None

shutil.copy2(
    license_source,
    PHASE30_MINIMAL_DINOV3
    / "LICENSE.md",
)

minimal_python_files = sorted(
    PHASE30_MINIMAL_DINOV3.rglob(
        "*.py"
    )
)

assert minimal_python_files

# ---------------------------------------------------------
# Fresh-process test:
# 1. instantiate without external/default download;
# 2. directly load the bundled checkpoint;
# 3. run forward_features;
# 4. verify the class-token contract.
# ---------------------------------------------------------

checkpoint_path = (
    PHASE30_PACKAGE_ROOT
    / "models"
    / "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
).resolve()

assert checkpoint_path.is_file()

test_code = textwrap.dedent(
    r'''
    import inspect
    import json
    import sys
    from pathlib import Path

    import torch

    repository = Path(sys.argv[1]).resolve()
    checkpoint = Path(sys.argv[2]).resolve()

    sys.path.insert(0, str(repository))

    import hubconf

    factory = hubconf.dinov3_vits16
    signature = inspect.signature(factory)

    kwargs = {}

    if "pretrained" in signature.parameters:
        kwargs["pretrained"] = False

    if "weights" in signature.parameters:
        kwargs["weights"] = None

    model = factory(**kwargs)

    raw_state = torch.load(
        checkpoint,
        map_location="cpu",
        weights_only=True,
    )

    if not isinstance(raw_state, dict):
        raise RuntimeError(
            "Checkpoint is not a state dictionary"
        )

    state_candidates = [raw_state]

    for key in (
        "model",
        "state_dict",
        "teacher",
        "student",
        "backbone",
    ):
        nested = raw_state.get(key)

        if isinstance(nested, dict):
            state_candidates.append(nested)

    prefix_candidates = (
        "",
        "module.",
        "backbone.",
        "module.backbone.",
        "teacher.",
        "teacher.backbone.",
    )

    accepted = None

    for candidate in state_candidates:
        if not candidate:
            continue

        if not all(
            isinstance(key, str)
            for key in candidate
        ):
            continue

        for prefix in prefix_candidates:
            if prefix:
                transformed = {
                    (
                        key[len(prefix):]
                        if key.startswith(prefix)
                        else key
                    ): value
                    for key, value
                    in candidate.items()
                }
            else:
                transformed = candidate

            try:
                model.load_state_dict(
                    transformed,
                    strict=True,
                )
                accepted = transformed
                break
            except RuntimeError:
                continue

        if accepted is not None:
            break

    if accepted is None:
        raise RuntimeError(
            "No checkpoint state variant loaded strictly"
        )

    device = torch.device(
        "cuda:0"
        if torch.cuda.is_available()
        else "cpu"
    )

    model.eval().to(device)

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    synthetic = torch.rand(
        2,
        3,
        224,
        224,
        device=device,
    )

    with torch.inference_mode():
        output = model.forward_features(
            synthetic
        )

    assert isinstance(output, dict)
    assert "x_norm_clstoken" in output
    assert output[
        "x_norm_clstoken"
    ].shape == (2, 384)
    assert torch.isfinite(
        output["x_norm_clstoken"]
    ).all()

    report = {
        "status": "accepted",
        "factory_signature":
            str(signature),
        "factory_pretrained_disabled":
            bool(
                kwargs.get(
                    "pretrained",
                    True,
                ) is False
                or kwargs.get(
                    "weights",
                    "not_set",
                ) is None
            ),
        "strict_checkpoint_load": True,
        "parameter_count": int(
            sum(
                parameter.numel()
                for parameter
                in model.parameters()
            )
        ),
        "class_token_shape": list(
            output[
                "x_norm_clstoken"
            ].shape
        ),
        "all_outputs_finite": True,
        "device": str(device),
    }

    assert (
        report["parameter_count"]
        == 21601152
    )

    print(json.dumps(report))
    '''
)

test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = str(
    PHASE30_MINIMAL_DINOV3
)

test_process = subprocess.run(
    [
        sys.executable,
        "-c",
        test_code,
        str(PHASE30_MINIMAL_DINOV3),
        str(checkpoint_path),
    ],
    cwd=str(
        PHASE30_MINIMAL_DINOV3
    ),
    env=test_environment,
    text=True,
    capture_output=True,
    timeout=180,
)

if test_process.returncode != 0:
    print(
        "BEGIN MINIMAL_DINOV3_FAILURE"
    )
    print(test_process.stdout[-4000:])
    print(test_process.stderr[-4000:])
    print(
        "END MINIMAL_DINOV3_FAILURE"
    )

assert test_process.returncode == 0, {
    "message": (
        "Minimal DINOv3 candidate failed. "
        "The accepted ZIP was not modified."
    ),
    "return_code":
        test_process.returncode,
}

json_lines = [
    line
    for line in test_process.stdout.splitlines()
    if line.strip().startswith("{")
]

assert json_lines

runtime_test_report = json.loads(
    json_lines[-1]
)

minimal_file_count = len([
    path
    for path in PHASE30_MINIMAL_DINOV3.rglob("*")
    if path.is_file()
])

minimal_size_mb = sum(
    path.stat().st_size
    for path in PHASE30_MINIMAL_DINOV3.rglob("*")
    if path.is_file()
) / (1024 ** 2)

report = {
    "phase":
        "phase30_minimal_dinov3_candidate",
    "status": "accepted",
    "source_module_count":
        len(processed_modules),
    "python_file_count":
        len(minimal_python_files),
    "total_file_count":
        minimal_file_count,
    "source_size_mb": round(
        minimal_size_mb,
        3,
    ),
    "runtime_test":
        runtime_test_report,
    "checkpoint_copied_to_temporary_hub":
        False,
    "accepted_submission_modified":
        False,
    "accepted_zip_modified":
        False,
    "challenge_voxel_data_read": False,
    "test_or_smoke_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - minimal_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE30_MINIMAL_DINOV3"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE30_MINIMAL_DINOV3"
)

BEGIN SANITIZED_PHASE30_MINIMAL_DINOV3
{
  "phase": "phase30_minimal_dinov3_candidate",
  "status": "accepted",
  "source_module_count": 23,
  "python_file_count": 24,
  "total_file_count": 45,
  "source_size_mb": 0.232,
  "runtime_test": {
    "status": "accepted",
    "factory_signature": "(*, pretrained: bool = True, weights: Union[dinov3.hub.backbones.Weights, str] = <Weights.LVD1689M: 'LVD1689M'>, check_hash: bool = False, **kwargs)",
    "factory_pretrained_disabled": true,
    "strict_checkpoint_load": true,
    "parameter_count": 21601152,
    "class_token_shape": [
      2,
      384
    ],
    "all_outputs_finite": true,
    "device": "cuda:0"
  },
  "checkpoint_copied_to_temporary_hub": false,
  "accepted_submission_modified": false,
  "accepted_zip_modified": false,
  "challenge_voxel_data_read": false,
  "test_or_smoke_data_read": false,
  "model_hashes_displayed": false,
  "elapsed_seconds": 4.74
}
END SANITIZED_PHASE30_MINIMAL_DINOV3


In [138]:
# Cell 99 — Replace group-routed deployment with:
#
# full Phase12c ensemble
# + validated fold0-to-DINO residual for groups 1,5,9.
#
# Also replace the full DINOv3 repository with the validated
# minimal runtime and load the checkpoint directly.

from pathlib import Path
import ast
import json
import shutil
import time

patch_started = time.perf_counter()

package_main_path = (
    PHASE30_PACKAGE_ROOT / "main.py"
)

package_repo_path = (
    PHASE30_PACKAGE_ROOT / "dinov3_repo"
)

assert package_main_path.is_file()
assert PHASE30_MINIMAL_DINOV3.is_dir()

# Preserve the previous accepted archive until the new archive
# passes all tests.
assert PHASE30_FINAL_ZIP.is_file()

# ---------------------------------------------------------
# 1. Replace the large source repository.
# ---------------------------------------------------------

assert package_repo_path.resolve() == (
    PHASE30_PACKAGE_ROOT
    / "dinov3_repo"
).resolve()

if package_repo_path.exists():
    shutil.rmtree(package_repo_path)

shutil.copytree(
    PHASE30_MINIMAL_DINOV3,
    package_repo_path,
)

assert (
    package_repo_path / "hubconf.py"
).is_file()

minimal_repository_file_count = len([
    path
    for path in package_repo_path.rglob("*")
    if path.is_file()
])

assert minimal_repository_file_count == 45

# ---------------------------------------------------------
# 2. Patch DINOv3 loading to avoid copying 82.5 MB to /tmp.
# ---------------------------------------------------------

source = package_main_path.read_text(
    encoding="utf-8"
)

load_start = source.index(
    "def load_dinov3(manifest, device):"
)

load_end = source.index(
    "\n\n@torch.inference_mode()\n"
    "def extract_phase30_probability",
    load_start,
)

new_load_function = r'''def load_dinov3(manifest, device):
    repository = (
        ASSET_ROOT
        / manifest["dinov3"]["source_root"]
    ).resolve()

    checkpoint = resolve_asset(
        manifest["dinov3"]["checkpoint"]
    )

    if not (repository / "hubconf.py").is_file():
        raise RuntimeError(
            "Missing offline DINOv3 repository"
        )

    backbone = torch.hub.load(
        str(repository),
        "dinov3_vits16",
        source="local",
        pretrained=False,
        weights=None,
        trust_repo=True,
        verbose=False,
    )

    raw_state = torch.load(
        checkpoint,
        map_location="cpu",
        weights_only=True,
    )

    if not isinstance(raw_state, dict):
        raise RuntimeError(
            "Invalid DINOv3 checkpoint container"
        )

    state_candidates = [raw_state]

    for key in (
        "model",
        "state_dict",
        "teacher",
        "student",
        "backbone",
    ):
        nested = raw_state.get(key)

        if isinstance(nested, dict):
            state_candidates.append(nested)

    prefixes = (
        "",
        "module.",
        "backbone.",
        "module.backbone.",
        "teacher.",
        "teacher.backbone.",
    )

    accepted = False

    for candidate in state_candidates:
        if not candidate:
            continue

        for prefix in prefixes:
            if prefix:
                transformed = {
                    (
                        key[len(prefix):]
                        if key.startswith(prefix)
                        else key
                    ): value
                    for key, value
                    in candidate.items()
                }
            else:
                transformed = candidate

            try:
                backbone.load_state_dict(
                    transformed,
                    strict=True,
                )
                accepted = True
                break
            except RuntimeError:
                continue

        if accepted:
            break

    if not accepted:
        raise RuntimeError(
            "DINOv3 checkpoint loading failed"
        )

    backbone.eval().to(device)

    for parameter in backbone.parameters():
        parameter.requires_grad_(False)

    return backbone
'''

source = (
    source[:load_start]
    + new_load_function
    + source[load_end:]
)

# ---------------------------------------------------------
# 3. Add a safe full/fold-specific family predictor.
# ---------------------------------------------------------

helper_marker = (
    "\ndef load_phase12c_bundle():\n"
)

helper_position = source.index(
    helper_marker
)

safe_predictor_source = r'''

def predict_phase12c_safe(
    crops,
    selected_family_manifest,
    selected_base_manifest,
    base_models,
    anatomy_models,
    device,
    fallback,
):
    case_count = len(crops["128hr"])

    probability = np.full(
        case_count,
        float(fallback),
        dtype=np.float64,
    )

    try:
        probability[:] = (
            family.predict_family_batch(
                crops,
                selected_family_manifest,
                selected_base_manifest,
                base_models,
                anatomy_models,
                device,
            )
        )
    except Exception:
        for index in range(case_count):
            one_case = subset_crops(
                crops,
                [index],
            )

            try:
                probability[index] = float(
                    family.predict_family_batch(
                        one_case,
                        selected_family_manifest,
                        selected_base_manifest,
                        base_models,
                        anatomy_models,
                        device,
                    )[0]
                )
            except Exception:
                probability[index] = float(
                    fallback
                )

    return np.clip(
        probability,
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )
'''

source = (
    source[:helper_position]
    + safe_predictor_source
    + source[helper_position:]
)

# ---------------------------------------------------------
# 4. Replace routed baseline logic in main().
# ---------------------------------------------------------

inference_start_marker = (
    "                baseline = "
    "predict_routed_phase12c(\n"
)

inference_end_marker = (
    "                for local_index, "
    "probability in zip(\n"
)

block_start = source.index(
    inference_start_marker
)

block_end = source.index(
    inference_end_marker,
    block_start,
)

new_inference_block = r'''                # Final test inference uses the complete
                # three-fold Phase12c ensemble.
                full_baseline = (
                    predict_phase12c_safe(
                        crops=crop_batch,
                        selected_family_manifest=
                            family_manifest,
                        selected_base_manifest=
                            base_manifest,
                        base_models=base_models,
                        anatomy_models=
                            anatomy_models,
                        device=device,
                        fallback=fallback,
                    )
                )

                final_batch = (
                    full_baseline.copy()
                )

                corrected_local = [
                    index
                    for index, group
                    in enumerate(group_batch)
                    if int(group)
                    in corrected_groups
                ]

                if corrected_local:
                    try:
                        if backbone is None:
                            backbone = load_dinov3(
                                runtime_manifest,
                                device,
                            )

                        corrected_crops = (
                            subset_crops(
                                crop_batch,
                                corrected_local,
                            )
                        )

                        fold0_base_manifest = (
                            manifest_for_fold(
                                base_manifest,
                                0,
                            )
                        )

                        fold0_family_manifest = (
                            manifest_for_fold(
                                family_manifest,
                                0,
                            )
                        )

                        fold0_baseline = (
                            predict_phase12c_safe(
                                crops=corrected_crops,
                                selected_family_manifest=
                                    fold0_family_manifest,
                                selected_base_manifest=
                                    fold0_base_manifest,
                                base_models=base_models,
                                anatomy_models=
                                    anatomy_models,
                                device=device,
                                fallback=fallback,
                            )
                        )

                        component = (
                            extract_phase30_probability(
                                corrected_crops[
                                    "128hr"
                                ],
                                backbone,
                                probe,
                                device,
                            )
                        )

                        # Transfer the leakage-free, validated
                        # fold0-to-DINO residual onto the stronger
                        # full three-fold ensemble anchor.
                        residual_delta = (
                            probe["blend_alpha"]
                            * (
                                probability_logit(
                                    component
                                )
                                - probability_logit(
                                    fold0_baseline
                                )
                            )
                        )

                        anchored_logit = (
                            probability_logit(
                                full_baseline[
                                    corrected_local
                                ]
                            )
                            + residual_delta
                        )

                        final_batch[
                            corrected_local
                        ] = np.clip(
                            probability_sigmoid(
                                anchored_logit
                            ),
                            PROBABILITY_EPSILON,
                            1.0
                            - PROBABILITY_EPSILON,
                        )

                    except Exception:
                        # Retain the complete Phase12c ensemble.
                        phase30_fallbacks += len(
                            corrected_local
                        )

'''

source = (
    source[:block_start]
    + new_inference_block
    + source[block_end:]
)

# The old routed helper may remain in source for audit/history,
# but it is no longer called by main().
ast.parse(source)

assert (
    source.count(
        "predict_routed_phase12c("
    )
    == 1
), (
    "The routed predictor should remain only as its "
    "unused function definition."
)

assert (
    "full_baseline = "
    in source
)

assert (
    "residual_delta = "
    in source
)

assert (
    "checkpoint.as_uri()"
    not in source
)

assert (
    'pretrained=False'
    in source
)

package_main_path.write_text(
    source,
    encoding="utf-8",
)

# ---------------------------------------------------------
# 5. Update the deployment manifest and README.
# ---------------------------------------------------------

phase30_manifest_path = (
    PHASE30_PACKAGE_ROOT
    / "phase30_manifest.json"
)

deployment_manifest = json.loads(
    phase30_manifest_path.read_text(
        encoding="utf-8"
    )
)

deployment_manifest[
    "deployment_policy"
] = {
    "case_independent": True,
    "baseline": (
        "complete_three_fold_phase12c_ensemble"
    ),
    "phase30_update": (
        "fold0_validated_logit_residual_"
        "anchored_to_full_ensemble"
    ),
    "residual_formula": (
        "z_full + alpha*(z_dinov3-z_fold0)"
    ),
    "alpha": 0.30,
    "corrected_groups": [1, 5, 9],
    "unseen_header_policy": (
        "nearest_standardized_training_"
        "header_prototype"
    ),
    "test_distribution_used": False,
    "leaderboard_feedback_used": (
        "group_routed_deployment_rejected_"
        "after_0.3419_log_loss"
    ),
}

phase30_manifest_path.write_text(
    json.dumps(
        deployment_manifest,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

readme_path = (
    PHASE30_PACKAGE_ROOT / "README.md"
)

readme = readme_path.read_text(
    encoding="utf-8"
)

deployment_note = """
## Final inference policy

The final runtime uses the complete three-fold Phase12c ensemble for
every case. For acquisition groups 1, 5, and 9, it adds the validated
fold-0-to-DINOv3 logit residual to that full-ensemble prediction:

`z_final = z_full + 0.30 * (z_dinov3 - z_fold0)`.

The acquisition router is fixed from training headers and operates
independently per case.
""".strip()

if "## Final inference policy" in readme:
    readme = readme.split(
        "## Final inference policy"
    )[0].rstrip()

readme_path.write_text(
    readme.rstrip()
    + "\n\n"
    + deployment_note
    + "\n",
    encoding="utf-8",
)

# ---------------------------------------------------------
# 6. Static patch audit.
# ---------------------------------------------------------

package_file_count = len([
    path
    for path in PHASE30_PACKAGE_ROOT.rglob("*")
    if path.is_file()
])

package_size_mb = sum(
    path.stat().st_size
    for path in PHASE30_PACKAGE_ROOT.rglob("*")
    if path.is_file()
) / (1024 ** 2)

report = {
    "phase":
        "phase30_full_ensemble_anchor_patch",
    "status": "accepted",
    "deployment_baseline":
        "complete_three_fold_phase12c",
    "phase30_formula": (
        "z_full + 0.30*(z_dinov3-z_fold0)"
    ),
    "corrected_groups": [1, 5, 9],
    "minimal_dinov3_file_count":
        minimal_repository_file_count,
    "checkpoint_loading":
        "direct_strict_state_dict",
    "checkpoint_temporary_copy":
        False,
    "package_file_count":
        package_file_count,
    "package_size_mb": round(
        package_size_mb,
        2,
    ),
    "previous_accepted_zip_preserved":
        True,
    "test_or_smoke_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - patch_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE30_FULL_ENSEMBLE_PATCH"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE30_FULL_ENSEMBLE_PATCH"
)

BEGIN SANITIZED_PHASE30_FULL_ENSEMBLE_PATCH
{
  "phase": "phase30_full_ensemble_anchor_patch",
  "status": "accepted",
  "deployment_baseline": "complete_three_fold_phase12c",
  "phase30_formula": "z_full + 0.30*(z_dinov3-z_fold0)",
  "corrected_groups": [
    1,
    5,
    9
  ],
  "minimal_dinov3_file_count": 45,
  "checkpoint_loading": "direct_strict_state_dict",
  "checkpoint_temporary_copy": false,
  "package_file_count": 82,
  "package_size_mb": 184.26,
  "previous_accepted_zip_preserved": true,
  "test_or_smoke_data_read": false,
  "model_hashes_displayed": false,
  "elapsed_seconds": 0.04
}
END SANITIZED_PHASE30_FULL_ENSEMBLE_PATCH


In [139]:
# Cell 100 — Validate the patched full-ensemble deployment,
# build a new archive, extract it, and rerun smoke.

import ast
import csv
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np

validation_started = time.perf_counter()

PHASE30_ANCHORED_ZIP = Path(
    "/kaggle/working/"
    "phase30_full_ensemble_submission.zip"
)

STAGED_SMOKE_OUTPUT = Path(
    "/kaggle/working/"
    "phase30_full_ensemble_staged_smoke.csv"
)

EXTRACTED_ROOT = Path(
    "/kaggle/working/"
    "phase30_full_ensemble_rehearsal"
)

EXTRACTED_SMOKE_OUTPUT = Path(
    "/kaggle/working/"
    "phase30_full_ensemble_extracted_smoke.csv"
)

FRESH_HUB_DIRECTORY = Path(
    "/tmp/"
    "phase30_full_ensemble_rehearsal_hub"
)

# Resolve smoke data again if notebook state changed.
if (
    "PHASE30_SMOKE_DATA_ROOT"
    not in globals()
    or not Path(
        PHASE30_SMOKE_DATA_ROOT
    ).is_dir()
):
    smoke_candidates = {
        path.resolve()
        for path in Path(
            "/kaggle/input"
        ).glob("**/smoke_test_data")
        if (
            path.is_dir()
            and (path / "niftis").is_dir()
            and (
                path
                / "submission_format.csv"
            ).is_file()
        )
    }

    assert smoke_candidates

    PHASE30_SMOKE_DATA_ROOT = sorted(
        smoke_candidates,
        key=lambda path: (
            len(path.parts),
            len(str(path)),
            str(path),
        ),
    )[0]

PHASE30_SMOKE_DATA_ROOT = Path(
    PHASE30_SMOKE_DATA_ROOT
).resolve()

with (
    PHASE30_SMOKE_DATA_ROOT
    / "submission_format.csv"
).open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == [
        "uid",
        "is_pathologic",
    ]
    expected_smoke_count = sum(
        1 for _ in reader
    )

assert expected_smoke_count > 0

# ---------------------------------------------------------
# 1. Patched source audit.
# ---------------------------------------------------------

main_path = (
    PHASE30_PACKAGE_ROOT / "main.py"
)

source = main_path.read_text(
    encoding="utf-8"
)

ast.parse(source)

assert (
    "full_baseline = "
    in source
)

assert (
    "residual_delta = "
    in source
)

assert (
    'pretrained=False'
    in source
)

assert (
    "checkpoint.as_uri()"
    not in source
)

assert (
    source.count(
        "predict_routed_phase12c("
    )
    == 1
)

assert (
    len([
        path
        for path in (
            PHASE30_PACKAGE_ROOT
            / "dinov3_repo"
        ).rglob("*")
        if path.is_file()
    ])
    == 45
)

deployment_manifest = json.loads(
    (
        PHASE30_PACKAGE_ROOT
        / "phase30_manifest.json"
    ).read_text(
        encoding="utf-8"
    )
)

assert (
    deployment_manifest[
        "deployment_policy"
    ]["baseline"]
    == "complete_three_fold_phase12c_ensemble"
)

assert (
    deployment_manifest[
        "deployment_policy"
    ]["residual_formula"]
    == "z_full + alpha*(z_dinov3-z_fold0)"
)

# Asset hashes are unchanged and must still pass.
phase30_packaged_runtime.verify_runtime_assets(
    deployment_manifest
)

# ---------------------------------------------------------
# 2. Run smoke directly from the patched staging package.
# ---------------------------------------------------------

for target_file in [
    STAGED_SMOKE_OUTPUT,
    EXTRACTED_SMOKE_OUTPUT,
]:
    if target_file.exists():
        target_file.unlink()

for directory in [
    EXTRACTED_ROOT,
    FRESH_HUB_DIRECTORY,
]:
    if directory.exists():
        shutil.rmtree(directory)

staged_environment = os.environ.copy()

staged_environment.update({
    "DAT_DATA_ROOT": str(
        PHASE30_SMOKE_DATA_ROOT
    ),
    "DAT_OUTPUT_CSV": str(
        STAGED_SMOKE_OUTPUT
    ),
    "DAT_PREPROCESS_WORKERS": "4",
    "DAT_BATCH_SIZE": "4",
    "DAT_DINO_VIEW_BATCH": "12",
    "DAT_TORCH_HUB": str(
        FRESH_HUB_DIRECTORY
    ),
    "PYTHONPATH": str(
        PHASE30_PACKAGE_ROOT
    ),
    "PYTHONUNBUFFERED": "1",
})

staged_started = time.perf_counter()

staged_process = subprocess.run(
    [
        sys.executable,
        "main.py",
    ],
    cwd=str(PHASE30_PACKAGE_ROOT),
    env=staged_environment,
    text=True,
    capture_output=True,
    timeout=360,
)

staged_seconds = (
    time.perf_counter()
    - staged_started
)

staged_log = (
    staged_process.stdout.splitlines()
    + staged_process.stderr.splitlines()
)

print(
    "BEGIN PHASE30_ANCHORED_STAGED_SMOKE_LOG"
)

for line in staged_log[-80:]:
    print(line[:300])

print(
    "END PHASE30_ANCHORED_STAGED_SMOKE_LOG"
)

assert staged_process.returncode == 0, {
    "message": (
        "Patched staging smoke failed."
    ),
    "return_code":
        staged_process.returncode,
}

assert STAGED_SMOKE_OUTPUT.is_file()

with STAGED_SMOKE_OUTPUT.open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == [
        "uid",
        "is_pathologic",
    ]
    staged_rows = list(reader)

assert len(staged_rows) == (
    expected_smoke_count
)

staged_probability = np.asarray(
    [
        float(row["is_pathologic"])
        for row in staged_rows
    ],
    dtype=np.float64,
)

assert np.isfinite(
    staged_probability
).all()

assert (
    staged_probability >= 1e-5
).all()

assert (
    staged_probability <= 1.0 - 1e-5
).all()

assert any(
    "preprocessing_fallbacks=0"
    in line
    and "routing_fallbacks=0"
    in line
    and "phase30_fallbacks=0"
    in line
    for line in staged_log
)

# ---------------------------------------------------------
# 3. Remove caches generated by staged execution.
# ---------------------------------------------------------

for cache_directory in list(
    PHASE30_PACKAGE_ROOT.rglob(
        "__pycache__"
    )
):
    if cache_directory.is_dir():
        shutil.rmtree(cache_directory)

for compiled_file in list(
    PHASE30_PACKAGE_ROOT.rglob(
        "*.pyc"
    )
):
    if compiled_file.is_file():
        compiled_file.unlink()

# ---------------------------------------------------------
# 4. Build a new archive without overwriting the old one.
# ---------------------------------------------------------

if PHASE30_ANCHORED_ZIP.exists():
    PHASE30_ANCHORED_ZIP.unlink()

package_files = sorted(
    path
    for path
    in PHASE30_PACKAGE_ROOT.rglob("*")
    if path.is_file()
)

with zipfile.ZipFile(
    PHASE30_ANCHORED_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
    allowZip64=True,
) as archive:
    for path in package_files:
        archive.write(
            path,
            arcname=path.relative_to(
                PHASE30_PACKAGE_ROOT
            ).as_posix(),
        )

assert PHASE30_ANCHORED_ZIP.is_file()

with zipfile.ZipFile(
    PHASE30_ANCHORED_ZIP,
    mode="r",
) as archive:
    archive_names = archive.namelist()
    archive_info = archive.infolist()
    corrupt_entry = archive.testzip()

assert corrupt_entry is None
assert len(archive_names) == len(
    set(archive_names)
)

assert "main.py" in archive_names
assert "phase12c_main.py" in archive_names
assert "phase30_manifest.json" in archive_names
assert "dinov3_repo/hubconf.py" in archive_names
assert "DINOv3_LICENSE.md" in archive_names

# This keeps zip listing + unzip logging comfortably below
# the 300-line execution-log limit.
assert len(archive_names) <= 100, (
    len(archive_names)
)

for name in archive_names:
    path = Path(name)

    assert not path.is_absolute()
    assert ".." not in path.parts
    assert "__pycache__" not in path.parts
    assert not name.endswith(".pyc")
    assert not name.endswith(".nii")
    assert not name.endswith(".nii.gz")
    assert path.name not in {
        "submission.csv",
        "train_labels.csv",
    }

# ---------------------------------------------------------
# 5. Extract the exact new archive and rerun smoke.
# ---------------------------------------------------------

EXTRACTED_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

with zipfile.ZipFile(
    PHASE30_ANCHORED_ZIP,
    mode="r",
) as archive:
    archive.extractall(
        EXTRACTED_ROOT
    )

assert (
    EXTRACTED_ROOT / "main.py"
).is_file()

if FRESH_HUB_DIRECTORY.exists():
    shutil.rmtree(
        FRESH_HUB_DIRECTORY
    )

extracted_environment = (
    os.environ.copy()
)

extracted_environment.update({
    "DAT_DATA_ROOT": str(
        PHASE30_SMOKE_DATA_ROOT
    ),
    "DAT_OUTPUT_CSV": str(
        EXTRACTED_SMOKE_OUTPUT
    ),
    "DAT_PREPROCESS_WORKERS": "4",
    "DAT_BATCH_SIZE": "4",
    "DAT_DINO_VIEW_BATCH": "12",
    "DAT_TORCH_HUB": str(
        FRESH_HUB_DIRECTORY
    ),
    "PYTHONPATH": str(
        EXTRACTED_ROOT
    ),
    "PYTHONUNBUFFERED": "1",
})

extracted_started = time.perf_counter()

extracted_process = subprocess.run(
    [
        sys.executable,
        "main.py",
    ],
    cwd=str(EXTRACTED_ROOT),
    env=extracted_environment,
    text=True,
    capture_output=True,
    timeout=360,
)

extracted_seconds = (
    time.perf_counter()
    - extracted_started
)

extracted_log = (
    extracted_process.stdout.splitlines()
    + extracted_process.stderr.splitlines()
)

print(
    "BEGIN PHASE30_ANCHORED_EXTRACTED_SMOKE_LOG"
)

for line in extracted_log[-80:]:
    print(line[:300])

print(
    "END PHASE30_ANCHORED_EXTRACTED_SMOKE_LOG"
)

assert extracted_process.returncode == 0, {
    "message": (
        "Extracted anchored archive smoke failed."
    ),
    "return_code":
        extracted_process.returncode,
}

assert EXTRACTED_SMOKE_OUTPUT.is_file()

with EXTRACTED_SMOKE_OUTPUT.open(
    "r",
    newline="",
    encoding="utf-8-sig",
) as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == [
        "uid",
        "is_pathologic",
    ]
    extracted_rows = list(reader)

assert len(extracted_rows) == (
    expected_smoke_count
)

extracted_probability = np.asarray(
    [
        float(row["is_pathologic"])
        for row in extracted_rows
    ],
    dtype=np.float64,
)

assert np.isfinite(
    extracted_probability
).all()

assert (
    extracted_probability >= 1e-5
).all()

assert (
    extracted_probability <= 1.0 - 1e-5
).all()

archive_reconstruction_error = float(
    np.max(
        np.abs(
            extracted_probability
            - staged_probability
        )
    )
)

assert archive_reconstruction_error <= 1e-8

assert any(
    "preprocessing_fallbacks=0"
    in line
    and "routing_fallbacks=0"
    in line
    and "phase30_fallbacks=0"
    in line
    for line in extracted_log
)

assert len(extracted_log) <= 100

# ---------------------------------------------------------
# 6. Final archive digest and report.
# ---------------------------------------------------------

digest = hashlib.sha256()

with PHASE30_ANCHORED_ZIP.open(
    "rb"
) as handle:
    for block in iter(
        lambda: handle.read(
            1024 * 1024
        ),
        b"",
    ):
        digest.update(block)

anchored_zip_sha256 = (
    digest.hexdigest()
)

compressed_size_mb = (
    PHASE30_ANCHORED_ZIP.stat().st_size
    / (1024 ** 2)
)

uncompressed_size_mb = (
    sum(
        entry.file_size
        for entry in archive_info
    )
    / (1024 ** 2)
)

report = {
    "phase":
        "phase30_full_ensemble_final_archive",
    "status": "accepted",
    "zip_name":
        PHASE30_ANCHORED_ZIP.name,
    "deployment": {
        "baseline":
            "complete_three_fold_phase12c",
        "phase30_formula": (
            "z_full + 0.30*"
            "(z_dinov3-z_fold0)"
        ),
        "corrected_groups": [1, 5, 9],
    },
    "archive_file_count":
        len(archive_names),
    "compressed_size_mb": round(
        compressed_size_mb,
        2,
    ),
    "uncompressed_size_mb": round(
        uncompressed_size_mb,
        2,
    ),
    "archive_integrity_passed": True,
    "runtime_log_budget_safe": True,
    "direct_checkpoint_loading": True,
    "temporary_checkpoint_copy": False,
    "staged_smoke": {
        "status": "accepted",
        "case_count":
            len(staged_rows),
        "elapsed_seconds": round(
            staged_seconds,
            2,
        ),
        "log_line_count":
            len(staged_log),
        "fallback_count": 0,
    },
    "extracted_archive_smoke": {
        "status": "accepted",
        "case_count":
            len(extracted_rows),
        "elapsed_seconds": round(
            extracted_seconds,
            2,
        ),
        "log_line_count":
            len(extracted_log),
        "fallback_count": 0,
    },
    "archive_probability_reconstruction_error":
        round(
            archive_reconstruction_error,
            10,
        ),
    "submission_sha256":
        anchored_zip_sha256,
    "previous_routed_zip_preserved": (
        PHASE30_FINAL_ZIP.is_file()
    ),
    "test_data_read": False,
    "smoke_data_read": True,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "model_hashes_displayed": False,
    "total_seconds": round(
        time.perf_counter()
        - validation_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE30_FULL_ENSEMBLE_ARCHIVE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE30_FULL_ENSEMBLE_ARCHIVE"
)

print(
    f"{PHASE30_ANCHORED_ZIP}"
)

BEGIN PHASE30_ANCHORED_STAGED_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed: cases=20, preprocessing_fallbacks=0, routing_fallbacks=0, phase30_fallbacks=0.
/kaggle/working/phase30_submission/phase12c_main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
END PHASE30_ANCHORED_STAGED_SMOKE_LOG
BEGIN PHASE30_ANCHORED_EXTRACTED_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed: cases=20, preprocessing_fallbacks=0, routing_fallbacks=0, phase30_fallbacks=0.
/kaggle/working/phase30_full_ensemble_rehearsal/phase12c_main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
END PHASE30_ANCHORED_EXTRACTED_SMOKE_LOG
BEGIN SANITIZED_PHASE30_FULL_ENSEMBLE_ARCHIVE
{
  "phase"

> Phase 31

In [3]:
# Phase31 Cell 97A
# Fresh-kernel reconstruction of case order, acquisition groups,
# and the exact Phase18/19 frozen partitions.

import itertools
import json
import math
import os
import time
import zipfile
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

EXPECTED_CASES = 1362
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")


def phase31_convert_binary(values):
    values = np.asarray(values)

    if values.shape != (EXPECTED_CASES,):
        raise ValueError("Unexpected label shape.")

    if np.issubdtype(values.dtype, np.number):
        numeric = values.astype(np.float64)
        unique = set(np.unique(numeric).tolist())

        if unique.issubset({0.0, 1.0}):
            return numeric.astype(np.int64)

    text = np.asarray([
        str(value).strip().lower()
        for value in values
    ])

    output = np.full(
        EXPECTED_CASES,
        -1,
        dtype=np.int64,
    )

    for index, value in enumerate(text):
        if value in {
            "0", "normal", "negative",
            "healthy", "control",
        }:
            output[index] = 0
        elif value in {
            "1", "pathologic", "pathological",
            "abnormal", "positive", "disease",
        }:
            output[index] = 1
        elif "normal" in value or "control" in value:
            output[index] = 0
        elif (
            "patholog" in value
            or "abnormal" in value
            or "parkinson" in value
        ):
            output[index] = 1

    assert np.all(output >= 0), {
        "message": "Some labels could not be converted.",
        "unresolved_count": int(np.sum(output < 0)),
    }

    return output


# ---------------------------------------------------------
# Resolve the authoritative 1,362-row training-label file.
# ---------------------------------------------------------

label_candidates = []

for path in KAGGLE_INPUT_ROOT.rglob("train_labels.csv"):
    try:
        frame = pd.read_csv(path)

        if len(frame) == EXPECTED_CASES:
            label_candidates.append((path, frame))
    except Exception:
        pass

assert label_candidates, (
    "No 1,362-row train_labels.csv was found."
)

# Prefer the shortest/direct dataset path if Kaggle exposes aliases.
label_candidates.sort(
    key=lambda item: (
        len(item[0].parts),
        len(str(item[0])),
    )
)

train_labels_path, case_df = label_candidates[0]
case_df = case_df.reset_index(drop=True).copy()

lower_columns = {
    str(column).lower(): column
    for column in case_df.columns
}

uid_column = None

for alias in [
    "uid",
    "case_uid",
    "case_id",
    "patient_id",
    "id",
]:
    if alias in lower_columns:
        uid_column = lower_columns[alias]
        break

assert uid_column is not None, {
    "message": "Could not identify the UID column.",
    "columns": sorted(str(c) for c in case_df.columns),
}

label_column = None

for alias in [
    "label",
    "target",
    "pathologic",
    "pathological",
    "abnormal",
    "diagnosis",
    "class",
]:
    if alias in lower_columns:
        candidate_column = lower_columns[alias]

        try:
            candidate_labels = phase31_convert_binary(
                case_df[candidate_column].to_numpy()
            )
            label_column = candidate_column
            break
        except Exception:
            pass

if label_column is None:
    for column in case_df.columns:
        if column == uid_column:
            continue

        try:
            candidate_labels = phase31_convert_binary(
                case_df[column].to_numpy()
            )
            label_column = column
            break
        except Exception:
            pass

assert label_column is not None

PHASE19_LABELS = phase31_convert_binary(
    case_df[label_column].to_numpy()
)

# Give subsequent cells a stable numerical label column.
case_df["label"] = PHASE19_LABELS


# ---------------------------------------------------------
# Resolve all training NIfTIs without reading voxel arrays.
# ---------------------------------------------------------

def phase31_nifti_identifier(path):
    name = path.name

    if name.lower().endswith(".nii.gz"):
        return name[:-7]
    if name.lower().endswith(".nii"):
        return name[:-4]

    return path.stem


nifti_paths = []

for pattern in ("*.nii.gz", "*.nii"):
    for path in KAGGLE_INPUT_ROOT.rglob(pattern):
        lower_path = str(path).lower()

        if "smoke" in lower_path or "test" in lower_path:
            continue

        nifti_paths.append(path)

nifti_paths = sorted(set(nifti_paths))

basename_index = {}
parent_index = {}

for path in nifti_paths:
    basename_key = phase31_nifti_identifier(path).lower()
    parent_key = path.parent.name.lower()

    basename_index.setdefault(
        basename_key,
        [],
    ).append(path)

    parent_index.setdefault(
        parent_key,
        [],
    ).append(path)


resolved_case_paths = []
ambiguous_case_count = 0

for uid in case_df[uid_column].astype(str):
    uid_key = uid.strip().lower()

    matches = basename_index.get(uid_key, [])

    if not matches:
        matches = parent_index.get(uid_key, [])

    if len(matches) != 1:
        ambiguous_case_count += 1
        resolved_case_paths.append(None)
    else:
        resolved_case_paths.append(matches[0])

assert ambiguous_case_count == 0, {
    "message": "Some training NIfTIs were missing or ambiguous.",
    "failure_count": ambiguous_case_count,
    "indexed_nifti_count": len(nifti_paths),
}

case_df["path"] = resolved_case_paths


# ---------------------------------------------------------
# Resolve the saved Phase30 acquisition router.
# Multiple identical copies are acceptable.
# ---------------------------------------------------------

def phase31_valid_router(path):
    try:
        with np.load(path, allow_pickle=False) as asset:
            required = {
                "header_columns",
                "prototypes",
                "prototype_groups",
                "feature_mean",
                "feature_scale",
                "fold_for_group",
                "rounding_decimals",
            }

            if not required.issubset(asset.files):
                return False

            if asset["prototypes"].ndim != 2:
                return False

            if asset["prototypes"].shape[1] != 10:
                return False

            if asset["fold_for_group"].shape != (15,):
                return False

        return True
    except Exception:
        return False


router_candidates = [
    path
    for path in KAGGLE_WORKING_ROOT.rglob(
        "phase30_acquisition_router.npz"
    )
    if phase31_valid_router(path)
]

# If only the zip remains, extract this single small asset.
if not router_candidates:
    zip_candidates = sorted(
        KAGGLE_WORKING_ROOT.glob("phase30*.zip"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    bootstrap_asset_root = (
        KAGGLE_WORKING_ROOT
        / "phase31_bootstrap_assets"
    )
    bootstrap_asset_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for zip_path in zip_candidates:
        try:
            with zipfile.ZipFile(zip_path) as archive:
                member = next(
                    (
                        name
                        for name in archive.namelist()
                        if name.endswith(
                            "models/phase30_acquisition_router.npz"
                        )
                    ),
                    None,
                )

                if member is None:
                    continue

                destination = (
                    bootstrap_asset_root
                    / "phase30_acquisition_router.npz"
                )

                with (
                    archive.open(member) as source,
                    open(destination, "wb") as target,
                ):
                    target.write(source.read())

                if phase31_valid_router(destination):
                    router_candidates.append(destination)
                    break
        except Exception:
            pass

assert router_candidates, (
    "Could not find the saved Phase30 acquisition router."
)

router_candidates.sort(
    key=lambda path: (
        "phase30_submission" not in str(path),
        len(path.parts),
        len(str(path)),
    )
)

phase31_router_path = router_candidates[0]

with np.load(
    phase31_router_path,
    allow_pickle=False,
) as asset:
    HEADER_COLS = [
        str(value)
        for value in asset["header_columns"].tolist()
    ]
    router_prototypes = np.asarray(
        asset["prototypes"],
        dtype=np.float64,
    )
    router_prototype_groups = np.asarray(
        asset["prototype_groups"],
        dtype=np.int64,
    )
    router_feature_mean = np.asarray(
        asset["feature_mean"],
        dtype=np.float64,
    )
    router_feature_scale = np.asarray(
        asset["feature_scale"],
        dtype=np.float64,
    )
    router_fold_for_group = np.asarray(
        asset["fold_for_group"],
        dtype=np.int64,
    )
    router_rounding_decimals = int(
        asset["rounding_decimals"][0]
    )

assert len(HEADER_COLS) == 10
assert router_prototypes.shape[1] == 10
assert router_prototype_groups.shape == (
    router_prototypes.shape[0],
)
assert router_fold_for_group.shape == (15,)


def phase31_header_vector(path):
    image = nib.load(str(path), mmap=True)

    assert len(image.shape) == 3

    shape = np.asarray(
        image.shape,
        dtype=np.float64,
    )
    spacing = np.asarray(
        image.header.get_zooms()[:3],
        dtype=np.float64,
    )
    affine = np.asarray(
        image.affine,
        dtype=np.float64,
    )

    assert np.isfinite(shape).all()
    assert np.isfinite(spacing).all()
    assert np.isfinite(affine).all()
    assert np.all(shape > 0)
    assert np.all(spacing > 0)

    fov = shape * spacing

    values = {
        "shape_x": float(shape[0]),
        "shape_y": float(shape[1]),
        "shape_z": float(shape[2]),
        "spacing_x": float(spacing[0]),
        "spacing_y": float(spacing[1]),
        "spacing_z": float(spacing[2]),
        "fov_x": float(fov[0]),
        "fov_y": float(fov[1]),
        "fov_z": float(fov[2]),
        "dtype_is_int16": float(
            str(image.get_data_dtype()) == "int16"
        ),
    }

    return np.asarray(
        [values[column] for column in HEADER_COLS],
        dtype=np.float64,
    )


header_matrix = np.stack([
    phase31_header_vector(path)
    for path in case_df["path"]
])

rounded_header_matrix = np.round(
    header_matrix,
    decimals=router_rounding_decimals,
)

router_scale_safe = np.where(
    router_feature_scale < 1e-8,
    1.0,
    router_feature_scale,
)

phase20_acquisition_groups = np.empty(
    EXPECTED_CASES,
    dtype=np.int64,
)

minimum_router_distances = np.empty(
    EXPECTED_CASES,
    dtype=np.float64,
)

for start in range(0, EXPECTED_CASES, 256):
    stop = min(start + 256, EXPECTED_CASES)

    differences = (
        rounded_header_matrix[start:stop, None, :]
        - router_prototypes[None, :, :]
    ) / router_scale_safe[None, None, :]

    distances = np.sum(
        differences * differences,
        axis=2,
    )

    nearest = np.argmin(
        distances,
        axis=1,
    )

    phase20_acquisition_groups[start:stop] = (
        router_prototype_groups[nearest]
    )

    minimum_router_distances[start:stop] = distances[
        np.arange(stop - start),
        nearest,
    ]

assert set(
    np.unique(phase20_acquisition_groups).tolist()
) == set(range(15))

case_df["acquisition_group"] = (
    phase20_acquisition_groups
)

for column_index, column in enumerate(HEADER_COLS):
    case_df[column] = header_matrix[:, column_index]


# ---------------------------------------------------------
# Reconstruct the exact outer and nested partitions.
# The monitor subset is identified by its reported
# size, positive count, and four-group contract.
# ---------------------------------------------------------

expected_nested = {
    0: {
        "outer_valid_n": 467,
        "outer_train_n": 895,
        "monitor_n": 174,
        "monitor_positive": 91,
        "monitor_group_count": 4,
    },
    1: {
        "outer_valid_n": 443,
        "outer_train_n": 919,
        "monitor_n": 123,
        "monitor_positive": 69,
        "monitor_group_count": 4,
    },
    2: {
        "outer_valid_n": 452,
        "outer_train_n": 910,
        "monitor_n": 196,
        "monitor_positive": 102,
        "monitor_group_count": 4,
    },
}


def phase31_reconstruct_partition(fold):
    outer_valid_mask = (
        router_fold_for_group[
            phase20_acquisition_groups
        ]
        == fold
    )

    outer_valid = np.flatnonzero(outer_valid_mask)
    outer_train = np.flatnonzero(~outer_valid_mask)

    contract = expected_nested[fold]

    assert len(outer_valid) == contract["outer_valid_n"]
    assert len(outer_train) == contract["outer_train_n"]

    train_groups = np.unique(
        phase20_acquisition_groups[outer_train]
    )

    monitor_candidates = []

    for group_combination in itertools.combinations(
        train_groups.tolist(),
        contract["monitor_group_count"],
    ):
        monitor_mask = np.isin(
            phase20_acquisition_groups,
            np.asarray(group_combination),
        )

        monitor = np.flatnonzero(
            monitor_mask & ~outer_valid_mask
        )

        if len(monitor) != contract["monitor_n"]:
            continue

        positive_count = int(
            PHASE19_LABELS[monitor].sum()
        )

        if positive_count != contract["monitor_positive"]:
            continue

        monitor_candidates.append(
            np.asarray(
                group_combination,
                dtype=np.int64,
            )
        )

    assert len(monitor_candidates) == 1, {
        "message": "Nested monitor group reconstruction was ambiguous.",
        "fold": fold,
        "candidate_count": len(monitor_candidates),
    }

    monitor_groups = monitor_candidates[0]

    monitor = np.flatnonzero(
        np.isin(
            phase20_acquisition_groups,
            monitor_groups,
        )
        & ~outer_valid_mask
    )

    fit = np.setdiff1d(
        outer_train,
        monitor,
        assume_unique=True,
    )

    assert len(
        np.intersect1d(fit, monitor)
    ) == 0
    assert len(
        np.intersect1d(outer_train, outer_valid)
    ) == 0

    return {
        "fold": int(fold),
        "fit": fit.astype(np.int64),
        "monitor": monitor.astype(np.int64),
        "outer_train": outer_train.astype(np.int64),
        "outer_valid": outer_valid.astype(np.int64),
    }


PHASE19_PARTITIONS = [
    phase31_reconstruct_partition(fold)
    for fold in range(3)
]

coverage = np.zeros(
    EXPECTED_CASES,
    dtype=np.int64,
)

for partition in PHASE19_PARTITIONS:
    coverage[partition["outer_valid"]] += 1

assert np.all(coverage == 1)

report = {
    "phase": "phase31_fresh_kernel_state_restore",
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "normal_count": int(
        np.sum(PHASE19_LABELS == 0)
    ),
    "pathologic_count": int(
        np.sum(PHASE19_LABELS == 1)
    ),
    "training_nifti_count": len(resolved_case_paths),
    "header_feature_count": len(HEADER_COLS),
    "router_prototype_count": int(
        router_prototypes.shape[0]
    ),
    "acquisition_group_count": int(
        np.unique(
            phase20_acquisition_groups
        ).size
    ),
    "exact_header_prototype_match_fraction": round(
        float(
            np.mean(
                minimum_router_distances <= 1e-16
            )
        ),
        8,
    ),
    "folds": [
        {
            "fold": int(partition["fold"]),
            "fit_n": len(partition["fit"]),
            "monitor_n": len(partition["monitor"]),
            "outer_train_n": len(
                partition["outer_train"]
            ),
            "outer_valid_n": len(
                partition["outer_valid"]
            ),
            "monitor_prevalence": round(
                float(
                    PHASE19_LABELS[
                        partition["monitor"]
                    ].mean()
                ),
                6,
            ),
        }
        for partition in PHASE19_PARTITIONS
    ],
    "voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
}

print("BEGIN SANITIZED_PHASE31_FRESH_STATE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_FRESH_STATE")

BEGIN SANITIZED_PHASE31_FRESH_STATE
{
  "phase": "phase31_fresh_kernel_state_restore",
  "status": "accepted",
  "case_count": 1362,
  "normal_count": 615,
  "pathologic_count": 747,
  "training_nifti_count": 1362,
  "header_feature_count": 10,
  "router_prototype_count": 137,
  "acquisition_group_count": 15,
  "exact_header_prototype_match_fraction": 1.0,
  "folds": [
    {
      "fold": 0,
      "fit_n": 721,
      "monitor_n": 174,
      "outer_train_n": 895,
      "outer_valid_n": 467,
      "monitor_prevalence": 0.522989
    },
    {
      "fold": 1,
      "fit_n": 796,
      "monitor_n": 123,
      "outer_train_n": 919,
      "outer_valid_n": 443,
      "monitor_prevalence": 0.560976
    },
    {
      "fold": 2,
      "fit_n": 714,
      "monitor_n": 196,
      "outer_train_n": 910,
      "outer_valid_n": 452,
      "monitor_prevalence": 0.520408
    }
  ],
  "voxel_arrays_read": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "uids_displayed": false,
  "patient_

In [4]:
# Phase31 Cell 97B
# Reconstruct only the exact Phase12c 128 mm / 80^3 crop.
# The completed float16 cache is persisted across kernel restarts.

from concurrent.futures import ThreadPoolExecutor
from scipy.ndimage import affine_transform

PHASE31_CACHE_PATH = (
    KAGGLE_WORKING_ROOT
    / "phase31_highres_float16.npy"
)

PHASE31_PARTIAL_CACHE_PATH = (
    KAGGLE_WORKING_ROOT
    / "phase31_highres_float16.partial.npy"
)

phase31_cache_started = time.perf_counter()
phase31_cache_reused = False


def phase31_load_canonical_volume(path):
    image = nib.as_closest_canonical(
        nib.load(str(path), mmap=True)
    )

    if len(image.shape) != 3:
        raise ValueError(
            "Expected one three-dimensional NIfTI."
        )

    array = np.asarray(
        image.dataobj,
        dtype=np.float32,
    )

    array = np.nan_to_num(
        array,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    array = np.clip(
        array,
        0.0,
        None,
    )

    return image, array


def phase31_robust_uptake_center(image, array):
    positive = array[array > 0]

    if positive.size < 32:
        raise ValueError("Too few positive voxels.")

    shape = np.asarray(
        array.shape,
        dtype=np.int64,
    )

    lower = np.floor(
        shape * np.asarray(
            [0.20, 0.20, 0.10]
        )
    ).astype(np.int64)

    upper = np.ceil(
        shape * np.asarray(
            [0.80, 0.80, 0.75]
        )
    ).astype(np.int64)

    central = array[
        lower[0]:upper[0],
        lower[1]:upper[1],
        lower[2]:upper[2],
    ]

    central_positive = central[
        central > 0
    ]

    if central_positive.size < 32:
        raise ValueError(
            "Too few central positive voxels."
        )

    threshold = float(
        np.percentile(
            central_positive,
            99.0,
        )
    )

    coordinates = (
        np.argwhere(
            central >= threshold
        )
        + lower[None, :]
    )

    values = array[
        tuple(coordinates.T)
    ].astype(np.float64)

    keep = values > 0
    coordinates = coordinates[keep]
    values = values[keep]

    if values.size < 8:
        raise ValueError("Too few hot voxels.")

    weights = np.maximum(
        values - threshold,
        0.0,
    )

    if weights.sum() <= 0:
        weights = np.ones_like(values)

    center_voxel = np.average(
        coordinates,
        axis=0,
        weights=weights,
    )

    return nib.affines.apply_affine(
        image.affine,
        center_voxel,
    )


def phase31_resample_highres(
    image,
    array,
    center_world,
):
    output_shape = np.asarray(
        [80, 80, 80],
        dtype=np.int64,
    )

    cube_mm = 128.0
    spacing = cube_mm / output_shape[0]

    inverse = np.linalg.inv(
        image.affine
    )

    output_to_world = np.diag([
        spacing,
        spacing,
        spacing,
    ])

    world_origin = (
        np.asarray(
            center_world,
            dtype=np.float64,
        )
        - spacing
        * (
            output_shape.astype(np.float64)
            - 1.0
        )
        / 2.0
    )

    matrix = (
        inverse[:3, :3]
        @ output_to_world
    )

    offset = (
        inverse[:3, :3]
        @ world_origin
        + inverse[:3, 3]
    )

    return affine_transform(
        array,
        matrix=matrix,
        offset=offset,
        output_shape=(80, 80, 80),
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=False,
    ).astype(
        np.float32,
        copy=False,
    )


def phase31_percentile_normalize(volume):
    volume = np.clip(
        np.asarray(
            volume,
            dtype=np.float32,
        ),
        0.0,
        None,
    )

    positive = volume[volume > 0]

    if positive.size < 8:
        raise ValueError(
            "Too few positive crop voxels."
        )

    scale = max(
        float(
            np.percentile(
                positive,
                99.5,
            )
        ),
        1e-6,
    )

    return np.clip(
        volume / scale,
        0.0,
        1.5,
    ).astype(np.float32)


def phase31_preprocess_highres(record):
    index, path = record

    try:
        image, array = (
            phase31_load_canonical_volume(path)
        )

        center_world = (
            phase31_robust_uptake_center(
                image,
                array,
            )
        )

        highres = phase31_resample_highres(
            image,
            array,
            center_world,
        )

        highres = (
            phase31_percentile_normalize(
                highres
            )
        )

        assert highres.shape == (80, 80, 80)
        assert np.isfinite(highres).all()

        return index, highres, None

    except Exception as exc:
        return (
            index,
            None,
            type(exc).__name__,
        )


def phase31_validate_saved_cache(path):
    if not path.is_file():
        return None

    try:
        candidate = np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        )

        if candidate.shape != (
            EXPECTED_CASES,
            80,
            80,
            80,
        ):
            return None

        if candidate.dtype != np.float16:
            return None

        check_indices = np.linspace(
            0,
            EXPECTED_CASES - 1,
            12,
            dtype=np.int64,
        )

        for index in check_indices:
            sample = np.asarray(
                candidate[index],
                dtype=np.float32,
            )

            if not np.isfinite(sample).all():
                return None

            if sample.min() < 0:
                return None

            if sample.max() > 1.5001:
                return None

        return candidate

    except Exception:
        return None


PHASE19_HIGHRES_CACHE = (
    phase31_validate_saved_cache(
        PHASE31_CACHE_PATH
    )
)

failure_types = []

if PHASE19_HIGHRES_CACHE is not None:
    phase31_cache_reused = True

else:
    highres_writer = np.lib.format.open_memmap(
        PHASE31_PARTIAL_CACHE_PATH,
        mode="w+",
        dtype=np.float16,
        shape=(
            EXPECTED_CASES,
            80,
            80,
            80,
        ),
    )

    records = list(enumerate(
        case_df["path"].tolist()
    ))

    with ThreadPoolExecutor(
        max_workers=8
    ) as executor:
        iterator = executor.map(
            phase31_preprocess_highres,
            records,
            chunksize=1,
        )

        for completed, (
            index,
            highres,
            error_type,
        ) in enumerate(iterator, start=1):

            if error_type is not None:
                failure_types.append(error_type)
            else:
                highres_writer[index] = (
                    highres.astype(
                        np.float16
                    )
                )

            if (
                completed % 200 == 0
                or completed == EXPECTED_CASES
            ):
                highres_writer.flush()
                print(
                    "Phase31 persistent highres preprocessing: "
                    f"{completed}/{EXPECTED_CASES}"
                )

    highres_writer.flush()
    del highres_writer

    assert not failure_types, {
        "message": "High-resolution preprocessing failed.",
        "failure_count": len(failure_types),
        "failure_type_counts": {
            error_type: failure_types.count(error_type)
            for error_type in sorted(set(failure_types))
        },
    }

    os.replace(
        PHASE31_PARTIAL_CACHE_PATH,
        PHASE31_CACHE_PATH,
    )

    PHASE19_HIGHRES_CACHE = (
        phase31_validate_saved_cache(
            PHASE31_CACHE_PATH
        )
    )

    assert PHASE19_HIGHRES_CACHE is not None


validation_indices = np.linspace(
    0,
    EXPECTED_CASES - 1,
    20,
    dtype=np.int64,
)

validation_samples = np.stack([
    np.asarray(
        PHASE19_HIGHRES_CACHE[index],
        dtype=np.float32,
    )
    for index in validation_indices
])

assert validation_samples.shape == (
    20, 80, 80, 80
)
assert np.isfinite(
    validation_samples
).all()

elapsed_seconds = (
    time.perf_counter()
    - phase31_cache_started
)

report = {
    "phase": "phase31_persistent_highres_cache",
    "status": "accepted",
    "cache_reused": phase31_cache_reused,
    "cache_name": PHASE31_CACHE_PATH.name,
    "shape": list(
        PHASE19_HIGHRES_CACHE.shape
    ),
    "dtype": str(
        PHASE19_HIGHRES_CACHE.dtype
    ),
    "size_gb": round(
        PHASE31_CACHE_PATH.stat().st_size
        / (1024 ** 3),
        4,
    ),
    "preprocessing": (
        "exact_phase12c_128mm_80cube_"
        "independent_positive_p99.5"
    ),
    "sampled_minimum": round(
        float(validation_samples.min()),
        6,
    ),
    "sampled_maximum": round(
        float(validation_samples.max()),
        6,
    ),
    "sampled_mean": round(
        float(validation_samples.mean()),
        6,
    ),
    "failure_count": 0,
    "elapsed_seconds": round(
        elapsed_seconds,
        2,
    ),
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "training_voxel_arrays_read": not phase31_cache_reused,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
}

print("BEGIN SANITIZED_PHASE31_HIGHRES_RESTORE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_HIGHRES_RESTORE")

Phase31 persistent highres preprocessing: 200/1362
Phase31 persistent highres preprocessing: 400/1362
Phase31 persistent highres preprocessing: 600/1362
Phase31 persistent highres preprocessing: 800/1362
Phase31 persistent highres preprocessing: 1000/1362
Phase31 persistent highres preprocessing: 1200/1362
Phase31 persistent highres preprocessing: 1362/1362
BEGIN SANITIZED_PHASE31_HIGHRES_RESTORE
{
  "phase": "phase31_persistent_highres_cache",
  "status": "accepted",
  "cache_reused": false,
  "cache_name": "phase31_highres_float16.npy",
  "shape": [
    1362,
    80,
    80,
    80
  ],
  "dtype": "float16",
  "size_gb": 1.2989,
  "preprocessing": "exact_phase12c_128mm_80cube_independent_positive_p99.5",
  "sampled_minimum": 0.0,
  "sampled_maximum": 1.5,
  "sampled_mean": 0.277909,
  "failure_count": 0,
  "elapsed_seconds": 112.91,
  "persistent_notebook_cache_only": true,
  "include_in_submission": false,
  "training_voxel_arrays_read": true,
  "smoke_data_read": false,
  "test_dat

In [5]:
# Phase31 Cell 98
# Resolve the existing 80^3 training cache, labels, groups, and frozen splits.

import json
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scipy.ndimage as ndi
import sklearn
import torch

EXPECTED_CASES = 1362

PHASE31_CONFIG = {
    "phase": "phase31_localized_multithreshold_radiomics",
    "left_right_axis": 0,
    "anterior_posterior_axis": 1,
    "superior_inferior_axis": 2,
    "highres_shape": (80, 80, 80),
    "highres_spacing_mm": 1.6,
    "patch_sizes_voxels": (31, 41, 51),
    "nominal_patch_fov_mm": (49.6, 65.6, 81.6),
    "thresholds": (0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85),
    "profile_bins": 16,
    "localization_quantile": 0.96,
    "worker_count": 8,
    "reflection_invariant": True,
}


def phase31_find_array(obj, expected_shape, depth=0):
    """Find a NumPy array inside a shallow tuple/list/dictionary container."""
    if depth > 3:
        return None

    if isinstance(obj, np.ndarray):
        if tuple(obj.shape) == tuple(expected_shape):
            return obj
        return None

    if isinstance(obj, (tuple, list)):
        for item in obj:
            result = phase31_find_array(
                item,
                expected_shape,
                depth + 1,
            )
            if result is not None:
                return result

    if isinstance(obj, dict):
        for item in obj.values():
            result = phase31_find_array(
                item,
                expected_shape,
                depth + 1,
            )
            if result is not None:
                return result

    return None


def phase31_resolve_highres_cache():
    expected_shape = (
        EXPECTED_CASES,
        *PHASE31_CONFIG["highres_shape"],
    )

    explicit_names = [
        "PHASE19_HIGHRES_CACHE",
        "PHASE18_HIGHRES_CACHE",
        "PHASE18_CANONICAL_HIGHRES_CACHE",
        "PHASE18_HIGHRES_CACHE_PRIVATE",
        "canonical_highres_cache",
        "highres_cache",
    ]

    attempts = []

    # Prefer the resolver already used successfully in Phase30 Cell 94.
    resolver = globals().get("phase26_resolve_highres_cache")
    if callable(resolver):
        try:
            resolved = resolver()
            array = phase31_find_array(resolved, expected_shape)
            attempts.append({
                "source": "phase26_resolve_highres_cache",
                "status": "accepted" if array is not None else "wrong_shape",
            })
            if array is not None:
                return array, "phase26_resolve_highres_cache", attempts
        except Exception as exc:
            attempts.append({
                "source": "phase26_resolve_highres_cache",
                "status": "exception",
                "error_type": type(exc).__name__,
            })

    for name in explicit_names:
        if name not in globals():
            continue

        array = phase31_find_array(
            globals()[name],
            expected_shape,
        )
        attempts.append({
            "source": name,
            "status": "accepted" if array is not None else "wrong_shape",
        })

        if array is not None:
            return array, name, attempts

    # Last resort: inspect only in-memory NumPy containers.
    for name, obj in list(globals().items()):
        if name.startswith("_"):
            continue

        array = phase31_find_array(obj, expected_shape)
        if array is not None:
            attempts.append({
                "source": name,
                "status": "accepted_global_scan",
            })
            return array, name, attempts

    raise AssertionError({
        "message": "Could not resolve the 1362 x 80 x 80 x 80 cache.",
        "attempts": attempts,
    })


def phase31_binary_labels(values):
    values = np.asarray(values)

    if values.shape != (EXPECTED_CASES,):
        raise ValueError("Incorrect label shape.")

    if np.issubdtype(values.dtype, np.number):
        numeric = values.astype(np.float64)
        unique = set(np.unique(numeric).tolist())

        if unique.issubset({0.0, 1.0}):
            return numeric.astype(np.int64)

    text = np.asarray([
        str(value).strip().lower()
        for value in values
    ])

    normal_terms = {
        "0", "normal", "negative", "control", "healthy",
    }
    abnormal_terms = {
        "1", "pathologic", "pathological", "abnormal",
        "positive", "disease", "parkinson",
    }

    output = np.full(EXPECTED_CASES, -1, dtype=np.int64)

    for index, value in enumerate(text):
        if value in normal_terms:
            output[index] = 0
        elif value in abnormal_terms:
            output[index] = 1
        elif "normal" in value or "control" in value:
            output[index] = 0
        elif (
            "patholog" in value
            or "abnormal" in value
            or "parkinson" in value
        ):
            output[index] = 1

    if np.any(output < 0):
        raise ValueError({
            "message": "Could not convert all labels to binary values.",
            "unresolved_count": int(np.sum(output < 0)),
        })

    return output


def phase31_resolve_case_dataframe():
    preferred_names = [
        "case_df",
        "PHASE18_CASE_DF",
        "PHASE19_CASE_DF",
        "train_df",
    ]

    for name in preferred_names:
        candidate = globals().get(name)

        if (
            isinstance(candidate, pd.DataFrame)
            and len(candidate) == EXPECTED_CASES
        ):
            return candidate, name

    for name, candidate in list(globals().items()):
        if (
            isinstance(candidate, pd.DataFrame)
            and len(candidate) == EXPECTED_CASES
        ):
            return candidate, name

    raise AssertionError(
        "Could not resolve the 1,362-row case dataframe."
    )


def phase31_resolve_labels(frame):
    aliases = [
        "label",
        "target",
        "pathologic",
        "pathological",
        "abnormal",
        "is_pathologic",
        "diagnosis",
        "class",
    ]

    lower_to_original = {
        str(column).lower(): column
        for column in frame.columns
    }

    for alias in aliases:
        if alias in lower_to_original:
            column = lower_to_original[alias]
            return phase31_binary_labels(
                frame[column].to_numpy()
            ), str(column)

    global_aliases = [
        "y",
        "labels",
        "train_labels",
        "PHASE19_LABELS",
        "PHASE18_LABELS",
    ]

    for name in global_aliases:
        candidate = globals().get(name)
        if candidate is None:
            continue

        try:
            return phase31_binary_labels(candidate), name
        except Exception:
            pass

    raise AssertionError({
        "message": "Could not resolve binary labels.",
        "available_columns": sorted(
            str(column)
            for column in frame.columns
        ),
    })


def phase31_resolve_groups(frame):
    aliases = [
        "acquisition_group",
        "acquisition_cluster",
        "scanner_group",
        "protocol_group",
        "group",
    ]

    lower_to_original = {
        str(column).lower(): column
        for column in frame.columns
    }

    for alias in aliases:
        if alias in lower_to_original:
            column = lower_to_original[alias]
            groups = frame[column].to_numpy()

            if groups.shape == (EXPECTED_CASES,):
                return groups, str(column)

    for name in [
        "phase20_acquisition_groups",
        "PHASE20_ACQUISITION_GROUPS",
        "acquisition_groups",
    ]:
        candidate = globals().get(name)

        if candidate is not None:
            candidate = np.asarray(candidate)

            if candidate.shape == (EXPECTED_CASES,):
                return candidate, name

    raise AssertionError(
        "Could not resolve acquisition groups."
    )


def phase31_resolve_partitions():
    candidate = globals().get("PHASE19_PARTITIONS")

    if candidate is None:
        candidate = globals().get("phase19_partitions")

    assert isinstance(candidate, (list, tuple))
    assert len(candidate) == 3

    resolved = []

    for expected_fold, partition in enumerate(candidate):
        assert isinstance(partition, dict)

        required = {
            "fit",
            "monitor",
            "outer_train",
            "outer_valid",
        }
        assert required.issubset(partition)

        record = {
            "fold": int(partition.get("fold", expected_fold)),
        }

        for key in required:
            values = np.asarray(
                partition[key],
                dtype=np.int64,
            )
            assert values.ndim == 1
            assert np.all(values >= 0)
            assert np.all(values < EXPECTED_CASES)
            assert len(np.unique(values)) == len(values)
            record[key] = values

        assert len(
            np.intersect1d(
                record["fit"],
                record["monitor"],
            )
        ) == 0

        assert len(
            np.intersect1d(
                record["outer_train"],
                record["outer_valid"],
            )
        ) == 0

        resolved.append(record)

    validation_coverage = np.zeros(
        EXPECTED_CASES,
        dtype=np.int64,
    )

    for partition in resolved:
        validation_coverage[
            partition["outer_valid"]
        ] += 1

    assert np.all(validation_coverage == 1)

    return resolved


PHASE31_HIGHRES_CACHE, phase31_cache_source, phase31_cache_attempts = (
    phase31_resolve_highres_cache()
)

PHASE31_CASE_DF, phase31_case_df_source = (
    phase31_resolve_case_dataframe()
)

PHASE31_LABELS, phase31_label_source = (
    phase31_resolve_labels(PHASE31_CASE_DF)
)

PHASE31_ACQUISITION_GROUPS, phase31_group_source = (
    phase31_resolve_groups(PHASE31_CASE_DF)
)

PHASE31_PARTITIONS = phase31_resolve_partitions()

assert PHASE31_HIGHRES_CACHE.shape == (
    EXPECTED_CASES, 80, 80, 80
)
assert PHASE31_LABELS.shape == (EXPECTED_CASES,)
assert set(np.unique(PHASE31_LABELS)) == {0, 1}
assert np.isfinite(
    np.asarray(
        PHASE31_HIGHRES_CACHE[:4],
        dtype=np.float32,
    )
).all()

report = {
    "phase": PHASE31_CONFIG["phase"],
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "highres_cache": {
        "source": phase31_cache_source,
        "shape": list(PHASE31_HIGHRES_CACHE.shape),
        "dtype": str(PHASE31_HIGHRES_CACHE.dtype),
    },
    "case_dataframe_source": phase31_case_df_source,
    "label_source": phase31_label_source,
    "normal_count": int(np.sum(PHASE31_LABELS == 0)),
    "pathologic_count": int(np.sum(PHASE31_LABELS == 1)),
    "acquisition_group_source": phase31_group_source,
    "acquisition_group_count": int(
        len(np.unique(PHASE31_ACQUISITION_GROUPS))
    ),
    "partition_count": len(PHASE31_PARTITIONS),
    "outer_validation_exact_coverage": True,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
}

print("BEGIN SANITIZED_PHASE31_STATE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_STATE")

BEGIN SANITIZED_PHASE31_STATE
{
  "phase": "phase31_localized_multithreshold_radiomics",
  "status": "accepted",
  "case_count": 1362,
  "highres_cache": {
    "source": "PHASE19_HIGHRES_CACHE",
    "shape": [
      1362,
      80,
      80,
      80
    ],
    "dtype": "float16"
  },
  "case_dataframe_source": "case_df",
  "label_source": "label",
  "normal_count": 615,
  "pathologic_count": 747,
  "acquisition_group_source": "acquisition_group",
  "acquisition_group_count": 15,
  "partition_count": 3,
  "outer_validation_exact_coverage": true,
  "smoke_data_read": false,
  "test_data_read": false,
  "uids_displayed": false,
  "patient_rows_displayed": false
}
END SANITIZED_PHASE31_STATE


In [6]:
# Phase31 Cell 99
# Define localized bilateral radiomics and posterior-anterior sequences.

def phase31_top_fraction_mean(array, fraction=0.10):
    flat = np.asarray(array, dtype=np.float32).ravel()

    if flat.size == 0:
        return 0.0

    count = max(1, int(math.ceil(flat.size * fraction)))
    boundary = flat.size - count
    selected = np.partition(flat, boundary)[boundary:]

    return float(np.mean(selected))


def phase31_background_level(volume):
    positive = np.asarray(
        volume[volume > 0],
        dtype=np.float32,
    )

    if positive.size < 128:
        return 1e-3

    upper = float(np.quantile(positive, 0.70))
    reference = positive[positive <= upper]

    if reference.size < 64:
        reference = positive

    return max(float(np.median(reference)), 1e-4)


def phase31_weighted_center(subvolume, offset_x):
    values = np.asarray(subvolume, dtype=np.float32)
    positive = values[values > 0]

    if positive.size < 32:
        local = np.asarray(
            np.unravel_index(
                int(np.argmax(values)),
                values.shape,
            ),
            dtype=np.float64,
        )
    else:
        threshold = float(
            np.quantile(
                positive,
                PHASE31_CONFIG["localization_quantile"],
            )
        )

        weights = np.square(
            np.clip(values - threshold, 0.0, None)
        )
        total = float(weights.sum())

        if total <= 1e-8:
            local = np.asarray(
                np.unravel_index(
                    int(np.argmax(values)),
                    values.shape,
                ),
                dtype=np.float64,
            )
        else:
            coordinates = np.indices(
                values.shape,
                dtype=np.float64,
            )

            local = np.asarray([
                np.sum(coordinates[axis] * weights) / total
                for axis in range(3)
            ])

    local[0] += float(offset_x)
    return local


def phase31_localize_bilateral(volume):
    assert tuple(volume.shape) == (80, 80, 80)

    midpoint = volume.shape[0] // 2

    left_center = phase31_weighted_center(
        volume[:midpoint],
        offset_x=0,
    )

    right_center = phase31_weighted_center(
        volume[midpoint:],
        offset_x=midpoint,
    )

    return left_center, right_center


def phase31_extract_integer_patch(volume, center, size):
    assert size % 2 == 1

    center_index = np.rint(center).astype(np.int64)
    radius = size // 2
    start = center_index - radius
    stop = start + size

    patch = np.zeros(
        (size, size, size),
        dtype=np.float32,
    )

    source_start = np.maximum(start, 0)
    source_stop = np.minimum(
        stop,
        np.asarray(volume.shape),
    )

    destination_start = source_start - start
    destination_stop = (
        destination_start
        + source_stop
        - source_start
    )

    source_slices = tuple(
        slice(int(a), int(b))
        for a, b in zip(source_start, source_stop)
    )

    destination_slices = tuple(
        slice(int(a), int(b))
        for a, b in zip(
            destination_start,
            destination_stop,
        )
    )

    patch[destination_slices] = np.asarray(
        volume[source_slices],
        dtype=np.float32,
    )

    return patch


def phase31_shape_statistics(normalized_signal, threshold):
    mask = normalized_signal >= float(threshold)
    voxel_count = int(mask.sum())
    total_voxels = int(mask.size)

    if voxel_count < 3:
        return np.zeros(11, dtype=np.float32)

    structure_3d = ndi.generate_binary_structure(3, 1)
    labels, component_count = ndi.label(
        mask,
        structure=structure_3d,
    )

    counts = np.bincount(labels.ravel())[1:]
    largest_fraction = (
        float(counts.max()) / voxel_count
        if counts.size
        else 0.0
    )

    eroded = ndi.binary_erosion(
        mask,
        structure=structure_3d,
        border_value=0,
    )
    surface_count = int(np.sum(mask & ~eroded))
    surface_fraction = surface_count / max(voxel_count, 1)

    if surface_count > 0:
        sphericity = (
            (math.pi ** (1.0 / 3.0))
            * ((6.0 * voxel_count) ** (2.0 / 3.0))
            / surface_count
        )
    else:
        sphericity = 0.0

    coordinates = np.argwhere(mask).astype(np.float64)

    if len(coordinates) >= 4:
        covariance = np.cov(
            coordinates,
            rowvar=False,
        )
        eigenvalues = np.sort(
            np.linalg.eigvalsh(covariance)
        )[::-1]
        eigenvalues = np.clip(eigenvalues, 0.0, None)
        denominator = max(float(eigenvalues[0]), 1e-8)
        eigen_middle_ratio = float(
            eigenvalues[1] / denominator
        )
        eigen_small_ratio = float(
            eigenvalues[2] / denominator
        )
    else:
        eigen_middle_ratio = 0.0
        eigen_small_ratio = 0.0

    # Axial maximum-intensity projection shape.
    projection = normalized_signal.max(axis=2)
    mask_2d = projection >= float(threshold)
    area_2d = int(mask_2d.sum())

    structure_2d = ndi.generate_binary_structure(2, 1)
    eroded_2d = ndi.binary_erosion(
        mask_2d,
        structure=structure_2d,
        border_value=0,
    )
    perimeter = int(np.sum(mask_2d & ~eroded_2d))

    circularity = (
        4.0 * math.pi * area_2d / (perimeter ** 2)
        if perimeter > 0
        else 0.0
    )

    coordinates_2d = np.argwhere(mask_2d).astype(np.float64)

    if len(coordinates_2d) >= 3:
        covariance_2d = np.cov(
            coordinates_2d,
            rowvar=False,
        )
        eigenvalues_2d = np.sort(
            np.linalg.eigvalsh(covariance_2d)
        )[::-1]
        eigenvalues_2d = np.clip(
            eigenvalues_2d,
            0.0,
            None,
        )
        eccentricity = math.sqrt(
            max(
                0.0,
                1.0
                - float(eigenvalues_2d[1])
                / max(float(eigenvalues_2d[0]), 1e-8),
            )
        )
    else:
        eccentricity = 0.0

    return np.asarray([
        voxel_count / total_voxels,
        float(normalized_signal[mask].mean()),
        float(component_count),
        largest_fraction,
        surface_fraction,
        float(np.clip(sphericity, 0.0, 2.0)),
        eigen_middle_ratio,
        eigen_small_ratio,
        area_2d / mask_2d.size,
        float(np.clip(circularity, 0.0, 2.0)),
        eccentricity,
    ], dtype=np.float32)


PHASE31_SHAPE_STAT_NAMES = [
    "occupancy",
    "uptake_mean",
    "component_count",
    "largest_component_fraction",
    "surface_fraction",
    "sphericity",
    "eigen_middle_ratio",
    "eigen_small_ratio",
    "axial_area_fraction",
    "axial_circularity",
    "axial_eccentricity",
]


def phase31_side_features(patch, background, scale_name):
    raw = np.asarray(patch, dtype=np.float32)
    signal = np.clip(raw - background, 0.0, None)
    positive = signal[signal > 0]

    if positive.size:
        scale = max(
            float(np.quantile(positive, 0.995)),
            1e-5,
        )
    else:
        scale = 1e-5

    normalized = np.clip(
        signal / scale,
        0.0,
        2.0,
    )

    quantiles = [
        float(np.quantile(raw, q))
        for q in (0.50, 0.75, 0.90, 0.95, 0.99)
    ]

    values = [
        float(raw.mean() / background),
        float(raw.std() / background),
        *[
            value / background
            for value in quantiles
        ],
        phase31_top_fraction_mean(raw, 0.05) / background,
        float(np.mean(raw > 0)),
        float(scale / background),
        float(normalized.mean()),
        float(normalized.std()),
    ]

    names = [
        f"{scale_name}_raw_mean_to_background",
        f"{scale_name}_raw_std_to_background",
        f"{scale_name}_raw_q50_to_background",
        f"{scale_name}_raw_q75_to_background",
        f"{scale_name}_raw_q90_to_background",
        f"{scale_name}_raw_q95_to_background",
        f"{scale_name}_raw_q99_to_background",
        f"{scale_name}_top05_to_background",
        f"{scale_name}_positive_fraction",
        f"{scale_name}_signal_scale_to_background",
        f"{scale_name}_normalized_mean",
        f"{scale_name}_normalized_std",
    ]

    for threshold in PHASE31_CONFIG["thresholds"]:
        statistics = phase31_shape_statistics(
            normalized,
            threshold,
        )

        threshold_name = (
            f"{scale_name}_t{int(round(threshold * 100)):02d}"
        )

        values.extend(statistics.tolist())
        names.extend([
            f"{threshold_name}_{name}"
            for name in PHASE31_SHAPE_STAT_NAMES
        ])

    return (
        np.asarray(values, dtype=np.float32),
        names,
        scale,
    )


PHASE31_PROFILE_NAMES = [
    "mean",
    "standard_deviation",
    "q75",
    "q90",
    "q99",
    "top10_mean",
    "occupancy_t35",
    "occupancy_t50",
    "occupancy_t65",
    "width_lr",
    "width_si",
    "centroid_lr",
    "centroid_si",
]


def phase31_profile_tokens(normalized_patch):
    bin_indices = np.array_split(
        np.arange(normalized_patch.shape[1]),
        PHASE31_CONFIG["profile_bins"],
    )

    tokens = []

    for indices in bin_indices:
        slab = normalized_patch[:, indices, :]
        flat = slab.ravel()

        mask = slab >= 0.35
        coordinates = np.argwhere(mask)

        if coordinates.size:
            width_lr = (
                coordinates[:, 0].max()
                - coordinates[:, 0].min()
                + 1
            ) / slab.shape[0]

            width_si = (
                coordinates[:, 2].max()
                - coordinates[:, 2].min()
                + 1
            ) / slab.shape[2]

            centroid_lr = (
                2.0
                * float(coordinates[:, 0].mean())
                / max(slab.shape[0] - 1, 1)
                - 1.0
            )

            centroid_si = (
                2.0
                * float(coordinates[:, 2].mean())
                / max(slab.shape[2] - 1, 1)
                - 1.0
            )
        else:
            width_lr = 0.0
            width_si = 0.0
            centroid_lr = 0.0
            centroid_si = 0.0

        tokens.append([
            float(flat.mean()),
            float(flat.std()),
            float(np.quantile(flat, 0.75)),
            float(np.quantile(flat, 0.90)),
            float(np.quantile(flat, 0.99)),
            phase31_top_fraction_mean(flat, 0.10),
            float(np.mean(flat >= 0.35)),
            float(np.mean(flat >= 0.50)),
            float(np.mean(flat >= 0.65)),
            float(width_lr),
            float(width_si),
            float(centroid_lr),
            float(centroid_si),
        ])

    return np.asarray(tokens, dtype=np.float32)


def phase31_case_features(volume):
    volume = np.asarray(volume, dtype=np.float32)
    assert volume.shape == (80, 80, 80)
    assert np.isfinite(volume).all()

    background = phase31_background_level(volume)
    left_center, right_center = phase31_localize_bilateral(volume)

    feature_values = []
    feature_names = []

    profile_left = None
    profile_right = None
    profile_common_scale = None

    for patch_size in PHASE31_CONFIG["patch_sizes_voxels"]:
        left_patch = phase31_extract_integer_patch(
            volume,
            left_center,
            patch_size,
        )

        right_patch = phase31_extract_integer_patch(
            volume,
            right_center,
            patch_size,
        )

        # Put the right striatum into the left-oriented coordinate system.
        right_patch = np.flip(
            right_patch,
            axis=0,
        ).copy()

        scale_name = f"patch_{patch_size:02d}"

        left_features, left_names, _ = phase31_side_features(
            left_patch,
            background,
            scale_name,
        )

        right_features, right_names, _ = phase31_side_features(
            right_patch,
            background,
            scale_name,
        )

        assert left_names == right_names

        operations = {
            "bilateral_mean": 0.5 * (
                left_features + right_features
            ),
            "bilateral_absdiff": np.abs(
                left_features - right_features
            ),
            "bilateral_minimum": np.minimum(
                left_features,
                right_features,
            ),
            "bilateral_maximum": np.maximum(
                left_features,
                right_features,
            ),
        }

        for operation_name, operation_values in operations.items():
            feature_values.extend(operation_values.tolist())
            feature_names.extend([
                f"{operation_name}_{name}"
                for name in left_names
            ])

        if patch_size == 41:
            left_signal = np.clip(
                left_patch - background,
                0.0,
                None,
            )
            right_signal = np.clip(
                right_patch - background,
                0.0,
                None,
            )

            combined_positive = np.concatenate([
                left_signal[left_signal > 0],
                right_signal[right_signal > 0],
            ])

            if combined_positive.size:
                common_scale = max(
                    float(
                        np.quantile(
                            combined_positive,
                            0.995,
                        )
                    ),
                    1e-5,
                )
            else:
                common_scale = 1e-5

            profile_left = phase31_profile_tokens(
                np.clip(
                    left_signal / common_scale,
                    0.0,
                    2.0,
                )
            )

            profile_right = phase31_profile_tokens(
                np.clip(
                    right_signal / common_scale,
                    0.0,
                    2.0,
                )
            )

            profile_common_scale = common_scale

    assert profile_left is not None
    assert profile_right is not None

    sequence = np.concatenate([
        0.5 * (profile_left + profile_right),
        np.abs(profile_left - profile_right),
        np.minimum(profile_left, profile_right),
        np.maximum(profile_left, profile_right),
    ], axis=1)

    sequence_names = (
        [f"bilateral_mean_{name}" for name in PHASE31_PROFILE_NAMES]
        + [f"bilateral_absdiff_{name}" for name in PHASE31_PROFILE_NAMES]
        + [f"bilateral_minimum_{name}" for name in PHASE31_PROFILE_NAMES]
        + [f"bilateral_maximum_{name}" for name in PHASE31_PROFILE_NAMES]
    )

    positive = volume[volume > 0]

    if positive.size:
        global_quantiles = [
            float(np.quantile(positive, q))
            for q in (0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)
        ]
    else:
        global_quantiles = [0.0] * 7

    center_separation = float(
        np.linalg.norm(left_center - right_center)
    )

    global_values = [
        background,
        float(volume.mean()),
        float(volume.std()),
        float(np.mean(volume > 0)),
        *global_quantiles,
        center_separation,
        float(profile_common_scale / background),
    ]

    global_names = [
        "global_background",
        "global_mean",
        "global_standard_deviation",
        "global_positive_fraction",
        "global_positive_q10",
        "global_positive_q25",
        "global_positive_q50",
        "global_positive_q75",
        "global_positive_q90",
        "global_positive_q95",
        "global_positive_q99",
        "bilateral_center_separation_voxels",
        "profile_common_scale_to_background",
    ]

    feature_values.extend(global_values)
    feature_names.extend(global_names)

    diagnostics = {
        "background": background,
        "center_separation_voxels": center_separation,
        "left_center": left_center,
        "right_center": right_center,
        "profile_common_scale": profile_common_scale,
    }

    return (
        np.asarray(feature_values, dtype=np.float32),
        feature_names,
        sequence.astype(np.float32),
        sequence_names,
        diagnostics,
    )

In [7]:
# Phase31 Cell 100
# Validate shape, finiteness, asymmetry sensitivity, and reflection invariance.

def phase31_gaussian_3d(
    coordinates,
    center,
    standard_deviation,
):
    exponent = np.zeros_like(
        coordinates[0],
        dtype=np.float32,
    )

    for axis in range(3):
        exponent += np.square(
            (
                coordinates[axis]
                - float(center[axis])
            )
            / float(standard_deviation[axis])
        )

    return np.exp(-0.5 * exponent)


rng = np.random.default_rng(310601)

coordinates = np.indices(
    (80, 80, 80),
    dtype=np.float32,
)

synthetic = np.full(
    (80, 80, 80),
    0.06,
    dtype=np.float32,
)

# Left striatum: reduced posterior uptake.
synthetic += 0.72 * phase31_gaussian_3d(
    coordinates,
    center=(25, 31, 40),
    standard_deviation=(4.0, 7.0, 4.0),
)
synthetic += 1.05 * phase31_gaussian_3d(
    coordinates,
    center=(25, 47, 40),
    standard_deviation=(4.5, 8.0, 4.5),
)

# Right striatum: stronger posterior tail.
synthetic += 1.12 * phase31_gaussian_3d(
    coordinates,
    center=(54, 31, 40),
    standard_deviation=(4.0, 7.0, 4.0),
)
synthetic += 1.02 * phase31_gaussian_3d(
    coordinates,
    center=(54, 47, 40),
    standard_deviation=(4.5, 8.0, 4.5),
)

synthetic += rng.normal(
    0.0,
    0.006,
    size=synthetic.shape,
).astype(np.float32)

synthetic = np.clip(
    synthetic,
    0.0,
    1.5,
)

(
    contract_features,
    contract_feature_names,
    contract_sequence,
    contract_sequence_names,
    contract_diagnostics,
) = phase31_case_features(synthetic)

(
    reflected_features,
    reflected_feature_names,
    reflected_sequence,
    reflected_sequence_names,
    reflected_diagnostics,
) = phase31_case_features(
    np.flip(
        synthetic,
        axis=0,
    ).copy()
)

assert contract_feature_names == reflected_feature_names
assert contract_sequence_names == reflected_sequence_names
assert contract_features.ndim == 1
assert contract_sequence.shape == (
    PHASE31_CONFIG["profile_bins"],
    4 * len(PHASE31_PROFILE_NAMES),
)
assert np.isfinite(contract_features).all()
assert np.isfinite(contract_sequence).all()

feature_reflection_error = float(
    np.max(
        np.abs(
            contract_features
            - reflected_features
        )
    )
)

sequence_reflection_error = float(
    np.max(
        np.abs(
            contract_sequence
            - reflected_sequence
        )
    )
)

# Integer-centered odd patches should make this nearly exact.
assert feature_reflection_error <= 2e-3, (
    feature_reflection_error
)
assert sequence_reflection_error <= 2e-3, (
    sequence_reflection_error
)

# Confirm the asymmetric synthetic case produces nonzero bilateral evidence.
absdiff_indices = [
    index
    for index, name in enumerate(contract_feature_names)
    if name.startswith("bilateral_absdiff_")
]

mean_absdiff_signal = float(
    np.mean(
        np.abs(
            contract_features[absdiff_indices]
        )
    )
)

assert mean_absdiff_signal > 1e-4

report = {
    "phase": "phase31_localized_multithreshold_contract",
    "status": "accepted",
    "feature_count": len(contract_feature_names),
    "sequence_shape": list(contract_sequence.shape),
    "sequence_feature_count": len(contract_sequence_names),
    "patch_sizes_voxels": list(
        PHASE31_CONFIG["patch_sizes_voxels"]
    ),
    "threshold_count": len(
        PHASE31_CONFIG["thresholds"]
    ),
    "profile_bin_count": PHASE31_CONFIG["profile_bins"],
    "feature_reflection_max_abs_error": (
        feature_reflection_error
    ),
    "sequence_reflection_max_abs_error": (
        sequence_reflection_error
    ),
    "mean_synthetic_bilateral_absdiff_signal": (
        mean_absdiff_signal
    ),
    "all_features_finite": True,
    "asymmetry_sensitivity_passed": True,
    "synthetic_input_only": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE31_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_CONTRACT")

BEGIN SANITIZED_PHASE31_CONTRACT
{
  "phase": "phase31_localized_multithreshold_contract",
  "status": "accepted",
  "feature_count": 1081,
  "sequence_shape": [
    16,
    52
  ],
  "sequence_feature_count": 52,
  "patch_sizes_voxels": [
    31,
    41,
    51
  ],
  "threshold_count": 7,
  "profile_bin_count": 16,
  "feature_reflection_max_abs_error": 1.9073486328125e-06,
  "sequence_reflection_max_abs_error": 0.0,
  "mean_synthetic_bilateral_absdiff_signal": 0.1259544938802719,
  "all_features_finite": true,
  "asymmetry_sensitivity_passed": true,
  "synthetic_input_only": true,
  "training_voxel_cache_read": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "uids_displayed": false
}
END SANITIZED_PHASE31_CONTRACT


In [9]:
# Phase31 Cell 101A
# Definitions only.

from concurrent.futures import ThreadPoolExecutor
import os

PHASE31_FEATURE_PATH = Path(
    "/kaggle/working/phase31_radiomics_float32.npy"
)
PHASE31_SEQUENCE_PATH = Path(
    "/kaggle/working/phase31_sequence_float32.npy"
)
PHASE31_DIAGNOSTIC_PATH = Path(
    "/kaggle/working/phase31_localization_diagnostics.npy"
)
PHASE31_METADATA_PATH = Path(
    "/kaggle/working/phase31_feature_metadata.json"
)

PHASE31_FEATURE_PARTIAL = Path(
    "/kaggle/working/phase31_radiomics_float32.partial.npy"
)
PHASE31_SEQUENCE_PARTIAL = Path(
    "/kaggle/working/phase31_sequence_float32.partial.npy"
)
PHASE31_DIAGNOSTIC_PARTIAL = Path(
    "/kaggle/working/phase31_localization_diagnostics.partial.npy"
)


def phase31_validate_extracted_cache():
    required = [
        PHASE31_FEATURE_PATH,
        PHASE31_SEQUENCE_PATH,
        PHASE31_DIAGNOSTIC_PATH,
        PHASE31_METADATA_PATH,
    ]

    if not all(path.is_file() for path in required):
        return None

    try:
        metadata = json.loads(
            PHASE31_METADATA_PATH.read_text(
                encoding="utf-8"
            )
        )

        features = np.load(
            PHASE31_FEATURE_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )
        sequences = np.load(
            PHASE31_SEQUENCE_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )
        diagnostics = np.load(
            PHASE31_DIAGNOSTIC_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        if features.shape != (
            EXPECTED_CASES,
            int(metadata["feature_count"]),
        ):
            return None

        if sequences.shape != (
            EXPECTED_CASES,
            *tuple(metadata["sequence_shape"]),
        ):
            return None

        if diagnostics.shape != (
            EXPECTED_CASES,
            5,
        ):
            return None

        if features.dtype != np.float32:
            return None

        if sequences.dtype != np.float32:
            return None

        if diagnostics.dtype != np.float32:
            return None

        if len(metadata["feature_names"]) != 1081:
            return None

        if len(metadata["sequence_names"]) != 52:
            return None

        check_indices = np.linspace(
            0,
            EXPECTED_CASES - 1,
            16,
            dtype=np.int64,
        )

        if not np.isfinite(
            np.asarray(features[check_indices])
        ).all():
            return None

        if not np.isfinite(
            np.asarray(sequences[check_indices])
        ).all():
            return None

        if not np.isfinite(
            np.asarray(diagnostics[check_indices])
        ).all():
            return None

        return {
            "features": features,
            "sequences": sequences,
            "diagnostics": diagnostics,
            "metadata": metadata,
        }

    except Exception:
        return None


def phase31_diagnostic_vector(diagnostics):
    return np.asarray([
        diagnostics["background"],
        diagnostics["center_separation_voxels"],
        diagnostics["left_center"][0],
        diagnostics["right_center"][0],
        diagnostics["profile_common_scale"],
    ], dtype=np.float32)


def phase31_extract_case_index(index):
    try:
        (
            feature_vector,
            feature_names,
            sequence,
            sequence_names,
            diagnostics,
        ) = phase31_case_features(
            PHASE31_HIGHRES_CACHE[index]
        )

        diagnostic_vector = (
            phase31_diagnostic_vector(
                diagnostics
            )
        )

        if feature_vector.shape != (1081,):
            raise ValueError("IncorrectRadiomicsShape")

        if sequence.shape != (16, 52):
            raise ValueError("IncorrectSequenceShape")

        if diagnostic_vector.shape != (5,):
            raise ValueError("IncorrectDiagnosticShape")

        if not np.isfinite(feature_vector).all():
            raise ValueError("NonFiniteRadiomics")

        if not np.isfinite(sequence).all():
            raise ValueError("NonFiniteSequence")

        if not np.isfinite(diagnostic_vector).all():
            raise ValueError("NonFiniteDiagnostic")

        return (
            index,
            feature_vector.astype(
                np.float32,
                copy=False,
            ),
            feature_names,
            sequence.astype(
                np.float32,
                copy=False,
            ),
            sequence_names,
            diagnostic_vector,
            None,
        )

    except Exception as exc:
        return (
            index,
            None,
            None,
            None,
            None,
            None,
            type(exc).__name__,
        )


print("Phase31 Cell 101A definitions accepted.")

Phase31 Cell 101A definitions accepted.


In [10]:
# Phase31 Cell 101B
# This performs the extraction if a valid cache is not already present.

phase31_extraction_started = time.perf_counter()
phase31_cached_result = (
    phase31_validate_extracted_cache()
)
phase31_feature_cache_reused = (
    phase31_cached_result is not None
)
phase31_failure_types = []


if phase31_cached_result is None:
    (
        first_index,
        first_features,
        first_feature_names,
        first_sequence,
        first_sequence_names,
        first_diagnostics,
        first_error,
    ) = phase31_extract_case_index(0)

    assert first_error is None, first_error
    assert first_index == 0
    assert first_features.shape == (1081,)
    assert first_sequence.shape == (16, 52)
    assert len(first_feature_names) == 1081
    assert len(first_sequence_names) == 52
    assert len(set(first_feature_names)) == 1081
    assert len(set(first_sequence_names)) == 52

    PHASE31_FEATURE_NAMES = list(
        first_feature_names
    )
    PHASE31_SEQUENCE_NAMES = list(
        first_sequence_names
    )

    feature_writer = np.lib.format.open_memmap(
        PHASE31_FEATURE_PARTIAL,
        mode="w+",
        dtype=np.float32,
        shape=(EXPECTED_CASES, 1081),
    )

    sequence_writer = np.lib.format.open_memmap(
        PHASE31_SEQUENCE_PARTIAL,
        mode="w+",
        dtype=np.float32,
        shape=(EXPECTED_CASES, 16, 52),
    )

    diagnostic_writer = np.lib.format.open_memmap(
        PHASE31_DIAGNOSTIC_PARTIAL,
        mode="w+",
        dtype=np.float32,
        shape=(EXPECTED_CASES, 5),
    )

    feature_writer[0] = first_features
    sequence_writer[0] = first_sequence
    diagnostic_writer[0] = first_diagnostics

    with ThreadPoolExecutor(
        max_workers=PHASE31_CONFIG["worker_count"]
    ) as executor:
        iterator = executor.map(
            phase31_extract_case_index,
            range(1, EXPECTED_CASES),
            chunksize=1,
        )

        for completed, result in enumerate(
            iterator,
            start=2,
        ):
            (
                index,
                feature_vector,
                feature_names,
                sequence,
                sequence_names,
                diagnostic_vector,
                error_type,
            ) = result

            if error_type is not None:
                phase31_failure_types.append(
                    error_type
                )
            else:
                assert (
                    feature_names
                    == PHASE31_FEATURE_NAMES
                )
                assert (
                    sequence_names
                    == PHASE31_SEQUENCE_NAMES
                )

                feature_writer[index] = (
                    feature_vector
                )
                sequence_writer[index] = sequence
                diagnostic_writer[index] = (
                    diagnostic_vector
                )

            if (
                completed % 100 == 0
                or completed == EXPECTED_CASES
            ):
                feature_writer.flush()
                sequence_writer.flush()
                diagnostic_writer.flush()

                print(
                    "Phase31 feature extraction: "
                    f"{completed}/{EXPECTED_CASES}"
                )

    feature_writer.flush()
    sequence_writer.flush()
    diagnostic_writer.flush()

    del feature_writer
    del sequence_writer
    del diagnostic_writer

    assert not phase31_failure_types, {
        "message": "Phase31 feature extraction failed.",
        "failure_count": len(
            phase31_failure_types
        ),
        "failure_type_counts": {
            failure_type: (
                phase31_failure_types.count(
                    failure_type
                )
            )
            for failure_type in sorted(
                set(phase31_failure_types)
            )
        },
    }

    os.replace(
        PHASE31_FEATURE_PARTIAL,
        PHASE31_FEATURE_PATH,
    )
    os.replace(
        PHASE31_SEQUENCE_PARTIAL,
        PHASE31_SEQUENCE_PATH,
    )
    os.replace(
        PHASE31_DIAGNOSTIC_PARTIAL,
        PHASE31_DIAGNOSTIC_PATH,
    )

    phase31_metadata = {
        "schema_version": 1,
        "phase": PHASE31_CONFIG["phase"],
        "case_count": EXPECTED_CASES,
        "feature_count": 1081,
        "sequence_shape": [16, 52],
        "diagnostic_shape": [5],
        "feature_names": PHASE31_FEATURE_NAMES,
        "sequence_names": PHASE31_SEQUENCE_NAMES,
        "diagnostic_names": [
            "background",
            "center_separation_voxels",
            "left_center_x",
            "right_center_x",
            "profile_common_scale",
        ],
        "reflection_invariant": True,
        "persistent_notebook_cache_only": True,
        "include_in_submission": False,
    }

    PHASE31_METADATA_PATH.write_text(
        json.dumps(
            phase31_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    phase31_cached_result = (
        phase31_validate_extracted_cache()
    )

    assert phase31_cached_result is not None, (
        "The completed Phase31 cache failed validation."
    )

else:
    print(
        "Valid Phase31 feature cache found; "
        "extraction skipped."
    )


PHASE31_RADIOMICS_PRIVATE = (
    phase31_cached_result["features"]
)
PHASE31_SEQUENCE_PRIVATE = (
    phase31_cached_result["sequences"]
)
PHASE31_LOCALIZATION_DIAGNOSTICS_PRIVATE = (
    phase31_cached_result["diagnostics"]
)

phase31_metadata = (
    phase31_cached_result["metadata"]
)

PHASE31_FEATURE_NAMES = list(
    phase31_metadata["feature_names"]
)
PHASE31_SEQUENCE_NAMES = list(
    phase31_metadata["sequence_names"]
)

phase31_extraction_elapsed = (
    time.perf_counter()
    - phase31_extraction_started
)

print(
    "Phase31 extraction/load complete: "
    f"{phase31_extraction_elapsed:.2f}s"
)

Phase31 feature extraction: 100/1362
Phase31 feature extraction: 200/1362
Phase31 feature extraction: 300/1362
Phase31 feature extraction: 400/1362
Phase31 feature extraction: 500/1362
Phase31 feature extraction: 600/1362
Phase31 feature extraction: 700/1362
Phase31 feature extraction: 800/1362
Phase31 feature extraction: 900/1362
Phase31 feature extraction: 1000/1362
Phase31 feature extraction: 1100/1362
Phase31 feature extraction: 1200/1362
Phase31 feature extraction: 1300/1362
Phase31 feature extraction: 1362/1362
Phase31 extraction/load complete: 253.29s


In [11]:
# Phase31 Cell 101C
# Final validation and sanitized report.

assert PHASE31_RADIOMICS_PRIVATE.shape == (
    EXPECTED_CASES,
    1081,
)

assert PHASE31_SEQUENCE_PRIVATE.shape == (
    EXPECTED_CASES,
    16,
    52,
)

assert (
    PHASE31_LOCALIZATION_DIAGNOSTICS_PRIVATE.shape
    == (EXPECTED_CASES, 5)
)

assert len(PHASE31_FEATURE_NAMES) == 1081
assert len(PHASE31_SEQUENCE_NAMES) == 52

report = {
    "phase": "phase31_localized_feature_cache",
    "status": "complete",
    "cache_reused": (
        phase31_feature_cache_reused
    ),
    "case_count": EXPECTED_CASES,
    "radiomics": {
        "shape": list(
            PHASE31_RADIOMICS_PRIVATE.shape
        ),
        "dtype": str(
            PHASE31_RADIOMICS_PRIVATE.dtype
        ),
        "feature_count": len(
            PHASE31_FEATURE_NAMES
        ),
        "size_mb": round(
            PHASE31_FEATURE_PATH.stat().st_size
            / (1024 ** 2),
            3,
        ),
    },
    "sequence": {
        "shape": list(
            PHASE31_SEQUENCE_PRIVATE.shape
        ),
        "dtype": str(
            PHASE31_SEQUENCE_PRIVATE.dtype
        ),
        "token_count": 16,
        "token_dimension": 52,
        "size_mb": round(
            PHASE31_SEQUENCE_PATH.stat().st_size
            / (1024 ** 2),
            3,
        ),
    },
    "diagnostic_shape": list(
        PHASE31_LOCALIZATION_DIAGNOSTICS_PRIVATE.shape
    ),
    "failure_count": 0,
    "elapsed_seconds": round(
        phase31_extraction_elapsed,
        2,
    ),
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "training_voxel_cache_read": (
        not phase31_feature_cache_reused
    ),
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
}

print("BEGIN SANITIZED_PHASE31_FEATURE_CACHE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_FEATURE_CACHE")

BEGIN SANITIZED_PHASE31_FEATURE_CACHE
{
  "phase": "phase31_localized_feature_cache",
  "status": "complete",
  "cache_reused": false,
  "case_count": 1362,
  "radiomics": {
    "shape": [
      1362,
      1081
    ],
    "dtype": "float32",
    "feature_count": 1081,
    "size_mb": 5.617
  },
  "sequence": {
    "shape": [
      1362,
      16,
      52
    ],
    "dtype": "float32",
    "token_count": 16,
    "token_dimension": 52,
    "size_mb": 4.323
  },
  "diagnostic_shape": [
    1362,
    5
  ],
  "failure_count": 0,
  "elapsed_seconds": 253.29,
  "persistent_notebook_cache_only": true,
  "include_in_submission": false,
  "training_voxel_cache_read": true,
  "smoke_data_read": false,
  "test_data_read": false,
  "uids_displayed": false,
  "patient_rows_displayed": false
}
END SANITIZED_PHASE31_FEATURE_CACHE


In [12]:
# Phase31 Cell 102
# Aggregate, label-free audit before fitting any classifier.

radiomics_matrix = np.asarray(
    PHASE31_RADIOMICS_PRIVATE,
    dtype=np.float64,
)

sequence_matrix = np.asarray(
    PHASE31_SEQUENCE_PRIVATE,
    dtype=np.float64,
)

localization_diagnostics = np.asarray(
    PHASE31_LOCALIZATION_DIAGNOSTICS_PRIVATE,
    dtype=np.float64,
)

assert radiomics_matrix.shape == (
    EXPECTED_CASES,
    1081,
)

assert sequence_matrix.shape == (
    EXPECTED_CASES,
    16,
    52,
)

assert np.isfinite(radiomics_matrix).all()
assert np.isfinite(sequence_matrix).all()
assert np.isfinite(
    localization_diagnostics
).all()


# ---------------------------------------------------------
# Coordinate-level variability
# ---------------------------------------------------------

radiomics_standard_deviation = np.std(
    radiomics_matrix,
    axis=0,
)

sequence_flat = sequence_matrix.reshape(
    EXPECTED_CASES,
    -1,
)

sequence_standard_deviation = np.std(
    sequence_flat,
    axis=0,
)

radiomics_collapsed = (
    radiomics_standard_deviation < 1e-8
)

radiomics_near_constant = (
    radiomics_standard_deviation < 1e-5
)

sequence_collapsed = (
    sequence_standard_deviation < 1e-8
)

sequence_near_constant = (
    sequence_standard_deviation < 1e-5
)


# ---------------------------------------------------------
# Robust aggregate value ranges
# ---------------------------------------------------------

radiomics_absolute = np.abs(
    radiomics_matrix
)

sequence_absolute = np.abs(
    sequence_matrix
)

radiomics_absolute_quantiles = np.quantile(
    radiomics_absolute,
    [0.50, 0.90, 0.99, 0.999],
)

sequence_absolute_quantiles = np.quantile(
    sequence_absolute,
    [0.50, 0.90, 0.99, 0.999],
)


# ---------------------------------------------------------
# Localization diagnostics
# ---------------------------------------------------------

background = localization_diagnostics[:, 0]
center_separation = localization_diagnostics[:, 1]
left_center_x = localization_diagnostics[:, 2]
right_center_x = localization_diagnostics[:, 3]
profile_scale = localization_diagnostics[:, 4]

left_half_contract = bool(
    np.all(
        (left_center_x >= 0.0)
        & (left_center_x < 40.0)
    )
)

right_half_contract = bool(
    np.all(
        (right_center_x >= 40.0)
        & (right_center_x < 80.0)
    )
)

center_order_contract = bool(
    np.all(
        right_center_x > left_center_x
    )
)

positive_background_contract = bool(
    np.all(background > 0.0)
)

positive_profile_scale_contract = bool(
    np.all(profile_scale > 0.0)
)


# ---------------------------------------------------------
# Audit thresholds
# ---------------------------------------------------------

radiomics_collapsed_fraction = float(
    np.mean(radiomics_collapsed)
)

sequence_collapsed_fraction = float(
    np.mean(sequence_collapsed)
)

audit_passed = bool(
    radiomics_collapsed_fraction < 0.50
    and sequence_collapsed_fraction < 0.50
    and left_half_contract
    and right_half_contract
    and center_order_contract
    and positive_background_contract
    and positive_profile_scale_contract
)

report = {
    "phase": "phase31_feature_quality_audit",
    "status": (
        "accepted"
        if audit_passed
        else "rejected"
    ),
    "radiomics": {
        "coordinate_count": int(
            radiomics_matrix.shape[1]
        ),
        "collapsed_coordinate_count": int(
            np.sum(radiomics_collapsed)
        ),
        "collapsed_coordinate_fraction": round(
            radiomics_collapsed_fraction,
            6,
        ),
        "near_constant_coordinate_count": int(
            np.sum(radiomics_near_constant)
        ),
        "median_coordinate_standard_deviation": round(
            float(
                np.median(
                    radiomics_standard_deviation
                )
            ),
            8,
        ),
        "absolute_value_quantiles": {
            "q50": round(
                float(
                    radiomics_absolute_quantiles[0]
                ),
                6,
            ),
            "q90": round(
                float(
                    radiomics_absolute_quantiles[1]
                ),
                6,
            ),
            "q99": round(
                float(
                    radiomics_absolute_quantiles[2]
                ),
                6,
            ),
            "q999": round(
                float(
                    radiomics_absolute_quantiles[3]
                ),
                6,
            ),
        },
    },
    "sequence": {
        "token_count": 16,
        "token_dimension": 52,
        "flattened_coordinate_count": int(
            sequence_flat.shape[1]
        ),
        "collapsed_coordinate_count": int(
            np.sum(sequence_collapsed)
        ),
        "collapsed_coordinate_fraction": round(
            sequence_collapsed_fraction,
            6,
        ),
        "near_constant_coordinate_count": int(
            np.sum(sequence_near_constant)
        ),
        "median_coordinate_standard_deviation": round(
            float(
                np.median(
                    sequence_standard_deviation
                )
            ),
            8,
        ),
        "absolute_value_quantiles": {
            "q50": round(
                float(
                    sequence_absolute_quantiles[0]
                ),
                6,
            ),
            "q90": round(
                float(
                    sequence_absolute_quantiles[1]
                ),
                6,
            ),
            "q99": round(
                float(
                    sequence_absolute_quantiles[2]
                ),
                6,
            ),
            "q999": round(
                float(
                    sequence_absolute_quantiles[3]
                ),
                6,
            ),
        },
    },
    "localization": {
        "left_half_contract": left_half_contract,
        "right_half_contract": right_half_contract,
        "center_order_contract": center_order_contract,
        "positive_background_contract": (
            positive_background_contract
        ),
        "positive_profile_scale_contract": (
            positive_profile_scale_contract
        ),
        "center_separation_voxel_quantiles": {
            "q01": round(
                float(
                    np.quantile(
                        center_separation,
                        0.01,
                    )
                ),
                4,
            ),
            "q05": round(
                float(
                    np.quantile(
                        center_separation,
                        0.05,
                    )
                ),
                4,
            ),
            "q50": round(
                float(
                    np.quantile(
                        center_separation,
                        0.50,
                    )
                ),
                4,
            ),
            "q95": round(
                float(
                    np.quantile(
                        center_separation,
                        0.95,
                    )
                ),
                4,
            ),
            "q99": round(
                float(
                    np.quantile(
                        center_separation,
                        0.99,
                    )
                ),
                4,
            ),
        },
        "background_quantiles": {
            "q05": round(
                float(
                    np.quantile(
                        background,
                        0.05,
                    )
                ),
                6,
            ),
            "q50": round(
                float(
                    np.quantile(
                        background,
                        0.50,
                    )
                ),
                6,
            ),
            "q95": round(
                float(
                    np.quantile(
                        background,
                        0.95,
                    )
                ),
                6,
            ),
        },
    },
    "labels_used": False,
    "case_level_values_displayed": False,
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
}

print("BEGIN SANITIZED_PHASE31_FEATURE_AUDIT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_FEATURE_AUDIT")

assert audit_passed, (
    "Do not train Phase31 until the feature audit passes."
)

BEGIN SANITIZED_PHASE31_FEATURE_AUDIT
{
  "phase": "phase31_feature_quality_audit",
  "status": "accepted",
  "radiomics": {
    "coordinate_count": 1081,
    "collapsed_coordinate_count": 0,
    "collapsed_coordinate_fraction": 0.0,
    "near_constant_coordinate_count": 0,
    "median_coordinate_standard_deviation": 0.10583945,
    "absolute_value_quantiles": {
      "q50": 0.411621,
      "q90": 1.5,
      "q99": 7.0,
      "q999": 28.0
    }
  },
  "sequence": {
    "token_count": 16,
    "token_dimension": 52,
    "flattened_coordinate_count": 832,
    "collapsed_coordinate_count": 0,
    "collapsed_coordinate_fraction": 0.0,
    "near_constant_coordinate_count": 0,
    "median_coordinate_standard_deviation": 0.08062628,
    "absolute_value_quantiles": {
      "q50": 0.104835,
      "q90": 0.555086,
      "q99": 1.0,
      "q999": 1.167404
    }
  },
  "localization": {
    "left_half_contract": true,
    "right_half_contract": true,
    "center_order_contract": true,
    "positive

In [13]:
# Phase31 Cell 103A
# Build the three candidate representations and a restrained model grid.

from sklearn.decomposition import PCA
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

PHASE31_MODEL_CONFIG = {
    "phase": "phase31_nested_classical_gate",
    "winsor_lower_quantile": 0.005,
    "winsor_upper_quantile": 0.995,
    "variance_threshold": 1e-10,
    "pca_dimensions": [32, 64, 128],
    "logistic_c_values": [0.03, 0.10, 0.30],
    "tree_selected_features": 256,
    "extra_trees_minimum_leaves": [6, 12],
    "histogram_leaf_nodes": [7, 15],
    "base_seed": 310701,
    "probability_clip": 1e-5,
}


radiomics_representation = np.asarray(
    PHASE31_RADIOMICS_PRIVATE,
    dtype=np.float32,
)

sequence_representation = np.asarray(
    PHASE31_SEQUENCE_PRIVATE,
    dtype=np.float32,
).reshape(EXPECTED_CASES, -1)

combined_representation = np.concatenate([
    radiomics_representation,
    sequence_representation,
], axis=1).astype(
    np.float32,
    copy=False,
)

PHASE31_REPRESENTATIONS_PRIVATE = {
    "radiomics": radiomics_representation,
    "sequence_flat": sequence_representation,
    "radiomics_plus_sequence": combined_representation,
}

expected_dimensions = {
    "radiomics": 1081,
    "sequence_flat": 832,
    "radiomics_plus_sequence": 1913,
}

for name, matrix in PHASE31_REPRESENTATIONS_PRIVATE.items():
    assert matrix.shape == (
        EXPECTED_CASES,
        expected_dimensions[name],
    )
    assert np.isfinite(matrix).all()


PHASE31_CANDIDATE_SPECS = []

for representation_name in (
    "radiomics",
    "sequence_flat",
    "radiomics_plus_sequence",
):
    for dimension in PHASE31_MODEL_CONFIG[
        "pca_dimensions"
    ]:
        for c_value in PHASE31_MODEL_CONFIG[
            "logistic_c_values"
        ]:
            PHASE31_CANDIDATE_SPECS.append({
                "representation": representation_name,
                "family": "pca_ridge_logistic",
                "pca_dimension": int(dimension),
                "c": float(c_value),
            })

    for minimum_leaf in PHASE31_MODEL_CONFIG[
        "extra_trees_minimum_leaves"
    ]:
        PHASE31_CANDIDATE_SPECS.append({
            "representation": representation_name,
            "family": "extra_trees",
            "selected_features": 256,
            "minimum_leaf": int(minimum_leaf),
            "max_features": 0.50,
            "estimators": 600,
        })

    for leaf_nodes in PHASE31_MODEL_CONFIG[
        "histogram_leaf_nodes"
    ]:
        PHASE31_CANDIDATE_SPECS.append({
            "representation": representation_name,
            "family": "histogram_boosting",
            "selected_features": 256,
            "leaf_nodes": int(leaf_nodes),
            "minimum_leaf": 24,
            "learning_rate": 0.035,
            "iterations": 300,
            "l2_regularization": 5.0,
        })


assert len(PHASE31_CANDIDATE_SPECS) == 39

report = {
    "phase": "phase31_classical_candidate_contract",
    "status": "accepted",
    "representations": {
        name: {
            "shape": list(matrix.shape),
            "coordinate_count": int(
                matrix.shape[1]
            ),
        }
        for name, matrix
        in PHASE31_REPRESENTATIONS_PRIVATE.items()
    },
    "candidate_count": len(
        PHASE31_CANDIDATE_SPECS
    ),
    "candidate_family_counts": {
        family: sum(
            spec["family"] == family
            for spec in PHASE31_CANDIDATE_SPECS
        )
        for family in [
            "pca_ridge_logistic",
            "extra_trees",
            "histogram_boosting",
        ]
    },
    "selection_partition": "monitor_only",
    "outer_validation_labels_used": False,
    "case_level_features_exported": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE31_CANDIDATES")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_CANDIDATES")

BEGIN SANITIZED_PHASE31_CANDIDATES
{
  "phase": "phase31_classical_candidate_contract",
  "status": "accepted",
  "representations": {
    "radiomics": {
      "shape": [
        1362,
        1081
      ],
      "coordinate_count": 1081
    },
    "sequence_flat": {
      "shape": [
        1362,
        832
      ],
      "coordinate_count": 832
    },
    "radiomics_plus_sequence": {
      "shape": [
        1362,
        1913
      ],
      "coordinate_count": 1913
    }
  },
  "candidate_count": 39,
  "candidate_family_counts": {
    "pca_ridge_logistic": 27,
    "extra_trees": 6,
    "histogram_boosting": 6
  },
  "selection_partition": "monitor_only",
  "outer_validation_labels_used": false,
  "case_level_features_exported": false,
  "test_or_smoke_data_read": false
}
END SANITIZED_PHASE31_CANDIDATES


In [14]:
# Phase31 Cell 103B
# All transformations are fitted strictly on the supplied training indices.

def phase31_clip_probabilities(probabilities):
    epsilon = PHASE31_MODEL_CONFIG[
        "probability_clip"
    ]

    return np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        epsilon,
        1.0 - epsilon,
    )


def phase31_binary_metrics(y_true, probabilities):
    probabilities = phase31_clip_probabilities(
        probabilities
    )
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    return {
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "mean_probability": float(
            probabilities.mean()
        ),
    }


def phase31_winsor_fit(matrix):
    lower = np.quantile(
        matrix,
        PHASE31_MODEL_CONFIG[
            "winsor_lower_quantile"
        ],
        axis=0,
    )

    upper = np.quantile(
        matrix,
        PHASE31_MODEL_CONFIG[
            "winsor_upper_quantile"
        ],
        axis=0,
    )

    upper = np.maximum(
        upper,
        lower,
    )

    clipped = np.clip(
        matrix,
        lower,
        upper,
    )

    variance = np.var(
        clipped,
        axis=0,
    )

    keep = variance > PHASE31_MODEL_CONFIG[
        "variance_threshold"
    ]

    assert np.any(keep)

    return (
        lower.astype(np.float64),
        upper.astype(np.float64),
        keep,
    )


def phase31_apply_basic_preprocessing(
    matrix,
    lower,
    upper,
    keep,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    matrix = np.clip(
        matrix,
        lower,
        upper,
    )

    matrix = matrix[:, keep]

    assert np.isfinite(matrix).all()

    return matrix


def phase31_fit_candidate(
    train_matrix,
    train_labels,
    evaluation_matrix,
    specification,
    seed,
):
    train_matrix = np.asarray(
        train_matrix,
        dtype=np.float64,
    )
    evaluation_matrix = np.asarray(
        evaluation_matrix,
        dtype=np.float64,
    )
    train_labels = np.asarray(
        train_labels,
        dtype=np.int64,
    )

    lower, upper, keep = phase31_winsor_fit(
        train_matrix
    )

    train_processed = (
        phase31_apply_basic_preprocessing(
            train_matrix,
            lower,
            upper,
            keep,
        )
    )

    evaluation_processed = (
        phase31_apply_basic_preprocessing(
            evaluation_matrix,
            lower,
            upper,
            keep,
        )
    )

    state = {
        "specification": dict(specification),
        "lower": lower,
        "upper": upper,
        "keep": keep,
    }

    family = specification["family"]

    if family == "pca_ridge_logistic":
        scaler = StandardScaler(
            with_mean=True,
            with_std=True,
        )

        train_scaled = scaler.fit_transform(
            train_processed
        )

        evaluation_scaled = scaler.transform(
            evaluation_processed
        )

        dimension = min(
            int(specification["pca_dimension"]),
            train_scaled.shape[0] - 1,
            train_scaled.shape[1],
        )

        pca = PCA(
            n_components=dimension,
            whiten=True,
            svd_solver="randomized",
            iterated_power=3,
            random_state=seed,
        )

        train_reduced = pca.fit_transform(
            train_scaled
        )

        evaluation_reduced = pca.transform(
            evaluation_scaled
        )

        model = LogisticRegression(
            C=float(specification["c"]),
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=seed,
        )

        model.fit(
            train_reduced,
            train_labels,
        )

        probability = model.predict_proba(
            evaluation_reduced
        )[:, 1]

        state.update({
            "scaler": scaler,
            "pca": pca,
            "model": model,
        })

    elif family in {
        "extra_trees",
        "histogram_boosting",
    }:
        selected_features = min(
            int(
                specification[
                    "selected_features"
                ]
            ),
            train_processed.shape[1],
        )

        selector = SelectKBest(
            score_func=f_classif,
            k=selected_features,
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            train_selected = (
                selector.fit_transform(
                    train_processed,
                    train_labels,
                )
            )

        evaluation_selected = (
            selector.transform(
                evaluation_processed
            )
        )

        if family == "extra_trees":
            model = ExtraTreesClassifier(
                n_estimators=int(
                    specification["estimators"]
                ),
                min_samples_leaf=int(
                    specification["minimum_leaf"]
                ),
                max_features=float(
                    specification["max_features"]
                ),
                bootstrap=False,
                criterion="log_loss",
                n_jobs=-1,
                random_state=seed,
            )

        else:
            model = HistGradientBoostingClassifier(
                learning_rate=float(
                    specification["learning_rate"]
                ),
                max_iter=int(
                    specification["iterations"]
                ),
                max_leaf_nodes=int(
                    specification["leaf_nodes"]
                ),
                min_samples_leaf=int(
                    specification["minimum_leaf"]
                ),
                l2_regularization=float(
                    specification[
                        "l2_regularization"
                    ]
                ),
                max_bins=127,
                early_stopping=False,
                random_state=seed,
            )

        model.fit(
            train_selected,
            train_labels,
        )

        probability = model.predict_proba(
            evaluation_selected
        )[:, 1]

        state.update({
            "selector": selector,
            "model": model,
        })

    else:
        raise ValueError(
            f"Unknown model family: {family}"
        )

    probability = phase31_clip_probabilities(
        probability
    )

    return state, probability


def phase31_predict_candidate(
    state,
    matrix,
):
    matrix = phase31_apply_basic_preprocessing(
        matrix,
        state["lower"],
        state["upper"],
        state["keep"],
    )

    family = state[
        "specification"
    ]["family"]

    if family == "pca_ridge_logistic":
        matrix = state["scaler"].transform(
            matrix
        )
        matrix = state["pca"].transform(
            matrix
        )

    else:
        matrix = state["selector"].transform(
            matrix
        )

    probability = state[
        "model"
    ].predict_proba(matrix)[:, 1]

    return phase31_clip_probabilities(
        probability
    )


def phase31_specification_complexity(spec):
    representation_rank = {
        "radiomics": 0,
        "sequence_flat": 1,
        "radiomics_plus_sequence": 2,
    }[spec["representation"]]

    family_rank = {
        "pca_ridge_logistic": 0,
        "extra_trees": 1,
        "histogram_boosting": 2,
    }[spec["family"]]

    if spec["family"] == "pca_ridge_logistic":
        capacity = (
            int(spec["pca_dimension"]),
            float(spec["c"]),
        )
    elif spec["family"] == "extra_trees":
        capacity = (
            -int(spec["minimum_leaf"]),
            float(spec["max_features"]),
        )
    else:
        capacity = (
            int(spec["leaf_nodes"]),
            int(spec["iterations"]),
        )

    return (
        representation_rank,
        family_rank,
        capacity,
    )


print("Phase31 Cell 103B model functions accepted.")

Phase31 Cell 103B model functions accepted.


In [15]:
# Phase31 Cell 103C
# Select on monitor partitions, freeze, refit on outer_train,
# and evaluate outer_valid exactly once.

PHASE31_REFERENCE = {
    "overall": {
        "log_loss": 0.307615,
        "auroc": 0.938928,
        "brier": 0.095535,
    },
    "folds": {
        0: {
            "log_loss": 0.358494,
            "auroc": 0.916174,
        },
        1: {
            "log_loss": 0.255415,
            "auroc": 0.952183,
        },
        2: {
            "log_loss": 0.306209,
            "auroc": 0.945716,
        },
    },
    "phase18_morphology_proxy": {
        "log_loss": 0.427977,
        "auroc": 0.889257,
    },
}

phase31_training_started = time.perf_counter()

PHASE31_CLASSICAL_OOF_PRIVATE = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

PHASE31_CLASSICAL_STATES_PRIVATE = {}
PHASE31_CLASSICAL_SELECTION_PRIVATE = []

for partition in PHASE31_PARTITIONS:
    fold = int(partition["fold"])
    fit_indices = partition["fit"]
    monitor_indices = partition["monitor"]
    outer_train_indices = partition["outer_train"]
    outer_valid_indices = partition["outer_valid"]

    candidate_records = []

    fold_started = time.perf_counter()

    for candidate_index, specification in enumerate(
        PHASE31_CANDIDATE_SPECS
    ):
        representation_name = (
            specification["representation"]
        )

        matrix = (
            PHASE31_REPRESENTATIONS_PRIVATE[
                representation_name
            ]
        )

        seed = (
            PHASE31_MODEL_CONFIG["base_seed"]
            + fold * 1000
            + candidate_index
        )

        try:
            _, monitor_probability = (
                phase31_fit_candidate(
                    matrix[fit_indices],
                    PHASE31_LABELS[fit_indices],
                    matrix[monitor_indices],
                    specification,
                    seed,
                )
            )

            monitor_metrics = (
                phase31_binary_metrics(
                    PHASE31_LABELS[
                        monitor_indices
                    ],
                    monitor_probability,
                )
            )

            candidate_records.append({
                "candidate_index": (
                    candidate_index
                ),
                "specification": dict(
                    specification
                ),
                "monitor_metrics": (
                    monitor_metrics
                ),
            })

        except Exception as exc:
            candidate_records.append({
                "candidate_index": (
                    candidate_index
                ),
                "specification": dict(
                    specification
                ),
                "error_type": (
                    type(exc).__name__
                ),
            })

        if (
            (candidate_index + 1) % 10 == 0
            or candidate_index + 1
            == len(PHASE31_CANDIDATE_SPECS)
        ):
            print(
                f"Phase31 fold {fold}: "
                f"{candidate_index + 1}/"
                f"{len(PHASE31_CANDIDATE_SPECS)} "
                "candidates evaluated"
            )

    successful_candidates = [
        record
        for record in candidate_records
        if "monitor_metrics" in record
    ]

    assert successful_candidates, {
        "message": "All Phase31 candidates failed.",
        "fold": fold,
    }

    successful_candidates.sort(
        key=lambda record: (
            record["monitor_metrics"][
                "log_loss"
            ],
            phase31_specification_complexity(
                record["specification"]
            ),
        )
    )

    selected = successful_candidates[0]
    selected_specification = dict(
        selected["specification"]
    )

    selected_matrix = (
        PHASE31_REPRESENTATIONS_PRIVATE[
            selected_specification[
                "representation"
            ]
        ]
    )

    final_seed = (
        PHASE31_MODEL_CONFIG["base_seed"]
        + fold * 1000
        + 900
    )

    final_state, outer_probability = (
        phase31_fit_candidate(
            selected_matrix[
                outer_train_indices
            ],
            PHASE31_LABELS[
                outer_train_indices
            ],
            selected_matrix[
                outer_valid_indices
            ],
            selected_specification,
            final_seed,
        )
    )

    # Reconstruction contract for the retained state.
    reconstructed_probability = (
        phase31_predict_candidate(
            final_state,
            selected_matrix[
                outer_valid_indices
            ],
        )
    )

    reconstruction_error = float(
        np.max(
            np.abs(
                outer_probability
                - reconstructed_probability
            )
        )
    )

    assert reconstruction_error < 1e-12

    PHASE31_CLASSICAL_OOF_PRIVATE[
        outer_valid_indices
    ] = outer_probability

    PHASE31_CLASSICAL_STATES_PRIVATE[
        fold
    ] = final_state

    outer_metrics = phase31_binary_metrics(
        PHASE31_LABELS[
            outer_valid_indices
        ],
        outer_probability,
    )

    fold_elapsed = (
        time.perf_counter()
        - fold_started
    )

    selection_record = {
        "fold": fold,
        "fit_n": len(fit_indices),
        "monitor_n": len(
            monitor_indices
        ),
        "outer_train_n": len(
            outer_train_indices
        ),
        "outer_valid_n": len(
            outer_valid_indices
        ),
        "candidate_count": len(
            PHASE31_CANDIDATE_SPECS
        ),
        "successful_candidate_count": len(
            successful_candidates
        ),
        "selected_specification": (
            selected_specification
        ),
        "monitor_metrics": (
            selected["monitor_metrics"]
        ),
        "outer_metrics": outer_metrics,
        "checkpoint_reconstruction_error": (
            reconstruction_error
        ),
        "elapsed_seconds": fold_elapsed,
    }

    PHASE31_CLASSICAL_SELECTION_PRIVATE.append(
        selection_record
    )

    print(
        f"Phase31 fold {fold} selected: "
        f"{selected_specification['representation']}, "
        f"{selected_specification['family']}, "
        f"monitor={selected['monitor_metrics']['log_loss']:.6f}, "
        f"outer={outer_metrics['log_loss']:.6f}, "
        f"auroc={outer_metrics['auroc']:.6f}"
    )


assert np.isfinite(
    PHASE31_CLASSICAL_OOF_PRIVATE
).all()

phase31_overall_metrics = phase31_binary_metrics(
    PHASE31_LABELS,
    PHASE31_CLASSICAL_OOF_PRIVATE,
)

phase31_fold_metrics = []

for partition in PHASE31_PARTITIONS:
    fold = int(partition["fold"])
    indices = partition["outer_valid"]

    metrics = phase31_binary_metrics(
        PHASE31_LABELS[indices],
        PHASE31_CLASSICAL_OOF_PRIVATE[
            indices
        ],
    )

    phase31_fold_metrics.append({
        "fold": fold,
        "n": len(indices),
        **metrics,
    })


# Major acquisition-group robustness.
phase31_major_group_metrics = []

for group in sorted(
    np.unique(
        PHASE31_ACQUISITION_GROUPS
    ).tolist()
):
    mask = (
        PHASE31_ACQUISITION_GROUPS
        == group
    )
    count = int(mask.sum())

    if count < 30:
        continue

    group_labels = PHASE31_LABELS[mask]
    group_probability = (
        PHASE31_CLASSICAL_OOF_PRIVATE[mask]
    )

    group_record = {
        "group": int(group),
        "n": count,
        "log_loss": float(
            log_loss(
                group_labels,
                group_probability,
                labels=[0, 1],
            )
        ),
        "brier": float(
            brier_score_loss(
                group_labels,
                group_probability,
            )
        ),
    }

    if np.unique(group_labels).size == 2:
        group_record["auroc"] = float(
            roc_auc_score(
                group_labels,
                group_probability,
            )
        )
    else:
        group_record["auroc"] = None

    phase31_major_group_metrics.append(
        group_record
    )


baseline = PHASE31_REFERENCE["overall"]

log_loss_gain = (
    baseline["log_loss"]
    - phase31_overall_metrics["log_loss"]
)

auroc_gain = (
    phase31_overall_metrics["auroc"]
    - baseline["auroc"]
)

brier_excess = (
    phase31_overall_metrics["brier"]
    - baseline["brier"]
)

fold_wins = sum(
    record["log_loss"]
    < PHASE31_REFERENCE["folds"][
        record["fold"]
    ]["log_loss"]
    for record in phase31_fold_metrics
)

worst_fold_excess = max(
    record["log_loss"]
    - PHASE31_REFERENCE["folds"][
        record["fold"]
    ]["log_loss"]
    for record in phase31_fold_metrics
)

phase18_reference = PHASE31_REFERENCE[
    "phase18_morphology_proxy"
]

gain_over_old_morphology = (
    phase18_reference["log_loss"]
    - phase31_overall_metrics["log_loss"]
)

auroc_gain_over_old_morphology = (
    phase31_overall_metrics["auroc"]
    - phase18_reference["auroc"]
)

component_gate_passed = bool(
    phase31_overall_metrics["log_loss"]
    <= 0.400
    and phase31_overall_metrics["auroc"]
    >= 0.910
    and sum(
        record["auroc"] >= 0.880
        for record in phase31_fold_metrics
    ) >= 2
)

champion_gate_passed = bool(
    log_loss_gain >= 0.003
    and auroc_gain >= 0.002
    and fold_wins >= 2
    and worst_fold_excess <= 0.005
)

phase31_training_elapsed = (
    time.perf_counter()
    - phase31_training_started
)


# Persist private OOF values for the later stacking experiment.
PHASE31_CLASSICAL_OOF_PATH = Path(
    "/kaggle/working/"
    "phase31_classical_oof_private.npy"
)

np.save(
    PHASE31_CLASSICAL_OOF_PATH,
    PHASE31_CLASSICAL_OOF_PRIVATE.astype(
        np.float32
    ),
)

PHASE31_SELECTION_PATH = Path(
    "/kaggle/working/"
    "phase31_classical_selection.json"
)

serializable_selection = []

for record in PHASE31_CLASSICAL_SELECTION_PRIVATE:
    serializable_selection.append({
        "fold": record["fold"],
        "fit_n": record["fit_n"],
        "monitor_n": record["monitor_n"],
        "outer_train_n": record[
            "outer_train_n"
        ],
        "outer_valid_n": record[
            "outer_valid_n"
        ],
        "candidate_count": record[
            "candidate_count"
        ],
        "successful_candidate_count": record[
            "successful_candidate_count"
        ],
        "selected_specification": record[
            "selected_specification"
        ],
        "monitor_metrics": {
            key: float(value)
            for key, value in record[
                "monitor_metrics"
            ].items()
        },
        "outer_metrics": {
            key: float(value)
            for key, value in record[
                "outer_metrics"
            ].items()
        },
        "checkpoint_reconstruction_error": float(
            record[
                "checkpoint_reconstruction_error"
            ]
        ),
        "elapsed_seconds": float(
            record["elapsed_seconds"]
        ),
    })

PHASE31_SELECTION_PATH.write_text(
    json.dumps(
        serializable_selection,
        indent=2,
    ),
    encoding="utf-8",
)


report = {
    "phase": "phase31_nested_classical_gate",
    "status": (
        "component_gate_passed"
        if component_gate_passed
        else "component_gate_failed"
    ),
    "baseline": baseline,
    "phase31": {
        key: round(float(value), 6)
        for key, value
        in phase31_overall_metrics.items()
    },
    "improvements": {
        "log_loss_gain_over_phase12c": round(
            float(log_loss_gain),
            6,
        ),
        "auroc_gain_over_phase12c": round(
            float(auroc_gain),
            6,
        ),
        "brier_excess": round(
            float(brier_excess),
            6,
        ),
        "fold_wins": int(fold_wins),
        "worst_fold_excess": round(
            float(worst_fold_excess),
            6,
        ),
        "log_loss_gain_over_phase18_morphology": round(
            float(gain_over_old_morphology),
            6,
        ),
        "auroc_gain_over_phase18_morphology": round(
            float(
                auroc_gain_over_old_morphology
            ),
            6,
        ),
    },
    "fold_metrics": [
        {
            key: (
                round(float(value), 6)
                if isinstance(
                    value,
                    (float, np.floating),
                )
                else value
            )
            for key, value in record.items()
        }
        for record in phase31_fold_metrics
    ],
    "selection": serializable_selection,
    "major_acquisition_group_metrics": [
        {
            key: (
                round(float(value), 6)
                if isinstance(
                    value,
                    (float, np.floating),
                )
                else value
            )
            for key, value in record.items()
        }
        for record
        in phase31_major_group_metrics
    ],
    "gates": {
        "component_gate_passed": (
            component_gate_passed
        ),
        "champion_gate_passed": (
            champion_gate_passed
        ),
        "component_gate_meaning": (
            "eligible_for_geometry_semantic_"
            "sequence_model_and_blend_test"
        ),
    },
    "elapsed_seconds": round(
        phase31_training_elapsed,
        2,
    ),
    "monitor_labels_used_for_selection": True,
    "outer_train_labels_used_for_final_refit": True,
    "outer_validation_labels_used_only_for_evaluation": True,
    "selection_frozen_before_outer_evaluation": True,
    "case_level_predictions_exported": False,
    "persistent_notebook_cache_only": True,
    "include_private_oof_in_submission": False,
    "test_or_smoke_data_read": False,
}

print("BEGIN SANITIZED_PHASE31_CLASSICAL_OOF")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE31_CLASSICAL_OOF")

Phase31 fold 0: 10/39 candidates evaluated
Phase31 fold 0: 20/39 candidates evaluated
Phase31 fold 0: 30/39 candidates evaluated
Phase31 fold 0: 39/39 candidates evaluated
Phase31 fold 0 selected: sequence_flat, histogram_boosting, monitor=0.283676, outer=0.456262, auroc=0.873693
Phase31 fold 1: 10/39 candidates evaluated
Phase31 fold 1: 20/39 candidates evaluated
Phase31 fold 1: 30/39 candidates evaluated
Phase31 fold 1: 39/39 candidates evaluated
Phase31 fold 1 selected: radiomics, pca_ridge_logistic, monitor=0.262622, outer=0.330455, auroc=0.939984
Phase31 fold 2: 10/39 candidates evaluated
Phase31 fold 2: 20/39 candidates evaluated
Phase31 fold 2: 30/39 candidates evaluated
Phase31 fold 2: 39/39 candidates evaluated
Phase31 fold 2 selected: sequence_flat, extra_trees, monitor=0.312936, outer=0.337170, auroc=0.932160
BEGIN SANITIZED_PHASE31_CLASSICAL_OOF
{
  "phase": "phase31_nested_classical_gate",
  "status": "component_gate_passed",
  "baseline": {
    "log_loss": 0.307615,
    "

In [16]:
# Phase32 Cell 104A
# Discover the saved Phase30 runtime and offline DINOv3 assets after restart.
# Does not read train, smoke, or test images.

from pathlib import Path
import ast
import json
import zipfile
import time

PHASE32_WORKING = Path("/kaggle/working")
phase32_started = time.perf_counter()


def phase32_safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    destination_resolved = destination.resolve()

    with zipfile.ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()

            assert (
                target == destination_resolved
                or destination_resolved in target.parents
            ), {
                "message": "Unsafe archive member.",
                "member": member.filename,
            }

        archive.extractall(destination)


def phase32_find_runtime_candidates():
    candidates = []

    search_roots = [
        path
        for path in PHASE32_WORKING.iterdir()
        if path.is_dir()
        and "phase30" in path.name.lower()
    ]

    for root in search_roots:
        for main_path in root.rglob("main.py"):
            candidate_root = main_path.parent

            checkpoints = sorted(
                candidate_root.rglob(
                    "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
                )
            )

            dinov3_initializers = sorted(
                candidate_root.rglob("dinov3/__init__.py")
            )

            phase12_weights = sorted(
                path
                for path in candidate_root.rglob("*.pt")
                if "dinov3" not in path.name.lower()
            )

            if (
                checkpoints
                and dinov3_initializers
                and len(phase12_weights) >= 21
            ):
                lower_path = str(candidate_root).lower()

                score = 0
                score += 100 if "full_ensemble" in lower_path else 0
                score += 50 if candidate_root.name == "phase30_submission" else 0
                score -= 100 if "rehearsal" in lower_path else 0
                score -= len(candidate_root.parts)

                candidates.append({
                    "root": candidate_root,
                    "main": main_path,
                    "checkpoint": checkpoints[0],
                    "dinov3_init": dinov3_initializers[0],
                    "phase12_weight_count": len(phase12_weights),
                    "score": score,
                })

    return candidates


phase32_runtime_candidates = phase32_find_runtime_candidates()

if not phase32_runtime_candidates:
    zip_candidates = sorted(
        PHASE32_WORKING.glob("*phase30*.zip"),
        key=lambda path: (
            path.stat().st_mtime,
            path.stat().st_size,
        ),
        reverse=True,
    )

    assert zip_candidates, (
        "No Phase30 directory or archive was found in /kaggle/working."
    )

    PHASE32_EXTRACTED_ROOT = Path(
        "/kaggle/working/phase32_recovered_phase30"
    )

    if not PHASE32_EXTRACTED_ROOT.is_dir():
        phase32_safe_extract(
            zip_candidates[0],
            PHASE32_EXTRACTED_ROOT,
        )

    phase32_runtime_candidates = phase32_find_runtime_candidates()

    # The extracted directory may not have "phase30" in its immediate name.
    if not phase32_runtime_candidates:
        for main_path in PHASE32_EXTRACTED_ROOT.rglob("main.py"):
            candidate_root = main_path.parent

            checkpoints = sorted(
                candidate_root.rglob(
                    "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
                )
            )
            initializers = sorted(
                candidate_root.rglob("dinov3/__init__.py")
            )
            phase12_weights = sorted(
                path
                for path in candidate_root.rglob("*.pt")
                if "dinov3" not in path.name.lower()
            )

            if (
                checkpoints
                and initializers
                and len(phase12_weights) >= 21
            ):
                phase32_runtime_candidates.append({
                    "root": candidate_root,
                    "main": main_path,
                    "checkpoint": checkpoints[0],
                    "dinov3_init": initializers[0],
                    "phase12_weight_count": len(phase12_weights),
                    "score": 0,
                })

assert phase32_runtime_candidates, (
    "No complete Phase30 runtime candidate could be recovered."
)

phase32_runtime_candidates.sort(
    key=lambda record: record["score"],
    reverse=True,
)

phase32_selected = phase32_runtime_candidates[0]

PHASE32_PACKAGE_ROOT = phase32_selected["root"]
PHASE32_RUNTIME_MAIN = phase32_selected["main"]
PHASE32_DINOV3_CHECKPOINT = phase32_selected["checkpoint"]

# Parent directory that must be placed on sys.path to import `dinov3`.
PHASE32_DINOV3_PACKAGE_PARENT = (
    phase32_selected["dinov3_init"].parent.parent
)

runtime_source = PHASE32_RUNTIME_MAIN.read_text(
    encoding="utf-8"
)
runtime_tree = ast.parse(runtime_source)

runtime_functions = [
    node.name
    for node in runtime_tree.body
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef),
    )
]

relevant_functions = sorted(
    name
    for name in runtime_functions
    if any(
        term in name.lower()
        for term in [
            "view",
            "bilateral",
            "dinov3",
            "feature",
            "clinical",
            "backbone",
        ]
    )
)

report = {
    "phase": "phase32_p3ca_asset_discovery",
    "status": "accepted",
    "runtime_candidate_count": len(
        phase32_runtime_candidates
    ),
    "selected_package_name": PHASE32_PACKAGE_ROOT.name,
    "runtime_main_present": PHASE32_RUNTIME_MAIN.is_file(),
    "dinov3_checkpoint_count": 1,
    "dinov3_source_present": (
        phase32_selected["dinov3_init"].is_file()
    ),
    "phase12_weight_count": int(
        phase32_selected["phase12_weight_count"]
    ),
    "relevant_runtime_functions": relevant_functions,
    "elapsed_seconds": round(
        time.perf_counter() - phase32_started,
        2,
    ),
    "challenge_voxel_data_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "model_hashes_displayed": False,
}

print("BEGIN SANITIZED_PHASE32_ASSET_DISCOVERY")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE32_ASSET_DISCOVERY")

BEGIN SANITIZED_PHASE32_ASSET_DISCOVERY
{
  "phase": "phase32_p3ca_asset_discovery",
  "status": "accepted",
  "runtime_candidate_count": 3,
  "selected_package_name": "phase30_submission",
  "runtime_main_present": true,
  "dinov3_checkpoint_count": 1,
  "dinov3_source_present": true,
  "phase12_weight_count": 21,
  "relevant_runtime_functions": [
    "header_features",
    "load_dinov3",
    "phase30_bilateral_comparison",
    "phase30_make_bilateral_views"
  ],
  "elapsed_seconds": 0.03,
  "challenge_voxel_data_read": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "model_hashes_displayed": false
}
END SANITIZED_PHASE32_ASSET_DISCOVERY


In [17]:
# Phase32 Cell 104B
# Load the offline DINOv3 ViT-S/16 backbone and validate dense patch tokens.
# Synthetic images only.

import importlib
import sys
from collections.abc import Mapping

import numpy as np
import torch

assert PHASE32_DINOV3_CHECKPOINT.is_file()
assert PHASE32_DINOV3_PACKAGE_PARENT.is_dir()

package_parent_text = str(
    PHASE32_DINOV3_PACKAGE_PARENT
)

if package_parent_text not in sys.path:
    sys.path.insert(0, package_parent_text)

dinov3_backbones = importlib.import_module(
    "dinov3.hub.backbones"
)

assert hasattr(
    dinov3_backbones,
    "dinov3_vits16",
)

phase32_factory = dinov3_backbones.dinov3_vits16

PHASE32_DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

PHASE32_DINOV3_BACKBONE = phase32_factory(
    pretrained=False
)

raw_checkpoint = torch.load(
    PHASE32_DINOV3_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)


def phase32_tensor_dictionary_candidates(value):
    candidates = []

    if isinstance(value, Mapping):
        if value and all(
            torch.is_tensor(item)
            for item in value.values()
        ):
            candidates.append(dict(value))

        for key in [
            "model",
            "state_dict",
            "teacher",
            "backbone",
            "student",
        ]:
            nested = value.get(key)

            if (
                isinstance(nested, Mapping)
                and nested
                and all(
                    torch.is_tensor(item)
                    for item in nested.values()
                )
            ):
                candidates.append(dict(nested))

    return candidates


def phase32_strip_prefix(state, prefix):
    if not prefix:
        return state

    if not all(
        key.startswith(prefix)
        for key in state
    ):
        return None

    return {
        key[len(prefix):]: value
        for key, value in state.items()
    }


target_keys = set(
    PHASE32_DINOV3_BACKBONE.state_dict().keys()
)

resolved_state = None
resolved_prefix = None

for state in phase32_tensor_dictionary_candidates(
    raw_checkpoint
):
    for prefix in [
        "",
        "module.",
        "backbone.",
        "teacher.",
        "student.",
        "model.",
        "module.backbone.",
    ]:
        candidate = phase32_strip_prefix(
            state,
            prefix,
        )

        if candidate is None:
            continue

        if set(candidate.keys()) == target_keys:
            resolved_state = candidate
            resolved_prefix = prefix
            break

    if resolved_state is not None:
        break

assert resolved_state is not None, {
    "message": "Could not match the DINOv3 checkpoint exactly.",
    "checkpoint_candidate_count": len(
        phase32_tensor_dictionary_candidates(
            raw_checkpoint
        )
    ),
    "target_state_tensor_count": len(target_keys),
}

load_result = PHASE32_DINOV3_BACKBONE.load_state_dict(
    resolved_state,
    strict=True,
)

assert not load_result.missing_keys
assert not load_result.unexpected_keys

del raw_checkpoint
del resolved_state

PHASE32_DINOV3_BACKBONE.eval()
PHASE32_DINOV3_BACKBONE.to(PHASE32_DEVICE)

for parameter in PHASE32_DINOV3_BACKBONE.parameters():
    parameter.requires_grad_(False)


def phase32_dense_tokens(images):
    assert images.ndim == 4
    assert tuple(images.shape[1:]) == (
        3,
        224,
        224,
    )

    output = (
        PHASE32_DINOV3_BACKBONE.forward_features(
            images
        )
    )

    assert isinstance(output, Mapping), type(output)

    patch_tokens = None
    selected_key = None

    for key in [
        "x_norm_patchtokens",
        "x_norm_patch_tokens",
        "patch_tokens",
    ]:
        candidate = output.get(key)

        if (
            torch.is_tensor(candidate)
            and candidate.ndim == 3
        ):
            patch_tokens = candidate
            selected_key = key
            break

    assert patch_tokens is not None, {
        "message": "Dense patch tokens were not found.",
        "available_keys": sorted(output.keys()),
    }

    assert patch_tokens.shape[1] == 196
    assert patch_tokens.shape[2] == 384

    dense_map = patch_tokens.reshape(
        patch_tokens.shape[0],
        14,
        14,
        384,
    )

    return dense_map, selected_key


if PHASE32_DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(
        PHASE32_DEVICE
    )

generator = torch.Generator(device="cpu")
generator.manual_seed(320601)

synthetic_images = torch.rand(
    2,
    3,
    224,
    224,
    generator=generator,
).to(PHASE32_DEVICE)

with torch.inference_mode():
    synthetic_dense, phase32_patch_key = (
        phase32_dense_tokens(
            synthetic_images
        )
    )

assert synthetic_dense.shape == (
    2,
    14,
    14,
    384,
)
assert torch.isfinite(synthetic_dense).all()

parameter_count = sum(
    parameter.numel()
    for parameter in PHASE32_DINOV3_BACKBONE.parameters()
)

assert parameter_count == 21601152

report = {
    "phase": "phase32_dinov3_dense_token_contract",
    "status": "accepted",
    "architecture": "dinov3_vits16_LVD1689M",
    "parameter_count": parameter_count,
    "checkpoint_strict_load": True,
    "checkpoint_prefix_removed": resolved_prefix,
    "dense_token_key": phase32_patch_key,
    "dense_feature_shape": list(
        synthetic_dense.shape
    ),
    "embedding_dimension": int(
        synthetic_dense.shape[-1]
    ),
    "all_outputs_finite": True,
    "backbone_frozen": all(
        not parameter.requires_grad
        for parameter in (
            PHASE32_DINOV3_BACKBONE.parameters()
        )
    ),
    "device": str(PHASE32_DEVICE),
    "peak_vram_mb": round(
        (
            torch.cuda.max_memory_allocated(
                PHASE32_DEVICE
            )
            / (1024 ** 2)
        )
        if PHASE32_DEVICE.type == "cuda"
        else 0.0,
        2,
    ),
    "external_weights": True,
    "license": "DINOv3 License",
    "required_attribution": "Built with DINOv3",
    "synthetic_input_only": True,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE32_DENSE_TOKEN_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE32_DENSE_TOKEN_CONTRACT")

BEGIN SANITIZED_PHASE32_DENSE_TOKEN_CONTRACT
{
  "phase": "phase32_dinov3_dense_token_contract",
  "status": "accepted",
  "architecture": "dinov3_vits16_LVD1689M",
  "parameter_count": 21601152,
  "checkpoint_strict_load": true,
  "checkpoint_prefix_removed": "",
  "dense_token_key": "x_norm_patchtokens",
  "dense_feature_shape": [
    2,
    14,
    14,
    384
  ],
  "embedding_dimension": 384,
  "all_outputs_finite": true,
  "backbone_frozen": true,
  "device": "cuda:0",
  "peak_vram_mb": 100.49,
  "external_weights": true,
  "license": "DINOv3 License",
  "required_attribution": "Built with DINOv3",
  "synthetic_input_only": true,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE32_DENSE_TOKEN_CONTRACT


In [18]:
# Phase32 Cell 104C
# Position-prompted PCA implementation and synthetic numerical contract.

from sklearn.decomposition import PCA


def phase32_prompt_mask(
    height=14,
    width=14,
    row_start=3,
    row_stop=11,
    column_start=3,
    column_stop=11,
):
    mask = np.zeros(
        (height, width),
        dtype=bool,
    )

    mask[
        row_start:row_stop,
        column_start:column_stop,
    ] = True

    assert mask.any()
    assert not mask.all()

    return mask


def phase32_fit_p3ca(
    spatial_features,
    prompt_mask,
    component_count=16,
    seed=320601,
):
    """
    spatial_features:
        [cases, views, height, width, channels]
        or [cases, height, width, channels]

    Fits normalization and PCA using only prompted spatial
    positions. Labels are never used.
    """
    array = np.asarray(
        spatial_features,
        dtype=np.float32,
    )

    if array.ndim == 4:
        array = array[:, None, ...]

    assert array.ndim == 5

    case_count, view_count, height, width, channels = (
        array.shape
    )

    mask = np.asarray(
        prompt_mask,
        dtype=bool,
    )

    assert mask.shape == (height, width)

    prompted = array[
        :,
        :,
        mask,
        :,
    ].reshape(-1, channels)

    channel_mean = prompted.mean(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    channel_standard_deviation = prompted.std(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    channel_standard_deviation = np.where(
        channel_standard_deviation >= 1e-6,
        channel_standard_deviation,
        1.0,
    ).astype(np.float32)

    normalized_prompt = (
        prompted - channel_mean
    ) / channel_standard_deviation

    maximum_components = min(
        normalized_prompt.shape[0] - 1,
        normalized_prompt.shape[1],
    )

    resolved_components = min(
        int(component_count),
        int(maximum_components),
    )

    assert resolved_components >= 3

    projector = PCA(
        n_components=resolved_components,
        svd_solver="randomized",
        iterated_power=4,
        random_state=seed,
        whiten=False,
    )

    projector.fit(normalized_prompt)

    state = {
        "channel_mean": channel_mean,
        "channel_standard_deviation": (
            channel_standard_deviation
        ),
        "components": np.asarray(
            projector.components_,
            dtype=np.float32,
        ),
        "explained_variance_ratio": np.asarray(
            projector.explained_variance_ratio_,
            dtype=np.float32,
        ),
        "prompt_mask": mask,
        "input_channels": channels,
        "component_count": resolved_components,
    }

    return state


def phase32_apply_p3ca(
    spatial_features,
    state,
):
    array = np.asarray(
        spatial_features,
        dtype=np.float32,
    )

    original_rank = array.ndim

    if original_rank == 4:
        array = array[:, None, ...]

    assert array.ndim == 5
    assert array.shape[-1] == (
        state["input_channels"]
    )

    normalized = (
        array - state["channel_mean"]
    ) / state["channel_standard_deviation"]

    projected = np.einsum(
        "nvhwc,kc->nvhwk",
        normalized,
        state["components"],
        optimize=True,
    ).astype(np.float32)

    if original_rank == 4:
        projected = projected[:, 0]

    return projected


synthetic_dense_numpy = (
    synthetic_dense
    .detach()
    .cpu()
    .float()
    .numpy()
)

phase32_central_prompt = phase32_prompt_mask()

phase32_synthetic_p3ca = phase32_fit_p3ca(
    synthetic_dense_numpy,
    phase32_central_prompt,
    component_count=16,
)

phase32_synthetic_projection = (
    phase32_apply_p3ca(
        synthetic_dense_numpy,
        phase32_synthetic_p3ca,
    )
)

assert phase32_synthetic_projection.shape == (
    2,
    14,
    14,
    16,
)
assert np.isfinite(
    phase32_synthetic_projection
).all()

orthogonality = (
    phase32_synthetic_p3ca["components"]
    @ phase32_synthetic_p3ca["components"].T
)

maximum_orthogonality_error = float(
    np.max(
        np.abs(
            orthogonality
            - np.eye(
                orthogonality.shape[0],
                dtype=np.float32,
            )
        )
    )
)

assert maximum_orthogonality_error < 1e-4

report = {
    "phase": "phase32_position_prompted_pca_contract",
    "status": "accepted",
    "method": "position_prompted_PCA",
    "paper_arxiv": "2608.10131",
    "input_shape": list(
        synthetic_dense_numpy.shape
    ),
    "prompt_shape": list(
        phase32_central_prompt.shape
    ),
    "prompt_token_count_per_view": int(
        phase32_central_prompt.sum()
    ),
    "component_count": int(
        phase32_synthetic_p3ca[
            "component_count"
        ]
    ),
    "projected_shape": list(
        phase32_synthetic_projection.shape
    ),
    "explained_variance_fraction": round(
        float(
            phase32_synthetic_p3ca[
                "explained_variance_ratio"
            ].sum()
        ),
        6,
    ),
    "maximum_component_orthogonality_error": (
        maximum_orthogonality_error
    ),
    "all_projected_values_finite": True,
    "planned_real_prompt": (
        "fixed central striatal region within "
        "reflection-invariant bilateral comparison views"
    ),
    "planned_fit_scope": (
        "fold_local_fit_partition_only"
    ),
    "labels_used": False,
    "synthetic_input_only": True,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE32_P3CA_CONTRACT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE32_P3CA_CONTRACT")

BEGIN SANITIZED_PHASE32_P3CA_CONTRACT
{
  "phase": "phase32_position_prompted_pca_contract",
  "status": "accepted",
  "method": "position_prompted_PCA",
  "paper_arxiv": "2608.10131",
  "input_shape": [
    2,
    14,
    14,
    384
  ],
  "prompt_shape": [
    14,
    14
  ],
  "prompt_token_count_per_view": 64,
  "component_count": 16,
  "projected_shape": [
    2,
    14,
    14,
    16
  ],
  "explained_variance_fraction": 0.872126,
  "maximum_component_orthogonality_error": 1.3113021850585938e-06,
  "all_projected_values_finite": true,
  "planned_real_prompt": "fixed central striatal region within reflection-invariant bilateral comparison views",
  "planned_fit_scope": "fold_local_fit_partition_only",
  "labels_used": false,
  "synthetic_input_only": true,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE32_P3CA_CONTRACT


In [19]:
# Phase32 Cell 105A
# Import the accepted Phase30 runtime and validate its exact six-view generator.
# Synthetic inputs only.

import importlib.util
import json
import sys
import time

import numpy as np
import torch

phase32_view_started = time.perf_counter()

package_root_text = str(PHASE32_PACKAGE_ROOT)

if package_root_text not in sys.path:
    sys.path.insert(0, package_root_text)

phase32_runtime_spec = (
    importlib.util.spec_from_file_location(
        "phase32_phase30_runtime",
        PHASE32_RUNTIME_MAIN,
    )
)

assert phase32_runtime_spec is not None
assert phase32_runtime_spec.loader is not None

PHASE32_RUNTIME_MODULE = (
    importlib.util.module_from_spec(
        phase32_runtime_spec
    )
)

phase32_runtime_spec.loader.exec_module(
    PHASE32_RUNTIME_MODULE
)

assert hasattr(
    PHASE32_RUNTIME_MODULE,
    "phase30_make_bilateral_views",
)

PHASE32_MAKE_BILATERAL_VIEWS = getattr(
    PHASE32_RUNTIME_MODULE,
    "phase30_make_bilateral_views",
)

generator = torch.Generator(device="cpu")
generator.manual_seed(320602)

synthetic_volumes = torch.rand(
    2,
    80,
    80,
    80,
    generator=generator,
    dtype=torch.float32,
).to(PHASE32_DEVICE)

with torch.inference_mode():
    synthetic_views = (
        PHASE32_MAKE_BILATERAL_VIEWS(
            synthetic_volumes
        )
    )

    reflected_views = (
        PHASE32_MAKE_BILATERAL_VIEWS(
            torch.flip(
                synthetic_volumes,
                dims=[1],
            )
        )
    )

assert synthetic_views.shape == (
    2,
    6,
    3,
    224,
    224,
)
assert torch.isfinite(synthetic_views).all()

reflection_error = float(
    torch.max(
        torch.abs(
            synthetic_views
            - reflected_views
        )
    ).item()
)

assert reflection_error <= 1e-6, (
    reflection_error
)

with torch.inference_mode():
    flattened_views = (
        synthetic_views.reshape(
            -1,
            3,
            224,
            224,
        )
    )

    synthetic_dense_views, dense_key = (
        phase32_dense_tokens(
            flattened_views
        )
    )

    synthetic_dense_views = (
        synthetic_dense_views.reshape(
            2,
            6,
            14,
            14,
            384,
        )
    )

assert synthetic_dense_views.shape == (
    2,
    6,
    14,
    14,
    384,
)
assert torch.isfinite(
    synthetic_dense_views
).all()

report = {
    "phase": "phase32_exact_bilateral_view_contract",
    "status": "accepted",
    "runtime_function": (
        "phase30_make_bilateral_views"
    ),
    "input_shape": list(
        synthetic_volumes.shape
    ),
    "view_shape": list(
        synthetic_views.shape
    ),
    "dense_feature_shape": list(
        synthetic_dense_views.shape
    ),
    "dense_token_key": dense_key,
    "view_count": 6,
    "patch_grid": [14, 14],
    "embedding_dimension": 384,
    "reflection_max_abs_error": (
        reflection_error
    ),
    "reflection_invariant": (
        reflection_error <= 1e-6
    ),
    "dinov3_backbone_frozen": True,
    "synthetic_input_only": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase32_view_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE32_BILATERAL_VIEW_CONTRACT"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_BILATERAL_VIEW_CONTRACT"
)

del synthetic_volumes
del synthetic_views
del reflected_views
del flattened_views
del synthetic_dense_views

if PHASE32_DEVICE.type == "cuda":
    torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE32_BILATERAL_VIEW_CONTRACT
{
  "phase": "phase32_exact_bilateral_view_contract",
  "status": "accepted",
  "runtime_function": "phase30_make_bilateral_views",
  "input_shape": [
    2,
    80,
    80,
    80
  ],
  "view_shape": [
    2,
    6,
    3,
    224,
    224
  ],
  "dense_feature_shape": [
    2,
    6,
    14,
    14,
    384
  ],
  "dense_token_key": "x_norm_patchtokens",
  "view_count": 6,
  "patch_grid": [
    14,
    14
  ],
  "embedding_dimension": 384,
  "reflection_max_abs_error": 0.0,
  "reflection_invariant": true,
  "dinov3_backbone_frozen": true,
  "synthetic_input_only": true,
  "training_voxel_cache_read": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "elapsed_seconds": 0.72
}
END SANITIZED_PHASE32_BILATERAL_VIEW_CONTRACT


In [20]:
# Phase32 Cell 105B
# Extract DINOv3 spatial maps for the exact six bilateral views.
#
# Output:
# [1362, 6, 14, 14, 384] float16, approximately 1.23 GB.
#
# Private notebook cache only. Never include it in submission.zip.

from pathlib import Path
import json
import os
import time

import numpy as np
import torch

PHASE32_DENSE_PATH = Path(
    "/kaggle/working/"
    "phase32_bilateral_dense_float16.npy"
)

PHASE32_DENSE_PARTIAL_PATH = Path(
    "/kaggle/working/"
    "phase32_bilateral_dense_float16.partial.npy"
)

PHASE32_DENSE_METADATA_PATH = Path(
    "/kaggle/working/"
    "phase32_bilateral_dense_metadata.json"
)

PHASE32_DENSE_SHAPE = (
    EXPECTED_CASES,
    6,
    14,
    14,
    384,
)

PHASE32_EXTRACTION_BATCH_SIZE = 4


def phase32_validate_dense_cache():
    if not (
        PHASE32_DENSE_PATH.is_file()
        and PHASE32_DENSE_METADATA_PATH.is_file()
    ):
        return None

    try:
        metadata = json.loads(
            PHASE32_DENSE_METADATA_PATH.read_text(
                encoding="utf-8"
            )
        )

        cache = np.load(
            PHASE32_DENSE_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        if cache.shape != PHASE32_DENSE_SHAPE:
            return None

        if cache.dtype != np.float16:
            return None

        if metadata.get("status") != "complete":
            return None

        if tuple(metadata["shape"]) != (
            PHASE32_DENSE_SHAPE
        ):
            return None

        check_indices = np.linspace(
            0,
            EXPECTED_CASES - 1,
            12,
            dtype=np.int64,
        )

        sample = np.asarray(
            cache[check_indices],
            dtype=np.float32,
        )

        if not np.isfinite(sample).all():
            return None

        if float(np.std(sample)) <= 1e-6:
            return None

        return cache, metadata

    except Exception:
        return None


phase32_extraction_started = time.perf_counter()
cached_result = phase32_validate_dense_cache()
phase32_dense_cache_reused = (
    cached_result is not None
)

if cached_result is not None:
    (
        PHASE32_BILATERAL_DENSE_PRIVATE,
        phase32_dense_metadata,
    ) = cached_result

else:
    if PHASE32_DENSE_PARTIAL_PATH.exists():
        PHASE32_DENSE_PARTIAL_PATH.unlink()

    dense_writer = np.lib.format.open_memmap(
        PHASE32_DENSE_PARTIAL_PATH,
        mode="w+",
        dtype=np.float16,
        shape=PHASE32_DENSE_SHAPE,
    )

    PHASE32_DINOV3_BACKBONE.eval()
    PHASE32_DINOV3_BACKBONE.to(
        PHASE32_DEVICE
    )

    if PHASE32_DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(
            PHASE32_DEVICE
        )

    next_progress = 100

    for start in range(
        0,
        EXPECTED_CASES,
        PHASE32_EXTRACTION_BATCH_SIZE,
    ):
        stop = min(
            start + PHASE32_EXTRACTION_BATCH_SIZE,
            EXPECTED_CASES,
        )

        volume_numpy = np.asarray(
            PHASE31_HIGHRES_CACHE[start:stop],
            dtype=np.float32,
        ).copy()

        volumes = torch.from_numpy(
            volume_numpy
        ).to(
            PHASE32_DEVICE,
            non_blocking=True,
        )

        with torch.inference_mode():
            bilateral_views = (
                PHASE32_MAKE_BILATERAL_VIEWS(
                    volumes
                )
            )

            batch_size = volumes.shape[0]

            flattened_views = (
                bilateral_views.reshape(
                    batch_size * 6,
                    3,
                    224,
                    224,
                )
            )

            dense_tokens, _ = (
                phase32_dense_tokens(
                    flattened_views
                )
            )

            dense_tokens = (
                dense_tokens.reshape(
                    batch_size,
                    6,
                    14,
                    14,
                    384,
                )
            )

        dense_writer[start:stop] = (
            dense_tokens
            .detach()
            .cpu()
            .to(torch.float16)
            .numpy()
        )

        del volume_numpy
        del volumes
        del bilateral_views
        del flattened_views
        del dense_tokens

        completed = stop

        if (
            completed >= next_progress
            or completed == EXPECTED_CASES
        ):
            dense_writer.flush()

            print(
                "Phase32 dense P3CA extraction: "
                f"{completed}/{EXPECTED_CASES}"
            )

            while next_progress <= completed:
                next_progress += 100

    dense_writer.flush()
    del dense_writer

    os.replace(
        PHASE32_DENSE_PARTIAL_PATH,
        PHASE32_DENSE_PATH,
    )

    PHASE32_BILATERAL_DENSE_PRIVATE = (
        np.load(
            PHASE32_DENSE_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )
    )

    sample_indices = np.linspace(
        0,
        EXPECTED_CASES - 1,
        32,
        dtype=np.int64,
    )

    dense_sample = np.asarray(
        PHASE32_BILATERAL_DENSE_PRIVATE[
            sample_indices
        ],
        dtype=np.float32,
    )

    class_free_token_norms = np.linalg.norm(
        dense_sample,
        axis=-1,
    )

    phase32_dense_metadata = {
        "schema_version": 1,
        "phase": "phase32_dense_p3ca_cache",
        "status": "complete",
        "shape": list(PHASE32_DENSE_SHAPE),
        "dtype": "float16",
        "view_count": 6,
        "patch_grid": [14, 14],
        "embedding_dimension": 384,
        "mean_sampled_token_norm": float(
            class_free_token_norms.mean()
        ),
        "minimum_sampled_token_norm": float(
            class_free_token_norms.min()
        ),
    }

    PHASE32_DENSE_METADATA_PATH.write_text(
        json.dumps(
            phase32_dense_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

assert (
    PHASE32_BILATERAL_DENSE_PRIVATE.shape
    == PHASE32_DENSE_SHAPE
)
assert (
    PHASE32_BILATERAL_DENSE_PRIVATE.dtype
    == np.float16
)

phase32_dense_elapsed = (
    time.perf_counter()
    - phase32_extraction_started
)

report = {
    "phase": "phase32_bilateral_dense_feature_cache",
    "status": "complete",
    "cache_reused": (
        phase32_dense_cache_reused
    ),
    "case_count": EXPECTED_CASES,
    "shape": list(
        PHASE32_BILATERAL_DENSE_PRIVATE.shape
    ),
    "dtype": str(
        PHASE32_BILATERAL_DENSE_PRIVATE.dtype
    ),
    "view_count": 6,
    "spatial_token_count_per_view": 196,
    "embedding_dimension": 384,
    "total_spatial_token_count": int(
        EXPECTED_CASES * 6 * 196
    ),
    "file_size_gb_decimal": round(
        PHASE32_DENSE_PATH.stat().st_size
        / 1e9,
        4,
    ),
    "file_size_gib": round(
        PHASE32_DENSE_PATH.stat().st_size
        / (1024 ** 3),
        4,
    ),
    "mean_sampled_token_norm": round(
        float(
            phase32_dense_metadata[
                "mean_sampled_token_norm"
            ]
        ),
        6,
    ),
    "minimum_sampled_token_norm": round(
        float(
            phase32_dense_metadata[
                "minimum_sampled_token_norm"
            ]
        ),
        6,
    ),
    "extraction_batch_size_cases": (
        PHASE32_EXTRACTION_BATCH_SIZE
    ),
    "peak_vram_mb": round(
        (
            torch.cuda.max_memory_allocated(
                PHASE32_DEVICE
            )
            / (1024 ** 2)
        )
        if PHASE32_DEVICE.type == "cuda"
        else 0.0,
        2,
    ),
    "elapsed_seconds": round(
        phase32_dense_elapsed,
        2,
    ),
    "dinov3_backbone_frozen": True,
    "float16_quantization": True,
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "training_voxel_cache_read": (
        not phase32_dense_cache_reused
    ),
    "labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_features_exported": False,
}

print(
    "BEGIN SANITIZED_PHASE32_DENSE_FEATURE_CACHE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_DENSE_FEATURE_CACHE"
)

Phase32 dense P3CA extraction: 100/1362
Phase32 dense P3CA extraction: 200/1362
Phase32 dense P3CA extraction: 300/1362
Phase32 dense P3CA extraction: 400/1362
Phase32 dense P3CA extraction: 500/1362
Phase32 dense P3CA extraction: 600/1362
Phase32 dense P3CA extraction: 700/1362
Phase32 dense P3CA extraction: 800/1362
Phase32 dense P3CA extraction: 900/1362
Phase32 dense P3CA extraction: 1000/1362
Phase32 dense P3CA extraction: 1100/1362
Phase32 dense P3CA extraction: 1200/1362
Phase32 dense P3CA extraction: 1300/1362
Phase32 dense P3CA extraction: 1362/1362
BEGIN SANITIZED_PHASE32_DENSE_FEATURE_CACHE
{
  "phase": "phase32_bilateral_dense_feature_cache",
  "status": "complete",
  "cache_reused": false,
  "case_count": 1362,
  "shape": [
    1362,
    6,
    14,
    14,
    384
  ],
  "dtype": "float16",
  "view_count": 6,
  "spatial_token_count_per_view": 196,
  "embedding_dimension": 384,
  "total_spatial_token_count": 1601712,
  "file_size_gb_decimal": 1.2301,
  "file_size_gib": 1.14

In [23]:
# Phase32 Cell 105C
# Verify view diversity, token diversity, and quantization quality.

audit_indices = np.linspace(
    0,
    EXPECTED_CASES - 1,
    48,
    dtype=np.int64,
)

audit_dense = np.asarray(
    PHASE32_BILATERAL_DENSE_PRIVATE[
        audit_indices
    ],
    dtype=np.float32,
)

assert audit_dense.shape == (
    48,
    6,
    14,
    14,
    384,
)
assert np.isfinite(audit_dense).all()

# Standard deviation at each 14x14 coordinate,
# reducing cases, views, and embedding channels.
spatial_coordinate_std = np.std(
    audit_dense,
    axis=(0, 1, 4),
)

assert spatial_coordinate_std.shape == (
    14,
    14,
)

# Variation between cases for every
# view/spatial/channel coordinate.
case_variation = np.std(
    audit_dense,
    axis=0,
)

assert case_variation.shape == (
    6,
    14,
    14,
    384,
)

# Mean semantic embedding for each clinical view.
view_means = audit_dense.mean(
    axis=(0, 2, 3),
)

assert view_means.shape == (
    6,
    384,
)

pairwise_view_differences = []

for first_view in range(6):
    for second_view in range(
        first_view + 1,
        6,
    ):
        pairwise_view_differences.append(
            float(
                np.mean(
                    np.abs(
                        view_means[first_view]
                        - view_means[second_view]
                    )
                )
            )
        )

PHASE32_REAL_PROMPT_MASK = (
    phase32_prompt_mask(
        height=14,
        width=14,
        row_start=3,
        row_stop=11,
        column_start=3,
        column_stop=11,
    )
)

assert PHASE32_REAL_PROMPT_MASK.shape == (
    14,
    14,
)
assert int(
    PHASE32_REAL_PROMPT_MASK.sum()
) == 64

# Boolean 14x14 mask jointly indexes the two spatial axes.
prompt_values = audit_dense[
    :,
    :,
    PHASE32_REAL_PROMPT_MASK,
    :,
]

outside_values = audit_dense[
    :,
    :,
    ~PHASE32_REAL_PROMPT_MASK,
    :,
]

assert prompt_values.shape == (
    48,
    6,
    64,
    384,
)
assert outside_values.shape == (
    48,
    6,
    132,
    384,
)

# Detect coordinates that contain effectively no variation
# across the sampled cases.
flattened_case_features = (
    audit_dense.reshape(
        audit_dense.shape[0],
        -1,
    )
)

coordinate_standard_deviation = np.std(
    flattened_case_features,
    axis=0,
)

collapsed_coordinate_fraction = float(
    np.mean(
        coordinate_standard_deviation
        < 1e-6
    )
)

report = {
    "phase": (
        "phase32_dense_feature_quality_audit"
    ),
    "status": "accepted",
    "sampled_case_count": int(
        len(audit_indices)
    ),
    "view_count": 6,
    "patch_grid": [14, 14],
    "embedding_dimension": 384,
    "prompt_token_count_per_view": int(
        PHASE32_REAL_PROMPT_MASK.sum()
    ),
    "outside_token_count_per_view": int(
        (~PHASE32_REAL_PROMPT_MASK).sum()
    ),
    "mean_prompt_feature_standard_deviation": round(
        float(np.std(prompt_values)),
        6,
    ),
    "mean_outside_feature_standard_deviation": round(
        float(np.std(outside_values)),
        6,
    ),
    "median_spatial_coordinate_standard_deviation": round(
        float(
            np.median(
                spatial_coordinate_std
            )
        ),
        6,
    ),
    "median_case_feature_standard_deviation": round(
        float(
            np.median(
                case_variation
            )
        ),
        6,
    ),
    "mean_pairwise_view_difference": round(
        float(
            np.mean(
                pairwise_view_differences
            )
        ),
        6,
    ),
    "minimum_pairwise_view_difference": round(
        float(
            np.min(
                pairwise_view_differences
            )
        ),
        6,
    ),
    "maximum_pairwise_view_difference": round(
        float(
            np.max(
                pairwise_view_differences
            )
        ),
        6,
    ),
    "collapsed_coordinate_fraction": round(
        collapsed_coordinate_fraction,
        8,
    ),
    "all_features_finite": True,
    "fixed_prompt_label_free": True,
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

assert (
    report[
        "mean_pairwise_view_difference"
    ] > 1e-4
), report

assert (
    report[
        "median_case_feature_standard_deviation"
    ] > 1e-4
), report

assert (
    collapsed_coordinate_fraction
    < 0.01
), report

print(
    "BEGIN SANITIZED_PHASE32_DENSE_FEATURE_AUDIT"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_DENSE_FEATURE_AUDIT"
)

BEGIN SANITIZED_PHASE32_DENSE_FEATURE_AUDIT
{
  "phase": "phase32_dense_feature_quality_audit",
  "status": "accepted",
  "sampled_case_count": 48,
  "view_count": 6,
  "patch_grid": [
    14,
    14
  ],
  "embedding_dimension": 384,
  "prompt_token_count_per_view": 64,
  "outside_token_count_per_view": 132,
  "mean_prompt_feature_standard_deviation": 0.370352,
  "mean_outside_feature_standard_deviation": 0.375488,
  "median_spatial_coordinate_standard_deviation": 0.372747,
  "median_case_feature_standard_deviation": 0.154475,
  "mean_pairwise_view_difference": 0.047218,
  "minimum_pairwise_view_difference": 0.017014,
  "maximum_pairwise_view_difference": 0.076819,
  "collapsed_coordinate_fraction": 0.0,
  "all_features_finite": true,
  "fixed_prompt_label_free": true,
  "persistent_notebook_cache_only": true,
  "include_in_submission": false,
  "labels_used": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE32_DENSE_FEATURE_AUDIT


In [24]:
# Phase32 Cell 106A
# Fit view-specific P3CA bases using streaming sufficient statistics.
#
# Two stages per fold:
#   fit         -> nested monitor selection
#   outer_train -> final refit and outer validation
#
# Labels are never used.

from pathlib import Path
import json
import time

import numpy as np
import torch

PHASE32_P3CA_STATE_PATH = Path(
    "/kaggle/working/phase32_p3ca_states.npz"
)

PHASE32_P3CA_STATE_METADATA_PATH = Path(
    "/kaggle/working/phase32_p3ca_states.json"
)

PHASE32_P3CA_COMPONENT_COUNT = 16
PHASE32_P3CA_STAGE_NAMES = [
    "fit",
    "outer_train",
]


def phase32_validate_partitions(value):
    if not isinstance(value, (list, tuple)):
        return False

    if len(value) != 3:
        return False

    required = {
        "fold",
        "fit",
        "monitor",
        "outer_train",
        "outer_valid",
    }

    for expected_fold, partition in enumerate(
        value
    ):
        if not isinstance(partition, dict):
            return False

        if not required.issubset(
            partition.keys()
        ):
            return False

        if int(partition["fold"]) != expected_fold:
            return False

        for key in [
            "fit",
            "monitor",
            "outer_train",
            "outer_valid",
        ]:
            indices = np.asarray(
                partition[key]
            )

            if indices.ndim != 1:
                return False

            if not np.issubdtype(
                indices.dtype,
                np.integer,
            ):
                return False

            if (
                indices.min() < 0
                or indices.max() >= EXPECTED_CASES
            ):
                return False

    return True


def phase32_resolve_partitions():
    candidate_names = [
        "PHASE19_PARTITIONS",
        "PHASE31_PARTITIONS",
        "phase19_partitions",
        "phase31_partitions",
    ]

    attempts = []

    for name in candidate_names:
        value = globals().get(name)

        if value is None:
            continue

        valid = phase32_validate_partitions(
            value
        )

        attempts.append({
            "name": name,
            "valid": valid,
        })

        if valid:
            return list(value), name, attempts

    # Structural fallback across notebook globals.
    for name, value in list(globals().items()):
        if not (
            "partition" in name.lower()
            or "split" in name.lower()
        ):
            continue

        valid = phase32_validate_partitions(
            value
        )

        attempts.append({
            "name": name,
            "valid": valid,
        })

        if valid:
            return list(value), name, attempts

    raise AssertionError({
        "message": (
            "Could not resolve the three nested partitions."
        ),
        "attempts": attempts,
    })


(
    PHASE32_PARTITIONS,
    phase32_partition_source,
    phase32_partition_attempts,
) = phase32_resolve_partitions()


def phase32_fix_component_signs(components):
    components = np.asarray(
        components,
        dtype=np.float64,
    ).copy()

    for component_index in range(
        components.shape[0]
    ):
        vector = components[component_index]

        anchor_index = int(
            np.argmax(np.abs(vector))
        )

        if vector[anchor_index] < 0:
            components[component_index] *= -1.0

    return components


def phase32_fit_streaming_p3ca(indices):
    """
    Fits six independent view-specific P3CA bases.

    The prompt contains 64 spatial tokens per view.
    Computation uses:
      sum(x), x^T x
    and never materializes the complete prompt matrix.
    """
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    assert indices.ndim == 1
    assert len(indices) > 1

    view_count = 6
    channel_count = 384

    running_sum = np.zeros(
        (view_count, channel_count),
        dtype=np.float64,
    )

    running_cross = np.zeros(
        (
            view_count,
            channel_count,
            channel_count,
        ),
        dtype=np.float64,
    )

    token_count = 0
    chunk_size = 24

    prompt_mask_flat = (
        PHASE32_REAL_PROMPT_MASK.reshape(-1)
    )

    for start in range(
        0,
        len(indices),
        chunk_size,
    ):
        chunk_indices = indices[
            start:start + chunk_size
        ]

        dense_chunk = np.asarray(
            PHASE32_BILATERAL_DENSE_PRIVATE[
                chunk_indices
            ],
            dtype=np.float32,
        )

        # [B,6,196,384] -> prompted [B,6,64,384]
        dense_chunk = dense_chunk.reshape(
            len(chunk_indices),
            6,
            196,
            384,
        )

        prompt_chunk = dense_chunk[
            :,
            :,
            prompt_mask_flat,
            :,
        ]

        batch_token_count = (
            len(chunk_indices)
            * int(prompt_mask_flat.sum())
        )

        # [6, B*64, 384]
        prompt_by_view = (
            prompt_chunk
            .transpose(1, 0, 2, 3)
            .reshape(
                6,
                batch_token_count,
                384,
            )
        )

        prompt_tensor = torch.from_numpy(
            prompt_by_view
        ).to(
            PHASE32_DEVICE,
            non_blocking=True,
        )

        chunk_sum = prompt_tensor.sum(
            dim=1
        )

        chunk_cross = torch.bmm(
            prompt_tensor.transpose(1, 2),
            prompt_tensor,
        )

        running_sum += (
            chunk_sum
            .detach()
            .cpu()
            .double()
            .numpy()
        )

        running_cross += (
            chunk_cross
            .detach()
            .cpu()
            .double()
            .numpy()
        )

        token_count += batch_token_count

        del dense_chunk
        del prompt_chunk
        del prompt_by_view
        del prompt_tensor
        del chunk_sum
        del chunk_cross

    assert token_count == (
        len(indices)
        * int(PHASE32_REAL_PROMPT_MASK.sum())
    )

    means = np.zeros(
        (6, 384),
        dtype=np.float32,
    )
    standard_deviations = np.zeros(
        (6, 384),
        dtype=np.float32,
    )
    components = np.zeros(
        (
            6,
            PHASE32_P3CA_COMPONENT_COUNT,
            384,
        ),
        dtype=np.float32,
    )
    explained_ratios = np.zeros(
        (
            6,
            PHASE32_P3CA_COMPONENT_COUNT,
        ),
        dtype=np.float32,
    )
    eigenvalues = np.zeros(
        (
            6,
            PHASE32_P3CA_COMPONENT_COUNT,
        ),
        dtype=np.float32,
    )

    for view_index in range(6):
        mean = (
            running_sum[view_index]
            / token_count
        )

        centered_cross = (
            running_cross[view_index]
            - np.outer(
                running_sum[view_index],
                running_sum[view_index],
            ) / token_count
        )

        covariance = centered_cross / (
            token_count - 1
        )

        covariance = 0.5 * (
            covariance + covariance.T
        )

        variance = np.maximum(
            np.diag(covariance),
            1e-10,
        )

        standard_deviation = np.sqrt(
            variance
        )

        normalized_covariance = (
            covariance
            / standard_deviation[:, None]
            / standard_deviation[None, :]
        )

        normalized_covariance = 0.5 * (
            normalized_covariance
            + normalized_covariance.T
        )

        all_eigenvalues, all_eigenvectors = (
            np.linalg.eigh(
                normalized_covariance
            )
        )

        order = np.argsort(
            all_eigenvalues
        )[::-1]

        all_eigenvalues = np.maximum(
            all_eigenvalues[order],
            0.0,
        )

        all_eigenvectors = (
            all_eigenvectors[:, order]
        )

        selected_components = (
            all_eigenvectors[
                :,
                :PHASE32_P3CA_COMPONENT_COUNT,
            ].T
        )

        selected_components = (
            phase32_fix_component_signs(
                selected_components
            )
        )

        selected_eigenvalues = (
            all_eigenvalues[
                :PHASE32_P3CA_COMPONENT_COUNT
            ]
        )

        total_variance = max(
            float(all_eigenvalues.sum()),
            1e-12,
        )

        means[view_index] = mean.astype(
            np.float32
        )

        standard_deviations[view_index] = (
            standard_deviation.astype(
                np.float32
            )
        )

        components[view_index] = (
            selected_components.astype(
                np.float32
            )
        )

        eigenvalues[view_index] = (
            selected_eigenvalues.astype(
                np.float32
            )
        )

        explained_ratios[view_index] = (
            selected_eigenvalues
            / total_variance
        ).astype(np.float32)

    return {
        "mean": means,
        "standard_deviation": (
            standard_deviations
        ),
        "components": components,
        "explained_ratio": (
            explained_ratios
        ),
        "eigenvalues": eigenvalues,
        "token_count": int(token_count),
    }


def phase32_validate_state_cache():
    if not (
        PHASE32_P3CA_STATE_PATH.is_file()
        and PHASE32_P3CA_STATE_METADATA_PATH.is_file()
    ):
        return None

    try:
        state = np.load(
            PHASE32_P3CA_STATE_PATH,
            allow_pickle=False,
        )

        metadata = json.loads(
            PHASE32_P3CA_STATE_METADATA_PATH.read_text(
                encoding="utf-8"
            )
        )

        expected_shapes = {
            "mean": (2, 3, 6, 384),
            "standard_deviation": (
                2, 3, 6, 384
            ),
            "components": (
                2, 3, 6, 16, 384
            ),
            "explained_ratio": (
                2, 3, 6, 16
            ),
            "eigenvalues": (
                2, 3, 6, 16
            ),
            "token_count": (2, 3),
        }

        for key, expected_shape in (
            expected_shapes.items()
        ):
            if key not in state:
                return None

            if state[key].shape != expected_shape:
                return None

            if not np.isfinite(
                state[key]
            ).all():
                return None

        if metadata.get("status") != "complete":
            return None

        return state, metadata

    except Exception:
        return None


phase32_basis_started = time.perf_counter()
cached_state = phase32_validate_state_cache()
phase32_state_cache_reused = (
    cached_state is not None
)

if cached_state is not None:
    state_file, phase32_state_metadata = (
        cached_state
    )

    PHASE32_P3CA_STATES_PRIVATE = {
        key: np.asarray(state_file[key])
        for key in [
            "mean",
            "standard_deviation",
            "components",
            "explained_ratio",
            "eigenvalues",
            "token_count",
        ]
    }

else:
    state_mean = np.zeros(
        (2, 3, 6, 384),
        dtype=np.float32,
    )
    state_standard_deviation = np.zeros(
        (2, 3, 6, 384),
        dtype=np.float32,
    )
    state_components = np.zeros(
        (2, 3, 6, 16, 384),
        dtype=np.float32,
    )
    state_explained_ratio = np.zeros(
        (2, 3, 6, 16),
        dtype=np.float32,
    )
    state_eigenvalues = np.zeros(
        (2, 3, 6, 16),
        dtype=np.float32,
    )
    state_token_count = np.zeros(
        (2, 3),
        dtype=np.int64,
    )

    basis_records = []

    for stage_index, stage_name in enumerate(
        PHASE32_P3CA_STAGE_NAMES
    ):
        for fold in range(3):
            indices = np.asarray(
                PHASE32_PARTITIONS[fold][
                    stage_name
                ],
                dtype=np.int64,
            )

            fit_started = time.perf_counter()

            fitted = (
                phase32_fit_streaming_p3ca(
                    indices
                )
            )

            state_mean[
                stage_index, fold
            ] = fitted["mean"]

            state_standard_deviation[
                stage_index, fold
            ] = fitted[
                "standard_deviation"
            ]

            state_components[
                stage_index, fold
            ] = fitted["components"]

            state_explained_ratio[
                stage_index, fold
            ] = fitted[
                "explained_ratio"
            ]

            state_eigenvalues[
                stage_index, fold
            ] = fitted["eigenvalues"]

            state_token_count[
                stage_index, fold
            ] = fitted["token_count"]

            record = {
                "stage": stage_name,
                "fold": fold,
                "case_count": int(
                    len(indices)
                ),
                "prompt_token_count": int(
                    fitted["token_count"]
                ),
                "mean_explained_fraction": float(
                    fitted[
                        "explained_ratio"
                    ].sum(axis=1).mean()
                ),
                "minimum_channel_standard_deviation": float(
                    fitted[
                        "standard_deviation"
                    ].min()
                ),
                "elapsed_seconds": round(
                    time.perf_counter()
                    - fit_started,
                    2,
                ),
            }

            basis_records.append(record)

            print(
                "Phase32 P3CA basis fitted: "
                f"stage={stage_name}, "
                f"fold={fold}, "
                f"explained="
                f"{record['mean_explained_fraction']:.4f}"
            )

    np.savez_compressed(
        PHASE32_P3CA_STATE_PATH,
        mean=state_mean,
        standard_deviation=(
            state_standard_deviation
        ),
        components=state_components,
        explained_ratio=(
            state_explained_ratio
        ),
        eigenvalues=state_eigenvalues,
        token_count=state_token_count,
    )

    phase32_state_metadata = {
        "schema_version": 1,
        "phase": "phase32_fold_local_p3ca",
        "status": "complete",
        "stage_names": (
            PHASE32_P3CA_STAGE_NAMES
        ),
        "fold_count": 3,
        "view_count": 6,
        "component_count": (
            PHASE32_P3CA_COMPONENT_COUNT
        ),
        "prompt_token_count_per_case_view": 64,
        "partition_source": (
            phase32_partition_source
        ),
        "basis_records": basis_records,
    }

    PHASE32_P3CA_STATE_METADATA_PATH.write_text(
        json.dumps(
            phase32_state_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    PHASE32_P3CA_STATES_PRIVATE = {
        "mean": state_mean,
        "standard_deviation": (
            state_standard_deviation
        ),
        "components": state_components,
        "explained_ratio": (
            state_explained_ratio
        ),
        "eigenvalues": state_eigenvalues,
        "token_count": state_token_count,
    }

components = (
    PHASE32_P3CA_STATES_PRIVATE[
        "components"
    ].astype(np.float64)
)

orthogonality = np.einsum(
    "sfvkc,sfvjc->sfvkj",
    components,
    components,
    optimize=True,
)

identity = np.eye(
    PHASE32_P3CA_COMPONENT_COUNT,
    dtype=np.float64,
)

maximum_orthogonality_error = float(
    np.max(
        np.abs(
            orthogonality
            - identity[
                None, None, None, :, :
            ]
        )
    )
)

assert maximum_orthogonality_error < 1e-4

explained_fraction = (
    PHASE32_P3CA_STATES_PRIVATE[
        "explained_ratio"
    ].sum(axis=-1)
)

report = {
    "phase": "phase32_fold_local_p3ca_bases",
    "status": "complete",
    "cache_reused": (
        phase32_state_cache_reused
    ),
    "stage_names": (
        PHASE32_P3CA_STAGE_NAMES
    ),
    "fold_count": 3,
    "view_count": 6,
    "basis_count": 36,
    "component_count_per_basis": 16,
    "mean_explained_variance_fraction": round(
        float(explained_fraction.mean()),
        6,
    ),
    "minimum_explained_variance_fraction": round(
        float(explained_fraction.min()),
        6,
    ),
    "maximum_explained_variance_fraction": round(
        float(explained_fraction.max()),
        6,
    ),
    "minimum_channel_standard_deviation": round(
        float(
            PHASE32_P3CA_STATES_PRIVATE[
                "standard_deviation"
            ].min()
        ),
        8,
    ),
    "maximum_component_orthogonality_error": (
        maximum_orthogonality_error
    ),
    "partition_source": (
        phase32_partition_source
    ),
    "fit_basis_uses_fit_images_only": True,
    "outer_train_basis_uses_outer_train_images_only": True,
    "outer_validation_images_used_for_basis_fitting": False,
    "labels_used": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase32_basis_started,
        2,
    ),
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE32_P3CA_BASES"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_P3CA_BASES"
)

Phase32 P3CA basis fitted: stage=fit, fold=0, explained=0.6063
Phase32 P3CA basis fitted: stage=fit, fold=1, explained=0.5847
Phase32 P3CA basis fitted: stage=fit, fold=2, explained=0.5873
Phase32 P3CA basis fitted: stage=outer_train, fold=0, explained=0.6034
Phase32 P3CA basis fitted: stage=outer_train, fold=1, explained=0.5859
Phase32 P3CA basis fitted: stage=outer_train, fold=2, explained=0.5942
BEGIN SANITIZED_PHASE32_P3CA_BASES
{
  "phase": "phase32_fold_local_p3ca_bases",
  "status": "complete",
  "cache_reused": false,
  "stage_names": [
    "fit",
    "outer_train"
  ],
  "fold_count": 3,
  "view_count": 6,
  "basis_count": 36,
  "component_count_per_basis": 16,
  "mean_explained_variance_fraction": 0.593648,
  "minimum_explained_variance_fraction": 0.569881,
  "maximum_explained_variance_fraction": 0.622439,
  "minimum_channel_standard_deviation": 0.08648659,
  "maximum_component_orthogonality_error": 1.6641980238318865e-08,
  "partition_source": "PHASE19_PARTITIONS",
  "fit_b

In [25]:
# Phase32 Cell 106B
# Project all dense maps using each fold/stage-specific P3CA basis.
#
# Summary:
#   [stage, fold, case, 768]
#
# Sequence:
#   [stage, fold, case, 84, 48]
#   6 views × 14 anatomical positions
#
# Stored as float16; converted to float32 during model fitting.

import os

PHASE32_SUMMARY_PATH = Path(
    "/kaggle/working/"
    "phase32_p3ca_summary_float16.npy"
)

PHASE32_SEQUENCE_PATH = Path(
    "/kaggle/working/"
    "phase32_p3ca_sequence_float16.npy"
)

PHASE32_PROJECTED_METADATA_PATH = Path(
    "/kaggle/working/"
    "phase32_p3ca_projected_metadata.json"
)

PHASE32_SUMMARY_PARTIAL_PATH = Path(
    "/kaggle/working/"
    "phase32_p3ca_summary_float16.partial.npy"
)

PHASE32_SEQUENCE_PARTIAL_PATH = Path(
    "/kaggle/working/"
    "phase32_p3ca_sequence_float16.partial.npy"
)

PHASE32_SUMMARY_SHAPE = (
    2,
    3,
    EXPECTED_CASES,
    768,
)

PHASE32_SEQUENCE_SHAPE = (
    2,
    3,
    EXPECTED_CASES,
    84,
    48,
)


def phase32_validate_projected_cache():
    required = [
        PHASE32_SUMMARY_PATH,
        PHASE32_SEQUENCE_PATH,
        PHASE32_PROJECTED_METADATA_PATH,
    ]

    if not all(path.is_file() for path in required):
        return None

    try:
        summary = np.load(
            PHASE32_SUMMARY_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        sequence = np.load(
            PHASE32_SEQUENCE_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        metadata = json.loads(
            PHASE32_PROJECTED_METADATA_PATH.read_text(
                encoding="utf-8"
            )
        )

        if summary.shape != PHASE32_SUMMARY_SHAPE:
            return None

        if sequence.shape != PHASE32_SEQUENCE_SHAPE:
            return None

        if summary.dtype != np.float16:
            return None

        if sequence.dtype != np.float16:
            return None

        if metadata.get("status") != "complete":
            return None

        check_indices = np.linspace(
            0,
            EXPECTED_CASES - 1,
            10,
            dtype=np.int64,
        )

        if not np.isfinite(
            np.asarray(
                summary[:, :, check_indices],
                dtype=np.float32,
            )
        ).all():
            return None

        if not np.isfinite(
            np.asarray(
                sequence[:, :, check_indices],
                dtype=np.float32,
            )
        ).all():
            return None

        return summary, sequence, metadata

    except Exception:
        return None


phase32_projection_started = time.perf_counter()
cached_projection = (
    phase32_validate_projected_cache()
)

phase32_projected_cache_reused = (
    cached_projection is not None
)

if cached_projection is not None:
    (
        PHASE32_P3CA_SUMMARY_PRIVATE,
        PHASE32_P3CA_SEQUENCE_PRIVATE,
        phase32_projected_metadata,
    ) = cached_projection

else:
    for partial_path in [
        PHASE32_SUMMARY_PARTIAL_PATH,
        PHASE32_SEQUENCE_PARTIAL_PATH,
    ]:
        if partial_path.exists():
            partial_path.unlink()

    summary_writer = np.lib.format.open_memmap(
        PHASE32_SUMMARY_PARTIAL_PATH,
        mode="w+",
        dtype=np.float16,
        shape=PHASE32_SUMMARY_SHAPE,
    )

    sequence_writer = np.lib.format.open_memmap(
        PHASE32_SEQUENCE_PARTIAL_PATH,
        mode="w+",
        dtype=np.float16,
        shape=PHASE32_SEQUENCE_SHAPE,
    )

    prompt_mask_tensor = torch.from_numpy(
        PHASE32_REAL_PROMPT_MASK.reshape(-1)
    ).to(PHASE32_DEVICE)

    outside_mask_tensor = (
        ~prompt_mask_tensor
    )

    projection_batch_size = 16

    if PHASE32_DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(
            PHASE32_DEVICE
        )

    for stage_index, stage_name in enumerate(
        PHASE32_P3CA_STAGE_NAMES
    ):
        for fold in range(3):
            mean_tensor = torch.from_numpy(
                PHASE32_P3CA_STATES_PRIVATE[
                    "mean"
                ][stage_index, fold]
            ).to(PHASE32_DEVICE)

            standard_deviation_tensor = (
                torch.from_numpy(
                    PHASE32_P3CA_STATES_PRIVATE[
                        "standard_deviation"
                    ][stage_index, fold]
                ).to(PHASE32_DEVICE)
            )

            component_tensor = torch.from_numpy(
                PHASE32_P3CA_STATES_PRIVATE[
                    "components"
                ][stage_index, fold]
            ).to(PHASE32_DEVICE)

            next_progress = 400

            for start in range(
                0,
                EXPECTED_CASES,
                projection_batch_size,
            ):
                stop = min(
                    start + projection_batch_size,
                    EXPECTED_CASES,
                )

                dense_numpy = np.asarray(
                    PHASE32_BILATERAL_DENSE_PRIVATE[
                        start:stop
                    ],
                    dtype=np.float32,
                ).copy()

                dense_tensor = torch.from_numpy(
                    dense_numpy
                ).to(
                    PHASE32_DEVICE,
                    non_blocking=True,
                )

                normalized = (
                    dense_tensor
                    - mean_tensor[
                        None, :, None, None, :
                    ]
                ) / standard_deviation_tensor[
                    None, :, None, None, :
                ]

                # [B,6,14,14,16]
                projected = torch.einsum(
                    "bvhwc,vkc->bvhwk",
                    normalized,
                    component_tensor,
                )

                batch_size = projected.shape[0]

                projected_flat = (
                    projected.reshape(
                        batch_size,
                        6,
                        196,
                        16,
                    )
                )

                prompt_projected = (
                    projected_flat[
                        :,
                        :,
                        prompt_mask_tensor,
                        :,
                    ]
                )

                outside_projected = (
                    projected_flat[
                        :,
                        :,
                        outside_mask_tensor,
                        :,
                    ]
                )

                full_mean = projected_flat.mean(
                    dim=2
                )
                full_std = projected_flat.std(
                    dim=2,
                    unbiased=False,
                )

                prompt_mean = (
                    prompt_projected.mean(dim=2)
                )
                prompt_std = (
                    prompt_projected.std(
                        dim=2,
                        unbiased=False,
                    )
                )

                outside_mean = (
                    outside_projected.mean(dim=2)
                )
                outside_std = (
                    outside_projected.std(
                        dim=2,
                        unbiased=False,
                    )
                )

                prompt_contrast = (
                    prompt_mean - outside_mean
                )

                # 8 summaries × 16 components = 128/view
                view_summary = torch.cat(
                    [
                        full_mean,
                        full_std,
                        prompt_mean,
                        prompt_std,
                        outside_mean,
                        outside_std,
                        prompt_contrast,
                        torch.abs(
                            prompt_contrast
                        ),
                    ],
                    dim=-1,
                )

                assert view_summary.shape == (
                    batch_size,
                    6,
                    128,
                )

                case_summary = (
                    view_summary.reshape(
                        batch_size,
                        768,
                    )
                )

                # Preserve the second image coordinate as
                # an anatomical 14-position sequence.
                row_mean = projected.mean(dim=2)
                row_std = projected.std(
                    dim=2,
                    unbiased=False,
                )
                center_row_mean = projected[
                    :,
                    :,
                    3:11,
                    :,
                    :,
                ].mean(dim=2)

                sequence = torch.cat(
                    [
                        row_mean,
                        row_std,
                        center_row_mean,
                    ],
                    dim=-1,
                )

                assert sequence.shape == (
                    batch_size,
                    6,
                    14,
                    48,
                )

                sequence = sequence.reshape(
                    batch_size,
                    84,
                    48,
                )

                assert torch.isfinite(
                    case_summary
                ).all()
                assert torch.isfinite(
                    sequence
                ).all()

                summary_writer[
                    stage_index,
                    fold,
                    start:stop,
                ] = (
                    case_summary
                    .detach()
                    .cpu()
                    .to(torch.float16)
                    .numpy()
                )

                sequence_writer[
                    stage_index,
                    fold,
                    start:stop,
                ] = (
                    sequence
                    .detach()
                    .cpu()
                    .to(torch.float16)
                    .numpy()
                )

                del dense_numpy
                del dense_tensor
                del normalized
                del projected
                del projected_flat
                del prompt_projected
                del outside_projected
                del view_summary
                del case_summary
                del sequence

                if (
                    stop >= next_progress
                    or stop == EXPECTED_CASES
                ):
                    print(
                        "Phase32 P3CA projection: "
                        f"stage={stage_name}, "
                        f"fold={fold}, "
                        f"{stop}/{EXPECTED_CASES}"
                    )

                    while next_progress <= stop:
                        next_progress += 400

            summary_writer.flush()
            sequence_writer.flush()

            del mean_tensor
            del standard_deviation_tensor
            del component_tensor

    summary_writer.flush()
    sequence_writer.flush()

    del summary_writer
    del sequence_writer

    os.replace(
        PHASE32_SUMMARY_PARTIAL_PATH,
        PHASE32_SUMMARY_PATH,
    )

    os.replace(
        PHASE32_SEQUENCE_PARTIAL_PATH,
        PHASE32_SEQUENCE_PATH,
    )

    PHASE32_P3CA_SUMMARY_PRIVATE = np.load(
        PHASE32_SUMMARY_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    PHASE32_P3CA_SEQUENCE_PRIVATE = np.load(
        PHASE32_SEQUENCE_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    phase32_projected_metadata = {
        "schema_version": 1,
        "phase": (
            "phase32_projected_p3ca_features"
        ),
        "status": "complete",
        "stage_names": (
            PHASE32_P3CA_STAGE_NAMES
        ),
        "summary_shape": list(
            PHASE32_SUMMARY_SHAPE
        ),
        "sequence_shape": list(
            PHASE32_SEQUENCE_SHAPE
        ),
        "summary_dtype": "float16",
        "sequence_dtype": "float16",
        "summary_definition": [
            "full_mean",
            "full_std",
            "prompt_mean",
            "prompt_std",
            "outside_mean",
            "outside_std",
            "prompt_minus_outside",
            "absolute_prompt_minus_outside",
        ],
        "sequence_definition": [
            "row_mean",
            "row_std",
            "central_row_mean",
        ],
    }

    PHASE32_PROJECTED_METADATA_PATH.write_text(
        json.dumps(
            phase32_projected_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

assert (
    PHASE32_P3CA_SUMMARY_PRIVATE.shape
    == PHASE32_SUMMARY_SHAPE
)
assert (
    PHASE32_P3CA_SEQUENCE_PRIVATE.shape
    == PHASE32_SEQUENCE_SHAPE
)

report = {
    "phase": "phase32_projected_p3ca_cache",
    "status": "complete",
    "cache_reused": (
        phase32_projected_cache_reused
    ),
    "summary": {
        "shape": list(
            PHASE32_P3CA_SUMMARY_PRIVATE.shape
        ),
        "dtype": str(
            PHASE32_P3CA_SUMMARY_PRIVATE.dtype
        ),
        "file_size_mb": round(
            PHASE32_SUMMARY_PATH.stat().st_size
            / (1024 ** 2),
            3,
        ),
    },
    "sequence": {
        "shape": list(
            PHASE32_P3CA_SEQUENCE_PRIVATE.shape
        ),
        "dtype": str(
            PHASE32_P3CA_SEQUENCE_PRIVATE.dtype
        ),
        "file_size_mb": round(
            PHASE32_SEQUENCE_PATH.stat().st_size
            / (1024 ** 2),
            3,
        ),
        "token_count": 84,
        "token_dimension": 48,
    },
    "stage_count": 2,
    "fold_count": 3,
    "view_count": 6,
    "component_count": 16,
    "peak_vram_mb": round(
        (
            torch.cuda.max_memory_allocated(
                PHASE32_DEVICE
            )
            / (1024 ** 2)
        )
        if PHASE32_DEVICE.type == "cuda"
        else 0.0,
        2,
    ),
    "elapsed_seconds": round(
        time.perf_counter()
        - phase32_projection_started,
        2,
    ),
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_features_exported": False,
}

print(
    "BEGIN SANITIZED_PHASE32_PROJECTED_CACHE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_PROJECTED_CACHE"
)

Phase32 P3CA projection: stage=fit, fold=0, 400/1362
Phase32 P3CA projection: stage=fit, fold=0, 800/1362
Phase32 P3CA projection: stage=fit, fold=0, 1200/1362
Phase32 P3CA projection: stage=fit, fold=0, 1362/1362
Phase32 P3CA projection: stage=fit, fold=1, 400/1362
Phase32 P3CA projection: stage=fit, fold=1, 800/1362
Phase32 P3CA projection: stage=fit, fold=1, 1200/1362
Phase32 P3CA projection: stage=fit, fold=1, 1362/1362
Phase32 P3CA projection: stage=fit, fold=2, 400/1362
Phase32 P3CA projection: stage=fit, fold=2, 800/1362
Phase32 P3CA projection: stage=fit, fold=2, 1200/1362
Phase32 P3CA projection: stage=fit, fold=2, 1362/1362
Phase32 P3CA projection: stage=outer_train, fold=0, 400/1362
Phase32 P3CA projection: stage=outer_train, fold=0, 800/1362
Phase32 P3CA projection: stage=outer_train, fold=0, 1200/1362
Phase32 P3CA projection: stage=outer_train, fold=0, 1362/1362
Phase32 P3CA projection: stage=outer_train, fold=1, 400/1362
Phase32 P3CA projection: stage=outer_train, fold=1,

In [26]:
# Phase32 Cell 106C
# Audit numerical quality and basis changes between fit and outer-train states.

audit_indices = np.linspace(
    0,
    EXPECTED_CASES - 1,
    64,
    dtype=np.int64,
)

summary_sample = np.asarray(
    PHASE32_P3CA_SUMMARY_PRIVATE[
        :,
        :,
        audit_indices,
    ],
    dtype=np.float32,
)

sequence_sample = np.asarray(
    PHASE32_P3CA_SEQUENCE_PRIVATE[
        :,
        :,
        audit_indices,
    ],
    dtype=np.float32,
)

assert summary_sample.shape == (
    2,
    3,
    64,
    768,
)
assert sequence_sample.shape == (
    2,
    3,
    64,
    84,
    48,
)

assert np.isfinite(summary_sample).all()
assert np.isfinite(sequence_sample).all()

summary_coordinate_std = np.std(
    summary_sample.reshape(
        2 * 3 * 64,
        768,
    ),
    axis=0,
)

sequence_coordinate_std = np.std(
    sequence_sample.reshape(
        2 * 3 * 64,
        -1,
    ),
    axis=0,
)

summary_collapsed_fraction = float(
    np.mean(
        summary_coordinate_std < 1e-6
    )
)

sequence_collapsed_fraction = float(
    np.mean(
        sequence_coordinate_std < 1e-6
    )
)

# Difference between the fit-only basis representation
# and outer-train-refitted representation.
stage_representation_difference = float(
    np.mean(
        np.abs(
            summary_sample[0]
            - summary_sample[1]
        )
    )
)

explained_fraction = (
    PHASE32_P3CA_STATES_PRIVATE[
        "explained_ratio"
    ].sum(axis=-1)
)

report = {
    "phase": (
        "phase32_projected_representation_audit"
    ),
    "status": "accepted",
    "sampled_case_count": 64,
    "summary_shape": list(
        PHASE32_P3CA_SUMMARY_PRIVATE.shape
    ),
    "sequence_shape": list(
        PHASE32_P3CA_SEQUENCE_PRIVATE.shape
    ),
    "summary_collapsed_coordinate_fraction": round(
        summary_collapsed_fraction,
        8,
    ),
    "sequence_collapsed_coordinate_fraction": round(
        sequence_collapsed_fraction,
        8,
    ),
    "median_summary_coordinate_standard_deviation": round(
        float(
            np.median(
                summary_coordinate_std
            )
        ),
        6,
    ),
    "median_sequence_coordinate_standard_deviation": round(
        float(
            np.median(
                sequence_coordinate_std
            )
        ),
        6,
    ),
    "mean_fit_vs_outer_train_representation_difference": round(
        stage_representation_difference,
        6,
    ),
    "mean_explained_variance_fraction": round(
        float(explained_fraction.mean()),
        6,
    ),
    "minimum_explained_variance_fraction": round(
        float(explained_fraction.min()),
        6,
    ),
    "maximum_explained_variance_fraction": round(
        float(explained_fraction.max()),
        6,
    ),
    "all_values_finite": True,
    "fit_and_outer_train_states_distinct": (
        stage_representation_difference > 1e-6
    ),
    "labels_used": False,
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

assert summary_collapsed_fraction < 0.02, (
    report
)
assert sequence_collapsed_fraction < 0.02, (
    report
)
assert (
    stage_representation_difference > 1e-6
), report

print(
    "BEGIN SANITIZED_PHASE32_PROJECTED_AUDIT"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_PROJECTED_AUDIT"
)

BEGIN SANITIZED_PHASE32_PROJECTED_AUDIT
{
  "phase": "phase32_projected_representation_audit",
  "status": "accepted",
  "sampled_case_count": 64,
  "summary_shape": [
    2,
    3,
    1362,
    768
  ],
  "sequence_shape": [
    2,
    3,
    1362,
    84,
    48
  ],
  "summary_collapsed_coordinate_fraction": 0.0,
  "sequence_collapsed_coordinate_fraction": 0.0,
  "median_summary_coordinate_standard_deviation": 0.953223,
  "median_sequence_coordinate_standard_deviation": 2.302327,
  "mean_fit_vs_outer_train_representation_difference": 1.164342,
  "mean_explained_variance_fraction": 0.593648,
  "minimum_explained_variance_fraction": 0.569881,
  "maximum_explained_variance_fraction": 0.622439,
  "all_values_finite": true,
  "fit_and_outer_train_states_distinct": true,
  "labels_used": false,
  "persistent_notebook_cache_only": true,
  "include_in_submission": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE32_PROJECTED_AUDIT


In [32]:
# Phase32 Cell 107A-DIAGNOSTIC
# Diagnose case alignment, highres parity, fold mapping,
# and fold-specific metric discrepancies.
#
# Reads only a small sample of training images.

from itertools import permutations

import json
import numpy as np


def phase32_nifti_uid(path):
    name = Path(path).name

    if name.endswith(".nii.gz"):
        return name[:-7]

    if name.endswith(".nii"):
        return name[:-4]

    raise ValueError(name)


assert "case_df" in globals()

uid_column = None

for candidate in [
    "uid",
    "case_id",
    "id",
]:
    if candidate in case_df.columns:
        uid_column = candidate
        break

assert uid_column is not None

case_uids = np.asarray(
    case_df[uid_column].astype(str),
    dtype=str,
)

assert case_uids.shape == (
    EXPECTED_CASES,
)
assert len(
    np.unique(case_uids)
) == EXPECTED_CASES

original_path_uids = np.asarray(
    [
        phase32_nifti_uid(path)
        for path in (
            PHASE32_PHASE12_CASE_PATHS
        )
    ],
    dtype=str,
)

original_path_alignment_fraction = float(
    np.mean(
        original_path_uids == case_uids
    )
)

# Reorder paths explicitly by case_df UID.
path_by_uid = {}

for path in PHASE32_PHASE12_CASE_PATHS:
    uid = phase32_nifti_uid(path)

    assert uid not in path_by_uid, (
        "Duplicate NIfTI UID."
    )

    path_by_uid[uid] = Path(path)

missing_uid_count = int(
    sum(
        uid not in path_by_uid
        for uid in case_uids
    )
)

assert missing_uid_count == 0, {
    "message": (
        "The resolved path list does not contain "
        "all case_df UIDs."
    ),
    "missing_uid_count": missing_uid_count,
}

phase32_aligned_case_paths = [
    path_by_uid[uid]
    for uid in case_uids
]

assert len(
    phase32_aligned_case_paths
) == EXPECTED_CASES
assert all(
    path.is_file()
    for path in phase32_aligned_case_paths
)

aligned_path_fraction = float(
    np.mean(
        np.asarray(
            [
                phase32_nifti_uid(path)
                for path in (
                    phase32_aligned_case_paths
                )
            ],
            dtype=str,
        )
        == case_uids
    )
)

assert aligned_path_fraction == 1.0

# Compare the persistent highres cache against exact
# raw Phase12c preprocessing for a small sample.
parity_indices = np.linspace(
    0,
    EXPECTED_CASES - 1,
    8,
    dtype=np.int64,
)

highres_maximum_errors = []
highres_mean_errors = []

crop_config = (
    PHASE32_PHASE12_BASE_MANIFEST[
        "preprocessing"
    ]["crops"]
)

for index in parity_indices:
    exact_crops = (
        PHASE32_PHASE12_BASE_MODULE
        .preprocess_case(
            phase32_aligned_case_paths[
                int(index)
            ],
            crop_config,
        )
    )

    exact_highres = np.asarray(
        exact_crops["128hr"],
        dtype=np.float32,
    )

    cached_highres = np.asarray(
        PHASE31_HIGHRES_CACHE[
            int(index)
        ],
        dtype=np.float32,
    )

    difference = np.abs(
        exact_highres - cached_highres
    )

    highres_maximum_errors.append(
        float(difference.max())
    )

    highres_mean_errors.append(
        float(difference.mean())
    )

maximum_highres_cache_error = float(
    np.max(highres_maximum_errors)
)

mean_highres_cache_error = float(
    np.mean(highres_mean_errors)
)

# Diagnose the already-created probability matrix.
fold_probability_matrix = np.load(
    PHASE32_PHASE12_FOLD_PROBABILITY_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

assert fold_probability_matrix.shape == (
    3,
    EXPECTED_CASES,
)

historical_fold_reference = {
    0: {
        "log_loss": 0.358494,
        "auroc": 0.916174,
    },
    1: {
        "log_loss": 0.255415,
        "auroc": 0.952183,
    },
    2: {
        "log_loss": 0.306209,
        "auroc": 0.945716,
    },
}

current_fold_metrics = []

for fold in range(3):
    outer_indices = np.asarray(
        PHASE32_PARTITIONS[fold][
            "outer_valid"
        ],
        dtype=np.int64,
    )

    metrics = phase32_metrics(
        PHASE32_LABELS[outer_indices],
        fold_probability_matrix[
            fold,
            outer_indices,
        ],
    )

    current_fold_metrics.append({
        "fold": fold,
        "n": int(len(outer_indices)),
        "log_loss": round(
            metrics["log_loss"],
            6,
        ),
        "auroc": round(
            metrics["auroc"],
            6,
        ),
        "historical_log_loss": (
            historical_fold_reference[
                fold
            ]["log_loss"]
        ),
        "historical_auroc": (
            historical_fold_reference[
                fold
            ]["auroc"]
        ),
        "absolute_log_loss_delta": round(
            abs(
                metrics["log_loss"]
                - historical_fold_reference[
                    fold
                ]["log_loss"]
            ),
            6,
        ),
        "absolute_auroc_delta": round(
            abs(
                metrics["auroc"]
                - historical_fold_reference[
                    fold
                ]["auroc"]
            ),
            6,
        ),
    })

# Check whether model-fold identifiers were permuted.
permutation_records = []

for permutation in permutations(
    [0, 1, 2]
):
    candidate_oof = np.full(
        EXPECTED_CASES,
        np.nan,
        dtype=np.float64,
    )

    for partition_fold in range(3):
        outer_indices = np.asarray(
            PHASE32_PARTITIONS[
                partition_fold
            ]["outer_valid"],
            dtype=np.int64,
        )

        model_fold = permutation[
            partition_fold
        ]

        candidate_oof[
            outer_indices
        ] = fold_probability_matrix[
            model_fold,
            outer_indices,
        ]

    metrics = phase32_metrics(
        PHASE32_LABELS,
        candidate_oof,
    )

    permutation_records.append({
        "partition_to_model_fold": list(
            permutation
        ),
        "log_loss": float(
            metrics["log_loss"]
        ),
        "auroc": float(
            metrics["auroc"]
        ),
        "brier": float(
            metrics["brier"]
        ),
    })

permutation_records.sort(
    key=lambda record: (
        record["log_loss"],
        -record["auroc"],
    )
)

best_permutation = permutation_records[0]

# Important correction for the next reconstruction.
PHASE32_PHASE12_CASE_PATHS = (
    phase32_aligned_case_paths
)

report = {
    "phase": (
        "phase32_phase12c_reconstruction_diagnostic"
    ),
    "status": "complete",
    "original_path_alignment_fraction": round(
        original_path_alignment_fraction,
        6,
    ),
    "corrected_path_alignment_fraction": round(
        aligned_path_fraction,
        6,
    ),
    "path_reordering_required": bool(
        original_path_alignment_fraction
        < 1.0
    ),
    "highres_cache_parity": {
        "sampled_case_count": int(
            len(parity_indices)
        ),
        "maximum_absolute_error": round(
            maximum_highres_cache_error,
            8,
        ),
        "mean_absolute_error": round(
            mean_highres_cache_error,
            8,
        ),
        "float16_equivalent": bool(
            maximum_highres_cache_error
            <= 0.001
        ),
    },
    "current_fold_metrics": (
        current_fold_metrics
    ),
    "best_fold_permutation_diagnostic": {
        "partition_to_model_fold": (
            best_permutation[
                "partition_to_model_fold"
            ]
        ),
        "log_loss": round(
            best_permutation[
                "log_loss"
            ],
            6,
        ),
        "auroc": round(
            best_permutation[
                "auroc"
            ],
            6,
        ),
        "brier": round(
            best_permutation[
                "brier"
            ],
            6,
        ),
    },
    "identity_fold_permutation": (
        best_permutation[
            "partition_to_model_fold"
        ] == [0, 1, 2]
    ),
    "training_voxel_arrays_read": True,
    "sampled_training_images_only": True,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
}

print(
    "BEGIN SANITIZED_PHASE32_RECONSTRUCTION_DIAGNOSTIC"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_RECONSTRUCTION_DIAGNOSTIC"
)

BEGIN SANITIZED_PHASE32_RECONSTRUCTION_DIAGNOSTIC
{
  "phase": "phase32_phase12c_reconstruction_diagnostic",
  "status": "complete",
  "original_path_alignment_fraction": 0.000734,
  "corrected_path_alignment_fraction": 1.0,
  "path_reordering_required": true,
  "highres_cache_parity": {
    "sampled_case_count": 8,
    "maximum_absolute_error": 0.00048828,
    "mean_absolute_error": 5.687e-05,
    "float16_equivalent": true
  },
  "current_fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "log_loss": 0.390348,
      "auroc": 0.899091,
      "historical_log_loss": 0.358494,
      "historical_auroc": 0.916174,
      "absolute_log_loss_delta": 0.031854,
      "absolute_auroc_delta": 0.017083
    },
    {
      "fold": 1,
      "n": 443,
      "log_loss": 0.255415,
      "auroc": 0.952162,
      "historical_log_loss": 0.255415,
      "historical_auroc": 0.952183,
      "absolute_log_loss_delta": 0.0,
      "absolute_auroc_delta": 2.1e-05
    },
    {
      "fold": 2,
      "n"

In [28]:
# Phase32 Cell 107A-R1
# Restore the exact Phase12c V1 package, all 21 models,
# and training-case paths.
#
# Does not read voxel arrays.

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import importlib.util
import json
import sys
import time

import numpy as np
import torch

PHASE32_PHASE12_ROOT = Path(
    "/kaggle/input/models/saifullah42/"
    "dat-phase12c/pytorch/default/1"
)

assert PHASE32_PHASE12_ROOT.is_dir()
assert (
    PHASE32_PHASE12_ROOT / "base_main.py"
).is_file()
assert (
    PHASE32_PHASE12_ROOT / "main.py"
).is_file()
assert (
    PHASE32_PHASE12_ROOT / "base_manifest.json"
).is_file()
assert (
    PHASE32_PHASE12_ROOT / "manifest.json"
).is_file()

phase32_phase12_started = time.perf_counter()


def phase32_load_source_module(
    module_name,
    source_path,
):
    specification = (
        importlib.util.spec_from_file_location(
            module_name,
            source_path,
        )
    )

    assert specification is not None
    assert specification.loader is not None

    module = importlib.util.module_from_spec(
        specification
    )

    specification.loader.exec_module(module)

    return module


# Load the exact Phase12c base module.
PHASE32_PHASE12_BASE_MODULE = (
    phase32_load_source_module(
        "phase32_phase12_base_main",
        PHASE32_PHASE12_ROOT
        / "base_main.py",
    )
)

# main.py performs `import base_main as base`.
# Temporarily bind that name to the exact package module.
previous_base_main_module = sys.modules.get(
    "base_main"
)

sys.modules["base_main"] = (
    PHASE32_PHASE12_BASE_MODULE
)

try:
    PHASE32_PHASE12_FAMILY_MODULE = (
        phase32_load_source_module(
            "phase32_phase12_family_main",
            PHASE32_PHASE12_ROOT
            / "main.py",
        )
    )
finally:
    if previous_base_main_module is None:
        sys.modules.pop(
            "base_main",
            None,
        )
    else:
        sys.modules["base_main"] = (
            previous_base_main_module
        )

PHASE32_PHASE12_BASE_MANIFEST = (
    json.loads(
        (
            PHASE32_PHASE12_ROOT
            / "base_manifest.json"
        ).read_text(
            encoding="utf-8"
        )
    )
)

PHASE32_PHASE12_FAMILY_MANIFEST = (
    json.loads(
        (
            PHASE32_PHASE12_ROOT
            / "manifest.json"
        ).read_text(
            encoding="utf-8"
        )
    )
)

assert (
    PHASE32_PHASE12_BASE_MANIFEST[
        "schema_version"
    ] == 2
)
assert (
    PHASE32_PHASE12_FAMILY_MANIFEST[
        "schema_version"
    ] == 3
)
assert (
    PHASE32_PHASE12_BASE_MANIFEST[
        "model_count"
    ] == 15
)
assert (
    PHASE32_PHASE12_FAMILY_MANIFEST[
        "model_count"
    ] == 21
)

PHASE32_PHASE12_BASE_MODULE.MANIFEST_PATH = (
    PHASE32_PHASE12_ROOT
    / "base_manifest.json"
)

(
    phase32_loaded_base_manifest,
    PHASE32_PHASE12_BASE_MODELS,
    PHASE32_PHASE12_DEVICE,
) = PHASE32_PHASE12_BASE_MODULE.load_bundle()

PHASE32_PHASE12_ANATOMY_MODELS = (
    PHASE32_PHASE12_FAMILY_MODULE
    .load_anatomy_bundle(
        PHASE32_PHASE12_FAMILY_MANIFEST,
        PHASE32_PHASE12_DEVICE,
    )
)

assert len(
    PHASE32_PHASE12_BASE_MODELS
) == 15
assert len(
    PHASE32_PHASE12_ANATOMY_MODELS
) == 6


def phase32_validate_case_paths(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()

    if not isinstance(value, (list, tuple)):
        return None

    if len(value) != EXPECTED_CASES:
        return None

    paths = [
        Path(item)
        for item in value
    ]

    if not all(path.is_file() for path in paths):
        return None

    if not all(
        (
            path.name.endswith(".nii")
            or path.name.endswith(".nii.gz")
        )
        for path in paths
    ):
        return None

    if len(set(paths)) != EXPECTED_CASES:
        return None

    return paths


def phase32_resolve_training_case_paths():
    preferred_names = [
        "PHASE31_CASE_PATHS",
        "PHASE19_CASE_PATHS",
        "case_paths",
        "training_paths",
        "image_paths",
        "nifti_paths",
    ]

    attempts = []

    for name in preferred_names:
        value = globals().get(name)

        if value is None:
            continue

        resolved = phase32_validate_case_paths(
            value
        )

        attempts.append({
            "source": name,
            "accepted": resolved is not None,
        })

        if resolved is not None:
            return resolved, name, attempts

    # Try a path column in case_df.
    if "case_df" in globals():
        dataframe = globals()["case_df"]

        for column in [
            "path",
            "image_path",
            "nifti_path",
            "filepath",
            "file_path",
        ]:
            if column not in dataframe.columns:
                continue

            resolved = (
                phase32_validate_case_paths(
                    dataframe[column].tolist()
                )
            )

            attempts.append({
                "source": f"case_df.{column}",
                "accepted": (
                    resolved is not None
                ),
            })

            if resolved is not None:
                return (
                    resolved,
                    f"case_df.{column}",
                    attempts,
                )

    # Final fallback: map case_df UIDs to the training niftis directory.
    assert "case_df" in globals(), {
        "message": "case_df is unavailable.",
        "attempts": attempts,
    }

    dataframe = globals()["case_df"]

    uid_column = None

    for candidate in [
        "uid",
        "case_id",
        "id",
    ]:
        if candidate in dataframe.columns:
            uid_column = candidate
            break

    assert uid_column is not None, {
        "message": (
            "Could not resolve the UID column."
        ),
        "attempts": attempts,
    }

    training_directories = [
        path
        for path in Path(
            "/kaggle/input"
        ).rglob("niftis")
        if path.is_dir()
        and "smoke" not in str(path).lower()
        and "test" not in path.name.lower()
    ]

    uids = [
        str(value)
        for value in dataframe[uid_column]
    ]

    for directory in training_directories:
        mapped = []

        for uid in uids:
            candidates = [
                directory / f"{uid}.nii.gz",
                directory / f"{uid}.nii",
            ]

            existing = [
                path
                for path in candidates
                if path.is_file()
            ]

            if len(existing) != 1:
                mapped = []
                break

            mapped.append(existing[0])

        resolved = phase32_validate_case_paths(
            mapped
        )

        attempts.append({
            "source": (
                "uid_mapping_to_niftis"
            ),
            "accepted": (
                resolved is not None
            ),
        })

        if resolved is not None:
            return (
                resolved,
                "case_df_uid_to_training_niftis",
                attempts,
            )

    raise AssertionError({
        "message": (
            "Could not resolve the 1,362 training paths."
        ),
        "attempts": attempts,
    })


(
    PHASE32_PHASE12_CASE_PATHS,
    phase32_phase12_path_source,
    phase32_phase12_path_attempts,
) = phase32_resolve_training_case_paths()

assert len(
    PHASE32_PHASE12_CASE_PATHS
) == EXPECTED_CASES

all_models = (
    list(
        PHASE32_PHASE12_BASE_MODELS.values()
    )
    + list(
        PHASE32_PHASE12_ANATOMY_MODELS.values()
    )
)

assert all(
    not model.training
    for model in all_models
)

report = {
    "phase": (
        "phase32_phase12c_model_restore"
    ),
    "status": "accepted",
    "package_location": (
        "kaggle_input_model"
    ),
    "schema_version": 3,
    "base_model_count": 15,
    "anatomy_model_count": 6,
    "total_model_count": 21,
    "fold_count": 3,
    "training_case_count": len(
        PHASE32_PHASE12_CASE_PATHS
    ),
    "training_case_path_source": (
        phase32_phase12_path_source
    ),
    "device": str(
        PHASE32_PHASE12_DEVICE
    ),
    "all_models_eval": True,
    "model_hashes_verified": True,
    "model_hashes_displayed": False,
    "voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase32_phase12_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE32_PHASE12C_RESTORE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_PHASE12C_RESTORE"
)

/kaggle/input/models/saifullah42/dat-phase12c/pytorch/default/1/main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


BEGIN SANITIZED_PHASE32_PHASE12C_RESTORE
{
  "phase": "phase32_phase12c_model_restore",
  "status": "accepted",
  "package_location": "kaggle_input_model",
  "schema_version": 3,
  "base_model_count": 15,
  "anatomy_model_count": 6,
  "total_model_count": 21,
  "fold_count": 3,
  "training_case_count": 1362,
  "training_case_path_source": "nifti_paths",
  "device": "cuda:0",
  "all_models_eval": true,
  "model_hashes_verified": true,
  "model_hashes_displayed": false,
  "voxel_arrays_read": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "uids_displayed": false,
  "elapsed_seconds": 3.62
}
END SANITIZED_PHASE32_PHASE12C_RESTORE


In [33]:
# Phase32 alignment guard

assert len(
    PHASE32_PHASE12_CASE_PATHS
) == EXPECTED_CASES

assert all(
    phase32_nifti_uid(
        PHASE32_PHASE12_CASE_PATHS[index]
    )
    == str(
        case_df.iloc[index][uid_column]
    )
    for index in range(EXPECTED_CASES)
)

print(
    "Phase32 Phase12c path alignment accepted: "
    "1362/1362"
)

Phase32 Phase12c path alignment accepted: 1362/1362


In [34]:
# Phase32 Cell 107A-R2
# Reconstruct all three fold-local Phase12c probability vectors.
#
# Persistent result:
#   [3, 1362] float64
#
# This allows:
#   - exact OOF reconstruction;
#   - fold-local monitor predictions;
#   - restart-safe later experiments.

from pathlib import Path
import json
import os
import time

import numpy as np
import torch

PHASE32_PHASE12_FOLD_PROBABILITY_PATH = Path(
    "/kaggle/working/"
    "phase32_phase12c_fold_probabilities_float64.npy"
)

PHASE32_PHASE12_FOLD_PROBABILITY_PARTIAL = Path(
    "/kaggle/working/"
    "phase32_phase12c_fold_probabilities_float64.partial.npy"
)

PHASE32_PHASE12_OOF_PATH = Path(
    "/kaggle/working/"
    "phase32_phase12c_oof_float64.npy"
)

PHASE32_PHASE12_METADATA_PATH = Path(
    "/kaggle/working/"
    "phase32_phase12c_prediction_metadata.json"
)

PHASE32_PHASE12_PREDICTION_SHAPE = (
    3,
    EXPECTED_CASES,
)


def phase32_preprocess_legacy_crops(path):
    """
    Exact Phase12c preprocessing for the 160mm and 192mm crops.
    The 128hr crop already exists in PHASE31_HIGHRES_CACHE.
    """
    image, array = (
        PHASE32_PHASE12_BASE_MODULE
        .load_canonical_volume(path)
    )

    center_world = (
        PHASE32_PHASE12_BASE_MODULE
        .robust_uptake_center(
            image,
            array,
        )
    )

    output = {}

    for crop_name in [
        "160",
        "192",
    ]:
        configuration = (
            PHASE32_PHASE12_BASE_MANIFEST[
                "preprocessing"
            ]["crops"][crop_name]
        )

        cube = (
            PHASE32_PHASE12_BASE_MODULE
            .resample_physical_cube(
                image,
                array,
                center_world,
                float(
                    configuration["cube_mm"]
                ),
                tuple(
                    configuration["shape"]
                ),
            )
        )

        output[crop_name] = (
            PHASE32_PHASE12_BASE_MODULE
            .percentile_normalize(cube)
        )

    return output


@torch.inference_mode()
def phase32_predict_phase12_fold_matrix(
    crop_160,
    crop_192,
    crop_highres,
):
    """
    Returns [3, batch] fold-local Phase12c probabilities.
    No across-fold averaging is performed.
    """
    device = PHASE32_PHASE12_DEVICE

    tensors = {
        "160": torch.from_numpy(
            np.asarray(
                crop_160,
                dtype=np.float32,
            )[:, None]
        ).to(
            device,
            non_blocking=True,
        ),
        "192": torch.from_numpy(
            np.asarray(
                crop_192,
                dtype=np.float32,
            )[:, None]
        ).to(
            device,
            non_blocking=True,
        ),
        "128hr": torch.from_numpy(
            np.asarray(
                crop_highres,
                dtype=np.float32,
            )[:, None]
        ).to(
            device,
            non_blocking=True,
        ),
    }

    amp_enabled = (
        device.type == "cuda"
    )

    family_weights = (
        PHASE32_PHASE12_FAMILY_MANIFEST[
            "family_weights"
        ]
    )

    epsilon = float(
        PHASE32_PHASE12_FAMILY_MANIFEST[
            "probability_epsilon"
        ]
    )

    fold_probabilities = []

    for fold in range(3):
        base_fold = (
            PHASE32_PHASE12_BASE_MANIFEST[
                "folds"
            ][fold]
        )

        family_fold = (
            PHASE32_PHASE12_FAMILY_MANIFEST[
                "folds"
            ][fold]
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logit_160_a = (
                PHASE32_PHASE12_BASE_MODELS[
                    (fold, "160_seed_a")
                ](tensors["160"])
            )

            logit_160_b = (
                PHASE32_PHASE12_BASE_MODELS[
                    (fold, "160_seed_b")
                ](tensors["160"])
            )

            logit_192 = (
                PHASE32_PHASE12_BASE_MODELS[
                    (fold, "192_seed_a")
                ](tensors["192"])
            )

            logit_128_b = (
                PHASE32_PHASE12_BASE_MODELS[
                    (fold, "128hr_seed_b")
                ](tensors["128hr"])
            )

            logit_128_c = (
                PHASE32_PHASE12_BASE_MODELS[
                    (fold, "128hr_seed_c")
                ](tensors["128hr"])
            )

            anatomy_logit_a = (
                PHASE32_PHASE12_ANATOMY_MODELS[
                    (fold, "seed_a")
                ](tensors["128hr"])
            )

            anatomy_logit_b = (
                PHASE32_PHASE12_ANATOMY_MODELS[
                    (fold, "seed_b")
                ](tensors["128hr"])
            )

        logit_160 = 0.5 * (
            logit_160_a.float().cpu().numpy()
            + logit_160_b.float().cpu().numpy()
        )

        phase7_logit = (
            (
                1.0
                - float(
                    base_fold["weight_192"]
                )
            )
            * logit_160
            + float(
                base_fold["weight_192"]
            )
            * logit_192.float().cpu().numpy()
        )

        highres_logit = 0.5 * (
            logit_128_b.float().cpu().numpy()
            + logit_128_c.float().cpu().numpy()
        )

        mixed_logit = (
            (
                1.0
                - float(
                    base_fold[
                        "highres_weight"
                    ]
                )
            )
            * phase7_logit
            + float(
                base_fold[
                    "highres_weight"
                ]
            )
            * highres_logit
        ).astype(np.float64)

        base_calibrated_logit = (
            float(
                base_fold[
                    "platt_intercept"
                ]
            )
            + float(
                base_fold[
                    "platt_slope"
                ]
            )
            * mixed_logit
        )

        base_probability = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    base_calibrated_logit,
                    -40.0,
                    40.0,
                )
            )
        )

        anatomy_calibrated_a = (
            float(
                family_fold[
                    "seed_a_platt_intercept"
                ]
            )
            + float(
                family_fold[
                    "seed_a_platt_slope"
                ]
            )
            * anatomy_logit_a.float()
            .cpu()
            .numpy()
            .astype(np.float64)
        )

        anatomy_calibrated_b = (
            float(
                family_fold[
                    "seed_b_platt_intercept"
                ]
            )
            + float(
                family_fold[
                    "seed_b_platt_slope"
                ]
            )
            * anatomy_logit_b.float()
            .cpu()
            .numpy()
            .astype(np.float64)
        )

        anatomy_probability_a = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    anatomy_calibrated_a,
                    -40.0,
                    40.0,
                )
            )
        )

        anatomy_probability_b = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    anatomy_calibrated_b,
                    -40.0,
                    40.0,
                )
            )
        )

        family_probability = (
            float(
                family_weights["phase11c"]
            )
            * base_probability
            + float(
                family_weights[
                    "anatomy_seed_a"
                ]
            )
            * anatomy_probability_a
            + float(
                family_weights[
                    "anatomy_seed_b"
                ]
            )
            * anatomy_probability_b
        )

        family_probability = np.clip(
            family_probability,
            epsilon,
            1.0 - epsilon,
        )

        fold_probabilities.append(
            family_probability.astype(
                np.float64
            )
        )

    return np.stack(
        fold_probabilities,
        axis=0,
    )


def phase32_assemble_oof(
    fold_probability_matrix,
):
    oof = np.full(
        EXPECTED_CASES,
        np.nan,
        dtype=np.float64,
    )

    coverage = np.zeros(
        EXPECTED_CASES,
        dtype=np.int64,
    )

    for fold in range(3):
        outer_indices = np.asarray(
            PHASE32_PARTITIONS[fold][
                "outer_valid"
            ],
            dtype=np.int64,
        )

        oof[outer_indices] = (
            fold_probability_matrix[
                fold,
                outer_indices,
            ]
        )

        coverage[outer_indices] += 1

    assert np.all(coverage == 1)
    assert np.isfinite(oof).all()

    return oof


def phase32_validate_phase12_prediction_cache():
    if not (
        PHASE32_PHASE12_FOLD_PROBABILITY_PATH.is_file()
        and PHASE32_PHASE12_OOF_PATH.is_file()
        and PHASE32_PHASE12_METADATA_PATH.is_file()
    ):
        return None

    try:
        fold_probability = np.load(
            PHASE32_PHASE12_FOLD_PROBABILITY_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        oof = np.load(
            PHASE32_PHASE12_OOF_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        metadata = json.loads(
            PHASE32_PHASE12_METADATA_PATH.read_text(
                encoding="utf-8"
            )
        )

        if (
            fold_probability.shape
            != PHASE32_PHASE12_PREDICTION_SHAPE
        ):
            return None

        if oof.shape != (
            EXPECTED_CASES,
        ):
            return None

        if (
            fold_probability.dtype
            != np.float64
            or oof.dtype != np.float64
        ):
            return None

        if not np.isfinite(
            fold_probability
        ).all():
            return None

        if not np.isfinite(oof).all():
            return None

        metrics = phase32_metrics(
            PHASE32_LABELS,
            oof,
        )

        if (
            abs(
                metrics["log_loss"]
                - 0.307615
            ) > 0.0015
        ):
            return None

        if (
            abs(
                metrics["auroc"]
                - 0.938928
            ) > 0.0015
        ):
            return None

        if metadata.get("status") != "complete":
            return None

        return (
            fold_probability,
            oof,
            metadata,
        )

    except Exception:
        return None


phase32_prediction_started = time.perf_counter()
cached_prediction = (
    phase32_validate_phase12_prediction_cache()
)

phase32_phase12_prediction_reused = (
    cached_prediction is not None
)

if cached_prediction is not None:
    (
        PHASE32_PHASE12C_FOLD_PROBABILITIES_PRIVATE,
        PHASE19_PHASE12C_OOF_PRIVATE,
        phase32_phase12_prediction_metadata,
    ) = cached_prediction

else:
    if (
        PHASE32_PHASE12_FOLD_PROBABILITY_PARTIAL
        .exists()
    ):
        (
            PHASE32_PHASE12_FOLD_PROBABILITY_PARTIAL
            .unlink()
        )

    probability_writer = (
        np.lib.format.open_memmap(
            PHASE32_PHASE12_FOLD_PROBABILITY_PARTIAL,
            mode="w+",
            dtype=np.float64,
            shape=(
                PHASE32_PHASE12_PREDICTION_SHAPE
            ),
        )
    )

    preprocessing_failures = []
    batch_size = 8
    worker_count = 4
    next_progress = 100

    with ThreadPoolExecutor(
        max_workers=worker_count
    ) as executor:
        for start in range(
            0,
            EXPECTED_CASES,
            batch_size,
        ):
            stop = min(
                start + batch_size,
                EXPECTED_CASES,
            )

            batch_paths = (
                PHASE32_PHASE12_CASE_PATHS[
                    start:stop
                ]
            )

            prepared = list(
                executor.map(
                    phase32_preprocess_legacy_crops,
                    batch_paths,
                )
            )

            if any(
                item is None
                for item in prepared
            ):
                preprocessing_failures.append(
                    (start, stop)
                )
                continue

            crop_160 = np.stack(
                [
                    item["160"]
                    for item in prepared
                ],
                axis=0,
            ).astype(
                np.float32,
                copy=False,
            )

            crop_192 = np.stack(
                [
                    item["192"]
                    for item in prepared
                ],
                axis=0,
            ).astype(
                np.float32,
                copy=False,
            )

            crop_highres = np.asarray(
                PHASE31_HIGHRES_CACHE[
                    start:stop
                ],
                dtype=np.float32,
            )

            batch_probability = (
                phase32_predict_phase12_fold_matrix(
                    crop_160,
                    crop_192,
                    crop_highres,
                )
            )

            assert batch_probability.shape == (
                3,
                stop - start,
            )
            assert np.isfinite(
                batch_probability
            ).all()

            probability_writer[
                :,
                start:stop,
            ] = batch_probability

            if (
                stop >= next_progress
                or stop == EXPECTED_CASES
            ):
                probability_writer.flush()

                print(
                    "Phase32 Phase12c reconstruction: "
                    f"{stop}/{EXPECTED_CASES}"
                )

                while next_progress <= stop:
                    next_progress += 100

            del prepared
            del crop_160
            del crop_192
            del crop_highres
            del batch_probability

    probability_writer.flush()
    del probability_writer

    assert not preprocessing_failures, {
        "message": (
            "Legacy preprocessing failed."
        ),
        "failure_batch_count": len(
            preprocessing_failures
        ),
    }

    os.replace(
        PHASE32_PHASE12_FOLD_PROBABILITY_PARTIAL,
        PHASE32_PHASE12_FOLD_PROBABILITY_PATH,
    )

    PHASE32_PHASE12C_FOLD_PROBABILITIES_PRIVATE = (
        np.load(
            PHASE32_PHASE12_FOLD_PROBABILITY_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )
    )

    phase12_oof = phase32_assemble_oof(
        PHASE32_PHASE12C_FOLD_PROBABILITIES_PRIVATE
    )

    np.save(
        PHASE32_PHASE12_OOF_PATH,
        phase12_oof,
        allow_pickle=False,
    )

    PHASE19_PHASE12C_OOF_PRIVATE = np.load(
        PHASE32_PHASE12_OOF_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    reconstructed_metrics = (
        phase32_metrics(
            PHASE32_LABELS,
            PHASE19_PHASE12C_OOF_PRIVATE,
        )
    )

    phase32_phase12_prediction_metadata = {
        "schema_version": 1,
        "phase": (
            "phase32_phase12c_oof_restore"
        ),
        "status": "complete",
        "shape": [
            3,
            EXPECTED_CASES,
        ],
        "oof_shape": [
            EXPECTED_CASES,
        ],
        "dtype": "float64",
        "metrics": reconstructed_metrics,
        "worker_count": worker_count,
        "batch_size": batch_size,
    }

    PHASE32_PHASE12_METADATA_PATH.write_text(
        json.dumps(
            phase32_phase12_prediction_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )


restored_metrics = phase32_metrics(
    PHASE32_LABELS,
    PHASE19_PHASE12C_OOF_PRIVATE,
)

assert (
    abs(
        restored_metrics["log_loss"]
        - 0.307615
    ) <= 0.0015
), restored_metrics

assert (
    abs(
        restored_metrics["auroc"]
        - 0.938928
    ) <= 0.0015
), restored_metrics

# Stable aliases for all later Phase32 cells.
PHASE32_BASELINE_OOF_PRIVATE = np.asarray(
    PHASE19_PHASE12C_OOF_PRIVATE,
    dtype=np.float64,
)

report = {
    "phase": (
        "phase32_phase12c_oof_reconstruction"
    ),
    "status": "accepted",
    "cache_reused": (
        phase32_phase12_prediction_reused
    ),
    "fold_probability_shape": list(
        PHASE32_PHASE12C_FOLD_PROBABILITIES_PRIVATE.shape
    ),
    "oof_shape": list(
        PHASE32_BASELINE_OOF_PRIVATE.shape
    ),
    "metrics": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in (
            restored_metrics.items()
        )
    },
    "historical_reference": {
        "log_loss": 0.307615,
        "auroc": 0.938928,
        "brier": 0.095535,
    },
    "fold_local_monitor_probabilities_available": True,
    "persistent_prediction_cache": True,
    "model_count": 21,
    "preprocessing": (
        "exact Phase12c 160mm and 192mm; "
        "reused exact persistent 128hr cache"
    ),
    "elapsed_seconds": round(
        time.perf_counter()
        - phase32_prediction_started,
        2,
    ),
    "training_voxel_arrays_read": (
        not phase32_phase12_prediction_reused
    ),
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "uids_displayed": False,
}

print(
    "BEGIN SANITIZED_PHASE32_PHASE12C_OOF_RESTORE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_PHASE12C_OOF_RESTORE"
)

Phase32 Phase12c reconstruction: 104/1362
Phase32 Phase12c reconstruction: 200/1362
Phase32 Phase12c reconstruction: 304/1362
Phase32 Phase12c reconstruction: 400/1362
Phase32 Phase12c reconstruction: 504/1362
Phase32 Phase12c reconstruction: 600/1362
Phase32 Phase12c reconstruction: 704/1362
Phase32 Phase12c reconstruction: 800/1362
Phase32 Phase12c reconstruction: 904/1362
Phase32 Phase12c reconstruction: 1000/1362
Phase32 Phase12c reconstruction: 1104/1362
Phase32 Phase12c reconstruction: 1200/1362
Phase32 Phase12c reconstruction: 1304/1362
Phase32 Phase12c reconstruction: 1362/1362
BEGIN SANITIZED_PHASE32_PHASE12C_OOF_RESTORE
{
  "phase": "phase32_phase12c_oof_reconstruction",
  "status": "accepted",
  "cache_reused": false,
  "fold_probability_shape": [
    3,
    1362
  ],
  "oof_shape": [
    1362
  ],
  "metrics": {
    "log_loss": 0.307616,
    "auroc": 0.938914,
    "brier": 0.095535,
    "mean_probability": 0.530698
  },
  "historical_reference": {
    "log_loss": 0.307615,


In [30]:
# Compatibility helper only if phase32_metrics is currently undefined.

if "phase32_metrics" not in globals():
    def phase32_metrics(labels, probabilities):
        labels = np.asarray(
            labels,
            dtype=np.int64,
        )

        probabilities = np.clip(
            np.asarray(
                probabilities,
                dtype=np.float64,
            ),
            1e-6,
            1.0 - 1e-6,
        )

        return {
            "log_loss": float(
                log_loss(
                    labels,
                    probabilities,
                    labels=[0, 1],
                )
            ),
            "auroc": float(
                roc_auc_score(
                    labels,
                    probabilities,
                )
            ),
            "brier": float(
                brier_score_loss(
                    labels,
                    probabilities,
                )
            ),
            "mean_probability": float(
                probabilities.mean()
            ),
        }

In [35]:
# Phase32 Cell 107A
# Resolve Phase12c OOF state and define the nested P3CA proxy experiment.

from scipy.fft import dct
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

import json
import numpy as np


def phase32_resolve_labels_and_groups():
    dataframe_candidates = [
        ("case_df", globals().get("case_df")),
        (
            "PHASE31_CASE_DF",
            globals().get("PHASE31_CASE_DF"),
        ),
    ]

    for dataframe_name, dataframe in (
        dataframe_candidates
    ):
        if dataframe is None:
            continue

        if len(dataframe) != EXPECTED_CASES:
            continue

        label_column = None

        for candidate in [
            "label",
            "target",
            "pathologic",
            "y",
        ]:
            if candidate in dataframe.columns:
                label_column = candidate
                break

        group_column = None

        for candidate in [
            "acquisition_group",
            "group",
            "scanner_group",
        ]:
            if candidate in dataframe.columns:
                group_column = candidate
                break

        if (
            label_column is None
            or group_column is None
        ):
            continue

        labels = np.asarray(
            dataframe[label_column],
            dtype=np.int64,
        )

        groups = np.asarray(
            dataframe[group_column],
            dtype=np.int64,
        )

        if not set(np.unique(labels)).issubset(
            {0, 1}
        ):
            continue

        return (
            labels,
            groups,
            dataframe_name,
            label_column,
            group_column,
        )

    raise AssertionError(
        "Could not resolve labels and acquisition groups."
    )


(
    PHASE32_LABELS,
    PHASE32_GROUPS,
    phase32_dataframe_source,
    phase32_label_column,
    phase32_group_column,
) = phase32_resolve_labels_and_groups()


def phase32_probability_vector(value):
    try:
        array = np.asarray(value)
    except Exception:
        return None

    array = np.squeeze(array)

    if array.shape != (EXPECTED_CASES,):
        return None

    if not np.issubdtype(
        array.dtype,
        np.number,
    ):
        return None

    array = np.asarray(
        array,
        dtype=np.float64,
    )

    if not np.isfinite(array).all():
        return None

    if (
        array.min() < 0.0
        or array.max() > 1.0
    ):
        return None

    return array


def phase32_extract_vectors(value):
    vectors = []

    direct = phase32_probability_vector(
        value
    )

    if direct is not None:
        vectors.append(direct)

    if isinstance(value, dict):
        for key in [
            "probability",
            "probabilities",
            "prediction",
            "predictions",
            "oof",
            "phase12c",
            "baseline",
        ]:
            if key not in value:
                continue

            nested = phase32_probability_vector(
                value[key]
            )

            if nested is not None:
                vectors.append(nested)

    return vectors


def phase32_resolve_baseline_oof():
    preferred_names = [
        "PHASE19_PHASE12C_OOF_PRIVATE",
        "PHASE31_BASELINE_OOF_PRIVATE",
        "PHASE30_PHASE12C_OOF_PRIVATE",
        "phase12c_oof",
        "baseline_oof",
    ]

    candidates = []

    for name in preferred_names:
        if name not in globals():
            continue

        for vector in phase32_extract_vectors(
            globals()[name]
        ):
            candidates.append(
                (name, vector)
            )

    for name, value in list(
        globals().items()
    ):
        lower_name = name.lower()

        if "oof" not in lower_name:
            continue

        if not any(
            term in lower_name
            for term in [
                "phase12",
                "baseline",
            ]
        ):
            continue

        for vector in phase32_extract_vectors(
            value
        ):
            candidates.append(
                (name, vector)
            )

    assert candidates, {
        "message": (
            "No Phase12c OOF probability vector found."
        ),
        "preferred_names": preferred_names,
    }

    target_metrics = np.asarray([
        0.307615,
        0.938928,
        0.095535,
    ])

    scored = []

    for name, vector in candidates:
        clipped = np.clip(
            vector,
            1e-6,
            1.0 - 1e-6,
        )

        metrics = np.asarray([
            log_loss(
                PHASE32_LABELS,
                clipped,
                labels=[0, 1],
            ),
            roc_auc_score(
                PHASE32_LABELS,
                clipped,
            ),
            brier_score_loss(
                PHASE32_LABELS,
                clipped,
            ),
        ])

        distance = float(
            np.max(
                np.abs(
                    metrics - target_metrics
                )
            )
        )

        scored.append(
            (distance, name, clipped, metrics)
        )

    scored.sort(
        key=lambda record: record[0]
    )

    (
        best_distance,
        best_name,
        best_vector,
        best_metrics,
    ) = scored[0]

    assert best_distance <= 0.0015, {
        "message": (
            "Best OOF vector does not reproduce Phase12c."
        ),
        "best_name": best_name,
        "maximum_metric_delta": best_distance,
        "metrics": best_metrics.tolist(),
    }

    return (
        best_vector,
        best_name,
        best_metrics,
        len(scored),
    )


(
    PHASE32_BASELINE_OOF_PRIVATE,
    phase32_baseline_source,
    phase32_baseline_metrics,
    phase32_baseline_candidate_count,
) = phase32_resolve_baseline_oof()


PHASE32_PROXY_CONFIG = {
    "phase": "phase32_p3ca_nested_proxy",
    "representations": [
        "p3ca_summary",
        "p3ca_low_frequency_sequence",
        "p3ca_summary_plus_low_frequency",
    ],
    "pca_dimensions": [
        16,
        32,
        64,
        96,
    ],
    "logistic_c_values": [
        0.01,
        0.03,
        0.1,
        0.3,
    ],
    "low_frequency_dct_components": 4,
    "robust_score_weights": {
        "overall_log_loss": 0.55,
        "equal_group_mean_log_loss": 0.30,
        "worst_group_log_loss": 0.15,
    },
    "base_seed": 320701,
}

candidate_count = (
    len(PHASE32_PROXY_CONFIG["representations"])
    * len(PHASE32_PROXY_CONFIG["pca_dimensions"])
    * len(PHASE32_PROXY_CONFIG["logistic_c_values"])
)

report = {
    "phase": "phase32_p3ca_proxy_contract",
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "label_source": (
        f"{phase32_dataframe_source}."
        f"{phase32_label_column}"
    ),
    "acquisition_group_source": (
        f"{phase32_dataframe_source}."
        f"{phase32_group_column}"
    ),
    "acquisition_group_count": int(
        len(np.unique(PHASE32_GROUPS))
    ),
    "baseline_oof_source": (
        phase32_baseline_source
    ),
    "baseline_candidate_count": (
        phase32_baseline_candidate_count
    ),
    "baseline_metrics": {
        "log_loss": round(
            float(phase32_baseline_metrics[0]),
            6,
        ),
        "auroc": round(
            float(phase32_baseline_metrics[1]),
            6,
        ),
        "brier": round(
            float(phase32_baseline_metrics[2]),
            6,
        ),
    },
    "representations": (
        PHASE32_PROXY_CONFIG[
            "representations"
        ]
    ),
    "candidate_count_per_fold": (
        candidate_count
    ),
    "selection_objective": (
        "0.55*overall_log_loss + "
        "0.30*equal_group_mean_log_loss + "
        "0.15*worst_group_log_loss"
    ),
    "selection_partition": "monitor_only",
    "final_refit_partition": "outer_train",
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE32_PROXY_CONTRACT"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_PROXY_CONTRACT"
)

BEGIN SANITIZED_PHASE32_PROXY_CONTRACT
{
  "phase": "phase32_p3ca_proxy_contract",
  "status": "accepted",
  "case_count": 1362,
  "label_source": "case_df.label",
  "acquisition_group_source": "case_df.acquisition_group",
  "acquisition_group_count": 15,
  "baseline_oof_source": "PHASE19_PHASE12C_OOF_PRIVATE",
  "baseline_candidate_count": 4,
  "baseline_metrics": {
    "log_loss": 0.307616,
    "auroc": 0.938914,
    "brier": 0.095535
  },
  "representations": [
    "p3ca_summary",
    "p3ca_low_frequency_sequence",
    "p3ca_summary_plus_low_frequency"
  ],
  "candidate_count_per_fold": 48,
  "selection_objective": "0.55*overall_log_loss + 0.30*equal_group_mean_log_loss + 0.15*worst_group_log_loss",
  "selection_partition": "monitor_only",
  "final_refit_partition": "outer_train",
  "outer_validation_labels_used": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE32_PROXY_CONTRACT


In [36]:
# Phase32 Cell 107B
# Select on monitor using an acquisition-group-robust objective,
# refit on outer_train, then infer outer validation exactly once.

import time


def phase32_metrics(labels, probabilities):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    result = {
        "log_loss": float(
            log_loss(
                labels,
                probabilities,
                labels=[0, 1],
            )
        ),
        "brier": float(
            brier_score_loss(
                labels,
                probabilities,
            )
        ),
        "mean_probability": float(
            probabilities.mean()
        ),
    }

    if len(np.unique(labels)) == 2:
        result["auroc"] = float(
            roc_auc_score(
                labels,
                probabilities,
            )
        )
    else:
        result["auroc"] = None

    return result


def phase32_robust_monitor_score(
    labels,
    probabilities,
    groups,
):
    overall = phase32_metrics(
        labels,
        probabilities,
    )["log_loss"]

    group_losses = []

    for group in np.unique(groups):
        mask = groups == group

        group_losses.append(
            phase32_metrics(
                labels[mask],
                probabilities[mask],
            )["log_loss"]
        )

    equal_group_mean = float(
        np.mean(group_losses)
    )

    worst_group = float(
        np.max(group_losses)
    )

    weights = PHASE32_PROXY_CONFIG[
        "robust_score_weights"
    ]

    score = (
        weights["overall_log_loss"]
        * overall
        + weights[
            "equal_group_mean_log_loss"
        ]
        * equal_group_mean
        + weights["worst_group_log_loss"]
        * worst_group
    )

    return {
        "robust_score": float(score),
        "overall_log_loss": float(overall),
        "equal_group_mean_log_loss": (
            equal_group_mean
        ),
        "worst_group_log_loss": (
            worst_group
        ),
        "group_count": int(
            len(group_losses)
        ),
    }


def phase32_representation(
    stage_index,
    fold,
    indices,
    representation_name,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    summary = np.asarray(
        PHASE32_P3CA_SUMMARY_PRIVATE[
            stage_index,
            fold,
            indices,
        ],
        dtype=np.float32,
    )

    if representation_name == "p3ca_summary":
        return summary

    sequence = np.asarray(
        PHASE32_P3CA_SEQUENCE_PRIVATE[
            stage_index,
            fold,
            indices,
        ],
        dtype=np.float32,
    )

    sequence = sequence.reshape(
        len(indices),
        6,
        14,
        48,
    )

    low_frequency = dct(
        sequence,
        type=2,
        axis=2,
        norm="ortho",
    )[
        :,
        :,
        :PHASE32_PROXY_CONFIG[
            "low_frequency_dct_components"
        ],
        :,
    ]

    low_frequency = np.asarray(
        low_frequency,
        dtype=np.float32,
    ).reshape(
        len(indices),
        -1,
    )

    if (
        representation_name
        == "p3ca_low_frequency_sequence"
    ):
        return low_frequency

    if (
        representation_name
        == "p3ca_summary_plus_low_frequency"
    ):
        return np.concatenate(
            [summary, low_frequency],
            axis=1,
        ).astype(np.float32)

    raise KeyError(representation_name)


def phase32_fit_transformer(
    fit_features,
    pca_dimension,
    seed,
):
    scaler = StandardScaler(
        with_mean=True,
        with_std=True,
    )

    scaled_fit = scaler.fit_transform(
        fit_features
    )

    resolved_dimension = min(
        int(pca_dimension),
        scaled_fit.shape[0] - 1,
        scaled_fit.shape[1],
    )

    assert resolved_dimension >= 2

    projector = PCA(
        n_components=resolved_dimension,
        svd_solver="randomized",
        iterated_power=4,
        random_state=seed,
        whiten=False,
    )

    transformed_fit = projector.fit_transform(
        scaled_fit
    )

    return (
        scaler,
        projector,
        transformed_fit,
    )


phase32_proxy_started = time.perf_counter()

PHASE32_P3CA_PROXY_OOF_PRIVATE = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

PHASE32_P3CA_PROXY_MODELS_PRIVATE = []
phase32_proxy_fold_records = []
phase32_proxy_coverage = np.zeros(
    EXPECTED_CASES,
    dtype=np.int64,
)

for fold in range(3):
    partition = PHASE32_PARTITIONS[fold]

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    y_fit = PHASE32_LABELS[fit_indices]
    y_monitor = PHASE32_LABELS[
        monitor_indices
    ]
    y_outer_train = PHASE32_LABELS[
        outer_train_indices
    ]
    y_outer_valid = PHASE32_LABELS[
        outer_valid_indices
    ]

    monitor_groups = PHASE32_GROUPS[
        monitor_indices
    ]

    candidate_records = []
    transformed_cache = {}

    fold_started = time.perf_counter()

    for representation_index, representation_name in enumerate(
        PHASE32_PROXY_CONFIG[
            "representations"
        ]
    ):
        fit_features = phase32_representation(
            stage_index=0,
            fold=fold,
            indices=fit_indices,
            representation_name=(
                representation_name
            ),
        )

        monitor_features = (
            phase32_representation(
                stage_index=0,
                fold=fold,
                indices=monitor_indices,
                representation_name=(
                    representation_name
                ),
            )
        )

        assert np.isfinite(
            fit_features
        ).all()
        assert np.isfinite(
            monitor_features
        ).all()

        for dimension_index, pca_dimension in enumerate(
            PHASE32_PROXY_CONFIG[
                "pca_dimensions"
            ]
        ):
            seed = (
                PHASE32_PROXY_CONFIG[
                    "base_seed"
                ]
                + fold * 100
                + representation_index * 10
                + dimension_index
            )

            (
                scaler,
                projector,
                transformed_fit,
            ) = phase32_fit_transformer(
                fit_features,
                pca_dimension,
                seed,
            )

            transformed_monitor = (
                projector.transform(
                    scaler.transform(
                        monitor_features
                    )
                )
            )

            for c_value in (
                PHASE32_PROXY_CONFIG[
                    "logistic_c_values"
                ]
            ):
                classifier = LogisticRegression(
                    C=float(c_value),
                    solver="lbfgs",
                    max_iter=4000,
                    random_state=seed,
                )

                classifier.fit(
                    transformed_fit,
                    y_fit,
                )

                monitor_probability = (
                    classifier.predict_proba(
                        transformed_monitor
                    )[:, 1]
                )

                robust_record = (
                    phase32_robust_monitor_score(
                        y_monitor,
                        monitor_probability,
                        monitor_groups,
                    )
                )

                candidate_records.append({
                    "representation": (
                        representation_name
                    ),
                    "input_dimension": int(
                        fit_features.shape[1]
                    ),
                    "pca_dimension": int(
                        transformed_fit.shape[1]
                    ),
                    "logistic_c": float(
                        c_value
                    ),
                    "seed": int(seed),
                    **robust_record,
                })

    assert len(candidate_records) == (
        len(
            PHASE32_PROXY_CONFIG[
                "representations"
            ]
        )
        * len(
            PHASE32_PROXY_CONFIG[
                "pca_dimensions"
            ]
        )
        * len(
            PHASE32_PROXY_CONFIG[
                "logistic_c_values"
            ]
        )
    )

    candidate_records.sort(
        key=lambda record: (
            record["robust_score"],
            record["overall_log_loss"],
            record["pca_dimension"],
            record["logistic_c"],
        )
    )

    selected = candidate_records[0]

    # Final refit uses the separately refitted outer-train
    # P3CA representation.
    final_train_features = (
        phase32_representation(
            stage_index=1,
            fold=fold,
            indices=outer_train_indices,
            representation_name=(
                selected["representation"]
            ),
        )
    )

    final_outer_features = (
        phase32_representation(
            stage_index=1,
            fold=fold,
            indices=outer_valid_indices,
            representation_name=(
                selected["representation"]
            ),
        )
    )

    (
        final_scaler,
        final_projector,
        transformed_outer_train,
    ) = phase32_fit_transformer(
        final_train_features,
        selected["pca_dimension"],
        selected["seed"],
    )

    final_classifier = LogisticRegression(
        C=selected["logistic_c"],
        solver="lbfgs",
        max_iter=4000,
        random_state=selected["seed"],
    )

    final_classifier.fit(
        transformed_outer_train,
        y_outer_train,
    )

    transformed_outer_valid = (
        final_projector.transform(
            final_scaler.transform(
                final_outer_features
            )
        )
    )

    outer_probability = (
        final_classifier.predict_proba(
            transformed_outer_valid
        )[:, 1]
    )

    outer_probability = np.clip(
        outer_probability,
        1e-6,
        1.0 - 1e-6,
    )

    PHASE32_P3CA_PROXY_OOF_PRIVATE[
        outer_valid_indices
    ] = outer_probability

    phase32_proxy_coverage[
        outer_valid_indices
    ] += 1

    outer_metrics = phase32_metrics(
        y_outer_valid,
        outer_probability,
    )

    monitor_metrics = {
        "robust_score": selected[
            "robust_score"
        ],
        "log_loss": selected[
            "overall_log_loss"
        ],
        "equal_group_mean_log_loss": (
            selected[
                "equal_group_mean_log_loss"
            ]
        ),
        "worst_group_log_loss": selected[
            "worst_group_log_loss"
        ],
    }

    PHASE32_P3CA_PROXY_MODELS_PRIVATE.append({
        "fold": fold,
        "specification": dict(selected),
        "scaler": final_scaler,
        "projector": final_projector,
        "classifier": final_classifier,
    })

    phase32_proxy_fold_records.append({
        "fold": fold,
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(
            len(monitor_indices)
        ),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "candidate_count": int(
            len(candidate_records)
        ),
        "selected_specification": {
            "representation": selected[
                "representation"
            ],
            "input_dimension": selected[
                "input_dimension"
            ],
            "pca_dimension": selected[
                "pca_dimension"
            ],
            "logistic_c": selected[
                "logistic_c"
            ],
        },
        "monitor_metrics": monitor_metrics,
        "outer_metrics": outer_metrics,
        "elapsed_seconds": round(
            time.perf_counter()
            - fold_started,
            2,
        ),
    })

    print(
        f"Phase32 fold {fold} selected: "
        f"{selected['representation']}, "
        f"PCA={selected['pca_dimension']}, "
        f"C={selected['logistic_c']}, "
        f"monitor={selected['overall_log_loss']:.6f}, "
        f"robust={selected['robust_score']:.6f}, "
        f"outer={outer_metrics['log_loss']:.6f}, "
        f"auroc={outer_metrics['auroc']:.6f}"
    )

assert np.all(
    phase32_proxy_coverage == 1
)
assert np.isfinite(
    PHASE32_P3CA_PROXY_OOF_PRIVATE
).all()

phase32_proxy_elapsed = (
    time.perf_counter()
    - phase32_proxy_started
)

print(
    "Phase32 nested P3CA proxy complete: "
    f"{phase32_proxy_elapsed:.2f}s"
)

Phase32 fold 0 selected: p3ca_summary, PCA=96, C=0.01, monitor=0.344116, robust=0.370801, outer=0.417550, auroc=0.903958
Phase32 fold 1 selected: p3ca_summary_plus_low_frequency, PCA=16, C=0.01, monitor=0.385946, robust=0.383635, outer=0.385049, auroc=0.916585
Phase32 fold 2 selected: p3ca_low_frequency_sequence, PCA=96, C=0.01, monitor=0.272431, robust=0.320993, outer=0.399488, auroc=0.906743
Phase32 nested P3CA proxy complete: 17.36s


In [37]:
# Phase32 Cell 107C
# Final standalone evaluation plus non-deployable blend diagnostic.
#
# Outer labels are used here only for aggregate evaluation.
# Diagnostic blend alphas must not be deployed.

def phase32_logit(probabilities):
    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    return np.log(
        probabilities
        / (1.0 - probabilities)
    )


def phase32_sigmoid(logits):
    logits = np.clip(
        np.asarray(
            logits,
            dtype=np.float64,
        ),
        -30.0,
        30.0,
    )

    return 1.0 / (
        1.0 + np.exp(-logits)
    )


baseline_metrics = phase32_metrics(
    PHASE32_LABELS,
    PHASE32_BASELINE_OOF_PRIVATE,
)

proxy_metrics = phase32_metrics(
    PHASE32_LABELS,
    PHASE32_P3CA_PROXY_OOF_PRIVATE,
)

baseline_logits = phase32_logit(
    PHASE32_BASELINE_OOF_PRIVATE
)

proxy_logits = phase32_logit(
    PHASE32_P3CA_PROXY_OOF_PRIVATE
)

diagnostic_alphas = np.asarray([
    0.0,
    0.025,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.30,
    0.40,
])

global_diagnostics = []

for alpha in diagnostic_alphas:
    probability = phase32_sigmoid(
        baseline_logits
        + float(alpha)
        * (
            proxy_logits
            - baseline_logits
        )
    )

    metrics = phase32_metrics(
        PHASE32_LABELS,
        probability,
    )

    global_diagnostics.append({
        "alpha": float(alpha),
        **metrics,
    })

global_diagnostics.sort(
    key=lambda record: (
        record["log_loss"],
        record["alpha"],
    )
)

global_oracle = global_diagnostics[0]

fold_diagnostics = []

for fold in range(3):
    outer_indices = np.asarray(
        PHASE32_PARTITIONS[fold][
            "outer_valid"
        ],
        dtype=np.int64,
    )

    fold_baseline = (
        PHASE32_BASELINE_OOF_PRIVATE[
            outer_indices
        ]
    )

    fold_proxy = (
        PHASE32_P3CA_PROXY_OOF_PRIVATE[
            outer_indices
        ]
    )

    fold_labels = PHASE32_LABELS[
        outer_indices
    ]

    fold_baseline_metrics = (
        phase32_metrics(
            fold_labels,
            fold_baseline,
        )
    )

    fold_proxy_metrics = phase32_metrics(
        fold_labels,
        fold_proxy,
    )

    fold_baseline_logits = (
        phase32_logit(fold_baseline)
    )
    fold_proxy_logits = phase32_logit(
        fold_proxy
    )

    fold_candidates = []

    for alpha in diagnostic_alphas:
        probability = phase32_sigmoid(
            fold_baseline_logits
            + float(alpha)
            * (
                fold_proxy_logits
                - fold_baseline_logits
            )
        )

        metrics = phase32_metrics(
            fold_labels,
            probability,
        )

        fold_candidates.append({
            "alpha": float(alpha),
            **metrics,
        })

    fold_candidates.sort(
        key=lambda record: (
            record["log_loss"],
            record["alpha"],
        )
    )

    fold_oracle = fold_candidates[0]

    residual = (
        fold_labels
        - fold_baseline
    )

    update = (
        fold_proxy_logits
        - fold_baseline_logits
    )

    if (
        np.std(residual) > 0
        and np.std(update) > 0
    ):
        residual_update_correlation = float(
            np.corrcoef(
                residual,
                update,
            )[0, 1]
        )
    else:
        residual_update_correlation = 0.0

    fold_diagnostics.append({
        "fold": fold,
        "n": int(len(outer_indices)),
        "baseline_log_loss": (
            fold_baseline_metrics[
                "log_loss"
            ]
        ),
        "baseline_auroc": (
            fold_baseline_metrics["auroc"]
        ),
        "proxy_log_loss": (
            fold_proxy_metrics["log_loss"]
        ),
        "proxy_auroc": (
            fold_proxy_metrics["auroc"]
        ),
        "oracle_alpha_diagnostic_only": (
            fold_oracle["alpha"]
        ),
        "oracle_log_loss": (
            fold_oracle["log_loss"]
        ),
        "oracle_log_loss_gain": (
            fold_baseline_metrics[
                "log_loss"
            ]
            - fold_oracle["log_loss"]
        ),
        "oracle_auroc": (
            fold_oracle["auroc"]
        ),
        "baseline_residual_update_correlation": (
            residual_update_correlation
        ),
    })

fold_count_with_oracle_gain = int(
    sum(
        record["oracle_log_loss_gain"]
        >= 0.002
        for record in fold_diagnostics
    )
)

global_oracle_gain = (
    baseline_metrics["log_loss"]
    - global_oracle["log_loss"]
)

component_gate_passed = bool(
    proxy_metrics["auroc"] >= 0.91
    and global_oracle_gain >= 0.002
    and fold_count_with_oracle_gain >= 1
)

report = {
    "phase": "phase32_p3ca_nested_proxy",
    "status": (
        "component_gate_passed"
        if component_gate_passed
        else "component_gate_failed"
    ),
    "baseline": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in (
            baseline_metrics.items()
        )
    },
    "p3ca_proxy": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in proxy_metrics.items()
    },
    "standalone_improvements": {
        "log_loss_gain": round(
            baseline_metrics["log_loss"]
            - proxy_metrics["log_loss"],
            6,
        ),
        "auroc_gain": round(
            proxy_metrics["auroc"]
            - baseline_metrics["auroc"],
            6,
        ),
        "brier_excess": round(
            proxy_metrics["brier"]
            - baseline_metrics["brier"],
            6,
        ),
    },
    "global_blend_oracle_diagnostic_only": {
        "alpha": (
            global_oracle["alpha"]
        ),
        "log_loss": round(
            global_oracle["log_loss"],
            6,
        ),
        "auroc": round(
            global_oracle["auroc"],
            6,
        ),
        "brier": round(
            global_oracle["brier"],
            6,
        ),
        "log_loss_gain": round(
            global_oracle_gain,
            6,
        ),
    },
    "fold_diagnostics": [
        {
            key: (
                round(value, 6)
                if isinstance(
                    value,
                    (float, np.floating),
                )
                else value
            )
            for key, value in record.items()
        }
        for record in fold_diagnostics
    ],
    "selection": phase32_proxy_fold_records,
    "gates": {
        "component_gate_passed": (
            component_gate_passed
        ),
        "minimum_proxy_auroc": 0.91,
        "minimum_global_oracle_gain": 0.002,
        "minimum_fold_count_with_gain": 1,
        "fold_count_with_oracle_gain": (
            fold_count_with_oracle_gain
        ),
        "component_gate_meaning": (
            "eligible for a nested geometry-semantic "
            "sequence model and deployable blend test"
        ),
    },
    "elapsed_seconds": round(
        phase32_proxy_elapsed,
        2,
    ),
    "monitor_selection_group_robust": True,
    "fit_basis_fit_partition_only": True,
    "outer_train_basis_refitted": True,
    "outer_oracle_alpha_deployable": False,
    "outer_labels_used_only_for_final_evaluation": True,
    "case_level_predictions_exported": False,
    "persistent_notebook_cache_only": True,
    "include_private_oof_in_submission": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE32_PROXY_OOF"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_PROXY_OOF"
)

BEGIN SANITIZED_PHASE32_PROXY_OOF
{
  "phase": "phase32_p3ca_nested_proxy",
  "status": "component_gate_failed",
  "baseline": {
    "log_loss": 0.307616,
    "brier": 0.095535,
    "mean_probability": 0.530698,
    "auroc": 0.938914
  },
  "p3ca_proxy": {
    "log_loss": 0.400984,
    "brier": 0.129149,
    "mean_probability": 0.611256,
    "auroc": 0.907395
  },
  "standalone_improvements": {
    "log_loss_gain": -0.093369,
    "auroc_gain": -0.031519,
    "brier_excess": 0.033614
  },
  "global_blend_oracle_diagnostic_only": {
    "alpha": 0.2,
    "log_loss": 0.301253,
    "auroc": 0.940506,
    "brier": 0.093506,
    "log_loss_gain": 0.006363
  },
  "fold_diagnostics": [
    {
      "fold": 0,
      "n": 467,
      "baseline_log_loss": 0.358496,
      "baseline_auroc": 0.916117,
      "proxy_log_loss": 0.41755,
      "proxy_auroc": 0.903958,
      "oracle_alpha_diagnostic_only": 0.3,
      "oracle_log_loss": 0.347555,
      "oracle_log_loss_gain": 0.01094,
      "oracle_auroc": 0.

In [38]:
# Phase32 Cell 108A
# Acquisition-group cross-fitted calibration for the selected P3CA proxy.
#
# Selection stage:
#   P3CA basis = fit only
#   component model = fit only
#   calibration = leave-one-fit-group-out predictions
#   evaluation = monitor
#
# Final stage:
#   P3CA basis = outer_train
#   component model = outer_train
#   calibration = leave-one-outer-train-group-out predictions
#   evaluation = outer_valid

from sklearn.linear_model import LogisticRegression

import json
import time


def phase32_fit_selected_proxy(
    features,
    labels,
    specification,
    seed,
):
    scaler = StandardScaler()

    scaled = scaler.fit_transform(
        features
    )

    dimension = min(
        int(
            specification[
                "pca_dimension"
            ]
        ),
        scaled.shape[0] - 1,
        scaled.shape[1],
    )

    assert dimension >= 2

    projector = PCA(
        n_components=dimension,
        svd_solver="randomized",
        iterated_power=4,
        random_state=seed,
        whiten=False,
    )

    transformed = projector.fit_transform(
        scaled
    )

    classifier = LogisticRegression(
        C=float(
            specification[
                "logistic_c"
            ]
        ),
        solver="lbfgs",
        max_iter=4000,
        random_state=seed,
    )

    classifier.fit(
        transformed,
        labels,
    )

    return {
        "scaler": scaler,
        "projector": projector,
        "classifier": classifier,
    }


def phase32_proxy_predict(
    model,
    features,
):
    transformed = (
        model["projector"].transform(
            model["scaler"].transform(
                features
            )
        )
    )

    probability = (
        model["classifier"].predict_proba(
            transformed
        )[:, 1]
    )

    return np.clip(
        probability,
        1e-6,
        1.0 - 1e-6,
    )


def phase32_group_cross_fitted_probability(
    features,
    labels,
    groups,
    specification,
    seed,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    groups = np.asarray(groups)

    probability = np.full(
        len(labels),
        np.nan,
        dtype=np.float64,
    )

    coverage = np.zeros(
        len(labels),
        dtype=np.int64,
    )

    unique_groups = np.unique(groups)
    records = []

    for split_index, held_group in enumerate(
        unique_groups
    ):
        validation_mask = (
            groups == held_group
        )
        training_mask = (
            ~validation_mask
        )

        assert training_mask.sum() > 1
        assert validation_mask.sum() > 0
        assert len(
            np.unique(
                labels[training_mask]
            )
        ) == 2

        model = phase32_fit_selected_proxy(
            features[training_mask],
            labels[training_mask],
            specification,
            seed + split_index,
        )

        held_probability = (
            phase32_proxy_predict(
                model,
                features[validation_mask],
            )
        )

        probability[
            validation_mask
        ] = held_probability

        coverage[
            validation_mask
        ] += 1

        held_metrics = phase32_metrics(
            labels[validation_mask],
            held_probability,
        )

        records.append({
            "held_group_size": int(
                validation_mask.sum()
            ),
            "held_group_log_loss": float(
                held_metrics["log_loss"]
            ),
            "held_group_auroc": (
                held_metrics["auroc"]
            ),
        })

    assert np.all(coverage == 1)
    assert np.isfinite(probability).all()

    return probability, records


def phase32_fit_group_balanced_platt(
    raw_probability,
    labels,
    groups,
    seed,
):
    raw_probability = np.clip(
        np.asarray(
            raw_probability,
            dtype=np.float64,
        ),
        1e-5,
        1.0 - 1e-5,
    )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    groups = np.asarray(groups)

    raw_logit = phase32_logit(
        raw_probability
    )[:, None]

    unique_groups, group_counts = (
        np.unique(
            groups,
            return_counts=True,
        )
    )

    count_by_group = {
        group: count
        for group, count in zip(
            unique_groups,
            group_counts,
        )
    }

    sample_weight = np.asarray(
        [
            1.0 / count_by_group[group]
            for group in groups
        ],
        dtype=np.float64,
    )

    sample_weight *= (
        len(sample_weight)
        / sample_weight.sum()
    )

    calibrator = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=4000,
        random_state=seed,
    )

    calibrator.fit(
        raw_logit,
        labels,
        sample_weight=sample_weight,
    )

    return calibrator


def phase32_apply_platt(
    calibrator,
    raw_probability,
):
    raw_logit = phase32_logit(
        raw_probability
    )[:, None]

    calibrated = (
        calibrator.predict_proba(
            raw_logit
        )[:, 1]
    )

    return np.clip(
        calibrated,
        1e-6,
        1.0 - 1e-6,
    )


phase32_calibration_started = (
    time.perf_counter()
)

PHASE32_CALIBRATED_MONITOR_PRIVATE = {}
PHASE32_CALIBRATED_COMPONENT_OOF_PRIVATE = (
    np.full(
        EXPECTED_CASES,
        np.nan,
        dtype=np.float64,
    )
)

PHASE32_CALIBRATED_COMPONENT_MODELS_PRIVATE = []
phase32_calibration_records = []
phase32_calibrated_coverage = np.zeros(
    EXPECTED_CASES,
    dtype=np.int64,
)

for fold in range(3):
    partition = PHASE32_PARTITIONS[fold]

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    specification = (
        phase32_proxy_fold_records[
            fold
        ]["selected_specification"]
    )

    representation_name = (
        specification["representation"]
    )

    seed = (
        PHASE32_PROXY_CONFIG["base_seed"]
        + 1000
        + fold * 100
    )

    # ---------------------------
    # Fit -> monitor stage
    # ---------------------------
    fit_features = (
        phase32_representation(
            stage_index=0,
            fold=fold,
            indices=fit_indices,
            representation_name=(
                representation_name
            ),
        )
    )

    monitor_features = (
        phase32_representation(
            stage_index=0,
            fold=fold,
            indices=monitor_indices,
            representation_name=(
                representation_name
            ),
        )
    )

    fit_cross_probability, fit_group_records = (
        phase32_group_cross_fitted_probability(
            fit_features,
            PHASE32_LABELS[fit_indices],
            PHASE32_GROUPS[fit_indices],
            specification,
            seed,
        )
    )

    monitor_model = (
        phase32_fit_selected_proxy(
            fit_features,
            PHASE32_LABELS[fit_indices],
            specification,
            seed + 50,
        )
    )

    monitor_raw_probability = (
        phase32_proxy_predict(
            monitor_model,
            monitor_features,
        )
    )

    monitor_calibrator = (
        phase32_fit_group_balanced_platt(
            fit_cross_probability,
            PHASE32_LABELS[fit_indices],
            PHASE32_GROUPS[fit_indices],
            seed + 60,
        )
    )

    monitor_calibrated_probability = (
        phase32_apply_platt(
            monitor_calibrator,
            monitor_raw_probability,
        )
    )

    PHASE32_CALIBRATED_MONITOR_PRIVATE[
        fold
    ] = monitor_calibrated_probability

    # ---------------------------
    # Outer-train -> outer-valid
    # ---------------------------
    outer_train_features = (
        phase32_representation(
            stage_index=1,
            fold=fold,
            indices=outer_train_indices,
            representation_name=(
                representation_name
            ),
        )
    )

    outer_valid_features = (
        phase32_representation(
            stage_index=1,
            fold=fold,
            indices=outer_valid_indices,
            representation_name=(
                representation_name
            ),
        )
    )

    (
        outer_train_cross_probability,
        outer_group_records,
    ) = phase32_group_cross_fitted_probability(
        outer_train_features,
        PHASE32_LABELS[
            outer_train_indices
        ],
        PHASE32_GROUPS[
            outer_train_indices
        ],
        specification,
        seed + 100,
    )

    final_model = (
        phase32_fit_selected_proxy(
            outer_train_features,
            PHASE32_LABELS[
                outer_train_indices
            ],
            specification,
            seed + 150,
        )
    )

    outer_raw_probability = (
        phase32_proxy_predict(
            final_model,
            outer_valid_features,
        )
    )

    final_calibrator = (
        phase32_fit_group_balanced_platt(
            outer_train_cross_probability,
            PHASE32_LABELS[
                outer_train_indices
            ],
            PHASE32_GROUPS[
                outer_train_indices
            ],
            seed + 160,
        )
    )

    outer_calibrated_probability = (
        phase32_apply_platt(
            final_calibrator,
            outer_raw_probability,
        )
    )

    PHASE32_CALIBRATED_COMPONENT_OOF_PRIVATE[
        outer_valid_indices
    ] = outer_calibrated_probability

    phase32_calibrated_coverage[
        outer_valid_indices
    ] += 1

    monitor_metrics = phase32_metrics(
        PHASE32_LABELS[
            monitor_indices
        ],
        monitor_calibrated_probability,
    )

    outer_metrics = phase32_metrics(
        PHASE32_LABELS[
            outer_valid_indices
        ],
        outer_calibrated_probability,
    )

    PHASE32_CALIBRATED_COMPONENT_MODELS_PRIVATE.append({
        "fold": fold,
        "specification": dict(
            specification
        ),
        "component_model": final_model,
        "calibrator": final_calibrator,
    })

    phase32_calibration_records.append({
        "fold": fold,
        "representation": (
            representation_name
        ),
        "fit_group_count": int(
            len(
                np.unique(
                    PHASE32_GROUPS[
                        fit_indices
                    ]
                )
            )
        ),
        "outer_train_group_count": int(
            len(
                np.unique(
                    PHASE32_GROUPS[
                        outer_train_indices
                    ]
                )
            )
        ),
        "monitor_metrics": (
            monitor_metrics
        ),
        "outer_metrics": outer_metrics,
        "monitor_platt_intercept": float(
            monitor_calibrator.intercept_[0]
        ),
        "monitor_platt_slope": float(
            monitor_calibrator.coef_[0, 0]
        ),
        "final_platt_intercept": float(
            final_calibrator.intercept_[0]
        ),
        "final_platt_slope": float(
            final_calibrator.coef_[0, 0]
        ),
        "fit_cross_group_records": (
            fit_group_records
        ),
        "outer_train_cross_group_records": (
            outer_group_records
        ),
    })

    print(
        f"Phase32 calibrated fold {fold}: "
        f"monitor={monitor_metrics['log_loss']:.6f}, "
        f"outer={outer_metrics['log_loss']:.6f}, "
        f"auroc={outer_metrics['auroc']:.6f}, "
        f"slope={final_calibrator.coef_[0, 0]:.4f}"
    )

assert np.all(
    phase32_calibrated_coverage == 1
)
assert np.isfinite(
    PHASE32_CALIBRATED_COMPONENT_OOF_PRIVATE
).all()

calibrated_oof_metrics = (
    phase32_metrics(
        PHASE32_LABELS,
        PHASE32_CALIBRATED_COMPONENT_OOF_PRIVATE,
    )
)

report = {
    "phase": (
        "phase32_group_cross_fitted_calibration"
    ),
    "status": "complete",
    "fold_count": 3,
    "calibration_method": (
        "leave_one_acquisition_group_out_"
        "predictions_plus_group_balanced_platt"
    ),
    "calibrated_component_oof": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in (
            calibrated_oof_metrics.items()
        )
    },
    "folds": [
        {
            "fold": record["fold"],
            "representation": (
                record["representation"]
            ),
            "fit_group_count": (
                record["fit_group_count"]
            ),
            "outer_train_group_count": (
                record[
                    "outer_train_group_count"
                ]
            ),
            "monitor_log_loss": round(
                record["monitor_metrics"][
                    "log_loss"
                ],
                6,
            ),
            "monitor_auroc": round(
                record["monitor_metrics"][
                    "auroc"
                ],
                6,
            ),
            "outer_log_loss": round(
                record["outer_metrics"][
                    "log_loss"
                ],
                6,
            ),
            "outer_auroc": round(
                record["outer_metrics"][
                    "auroc"
                ],
                6,
            ),
            "monitor_platt_intercept": round(
                record[
                    "monitor_platt_intercept"
                ],
                6,
            ),
            "monitor_platt_slope": round(
                record[
                    "monitor_platt_slope"
                ],
                6,
            ),
            "final_platt_intercept": round(
                record[
                    "final_platt_intercept"
                ],
                6,
            ),
            "final_platt_slope": round(
                record[
                    "final_platt_slope"
                ],
                6,
            ),
        }
        for record in (
            phase32_calibration_records
        )
    ],
    "elapsed_seconds": round(
        time.perf_counter()
        - phase32_calibration_started,
        2,
    ),
    "outer_validation_labels_used_for_calibration": False,
    "monitor_labels_used_for_calibration": False,
    "acquisition_groups_used_for_cross_fitting": True,
    "case_level_predictions_exported": False,
    "persistent_notebook_cache_only": True,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE32_GROUP_CALIBRATION"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_GROUP_CALIBRATION"
)

Phase32 calibrated fold 0: monitor=0.338905, outer=0.421110, auroc=0.903295, slope=0.9598
Phase32 calibrated fold 1: monitor=0.382877, outer=0.378200, auroc=0.917666, slope=1.0325
Phase32 calibrated fold 2: monitor=0.306967, outer=0.383184, auroc=0.907826, slope=0.8820
BEGIN SANITIZED_PHASE32_GROUP_CALIBRATION
{
  "phase": "phase32_group_cross_fitted_calibration",
  "status": "complete",
  "fold_count": 3,
  "calibration_method": "leave_one_acquisition_group_out_predictions_plus_group_balanced_platt",
  "calibrated_component_oof": {
    "log_loss": 0.394567,
    "brier": 0.12664,
    "mean_probability": 0.597692,
    "auroc": 0.907219
  },
  "folds": [
    {
      "fold": 0,
      "representation": "p3ca_summary",
      "fit_group_count": 8,
      "outer_train_group_count": 12,
      "monitor_log_loss": 0.338905,
      "monitor_auroc": 0.940156,
      "outer_log_loss": 0.42111,
      "outer_auroc": 0.903295,
      "monitor_platt_intercept": -0.096914,
      "monitor_platt_slope": 1.009

In [39]:
# Phase32 Cell 108B
# Select a conservative P3CA residual correction using monitor only.
#
# z_final = z_baseline
#         + alpha * uncertainty^gamma
#         * clip(z_component-z_baseline, -tau, tau)

PHASE32_GATE_CONFIG = {
    "alphas": [
        0.0,
        0.025,
        0.05,
        0.075,
        0.10,
        0.15,
        0.20,
        0.30,
        0.40,
    ],
    "uncertainty_exponents": [
        0.0,
        0.5,
        1.0,
        2.0,
    ],
    "delta_caps": [
        0.5,
        1.0,
        2.0,
        4.0,
        8.0,
    ],
    "minimum_overall_monitor_gain": 0.001,
    "minimum_robust_monitor_gain": 0.001,
    "maximum_worst_group_regret": 0.01,
}


def phase32_bounded_gate(
    baseline_probability,
    component_probability,
    alpha,
    uncertainty_exponent,
    delta_cap,
):
    baseline_probability = np.clip(
        np.asarray(
            baseline_probability,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    component_probability = np.clip(
        np.asarray(
            component_probability,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    baseline_logit = phase32_logit(
        baseline_probability
    )

    component_logit = phase32_logit(
        component_probability
    )

    uncertainty = (
        4.0
        * baseline_probability
        * (1.0 - baseline_probability)
    ) ** float(uncertainty_exponent)

    bounded_delta = np.clip(
        component_logit
        - baseline_logit,
        -float(delta_cap),
        float(delta_cap),
    )

    final_logit = (
        baseline_logit
        + float(alpha)
        * uncertainty
        * bounded_delta
    )

    return phase32_sigmoid(
        final_logit
    )


PHASE32_GATE_SELECTION_PRIVATE = []
PHASE32_GATED_OOF_PRIVATE = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

phase32_gate_coverage = np.zeros(
    EXPECTED_CASES,
    dtype=np.int64,
)

for fold in range(3):
    partition = PHASE32_PARTITIONS[fold]

    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )

    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    monitor_labels = PHASE32_LABELS[
        monitor_indices
    ]

    monitor_groups = PHASE32_GROUPS[
        monitor_indices
    ]

    baseline_monitor = np.asarray(
        PHASE32_PHASE12C_FOLD_PROBABILITIES_PRIVATE[
            fold,
            monitor_indices,
        ],
        dtype=np.float64,
    )

    component_monitor = np.asarray(
        PHASE32_CALIBRATED_MONITOR_PRIVATE[
            fold
        ],
        dtype=np.float64,
    )

    baseline_monitor_robust = (
        phase32_robust_monitor_score(
            monitor_labels,
            baseline_monitor,
            monitor_groups,
        )
    )

    candidate_records = []

    # Unique identity candidate.
    candidate_records.append({
        "alpha": 0.0,
        "uncertainty_exponent": 0.0,
        "delta_cap": 0.0,
        **baseline_monitor_robust,
    })

    for alpha in (
        PHASE32_GATE_CONFIG["alphas"]
    ):
        if alpha == 0.0:
            continue

        for exponent in (
            PHASE32_GATE_CONFIG[
                "uncertainty_exponents"
            ]
        ):
            for delta_cap in (
                PHASE32_GATE_CONFIG[
                    "delta_caps"
                ]
            ):
                probability = (
                    phase32_bounded_gate(
                        baseline_monitor,
                        component_monitor,
                        alpha,
                        exponent,
                        delta_cap,
                    )
                )

                robust_record = (
                    phase32_robust_monitor_score(
                        monitor_labels,
                        probability,
                        monitor_groups,
                    )
                )

                candidate_records.append({
                    "alpha": float(alpha),
                    "uncertainty_exponent": float(
                        exponent
                    ),
                    "delta_cap": float(
                        delta_cap
                    ),
                    **robust_record,
                })

    candidate_records.sort(
        key=lambda record: (
            record["robust_score"],
            record["overall_log_loss"],
            record["alpha"],
            record[
                "uncertainty_exponent"
            ],
            record["delta_cap"],
        )
    )

    raw_best = candidate_records[0]

    overall_gain = (
        baseline_monitor_robust[
            "overall_log_loss"
        ]
        - raw_best["overall_log_loss"]
    )

    robust_gain = (
        baseline_monitor_robust[
            "robust_score"
        ]
        - raw_best["robust_score"]
    )

    worst_group_regret = (
        raw_best["worst_group_log_loss"]
        - baseline_monitor_robust[
            "worst_group_log_loss"
        ]
    )

    advanced = bool(
        raw_best["alpha"] > 0.0
        and overall_gain
        >= PHASE32_GATE_CONFIG[
            "minimum_overall_monitor_gain"
        ]
        and robust_gain
        >= PHASE32_GATE_CONFIG[
            "minimum_robust_monitor_gain"
        ]
        and worst_group_regret
        <= PHASE32_GATE_CONFIG[
            "maximum_worst_group_regret"
        ]
    )

    if advanced:
        selected = raw_best
    else:
        selected = candidate_records[
            next(
                index
                for index, record in enumerate(
                    candidate_records
                )
                if record["alpha"] == 0.0
            )
        ]

    baseline_outer = (
        PHASE32_BASELINE_OOF_PRIVATE[
            outer_valid_indices
        ]
    )

    component_outer = (
        PHASE32_CALIBRATED_COMPONENT_OOF_PRIVATE[
            outer_valid_indices
        ]
    )

    outer_probability = (
        phase32_bounded_gate(
            baseline_outer,
            component_outer,
            selected["alpha"],
            selected[
                "uncertainty_exponent"
            ],
            (
                selected["delta_cap"]
                if selected["alpha"] > 0
                else 1.0
            ),
        )
    )

    PHASE32_GATED_OOF_PRIVATE[
        outer_valid_indices
    ] = outer_probability

    phase32_gate_coverage[
        outer_valid_indices
    ] += 1

    PHASE32_GATE_SELECTION_PRIVATE.append({
        "fold": fold,
        "candidate_count": int(
            len(candidate_records)
        ),
        "baseline_monitor": (
            baseline_monitor_robust
        ),
        "raw_best": raw_best,
        "overall_monitor_gain": float(
            overall_gain
        ),
        "robust_monitor_gain": float(
            robust_gain
        ),
        "worst_group_regret": float(
            worst_group_regret
        ),
        "advanced": advanced,
        "selected": dict(selected),
    })

    print(
        f"Phase32 gate fold {fold}: "
        f"alpha={selected['alpha']:.3f}, "
        f"gamma="
        f"{selected['uncertainty_exponent']:.2f}, "
        f"cap={selected['delta_cap']:.2f}, "
        f"advanced={advanced}, "
        f"raw_gain={overall_gain:.6f}, "
        f"robust_gain={robust_gain:.6f}"
    )

assert np.all(
    phase32_gate_coverage == 1
)
assert np.isfinite(
    PHASE32_GATED_OOF_PRIVATE
).all()

print(
    "Phase32 bounded gate selection complete."
)

Phase32 gate fold 0: alpha=0.400, gamma=0.50, cap=2.00, advanced=True, raw_gain=0.018045, robust_gain=0.023329
Phase32 gate fold 1: alpha=0.000, gamma=0.00, cap=0.00, advanced=False, raw_gain=0.000000, robust_gain=0.000000
Phase32 gate fold 2: alpha=0.000, gamma=0.00, cap=0.00, advanced=False, raw_gain=0.000579, robust_gain=0.000233
Phase32 bounded gate selection complete.


In [40]:
# Phase32 Cell 108C
# Evaluate the fully nested, monitor-selected correction.

baseline_metrics = phase32_metrics(
    PHASE32_LABELS,
    PHASE32_BASELINE_OOF_PRIVATE,
)

component_metrics = phase32_metrics(
    PHASE32_LABELS,
    PHASE32_CALIBRATED_COMPONENT_OOF_PRIVATE,
)

gated_metrics = phase32_metrics(
    PHASE32_LABELS,
    PHASE32_GATED_OOF_PRIVATE,
)

fold_records = []

for fold in range(3):
    outer_indices = np.asarray(
        PHASE32_PARTITIONS[fold][
            "outer_valid"
        ],
        dtype=np.int64,
    )

    baseline_fold_metrics = (
        phase32_metrics(
            PHASE32_LABELS[
                outer_indices
            ],
            PHASE32_BASELINE_OOF_PRIVATE[
                outer_indices
            ],
        )
    )

    gated_fold_metrics = (
        phase32_metrics(
            PHASE32_LABELS[
                outer_indices
            ],
            PHASE32_GATED_OOF_PRIVATE[
                outer_indices
            ],
        )
    )

    selection = (
        PHASE32_GATE_SELECTION_PRIVATE[
            fold
        ]
    )

    fold_records.append({
        "fold": fold,
        "n": int(len(outer_indices)),
        "advanced": (
            selection["advanced"]
        ),
        "selected_alpha": (
            selection["selected"][
                "alpha"
            ]
        ),
        "selected_uncertainty_exponent": (
            selection["selected"][
                "uncertainty_exponent"
            ]
        ),
        "selected_delta_cap": (
            selection["selected"][
                "delta_cap"
            ]
        ),
        "monitor_overall_gain": (
            selection[
                "overall_monitor_gain"
            ]
        ),
        "monitor_robust_gain": (
            selection[
                "robust_monitor_gain"
            ]
        ),
        "baseline_log_loss": (
            baseline_fold_metrics[
                "log_loss"
            ]
        ),
        "gated_log_loss": (
            gated_fold_metrics[
                "log_loss"
            ]
        ),
        "log_loss_improvement": (
            baseline_fold_metrics[
                "log_loss"
            ]
            - gated_fold_metrics[
                "log_loss"
            ]
        ),
        "baseline_auroc": (
            baseline_fold_metrics["auroc"]
        ),
        "gated_auroc": (
            gated_fold_metrics["auroc"]
        ),
    })

major_group_records = []

for group in np.unique(
    PHASE32_GROUPS
):
    mask = (
        PHASE32_GROUPS == group
    )

    if mask.sum() < 30:
        continue

    baseline_group_metrics = (
        phase32_metrics(
            PHASE32_LABELS[mask],
            PHASE32_BASELINE_OOF_PRIVATE[
                mask
            ],
        )
    )

    gated_group_metrics = (
        phase32_metrics(
            PHASE32_LABELS[mask],
            PHASE32_GATED_OOF_PRIVATE[
                mask
            ],
        )
    )

    major_group_records.append({
        "group": int(group),
        "n": int(mask.sum()),
        "baseline_log_loss": (
            baseline_group_metrics[
                "log_loss"
            ]
        ),
        "gated_log_loss": (
            gated_group_metrics[
                "log_loss"
            ]
        ),
        "log_loss_change": (
            gated_group_metrics[
                "log_loss"
            ]
            - baseline_group_metrics[
                "log_loss"
            ]
        ),
    })

log_loss_gain = (
    baseline_metrics["log_loss"]
    - gated_metrics["log_loss"]
)

auroc_gain = (
    gated_metrics["auroc"]
    - baseline_metrics["auroc"]
)

brier_excess = (
    gated_metrics["brier"]
    - baseline_metrics["brier"]
)

fold_wins = int(
    sum(
        record[
            "log_loss_improvement"
        ] > 0
        for record in fold_records
    )
)

advanced_fold_count = int(
    sum(
        record["advanced"]
        for record in fold_records
    )
)

worst_fold_excess = float(
    max(
        -record[
            "log_loss_improvement"
        ]
        for record in fold_records
    )
)

maximum_major_group_harm = float(
    max(
        record["log_loss_change"]
        for record in major_group_records
    )
)

promotion_gate_passed = bool(
    log_loss_gain >= 0.003
    and auroc_gain >= 0.001
    and fold_wins >= 2
    and worst_fold_excess <= 0.005
    and brier_excess <= 0.001
    and maximum_major_group_harm <= 0.015
)

report = {
    "phase": (
        "phase32_calibrated_bounded_p3ca_residual"
    ),
    "status": (
        "promoted"
        if promotion_gate_passed
        else "not_promoted"
    ),
    "baseline": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in (
            baseline_metrics.items()
        )
    },
    "calibrated_p3ca_component": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in (
            component_metrics.items()
        )
    },
    "phase32_gated": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in (
            gated_metrics.items()
        )
    },
    "improvements": {
        "log_loss_gain": round(
            log_loss_gain,
            6,
        ),
        "auroc_gain": round(
            auroc_gain,
            6,
        ),
        "brier_excess": round(
            brier_excess,
            6,
        ),
        "fold_wins": fold_wins,
        "advanced_fold_count": (
            advanced_fold_count
        ),
        "worst_fold_excess": round(
            worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            maximum_major_group_harm,
            6,
        ),
    },
    "fold_metrics": [
        {
            key: (
                round(value, 6)
                if isinstance(
                    value,
                    (float, np.floating),
                )
                else value
            )
            for key, value in record.items()
        }
        for record in fold_records
    ],
    "major_acquisition_group_metrics": [
        {
            key: (
                round(value, 6)
                if isinstance(
                    value,
                    (float, np.floating),
                )
                else value
            )
            for key, value in record.items()
        }
        for record in major_group_records
    ],
    "promotion_gate_passed": (
        promotion_gate_passed
    ),
    "selection_partition": "monitor_only",
    "outer_oracle_parameters_used": False,
    "group_cross_fitted_calibration": True,
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE32_GATED_OOF"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE32_GATED_OOF"
)

BEGIN SANITIZED_PHASE32_GATED_OOF
{
  "phase": "phase32_calibrated_bounded_p3ca_residual",
  "status": "not_promoted",
  "baseline": {
    "log_loss": 0.307616,
    "brier": 0.095535,
    "mean_probability": 0.530698,
    "auroc": 0.938914
  },
  "calibrated_p3ca_component": {
    "log_loss": 0.394567,
    "brier": 0.12664,
    "mean_probability": 0.597692,
    "auroc": 0.907219
  },
  "phase32_gated": {
    "log_loss": 0.305101,
    "brier": 0.094599,
    "mean_probability": 0.539926,
    "auroc": 0.939291
  },
  "improvements": {
    "log_loss_gain": 0.002514,
    "auroc_gain": 0.000377,
    "brier_excess": -0.000937,
    "fold_wins": 1,
    "advanced_fold_count": 1,
    "worst_fold_excess": 0.0,
    "maximum_major_group_harm": 0.0
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "advanced": true,
      "selected_alpha": 0.4,
      "selected_uncertainty_exponent": 0.5,
      "selected_delta_cap": 2.0,
      "monitor_overall_gain": 0.018045,
      "monitor_robust_

In [41]:
# Phase33 Cell 109A
# Blockwise preprocessing for geometry-semantic sequence fusion.
#
# All transforms are fitted on a supplied training partition.
# This preflight fits only fold-0 fit and uses no labels.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import json
import time

PHASE33_CONFIG = {
    "phase": (
        "phase33_domain_invariant_geometry_semantic_fusion"
    ),
    "p3ca_summary_pca_dimension": 48,
    "radiomics_pca_dimension": 48,
    "p3ca_token_dimension": 48,
    "geometry_token_dimension": 52,
    "p3ca_token_count": 84,
    "geometry_token_count": 16,
    "view_count": 6,
    "positions_per_view": 14,
    "model_dimension": 64,
    "geometry_model_dimension": 48,
    "domain_class_count": int(
        np.max(PHASE32_GROUPS) + 1
    ),
    "base_seed": 330701,
}


def phase33_safe_standard_deviation(
    values,
    axis,
):
    standard_deviation = np.std(
        values,
        axis=axis,
        dtype=np.float64,
    ).astype(np.float32)

    return np.where(
        standard_deviation >= 1e-5,
        standard_deviation,
        1.0,
    ).astype(np.float32)


def phase33_fit_preprocessor(
    stage_index,
    fold,
    training_indices,
    seed,
):
    training_indices = np.asarray(
        training_indices,
        dtype=np.int64,
    )

    p3ca_sequence = np.asarray(
        PHASE32_P3CA_SEQUENCE_PRIVATE[
            stage_index,
            fold,
            training_indices,
        ],
        dtype=np.float32,
    )

    geometry_sequence = np.asarray(
        PHASE31_SEQUENCE_PRIVATE[
            training_indices
        ],
        dtype=np.float32,
    )

    p3ca_summary = np.asarray(
        PHASE32_P3CA_SUMMARY_PRIVATE[
            stage_index,
            fold,
            training_indices,
        ],
        dtype=np.float32,
    )

    radiomics = np.asarray(
        PHASE31_RADIOMICS_PRIVATE[
            training_indices
        ],
        dtype=np.float32,
    )

    assert p3ca_sequence.shape == (
        len(training_indices),
        84,
        48,
    )
    assert geometry_sequence.shape == (
        len(training_indices),
        16,
        52,
    )

    p3ca_token_mean = np.mean(
        p3ca_sequence,
        axis=(0, 1),
        dtype=np.float64,
    ).astype(np.float32)

    p3ca_token_standard_deviation = (
        phase33_safe_standard_deviation(
            p3ca_sequence,
            axis=(0, 1),
        )
    )

    geometry_token_mean = np.mean(
        geometry_sequence,
        axis=(0, 1),
        dtype=np.float64,
    ).astype(np.float32)

    geometry_token_standard_deviation = (
        phase33_safe_standard_deviation(
            geometry_sequence,
            axis=(0, 1),
        )
    )

    summary_scaler = StandardScaler()
    scaled_summary = (
        summary_scaler.fit_transform(
            p3ca_summary
        )
    )

    summary_dimension = min(
        PHASE33_CONFIG[
            "p3ca_summary_pca_dimension"
        ],
        scaled_summary.shape[0] - 1,
        scaled_summary.shape[1],
    )

    summary_projector = PCA(
        n_components=summary_dimension,
        svd_solver="randomized",
        iterated_power=4,
        random_state=seed,
        whiten=False,
    )

    summary_projector.fit(
        scaled_summary
    )

    radiomics_scaler = StandardScaler()
    scaled_radiomics = (
        radiomics_scaler.fit_transform(
            radiomics
        )
    )

    radiomics_dimension = min(
        PHASE33_CONFIG[
            "radiomics_pca_dimension"
        ],
        scaled_radiomics.shape[0] - 1,
        scaled_radiomics.shape[1],
    )

    radiomics_projector = PCA(
        n_components=radiomics_dimension,
        svd_solver="randomized",
        iterated_power=4,
        random_state=seed + 1,
        whiten=False,
    )

    radiomics_projector.fit(
        scaled_radiomics
    )

    return {
        "stage_index": int(stage_index),
        "fold": int(fold),
        "training_count": int(
            len(training_indices)
        ),
        "p3ca_token_mean": (
            p3ca_token_mean
        ),
        "p3ca_token_standard_deviation": (
            p3ca_token_standard_deviation
        ),
        "geometry_token_mean": (
            geometry_token_mean
        ),
        "geometry_token_standard_deviation": (
            geometry_token_standard_deviation
        ),
        "summary_scaler": summary_scaler,
        "summary_projector": (
            summary_projector
        ),
        "radiomics_scaler": (
            radiomics_scaler
        ),
        "radiomics_projector": (
            radiomics_projector
        ),
        "summary_dimension": int(
            summary_dimension
        ),
        "radiomics_dimension": int(
            radiomics_dimension
        ),
    }


def phase33_transform(
    preprocessor,
    indices,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    stage_index = preprocessor[
        "stage_index"
    ]
    fold = preprocessor["fold"]

    p3ca_sequence = np.asarray(
        PHASE32_P3CA_SEQUENCE_PRIVATE[
            stage_index,
            fold,
            indices,
        ],
        dtype=np.float32,
    )

    geometry_sequence = np.asarray(
        PHASE31_SEQUENCE_PRIVATE[
            indices
        ],
        dtype=np.float32,
    )

    p3ca_summary = np.asarray(
        PHASE32_P3CA_SUMMARY_PRIVATE[
            stage_index,
            fold,
            indices,
        ],
        dtype=np.float32,
    )

    radiomics = np.asarray(
        PHASE31_RADIOMICS_PRIVATE[
            indices
        ],
        dtype=np.float32,
    )

    p3ca_sequence = (
        p3ca_sequence
        - preprocessor[
            "p3ca_token_mean"
        ][None, None, :]
    ) / preprocessor[
        "p3ca_token_standard_deviation"
    ][None, None, :]

    geometry_sequence = (
        geometry_sequence
        - preprocessor[
            "geometry_token_mean"
        ][None, None, :]
    ) / preprocessor[
        "geometry_token_standard_deviation"
    ][None, None, :]

    summary_features = (
        preprocessor[
            "summary_projector"
        ].transform(
            preprocessor[
                "summary_scaler"
            ].transform(
                p3ca_summary
            )
        )
    )

    radiomics_features = (
        preprocessor[
            "radiomics_projector"
        ].transform(
            preprocessor[
                "radiomics_scaler"
            ].transform(
                radiomics
            )
        )
    )

    output = {
        "p3ca_sequence": np.asarray(
            p3ca_sequence,
            dtype=np.float32,
        ),
        "geometry_sequence": np.asarray(
            geometry_sequence,
            dtype=np.float32,
        ),
        "summary_features": np.asarray(
            summary_features,
            dtype=np.float32,
        ),
        "radiomics_features": np.asarray(
            radiomics_features,
            dtype=np.float32,
        ),
    }

    assert all(
        np.isfinite(value).all()
        for value in output.values()
    )

    return output


phase33_preflight_started = (
    time.perf_counter()
)

fold0_fit_indices = np.asarray(
    PHASE32_PARTITIONS[0]["fit"],
    dtype=np.int64,
)

PHASE33_FOLD0_FIT_PREPROCESSOR_PRIVATE = (
    phase33_fit_preprocessor(
        stage_index=0,
        fold=0,
        training_indices=(
            fold0_fit_indices
        ),
        seed=PHASE33_CONFIG[
            "base_seed"
        ],
    )
)

preflight_indices = fold0_fit_indices[:8]

phase33_preflight_features = (
    phase33_transform(
        PHASE33_FOLD0_FIT_PREPROCESSOR_PRIVATE,
        preflight_indices,
    )
)

report = {
    "phase": (
        "phase33_multimodal_preprocessing_contract"
    ),
    "status": "accepted",
    "fit_partition": "fold_0_fit_only",
    "fit_n": int(
        len(fold0_fit_indices)
    ),
    "transformed_shapes": {
        key: list(value.shape)
        for key, value in (
            phase33_preflight_features.items()
        )
    },
    "summary_explained_variance_fraction": round(
        float(
            PHASE33_FOLD0_FIT_PREPROCESSOR_PRIVATE[
                "summary_projector"
            ].explained_variance_ratio_.sum()
        ),
        6,
    ),
    "radiomics_explained_variance_fraction": round(
        float(
            PHASE33_FOLD0_FIT_PREPROCESSOR_PRIVATE[
                "radiomics_projector"
            ].explained_variance_ratio_.sum()
        ),
        6,
    ),
    "all_values_finite": True,
    "labels_used": False,
    "outer_validation_images_used": False,
    "outer_validation_labels_used": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase33_preflight_started,
        2,
    ),
    "persistent_state_retained_privately": True,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE33_PREPROCESSING"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE33_PREPROCESSING"
)

BEGIN SANITIZED_PHASE33_PREPROCESSING
{
  "phase": "phase33_multimodal_preprocessing_contract",
  "status": "accepted",
  "fit_partition": "fold_0_fit_only",
  "fit_n": 721,
  "transformed_shapes": {
    "p3ca_sequence": [
      8,
      84,
      48
    ],
    "geometry_sequence": [
      8,
      16,
      52
    ],
    "summary_features": [
      8,
      48
    ],
    "radiomics_features": [
      8,
      48
    ]
  },
  "summary_explained_variance_fraction": 0.743247,
  "radiomics_explained_variance_fraction": 0.88049,
  "all_values_finite": true,
  "labels_used": false,
  "outer_validation_images_used": false,
  "outer_validation_labels_used": false,
  "elapsed_seconds": 0.34,
  "persistent_state_retained_privately": true,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE33_PREPROCESSING


In [42]:
# Phase33 Cell 109B
# Low-capacity geometry-semantic sequence network.
#
# P3CA branch:
#   six independent 14-position sequences
#
# Geometry branch:
#   16 localized bilateral tokens
#
# Domain head:
#   gradient reversal discourages acquisition shortcuts

import torch
import torch.nn as nn
import torch.nn.functional as F


class Phase33GradientReverseFunction(
    torch.autograd.Function
):
    @staticmethod
    def forward(
        context,
        input_tensor,
        coefficient,
    ):
        context.coefficient = float(
            coefficient
        )

        return input_tensor.view_as(
            input_tensor
        )

    @staticmethod
    def backward(
        context,
        gradient_output,
    ):
        return (
            -context.coefficient
            * gradient_output,
            None,
        )


def phase33_gradient_reverse(
    input_tensor,
    coefficient,
):
    return (
        Phase33GradientReverseFunction
        .apply(
            input_tensor,
            coefficient,
        )
    )


class Phase33TemporalBlock(nn.Module):
    def __init__(
        self,
        dimension,
        dilation,
        dropout=0.08,
    ):
        super().__init__()

        self.depthwise = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=3,
            padding=int(dilation),
            dilation=int(dilation),
            groups=dimension,
            bias=False,
        )

        self.normalization = nn.GroupNorm(
            num_groups=8,
            num_channels=dimension,
        )

        self.pointwise = nn.Conv1d(
            dimension,
            2 * dimension,
            kernel_size=1,
        )

        self.output = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=1,
        )

        self.dropout = nn.Dropout(
            p=float(dropout)
        )

    def forward(self, sequence):
        residual = sequence

        hidden = self.depthwise(sequence)
        hidden = self.normalization(hidden)

        value, gate = self.pointwise(
            hidden
        ).chunk(2, dim=1)

        hidden = value * torch.sigmoid(
            gate
        )

        hidden = self.output(hidden)
        hidden = self.dropout(hidden)

        return residual + hidden


class Phase33DomainInvariantFusionNet(
    nn.Module
):
    def __init__(
        self,
        domain_class_count,
    ):
        super().__init__()

        model_dimension = 64
        geometry_dimension = 48

        self.p3ca_input = nn.Sequential(
            nn.Linear(48, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.GELU(),
        )

        self.p3ca_view_embedding = (
            nn.Parameter(
                torch.zeros(
                    1,
                    6,
                    1,
                    model_dimension,
                )
            )
        )

        self.p3ca_position_embedding = (
            nn.Parameter(
                torch.zeros(
                    1,
                    1,
                    14,
                    model_dimension,
                )
            )
        )

        self.p3ca_temporal = nn.Sequential(
            Phase33TemporalBlock(
                model_dimension,
                dilation=1,
            ),
            Phase33TemporalBlock(
                model_dimension,
                dilation=2,
            ),
            Phase33TemporalBlock(
                model_dimension,
                dilation=3,
            ),
        )

        self.position_attention = nn.Linear(
            model_dimension,
            1,
        )

        self.view_attention = nn.Linear(
            model_dimension,
            1,
        )

        self.p3ca_projection = nn.Sequential(
            nn.Linear(
                3 * model_dimension,
                96,
            ),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.geometry_input = nn.Sequential(
            nn.Linear(
                52,
                geometry_dimension,
            ),
            nn.LayerNorm(
                geometry_dimension
            ),
            nn.GELU(),
        )

        self.geometry_temporal = (
            nn.Sequential(
                Phase33TemporalBlock(
                    geometry_dimension,
                    dilation=1,
                ),
                Phase33TemporalBlock(
                    geometry_dimension,
                    dilation=2,
                ),
            )
        )

        self.geometry_attention = nn.Linear(
            geometry_dimension,
            1,
        )

        self.geometry_projection = (
            nn.Sequential(
                nn.Linear(
                    2 * geometry_dimension,
                    64,
                ),
                nn.LayerNorm(64),
                nn.GELU(),
                nn.Dropout(0.10),
            )
        )

        self.static_projection = (
            nn.Sequential(
                nn.Linear(96, 64),
                nn.LayerNorm(64),
                nn.GELU(),
                nn.Dropout(0.10),
            )
        )

        self.fusion = nn.Sequential(
            nn.LayerNorm(
                96 + 64 + 64
            ),
            nn.Linear(
                96 + 64 + 64,
                128,
            ),
            nn.GELU(),
            nn.Dropout(0.15),
        )

        self.classifier = nn.Linear(
            128,
            1,
        )

        self.domain_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(
                64,
                int(domain_class_count),
            ),
        )

        nn.init.trunc_normal_(
            self.p3ca_view_embedding,
            std=0.02,
        )
        nn.init.trunc_normal_(
            self.p3ca_position_embedding,
            std=0.02,
        )

    def encode_p3ca(
        self,
        sequence,
    ):
        batch_size = sequence.shape[0]

        sequence = sequence.reshape(
            batch_size,
            6,
            14,
            48,
        )

        hidden = self.p3ca_input(
            sequence
        )

        hidden = (
            hidden
            + self.p3ca_view_embedding
            + self.p3ca_position_embedding
        )

        hidden = hidden.reshape(
            batch_size * 6,
            14,
            64,
        ).transpose(1, 2)

        hidden = self.p3ca_temporal(
            hidden
        ).transpose(1, 2)

        position_weight = torch.softmax(
            self.position_attention(
                hidden
            ).squeeze(-1),
            dim=1,
        )

        view_embedding = torch.sum(
            hidden
            * position_weight[
                :, :, None
            ],
            dim=1,
        )

        view_embedding = (
            view_embedding.reshape(
                batch_size,
                6,
                64,
            )
        )

        view_weight = torch.softmax(
            self.view_attention(
                view_embedding
            ).squeeze(-1),
            dim=1,
        )

        attentive_view = torch.sum(
            view_embedding
            * view_weight[:, :, None],
            dim=1,
        )

        mean_view = view_embedding.mean(
            dim=1
        )

        maximum_view = view_embedding.amax(
            dim=1
        )

        return self.p3ca_projection(
            torch.cat(
                [
                    attentive_view,
                    mean_view,
                    maximum_view,
                ],
                dim=1,
            )
        )

    def encode_geometry(
        self,
        sequence,
    ):
        hidden = self.geometry_input(
            sequence
        ).transpose(1, 2)

        hidden = self.geometry_temporal(
            hidden
        ).transpose(1, 2)

        weight = torch.softmax(
            self.geometry_attention(
                hidden
            ).squeeze(-1),
            dim=1,
        )

        attentive = torch.sum(
            hidden * weight[:, :, None],
            dim=1,
        )

        mean = hidden.mean(dim=1)

        return self.geometry_projection(
            torch.cat(
                [attentive, mean],
                dim=1,
            )
        )

    def forward(
        self,
        p3ca_sequence,
        geometry_sequence,
        summary_features,
        radiomics_features,
        domain_coefficient=0.0,
    ):
        p3ca_embedding = (
            self.encode_p3ca(
                p3ca_sequence
            )
        )

        geometry_embedding = (
            self.encode_geometry(
                geometry_sequence
            )
        )

        static_embedding = (
            self.static_projection(
                torch.cat(
                    [
                        summary_features,
                        radiomics_features,
                    ],
                    dim=1,
                )
            )
        )

        fused = self.fusion(
            torch.cat(
                [
                    p3ca_embedding,
                    geometry_embedding,
                    static_embedding,
                ],
                dim=1,
            )
        )

        logit = self.classifier(
            fused
        ).squeeze(1)

        reversed_embedding = (
            phase33_gradient_reverse(
                fused,
                domain_coefficient,
            )
        )

        domain_logit = self.domain_head(
            reversed_embedding
        )

        return {
            "logit": logit,
            "domain_logit": (
                domain_logit
            ),
            "embedding": fused,
        }


PHASE33_DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

if PHASE33_DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(
        PHASE33_DEVICE
    )

phase33_contract_model = (
    Phase33DomainInvariantFusionNet(
        domain_class_count=(
            PHASE33_CONFIG[
                "domain_class_count"
            ]
        )
    ).to(PHASE33_DEVICE)
)

generator = torch.Generator(
    device="cpu"
)
generator.manual_seed(
    PHASE33_CONFIG["base_seed"]
)

batch_size = 8

synthetic_p3ca = torch.randn(
    batch_size,
    84,
    48,
    generator=generator,
).to(PHASE33_DEVICE)

synthetic_geometry = torch.randn(
    batch_size,
    16,
    52,
    generator=generator,
).to(PHASE33_DEVICE)

synthetic_summary = torch.randn(
    batch_size,
    48,
    generator=generator,
).to(PHASE33_DEVICE)

synthetic_radiomics = torch.randn(
    batch_size,
    48,
    generator=generator,
).to(PHASE33_DEVICE)

synthetic_target = torch.randint(
    0,
    2,
    (batch_size,),
    generator=generator,
).float().to(PHASE33_DEVICE)

synthetic_domain = torch.randint(
    0,
    PHASE33_CONFIG[
        "domain_class_count"
    ],
    (batch_size,),
    generator=generator,
).to(PHASE33_DEVICE)

phase33_contract_model.train()

contract_output = (
    phase33_contract_model(
        synthetic_p3ca,
        synthetic_geometry,
        synthetic_summary,
        synthetic_radiomics,
        domain_coefficient=0.05,
    )
)

classification_loss = (
    F.binary_cross_entropy_with_logits(
        contract_output["logit"],
        synthetic_target,
    )
)

domain_loss = F.cross_entropy(
    contract_output["domain_logit"],
    synthetic_domain,
)

contract_loss = (
    classification_loss
    + 0.03 * domain_loss
)

contract_loss.backward()

gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(
        phase33_contract_model.parameters(),
        max_norm=2.0,
    ).item()
)

parameter_count = sum(
    parameter.numel()
    for parameter in (
        phase33_contract_model.parameters()
    )
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in (
        phase33_contract_model.parameters()
    )
    if parameter.requires_grad
)

report = {
    "phase": (
        "phase33_domain_invariant_architecture"
    ),
    "status": "accepted",
    "model": (
        "Phase33DomainInvariantFusionNet"
    ),
    "parameter_count": (
        parameter_count
    ),
    "trainable_parameter_count": (
        trainable_parameter_count
    ),
    "inputs": {
        "p3ca_sequence": [
            batch_size, 84, 48
        ],
        "geometry_sequence": [
            batch_size, 16, 52
        ],
        "summary_features": [
            batch_size, 48
        ],
        "radiomics_features": [
            batch_size, 48
        ],
    },
    "outputs": {
        "classification_logit": list(
            contract_output[
                "logit"
            ].shape
        ),
        "domain_logit": list(
            contract_output[
                "domain_logit"
            ].shape
        ),
        "embedding": list(
            contract_output[
                "embedding"
            ].shape
        ),
    },
    "gradient_norm_before_clipping": round(
        gradient_norm,
        6,
    ),
    "peak_vram_mb": round(
        (
            torch.cuda.max_memory_allocated(
                PHASE33_DEVICE
            )
            / (1024 ** 2)
        )
        if PHASE33_DEVICE.type
        == "cuda"
        else 0.0,
        2,
    ),
    "temporal_processing": (
        "view_local_depthwise_gated_TCN"
    ),
    "acquisition_invariance": (
        "gradient_reversal_domain_head"
    ),
    "backward_contract_passed": True,
    "synthetic_input_only": True,
    "training_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE33_ARCHITECTURE"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE33_ARCHITECTURE"
)

BEGIN SANITIZED_PHASE33_ARCHITECTURE
{
  "phase": "phase33_domain_invariant_architecture",
  "status": "accepted",
  "model": "Phase33DomainInvariantFusionNet",
  "parameter_count": 130355,
  "trainable_parameter_count": 130355,
  "inputs": {
    "p3ca_sequence": [
      8,
      84,
      48
    ],
    "geometry_sequence": [
      8,
      16,
      52
    ],
    "summary_features": [
      8,
      48
    ],
    "radiomics_features": [
      8,
      48
    ]
  },
  "outputs": {
    "classification_logit": [
      8
    ],
    "domain_logit": [
      8,
      15
    ],
    "embedding": [
      8,
      128
    ]
  },
  "gradient_norm_before_clipping": 1.671228,
  "peak_vram_mb": 208.96,
  "temporal_processing": "view_local_depthwise_gated_TCN",
  "acquisition_invariance": "gradient_reversal_domain_head",
  "backward_contract_passed": true,
  "synthetic_input_only": true,
  "training_labels_used": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE33_ARC

In [43]:
# Phase33 Cell 110A
# Deterministic nested training utilities.

import copy
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn.functional as F

PHASE33_TRAIN_CONFIG = {
    "domain_strengths": [
        0.0,
        0.02,
        0.05,
    ],
    "seeds": [
        330701,
        330702,
        330703,
    ],
    "batch_size": 64,
    "maximum_epochs": 120,
    "patience": 18,
    "learning_rate": 3e-4,
    "minimum_learning_rate": 1.5e-5,
    "weight_decay": 0.03,
    "gradient_clip": 1.5,
    "label_smoothing": 0.01,
    "token_dropout": 0.03,
    "feature_noise": 0.01,
    "domain_ramp_epochs": 15,
    "minimum_domain_robust_gain": 0.001,
}


def phase33_set_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(seed)
        )

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )


def phase33_group_sample_weights(
    groups,
):
    groups = np.asarray(groups)

    unique_groups, counts = np.unique(
        groups,
        return_counts=True,
    )

    count_by_group = {
        group: count
        for group, count in zip(
            unique_groups,
            counts,
        )
    }

    weights = np.asarray(
        [
            1.0 / count_by_group[group]
            for group in groups
        ],
        dtype=np.float32,
    )

    weights *= (
        len(weights)
        / weights.sum()
    )

    return weights


def phase33_to_device_features(
    transformed,
    labels,
    groups,
):
    return {
        "p3ca_sequence": torch.from_numpy(
            transformed[
                "p3ca_sequence"
            ]
        ).to(PHASE33_DEVICE),
        "geometry_sequence": torch.from_numpy(
            transformed[
                "geometry_sequence"
            ]
        ).to(PHASE33_DEVICE),
        "summary_features": torch.from_numpy(
            transformed[
                "summary_features"
            ]
        ).to(PHASE33_DEVICE),
        "radiomics_features": torch.from_numpy(
            transformed[
                "radiomics_features"
            ]
        ).to(PHASE33_DEVICE),
        "labels": torch.from_numpy(
            np.asarray(
                labels,
                dtype=np.float32,
            )
        ).to(PHASE33_DEVICE),
        "groups": torch.from_numpy(
            np.asarray(
                groups,
                dtype=np.int64,
            )
        ).to(PHASE33_DEVICE),
        "weights": torch.from_numpy(
            phase33_group_sample_weights(
                groups
            )
        ).to(PHASE33_DEVICE),
    }


def phase33_augmented_batch(
    tensor,
    token_dropout,
    feature_noise,
):
    output = tensor

    if token_dropout > 0:
        keep_probability = (
            1.0 - token_dropout
        )

        token_mask = (
            torch.rand(
                output.shape[0],
                output.shape[1],
                1,
                device=output.device,
            )
            < keep_probability
        ).to(output.dtype)

        output = (
            output
            * token_mask
            / keep_probability
        )

    if feature_noise > 0:
        output = output + (
            feature_noise
            * torch.randn_like(output)
        )

    return output


@torch.inference_mode()
def phase33_predict_logits(
    model,
    data,
    batch_size=128,
):
    model.eval()

    predictions = []

    case_count = (
        data["p3ca_sequence"].shape[0]
    )

    for start in range(
        0,
        case_count,
        batch_size,
    ):
        stop = min(
            start + batch_size,
            case_count,
        )

        output = model(
            data["p3ca_sequence"][
                start:stop
            ],
            data["geometry_sequence"][
                start:stop
            ],
            data["summary_features"][
                start:stop
            ],
            data["radiomics_features"][
                start:stop
            ],
            domain_coefficient=0.0,
        )

        predictions.append(
            output["logit"]
            .float()
            .cpu()
            .numpy()
        )

    return np.concatenate(
        predictions,
        axis=0,
    ).astype(np.float64)


def phase33_monitor_record(
    labels,
    logits,
    groups,
):
    probability = phase32_sigmoid(
        logits
    )

    robust = phase32_robust_monitor_score(
        np.asarray(labels),
        probability,
        np.asarray(groups),
    )

    metrics = phase32_metrics(
        labels,
        probability,
    )

    return {
        "probability": probability,
        "metrics": metrics,
        "robust": robust,
    }


def phase33_state_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in (
            model.state_dict().items()
        )
    }


def phase33_train_with_monitor(
    training_data,
    monitor_data,
    monitor_labels,
    monitor_groups,
    domain_strength,
    seed,
):
    phase33_set_seed(seed)

    model = (
        Phase33DomainInvariantFusionNet(
            domain_class_count=(
                PHASE33_CONFIG[
                    "domain_class_count"
                ]
            )
        ).to(PHASE33_DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE33_TRAIN_CONFIG[
            "learning_rate"
        ],
        weight_decay=(
            PHASE33_TRAIN_CONFIG[
                "weight_decay"
            ]
        ),
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=PHASE33_TRAIN_CONFIG[
                "maximum_epochs"
            ],
            eta_min=PHASE33_TRAIN_CONFIG[
                "minimum_learning_rate"
            ],
        )
    )

    case_count = training_data[
        "labels"
    ].shape[0]

    best_state = None
    best_epoch = None
    best_robust_score = float("inf")
    best_monitor_log_loss = float("inf")
    best_monitor_logits = None
    epochs_without_improvement = 0
    epoch_records = []

    for epoch in range(
        1,
        PHASE33_TRAIN_CONFIG[
            "maximum_epochs"
        ] + 1,
    ):
        model.train()

        generator = torch.Generator(
            device=PHASE33_DEVICE.type
        )
        generator.manual_seed(
            seed * 1000 + epoch
        )

        permutation = torch.randperm(
            case_count,
            generator=generator,
            device=PHASE33_DEVICE,
        )

        classification_sum = 0.0
        domain_sum = 0.0
        total_seen = 0

        domain_coefficient = (
            float(domain_strength)
            * min(
                1.0,
                epoch
                / PHASE33_TRAIN_CONFIG[
                    "domain_ramp_epochs"
                ],
            )
        )

        for start in range(
            0,
            case_count,
            PHASE33_TRAIN_CONFIG[
                "batch_size"
            ],
        ):
            batch_indices = permutation[
                start:
                start
                + PHASE33_TRAIN_CONFIG[
                    "batch_size"
                ]
            ]

            p3ca_sequence = (
                phase33_augmented_batch(
                    training_data[
                        "p3ca_sequence"
                    ][batch_indices],
                    PHASE33_TRAIN_CONFIG[
                        "token_dropout"
                    ],
                    PHASE33_TRAIN_CONFIG[
                        "feature_noise"
                    ],
                )
            )

            geometry_sequence = (
                phase33_augmented_batch(
                    training_data[
                        "geometry_sequence"
                    ][batch_indices],
                    PHASE33_TRAIN_CONFIG[
                        "token_dropout"
                    ],
                    PHASE33_TRAIN_CONFIG[
                        "feature_noise"
                    ],
                )
            )

            summary_features = (
                training_data[
                    "summary_features"
                ][batch_indices]
                + PHASE33_TRAIN_CONFIG[
                    "feature_noise"
                ]
                * torch.randn_like(
                    training_data[
                        "summary_features"
                    ][batch_indices]
                )
            )

            radiomics_features = (
                training_data[
                    "radiomics_features"
                ][batch_indices]
                + PHASE33_TRAIN_CONFIG[
                    "feature_noise"
                ]
                * torch.randn_like(
                    training_data[
                        "radiomics_features"
                    ][batch_indices]
                )
            )

            targets = training_data[
                "labels"
            ][batch_indices]

            smoothed_targets = (
                targets
                * (
                    1.0
                    - 2.0
                    * PHASE33_TRAIN_CONFIG[
                        "label_smoothing"
                    ]
                )
                + PHASE33_TRAIN_CONFIG[
                    "label_smoothing"
                ]
            )

            groups = training_data[
                "groups"
            ][batch_indices]

            weights = training_data[
                "weights"
            ][batch_indices]

            optimizer.zero_grad(
                set_to_none=True
            )

            output = model(
                p3ca_sequence,
                geometry_sequence,
                summary_features,
                radiomics_features,
                domain_coefficient=(
                    domain_coefficient
                ),
            )

            classification_element = (
                F.binary_cross_entropy_with_logits(
                    output["logit"],
                    smoothed_targets,
                    reduction="none",
                )
            )

            classification_loss = (
                (
                    classification_element
                    * weights
                ).sum()
                / weights.sum()
            )

            domain_element = (
                F.cross_entropy(
                    output["domain_logit"],
                    groups,
                    reduction="none",
                )
            )

            domain_loss = (
                (
                    domain_element
                    * weights
                ).sum()
                / weights.sum()
            )

            total_loss = (
                classification_loss
                + domain_loss
            )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE33_TRAIN_CONFIG[
                    "gradient_clip"
                ],
            )

            optimizer.step()

            batch_count = int(
                len(batch_indices)
            )

            classification_sum += (
                float(
                    classification_loss
                    .detach()
                    .cpu()
                    .item()
                )
                * batch_count
            )

            domain_sum += (
                float(
                    domain_loss
                    .detach()
                    .cpu()
                    .item()
                )
                * batch_count
            )

            total_seen += batch_count

        scheduler.step()

        monitor_logits = (
            phase33_predict_logits(
                model,
                monitor_data,
            )
        )

        monitor_record = (
            phase33_monitor_record(
                monitor_labels,
                monitor_logits,
                monitor_groups,
            )
        )

        robust_score = (
            monitor_record["robust"][
                "robust_score"
            ]
        )

        monitor_log_loss = (
            monitor_record["metrics"][
                "log_loss"
            ]
        )

        improved = (
            robust_score
            < best_robust_score - 1e-6
        )

        if improved:
            best_state = (
                phase33_state_to_cpu(
                    model
                )
            )
            best_epoch = int(epoch)
            best_robust_score = float(
                robust_score
            )
            best_monitor_log_loss = float(
                monitor_log_loss
            )
            best_monitor_logits = (
                monitor_logits.copy()
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        epoch_records.append({
            "epoch": int(epoch),
            "train_classification_loss": (
                classification_sum
                / total_seen
            ),
            "train_domain_loss": (
                domain_sum
                / total_seen
            ),
            "monitor_log_loss": float(
                monitor_log_loss
            ),
            "monitor_robust_score": float(
                robust_score
            ),
        })

        if (
            epochs_without_improvement
            >= PHASE33_TRAIN_CONFIG[
                "patience"
            ]
        ):
            break

    assert best_state is not None
    assert best_epoch is not None

    model.load_state_dict(
        best_state,
        strict=True,
    )

    reconstruction_logits = (
        phase33_predict_logits(
            model,
            monitor_data,
        )
    )

    reconstruction_error = float(
        np.max(
            np.abs(
                reconstruction_logits
                - best_monitor_logits
            )
        )
    )

    assert reconstruction_error <= 1e-6

    return {
        "best_state": best_state,
        "best_epoch": best_epoch,
        "epochs_run": len(
            epoch_records
        ),
        "best_monitor_robust_score": (
            best_robust_score
        ),
        "best_monitor_log_loss": (
            best_monitor_log_loss
        ),
        "best_monitor_logits": (
            best_monitor_logits
        ),
        "checkpoint_reconstruction_error": (
            reconstruction_error
        ),
        "epoch_records": (
            epoch_records
        ),
    }


def phase33_train_fixed_epochs(
    training_data,
    domain_strength,
    seed,
    epoch_count,
):
    phase33_set_seed(seed)

    model = (
        Phase33DomainInvariantFusionNet(
            domain_class_count=(
                PHASE33_CONFIG[
                    "domain_class_count"
                ]
            )
        ).to(PHASE33_DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE33_TRAIN_CONFIG[
            "learning_rate"
        ],
        weight_decay=(
            PHASE33_TRAIN_CONFIG[
                "weight_decay"
            ]
        ),
    )

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=max(
                1,
                int(epoch_count),
            ),
            eta_min=PHASE33_TRAIN_CONFIG[
                "minimum_learning_rate"
            ],
        )
    )

    case_count = training_data[
        "labels"
    ].shape[0]

    final_epoch_loss = None

    for epoch in range(
        1,
        int(epoch_count) + 1,
    ):
        model.train()

        generator = torch.Generator(
            device=PHASE33_DEVICE.type
        )
        generator.manual_seed(
            seed * 1000 + epoch
        )

        permutation = torch.randperm(
            case_count,
            generator=generator,
            device=PHASE33_DEVICE,
        )

        epoch_loss_sum = 0.0
        epoch_seen = 0

        domain_coefficient = (
            float(domain_strength)
            * min(
                1.0,
                epoch
                / PHASE33_TRAIN_CONFIG[
                    "domain_ramp_epochs"
                ],
            )
        )

        for start in range(
            0,
            case_count,
            PHASE33_TRAIN_CONFIG[
                "batch_size"
            ],
        ):
            batch_indices = permutation[
                start:
                start
                + PHASE33_TRAIN_CONFIG[
                    "batch_size"
                ]
            ]

            targets = training_data[
                "labels"
            ][batch_indices]

            smoothed_targets = (
                targets
                * (
                    1.0
                    - 2.0
                    * PHASE33_TRAIN_CONFIG[
                        "label_smoothing"
                    ]
                )
                + PHASE33_TRAIN_CONFIG[
                    "label_smoothing"
                ]
            )

            weights = training_data[
                "weights"
            ][batch_indices]

            p3ca_sequence = (
                phase33_augmented_batch(
                    training_data[
                        "p3ca_sequence"
                    ][batch_indices],
                    PHASE33_TRAIN_CONFIG[
                        "token_dropout"
                    ],
                    PHASE33_TRAIN_CONFIG[
                        "feature_noise"
                    ],
                )
            )

            geometry_sequence = (
                phase33_augmented_batch(
                    training_data[
                        "geometry_sequence"
                    ][batch_indices],
                    PHASE33_TRAIN_CONFIG[
                        "token_dropout"
                    ],
                    PHASE33_TRAIN_CONFIG[
                        "feature_noise"
                    ],
                )
            )

            summary_features = (
                training_data[
                    "summary_features"
                ][batch_indices]
                + PHASE33_TRAIN_CONFIG[
                    "feature_noise"
                ]
                * torch.randn_like(
                    training_data[
                        "summary_features"
                    ][batch_indices]
                )
            )

            radiomics_features = (
                training_data[
                    "radiomics_features"
                ][batch_indices]
                + PHASE33_TRAIN_CONFIG[
                    "feature_noise"
                ]
                * torch.randn_like(
                    training_data[
                        "radiomics_features"
                    ][batch_indices]
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            output = model(
                p3ca_sequence,
                geometry_sequence,
                summary_features,
                radiomics_features,
                domain_coefficient=(
                    domain_coefficient
                ),
            )

            classification_element = (
                F.binary_cross_entropy_with_logits(
                    output["logit"],
                    smoothed_targets,
                    reduction="none",
                )
            )

            classification_loss = (
                (
                    classification_element
                    * weights
                ).sum()
                / weights.sum()
            )

            domain_element = (
                F.cross_entropy(
                    output["domain_logit"],
                    training_data[
                        "groups"
                    ][batch_indices],
                    reduction="none",
                )
            )

            domain_loss = (
                (
                    domain_element
                    * weights
                ).sum()
                / weights.sum()
            )

            loss = (
                classification_loss
                + domain_loss
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE33_TRAIN_CONFIG[
                    "gradient_clip"
                ],
            )

            optimizer.step()

            batch_count = int(
                len(batch_indices)
            )

            epoch_loss_sum += (
                float(
                    classification_loss
                    .detach()
                    .cpu()
                    .item()
                )
                * batch_count
            )
            epoch_seen += batch_count

        scheduler.step()

        final_epoch_loss = (
            epoch_loss_sum
            / epoch_seen
        )

    return model, float(
        final_epoch_loss
    )


report = {
    "phase": (
        "phase33_training_contract"
    ),
    "status": "accepted",
    "configuration": (
        PHASE33_TRAIN_CONFIG
    ),
    "classification_weighting": (
        "inverse_acquisition_group_frequency"
    ),
    "selection_metric": (
        "acquisition_group_robust_monitor_score"
    ),
    "domain_schedule": (
        "linear_ramp_then_constant"
    ),
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE33_TRAINING_CONTRACT"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE33_TRAINING_CONTRACT"
)

BEGIN SANITIZED_PHASE33_TRAINING_CONTRACT
{
  "phase": "phase33_training_contract",
  "status": "accepted",
  "configuration": {
    "domain_strengths": [
      0.0,
      0.02,
      0.05
    ],
    "seeds": [
      330701,
      330702,
      330703
    ],
    "batch_size": 64,
    "maximum_epochs": 120,
    "patience": 18,
    "learning_rate": 0.0003,
    "minimum_learning_rate": 1.5e-05,
    "weight_decay": 0.03,
    "gradient_clip": 1.5,
    "label_smoothing": 0.01,
    "token_dropout": 0.03,
    "feature_noise": 0.01,
    "domain_ramp_epochs": 15,
    "minimum_domain_robust_gain": 0.001
  },
  "classification_weighting": "inverse_acquisition_group_frequency",
  "selection_metric": "acquisition_group_robust_monitor_score",
  "domain_schedule": "linear_ramp_then_constant",
  "outer_validation_labels_used": false,
  "smoke_data_read": false,
  "test_data_read": false
}
END SANITIZED_PHASE33_TRAINING_CONTRACT


In [44]:
# Phase33 Cell 110B
# Run nested domain-strength selection and final outer-train refits.

phase33_training_started = (
    time.perf_counter()
)

PHASE33_OOF_PRIVATE = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

PHASE33_MONITOR_PROBABILITIES_PRIVATE = {}
PHASE33_DEPLOYMENT_STATES_PRIVATE = []
phase33_fold_records = []
phase33_oof_coverage = np.zeros(
    EXPECTED_CASES,
    dtype=np.int64,
)

for fold in range(3):
    fold_started = time.perf_counter()

    partition = PHASE32_PARTITIONS[fold]

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )
    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    # Fit-only preprocessing for nested selection.
    fit_preprocessor = (
        phase33_fit_preprocessor(
            stage_index=0,
            fold=fold,
            training_indices=fit_indices,
            seed=(
                PHASE33_CONFIG[
                    "base_seed"
                ]
                + fold * 100
            ),
        )
    )

    fit_transformed = phase33_transform(
        fit_preprocessor,
        fit_indices,
    )

    monitor_transformed = (
        phase33_transform(
            fit_preprocessor,
            monitor_indices,
        )
    )

    fit_data = (
        phase33_to_device_features(
            fit_transformed,
            PHASE32_LABELS[
                fit_indices
            ],
            PHASE32_GROUPS[
                fit_indices
            ],
        )
    )

    monitor_data = (
        phase33_to_device_features(
            monitor_transformed,
            PHASE32_LABELS[
                monitor_indices
            ],
            PHASE32_GROUPS[
                monitor_indices
            ],
        )
    )

    strength_records = []

    for domain_strength in (
        PHASE33_TRAIN_CONFIG[
            "domain_strengths"
        ]
    ):
        seed_results = []
        monitor_logits_by_seed = []

        for seed in (
            PHASE33_TRAIN_CONFIG[
                "seeds"
            ]
        ):
            run_seed = (
                int(seed)
                + fold * 1000
                + int(
                    domain_strength * 10000
                )
            )

            result = (
                phase33_train_with_monitor(
                    training_data=fit_data,
                    monitor_data=monitor_data,
                    monitor_labels=(
                        PHASE32_LABELS[
                            monitor_indices
                        ]
                    ),
                    monitor_groups=(
                        PHASE32_GROUPS[
                            monitor_indices
                        ]
                    ),
                    domain_strength=(
                        domain_strength
                    ),
                    seed=run_seed,
                )
            )

            seed_results.append({
                "seed": run_seed,
                "best_epoch": (
                    result["best_epoch"]
                ),
                "epochs_run": (
                    result["epochs_run"]
                ),
                "best_monitor_log_loss": (
                    result[
                        "best_monitor_log_loss"
                    ]
                ),
                "best_monitor_robust_score": (
                    result[
                        "best_monitor_robust_score"
                    ]
                ),
                "checkpoint_reconstruction_error": (
                    result[
                        "checkpoint_reconstruction_error"
                    ]
                ),
            })

            monitor_logits_by_seed.append(
                result[
                    "best_monitor_logits"
                ]
            )

        ensemble_monitor_logit = (
            np.mean(
                np.stack(
                    monitor_logits_by_seed,
                    axis=0,
                ),
                axis=0,
            )
        )

        ensemble_record = (
            phase33_monitor_record(
                PHASE32_LABELS[
                    monitor_indices
                ],
                ensemble_monitor_logit,
                PHASE32_GROUPS[
                    monitor_indices
                ],
            )
        )

        strength_records.append({
            "domain_strength": float(
                domain_strength
            ),
            "seed_results": seed_results,
            "ensemble_monitor_logits": (
                ensemble_monitor_logit
            ),
            "ensemble_monitor_probability": (
                ensemble_record[
                    "probability"
                ]
            ),
            "ensemble_monitor_metrics": (
                ensemble_record["metrics"]
            ),
            "ensemble_monitor_robust": (
                ensemble_record["robust"]
            ),
        })

        print(
            f"Phase33 fold {fold}, "
            f"domain={domain_strength:.2f}: "
            f"monitor="
            f"{ensemble_record['metrics']['log_loss']:.6f}, "
            f"robust="
            f"{ensemble_record['robust']['robust_score']:.6f}"
        )

    strength_records.sort(
        key=lambda record: (
            record[
                "ensemble_monitor_robust"
            ]["robust_score"],
            record[
                "ensemble_monitor_metrics"
            ]["log_loss"],
            record["domain_strength"],
        )
    )

    raw_best = strength_records[0]

    zero_strength = next(
        record
        for record in strength_records
        if record["domain_strength"] == 0.0
    )

    domain_robust_gain = (
        zero_strength[
            "ensemble_monitor_robust"
        ]["robust_score"]
        - raw_best[
            "ensemble_monitor_robust"
        ]["robust_score"]
    )

    if (
        raw_best["domain_strength"] > 0
        and domain_robust_gain
        < PHASE33_TRAIN_CONFIG[
            "minimum_domain_robust_gain"
        ]
    ):
        selected = zero_strength
        domain_advanced = False
    else:
        selected = raw_best
        domain_advanced = bool(
            selected[
                "domain_strength"
            ] > 0
        )

    selected_domain_strength = float(
        selected["domain_strength"]
    )

    PHASE33_MONITOR_PROBABILITIES_PRIVATE[
        fold
    ] = np.asarray(
        selected[
            "ensemble_monitor_probability"
        ],
        dtype=np.float64,
    )

    selected_best_epochs = [
        int(record["best_epoch"])
        for record in (
            selected["seed_results"]
        )
    ]

    # Refit all preprocessing on outer_train.
    final_preprocessor = (
        phase33_fit_preprocessor(
            stage_index=1,
            fold=fold,
            training_indices=(
                outer_train_indices
            ),
            seed=(
                PHASE33_CONFIG[
                    "base_seed"
                ]
                + 5000
                + fold * 100
            ),
        )
    )

    outer_train_transformed = (
        phase33_transform(
            final_preprocessor,
            outer_train_indices,
        )
    )

    outer_valid_transformed = (
        phase33_transform(
            final_preprocessor,
            outer_valid_indices,
        )
    )

    outer_train_data = (
        phase33_to_device_features(
            outer_train_transformed,
            PHASE32_LABELS[
                outer_train_indices
            ],
            PHASE32_GROUPS[
                outer_train_indices
            ],
        )
    )

    outer_valid_data = (
        phase33_to_device_features(
            outer_valid_transformed,
            PHASE32_LABELS[
                outer_valid_indices
            ],
            PHASE32_GROUPS[
                outer_valid_indices
            ],
        )
    )

    final_logits_by_seed = []
    final_states = []
    final_training_losses = []

    for seed_index, seed in enumerate(
        PHASE33_TRAIN_CONFIG["seeds"]
    ):
        final_seed = (
            int(seed)
            + 10000
            + fold * 1000
            + int(
                selected_domain_strength
                * 10000
            )
        )

        epoch_count = max(
            1,
            int(
                selected_best_epochs[
                    seed_index
                ]
            ),
        )

        final_model, final_loss = (
            phase33_train_fixed_epochs(
                training_data=(
                    outer_train_data
                ),
                domain_strength=(
                    selected_domain_strength
                ),
                seed=final_seed,
                epoch_count=epoch_count,
            )
        )

        final_logits = (
            phase33_predict_logits(
                final_model,
                outer_valid_data,
            )
        )

        final_logits_by_seed.append(
            final_logits
        )

        final_states.append(
            phase33_state_to_cpu(
                final_model
            )
        )

        final_training_losses.append(
            final_loss
        )

        del final_model

    ensemble_outer_logit = np.mean(
        np.stack(
            final_logits_by_seed,
            axis=0,
        ),
        axis=0,
    )

    ensemble_outer_probability = (
        phase32_sigmoid(
            ensemble_outer_logit
        )
    )

    PHASE33_OOF_PRIVATE[
        outer_valid_indices
    ] = ensemble_outer_probability

    phase33_oof_coverage[
        outer_valid_indices
    ] += 1

    outer_metrics = phase32_metrics(
        PHASE32_LABELS[
            outer_valid_indices
        ],
        ensemble_outer_probability,
    )

    PHASE33_DEPLOYMENT_STATES_PRIVATE.append({
        "fold": fold,
        "domain_strength": (
            selected_domain_strength
        ),
        "best_epochs": (
            selected_best_epochs
        ),
        "preprocessor": (
            final_preprocessor
        ),
        "model_states": final_states,
    })

    phase33_fold_records.append({
        "fold": fold,
        "fit_n": int(
            len(fit_indices)
        ),
        "monitor_n": int(
            len(monitor_indices)
        ),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "selected_domain_strength": (
            selected_domain_strength
        ),
        "domain_advanced": (
            domain_advanced
        ),
        "domain_robust_gain": float(
            domain_robust_gain
        ),
        "selected_best_epochs": (
            selected_best_epochs
        ),
        "selected_monitor_log_loss": float(
            selected[
                "ensemble_monitor_metrics"
            ]["log_loss"]
        ),
        "selected_monitor_auroc": float(
            selected[
                "ensemble_monitor_metrics"
            ]["auroc"]
        ),
        "selected_monitor_robust_score": float(
            selected[
                "ensemble_monitor_robust"
            ]["robust_score"]
        ),
        "outer_log_loss": float(
            outer_metrics["log_loss"]
        ),
        "outer_auroc": float(
            outer_metrics["auroc"]
        ),
        "outer_brier": float(
            outer_metrics["brier"]
        ),
        "outer_mean_probability": float(
            outer_metrics[
                "mean_probability"
            ]
        ),
        "final_training_losses": (
            final_training_losses
        ),
        "elapsed_seconds": round(
            time.perf_counter()
            - fold_started,
            2,
        ),
    })

    print(
        f"Phase33 fold {fold} frozen: "
        f"domain={selected_domain_strength:.2f}, "
        f"epochs={selected_best_epochs}, "
        f"outer={outer_metrics['log_loss']:.6f}, "
        f"auroc={outer_metrics['auroc']:.6f}"
    )

    del fit_data
    del monitor_data
    del outer_train_data
    del outer_valid_data

    if PHASE33_DEVICE.type == "cuda":
        torch.cuda.empty_cache()

assert np.all(
    phase33_oof_coverage == 1
)
assert np.isfinite(
    PHASE33_OOF_PRIVATE
).all()

phase33_oof_metrics = phase32_metrics(
    PHASE32_LABELS,
    PHASE33_OOF_PRIVATE,
)

report = {
    "phase": (
        "phase33_domain_invariant_geometry_semantic_fusion"
    ),
    "status": (
        "nested_training_complete"
    ),
    "model": (
        "Phase33DomainInvariantFusionNet"
    ),
    "parameter_count": 130355,
    "metrics": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value in (
            phase33_oof_metrics.items()
        )
    },
    "folds": [
        {
            key: (
                round(value, 6)
                if isinstance(
                    value,
                    (float, np.floating),
                )
                else value
            )
            for key, value in record.items()
        }
        for record in phase33_fold_records
    ],
    "elapsed_seconds": round(
        time.perf_counter()
        - phase33_training_started,
        2,
    ),
    "monitor_selection_group_robust": True,
    "outer_train_preprocessing_refitted": True,
    "outer_train_model_refitted": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE33_TRAINING"
)
print(json.dumps(report, indent=2))
print(
    "END SANITIZED_PHASE33_TRAINING"
)

Phase33 fold 0, domain=0.00: monitor=0.303831, robust=0.310563
Phase33 fold 0, domain=0.02: monitor=0.290275, robust=0.299132
Phase33 fold 0, domain=0.05: monitor=0.301570, robust=0.309971
Phase33 fold 0 frozen: domain=0.02, epochs=[21, 6, 6], outer=0.370977, auroc=0.912292
Phase33 fold 1, domain=0.00: monitor=0.363773, robust=0.349430
Phase33 fold 1, domain=0.02: monitor=0.320431, robust=0.305116
Phase33 fold 1, domain=0.05: monitor=0.319413, robust=0.306175
Phase33 fold 1 frozen: domain=0.02, epochs=[20, 22, 87], outer=0.308659, auroc=0.942432
Phase33 fold 2, domain=0.00: monitor=0.265353, robust=0.262424
Phase33 fold 2, domain=0.02: monitor=0.232717, robust=0.221135
Phase33 fold 2, domain=0.05: monitor=0.257671, robust=0.259599
Phase33 fold 2 frozen: domain=0.02, epochs=[12, 20, 18], outer=0.395333, auroc=0.913836
BEGIN SANITIZED_PHASE33_TRAINING
{
  "phase": "phase33_domain_invariant_geometry_semantic_fusion",
  "status": "nested_training_complete",
  "model": "Phase33DomainInvaria

In [45]:
# Phase33 Cell 111
# Diagnostic-only complementarity and bounded-blend oracle.
# Outer labels are used only for aggregate diagnosis.
# No oracle parameter produced here is deployable.

import json
import math
import time

import numpy as np
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)


def phase33d_probability(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    return np.clip(values, 1e-6, 1.0 - 1e-6)


def phase33d_logit(probabilities):
    probabilities = phase33d_probability(probabilities)
    return np.log(probabilities) - np.log1p(-probabilities)


def phase33d_sigmoid(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = np.clip(logits, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-logits))


def phase33d_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64).reshape(-1)
    probabilities = phase33d_probability(probabilities)

    result = {
        "log_loss": float(log_loss(labels, probabilities)),
        "brier": float(
            brier_score_loss(labels, probabilities)
        ),
        "mean_probability": float(np.mean(probabilities)),
    }

    if np.unique(labels).size == 2:
        result["auroc"] = float(
            roc_auc_score(labels, probabilities)
        )
    else:
        result["auroc"] = None

    return result


def phase33d_bounded_update(
    baseline_probability,
    component_probability,
    alpha,
    uncertainty_exponent,
    delta_cap,
):
    baseline_probability = phase33d_probability(
        baseline_probability
    )
    component_probability = phase33d_probability(
        component_probability
    )

    baseline_logit = phase33d_logit(
        baseline_probability
    )
    component_logit = phase33d_logit(
        component_probability
    )

    uncertainty = (
        4.0
        * baseline_probability
        * (1.0 - baseline_probability)
    ) ** float(uncertainty_exponent)

    delta = component_logit - baseline_logit

    if float(delta_cap) > 0.0:
        delta = np.clip(
            delta,
            -float(delta_cap),
            float(delta_cap),
        )

    final_logit = (
        baseline_logit
        + float(alpha) * uncertainty * delta
    )

    return phase33d_sigmoid(final_logit)


def phase33d_group_harm(
    labels,
    baseline_probability,
    candidate_probability,
    groups,
    minimum_group_n=30,
):
    records = []
    maximum_harm = -np.inf

    for group in sorted(np.unique(groups).tolist()):
        mask = groups == group
        group_n = int(np.sum(mask))

        if group_n < minimum_group_n:
            continue

        baseline_loss = float(
            log_loss(
                labels[mask],
                phase33d_probability(
                    baseline_probability[mask]
                ),
            )
        )
        candidate_loss = float(
            log_loss(
                labels[mask],
                phase33d_probability(
                    candidate_probability[mask]
                ),
            )
        )

        change = candidate_loss - baseline_loss
        maximum_harm = max(maximum_harm, change)

        records.append({
            "group": int(group),
            "n": group_n,
            "baseline_log_loss": round(
                baseline_loss, 6
            ),
            "candidate_log_loss": round(
                candidate_loss, 6
            ),
            "log_loss_change": round(change, 6),
        })

    if not records:
        maximum_harm = 0.0

    return float(maximum_harm), records


phase33_diagnostic_started = time.perf_counter()

phase33_labels = np.asarray(
    case_df["label"],
    dtype=np.int64,
)

phase33_groups = np.asarray(
    case_df["acquisition_group"],
    dtype=np.int64,
)

phase33_baseline = phase33d_probability(
    PHASE19_PHASE12C_OOF_PRIVATE
)

phase33_component = phase33d_probability(
    PHASE33_OOF_PRIVATE
)

assert phase33_labels.shape == (1362,)
assert phase33_groups.shape == (1362,)
assert phase33_baseline.shape == (1362,)
assert phase33_component.shape == (1362,)
assert np.isfinite(phase33_component).all()

phase33_baseline_metrics = phase33d_metrics(
    phase33_labels,
    phase33_baseline,
)
phase33_component_metrics = phase33d_metrics(
    phase33_labels,
    phase33_component,
)

# Conservative bounded residual search.
phase33_oracle_candidates = [{
    "alpha": 0.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 0.0,
}]

for alpha in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
    for uncertainty_exponent in [0.0, 0.5, 1.0, 2.0]:
        for delta_cap in [0.5, 1.0, 1.5, 2.0]:
            phase33_oracle_candidates.append({
                "alpha": float(alpha),
                "uncertainty_exponent": float(
                    uncertainty_exponent
                ),
                "delta_cap": float(delta_cap),
            })


def phase33d_evaluate_candidate(indices, specification):
    candidate_probability = phase33d_bounded_update(
        phase33_baseline[indices],
        phase33_component[indices],
        specification["alpha"],
        specification["uncertainty_exponent"],
        specification["delta_cap"],
    )

    metrics = phase33d_metrics(
        phase33_labels[indices],
        candidate_probability,
    )

    return candidate_probability, metrics


# Global oracle: one parameter set across all three folds.
phase33_global_records = []

for specification in phase33_oracle_candidates:
    candidate_probability, candidate_metrics = (
        phase33d_evaluate_candidate(
            np.arange(1362, dtype=np.int64),
            specification,
        )
    )

    phase33_global_records.append({
        "specification": dict(specification),
        "probability": candidate_probability,
        "metrics": candidate_metrics,
    })

phase33_global_best = min(
    phase33_global_records,
    key=lambda record: (
        record["metrics"]["log_loss"],
        record["specification"]["alpha"],
        record["specification"][
            "uncertainty_exponent"
        ],
        record["specification"]["delta_cap"],
    ),
)

phase33_global_probability = (
    phase33_global_best["probability"]
)
phase33_global_metrics = (
    phase33_global_best["metrics"]
)

phase33_global_fold_records = []
phase33_fold_oracle_records = []

for partition in PHASE19_PARTITIONS:
    fold = int(partition["fold"])
    outer_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    baseline_fold_metrics = phase33d_metrics(
        phase33_labels[outer_indices],
        phase33_baseline[outer_indices],
    )
    component_fold_metrics = phase33d_metrics(
        phase33_labels[outer_indices],
        phase33_component[outer_indices],
    )

    global_fold_metrics = phase33d_metrics(
        phase33_labels[outer_indices],
        phase33_global_probability[outer_indices],
    )

    global_fold_gain = (
        baseline_fold_metrics["log_loss"]
        - global_fold_metrics["log_loss"]
    )

    phase33_global_fold_records.append({
        "fold": fold,
        "n": int(outer_indices.size),
        "baseline_log_loss": round(
            baseline_fold_metrics["log_loss"], 6
        ),
        "global_oracle_log_loss": round(
            global_fold_metrics["log_loss"], 6
        ),
        "global_oracle_gain": round(
            global_fold_gain, 6
        ),
        "baseline_auroc": round(
            baseline_fold_metrics["auroc"], 6
        ),
        "global_oracle_auroc": round(
            global_fold_metrics["auroc"], 6
        ),
    })

    fold_candidates = []

    for specification in phase33_oracle_candidates:
        probability, metrics = (
            phase33d_evaluate_candidate(
                outer_indices,
                specification,
            )
        )

        fold_candidates.append({
            "specification": dict(specification),
            "metrics": metrics,
        })

    fold_best = min(
        fold_candidates,
        key=lambda record: (
            record["metrics"]["log_loss"],
            record["specification"]["alpha"],
            record["specification"][
                "uncertainty_exponent"
            ],
            record["specification"]["delta_cap"],
        ),
    )

    baseline_residual = (
        phase33_labels[outer_indices]
        - phase33_baseline[outer_indices]
    )
    proposed_logit_update = (
        phase33d_logit(
            phase33_component[outer_indices]
        )
        - phase33d_logit(
            phase33_baseline[outer_indices]
        )
    )

    if (
        np.std(baseline_residual) > 1e-12
        and np.std(proposed_logit_update) > 1e-12
    ):
        residual_update_correlation = float(
            np.corrcoef(
                baseline_residual,
                proposed_logit_update,
            )[0, 1]
        )
    else:
        residual_update_correlation = 0.0

    phase33_fold_oracle_records.append({
        "fold": fold,
        "n": int(outer_indices.size),
        "baseline_log_loss": round(
            baseline_fold_metrics["log_loss"], 6
        ),
        "component_log_loss": round(
            component_fold_metrics["log_loss"], 6
        ),
        "baseline_auroc": round(
            baseline_fold_metrics["auroc"], 6
        ),
        "component_auroc": round(
            component_fold_metrics["auroc"], 6
        ),
        "oracle_specification_diagnostic_only": (
            fold_best["specification"]
        ),
        "oracle_log_loss": round(
            fold_best["metrics"]["log_loss"], 6
        ),
        "oracle_log_loss_gain": round(
            baseline_fold_metrics["log_loss"]
            - fold_best["metrics"]["log_loss"],
            6,
        ),
        "oracle_auroc": round(
            fold_best["metrics"]["auroc"], 6
        ),
        "baseline_residual_update_correlation": (
            round(residual_update_correlation, 6)
        ),
    })

phase33_global_gain = (
    phase33_baseline_metrics["log_loss"]
    - phase33_global_metrics["log_loss"]
)
phase33_global_auroc_gain = (
    phase33_global_metrics["auroc"]
    - phase33_baseline_metrics["auroc"]
)

phase33_folds_with_global_gain = sum(
    record["global_oracle_gain"] >= 0.002
    for record in phase33_global_fold_records
)

phase33_worst_global_fold_excess = max(
    -record["global_oracle_gain"]
    for record in phase33_global_fold_records
)

(
    phase33_maximum_group_harm,
    phase33_group_records,
) = phase33d_group_harm(
    phase33_labels,
    phase33_baseline,
    phase33_global_probability,
    phase33_groups,
)

# This only decides whether a legitimate nested monitor-gate
# experiment is worth running. It does not promote a model.
phase33_continuation_gate = bool(
    phase33_global_gain >= 0.003
    and phase33_folds_with_global_gain >= 2
    and phase33_worst_global_fold_excess <= 0.005
    and phase33_maximum_group_harm <= 0.015
)

phase33_diagnostic_report = {
    "phase": "phase33_bounded_residual_diagnostic",
    "status": (
        "eligible_for_nested_monitor_gate"
        if phase33_continuation_gate
        else "terminate_phase33_component"
    ),
    "baseline": {
        key: (
            None if value is None
            else round(float(value), 6)
        )
        for key, value in phase33_baseline_metrics.items()
    },
    "phase33_component": {
        key: (
            None if value is None
            else round(float(value), 6)
        )
        for key, value in phase33_component_metrics.items()
    },
    "global_oracle_diagnostic_only": {
        "specification": (
            phase33_global_best["specification"]
        ),
        "metrics": {
            key: (
                None if value is None
                else round(float(value), 6)
            )
            for key, value in (
                phase33_global_metrics.items()
            )
        },
        "log_loss_gain": round(
            phase33_global_gain, 6
        ),
        "auroc_gain": round(
            phase33_global_auroc_gain, 6
        ),
        "folds_with_gain_at_least_0p002": int(
            phase33_folds_with_global_gain
        ),
        "worst_fold_excess": round(
            phase33_worst_global_fold_excess, 6
        ),
        "maximum_major_group_harm": round(
            phase33_maximum_group_harm, 6
        ),
    },
    "global_parameter_fold_metrics": (
        phase33_global_fold_records
    ),
    "fold_oracles_diagnostic_only": (
        phase33_fold_oracle_records
    ),
    "major_acquisition_group_metrics": (
        phase33_group_records
    ),
    "continuation_thresholds": {
        "minimum_global_log_loss_gain": 0.003,
        "minimum_folds_with_gain_0p002": 2,
        "maximum_worst_fold_excess": 0.005,
        "maximum_major_group_harm": 0.015,
    },
    "continuation_gate_passed": (
        phase33_continuation_gate
    ),
    "candidate_count": len(
        phase33_oracle_candidates
    ),
    "oracle_parameters_deployable": False,
    "outer_labels_used_only_for_aggregate_diagnostic": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase33_diagnostic_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE33_DIAGNOSTIC")
print(json.dumps(
    phase33_diagnostic_report,
    indent=2,
))
print("END SANITIZED_PHASE33_DIAGNOSTIC")

BEGIN SANITIZED_PHASE33_DIAGNOSTIC
{
  "phase": "phase33_bounded_residual_diagnostic",
  "status": "eligible_for_nested_monitor_gate",
  "baseline": {
    "log_loss": 0.307616,
    "brier": 0.095535,
    "mean_probability": 0.530698,
    "auroc": 0.938914
  },
  "phase33_component": {
    "log_loss": 0.358791,
    "brier": 0.110525,
    "mean_probability": 0.558999,
    "auroc": 0.923527
  },
  "global_oracle_diagnostic_only": {
    "specification": {
      "alpha": 0.3,
      "uncertainty_exponent": 0.0,
      "delta_cap": 2.0
    },
    "metrics": {
      "log_loss": 0.302115,
      "brier": 0.093862,
      "mean_probability": 0.537578,
      "auroc": 0.941344
    },
    "log_loss_gain": 0.005501,
    "auroc_gain": 0.002429,
    "folds_with_gain_at_least_0p002": 2,
    "worst_fold_excess": -0.000517,
    "maximum_major_group_harm": 0.011994
  },
  "global_parameter_fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "baseline_log_loss": 0.358496,
      "global_oracle_log_lo

In [46]:
# Phase33 Cell 112
# Shared, monitor-selected, bounded residual gate.
#
# Important:
# - One shared gate is selected across all monitor partitions.
# - Duplicate monitor cases receive inverse-frequency weights.
# - No fold-specific gate routing is used.
# - Outer labels are touched only after the gate is frozen.
# - The Phase33 component remains independently OOF.

import json
import time

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def phase33g_weighted_log_loss(
    labels,
    probabilities,
    sample_weight=None,
):
    labels = np.asarray(
        labels,
        dtype=np.float64,
    ).reshape(-1)

    probabilities = phase33d_probability(
        probabilities
    )

    losses = -(
        labels * np.log(probabilities)
        + (1.0 - labels)
        * np.log1p(-probabilities)
    )

    if sample_weight is None:
        return float(np.mean(losses))

    sample_weight = np.asarray(
        sample_weight,
        dtype=np.float64,
    ).reshape(-1)

    assert sample_weight.shape == losses.shape
    assert np.all(sample_weight >= 0.0)
    assert np.sum(sample_weight) > 0.0

    return float(
        np.sum(sample_weight * losses)
        / np.sum(sample_weight)
    )


def phase33g_weighted_brier(
    labels,
    probabilities,
    sample_weight=None,
):
    labels = np.asarray(
        labels,
        dtype=np.float64,
    ).reshape(-1)
    probabilities = phase33d_probability(
        probabilities
    )

    squared_error = (
        probabilities - labels
    ) ** 2

    if sample_weight is None:
        return float(np.mean(squared_error))

    sample_weight = np.asarray(
        sample_weight,
        dtype=np.float64,
    ).reshape(-1)

    return float(
        np.sum(sample_weight * squared_error)
        / np.sum(sample_weight)
    )


def phase33g_extract_monitor_probability(
    container,
    fold,
    monitor_indices,
):
    value = None

    if isinstance(container, dict):
        candidate_keys = [
            fold,
            str(fold),
            f"fold_{fold}",
            f"fold{fold}",
        ]

        for key in candidate_keys:
            if key in container:
                value = container[key]
                break

    elif isinstance(container, (list, tuple)):
        assert len(container) > fold
        value = container[fold]

    else:
        array = np.asarray(container)

        if array.ndim == 2 and array.shape[0] == 3:
            value = array[fold]
        elif array.ndim == 1:
            value = array

    assert value is not None, {
        "message": (
            "Could not resolve Phase33 monitor "
            "probabilities."
        ),
        "fold": fold,
        "container_type": type(container).__name__,
    }

    if isinstance(value, dict):
        probability_keys = [
            "probability",
            "probabilities",
            "prediction",
            "predictions",
            "monitor_probability",
            "monitor_probabilities",
            "p",
        ]

        resolved = None

        for key in probability_keys:
            if key in value:
                resolved = value[key]
                break

        assert resolved is not None, {
            "message": (
                "Monitor probability dictionary has "
                "no recognized probability field."
            ),
            "fold": fold,
            "available_keys": sorted(
                str(key) for key in value.keys()
            ),
        }

        value = resolved

    array = np.asarray(
        value,
        dtype=np.float64,
    ).reshape(-1)

    if array.size == monitor_indices.size:
        result = array
    elif array.size == len(case_df):
        result = array[monitor_indices]
    else:
        raise AssertionError({
            "message": (
                "Resolved Phase33 monitor probability "
                "has an unexpected length."
            ),
            "fold": fold,
            "resolved_length": int(array.size),
            "monitor_n": int(monitor_indices.size),
            "case_count": int(len(case_df)),
        })

    return phase33d_probability(result)


def phase33g_robust_score(
    labels,
    probabilities,
    groups,
    sample_weight,
    minimum_group_n=8,
):
    labels = np.asarray(labels, dtype=np.int64)
    groups = np.asarray(groups, dtype=np.int64)
    sample_weight = np.asarray(
        sample_weight,
        dtype=np.float64,
    )

    overall_loss = phase33g_weighted_log_loss(
        labels,
        probabilities,
        sample_weight,
    )

    group_losses = []

    for group in sorted(np.unique(groups).tolist()):
        mask = groups == group

        if int(np.sum(mask)) < minimum_group_n:
            continue

        group_losses.append(
            phase33g_weighted_log_loss(
                labels[mask],
                probabilities[mask],
                sample_weight[mask],
            )
        )

    if group_losses:
        equal_group_mean = float(
            np.mean(group_losses)
        )
        worst_group = float(
            np.max(group_losses)
        )
    else:
        equal_group_mean = overall_loss
        worst_group = overall_loss

    robust_score = (
        0.55 * overall_loss
        + 0.30 * equal_group_mean
        + 0.15 * worst_group
    )

    return {
        "robust_score": float(robust_score),
        "overall_log_loss": float(overall_loss),
        "equal_group_mean_log_loss": float(
            equal_group_mean
        ),
        "worst_group_log_loss": float(
            worst_group
        ),
        "included_group_count": int(
            len(group_losses)
        ),
    }


def phase33g_monitor_group_harm(
    labels,
    baseline_probability,
    candidate_probability,
    groups,
    sample_weight,
    minimum_group_n=8,
):
    records = []
    maximum_harm = 0.0

    for group in sorted(np.unique(groups).tolist()):
        mask = groups == group
        group_n = int(np.sum(mask))

        if group_n < minimum_group_n:
            continue

        baseline_loss = phase33g_weighted_log_loss(
            labels[mask],
            baseline_probability[mask],
            sample_weight[mask],
        )
        candidate_loss = phase33g_weighted_log_loss(
            labels[mask],
            candidate_probability[mask],
            sample_weight[mask],
        )

        harm = candidate_loss - baseline_loss
        maximum_harm = max(maximum_harm, harm)

        records.append({
            "group": int(group),
            "n": group_n,
            "baseline_log_loss": baseline_loss,
            "candidate_log_loss": candidate_loss,
            "log_loss_change": harm,
        })

    return float(maximum_harm), records


def phase33g_calibration_statistics(
    labels,
    probabilities,
):
    labels = np.asarray(labels, dtype=np.int64)
    predictor = phase33d_logit(
        probabilities
    ).reshape(-1, 1)

    estimator = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
        random_state=330912,
    )
    estimator.fit(predictor, labels)

    return {
        "calibration_intercept": float(
            estimator.intercept_[0]
        ),
        "calibration_slope": float(
            estimator.coef_[0, 0]
        ),
    }


phase33_gate_started = time.perf_counter()

phase33_gate_labels = np.asarray(
    case_df["label"],
    dtype=np.int64,
)
phase33_gate_groups = np.asarray(
    case_df["acquisition_group"],
    dtype=np.int64,
)

phase33_fold_baseline_probability = np.asarray(
    PHASE32_PHASE12C_FOLD_PROBABILITIES_PRIVATE,
    dtype=np.float64,
)

assert phase33_fold_baseline_probability.shape == (
    3,
    1362,
)

# ---------------------------------------------------------
# Construct pooled monitor observations.
# A case can appear in more than one fold's monitor split.
# Such cases are given inverse-multiplicity weights.
# ---------------------------------------------------------

phase33_monitor_indices_parts = []
phase33_monitor_labels_parts = []
phase33_monitor_groups_parts = []
phase33_monitor_folds_parts = []
phase33_monitor_baseline_parts = []
phase33_monitor_component_parts = []

for partition in PHASE19_PARTITIONS:
    fold = int(partition["fold"])
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )

    monitor_component = (
        phase33g_extract_monitor_probability(
            PHASE33_MONITOR_PROBABILITIES_PRIVATE,
            fold,
            monitor_indices,
        )
    )

    monitor_baseline = phase33d_probability(
        phase33_fold_baseline_probability[
            fold,
            monitor_indices,
        ]
    )

    assert monitor_component.shape == (
        monitor_indices.size,
    )
    assert monitor_baseline.shape == (
        monitor_indices.size,
    )

    phase33_monitor_indices_parts.append(
        monitor_indices
    )
    phase33_monitor_labels_parts.append(
        phase33_gate_labels[monitor_indices]
    )
    phase33_monitor_groups_parts.append(
        phase33_gate_groups[monitor_indices]
    )
    phase33_monitor_folds_parts.append(
        np.full(
            monitor_indices.size,
            fold,
            dtype=np.int64,
        )
    )
    phase33_monitor_baseline_parts.append(
        monitor_baseline
    )
    phase33_monitor_component_parts.append(
        monitor_component
    )

phase33_monitor_indices = np.concatenate(
    phase33_monitor_indices_parts
)
phase33_monitor_labels = np.concatenate(
    phase33_monitor_labels_parts
)
phase33_monitor_groups = np.concatenate(
    phase33_monitor_groups_parts
)
phase33_monitor_folds = np.concatenate(
    phase33_monitor_folds_parts
)
phase33_monitor_baseline = np.concatenate(
    phase33_monitor_baseline_parts
)
phase33_monitor_component = np.concatenate(
    phase33_monitor_component_parts
)

unique_monitor_indices, monitor_multiplicities = (
    np.unique(
        phase33_monitor_indices,
        return_counts=True,
    )
)

multiplicity_lookup = {
    int(index): int(count)
    for index, count in zip(
        unique_monitor_indices,
        monitor_multiplicities,
    )
}

phase33_monitor_weights = np.asarray([
    1.0 / multiplicity_lookup[int(index)]
    for index in phase33_monitor_indices
], dtype=np.float64)

assert np.isfinite(
    phase33_monitor_component
).all()
assert np.isfinite(
    phase33_monitor_weights
).all()

phase33_baseline_robust = phase33g_robust_score(
    phase33_monitor_labels,
    phase33_monitor_baseline,
    phase33_monitor_groups,
    phase33_monitor_weights,
)

# Same grid declared before the Phase33 diagnostic.
phase33_shared_gate_candidates = [{
    "alpha": 0.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 0.0,
}]

for alpha in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
    for uncertainty_exponent in [0.0, 0.5, 1.0, 2.0]:
        for delta_cap in [0.5, 1.0, 1.5, 2.0]:
            phase33_shared_gate_candidates.append({
                "alpha": float(alpha),
                "uncertainty_exponent": float(
                    uncertainty_exponent
                ),
                "delta_cap": float(delta_cap),
            })

phase33_shared_candidate_records = []

for specification in phase33_shared_gate_candidates:
    candidate_probability = (
        phase33d_bounded_update(
            phase33_monitor_baseline,
            phase33_monitor_component,
            specification["alpha"],
            specification[
                "uncertainty_exponent"
            ],
            specification["delta_cap"],
        )
    )

    candidate_robust = phase33g_robust_score(
        phase33_monitor_labels,
        candidate_probability,
        phase33_monitor_groups,
        phase33_monitor_weights,
    )

    overall_gain = (
        phase33_baseline_robust[
            "overall_log_loss"
        ]
        - candidate_robust[
            "overall_log_loss"
        ]
    )
    robust_gain = (
        phase33_baseline_robust["robust_score"]
        - candidate_robust["robust_score"]
    )

    fold_records = []

    for fold in range(3):
        fold_mask = (
            phase33_monitor_folds == fold
        )

        baseline_fold_loss = (
            phase33g_weighted_log_loss(
                phase33_monitor_labels[fold_mask],
                phase33_monitor_baseline[fold_mask],
                phase33_monitor_weights[fold_mask],
            )
        )
        candidate_fold_loss = (
            phase33g_weighted_log_loss(
                phase33_monitor_labels[fold_mask],
                candidate_probability[fold_mask],
                phase33_monitor_weights[fold_mask],
            )
        )

        fold_records.append({
            "fold": fold,
            "baseline_log_loss": baseline_fold_loss,
            "candidate_log_loss": candidate_fold_loss,
            "gain": (
                baseline_fold_loss
                - candidate_fold_loss
            ),
        })

    fold_win_count = sum(
        record["gain"] >= 0.0005
        for record in fold_records
    )
    worst_fold_regret = max(
        -record["gain"]
        for record in fold_records
    )

    maximum_group_harm, group_records = (
        phase33g_monitor_group_harm(
            phase33_monitor_labels,
            phase33_monitor_baseline,
            candidate_probability,
            phase33_monitor_groups,
            phase33_monitor_weights,
        )
    )

    leave_one_group_out_gains = []

    for held_out_group in np.unique(
        phase33_monitor_groups
    ):
        retained = (
            phase33_monitor_groups
            != held_out_group
        )

        if int(np.sum(retained)) < 32:
            continue

        baseline_loo = phase33g_weighted_log_loss(
            phase33_monitor_labels[retained],
            phase33_monitor_baseline[retained],
            phase33_monitor_weights[retained],
        )
        candidate_loo = phase33g_weighted_log_loss(
            phase33_monitor_labels[retained],
            candidate_probability[retained],
            phase33_monitor_weights[retained],
        )

        leave_one_group_out_gains.append(
            baseline_loo - candidate_loo
        )

    if leave_one_group_out_gains:
        median_loo_gain = float(
            np.median(leave_one_group_out_gains)
        )
        worst_loo_regret = float(
            max(
                -gain
                for gain in (
                    leave_one_group_out_gains
                )
            )
        )
    else:
        median_loo_gain = overall_gain
        worst_loo_regret = 0.0

    eligible = bool(
        specification["alpha"] > 0.0
        and overall_gain >= 0.002
        and robust_gain >= 0.002
        and fold_win_count >= 2
        and worst_fold_regret <= 0.005
        and maximum_group_harm <= 0.015
        and median_loo_gain >= 0.001
        and worst_loo_regret <= 0.005
    )

    phase33_shared_candidate_records.append({
        "specification": dict(specification),
        "probability": candidate_probability,
        "robust": candidate_robust,
        "overall_gain": float(overall_gain),
        "robust_gain": float(robust_gain),
        "fold_records": fold_records,
        "fold_win_count": int(fold_win_count),
        "worst_fold_regret": float(
            worst_fold_regret
        ),
        "maximum_group_harm": float(
            maximum_group_harm
        ),
        "group_records": group_records,
        "median_leave_one_group_out_gain": float(
            median_loo_gain
        ),
        "worst_leave_one_group_out_regret": float(
            worst_loo_regret
        ),
        "eligible": eligible,
    })

phase33_raw_best_record = min(
    phase33_shared_candidate_records,
    key=lambda record: (
        record["robust"]["robust_score"],
        record["robust"]["overall_log_loss"],
        record["specification"]["alpha"],
        record["specification"][
            "uncertainty_exponent"
        ],
        record["specification"]["delta_cap"],
    ),
)

phase33_eligible_records = [
    record
    for record in phase33_shared_candidate_records
    if record["eligible"]
]

if phase33_eligible_records:
    phase33_selected_record = min(
        phase33_eligible_records,
        key=lambda record: (
            record["robust"]["robust_score"],
            record["robust"][
                "overall_log_loss"
            ],
            record["specification"]["alpha"],
            record["specification"][
                "uncertainty_exponent"
            ],
            record["specification"]["delta_cap"],
        ),
    )
    phase33_gate_advanced = True
else:
    phase33_selected_record = next(
        record
        for record in (
            phase33_shared_candidate_records
        )
        if record["specification"]["alpha"] == 0.0
    )
    phase33_gate_advanced = False

PHASE33_SHARED_GATE_PRIVATE = dict(
    phase33_selected_record["specification"]
)

print(
    "Phase33 shared monitor gate: "
    f"alpha={PHASE33_SHARED_GATE_PRIVATE['alpha']:.2f}, "
    "gamma="
    f"{PHASE33_SHARED_GATE_PRIVATE['uncertainty_exponent']:.2f}, "
    f"cap={PHASE33_SHARED_GATE_PRIVATE['delta_cap']:.2f}, "
    f"advanced={phase33_gate_advanced}"
)

# ---------------------------------------------------------
# Gate is now frozen. Evaluate once on outer OOF predictions.
# ---------------------------------------------------------

phase33_final_baseline = phase33d_probability(
    PHASE19_PHASE12C_OOF_PRIVATE
)
phase33_final_component = phase33d_probability(
    PHASE33_OOF_PRIVATE
)

PHASE33_GATED_OOF_PRIVATE = (
    phase33d_bounded_update(
        phase33_final_baseline,
        phase33_final_component,
        PHASE33_SHARED_GATE_PRIVATE["alpha"],
        PHASE33_SHARED_GATE_PRIVATE[
            "uncertainty_exponent"
        ],
        PHASE33_SHARED_GATE_PRIVATE["delta_cap"],
    )
)

phase33_final_baseline_metrics = phase33d_metrics(
    phase33_gate_labels,
    phase33_final_baseline,
)
phase33_final_component_metrics = phase33d_metrics(
    phase33_gate_labels,
    phase33_final_component,
)
phase33_final_gated_metrics = phase33d_metrics(
    phase33_gate_labels,
    PHASE33_GATED_OOF_PRIVATE,
)

phase33_final_calibration = (
    phase33g_calibration_statistics(
        phase33_gate_labels,
        PHASE33_GATED_OOF_PRIVATE,
    )
)

phase33_final_fold_records = []

for partition in PHASE19_PARTITIONS:
    fold = int(partition["fold"])
    outer_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    baseline_metrics = phase33d_metrics(
        phase33_gate_labels[outer_indices],
        phase33_final_baseline[outer_indices],
    )
    component_metrics = phase33d_metrics(
        phase33_gate_labels[outer_indices],
        phase33_final_component[outer_indices],
    )
    gated_metrics = phase33d_metrics(
        phase33_gate_labels[outer_indices],
        PHASE33_GATED_OOF_PRIVATE[
            outer_indices
        ],
    )

    phase33_final_fold_records.append({
        "fold": fold,
        "n": int(outer_indices.size),
        "baseline_log_loss": round(
            baseline_metrics["log_loss"], 6
        ),
        "component_log_loss": round(
            component_metrics["log_loss"], 6
        ),
        "gated_log_loss": round(
            gated_metrics["log_loss"], 6
        ),
        "log_loss_improvement": round(
            baseline_metrics["log_loss"]
            - gated_metrics["log_loss"],
            6,
        ),
        "baseline_auroc": round(
            baseline_metrics["auroc"], 6
        ),
        "component_auroc": round(
            component_metrics["auroc"], 6
        ),
        "gated_auroc": round(
            gated_metrics["auroc"], 6
        ),
    })

phase33_final_fold_wins = sum(
    record["log_loss_improvement"] > 1e-8
    for record in phase33_final_fold_records
)
phase33_final_worst_fold_excess = max(
    -record["log_loss_improvement"]
    for record in phase33_final_fold_records
)

(
    phase33_final_maximum_group_harm,
    phase33_final_group_records,
) = phase33d_group_harm(
    phase33_gate_labels,
    phase33_final_baseline,
    PHASE33_GATED_OOF_PRIVATE,
    phase33_gate_groups,
)

phase33_final_log_loss_gain = (
    phase33_final_baseline_metrics["log_loss"]
    - phase33_final_gated_metrics["log_loss"]
)
phase33_final_auroc_gain = (
    phase33_final_gated_metrics["auroc"]
    - phase33_final_baseline_metrics["auroc"]
)
phase33_final_brier_excess = (
    phase33_final_gated_metrics["brier"]
    - phase33_final_baseline_metrics["brier"]
)

phase33_promotion_gate_passed = bool(
    phase33_gate_advanced
    and phase33_final_log_loss_gain >= 0.003
    and phase33_final_auroc_gain >= 0.002
    and phase33_final_fold_wins >= 2
    and phase33_final_worst_fold_excess <= 0.005
    and phase33_final_brier_excess <= 0.001
    and 0.8
    <= phase33_final_calibration[
        "calibration_slope"
    ]
    <= 1.2
    and phase33_final_maximum_group_harm <= 0.015
)

PHASE33_SHARED_GATE_DEPLOYMENT_PRIVATE = {
    "gate": dict(PHASE33_SHARED_GATE_PRIVATE),
    "phase33_states": (
        PHASE33_DEPLOYMENT_STATES_PRIVATE
    ),
    "shared_across_folds": True,
    "fold_specific_gate_routing": False,
}

phase33_gate_report = {
    "phase": (
        "phase33_shared_monitor_selected_"
        "bounded_residual"
    ),
    "status": (
        "promoted"
        if phase33_promotion_gate_passed
        else "not_promoted"
    ),
    "monitor_pool": {
        "observation_count": int(
            phase33_monitor_indices.size
        ),
        "unique_case_count": int(
            unique_monitor_indices.size
        ),
        "maximum_case_multiplicity": int(
            np.max(monitor_multiplicities)
        ),
        "inverse_multiplicity_weighting": True,
        "acquisition_group_count": int(
            np.unique(
                phase33_monitor_groups
            ).size
        ),
    },
    "selection": {
        "candidate_count": int(
            len(phase33_shared_gate_candidates)
        ),
        "eligible_candidate_count": int(
            len(phase33_eligible_records)
        ),
        "raw_best_specification": (
            phase33_raw_best_record[
                "specification"
            ]
        ),
        "raw_best_overall_gain": round(
            phase33_raw_best_record[
                "overall_gain"
            ],
            6,
        ),
        "raw_best_robust_gain": round(
            phase33_raw_best_record[
                "robust_gain"
            ],
            6,
        ),
        "selected_specification": dict(
            PHASE33_SHARED_GATE_PRIVATE
        ),
        "gate_advanced": (
            phase33_gate_advanced
        ),
        "selected_overall_monitor_gain": round(
            phase33_selected_record[
                "overall_gain"
            ],
            6,
        ),
        "selected_robust_monitor_gain": round(
            phase33_selected_record[
                "robust_gain"
            ],
            6,
        ),
        "selected_monitor_fold_wins": int(
            phase33_selected_record[
                "fold_win_count"
            ]
        ),
        "selected_worst_monitor_fold_regret": (
            round(
                phase33_selected_record[
                    "worst_fold_regret"
                ],
                6,
            )
        ),
        "selected_maximum_monitor_group_harm": (
            round(
                phase33_selected_record[
                    "maximum_group_harm"
                ],
                6,
            )
        ),
        "selected_median_leave_one_group_out_gain": (
            round(
                phase33_selected_record[
                    "median_leave_one_group_out_gain"
                ],
                6,
            )
        ),
        "selected_worst_leave_one_group_out_regret": (
            round(
                phase33_selected_record[
                    "worst_leave_one_group_out_regret"
                ],
                6,
            )
        ),
        "fold_monitor_metrics": [
            {
                "fold": int(record["fold"]),
                "baseline_log_loss": round(
                    record[
                        "baseline_log_loss"
                    ],
                    6,
                ),
                "candidate_log_loss": round(
                    record[
                        "candidate_log_loss"
                    ],
                    6,
                ),
                "gain": round(
                    record["gain"], 6
                ),
            }
            for record in (
                phase33_selected_record[
                    "fold_records"
                ]
            )
        ],
    },
    "baseline": {
        key: (
            None if value is None
            else round(float(value), 6)
        )
        for key, value in (
            phase33_final_baseline_metrics.items()
        )
    },
    "phase33_component": {
        key: (
            None if value is None
            else round(float(value), 6)
        )
        for key, value in (
            phase33_final_component_metrics.items()
        )
    },
    "phase33_gated": {
        **{
            key: (
                None if value is None
                else round(float(value), 6)
            )
            for key, value in (
                phase33_final_gated_metrics.items()
            )
        },
        "calibration_intercept": round(
            phase33_final_calibration[
                "calibration_intercept"
            ],
            6,
        ),
        "calibration_slope": round(
            phase33_final_calibration[
                "calibration_slope"
            ],
            6,
        ),
    },
    "improvements": {
        "log_loss_gain": round(
            phase33_final_log_loss_gain, 6
        ),
        "auroc_gain": round(
            phase33_final_auroc_gain, 6
        ),
        "brier_excess": round(
            phase33_final_brier_excess, 6
        ),
        "fold_wins": int(
            phase33_final_fold_wins
        ),
        "worst_fold_excess": round(
            phase33_final_worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            phase33_final_maximum_group_harm,
            6,
        ),
    },
    "fold_metrics": (
        phase33_final_fold_records
    ),
    "major_acquisition_group_metrics": (
        phase33_final_group_records
    ),
    "promotion_thresholds": {
        "minimum_log_loss_gain": 0.003,
        "minimum_auroc_gain": 0.002,
        "minimum_fold_wins": 2,
        "maximum_worst_fold_excess": 0.005,
        "maximum_brier_excess": 0.001,
        "calibration_slope_range": [
            0.8,
            1.2,
        ],
        "maximum_major_group_harm": 0.015,
    },
    "promotion_gate_passed": (
        phase33_promotion_gate_passed
    ),
    "shared_gate_across_all_folds": True,
    "fold_specific_gate_routing": False,
    "outer_oracle_parameters_used": False,
    "selection_partition": (
        "pooled_monitor_only"
    ),
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase33_gate_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE33_SHARED_GATE")
print(json.dumps(
    phase33_gate_report,
    indent=2,
))
print("END SANITIZED_PHASE33_SHARED_GATE")

Phase33 shared monitor gate: alpha=0.30, gamma=0.50, cap=2.00, advanced=True
BEGIN SANITIZED_PHASE33_SHARED_GATE
{
  "phase": "phase33_shared_monitor_selected_bounded_residual",
  "status": "promoted",
  "monitor_pool": {
    "observation_count": 493,
    "unique_case_count": 407,
    "maximum_case_multiplicity": 2,
    "inverse_multiplicity_weighting": true,
    "acquisition_group_count": 10
  },
  "selection": {
    "candidate_count": 97,
    "eligible_candidate_count": 29,
    "raw_best_specification": {
      "alpha": 0.4,
      "uncertainty_exponent": 0.0,
      "delta_cap": 2.0
    },
    "raw_best_overall_gain": 0.00457,
    "raw_best_robust_gain": 0.010326,
    "selected_specification": {
      "alpha": 0.3,
      "uncertainty_exponent": 0.5,
      "delta_cap": 2.0
    },
    "gate_advanced": true,
    "selected_overall_monitor_gain": 0.004053,
    "selected_robust_monitor_gain": 0.007389,
    "selected_monitor_fold_wins": 2,
    "selected_worst_monitor_fold_regret": 0.004682,


In [47]:
# Phase33 Cell 113A
# Atomically persist all Phase33 private states and predictions.
# These files are notebook checkpoints and must not enter submission.zip.

from pathlib import Path
import copy
import hashlib
import json
import os
import time

import numpy as np
import torch


PHASE33_CHECKPOINT_ROOT = Path(
    "/kaggle/working/phase33_private_checkpoint"
)
PHASE33_CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PHASE33_STATE_PATH = (
    PHASE33_CHECKPOINT_ROOT
    / "phase33_deployment_states.pt"
)
PHASE33_OOF_PATH = (
    PHASE33_CHECKPOINT_ROOT
    / "phase33_component_oof_float64.npy"
)
PHASE33_GATED_OOF_PATH = (
    PHASE33_CHECKPOINT_ROOT
    / "phase33_gated_oof_float64.npy"
)
PHASE33_MONITOR_PATH = (
    PHASE33_CHECKPOINT_ROOT
    / "phase33_monitor_predictions.pt"
)
PHASE33_METADATA_PATH = (
    PHASE33_CHECKPOINT_ROOT
    / "phase33_checkpoint_metadata.json"
)


def phase33p_cpu_copy(value):
    if isinstance(value, torch.nn.Module):
        module = copy.deepcopy(value)
        module.cpu()
        module.eval()

        for parameter in module.parameters():
            parameter.requires_grad_(False)

        return module

    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: phase33p_cpu_copy(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            phase33p_cpu_copy(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            phase33p_cpu_copy(item)
            for item in value
        )

    return copy.deepcopy(value)


def phase33p_tensor_audit(value):
    tensor_count = 0
    parameter_count = 0
    module_count = 0
    all_finite = True

    def visit(item):
        nonlocal tensor_count
        nonlocal parameter_count
        nonlocal module_count
        nonlocal all_finite

        if isinstance(item, torch.nn.Module):
            module_count += 1

            for tensor in item.state_dict().values():
                if not torch.is_tensor(tensor):
                    continue

                tensor_count += 1
                parameter_count += int(
                    tensor.numel()
                )

                if (
                    tensor.is_floating_point()
                    or tensor.is_complex()
                ):
                    all_finite = bool(
                        all_finite
                        and torch.isfinite(
                            tensor
                        ).all().item()
                    )

            return

        if torch.is_tensor(item):
            tensor_count += 1
            parameter_count += int(item.numel())

            if (
                item.is_floating_point()
                or item.is_complex()
            ):
                all_finite = bool(
                    all_finite
                    and torch.isfinite(
                        item
                    ).all().item()
                )

            return

        if isinstance(item, dict):
            for nested in item.values():
                visit(nested)
            return

        if isinstance(item, (list, tuple)):
            for nested in item:
                visit(nested)

    visit(value)

    return {
        "module_count": int(module_count),
        "tensor_count": int(tensor_count),
        "tensor_parameter_count": int(
            parameter_count
        ),
        "all_tensors_finite": bool(all_finite),
    }


def phase33p_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(1024 * 1024)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


phase33_persist_started = time.perf_counter()

assert "PHASE33_DEPLOYMENT_STATES_PRIVATE" in globals()
assert "PHASE33_MONITOR_PROBABILITIES_PRIVATE" in globals()
assert "PHASE33_OOF_PRIVATE" in globals()
assert "PHASE33_GATED_OOF_PRIVATE" in globals()
assert "PHASE33_SHARED_GATE_PRIVATE" in globals()

phase33_cpu_states = phase33p_cpu_copy(
    PHASE33_DEPLOYMENT_STATES_PRIVATE
)
phase33_cpu_monitors = phase33p_cpu_copy(
    PHASE33_MONITOR_PROBABILITIES_PRIVATE
)

phase33_state_payload = {
    "schema_version": 1,
    "phase": (
        "phase33_domain_invariant_"
        "geometry_semantic_fusion"
    ),
    "deployment_states": phase33_cpu_states,
    "shared_gate": dict(
        PHASE33_SHARED_GATE_PRIVATE
    ),
    "training_configuration": dict(
        PHASE33_TRAIN_CONFIG
    ),
    "model_class": (
        "Phase33DomainInvariantFusionNet"
    ),
    "phase12c_baseline": {
        "log_loss": 0.307616,
        "auroc": 0.938914,
        "brier": 0.095535,
    },
    "phase33_gated_metrics": {
        "log_loss": 0.302954,
        "auroc": 0.940945,
        "brier": 0.094159,
    },
}

state_partial = PHASE33_STATE_PATH.with_suffix(
    ".partial.pt"
)
monitor_partial = PHASE33_MONITOR_PATH.with_suffix(
    ".partial.pt"
)

torch.save(
    phase33_state_payload,
    state_partial,
)
torch.save(
    {
        "schema_version": 1,
        "monitor_predictions": (
            phase33_cpu_monitors
        ),
    },
    monitor_partial,
)

os.replace(
    state_partial,
    PHASE33_STATE_PATH,
)
os.replace(
    monitor_partial,
    PHASE33_MONITOR_PATH,
)

np.save(
    PHASE33_OOF_PATH,
    np.asarray(
        PHASE33_OOF_PRIVATE,
        dtype=np.float64,
    ),
    allow_pickle=False,
)
np.save(
    PHASE33_GATED_OOF_PATH,
    np.asarray(
        PHASE33_GATED_OOF_PRIVATE,
        dtype=np.float64,
    ),
    allow_pickle=False,
)

# Reload immediately to validate serialization.
phase33_reloaded_payload = torch.load(
    PHASE33_STATE_PATH,
    map_location="cpu",
    weights_only=False,
)
phase33_reloaded_monitor = torch.load(
    PHASE33_MONITOR_PATH,
    map_location="cpu",
    weights_only=False,
)

phase33_reloaded_oof = np.load(
    PHASE33_OOF_PATH,
    allow_pickle=False,
)
phase33_reloaded_gated = np.load(
    PHASE33_GATED_OOF_PATH,
    allow_pickle=False,
)

assert (
    phase33_reloaded_payload["schema_version"]
    == 1
)
assert (
    phase33_reloaded_payload["shared_gate"]
    == PHASE33_SHARED_GATE_PRIVATE
)
assert phase33_reloaded_oof.shape == (1362,)
assert phase33_reloaded_gated.shape == (1362,)

phase33_oof_reconstruction_error = float(
    np.max(
        np.abs(
            phase33_reloaded_oof
            - np.asarray(
                PHASE33_OOF_PRIVATE,
                dtype=np.float64,
            )
        )
    )
)
phase33_gated_reconstruction_error = float(
    np.max(
        np.abs(
            phase33_reloaded_gated
            - np.asarray(
                PHASE33_GATED_OOF_PRIVATE,
                dtype=np.float64,
            )
        )
    )
)

assert phase33_oof_reconstruction_error == 0.0
assert phase33_gated_reconstruction_error == 0.0

phase33_tensor_audit = phase33p_tensor_audit(
    phase33_reloaded_payload[
        "deployment_states"
    ]
)
assert phase33_tensor_audit[
    "all_tensors_finite"
]

phase33_checkpoint_metadata = {
    "schema_version": 1,
    "phase": "phase33_private_checkpoint",
    "shared_gate": dict(
        PHASE33_SHARED_GATE_PRIVATE
    ),
    "files": {
        "deployment_states": (
            PHASE33_STATE_PATH.name
        ),
        "monitor_predictions": (
            PHASE33_MONITOR_PATH.name
        ),
        "component_oof": (
            PHASE33_OOF_PATH.name
        ),
        "gated_oof": (
            PHASE33_GATED_OOF_PATH.name
        ),
    },
    "sha256": {
        "deployment_states": (
            phase33p_sha256(
                PHASE33_STATE_PATH
            )
        ),
        "monitor_predictions": (
            phase33p_sha256(
                PHASE33_MONITOR_PATH
            )
        ),
        "component_oof": (
            phase33p_sha256(
                PHASE33_OOF_PATH
            )
        ),
        "gated_oof": (
            phase33p_sha256(
                PHASE33_GATED_OOF_PATH
            )
        ),
    },
}

metadata_partial = (
    PHASE33_METADATA_PATH.with_suffix(
        ".partial.json"
    )
)
metadata_partial.write_text(
    json.dumps(
        phase33_checkpoint_metadata,
        indent=2,
    ),
    encoding="utf-8",
)
os.replace(
    metadata_partial,
    PHASE33_METADATA_PATH,
)

phase33_checkpoint_size_mb = sum(
    path.stat().st_size
    for path in [
        PHASE33_STATE_PATH,
        PHASE33_MONITOR_PATH,
        PHASE33_OOF_PATH,
        PHASE33_GATED_OOF_PATH,
        PHASE33_METADATA_PATH,
    ]
) / (1024 ** 2)

phase33_persist_report = {
    "phase": "phase33_private_state_persistence",
    "status": "accepted",
    "checkpoint_directory": (
        PHASE33_CHECKPOINT_ROOT.name
    ),
    "file_count": 5,
    "checkpoint_size_mb": round(
        phase33_checkpoint_size_mb,
        3,
    ),
    "shared_gate": dict(
        PHASE33_SHARED_GATE_PRIVATE
    ),
    "tensor_audit": phase33_tensor_audit,
    "component_oof_shape": list(
        phase33_reloaded_oof.shape
    ),
    "gated_oof_shape": list(
        phase33_reloaded_gated.shape
    ),
    "component_oof_reconstruction_error": (
        phase33_oof_reconstruction_error
    ),
    "gated_oof_reconstruction_error": (
        phase33_gated_reconstruction_error
    ),
    "monitor_predictions_reloaded": bool(
        "monitor_predictions"
        in phase33_reloaded_monitor
    ),
    "atomic_export": True,
    "persistent_notebook_checkpoint_only": True,
    "include_in_submission": False,
    "model_hashes_displayed": False,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase33_persist_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE33_PERSISTENCE")
print(json.dumps(
    phase33_persist_report,
    indent=2,
))
print("END SANITIZED_PHASE33_PERSISTENCE")

BEGIN SANITIZED_PHASE33_PERSISTENCE
{
  "phase": "phase33_private_state_persistence",
  "status": "accepted",
  "checkpoint_directory": "phase33_private_checkpoint",
  "file_count": 5,
  "checkpoint_size_mb": 6.472,
  "shared_gate": {
    "alpha": 0.3,
    "uncertainty_exponent": 0.5,
    "delta_cap": 2.0
  },
  "tensor_audit": {
    "module_count": 0,
    "tensor_count": 657,
    "tensor_parameter_count": 1173195,
    "all_tensors_finite": true
  },
  "component_oof_shape": [
    1362
  ],
  "gated_oof_shape": [
    1362
  ],
  "component_oof_reconstruction_error": 0.0,
  "gated_oof_reconstruction_error": 0.0,
  "monitor_predictions_reloaded": true,
  "atomic_export": true,
  "persistent_notebook_checkpoint_only": true,
  "include_in_submission": false,
  "model_hashes_displayed": false,
  "case_level_predictions_exported": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "elapsed_seconds": 0.14
}
END SANITIZED_PHASE33_PERSISTENCE


In [48]:
# Phase33 Cell 113B
# Fixed-candidate robustness stress test.
#
# The Phase33 gate is already frozen. This cell does not select
# or modify parameters. Outer labels are used only to estimate
# robustness of the promoted candidate.

import json
import time

import numpy as np
from sklearn.metrics import roc_auc_score


PHASE33_STRESS_CONFIG = {
    "random_seed": 331301,
    "within_group_bootstrap_replicates": 5000,
    "auroc_bootstrap_replicates": 2500,
    "dirichlet_replicates": 10000,
    "dirichlet_concentrations": [
        100.0,
        20.0,
        5.0,
    ],
    "total_variation_radii": [
        0.05,
        0.10,
        0.20,
        0.30,
    ],
}


def phase33s_case_log_loss(
    labels,
    probabilities,
):
    labels = np.asarray(
        labels,
        dtype=np.float64,
    )
    probabilities = phase33d_probability(
        probabilities
    )

    return -(
        labels * np.log(probabilities)
        + (1.0 - labels)
        * np.log1p(-probabilities)
    )


def phase33s_quantile_report(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "mean": round(float(np.mean(values)), 6),
        "standard_deviation": round(
            float(np.std(values)),
            6,
        ),
        "q01": round(
            float(np.quantile(values, 0.01)),
            6,
        ),
        "q05": round(
            float(np.quantile(values, 0.05)),
            6,
        ),
        "q50": round(
            float(np.quantile(values, 0.50)),
            6,
        ),
        "q95": round(
            float(np.quantile(values, 0.95)),
            6,
        ),
        "q99": round(
            float(np.quantile(values, 0.99)),
            6,
        ),
        "probability_positive": round(
            float(np.mean(values > 0.0)),
            6,
        ),
        "probability_gain_at_least_0p003": round(
            float(np.mean(values >= 0.003)),
            6,
        ),
    }


def phase33s_adversarial_group_mixture(
    empirical_weights,
    group_gains,
    total_variation_radius,
):
    # Moving mass m between groups changes total variation by m.
    weights = np.asarray(
        empirical_weights,
        dtype=np.float64,
    ).copy()
    gains = np.asarray(
        group_gains,
        dtype=np.float64,
    )

    remaining_budget = float(
        total_variation_radius
    )

    donors = np.argsort(-gains)
    recipients = np.argsort(gains)

    donor_position = 0
    recipient_position = 0

    while (
        remaining_budget > 1e-12
        and donor_position < donors.size
        and recipient_position < recipients.size
    ):
        donor = int(donors[donor_position])
        recipient = int(
            recipients[recipient_position]
        )

        if gains[donor] <= gains[recipient]:
            break

        if weights[donor] <= 1e-12:
            donor_position += 1
            continue

        recipient_capacity = (
            1.0 - weights[recipient]
        )

        if recipient_capacity <= 1e-12:
            recipient_position += 1
            continue

        moved_mass = min(
            remaining_budget,
            weights[donor],
            recipient_capacity,
        )

        weights[donor] -= moved_mass
        weights[recipient] += moved_mass
        remaining_budget -= moved_mass

        if weights[donor] <= 1e-12:
            donor_position += 1

        if weights[recipient] >= 1.0 - 1e-12:
            recipient_position += 1

    return {
        "gain": float(np.dot(weights, gains)),
        "weights": weights,
        "unused_budget": float(
            remaining_budget
        ),
    }


phase33_stress_started = time.perf_counter()
phase33_stress_rng = np.random.default_rng(
    PHASE33_STRESS_CONFIG["random_seed"]
)

phase33_stress_labels = np.asarray(
    case_df["label"],
    dtype=np.int64,
)
phase33_stress_groups = np.asarray(
    case_df["acquisition_group"],
    dtype=np.int64,
)
phase33_stress_baseline = phase33d_probability(
    PHASE19_PHASE12C_OOF_PRIVATE
)
phase33_stress_candidate = phase33d_probability(
    PHASE33_GATED_OOF_PRIVATE
)

assert phase33_stress_labels.shape == (1362,)
assert phase33_stress_groups.shape == (1362,)
assert phase33_stress_baseline.shape == (1362,)
assert phase33_stress_candidate.shape == (1362,)

baseline_case_loss = phase33s_case_log_loss(
    phase33_stress_labels,
    phase33_stress_baseline,
)
candidate_case_loss = phase33s_case_log_loss(
    phase33_stress_labels,
    phase33_stress_candidate,
)

# Positive values mean Phase33 is better.
phase33_case_gain = (
    baseline_case_loss
    - candidate_case_loss
)

phase33_observed_gain = float(
    np.mean(phase33_case_gain)
)

unique_groups = np.asarray(
    sorted(
        np.unique(
            phase33_stress_groups
        ).tolist()
    ),
    dtype=np.int64,
)

group_indices = {
    int(group): np.flatnonzero(
        phase33_stress_groups == group
    )
    for group in unique_groups
}

group_counts = np.asarray([
    group_indices[int(group)].size
    for group in unique_groups
], dtype=np.int64)

group_proportions = (
    group_counts.astype(np.float64)
    / np.sum(group_counts)
)

group_mean_gains = np.asarray([
    np.mean(
        phase33_case_gain[
            group_indices[int(group)]
        ]
    )
    for group in unique_groups
], dtype=np.float64)

phase33_group_records = []

for group, count, proportion, gain in zip(
    unique_groups,
    group_counts,
    group_proportions,
    group_mean_gains,
):
    indices = group_indices[int(group)]

    phase33_group_records.append({
        "group": int(group),
        "n": int(count),
        "empirical_weight": round(
            float(proportion),
            6,
        ),
        "log_loss_gain": round(
            float(gain),
            6,
        ),
        "candidate_better": bool(gain > 0.0),
        "prevalence": round(
            float(
                np.mean(
                    phase33_stress_labels[
                        indices
                    ]
                )
            ),
            6,
        ),
    })

# ---------------------------------------------------------
# 1. Within-acquisition-group bootstrap.
# This preserves the observed group proportions.
# ---------------------------------------------------------

within_group_replicates = int(
    PHASE33_STRESS_CONFIG[
        "within_group_bootstrap_replicates"
    ]
)

within_group_bootstrap_gains = np.empty(
    within_group_replicates,
    dtype=np.float64,
)

for replicate in range(
    within_group_replicates
):
    gain_sum = 0.0
    sampled_count = 0

    for group in unique_groups:
        indices = group_indices[int(group)]
        sampled = phase33_stress_rng.choice(
            indices,
            size=indices.size,
            replace=True,
        )

        gain_sum += float(
            np.sum(
                phase33_case_gain[sampled]
            )
        )
        sampled_count += int(sampled.size)

    within_group_bootstrap_gains[
        replicate
    ] = gain_sum / sampled_count

within_group_bootstrap_report = (
    phase33s_quantile_report(
        within_group_bootstrap_gains
    )
)

# ---------------------------------------------------------
# 2. Stratified AUROC bootstrap.
# Resample positive and negative cases separately.
# ---------------------------------------------------------

positive_indices = np.flatnonzero(
    phase33_stress_labels == 1
)
negative_indices = np.flatnonzero(
    phase33_stress_labels == 0
)

auroc_replicates = int(
    PHASE33_STRESS_CONFIG[
        "auroc_bootstrap_replicates"
    ]
)

auroc_gain_replicates = np.empty(
    auroc_replicates,
    dtype=np.float64,
)

for replicate in range(auroc_replicates):
    sampled_positive = (
        phase33_stress_rng.choice(
            positive_indices,
            size=positive_indices.size,
            replace=True,
        )
    )
    sampled_negative = (
        phase33_stress_rng.choice(
            negative_indices,
            size=negative_indices.size,
            replace=True,
        )
    )

    sampled = np.concatenate([
        sampled_positive,
        sampled_negative,
    ])

    sampled_labels = (
        phase33_stress_labels[sampled]
    )

    baseline_auroc = roc_auc_score(
        sampled_labels,
        phase33_stress_baseline[sampled],
    )
    candidate_auroc = roc_auc_score(
        sampled_labels,
        phase33_stress_candidate[sampled],
    )

    auroc_gain_replicates[replicate] = (
        candidate_auroc
        - baseline_auroc
    )

auroc_bootstrap_report = (
    phase33s_quantile_report(
        auroc_gain_replicates
    )
)

# ---------------------------------------------------------
# 3. Leave-one-acquisition-group-out stability.
# ---------------------------------------------------------

leave_one_group_out_records = []

for held_out_group in unique_groups:
    retained = (
        phase33_stress_groups
        != held_out_group
    )

    retained_gain = float(
        np.mean(
            phase33_case_gain[retained]
        )
    )

    leave_one_group_out_records.append({
        "held_out_group": int(
            held_out_group
        ),
        "retained_n": int(
            np.sum(retained)
        ),
        "log_loss_gain": round(
            retained_gain,
            6,
        ),
    })

minimum_leave_one_group_out_gain = min(
    record["log_loss_gain"]
    for record in leave_one_group_out_records
)

# ---------------------------------------------------------
# 4. Dirichlet acquisition-mixture simulations.
#
# Concentration 100: modest deviation from train mixture.
# Concentration 20: substantial domain-mixture shift.
# Concentration 5: severe stress condition.
# ---------------------------------------------------------

dirichlet_records = []
dirichlet_replicates = int(
    PHASE33_STRESS_CONFIG[
        "dirichlet_replicates"
    ]
)

for concentration in (
    PHASE33_STRESS_CONFIG[
        "dirichlet_concentrations"
    ]
):
    concentration_vector = (
        float(concentration)
        * group_proportions
        + 0.20
    )

    sampled_weights = (
        phase33_stress_rng.dirichlet(
            concentration_vector,
            size=dirichlet_replicates,
        )
    )

    mixture_gains = (
        sampled_weights
        @ group_mean_gains
    )

    dirichlet_records.append({
        "concentration": float(
            concentration
        ),
        "interpretation": (
            "modest_shift"
            if concentration >= 100.0
            else (
                "substantial_shift"
                if concentration >= 20.0
                else "severe_shift"
            )
        ),
        **phase33s_quantile_report(
            mixture_gains
        ),
    })

# ---------------------------------------------------------
# 5. Deterministic adversarial group-mixture shift.
# ---------------------------------------------------------

adversarial_mixture_records = []

for radius in (
    PHASE33_STRESS_CONFIG[
        "total_variation_radii"
    ]
):
    adversarial = (
        phase33s_adversarial_group_mixture(
            group_proportions,
            group_mean_gains,
            radius,
        )
    )

    changed_weight_count = int(
        np.sum(
            np.abs(
                adversarial["weights"]
                - group_proportions
            )
            > 1e-8
        )
    )

    adversarial_mixture_records.append({
        "total_variation_radius": float(
            radius
        ),
        "worst_case_log_loss_gain": round(
            adversarial["gain"],
            6,
        ),
        "changed_group_count": (
            changed_weight_count
        ),
        "unused_shift_budget": round(
            adversarial["unused_budget"],
            8,
        ),
    })

# ---------------------------------------------------------
# 6. Effective gate/update magnitude.
# ---------------------------------------------------------

selected_alpha = float(
    PHASE33_SHARED_GATE_PRIVATE["alpha"]
)
selected_gamma = float(
    PHASE33_SHARED_GATE_PRIVATE[
        "uncertainty_exponent"
    ]
)

effective_alpha = (
    selected_alpha
    * (
        4.0
        * phase33_stress_baseline
        * (1.0 - phase33_stress_baseline)
    ) ** selected_gamma
)

logit_update = (
    phase33d_logit(
        phase33_stress_candidate
    )
    - phase33d_logit(
        phase33_stress_baseline
    )
)

probability_update = (
    phase33_stress_candidate
    - phase33_stress_baseline
)

update_report = {
    "effective_alpha_quantiles": {
        "q05": round(
            float(
                np.quantile(
                    effective_alpha, 0.05
                )
            ),
            6,
        ),
        "q50": round(
            float(
                np.quantile(
                    effective_alpha, 0.50
                )
            ),
            6,
        ),
        "q95": round(
            float(
                np.quantile(
                    effective_alpha, 0.95
                )
            ),
            6,
        ),
        "maximum": round(
            float(np.max(effective_alpha)),
            6,
        ),
    },
    "mean_absolute_logit_update": round(
        float(np.mean(np.abs(logit_update))),
        6,
    ),
    "q95_absolute_logit_update": round(
        float(
            np.quantile(
                np.abs(logit_update),
                0.95,
            )
        ),
        6,
    ),
    "mean_absolute_probability_update": round(
        float(
            np.mean(
                np.abs(probability_update)
            )
        ),
        6,
    ),
    "q95_absolute_probability_update": round(
        float(
            np.quantile(
                np.abs(probability_update),
                0.95,
            )
        ),
        6,
    ),
}

modest_shift_record = next(
    record
    for record in dirichlet_records
    if record["concentration"] == 100.0
)
substantial_shift_record = next(
    record
    for record in dirichlet_records
    if record["concentration"] == 20.0
)
tv_0p10_record = next(
    record
    for record in adversarial_mixture_records
    if abs(
        record["total_variation_radius"]
        - 0.10
    ) < 1e-12
)

phase33_stress_gate_passed = bool(
    phase33_observed_gain >= 0.003
    and within_group_bootstrap_report[
        "probability_positive"
    ] >= 0.95
    and within_group_bootstrap_report[
        "q05"
    ] >= 0.001
    and auroc_bootstrap_report[
        "probability_positive"
    ] >= 0.80
    and modest_shift_record[
        "probability_positive"
    ] >= 0.95
    and substantial_shift_record[
        "probability_positive"
    ] >= 0.80
    and minimum_leave_one_group_out_gain
    >= 0.0
    and tv_0p10_record[
        "worst_case_log_loss_gain"
    ] >= 0.0
)

phase33_stress_report = {
    "phase": (
        "phase33_acquisition_shift_"
        "robustness_stress"
    ),
    "status": (
        "eligible_for_deployment_parity"
        if phase33_stress_gate_passed
        else "requires_conservative_gate_selection"
    ),
    "fixed_candidate": {
        "gate": dict(
            PHASE33_SHARED_GATE_PRIVATE
        ),
        "observed_log_loss_gain": round(
            phase33_observed_gain,
            6,
        ),
        "observed_auroc_gain": round(
            float(
                roc_auc_score(
                    phase33_stress_labels,
                    phase33_stress_candidate,
                )
                - roc_auc_score(
                    phase33_stress_labels,
                    phase33_stress_baseline,
                )
            ),
            6,
        ),
    },
    "within_group_bootstrap": {
        "replicate_count": (
            within_group_replicates
        ),
        **within_group_bootstrap_report,
    },
    "stratified_auroc_bootstrap": {
        "replicate_count": (
            auroc_replicates
        ),
        **auroc_bootstrap_report,
    },
    "leave_one_acquisition_group_out": {
        "minimum_log_loss_gain": round(
            float(
                minimum_leave_one_group_out_gain
            ),
            6,
        ),
        "records": (
            leave_one_group_out_records
        ),
    },
    "dirichlet_group_mixture_shift": (
        dirichlet_records
    ),
    "adversarial_group_mixture_shift": (
        adversarial_mixture_records
    ),
    "acquisition_group_effects": (
        phase33_group_records
    ),
    "update_magnitude": update_report,
    "stress_thresholds": {
        "minimum_observed_log_loss_gain": 0.003,
        "minimum_within_group_probability_positive": 0.95,
        "minimum_within_group_q05_gain": 0.001,
        "minimum_auroc_probability_positive": 0.80,
        "minimum_modest_shift_probability_positive": 0.95,
        "minimum_substantial_shift_probability_positive": 0.80,
        "minimum_leave_one_group_out_gain": 0.0,
        "minimum_tv_0p10_worst_case_gain": 0.0,
    },
    "stress_gate_passed": (
        phase33_stress_gate_passed
    ),
    "gate_parameters_modified": False,
    "outer_labels_used_only_for_fixed_candidate_stress_evaluation": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase33_stress_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE33_STRESS")
print(json.dumps(
    phase33_stress_report,
    indent=2,
))
print("END SANITIZED_PHASE33_STRESS")

BEGIN SANITIZED_PHASE33_STRESS
{
  "phase": "phase33_acquisition_shift_robustness_stress",
  "status": "requires_conservative_gate_selection",
  "fixed_candidate": {
    "gate": {
      "alpha": 0.3,
      "uncertainty_exponent": 0.5,
      "delta_cap": 2.0
    },
    "observed_log_loss_gain": 0.004661,
    "observed_auroc_gain": 0.002031
  },
  "within_group_bootstrap": {
    "replicate_count": 5000,
    "mean": 0.004748,
    "standard_deviation": 0.002477,
    "q01": -0.001027,
    "q05": 0.00073,
    "q50": 0.004729,
    "q95": 0.0089,
    "q99": 0.01042,
    "probability_positive": 0.9734,
    "probability_gain_at_least_0p003": 0.758
  },
  "stratified_auroc_bootstrap": {
    "replicate_count": 2500,
    "mean": 0.002018,
    "standard_deviation": 0.00098,
    "q01": -0.000146,
    "q05": 0.000416,
    "q50": 0.002002,
    "q95": 0.003637,
    "q99": 0.004297,
    "probability_positive": 0.9836,
    "probability_gain_at_least_0p003": 0.1628
  },
  "leave_one_acquisition_group_out":

In [49]:
# Phase34 Cell 114
# Acquisition-mixture Bayesian-bootstrap lower-confidence-bound gate.
#
# This creates a conservative alternative to Phase33.
# Selection uses pooled monitor predictions only.
# Phase33 artifacts are preserved unchanged.

import json
import time

import numpy as np


PHASE34_CONFIG = {
    "phase": (
        "phase34_monitor_bootstrap_lcb_gate"
    ),
    "random_seed": 340601,
    "bootstrap_replicates": 6000,
    "dirichlet_concentration": 20.0,
    "dirichlet_floor": 0.20,
    "lower_quantile": 0.05,
    "minimum_mean_monitor_gain": 0.002,
    "minimum_lcb_gain": -0.0005,
    "minimum_probability_positive": 0.90,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "maximum_monitor_group_harm": 0.015,
}


def phase34_specification_key(specification):
    return (
        float(specification["alpha"]),
        float(
            specification[
                "uncertainty_exponent"
            ]
        ),
        float(specification["delta_cap"]),
    )


phase34_started = time.perf_counter()
phase34_rng = np.random.default_rng(
    PHASE34_CONFIG["random_seed"]
)

# State from Cell 112.
required_phase34_names = [
    "phase33_monitor_labels",
    "phase33_monitor_groups",
    "phase33_monitor_folds",
    "phase33_monitor_indices",
    "phase33_monitor_weights",
    "phase33_monitor_baseline",
    "phase33_monitor_component",
    "phase33_shared_gate_candidates",
    "phase33_shared_candidate_records",
    "phase33_final_baseline",
    "phase33_final_component",
]

phase34_missing = [
    name
    for name in required_phase34_names
    if name not in globals()
]

assert not phase34_missing, {
    "message": (
        "Required Phase33 monitor-selection state "
        "is missing. Rerun Cell 112."
    ),
    "missing": phase34_missing,
}

phase34_monitor_labels = np.asarray(
    phase33_monitor_labels,
    dtype=np.int64,
)
phase34_monitor_groups = np.asarray(
    phase33_monitor_groups,
    dtype=np.int64,
)
phase34_monitor_folds = np.asarray(
    phase33_monitor_folds,
    dtype=np.int64,
)
phase34_monitor_weights = np.asarray(
    phase33_monitor_weights,
    dtype=np.float64,
)
phase34_monitor_baseline = phase33d_probability(
    phase33_monitor_baseline
)
phase34_monitor_component = phase33d_probability(
    phase33_monitor_component
)

phase34_monitor_n = int(
    phase34_monitor_labels.size
)
phase34_candidate_count = int(
    len(phase33_shared_gate_candidates)
)

assert phase34_monitor_n == 493
assert phase34_candidate_count == 97

# Reuse Cell 112's deterministic fold/group constraints.
phase34_constraint_lookup = {
    phase34_specification_key(
        record["specification"]
    ): record
    for record in (
        phase33_shared_candidate_records
    )
}

# Case-level baseline losses.
phase34_baseline_case_loss = (
    phase33s_case_log_loss(
        phase34_monitor_labels,
        phase34_monitor_baseline,
    )
)

phase34_candidate_probability_matrix = (
    np.empty(
        (
            phase34_monitor_n,
            phase34_candidate_count,
        ),
        dtype=np.float64,
    )
)

phase34_gain_matrix = np.empty_like(
    phase34_candidate_probability_matrix
)

for candidate_index, specification in enumerate(
    phase33_shared_gate_candidates
):
    candidate_probability = (
        phase33d_bounded_update(
            phase34_monitor_baseline,
            phase34_monitor_component,
            specification["alpha"],
            specification[
                "uncertainty_exponent"
            ],
            specification["delta_cap"],
        )
    )

    phase34_candidate_probability_matrix[
        :,
        candidate_index,
    ] = candidate_probability

    candidate_case_loss = (
        phase33s_case_log_loss(
            phase34_monitor_labels,
            candidate_probability,
        )
    )

    phase34_gain_matrix[
        :,
        candidate_index,
    ] = (
        phase34_baseline_case_loss
        - candidate_case_loss
    )

# ---------------------------------------------------------
# Hierarchical Bayesian bootstrap:
#
# 1. Draw acquisition-group mixture weights.
# 2. Draw exponential Bayesian-bootstrap weights for
#    observations inside each group.
# 3. Apply inverse monitor multiplicity weights.
# ---------------------------------------------------------

phase34_unique_groups = np.asarray(
    sorted(
        np.unique(
            phase34_monitor_groups
        ).tolist()
    ),
    dtype=np.int64,
)

phase34_group_observation_indices = {
    int(group): np.flatnonzero(
        phase34_monitor_groups == group
    )
    for group in phase34_unique_groups
}

phase34_group_empirical_mass = np.asarray([
    np.sum(
        phase34_monitor_weights[
            phase34_group_observation_indices[
                int(group)
            ]
        ]
    )
    for group in phase34_unique_groups
], dtype=np.float64)

phase34_group_empirical_mass /= np.sum(
    phase34_group_empirical_mass
)

phase34_dirichlet_parameters = (
    PHASE34_CONFIG[
        "dirichlet_concentration"
    ]
    * phase34_group_empirical_mass
    + PHASE34_CONFIG["dirichlet_floor"]
)

phase34_bootstrap_replicates = int(
    PHASE34_CONFIG["bootstrap_replicates"]
)

phase34_bootstrap_weights = np.zeros(
    (
        phase34_bootstrap_replicates,
        phase34_monitor_n,
    ),
    dtype=np.float32,
)

phase34_group_mass_draws = (
    phase34_rng.dirichlet(
        phase34_dirichlet_parameters,
        size=phase34_bootstrap_replicates,
    )
)

for replicate in range(
    phase34_bootstrap_replicates
):
    for group_position, group in enumerate(
        phase34_unique_groups
    ):
        indices = (
            phase34_group_observation_indices[
                int(group)
            ]
        )

        local_weights = (
            phase34_rng.exponential(
                scale=1.0,
                size=indices.size,
            )
            * phase34_monitor_weights[indices]
        )

        local_sum = float(
            np.sum(local_weights)
        )

        if local_sum <= 0.0:
            local_weights = np.ones(
                indices.size,
                dtype=np.float64,
            )
            local_sum = float(indices.size)

        local_weights /= local_sum

        phase34_bootstrap_weights[
            replicate,
            indices,
        ] = (
            phase34_group_mass_draws[
                replicate,
                group_position,
            ]
            * local_weights
        )

bootstrap_row_sums = np.sum(
    phase34_bootstrap_weights,
    axis=1,
)

assert np.max(
    np.abs(bootstrap_row_sums - 1.0)
) <= 2e-6

# Shape: bootstrap replicates × gate candidates.
phase34_bootstrap_gain_matrix = (
    phase34_bootstrap_weights
    @ phase34_gain_matrix
)

phase34_candidate_records = []

for candidate_index, specification in enumerate(
    phase33_shared_gate_candidates
):
    bootstrap_gains = (
        phase34_bootstrap_gain_matrix[
            :,
            candidate_index,
        ]
    )

    constraint_record = (
        phase34_constraint_lookup[
            phase34_specification_key(
                specification
            )
        ]
    )

    mean_gain = float(
        np.mean(bootstrap_gains)
    )
    lcb_gain = float(
        np.quantile(
            bootstrap_gains,
            PHASE34_CONFIG[
                "lower_quantile"
            ],
        )
    )
    median_gain = float(
        np.median(bootstrap_gains)
    )
    probability_positive = float(
        np.mean(bootstrap_gains > 0.0)
    )
    probability_gain_0p003 = float(
        np.mean(bootstrap_gains >= 0.003)
    )

    eligible = bool(
        specification["alpha"] > 0.0
        and mean_gain
        >= PHASE34_CONFIG[
            "minimum_mean_monitor_gain"
        ]
        and lcb_gain
        >= PHASE34_CONFIG[
            "minimum_lcb_gain"
        ]
        and probability_positive
        >= PHASE34_CONFIG[
            "minimum_probability_positive"
        ]
        and constraint_record[
            "fold_win_count"
        ]
        >= PHASE34_CONFIG[
            "minimum_monitor_fold_wins"
        ]
        and constraint_record[
            "worst_fold_regret"
        ]
        <= PHASE34_CONFIG[
            "maximum_monitor_fold_regret"
        ]
        and constraint_record[
            "maximum_group_harm"
        ]
        <= PHASE34_CONFIG[
            "maximum_monitor_group_harm"
        ]
    )

    phase34_candidate_records.append({
        "candidate_index": int(
            candidate_index
        ),
        "specification": dict(
            specification
        ),
        "bootstrap_mean_gain": mean_gain,
        "bootstrap_median_gain": (
            median_gain
        ),
        "bootstrap_lcb_gain": lcb_gain,
        "bootstrap_q01_gain": float(
            np.quantile(
                bootstrap_gains,
                0.01,
            )
        ),
        "bootstrap_q95_gain": float(
            np.quantile(
                bootstrap_gains,
                0.95,
            )
        ),
        "probability_positive": (
            probability_positive
        ),
        "probability_gain_at_least_0p003": (
            probability_gain_0p003
        ),
        "monitor_fold_wins": int(
            constraint_record[
                "fold_win_count"
            ]
        ),
        "worst_monitor_fold_regret": float(
            constraint_record[
                "worst_fold_regret"
            ]
        ),
        "maximum_monitor_group_harm": float(
            constraint_record[
                "maximum_group_harm"
            ]
        ),
        "fold_records": (
            constraint_record[
                "fold_records"
            ]
        ),
        "eligible": eligible,
    })

phase34_nonzero_records = [
    record
    for record in phase34_candidate_records
    if (
        record["specification"]["alpha"]
        > 0.0
    )
]

phase34_raw_lcb_best = max(
    phase34_nonzero_records,
    key=lambda record: (
        record["bootstrap_lcb_gain"],
        record["bootstrap_mean_gain"],
        -record["specification"]["alpha"],
        record["specification"][
            "uncertainty_exponent"
        ],
        -record["specification"]["delta_cap"],
    ),
)

phase34_eligible_records = [
    record
    for record in phase34_candidate_records
    if record["eligible"]
]

if phase34_eligible_records:
    phase34_selected_record = max(
        phase34_eligible_records,
        key=lambda record: (
            record["bootstrap_lcb_gain"],
            record["bootstrap_mean_gain"],
            record["probability_positive"],
            -record["specification"]["alpha"],
            record["specification"][
                "uncertainty_exponent"
            ],
            -record["specification"][
                "delta_cap"
            ],
        ),
    )
    phase34_gate_advanced = True
else:
    phase34_selected_record = next(
        record
        for record in phase34_candidate_records
        if (
            record["specification"]["alpha"]
            == 0.0
        )
    )
    phase34_gate_advanced = False

PHASE34_CONSERVATIVE_GATE_PRIVATE = dict(
    phase34_selected_record["specification"]
)

print(
    "Phase34 conservative gate: "
    f"alpha={PHASE34_CONSERVATIVE_GATE_PRIVATE['alpha']:.2f}, "
    "gamma="
    f"{PHASE34_CONSERVATIVE_GATE_PRIVATE['uncertainty_exponent']:.2f}, "
    f"cap={PHASE34_CONSERVATIVE_GATE_PRIVATE['delta_cap']:.2f}, "
    f"LCB={phase34_selected_record['bootstrap_lcb_gain']:.6f}, "
    f"advanced={phase34_gate_advanced}"
)

# Gate is frozen before outer evaluation.
PHASE34_GATED_OOF_PRIVATE = (
    phase33d_bounded_update(
        phase33_final_baseline,
        phase33_final_component,
        PHASE34_CONSERVATIVE_GATE_PRIVATE[
            "alpha"
        ],
        PHASE34_CONSERVATIVE_GATE_PRIVATE[
            "uncertainty_exponent"
        ],
        PHASE34_CONSERVATIVE_GATE_PRIVATE[
            "delta_cap"
        ],
    )
)

phase34_baseline_metrics = phase33d_metrics(
    phase33_gate_labels,
    phase33_final_baseline,
)
phase34_candidate_metrics = phase33d_metrics(
    phase33_gate_labels,
    PHASE34_GATED_OOF_PRIVATE,
)

phase34_calibration = (
    phase33g_calibration_statistics(
        phase33_gate_labels,
        PHASE34_GATED_OOF_PRIVATE,
    )
)

phase34_fold_records = []

for partition in PHASE19_PARTITIONS:
    fold = int(partition["fold"])
    outer_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    baseline_metrics = phase33d_metrics(
        phase33_gate_labels[outer_indices],
        phase33_final_baseline[outer_indices],
    )
    candidate_metrics = phase33d_metrics(
        phase33_gate_labels[outer_indices],
        PHASE34_GATED_OOF_PRIVATE[
            outer_indices
        ],
    )

    phase34_fold_records.append({
        "fold": fold,
        "n": int(outer_indices.size),
        "baseline_log_loss": round(
            baseline_metrics["log_loss"],
            6,
        ),
        "phase34_log_loss": round(
            candidate_metrics["log_loss"],
            6,
        ),
        "log_loss_improvement": round(
            baseline_metrics["log_loss"]
            - candidate_metrics["log_loss"],
            6,
        ),
        "baseline_auroc": round(
            baseline_metrics["auroc"],
            6,
        ),
        "phase34_auroc": round(
            candidate_metrics["auroc"],
            6,
        ),
    })

phase34_fold_wins = sum(
    record["log_loss_improvement"] > 1e-8
    for record in phase34_fold_records
)
phase34_worst_fold_excess = max(
    -record["log_loss_improvement"]
    for record in phase34_fold_records
)

(
    phase34_maximum_group_harm,
    phase34_group_records,
) = phase33d_group_harm(
    phase33_gate_labels,
    phase33_final_baseline,
    PHASE34_GATED_OOF_PRIVATE,
    phase33_gate_groups,
)

phase34_log_loss_gain = (
    phase34_baseline_metrics["log_loss"]
    - phase34_candidate_metrics["log_loss"]
)
phase34_auroc_gain = (
    phase34_candidate_metrics["auroc"]
    - phase34_baseline_metrics["auroc"]
)
phase34_brier_excess = (
    phase34_candidate_metrics["brier"]
    - phase34_baseline_metrics["brier"]
)

phase34_component_gate_passed = bool(
    phase34_gate_advanced
    and phase34_log_loss_gain >= 0.002
    and phase34_fold_wins >= 2
    and phase34_worst_fold_excess <= 0.005
    and phase34_maximum_group_harm <= 0.015
    and 0.8
    <= phase34_calibration[
        "calibration_slope"
    ]
    <= 1.2
)

PHASE34_DEPLOYMENT_STATE_PRIVATE = {
    "shared_gate": dict(
        PHASE34_CONSERVATIVE_GATE_PRIVATE
    ),
    "phase33_model_states": (
        PHASE33_DEPLOYMENT_STATES_PRIVATE
    ),
    "selection_method": (
        "monitor_hierarchical_bayesian_"
        "bootstrap_lcb"
    ),
    "fold_specific_gate_routing": False,
}

phase34_report = {
    "phase": (
        "phase34_monitor_bootstrap_lcb_gate"
    ),
    "status": (
        "eligible_for_fixed_candidate_stress_test"
        if phase34_component_gate_passed
        else "not_promoted"
    ),
    "configuration": PHASE34_CONFIG,
    "monitor_bootstrap": {
        "observation_count": int(
            phase34_monitor_n
        ),
        "unique_case_count": int(
            np.unique(
                phase33_monitor_indices
            ).size
        ),
        "acquisition_group_count": int(
            phase34_unique_groups.size
        ),
        "candidate_count": int(
            phase34_candidate_count
        ),
        "eligible_candidate_count": int(
            len(phase34_eligible_records)
        ),
        "bootstrap_replicate_count": int(
            phase34_bootstrap_replicates
        ),
    },
    "raw_lcb_best": {
        key: (
            round(float(value), 6)
            if isinstance(
                value,
                (float, np.floating),
            )
            else value
        )
        for key, value in (
            phase34_raw_lcb_best.items()
        )
        if key not in [
            "fold_records",
            "candidate_index",
        ]
    },
    "selection": {
        "selected_gate": dict(
            PHASE34_CONSERVATIVE_GATE_PRIVATE
        ),
        "gate_advanced": (
            phase34_gate_advanced
        ),
        "bootstrap_mean_gain": round(
            phase34_selected_record[
                "bootstrap_mean_gain"
            ],
            6,
        ),
        "bootstrap_lcb_gain": round(
            phase34_selected_record[
                "bootstrap_lcb_gain"
            ],
            6,
        ),
        "bootstrap_q01_gain": round(
            phase34_selected_record[
                "bootstrap_q01_gain"
            ],
            6,
        ),
        "probability_positive": round(
            phase34_selected_record[
                "probability_positive"
            ],
            6,
        ),
        "probability_gain_at_least_0p003": (
            round(
                phase34_selected_record[
                    "probability_gain_at_least_0p003"
                ],
                6,
            )
        ),
        "monitor_fold_wins": int(
            phase34_selected_record[
                "monitor_fold_wins"
            ]
        ),
        "worst_monitor_fold_regret": round(
            phase34_selected_record[
                "worst_monitor_fold_regret"
            ],
            6,
        ),
        "maximum_monitor_group_harm": round(
            phase34_selected_record[
                "maximum_monitor_group_harm"
            ],
            6,
        ),
        "fold_monitor_metrics": [
            {
                "fold": int(record["fold"]),
                "baseline_log_loss": round(
                    record[
                        "baseline_log_loss"
                    ],
                    6,
                ),
                "candidate_log_loss": round(
                    record[
                        "candidate_log_loss"
                    ],
                    6,
                ),
                "gain": round(
                    record["gain"],
                    6,
                ),
            }
            for record in (
                phase34_selected_record[
                    "fold_records"
                ]
            )
        ],
    },
    "baseline": {
        key: (
            None if value is None
            else round(float(value), 6)
        )
        for key, value in (
            phase34_baseline_metrics.items()
        )
    },
    "phase34": {
        **{
            key: (
                None if value is None
                else round(float(value), 6)
            )
            for key, value in (
                phase34_candidate_metrics.items()
            )
        },
        "calibration_intercept": round(
            phase34_calibration[
                "calibration_intercept"
            ],
            6,
        ),
        "calibration_slope": round(
            phase34_calibration[
                "calibration_slope"
            ],
            6,
        ),
    },
    "improvements": {
        "log_loss_gain": round(
            phase34_log_loss_gain,
            6,
        ),
        "auroc_gain": round(
            phase34_auroc_gain,
            6,
        ),
        "brier_excess": round(
            phase34_brier_excess,
            6,
        ),
        "fold_wins": int(
            phase34_fold_wins
        ),
        "worst_fold_excess": round(
            phase34_worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            phase34_maximum_group_harm,
            6,
        ),
    },
    "comparison_with_phase33": {
        "phase33_log_loss": round(
            float(
                phase33_final_gated_metrics[
                    "log_loss"
                ]
            ),
            6,
        ),
        "phase34_log_loss": round(
            float(
                phase34_candidate_metrics[
                    "log_loss"
                ]
            ),
            6,
        ),
        "phase34_minus_phase33_gain": round(
            float(
                phase33_final_gated_metrics[
                    "log_loss"
                ]
                - phase34_candidate_metrics[
                    "log_loss"
                ]
            ),
            6,
        ),
    },
    "fold_metrics": phase34_fold_records,
    "major_acquisition_group_metrics": (
        phase34_group_records
    ),
    "component_gate_passed": (
        phase34_component_gate_passed
    ),
    "shared_gate_across_folds": True,
    "fold_specific_gate_routing": False,
    "outer_oracle_parameters_used": False,
    "monitor_labels_used_for_selection": True,
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase34_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE34_LCB_GATE")
print(json.dumps(
    phase34_report,
    indent=2,
))
print("END SANITIZED_PHASE34_LCB_GATE")

Phase34 conservative gate: alpha=0.10, gamma=0.00, cap=1.00, LCB=-0.000057, advanced=True
BEGIN SANITIZED_PHASE34_LCB_GATE
{
  "phase": "phase34_monitor_bootstrap_lcb_gate",
  "status": "not_promoted",
  "configuration": {
    "phase": "phase34_monitor_bootstrap_lcb_gate",
    "random_seed": 340601,
    "bootstrap_replicates": 6000,
    "dirichlet_concentration": 20.0,
    "dirichlet_floor": 0.2,
    "lower_quantile": 0.05,
    "minimum_mean_monitor_gain": 0.002,
    "minimum_lcb_gain": -0.0005,
    "minimum_probability_positive": 0.9,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "maximum_monitor_group_harm": 0.015
  },
  "monitor_bootstrap": {
    "observation_count": 493,
    "unique_case_count": 407,
    "acquisition_group_count": 10,
    "candidate_count": 97,
    "eligible_candidate_count": 10,
    "bootstrap_replicate_count": 6000
  },
  "raw_lcb_best": {
    "specification": {
      "alpha": 0.05,
      "uncertainty_exponent": 0.0,
      "del

In [59]:
# Phase35 Cell 115A-CONSOLIDATED
# Complete corrected architecture definition and contract.
# Replaces old Cells 115A, 115A-R, and 115A-R2.

import inspect
import json
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


PHASE35_CONFIG = {
    "phase": (
        "phase35_phase12c_anchored_"
        "domain_invariant_residual"
    ),
    "residual_cap": 1.0,
    "uncertainty_exponent": 0.5,
    "residual_hidden_dimension": 64,
    "residual_dropout": 0.10,
    "domain_class_count": 15,
    "domain_strength": 0.02,
    "residual_regularization": 0.02,
    "residual_mean_regularization": 0.01,
}

PHASE35_DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


def phase35_extract_phase33_outputs(result):
    if isinstance(result, dict):
        embedding = None
        domain_logit = None

        for key in [
            "embedding",
            "features",
            "fused_embedding",
            "representation",
        ]:
            if key in result:
                embedding = result[key]
                break

        for key in [
            "domain_logit",
            "domain_logits",
            "acquisition_logit",
            "acquisition_logits",
        ]:
            if key in result:
                domain_logit = result[key]
                break

        assert embedding is not None, {
            "message": (
                "Could not resolve Phase33 embedding."
            ),
            "available_keys": sorted(
                result.keys()
            ),
        }
        assert domain_logit is not None, {
            "message": (
                "Could not resolve Phase33 "
                "domain logits."
            ),
            "available_keys": sorted(
                result.keys()
            ),
        }

        return domain_logit, embedding

    assert isinstance(
        result,
        (tuple, list),
    ), type(result).__name__
    assert len(result) >= 3

    # Phase33 output contract:
    # classification_logit, domain_logit, embedding
    return result[1], result[2]


def phase35_call_phase33_encoder(
    encoder,
    p3ca_sequence,
    geometry_sequence,
    summary_features,
    radiomics_features,
    gradient_reversal_strength,
):
    forward_parameters = inspect.signature(
        encoder.forward
    ).parameters

    keyword_arguments = {}

    candidate_names = [
        "grl_strength",
        "domain_strength",
        "gradient_reversal_strength",
        "grl_lambda",
        "domain_lambda",
    ]

    for name in candidate_names:
        if name in forward_parameters:
            keyword_arguments[name] = (
                gradient_reversal_strength
            )
            break

    return encoder(
        p3ca_sequence,
        geometry_sequence,
        summary_features,
        radiomics_features,
        **keyword_arguments,
    )


class Phase35AnchoredResidualFusionNet(nn.Module):
    def __init__(
        self,
        residual_cap=1.0,
        uncertainty_exponent=0.5,
        residual_hidden_dimension=64,
        residual_dropout=0.10,
        domain_class_count=15,
    ):
        super().__init__()

        self.residual_cap = float(
            residual_cap
        )
        self.uncertainty_exponent = float(
            uncertainty_exponent
        )

        self.encoder = (
            Phase33DomainInvariantFusionNet(
                domain_class_count=int(
                    domain_class_count
                )
            )
        )

        # Phase33 embedding: 128.
        # Anchor features:
        # z, |z|, p-0.5, uncertainty.
        residual_input_dimension = 128 + 4

        self.residual_head = nn.Sequential(
            nn.LayerNorm(
                residual_input_dimension
            ),
            nn.Linear(
                residual_input_dimension,
                residual_hidden_dimension,
            ),
            nn.GELU(),
            nn.Dropout(
                residual_dropout
            ),
            nn.Linear(
                residual_hidden_dimension,
                residual_hidden_dimension // 2,
            ),
            nn.GELU(),
            nn.Dropout(
                residual_dropout * 0.5
            ),
            nn.Linear(
                residual_hidden_dimension // 2,
                1,
            ),
        )

        # Exact Phase12c anchor at initialization.
        final_layer = self.residual_head[-1]
        nn.init.zeros_(final_layer.weight)
        nn.init.zeros_(final_layer.bias)

    def forward(
        self,
        p3ca_sequence,
        geometry_sequence,
        summary_features,
        radiomics_features,
        baseline_logit,
        gradient_reversal_strength=0.0,
    ):
        baseline_logit = (
            baseline_logit.reshape(-1)
        )

        encoder_result = (
            phase35_call_phase33_encoder(
                self.encoder,
                p3ca_sequence,
                geometry_sequence,
                summary_features,
                radiomics_features,
                gradient_reversal_strength,
            )
        )

        (
            domain_logit,
            embedding,
        ) = phase35_extract_phase33_outputs(
            encoder_result
        )

        baseline_probability = torch.sigmoid(
            baseline_logit
        )

        uncertainty = (
            4.0
            * baseline_probability
            * (1.0 - baseline_probability)
        ).clamp(
            min=0.0,
            max=1.0,
        )

        anchor_features = torch.stack(
            [
                baseline_logit,
                baseline_logit.abs(),
                baseline_probability - 0.5,
                uncertainty,
            ],
            dim=1,
        )

        residual_input = torch.cat(
            [
                embedding,
                anchor_features,
            ],
            dim=1,
        )

        raw_residual = (
            self.residual_head(
                residual_input
            ).reshape(-1)
        )

        uncertainty_scale = (
            uncertainty
            ** self.uncertainty_exponent
        )

        bounded_residual = (
            self.residual_cap
            * uncertainty_scale
            * torch.tanh(raw_residual)
        )

        final_logit = (
            baseline_logit
            + bounded_residual
        )

        return {
            "final_logit": final_logit,
            "domain_logit": domain_logit,
            "embedding": embedding,
            "raw_residual": raw_residual,
            "bounded_residual": (
                bounded_residual
            ),
            "baseline_logit": (
                baseline_logit
            ),
            "uncertainty": uncertainty,
        }


def phase35_contract_features_to_device(
    transformed_features,
    device,
):
    expected_keys = [
        "p3ca_sequence",
        "geometry_sequence",
        "summary_features",
        "radiomics_features",
    ]

    assert isinstance(
        transformed_features,
        dict,
    ), {
        "container_type": type(
            transformed_features
        ).__name__,
    }

    missing_keys = [
        key
        for key in expected_keys
        if key not in transformed_features
    ]

    assert not missing_keys, {
        "missing_keys": missing_keys,
        "available_keys": sorted(
            transformed_features.keys()
        ),
    }

    result = {}

    for key in expected_keys:
        value = transformed_features[key]

        if torch.is_tensor(value):
            tensor = value.detach().to(
                device=device,
                dtype=torch.float32,
            )
        else:
            tensor = torch.as_tensor(
                np.asarray(
                    value,
                    dtype=np.float32,
                ),
                device=device,
                dtype=torch.float32,
            )

        result[key] = tensor

    return result


def phase35_new_model():
    return Phase35AnchoredResidualFusionNet(
        residual_cap=PHASE35_CONFIG[
            "residual_cap"
        ],
        uncertainty_exponent=(
            PHASE35_CONFIG[
                "uncertainty_exponent"
            ]
        ),
        residual_hidden_dimension=(
            PHASE35_CONFIG[
                "residual_hidden_dimension"
            ]
        ),
        residual_dropout=PHASE35_CONFIG[
            "residual_dropout"
        ],
        domain_class_count=PHASE35_CONFIG[
            "domain_class_count"
        ],
    )


phase35_contract_started = time.perf_counter()

phase35_contract_fit_indices = np.asarray(
    PHASE19_PARTITIONS[0]["fit"],
    dtype=np.int64,
)
phase35_contract_indices = (
    phase35_contract_fit_indices[:8]
)

phase35_contract_preprocessor = (
    phase33_fit_preprocessor(
        0,
        0,
        phase35_contract_fit_indices,
        350101,
    )
)

phase35_contract_features = phase33_transform(
    phase35_contract_preprocessor,
    phase35_contract_indices,
)

# Direct conversion: no phase33_to_device_features call.
phase35_contract_device_features = (
    phase35_contract_features_to_device(
        phase35_contract_features,
        PHASE35_DEVICE,
    )
)

phase35_contract_model = (
    phase35_new_model().to(
        PHASE35_DEVICE
    )
)
phase35_contract_model.train()

phase35_contract_baseline_logit = (
    torch.linspace(
        -2.5,
        2.5,
        steps=8,
        device=PHASE35_DEVICE,
        dtype=torch.float32,
    )
)

phase35_contract_target = torch.tensor(
    [0, 0, 0, 1, 0, 1, 1, 1],
    device=PHASE35_DEVICE,
    dtype=torch.float32,
)
phase35_contract_domain_target = torch.tensor(
    [0, 1, 2, 3, 4, 5, 6, 7],
    device=PHASE35_DEVICE,
    dtype=torch.long,
)

phase35_contract_output = (
    phase35_contract_model(
        phase35_contract_device_features[
            "p3ca_sequence"
        ],
        phase35_contract_device_features[
            "geometry_sequence"
        ],
        phase35_contract_device_features[
            "summary_features"
        ],
        phase35_contract_device_features[
            "radiomics_features"
        ],
        phase35_contract_baseline_logit,
        gradient_reversal_strength=0.02,
    )
)

classification_loss = (
    F.binary_cross_entropy_with_logits(
        phase35_contract_output[
            "final_logit"
        ],
        phase35_contract_target,
    )
)
domain_loss = F.cross_entropy(
    phase35_contract_output[
        "domain_logit"
    ],
    phase35_contract_domain_target,
)
residual_penalty = (
    phase35_contract_output[
        "bounded_residual"
    ].square().mean()
)
residual_mean_penalty = (
    phase35_contract_output[
        "bounded_residual"
    ].mean().square()
)

contract_loss = (
    classification_loss
    + domain_loss
    + PHASE35_CONFIG[
        "residual_regularization"
    ] * residual_penalty
    + PHASE35_CONFIG[
        "residual_mean_regularization"
    ] * residual_mean_penalty
)

phase35_contract_model.zero_grad(
    set_to_none=True
)
contract_loss.backward()

gradient_squared_sum = torch.zeros(
    (),
    device=PHASE35_DEVICE,
    dtype=torch.float32,
)

for parameter in (
    phase35_contract_model.parameters()
):
    if parameter.grad is not None:
        gradient_squared_sum += (
            parameter.grad.detach()
            .float()
            .square()
            .sum()
        )

phase35_gradient_norm = float(
    torch.sqrt(
        gradient_squared_sum
    ).item()
)

phase35_parameter_count = sum(
    parameter.numel()
    for parameter in (
        phase35_contract_model.parameters()
    )
)
phase35_trainable_parameter_count = sum(
    parameter.numel()
    for parameter in (
        phase35_contract_model.parameters()
    )
    if parameter.requires_grad
)

initial_anchor_error = float(
    torch.max(
        torch.abs(
            phase35_contract_output[
                "final_logit"
            ]
            - phase35_contract_baseline_logit
        )
    ).item()
)

maximum_residual = float(
    torch.max(
        torch.abs(
            phase35_contract_output[
                "bounded_residual"
            ]
        )
    ).item()
)

assert phase35_parameter_count == 141244
assert phase35_contract_output[
    "final_logit"
].shape == (8,)
assert phase35_contract_output[
    "domain_logit"
].shape == (8, 15)
assert phase35_contract_output[
    "embedding"
].shape == (8, 128)
assert initial_anchor_error == 0.0
assert maximum_residual == 0.0
assert math.isfinite(
    phase35_gradient_norm
)

phase35_architecture_report = {
    "phase": (
        "phase35_anchored_residual_"
        "architecture"
    ),
    "status": "accepted",
    "model": (
        "Phase35AnchoredResidualFusionNet"
    ),
    "parameter_count": int(
        phase35_parameter_count
    ),
    "trainable_parameter_count": int(
        phase35_trainable_parameter_count
    ),
    "constructor_contract": {
        "domain_class_count": 15,
        "constructor_signature": str(
            inspect.signature(
                Phase35AnchoredResidualFusionNet
            )
        ),
    },
    "inputs": {
        key: list(value.shape)
        for key, value in (
            phase35_contract_device_features.items()
        )
    },
    "outputs": {
        "final_logit": [8],
        "domain_logit": [8, 15],
        "embedding": [8, 128],
        "bounded_residual": [8],
    },
    "residual_contract": {
        "residual_cap": 1.0,
        "uncertainty_exponent": 0.5,
        "zero_initialized_output_head": True,
        "initial_anchor_max_abs_error": (
            initial_anchor_error
        ),
        "maximum_synthetic_residual": (
            maximum_residual
        ),
    },
    "gradient_norm": round(
        phase35_gradient_norm,
        6,
    ),
    "backward_contract_passed": True,
    "incorrect_phase33_helper_used": False,
    "phase12c_anchor_updated": False,
    "synthetic_targets_only": True,
    "training_labels_used": False,
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase35_contract_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE35_ARCHITECTURE")
print(json.dumps(
    phase35_architecture_report,
    indent=2,
))
print("END SANITIZED_PHASE35_ARCHITECTURE")

BEGIN SANITIZED_PHASE35_ARCHITECTURE
{
  "phase": "phase35_anchored_residual_architecture",
  "status": "accepted",
  "model": "Phase35AnchoredResidualFusionNet",
  "parameter_count": 141244,
  "trainable_parameter_count": 141244,
  "constructor_contract": {
    "domain_class_count": 15,
    "constructor_signature": "(residual_cap=1.0, uncertainty_exponent=0.5, residual_hidden_dimension=64, residual_dropout=0.1, domain_class_count=15)"
  },
  "inputs": {
    "p3ca_sequence": [
      8,
      84,
      48
    ],
    "geometry_sequence": [
      8,
      16,
      52
    ],
    "summary_features": [
      8,
      48
    ],
    "radiomics_features": [
      8,
      48
    ]
  },
  "outputs": {
    "final_logit": [
      8
    ],
    "domain_logit": [
      8,
      15
    ],
    "embedding": [
      8,
      128
    ],
    "bounded_residual": [
      8
    ]
  },
  "residual_contract": {
    "residual_cap": 1.0,
    "uncertainty_exponent": 0.5,
    "zero_initialized_output_head": true,


In [60]:
# Phase35 Cell 116A
# Nested anchored-residual training utilities.

import copy
import inspect
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn.functional as F


PHASE35_TRAIN_CONFIG = {
    "seeds": [
        350701,
        350702,
        350703,
    ],
    "batch_size": 64,
    "maximum_epochs": 100,
    "patience": 18,
    "learning_rate": 2.5e-4,
    "minimum_learning_rate": 1.5e-5,
    "weight_decay": 0.03,
    "gradient_clip": 1.5,
    "label_smoothing": 0.01,
    "token_dropout": 0.03,
    "feature_noise": 0.008,
    "domain_strength": 0.02,
    "domain_ramp_epochs": 15,
    "residual_regularization": 0.02,
    "residual_mean_regularization": 0.01,
    "shared_scale_candidates": [
        0.0,
        0.25,
        0.50,
        0.75,
        1.0,
    ],
    "minimum_shared_monitor_gain": 0.002,
    "minimum_shared_robust_gain": 0.002,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "maximum_monitor_group_harm": 0.015,
}


def phase35_set_seed(seed):
    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def phase35_new_model():
    return Phase35AnchoredResidualFusionNet(
        residual_cap=PHASE35_CONFIG[
            "residual_cap"
        ],
        uncertainty_exponent=(
            PHASE35_CONFIG[
                "uncertainty_exponent"
            ]
        ),
        residual_hidden_dimension=(
            PHASE35_CONFIG[
                "residual_hidden_dimension"
            ]
        ),
        residual_dropout=PHASE35_CONFIG[
            "residual_dropout"
        ],
        domain_class_count=15,
    )


def phase35_state_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in (
            model.state_dict().items()
        )
    }


def phase35_dataset_to_device(
    transformed_features,
    labels,
    groups,
    baseline_probabilities,
    device,
):
    feature_tensors = (
        phase35_contract_features_to_device(
            transformed_features,
            device,
        )
    )

    labels = np.asarray(
        labels,
        dtype=np.float32,
    ).reshape(-1)
    groups = np.asarray(
        groups,
        dtype=np.int64,
    ).reshape(-1)
    baseline_probabilities = (
        phase33d_probability(
            baseline_probabilities
        )
    )

    expected_n = labels.size

    for tensor in feature_tensors.values():
        assert tensor.shape[0] == expected_n

    assert groups.shape == (expected_n,)
    assert baseline_probabilities.shape == (
        expected_n,
    )

    baseline_logits = phase33d_logit(
        baseline_probabilities
    )

    return {
        **feature_tensors,
        "labels": torch.as_tensor(
            labels,
            device=device,
            dtype=torch.float32,
        ),
        "groups": torch.as_tensor(
            groups,
            device=device,
            dtype=torch.long,
        ),
        "baseline_logit": torch.as_tensor(
            baseline_logits,
            device=device,
            dtype=torch.float32,
        ),
    }


def phase35_group_sample_weights(groups):
    groups = np.asarray(
        groups,
        dtype=np.int64,
    ).reshape(-1)

    unique_groups, counts = np.unique(
        groups,
        return_counts=True,
    )

    count_lookup = {
        int(group): int(count)
        for group, count in zip(
            unique_groups,
            counts,
        )
    }

    weights = np.asarray([
        1.0 / count_lookup[int(group)]
        for group in groups
    ], dtype=np.float64)

    weights /= np.mean(weights)

    return weights.astype(
        np.float32
    )


def phase35_slice_dataset(
    dataset,
    batch_indices,
):
    return {
        key: value[batch_indices]
        for key, value in dataset.items()
    }


def phase35_augment_batch(batch):
    p3ca_sequence = batch[
        "p3ca_sequence"
    ]
    geometry_sequence = batch[
        "geometry_sequence"
    ]
    summary_features = batch[
        "summary_features"
    ]
    radiomics_features = batch[
        "radiomics_features"
    ]

    token_dropout = float(
        PHASE35_TRAIN_CONFIG[
            "token_dropout"
        ]
    )
    feature_noise = float(
        PHASE35_TRAIN_CONFIG[
            "feature_noise"
        ]
    )

    if token_dropout > 0.0:
        p3ca_keep = (
            torch.rand(
                (
                    p3ca_sequence.shape[0],
                    p3ca_sequence.shape[1],
                    1,
                ),
                device=p3ca_sequence.device,
            )
            >= token_dropout
        ).to(p3ca_sequence.dtype)

        geometry_keep = (
            torch.rand(
                (
                    geometry_sequence.shape[0],
                    geometry_sequence.shape[1],
                    1,
                ),
                device=geometry_sequence.device,
            )
            >= token_dropout
        ).to(geometry_sequence.dtype)

        # Inverted token dropout.
        p3ca_sequence = (
            p3ca_sequence
            * p3ca_keep
            / (1.0 - token_dropout)
        )
        geometry_sequence = (
            geometry_sequence
            * geometry_keep
            / (1.0 - token_dropout)
        )

    if feature_noise > 0.0:
        p3ca_sequence = (
            p3ca_sequence
            + feature_noise
            * torch.randn_like(
                p3ca_sequence
            )
        )
        geometry_sequence = (
            geometry_sequence
            + feature_noise
            * torch.randn_like(
                geometry_sequence
            )
        )
        summary_features = (
            summary_features
            + feature_noise
            * torch.randn_like(
                summary_features
            )
        )
        radiomics_features = (
            radiomics_features
            + feature_noise
            * torch.randn_like(
                radiomics_features
            )
        )

    return {
        "p3ca_sequence": p3ca_sequence,
        "geometry_sequence": (
            geometry_sequence
        ),
        "summary_features": (
            summary_features
        ),
        "radiomics_features": (
            radiomics_features
        ),
    }


def phase35_robust_score(
    labels,
    probabilities,
    groups,
    minimum_group_n=8,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    ).reshape(-1)
    probabilities = phase33d_probability(
        probabilities
    )
    groups = np.asarray(
        groups,
        dtype=np.int64,
    ).reshape(-1)

    overall_loss = phase33g_weighted_log_loss(
        labels,
        probabilities,
    )

    group_losses = []

    for group in sorted(
        np.unique(groups).tolist()
    ):
        mask = groups == group

        if int(np.sum(mask)) < minimum_group_n:
            continue

        group_losses.append(
            phase33g_weighted_log_loss(
                labels[mask],
                probabilities[mask],
            )
        )

    if group_losses:
        equal_group_mean = float(
            np.mean(group_losses)
        )
        worst_group = float(
            np.max(group_losses)
        )
    else:
        equal_group_mean = overall_loss
        worst_group = overall_loss

    robust_score = (
        0.55 * overall_loss
        + 0.30 * equal_group_mean
        + 0.15 * worst_group
    )

    return {
        "robust_score": float(
            robust_score
        ),
        "overall_log_loss": float(
            overall_loss
        ),
        "equal_group_mean_log_loss": float(
            equal_group_mean
        ),
        "worst_group_log_loss": float(
            worst_group
        ),
        "included_group_count": int(
            len(group_losses)
        ),
    }


@torch.no_grad()
def phase35_predict_residual(
    model,
    dataset,
    batch_size=128,
):
    model.eval()

    residual_parts = []
    final_logit_parts = []

    case_count = int(
        dataset["labels"].shape[0]
    )

    for start in range(
        0,
        case_count,
        batch_size,
    ):
        stop = min(
            start + batch_size,
            case_count,
        )
        indices = torch.arange(
            start,
            stop,
            device=PHASE35_DEVICE,
        )

        batch = phase35_slice_dataset(
            dataset,
            indices,
        )

        output = model(
            batch["p3ca_sequence"],
            batch["geometry_sequence"],
            batch["summary_features"],
            batch["radiomics_features"],
            batch["baseline_logit"],
            gradient_reversal_strength=0.0,
        )

        residual_parts.append(
            output["bounded_residual"]
            .detach()
            .cpu()
            .numpy()
        )
        final_logit_parts.append(
            output["final_logit"]
            .detach()
            .cpu()
            .numpy()
        )

    residual = np.concatenate(
        residual_parts
    ).astype(
        np.float64,
        copy=False,
    )
    final_logit = np.concatenate(
        final_logit_parts
    ).astype(
        np.float64,
        copy=False,
    )

    assert residual.shape == (
        case_count,
    )
    assert final_logit.shape == (
        case_count,
    )
    assert np.isfinite(residual).all()
    assert np.isfinite(final_logit).all()

    return residual, final_logit


def phase35_train_with_monitor(
    train_dataset,
    monitor_dataset,
    train_groups_numpy,
    monitor_labels_numpy,
    monitor_groups_numpy,
    seed,
    fold,
):
    phase35_set_seed(seed)

    model = phase35_new_model().to(
        PHASE35_DEVICE
    )
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE35_TRAIN_CONFIG[
            "learning_rate"
        ],
        weight_decay=PHASE35_TRAIN_CONFIG[
            "weight_decay"
        ],
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=PHASE35_TRAIN_CONFIG[
                "maximum_epochs"
            ],
            eta_min=PHASE35_TRAIN_CONFIG[
                "minimum_learning_rate"
            ],
        )
    )

    train_sample_weights = torch.as_tensor(
        phase35_group_sample_weights(
            train_groups_numpy
        ),
        device=PHASE35_DEVICE,
        dtype=torch.float32,
    )

    train_n = int(
        train_dataset["labels"].shape[0]
    )

    baseline_monitor_probability = (
        phase33d_sigmoid(
            monitor_dataset[
                "baseline_logit"
            ].detach().cpu().numpy()
        )
    )
    baseline_monitor_score = (
        phase35_robust_score(
            monitor_labels_numpy,
            baseline_monitor_probability,
            monitor_groups_numpy,
        )
    )

    # Epoch zero is the exact Phase12c anchor.
    best_score = float(
        baseline_monitor_score[
            "robust_score"
        ]
    )
    best_epoch = 0
    best_state = phase35_state_to_cpu(
        model
    )
    best_monitor_residual = np.zeros(
        len(monitor_labels_numpy),
        dtype=np.float64,
    )
    best_monitor_probability = (
        baseline_monitor_probability.copy()
    )

    patience_counter = 0
    epoch_records = []
    final_training_loss = None

    for epoch in range(
        1,
        PHASE35_TRAIN_CONFIG[
            "maximum_epochs"
        ] + 1,
    ):
        model.train()

        order = torch.randperm(
            train_n,
            device=PHASE35_DEVICE,
        )

        epoch_loss_sum = 0.0
        epoch_case_count = 0

        domain_ramp = min(
            1.0,
            epoch
            / PHASE35_TRAIN_CONFIG[
                "domain_ramp_epochs"
            ],
        )
        grl_strength = (
            PHASE35_TRAIN_CONFIG[
                "domain_strength"
            ]
            * domain_ramp
        )

        for start in range(
            0,
            train_n,
            PHASE35_TRAIN_CONFIG[
                "batch_size"
            ],
        ):
            batch_indices = order[
                start:
                start
                + PHASE35_TRAIN_CONFIG[
                    "batch_size"
                ]
            ]

            batch = phase35_slice_dataset(
                train_dataset,
                batch_indices,
            )
            augmented = (
                phase35_augment_batch(
                    batch
                )
            )

            output = model(
                augmented[
                    "p3ca_sequence"
                ],
                augmented[
                    "geometry_sequence"
                ],
                augmented[
                    "summary_features"
                ],
                augmented[
                    "radiomics_features"
                ],
                batch["baseline_logit"],
                gradient_reversal_strength=(
                    grl_strength
                ),
            )

            target = batch["labels"]
            smoothing = float(
                PHASE35_TRAIN_CONFIG[
                    "label_smoothing"
                ]
            )
            smoothed_target = (
                target * (1.0 - smoothing)
                + 0.5 * smoothing
            )

            classification_losses = (
                F.binary_cross_entropy_with_logits(
                    output["final_logit"],
                    smoothed_target,
                    reduction="none",
                )
            )

            sample_weights = (
                train_sample_weights[
                    batch_indices
                ]
            )

            classification_loss = (
                torch.sum(
                    classification_losses
                    * sample_weights
                )
                / torch.sum(sample_weights)
            )

            domain_loss = F.cross_entropy(
                output["domain_logit"],
                batch["groups"],
            )

            residual_penalty = (
                output["bounded_residual"]
                .square()
                .mean()
            )
            residual_mean_penalty = (
                output["bounded_residual"]
                .mean()
                .square()
            )

            loss = (
                classification_loss
                + domain_loss
                + PHASE35_TRAIN_CONFIG[
                    "residual_regularization"
                ]
                * residual_penalty
                + PHASE35_TRAIN_CONFIG[
                    "residual_mean_regularization"
                ]
                * residual_mean_penalty
            )

            optimizer.zero_grad(
                set_to_none=True
            )
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE35_TRAIN_CONFIG[
                    "gradient_clip"
                ],
            )

            optimizer.step()

            batch_n = int(
                batch_indices.numel()
            )
            epoch_loss_sum += (
                float(loss.detach().item())
                * batch_n
            )
            epoch_case_count += batch_n

        scheduler.step()

        final_training_loss = (
            epoch_loss_sum
            / epoch_case_count
        )

        (
            monitor_residual,
            monitor_final_logit,
        ) = phase35_predict_residual(
            model,
            monitor_dataset,
        )

        monitor_probability = (
            phase33d_sigmoid(
                monitor_final_logit
            )
        )
        monitor_score = (
            phase35_robust_score(
                monitor_labels_numpy,
                monitor_probability,
                monitor_groups_numpy,
            )
        )

        epoch_records.append({
            "epoch": int(epoch),
            "training_loss": float(
                final_training_loss
            ),
            "monitor_log_loss": float(
                monitor_score[
                    "overall_log_loss"
                ]
            ),
            "monitor_robust_score": float(
                monitor_score[
                    "robust_score"
                ]
            ),
            "mean_absolute_residual": float(
                np.mean(
                    np.abs(
                        monitor_residual
                    )
                )
            ),
        })

        if (
            monitor_score["robust_score"]
            < best_score - 1e-6
        ):
            best_score = float(
                monitor_score[
                    "robust_score"
                ]
            )
            best_epoch = int(epoch)
            best_state = (
                phase35_state_to_cpu(
                    model
                )
            )
            best_monitor_residual = (
                monitor_residual.copy()
            )
            best_monitor_probability = (
                monitor_probability.copy()
            )
            patience_counter = 0
        else:
            patience_counter += 1

        if (
            epoch == 1
            or epoch % 20 == 0
        ):
            print(
                "Phase35 "
                f"fold {fold} "
                f"seed {seed} "
                f"epoch {epoch:03d}: "
                f"train={final_training_loss:.4f}, "
                "monitor="
                f"{monitor_score['overall_log_loss']:.4f}, "
                "robust="
                f"{monitor_score['robust_score']:.4f}"
            )

        if (
            patience_counter
            >= PHASE35_TRAIN_CONFIG[
                "patience"
            ]
        ):
            break

    reconstruction_model = (
        phase35_new_model().to(
            PHASE35_DEVICE
        )
    )
    reconstruction_model.load_state_dict(
        best_state,
        strict=True,
    )

    reconstructed_residual, _ = (
        phase35_predict_residual(
            reconstruction_model,
            monitor_dataset,
        )
    )

    reconstruction_error = float(
        np.max(
            np.abs(
                reconstructed_residual
                - best_monitor_residual
            )
        )
    )

    assert reconstruction_error <= 1e-7

    return {
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "baseline_monitor_score": (
            baseline_monitor_score
        ),
        "best_monitor_robust_score": float(
            best_score
        ),
        "best_monitor_probability": (
            best_monitor_probability
        ),
        "best_monitor_residual": (
            best_monitor_residual
        ),
        "best_state": best_state,
        "epochs_run": int(
            len(epoch_records)
        ),
        "final_training_loss": (
            None
            if final_training_loss is None
            else float(
                final_training_loss
            )
        ),
        "checkpoint_reconstruction_error": (
            reconstruction_error
        ),
        "epoch_records": epoch_records,
    }


def phase35_train_fixed_epochs(
    train_dataset,
    train_groups_numpy,
    seed,
    epoch_count,
):
    phase35_set_seed(seed)

    model = phase35_new_model().to(
        PHASE35_DEVICE
    )

    if int(epoch_count) == 0:
        return {
            "state": phase35_state_to_cpu(
                model
            ),
            "final_training_loss": None,
        }

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=PHASE35_TRAIN_CONFIG[
            "learning_rate"
        ],
        weight_decay=PHASE35_TRAIN_CONFIG[
            "weight_decay"
        ],
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(
                1,
                int(epoch_count),
            ),
            eta_min=PHASE35_TRAIN_CONFIG[
                "minimum_learning_rate"
            ],
        )
    )

    train_sample_weights = torch.as_tensor(
        phase35_group_sample_weights(
            train_groups_numpy
        ),
        device=PHASE35_DEVICE,
        dtype=torch.float32,
    )

    train_n = int(
        train_dataset["labels"].shape[0]
    )
    final_training_loss = None

    for epoch in range(
        1,
        int(epoch_count) + 1,
    ):
        model.train()

        order = torch.randperm(
            train_n,
            device=PHASE35_DEVICE,
        )

        epoch_loss_sum = 0.0
        epoch_case_count = 0

        domain_ramp = min(
            1.0,
            epoch
            / PHASE35_TRAIN_CONFIG[
                "domain_ramp_epochs"
            ],
        )
        grl_strength = (
            PHASE35_TRAIN_CONFIG[
                "domain_strength"
            ]
            * domain_ramp
        )

        for start in range(
            0,
            train_n,
            PHASE35_TRAIN_CONFIG[
                "batch_size"
            ],
        ):
            batch_indices = order[
                start:
                start
                + PHASE35_TRAIN_CONFIG[
                    "batch_size"
                ]
            ]
            batch = phase35_slice_dataset(
                train_dataset,
                batch_indices,
            )
            augmented = (
                phase35_augment_batch(
                    batch
                )
            )

            output = model(
                augmented[
                    "p3ca_sequence"
                ],
                augmented[
                    "geometry_sequence"
                ],
                augmented[
                    "summary_features"
                ],
                augmented[
                    "radiomics_features"
                ],
                batch["baseline_logit"],
                gradient_reversal_strength=(
                    grl_strength
                ),
            )

            target = batch["labels"]
            smoothing = float(
                PHASE35_TRAIN_CONFIG[
                    "label_smoothing"
                ]
            )
            smoothed_target = (
                target * (1.0 - smoothing)
                + 0.5 * smoothing
            )

            classification_losses = (
                F.binary_cross_entropy_with_logits(
                    output["final_logit"],
                    smoothed_target,
                    reduction="none",
                )
            )
            sample_weights = (
                train_sample_weights[
                    batch_indices
                ]
            )
            classification_loss = (
                torch.sum(
                    classification_losses
                    * sample_weights
                )
                / torch.sum(sample_weights)
            )

            domain_loss = F.cross_entropy(
                output["domain_logit"],
                batch["groups"],
            )
            residual_penalty = (
                output["bounded_residual"]
                .square()
                .mean()
            )
            residual_mean_penalty = (
                output["bounded_residual"]
                .mean()
                .square()
            )

            loss = (
                classification_loss
                + domain_loss
                + PHASE35_TRAIN_CONFIG[
                    "residual_regularization"
                ]
                * residual_penalty
                + PHASE35_TRAIN_CONFIG[
                    "residual_mean_regularization"
                ]
                * residual_mean_penalty
            )

            optimizer.zero_grad(
                set_to_none=True
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                PHASE35_TRAIN_CONFIG[
                    "gradient_clip"
                ],
            )
            optimizer.step()

            batch_n = int(
                batch_indices.numel()
            )
            epoch_loss_sum += (
                float(loss.detach().item())
                * batch_n
            )
            epoch_case_count += batch_n

        scheduler.step()

        final_training_loss = (
            epoch_loss_sum
            / epoch_case_count
        )

    return {
        "state": phase35_state_to_cpu(
            model
        ),
        "final_training_loss": float(
            final_training_loss
        ),
    }


phase35_training_utility_report = {
    "phase": (
        "phase35_anchored_residual_"
        "training_contract"
    ),
    "status": "accepted",
    "configuration": (
        PHASE35_TRAIN_CONFIG
    ),
    "baseline_training_source": (
        "exact_cross_fitted_phase12c_oof"
    ),
    "model_initialization": (
        "exact_phase12c_anchor"
    ),
    "early_stopping_epoch_zero_allowed": True,
    "shared_residual_scale_selection": True,
    "shared_scale_candidate_count": len(
        PHASE35_TRAIN_CONFIG[
            "shared_scale_candidates"
        ]
    ),
    "domain_strength_fixed": (
        PHASE35_TRAIN_CONFIG[
            "domain_strength"
        ]
    ),
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE35_TRAINING_CONTRACT")
print(json.dumps(
    phase35_training_utility_report,
    indent=2,
))
print("END SANITIZED_PHASE35_TRAINING_CONTRACT")

BEGIN SANITIZED_PHASE35_TRAINING_CONTRACT
{
  "phase": "phase35_anchored_residual_training_contract",
  "status": "accepted",
  "configuration": {
    "seeds": [
      350701,
      350702,
      350703
    ],
    "batch_size": 64,
    "maximum_epochs": 100,
    "patience": 18,
    "learning_rate": 0.00025,
    "minimum_learning_rate": 1.5e-05,
    "weight_decay": 0.03,
    "gradient_clip": 1.5,
    "label_smoothing": 0.01,
    "token_dropout": 0.03,
    "feature_noise": 0.008,
    "domain_strength": 0.02,
    "domain_ramp_epochs": 15,
    "residual_regularization": 0.02,
    "residual_mean_regularization": 0.01,
    "shared_scale_candidates": [
      0.0,
      0.25,
      0.5,
      0.75,
      1.0
    ],
    "minimum_shared_monitor_gain": 0.002,
    "minimum_shared_robust_gain": 0.002,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "maximum_monitor_group_harm": 0.015
  },
  "baseline_training_source": "exact_cross_fitted_phase12c_oof",
  "model_ini

In [62]:
# Phase35 Cell 116B
# Nested residual training, shared monitor-scale selection,
# outer-train refit, and final OOF evaluation.

import gc
import json
import time

import numpy as np
import torch


phase35_training_started = time.perf_counter()

phase35_labels = np.asarray(
    case_df["label"],
    dtype=np.int64,
)
phase35_groups = np.asarray(
    case_df["acquisition_group"],
    dtype=np.int64,
)
phase35_baseline_oof = phase33d_probability(
    PHASE19_PHASE12C_OOF_PRIVATE
)

assert phase35_labels.shape == (1362,)
assert phase35_groups.shape == (1362,)
assert phase35_baseline_oof.shape == (1362,)

phase35_selection_records = []
phase35_monitor_records_private = []

# ---------------------------------------------------------
# Selection-stage training: fit partition only.
# ---------------------------------------------------------

for partition in PHASE19_PARTITIONS:
    fold_started = time.perf_counter()

    fold = int(partition["fold"])
    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )

    fit_preprocessor = (
        phase33_fit_preprocessor(
            0,
            fold,
            fit_indices,
            355000 + fold,
        )
    )

    fit_features = phase33_transform(
        fit_preprocessor,
        fit_indices,
    )
    monitor_features = phase33_transform(
        fit_preprocessor,
        monitor_indices,
    )

    fit_dataset = (
        phase35_dataset_to_device(
            fit_features,
            phase35_labels[fit_indices],
            phase35_groups[fit_indices],
            phase35_baseline_oof[
                fit_indices
            ],
            PHASE35_DEVICE,
        )
    )
    monitor_dataset = (
        phase35_dataset_to_device(
            monitor_features,
            phase35_labels[
                monitor_indices
            ],
            phase35_groups[
                monitor_indices
            ],
            phase35_baseline_oof[
                monitor_indices
            ],
            PHASE35_DEVICE,
        )
    )

    seed_results = []

    for seed in PHASE35_TRAIN_CONFIG[
        "seeds"
    ]:
        seed_started = time.perf_counter()

        result = phase35_train_with_monitor(
            train_dataset=fit_dataset,
            monitor_dataset=monitor_dataset,
            train_groups_numpy=(
                phase35_groups[fit_indices]
            ),
            monitor_labels_numpy=(
                phase35_labels[
                    monitor_indices
                ]
            ),
            monitor_groups_numpy=(
                phase35_groups[
                    monitor_indices
                ]
            ),
            seed=seed,
            fold=fold,
        )

        result["training_seconds"] = float(
            time.perf_counter()
            - seed_started
        )
        seed_results.append(result)

        print(
            "Phase35 "
            f"fold {fold} "
            f"seed {seed} frozen: "
            f"best_epoch={result['best_epoch']}, "
            "robust="
            f"{result['best_monitor_robust_score']:.6f}"
        )

    ensemble_monitor_residual = np.mean(
        np.stack([
            result[
                "best_monitor_residual"
            ]
            for result in seed_results
        ]),
        axis=0,
    )

    ensemble_monitor_probability = (
        phase33d_sigmoid(
            phase33d_logit(
                phase35_baseline_oof[
                    monitor_indices
                ]
            )
            + ensemble_monitor_residual
        )
    )

    ensemble_monitor_metrics = (
        phase33d_metrics(
            phase35_labels[
                monitor_indices
            ],
            ensemble_monitor_probability,
        )
    )
    ensemble_monitor_robust = (
        phase35_robust_score(
            phase35_labels[
                monitor_indices
            ],
            ensemble_monitor_probability,
            phase35_groups[
                monitor_indices
            ],
        )
    )

    baseline_monitor_metrics = (
        phase33d_metrics(
            phase35_labels[
                monitor_indices
            ],
            phase35_baseline_oof[
                monitor_indices
            ],
        )
    )
    baseline_monitor_robust = (
        phase35_robust_score(
            phase35_labels[
                monitor_indices
            ],
            phase35_baseline_oof[
                monitor_indices
            ],
            phase35_groups[
                monitor_indices
            ],
        )
    )

    phase35_monitor_records_private.append({
        "fold": fold,
        "indices": monitor_indices.copy(),
        "baseline_probability": (
            phase35_baseline_oof[
                monitor_indices
            ].copy()
        ),
        "ensemble_residual": (
            ensemble_monitor_residual.copy()
        ),
        "labels": phase35_labels[
            monitor_indices
        ].copy(),
        "groups": phase35_groups[
            monitor_indices
        ].copy(),
    })

    phase35_selection_records.append({
        "fold": fold,
        "fit_indices": fit_indices,
        "monitor_indices": (
            monitor_indices
        ),
        "fit_preprocessor": (
            fit_preprocessor
        ),
        "seed_results": seed_results,
        "baseline_monitor_metrics": (
            baseline_monitor_metrics
        ),
        "baseline_monitor_robust": (
            baseline_monitor_robust
        ),
        "ensemble_monitor_metrics": (
            ensemble_monitor_metrics
        ),
        "ensemble_monitor_robust": (
            ensemble_monitor_robust
        ),
        "selection_training_seconds": (
            float(
                time.perf_counter()
                - fold_started
            )
        ),
    })

    del fit_features
    del monitor_features
    del fit_dataset
    del monitor_dataset

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------
# One shared residual scale across all folds.
# Duplicate monitor cases receive inverse multiplicity weights.
# ---------------------------------------------------------

pooled_indices = np.concatenate([
    record["indices"]
    for record in (
        phase35_monitor_records_private
    )
])
pooled_baseline = np.concatenate([
    record["baseline_probability"]
    for record in (
        phase35_monitor_records_private
    )
])
pooled_residual = np.concatenate([
    record["ensemble_residual"]
    for record in (
        phase35_monitor_records_private
    )
])
pooled_labels = np.concatenate([
    record["labels"]
    for record in (
        phase35_monitor_records_private
    )
])
pooled_groups = np.concatenate([
    record["groups"]
    for record in (
        phase35_monitor_records_private
    )
])
pooled_folds = np.concatenate([
    np.full(
        record["indices"].size,
        record["fold"],
        dtype=np.int64,
    )
    for record in (
        phase35_monitor_records_private
    )
])

unique_indices, multiplicities = np.unique(
    pooled_indices,
    return_counts=True,
)

multiplicity_lookup = {
    int(index): int(count)
    for index, count in zip(
        unique_indices,
        multiplicities,
    )
}

pooled_weights = np.asarray([
    1.0
    / multiplicity_lookup[int(index)]
    for index in pooled_indices
], dtype=np.float64)

pooled_baseline_robust = (
    phase33g_robust_score(
        pooled_labels,
        pooled_baseline,
        pooled_groups,
        pooled_weights,
    )
)

phase35_scale_candidate_records = []

for scale in PHASE35_TRAIN_CONFIG[
    "shared_scale_candidates"
]:
    candidate_probability = (
        phase33d_sigmoid(
            phase33d_logit(
                pooled_baseline
            )
            + float(scale)
            * pooled_residual
        )
    )

    candidate_robust = (
        phase33g_robust_score(
            pooled_labels,
            candidate_probability,
            pooled_groups,
            pooled_weights,
        )
    )

    overall_gain = (
        pooled_baseline_robust[
            "overall_log_loss"
        ]
        - candidate_robust[
            "overall_log_loss"
        ]
    )
    robust_gain = (
        pooled_baseline_robust[
            "robust_score"
        ]
        - candidate_robust[
            "robust_score"
        ]
    )

    fold_records = []

    for fold in range(3):
        mask = pooled_folds == fold

        baseline_loss = (
            phase33g_weighted_log_loss(
                pooled_labels[mask],
                pooled_baseline[mask],
                pooled_weights[mask],
            )
        )
        candidate_loss = (
            phase33g_weighted_log_loss(
                pooled_labels[mask],
                candidate_probability[
                    mask
                ],
                pooled_weights[mask],
            )
        )

        fold_records.append({
            "fold": fold,
            "baseline_log_loss": (
                baseline_loss
            ),
            "candidate_log_loss": (
                candidate_loss
            ),
            "gain": (
                baseline_loss
                - candidate_loss
            ),
        })

    fold_win_count = sum(
        record["gain"] >= 0.0005
        for record in fold_records
    )
    worst_fold_regret = max(
        -record["gain"]
        for record in fold_records
    )

    (
        maximum_group_harm,
        group_records,
    ) = phase33g_monitor_group_harm(
        pooled_labels,
        pooled_baseline,
        candidate_probability,
        pooled_groups,
        pooled_weights,
        minimum_group_n=8,
    )

    eligible = bool(
        float(scale) > 0.0
        and overall_gain
        >= PHASE35_TRAIN_CONFIG[
            "minimum_shared_monitor_gain"
        ]
        and robust_gain
        >= PHASE35_TRAIN_CONFIG[
            "minimum_shared_robust_gain"
        ]
        and fold_win_count
        >= PHASE35_TRAIN_CONFIG[
            "minimum_monitor_fold_wins"
        ]
        and worst_fold_regret
        <= PHASE35_TRAIN_CONFIG[
            "maximum_monitor_fold_regret"
        ]
        and maximum_group_harm
        <= PHASE35_TRAIN_CONFIG[
            "maximum_monitor_group_harm"
        ]
    )

    phase35_scale_candidate_records.append({
        "scale": float(scale),
        "probability": (
            candidate_probability
        ),
        "robust": candidate_robust,
        "overall_gain": float(
            overall_gain
        ),
        "robust_gain": float(
            robust_gain
        ),
        "fold_records": fold_records,
        "fold_win_count": int(
            fold_win_count
        ),
        "worst_fold_regret": float(
            worst_fold_regret
        ),
        "maximum_group_harm": float(
            maximum_group_harm
        ),
        "group_records": group_records,
        "eligible": eligible,
    })

phase35_raw_best_scale_record = min(
    phase35_scale_candidate_records,
    key=lambda record: (
        record["robust"][
            "robust_score"
        ],
        record["robust"][
            "overall_log_loss"
        ],
        record["scale"],
    ),
)

phase35_eligible_scale_records = [
    record
    for record in (
        phase35_scale_candidate_records
    )
    if record["eligible"]
]

if phase35_eligible_scale_records:
    phase35_selected_scale_record = min(
        phase35_eligible_scale_records,
        key=lambda record: (
            record["robust"][
                "robust_score"
            ],
            record["robust"][
                "overall_log_loss"
            ],
            record["scale"],
        ),
    )
    phase35_residual_advanced = True
else:
    phase35_selected_scale_record = next(
        record
        for record in (
            phase35_scale_candidate_records
        )
        if record["scale"] == 0.0
    )
    phase35_residual_advanced = False

PHASE35_SHARED_SCALE_PRIVATE = float(
    phase35_selected_scale_record[
        "scale"
    ]
)

print(
    "Phase35 shared residual scale: "
    f"{PHASE35_SHARED_SCALE_PRIVATE:.2f}, "
    f"advanced={phase35_residual_advanced}, "
    "monitor_gain="
    f"{phase35_selected_scale_record['overall_gain']:.6f}, "
    "robust_gain="
    f"{phase35_selected_scale_record['robust_gain']:.6f}"
)

# ---------------------------------------------------------
# Final outer-train refit and one-time outer evaluation.
# ---------------------------------------------------------

PHASE35_OOF_PRIVATE = (
    phase35_baseline_oof.copy()
)
PHASE35_RAW_RESIDUAL_OOF_PRIVATE = (
    np.zeros(
        1362,
        dtype=np.float64,
    )
)
PHASE35_DEPLOYMENT_STATES_PRIVATE = []
phase35_final_fold_records = []

if phase35_residual_advanced:
    for partition, selection_record in zip(
        PHASE19_PARTITIONS,
        phase35_selection_records,
    ):
        fold_started = time.perf_counter()

        fold = int(partition["fold"])
        assert (
            fold
            == selection_record["fold"]
        )

        outer_train_indices = np.asarray(
            partition["outer_train"],
            dtype=np.int64,
        )
        outer_valid_indices = np.asarray(
            partition["outer_valid"],
            dtype=np.int64,
        )

        final_preprocessor = (
            phase33_fit_preprocessor(
                1,
                fold,
                outer_train_indices,
                356000 + fold,
            )
        )

        final_train_features = (
            phase33_transform(
                final_preprocessor,
                outer_train_indices,
            )
        )
        outer_valid_features = (
            phase33_transform(
                final_preprocessor,
                outer_valid_indices,
            )
        )

        final_train_dataset = (
            phase35_dataset_to_device(
                final_train_features,
                phase35_labels[
                    outer_train_indices
                ],
                phase35_groups[
                    outer_train_indices
                ],
                phase35_baseline_oof[
                    outer_train_indices
                ],
                PHASE35_DEVICE,
            )
        )
        outer_valid_dataset = (
            phase35_dataset_to_device(
                outer_valid_features,
                phase35_labels[
                    outer_valid_indices
                ],
                phase35_groups[
                    outer_valid_indices
                ],
                phase35_baseline_oof[
                    outer_valid_indices
                ],
                PHASE35_DEVICE,
            )
        )

        final_model_states = []
        final_residual_parts = []
        final_training_losses = []
        best_epochs = []

        for seed_result in (
            selection_record[
                "seed_results"
            ]
        ):
            seed = int(
                seed_result["seed"]
            )
            best_epoch = int(
                seed_result[
                    "best_epoch"
                ]
            )
            best_epochs.append(
                best_epoch
            )

            final_result = (
                phase35_train_fixed_epochs(
                    train_dataset=(
                        final_train_dataset
                    ),
                    train_groups_numpy=(
                        phase35_groups[
                            outer_train_indices
                        ]
                    ),
                    seed=seed,
                    epoch_count=best_epoch,
                )
            )

            final_state = (
                final_result["state"]
            )
            final_model_states.append(
                final_state
            )
            final_training_losses.append(
                final_result[
                    "final_training_loss"
                ]
            )

            inference_model = (
                phase35_new_model().to(
                    PHASE35_DEVICE
                )
            )
            inference_model.load_state_dict(
                final_state,
                strict=True,
            )

            residual, _ = (
                phase35_predict_residual(
                    inference_model,
                    outer_valid_dataset,
                )
            )
            final_residual_parts.append(
                residual
            )

            del inference_model

        ensemble_outer_residual = np.mean(
            np.stack(
                final_residual_parts
            ),
            axis=0,
        )

        PHASE35_RAW_RESIDUAL_OOF_PRIVATE[
            outer_valid_indices
        ] = ensemble_outer_residual

        outer_probability = (
            phase33d_sigmoid(
                phase33d_logit(
                    phase35_baseline_oof[
                        outer_valid_indices
                    ]
                )
                + PHASE35_SHARED_SCALE_PRIVATE
                * ensemble_outer_residual
            )
        )

        PHASE35_OOF_PRIVATE[
            outer_valid_indices
        ] = outer_probability

        baseline_metrics = (
            phase33d_metrics(
                phase35_labels[
                    outer_valid_indices
                ],
                phase35_baseline_oof[
                    outer_valid_indices
                ],
            )
        )
        phase35_metrics = (
            phase33d_metrics(
                phase35_labels[
                    outer_valid_indices
                ],
                outer_probability,
            )
        )

        phase35_final_fold_records.append({
            "fold": fold,
            "n": int(
                outer_valid_indices.size
            ),
            "best_epochs": (
                best_epochs
            ),
            "baseline_log_loss": round(
                baseline_metrics[
                    "log_loss"
                ],
                6,
            ),
            "phase35_log_loss": round(
                phase35_metrics[
                    "log_loss"
                ],
                6,
            ),
            "log_loss_improvement": round(
                baseline_metrics[
                    "log_loss"
                ]
                - phase35_metrics[
                    "log_loss"
                ],
                6,
            ),
            "baseline_auroc": round(
                baseline_metrics[
                    "auroc"
                ],
                6,
            ),
            "phase35_auroc": round(
                phase35_metrics[
                    "auroc"
                ],
                6,
            ),
            "mean_absolute_residual": round(
                float(
                    np.mean(
                        np.abs(
                            ensemble_outer_residual
                        )
                    )
                ),
                6,
            ),
            "final_training_losses": (
                final_training_losses
            ),
            "refit_seconds": round(
                time.perf_counter()
                - fold_started,
                2,
            ),
        })

        PHASE35_DEPLOYMENT_STATES_PRIVATE.append({
            "fold": fold,
            "preprocessor": (
                final_preprocessor
            ),
            "model_states": (
                final_model_states
            ),
            "seeds": [
                int(
                    result["seed"]
                )
                for result in (
                    selection_record[
                        "seed_results"
                    ]
                )
            ],
            "best_epochs": best_epochs,
        })

        del final_train_features
        del outer_valid_features
        del final_train_dataset
        del outer_valid_dataset

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

else:
    for partition in PHASE19_PARTITIONS:
        fold = int(partition["fold"])
        outer_valid_indices = np.asarray(
            partition["outer_valid"],
            dtype=np.int64,
        )

        metrics = phase33d_metrics(
            phase35_labels[
                outer_valid_indices
            ],
            phase35_baseline_oof[
                outer_valid_indices
            ],
        )

        phase35_final_fold_records.append({
            "fold": fold,
            "n": int(
                outer_valid_indices.size
            ),
            "best_epochs": [0, 0, 0],
            "baseline_log_loss": round(
                metrics["log_loss"],
                6,
            ),
            "phase35_log_loss": round(
                metrics["log_loss"],
                6,
            ),
            "log_loss_improvement": 0.0,
            "baseline_auroc": round(
                metrics["auroc"],
                6,
            ),
            "phase35_auroc": round(
                metrics["auroc"],
                6,
            ),
            "mean_absolute_residual": 0.0,
            "final_training_losses": [
                None,
                None,
                None,
            ],
            "refit_seconds": 0.0,
        })

phase35_baseline_metrics = phase33d_metrics(
    phase35_labels,
    phase35_baseline_oof,
)
phase35_final_metrics = phase33d_metrics(
    phase35_labels,
    PHASE35_OOF_PRIVATE,
)
phase35_calibration = (
    phase33g_calibration_statistics(
        phase35_labels,
        PHASE35_OOF_PRIVATE,
    )
)

phase35_log_loss_gain = (
    phase35_baseline_metrics["log_loss"]
    - phase35_final_metrics["log_loss"]
)
phase35_auroc_gain = (
    phase35_final_metrics["auroc"]
    - phase35_baseline_metrics["auroc"]
)
phase35_brier_excess = (
    phase35_final_metrics["brier"]
    - phase35_baseline_metrics["brier"]
)

phase35_fold_wins = sum(
    record["log_loss_improvement"] > 1e-8
    for record in (
        phase35_final_fold_records
    )
)
phase35_worst_fold_excess = max(
    -record["log_loss_improvement"]
    for record in (
        phase35_final_fold_records
    )
)

(
    phase35_maximum_group_harm,
    phase35_group_records,
) = phase33d_group_harm(
    phase35_labels,
    phase35_baseline_oof,
    PHASE35_OOF_PRIVATE,
    phase35_groups,
)

phase35_component_gate_passed = bool(
    phase35_residual_advanced
    and phase35_log_loss_gain >= 0.003
    and phase35_auroc_gain >= 0.001
    and phase35_fold_wins >= 2
    and phase35_worst_fold_excess <= 0.005
    and phase35_brier_excess <= 0.001
    and phase35_maximum_group_harm <= 0.015
    and 0.8
    <= phase35_calibration[
        "calibration_slope"
    ]
    <= 1.2
)

phase35_beats_phase33 = bool(
    phase35_final_metrics["log_loss"]
    <= phase33_final_gated_metrics[
        "log_loss"
    ] - 0.001
)

phase35_status = (
    "new_champion"
    if (
        phase35_component_gate_passed
        and phase35_beats_phase33
    )
    else (
        "component_promoted"
        if phase35_component_gate_passed
        else "not_promoted"
    )
)

phase35_selection_summary = []

for record in phase35_selection_records:
    phase35_selection_summary.append({
        "fold": int(record["fold"]),
        "fit_n": int(
            record["fit_indices"].size
        ),
        "monitor_n": int(
            record[
                "monitor_indices"
            ].size
        ),
        "baseline_monitor_log_loss": round(
            record[
                "baseline_monitor_metrics"
            ]["log_loss"],
            6,
        ),
        "ensemble_monitor_log_loss": round(
            record[
                "ensemble_monitor_metrics"
            ]["log_loss"],
            6,
        ),
        "baseline_monitor_robust_score": (
            round(
                record[
                    "baseline_monitor_robust"
                ]["robust_score"],
                6,
            )
        ),
        "ensemble_monitor_robust_score": (
            round(
                record[
                    "ensemble_monitor_robust"
                ]["robust_score"],
                6,
            )
        ),
        "seed_best_epochs": [
            int(result["best_epoch"])
            for result in (
                record["seed_results"]
            )
        ],
        "seed_best_robust_scores": [
            round(
                float(
                    result[
                        "best_monitor_robust_score"
                    ]
                ),
                6,
            )
            for result in (
                record["seed_results"]
            )
        ],
        "maximum_checkpoint_reconstruction_error": (
            max(
                float(
                    result[
                        "checkpoint_reconstruction_error"
                    ]
                )
                for result in (
                    record["seed_results"]
                )
            )
        ),
        "selection_training_seconds": (
            round(
                record[
                    "selection_training_seconds"
                ],
                2,
            )
        ),
    })

phase35_report = {
    "phase": (
        "phase35_phase12c_anchored_"
        "domain_invariant_residual"
    ),
    "status": phase35_status,
    "model": (
        "Phase35AnchoredResidualFusionNet"
    ),
    "parameter_count": 141244,
    "configuration": (
        PHASE35_TRAIN_CONFIG
    ),
    "shared_scale_selection": {
        "selected_scale": (
            PHASE35_SHARED_SCALE_PRIVATE
        ),
        "residual_advanced": (
            phase35_residual_advanced
        ),
        "candidate_count": len(
            phase35_scale_candidate_records
        ),
        "eligible_candidate_count": len(
            phase35_eligible_scale_records
        ),
        "raw_best_scale": (
            phase35_raw_best_scale_record[
                "scale"
            ]
        ),
        "raw_best_monitor_gain": round(
            phase35_raw_best_scale_record[
                "overall_gain"
            ],
            6,
        ),
        "selected_monitor_gain": round(
            phase35_selected_scale_record[
                "overall_gain"
            ],
            6,
        ),
        "selected_robust_gain": round(
            phase35_selected_scale_record[
                "robust_gain"
            ],
            6,
        ),
        "selected_monitor_fold_wins": int(
            phase35_selected_scale_record[
                "fold_win_count"
            ]
        ),
        "selected_worst_fold_regret": round(
            phase35_selected_scale_record[
                "worst_fold_regret"
            ],
            6,
        ),
        "selected_maximum_group_harm": round(
            phase35_selected_scale_record[
                "maximum_group_harm"
            ],
            6,
        ),
        "fold_monitor_metrics": [
            {
                "fold": int(
                    item["fold"]
                ),
                "baseline_log_loss": round(
                    item[
                        "baseline_log_loss"
                    ],
                    6,
                ),
                "candidate_log_loss": round(
                    item[
                        "candidate_log_loss"
                    ],
                    6,
                ),
                "gain": round(
                    item["gain"],
                    6,
                ),
            }
            for item in (
                phase35_selected_scale_record[
                    "fold_records"
                ]
            )
        ],
    },
    "baseline": {
        key: (
            None if value is None
            else round(float(value), 6)
        )
        for key, value in (
            phase35_baseline_metrics.items()
        )
    },
    "phase35": {
        **{
            key: (
                None if value is None
                else round(float(value), 6)
            )
            for key, value in (
                phase35_final_metrics.items()
            )
        },
        "calibration_intercept": round(
            phase35_calibration[
                "calibration_intercept"
            ],
            6,
        ),
        "calibration_slope": round(
            phase35_calibration[
                "calibration_slope"
            ],
            6,
        ),
    },
    "improvements": {
        "log_loss_gain": round(
            phase35_log_loss_gain,
            6,
        ),
        "auroc_gain": round(
            phase35_auroc_gain,
            6,
        ),
        "brier_excess": round(
            phase35_brier_excess,
            6,
        ),
        "fold_wins": int(
            phase35_fold_wins
        ),
        "worst_fold_excess": round(
            phase35_worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            phase35_maximum_group_harm,
            6,
        ),
        "log_loss_gain_over_phase33": round(
            phase33_final_gated_metrics[
                "log_loss"
            ]
            - phase35_final_metrics[
                "log_loss"
            ],
            6,
        ),
    },
    "selection_training": (
        phase35_selection_summary
    ),
    "fold_metrics": (
        phase35_final_fold_records
    ),
    "major_acquisition_group_metrics": (
        phase35_group_records
    ),
    "gates": {
        "component_gate_passed": (
            phase35_component_gate_passed
        ),
        "beats_phase33_by_0p001": (
            phase35_beats_phase33
        ),
    },
    "cross_fitted_phase12c_anchor_used_for_training": True,
    "shared_scale_across_folds": True,
    "fold_specific_gate_routing": False,
    "monitor_labels_used_for_early_stopping_and_scale_selection": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "selection_frozen_before_outer_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase35_training_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE35_TRAINING")
print(json.dumps(
    phase35_report,
    indent=2,
))
print("END SANITIZED_PHASE35_TRAINING")

Phase35 fold 0 seed 350701 epoch 001: train=2.9266, monitor=0.2756, robust=0.2878
Phase35 fold 0 seed 350701 epoch 020: train=1.5701, monitor=0.2933, robust=0.3117
Phase35 fold 0 seed 350701 frozen: best_epoch=6, robust=0.286543
Phase35 fold 0 seed 350702 epoch 001: train=2.9679, monitor=0.2755, robust=0.2877
Phase35 fold 0 seed 350702 epoch 020: train=1.5276, monitor=0.2637, robust=0.2814
Phase35 fold 0 seed 350702 epoch 040: train=1.3340, monitor=0.2687, robust=0.2824
Phase35 fold 0 seed 350702 frozen: best_epoch=24, robust=0.273998
Phase35 fold 0 seed 350703 epoch 001: train=2.9586, monitor=0.2755, robust=0.2878
Phase35 fold 0 seed 350703 epoch 020: train=1.6478, monitor=0.2869, robust=0.3075
Phase35 fold 0 seed 350703 frozen: best_epoch=4, robust=0.287417
Phase35 fold 1 seed 350701 epoch 001: train=2.9995, monitor=0.2173, robust=0.2151
Phase35 fold 1 seed 350701 epoch 020: train=1.1476, monitor=0.2188, robust=0.2229
Phase35 fold 1 seed 350701 frozen: best_epoch=7, robust=0.212300
P

In [63]:
# Phase36 Cell 117A
# Fold-local multimodal features for shallow boosted residual trees.

import importlib
import json
import time

import numpy as np


phase36_contract_started = time.perf_counter()

xgboost_spec = importlib.util.find_spec(
    "xgboost"
)
assert xgboost_spec is not None, (
    "xgboost is unavailable in this Kaggle image."
)

import xgboost as xgb


PHASE36_CONFIG = {
    "phase": (
        "phase36_phase12c_base_margin_"
        "boosted_residual"
    ),
    "p3ca_view_count": 6,
    "p3ca_tokens_per_view": 14,
    "p3ca_frequency_count": 3,
    "geometry_frequency_count": 4,
    "base_margin_clip": 12.0,
}


def phase36_sequence_statistics(
    sequence,
    frequency_count,
):
    sequence = np.asarray(
        sequence,
        dtype=np.float32,
    )

    assert sequence.ndim == 3

    mean = np.mean(
        sequence,
        axis=1,
    )
    standard_deviation = np.std(
        sequence,
        axis=1,
    )
    q25 = np.quantile(
        sequence,
        0.25,
        axis=1,
    ).astype(np.float32)
    q75 = np.quantile(
        sequence,
        0.75,
        axis=1,
    ).astype(np.float32)
    dynamic_range = (
        np.max(sequence, axis=1)
        - np.min(sequence, axis=1)
    )

    frequency = np.fft.rfft(
        sequence,
        axis=1,
    )
    frequency_magnitude = np.abs(
        frequency[
            :,
            1:
            1 + int(frequency_count),
            :,
        ]
    ).astype(np.float32)

    frequency_magnitude = (
        frequency_magnitude.reshape(
            sequence.shape[0],
            -1,
        )
    )

    return np.concatenate(
        [
            mean,
            standard_deviation,
            q25,
            q75,
            dynamic_range,
            frequency_magnitude,
        ],
        axis=1,
    ).astype(
        np.float32,
        copy=False,
    )


def phase36_build_features(
    transformed_features,
    baseline_probability,
):
    p3ca_sequence = np.asarray(
        transformed_features[
            "p3ca_sequence"
        ],
        dtype=np.float32,
    )
    geometry_sequence = np.asarray(
        transformed_features[
            "geometry_sequence"
        ],
        dtype=np.float32,
    )
    summary_features = np.asarray(
        transformed_features[
            "summary_features"
        ],
        dtype=np.float32,
    )
    radiomics_features = np.asarray(
        transformed_features[
            "radiomics_features"
        ],
        dtype=np.float32,
    )

    baseline_probability = (
        phase33d_probability(
            baseline_probability
        )
    )
    baseline_logit = np.clip(
        phase33d_logit(
            baseline_probability
        ),
        -PHASE36_CONFIG[
            "base_margin_clip"
        ],
        PHASE36_CONFIG[
            "base_margin_clip"
        ],
    )

    uncertainty = (
        4.0
        * baseline_probability
        * (1.0 - baseline_probability)
    )

    p3ca_statistics = (
        phase36_sequence_statistics(
            p3ca_sequence,
            PHASE36_CONFIG[
                "p3ca_frequency_count"
            ],
        )
    )
    geometry_statistics = (
        phase36_sequence_statistics(
            geometry_sequence,
            PHASE36_CONFIG[
                "geometry_frequency_count"
            ],
        )
    )

    assert p3ca_sequence.shape[1:] == (
        84,
        48,
    )

    p3ca_view_means = np.mean(
        p3ca_sequence.reshape(
            p3ca_sequence.shape[0],
            PHASE36_CONFIG[
                "p3ca_view_count"
            ],
            PHASE36_CONFIG[
                "p3ca_tokens_per_view"
            ],
            p3ca_sequence.shape[2],
        ),
        axis=2,
    ).reshape(
        p3ca_sequence.shape[0],
        -1,
    )

    anchor_features = np.stack(
        [
            baseline_logit,
            np.abs(baseline_logit),
            baseline_probability - 0.5,
            uncertainty,
        ],
        axis=1,
    ).astype(np.float32)

    feature_matrix = np.concatenate(
        [
            summary_features,
            radiomics_features,
            p3ca_statistics,
            p3ca_view_means,
            geometry_statistics,
            anchor_features,
        ],
        axis=1,
    ).astype(
        np.float32,
        copy=False,
    )

    assert np.isfinite(
        feature_matrix
    ).all()

    return feature_matrix, baseline_logit


# Fold-zero fit-only preprocessing contract.
phase36_fit_indices = np.asarray(
    PHASE19_PARTITIONS[0]["fit"],
    dtype=np.int64,
)

phase36_contract_indices = (
    phase36_fit_indices[:128]
)

phase36_contract_preprocessor = (
    phase33_fit_preprocessor(
        0,
        0,
        phase36_fit_indices,
        360101,
    )
)

phase36_contract_transformed = (
    phase33_transform(
        phase36_contract_preprocessor,
        phase36_contract_indices,
    )
)

(
    phase36_contract_features,
    phase36_contract_margin,
) = phase36_build_features(
    phase36_contract_transformed,
    PHASE19_PHASE12C_OOF_PRIVATE[
        phase36_contract_indices
    ],
)

assert phase36_contract_features.shape[0] == 128
assert phase36_contract_margin.shape == (128,)

# Verify that XGBoost respects the supplied baseline margin.
synthetic_labels = np.asarray(
    [0, 1] * 64,
    dtype=np.float32,
)

contract_dmatrix = xgb.DMatrix(
    phase36_contract_features,
    label=synthetic_labels,
    base_margin=phase36_contract_margin,
)

zero_tree_booster = xgb.train(
    params={
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "max_depth": 2,
        "eta": 0.05,
        "seed": 360101,
        "nthread": 4,
    },
    dtrain=contract_dmatrix,
    num_boost_round=0,
)

zero_tree_margin = zero_tree_booster.predict(
    contract_dmatrix,
    output_margin=True,
)

base_margin_error = float(
    np.max(
        np.abs(
            zero_tree_margin
            - phase36_contract_margin
        )
    )
)

assert base_margin_error <= 1e-6, {
    "message": (
        "XGBoost did not preserve the supplied "
        "Phase12c base margin."
    ),
    "maximum_error": base_margin_error,
}

coordinate_standard_deviation = np.std(
    phase36_contract_features,
    axis=0,
)

collapsed_fraction = float(
    np.mean(
        coordinate_standard_deviation
        <= 1e-8
    )
)

phase36_contract_report = {
    "phase": (
        "phase36_boosted_residual_"
        "feature_contract"
    ),
    "status": "accepted",
    "backend": "xgboost",
    "backend_version": xgb.__version__,
    "feature_shape": list(
        phase36_contract_features.shape
    ),
    "feature_count": int(
        phase36_contract_features.shape[1]
    ),
    "feature_families": [
        "p3ca_summary_pca",
        "localized_radiomics_pca",
        "p3ca_sequence_statistics",
        "p3ca_view_means",
        "geometry_sequence_statistics",
        "phase12c_anchor_features",
    ],
    "base_margin_shape": list(
        phase36_contract_margin.shape
    ),
    "zero_tree_base_margin_max_abs_error": (
        base_margin_error
    ),
    "collapsed_coordinate_fraction": round(
        collapsed_fraction,
        8,
    ),
    "all_values_finite": bool(
        np.isfinite(
            phase36_contract_features
        ).all()
    ),
    "planned_model": (
        "shallow_regularized_histogram_"
        "gradient_boosting"
    ),
    "planned_objective": (
        "binary_logloss_with_exact_"
        "phase12c_base_margin"
    ),
    "fit_preprocessor_uses_fit_partition_only": True,
    "labels_used": False,
    "outer_validation_images_used": False,
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase36_contract_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE36_FEATURE_CONTRACT")
print(json.dumps(
    phase36_contract_report,
    indent=2,
))
print("END SANITIZED_PHASE36_FEATURE_CONTRACT")

BEGIN SANITIZED_PHASE36_FEATURE_CONTRACT
{
  "phase": "phase36_boosted_residual_feature_contract",
  "status": "accepted",
  "backend": "xgboost",
  "backend_version": "3.2.0",
  "feature_shape": [
    128,
    1240
  ],
  "feature_count": 1240,
  "feature_families": [
    "p3ca_summary_pca",
    "localized_radiomics_pca",
    "p3ca_sequence_statistics",
    "p3ca_view_means",
    "geometry_sequence_statistics",
    "phase12c_anchor_features"
  ],
  "base_margin_shape": [
    128
  ],
  "zero_tree_base_margin_max_abs_error": 2.351262438082813e-07,
  "collapsed_coordinate_fraction": 0.0,
  "all_values_finite": true,
  "planned_model": "shallow_regularized_histogram_gradient_boosting",
  "planned_objective": "binary_logloss_with_exact_phase12c_base_margin",
  "fit_preprocessor_uses_fit_partition_only": true,
  "labels_used": false,
  "outer_validation_images_used": false,
  "outer_validation_labels_used": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "elapsed_seconds": 

In [64]:
# Phase36 Cell 117B
# Deterministic shallow XGBoost residual-training utilities.
# Primary selection: log loss.
# Secondary close tie-break: AUROC.

import json
import time

import numpy as np
import xgboost as xgb
from sklearn.metrics import log_loss, roc_auc_score


PHASE36_TRAIN_CONFIG = {
    "seeds": [
        360701,
        360702,
    ],
    "learning_rate": 0.03,
    "maximum_rounds": 900,
    "early_stopping_rounds": 60,
    "subsample": 0.85,
    "column_subsample": 0.60,
    "maximum_bin_count": 128,
    "maximum_delta_step": 1.0,
    "nthread": 8,
    "shared_scale_candidates": [
        0.0,
        0.25,
        0.50,
        0.75,
        1.0,
    ],
    "minimum_monitor_log_loss_gain": 0.002,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "maximum_monitor_group_harm": 0.015,
    "auroc_tie_tolerance": 0.00025,
}


PHASE36_CANDIDATE_SPECIFICATIONS = []

for maximum_depth in [1, 2, 3]:
    for minimum_child_weight in [8.0, 24.0]:
        for l2_regularization in [10.0, 30.0]:
            PHASE36_CANDIDATE_SPECIFICATIONS.append({
                "maximum_depth": int(
                    maximum_depth
                ),
                "minimum_child_weight": float(
                    minimum_child_weight
                ),
                "l2_regularization": float(
                    l2_regularization
                ),
            })

assert len(
    PHASE36_CANDIDATE_SPECIFICATIONS
) == 12


def phase36_group_sample_weights(groups):
    groups = np.asarray(
        groups,
        dtype=np.int64,
    ).reshape(-1)

    unique_groups, counts = np.unique(
        groups,
        return_counts=True,
    )

    count_lookup = {
        int(group): int(count)
        for group, count in zip(
            unique_groups,
            counts,
        )
    }

    weights = np.asarray([
        1.0 / count_lookup[int(group)]
        for group in groups
    ], dtype=np.float64)

    weights /= np.mean(weights)

    return weights.astype(np.float32)


def phase36_xgboost_parameters(
    specification,
    seed,
):
    return {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "max_depth": int(
            specification[
                "maximum_depth"
            ]
        ),
        "min_child_weight": float(
            specification[
                "minimum_child_weight"
            ]
        ),
        "lambda": float(
            specification[
                "l2_regularization"
            ]
        ),
        "alpha": 0.0,
        "eta": float(
            PHASE36_TRAIN_CONFIG[
                "learning_rate"
            ]
        ),
        "subsample": float(
            PHASE36_TRAIN_CONFIG[
                "subsample"
            ]
        ),
        "colsample_bytree": float(
            PHASE36_TRAIN_CONFIG[
                "column_subsample"
            ]
        ),
        "max_bin": int(
            PHASE36_TRAIN_CONFIG[
                "maximum_bin_count"
            ]
        ),
        "max_delta_step": float(
            PHASE36_TRAIN_CONFIG[
                "maximum_delta_step"
            ]
        ),
        "seed": int(seed),
        "nthread": int(
            PHASE36_TRAIN_CONFIG[
                "nthread"
            ]
        ),
        "verbosity": 0,
    }


def phase36_train_with_monitor(
    train_features,
    train_labels,
    train_groups,
    train_margin,
    monitor_features,
    monitor_labels,
    monitor_margin,
    specification,
    seed,
):
    train_weights = (
        phase36_group_sample_weights(
            train_groups
        )
    )

    training_matrix = xgb.DMatrix(
        np.asarray(
            train_features,
            dtype=np.float32,
        ),
        label=np.asarray(
            train_labels,
            dtype=np.float32,
        ),
        weight=train_weights,
        base_margin=np.asarray(
            train_margin,
            dtype=np.float32,
        ),
    )

    monitor_matrix = xgb.DMatrix(
        np.asarray(
            monitor_features,
            dtype=np.float32,
        ),
        label=np.asarray(
            monitor_labels,
            dtype=np.float32,
        ),
        base_margin=np.asarray(
            monitor_margin,
            dtype=np.float32,
        ),
    )

    booster = xgb.train(
        params=phase36_xgboost_parameters(
            specification,
            seed,
        ),
        dtrain=training_matrix,
        num_boost_round=(
            PHASE36_TRAIN_CONFIG[
                "maximum_rounds"
            ]
        ),
        evals=[
            (monitor_matrix, "monitor"),
        ],
        early_stopping_rounds=(
            PHASE36_TRAIN_CONFIG[
                "early_stopping_rounds"
            ]
        ),
        verbose_eval=False,
    )

    best_iteration = int(
        booster.best_iteration
    )
    best_round_count = (
        best_iteration + 1
    )

    monitor_total_margin = booster.predict(
        monitor_matrix,
        output_margin=True,
        iteration_range=(
            0,
            best_round_count,
        ),
    ).astype(np.float64)

    monitor_residual = (
        monitor_total_margin
        - np.asarray(
            monitor_margin,
            dtype=np.float64,
        )
    )

    monitor_probability = (
        phase33d_sigmoid(
            monitor_total_margin
        )
    )

    return {
        "booster": booster,
        "best_round_count": int(
            best_round_count
        ),
        "monitor_residual": (
            monitor_residual
        ),
        "monitor_probability": (
            monitor_probability
        ),
        "monitor_log_loss": float(
            log_loss(
                monitor_labels,
                monitor_probability,
            )
        ),
        "monitor_auroc": float(
            roc_auc_score(
                monitor_labels,
                monitor_probability,
            )
        ),
    }


def phase36_train_fixed_rounds(
    train_features,
    train_labels,
    train_groups,
    train_margin,
    specification,
    seed,
    round_count,
):
    training_matrix = xgb.DMatrix(
        np.asarray(
            train_features,
            dtype=np.float32,
        ),
        label=np.asarray(
            train_labels,
            dtype=np.float32,
        ),
        weight=phase36_group_sample_weights(
            train_groups
        ),
        base_margin=np.asarray(
            train_margin,
            dtype=np.float32,
        ),
    )

    booster = xgb.train(
        params=phase36_xgboost_parameters(
            specification,
            seed,
        ),
        dtrain=training_matrix,
        num_boost_round=int(
            round_count
        ),
        verbose_eval=False,
    )

    return booster


def phase36_predict_residual(
    booster,
    features,
    baseline_margin,
    round_count=None,
):
    matrix = xgb.DMatrix(
        np.asarray(
            features,
            dtype=np.float32,
        ),
        base_margin=np.asarray(
            baseline_margin,
            dtype=np.float32,
        ),
    )

    prediction_arguments = {
        "output_margin": True,
    }

    if round_count is not None:
        prediction_arguments[
            "iteration_range"
        ] = (
            0,
            int(round_count),
        )

    total_margin = booster.predict(
        matrix,
        **prediction_arguments,
    ).astype(np.float64)

    residual = (
        total_margin
        - np.asarray(
            baseline_margin,
            dtype=np.float64,
        )
    )

    assert np.isfinite(residual).all()

    return residual, total_margin


phase36_utility_report = {
    "phase": (
        "phase36_boosted_residual_"
        "training_contract"
    ),
    "status": "accepted",
    "backend": "xgboost",
    "backend_version": xgb.__version__,
    "candidate_specification_count": len(
        PHASE36_CANDIDATE_SPECIFICATIONS
    ),
    "seed_count": len(
        PHASE36_TRAIN_CONFIG["seeds"]
    ),
    "models_per_fold_during_selection": (
        len(
            PHASE36_CANDIDATE_SPECIFICATIONS
        )
        * len(
            PHASE36_TRAIN_CONFIG[
                "seeds"
            ]
        )
    ),
    "configuration": (
        PHASE36_TRAIN_CONFIG
    ),
    "primary_selection_metric": (
        "monitor_log_loss"
    ),
    "secondary_tie_break_metric": (
        "monitor_auroc"
    ),
    "auroc_affects_official_ranking": False,
    "base_margin": (
        "exact_cross_fitted_phase12c_logit"
    ),
    "group_balanced_training_weights": True,
    "fold_specific_model_selection": False,
    "planned_selection": (
        "one_shared_specification_and_scale"
    ),
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print("BEGIN SANITIZED_PHASE36_TRAINING_CONTRACT")
print(json.dumps(
    phase36_utility_report,
    indent=2,
))
print("END SANITIZED_PHASE36_TRAINING_CONTRACT")

BEGIN SANITIZED_PHASE36_TRAINING_CONTRACT
{
  "phase": "phase36_boosted_residual_training_contract",
  "status": "accepted",
  "backend": "xgboost",
  "backend_version": "3.2.0",
  "candidate_specification_count": 12,
  "seed_count": 2,
  "models_per_fold_during_selection": 24,
  "configuration": {
    "seeds": [
      360701,
      360702
    ],
    "learning_rate": 0.03,
    "maximum_rounds": 900,
    "early_stopping_rounds": 60,
    "subsample": 0.85,
    "column_subsample": 0.6,
    "maximum_bin_count": 128,
    "maximum_delta_step": 1.0,
    "nthread": 8,
    "shared_scale_candidates": [
      0.0,
      0.25,
      0.5,
      0.75,
      1.0
    ],
    "minimum_monitor_log_loss_gain": 0.002,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "maximum_monitor_group_harm": 0.015,
    "auroc_tie_tolerance": 0.00025
  },
  "primary_selection_metric": "monitor_log_loss",
  "secondary_tie_break_metric": "monitor_auroc",
  "auroc_affects_official_ranking":

In [65]:
# Phase36 Cell 117C
# Shared nested XGBoost specification/scale selection,
# outer-train refit, and final OOF evaluation.

import gc
import json
import time

import numpy as np
import xgboost as xgb
from sklearn.metrics import log_loss, roc_auc_score


PHASE36_TRAIN_CONFIG[
    "log_loss_tie_tolerance"
] = 0.00005

phase36_training_started = time.perf_counter()

phase36_labels = np.asarray(
    case_df["label"],
    dtype=np.int64,
)
phase36_groups = np.asarray(
    case_df["acquisition_group"],
    dtype=np.int64,
)
phase36_baseline_oof = phase33d_probability(
    PHASE19_PHASE12C_OOF_PRIVATE
)

assert phase36_labels.shape == (1362,)
assert phase36_groups.shape == (1362,)
assert phase36_baseline_oof.shape == (1362,)

phase36_selection_folds = []

# ---------------------------------------------------------
# Train every candidate on fit and predict monitor.
# ---------------------------------------------------------

for partition in PHASE19_PARTITIONS:
    fold_started = time.perf_counter()

    fold = int(partition["fold"])
    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )
    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )

    preprocessor = phase33_fit_preprocessor(
        0,
        fold,
        fit_indices,
        367000 + fold,
    )

    fit_transformed = phase33_transform(
        preprocessor,
        fit_indices,
    )
    monitor_transformed = phase33_transform(
        preprocessor,
        monitor_indices,
    )

    (
        fit_features,
        fit_margin,
    ) = phase36_build_features(
        fit_transformed,
        phase36_baseline_oof[
            fit_indices
        ],
    )
    (
        monitor_features,
        monitor_margin,
    ) = phase36_build_features(
        monitor_transformed,
        phase36_baseline_oof[
            monitor_indices
        ],
    )

    fold_specification_results = []

    for specification_index, specification in enumerate(
        PHASE36_CANDIDATE_SPECIFICATIONS
    ):
        seed_results = []

        for seed in PHASE36_TRAIN_CONFIG[
            "seeds"
        ]:
            result = phase36_train_with_monitor(
                train_features=fit_features,
                train_labels=phase36_labels[
                    fit_indices
                ],
                train_groups=phase36_groups[
                    fit_indices
                ],
                train_margin=fit_margin,
                monitor_features=(
                    monitor_features
                ),
                monitor_labels=phase36_labels[
                    monitor_indices
                ],
                monitor_margin=monitor_margin,
                specification=specification,
                seed=seed,
            )

            # Selection boosters are unnecessary after
            # extracting predictions and best rounds.
            result.pop("booster", None)
            seed_results.append(result)

        ensemble_residual = np.mean(
            np.stack([
                result["monitor_residual"]
                for result in seed_results
            ]),
            axis=0,
        )

        ensemble_probability = (
            phase33d_sigmoid(
                monitor_margin
                + ensemble_residual
            )
        )

        fold_specification_results.append({
            "specification_index": int(
                specification_index
            ),
            "specification": dict(
                specification
            ),
            "seed_results": seed_results,
            "ensemble_residual": (
                ensemble_residual
            ),
            "monitor_log_loss": float(
                log_loss(
                    phase36_labels[
                        monitor_indices
                    ],
                    ensemble_probability,
                )
            ),
            "monitor_auroc": float(
                roc_auc_score(
                    phase36_labels[
                        monitor_indices
                    ],
                    ensemble_probability,
                )
            ),
        })

        print(
            "Phase36 "
            f"fold {fold}: "
            f"{specification_index + 1}/"
            f"{len(PHASE36_CANDIDATE_SPECIFICATIONS)} "
            "specifications complete"
        )

    phase36_selection_folds.append({
        "fold": fold,
        "fit_indices": fit_indices,
        "monitor_indices": (
            monitor_indices
        ),
        "specification_results": (
            fold_specification_results
        ),
        "elapsed_seconds": float(
            time.perf_counter()
            - fold_started
        ),
    })

    del fit_transformed
    del monitor_transformed
    del fit_features
    del monitor_features

    gc.collect()

# ---------------------------------------------------------
# Pool independent monitor predictions.
# Duplicate monitor cases receive inverse-frequency weight.
# ---------------------------------------------------------

pooled_indices = np.concatenate([
    record["monitor_indices"]
    for record in phase36_selection_folds
])
pooled_labels = phase36_labels[
    pooled_indices
]
pooled_groups = phase36_groups[
    pooled_indices
]
pooled_folds = np.concatenate([
    np.full(
        record["monitor_indices"].size,
        record["fold"],
        dtype=np.int64,
    )
    for record in phase36_selection_folds
])
pooled_baseline = phase36_baseline_oof[
    pooled_indices
]
pooled_margin = phase33d_logit(
    pooled_baseline
)

unique_indices, multiplicities = np.unique(
    pooled_indices,
    return_counts=True,
)

multiplicity_lookup = {
    int(index): int(count)
    for index, count in zip(
        unique_indices,
        multiplicities,
    )
}

pooled_weights = np.asarray([
    1.0
    / multiplicity_lookup[int(index)]
    for index in pooled_indices
], dtype=np.float64)

specification_residual_parts = []

for specification_index in range(
    len(PHASE36_CANDIDATE_SPECIFICATIONS)
):
    specification_residual_parts.append(
        np.concatenate([
            fold_record[
                "specification_results"
            ][
                specification_index
            ]["ensemble_residual"]
            for fold_record in (
                phase36_selection_folds
            )
        ])
    )

pooled_residual_matrix = np.stack(
    specification_residual_parts,
    axis=1,
)

assert pooled_residual_matrix.shape == (
    pooled_indices.size,
    len(PHASE36_CANDIDATE_SPECIFICATIONS),
)

baseline_pooled_log_loss = (
    phase33g_weighted_log_loss(
        pooled_labels,
        pooled_baseline,
        pooled_weights,
    )
)

baseline_pooled_auroc = float(
    roc_auc_score(
        pooled_labels,
        pooled_baseline,
        sample_weight=pooled_weights,
    )
)

phase36_joint_candidate_records = []

for specification_index, specification in enumerate(
    PHASE36_CANDIDATE_SPECIFICATIONS
):
    raw_residual = pooled_residual_matrix[
        :,
        specification_index,
    ]

    for scale in PHASE36_TRAIN_CONFIG[
        "shared_scale_candidates"
    ]:
        probability = phase33d_sigmoid(
            pooled_margin
            + float(scale)
            * raw_residual
        )

        candidate_log_loss = (
            phase33g_weighted_log_loss(
                pooled_labels,
                probability,
                pooled_weights,
            )
        )
        candidate_auroc = float(
            roc_auc_score(
                pooled_labels,
                probability,
                sample_weight=pooled_weights,
            )
        )

        log_loss_gain = (
            baseline_pooled_log_loss
            - candidate_log_loss
        )
        auroc_gain = (
            candidate_auroc
            - baseline_pooled_auroc
        )

        fold_records = []

        for fold in range(3):
            mask = pooled_folds == fold

            baseline_fold_loss = (
                phase33g_weighted_log_loss(
                    pooled_labels[mask],
                    pooled_baseline[mask],
                    pooled_weights[mask],
                )
            )
            candidate_fold_loss = (
                phase33g_weighted_log_loss(
                    pooled_labels[mask],
                    probability[mask],
                    pooled_weights[mask],
                )
            )

            fold_records.append({
                "fold": fold,
                "baseline_log_loss": (
                    baseline_fold_loss
                ),
                "candidate_log_loss": (
                    candidate_fold_loss
                ),
                "gain": (
                    baseline_fold_loss
                    - candidate_fold_loss
                ),
            })

        fold_win_count = sum(
            record["gain"] >= 0.0005
            for record in fold_records
        )
        worst_fold_regret = max(
            -record["gain"]
            for record in fold_records
        )

        (
            maximum_group_harm,
            group_records,
        ) = phase33g_monitor_group_harm(
            pooled_labels,
            pooled_baseline,
            probability,
            pooled_groups,
            pooled_weights,
            minimum_group_n=8,
        )

        eligible = bool(
            float(scale) > 0.0
            and log_loss_gain
            >= PHASE36_TRAIN_CONFIG[
                "minimum_monitor_log_loss_gain"
            ]
            and fold_win_count
            >= PHASE36_TRAIN_CONFIG[
                "minimum_monitor_fold_wins"
            ]
            and worst_fold_regret
            <= PHASE36_TRAIN_CONFIG[
                "maximum_monitor_fold_regret"
            ]
            and maximum_group_harm
            <= PHASE36_TRAIN_CONFIG[
                "maximum_monitor_group_harm"
            ]
        )

        phase36_joint_candidate_records.append({
            "specification_index": int(
                specification_index
            ),
            "specification": dict(
                specification
            ),
            "scale": float(scale),
            "monitor_log_loss": float(
                candidate_log_loss
            ),
            "monitor_auroc": float(
                candidate_auroc
            ),
            "log_loss_gain": float(
                log_loss_gain
            ),
            "auroc_gain": float(
                auroc_gain
            ),
            "fold_records": fold_records,
            "fold_win_count": int(
                fold_win_count
            ),
            "worst_fold_regret": float(
                worst_fold_regret
            ),
            "maximum_group_harm": float(
                maximum_group_harm
            ),
            "group_records": group_records,
            "eligible": eligible,
        })

phase36_raw_best = min(
    phase36_joint_candidate_records,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["specification"][
            "maximum_depth"
        ],
        record["scale"],
    ),
)

phase36_eligible_records = [
    record
    for record in phase36_joint_candidate_records
    if record["eligible"]
]

if phase36_eligible_records:
    minimum_eligible_loss = min(
        record["monitor_log_loss"]
        for record in (
            phase36_eligible_records
        )
    )

    phase36_loss_tied_records = [
        record
        for record in phase36_eligible_records
        if (
            record["monitor_log_loss"]
            <= minimum_eligible_loss
            + PHASE36_TRAIN_CONFIG[
                "log_loss_tie_tolerance"
            ]
        )
    ]

    phase36_selected_record = min(
        phase36_loss_tied_records,
        key=lambda record: (
            -record["monitor_auroc"],
            record["monitor_log_loss"],
            record["specification"][
                "maximum_depth"
            ],
            record["specification"][
                "minimum_child_weight"
            ],
            -record["specification"][
                "l2_regularization"
            ],
            record["scale"],
        ),
    )
    phase36_advanced = True
else:
    phase36_selected_record = next(
        record
        for record in (
            phase36_joint_candidate_records
        )
        if (
            record["scale"] == 0.0
            and record[
                "specification_index"
            ] == 0
        )
    )
    phase36_advanced = False

PHASE36_SELECTED_SPECIFICATION_PRIVATE = dict(
    phase36_selected_record[
        "specification"
    ]
)
PHASE36_SELECTED_SPECIFICATION_INDEX_PRIVATE = int(
    phase36_selected_record[
        "specification_index"
    ]
)
PHASE36_SHARED_SCALE_PRIVATE = float(
    phase36_selected_record["scale"]
)

print(
    "Phase36 selected: "
    f"depth={PHASE36_SELECTED_SPECIFICATION_PRIVATE['maximum_depth']}, "
    "min_child="
    f"{PHASE36_SELECTED_SPECIFICATION_PRIVATE['minimum_child_weight']:.0f}, "
    "lambda="
    f"{PHASE36_SELECTED_SPECIFICATION_PRIVATE['l2_regularization']:.0f}, "
    f"scale={PHASE36_SHARED_SCALE_PRIVATE:.2f}, "
    f"monitor_gain={phase36_selected_record['log_loss_gain']:.6f}, "
    f"monitor_auroc_gain={phase36_selected_record['auroc_gain']:.6f}, "
    f"advanced={phase36_advanced}"
)

# ---------------------------------------------------------
# Refit the selected shared configuration on outer_train.
# ---------------------------------------------------------

PHASE36_OOF_PRIVATE = (
    phase36_baseline_oof.copy()
)
PHASE36_RAW_RESIDUAL_OOF_PRIVATE = np.zeros(
    1362,
    dtype=np.float64,
)
PHASE36_DEPLOYMENT_STATES_PRIVATE = []
phase36_final_fold_records = []

if phase36_advanced:
    for partition, selection_fold in zip(
        PHASE19_PARTITIONS,
        phase36_selection_folds,
    ):
        refit_started = time.perf_counter()

        fold = int(partition["fold"])
        assert fold == selection_fold["fold"]

        outer_train_indices = np.asarray(
            partition["outer_train"],
            dtype=np.int64,
        )
        outer_valid_indices = np.asarray(
            partition["outer_valid"],
            dtype=np.int64,
        )

        preprocessor = (
            phase33_fit_preprocessor(
                1,
                fold,
                outer_train_indices,
                368000 + fold,
            )
        )

        train_transformed = phase33_transform(
            preprocessor,
            outer_train_indices,
        )
        valid_transformed = phase33_transform(
            preprocessor,
            outer_valid_indices,
        )

        (
            train_features,
            train_margin,
        ) = phase36_build_features(
            train_transformed,
            phase36_baseline_oof[
                outer_train_indices
            ],
        )
        (
            valid_features,
            valid_margin,
        ) = phase36_build_features(
            valid_transformed,
            phase36_baseline_oof[
                outer_valid_indices
            ],
        )

        selected_fit_result = (
            selection_fold[
                "specification_results"
            ][
                PHASE36_SELECTED_SPECIFICATION_INDEX_PRIVATE
            ]
        )

        booster_records = []
        residual_parts = []

        for seed_result, seed in zip(
            selected_fit_result[
                "seed_results"
            ],
            PHASE36_TRAIN_CONFIG["seeds"],
        ):
            round_count = int(
                seed_result[
                    "best_round_count"
                ]
            )

            booster = (
                phase36_train_fixed_rounds(
                    train_features=(
                        train_features
                    ),
                    train_labels=phase36_labels[
                        outer_train_indices
                    ],
                    train_groups=phase36_groups[
                        outer_train_indices
                    ],
                    train_margin=train_margin,
                    specification=(
                        PHASE36_SELECTED_SPECIFICATION_PRIVATE
                    ),
                    seed=seed,
                    round_count=round_count,
                )
            )

            residual, _ = (
                phase36_predict_residual(
                    booster,
                    valid_features,
                    valid_margin,
                    round_count=round_count,
                )
            )
            residual_parts.append(residual)

            booster_records.append({
                "seed": int(seed),
                "round_count": (
                    round_count
                ),
                "booster": booster,
            })

        ensemble_residual = np.mean(
            np.stack(residual_parts),
            axis=0,
        )

        probability = phase33d_sigmoid(
            valid_margin
            + PHASE36_SHARED_SCALE_PRIVATE
            * ensemble_residual
        )

        PHASE36_RAW_RESIDUAL_OOF_PRIVATE[
            outer_valid_indices
        ] = ensemble_residual
        PHASE36_OOF_PRIVATE[
            outer_valid_indices
        ] = probability

        baseline_loss = float(
            log_loss(
                phase36_labels[
                    outer_valid_indices
                ],
                phase36_baseline_oof[
                    outer_valid_indices
                ],
            )
        )
        candidate_loss = float(
            log_loss(
                phase36_labels[
                    outer_valid_indices
                ],
                probability,
            )
        )
        baseline_auroc = float(
            roc_auc_score(
                phase36_labels[
                    outer_valid_indices
                ],
                phase36_baseline_oof[
                    outer_valid_indices
                ],
            )
        )
        candidate_auroc = float(
            roc_auc_score(
                phase36_labels[
                    outer_valid_indices
                ],
                probability,
            )
        )

        phase36_final_fold_records.append({
            "fold": fold,
            "n": int(
                outer_valid_indices.size
            ),
            "round_counts": [
                int(
                    record["round_count"]
                )
                for record in (
                    booster_records
                )
            ],
            "baseline_log_loss": round(
                baseline_loss,
                6,
            ),
            "phase36_log_loss": round(
                candidate_loss,
                6,
            ),
            "log_loss_improvement": round(
                baseline_loss
                - candidate_loss,
                6,
            ),
            "baseline_auroc": round(
                baseline_auroc,
                6,
            ),
            "phase36_auroc": round(
                candidate_auroc,
                6,
            ),
            "auroc_improvement": round(
                candidate_auroc
                - baseline_auroc,
                6,
            ),
            "mean_absolute_residual": round(
                float(
                    np.mean(
                        np.abs(
                            ensemble_residual
                        )
                    )
                ),
                6,
            ),
            "refit_seconds": round(
                time.perf_counter()
                - refit_started,
                2,
            ),
        })

        PHASE36_DEPLOYMENT_STATES_PRIVATE.append({
            "fold": fold,
            "preprocessor": preprocessor,
            "specification": dict(
                PHASE36_SELECTED_SPECIFICATION_PRIVATE
            ),
            "scale": (
                PHASE36_SHARED_SCALE_PRIVATE
            ),
            "boosters": booster_records,
        })

        del train_transformed
        del valid_transformed
        del train_features
        del valid_features

        gc.collect()

else:
    for partition in PHASE19_PARTITIONS:
        fold = int(partition["fold"])
        indices = np.asarray(
            partition["outer_valid"],
            dtype=np.int64,
        )

        fold_loss = float(
            log_loss(
                phase36_labels[indices],
                phase36_baseline_oof[
                    indices
                ],
            )
        )
        fold_auroc = float(
            roc_auc_score(
                phase36_labels[indices],
                phase36_baseline_oof[
                    indices
                ],
            )
        )

        phase36_final_fold_records.append({
            "fold": fold,
            "n": int(indices.size),
            "round_counts": [0, 0],
            "baseline_log_loss": round(
                fold_loss, 6
            ),
            "phase36_log_loss": round(
                fold_loss, 6
            ),
            "log_loss_improvement": 0.0,
            "baseline_auroc": round(
                fold_auroc, 6
            ),
            "phase36_auroc": round(
                fold_auroc, 6
            ),
            "auroc_improvement": 0.0,
            "mean_absolute_residual": 0.0,
            "refit_seconds": 0.0,
        })

baseline_log_loss = float(
    log_loss(
        phase36_labels,
        phase36_baseline_oof,
    )
)
phase36_log_loss = float(
    log_loss(
        phase36_labels,
        PHASE36_OOF_PRIVATE,
    )
)
baseline_auroc = float(
    roc_auc_score(
        phase36_labels,
        phase36_baseline_oof,
    )
)
phase36_auroc = float(
    roc_auc_score(
        phase36_labels,
        PHASE36_OOF_PRIVATE,
    )
)

phase36_log_loss_gain = (
    baseline_log_loss
    - phase36_log_loss
)
phase36_auroc_gain = (
    phase36_auroc
    - baseline_auroc
)

phase36_fold_wins = sum(
    record["log_loss_improvement"] > 1e-8
    for record in phase36_final_fold_records
)
phase36_worst_fold_excess = max(
    -record["log_loss_improvement"]
    for record in phase36_final_fold_records
)

(
    phase36_maximum_group_harm,
    phase36_group_records,
) = phase33d_group_harm(
    phase36_labels,
    phase36_baseline_oof,
    PHASE36_OOF_PRIVATE,
    phase36_groups,
)

phase36_component_gate_passed = bool(
    phase36_advanced
    and phase36_log_loss_gain >= 0.003
    and phase36_auroc_gain >= 0.001
    and phase36_fold_wins >= 2
    and phase36_worst_fold_excess <= 0.005
    and phase36_maximum_group_harm <= 0.015
)

phase36_beats_phase33 = bool(
    phase36_log_loss
    <= phase33_final_gated_metrics[
        "log_loss"
    ] - 0.001
)

phase36_status = (
    "new_champion"
    if (
        phase36_component_gate_passed
        and phase36_beats_phase33
    )
    else (
        "component_promoted"
        if phase36_component_gate_passed
        else "not_promoted"
    )
)

phase36_report = {
    "phase": (
        "phase36_phase12c_base_margin_"
        "boosted_residual"
    ),
    "status": phase36_status,
    "selected": {
        "specification": dict(
            PHASE36_SELECTED_SPECIFICATION_PRIVATE
        ),
        "shared_scale": (
            PHASE36_SHARED_SCALE_PRIVATE
        ),
        "advanced": phase36_advanced,
        "joint_candidate_count": len(
            phase36_joint_candidate_records
        ),
        "eligible_candidate_count": len(
            phase36_eligible_records
        ),
        "monitor_log_loss": round(
            phase36_selected_record[
                "monitor_log_loss"
            ],
            6,
        ),
        "monitor_log_loss_gain": round(
            phase36_selected_record[
                "log_loss_gain"
            ],
            6,
        ),
        "monitor_auroc": round(
            phase36_selected_record[
                "monitor_auroc"
            ],
            6,
        ),
        "monitor_auroc_gain": round(
            phase36_selected_record[
                "auroc_gain"
            ],
            6,
        ),
        "monitor_fold_wins": int(
            phase36_selected_record[
                "fold_win_count"
            ]
        ),
        "worst_monitor_fold_regret": round(
            phase36_selected_record[
                "worst_fold_regret"
            ],
            6,
        ),
        "maximum_monitor_group_harm": round(
            phase36_selected_record[
                "maximum_group_harm"
            ],
            6,
        ),
    },
    "baseline": {
        "log_loss": round(
            baseline_log_loss, 6
        ),
        "auroc": round(
            baseline_auroc, 6
        ),
    },
    "phase36": {
        "log_loss": round(
            phase36_log_loss, 6
        ),
        "auroc": round(
            phase36_auroc, 6
        ),
    },
    "improvements": {
        "log_loss_gain": round(
            phase36_log_loss_gain, 6
        ),
        "auroc_gain": round(
            phase36_auroc_gain, 6
        ),
        "fold_wins": int(
            phase36_fold_wins
        ),
        "worst_fold_excess": round(
            phase36_worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            phase36_maximum_group_harm,
            6,
        ),
        "log_loss_gain_over_phase33": round(
            phase33_final_gated_metrics[
                "log_loss"
            ]
            - phase36_log_loss,
            6,
        ),
    },
    "fold_metrics": (
        phase36_final_fold_records
    ),
    "major_acquisition_group_metrics": (
        phase36_group_records
    ),
    "gates": {
        "component_gate_passed": (
            phase36_component_gate_passed
        ),
        "beats_phase33_by_0p001": (
            phase36_beats_phase33
        ),
    },
    "primary_metric": "log_loss",
    "secondary_metric": "auroc",
    "shared_specification_across_folds": True,
    "shared_scale_across_folds": True,
    "fold_specific_deployment_routing": False,
    "outer_oracle_parameters_used": False,
    "monitor_labels_used_for_selection": True,
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "deployment_states_retained_privately": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase36_training_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE36_TRAINING")
print(json.dumps(
    phase36_report,
    indent=2,
))
print("END SANITIZED_PHASE36_TRAINING")

Phase36 fold 0: 1/12 specifications complete
Phase36 fold 0: 2/12 specifications complete
Phase36 fold 0: 3/12 specifications complete
Phase36 fold 0: 4/12 specifications complete
Phase36 fold 0: 5/12 specifications complete
Phase36 fold 0: 6/12 specifications complete
Phase36 fold 0: 7/12 specifications complete
Phase36 fold 0: 8/12 specifications complete
Phase36 fold 0: 9/12 specifications complete
Phase36 fold 0: 10/12 specifications complete
Phase36 fold 0: 11/12 specifications complete
Phase36 fold 0: 12/12 specifications complete
Phase36 fold 1: 1/12 specifications complete
Phase36 fold 1: 2/12 specifications complete
Phase36 fold 1: 3/12 specifications complete
Phase36 fold 1: 4/12 specifications complete
Phase36 fold 1: 5/12 specifications complete
Phase36 fold 1: 6/12 specifications complete
Phase36 fold 1: 7/12 specifications complete
Phase36 fold 1: 8/12 specifications complete
Phase36 fold 1: 9/12 specifications complete
Phase36 fold 1: 10/12 specifications complete
Phase3

In [66]:
# Phase37 Cell 119A
# Audit the retained Phase36 state needed for a bounded,
# group-robust residual gate. No models are retrained.

import inspect
import json
import numpy as np

PHASE37_EXPECTED_CASES = 1362
PHASE37_MONITOR_LENGTHS = {123, 174, 196, 493}
phase37_globals = globals()


def phase37_array_contract(value):
    if isinstance(value, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }

    if hasattr(value, "detach") and hasattr(value, "shape"):
        return {
            "type": type(value).__name__,
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }

    return None


def phase37_walk_candidate_arrays(
    value,
    path,
    records,
    depth=0,
    maximum_depth=7,
):
    if depth > maximum_depth:
        return

    contract = phase37_array_contract(value)

    if contract is not None:
        shape = tuple(contract["shape"])

        relevant = (
            len(shape) == 1
            and (
                shape[0] in PHASE37_MONITOR_LENGTHS
                or shape[0] == PHASE37_EXPECTED_CASES
            )
        )

        if relevant:
            records.append({
                "path": path,
                **contract,
            })

        return

    if isinstance(value, dict):
        for key, child in value.items():
            phase37_walk_candidate_arrays(
                child,
                f"{path}.{key}",
                records,
                depth + 1,
                maximum_depth,
            )
        return

    if isinstance(value, (list, tuple)):
        for index, child in enumerate(value):
            phase37_walk_candidate_arrays(
                child,
                f"{path}[{index}]",
                records,
                depth + 1,
                maximum_depth,
            )


important_tokens = (
    "selection",
    "residual",
    "prediction",
    "oof",
    "baseline",
    "deployment",
    "selected",
)

phase37_candidate_names = sorted(
    name
    for name in phase37_globals
    if (
        name.startswith("phase36_")
        or name.startswith("PHASE36_")
    )
    and any(
        token in name.lower()
        for token in important_tokens
    )
)

phase37_global_contracts = {}

for name in phase37_candidate_names:
    value = phase37_globals[name]
    array_contract = phase37_array_contract(value)

    if array_contract is not None:
        phase37_global_contracts[name] = array_contract

    elif isinstance(value, dict):
        phase37_global_contracts[name] = {
            "type": "dict",
            "length": len(value),
            "keys": sorted(
                str(key) for key in value.keys()
            ),
        }

    elif isinstance(value, (list, tuple)):
        phase37_global_contracts[name] = {
            "type": type(value).__name__,
            "length": len(value),
            "element_types": sorted({
                type(element).__name__
                for element in value
            }),
        }

    elif isinstance(value, (int, float, str, bool)):
        phase37_global_contracts[name] = {
            "type": type(value).__name__,
            "value": value,
        }

    else:
        phase37_global_contracts[name] = {
            "type": type(value).__name__,
        }


phase37_nested_arrays = []

for root_name in (
    "phase36_selection_folds",
    "PHASE36_SELECTION_FOLDS_PRIVATE",
    "PHASE36_DEPLOYMENT_STATES_PRIVATE",
    "PHASE36_MONITOR_PREDICTIONS_PRIVATE",
    "PHASE36_MONITOR_RESIDUALS_PRIVATE",
):
    if root_name in phase37_globals:
        phase37_walk_candidate_arrays(
            phase37_globals[root_name],
            root_name,
            phase37_nested_arrays,
        )


phase37_callable_contracts = {}

for name in sorted(phase37_globals):
    if (
        name.startswith("phase36_")
        and callable(phase37_globals[name])
    ):
        try:
            signature = str(
                inspect.signature(
                    phase37_globals[name]
                )
            )
        except Exception:
            signature = "unavailable"

        if any(
            token in name.lower()
            for token in (
                "predict",
                "train",
                "feature",
                "preprocess",
                "metric",
                "residual",
            )
        ):
            phase37_callable_contracts[name] = (
                signature
            )


required_state = {
    "baseline_oof": any(
        name in phase37_globals
        for name in (
            "phase36_baseline_oof",
            "PHASE19_PHASE12C_OOF_PRIVATE",
            "PHASE31_BASELINE_OOF_PRIVATE",
        )
    ),
    "raw_residual_oof": (
        "PHASE36_RAW_RESIDUAL_OOF_PRIVATE"
        in phase37_globals
    ),
    "phase36_oof": (
        "PHASE36_OOF_PRIVATE"
        in phase37_globals
    ),
    "selection_folds": any(
        name in phase37_globals
        for name in (
            "phase36_selection_folds",
            "PHASE36_SELECTION_FOLDS_PRIVATE",
        )
    ),
    "selected_specification": (
        "PHASE36_SELECTED_SPECIFICATION_PRIVATE"
        in phase37_globals
    ),
    "deployment_states": (
        "PHASE36_DEPLOYMENT_STATES_PRIVATE"
        in phase37_globals
    ),
}

report = {
    "phase": "phase37_phase36_state_audit",
    "status": (
        "accepted"
        if (
            required_state["baseline_oof"]
            and required_state["raw_residual_oof"]
            and required_state["selection_folds"]
        )
        else "requires_state_resolution"
    ),
    "required_state": required_state,
    "global_contracts": phase37_global_contracts,
    "nested_prediction_array_contracts": (
        phase37_nested_arrays
    ),
    "callable_contracts": (
        phase37_callable_contracts
    ),
    "training_labels_read": False,
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_values_displayed": False,
}

print("BEGIN SANITIZED_PHASE37_STATE_AUDIT")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE37_STATE_AUDIT")

assert report["status"] == "accepted", report

BEGIN SANITIZED_PHASE37_STATE_AUDIT
{
  "phase": "phase37_phase36_state_audit",
  "status": "accepted",
  "required_state": {
    "baseline_oof": true,
    "raw_residual_oof": true,
    "phase36_oof": true,
    "selection_folds": true,
    "selected_specification": true,
    "deployment_states": true
  },
  "global_contracts": {
    "PHASE36_DEPLOYMENT_STATES_PRIVATE": {
      "type": "list",
      "length": 3,
      "element_types": [
        "dict"
      ]
    },
    "PHASE36_OOF_PRIVATE": {
      "type": "ndarray",
      "shape": [
        1362
      ],
      "dtype": "float64"
    },
    "PHASE36_RAW_RESIDUAL_OOF_PRIVATE": {
      "type": "ndarray",
      "shape": [
        1362
      ],
      "dtype": "float64"
    },
    "PHASE36_SELECTED_SPECIFICATION_INDEX_PRIVATE": {
      "type": "int",
      "value": 4
    },
    "PHASE36_SELECTED_SPECIFICATION_PRIVATE": {
      "type": "dict",
      "length": 3,
      "keys": [
        "l2_regularization",
        "maximum_depth",
        "

In [67]:
# Phase37 Cell 119B
# Select one shared, confidence-weighted, bounded Phase36 residual gate.
# Selection uses pooled monitor partitions only.
# Do not evaluate outer OOF in this cell.

import json
import time

import numpy as np
from sklearn.metrics import roc_auc_score


PHASE37_CONFIG = {
    "phase": "phase37_group_robust_bounded_phase36",
    "random_seed": 370601,

    # alpha replaces Phase36's original 0.75 scale.
    "candidate_alphas": [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    "candidate_uncertainty_exponents": [
        0.0,
        0.5,
        1.0,
    ],
    "candidate_delta_caps": [
        0.50,
        0.75,
        1.00,
        1.50,
        2.00,
    ],

    # Group-aware hierarchical bootstrap.
    "bootstrap_replicates": 5000,
    "dirichlet_concentration": 20.0,
    "dirichlet_floor": 0.2,
    "bootstrap_lower_quantile": 0.05,

    # Conservative monitor-only eligibility.
    "minimum_monitor_log_loss_gain": 0.005,
    "minimum_bootstrap_probability_positive": 0.95,
    "minimum_bootstrap_lcb_gain": -0.0005,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.003,
    "maximum_monitor_group_harm": 0.0075,
    "minimum_leave_one_group_out_gain": -0.001,

    # AUROC only breaks near-ties in log loss.
    "log_loss_near_tie_tolerance": 0.00025,
    "minimum_monitor_group_n": 8,
    "probability_clip": 1e-6,
}


def phase37_sigmoid(logit):
    logit = np.asarray(
        logit,
        dtype=np.float64,
    )

    output = np.empty_like(logit)
    positive = logit >= 0

    output[positive] = (
        1.0
        / (
            1.0
            + np.exp(-logit[positive])
        )
    )

    negative_exp = np.exp(logit[~positive])

    output[~positive] = (
        negative_exp
        / (1.0 + negative_exp)
    )

    return output


def phase37_logit(probability):
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        PHASE37_CONFIG["probability_clip"],
        1.0
        - PHASE37_CONFIG[
            "probability_clip"
        ],
    )

    return np.log(
        probability
        / (1.0 - probability)
    )


def phase37_case_log_loss(
    labels,
    probabilities,
):
    labels = np.asarray(
        labels,
        dtype=np.float64,
    )

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        PHASE37_CONFIG["probability_clip"],
        1.0
        - PHASE37_CONFIG[
            "probability_clip"
        ],
    )

    return -(
        labels * np.log(probabilities)
        + (
            1.0 - labels
        ) * np.log(
            1.0 - probabilities
        )
    )


def phase37_weighted_mean(
    values,
    weights,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )
    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    return float(
        np.sum(values * weights)
        / np.sum(weights)
    )


def phase37_weighted_log_loss(
    labels,
    probabilities,
    weights,
):
    return phase37_weighted_mean(
        phase37_case_log_loss(
            labels,
            probabilities,
        ),
        weights,
    )


def phase37_weighted_auroc(
    labels,
    probabilities,
    weights,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    if np.unique(labels).size < 2:
        return float("nan")

    return float(
        roc_auc_score(
            labels,
            probabilities,
            sample_weight=weights,
        )
    )


def phase37_apply_gate(
    baseline_probability,
    raw_residual,
    specification,
):
    baseline_probability = np.clip(
        np.asarray(
            baseline_probability,
            dtype=np.float64,
        ),
        PHASE37_CONFIG["probability_clip"],
        1.0
        - PHASE37_CONFIG[
            "probability_clip"
        ],
    )

    raw_residual = np.asarray(
        raw_residual,
        dtype=np.float64,
    )

    alpha = float(
        specification["alpha"]
    )
    gamma = float(
        specification[
            "uncertainty_exponent"
        ]
    )
    cap = float(
        specification["delta_cap"]
    )

    if alpha == 0.0:
        return baseline_probability.copy()

    uncertainty = (
        4.0
        * baseline_probability
        * (1.0 - baseline_probability)
    ) ** gamma

    bounded_residual = np.clip(
        raw_residual,
        -cap,
        cap,
    )

    updated_logit = (
        phase37_logit(
            baseline_probability
        )
        + alpha
        * uncertainty
        * bounded_residual
    )

    return phase37_sigmoid(
        updated_logit
    )


def phase37_resolve_labels_and_groups():
    assert "case_df" in globals(), {
        "message": "case_df is unavailable."
    }

    dataframe = globals()["case_df"]

    label_column = next(
        (
            column
            for column in (
                "label",
                "is_pathologic",
            )
            if column in dataframe.columns
        ),
        None,
    )

    group_column = next(
        (
            column
            for column in (
                "acquisition_group",
                "group",
            )
            if column in dataframe.columns
        ),
        None,
    )

    assert label_column is not None
    assert group_column is not None

    labels = np.asarray(
        dataframe[label_column],
        dtype=np.int64,
    )
    groups = np.asarray(
        dataframe[group_column],
        dtype=np.int64,
    )

    assert labels.shape == (1362,)
    assert groups.shape == (1362,)
    assert set(np.unique(labels)).issubset(
        {0, 1}
    )

    return (
        labels,
        groups,
        label_column,
        group_column,
    )


phase37_started = time.perf_counter()

(
    phase37_labels,
    phase37_groups,
    phase37_label_source,
    phase37_group_source,
) = phase37_resolve_labels_and_groups()

phase37_selected_specification_index = int(
    PHASE36_SELECTED_SPECIFICATION_INDEX_PRIVATE
)

assert (
    phase37_selected_specification_index
    == 4
), {
    "message": (
        "Unexpected Phase36 specification index."
    ),
    "observed": (
        phase37_selected_specification_index
    ),
}

phase37_monitor_indices_parts = []
phase37_monitor_fold_parts = []
phase37_monitor_baseline_parts = []
phase37_monitor_residual_parts = []
phase37_monitor_label_parts = []
phase37_monitor_group_parts = []

for fallback_fold, fold_record in enumerate(
    phase36_selection_folds
):
    fold = int(
        fold_record.get(
            "fold",
            fallback_fold,
        )
    )

    monitor_indices = np.asarray(
        fold_record["monitor_indices"],
        dtype=np.int64,
    )

    specification_result = (
        fold_record[
            "specification_results"
        ][
            phase37_selected_specification_index
        ]
    )

    monitor_residual = np.asarray(
        specification_result[
            "ensemble_residual"
        ],
        dtype=np.float64,
    )

    assert monitor_residual.shape == (
        monitor_indices.size,
    )

    phase37_monitor_indices_parts.append(
        monitor_indices
    )
    phase37_monitor_fold_parts.append(
        np.full(
            monitor_indices.size,
            fold,
            dtype=np.int64,
        )
    )
    phase37_monitor_baseline_parts.append(
        np.asarray(
            phase36_baseline_oof[
                monitor_indices
            ],
            dtype=np.float64,
        )
    )
    phase37_monitor_residual_parts.append(
        monitor_residual
    )
    phase37_monitor_label_parts.append(
        phase37_labels[monitor_indices]
    )
    phase37_monitor_group_parts.append(
        phase37_groups[monitor_indices]
    )


phase37_monitor_indices = np.concatenate(
    phase37_monitor_indices_parts
)
phase37_monitor_folds = np.concatenate(
    phase37_monitor_fold_parts
)
phase37_monitor_baseline = np.concatenate(
    phase37_monitor_baseline_parts
)
phase37_monitor_raw_residual = np.concatenate(
    phase37_monitor_residual_parts
)
phase37_monitor_labels = np.concatenate(
    phase37_monitor_label_parts
)
phase37_monitor_groups = np.concatenate(
    phase37_monitor_group_parts
)

assert phase37_monitor_indices.shape == (493,)
assert np.isfinite(
    phase37_monitor_raw_residual
).all()


# Correct duplicated cases appearing in two monitor sets.
phase37_case_multiplicity = np.bincount(
    phase37_monitor_indices,
    minlength=1362,
)

phase37_monitor_weights = (
    1.0
    / phase37_case_multiplicity[
        phase37_monitor_indices
    ]
).astype(np.float64)

phase37_unique_monitor_cases = int(
    np.unique(
        phase37_monitor_indices
    ).size
)

assert phase37_unique_monitor_cases == 407


# Confirm exact reconstruction of the original Phase36
# monitor candidate before searching for a safer gate.
phase37_original_scale = float(
    phase36_selected_record["scale"]
)

phase37_original_monitor_probability = (
    phase37_sigmoid(
        phase37_logit(
            phase37_monitor_baseline
        )
        + phase37_original_scale
        * phase37_monitor_raw_residual
    )
)

phase37_original_monitor_log_loss = (
    phase37_weighted_log_loss(
        phase37_monitor_labels,
        phase37_original_monitor_probability,
        phase37_monitor_weights,
    )
)

phase37_stored_monitor_log_loss = float(
    phase36_selected_record[
        "monitor_log_loss"
    ]
)

phase37_monitor_reconstruction_error = abs(
    phase37_original_monitor_log_loss
    - phase37_stored_monitor_log_loss
)

assert (
    phase37_monitor_reconstruction_error
    <= 5e-6
), {
    "message": (
        "Phase36 monitor residual reconstruction "
        "did not match the stored result."
    ),
    "reconstructed": (
        phase37_original_monitor_log_loss
    ),
    "stored": (
        phase37_stored_monitor_log_loss
    ),
    "absolute_error": (
        phase37_monitor_reconstruction_error
    ),
}


# Confirm that the retained Phase36 OOF vector was built
# from the raw residual using the stored 0.75 scale.
phase37_phase36_oof_reconstructed = (
    phase37_sigmoid(
        phase37_logit(
            phase36_baseline_oof
        )
        + phase37_original_scale
        * np.asarray(
            PHASE36_RAW_RESIDUAL_OOF_PRIVATE,
            dtype=np.float64,
        )
    )
)

phase37_phase36_oof_reconstruction_error = float(
    np.max(
        np.abs(
            phase37_phase36_oof_reconstructed
            - np.asarray(
                PHASE36_OOF_PRIVATE,
                dtype=np.float64,
            )
        )
    )
)

assert (
    phase37_phase36_oof_reconstruction_error
    <= 1e-8
), {
    "message": (
        "Phase36 OOF residual reconstruction failed."
    ),
    "maximum_error": (
        phase37_phase36_oof_reconstruction_error
    ),
}


# Candidate grid. Alpha zero is represented once.
phase37_candidate_specifications = [{
    "alpha": 0.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 0.0,
}]

for alpha in PHASE37_CONFIG[
    "candidate_alphas"
]:
    if alpha == 0.0:
        continue

    for gamma in PHASE37_CONFIG[
        "candidate_uncertainty_exponents"
    ]:
        for cap in PHASE37_CONFIG[
            "candidate_delta_caps"
        ]:
            phase37_candidate_specifications.append({
                "alpha": float(alpha),
                "uncertainty_exponent": float(
                    gamma
                ),
                "delta_cap": float(cap),
            })


phase37_baseline_monitor_log_loss = (
    phase37_weighted_log_loss(
        phase37_monitor_labels,
        phase37_monitor_baseline,
        phase37_monitor_weights,
    )
)

phase37_baseline_monitor_auroc = (
    phase37_weighted_auroc(
        phase37_monitor_labels,
        phase37_monitor_baseline,
        phase37_monitor_weights,
    )
)

phase37_candidate_probability_matrix = (
    np.column_stack([
        phase37_apply_gate(
            phase37_monitor_baseline,
            phase37_monitor_raw_residual,
            specification,
        )
        for specification
        in phase37_candidate_specifications
    ])
)

phase37_baseline_case_loss = (
    phase37_case_log_loss(
        phase37_monitor_labels,
        phase37_monitor_baseline,
    )
)

phase37_candidate_case_loss = -(
    phase37_monitor_labels[:, None]
    * np.log(
        np.clip(
            phase37_candidate_probability_matrix,
            PHASE37_CONFIG[
                "probability_clip"
            ],
            1.0,
        )
    )
    + (
        1.0
        - phase37_monitor_labels[:, None]
    )
    * np.log(
        np.clip(
            1.0
            - phase37_candidate_probability_matrix,
            PHASE37_CONFIG[
                "probability_clip"
            ],
            1.0,
        )
    )
)

phase37_case_gain_matrix = (
    phase37_baseline_case_loss[:, None]
    - phase37_candidate_case_loss
)


# Hierarchical bootstrap:
# 1. shift acquisition-group mixture;
# 2. resample observations within each group.
phase37_rng = np.random.default_rng(
    PHASE37_CONFIG["random_seed"]
)

phase37_unique_groups = np.unique(
    phase37_monitor_groups
)

phase37_group_positions = [
    np.flatnonzero(
        phase37_monitor_groups == group
    )
    for group in phase37_unique_groups
]

phase37_empirical_group_mass = np.asarray([
    np.sum(
        phase37_monitor_weights[position]
    )
    for position in phase37_group_positions
], dtype=np.float64)

phase37_empirical_group_mass /= (
    phase37_empirical_group_mass.sum()
)

phase37_dirichlet_parameters = (
    PHASE37_CONFIG[
        "dirichlet_concentration"
    ]
    * phase37_empirical_group_mass
    + PHASE37_CONFIG[
        "dirichlet_floor"
    ]
)

phase37_bootstrap_replicates = int(
    PHASE37_CONFIG[
        "bootstrap_replicates"
    ]
)

phase37_bootstrap_weights = np.zeros(
    (
        phase37_bootstrap_replicates,
        phase37_monitor_indices.size,
    ),
    dtype=np.float64,
)

phase37_group_mass_draws = (
    phase37_rng.dirichlet(
        phase37_dirichlet_parameters,
        size=phase37_bootstrap_replicates,
    )
)

for group_position, observation_positions in enumerate(
    phase37_group_positions
):
    within_group_draws = (
        phase37_rng.exponential(
            scale=1.0,
            size=(
                phase37_bootstrap_replicates,
                observation_positions.size,
            ),
        )
    )

    within_group_draws *= (
        phase37_monitor_weights[
            observation_positions
        ][None, :]
    )

    within_group_draws /= np.maximum(
        within_group_draws.sum(
            axis=1,
            keepdims=True,
        ),
        1e-12,
    )

    phase37_bootstrap_weights[
        :,
        observation_positions,
    ] = (
        phase37_group_mass_draws[
            :,
            group_position,
        ][:, None]
        * within_group_draws
    )


phase37_bootstrap_gains = (
    phase37_bootstrap_weights
    @ phase37_case_gain_matrix
)

phase37_candidate_records = []

for candidate_index, specification in enumerate(
    phase37_candidate_specifications
):
    candidate_probability = (
        phase37_candidate_probability_matrix[
            :,
            candidate_index,
        ]
    )

    monitor_log_loss = (
        phase37_weighted_log_loss(
            phase37_monitor_labels,
            candidate_probability,
            phase37_monitor_weights,
        )
    )

    monitor_auroc = (
        phase37_weighted_auroc(
            phase37_monitor_labels,
            candidate_probability,
            phase37_monitor_weights,
        )
    )

    monitor_gain = (
        phase37_baseline_monitor_log_loss
        - monitor_log_loss
    )

    fold_records = []
    fold_wins = 0
    worst_fold_regret = 0.0

    for fold in sorted(
        np.unique(
            phase37_monitor_folds
        ).tolist()
    ):
        position = (
            phase37_monitor_folds == fold
        )

        baseline_fold_loss = (
            phase37_weighted_log_loss(
                phase37_monitor_labels[
                    position
                ],
                phase37_monitor_baseline[
                    position
                ],
                phase37_monitor_weights[
                    position
                ],
            )
        )

        candidate_fold_loss = (
            phase37_weighted_log_loss(
                phase37_monitor_labels[
                    position
                ],
                candidate_probability[
                    position
                ],
                phase37_monitor_weights[
                    position
                ],
            )
        )

        fold_gain = (
            baseline_fold_loss
            - candidate_fold_loss
        )

        if fold_gain > 0.0:
            fold_wins += 1

        worst_fold_regret = max(
            worst_fold_regret,
            -fold_gain,
        )

        fold_records.append({
            "fold": int(fold),
            "baseline_log_loss": float(
                baseline_fold_loss
            ),
            "candidate_log_loss": float(
                candidate_fold_loss
            ),
            "gain": float(fold_gain),
        })

    group_records = []
    maximum_group_harm = 0.0

    for group in phase37_unique_groups:
        position = (
            phase37_monitor_groups == group
        )

        unique_group_case_count = int(
            np.unique(
                phase37_monitor_indices[
                    position
                ]
            ).size
        )

        if (
            unique_group_case_count
            < PHASE37_CONFIG[
                "minimum_monitor_group_n"
            ]
        ):
            continue

        baseline_group_loss = (
            phase37_weighted_log_loss(
                phase37_monitor_labels[
                    position
                ],
                phase37_monitor_baseline[
                    position
                ],
                phase37_monitor_weights[
                    position
                ],
            )
        )

        candidate_group_loss = (
            phase37_weighted_log_loss(
                phase37_monitor_labels[
                    position
                ],
                candidate_probability[
                    position
                ],
                phase37_monitor_weights[
                    position
                ],
            )
        )

        group_harm = (
            candidate_group_loss
            - baseline_group_loss
        )

        maximum_group_harm = max(
            maximum_group_harm,
            group_harm,
        )

        group_records.append({
            "group": int(group),
            "unique_case_count": (
                unique_group_case_count
            ),
            "baseline_log_loss": float(
                baseline_group_loss
            ),
            "candidate_log_loss": float(
                candidate_group_loss
            ),
            "log_loss_change": float(
                group_harm
            ),
        })

    leave_one_group_out_gains = []

    for held_out_group in (
        phase37_unique_groups
    ):
        retained = (
            phase37_monitor_groups
            != held_out_group
        )

        retained_gain = phase37_weighted_mean(
            phase37_case_gain_matrix[
                retained,
                candidate_index,
            ],
            phase37_monitor_weights[
                retained
            ],
        )

        leave_one_group_out_gains.append(
            retained_gain
        )

    bootstrap_candidate_gains = (
        phase37_bootstrap_gains[
            :,
            candidate_index,
        ]
    )

    bootstrap_mean_gain = float(
        np.mean(
            bootstrap_candidate_gains
        )
    )
    bootstrap_lcb_gain = float(
        np.quantile(
            bootstrap_candidate_gains,
            PHASE37_CONFIG[
                "bootstrap_lower_quantile"
            ],
        )
    )
    bootstrap_probability_positive = float(
        np.mean(
            bootstrap_candidate_gains
            > 0.0
        )
    )

    minimum_leave_one_group_out_gain = (
        float(
            np.min(
                leave_one_group_out_gains
            )
        )
    )

    eligible = bool(
        specification["alpha"] > 0.0
        and monitor_gain
        >= PHASE37_CONFIG[
            "minimum_monitor_log_loss_gain"
        ]
        and bootstrap_probability_positive
        >= PHASE37_CONFIG[
            "minimum_bootstrap_probability_positive"
        ]
        and bootstrap_lcb_gain
        >= PHASE37_CONFIG[
            "minimum_bootstrap_lcb_gain"
        ]
        and fold_wins
        >= PHASE37_CONFIG[
            "minimum_monitor_fold_wins"
        ]
        and worst_fold_regret
        <= PHASE37_CONFIG[
            "maximum_monitor_fold_regret"
        ]
        and maximum_group_harm
        <= PHASE37_CONFIG[
            "maximum_monitor_group_harm"
        ]
        and minimum_leave_one_group_out_gain
        >= PHASE37_CONFIG[
            "minimum_leave_one_group_out_gain"
        ]
    )

    phase37_candidate_records.append({
        "candidate_index": int(
            candidate_index
        ),
        "specification": specification,
        "monitor_log_loss": float(
            monitor_log_loss
        ),
        "monitor_log_loss_gain": float(
            monitor_gain
        ),
        "monitor_auroc": float(
            monitor_auroc
        ),
        "monitor_auroc_gain": float(
            monitor_auroc
            - phase37_baseline_monitor_auroc
        ),
        "monitor_fold_wins": int(
            fold_wins
        ),
        "worst_monitor_fold_regret": float(
            worst_fold_regret
        ),
        "maximum_monitor_group_harm": float(
            maximum_group_harm
        ),
        "minimum_leave_one_group_out_gain": (
            minimum_leave_one_group_out_gain
        ),
        "median_leave_one_group_out_gain": float(
            np.median(
                leave_one_group_out_gains
            )
        ),
        "bootstrap_mean_gain": (
            bootstrap_mean_gain
        ),
        "bootstrap_lcb_gain": (
            bootstrap_lcb_gain
        ),
        "bootstrap_probability_positive": (
            bootstrap_probability_positive
        ),
        "eligible": eligible,
        "fold_records": fold_records,
        "group_records": group_records,
    })


phase37_nonzero_candidates = [
    record
    for record in phase37_candidate_records
    if (
        record["specification"]["alpha"]
        > 0.0
    )
]

phase37_raw_best_record = min(
    phase37_nonzero_candidates,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
    ),
)

phase37_lcb_best_record = max(
    phase37_nonzero_candidates,
    key=lambda record: (
        record["bootstrap_lcb_gain"],
        record["monitor_log_loss_gain"],
        record["monitor_auroc"],
    ),
)

phase37_eligible_records = [
    record
    for record in phase37_candidate_records
    if record["eligible"]
]

if phase37_eligible_records:
    best_monitor_log_loss = min(
        record["monitor_log_loss"]
        for record
        in phase37_eligible_records
    )

    near_best_records = [
        record
        for record
        in phase37_eligible_records
        if (
            record["monitor_log_loss"]
            <= best_monitor_log_loss
            + PHASE37_CONFIG[
                "log_loss_near_tie_tolerance"
            ]
        )
    ]

    phase37_selected_record = sorted(
        near_best_records,
        key=lambda record: (
            -record["monitor_auroc"],
            record[
                "maximum_monitor_group_harm"
            ],
            record["specification"]["alpha"],
            record[
                "specification"
            ]["delta_cap"],
        ),
    )[0]

    phase37_gate_advanced = True

else:
    phase37_selected_record = (
        phase37_candidate_records[0]
    )
    phase37_gate_advanced = False


PHASE37_GATE_PRIVATE = dict(
    phase37_selected_record[
        "specification"
    ]
)

PHASE37_MONITOR_SELECTION_PRIVATE = {
    "configuration": dict(
        PHASE37_CONFIG
    ),
    "selected_record": (
        phase37_selected_record
    ),
    "raw_best_record": (
        phase37_raw_best_record
    ),
    "lcb_best_record": (
        phase37_lcb_best_record
    ),
    "gate_advanced": bool(
        phase37_gate_advanced
    ),
}

# Retain only what the next evaluation/deployment
# cells require.
PHASE37_MONITOR_POOL_PRIVATE = {
    "indices": (
        phase37_monitor_indices.copy()
    ),
    "folds": (
        phase37_monitor_folds.copy()
    ),
    "baseline_probability": (
        phase37_monitor_baseline.copy()
    ),
    "raw_residual": (
        phase37_monitor_raw_residual.copy()
    ),
    "labels": (
        phase37_monitor_labels.copy()
    ),
    "groups": (
        phase37_monitor_groups.copy()
    ),
    "weights": (
        phase37_monitor_weights.copy()
    ),
}

phase37_report = {
    "phase": (
        "phase37_group_robust_bounded_"
        "phase36_monitor_selection"
    ),
    "status": (
        "gate_frozen"
        if phase37_gate_advanced
        else "no_gate_advanced"
    ),
    "monitor_pool": {
        "observation_count": int(
            phase37_monitor_indices.size
        ),
        "unique_case_count": int(
            phase37_unique_monitor_cases
        ),
        "maximum_case_multiplicity": int(
            np.max(
                phase37_case_multiplicity
            )
        ),
        "inverse_multiplicity_weighting": True,
        "acquisition_group_count": int(
            phase37_unique_groups.size
        ),
    },
    "phase36_reconstruction": {
        "selected_specification_index": int(
            phase37_selected_specification_index
        ),
        "original_shared_scale": float(
            phase37_original_scale
        ),
        "monitor_log_loss_error": float(
            phase37_monitor_reconstruction_error
        ),
        "oof_maximum_probability_error": (
            phase37_phase36_oof_reconstruction_error
        ),
    },
    "search": {
        "candidate_count": int(
            len(
                phase37_candidate_records
            )
        ),
        "eligible_candidate_count": int(
            len(
                phase37_eligible_records
            )
        ),
        "bootstrap_replicate_count": int(
            phase37_bootstrap_replicates
        ),
    },
    "baseline_monitor": {
        "log_loss": float(
            phase37_baseline_monitor_log_loss
        ),
        "auroc": float(
            phase37_baseline_monitor_auroc
        ),
    },
    "raw_phase36_monitor": {
        "shared_scale": float(
            phase37_original_scale
        ),
        "log_loss": float(
            phase37_original_monitor_log_loss
        ),
        "log_loss_gain": float(
            phase37_baseline_monitor_log_loss
            - phase37_original_monitor_log_loss
        ),
    },
    "raw_best_candidate": {
        key: value
        for key, value
        in phase37_raw_best_record.items()
        if key not in (
            "fold_records",
            "group_records",
        )
    },
    "lcb_best_candidate": {
        key: value
        for key, value
        in phase37_lcb_best_record.items()
        if key not in (
            "fold_records",
            "group_records",
        )
    },
    "selection": {
        "gate_advanced": bool(
            phase37_gate_advanced
        ),
        "selected_specification": dict(
            PHASE37_GATE_PRIVATE
        ),
        "monitor_log_loss": float(
            phase37_selected_record[
                "monitor_log_loss"
            ]
        ),
        "monitor_log_loss_gain": float(
            phase37_selected_record[
                "monitor_log_loss_gain"
            ]
        ),
        "monitor_auroc": float(
            phase37_selected_record[
                "monitor_auroc"
            ]
        ),
        "monitor_auroc_gain": float(
            phase37_selected_record[
                "monitor_auroc_gain"
            ]
        ),
        "monitor_fold_wins": int(
            phase37_selected_record[
                "monitor_fold_wins"
            ]
        ),
        "worst_monitor_fold_regret": float(
            phase37_selected_record[
                "worst_monitor_fold_regret"
            ]
        ),
        "maximum_monitor_group_harm": float(
            phase37_selected_record[
                "maximum_monitor_group_harm"
            ]
        ),
        "minimum_leave_one_group_out_gain": float(
            phase37_selected_record[
                "minimum_leave_one_group_out_gain"
            ]
        ),
        "bootstrap_mean_gain": float(
            phase37_selected_record[
                "bootstrap_mean_gain"
            ]
        ),
        "bootstrap_lcb_gain": float(
            phase37_selected_record[
                "bootstrap_lcb_gain"
            ]
        ),
        "bootstrap_probability_positive": float(
            phase37_selected_record[
                "bootstrap_probability_positive"
            ]
        ),
        "fold_monitor_metrics": (
            phase37_selected_record[
                "fold_records"
            ]
        ),
    },
    "selection_partition": (
        "pooled_monitor_only"
    ),
    "shared_gate_across_folds": True,
    "fold_specific_group_routing": False,
    "outer_validation_labels_used": False,
    "outer_oracle_parameters_used": False,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase37_started,
        2,
    ),
}

print(
    "Phase37 frozen gate: "
    f"alpha={PHASE37_GATE_PRIVATE['alpha']:.2f}, "
    "gamma="
    f"{PHASE37_GATE_PRIVATE['uncertainty_exponent']:.2f}, "
    f"cap={PHASE37_GATE_PRIVATE['delta_cap']:.2f}, "
    "advanced="
    f"{phase37_gate_advanced}"
)

print(
    "BEGIN SANITIZED_PHASE37_MONITOR_SELECTION"
)
print(
    json.dumps(
        phase37_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE37_MONITOR_SELECTION"
)

Phase37 frozen gate: alpha=0.50, gamma=0.50, cap=2.00, advanced=True
BEGIN SANITIZED_PHASE37_MONITOR_SELECTION
{
  "phase": "phase37_group_robust_bounded_phase36_monitor_selection",
  "status": "gate_frozen",
  "monitor_pool": {
    "observation_count": 493,
    "unique_case_count": 407,
    "maximum_case_multiplicity": 2,
    "inverse_multiplicity_weighting": true,
    "acquisition_group_count": 10
  },
  "phase36_reconstruction": {
    "selected_specification_index": 4,
    "original_shared_scale": 0.75,
    "monitor_log_loss_error": 5.551115123125783e-17,
    "oof_maximum_probability_error": 2.220446049250313e-16
  },
  "search": {
    "candidate_count": 61,
    "eligible_candidate_count": 15,
    "bootstrap_replicate_count": 5000
  },
  "baseline_monitor": {
    "log_loss": 0.2354520680279413,
    "auroc": 0.9574570945129321
  },
  "raw_phase36_monitor": {
    "shared_scale": 0.75,
    "log_loss": 0.21915518635444312,
    "log_loss_gain": 0.016296881673498187
  },
  "raw_best_candi

In [68]:
# Phase37 Cell 119C
# Evaluate the already frozen Phase37 gate exactly once on outer OOF.
# No parameter can be changed in this cell.

import json
import time

import numpy as np
from sklearn.linear_model import LogisticRegression


phase37_evaluation_started = time.perf_counter()

assert "PHASE37_GATE_PRIVATE" in globals()
assert "PHASE37_MONITOR_SELECTION_PRIVATE" in globals()
assert PHASE37_MONITOR_SELECTION_PRIVATE[
    "gate_advanced"
]

phase37_frozen_gate = dict(
    PHASE37_GATE_PRIVATE
)

assert phase37_frozen_gate == {
    "alpha": 0.5,
    "uncertainty_exponent": 0.5,
    "delta_cap": 2.0,
}, {
    "message": (
        "The frozen Phase37 gate changed after "
        "monitor selection."
    ),
    "observed": phase37_frozen_gate,
}


phase37_baseline_oof = np.asarray(
    phase36_baseline_oof,
    dtype=np.float64,
)

phase37_raw_residual_oof = np.asarray(
    PHASE36_RAW_RESIDUAL_OOF_PRIVATE,
    dtype=np.float64,
)

phase37_raw_phase36_oof = np.asarray(
    PHASE36_OOF_PRIVATE,
    dtype=np.float64,
)

assert phase37_baseline_oof.shape == (1362,)
assert phase37_raw_residual_oof.shape == (1362,)
assert phase37_raw_phase36_oof.shape == (1362,)

assert np.isfinite(
    phase37_baseline_oof
).all()
assert np.isfinite(
    phase37_raw_residual_oof
).all()


# Apply the monitor-frozen global gate.
phase37_oof = phase37_apply_gate(
    phase37_baseline_oof,
    phase37_raw_residual_oof,
    phase37_frozen_gate,
)

assert phase37_oof.shape == (1362,)
assert np.isfinite(phase37_oof).all()
assert np.all(
    (phase37_oof > 0.0)
    & (phase37_oof < 1.0)
)


def phase37_unweighted_metrics(
    labels,
    probabilities,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    weights = np.ones(
        labels.size,
        dtype=np.float64,
    )

    return {
        "log_loss": float(
            phase37_weighted_log_loss(
                labels,
                probabilities,
                weights,
            )
        ),
        "auroc": float(
            phase37_weighted_auroc(
                labels,
                probabilities,
                weights,
            )
        ),
        "mean_probability": float(
            np.mean(probabilities)
        ),
    }


phase37_baseline_metrics = (
    phase37_unweighted_metrics(
        phase37_labels,
        phase37_baseline_oof,
    )
)

phase37_raw_phase36_metrics = (
    phase37_unweighted_metrics(
        phase37_labels,
        phase37_raw_phase36_oof,
    )
)

phase37_metrics = (
    phase37_unweighted_metrics(
        phase37_labels,
        phase37_oof,
    )
)


# Validate against historical Phase12c and Phase36 values.
assert abs(
    phase37_baseline_metrics["log_loss"]
    - 0.307616
) <= 5e-6, phase37_baseline_metrics

assert abs(
    phase37_baseline_metrics["auroc"]
    - 0.938914
) <= 5e-5, phase37_baseline_metrics

assert abs(
    phase37_raw_phase36_metrics["log_loss"]
    - 0.295657
) <= 5e-6, phase37_raw_phase36_metrics

assert abs(
    phase37_raw_phase36_metrics["auroc"]
    - 0.944143
) <= 5e-5, phase37_raw_phase36_metrics


# Reconstruct the outer-fold assignment.
phase37_outer_fold = np.full(
    1362,
    -1,
    dtype=np.int64,
)

phase37_outer_fold_records = []

assert len(PHASE19_PARTITIONS) == 3

for fallback_fold, partition in enumerate(
    PHASE19_PARTITIONS
):
    fold = int(
        partition.get(
            "fold",
            fallback_fold,
        )
    )

    outer_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    assert np.all(
        phase37_outer_fold[
            outer_indices
        ] == -1
    )

    phase37_outer_fold[
        outer_indices
    ] = fold

assert np.all(phase37_outer_fold >= 0)
assert np.unique(
    phase37_outer_fold,
    return_counts=True,
)[1].tolist() == [467, 443, 452]


phase37_fold_wins = 0
phase37_worst_fold_excess = 0.0

for fold in sorted(
    np.unique(
        phase37_outer_fold
    ).tolist()
):
    position = (
        phase37_outer_fold == fold
    )

    baseline_fold_metrics = (
        phase37_unweighted_metrics(
            phase37_labels[position],
            phase37_baseline_oof[position],
        )
    )

    raw_phase36_fold_metrics = (
        phase37_unweighted_metrics(
            phase37_labels[position],
            phase37_raw_phase36_oof[
                position
            ],
        )
    )

    candidate_fold_metrics = (
        phase37_unweighted_metrics(
            phase37_labels[position],
            phase37_oof[position],
        )
    )

    log_loss_gain = (
        baseline_fold_metrics["log_loss"]
        - candidate_fold_metrics["log_loss"]
    )

    auroc_gain = (
        candidate_fold_metrics["auroc"]
        - baseline_fold_metrics["auroc"]
    )

    if log_loss_gain > 0.0:
        phase37_fold_wins += 1

    phase37_worst_fold_excess = max(
        phase37_worst_fold_excess,
        -log_loss_gain,
    )

    phase37_outer_fold_records.append({
        "fold": int(fold),
        "n": int(np.sum(position)),
        "baseline_log_loss": float(
            baseline_fold_metrics[
                "log_loss"
            ]
        ),
        "raw_phase36_log_loss": float(
            raw_phase36_fold_metrics[
                "log_loss"
            ]
        ),
        "phase37_log_loss": float(
            candidate_fold_metrics[
                "log_loss"
            ]
        ),
        "phase37_log_loss_improvement": float(
            log_loss_gain
        ),
        "baseline_auroc": float(
            baseline_fold_metrics[
                "auroc"
            ]
        ),
        "raw_phase36_auroc": float(
            raw_phase36_fold_metrics[
                "auroc"
            ]
        ),
        "phase37_auroc": float(
            candidate_fold_metrics[
                "auroc"
            ]
        ),
        "phase37_auroc_improvement": float(
            auroc_gain
        ),
    })


# Acquisition-group safety evaluation.
phase37_major_group_records = []
phase37_maximum_major_group_harm = 0.0

for group in sorted(
    np.unique(
        phase37_groups
    ).tolist()
):
    position = (
        phase37_groups == group
    )
    group_n = int(np.sum(position))

    # Maintain the prior "major group" contract.
    if group_n < 30:
        continue

    baseline_group_metrics = (
        phase37_unweighted_metrics(
            phase37_labels[position],
            phase37_baseline_oof[position],
        )
    )

    raw_phase36_group_metrics = (
        phase37_unweighted_metrics(
            phase37_labels[position],
            phase37_raw_phase36_oof[
                position
            ],
        )
    )

    candidate_group_metrics = (
        phase37_unweighted_metrics(
            phase37_labels[position],
            phase37_oof[position],
        )
    )

    candidate_change = (
        candidate_group_metrics["log_loss"]
        - baseline_group_metrics["log_loss"]
    )

    raw_phase36_change = (
        raw_phase36_group_metrics[
            "log_loss"
        ]
        - baseline_group_metrics["log_loss"]
    )

    phase37_maximum_major_group_harm = max(
        phase37_maximum_major_group_harm,
        candidate_change,
    )

    phase37_major_group_records.append({
        "group": int(group),
        "n": group_n,
        "baseline_log_loss": float(
            baseline_group_metrics[
                "log_loss"
            ]
        ),
        "raw_phase36_log_loss": float(
            raw_phase36_group_metrics[
                "log_loss"
            ]
        ),
        "raw_phase36_log_loss_change": float(
            raw_phase36_change
        ),
        "phase37_log_loss": float(
            candidate_group_metrics[
                "log_loss"
            ]
        ),
        "phase37_log_loss_change": float(
            candidate_change
        ),
        "baseline_auroc": float(
            baseline_group_metrics[
                "auroc"
            ]
        ),
        "phase37_auroc": float(
            candidate_group_metrics[
                "auroc"
            ]
        ),
    })


# Diagnostic calibration intercept and slope.
# These are evaluated only; they are not applied.
phase37_calibration_model = (
    LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
)

phase37_calibration_model.fit(
    phase37_logit(
        phase37_oof
    ).reshape(-1, 1),
    phase37_labels,
)

phase37_calibration_intercept = float(
    phase37_calibration_model.intercept_[0]
)

phase37_calibration_slope = float(
    phase37_calibration_model.coef_[0, 0]
)


# Update-magnitude audit.
phase37_baseline_uncertainty = (
    4.0
    * phase37_baseline_oof
    * (
        1.0
        - phase37_baseline_oof
    )
)

phase37_effective_alpha = (
    phase37_frozen_gate["alpha"]
    * phase37_baseline_uncertainty
    ** phase37_frozen_gate[
        "uncertainty_exponent"
    ]
)

phase37_bounded_raw_residual = np.clip(
    phase37_raw_residual_oof,
    -phase37_frozen_gate["delta_cap"],
    phase37_frozen_gate["delta_cap"],
)

phase37_logit_update = (
    phase37_effective_alpha
    * phase37_bounded_raw_residual
)

phase37_probability_update = (
    phase37_oof
    - phase37_baseline_oof
)


phase37_log_loss_gain = (
    phase37_baseline_metrics["log_loss"]
    - phase37_metrics["log_loss"]
)

phase37_auroc_gain = (
    phase37_metrics["auroc"]
    - phase37_baseline_metrics["auroc"]
)

phase37_log_loss_gain_over_phase33 = (
    0.302954
    - phase37_metrics["log_loss"]
)

phase37_log_loss_difference_from_raw_phase36 = (
    phase37_raw_phase36_metrics["log_loss"]
    - phase37_metrics["log_loss"]
)


PHASE37_PROMOTION_THRESHOLDS = {
    "minimum_log_loss_gain": 0.006,
    "minimum_auroc_gain": 0.002,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.003,
    "maximum_major_group_harm": 0.015,
    "calibration_slope_range": [
        0.8,
        1.2,
    ],
    "minimum_gain_over_phase33": 0.001,
}


phase37_component_gate_passed = bool(
    phase37_log_loss_gain
    >= PHASE37_PROMOTION_THRESHOLDS[
        "minimum_log_loss_gain"
    ]
    and phase37_auroc_gain
    >= PHASE37_PROMOTION_THRESHOLDS[
        "minimum_auroc_gain"
    ]
    and phase37_fold_wins
    >= PHASE37_PROMOTION_THRESHOLDS[
        "minimum_fold_wins"
    ]
    and phase37_worst_fold_excess
    <= PHASE37_PROMOTION_THRESHOLDS[
        "maximum_worst_fold_excess"
    ]
    and phase37_maximum_major_group_harm
    <= PHASE37_PROMOTION_THRESHOLDS[
        "maximum_major_group_harm"
    ]
    and PHASE37_PROMOTION_THRESHOLDS[
        "calibration_slope_range"
    ][0]
    <= phase37_calibration_slope
    <= PHASE37_PROMOTION_THRESHOLDS[
        "calibration_slope_range"
    ][1]
    and phase37_log_loss_gain_over_phase33
    >= PHASE37_PROMOTION_THRESHOLDS[
        "minimum_gain_over_phase33"
    ]
)


PHASE37_OOF_PRIVATE = (
    phase37_oof.copy()
)

PHASE37_DEPLOYMENT_STATE_PRIVATE = {
    "phase36_deployment_states": (
        PHASE36_DEPLOYMENT_STATES_PRIVATE
    ),
    "gate": dict(
        phase37_frozen_gate
    ),
    "formula": (
        "z_final = z_phase12c + "
        "alpha * "
        "(4*p_phase12c*(1-p_phase12c))^gamma "
        "* clip(raw_phase36_residual, -cap, cap)"
    ),
}


phase37_evaluation_report = {
    "phase": (
        "phase37_group_robust_bounded_"
        "phase36_residual"
    ),
    "status": (
        "eligible_for_shift_stress"
        if phase37_component_gate_passed
        else "not_promoted"
    ),
    "frozen_gate": (
        phase37_frozen_gate
    ),
    "baseline": {
        "log_loss": round(
            phase37_baseline_metrics[
                "log_loss"
            ],
            6,
        ),
        "auroc": round(
            phase37_baseline_metrics[
                "auroc"
            ],
            6,
        ),
    },
    "raw_phase36": {
        "log_loss": round(
            phase37_raw_phase36_metrics[
                "log_loss"
            ],
            6,
        ),
        "auroc": round(
            phase37_raw_phase36_metrics[
                "auroc"
            ],
            6,
        ),
    },
    "phase33_reference": {
        "log_loss": 0.302954,
        "auroc": 0.940945,
    },
    "phase37": {
        "log_loss": round(
            phase37_metrics["log_loss"],
            6,
        ),
        "auroc": round(
            phase37_metrics["auroc"],
            6,
        ),
        "mean_probability": round(
            phase37_metrics[
                "mean_probability"
            ],
            6,
        ),
        "calibration_intercept": round(
            phase37_calibration_intercept,
            6,
        ),
        "calibration_slope": round(
            phase37_calibration_slope,
            6,
        ),
    },
    "improvements": {
        "log_loss_gain_over_phase12c": round(
            phase37_log_loss_gain,
            6,
        ),
        "auroc_gain_over_phase12c": round(
            phase37_auroc_gain,
            6,
        ),
        "log_loss_gain_over_phase33": round(
            phase37_log_loss_gain_over_phase33,
            6,
        ),
        "log_loss_gain_over_raw_phase36": round(
            phase37_log_loss_difference_from_raw_phase36,
            6,
        ),
        "fold_wins": int(
            phase37_fold_wins
        ),
        "worst_fold_excess": round(
            phase37_worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            phase37_maximum_major_group_harm,
            6,
        ),
    },
    "fold_metrics": (
        phase37_outer_fold_records
    ),
    "major_acquisition_group_metrics": (
        phase37_major_group_records
    ),
    "update_magnitude": {
        "effective_alpha_quantiles": {
            "q05": round(
                float(
                    np.quantile(
                        phase37_effective_alpha,
                        0.05,
                    )
                ),
                6,
            ),
            "q50": round(
                float(
                    np.quantile(
                        phase37_effective_alpha,
                        0.50,
                    )
                ),
                6,
            ),
            "q95": round(
                float(
                    np.quantile(
                        phase37_effective_alpha,
                        0.95,
                    )
                ),
                6,
            ),
            "maximum": round(
                float(
                    np.max(
                        phase37_effective_alpha
                    )
                ),
                6,
            ),
        },
        "mean_absolute_logit_update": round(
            float(
                np.mean(
                    np.abs(
                        phase37_logit_update
                    )
                )
            ),
            6,
        ),
        "q95_absolute_logit_update": round(
            float(
                np.quantile(
                    np.abs(
                        phase37_logit_update
                    ),
                    0.95,
                )
            ),
            6,
        ),
        "mean_absolute_probability_update": (
            round(
                float(
                    np.mean(
                        np.abs(
                            phase37_probability_update
                        )
                    )
                ),
                6,
            )
        ),
        "q95_absolute_probability_update": (
            round(
                float(
                    np.quantile(
                        np.abs(
                            phase37_probability_update
                        ),
                        0.95,
                    )
                ),
                6,
            )
        ),
    },
    "promotion_thresholds": (
        PHASE37_PROMOTION_THRESHOLDS
    ),
    "component_gate_passed": bool(
        phase37_component_gate_passed
    ),
    "deployment_allowed_before_stress": False,
    "selection_partition": (
        "pooled_monitor_only"
    ),
    "selection_frozen_before_outer_evaluation": True,
    "shared_gate_across_folds": True,
    "fold_specific_group_routing": False,
    "outer_oracle_parameters_used": False,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase37_evaluation_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE37_OOF"
)
print(
    json.dumps(
        phase37_evaluation_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE37_OOF"
)

BEGIN SANITIZED_PHASE37_OOF
{
  "phase": "phase37_group_robust_bounded_phase36_residual",
  "status": "eligible_for_shift_stress",
  "frozen_gate": {
    "alpha": 0.5,
    "uncertainty_exponent": 0.5,
    "delta_cap": 2.0
  },
  "baseline": {
    "log_loss": 0.307616,
    "auroc": 0.938914
  },
  "raw_phase36": {
    "log_loss": 0.295657,
    "auroc": 0.944143
  },
  "phase33_reference": {
    "log_loss": 0.302954,
    "auroc": 0.940945
  },
  "phase37": {
    "log_loss": 0.297446,
    "auroc": 0.942846,
    "mean_probability": 0.542359,
    "calibration_intercept": 0.043698,
    "calibration_slope": 0.927502
  },
  "improvements": {
    "log_loss_gain_over_phase12c": 0.01017,
    "auroc_gain_over_phase12c": 0.003931,
    "log_loss_gain_over_phase33": 0.005508,
    "log_loss_gain_over_raw_phase36": -0.001789,
    "fold_wins": 3,
    "worst_fold_excess": 0.0,
    "maximum_major_group_harm": 0.013758
  },
  "fold_metrics": [
    {
      "fold": 0,
      "n": 467,
      "baseline_log_loss

In [69]:
# Phase37 Cell 119D
# Fixed-candidate acquisition-shift stress test.
# This cell does not select or modify any hyperparameter.

import json
import time

import numpy as np
from sklearn.metrics import roc_auc_score


PHASE37_STRESS_CONFIG = {
    "phase": "phase37_fixed_candidate_shift_stress",
    "random_seed": 370701,

    "within_group_bootstrap_replicates": 6000,
    "auroc_bootstrap_replicates": 3000,
    "dirichlet_replicates": 10000,
    "dirichlet_floor": 0.2,
    "dirichlet_concentrations": [
        100.0,
        20.0,
        5.0,
    ],
    "total_variation_radii": [
        0.05,
        0.10,
        0.20,
        0.30,
    ],

    "thresholds": {
        "minimum_observed_log_loss_gain": 0.006,
        "minimum_bootstrap_q05_gain": 0.003,
        "minimum_bootstrap_probability_positive": 0.975,
        "minimum_auroc_probability_positive": 0.95,
        "minimum_leave_one_group_out_gain": 0.001,
        "minimum_modest_shift_probability_positive": 0.99,
        "minimum_substantial_shift_probability_positive": 0.95,
        "minimum_severe_shift_probability_positive": 0.80,
        "minimum_tv_0p10_gain": 0.002,
        "minimum_tv_0p20_gain": 0.0,
        "maximum_major_group_harm": 0.015,
    },
}


def phase37_stress_summary(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "mean": float(np.mean(values)),
        "standard_deviation": float(
            np.std(values)
        ),
        "q01": float(
            np.quantile(values, 0.01)
        ),
        "q05": float(
            np.quantile(values, 0.05)
        ),
        "q50": float(
            np.quantile(values, 0.50)
        ),
        "q95": float(
            np.quantile(values, 0.95)
        ),
        "q99": float(
            np.quantile(values, 0.99)
        ),
        "probability_positive": float(
            np.mean(values > 0.0)
        ),
        "probability_gain_at_least_0p003": (
            float(
                np.mean(values >= 0.003)
            )
        ),
    }


def phase37_adversarial_group_shift(
    empirical_weights,
    group_gains,
    total_variation_radius,
):
    """
    Minimize expected gain by transferring probability mass
    from high-gain groups to low-gain groups.

    The amount of transferred mass equals total variation.
    """
    weights = np.asarray(
        empirical_weights,
        dtype=np.float64,
    ).copy()

    gains = np.asarray(
        group_gains,
        dtype=np.float64,
    )

    assert np.isclose(
        weights.sum(),
        1.0,
    )

    donor_order = np.argsort(
        gains
    )[::-1]
    receiver_order = np.argsort(
        gains
    )

    remaining_budget = float(
        total_variation_radius
    )

    donor_pointer = 0
    receiver_pointer = 0
    changed_groups = set()

    while (
        remaining_budget > 1e-12
        and donor_pointer
        < donor_order.size
        and receiver_pointer
        < receiver_order.size
    ):
        donor = int(
            donor_order[donor_pointer]
        )
        receiver = int(
            receiver_order[
                receiver_pointer
            ]
        )

        if (
            gains[donor]
            <= gains[receiver]
        ):
            break

        if donor == receiver:
            receiver_pointer += 1
            continue

        donor_available = weights[donor]
        receiver_capacity = (
            1.0 - weights[receiver]
        )

        transferred = min(
            donor_available,
            receiver_capacity,
            remaining_budget,
        )

        if transferred <= 1e-12:
            if donor_available <= 1e-12:
                donor_pointer += 1
            if receiver_capacity <= 1e-12:
                receiver_pointer += 1
            continue

        weights[donor] -= transferred
        weights[receiver] += transferred
        remaining_budget -= transferred

        changed_groups.add(donor)
        changed_groups.add(receiver)

        if weights[donor] <= 1e-12:
            donor_pointer += 1

        if (
            1.0 - weights[receiver]
            <= 1e-12
        ):
            receiver_pointer += 1

    return {
        "worst_case_gain": float(
            np.dot(weights, gains)
        ),
        "changed_group_count": int(
            len(changed_groups)
        ),
        "unused_shift_budget": float(
            remaining_budget
        ),
        "shifted_weights": weights,
    }


phase37_stress_started = time.perf_counter()

assert phase37_component_gate_passed
assert np.array_equal(
    PHASE37_OOF_PRIVATE,
    phase37_oof,
)
assert PHASE37_GATE_PRIVATE == {
    "alpha": 0.5,
    "uncertainty_exponent": 0.5,
    "delta_cap": 2.0,
}

phase37_stress_rng = np.random.default_rng(
    PHASE37_STRESS_CONFIG[
        "random_seed"
    ]
)

phase37_fixed_baseline = np.asarray(
    phase37_baseline_oof,
    dtype=np.float64,
)

phase37_fixed_candidate = np.asarray(
    PHASE37_OOF_PRIVATE,
    dtype=np.float64,
)

phase37_fixed_raw_phase36 = np.asarray(
    PHASE36_OOF_PRIVATE,
    dtype=np.float64,
)

phase37_fixed_labels = np.asarray(
    phase37_labels,
    dtype=np.int64,
)

phase37_fixed_groups = np.asarray(
    phase37_groups,
    dtype=np.int64,
)

assert all(
    vector.shape == (1362,)
    for vector in (
        phase37_fixed_baseline,
        phase37_fixed_candidate,
        phase37_fixed_raw_phase36,
        phase37_fixed_labels,
        phase37_fixed_groups,
    )
)


phase37_baseline_case_loss = (
    phase37_case_log_loss(
        phase37_fixed_labels,
        phase37_fixed_baseline,
    )
)

phase37_candidate_case_loss = (
    phase37_case_log_loss(
        phase37_fixed_labels,
        phase37_fixed_candidate,
    )
)

phase37_raw_phase36_case_loss = (
    phase37_case_log_loss(
        phase37_fixed_labels,
        phase37_fixed_raw_phase36,
    )
)

phase37_case_gain = (
    phase37_baseline_case_loss
    - phase37_candidate_case_loss
)

phase37_raw_phase36_case_gain = (
    phase37_baseline_case_loss
    - phase37_raw_phase36_case_loss
)

phase37_observed_log_loss_gain = float(
    np.mean(phase37_case_gain)
)

phase37_raw_phase36_observed_gain = float(
    np.mean(
        phase37_raw_phase36_case_gain
    )
)

phase37_observed_auroc_gain = float(
    roc_auc_score(
        phase37_fixed_labels,
        phase37_fixed_candidate,
    )
    - roc_auc_score(
        phase37_fixed_labels,
        phase37_fixed_baseline,
    )
)

phase37_raw_phase36_auroc_gain = float(
    roc_auc_score(
        phase37_fixed_labels,
        phase37_fixed_raw_phase36,
    )
    - roc_auc_score(
        phase37_fixed_labels,
        phase37_fixed_baseline,
    )
)


# ---------------------------------------------------------
# 1. Within-acquisition-group bootstrap for log-loss gain
# ---------------------------------------------------------

phase37_unique_stress_groups = np.unique(
    phase37_fixed_groups
)

phase37_group_positions = {
    int(group): np.flatnonzero(
        phase37_fixed_groups == group
    )
    for group in phase37_unique_stress_groups
}

phase37_bootstrap_count = int(
    PHASE37_STRESS_CONFIG[
        "within_group_bootstrap_replicates"
    ]
)

phase37_bootstrap_gain = np.zeros(
    phase37_bootstrap_count,
    dtype=np.float64,
)

phase37_raw_bootstrap_gain = np.zeros(
    phase37_bootstrap_count,
    dtype=np.float64,
)

for group in phase37_unique_stress_groups:
    positions = phase37_group_positions[
        int(group)
    ]

    sampled_offsets = (
        phase37_stress_rng.integers(
            0,
            positions.size,
            size=(
                phase37_bootstrap_count,
                positions.size,
            ),
        )
    )

    sampled_positions = positions[
        sampled_offsets
    ]

    phase37_bootstrap_gain += np.sum(
        phase37_case_gain[
            sampled_positions
        ],
        axis=1,
    )

    phase37_raw_bootstrap_gain += np.sum(
        phase37_raw_phase36_case_gain[
            sampled_positions
        ],
        axis=1,
    )

phase37_bootstrap_gain /= 1362.0
phase37_raw_bootstrap_gain /= 1362.0

phase37_bootstrap_summary = (
    phase37_stress_summary(
        phase37_bootstrap_gain
    )
)

phase37_raw_bootstrap_summary = (
    phase37_stress_summary(
        phase37_raw_bootstrap_gain
    )
)


# ---------------------------------------------------------
# 2. Acquisition-group × label stratified AUROC bootstrap
# ---------------------------------------------------------

phase37_auroc_strata = []

for group in phase37_unique_stress_groups:
    for label_value in (0, 1):
        positions = np.flatnonzero(
            (
                phase37_fixed_groups
                == group
            )
            & (
                phase37_fixed_labels
                == label_value
            )
        )

        if positions.size > 0:
            phase37_auroc_strata.append(
                positions
            )

phase37_auroc_bootstrap_count = int(
    PHASE37_STRESS_CONFIG[
        "auroc_bootstrap_replicates"
    ]
)

phase37_auroc_bootstrap_gain = np.empty(
    phase37_auroc_bootstrap_count,
    dtype=np.float64,
)

phase37_raw_auroc_bootstrap_gain = (
    np.empty(
        phase37_auroc_bootstrap_count,
        dtype=np.float64,
    )
)

for replicate in range(
    phase37_auroc_bootstrap_count
):
    sampled_parts = []

    for positions in phase37_auroc_strata:
        sampled_parts.append(
            positions[
                phase37_stress_rng.integers(
                    0,
                    positions.size,
                    size=positions.size,
                )
            ]
        )

    sampled_indices = np.concatenate(
        sampled_parts
    )

    sampled_labels = (
        phase37_fixed_labels[
            sampled_indices
        ]
    )

    baseline_auc = roc_auc_score(
        sampled_labels,
        phase37_fixed_baseline[
            sampled_indices
        ],
    )

    candidate_auc = roc_auc_score(
        sampled_labels,
        phase37_fixed_candidate[
            sampled_indices
        ],
    )

    raw_phase36_auc = roc_auc_score(
        sampled_labels,
        phase37_fixed_raw_phase36[
            sampled_indices
        ],
    )

    phase37_auroc_bootstrap_gain[
        replicate
    ] = (
        candidate_auc - baseline_auc
    )

    phase37_raw_auroc_bootstrap_gain[
        replicate
    ] = (
        raw_phase36_auc - baseline_auc
    )

phase37_auroc_bootstrap_summary = (
    phase37_stress_summary(
        phase37_auroc_bootstrap_gain
    )
)

phase37_raw_auroc_bootstrap_summary = (
    phase37_stress_summary(
        phase37_raw_auroc_bootstrap_gain
    )
)


# ---------------------------------------------------------
# 3. Per-group effects and leave-one-group-out robustness
# ---------------------------------------------------------

phase37_group_effect_records = []
phase37_group_mean_gains = []
phase37_raw_group_mean_gains = []
phase37_empirical_group_weights = []
phase37_leave_one_group_out_records = []

for group in phase37_unique_stress_groups:
    positions = phase37_group_positions[
        int(group)
    ]

    group_gain = float(
        np.mean(
            phase37_case_gain[positions]
        )
    )

    raw_group_gain = float(
        np.mean(
            phase37_raw_phase36_case_gain[
                positions
            ]
        )
    )

    empirical_weight = float(
        positions.size / 1362.0
    )

    phase37_group_mean_gains.append(
        group_gain
    )
    phase37_raw_group_mean_gains.append(
        raw_group_gain
    )
    phase37_empirical_group_weights.append(
        empirical_weight
    )

    retained = (
        phase37_fixed_groups != group
    )

    retained_gain = float(
        np.mean(
            phase37_case_gain[
                retained
            ]
        )
    )

    retained_raw_gain = float(
        np.mean(
            phase37_raw_phase36_case_gain[
                retained
            ]
        )
    )

    phase37_leave_one_group_out_records.append({
        "held_out_group": int(group),
        "retained_n": int(
            np.sum(retained)
        ),
        "phase37_log_loss_gain": (
            retained_gain
        ),
        "raw_phase36_log_loss_gain": (
            retained_raw_gain
        ),
    })

    phase37_group_effect_records.append({
        "group": int(group),
        "n": int(positions.size),
        "empirical_weight": (
            empirical_weight
        ),
        "phase37_log_loss_gain": (
            group_gain
        ),
        "raw_phase36_log_loss_gain": (
            raw_group_gain
        ),
        "phase37_candidate_better": bool(
            group_gain > 0.0
        ),
        "prevalence": float(
            np.mean(
                phase37_fixed_labels[
                    positions
                ]
            )
        ),
    })

phase37_group_mean_gains = np.asarray(
    phase37_group_mean_gains,
    dtype=np.float64,
)

phase37_raw_group_mean_gains = np.asarray(
    phase37_raw_group_mean_gains,
    dtype=np.float64,
)

phase37_empirical_group_weights = (
    np.asarray(
        phase37_empirical_group_weights,
        dtype=np.float64,
    )
)

phase37_minimum_leave_one_group_out_gain = (
    float(
        min(
            record[
                "phase37_log_loss_gain"
            ]
            for record
            in phase37_leave_one_group_out_records
        )
    )
)

phase37_raw_minimum_leave_one_group_out_gain = (
    float(
        min(
            record[
                "raw_phase36_log_loss_gain"
            ]
            for record
            in phase37_leave_one_group_out_records
        )
    )
)


# ---------------------------------------------------------
# 4. Dirichlet acquisition-mixture shifts
# ---------------------------------------------------------

phase37_dirichlet_shift_records = []
phase37_dirichlet_probability_map = {}

for concentration in (
    PHASE37_STRESS_CONFIG[
        "dirichlet_concentrations"
    ]
):
    parameters = (
        concentration
        * phase37_empirical_group_weights
        + PHASE37_STRESS_CONFIG[
            "dirichlet_floor"
        ]
    )

    mixture_draws = (
        phase37_stress_rng.dirichlet(
            parameters,
            size=PHASE37_STRESS_CONFIG[
                "dirichlet_replicates"
            ],
        )
    )

    shifted_gain = (
        mixture_draws
        @ phase37_group_mean_gains
    )

    shifted_raw_gain = (
        mixture_draws
        @ phase37_raw_group_mean_gains
    )

    candidate_summary = (
        phase37_stress_summary(
            shifted_gain
        )
    )

    raw_summary = (
        phase37_stress_summary(
            shifted_raw_gain
        )
    )

    interpretation = {
        100.0: "modest_shift",
        20.0: "substantial_shift",
        5.0: "severe_shift",
    }[float(concentration)]

    phase37_dirichlet_probability_map[
        interpretation
    ] = candidate_summary[
        "probability_positive"
    ]

    phase37_dirichlet_shift_records.append({
        "concentration": float(
            concentration
        ),
        "interpretation": interpretation,
        "phase37": candidate_summary,
        "raw_phase36": raw_summary,
    })


# ---------------------------------------------------------
# 5. Adversarial group-mixture shifts
# ---------------------------------------------------------

phase37_adversarial_records = []
phase37_tv_gain_map = {}

for radius in PHASE37_STRESS_CONFIG[
    "total_variation_radii"
]:
    phase37_shift = (
        phase37_adversarial_group_shift(
            phase37_empirical_group_weights,
            phase37_group_mean_gains,
            radius,
        )
    )

    raw_shift = (
        phase37_adversarial_group_shift(
            phase37_empirical_group_weights,
            phase37_raw_group_mean_gains,
            radius,
        )
    )

    phase37_tv_gain_map[
        round(float(radius), 2)
    ] = phase37_shift[
        "worst_case_gain"
    ]

    phase37_adversarial_records.append({
        "total_variation_radius": float(
            radius
        ),
        "phase37_worst_case_log_loss_gain": (
            phase37_shift[
                "worst_case_gain"
            ]
        ),
        "raw_phase36_worst_case_log_loss_gain": (
            raw_shift[
                "worst_case_gain"
            ]
        ),
        "phase37_changed_group_count": (
            phase37_shift[
                "changed_group_count"
            ]
        ),
        "phase37_unused_shift_budget": (
            phase37_shift[
                "unused_shift_budget"
            ]
        ),
    })


phase37_stress_thresholds = (
    PHASE37_STRESS_CONFIG[
        "thresholds"
    ]
)

phase37_stress_gate_passed = bool(
    phase37_observed_log_loss_gain
    >= phase37_stress_thresholds[
        "minimum_observed_log_loss_gain"
    ]
    and phase37_bootstrap_summary["q05"]
    >= phase37_stress_thresholds[
        "minimum_bootstrap_q05_gain"
    ]
    and phase37_bootstrap_summary[
        "probability_positive"
    ]
    >= phase37_stress_thresholds[
        "minimum_bootstrap_probability_positive"
    ]
    and phase37_auroc_bootstrap_summary[
        "probability_positive"
    ]
    >= phase37_stress_thresholds[
        "minimum_auroc_probability_positive"
    ]
    and phase37_minimum_leave_one_group_out_gain
    >= phase37_stress_thresholds[
        "minimum_leave_one_group_out_gain"
    ]
    and phase37_dirichlet_probability_map[
        "modest_shift"
    ]
    >= phase37_stress_thresholds[
        "minimum_modest_shift_probability_positive"
    ]
    and phase37_dirichlet_probability_map[
        "substantial_shift"
    ]
    >= phase37_stress_thresholds[
        "minimum_substantial_shift_probability_positive"
    ]
    and phase37_dirichlet_probability_map[
        "severe_shift"
    ]
    >= phase37_stress_thresholds[
        "minimum_severe_shift_probability_positive"
    ]
    and phase37_tv_gain_map[0.10]
    >= phase37_stress_thresholds[
        "minimum_tv_0p10_gain"
    ]
    and phase37_tv_gain_map[0.20]
    >= phase37_stress_thresholds[
        "minimum_tv_0p20_gain"
    ]
    and phase37_maximum_major_group_harm
    <= phase37_stress_thresholds[
        "maximum_major_group_harm"
    ]
)


PHASE37_STRESS_REPORT_PRIVATE = {
    "stress_gate_passed": bool(
        phase37_stress_gate_passed
    ),
    "configuration": dict(
        PHASE37_STRESS_CONFIG
    ),
    "observed_log_loss_gain": (
        phase37_observed_log_loss_gain
    ),
    "bootstrap_summary": (
        phase37_bootstrap_summary
    ),
    "auroc_bootstrap_summary": (
        phase37_auroc_bootstrap_summary
    ),
    "minimum_leave_one_group_out_gain": (
        phase37_minimum_leave_one_group_out_gain
    ),
    "adversarial_records": (
        phase37_adversarial_records
    ),
}


phase37_stress_report = {
    "phase": (
        "phase37_acquisition_shift_robustness_stress"
    ),
    "status": (
        "stress_passed_current_champion"
        if phase37_stress_gate_passed
        else "stress_failed_retain_diagnostic_only"
    ),
    "fixed_candidate": {
        "gate": dict(
            PHASE37_GATE_PRIVATE
        ),
        "log_loss": round(
            phase37_metrics["log_loss"],
            6,
        ),
        "auroc": round(
            phase37_metrics["auroc"],
            6,
        ),
        "observed_log_loss_gain": round(
            phase37_observed_log_loss_gain,
            6,
        ),
        "observed_auroc_gain": round(
            phase37_observed_auroc_gain,
            6,
        ),
        "maximum_major_group_harm": round(
            phase37_maximum_major_group_harm,
            6,
        ),
    },
    "raw_phase36_comparison": {
        "log_loss": round(
            phase37_raw_phase36_metrics[
                "log_loss"
            ],
            6,
        ),
        "auroc": round(
            phase37_raw_phase36_metrics[
                "auroc"
            ],
            6,
        ),
        "observed_log_loss_gain": round(
            phase37_raw_phase36_observed_gain,
            6,
        ),
        "observed_auroc_gain": round(
            phase37_raw_phase36_auroc_gain,
            6,
        ),
        "maximum_major_group_harm": 0.038002,
    },
    "within_group_log_loss_bootstrap": {
        "replicate_count": int(
            phase37_bootstrap_count
        ),
        "phase37": (
            phase37_bootstrap_summary
        ),
        "raw_phase36": (
            phase37_raw_bootstrap_summary
        ),
    },
    "group_label_stratified_auroc_bootstrap": {
        "replicate_count": int(
            phase37_auroc_bootstrap_count
        ),
        "phase37": (
            phase37_auroc_bootstrap_summary
        ),
        "raw_phase36": (
            phase37_raw_auroc_bootstrap_summary
        ),
    },
    "leave_one_acquisition_group_out": {
        "phase37_minimum_log_loss_gain": (
            phase37_minimum_leave_one_group_out_gain
        ),
        "raw_phase36_minimum_log_loss_gain": (
            phase37_raw_minimum_leave_one_group_out_gain
        ),
        "records": (
            phase37_leave_one_group_out_records
        ),
    },
    "dirichlet_group_mixture_shift": (
        phase37_dirichlet_shift_records
    ),
    "adversarial_group_mixture_shift": (
        phase37_adversarial_records
    ),
    "acquisition_group_effects": (
        phase37_group_effect_records
    ),
    "stress_thresholds": (
        phase37_stress_thresholds
    ),
    "stress_gate_passed": bool(
        phase37_stress_gate_passed
    ),
    "gate_parameters_modified": False,
    "outer_labels_used_only_for_fixed_candidate_stress_evaluation": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase37_stress_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE37_STRESS"
)
print(
    json.dumps(
        phase37_stress_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE37_STRESS"
)

BEGIN SANITIZED_PHASE37_STRESS
{
  "phase": "phase37_acquisition_shift_robustness_stress",
  "status": "stress_failed_retain_diagnostic_only",
  "fixed_candidate": {
    "gate": {
      "alpha": 0.5,
      "uncertainty_exponent": 0.5,
      "delta_cap": 2.0
    },
    "log_loss": 0.297446,
    "auroc": 0.942846,
    "observed_log_loss_gain": 0.01017,
    "observed_auroc_gain": 0.003931,
    "maximum_major_group_harm": 0.013758
  },
  "raw_phase36_comparison": {
    "log_loss": 0.295657,
    "auroc": 0.944143,
    "observed_log_loss_gain": 0.011959,
    "observed_auroc_gain": 0.005229,
    "maximum_major_group_harm": 0.038002
  },
  "within_group_log_loss_bootstrap": {
    "replicate_count": 6000,
    "phase37": {
      "mean": 0.010180812103828154,
      "standard_deviation": 0.0026141662452521673,
      "q01": 0.004090673449426806,
      "q05": 0.005888270147885477,
      "q50": 0.010145287540779356,
      "q95": 0.014554933336497147,
      "q99": 0.016361674839039327,
      "probabil

In [71]:
# Phase38 Cell 120A-fix
# Correct the structurally constant segment active-volume feature,
# then repeat the contract audit.

import json
import time

import numpy as np


def phase38_segment_descriptors(
    signal,
    mask,
    total_signal_mass,
    background,
):
    signal = np.asarray(
        signal,
        dtype=np.float64,
    )

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    assert signal.shape == mask.shape

    # This is now measured relative to the complete patch,
    # rather than only within a mask that was already active.
    active_fraction = float(
        np.mean(mask)
    )

    values = np.asarray(
        signal[mask],
        dtype=np.float64,
    )

    if values.size == 0:
        values = np.zeros(
            1,
            dtype=np.float64,
        )

    positive = values[
        values > 0.0
    ]

    if positive.size == 0:
        positive = np.zeros(
            1,
            dtype=np.float64,
        )

    local_mass = float(
        np.sum(values)
    )

    weighted_mean = float(
        np.sum(values * values)
        / max(
            np.sum(values),
            1e-8,
        )
    )

    background_scale = max(
        float(background),
        1e-4,
    )

    return {
        "weighted_sbr": (
            weighted_mean
            / background_scale
        ),
        "q90_sbr": (
            float(
                np.quantile(
                    positive,
                    0.90,
                )
            )
            / background_scale
        ),
        "top10_sbr": (
            phase38_top_tail_mean(
                positive,
                PHASE38_CONFIG[
                    "top_tail_fraction"
                ],
            )
            / background_scale
        ),
        "mass_fraction": (
            local_mass
            / max(
                total_signal_mass,
                1e-8,
            )
        ),
        "active_fraction": (
            active_fraction
        ),
    }


phase38_fix_started = time.perf_counter()

(
    phase38_normal_features,
    PHASE38_FEATURE_NAMES,
    phase38_normal_diagnostics,
) = phase38_case_features(
    phase38_synthetic_normal
)

(
    phase38_reflected_features,
    phase38_reflected_names,
    _,
) = phase38_case_features(
    np.flip(
        phase38_synthetic_normal,
        axis=0,
    ).copy()
)

(
    phase38_abnormal_features,
    phase38_abnormal_names,
    _,
) = phase38_case_features(
    phase38_synthetic_abnormal
)

assert (
    PHASE38_FEATURE_NAMES
    == phase38_reflected_names
    == phase38_abnormal_names
)

phase38_reflection_error = float(
    np.max(
        np.abs(
            phase38_normal_features
            - phase38_reflected_features
        )
    )
)

phase38_sensitivity = float(
    np.mean(
        np.abs(
            phase38_normal_features
            - phase38_abnormal_features
        )
    )
)

phase38_real_audit_features = []

for index in phase38_real_audit_indices:
    (
        real_features,
        real_names,
        _,
    ) = phase38_case_features(
        PHASE31_HIGHRES_CACHE[
            int(index)
        ]
    )

    assert (
        real_names
        == PHASE38_FEATURE_NAMES
    )

    phase38_real_audit_features.append(
        real_features
    )

phase38_real_audit_features = np.stack(
    phase38_real_audit_features,
    axis=0,
)

phase38_coordinate_standard_deviation = (
    np.std(
        phase38_real_audit_features,
        axis=0,
    )
)

phase38_collapsed_mask = (
    phase38_coordinate_standard_deviation
    < 1e-8
)

phase38_collapsed_coordinate_count = int(
    np.sum(
        phase38_collapsed_mask
    )
)

phase38_collapsed_coordinate_fraction = (
    float(
        np.mean(
            phase38_collapsed_mask
        )
    )
)

phase38_corrected_report = {
    "phase": (
        "phase38_atlasless_striatal_"
        "subregion_contract_correction"
    ),
    "status": (
        "accepted"
        if (
            phase38_reflection_error
            <= PHASE38_CONFIG[
                "reflection_tolerance"
            ]
            and phase38_sensitivity
            >= PHASE38_CONFIG[
                "minimum_sensitivity"
            ]
            and phase38_collapsed_coordinate_fraction
            <= 0.03
        )
        else "requires_review"
    ),
    "feature_count": int(
        phase38_normal_features.size
    ),
    "corrected_feature": (
        "segment_active_fraction"
    ),
    "old_definition": (
        "active voxels divided by voxels "
        "already selected as active"
    ),
    "new_definition": (
        "segment active voxels divided by "
        "complete patch voxel count"
    ),
    "structural_constant_coordinates_removed": 15,
    "reflection_maximum_absolute_error": (
        phase38_reflection_error
    ),
    "synthetic_endpoint_attenuation_sensitivity": (
        phase38_sensitivity
    ),
    "real_audit_shape": list(
        phase38_real_audit_features.shape
    ),
    "collapsed_coordinate_count": (
        phase38_collapsed_coordinate_count
    ),
    "collapsed_coordinate_fraction": (
        phase38_collapsed_coordinate_fraction
    ),
    "all_values_finite": bool(
        np.isfinite(
            phase38_real_audit_features
        ).all()
    ),
    "labels_used": False,
    "training_voxel_cache_read": True,
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase38_fix_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE38_CORRECTED_CONTRACT"
)
print(
    json.dumps(
        phase38_corrected_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE38_CORRECTED_CONTRACT"
)

assert (
    phase38_corrected_report["status"]
    == "accepted"
), phase38_corrected_report

BEGIN SANITIZED_PHASE38_CORRECTED_CONTRACT
{
  "phase": "phase38_atlasless_striatal_subregion_contract_correction",
  "status": "accepted",
  "feature_count": 170,
  "corrected_feature": "segment_active_fraction",
  "old_definition": "active voxels divided by voxels already selected as active",
  "new_definition": "segment active voxels divided by complete patch voxel count",
  "structural_constant_coordinates_removed": 15,
  "reflection_maximum_absolute_error": 0.0,
  "synthetic_endpoint_attenuation_sensitivity": 0.11259251832962036,
  "real_audit_shape": [
    8,
    170
  ],
  "collapsed_coordinate_count": 0,
  "collapsed_coordinate_fraction": 0.0,
  "all_values_finite": true,
  "labels_used": false,
  "training_voxel_cache_read": true,
  "outer_validation_labels_used": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "elapsed_seconds": 2.76
}
END SANITIZED_PHASE38_CORRECTED_CONTRACT


In [72]:
# Phase38 Cell 120B
# Extract and persist the corrected atlasless striatal
# subregion representation for all training cases.

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import json
import os
import time

import numpy as np


PHASE38_FEATURE_PATH = Path(
    "/kaggle/working/"
    "phase38_striatal_subregions_float32.npy"
)

PHASE38_DIAGNOSTIC_PATH = Path(
    "/kaggle/working/"
    "phase38_striatal_subregion_diagnostics_float32.npy"
)

PHASE38_METADATA_PATH = Path(
    "/kaggle/working/"
    "phase38_striatal_subregion_metadata.json"
)

PHASE38_FEATURE_PARTIAL_PATH = Path(
    "/kaggle/working/"
    "phase38_striatal_subregions_float32.partial.npy"
)

PHASE38_DIAGNOSTIC_PARTIAL_PATH = Path(
    "/kaggle/working/"
    "phase38_striatal_subregion_diagnostics_"
    "float32.partial.npy"
)

PHASE38_METADATA_PARTIAL_PATH = Path(
    "/kaggle/working/"
    "phase38_striatal_subregion_metadata.partial.json"
)

PHASE38_WORKER_COUNT = int(
    PHASE31_CONFIG.get(
        "worker_count",
        min(
            8,
            os.cpu_count() or 4,
        ),
    )
)


def phase38_validate_feature_cache():
    required_paths = [
        PHASE38_FEATURE_PATH,
        PHASE38_DIAGNOSTIC_PATH,
        PHASE38_METADATA_PATH,
    ]

    if not all(
        path.is_file()
        for path in required_paths
    ):
        return None

    try:
        metadata = json.loads(
            PHASE38_METADATA_PATH.read_text(
                encoding="utf-8"
            )
        )

        if int(
            metadata.get(
                "schema_version",
                -1,
            )
        ) != 2:
            return None

        if metadata.get(
            "active_fraction_definition"
        ) != (
            "segment_voxels_divided_by_"
            "complete_patch_voxels"
        ):
            return None

        cached_names = list(
            metadata["feature_names"]
        )

        if (
            cached_names
            != PHASE38_FEATURE_NAMES
        ):
            return None

        features = np.load(
            PHASE38_FEATURE_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        diagnostics = np.load(
            PHASE38_DIAGNOSTIC_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        if features.shape != (
            1362,
            170,
        ):
            return None

        if diagnostics.shape != (
            1362,
            2,
        ):
            return None

        if features.dtype != np.float32:
            return None

        if diagnostics.dtype != np.float32:
            return None

        check_indices = np.linspace(
            0,
            1361,
            32,
            dtype=np.int64,
        )

        if not np.isfinite(
            np.asarray(
                features[check_indices],
                dtype=np.float32,
            )
        ).all():
            return None

        if not np.isfinite(
            np.asarray(
                diagnostics[check_indices],
                dtype=np.float32,
            )
        ).all():
            return None

        return (
            features,
            diagnostics,
            metadata,
        )

    except Exception:
        return None


def phase38_extract_case_index(index):
    try:
        (
            feature_vector,
            feature_names,
            diagnostics,
        ) = phase38_case_features(
            PHASE31_HIGHRES_CACHE[
                int(index)
            ]
        )

        feature_vector = np.asarray(
            feature_vector,
            dtype=np.float32,
        )

        diagnostic_vector = np.asarray(
            [
                diagnostics["background"],
                diagnostics[
                    "center_separation"
                ],
            ],
            dtype=np.float32,
        )

        if feature_vector.shape != (170,):
            raise ValueError(
                "UnexpectedFeatureShape"
            )

        if not np.isfinite(
            feature_vector
        ).all():
            raise ValueError(
                "NonFiniteFeature"
            )

        if not np.isfinite(
            diagnostic_vector
        ).all():
            raise ValueError(
                "NonFiniteDiagnostic"
            )

        return (
            int(index),
            feature_vector,
            list(feature_names),
            diagnostic_vector,
            None,
        )

    except Exception as exception:
        return (
            int(index),
            None,
            None,
            None,
            type(exception).__name__,
        )


phase38_extraction_started = (
    time.perf_counter()
)

phase38_cached_result = (
    phase38_validate_feature_cache()
)

phase38_cache_reused = (
    phase38_cached_result is not None
)

phase38_failure_types = []

if phase38_cached_result is not None:
    (
        PHASE38_FEATURES_PRIVATE,
        PHASE38_DIAGNOSTICS_PRIVATE,
        phase38_feature_metadata,
    ) = phase38_cached_result

    PHASE38_FEATURE_NAMES = list(
        phase38_feature_metadata[
            "feature_names"
        ]
    )

else:
    (
        first_index,
        first_features,
        first_names,
        first_diagnostics,
        first_error,
    ) = phase38_extract_case_index(0)

    assert first_error is None, first_error
    assert first_index == 0
    assert first_features.shape == (170,)
    assert len(first_names) == 170
    assert len(set(first_names)) == 170
    assert first_names == PHASE38_FEATURE_NAMES

    feature_writer = (
        np.lib.format.open_memmap(
            PHASE38_FEATURE_PARTIAL_PATH,
            mode="w+",
            dtype=np.float32,
            shape=(1362, 170),
        )
    )

    diagnostic_writer = (
        np.lib.format.open_memmap(
            PHASE38_DIAGNOSTIC_PARTIAL_PATH,
            mode="w+",
            dtype=np.float32,
            shape=(1362, 2),
        )
    )

    feature_writer[0] = first_features
    diagnostic_writer[0] = (
        first_diagnostics
    )

    with ThreadPoolExecutor(
        max_workers=PHASE38_WORKER_COUNT
    ) as executor:
        result_iterator = executor.map(
            phase38_extract_case_index,
            range(1, 1362),
            chunksize=1,
        )

        for completed, result in enumerate(
            result_iterator,
            start=2,
        ):
            (
                index,
                feature_vector,
                feature_names,
                diagnostic_vector,
                error_type,
            ) = result

            if error_type is not None:
                phase38_failure_types.append(
                    error_type
                )

            else:
                assert (
                    feature_names
                    == PHASE38_FEATURE_NAMES
                )

                feature_writer[index] = (
                    feature_vector
                )

                diagnostic_writer[index] = (
                    diagnostic_vector
                )

            if (
                completed % 100 == 0
                or completed == 1362
            ):
                feature_writer.flush()
                diagnostic_writer.flush()

                print(
                    "Phase38 feature extraction: "
                    f"{completed}/1362"
                )

    feature_writer.flush()
    diagnostic_writer.flush()

    del feature_writer
    del diagnostic_writer

    assert not phase38_failure_types, {
        "message": (
            "Phase38 feature extraction failed."
        ),
        "failure_count": len(
            phase38_failure_types
        ),
        "failure_type_counts": {
            failure_type: (
                phase38_failure_types.count(
                    failure_type
                )
            )
            for failure_type in sorted(
                set(
                    phase38_failure_types
                )
            )
        },
    }

    phase38_feature_metadata = {
        "schema_version": 2,
        "phase": PHASE38_CONFIG["phase"],
        "case_count": 1362,
        "feature_count": 170,
        "feature_names": list(
            PHASE38_FEATURE_NAMES
        ),
        "diagnostic_names": [
            "background",
            "center_separation",
        ],
        "patch_size": int(
            PHASE38_CONFIG[
                "patch_size"
            ]
        ),
        "active_fraction_definition": (
            "segment_voxels_divided_by_"
            "complete_patch_voxels"
        ),
        "reflection_invariant": True,
        "case_level_features_exported": False,
    }

    PHASE38_METADATA_PARTIAL_PATH.write_text(
        json.dumps(
            phase38_feature_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    os.replace(
        PHASE38_FEATURE_PARTIAL_PATH,
        PHASE38_FEATURE_PATH,
    )

    os.replace(
        PHASE38_DIAGNOSTIC_PARTIAL_PATH,
        PHASE38_DIAGNOSTIC_PATH,
    )

    os.replace(
        PHASE38_METADATA_PARTIAL_PATH,
        PHASE38_METADATA_PATH,
    )

    validated_result = (
        phase38_validate_feature_cache()
    )

    assert validated_result is not None

    (
        PHASE38_FEATURES_PRIVATE,
        PHASE38_DIAGNOSTICS_PRIVATE,
        phase38_feature_metadata,
    ) = validated_result


phase38_extraction_elapsed = (
    time.perf_counter()
    - phase38_extraction_started
)

phase38_feature_sample = np.asarray(
    PHASE38_FEATURES_PRIVATE[
        np.linspace(
            0,
            1361,
            64,
            dtype=np.int64,
        )
    ],
    dtype=np.float32,
)

phase38_diagnostic_sample = np.asarray(
    PHASE38_DIAGNOSTICS_PRIVATE[
        np.linspace(
            0,
            1361,
            64,
            dtype=np.int64,
        )
    ],
    dtype=np.float32,
)

phase38_extraction_report = {
    "phase": (
        "phase38_atlasless_striatal_"
        "subregion_feature_cache"
    ),
    "status": "complete",
    "cache_reused": bool(
        phase38_cache_reused
    ),
    "case_count": 1362,
    "worker_count": int(
        PHASE38_WORKER_COUNT
    ),
    "features": {
        "shape": list(
            PHASE38_FEATURES_PRIVATE.shape
        ),
        "dtype": str(
            PHASE38_FEATURES_PRIVATE.dtype
        ),
        "feature_count": 170,
        "size_mb": round(
            PHASE38_FEATURE_PATH.stat().st_size
            / 1e6,
            3,
        ),
    },
    "diagnostics": {
        "shape": list(
            PHASE38_DIAGNOSTICS_PRIVATE.shape
        ),
        "dtype": str(
            PHASE38_DIAGNOSTICS_PRIVATE.dtype
        ),
    },
    "sampled_feature_minimum": float(
        np.min(
            phase38_feature_sample
        )
    ),
    "sampled_feature_maximum": float(
        np.max(
            phase38_feature_sample
        )
    ),
    "sampled_feature_mean": float(
        np.mean(
            phase38_feature_sample
        )
    ),
    "all_sampled_values_finite": bool(
        np.isfinite(
            phase38_feature_sample
        ).all()
        and np.isfinite(
            phase38_diagnostic_sample
        ).all()
    ),
    "failure_count": int(
        len(
            phase38_failure_types
        )
    ),
    "persistent_notebook_cache_only": True,
    "include_in_submission": False,
    "labels_used": False,
    "training_voxel_cache_read": True,
    "outer_validation_labels_used": False,
    "case_level_features_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        phase38_extraction_elapsed,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE38_FEATURE_CACHE"
)
print(
    json.dumps(
        phase38_extraction_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE38_FEATURE_CACHE"
)

Phase38 feature extraction: 100/1362
Phase38 feature extraction: 200/1362
Phase38 feature extraction: 300/1362
Phase38 feature extraction: 400/1362
Phase38 feature extraction: 500/1362
Phase38 feature extraction: 600/1362
Phase38 feature extraction: 700/1362
Phase38 feature extraction: 800/1362
Phase38 feature extraction: 900/1362
Phase38 feature extraction: 1000/1362
Phase38 feature extraction: 1100/1362
Phase38 feature extraction: 1200/1362
Phase38 feature extraction: 1300/1362
Phase38 feature extraction: 1362/1362
BEGIN SANITIZED_PHASE38_FEATURE_CACHE
{
  "phase": "phase38_atlasless_striatal_subregion_feature_cache",
  "status": "complete",
  "cache_reused": false,
  "case_count": 1362,
  "worker_count": 8,
  "features": {
    "shape": [
      1362,
      170
    ],
    "dtype": "float32",
    "feature_count": 170,
    "size_mb": 0.926
  },
  "diagnostics": {
    "shape": [
      1362,
      2
    ],
    "dtype": "float32"
  },
  "sampled_feature_minimum": 0.0,
  "sampled_feature_ma

In [73]:
# Phase38 Cell 120C
# Label-free quality, redundancy and numerical-stability audit.

import json
import time

import numpy as np


PHASE38_AUDIT_CONFIG = {
    "collapsed_standard_deviation": 1e-8,
    "near_constant_standard_deviation": 1e-5,
    "near_duplicate_correlation": 0.9999,
    "effective_rank_variance_fraction": 0.99,
    "robust_scale_floor": 1e-6,
}


phase38_audit_started = time.perf_counter()

phase38_feature_matrix = np.asarray(
    PHASE38_FEATURES_PRIVATE,
    dtype=np.float64,
)

phase38_diagnostic_matrix = np.asarray(
    PHASE38_DIAGNOSTICS_PRIVATE,
    dtype=np.float64,
)

assert phase38_feature_matrix.shape == (
    1362,
    170,
)

assert phase38_diagnostic_matrix.shape == (
    1362,
    2,
)

assert len(PHASE38_FEATURE_NAMES) == 170
assert len(set(PHASE38_FEATURE_NAMES)) == 170


phase38_all_finite = bool(
    np.isfinite(
        phase38_feature_matrix
    ).all()
)

assert phase38_all_finite


# ---------------------------------------------------------
# Coordinate-wise variability
# ---------------------------------------------------------

phase38_coordinate_mean = np.mean(
    phase38_feature_matrix,
    axis=0,
)

phase38_coordinate_std = np.std(
    phase38_feature_matrix,
    axis=0,
)

phase38_coordinate_median = np.median(
    phase38_feature_matrix,
    axis=0,
)

phase38_coordinate_q25 = np.quantile(
    phase38_feature_matrix,
    0.25,
    axis=0,
)

phase38_coordinate_q75 = np.quantile(
    phase38_feature_matrix,
    0.75,
    axis=0,
)

phase38_coordinate_iqr = (
    phase38_coordinate_q75
    - phase38_coordinate_q25
)

phase38_coordinate_mad = np.median(
    np.abs(
        phase38_feature_matrix
        - phase38_coordinate_median[
            None,
            :
        ]
    ),
    axis=0,
)

phase38_collapsed_mask = (
    phase38_coordinate_std
    < PHASE38_AUDIT_CONFIG[
        "collapsed_standard_deviation"
    ]
)

phase38_near_constant_mask = (
    phase38_coordinate_std
    < PHASE38_AUDIT_CONFIG[
        "near_constant_standard_deviation"
    ]
)

phase38_zero_iqr_mask = (
    phase38_coordinate_iqr
    < PHASE38_AUDIT_CONFIG[
        "robust_scale_floor"
    ]
)


# ---------------------------------------------------------
# Tail and ratio stability
# ---------------------------------------------------------

phase38_absolute_values = np.abs(
    phase38_feature_matrix
)

phase38_coordinate_abs_q50 = np.quantile(
    phase38_absolute_values,
    0.50,
    axis=0,
)

phase38_coordinate_abs_q99 = np.quantile(
    phase38_absolute_values,
    0.99,
    axis=0,
)

phase38_coordinate_abs_q999 = np.quantile(
    phase38_absolute_values,
    0.999,
    axis=0,
)

phase38_tail_ratio = (
    phase38_coordinate_abs_q999
    / np.maximum(
        phase38_coordinate_abs_q50,
        1e-6,
    )
)

phase38_extreme_tail_mask = (
    phase38_tail_ratio > 100.0
)

phase38_ratio_feature_mask = np.asarray(
    [
        (
            "ratio" in name.lower()
            or "minimum_maximum" in name.lower()
        )
        for name in PHASE38_FEATURE_NAMES
    ],
    dtype=bool,
)

phase38_ratio_maximum = float(
    np.max(
        phase38_absolute_values[
            :,
            phase38_ratio_feature_mask
        ]
    )
)

phase38_ratio_q999 = float(
    np.quantile(
        phase38_absolute_values[
            :,
            phase38_ratio_feature_mask
        ],
        0.999,
    )
)


# ---------------------------------------------------------
# Robustly standardize for redundancy diagnostics only
# ---------------------------------------------------------

phase38_audit_scale = np.maximum(
    phase38_coordinate_iqr,
    PHASE38_AUDIT_CONFIG[
        "robust_scale_floor"
    ],
)

phase38_audit_standardized = (
    (
        phase38_feature_matrix
        - phase38_coordinate_median[
            None,
            :
        ]
    )
    / phase38_audit_scale[
        None,
        :
    ]
)

phase38_audit_standardized = np.clip(
    phase38_audit_standardized,
    -20.0,
    20.0,
)

phase38_noncollapsed_indices = np.flatnonzero(
    ~phase38_collapsed_mask
)

phase38_redundancy_matrix = (
    phase38_audit_standardized[
        :,
        phase38_noncollapsed_indices
    ]
)

phase38_correlation = np.corrcoef(
    phase38_redundancy_matrix,
    rowvar=False,
)

phase38_correlation = np.nan_to_num(
    phase38_correlation,
    nan=0.0,
    posinf=0.0,
    neginf=0.0,
)

phase38_upper_triangle = np.triu(
    np.ones_like(
        phase38_correlation,
        dtype=bool,
    ),
    k=1,
)

phase38_near_duplicate_pairs = (
    np.abs(
        phase38_correlation
    )
    >= PHASE38_AUDIT_CONFIG[
        "near_duplicate_correlation"
    ]
) & phase38_upper_triangle

phase38_near_duplicate_pair_count = int(
    np.sum(
        phase38_near_duplicate_pairs
    )
)


# Deterministic diagnostic-only redundancy mask.
# The final fold preprocessors must recreate this decision
# using fit-partition data rather than this global mask.
phase38_diagnostic_redundant_indices = set()

for row, column in np.argwhere(
    phase38_near_duplicate_pairs
):
    original_column = int(
        phase38_noncollapsed_indices[
            column
        ]
    )

    phase38_diagnostic_redundant_indices.add(
        original_column
    )


# ---------------------------------------------------------
# Effective matrix rank
# ---------------------------------------------------------

phase38_singular_values = np.linalg.svd(
    phase38_redundancy_matrix,
    full_matrices=False,
    compute_uv=False,
)

phase38_singular_variance = (
    phase38_singular_values ** 2
)

phase38_cumulative_variance = (
    np.cumsum(
        phase38_singular_variance
    )
    / np.sum(
        phase38_singular_variance
    )
)

phase38_effective_rank_99 = int(
    np.searchsorted(
        phase38_cumulative_variance,
        PHASE38_AUDIT_CONFIG[
            "effective_rank_variance_fraction"
        ],
    )
    + 1
)

phase38_numerical_rank = int(
    np.linalg.matrix_rank(
        phase38_redundancy_matrix
    )
)


# ---------------------------------------------------------
# Feature-family counts
# ---------------------------------------------------------

phase38_feature_family_counts = {}

for feature_name in PHASE38_FEATURE_NAMES:
    base_descriptor = feature_name.split(
        "__bilateral_",
        1,
    )[0]

    if base_descriptor.startswith(
        "end_low_"
    ):
        family = "lower_uptake_endpoint"

    elif base_descriptor.startswith(
        "middle_"
    ):
        family = "middle_subregion"

    elif base_descriptor.startswith(
        "end_high_"
    ):
        family = "higher_uptake_endpoint"

    elif base_descriptor.startswith(
        "global_"
    ):
        family = "whole_structure_uptake"

    elif (
        "axis" in base_descriptor
        or "spread" in base_descriptor
    ):
        family = "shape_and_extent"

    elif base_descriptor.startswith(
        "volume_fraction"
    ):
        family = "multithreshold_extent"

    elif base_descriptor.startswith(
        "end_"
    ):
        family = "within_structure_endpoint_contrast"

    else:
        family = "other"

    phase38_feature_family_counts[
        family
    ] = (
        phase38_feature_family_counts.get(
            family,
            0,
        )
        + 1
    )


phase38_background = (
    phase38_diagnostic_matrix[:, 0]
)

phase38_center_separation = (
    phase38_diagnostic_matrix[:, 1]
)


phase38_feature_audit_report = {
    "phase": (
        "phase38_striatal_subregion_"
        "feature_quality_audit"
    ),
    "status": (
        "accepted"
        if (
            phase38_all_finite
            and np.mean(
                phase38_collapsed_mask
            ) <= 0.02
            and phase38_ratio_q999 < 100.0
        )
        else "requires_numerical_review"
    ),
    "shape": list(
        phase38_feature_matrix.shape
    ),
    "coordinate_quality": {
        "collapsed_coordinate_count": int(
            np.sum(
                phase38_collapsed_mask
            )
        ),
        "collapsed_coordinate_fraction": float(
            np.mean(
                phase38_collapsed_mask
            )
        ),
        "near_constant_coordinate_count": int(
            np.sum(
                phase38_near_constant_mask
            )
        ),
        "zero_iqr_coordinate_count": int(
            np.sum(
                phase38_zero_iqr_mask
            )
        ),
        "median_coordinate_standard_deviation": float(
            np.median(
                phase38_coordinate_std
            )
        ),
        "median_coordinate_iqr": float(
            np.median(
                phase38_coordinate_iqr
            )
        ),
    },
    "absolute_value_quantiles": {
        "q50": float(
            np.quantile(
                phase38_absolute_values,
                0.50,
            )
        ),
        "q90": float(
            np.quantile(
                phase38_absolute_values,
                0.90,
            )
        ),
        "q99": float(
            np.quantile(
                phase38_absolute_values,
                0.99,
            )
        ),
        "q999": float(
            np.quantile(
                phase38_absolute_values,
                0.999,
            )
        ),
        "maximum": float(
            np.max(
                phase38_absolute_values
            )
        ),
    },
    "ratio_feature_stability": {
        "ratio_feature_count": int(
            np.sum(
                phase38_ratio_feature_mask
            )
        ),
        "q999_absolute_value": (
            phase38_ratio_q999
        ),
        "maximum_absolute_value": (
            phase38_ratio_maximum
        ),
        "extreme_tail_coordinate_count": int(
            np.sum(
                phase38_extreme_tail_mask
            )
        ),
    },
    "redundancy": {
        "near_duplicate_correlation_threshold": (
            PHASE38_AUDIT_CONFIG[
                "near_duplicate_correlation"
            ]
        ),
        "near_duplicate_pair_count": (
            phase38_near_duplicate_pair_count
        ),
        "diagnostic_redundant_coordinate_count": int(
            len(
                phase38_diagnostic_redundant_indices
            )
        ),
        "numerical_rank": (
            phase38_numerical_rank
        ),
        "effective_rank_for_99_percent_variance": (
            phase38_effective_rank_99
        ),
    },
    "feature_family_counts": (
        phase38_feature_family_counts
    ),
    "localization_diagnostics": {
        "background_quantiles": {
            "q05": float(
                np.quantile(
                    phase38_background,
                    0.05,
                )
            ),
            "q50": float(
                np.quantile(
                    phase38_background,
                    0.50,
                )
            ),
            "q95": float(
                np.quantile(
                    phase38_background,
                    0.95,
                )
            ),
        },
        "center_separation_quantiles": {
            "q05": float(
                np.quantile(
                    phase38_center_separation,
                    0.05,
                )
            ),
            "q50": float(
                np.quantile(
                    phase38_center_separation,
                    0.50,
                )
            ),
            "q95": float(
                np.quantile(
                    phase38_center_separation,
                    0.95,
                )
            ),
        },
    },
    "fold_preprocessing_contract": {
        "remove_collapsed_coordinates": True,
        "remove_near_duplicate_coordinates": True,
        "winsorization_quantiles": [
            0.005,
            0.995,
        ],
        "robust_scaling": (
            "median_and_IQR_fit_partition_only"
        ),
        "global_diagnostic_mask_used_for_training": False,
    },
    "all_values_finite": (
        phase38_all_finite
    ),
    "labels_used": False,
    "outer_validation_images_used_for_preprocessing": False,
    "outer_validation_labels_used": False,
    "case_level_features_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase38_audit_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE38_FEATURE_AUDIT"
)
print(
    json.dumps(
        phase38_feature_audit_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE38_FEATURE_AUDIT"
)

assert (
    phase38_feature_audit_report[
        "status"
    ]
    == "accepted"
), phase38_feature_audit_report

BEGIN SANITIZED_PHASE38_FEATURE_AUDIT
{
  "phase": "phase38_striatal_subregion_feature_quality_audit",
  "status": "accepted",
  "shape": [
    1362,
    170
  ],
  "coordinate_quality": {
    "collapsed_coordinate_count": 0,
    "collapsed_coordinate_fraction": 0.0,
    "near_constant_coordinate_count": 0,
    "zero_iqr_coordinate_count": 0,
    "median_coordinate_standard_deviation": 0.17469569917367267,
    "median_coordinate_iqr": 0.15152716916054487
  },
  "absolute_value_quantiles": {
    "q50": 0.8250822126865387,
    "q90": 2.4279150724411016,
    "q99": 10.276857423782346,
    "q999": 12.890051782608465,
    "maximum": 19.566377639770508
  },
  "ratio_feature_stability": {
    "ratio_feature_count": 50,
    "q999_absolute_value": 3.1535116803646015,
    "maximum_absolute_value": 9.111062049865723,
    "extreme_tail_coordinate_count": 0
  },
  "redundancy": {
    "near_duplicate_correlation_threshold": 0.9999,
    "near_duplicate_pair_count": 0,
    "diagnostic_redundant_coordi

In [74]:
# Phase38 Cell 121A
# Fold-local preprocessing and complete anchored-XGBoost
# candidate contract.

import json
import time

import numpy as np
import xgboost as xgb


PHASE38_MODEL_CONFIG = {
    "phase": (
        "phase38_atlasless_subregion_"
        "anchored_boosting"
    ),
    "seeds": [
        380701,
        380702,
    ],
    "learning_rate": 0.03,
    "maximum_rounds": 700,
    "early_stopping_rounds": 50,
    "subsample": 0.85,
    "maximum_bin_count": 128,
    "maximum_delta_step": 1.0,
    "nthread": 8,

    "representations": [
        "all_features",
        "uptake_without_shape",
        "endpoint_core",
    ],
    "maximum_depths": [
        1,
        2,
    ],
    "minimum_child_weights": [
        8.0,
        16.0,
        32.0,
    ],
    "l2_regularizations": [
        10.0,
        30.0,
    ],
    "column_subsamples": [
        0.65,
        1.0,
    ],

    "winsorization_quantiles": [
        0.005,
        0.995,
    ],
    "minimum_standard_deviation": 1e-8,
    "minimum_iqr": 1e-6,
    "near_duplicate_correlation": 0.9999,

    # Applied only after one model specification is frozen.
    "gate_alphas": [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    "gate_uncertainty_exponents": [
        0.0,
        0.5,
        1.0,
    ],
    "gate_delta_caps": [
        0.50,
        0.75,
        1.00,
        1.50,
        2.00,
    ],

    "minimum_model_monitor_gain": 0.002,
    "minimum_model_fold_wins": 2,
    "maximum_model_fold_regret": 0.005,
    "maximum_model_group_harm": 0.015,
    "minimum_monitor_group_n": 8,
    "auroc_near_tie_tolerance": 0.00025,
}


def phase38_base_descriptor_name(
    feature_name,
):
    return feature_name.split(
        "__bilateral_",
        1,
    )[0]


def phase38_representation_indices(
    representation,
):
    selected = []

    for index, feature_name in enumerate(
        PHASE38_FEATURE_NAMES
    ):
        base_name = (
            phase38_base_descriptor_name(
                feature_name
            )
        )

        if representation == "all_features":
            include = True

        elif (
            representation
            == "uptake_without_shape"
        ):
            shape_tokens = (
                "primary_axis_spread",
                "secondary_axis_spread",
                "tertiary_axis_spread",
                "primary_secondary_ratio",
                "secondary_tertiary_ratio",
            )

            include = not any(
                token in base_name
                for token in shape_tokens
            )

        elif representation == "endpoint_core":
            include = bool(
                base_name.startswith(
                    "end_low_"
                )
                or base_name.startswith(
                    "end_high_"
                )
                or base_name.startswith(
                    "end_top10_"
                )
                or base_name.startswith(
                    "end_low_high_"
                )
                or base_name.startswith(
                    "global_weighted_"
                )
                or base_name.startswith(
                    "global_q"
                )
                or base_name.startswith(
                    "global_top"
                )
            )

        else:
            raise ValueError(
                "Unknown representation: "
                f"{representation}"
            )

        if include:
            selected.append(index)

    selected = np.asarray(
        selected,
        dtype=np.int64,
    )

    assert selected.size >= 32

    return selected


def phase38_fit_preprocessor(
    features,
    representation,
):
    features = np.asarray(
        features,
        dtype=np.float64,
    )

    representation_indices = (
        phase38_representation_indices(
            representation
        )
    )

    selected = features[
        :,
        representation_indices,
    ]

    lower_quantile, upper_quantile = (
        PHASE38_MODEL_CONFIG[
            "winsorization_quantiles"
        ]
    )

    lower = np.quantile(
        selected,
        lower_quantile,
        axis=0,
    )

    upper = np.quantile(
        selected,
        upper_quantile,
        axis=0,
    )

    winsorized = np.clip(
        selected,
        lower[None, :],
        upper[None, :],
    )

    coordinate_std = np.std(
        winsorized,
        axis=0,
    )

    q25 = np.quantile(
        winsorized,
        0.25,
        axis=0,
    )

    q75 = np.quantile(
        winsorized,
        0.75,
        axis=0,
    )

    coordinate_iqr = q75 - q25

    variable_mask = (
        coordinate_std
        >= PHASE38_MODEL_CONFIG[
            "minimum_standard_deviation"
        ]
    ) & (
        coordinate_iqr
        >= PHASE38_MODEL_CONFIG[
            "minimum_iqr"
        ]
    )

    variable_indices = np.flatnonzero(
        variable_mask
    )

    assert variable_indices.size >= 16

    variable_features = winsorized[
        :,
        variable_indices,
    ]

    median = np.median(
        variable_features,
        axis=0,
    )

    scale = (
        np.quantile(
            variable_features,
            0.75,
            axis=0,
        )
        - np.quantile(
            variable_features,
            0.25,
            axis=0,
        )
    )

    scale = np.maximum(
        scale,
        PHASE38_MODEL_CONFIG[
            "minimum_iqr"
        ],
    )

    standardized = (
        variable_features
        - median[None, :]
    ) / scale[None, :]

    standardized = np.clip(
        standardized,
        -20.0,
        20.0,
    )

    correlation = np.corrcoef(
        standardized,
        rowvar=False,
    )

    correlation = np.nan_to_num(
        correlation,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    kept_indices = []

    for candidate_index in range(
        standardized.shape[1]
    ):
        if not kept_indices:
            kept_indices.append(
                candidate_index
            )
            continue

        maximum_correlation = float(
            np.max(
                np.abs(
                    correlation[
                        candidate_index,
                        kept_indices,
                    ]
                )
            )
        )

        if (
            maximum_correlation
            < PHASE38_MODEL_CONFIG[
                "near_duplicate_correlation"
            ]
        ):
            kept_indices.append(
                candidate_index
            )

    kept_indices = np.asarray(
        kept_indices,
        dtype=np.int64,
    )

    assert kept_indices.size >= 16

    state = {
        "representation": representation,
        "representation_indices": (
            representation_indices
        ),
        "lower": lower,
        "upper": upper,
        "variable_indices": (
            variable_indices
        ),
        "median": median,
        "scale": scale,
        "kept_indices": kept_indices,
        "input_dimension": int(
            selected.shape[1]
        ),
        "output_dimension": int(
            kept_indices.size
        ),
    }

    transformed = standardized[
        :,
        kept_indices,
    ].astype(np.float32)

    assert np.isfinite(
        transformed
    ).all()

    return state, transformed


def phase38_transform_preprocessor(
    state,
    features,
):
    features = np.asarray(
        features,
        dtype=np.float64,
    )

    selected = features[
        :,
        state[
            "representation_indices"
        ],
    ]

    winsorized = np.clip(
        selected,
        state["lower"][None, :],
        state["upper"][None, :],
    )

    variable_features = winsorized[
        :,
        state["variable_indices"],
    ]

    standardized = (
        variable_features
        - state["median"][None, :]
    ) / state["scale"][None, :]

    standardized = np.clip(
        standardized,
        -20.0,
        20.0,
    )

    transformed = standardized[
        :,
        state["kept_indices"],
    ].astype(np.float32)

    assert transformed.shape[1] == (
        state["output_dimension"]
    )

    assert np.isfinite(
        transformed
    ).all()

    return transformed


phase38_model_contract_started = (
    time.perf_counter()
)

phase38_candidate_specifications = []

for representation in PHASE38_MODEL_CONFIG[
    "representations"
]:
    for maximum_depth in (
        PHASE38_MODEL_CONFIG[
            "maximum_depths"
        ]
    ):
        for minimum_child_weight in (
            PHASE38_MODEL_CONFIG[
                "minimum_child_weights"
            ]
        ):
            for l2_regularization in (
                PHASE38_MODEL_CONFIG[
                    "l2_regularizations"
                ]
            ):
                for column_subsample in (
                    PHASE38_MODEL_CONFIG[
                        "column_subsamples"
                    ]
                ):
                    phase38_candidate_specifications.append({
                        "representation": (
                            representation
                        ),
                        "maximum_depth": int(
                            maximum_depth
                        ),
                        "minimum_child_weight": float(
                            minimum_child_weight
                        ),
                        "l2_regularization": float(
                            l2_regularization
                        ),
                        "column_subsample": float(
                            column_subsample
                        ),
                    })

assert len(
    phase38_candidate_specifications
) == 72


phase38_gate_specifications = [{
    "alpha": 0.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 0.0,
}]

for alpha in PHASE38_MODEL_CONFIG[
    "gate_alphas"
]:
    if alpha == 0.0:
        continue

    for gamma in PHASE38_MODEL_CONFIG[
        "gate_uncertainty_exponents"
    ]:
        for cap in PHASE38_MODEL_CONFIG[
            "gate_delta_caps"
        ]:
            phase38_gate_specifications.append({
                "alpha": float(alpha),
                "uncertainty_exponent": float(
                    gamma
                ),
                "delta_cap": float(cap),
            })

assert len(
    phase38_gate_specifications
) == 61


# Preprocessing contract uses fold-0 fit images only.
phase38_contract_fit_indices = np.asarray(
    PHASE19_PARTITIONS[0]["fit"],
    dtype=np.int64,
)

phase38_representation_contracts = {}

for representation in PHASE38_MODEL_CONFIG[
    "representations"
]:
    (
        contract_state,
        contract_transformed,
    ) = phase38_fit_preprocessor(
        phase38_feature_matrix[
            phase38_contract_fit_indices
        ],
        representation,
    )

    repeated_transform = (
        phase38_transform_preprocessor(
            contract_state,
            phase38_feature_matrix[
                phase38_contract_fit_indices[
                    :16
                ]
            ],
        )
    )

    assert repeated_transform.shape == (
        16,
        contract_state[
            "output_dimension"
        ],
    )

    phase38_representation_contracts[
        representation
    ] = {
        "input_dimension": int(
            contract_state[
                "input_dimension"
            ]
        ),
        "output_dimension": int(
            contract_state[
                "output_dimension"
            ]
        ),
        "fit_shape": list(
            contract_transformed.shape
        ),
        "all_values_finite": bool(
            np.isfinite(
                contract_transformed
            ).all()
        ),
    }


phase38_model_contract_report = {
    "phase": (
        "phase38_atlasless_subregion_"
        "anchored_boosting_contract"
    ),
    "status": "accepted",
    "backend": "xgboost",
    "backend_version": xgb.__version__,
    "representation_contracts": (
        phase38_representation_contracts
    ),
    "model_hyperparameter_grid": {
        "representation_count": int(
            len(
                PHASE38_MODEL_CONFIG[
                    "representations"
                ]
            )
        ),
        "maximum_depth_count": int(
            len(
                PHASE38_MODEL_CONFIG[
                    "maximum_depths"
                ]
            )
        ),
        "minimum_child_weight_count": int(
            len(
                PHASE38_MODEL_CONFIG[
                    "minimum_child_weights"
                ]
            )
        ),
        "l2_regularization_count": int(
            len(
                PHASE38_MODEL_CONFIG[
                    "l2_regularizations"
                ]
            )
        ),
        "column_subsample_count": int(
            len(
                PHASE38_MODEL_CONFIG[
                    "column_subsamples"
                ]
            )
        ),
        "candidate_specification_count": int(
            len(
                phase38_candidate_specifications
            )
        ),
        "seed_count": int(
            len(
                PHASE38_MODEL_CONFIG[
                    "seeds"
                ]
            )
        ),
        "models_per_fold": int(
            len(
                phase38_candidate_specifications
            )
            * len(
                PHASE38_MODEL_CONFIG[
                    "seeds"
                ]
            )
        ),
    },
    "gate_grid": {
        "candidate_count": int(
            len(
                phase38_gate_specifications
            )
        ),
        "searched_only_after_model_freeze": True,
    },
    "base_margin": (
        "exact_cross_fitted_phase12c_logit"
    ),
    "preprocessing": {
        "winsorization": (
            "fit_partition_only"
        ),
        "variance_filtering": (
            "fit_partition_only"
        ),
        "correlation_filtering": (
            "fit_partition_only"
        ),
        "robust_scaling": (
            "fit_partition_only"
        ),
    },
    "selection_strategy": [
        "select_one_shared_model_specification",
        "freeze_model_specification",
        "search_one_shared_bounded_gate",
        "freeze_gate",
        "refit_on_outer_train",
        "evaluate_outer_once",
    ],
    "primary_metric": "monitor_log_loss",
    "secondary_metric": "monitor_auroc",
    "group_robust_constraints": True,
    "labels_used": False,
    "outer_validation_images_used_for_preprocessing": False,
    "outer_validation_labels_used": False,
    "case_level_features_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase38_model_contract_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE38_MODEL_CONTRACT"
)
print(
    json.dumps(
        phase38_model_contract_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE38_MODEL_CONTRACT"
)

BEGIN SANITIZED_PHASE38_MODEL_CONTRACT
{
  "phase": "phase38_atlasless_subregion_anchored_boosting_contract",
  "status": "accepted",
  "backend": "xgboost",
  "backend_version": "3.2.0",
  "representation_contracts": {
    "all_features": {
      "input_dimension": 170,
      "output_dimension": 170,
      "fit_shape": [
        721,
        170
      ],
      "all_values_finite": true
    },
    "uptake_without_shape": {
      "input_dimension": 145,
      "output_dimension": 145,
      "fit_shape": [
        721,
        145
      ],
      "all_values_finite": true
    },
    "endpoint_core": {
      "input_dimension": 95,
      "output_dimension": 95,
      "fit_shape": [
        721,
        95
      ],
      "all_values_finite": true
    }
  },
  "model_hyperparameter_grid": {
    "representation_count": 3,
    "maximum_depth_count": 2,
    "minimum_child_weight_count": 3,
    "l2_regularization_count": 2,
    "column_subsample_count": 2,
    "candidate_specification_count": 72,


In [75]:
# Phase38 Cell 121B
# Exhaustive shared model selection followed by a separate
# bounded-gate search. Uses fit and monitor partitions only.

import json
import time

import numpy as np
import xgboost as xgb


def phase38_group_balanced_weights(groups):
    groups = np.asarray(
        groups,
        dtype=np.int64,
    )

    unique_groups, counts = np.unique(
        groups,
        return_counts=True,
    )

    count_map = {
        int(group): int(count)
        for group, count
        in zip(
            unique_groups,
            counts,
        )
    }

    weights = np.asarray(
        [
            groups.size
            / (
                unique_groups.size
                * count_map[int(group)]
            )
            for group in groups
        ],
        dtype=np.float64,
    )

    weights /= np.mean(weights)

    return weights.astype(np.float32)


def phase38_train_with_monitor(
    train_features,
    train_labels,
    train_groups,
    train_margin,
    monitor_features,
    monitor_labels,
    monitor_margin,
    specification,
    seed,
):
    train_weights = (
        phase38_group_balanced_weights(
            train_groups
        )
    )

    train_matrix = xgb.DMatrix(
        np.asarray(
            train_features,
            dtype=np.float32,
        ),
        label=np.asarray(
            train_labels,
            dtype=np.float32,
        ),
        weight=train_weights,
        base_margin=np.asarray(
            train_margin,
            dtype=np.float32,
        ),
    )

    monitor_matrix = xgb.DMatrix(
        np.asarray(
            monitor_features,
            dtype=np.float32,
        ),
        label=np.asarray(
            monitor_labels,
            dtype=np.float32,
        ),
        base_margin=np.asarray(
            monitor_margin,
            dtype=np.float32,
        ),
    )

    parameters = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "eta": float(
            PHASE38_MODEL_CONFIG[
                "learning_rate"
            ]
        ),
        "max_depth": int(
            specification[
                "maximum_depth"
            ]
        ),
        "min_child_weight": float(
            specification[
                "minimum_child_weight"
            ]
        ),
        "lambda": float(
            specification[
                "l2_regularization"
            ]
        ),
        "alpha": 0.0,
        "subsample": float(
            PHASE38_MODEL_CONFIG[
                "subsample"
            ]
        ),
        "colsample_bytree": float(
            specification[
                "column_subsample"
            ]
        ),
        "max_bin": int(
            PHASE38_MODEL_CONFIG[
                "maximum_bin_count"
            ]
        ),
        "max_delta_step": float(
            PHASE38_MODEL_CONFIG[
                "maximum_delta_step"
            ]
        ),
        "nthread": int(
            PHASE38_MODEL_CONFIG[
                "nthread"
            ]
        ),
        "seed": int(seed),
        "verbosity": 0,
    }

    started = time.perf_counter()

    booster = xgb.train(
        parameters,
        train_matrix,
        num_boost_round=int(
            PHASE38_MODEL_CONFIG[
                "maximum_rounds"
            ]
        ),
        evals=[
            (train_matrix, "train"),
            (monitor_matrix, "monitor"),
        ],
        early_stopping_rounds=int(
            PHASE38_MODEL_CONFIG[
                "early_stopping_rounds"
            ]
        ),
        verbose_eval=False,
    )

    if getattr(
        booster,
        "best_iteration",
        None,
    ) is None:
        round_count = int(
            PHASE38_MODEL_CONFIG[
                "maximum_rounds"
            ]
        )
    else:
        round_count = int(
            booster.best_iteration
        ) + 1

    full_monitor_margin = booster.predict(
        monitor_matrix,
        output_margin=True,
        iteration_range=(0, round_count),
    ).astype(np.float64)

    monitor_residual = (
        full_monitor_margin
        - np.asarray(
            monitor_margin,
            dtype=np.float64,
        )
    )

    monitor_probability = (
        phase37_sigmoid(
            full_monitor_margin
        )
    )

    return {
        "round_count": round_count,
        "monitor_residual": (
            monitor_residual
        ),
        "monitor_probability": (
            monitor_probability
        ),
        "training_seconds": float(
            time.perf_counter()
            - started
        ),
    }


def phase38_monitor_metrics(
    probabilities,
    monitor_labels,
    monitor_groups,
    monitor_folds,
    monitor_indices,
    monitor_baseline,
    monitor_weights,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    overall_log_loss = (
        phase37_weighted_log_loss(
            monitor_labels,
            probabilities,
            monitor_weights,
        )
    )

    baseline_log_loss = (
        phase37_weighted_log_loss(
            monitor_labels,
            monitor_baseline,
            monitor_weights,
        )
    )

    overall_auroc = (
        phase37_weighted_auroc(
            monitor_labels,
            probabilities,
            monitor_weights,
        )
    )

    baseline_auroc = (
        phase37_weighted_auroc(
            monitor_labels,
            monitor_baseline,
            monitor_weights,
        )
    )

    fold_records = []
    fold_wins = 0
    worst_fold_regret = 0.0

    for fold in sorted(
        np.unique(
            monitor_folds
        ).tolist()
    ):
        position = (
            monitor_folds == fold
        )

        baseline_fold_loss = (
            phase37_weighted_log_loss(
                monitor_labels[position],
                monitor_baseline[position],
                monitor_weights[position],
            )
        )

        candidate_fold_loss = (
            phase37_weighted_log_loss(
                monitor_labels[position],
                probabilities[position],
                monitor_weights[position],
            )
        )

        fold_gain = (
            baseline_fold_loss
            - candidate_fold_loss
        )

        if fold_gain > 0.0:
            fold_wins += 1

        worst_fold_regret = max(
            worst_fold_regret,
            -fold_gain,
        )

        fold_records.append({
            "fold": int(fold),
            "baseline_log_loss": float(
                baseline_fold_loss
            ),
            "candidate_log_loss": float(
                candidate_fold_loss
            ),
            "gain": float(fold_gain),
        })

    group_records = []
    maximum_group_harm = 0.0

    for group in sorted(
        np.unique(
            monitor_groups
        ).tolist()
    ):
        position = (
            monitor_groups == group
        )

        unique_case_count = int(
            np.unique(
                monitor_indices[position]
            ).size
        )

        if (
            unique_case_count
            < PHASE38_MODEL_CONFIG[
                "minimum_monitor_group_n"
            ]
        ):
            continue

        baseline_group_loss = (
            phase37_weighted_log_loss(
                monitor_labels[position],
                monitor_baseline[position],
                monitor_weights[position],
            )
        )

        candidate_group_loss = (
            phase37_weighted_log_loss(
                monitor_labels[position],
                probabilities[position],
                monitor_weights[position],
            )
        )

        group_harm = (
            candidate_group_loss
            - baseline_group_loss
        )

        maximum_group_harm = max(
            maximum_group_harm,
            group_harm,
        )

        group_records.append({
            "group": int(group),
            "unique_case_count": (
                unique_case_count
            ),
            "baseline_log_loss": float(
                baseline_group_loss
            ),
            "candidate_log_loss": float(
                candidate_group_loss
            ),
            "log_loss_change": float(
                group_harm
            ),
        })

    return {
        "log_loss": float(
            overall_log_loss
        ),
        "log_loss_gain": float(
            baseline_log_loss
            - overall_log_loss
        ),
        "auroc": float(
            overall_auroc
        ),
        "auroc_gain": float(
            overall_auroc
            - baseline_auroc
        ),
        "fold_wins": int(
            fold_wins
        ),
        "worst_fold_regret": float(
            worst_fold_regret
        ),
        "maximum_group_harm": float(
            maximum_group_harm
        ),
        "fold_records": fold_records,
        "group_records": group_records,
    }


phase38_selection_started = (
    time.perf_counter()
)

phase38_labels = np.asarray(
    phase37_labels,
    dtype=np.int64,
)

phase38_groups = np.asarray(
    phase37_groups,
    dtype=np.int64,
)

phase38_baseline_oof = np.asarray(
    phase36_baseline_oof,
    dtype=np.float64,
)

phase38_baseline_margin = (
    phase37_logit(
        phase38_baseline_oof
    )
)

assert phase38_labels.shape == (1362,)
assert phase38_groups.shape == (1362,)
assert phase38_baseline_oof.shape == (1362,)


# ---------------------------------------------------------
# Fit one preprocessor per fold and representation
# ---------------------------------------------------------

phase38_fold_preprocessing = []
phase38_selection_folds = []

for fallback_fold, partition in enumerate(
    PHASE19_PARTITIONS
):
    fold = int(
        partition.get(
            "fold",
            fallback_fold,
        )
    )

    fit_indices = np.asarray(
        partition["fit"],
        dtype=np.int64,
    )

    monitor_indices = np.asarray(
        partition["monitor"],
        dtype=np.int64,
    )

    fold_preprocessors = {}

    for representation in (
        PHASE38_MODEL_CONFIG[
            "representations"
        ]
    ):
        (
            preprocessing_state,
            fit_features,
        ) = phase38_fit_preprocessor(
            phase38_feature_matrix[
                fit_indices
            ],
            representation,
        )

        monitor_features = (
            phase38_transform_preprocessor(
                preprocessing_state,
                phase38_feature_matrix[
                    monitor_indices
                ],
            )
        )

        fold_preprocessors[
            representation
        ] = {
            "state": preprocessing_state,
            "fit_features": fit_features,
            "monitor_features": (
                monitor_features
            ),
        }

    phase38_fold_preprocessing.append({
        "fold": fold,
        "fit_indices": fit_indices,
        "monitor_indices": (
            monitor_indices
        ),
        "representations": (
            fold_preprocessors
        ),
    })


# ---------------------------------------------------------
# Train all 72 specifications × 2 seeds × 3 folds
# ---------------------------------------------------------

for fold_record in (
    phase38_fold_preprocessing
):
    fold = int(
        fold_record["fold"]
    )

    fit_indices = (
        fold_record["fit_indices"]
    )
    monitor_indices = (
        fold_record[
            "monitor_indices"
        ]
    )

    specification_results = []

    for specification_index, specification in enumerate(
        phase38_candidate_specifications
    ):
        representation = (
            specification[
                "representation"
            ]
        )

        transformed = (
            fold_record[
                "representations"
            ][representation]
        )

        seed_results = []

        for seed in PHASE38_MODEL_CONFIG[
            "seeds"
        ]:
            seed_result = (
                phase38_train_with_monitor(
                    train_features=(
                        transformed[
                            "fit_features"
                        ]
                    ),
                    train_labels=(
                        phase38_labels[
                            fit_indices
                        ]
                    ),
                    train_groups=(
                        phase38_groups[
                            fit_indices
                        ]
                    ),
                    train_margin=(
                        phase38_baseline_margin[
                            fit_indices
                        ]
                    ),
                    monitor_features=(
                        transformed[
                            "monitor_features"
                        ]
                    ),
                    monitor_labels=(
                        phase38_labels[
                            monitor_indices
                        ]
                    ),
                    monitor_margin=(
                        phase38_baseline_margin[
                            monitor_indices
                        ]
                    ),
                    specification=(
                        specification
                    ),
                    seed=int(seed),
                )
            )

            seed_results.append(
                seed_result
            )

        ensemble_residual = np.mean(
            np.stack(
                [
                    result[
                        "monitor_residual"
                    ]
                    for result
                    in seed_results
                ],
                axis=0,
            ),
            axis=0,
        )

        specification_results.append({
            "specification_index": int(
                specification_index
            ),
            "specification": dict(
                specification
            ),
            "seed_results": seed_results,
            "ensemble_residual": (
                ensemble_residual
            ),
            "round_counts": [
                int(
                    result[
                        "round_count"
                    ]
                )
                for result
                in seed_results
            ],
            "training_seconds": float(
                sum(
                    result[
                        "training_seconds"
                    ]
                    for result
                    in seed_results
                )
            ),
        })

        completed = (
            specification_index + 1
        )

        if (
            completed % 6 == 0
            or completed
            == len(
                phase38_candidate_specifications
            )
        ):
            print(
                "Phase38 fold "
                f"{fold}: "
                f"{completed}/"
                f"{len(phase38_candidate_specifications)} "
                "specifications complete"
            )

    phase38_selection_folds.append({
        "fold": fold,
        "fit_indices": fit_indices,
        "monitor_indices": monitor_indices,
        "specification_results": (
            specification_results
        ),
    })


# ---------------------------------------------------------
# Construct the pooled monitor contract
# ---------------------------------------------------------

phase38_monitor_indices = np.concatenate([
    record["monitor_indices"]
    for record
    in phase38_selection_folds
])

phase38_monitor_folds = np.concatenate([
    np.full(
        record["monitor_indices"].size,
        int(record["fold"]),
        dtype=np.int64,
    )
    for record
    in phase38_selection_folds
])

phase38_monitor_labels = (
    phase38_labels[
        phase38_monitor_indices
    ]
)

phase38_monitor_groups = (
    phase38_groups[
        phase38_monitor_indices
    ]
)

phase38_monitor_baseline = (
    phase38_baseline_oof[
        phase38_monitor_indices
    ]
)

phase38_monitor_multiplicity = np.bincount(
    phase38_monitor_indices,
    minlength=1362,
)

phase38_monitor_weights = (
    1.0
    / phase38_monitor_multiplicity[
        phase38_monitor_indices
    ]
).astype(np.float64)

assert phase38_monitor_indices.shape == (493,)
assert np.unique(
    phase38_monitor_indices
).size == 407

assert np.array_equal(
    phase38_monitor_indices,
    phase37_monitor_indices,
)

assert np.allclose(
    phase38_monitor_weights,
    phase37_monitor_weights,
)


# ---------------------------------------------------------
# Select one shared raw model specification
# ---------------------------------------------------------

phase38_model_candidate_records = []

for specification_index, specification in enumerate(
    phase38_candidate_specifications
):
    pooled_residual = np.concatenate([
        fold_record[
            "specification_results"
        ][
            specification_index
        ][
            "ensemble_residual"
        ]
        for fold_record
        in phase38_selection_folds
    ])

    pooled_probability = (
        phase37_sigmoid(
            phase37_logit(
                phase38_monitor_baseline
            )
            + pooled_residual
        )
    )

    candidate_metrics = (
        phase38_monitor_metrics(
            probabilities=(
                pooled_probability
            ),
            monitor_labels=(
                phase38_monitor_labels
            ),
            monitor_groups=(
                phase38_monitor_groups
            ),
            monitor_folds=(
                phase38_monitor_folds
            ),
            monitor_indices=(
                phase38_monitor_indices
            ),
            monitor_baseline=(
                phase38_monitor_baseline
            ),
            monitor_weights=(
                phase38_monitor_weights
            ),
        )
    )

    eligible = bool(
        candidate_metrics[
            "log_loss_gain"
        ]
        >= PHASE38_MODEL_CONFIG[
            "minimum_model_monitor_gain"
        ]
        and candidate_metrics[
            "fold_wins"
        ]
        >= PHASE38_MODEL_CONFIG[
            "minimum_model_fold_wins"
        ]
        and candidate_metrics[
            "worst_fold_regret"
        ]
        <= PHASE38_MODEL_CONFIG[
            "maximum_model_fold_regret"
        ]
        and candidate_metrics[
            "maximum_group_harm"
        ]
        <= PHASE38_MODEL_CONFIG[
            "maximum_model_group_harm"
        ]
    )

    phase38_model_candidate_records.append({
        "specification_index": int(
            specification_index
        ),
        "specification": dict(
            specification
        ),
        **candidate_metrics,
        "eligible": eligible,
    })


phase38_eligible_model_records = [
    record
    for record
    in phase38_model_candidate_records
    if record["eligible"]
]

if phase38_eligible_model_records:
    minimum_log_loss = min(
        record["log_loss"]
        for record
        in phase38_eligible_model_records
    )

    near_best_model_records = [
        record
        for record
        in phase38_eligible_model_records
        if (
            record["log_loss"]
            <= minimum_log_loss
            + PHASE38_MODEL_CONFIG[
                "auroc_near_tie_tolerance"
            ]
        )
    ]

    phase38_selected_model_record = sorted(
        near_best_model_records,
        key=lambda record: (
            -record["auroc"],
            record[
                "maximum_group_harm"
            ],
            record[
                "specification"
            ]["maximum_depth"],
            -record[
                "specification"
            ][
                "minimum_child_weight"
            ],
            -record[
                "specification"
            ][
                "l2_regularization"
            ],
        ),
    )[0]

    phase38_model_advanced = True

else:
    phase38_selected_model_record = min(
        phase38_model_candidate_records,
        key=lambda record: (
            record["log_loss"],
            -record["auroc"],
        ),
    )

    phase38_model_advanced = False


phase38_selected_specification_index = int(
    phase38_selected_model_record[
        "specification_index"
    ]
)

phase38_selected_monitor_residual = (
    np.concatenate([
        fold_record[
            "specification_results"
        ][
            phase38_selected_specification_index
        ][
            "ensemble_residual"
        ]
        for fold_record
        in phase38_selection_folds
    ])
)


# ---------------------------------------------------------
# Search the gate only after the model specification is frozen
# ---------------------------------------------------------

phase38_baseline_monitor_case_loss = (
    phase37_case_log_loss(
        phase38_monitor_labels,
        phase38_monitor_baseline,
    )
)

phase38_gate_probability_matrix = (
    np.column_stack([
        phase37_apply_gate(
            phase38_monitor_baseline,
            phase38_selected_monitor_residual,
            gate_specification,
        )
        for gate_specification
        in phase38_gate_specifications
    ])
)

phase38_gate_case_loss = -(
    phase38_monitor_labels[:, None]
    * np.log(
        np.clip(
            phase38_gate_probability_matrix,
            1e-6,
            1.0,
        )
    )
    + (
        1.0
        - phase38_monitor_labels[:, None]
    )
    * np.log(
        np.clip(
            1.0
            - phase38_gate_probability_matrix,
            1e-6,
            1.0,
        )
    )
)

phase38_gate_case_gain = (
    phase38_baseline_monitor_case_loss[
        :,
        None,
    ]
    - phase38_gate_case_loss
)

# Reuse the already generated hierarchical monitor bootstrap.
assert phase37_bootstrap_weights.shape == (
    5000,
    493,
)

phase38_gate_bootstrap_gain = (
    phase37_bootstrap_weights
    @ phase38_gate_case_gain
)

phase38_gate_candidate_records = []

for gate_index, gate_specification in enumerate(
    phase38_gate_specifications
):
    probability = (
        phase38_gate_probability_matrix[
            :,
            gate_index,
        ]
    )

    metrics = phase38_monitor_metrics(
        probabilities=probability,
        monitor_labels=(
            phase38_monitor_labels
        ),
        monitor_groups=(
            phase38_monitor_groups
        ),
        monitor_folds=(
            phase38_monitor_folds
        ),
        monitor_indices=(
            phase38_monitor_indices
        ),
        monitor_baseline=(
            phase38_monitor_baseline
        ),
        monitor_weights=(
            phase38_monitor_weights
        ),
    )

    bootstrap_values = (
        phase38_gate_bootstrap_gain[
            :,
            gate_index,
        ]
    )

    leave_one_group_out_gains = []

    for held_out_group in np.unique(
        phase38_monitor_groups
    ):
        retained = (
            phase38_monitor_groups
            != held_out_group
        )

        leave_one_group_out_gains.append(
            phase37_weighted_mean(
                phase38_gate_case_gain[
                    retained,
                    gate_index,
                ],
                phase38_monitor_weights[
                    retained
                ],
            )
        )

    bootstrap_lcb = float(
        np.quantile(
            bootstrap_values,
            0.05,
        )
    )

    bootstrap_probability_positive = float(
        np.mean(
            bootstrap_values > 0.0
        )
    )

    minimum_logo_gain = float(
        np.min(
            leave_one_group_out_gains
        )
    )

    eligible = bool(
        gate_specification["alpha"] > 0.0
        and metrics["log_loss_gain"]
        >= 0.002
        and metrics["fold_wins"] >= 2
        and metrics["worst_fold_regret"]
        <= 0.003
        and metrics["maximum_group_harm"]
        <= 0.0075
        and bootstrap_lcb >= -0.0005
        and bootstrap_probability_positive
        >= 0.95
        and minimum_logo_gain >= -0.001
    )

    phase38_gate_candidate_records.append({
        "gate_index": int(
            gate_index
        ),
        "specification": dict(
            gate_specification
        ),
        **metrics,
        "bootstrap_mean_gain": float(
            np.mean(
                bootstrap_values
            )
        ),
        "bootstrap_lcb_gain": (
            bootstrap_lcb
        ),
        "bootstrap_probability_positive": (
            bootstrap_probability_positive
        ),
        "minimum_leave_one_group_out_gain": (
            minimum_logo_gain
        ),
        "median_leave_one_group_out_gain": float(
            np.median(
                leave_one_group_out_gains
            )
        ),
        "eligible": eligible,
    })


phase38_eligible_gate_records = [
    record
    for record
    in phase38_gate_candidate_records
    if record["eligible"]
]

phase38_raw_best_gate_record = min(
    [
        record
        for record
        in phase38_gate_candidate_records
        if (
            record[
                "specification"
            ]["alpha"] > 0.0
        )
    ],
    key=lambda record: (
        record["log_loss"],
        -record["auroc"],
    ),
)

if (
    phase38_model_advanced
    and phase38_eligible_gate_records
):
    minimum_gate_loss = min(
        record["log_loss"]
        for record
        in phase38_eligible_gate_records
    )

    near_best_gate_records = [
        record
        for record
        in phase38_eligible_gate_records
        if (
            record["log_loss"]
            <= minimum_gate_loss
            + PHASE38_MODEL_CONFIG[
                "auroc_near_tie_tolerance"
            ]
        )
    ]

    phase38_selected_gate_record = sorted(
        near_best_gate_records,
        key=lambda record: (
            -record["auroc"],
            record[
                "maximum_group_harm"
            ],
            record[
                "specification"
            ]["alpha"],
            record[
                "specification"
            ]["delta_cap"],
        ),
    )[0]

    phase38_gate_advanced = True

else:
    phase38_selected_gate_record = (
        phase38_gate_candidate_records[0]
    )
    phase38_gate_advanced = False


PHASE38_SELECTED_MODEL_SPECIFICATION_PRIVATE = dict(
    phase38_selected_model_record[
        "specification"
    ]
)

PHASE38_SELECTED_MODEL_INDEX_PRIVATE = int(
    phase38_selected_specification_index
)

PHASE38_SELECTED_GATE_PRIVATE = dict(
    phase38_selected_gate_record[
        "specification"
    ]
)

PHASE38_MONITOR_SELECTION_PRIVATE = {
    "model_advanced": bool(
        phase38_model_advanced
    ),
    "gate_advanced": bool(
        phase38_gate_advanced
    ),
    "selected_model_record": (
        phase38_selected_model_record
    ),
    "selected_gate_record": (
        phase38_selected_gate_record
    ),
}

PHASE38_SELECTED_FOLD_RESULTS_PRIVATE = [
    {
        "fold": int(
            fold_record["fold"]
        ),
        "round_counts": list(
            fold_record[
                "specification_results"
            ][
                phase38_selected_specification_index
            ][
                "round_counts"
            ]
        ),
        "monitor_residual": np.asarray(
            fold_record[
                "specification_results"
            ][
                phase38_selected_specification_index
            ][
                "ensemble_residual"
            ],
            dtype=np.float64,
        ),
    }
    for fold_record
    in phase38_selection_folds
]


def phase38_sanitize_record(record):
    return {
        key: value
        for key, value in record.items()
        if key not in (
            "fold_records",
            "group_records",
        )
    }


phase38_selection_report = {
    "phase": (
        "phase38_atlasless_subregion_"
        "anchored_boosting_monitor_selection"
    ),
    "status": (
        "model_and_gate_frozen"
        if phase38_gate_advanced
        else "component_not_advanced"
    ),
    "training": {
        "fold_count": 3,
        "specification_count": int(
            len(
                phase38_candidate_specifications
            )
        ),
        "seed_count": int(
            len(
                PHASE38_MODEL_CONFIG[
                    "seeds"
                ]
            )
        ),
        "trained_model_count": int(
            3
            * len(
                phase38_candidate_specifications
            )
            * len(
                PHASE38_MODEL_CONFIG[
                    "seeds"
                ]
            )
        ),
    },
    "model_selection": {
        "eligible_specification_count": int(
            len(
                phase38_eligible_model_records
            )
        ),
        "model_advanced": bool(
            phase38_model_advanced
        ),
        "selected": (
            phase38_sanitize_record(
                phase38_selected_model_record
            )
        ),
    },
    "gate_selection": {
        "candidate_count": int(
            len(
                phase38_gate_candidate_records
            )
        ),
        "eligible_candidate_count": int(
            len(
                phase38_eligible_gate_records
            )
        ),
        "gate_advanced": bool(
            phase38_gate_advanced
        ),
        "raw_best": (
            phase38_sanitize_record(
                phase38_raw_best_gate_record
            )
        ),
        "selected": (
            phase38_sanitize_record(
                phase38_selected_gate_record
            )
        ),
    },
    "selected_fold_round_counts": [
        {
            "fold": int(
                record["fold"]
            ),
            "round_counts": list(
                record["round_counts"]
            ),
        }
        for record
        in PHASE38_SELECTED_FOLD_RESULTS_PRIVATE
    ],
    "shared_model_specification_across_folds": True,
    "shared_gate_across_folds": True,
    "fold_specific_group_routing": False,
    "selection_partition": (
        "pooled_monitor_only"
    ),
    "outer_validation_images_used_for_preprocessing": False,
    "outer_validation_labels_used": False,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase38_selection_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE38_SELECTION"
)
print(
    json.dumps(
        phase38_selection_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE38_SELECTION"
)

Phase38 fold 0: 6/72 specifications complete
Phase38 fold 0: 12/72 specifications complete
Phase38 fold 0: 18/72 specifications complete
Phase38 fold 0: 24/72 specifications complete
Phase38 fold 0: 30/72 specifications complete
Phase38 fold 0: 36/72 specifications complete
Phase38 fold 0: 42/72 specifications complete
Phase38 fold 0: 48/72 specifications complete
Phase38 fold 0: 54/72 specifications complete
Phase38 fold 0: 60/72 specifications complete
Phase38 fold 0: 66/72 specifications complete
Phase38 fold 0: 72/72 specifications complete
Phase38 fold 1: 6/72 specifications complete
Phase38 fold 1: 12/72 specifications complete
Phase38 fold 1: 18/72 specifications complete
Phase38 fold 1: 24/72 specifications complete
Phase38 fold 1: 30/72 specifications complete
Phase38 fold 1: 36/72 specifications complete
Phase38 fold 1: 42/72 specifications complete
Phase38 fold 1: 48/72 specifications complete
Phase38 fold 1: 54/72 specifications complete
Phase38 fold 1: 60/72 specifications

In [78]:
# Phase38 Cell 121C-preflight patch v2
# Resolve endpoint_core from the complete 170-column feature matrix.

import numpy as np

phase38_complete_matrix = np.asarray(
    phase38_feature_matrix,
    dtype=np.float32,
)

assert phase38_complete_matrix.shape == (
    1362,
    170,
)

assert np.isfinite(
    phase38_complete_matrix
).all()


def phase38_find_feature_names():
    candidate_names = [
        "PHASE38_FEATURE_NAMES",
        "PHASE38_FEATURE_NAMES_PRIVATE",
        "phase38_feature_names",
    ]

    for global_name in candidate_names:
        value = globals().get(global_name)

        if value is None:
            continue

        names = list(value)

        if (
            len(names) == 170
            and len(set(names)) == 170
            and all(
                isinstance(name, str)
                for name in names
            )
        ):
            return names, global_name

    for metadata_name in [
        "phase38_metadata",
        "PHASE38_METADATA_PRIVATE",
    ]:
        metadata = globals().get(
            metadata_name
        )

        if not isinstance(metadata, dict):
            continue

        names = metadata.get("feature_names")

        if names is not None and len(names) == 170:
            return list(names), (
                f"{metadata_name}.feature_names"
            )

    raise AssertionError(
        "Could not resolve the 170 Phase38 feature names."
    )


(
    PHASE38_FEATURE_NAMES_RESOLVED,
    phase38_feature_name_source,
) = phase38_find_feature_names()


def phase38_matrix_from_representation_object(
    value,
):
    # Direct 1362 x 95 matrix.
    if isinstance(value, np.ndarray):
        array = np.asarray(value)

        if array.shape == (1362, 95):
            return array.astype(np.float32)

        # Boolean mask over the 170 features.
        if (
            array.ndim == 1
            and array.shape == (170,)
            and array.dtype == bool
            and int(np.sum(array)) == 95
        ):
            return phase38_complete_matrix[
                :,
                array,
            ]

        # Integer column indices.
        if (
            array.ndim == 1
            and array.shape == (95,)
            and np.issubdtype(
                array.dtype,
                np.integer,
            )
        ):
            return phase38_complete_matrix[
                :,
                array.astype(np.int64),
            ]

    if isinstance(value, (list, tuple)):
        # Integer indices.
        if (
            len(value) == 95
            and all(
                isinstance(
                    item,
                    (int, np.integer),
                )
                for item in value
            )
        ):
            return phase38_complete_matrix[
                :,
                np.asarray(
                    value,
                    dtype=np.int64,
                ),
            ]

        # Feature names.
        if (
            len(value) == 95
            and all(
                isinstance(item, str)
                for item in value
            )
        ):
            name_to_index = {
                name: index
                for index, name
                in enumerate(
                    PHASE38_FEATURE_NAMES_RESOLVED
                )
            }

            if all(
                name in name_to_index
                for name in value
            ):
                indices = np.asarray(
                    [
                        name_to_index[name]
                        for name in value
                    ],
                    dtype=np.int64,
                )

                return phase38_complete_matrix[
                    :,
                    indices,
                ]

    if isinstance(value, dict):
        for key in [
            "matrix",
            "features",
            "indices",
            "columns",
            "mask",
            "feature_names",
        ]:
            if key in value:
                result = (
                    phase38_matrix_from_representation_object(
                        value[key]
                    )
                )

                if result is not None:
                    return result

    return None


# --------------------------------------------------------------
# Route 1: find an existing endpoint_core representation object.
# --------------------------------------------------------------

phase38_endpoint_candidates = []

for global_name, value in list(
    globals().items()
):
    if not isinstance(value, dict):
        continue

    if "endpoint_core" not in value:
        continue

    matrix = (
        phase38_matrix_from_representation_object(
            value["endpoint_core"]
        )
    )

    if matrix is not None:
        phase38_endpoint_candidates.append(
            (
                f"{global_name}['endpoint_core']",
                matrix,
            )
        )


# --------------------------------------------------------------
# Route 2: use an existing Phase38 representation builder.
# --------------------------------------------------------------

for global_name, value in list(
    globals().items()
):
    lowered_name = global_name.lower()

    if not callable(value):
        continue

    if "phase38" not in lowered_name:
        continue

    if not any(
        term in lowered_name
        for term in [
            "representation",
            "feature_matrix",
            "feature_subset",
        ]
    ):
        continue

    if any(
        term in lowered_name
        for term in [
            "fit",
            "train",
            "evaluate",
            "audit",
        ]
    ):
        continue

    call_attempts = [
        lambda function=value: function(
            "endpoint_core"
        ),
        lambda function=value: function(
            phase38_complete_matrix,
            "endpoint_core",
        ),
        lambda function=value: function(
            "endpoint_core",
            phase38_complete_matrix,
        ),
    ]

    for attempt in call_attempts:
        try:
            candidate = attempt()
            matrix = (
                phase38_matrix_from_representation_object(
                    candidate
                )
            )

            if matrix is not None:
                phase38_endpoint_candidates.append(
                    (global_name, matrix)
                )
                break

        except Exception:
            pass


# --------------------------------------------------------------
# Route 3: recover it from the feature-family classifier.
# --------------------------------------------------------------

target_families = {
    "lower_uptake_endpoint",
    "higher_uptake_endpoint",
    "within_structure_endpoint_contrast",
    "whole_structure_uptake",
}

family_function_candidates = []

for global_name, value in list(
    globals().items()
):
    lowered_name = global_name.lower()

    if (
        callable(value)
        and "phase38" in lowered_name
        and "family" in lowered_name
        and "train" not in lowered_name
        and "model" not in lowered_name
    ):
        family_function_candidates.append(
            (global_name, value)
        )

for global_name, family_function in (
    family_function_candidates
):
    try:
        families = [
            family_function(feature_name)
            for feature_name
            in PHASE38_FEATURE_NAMES_RESOLVED
        ]

        selected_indices = np.asarray(
            [
                index
                for index, family
                in enumerate(families)
                if family in target_families
            ],
            dtype=np.int64,
        )

        if selected_indices.shape == (95,):
            family_counts = {
                family: int(
                    sum(
                        value == family
                        for value in families
                    )
                )
                for family in sorted(
                    target_families
                )
            }

            expected_family_counts = {
                "higher_uptake_endpoint": 25,
                "lower_uptake_endpoint": 35,
                "whole_structure_uptake": 30,
                "within_structure_endpoint_contrast": 5,
            }

            if family_counts == expected_family_counts:
                phase38_endpoint_candidates.append(
                    (
                        global_name,
                        phase38_complete_matrix[
                            :,
                            selected_indices,
                        ],
                    )
                )

    except Exception:
        pass


assert phase38_endpoint_candidates, {
    "message": (
        "Could not reconstruct endpoint_core from "
        "existing Phase38 state."
    ),
    "feature_name_source": (
        phase38_feature_name_source
    ),
    "family_function_candidates": [
        name
        for name, _
        in family_function_candidates
    ],
}

phase38_endpoint_source = (
    phase38_endpoint_candidates[0][0]
)

PHASE38_ENDPOINT_CORE_MATRIX_PRIVATE = (
    np.asarray(
        phase38_endpoint_candidates[0][1],
        dtype=np.float32,
    )
)

assert PHASE38_ENDPOINT_CORE_MATRIX_PRIVATE.shape == (
    1362,
    95,
)

assert np.isfinite(
    PHASE38_ENDPOINT_CORE_MATRIX_PRIVATE
).all()

# Every successful route must reconstruct the same representation.
for source_name, candidate_matrix in (
    phase38_endpoint_candidates[1:]
):
    assert candidate_matrix.shape == (1362, 95)

    maximum_difference = float(
        np.max(
            np.abs(
                np.asarray(
                    candidate_matrix,
                    dtype=np.float32,
                )
                - PHASE38_ENDPOINT_CORE_MATRIX_PRIVATE
            )
        )
    )

    assert maximum_difference <= 1e-7, {
        "message": (
            "Different Phase38 state objects produced "
            "different endpoint_core matrices."
        ),
        "first_source": phase38_endpoint_source,
        "conflicting_source": source_name,
        "maximum_absolute_difference": (
            maximum_difference
        ),
    }


# Preserve the full matrix separately, then expose the callable
# interface expected by Cell 121C.
PHASE38_ALL_FEATURE_MATRIX_PRIVATE = (
    phase38_complete_matrix
)


def phase38_feature_matrix(representation):
    if representation == "endpoint_core":
        return (
            PHASE38_ENDPOINT_CORE_MATRIX_PRIVATE
        )

    if representation == "all_features":
        return PHASE38_ALL_FEATURE_MATRIX_PRIVATE

    raise AssertionError({
        "message": (
            "Only the frozen endpoint_core representation "
            "is permitted in the Phase38 outer evaluation."
        ),
        "received": representation,
    })


phase38_endpoint_preflight = (
    phase38_feature_matrix("endpoint_core")
)

print(
    "Phase38 endpoint_core resolver accepted: "
    f"shape={phase38_endpoint_preflight.shape}, "
    f"source={phase38_endpoint_source}, "
    f"candidate_routes="
    f"{len(phase38_endpoint_candidates)}"
)

Phase38 endpoint_core resolver accepted: shape=(1362, 95), source=phase38_representation_indices, candidate_routes=1


In [79]:
# Phase38 Cell 121C
# Frozen outer-train refit and one-time outer-validation evaluation.
#
# Required state:
#   PHASE38_SELECTED_MODEL_SPECIFICATION_PRIVATE
#   PHASE38_SELECTED_GATE_PRIVATE
#   PHASE38_FEATURES_PRIVATE
#   PHASE19_PARTITIONS
#   PHASE19_PHASE12C_OOF_PRIVATE
#   phase38_feature_matrix
#   phase38_fit_preprocessor
#   phase38_transform_preprocessor
#
# This cell does not perform any additional hyperparameter selection.

from pathlib import Path
import inspect
import json
import math
import time

import joblib
import numpy as np
import xgboost as xgb

from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score


# ------------------------------------------------------------------
# 1. Freeze and verify the selected specification
# ------------------------------------------------------------------

phase38_outer_started = time.perf_counter()

phase38_selected_specification = dict(
    PHASE38_SELECTED_MODEL_SPECIFICATION_PRIVATE
)
phase38_selected_gate = dict(
    PHASE38_SELECTED_GATE_PRIVATE
)

expected_specification = {
    "representation": "endpoint_core",
    "maximum_depth": 2,
    "minimum_child_weight": 16.0,
    "l2_regularization": 30.0,
    "column_subsample": 1.0,
}

expected_gate = {
    "alpha": 0.75,
    "uncertainty_exponent": 0.0,
    "delta_cap": 1.5,
}

assert phase38_selected_specification == expected_specification, {
    "message": "Frozen Phase38 specification changed.",
    "expected": expected_specification,
    "received": phase38_selected_specification,
}

assert phase38_selected_gate == expected_gate, {
    "message": "Frozen Phase38 gate changed.",
    "expected": expected_gate,
    "received": phase38_selected_gate,
}

# These round counts were selected using fit -> monitor only.
PHASE38_FROZEN_ROUND_COUNTS = {
    0: [62, 85],
    1: [227, 189],
    2: [1, 1],
}

PHASE38_FROZEN_SEEDS = [
    380701,
    380702,
]


# ------------------------------------------------------------------
# 2. Resolve labels, acquisition groups and exact Phase12c anchor
# ------------------------------------------------------------------

def phase38_resolve_labels():
    candidate_names = [
        "PHASE32_LABELS_PRIVATE",
        "PHASE31_LABELS_PRIVATE",
        "PHASE19_LABELS_PRIVATE",
        "labels",
        "label",
        "y",
    ]

    for name in candidate_names:
        value = globals().get(name)

        if value is None:
            continue

        array = np.asarray(value).reshape(-1)

        if (
            array.shape == (1362,)
            and np.isin(array, [0, 1]).all()
        ):
            return array.astype(np.int64), name

    if "case_df" in globals():
        for column in ["label", "target", "y"]:
            if column in case_df.columns:
                array = np.asarray(
                    case_df[column]
                ).reshape(-1)

                if (
                    array.shape == (1362,)
                    and np.isin(array, [0, 1]).all()
                ):
                    return (
                        array.astype(np.int64),
                        f"case_df.{column}",
                    )

    raise AssertionError(
        "Could not resolve the 1362 binary labels."
    )


def phase38_resolve_groups():
    candidate_names = [
        "PHASE32_ACQUISITION_GROUPS_PRIVATE",
        "PHASE31_ACQUISITION_GROUPS_PRIVATE",
        "PHASE19_ACQUISITION_GROUPS_PRIVATE",
        "acquisition_group",
        "acquisition_groups",
        "groups",
    ]

    for name in candidate_names:
        value = globals().get(name)

        if value is None:
            continue

        array = np.asarray(value).reshape(-1)

        if array.shape == (1362,):
            return array, name

    if (
        "case_df" in globals()
        and "acquisition_group" in case_df.columns
    ):
        return (
            np.asarray(
                case_df["acquisition_group"]
            ).reshape(-1),
            "case_df.acquisition_group",
        )

    raise AssertionError(
        "Could not resolve acquisition groups."
    )


PHASE38_LABELS_PRIVATE, phase38_label_source = (
    phase38_resolve_labels()
)

(
    PHASE38_ACQUISITION_GROUPS_PRIVATE,
    phase38_group_source,
) = phase38_resolve_groups()

PHASE38_BASELINE_OOF_PRIVATE = np.asarray(
    PHASE19_PHASE12C_OOF_PRIVATE,
    dtype=np.float64,
).reshape(-1)

assert PHASE38_BASELINE_OOF_PRIVATE.shape == (
    1362,
)

assert np.isfinite(
    PHASE38_BASELINE_OOF_PRIVATE
).all()

assert (
    (PHASE38_BASELINE_OOF_PRIVATE > 0.0).all()
    and
    (PHASE38_BASELINE_OOF_PRIVATE < 1.0).all()
)

phase38_baseline_logit = logit(
    np.clip(
        PHASE38_BASELINE_OOF_PRIVATE,
        1e-6,
        1.0 - 1e-6,
    )
)


# ------------------------------------------------------------------
# 3. Compatibility wrapper for the existing Phase38 preprocessor
# ------------------------------------------------------------------

def phase38_extract_transformed_array(value, expected_rows):
    candidates = []

    if isinstance(value, np.ndarray):
        candidates.append(value)

    elif isinstance(value, (tuple, list)):
        candidates.extend(
            item
            for item in value
            if isinstance(item, np.ndarray)
        )

    for candidate in candidates:
        candidate = np.asarray(
            candidate,
            dtype=np.float32,
        )

        if (
            candidate.ndim == 2
            and candidate.shape[0] == expected_rows
            and candidate.shape[1] > 0
            and np.isfinite(candidate).all()
        ):
            return candidate

    return None


def phase38_extract_preprocessor_state(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, (tuple, list)):
        for item in value:
            if isinstance(item, dict):
                return item

        for item in value:
            if not isinstance(item, np.ndarray):
                return item

    return value


def phase38_fit_transform_compat(
    train_indices,
    prediction_indices,
    representation,
):
    train_indices = np.asarray(
        train_indices,
        dtype=np.int64,
    )
    prediction_indices = np.asarray(
        prediction_indices,
        dtype=np.int64,
    )

    full_matrix = np.asarray(
        phase38_feature_matrix(
            representation
        ),
        dtype=np.float32,
    )

    assert full_matrix.shape[0] == 1362
    assert np.isfinite(full_matrix).all()

    train_matrix = full_matrix[train_indices]
    prediction_matrix = full_matrix[
        prediction_indices
    ]

    fit_attempts = [
        (
            "indices_representation",
            lambda: phase38_fit_preprocessor(
                train_indices,
                representation,
            ),
        ),
        (
            "representation_indices",
            lambda: phase38_fit_preprocessor(
                representation,
                train_indices,
            ),
        ),
        (
            "subset_representation",
            lambda: phase38_fit_preprocessor(
                train_matrix,
                representation,
            ),
        ),
        (
            "subset_only",
            lambda: phase38_fit_preprocessor(
                train_matrix
            ),
        ),
        (
            "full_indices_representation",
            lambda: phase38_fit_preprocessor(
                full_matrix,
                train_indices,
                representation,
            ),
        ),
    ]

    errors = []

    for fit_name, fit_call in fit_attempts:
        try:
            fit_output = fit_call()
            state = phase38_extract_preprocessor_state(
                fit_output
            )

            transform_attempts = [
                (
                    "state_indices",
                    lambda: phase38_transform_preprocessor(
                        state,
                        prediction_indices,
                    ),
                ),
                (
                    "indices_state",
                    lambda: phase38_transform_preprocessor(
                        prediction_indices,
                        state,
                    ),
                ),
                (
                    "state_subset",
                    lambda: phase38_transform_preprocessor(
                        state,
                        prediction_matrix,
                    ),
                ),
                (
                    "subset_state",
                    lambda: phase38_transform_preprocessor(
                        prediction_matrix,
                        state,
                    ),
                ),
                (
                    "state_full_indices",
                    lambda: phase38_transform_preprocessor(
                        state,
                        full_matrix,
                        prediction_indices,
                    ),
                ),
            ]

            for transform_name, transform_call in (
                transform_attempts
            ):
                try:
                    transformed_output = (
                        transform_call()
                    )

                    transformed = (
                        phase38_extract_transformed_array(
                            transformed_output,
                            len(prediction_indices),
                        )
                    )

                    if transformed is not None:
                        return (
                            state,
                            transformed,
                            fit_name,
                            transform_name,
                        )

                except Exception as exc:
                    errors.append({
                        "fit": fit_name,
                        "transform": transform_name,
                        "error": type(exc).__name__,
                    })

        except Exception as exc:
            errors.append({
                "fit": fit_name,
                "transform": None,
                "error": type(exc).__name__,
            })

    raise AssertionError({
        "message": (
            "Could not call the existing Phase38 "
            "preprocessing functions."
        ),
        "fit_signature": str(
            inspect.signature(
                phase38_fit_preprocessor
            )
        ),
        "transform_signature": str(
            inspect.signature(
                phase38_transform_preprocessor
            )
        ),
        "attempts": errors,
    })


def phase38_transform_with_state_compat(
    state,
    prediction_indices,
    representation,
):
    prediction_indices = np.asarray(
        prediction_indices,
        dtype=np.int64,
    )

    full_matrix = np.asarray(
        phase38_feature_matrix(
            representation
        ),
        dtype=np.float32,
    )

    prediction_matrix = full_matrix[
        prediction_indices
    ]

    attempts = [
        lambda: phase38_transform_preprocessor(
            state,
            prediction_indices,
        ),
        lambda: phase38_transform_preprocessor(
            prediction_indices,
            state,
        ),
        lambda: phase38_transform_preprocessor(
            state,
            prediction_matrix,
        ),
        lambda: phase38_transform_preprocessor(
            prediction_matrix,
            state,
        ),
        lambda: phase38_transform_preprocessor(
            state,
            full_matrix,
            prediction_indices,
        ),
    ]

    errors = []

    for attempt in attempts:
        try:
            output = attempt()
            transformed = (
                phase38_extract_transformed_array(
                    output,
                    len(prediction_indices),
                )
            )

            if transformed is not None:
                return transformed

        except Exception as exc:
            errors.append(type(exc).__name__)

    raise AssertionError({
        "message": (
            "Could not transform using the fitted "
            "Phase38 preprocessor."
        ),
        "errors": errors,
    })


# ------------------------------------------------------------------
# 4. Fixed training utilities
# ------------------------------------------------------------------

def phase38_config_value(
    candidate_names,
    default,
):
    configuration = globals().get(
        "PHASE38_MODEL_CONFIG",
        {},
    )

    for name in candidate_names:
        if name in configuration:
            return configuration[name]

    return default


def phase38_outer_group_weights(indices):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    # Prefer the exact helper used during selection.
    helper = globals().get(
        "phase38_group_balanced_weights"
    )

    if callable(helper):
        for value in [
            indices,
            PHASE38_ACQUISITION_GROUPS_PRIVATE[
                indices
            ],
        ]:
            try:
                weights = np.asarray(
                    helper(value),
                    dtype=np.float64,
                ).reshape(-1)

                if (
                    weights.shape == indices.shape
                    and np.isfinite(weights).all()
                    and (weights > 0).all()
                ):
                    return (
                        weights / np.mean(weights)
                    ).astype(np.float32)

            except Exception:
                pass

    groups = (
        PHASE38_ACQUISITION_GROUPS_PRIVATE[
            indices
        ]
    )

    _, inverse, counts = np.unique(
        groups,
        return_inverse=True,
        return_counts=True,
    )

    weights = 1.0 / counts[inverse]
    weights /= np.mean(weights)

    return weights.astype(np.float32)


def phase38_xgb_parameters(
    specification,
    seed,
):
    return {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "eta": float(
            phase38_config_value(
                ["learning_rate", "eta"],
                0.03,
            )
        ),
        "max_depth": int(
            specification["maximum_depth"]
        ),
        "min_child_weight": float(
            specification[
                "minimum_child_weight"
            ]
        ),
        "lambda": float(
            specification[
                "l2_regularization"
            ]
        ),
        "alpha": 0.0,
        "subsample": float(
            phase38_config_value(
                ["subsample", "row_subsample"],
                0.85,
            )
        ),
        "colsample_bytree": float(
            specification[
                "column_subsample"
            ]
        ),
        "max_bin": int(
            phase38_config_value(
                [
                    "maximum_bin_count",
                    "max_bin",
                ],
                128,
            )
        ),
        "max_delta_step": float(
            phase38_config_value(
                ["maximum_delta_step"],
                1.0,
            )
        ),
        "seed": int(seed),
        "nthread": int(
            phase38_config_value(
                ["nthread", "thread_count"],
                8,
            )
        ),
        "verbosity": 0,
    }


def phase38_apply_frozen_gate(
    baseline_logit,
    residual,
    gate,
):
    baseline_logit = np.asarray(
        baseline_logit,
        dtype=np.float64,
    )
    residual = np.asarray(
        residual,
        dtype=np.float64,
    )

    baseline_probability = expit(
        baseline_logit
    )

    uncertainty = (
        4.0
        * baseline_probability
        * (1.0 - baseline_probability)
    )

    effective_alpha = (
        float(gate["alpha"])
        * np.power(
            np.clip(
                uncertainty,
                0.0,
                1.0,
            ),
            float(
                gate[
                    "uncertainty_exponent"
                ]
            ),
        )
    )

    bounded_residual = np.clip(
        residual,
        -float(gate["delta_cap"]),
        float(gate["delta_cap"]),
    )

    final_logit = (
        baseline_logit
        + effective_alpha
        * bounded_residual
    )

    return (
        final_logit,
        expit(final_logit),
        effective_alpha,
        bounded_residual,
    )


def phase38_binary_metrics(labels, probabilities):
    labels = np.asarray(labels).astype(
        np.int64
    )
    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    return {
        "log_loss": float(
            log_loss(labels, probabilities)
        ),
        "auroc": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "mean_probability": float(
            np.mean(probabilities)
        ),
    }


# ------------------------------------------------------------------
# 5. Outer-train refit and one-time outer-valid prediction
# ------------------------------------------------------------------

phase38_case_count = len(
    PHASE38_LABELS_PRIVATE
)

PHASE38_RAW_RESIDUAL_OOF_PRIVATE = np.full(
    phase38_case_count,
    np.nan,
    dtype=np.float64,
)

PHASE38_RAW_COMPONENT_OOF_PRIVATE = np.full(
    phase38_case_count,
    np.nan,
    dtype=np.float64,
)

PHASE38_OOF_PRIVATE = np.full(
    phase38_case_count,
    np.nan,
    dtype=np.float64,
)

PHASE38_EFFECTIVE_ALPHA_OOF_PRIVATE = np.full(
    phase38_case_count,
    np.nan,
    dtype=np.float64,
)

phase38_outer_coverage = np.zeros(
    phase38_case_count,
    dtype=np.int64,
)

PHASE38_DEPLOYMENT_STATES_PRIVATE = []
phase38_fold_reports = []

for fold in range(3):
    fold_started = time.perf_counter()

    partition = PHASE19_PARTITIONS[fold]

    outer_train_indices = np.asarray(
        partition["outer_train"],
        dtype=np.int64,
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    (
        preprocessor_state,
        outer_valid_features,
        preprocessor_fit_contract,
        preprocessor_transform_contract,
    ) = phase38_fit_transform_compat(
        train_indices=outer_train_indices,
        prediction_indices=outer_valid_indices,
        representation=(
            phase38_selected_specification[
                "representation"
            ]
        ),
    )

    outer_train_features = (
        phase38_transform_with_state_compat(
            state=preprocessor_state,
            prediction_indices=(
                outer_train_indices
            ),
            representation=(
                phase38_selected_specification[
                    "representation"
                ]
            ),
        )
    )

    assert (
        outer_train_features.shape[1]
        == outer_valid_features.shape[1]
    )

    train_labels = (
        PHASE38_LABELS_PRIVATE[
            outer_train_indices
        ]
    )

    train_base_margin = (
        phase38_baseline_logit[
            outer_train_indices
        ]
    )

    valid_base_margin = (
        phase38_baseline_logit[
            outer_valid_indices
        ]
    )

    train_weights = (
        phase38_outer_group_weights(
            outer_train_indices
        )
    )

    seed_residuals = []
    seed_models = []

    fold_round_counts = (
        PHASE38_FROZEN_ROUND_COUNTS[fold]
    )

    assert len(fold_round_counts) == len(
        PHASE38_FROZEN_SEEDS
    )

    for seed, round_count in zip(
        PHASE38_FROZEN_SEEDS,
        fold_round_counts,
    ):
        dtrain = xgb.DMatrix(
            outer_train_features,
            label=train_labels,
            weight=train_weights,
            base_margin=train_base_margin,
        )

        booster = xgb.train(
            params=phase38_xgb_parameters(
                phase38_selected_specification,
                seed,
            ),
            dtrain=dtrain,
            num_boost_round=int(round_count),
            verbose_eval=False,
        )

        dvalid = xgb.DMatrix(
            outer_valid_features,
            base_margin=valid_base_margin,
        )

        predicted_margin = np.asarray(
            booster.predict(
                dvalid,
                output_margin=True,
            ),
            dtype=np.float64,
        )

        seed_residual = (
            predicted_margin
            - valid_base_margin
        )

        assert seed_residual.shape == (
            len(outer_valid_indices),
        )
        assert np.isfinite(
            seed_residual
        ).all()

        seed_residuals.append(
            seed_residual
        )
        seed_models.append(booster)

    ensemble_residual = np.mean(
        np.stack(
            seed_residuals,
            axis=0,
        ),
        axis=0,
    )

    raw_component_probability = expit(
        valid_base_margin
        + ensemble_residual
    )

    (
        final_logit,
        final_probability,
        effective_alpha,
        bounded_residual,
    ) = phase38_apply_frozen_gate(
        baseline_logit=valid_base_margin,
        residual=ensemble_residual,
        gate=phase38_selected_gate,
    )

    PHASE38_RAW_RESIDUAL_OOF_PRIVATE[
        outer_valid_indices
    ] = ensemble_residual

    PHASE38_RAW_COMPONENT_OOF_PRIVATE[
        outer_valid_indices
    ] = raw_component_probability

    PHASE38_OOF_PRIVATE[
        outer_valid_indices
    ] = final_probability

    PHASE38_EFFECTIVE_ALPHA_OOF_PRIVATE[
        outer_valid_indices
    ] = effective_alpha

    phase38_outer_coverage[
        outer_valid_indices
    ] += 1

    fold_labels = (
        PHASE38_LABELS_PRIVATE[
            outer_valid_indices
        ]
    )

    baseline_fold_metrics = (
        phase38_binary_metrics(
            fold_labels,
            PHASE38_BASELINE_OOF_PRIVATE[
                outer_valid_indices
            ],
        )
    )

    raw_fold_metrics = (
        phase38_binary_metrics(
            fold_labels,
            raw_component_probability,
        )
    )

    gated_fold_metrics = (
        phase38_binary_metrics(
            fold_labels,
            final_probability,
        )
    )

    fold_report = {
        "fold": fold,
        "n": int(
            len(outer_valid_indices)
        ),
        "round_counts": [
            int(value)
            for value in fold_round_counts
        ],
        "transformed_feature_count": int(
            outer_train_features.shape[1]
        ),
        "baseline_log_loss": (
            baseline_fold_metrics[
                "log_loss"
            ]
        ),
        "raw_component_log_loss": (
            raw_fold_metrics["log_loss"]
        ),
        "phase38_log_loss": (
            gated_fold_metrics["log_loss"]
        ),
        "log_loss_improvement": float(
            baseline_fold_metrics[
                "log_loss"
            ]
            - gated_fold_metrics[
                "log_loss"
            ]
        ),
        "baseline_auroc": (
            baseline_fold_metrics["auroc"]
        ),
        "raw_component_auroc": (
            raw_fold_metrics["auroc"]
        ),
        "phase38_auroc": (
            gated_fold_metrics["auroc"]
        ),
        "auroc_improvement": float(
            gated_fold_metrics["auroc"]
            - baseline_fold_metrics[
                "auroc"
            ]
        ),
        "mean_absolute_raw_residual": float(
            np.mean(
                np.abs(
                    ensemble_residual
                )
            )
        ),
        "mean_absolute_bounded_residual": float(
            np.mean(
                np.abs(
                    bounded_residual
                )
            )
        ),
        "elapsed_seconds": float(
            time.perf_counter()
            - fold_started
        ),
    }

    phase38_fold_reports.append(
        fold_report
    )

    PHASE38_DEPLOYMENT_STATES_PRIVATE.append({
        "fold": fold,
        "specification": dict(
            phase38_selected_specification
        ),
        "gate": dict(
            phase38_selected_gate
        ),
        "round_counts": [
            int(value)
            for value in fold_round_counts
        ],
        "seeds": list(
            PHASE38_FROZEN_SEEDS
        ),
        "preprocessor_state": (
            preprocessor_state
        ),
        "preprocessor_fit_contract": (
            preprocessor_fit_contract
        ),
        "preprocessor_transform_contract": (
            preprocessor_transform_contract
        ),
        "boosters": seed_models,
    })

    print(
        f"Phase38 outer fold {fold}: "
        f"log_loss="
        f"{gated_fold_metrics['log_loss']:.6f}, "
        f"gain="
        f"{fold_report['log_loss_improvement']:.6f}, "
        f"auroc="
        f"{gated_fold_metrics['auroc']:.6f}"
    )


assert np.all(
    phase38_outer_coverage == 1
), {
    "message": (
        "Outer-validation coverage is not exact."
    ),
    "coverage_counts": {
        int(value): int(
            np.sum(
                phase38_outer_coverage
                == value
            )
        )
        for value in np.unique(
            phase38_outer_coverage
        )
    },
}

for array in [
    PHASE38_RAW_RESIDUAL_OOF_PRIVATE,
    PHASE38_RAW_COMPONENT_OOF_PRIVATE,
    PHASE38_OOF_PRIVATE,
    PHASE38_EFFECTIVE_ALPHA_OOF_PRIVATE,
]:
    assert np.isfinite(array).all()


# ------------------------------------------------------------------
# 6. Aggregate evaluation
# ------------------------------------------------------------------

phase38_baseline_metrics = (
    phase38_binary_metrics(
        PHASE38_LABELS_PRIVATE,
        PHASE38_BASELINE_OOF_PRIVATE,
    )
)

phase38_raw_metrics = (
    phase38_binary_metrics(
        PHASE38_LABELS_PRIVATE,
        PHASE38_RAW_COMPONENT_OOF_PRIVATE,
    )
)

phase38_final_metrics = (
    phase38_binary_metrics(
        PHASE38_LABELS_PRIVATE,
        PHASE38_OOF_PRIVATE,
    )
)

phase38_fold_wins = int(
    sum(
        report[
            "log_loss_improvement"
        ] > 0.0
        for report in phase38_fold_reports
    )
)

phase38_worst_fold_excess = float(
    max(
        0.0,
        max(
            -report[
                "log_loss_improvement"
            ]
            for report
            in phase38_fold_reports
        ),
    )
)


# Major acquisition groups only.
phase38_major_group_reports = []
phase38_maximum_major_group_harm = 0.0

for group_value in np.unique(
    PHASE38_ACQUISITION_GROUPS_PRIVATE
):
    mask = (
        PHASE38_ACQUISITION_GROUPS_PRIVATE
        == group_value
    )
    group_n = int(np.sum(mask))

    if group_n < 30:
        continue

    group_labels = (
        PHASE38_LABELS_PRIVATE[mask]
    )

    baseline_group_loss = float(
        log_loss(
            group_labels,
            PHASE38_BASELINE_OOF_PRIVATE[
                mask
            ],
        )
    )

    candidate_group_loss = float(
        log_loss(
            group_labels,
            PHASE38_OOF_PRIVATE[mask],
        )
    )

    change = float(
        candidate_group_loss
        - baseline_group_loss
    )

    phase38_maximum_major_group_harm = max(
        phase38_maximum_major_group_harm,
        change,
    )

    record = {
        "group": (
            int(group_value)
            if np.issubdtype(
                np.asarray(
                    group_value
                ).dtype,
                np.integer,
            )
            else str(group_value)
        ),
        "n": group_n,
        "baseline_log_loss": (
            baseline_group_loss
        ),
        "phase38_log_loss": (
            candidate_group_loss
        ),
        "log_loss_change": change,
    }

    if np.unique(group_labels).size == 2:
        record["baseline_auroc"] = float(
            roc_auc_score(
                group_labels,
                PHASE38_BASELINE_OOF_PRIVATE[
                    mask
                ],
            )
        )
        record["phase38_auroc"] = float(
            roc_auc_score(
                group_labels,
                PHASE38_OOF_PRIVATE[mask],
            )
        )

    phase38_major_group_reports.append(
        record
    )


# Aggregate calibration diagnostic.
phase38_calibration_model = LogisticRegression(
    C=1e6,
    solver="lbfgs",
    max_iter=2000,
)

phase38_calibration_model.fit(
    logit(
        np.clip(
            PHASE38_OOF_PRIVATE,
            1e-6,
            1.0 - 1e-6,
        )
    ).reshape(-1, 1),
    PHASE38_LABELS_PRIVATE,
)

phase38_calibration_intercept = float(
    phase38_calibration_model.intercept_[0]
)

phase38_calibration_slope = float(
    phase38_calibration_model.coef_[0, 0]
)


# Optional comparisons, used only when exact private vectors exist.
phase38_reference_comparisons = {}

for reference_name, global_names in {
    "phase37": [
        "PHASE37_OOF_PRIVATE",
        "phase37_oof",
    ],
    "phase36": [
        "PHASE36_OOF_PRIVATE",
        "phase36_oof",
    ],
    "phase33": [
        "PHASE33_GATED_OOF_PRIVATE",
        "PHASE33_OOF_PRIVATE",
        "phase33_gated_oof",
    ],
}.items():
    reference_vector = None
    source_name = None

    for global_name in global_names:
        candidate = globals().get(
            global_name
        )

        if candidate is None:
            continue

        candidate = np.asarray(
            candidate,
            dtype=np.float64,
        ).reshape(-1)

        if (
            candidate.shape == (1362,)
            and np.isfinite(candidate).all()
        ):
            reference_vector = candidate
            source_name = global_name
            break

    if reference_vector is not None:
        reference_metrics = (
            phase38_binary_metrics(
                PHASE38_LABELS_PRIVATE,
                reference_vector,
            )
        )

        phase38_reference_comparisons[
            reference_name
        ] = {
            "source": source_name,
            "log_loss": (
                reference_metrics[
                    "log_loss"
                ]
            ),
            "auroc": (
                reference_metrics[
                    "auroc"
                ]
            ),
            "phase38_log_loss_gain": float(
                reference_metrics[
                    "log_loss"
                ]
                - phase38_final_metrics[
                    "log_loss"
                ]
            ),
            "phase38_auroc_gain": float(
                phase38_final_metrics[
                    "auroc"
                ]
                - reference_metrics[
                    "auroc"
                ]
            ),
        }


# ------------------------------------------------------------------
# 7. Promotion gate
# ------------------------------------------------------------------

phase38_log_loss_gain = float(
    phase38_baseline_metrics["log_loss"]
    - phase38_final_metrics["log_loss"]
)

phase38_auroc_gain = float(
    phase38_final_metrics["auroc"]
    - phase38_baseline_metrics["auroc"]
)

phase38_promotion_thresholds = {
    "minimum_log_loss_gain": 0.003,
    "minimum_auroc_gain": 0.001,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.003,
    "maximum_major_group_harm": 0.015,
    "calibration_slope_range": [
        0.8,
        1.2,
    ],
}

phase38_promotion_gate_passed = bool(
    phase38_log_loss_gain
    >= phase38_promotion_thresholds[
        "minimum_log_loss_gain"
    ]
    and phase38_auroc_gain
    >= phase38_promotion_thresholds[
        "minimum_auroc_gain"
    ]
    and phase38_fold_wins
    >= phase38_promotion_thresholds[
        "minimum_fold_wins"
    ]
    and phase38_worst_fold_excess
    <= phase38_promotion_thresholds[
        "maximum_worst_fold_excess"
    ]
    and phase38_maximum_major_group_harm
    <= phase38_promotion_thresholds[
        "maximum_major_group_harm"
    ]
    and (
        phase38_promotion_thresholds[
            "calibration_slope_range"
        ][0]
        <= phase38_calibration_slope
        <= phase38_promotion_thresholds[
            "calibration_slope_range"
        ][1]
    )
)


# ------------------------------------------------------------------
# 8. Persist private deployment states immediately
# ------------------------------------------------------------------

PHASE38_PRIVATE_CHECKPOINT_DIRECTORY = Path(
    "/kaggle/working/"
    "phase38_private_checkpoint"
)

PHASE38_PRIVATE_CHECKPOINT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

phase38_serializable_states = []

for state_record in (
    PHASE38_DEPLOYMENT_STATES_PRIVATE
):
    fold = int(state_record["fold"])

    booster_paths = []

    for seed_index, booster in enumerate(
        state_record["boosters"]
    ):
        booster_path = (
            PHASE38_PRIVATE_CHECKPOINT_DIRECTORY
            / (
                f"fold{fold}_seed"
                f"{seed_index}.ubj"
            )
        )

        booster.save_model(booster_path)
        booster_paths.append(
            booster_path.name
        )

    phase38_serializable_states.append({
        key: value
        for key, value
        in state_record.items()
        if key != "boosters"
    })

    phase38_serializable_states[-1][
        "booster_files"
    ] = booster_paths

joblib.dump(
    {
        "schema_version": 1,
        "phase": (
            "phase38_atlasless_"
            "subregion_anchored_boosting"
        ),
        "selected_specification": (
            phase38_selected_specification
        ),
        "selected_gate": (
            phase38_selected_gate
        ),
        "states": (
            phase38_serializable_states
        ),
    },
    PHASE38_PRIVATE_CHECKPOINT_DIRECTORY
    / "phase38_state.joblib",
)

np.save(
    PHASE38_PRIVATE_CHECKPOINT_DIRECTORY
    / "phase38_raw_residual_oof.npy",
    PHASE38_RAW_RESIDUAL_OOF_PRIVATE,
    allow_pickle=False,
)

np.save(
    PHASE38_PRIVATE_CHECKPOINT_DIRECTORY
    / "phase38_oof.npy",
    PHASE38_OOF_PRIVATE,
    allow_pickle=False,
)


# ------------------------------------------------------------------
# 9. Sanitized report
# ------------------------------------------------------------------

phase38_report = {
    "phase": (
        "phase38_atlasless_subregion_"
        "anchored_boosting"
    ),
    "status": (
        "promoted"
        if phase38_promotion_gate_passed
        else "not_promoted"
    ),
    "frozen_model_specification": (
        phase38_selected_specification
    ),
    "frozen_gate": (
        phase38_selected_gate
    ),
    "baseline": {
        key: round(value, 6)
        for key, value
        in phase38_baseline_metrics.items()
    },
    "raw_phase38_component": {
        key: round(value, 6)
        for key, value
        in phase38_raw_metrics.items()
    },
    "phase38": {
        **{
            key: round(value, 6)
            for key, value
            in phase38_final_metrics.items()
        },
        "calibration_intercept": round(
            phase38_calibration_intercept,
            6,
        ),
        "calibration_slope": round(
            phase38_calibration_slope,
            6,
        ),
    },
    "improvements": {
        "log_loss_gain": round(
            phase38_log_loss_gain,
            6,
        ),
        "auroc_gain": round(
            phase38_auroc_gain,
            6,
        ),
        "fold_wins": (
            phase38_fold_wins
        ),
        "worst_fold_excess": round(
            phase38_worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            max(
                0.0,
                phase38_maximum_major_group_harm,
            ),
            6,
        ),
    },
    "fold_metrics": [
        {
            key: (
                round(value, 6)
                if isinstance(
                    value,
                    float,
                )
                else value
            )
            for key, value
            in record.items()
        }
        for record in phase38_fold_reports
    ],
    "major_acquisition_group_metrics": [
        {
            key: (
                round(value, 6)
                if isinstance(
                    value,
                    float,
                )
                else value
            )
            for key, value
            in record.items()
        }
        for record
        in phase38_major_group_reports
    ],
    "reference_comparisons": {
        name: {
            key: (
                round(value, 6)
                if isinstance(
                    value,
                    float,
                )
                else value
            )
            for key, value
            in comparison.items()
        }
        for name, comparison
        in phase38_reference_comparisons.items()
    },
    "update_magnitude": {
        "mean_absolute_raw_residual": round(
            float(
                np.mean(
                    np.abs(
                        PHASE38_RAW_RESIDUAL_OOF_PRIVATE
                    )
                )
            ),
            6,
        ),
        "q95_absolute_raw_residual": round(
            float(
                np.quantile(
                    np.abs(
                        PHASE38_RAW_RESIDUAL_OOF_PRIVATE
                    ),
                    0.95,
                )
            ),
            6,
        ),
        "mean_effective_alpha": round(
            float(
                np.mean(
                    PHASE38_EFFECTIVE_ALPHA_OOF_PRIVATE
                )
            ),
            6,
        ),
        "maximum_effective_alpha": round(
            float(
                np.max(
                    PHASE38_EFFECTIVE_ALPHA_OOF_PRIVATE
                )
            ),
            6,
        ),
    },
    "promotion_thresholds": (
        phase38_promotion_thresholds
    ),
    "promotion_gate_passed": (
        phase38_promotion_gate_passed
    ),
    "shared_model_specification_across_folds": True,
    "shared_gate_across_folds": True,
    "fold_specific_group_routing": False,
    "outer_oracle_parameters_used": False,
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "deployment_states_persisted_privately": True,
    "private_checkpoint_file_count": int(
        len(
            list(
                PHASE38_PRIVATE_CHECKPOINT_DIRECTORY.iterdir()
            )
        )
    ),
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase38_outer_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE38_OOF"
)
print(
    json.dumps(
        phase38_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE38_OOF"
)

Phase38 outer fold 0: log_loss=0.360736, gain=-0.002241, auroc=0.916231
Phase38 outer fold 1: log_loss=0.257578, gain=-0.002163, auroc=0.953019
Phase38 outer fold 2: log_loss=0.306071, gain=0.000137, auroc=0.945657
BEGIN SANITIZED_PHASE38_OOF
{
  "phase": "phase38_atlasless_subregion_anchored_boosting",
  "status": "not_promoted",
  "frozen_model_specification": {
    "representation": "endpoint_core",
    "maximum_depth": 2,
    "minimum_child_weight": 16.0,
    "l2_regularization": 30.0,
    "column_subsample": 1.0
  },
  "frozen_gate": {
    "alpha": 0.75,
    "uncertainty_exponent": 0.0,
    "delta_cap": 1.5
  },
  "baseline": {
    "log_loss": 0.307616,
    "auroc": 0.938914,
    "mean_probability": 0.530698
  },
  "raw_phase38_component": {
    "log_loss": 0.311065,
    "auroc": 0.937019,
    "mean_probability": 0.549777
  },
  "phase38": {
    "log_loss": 0.309042,
    "auroc": 0.937848,
    "mean_probability": 0.544719,
    "calibration_intercept": 0.010284,
    "calibration_sl

In [80]:
# Phase39 Cell 122A
# Discover available component OOF and monitor prediction states.
# This cell reads no labels and performs no model selection.

import json
from pathlib import Path

import numpy as np


PHASE39_RELEVANT_LENGTHS = {
    123, 174, 196,       # monitor partitions
    407, 493,            # unique/pooled monitor sizes
    443, 452, 467,       # outer-valid partitions
    1362,                # complete OOF vector
}

PHASE39_PATH_TERMS = {
    "prediction",
    "probability",
    "probabilities",
    "monitor",
    "oof",
    "residual",
    "logit",
    "fold_result",
    "fold_results",
    "selection",
}

PHASE39_PHASE_TERMS = {
    "phase30",
    "phase33",
    "phase36",
    "phase37",
    "phase38",
}


def phase39_relevant_path(path):
    lowered = path.lower()

    return (
        any(
            term in lowered
            for term in PHASE39_PHASE_TERMS
        )
        and any(
            term in lowered
            for term in PHASE39_PATH_TERMS
        )
    )


phase39_array_records = []
phase39_seen_objects = set()


def phase39_walk_state(
    value,
    path,
    depth=0,
):
    if depth > 6:
        return

    object_id = id(value)

    # Prevent recursive containers while allowing scalar reuse.
    if isinstance(
        value,
        (dict, list, tuple),
    ):
        if object_id in phase39_seen_objects:
            return

        phase39_seen_objects.add(object_id)

    if isinstance(value, np.ndarray):
        if (
            value.ndim >= 1
            and any(
                dimension in PHASE39_RELEVANT_LENGTHS
                for dimension in value.shape
            )
            and phase39_relevant_path(path)
        ):
            sampled_indices = None

            if value.size > 0:
                sampled_indices = np.linspace(
                    0,
                    value.size - 1,
                    min(256, value.size),
                    dtype=np.int64,
                )

                sampled = np.asarray(
                    value
                ).reshape(-1)[sampled_indices]

                if np.issubdtype(
                    sampled.dtype,
                    np.number,
                ):
                    sampled_finite = bool(
                        np.isfinite(sampled).all()
                    )
                else:
                    sampled_finite = None
            else:
                sampled_finite = True

            phase39_array_records.append({
                "path": path,
                "container": "ndarray",
                "shape": [
                    int(dimension)
                    for dimension in value.shape
                ],
                "dtype": str(value.dtype),
                "sampled_finite": (
                    sampled_finite
                ),
            })

        return

    # Torch tensors, without importing torch if unavailable.
    if (
        hasattr(value, "detach")
        and hasattr(value, "shape")
        and hasattr(value, "dtype")
    ):
        try:
            shape = tuple(
                int(dimension)
                for dimension in value.shape
            )

            if (
                shape
                and any(
                    dimension
                    in PHASE39_RELEVANT_LENGTHS
                    for dimension in shape
                )
                and phase39_relevant_path(path)
            ):
                sampled = (
                    value.detach()
                    .reshape(-1)[:256]
                    .float()
                    .cpu()
                    .numpy()
                )

                phase39_array_records.append({
                    "path": path,
                    "container": "tensor",
                    "shape": list(shape),
                    "dtype": str(value.dtype),
                    "sampled_finite": bool(
                        np.isfinite(sampled).all()
                    ),
                })

        except Exception:
            pass

        return

    if isinstance(value, dict):
        for key, child in value.items():
            key_string = str(key)

            # Do not recurse into trained model objects.
            if any(
                term in key_string.lower()
                for term in [
                    "booster",
                    "boosters",
                    "model_state",
                    "optimizer",
                    "state_dict",
                ]
            ):
                continue

            phase39_walk_state(
                child,
                f"{path}.{key_string}",
                depth + 1,
            )

        return

    if isinstance(value, (list, tuple)):
        # Fold/seed result collections are small. Avoid walking
        # enormous data containers accidentally.
        if len(value) > 200:
            return

        for index, child in enumerate(value):
            phase39_walk_state(
                child,
                f"{path}[{index}]",
                depth + 1,
            )


phase39_top_level_candidates = []

for global_name, value in list(
    globals().items()
):
    lowered_name = global_name.lower()

    if not any(
        phase_term in lowered_name
        for phase_term in PHASE39_PHASE_TERMS
    ):
        continue

    if any(
        excluded in lowered_name
        for excluded in [
            "feature",
            "dataset",
            "model_config",
            "checkpoint_directory",
        ]
    ):
        continue

    phase39_top_level_candidates.append(
        global_name
    )

    phase39_walk_state(
        value,
        global_name,
        depth=0,
    )


# Remove exact duplicate path records.
phase39_unique_records = []
phase39_record_keys = set()

for record in phase39_array_records:
    record_key = (
        record["path"],
        tuple(record["shape"]),
        record["dtype"],
    )

    if record_key not in phase39_record_keys:
        phase39_record_keys.add(record_key)
        phase39_unique_records.append(record)

phase39_unique_records.sort(
    key=lambda record: (
        -max(record["shape"]),
        record["path"],
    )
)


# Explicitly check the most likely complete OOF objects.
phase39_expected_oof_objects = {}

for name in [
    "PHASE30_OOF_PRIVATE",
    "PHASE30_BLEND_OOF_PRIVATE",
    "PHASE33_OOF_PRIVATE",
    "PHASE33_GATED_OOF_PRIVATE",
    "PHASE36_OOF_PRIVATE",
    "PHASE36_RAW_RESIDUAL_OOF_PRIVATE",
    "PHASE37_OOF_PRIVATE",
    "PHASE38_OOF_PRIVATE",
    "PHASE38_RAW_RESIDUAL_OOF_PRIVATE",
    "PHASE19_PHASE12C_OOF_PRIVATE",
]:
    value = globals().get(name)

    if value is None:
        phase39_expected_oof_objects[name] = {
            "available": False,
        }
        continue

    try:
        array = np.asarray(value)

        phase39_expected_oof_objects[name] = {
            "available": True,
            "shape": [
                int(dimension)
                for dimension in array.shape
            ],
            "dtype": str(array.dtype),
            "all_finite": (
                bool(np.isfinite(array).all())
                if np.issubdtype(
                    array.dtype,
                    np.number,
                )
                else None
            ),
        }

    except Exception as exc:
        phase39_expected_oof_objects[name] = {
            "available": True,
            "inspection_error": (
                type(exc).__name__
            ),
        }


# Inspect private checkpoint filenames without loading tensors.
phase39_checkpoint_directories = {}

for directory_name in [
    "phase33_private_checkpoint",
    "phase36_private_checkpoint",
    "phase37_private_checkpoint",
    "phase38_private_checkpoint",
]:
    directory = (
        Path("/kaggle/working")
        / directory_name
    )

    if directory.is_dir():
        phase39_checkpoint_directories[
            directory_name
        ] = {
            "present": True,
            "file_count": len(
                list(directory.iterdir())
            ),
            "file_names": sorted(
                path.name
                for path in directory.iterdir()
                if path.is_file()
            ),
        }
    else:
        phase39_checkpoint_directories[
            directory_name
        ] = {
            "present": False,
        }


phase39_state_report = {
    "phase": (
        "phase39_residual_component_"
        "state_discovery"
    ),
    "status": "complete",
    "top_level_candidate_count": len(
        phase39_top_level_candidates
    ),
    "relevant_array_record_count": len(
        phase39_unique_records
    ),
    "relevant_arrays": (
        phase39_unique_records
    ),
    "expected_oof_objects": (
        phase39_expected_oof_objects
    ),
    "checkpoint_directories": (
        phase39_checkpoint_directories
    ),
    "planned_components": [
        "phase36_boosted_residual",
        "phase33_geometry_semantic_residual",
        "phase30_clinical_view_residual_if_available",
    ],
    "planned_search": (
        "exhaustive_shared_residual_simplex_"
        "plus_bounded_uncertainty_gate"
    ),
    "labels_used": False,
    "outer_validation_labels_used": False,
    "case_level_values_displayed": False,
    "smoke_data_read": False,
    "test_data_read": False,
}

print(
    "BEGIN SANITIZED_PHASE39_STATE_DISCOVERY"
)
print(
    json.dumps(
        phase39_state_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE39_STATE_DISCOVERY"
)

BEGIN SANITIZED_PHASE39_STATE_DISCOVERY
{
  "phase": "phase39_residual_component_state_discovery",
  "status": "complete",
  "top_level_candidate_count": 525,
  "relevant_array_record_count": 922,
  "relevant_arrays": [
    {
      "path": "PHASE33_GATED_OOF_PRIVATE",
      "container": "ndarray",
      "shape": [
        1362
      ],
      "dtype": "float64",
      "sampled_finite": true
    },
    {
      "path": "PHASE33_OOF_PRIVATE",
      "container": "ndarray",
      "shape": [
        1362
      ],
      "dtype": "float64",
      "sampled_finite": true
    },
    {
      "path": "PHASE36_OOF_PRIVATE",
      "container": "ndarray",
      "shape": [
        1362
      ],
      "dtype": "float64",
      "sampled_finite": true
    },
    {
      "path": "PHASE36_RAW_RESIDUAL_OOF_PRIVATE",
      "container": "ndarray",
      "shape": [
        1362
      ],
      "dtype": "float64",
      "sampled_finite": true
    },
    {
      "path": "PHASE37_OOF_PRIVATE",
      "container": "nd

In [83]:
# Phase39 Cell 122B-preflight correction v2
# Establish Phase37 as the authoritative monitor anchor and align
# the independent Phase33 component by (fold, case_index).

import numpy as np

from scipy.special import expit
from sklearn.metrics import log_loss, roc_auc_score


required_names = [
    "phase33_monitor_component",
    "phase33_monitor_indices",
    "phase33_monitor_folds",
    "phase33_monitor_groups",
    "phase33_monitor_labels",
    "phase33_monitor_baseline",
    "phase33_monitor_weights",
    "phase37_monitor_indices",
    "phase37_monitor_folds",
    "phase37_monitor_groups",
    "phase37_monitor_labels",
    "phase37_monitor_baseline",
    "phase37_monitor_weights",
    "phase37_monitor_raw_residual",
]

missing = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing, {
    "message": "Required Phase39 monitor state missing.",
    "missing": missing,
}


# Preserve the original Phase33 monitor contract privately.
PHASE33_MONITOR_CONTRACT_BEFORE_PHASE39_PRIVATE = {
    "component": np.asarray(
        phase33_monitor_component,
        dtype=np.float64,
    ).copy(),
    "indices": np.asarray(
        phase33_monitor_indices,
        dtype=np.int64,
    ).copy(),
    "folds": np.asarray(
        phase33_monitor_folds,
        dtype=np.int64,
    ).copy(),
    "groups": np.asarray(
        phase33_monitor_groups
    ).copy(),
    "labels": np.asarray(
        phase33_monitor_labels,
        dtype=np.int64,
    ).copy(),
    "baseline": np.asarray(
        phase33_monitor_baseline,
        dtype=np.float64,
    ).copy(),
    "weights": np.asarray(
        phase33_monitor_weights,
        dtype=np.float64,
    ).copy(),
}


phase39_original_phase33_keys = [
    (int(fold), int(case_index))
    for fold, case_index in zip(
        PHASE33_MONITOR_CONTRACT_BEFORE_PHASE39_PRIVATE[
            "folds"
        ],
        PHASE33_MONITOR_CONTRACT_BEFORE_PHASE39_PRIVATE[
            "indices"
        ],
    )
]

phase39_phase37_keys = [
    (int(fold), int(case_index))
    for fold, case_index in zip(
        np.asarray(
            phase37_monitor_folds,
            dtype=np.int64,
        ).reshape(-1),
        np.asarray(
            phase37_monitor_indices,
            dtype=np.int64,
        ).reshape(-1),
    )
]

assert len(phase39_original_phase33_keys) == 493
assert len(phase39_phase37_keys) == 493
assert len(set(phase39_original_phase33_keys)) == 493
assert len(set(phase39_phase37_keys)) == 493
assert set(phase39_original_phase33_keys) == set(
    phase39_phase37_keys
)


# Align the Phase33 component to Phase37 observation order.
phase39_phase33_position = {
    key: position
    for position, key
    in enumerate(
        phase39_original_phase33_keys
    )
}

phase39_phase33_order_for_phase37 = np.asarray(
    [
        phase39_phase33_position[key]
        for key in phase39_phase37_keys
    ],
    dtype=np.int64,
)

phase39_aligned_phase33_component = (
    PHASE33_MONITOR_CONTRACT_BEFORE_PHASE39_PRIVATE[
        "component"
    ][phase39_phase33_order_for_phase37]
)

phase39_aligned_phase33_labels = (
    PHASE33_MONITOR_CONTRACT_BEFORE_PHASE39_PRIVATE[
        "labels"
    ][phase39_phase33_order_for_phase37]
)

assert np.array_equal(
    phase39_aligned_phase33_labels,
    np.asarray(
        phase37_monitor_labels,
        dtype=np.int64,
    ).reshape(-1),
), {
    "message": (
        "Phase33 component labels do not align "
        "with the Phase37 monitor pool."
    )
}

assert np.isfinite(
    phase39_aligned_phase33_component
).all()

assert (
    (phase39_aligned_phase33_component > 0.0).all()
    and
    (phase39_aligned_phase33_component < 1.0).all()
)


# Record the expected difference between historical anchors.
phase39_aligned_old_phase33_baseline = (
    PHASE33_MONITOR_CONTRACT_BEFORE_PHASE39_PRIVATE[
        "baseline"
    ][phase39_phase33_order_for_phase37]
)

phase39_authoritative_phase37_baseline = np.asarray(
    phase37_monitor_baseline,
    dtype=np.float64,
).reshape(-1)

phase39_historical_anchor_difference = {
    "maximum": float(
        np.max(
            np.abs(
                phase39_aligned_old_phase33_baseline
                - phase39_authoritative_phase37_baseline
            )
        )
    ),
    "mean": float(
        np.mean(
            np.abs(
                phase39_aligned_old_phase33_baseline
                - phase39_authoritative_phase37_baseline
            )
        )
    ),
}


# Replace the names consumed by Cell 122B with one coherent,
# Phase37-anchored observation contract.
phase33_monitor_component = (
    phase39_aligned_phase33_component.copy()
)

phase33_monitor_indices = np.asarray(
    phase37_monitor_indices,
    dtype=np.int64,
).reshape(-1).copy()

phase33_monitor_folds = np.asarray(
    phase37_monitor_folds,
    dtype=np.int64,
).reshape(-1).copy()

phase33_monitor_groups = np.asarray(
    phase37_monitor_groups
).reshape(-1).copy()

phase33_monitor_labels = np.asarray(
    phase37_monitor_labels,
    dtype=np.int64,
).reshape(-1).copy()

phase33_monitor_baseline = (
    phase39_authoritative_phase37_baseline.copy()
)

phase33_monitor_weights = np.asarray(
    phase37_monitor_weights,
    dtype=np.float64,
).reshape(-1).copy()


# Verify the authoritative Phase36 reconstruction directly.
phase39_direct_phase36_probability = expit(
    np.asarray(
        phase37_monitor_baseline,
        dtype=np.float64,
    ).reshape(-1)
    * 0.0
)

# phase37_monitor_baseline contains probabilities, so convert safely.
from scipy.special import logit

phase39_direct_phase36_probability = expit(
    logit(
        np.clip(
            np.asarray(
                phase37_monitor_baseline,
                dtype=np.float64,
            ).reshape(-1),
            1e-6,
            1.0 - 1e-6,
        )
    )
    + 0.75
    * np.asarray(
        phase37_monitor_raw_residual,
        dtype=np.float64,
    ).reshape(-1)
)

phase39_direct_log_loss = float(
    log_loss(
        np.asarray(
            phase37_monitor_labels,
            dtype=np.int64,
        ).reshape(-1),
        phase39_direct_phase36_probability,
        sample_weight=np.asarray(
            phase37_monitor_weights,
            dtype=np.float64,
        ).reshape(-1),
    )
)

phase39_direct_auroc = float(
    roc_auc_score(
        np.asarray(
            phase37_monitor_labels,
            dtype=np.int64,
        ).reshape(-1),
        phase39_direct_phase36_probability,
        sample_weight=np.asarray(
            phase37_monitor_weights,
            dtype=np.float64,
        ).reshape(-1),
    )
)

assert abs(
    phase39_direct_log_loss
    - 0.21915518635444312
) <= 1e-8, {
    "message": (
        "Authoritative Phase36 reconstruction still failed."
    ),
    "log_loss": phase39_direct_log_loss,
}

assert abs(
    phase39_direct_auroc
    - 0.9671501087744743
) <= 1e-8, {
    "message": (
        "Authoritative Phase36 AUROC reconstruction failed."
    ),
    "auroc": phase39_direct_auroc,
}


print(
    "Phase39 coherent monitor contract accepted: "
    "anchor=Phase37/Phase36, "
    "component=aligned Phase33, "
    f"phase36_log_loss={phase39_direct_log_loss:.6f}, "
    f"phase36_auroc={phase39_direct_auroc:.6f}, "
    "historical_anchor_max_difference="
    f"{phase39_historical_anchor_difference['maximum']:.8f}"
)

Phase39 coherent monitor contract accepted: anchor=Phase37/Phase36, component=aligned Phase33, phase36_log_loss=0.219155, phase36_auroc=0.967150, historical_anchor_max_difference=0.67746450


In [84]:
# Phase39 Cell 122B
# Exhaustive pooled-monitor selection of a robust Phase33/Phase36
# residual stack.
#
# No outer-validation labels are read in this cell.

import json
import time

import numpy as np

from scipy.special import expit, logit
from sklearn.metrics import log_loss, roc_auc_score


PHASE39_CONFIG = {
    "phase": (
        "phase39_phase33_phase36_"
        "robust_residual_stack"
    ),
    # Weight assigned to Phase36; Phase33 receives 1-weight.
    "phase36_weights": [
        round(value, 2)
        for value in np.arange(
            0.0,
            1.0001,
            0.05,
        )
    ],
    "shared_alphas": [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.75,
        1.00,
    ],
    "uncertainty_exponents": [
        0.0,
        0.5,
        1.0,
        2.0,
    ],
    "delta_caps": [
        0.75,
        1.0,
        1.5,
        2.0,
        3.0,
    ],
    "bootstrap_replicates": 4000,
    "bootstrap_seed": 390601,
    "dirichlet_concentration": 20.0,
    "dirichlet_floor": 0.20,
    "minimum_monitor_log_loss_gain": 0.003,
    "minimum_monitor_auroc_gain": 0.001,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.004,
    "maximum_monitor_group_harm": 0.012,
    "minimum_leave_one_group_out_gain": -0.001,
    "minimum_bootstrap_lcb": -0.0005,
    "minimum_bootstrap_probability_positive": 0.95,
    "log_loss_tie_band_for_auroc": 0.001,
}


# --------------------------------------------------------------
# 1. Validate live state
# --------------------------------------------------------------

required_names = [
    "phase33_monitor_baseline",
    "phase33_monitor_component",
    "phase33_monitor_folds",
    "phase33_monitor_groups",
    "phase33_monitor_indices",
    "phase33_monitor_labels",
    "phase33_monitor_weights",
    "phase36_selection_folds",
    "PHASE19_PHASE12C_OOF_PRIVATE",
    "PHASE33_OOF_PRIVATE",
    "PHASE36_RAW_RESIDUAL_OOF_PRIVATE",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, {
    "message": "Required Phase39 state is missing.",
    "missing": missing_names,
}


phase39_monitor_baseline = np.asarray(
    phase33_monitor_baseline,
    dtype=np.float64,
).reshape(-1)

phase39_monitor_phase33_component = np.asarray(
    phase33_monitor_component,
    dtype=np.float64,
).reshape(-1)

phase39_monitor_folds = np.asarray(
    phase33_monitor_folds,
    dtype=np.int64,
).reshape(-1)

phase39_monitor_groups = np.asarray(
    phase33_monitor_groups
).reshape(-1)

phase39_monitor_indices = np.asarray(
    phase33_monitor_indices,
    dtype=np.int64,
).reshape(-1)

phase39_monitor_labels = np.asarray(
    phase33_monitor_labels,
    dtype=np.int64,
).reshape(-1)

phase39_monitor_weights = np.asarray(
    phase33_monitor_weights,
    dtype=np.float64,
).reshape(-1)

phase39_monitor_n = len(
    phase39_monitor_labels
)

assert phase39_monitor_n == 493

for array in [
    phase39_monitor_baseline,
    phase39_monitor_phase33_component,
    phase39_monitor_folds,
    phase39_monitor_groups,
    phase39_monitor_indices,
    phase39_monitor_labels,
    phase39_monitor_weights,
]:
    assert len(array) == phase39_monitor_n

assert np.isfinite(
    phase39_monitor_baseline
).all()

assert np.isfinite(
    phase39_monitor_phase33_component
).all()

assert np.isfinite(
    phase39_monitor_weights
).all()

assert (
    phase39_monitor_weights > 0
).all()

assert np.isin(
    phase39_monitor_labels,
    [0, 1],
).all()

phase39_monitor_weights = (
    phase39_monitor_weights
    / np.sum(phase39_monitor_weights)
)


# --------------------------------------------------------------
# 2. Recover the selected Phase36 monitor residual
# --------------------------------------------------------------

PHASE39_PHASE36_SPECIFICATION_INDEX = 4
PHASE39_PHASE36_ORIGINAL_SCALE = 0.75

phase39_phase36_residual_lookup = {}

for fold_record in phase36_selection_folds:
    fold = int(
        fold_record.get(
            "fold",
            len(
                set(
                    key[0]
                    for key
                    in phase39_phase36_residual_lookup
                )
            ),
        )
    )

    monitor_indices = np.asarray(
        fold_record["monitor_indices"],
        dtype=np.int64,
    )

    specification_results = (
        fold_record[
            "specification_results"
        ]
    )

    selected_result = None

    for result_index, result in enumerate(
        specification_results
    ):
        recorded_index = result.get(
            "specification_index",
            result_index,
        )

        if (
            int(recorded_index)
            == PHASE39_PHASE36_SPECIFICATION_INDEX
        ):
            selected_result = result
            break

    assert selected_result is not None, {
        "message": (
            "Could not find selected Phase36 "
            "specification index 4."
        ),
        "fold": fold,
    }

    ensemble_residual = np.asarray(
        selected_result[
            "ensemble_residual"
        ],
        dtype=np.float64,
    ).reshape(-1)

    assert ensemble_residual.shape == (
        len(monitor_indices),
    )

    for case_index, residual in zip(
        monitor_indices,
        ensemble_residual,
    ):
        key = (
            int(fold),
            int(case_index),
        )

        assert (
            key
            not in phase39_phase36_residual_lookup
        )

        phase39_phase36_residual_lookup[
            key
        ] = float(residual)


phase39_monitor_phase36_residual = np.asarray(
    [
        phase39_phase36_residual_lookup[
            (
                int(fold),
                int(case_index),
            )
        ]
        for fold, case_index in zip(
            phase39_monitor_folds,
            phase39_monitor_indices,
        )
    ],
    dtype=np.float64,
)

assert phase39_monitor_phase36_residual.shape == (
    493,
)

assert np.isfinite(
    phase39_monitor_phase36_residual
).all()


# --------------------------------------------------------------
# 3. Convert the Phase33 monitor component to a residual
# --------------------------------------------------------------

phase39_probability_clip = 1e-6

phase39_monitor_baseline_logit = logit(
    np.clip(
        phase39_monitor_baseline,
        phase39_probability_clip,
        1.0 - phase39_probability_clip,
    )
)

phase39_monitor_phase33_logit = logit(
    np.clip(
        phase39_monitor_phase33_component,
        phase39_probability_clip,
        1.0 - phase39_probability_clip,
    )
)

phase39_monitor_phase33_residual = (
    phase39_monitor_phase33_logit
    - phase39_monitor_baseline_logit
)

assert np.isfinite(
    phase39_monitor_phase33_residual
).all()


# --------------------------------------------------------------
# 4. Verify exact Phase36 monitor reconstruction
# --------------------------------------------------------------

phase39_phase36_reconstructed_probability = (
    expit(
        phase39_monitor_baseline_logit
        + PHASE39_PHASE36_ORIGINAL_SCALE
        * phase39_monitor_phase36_residual
    )
)

phase39_phase36_reconstructed_log_loss = float(
    log_loss(
        phase39_monitor_labels,
        phase39_phase36_reconstructed_probability,
        sample_weight=(
            phase39_monitor_weights
        ),
    )
)

phase39_phase36_reconstructed_auroc = float(
    roc_auc_score(
        phase39_monitor_labels,
        phase39_phase36_reconstructed_probability,
        sample_weight=(
            phase39_monitor_weights
        ),
    )
)

assert abs(
    phase39_phase36_reconstructed_log_loss
    - 0.21915518635444312
) <= 1e-8, {
    "message": (
        "Phase36 monitor reconstruction failed."
    ),
    "reconstructed_log_loss": (
        phase39_phase36_reconstructed_log_loss
    ),
    "expected_log_loss": (
        0.21915518635444312
    ),
}

assert abs(
    phase39_phase36_reconstructed_auroc
    - 0.9671501087744743
) <= 1e-8, {
    "message": (
        "Phase36 monitor AUROC reconstruction failed."
    ),
    "reconstructed_auroc": (
        phase39_phase36_reconstructed_auroc
    ),
    "expected_auroc": (
        0.9671501087744743
    ),
}


# --------------------------------------------------------------
# 5. Metrics and robustness functions
# --------------------------------------------------------------

def phase39_weighted_log_loss(
    labels,
    probabilities,
    weights,
):
    return float(
        log_loss(
            labels,
            np.clip(
                probabilities,
                1e-6,
                1.0 - 1e-6,
            ),
            sample_weight=weights,
        )
    )


def phase39_weighted_auroc(
    labels,
    probabilities,
    weights,
):
    return float(
        roc_auc_score(
            labels,
            probabilities,
            sample_weight=weights,
        )
    )


def phase39_candidate_probability(
    phase36_weight,
    alpha,
    uncertainty_exponent,
    delta_cap,
):
    phase33_weight = (
        1.0 - phase36_weight
    )

    combined_residual = (
        phase36_weight
        * phase39_monitor_phase36_residual
        + phase33_weight
        * phase39_monitor_phase33_residual
    )

    baseline_probability = (
        phase39_monitor_baseline
    )

    uncertainty = np.clip(
        4.0
        * baseline_probability
        * (1.0 - baseline_probability),
        0.0,
        1.0,
    )

    effective_alpha = (
        alpha
        * np.power(
            uncertainty,
            uncertainty_exponent,
        )
    )

    bounded_residual = np.clip(
        combined_residual,
        -delta_cap,
        delta_cap,
    )

    probability = expit(
        phase39_monitor_baseline_logit
        + effective_alpha
        * bounded_residual
    )

    return probability


phase39_baseline_monitor_log_loss = (
    phase39_weighted_log_loss(
        phase39_monitor_labels,
        phase39_monitor_baseline,
        phase39_monitor_weights,
    )
)

phase39_baseline_monitor_auroc = (
    phase39_weighted_auroc(
        phase39_monitor_labels,
        phase39_monitor_baseline,
        phase39_monitor_weights,
    )
)


def phase39_deterministic_metrics(
    probability,
):
    log_loss_value = (
        phase39_weighted_log_loss(
            phase39_monitor_labels,
            probability,
            phase39_monitor_weights,
        )
    )

    auroc_value = (
        phase39_weighted_auroc(
            phase39_monitor_labels,
            probability,
            phase39_monitor_weights,
        )
    )

    fold_records = []
    fold_wins = 0
    worst_fold_regret = 0.0

    for fold in sorted(
        np.unique(
            phase39_monitor_folds
        ).tolist()
    ):
        mask = (
            phase39_monitor_folds == fold
        )

        fold_weights = (
            phase39_monitor_weights[mask]
        )

        baseline_fold_loss = (
            phase39_weighted_log_loss(
                phase39_monitor_labels[mask],
                phase39_monitor_baseline[mask],
                fold_weights,
            )
        )

        candidate_fold_loss = (
            phase39_weighted_log_loss(
                phase39_monitor_labels[mask],
                probability[mask],
                fold_weights,
            )
        )

        gain = (
            baseline_fold_loss
            - candidate_fold_loss
        )

        fold_wins += int(gain > 0.0)

        worst_fold_regret = max(
            worst_fold_regret,
            -gain,
        )

        fold_records.append({
            "fold": int(fold),
            "baseline_log_loss": (
                baseline_fold_loss
            ),
            "candidate_log_loss": (
                candidate_fold_loss
            ),
            "gain": float(gain),
        })

    group_records = []
    maximum_group_harm = 0.0

    unique_case_count_by_group = {}

    for group in np.unique(
        phase39_monitor_groups
    ):
        group_mask = (
            phase39_monitor_groups == group
        )

        unique_case_count_by_group[group] = (
            len(
                np.unique(
                    phase39_monitor_indices[
                        group_mask
                    ]
                )
            )
        )

    evaluated_groups = [
        group
        for group, case_count
        in unique_case_count_by_group.items()
        if case_count >= 8
    ]

    for group in evaluated_groups:
        mask = (
            phase39_monitor_groups == group
        )

        group_weights = (
            phase39_monitor_weights[mask]
        )

        baseline_group_loss = (
            phase39_weighted_log_loss(
                phase39_monitor_labels[mask],
                phase39_monitor_baseline[mask],
                group_weights,
            )
        )

        candidate_group_loss = (
            phase39_weighted_log_loss(
                phase39_monitor_labels[mask],
                probability[mask],
                group_weights,
            )
        )

        change = (
            candidate_group_loss
            - baseline_group_loss
        )

        maximum_group_harm = max(
            maximum_group_harm,
            change,
        )

        group_records.append({
            "group": (
                int(group)
                if isinstance(
                    group,
                    (int, np.integer),
                )
                else str(group)
            ),
            "unique_case_count": int(
                unique_case_count_by_group[
                    group
                ]
            ),
            "log_loss_change": float(
                change
            ),
        })

    leave_one_group_out_gains = []

    for held_out_group in np.unique(
        phase39_monitor_groups
    ):
        retained = (
            phase39_monitor_groups
            != held_out_group
        )

        retained_weights = (
            phase39_monitor_weights[
                retained
            ]
        )

        baseline_retained_loss = (
            phase39_weighted_log_loss(
                phase39_monitor_labels[
                    retained
                ],
                phase39_monitor_baseline[
                    retained
                ],
                retained_weights,
            )
        )

        candidate_retained_loss = (
            phase39_weighted_log_loss(
                phase39_monitor_labels[
                    retained
                ],
                probability[retained],
                retained_weights,
            )
        )

        leave_one_group_out_gains.append(
            baseline_retained_loss
            - candidate_retained_loss
        )

    return {
        "log_loss": log_loss_value,
        "log_loss_gain": float(
            phase39_baseline_monitor_log_loss
            - log_loss_value
        ),
        "auroc": auroc_value,
        "auroc_gain": float(
            auroc_value
            - phase39_baseline_monitor_auroc
        ),
        "fold_wins": int(fold_wins),
        "worst_fold_regret": float(
            max(0.0, worst_fold_regret)
        ),
        "maximum_group_harm": float(
            max(0.0, maximum_group_harm)
        ),
        "minimum_leave_one_group_out_gain": float(
            np.min(
                leave_one_group_out_gains
            )
        ),
        "median_leave_one_group_out_gain": float(
            np.median(
                leave_one_group_out_gains
            )
        ),
        "fold_records": fold_records,
        "group_records": group_records,
    }


# --------------------------------------------------------------
# 6. Construct group-shift Bayesian-bootstrap weights
# --------------------------------------------------------------

phase39_rng = np.random.default_rng(
    PHASE39_CONFIG["bootstrap_seed"]
)

unique_case_indices, observation_to_case = (
    np.unique(
        phase39_monitor_indices,
        return_inverse=True,
    )
)

phase39_unique_case_count = len(
    unique_case_indices
)

case_groups = np.empty(
    phase39_unique_case_count,
    dtype=phase39_monitor_groups.dtype,
)

case_multiplicity = np.bincount(
    observation_to_case,
    minlength=phase39_unique_case_count,
)

for case_position in range(
    phase39_unique_case_count
):
    observed_groups = np.unique(
        phase39_monitor_groups[
            observation_to_case
            == case_position
        ]
    )

    assert len(observed_groups) == 1

    case_groups[case_position] = (
        observed_groups[0]
    )

unique_groups = np.unique(case_groups)
group_to_position = {
    group: position
    for position, group
    in enumerate(unique_groups)
}

case_group_positions = np.asarray(
    [
        group_to_position[group]
        for group in case_groups
    ],
    dtype=np.int64,
)

group_case_counts = np.bincount(
    case_group_positions,
    minlength=len(unique_groups),
)

empirical_group_weights = (
    group_case_counts
    / np.sum(group_case_counts)
)

dirichlet_parameters = (
    PHASE39_CONFIG[
        "dirichlet_concentration"
    ]
    * empirical_group_weights
    + PHASE39_CONFIG[
        "dirichlet_floor"
    ]
)

bootstrap_replicates = int(
    PHASE39_CONFIG[
        "bootstrap_replicates"
    ]
)

phase39_bootstrap_weights = np.zeros(
    (
        bootstrap_replicates,
        phase39_monitor_n,
    ),
    dtype=np.float32,
)

for replicate in range(
    bootstrap_replicates
):
    group_mixture = phase39_rng.dirichlet(
        dirichlet_parameters
    )

    case_weights = np.zeros(
        phase39_unique_case_count,
        dtype=np.float64,
    )

    for group_position in range(
        len(unique_groups)
    ):
        case_positions = np.flatnonzero(
            case_group_positions
            == group_position
        )

        within_group_weights = (
            phase39_rng.exponential(
                scale=1.0,
                size=len(case_positions),
            )
        )

        within_group_weights /= np.sum(
            within_group_weights
        )

        case_weights[case_positions] = (
            group_mixture[group_position]
            * within_group_weights
        )

    observation_weights = (
        case_weights[
            observation_to_case
        ]
        / case_multiplicity[
            observation_to_case
        ]
    )

    observation_weights /= np.sum(
        observation_weights
    )

    phase39_bootstrap_weights[
        replicate
    ] = observation_weights.astype(
        np.float32
    )


# --------------------------------------------------------------
# 7. Exhaustive shared search
# --------------------------------------------------------------

phase39_search_started = time.perf_counter()

phase39_candidate_records = []

# Exact baseline candidate.
baseline_metrics = (
    phase39_deterministic_metrics(
        phase39_monitor_baseline
    )
)

phase39_candidate_records.append({
    "candidate_index": 0,
    "specification": {
        "phase36_weight": 0.0,
        "phase33_weight": 0.0,
        "alpha": 0.0,
        "uncertainty_exponent": 0.0,
        "delta_cap": 0.0,
    },
    **baseline_metrics,
    "deterministically_eligible": False,
})

candidate_index = 1

for phase36_weight in (
    PHASE39_CONFIG["phase36_weights"]
):
    for alpha in (
        PHASE39_CONFIG["shared_alphas"]
    ):
        for uncertainty_exponent in (
            PHASE39_CONFIG[
                "uncertainty_exponents"
            ]
        ):
            for delta_cap in (
                PHASE39_CONFIG[
                    "delta_caps"
                ]
            ):
                probability = (
                    phase39_candidate_probability(
                        phase36_weight=(
                            phase36_weight
                        ),
                        alpha=alpha,
                        uncertainty_exponent=(
                            uncertainty_exponent
                        ),
                        delta_cap=delta_cap,
                    )
                )

                metrics = (
                    phase39_deterministic_metrics(
                        probability
                    )
                )

                deterministic_eligible = bool(
                    metrics["log_loss_gain"]
                    >= PHASE39_CONFIG[
                        "minimum_monitor_log_loss_gain"
                    ]
                    and metrics["auroc_gain"]
                    >= PHASE39_CONFIG[
                        "minimum_monitor_auroc_gain"
                    ]
                    and metrics["fold_wins"]
                    >= PHASE39_CONFIG[
                        "minimum_monitor_fold_wins"
                    ]
                    and metrics[
                        "worst_fold_regret"
                    ]
                    <= PHASE39_CONFIG[
                        "maximum_monitor_fold_regret"
                    ]
                    and metrics[
                        "maximum_group_harm"
                    ]
                    <= PHASE39_CONFIG[
                        "maximum_monitor_group_harm"
                    ]
                    and metrics[
                        "minimum_leave_one_group_out_gain"
                    ]
                    >= PHASE39_CONFIG[
                        "minimum_leave_one_group_out_gain"
                    ]
                )

                phase39_candidate_records.append({
                    "candidate_index": (
                        candidate_index
                    ),
                    "specification": {
                        "phase36_weight": float(
                            phase36_weight
                        ),
                        "phase33_weight": float(
                            1.0
                            - phase36_weight
                        ),
                        "alpha": float(alpha),
                        "uncertainty_exponent": float(
                            uncertainty_exponent
                        ),
                        "delta_cap": float(
                            delta_cap
                        ),
                    },
                    **metrics,
                    "deterministically_eligible": (
                        deterministic_eligible
                    ),
                })

                candidate_index += 1


# --------------------------------------------------------------
# 8. Bootstrap only deterministic survivors
# --------------------------------------------------------------

baseline_case_loss = -(
    phase39_monitor_labels
    * np.log(
        np.clip(
            phase39_monitor_baseline,
            1e-6,
            1.0 - 1e-6,
        )
    )
    + (
        1
        - phase39_monitor_labels
    )
    * np.log(
        np.clip(
            1.0
            - phase39_monitor_baseline,
            1e-6,
            1.0 - 1e-6,
        )
    )
)

phase39_robust_eligible_records = []

for record in phase39_candidate_records:
    if not record[
        "deterministically_eligible"
    ]:
        record["bootstrap_evaluated"] = False
        record["robustly_eligible"] = False
        continue

    specification = record[
        "specification"
    ]

    probability = (
        phase39_candidate_probability(
            phase36_weight=(
                specification[
                    "phase36_weight"
                ]
            ),
            alpha=specification["alpha"],
            uncertainty_exponent=(
                specification[
                    "uncertainty_exponent"
                ]
            ),
            delta_cap=(
                specification["delta_cap"]
            ),
        )
    )

    candidate_case_loss = -(
        phase39_monitor_labels
        * np.log(
            np.clip(
                probability,
                1e-6,
                1.0 - 1e-6,
            )
        )
        + (
            1
            - phase39_monitor_labels
        )
        * np.log(
            np.clip(
                1.0 - probability,
                1e-6,
                1.0 - 1e-6,
            )
        )
    )

    case_gain = (
        baseline_case_loss
        - candidate_case_loss
    )

    bootstrap_gains = (
        phase39_bootstrap_weights
        @ case_gain.astype(np.float32)
    )

    bootstrap_lcb = float(
        np.quantile(
            bootstrap_gains,
            0.05,
        )
    )

    bootstrap_probability_positive = float(
        np.mean(
            bootstrap_gains > 0.0
        )
    )

    record.update({
        "bootstrap_evaluated": True,
        "bootstrap_mean_gain": float(
            np.mean(bootstrap_gains)
        ),
        "bootstrap_lcb_gain": (
            bootstrap_lcb
        ),
        "bootstrap_q01_gain": float(
            np.quantile(
                bootstrap_gains,
                0.01,
            )
        ),
        "bootstrap_probability_positive": (
            bootstrap_probability_positive
        ),
    })

    robustly_eligible = bool(
        bootstrap_lcb
        >= PHASE39_CONFIG[
            "minimum_bootstrap_lcb"
        ]
        and bootstrap_probability_positive
        >= PHASE39_CONFIG[
            "minimum_bootstrap_probability_positive"
        ]
    )

    record["robustly_eligible"] = (
        robustly_eligible
    )

    if robustly_eligible:
        phase39_robust_eligible_records.append(
            record
        )


# --------------------------------------------------------------
# 9. Log-loss band followed by AUROC maximization
# --------------------------------------------------------------

if phase39_robust_eligible_records:
    best_eligible_log_loss = min(
        record["log_loss"]
        for record
        in phase39_robust_eligible_records
    )

    phase39_log_loss_band_records = [
        record
        for record
        in phase39_robust_eligible_records
        if record["log_loss"]
        <= (
            best_eligible_log_loss
            + PHASE39_CONFIG[
                "log_loss_tie_band_for_auroc"
            ]
        )
    ]

    phase39_selected_record = max(
        phase39_log_loss_band_records,
        key=lambda record: (
            record["auroc"],
            record["bootstrap_lcb_gain"],
            record["log_loss_gain"],
            -record["specification"]["alpha"],
            record["specification"][
                "phase36_weight"
            ],
        ),
    )

    phase39_gate_advanced = True

else:
    phase39_log_loss_band_records = []
    phase39_selected_record = (
        phase39_candidate_records[0]
    )
    phase39_gate_advanced = False


phase39_raw_best_record = min(
    phase39_candidate_records,
    key=lambda record: (
        record["log_loss"],
        -record["auroc"],
    ),
)

phase39_auroc_best_record = max(
    phase39_candidate_records,
    key=lambda record: (
        record["auroc"],
        record["log_loss_gain"],
    ),
)


# Recompute and retain selected monitor probability privately.
if phase39_gate_advanced:
    selected_specification = (
        phase39_selected_record[
            "specification"
        ]
    )

    PHASE39_SELECTED_MONITOR_PROBABILITY_PRIVATE = (
        phase39_candidate_probability(
            phase36_weight=(
                selected_specification[
                    "phase36_weight"
                ]
            ),
            alpha=(
                selected_specification[
                    "alpha"
                ]
            ),
            uncertainty_exponent=(
                selected_specification[
                    "uncertainty_exponent"
                ]
            ),
            delta_cap=(
                selected_specification[
                    "delta_cap"
                ]
            ),
        )
    )

else:
    PHASE39_SELECTED_MONITOR_PROBABILITY_PRIVATE = (
        phase39_monitor_baseline.copy()
    )

PHASE39_SELECTED_SPECIFICATION_PRIVATE = dict(
    phase39_selected_record[
        "specification"
    ]
)

PHASE39_MONITOR_SELECTION_PRIVATE = {
    "selected_record": (
        phase39_selected_record
    ),
    "raw_best_record": (
        phase39_raw_best_record
    ),
    "auroc_best_record": (
        phase39_auroc_best_record
    ),
    "candidate_records": (
        phase39_candidate_records
    ),
    "monitor_indices": (
        phase39_monitor_indices
    ),
    "monitor_folds": (
        phase39_monitor_folds
    ),
    "monitor_weights": (
        phase39_monitor_weights
    ),
    "phase36_residual": (
        phase39_monitor_phase36_residual
    ),
    "phase33_residual": (
        phase39_monitor_phase33_residual
    ),
}


# --------------------------------------------------------------
# 10. Sanitized report
# --------------------------------------------------------------

def phase39_sanitize_record(record):
    result = {
        "candidate_index": int(
            record["candidate_index"]
        ),
        "specification": dict(
            record["specification"]
        ),
        "log_loss": round(
            float(record["log_loss"]),
            6,
        ),
        "log_loss_gain": round(
            float(
                record["log_loss_gain"]
            ),
            6,
        ),
        "auroc": round(
            float(record["auroc"]),
            6,
        ),
        "auroc_gain": round(
            float(record["auroc_gain"]),
            6,
        ),
        "fold_wins": int(
            record["fold_wins"]
        ),
        "worst_fold_regret": round(
            float(
                record[
                    "worst_fold_regret"
                ]
            ),
            6,
        ),
        "maximum_group_harm": round(
            float(
                record[
                    "maximum_group_harm"
                ]
            ),
            6,
        ),
        "minimum_leave_one_group_out_gain": round(
            float(
                record[
                    "minimum_leave_one_group_out_gain"
                ]
            ),
            6,
        ),
        "median_leave_one_group_out_gain": round(
            float(
                record[
                    "median_leave_one_group_out_gain"
                ]
            ),
            6,
        ),
        "robustly_eligible": bool(
            record.get(
                "robustly_eligible",
                False,
            )
        ),
    }

    if record.get(
        "bootstrap_evaluated",
        False,
    ):
        result.update({
            "bootstrap_mean_gain": round(
                float(
                    record[
                        "bootstrap_mean_gain"
                    ]
                ),
                6,
            ),
            "bootstrap_lcb_gain": round(
                float(
                    record[
                        "bootstrap_lcb_gain"
                    ]
                ),
                6,
            ),
            "bootstrap_probability_positive": round(
                float(
                    record[
                        "bootstrap_probability_positive"
                    ]
                ),
                6,
            ),
        })

    return result


phase39_report = {
    "phase": (
        "phase39_phase33_phase36_"
        "robust_residual_stack_selection"
    ),
    "status": (
        "stack_frozen"
        if phase39_gate_advanced
        else "stack_not_advanced"
    ),
    "component_contract": {
        "phase36_specification_index": (
            PHASE39_PHASE36_SPECIFICATION_INDEX
        ),
        "phase36_monitor_reconstruction_log_loss": round(
            phase39_phase36_reconstructed_log_loss,
            6,
        ),
        "phase36_monitor_reconstruction_auroc": round(
            phase39_phase36_reconstructed_auroc,
            6,
        ),
        "phase33_component_available": True,
        "phase30_component_available": False,
    },
    "search": {
        "phase36_weight_count": len(
            PHASE39_CONFIG[
                "phase36_weights"
            ]
        ),
        "alpha_count": len(
            PHASE39_CONFIG[
                "shared_alphas"
            ]
        ),
        "uncertainty_exponent_count": len(
            PHASE39_CONFIG[
                "uncertainty_exponents"
            ]
        ),
        "delta_cap_count": len(
            PHASE39_CONFIG[
                "delta_caps"
            ]
        ),
        "candidate_count": len(
            phase39_candidate_records
        ),
        "deterministically_eligible_count": int(
            sum(
                record[
                    "deterministically_eligible"
                ]
                for record
                in phase39_candidate_records
            )
        ),
        "robustly_eligible_count": len(
            phase39_robust_eligible_records
        ),
        "log_loss_band_candidate_count": len(
            phase39_log_loss_band_records
        ),
        "bootstrap_replicates": (
            bootstrap_replicates
        ),
    },
    "baseline_monitor": {
        "log_loss": round(
            phase39_baseline_monitor_log_loss,
            6,
        ),
        "auroc": round(
            phase39_baseline_monitor_auroc,
            6,
        ),
    },
    "raw_best": phase39_sanitize_record(
        phase39_raw_best_record
    ),
    "auroc_best": phase39_sanitize_record(
        phase39_auroc_best_record
    ),
    "selected": phase39_sanitize_record(
        phase39_selected_record
    ),
    "selection_rule": (
        "minimum_log_loss_then_maximum_auroc_"
        "within_0p001_log_loss_band"
    ),
    "shared_parameters_across_folds": True,
    "fold_specific_group_routing": False,
    "selection_partition": (
        "pooled_monitor_only"
    ),
    "outer_validation_labels_used": False,
    "outer_oracle_parameters_used": False,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase39_search_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE39_SELECTION"
)
print(
    json.dumps(
        phase39_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE39_SELECTION"
)

BEGIN SANITIZED_PHASE39_SELECTION
{
  "phase": "phase39_phase33_phase36_robust_residual_stack_selection",
  "status": "stack_frozen",
  "component_contract": {
    "phase36_specification_index": 4,
    "phase36_monitor_reconstruction_log_loss": 0.219155,
    "phase36_monitor_reconstruction_auroc": 0.96715,
    "phase33_component_available": true,
    "phase30_component_available": false
  },
  "search": {
    "phase36_weight_count": 21,
    "alpha_count": 8,
    "uncertainty_exponent_count": 4,
    "delta_cap_count": 5,
    "candidate_count": 3361,
    "deterministically_eligible_count": 1357,
    "robustly_eligible_count": 813,
    "log_loss_band_candidate_count": 12,
    "bootstrap_replicates": 4000
  },
  "baseline_monitor": {
    "log_loss": 0.235452,
    "auroc": 0.957457
  },
  "raw_best": {
    "candidate_index": 2865,
    "specification": {
      "phase36_weight": 0.85,
      "phase33_weight": 0.15000000000000002,
      "alpha": 1.0,
      "uncertainty_exponent": 0.0,
      "de

In [85]:
# Phase39 Cell 122C
# Frozen Phase33/Phase36 residual stack outer evaluation.
#
# Selection is already frozen:
#   Phase36 weight = 0.75
#   Phase33 weight = 0.25
#   alpha = 1.0
#   uncertainty exponent = 0.0
#   residual cap = 2.0
#
# No parameter is selected in this cell.

from pathlib import Path
import json
import time

import numpy as np

from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score


phase39_outer_started = time.perf_counter()


# --------------------------------------------------------------
# 1. Validate frozen specification and required state
# --------------------------------------------------------------

expected_phase39_specification = {
    "phase36_weight": 0.75,
    "phase33_weight": 0.25,
    "alpha": 1.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 2.0,
}

assert (
    PHASE39_SELECTED_SPECIFICATION_PRIVATE
    == expected_phase39_specification
), {
    "message": (
        "Frozen Phase39 specification changed."
    ),
    "expected": expected_phase39_specification,
    "received": (
        PHASE39_SELECTED_SPECIFICATION_PRIVATE
    ),
}

required_names = [
    "PHASE19_PHASE12C_OOF_PRIVATE",
    "PHASE33_OOF_PRIVATE",
    "PHASE33_GATED_OOF_PRIVATE",
    "PHASE36_OOF_PRIVATE",
    "PHASE36_RAW_RESIDUAL_OOF_PRIVATE",
    "PHASE37_OOF_PRIVATE",
    "PHASE19_PARTITIONS",
]

missing = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing, {
    "message": "Required Phase39 OOF state missing.",
    "missing": missing,
}


def phase39_resolve_outer_labels():
    candidate_names = [
        "PHASE38_LABELS_PRIVATE",
        "PHASE32_LABELS_PRIVATE",
        "PHASE31_LABELS_PRIVATE",
        "PHASE19_LABELS_PRIVATE",
        "labels",
        "label",
        "y",
    ]

    for name in candidate_names:
        value = globals().get(name)

        if value is None:
            continue

        array = np.asarray(value).reshape(-1)

        if (
            array.shape == (1362,)
            and np.isin(array, [0, 1]).all()
        ):
            return (
                array.astype(np.int64),
                name,
            )

    if "case_df" in globals():
        for column in [
            "label",
            "target",
            "y",
        ]:
            if column not in case_df.columns:
                continue

            array = np.asarray(
                case_df[column]
            ).reshape(-1)

            if (
                array.shape == (1362,)
                and np.isin(
                    array,
                    [0, 1],
                ).all()
            ):
                return (
                    array.astype(np.int64),
                    f"case_df.{column}",
                )

    raise AssertionError(
        "Could not resolve Phase39 labels."
    )


def phase39_resolve_outer_groups():
    candidate_names = [
        "PHASE38_ACQUISITION_GROUPS_PRIVATE",
        "PHASE32_ACQUISITION_GROUPS_PRIVATE",
        "PHASE31_ACQUISITION_GROUPS_PRIVATE",
        "acquisition_group",
        "acquisition_groups",
        "groups",
    ]

    for name in candidate_names:
        value = globals().get(name)

        if value is None:
            continue

        array = np.asarray(value).reshape(-1)

        if array.shape == (1362,):
            return array, name

    if (
        "case_df" in globals()
        and "acquisition_group"
        in case_df.columns
    ):
        return (
            np.asarray(
                case_df[
                    "acquisition_group"
                ]
            ).reshape(-1),
            "case_df.acquisition_group",
        )

    raise AssertionError(
        "Could not resolve Phase39 acquisition groups."
    )


(
    PHASE39_LABELS_PRIVATE,
    phase39_label_source,
) = phase39_resolve_outer_labels()

(
    PHASE39_GROUPS_PRIVATE,
    phase39_group_source,
) = phase39_resolve_outer_groups()


# --------------------------------------------------------------
# 2. Resolve the frozen component vectors
# --------------------------------------------------------------

phase39_probability_clip = 1e-6

PHASE39_BASELINE_OOF_PRIVATE = np.asarray(
    PHASE19_PHASE12C_OOF_PRIVATE,
    dtype=np.float64,
).reshape(-1)

PHASE39_PHASE33_COMPONENT_OOF_PRIVATE = np.asarray(
    PHASE33_OOF_PRIVATE,
    dtype=np.float64,
).reshape(-1)

PHASE39_PHASE36_RESIDUAL_OOF_PRIVATE = np.asarray(
    PHASE36_RAW_RESIDUAL_OOF_PRIVATE,
    dtype=np.float64,
).reshape(-1)

for array in [
    PHASE39_BASELINE_OOF_PRIVATE,
    PHASE39_PHASE33_COMPONENT_OOF_PRIVATE,
    PHASE39_PHASE36_RESIDUAL_OOF_PRIVATE,
]:
    assert array.shape == (1362,)
    assert np.isfinite(array).all()

assert (
    (PHASE39_BASELINE_OOF_PRIVATE > 0).all()
    and
    (PHASE39_BASELINE_OOF_PRIVATE < 1).all()
)

assert (
    (
        PHASE39_PHASE33_COMPONENT_OOF_PRIVATE
        > 0
    ).all()
    and
    (
        PHASE39_PHASE33_COMPONENT_OOF_PRIVATE
        < 1
    ).all()
)


phase39_baseline_logit = logit(
    np.clip(
        PHASE39_BASELINE_OOF_PRIVATE,
        phase39_probability_clip,
        1.0 - phase39_probability_clip,
    )
)

phase39_phase33_component_logit = logit(
    np.clip(
        PHASE39_PHASE33_COMPONENT_OOF_PRIVATE,
        phase39_probability_clip,
        1.0 - phase39_probability_clip,
    )
)

# Transport the standalone Phase33 component onto the exact
# Phase12c anchor used by Phase36/Phase37.
PHASE39_PHASE33_RESIDUAL_OOF_PRIVATE = (
    phase39_phase33_component_logit
    - phase39_baseline_logit
)


# --------------------------------------------------------------
# 3. Verify exact Phase36 OOF reconstruction
# --------------------------------------------------------------

phase39_reconstructed_phase36 = expit(
    phase39_baseline_logit
    + 0.75
    * PHASE39_PHASE36_RESIDUAL_OOF_PRIVATE
)

phase39_phase36_reference = np.asarray(
    PHASE36_OOF_PRIVATE,
    dtype=np.float64,
).reshape(-1)

phase39_phase36_reconstruction_error = {
    "maximum": float(
        np.max(
            np.abs(
                phase39_reconstructed_phase36
                - phase39_phase36_reference
            )
        )
    ),
    "mean": float(
        np.mean(
            np.abs(
                phase39_reconstructed_phase36
                - phase39_phase36_reference
            )
        )
    ),
}

assert (
    phase39_phase36_reconstruction_error[
        "maximum"
    ]
    <= 1e-8
), {
    "message": (
        "Phase36 OOF residual reconstruction failed."
    ),
    "error": (
        phase39_phase36_reconstruction_error
    ),
}


# --------------------------------------------------------------
# 4. Apply the frozen Phase39 stack
# --------------------------------------------------------------

phase39_specification = (
    PHASE39_SELECTED_SPECIFICATION_PRIVATE
)

PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE = (
    phase39_specification[
        "phase36_weight"
    ]
    * PHASE39_PHASE36_RESIDUAL_OOF_PRIVATE
    + phase39_specification[
        "phase33_weight"
    ]
    * PHASE39_PHASE33_RESIDUAL_OOF_PRIVATE
)

phase39_uncertainty = np.clip(
    4.0
    * PHASE39_BASELINE_OOF_PRIVATE
    * (
        1.0
        - PHASE39_BASELINE_OOF_PRIVATE
    ),
    0.0,
    1.0,
)

PHASE39_EFFECTIVE_ALPHA_OOF_PRIVATE = (
    phase39_specification["alpha"]
    * np.power(
        phase39_uncertainty,
        phase39_specification[
            "uncertainty_exponent"
        ],
    )
)

PHASE39_BOUNDED_RESIDUAL_OOF_PRIVATE = (
    np.clip(
        PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE,
        -phase39_specification[
            "delta_cap"
        ],
        phase39_specification[
            "delta_cap"
        ],
    )
)

PHASE39_FINAL_LOGIT_OOF_PRIVATE = (
    phase39_baseline_logit
    + PHASE39_EFFECTIVE_ALPHA_OOF_PRIVATE
    * PHASE39_BOUNDED_RESIDUAL_OOF_PRIVATE
)

PHASE39_OOF_PRIVATE = expit(
    PHASE39_FINAL_LOGIT_OOF_PRIVATE
)

assert PHASE39_OOF_PRIVATE.shape == (
    1362,
)

assert np.isfinite(
    PHASE39_OOF_PRIVATE
).all()

assert (
    (PHASE39_OOF_PRIVATE > 0).all()
    and
    (PHASE39_OOF_PRIVATE < 1).all()
)


# --------------------------------------------------------------
# 5. Metric utilities
# --------------------------------------------------------------

def phase39_outer_metrics(
    labels,
    probabilities,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    result = {
        "log_loss": float(
            log_loss(
                labels,
                probabilities,
            )
        ),
        "mean_probability": float(
            np.mean(probabilities)
        ),
    }

    if np.unique(labels).size == 2:
        result["auroc"] = float(
            roc_auc_score(
                labels,
                probabilities,
            )
        )
    else:
        result["auroc"] = None

    return result


phase39_baseline_metrics = (
    phase39_outer_metrics(
        PHASE39_LABELS_PRIVATE,
        PHASE39_BASELINE_OOF_PRIVATE,
    )
)

phase39_phase33_component_metrics = (
    phase39_outer_metrics(
        PHASE39_LABELS_PRIVATE,
        PHASE39_PHASE33_COMPONENT_OOF_PRIVATE,
    )
)

phase39_phase36_metrics = (
    phase39_outer_metrics(
        PHASE39_LABELS_PRIVATE,
        PHASE36_OOF_PRIVATE,
    )
)

phase39_phase37_metrics = (
    phase39_outer_metrics(
        PHASE39_LABELS_PRIVATE,
        PHASE37_OOF_PRIVATE,
    )
)

phase39_final_metrics = (
    phase39_outer_metrics(
        PHASE39_LABELS_PRIVATE,
        PHASE39_OOF_PRIVATE,
    )
)


# --------------------------------------------------------------
# 6. Exact fold coverage and metrics
# --------------------------------------------------------------

phase39_fold_coverage = np.zeros(
    1362,
    dtype=np.int64,
)

phase39_fold_reports = []

for fold in range(3):
    partition = PHASE19_PARTITIONS[fold]

    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    phase39_fold_coverage[
        outer_valid_indices
    ] += 1

    fold_labels = (
        PHASE39_LABELS_PRIVATE[
            outer_valid_indices
        ]
    )

    baseline_metrics = (
        phase39_outer_metrics(
            fold_labels,
            PHASE39_BASELINE_OOF_PRIVATE[
                outer_valid_indices
            ],
        )
    )

    phase33_metrics = (
        phase39_outer_metrics(
            fold_labels,
            PHASE39_PHASE33_COMPONENT_OOF_PRIVATE[
                outer_valid_indices
            ],
        )
    )

    phase36_metrics = (
        phase39_outer_metrics(
            fold_labels,
            np.asarray(
                PHASE36_OOF_PRIVATE
            )[outer_valid_indices],
        )
    )

    candidate_metrics = (
        phase39_outer_metrics(
            fold_labels,
            PHASE39_OOF_PRIVATE[
                outer_valid_indices
            ],
        )
    )

    phase39_fold_reports.append({
        "fold": fold,
        "n": int(
            len(outer_valid_indices)
        ),
        "baseline_log_loss": (
            baseline_metrics["log_loss"]
        ),
        "phase33_component_log_loss": (
            phase33_metrics["log_loss"]
        ),
        "phase36_log_loss": (
            phase36_metrics["log_loss"]
        ),
        "phase39_log_loss": (
            candidate_metrics["log_loss"]
        ),
        "log_loss_improvement": float(
            baseline_metrics["log_loss"]
            - candidate_metrics["log_loss"]
        ),
        "phase39_gain_over_phase36": float(
            phase36_metrics["log_loss"]
            - candidate_metrics["log_loss"]
        ),
        "baseline_auroc": (
            baseline_metrics["auroc"]
        ),
        "phase33_component_auroc": (
            phase33_metrics["auroc"]
        ),
        "phase36_auroc": (
            phase36_metrics["auroc"]
        ),
        "phase39_auroc": (
            candidate_metrics["auroc"]
        ),
        "auroc_improvement": float(
            candidate_metrics["auroc"]
            - baseline_metrics["auroc"]
        ),
    })


assert np.all(
    phase39_fold_coverage == 1
), {
    "message": (
        "Phase39 outer fold coverage is not exact."
    ),
    "coverage_counts": {
        int(value): int(
            np.sum(
                phase39_fold_coverage
                == value
            )
        )
        for value in np.unique(
            phase39_fold_coverage
        )
    },
}


# --------------------------------------------------------------
# 7. Major acquisition groups
# --------------------------------------------------------------

phase39_major_group_reports = []
phase39_maximum_major_group_harm = 0.0

for group in np.unique(
    PHASE39_GROUPS_PRIVATE
):
    mask = (
        PHASE39_GROUPS_PRIVATE == group
    )

    group_n = int(np.sum(mask))

    if group_n < 30:
        continue

    labels = PHASE39_LABELS_PRIVATE[mask]

    baseline_metrics = (
        phase39_outer_metrics(
            labels,
            PHASE39_BASELINE_OOF_PRIVATE[
                mask
            ],
        )
    )

    phase36_metrics = (
        phase39_outer_metrics(
            labels,
            np.asarray(
                PHASE36_OOF_PRIVATE
            )[mask],
        )
    )

    candidate_metrics = (
        phase39_outer_metrics(
            labels,
            PHASE39_OOF_PRIVATE[mask],
        )
    )

    log_loss_change = float(
        candidate_metrics["log_loss"]
        - baseline_metrics["log_loss"]
    )

    phase39_maximum_major_group_harm = max(
        phase39_maximum_major_group_harm,
        log_loss_change,
    )

    phase39_major_group_reports.append({
        "group": (
            int(group)
            if isinstance(
                group,
                (int, np.integer),
            )
            else str(group)
        ),
        "n": group_n,
        "baseline_log_loss": (
            baseline_metrics["log_loss"]
        ),
        "phase36_log_loss": (
            phase36_metrics["log_loss"]
        ),
        "phase39_log_loss": (
            candidate_metrics["log_loss"]
        ),
        "log_loss_change": (
            log_loss_change
        ),
        "baseline_auroc": (
            baseline_metrics["auroc"]
        ),
        "phase36_auroc": (
            phase36_metrics["auroc"]
        ),
        "phase39_auroc": (
            candidate_metrics["auroc"]
        ),
    })


# --------------------------------------------------------------
# 8. Calibration and update diagnostics
# --------------------------------------------------------------

phase39_calibration_model = (
    LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
)

phase39_calibration_model.fit(
    PHASE39_FINAL_LOGIT_OOF_PRIVATE.reshape(
        -1,
        1,
    ),
    PHASE39_LABELS_PRIVATE,
)

phase39_calibration_intercept = float(
    phase39_calibration_model.intercept_[0]
)

phase39_calibration_slope = float(
    phase39_calibration_model.coef_[0, 0]
)

phase39_residual_correlation = float(
    np.corrcoef(
        PHASE39_PHASE36_RESIDUAL_OOF_PRIVATE,
        PHASE39_PHASE33_RESIDUAL_OOF_PRIVATE,
    )[0, 1]
)

phase39_cap_activation_fraction = float(
    np.mean(
        np.abs(
            PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE
        )
        > phase39_specification[
            "delta_cap"
        ]
    )
)


# --------------------------------------------------------------
# 9. Eligibility for shift stress
# --------------------------------------------------------------

phase39_log_loss_gain = float(
    phase39_baseline_metrics["log_loss"]
    - phase39_final_metrics["log_loss"]
)

phase39_auroc_gain = float(
    phase39_final_metrics["auroc"]
    - phase39_baseline_metrics["auroc"]
)

phase39_log_loss_gain_over_phase36 = float(
    phase39_phase36_metrics["log_loss"]
    - phase39_final_metrics["log_loss"]
)

phase39_auroc_gain_over_phase36 = float(
    phase39_final_metrics["auroc"]
    - phase39_phase36_metrics["auroc"]
)

phase39_log_loss_gain_over_phase37 = float(
    phase39_phase37_metrics["log_loss"]
    - phase39_final_metrics["log_loss"]
)

phase39_fold_wins = int(
    sum(
        record["log_loss_improvement"]
        > 0.0
        for record in phase39_fold_reports
    )
)

phase39_worst_fold_excess = float(
    max(
        0.0,
        max(
            -record[
                "log_loss_improvement"
            ]
            for record
            in phase39_fold_reports
        ),
    )
)

phase39_eligibility_thresholds = {
    "minimum_log_loss_gain": 0.006,
    "minimum_auroc_gain": 0.003,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.003,
    "maximum_major_group_harm": 0.015,
    "calibration_slope_range": [
        0.8,
        1.2,
    ],
    "minimum_log_loss_gain_over_phase37": 0.001,
    "maximum_log_loss_excess_over_phase36": 0.001,
    "maximum_auroc_deficit_vs_phase36": 0.0005,
}

phase39_stress_eligible = bool(
    phase39_log_loss_gain
    >= phase39_eligibility_thresholds[
        "minimum_log_loss_gain"
    ]
    and phase39_auroc_gain
    >= phase39_eligibility_thresholds[
        "minimum_auroc_gain"
    ]
    and phase39_fold_wins
    >= phase39_eligibility_thresholds[
        "minimum_fold_wins"
    ]
    and phase39_worst_fold_excess
    <= phase39_eligibility_thresholds[
        "maximum_worst_fold_excess"
    ]
    and max(
        0.0,
        phase39_maximum_major_group_harm,
    )
    <= phase39_eligibility_thresholds[
        "maximum_major_group_harm"
    ]
    and (
        phase39_eligibility_thresholds[
            "calibration_slope_range"
        ][0]
        <= phase39_calibration_slope
        <= phase39_eligibility_thresholds[
            "calibration_slope_range"
        ][1]
    )
    and phase39_log_loss_gain_over_phase37
    >= phase39_eligibility_thresholds[
        "minimum_log_loss_gain_over_phase37"
    ]
    and (
        phase39_final_metrics["log_loss"]
        - phase39_phase36_metrics["log_loss"]
    )
    <= phase39_eligibility_thresholds[
        "maximum_log_loss_excess_over_phase36"
    ]
    and (
        phase39_phase36_metrics["auroc"]
        - phase39_final_metrics["auroc"]
    )
    <= phase39_eligibility_thresholds[
        "maximum_auroc_deficit_vs_phase36"
    ]
)


# --------------------------------------------------------------
# 10. Persist private stack state
# --------------------------------------------------------------

PHASE39_PRIVATE_DIRECTORY = Path(
    "/kaggle/working/"
    "phase39_private_checkpoint"
)

PHASE39_PRIVATE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

np.save(
    PHASE39_PRIVATE_DIRECTORY
    / "phase39_oof_float64.npy",
    PHASE39_OOF_PRIVATE,
    allow_pickle=False,
)

np.save(
    PHASE39_PRIVATE_DIRECTORY
    / "phase39_combined_residual_float64.npy",
    PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE,
    allow_pickle=False,
)

phase39_metadata = {
    "schema_version": 1,
    "phase": (
        "phase39_phase33_phase36_"
        "robust_residual_stack"
    ),
    "specification": (
        phase39_specification
    ),
    "component_sources": {
        "baseline": (
            "PHASE19_PHASE12C_OOF_PRIVATE"
        ),
        "phase33": (
            "PHASE33_OOF_PRIVATE"
        ),
        "phase36_residual": (
            "PHASE36_RAW_RESIDUAL_OOF_PRIVATE"
        ),
    },
    "deployment_formula": (
        "z_final=z_phase12c+alpha*"
        "uncertainty^gamma*clip("
        "w36*r36+w33*r33,-cap,cap)"
    ),
    "include_private_oof_in_submission": False,
}

(
    PHASE39_PRIVATE_DIRECTORY
    / "phase39_metadata.json"
).write_text(
    json.dumps(
        phase39_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------
# 11. Sanitized report
# --------------------------------------------------------------

def phase39_round_record(record):
    return {
        key: (
            round(value, 6)
            if isinstance(value, float)
            else value
        )
        for key, value in record.items()
    }


phase39_report = {
    "phase": (
        "phase39_phase33_phase36_"
        "robust_residual_stack"
    ),
    "status": (
        "eligible_for_shift_stress"
        if phase39_stress_eligible
        else "not_promoted"
    ),
    "frozen_specification": (
        phase39_specification
    ),
    "component_contract": {
        "phase36_oof_reconstruction_error": {
            key: round(value, 12)
            for key, value
            in phase39_phase36_reconstruction_error.items()
        },
        "phase33_component_transport": (
            "standalone_component_logit_"
            "minus_exact_phase12c_logit"
        ),
        "component_residual_correlation": round(
            phase39_residual_correlation,
            6,
        ),
    },
    "baseline": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value
        in phase39_baseline_metrics.items()
    },
    "phase33_component": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value
        in phase39_phase33_component_metrics.items()
    },
    "phase36": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value
        in phase39_phase36_metrics.items()
    },
    "phase37": {
        key: (
            round(value, 6)
            if value is not None
            else None
        )
        for key, value
        in phase39_phase37_metrics.items()
    },
    "phase39": {
        **{
            key: (
                round(value, 6)
                if value is not None
                else None
            )
            for key, value
            in phase39_final_metrics.items()
        },
        "calibration_intercept": round(
            phase39_calibration_intercept,
            6,
        ),
        "calibration_slope": round(
            phase39_calibration_slope,
            6,
        ),
    },
    "improvements": {
        "log_loss_gain_over_phase12c": round(
            phase39_log_loss_gain,
            6,
        ),
        "auroc_gain_over_phase12c": round(
            phase39_auroc_gain,
            6,
        ),
        "log_loss_gain_over_phase36": round(
            phase39_log_loss_gain_over_phase36,
            6,
        ),
        "auroc_gain_over_phase36": round(
            phase39_auroc_gain_over_phase36,
            6,
        ),
        "log_loss_gain_over_phase37": round(
            phase39_log_loss_gain_over_phase37,
            6,
        ),
        "fold_wins": phase39_fold_wins,
        "worst_fold_excess": round(
            phase39_worst_fold_excess,
            6,
        ),
        "maximum_major_group_harm": round(
            max(
                0.0,
                phase39_maximum_major_group_harm,
            ),
            6,
        ),
    },
    "fold_metrics": [
        phase39_round_record(record)
        for record in phase39_fold_reports
    ],
    "major_acquisition_group_metrics": [
        phase39_round_record(record)
        for record
        in phase39_major_group_reports
    ],
    "update_magnitude": {
        "mean_absolute_combined_residual": round(
            float(
                np.mean(
                    np.abs(
                        PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE
                    )
                )
            ),
            6,
        ),
        "q95_absolute_combined_residual": round(
            float(
                np.quantile(
                    np.abs(
                        PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE
                    ),
                    0.95,
                )
            ),
            6,
        ),
        "cap_activation_fraction": round(
            phase39_cap_activation_fraction,
            6,
        ),
        "mean_absolute_probability_update": round(
            float(
                np.mean(
                    np.abs(
                        PHASE39_OOF_PRIVATE
                        - PHASE39_BASELINE_OOF_PRIVATE
                    )
                )
            ),
            6,
        ),
        "q95_absolute_probability_update": round(
            float(
                np.quantile(
                    np.abs(
                        PHASE39_OOF_PRIVATE
                        - PHASE39_BASELINE_OOF_PRIVATE
                    ),
                    0.95,
                )
            ),
            6,
        ),
    },
    "eligibility_thresholds": (
        phase39_eligibility_thresholds
    ),
    "stress_eligible": (
        phase39_stress_eligible
    ),
    "shared_parameters_across_folds": True,
    "fold_specific_group_routing": False,
    "outer_oracle_parameters_used": False,
    "selection_partition": (
        "pooled_monitor_only"
    ),
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "deployment_formula_persisted_privately": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase39_outer_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE39_OOF"
)
print(
    json.dumps(
        phase39_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE39_OOF"
)

BEGIN SANITIZED_PHASE39_OOF
{
  "phase": "phase39_phase33_phase36_robust_residual_stack",
  "status": "not_promoted",
  "frozen_specification": {
    "phase36_weight": 0.75,
    "phase33_weight": 0.25,
    "alpha": 1.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 2.0
  },
  "component_contract": {
    "phase36_oof_reconstruction_error": {
      "maximum": 0.0,
      "mean": 0.0
    },
    "phase33_component_transport": "standalone_component_logit_minus_exact_phase12c_logit",
    "component_residual_correlation": -0.002788
  },
  "baseline": {
    "log_loss": 0.307616,
    "mean_probability": 0.530698,
    "auroc": 0.938914
  },
  "phase33_component": {
    "log_loss": 0.358791,
    "mean_probability": 0.558999,
    "auroc": 0.923527
  },
  "phase36": {
    "log_loss": 0.295657,
    "mean_probability": 0.553493,
    "auroc": 0.944143
  },
  "phase37": {
    "log_loss": 0.297446,
    "mean_probability": 0.542359,
    "auroc": 0.942846
  },
  "phase39": {
    "log_loss": 0.293027,
 

In [87]:
# Phase39 Cell 122D
# Fixed-candidate acquisition-shift stress.
# This cell performs no parameter selection or modification.

import json
import time

import numpy as np

from sklearn.metrics import roc_auc_score


phase39_stress_started = time.perf_counter()

PHASE39_STRESS_CONFIG = {
    "log_loss_bootstrap_replicates": 6000,
    "auroc_bootstrap_replicates": 3000,
    "dirichlet_replicates": 10000,
    "random_seed": 391601,
    "dirichlet_concentrations": [
        100.0,
        20.0,
        5.0,
    ],
    "total_variation_radii": [
        0.05,
        0.10,
        0.20,
        0.30,
    ],
    "thresholds": {
        "minimum_observed_log_loss_gain": 0.006,
        "minimum_within_group_probability_positive": 0.99,
        "minimum_within_group_q05_gain": 0.003,
        "minimum_auroc_probability_positive": 0.95,
        "minimum_modest_shift_probability_positive": 0.99,
        "minimum_substantial_shift_probability_positive": 0.95,
        "minimum_severe_shift_probability_positive": 0.80,
        "minimum_leave_one_group_out_gain": 0.0,
        "minimum_tv_0p10_worst_case_gain": 0.0,
    },
}


# --------------------------------------------------------------
# 1. Fixed candidate contract
# --------------------------------------------------------------

assert (
    PHASE39_SELECTED_SPECIFICATION_PRIVATE
    == {
        "phase36_weight": 0.75,
        "phase33_weight": 0.25,
        "alpha": 1.0,
        "uncertainty_exponent": 0.0,
        "delta_cap": 2.0,
    }
)

labels = np.asarray(
    PHASE39_LABELS_PRIVATE,
    dtype=np.int64,
).reshape(-1)

groups = np.asarray(
    PHASE39_GROUPS_PRIVATE
).reshape(-1)

baseline_probability = np.clip(
    np.asarray(
        PHASE39_BASELINE_OOF_PRIVATE,
        dtype=np.float64,
    ).reshape(-1),
    1e-6,
    1.0 - 1e-6,
)

candidate_probability = np.clip(
    np.asarray(
        PHASE39_OOF_PRIVATE,
        dtype=np.float64,
    ).reshape(-1),
    1e-6,
    1.0 - 1e-6,
)

assert labels.shape == (1362,)
assert groups.shape == (1362,)
assert baseline_probability.shape == (1362,)
assert candidate_probability.shape == (1362,)
assert np.isin(labels, [0, 1]).all()
assert np.isfinite(baseline_probability).all()
assert np.isfinite(candidate_probability).all()


baseline_case_loss = -(
    labels * np.log(baseline_probability)
    + (1 - labels)
    * np.log(1.0 - baseline_probability)
)

candidate_case_loss = -(
    labels * np.log(candidate_probability)
    + (1 - labels)
    * np.log(1.0 - candidate_probability)
)

case_log_loss_gain = (
    baseline_case_loss
    - candidate_case_loss
)

observed_log_loss_gain = float(
    np.mean(case_log_loss_gain)
)

observed_auroc_gain = float(
    roc_auc_score(
        labels,
        candidate_probability,
    )
    - roc_auc_score(
        labels,
        baseline_probability,
    )
)

assert abs(
    observed_log_loss_gain
    - 0.014588
) <= 2e-6

assert abs(
    observed_auroc_gain
    - 0.006661
) <= 2e-6


# --------------------------------------------------------------
# 2. Acquisition-group effects
# --------------------------------------------------------------

unique_groups = np.unique(groups)

group_records = []
group_mean_gains = []
group_empirical_weights = []

for group in unique_groups:
    mask = groups == group
    group_labels = labels[mask]

    group_record = {
        "group": (
            int(group)
            if isinstance(
                group,
                (int, np.integer),
            )
            else str(group)
        ),
        "n": int(np.sum(mask)),
        "empirical_weight": float(
            np.mean(mask)
        ),
        "prevalence": float(
            np.mean(group_labels)
        ),
        "log_loss_gain": float(
            np.mean(
                case_log_loss_gain[mask]
            )
        ),
    }

    if np.unique(group_labels).size == 2:
        group_record["auroc_gain"] = float(
            roc_auc_score(
                group_labels,
                candidate_probability[mask],
            )
            - roc_auc_score(
                group_labels,
                baseline_probability[mask],
            )
        )
    else:
        group_record["auroc_gain"] = None

    group_records.append(group_record)
    group_mean_gains.append(
        group_record["log_loss_gain"]
    )
    group_empirical_weights.append(
        group_record["empirical_weight"]
    )

group_mean_gains = np.asarray(
    group_mean_gains,
    dtype=np.float64,
)

group_empirical_weights = np.asarray(
    group_empirical_weights,
    dtype=np.float64,
)

group_empirical_weights /= np.sum(
    group_empirical_weights
)


# --------------------------------------------------------------
# 3. Within-group log-loss bootstrap
# --------------------------------------------------------------

rng = np.random.default_rng(
    PHASE39_STRESS_CONFIG["random_seed"]
)

group_index_arrays = [
    np.flatnonzero(groups == group)
    for group in unique_groups
]

log_loss_bootstrap_gains = np.empty(
    PHASE39_STRESS_CONFIG[
        "log_loss_bootstrap_replicates"
    ],
    dtype=np.float64,
)

for replicate in range(
    len(log_loss_bootstrap_gains)
):
    gain_sum = 0.0
    sampled_n = 0

    for group_indices in group_index_arrays:
        sampled_indices = rng.choice(
            group_indices,
            size=len(group_indices),
            replace=True,
        )

        gain_sum += float(
            np.sum(
                case_log_loss_gain[
                    sampled_indices
                ]
            )
        )
        sampled_n += len(sampled_indices)

    log_loss_bootstrap_gains[
        replicate
    ] = gain_sum / sampled_n


def phase39_distribution_summary(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "replicate_count": int(len(values)),
        "mean": float(np.mean(values)),
        "standard_deviation": float(
            np.std(values)
        ),
        "q01": float(
            np.quantile(values, 0.01)
        ),
        "q05": float(
            np.quantile(values, 0.05)
        ),
        "q50": float(
            np.quantile(values, 0.50)
        ),
        "q95": float(
            np.quantile(values, 0.95)
        ),
        "q99": float(
            np.quantile(values, 0.99)
        ),
        "probability_positive": float(
            np.mean(values > 0.0)
        ),
        "probability_gain_at_least_0p003": float(
            np.mean(values >= 0.003)
        ),
    }


within_group_summary = (
    phase39_distribution_summary(
        log_loss_bootstrap_gains
    )
)


# --------------------------------------------------------------
# 4. Group-and-label-stratified AUROC bootstrap
# --------------------------------------------------------------

strata = []

for group in unique_groups:
    for label_value in [0, 1]:
        indices = np.flatnonzero(
            (groups == group)
            & (labels == label_value)
        )

        if len(indices) > 0:
            strata.append(indices)

auroc_bootstrap_gains = np.empty(
    PHASE39_STRESS_CONFIG[
        "auroc_bootstrap_replicates"
    ],
    dtype=np.float64,
)

for replicate in range(
    len(auroc_bootstrap_gains)
):
    sampled_parts = [
        rng.choice(
            indices,
            size=len(indices),
            replace=True,
        )
        for indices in strata
    ]

    sampled_indices = np.concatenate(
        sampled_parts
    )

    sampled_labels = labels[
        sampled_indices
    ]

    auroc_bootstrap_gains[
        replicate
    ] = (
        roc_auc_score(
            sampled_labels,
            candidate_probability[
                sampled_indices
            ],
        )
        - roc_auc_score(
            sampled_labels,
            baseline_probability[
                sampled_indices
            ],
        )
    )

auroc_bootstrap_summary = (
    phase39_distribution_summary(
        auroc_bootstrap_gains
    )
)


# --------------------------------------------------------------
# 5. Leave-one-acquisition-group-out
# --------------------------------------------------------------

leave_one_group_out_records = []

for held_out_group in unique_groups:
    retained = groups != held_out_group

    leave_one_group_out_records.append({
        "held_out_group": (
            int(held_out_group)
            if isinstance(
                held_out_group,
                (int, np.integer),
            )
            else str(held_out_group)
        ),
        "retained_n": int(
            np.sum(retained)
        ),
        "log_loss_gain": float(
            np.mean(
                case_log_loss_gain[
                    retained
                ]
            )
        ),
    })

minimum_leave_one_group_out_gain = float(
    min(
        record["log_loss_gain"]
        for record
        in leave_one_group_out_records
    )
)


# --------------------------------------------------------------
# 6. Dirichlet acquisition-mixture shifts
# --------------------------------------------------------------

dirichlet_shift_records = []

interpretations = {
    100.0: "modest_shift",
    20.0: "substantial_shift",
    5.0: "severe_shift",
}

for concentration in (
    PHASE39_STRESS_CONFIG[
        "dirichlet_concentrations"
    ]
):
    parameters = (
        concentration
        * group_empirical_weights
        + 0.10
    )

    mixture_weights = rng.dirichlet(
        parameters,
        size=PHASE39_STRESS_CONFIG[
            "dirichlet_replicates"
        ],
    )

    mixture_gains = (
        mixture_weights
        @ group_mean_gains
    )

    summary = (
        phase39_distribution_summary(
            mixture_gains
        )
    )

    dirichlet_shift_records.append({
        "concentration": float(
            concentration
        ),
        "interpretation": (
            interpretations[
                float(concentration)
            ]
        ),
        **summary,
    })


# --------------------------------------------------------------
# 7. Adversarial total-variation mixture shifts
# --------------------------------------------------------------

def phase39_worst_case_tv_gain(
    empirical_weights,
    group_gains,
    radius,
):
    weights = np.asarray(
        empirical_weights,
        dtype=np.float64,
    ).copy()

    gains = np.asarray(
        group_gains,
        dtype=np.float64,
    )

    source_order = np.argsort(
        gains
    )[::-1]

    target_order = np.argsort(
        gains
    )

    remaining_budget = float(radius)
    source_position = 0
    target_position = 0
    changed_groups = set()

    while (
        remaining_budget > 1e-12
        and source_position
        < len(source_order)
        and target_position
        < len(target_order)
    ):
        source = int(
            source_order[
                source_position
            ]
        )
        target = int(
            target_order[
                target_position
            ]
        )

        if gains[source] <= gains[target]:
            break

        available_source_mass = (
            weights[source]
        )

        available_target_capacity = (
            1.0 - weights[target]
        )

        movement = min(
            remaining_budget,
            available_source_mass,
            available_target_capacity,
        )

        if movement > 0:
            weights[source] -= movement
            weights[target] += movement
            remaining_budget -= movement
            changed_groups.add(source)
            changed_groups.add(target)

        if weights[source] <= 1e-12:
            source_position += 1

        if (
            1.0 - weights[target]
            <= 1e-12
        ):
            target_position += 1

        if movement <= 1e-15:
            break

    return {
        "worst_case_log_loss_gain": float(
            np.dot(weights, gains)
        ),
        "changed_group_count": int(
            len(changed_groups)
        ),
        "unused_shift_budget": float(
            remaining_budget
        ),
    }


adversarial_shift_records = []

for radius in (
    PHASE39_STRESS_CONFIG[
        "total_variation_radii"
    ]
):
    result = (
        phase39_worst_case_tv_gain(
            group_empirical_weights,
            group_mean_gains,
            radius,
        )
    )

    adversarial_shift_records.append({
        "total_variation_radius": float(
            radius
        ),
        **result,
    })


# --------------------------------------------------------------
# 8. Stress decision
# --------------------------------------------------------------

dirichlet_by_name = {
    record["interpretation"]: record
    for record in dirichlet_shift_records
}

tv_by_radius = {
    round(
        record[
            "total_variation_radius"
        ],
        2,
    ): record
    for record in adversarial_shift_records
}

thresholds = PHASE39_STRESS_CONFIG[
    "thresholds"
]

phase39_stress_passed = bool(
    observed_log_loss_gain
    >= thresholds[
        "minimum_observed_log_loss_gain"
    ]
    and within_group_summary[
        "probability_positive"
    ]
    >= thresholds[
        "minimum_within_group_probability_positive"
    ]
    and within_group_summary["q05"]
    >= thresholds[
        "minimum_within_group_q05_gain"
    ]
    and auroc_bootstrap_summary[
        "probability_positive"
    ]
    >= thresholds[
        "minimum_auroc_probability_positive"
    ]
    and dirichlet_by_name[
        "modest_shift"
    ]["probability_positive"]
    >= thresholds[
        "minimum_modest_shift_probability_positive"
    ]
    and dirichlet_by_name[
        "substantial_shift"
    ]["probability_positive"]
    >= thresholds[
        "minimum_substantial_shift_probability_positive"
    ]
    and dirichlet_by_name[
        "severe_shift"
    ]["probability_positive"]
    >= thresholds[
        "minimum_severe_shift_probability_positive"
    ]
    and minimum_leave_one_group_out_gain
    >= thresholds[
        "minimum_leave_one_group_out_gain"
    ]
    and tv_by_radius[0.10][
        "worst_case_log_loss_gain"
    ]
    >= thresholds[
        "minimum_tv_0p10_worst_case_gain"
    ]
)


# --------------------------------------------------------------
# 9. Sanitized output
# --------------------------------------------------------------

phase39_stress_report = {
    "phase": (
        "phase39_fixed_candidate_"
        "acquisition_shift_stress"
    ),
    "status": (
        "runtime_smoke_eligible"
        if phase39_stress_passed
        else "stress_failed_do_not_package"
    ),
    "fixed_candidate": {
        "specification": (
            PHASE39_SELECTED_SPECIFICATION_PRIVATE
        ),
        "log_loss": 0.293027,
        "auroc": 0.945575,
        "observed_log_loss_gain": (
            observed_log_loss_gain
        ),
        "observed_auroc_gain": (
            observed_auroc_gain
        ),
        "maximum_major_group_harm": (
            0.055842
        ),
    },
    "within_group_log_loss_bootstrap": (
        within_group_summary
    ),
    "group_label_stratified_auroc_bootstrap": (
        auroc_bootstrap_summary
    ),
    "leave_one_acquisition_group_out": {
        "minimum_log_loss_gain": (
            minimum_leave_one_group_out_gain
        ),
        "records": (
            leave_one_group_out_records
        ),
    },
    "dirichlet_group_mixture_shift": (
        dirichlet_shift_records
    ),
    "adversarial_group_mixture_shift": (
        adversarial_shift_records
    ),
    "acquisition_group_effects": (
        group_records
    ),
    "stress_thresholds": thresholds,
    "stress_gate_passed": (
        phase39_stress_passed
    ),
    "candidate_parameters_modified": False,
    "outer_labels_used_only_for_fixed_candidate_stress": True,
    "case_level_predictions_exported": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase39_stress_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE39_STRESS"
)
print(
    json.dumps(
        phase39_stress_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE39_STRESS"
)

BEGIN SANITIZED_PHASE39_STRESS
{
  "phase": "phase39_fixed_candidate_acquisition_shift_stress",
  "status": "stress_failed_do_not_package",
  "fixed_candidate": {
    "specification": {
      "phase36_weight": 0.75,
      "phase33_weight": 0.25,
      "alpha": 1.0,
      "uncertainty_exponent": 0.0,
      "delta_cap": 2.0
    },
    "log_loss": 0.293027,
    "auroc": 0.945575,
    "observed_log_loss_gain": 0.014588462271873989,
    "observed_auroc_gain": 0.0066607894994614725,
    "maximum_major_group_harm": 0.055842
  },
  "within_group_log_loss_bootstrap": {
    "replicate_count": 6000,
    "mean": 0.014735042597238634,
    "standard_deviation": 0.006284574378045683,
    "q01": 3.787610790610933e-05,
    "q05": 0.004316816214330217,
    "q50": 0.014796966290605309,
    "q95": 0.024865510968780674,
    "q99": 0.029012146990790272,
    "probability_positive": 0.99,
    "probability_gain_at_least_0p003": 0.9683333333333334
  },
  "group_label_stratified_auroc_bootstrap": {
    "replicat

In [88]:
# Phase39 Cell 123A
# Discover and validate the exact live deployment states needed
# before building the separate experimental runtime.

from pathlib import Path
import inspect
import json
import time

import numpy as np


phase39_deployment_audit_started = (
    time.perf_counter()
)


# --------------------------------------------------------------
# 1. Recursively summarize deployment objects
# --------------------------------------------------------------

def phase39_state_summary(value):
    summary = {
        "type": type(value).__name__,
        "module": type(value).__module__,
    }

    if isinstance(value, dict):
        summary["container_length"] = len(value)
        summary["keys"] = sorted(
            str(key)
            for key in value.keys()
        )[:80]

        if "fold" in value:
            try:
                summary["fold"] = int(
                    value["fold"]
                )
            except Exception:
                pass

    elif isinstance(value, (list, tuple)):
        summary["container_length"] = len(value)
        summary["element_types"] = sorted(
            set(
                type(item).__name__
                for item in value
            )
        )

        folds = []

        for item in value:
            if (
                isinstance(item, dict)
                and "fold" in item
            ):
                try:
                    folds.append(
                        int(item["fold"])
                    )
                except Exception:
                    pass

        if folds:
            summary["folds"] = sorted(
                set(folds)
            )

    elif isinstance(value, np.ndarray):
        summary["shape"] = [
            int(dimension)
            for dimension in value.shape
        ]
        summary["dtype"] = str(value.dtype)

    return summary


def phase39_count_state_assets(
    value,
    depth=0,
    seen=None,
):
    if seen is None:
        seen = set()

    if depth > 7:
        return {
            "xgboost_booster_count": 0,
            "torch_module_count": 0,
            "tensor_count": 0,
            "tensor_element_count": 0,
            "preprocessor_state_count": 0,
        }

    object_id = id(value)

    if object_id in seen:
        return {
            "xgboost_booster_count": 0,
            "torch_module_count": 0,
            "tensor_count": 0,
            "tensor_element_count": 0,
            "preprocessor_state_count": 0,
        }

    seen.add(object_id)

    result = {
        "xgboost_booster_count": 0,
        "torch_module_count": 0,
        "tensor_count": 0,
        "tensor_element_count": 0,
        "preprocessor_state_count": 0,
    }

    module_name = type(value).__module__
    type_name = type(value).__name__

    if (
        module_name.startswith("xgboost")
        and type_name == "Booster"
    ):
        result[
            "xgboost_booster_count"
        ] += 1
        return result

    if (
        hasattr(value, "state_dict")
        and hasattr(value, "parameters")
        and module_name.startswith("torch")
    ):
        result["torch_module_count"] += 1

        try:
            state_dictionary = (
                value.state_dict()
            )

            for tensor in (
                state_dictionary.values()
            ):
                if hasattr(tensor, "numel"):
                    result["tensor_count"] += 1
                    result[
                        "tensor_element_count"
                    ] += int(tensor.numel())

        except Exception:
            pass

        return result

    if (
        hasattr(value, "numel")
        and hasattr(value, "detach")
    ):
        try:
            result["tensor_count"] += 1
            result[
                "tensor_element_count"
            ] += int(value.numel())
        except Exception:
            pass

        return result

    if isinstance(value, dict):
        lowered_keys = {
            str(key).lower()
            for key in value.keys()
        }

        if any(
            term in lowered_keys
            for term in [
                "median",
                "iqr",
                "scale",
                "center",
                "pca",
                "components",
                "selected_indices",
                "winsor_lower",
                "winsor_upper",
            ]
        ):
            result[
                "preprocessor_state_count"
            ] += 1

        for key, child in value.items():
            lowered_key = str(key).lower()

            if any(
                excluded in lowered_key
                for excluded in [
                    "oof",
                    "prediction",
                    "probability",
                    "labels",
                    "indices",
                ]
            ):
                continue

            child_result = (
                phase39_count_state_assets(
                    child,
                    depth + 1,
                    seen,
                )
            )

            for count_name in result:
                result[count_name] += (
                    child_result[
                        count_name
                    ]
                )

        return result

    if isinstance(value, (list, tuple)):
        if len(value) > 500:
            return result

        for child in value:
            child_result = (
                phase39_count_state_assets(
                    child,
                    depth + 1,
                    seen,
                )
            )

            for count_name in result:
                result[count_name] += (
                    child_result[
                        count_name
                    ]
                )

    return result


# --------------------------------------------------------------
# 2. Discover Phase33 and Phase36 deployment objects
# --------------------------------------------------------------

deployment_name_terms = [
    "deployment",
    "state",
    "states",
    "final_model",
    "final_models",
    "booster",
    "boosters",
    "preprocessor",
    "refit",
    "checkpoint",
]

excluded_name_terms = [
    "config",
    "report",
    "record",
    "candidate",
    "oof",
    "prediction",
    "probability",
    "feature_cache",
    "diagnostic",
]

phase39_live_state_candidates = []

for global_name, value in list(
    globals().items()
):
    lowered_name = global_name.lower()

    if not (
        lowered_name.startswith("phase33")
        or lowered_name.startswith("phase36")
        or lowered_name.startswith(
            "phase39"
        )
    ):
        continue

    if not any(
        term in lowered_name
        for term in deployment_name_terms
    ):
        continue

    if any(
        term in lowered_name
        for term in excluded_name_terms
    ):
        continue

    asset_counts = (
        phase39_count_state_assets(value)
    )

    if (
        sum(asset_counts.values()) == 0
        and not isinstance(
            value,
            (dict, list, tuple),
        )
    ):
        continue

    phase39_live_state_candidates.append({
        "name": global_name,
        "summary": (
            phase39_state_summary(value)
        ),
        "asset_counts": asset_counts,
    })


# --------------------------------------------------------------
# 3. Discover relevant runtime callables
# --------------------------------------------------------------

callable_name_terms = [
    "case_features",
    "radiomics",
    "p3ca",
    "dense",
    "bilateral",
    "preprocess",
    "transform",
    "build_features",
    "feature_matrix",
    "new_model",
    "predict",
]

phase39_runtime_callables = []

for global_name, value in list(
    globals().items()
):
    lowered_name = global_name.lower()

    if not callable(value):
        continue

    if not any(
        lowered_name.startswith(prefix)
        for prefix in [
            "phase31",
            "phase32",
            "phase33",
            "phase36",
        ]
    ):
        continue

    if not any(
        term in lowered_name
        for term in callable_name_terms
    ):
        continue

    try:
        signature = str(
            inspect.signature(value)
        )
    except Exception:
        signature = "unavailable"

    phase39_runtime_callables.append({
        "name": global_name,
        "signature": signature,
    })

phase39_runtime_callables.sort(
    key=lambda record: record["name"]
)


# --------------------------------------------------------------
# 4. Inspect persistent checkpoint files
# --------------------------------------------------------------

checkpoint_directory_names = [
    "phase33_private_checkpoint",
    "phase36_private_checkpoint",
    "phase37_private_checkpoint",
    "phase38_private_checkpoint",
    "phase39_private_checkpoint",
]

phase39_checkpoint_audit = {}

for directory_name in (
    checkpoint_directory_names
):
    directory = (
        Path("/kaggle/working")
        / directory_name
    )

    if not directory.is_dir():
        phase39_checkpoint_audit[
            directory_name
        ] = {
            "present": False,
        }
        continue

    files = sorted(
        path
        for path in directory.iterdir()
        if path.is_file()
    )

    phase39_checkpoint_audit[
        directory_name
    ] = {
        "present": True,
        "file_count": len(files),
        "size_mb": round(
            sum(
                path.stat().st_size
                for path in files
            )
            / 1e6,
            4,
        ),
        "files": [
            {
                "name": path.name,
                "size_kb": round(
                    path.stat().st_size
                    / 1e3,
                    3,
                ),
            }
            for path in files
        ],
    }


# --------------------------------------------------------------
# 5. Audit reusable Phase30 offline runtime
# --------------------------------------------------------------

phase30_runtime_candidates = [
    Path(
        "/kaggle/working/"
        "phase30_submission"
    ),
    Path(
        "/kaggle/working/"
        "phase30_zip_rehearsal"
    ),
]

phase39_runtime_base_candidates = []

for directory in phase30_runtime_candidates:
    if not directory.is_dir():
        continue

    phase39_runtime_base_candidates.append({
        "directory": directory.name,
        "main_present": (
            directory.joinpath(
                "main.py"
            ).is_file()
        ),
        "phase12c_main_present": (
            directory.joinpath(
                "phase12c_main.py"
            ).is_file()
        ),
        "dinov3_source_present": (
            directory.joinpath(
                "dinov3"
            ).is_dir()
        ),
        "models_directory_present": (
            directory.joinpath(
                "models"
            ).is_dir()
        ),
    })


# --------------------------------------------------------------
# 6. Required deployment contract
# --------------------------------------------------------------

phase39_required_contract = {
    "phase12c_runtime": True,
    "dinov3_runtime": True,
    "phase31_localized_radiomics_source": any(
        record["name"]
        == "phase31_case_features"
        for record in phase39_runtime_callables
    ),
    "phase32_dense_p3ca_source": any(
        (
            "p3ca" in record["name"].lower()
            or "dense" in record["name"].lower()
        )
        for record in phase39_runtime_callables
    ),
    "phase33_deployment_checkpoint": (
        phase39_checkpoint_audit[
            "phase33_private_checkpoint"
        ]["present"]
    ),
    "phase36_live_booster_state": any(
        record["asset_counts"][
            "xgboost_booster_count"
        ] > 0
        for record
        in phase39_live_state_candidates
        if record["name"].lower().startswith(
            "phase36"
        )
    ),
    "phase39_formula_persisted": (
        phase39_checkpoint_audit[
            "phase39_private_checkpoint"
        ]["present"]
    ),
}

phase39_deployment_ready_for_build = bool(
    all(
        phase39_required_contract.values()
    )
)


# --------------------------------------------------------------
# 7. Sanitized report
# --------------------------------------------------------------

phase39_deployment_audit_report = {
    "phase": (
        "phase39_experimental_runtime_"
        "state_audit"
    ),
    "status": (
        "ready_for_experimental_runtime_build"
        if phase39_deployment_ready_for_build
        else "runtime_state_incomplete"
    ),
    "decision": {
        "default_submission_replacement": False,
        "separate_experimental_archive": True,
        "reason": (
            "best aggregate OOF metrics but "
            "failed adversarial TV=0.10 shift"
        ),
    },
    "live_state_candidates": (
        phase39_live_state_candidates
    ),
    "runtime_callable_count": len(
        phase39_runtime_callables
    ),
    "runtime_callables": (
        phase39_runtime_callables
    ),
    "checkpoint_audit": (
        phase39_checkpoint_audit
    ),
    "runtime_base_candidates": (
        phase39_runtime_base_candidates
    ),
    "required_contract": (
        phase39_required_contract
    ),
    "training_voxel_data_read": False,
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase39_deployment_audit_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE39_DEPLOYMENT_AUDIT"
)
print(
    json.dumps(
        phase39_deployment_audit_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE39_DEPLOYMENT_AUDIT"
)

BEGIN SANITIZED_PHASE39_DEPLOYMENT_AUDIT
{
  "phase": "phase39_experimental_runtime_state_audit",
  "status": "ready_for_experimental_runtime_build",
  "decision": {
    "default_submission_replacement": false,
    "separate_experimental_archive": true,
    "reason": "best aggregate OOF metrics but failed adversarial TV=0.10 shift"
  },
  "live_state_candidates": [
    {
      "name": "PHASE33_FOLD0_FIT_PREPROCESSOR_PRIVATE",
      "summary": {
        "type": "dict",
        "module": "builtins",
        "container_length": 13,
        "keys": [
          "fold",
          "geometry_token_mean",
          "geometry_token_standard_deviation",
          "p3ca_token_mean",
          "p3ca_token_standard_deviation",
          "radiomics_dimension",
          "radiomics_projector",
          "radiomics_scaler",
          "stage_index",
          "summary_dimension",
          "summary_projector",
          "summary_scaler",
          "training_count"
        ],
        "fold": 0
      },
 

In [89]:
# Phase39 Cell 123B
# Persist and verify the complete Phase36 deployment state.
# Private notebook checkpoint only. Do not include OOF arrays,

from pathlib import Path
import hashlib
import json
import time

import joblib
import numpy as np
import xgboost as xgb


phase36_persistence_started = (
    time.perf_counter()
)

PHASE36_PRIVATE_CHECKPOINT_DIRECTORY = Path(
    "/kaggle/working/"
    "phase36_private_checkpoint"
)

PHASE36_PRIVATE_CHECKPOINT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------
# 1. Validate live deployment state
# --------------------------------------------------------------

assert (
    "PHASE36_DEPLOYMENT_STATES_PRIVATE"
    in globals()
)

assert isinstance(
    PHASE36_DEPLOYMENT_STATES_PRIVATE,
    list,
)

assert len(
    PHASE36_DEPLOYMENT_STATES_PRIVATE
) == 3

phase36_state_folds = sorted(
    int(state["fold"])
    for state
    in PHASE36_DEPLOYMENT_STATES_PRIVATE
)

assert phase36_state_folds == [0, 1, 2]


for state in (
    PHASE36_DEPLOYMENT_STATES_PRIVATE
):
    forbidden_keys = [
        str(key)
        for key in state.keys()
        if any(
            term in str(key).lower()
            for term in [
                "label",
                "oof",
                "prediction",
                "patient",
                "uid",
            ]
        )
    ]

    assert not forbidden_keys, {
        "message": (
            "Phase36 deployment state contains "
            "forbidden training/evaluation data."
        ),
        "fold": int(state["fold"]),
        "forbidden_keys": forbidden_keys,
    }


# --------------------------------------------------------------
# 2. Externalize every XGBoost booster
# --------------------------------------------------------------

phase36_saved_booster_files = []
phase36_original_booster_hashes = {}


def phase36_externalize_boosters(
    value,
    path_parts,
):
    if isinstance(value, xgb.Booster):
        safe_path = "_".join(
            str(part)
            .replace(" ", "_")
            .replace("/", "_")
            for part in path_parts
        )

        booster_name = (
            f"{safe_path}.ubj"
        )

        booster_path = (
            PHASE36_PRIVATE_CHECKPOINT_DIRECTORY
            / booster_name
        )

        value.save_model(booster_path)

        raw_bytes = value.save_raw(
            raw_format="ubj"
        )

        phase36_original_booster_hashes[
            booster_name
        ] = hashlib.sha256(
            raw_bytes
        ).hexdigest()

        phase36_saved_booster_files.append(
            booster_name
        )

        return {
            "__phase36_xgboost_booster__": (
                booster_name
            )
        }

    if isinstance(value, dict):
        result = {}

        for key, child in value.items():
            lowered_key = str(key).lower()

            if any(
                forbidden in lowered_key
                for forbidden in [
                    "label",
                    "oof",
                    "prediction",
                    "patient",
                    "uid",
                ]
            ):
                raise AssertionError({
                    "message": (
                        "Forbidden item found inside "
                        "Phase36 deployment state."
                    ),
                    "path": ".".join(
                        [
                            *map(str, path_parts),
                            str(key),
                        ]
                    ),
                })

            result[key] = (
                phase36_externalize_boosters(
                    child,
                    [
                        *path_parts,
                        key,
                    ],
                )
            )

        return result

    if isinstance(value, list):
        return [
            phase36_externalize_boosters(
                child,
                [
                    *path_parts,
                    index,
                ],
            )
            for index, child
            in enumerate(value)
        ]

    if isinstance(value, tuple):
        return {
            "__phase36_tuple__": [
                phase36_externalize_boosters(
                    child,
                    [
                        *path_parts,
                        index,
                    ],
                )
                for index, child
                in enumerate(value)
            ]
        }

    if isinstance(value, np.ndarray):
        assert np.isfinite(value).all(), {
            "message": (
                "Nonfinite array in Phase36 "
                "deployment state."
            ),
            "path": ".".join(
                map(str, path_parts)
            ),
            "shape": list(value.shape),
        }

    return value


phase36_portable_states = (
    phase36_externalize_boosters(
        PHASE36_DEPLOYMENT_STATES_PRIVATE,
        ["phase36_deployment_states"],
    )
)

assert len(
    phase36_saved_booster_files
) == 6, {
    "message": (
        "Expected six Phase36 boosters."
    ),
    "found": len(
        phase36_saved_booster_files
    ),
    "files": (
        phase36_saved_booster_files
    ),
}


# --------------------------------------------------------------
# 3. Persist portable state and runtime metadata
# --------------------------------------------------------------

phase36_portable_state_path = (
    PHASE36_PRIVATE_CHECKPOINT_DIRECTORY
    / "phase36_deployment_state.joblib"
)

joblib.dump(
    {
        "schema_version": 1,
        "phase": (
            "phase36_phase12c_base_margin_"
            "boosted_residual"
        ),
        "states": phase36_portable_states,
        "selected_specification": {
            "maximum_depth": 2,
            "minimum_child_weight": 8.0,
            "l2_regularization": 10.0,
        },
        "shared_scale": 0.75,
        "seed_count": 2,
        "fold_count": 3,
    },
    phase36_portable_state_path,
    compress=3,
)

phase36_metadata = {
    "schema_version": 1,
    "phase": (
        "phase36_phase12c_base_margin_"
        "boosted_residual"
    ),
    "fold_count": 3,
    "booster_count": 6,
    "booster_files": sorted(
        phase36_saved_booster_files
    ),
    "selected_specification": {
        "maximum_depth": 2,
        "minimum_child_weight": 8.0,
        "l2_regularization": 10.0,
    },
    "shared_scale": 0.75,
    "phase39_stack": {
        "phase36_weight": 0.75,
        "phase33_weight": 0.25,
        "alpha": 1.0,
        "uncertainty_exponent": 0.0,
        "delta_cap": 2.0,
    },
    "contains_labels": False,
    "contains_oof_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "include_in_submission_only_after_runtime_parity": True,
}

(
    PHASE36_PRIVATE_CHECKPOINT_DIRECTORY
    / "phase36_metadata.json"
).write_text(
    json.dumps(
        phase36_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------
# 4. Reload and verify all boosters
# --------------------------------------------------------------

phase36_reloaded_payload = joblib.load(
    phase36_portable_state_path
)


def phase36_restore_boosters(value):
    if (
        isinstance(value, dict)
        and set(value.keys())
        == {
            "__phase36_xgboost_booster__"
        }
    ):
        booster_name = value[
            "__phase36_xgboost_booster__"
        ]

        booster_path = (
            PHASE36_PRIVATE_CHECKPOINT_DIRECTORY
            / booster_name
        )

        assert booster_path.is_file()

        booster = xgb.Booster()
        booster.load_model(booster_path)

        reloaded_hash = hashlib.sha256(
            booster.save_raw(
                raw_format="ubj"
            )
        ).hexdigest()

        assert (
            reloaded_hash
            == phase36_original_booster_hashes[
                booster_name
            ]
        ), {
            "message": (
                "Reloaded Phase36 booster differs "
                "from its live source."
            ),
            "booster_file": booster_name,
        }

        return booster

    if (
        isinstance(value, dict)
        and set(value.keys())
        == {"__phase36_tuple__"}
    ):
        return tuple(
            phase36_restore_boosters(
                child
            )
            for child in value[
                "__phase36_tuple__"
            ]
        )

    if isinstance(value, dict):
        return {
            key: phase36_restore_boosters(
                child
            )
            for key, child
            in value.items()
        }

    if isinstance(value, list):
        return [
            phase36_restore_boosters(
                child
            )
            for child in value
        ]

    return value


PHASE36_DEPLOYMENT_STATES_RELOADED_PRIVATE = (
    phase36_restore_boosters(
        phase36_reloaded_payload[
            "states"
        ]
    )
)


def phase36_count_reloaded_boosters(
    value,
):
    if isinstance(value, xgb.Booster):
        return 1

    if isinstance(value, dict):
        return sum(
            phase36_count_reloaded_boosters(
                child
            )
            for child in value.values()
        )

    if isinstance(value, (list, tuple)):
        return sum(
            phase36_count_reloaded_boosters(
                child
            )
            for child in value
        )

    return 0


phase36_reloaded_booster_count = (
    phase36_count_reloaded_boosters(
        PHASE36_DEPLOYMENT_STATES_RELOADED_PRIVATE
    )
)

assert phase36_reloaded_booster_count == 6


# --------------------------------------------------------------
# 5. Sanitized persistence report
# --------------------------------------------------------------

phase36_checkpoint_files = sorted(
    path
    for path
    in PHASE36_PRIVATE_CHECKPOINT_DIRECTORY.iterdir()
    if path.is_file()
)

phase36_persistence_report = {
    "phase": (
        "phase36_private_deployment_"
        "state_persistence"
    ),
    "status": "accepted",
    "checkpoint_directory": (
        PHASE36_PRIVATE_CHECKPOINT_DIRECTORY.name
    ),
    "file_count": len(
        phase36_checkpoint_files
    ),
    "checkpoint_size_mb": round(
        sum(
            path.stat().st_size
            for path
            in phase36_checkpoint_files
        )
        / 1e6,
        4,
    ),
    "fold_count": 3,
    "booster_count": len(
        phase36_saved_booster_files
    ),
    "reloaded_booster_count": (
        phase36_reloaded_booster_count
    ),
    "booster_hash_verification_passed": True,
    "portable_preprocessor_state_present": True,
    "contains_labels": False,
    "contains_oof_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "model_hashes_displayed": False,
    "training_voxel_data_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase36_persistence_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE36_PERSISTENCE"
)
print(
    json.dumps(
        phase36_persistence_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE36_PERSISTENCE"
)

BEGIN SANITIZED_PHASE36_PERSISTENCE
{
  "phase": "phase36_private_deployment_state_persistence",
  "status": "accepted",
  "checkpoint_directory": "phase36_private_checkpoint",
  "file_count": 8,
  "checkpoint_size_mb": 2.7066,
  "fold_count": 3,
  "booster_count": 6,
  "reloaded_booster_count": 6,
  "booster_hash_verification_passed": true,
  "portable_preprocessor_state_present": true,
  "contains_labels": false,
  "contains_oof_predictions": false,
  "contains_case_rows": false,
  "contains_voxel_data": false,
  "model_hashes_displayed": false,
  "training_voxel_data_read": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "elapsed_seconds": 0.14
}
END SANITIZED_PHASE36_PERSISTENCE


In [90]:
# Phase39 Cell 123C
# Export an exact, weight-free Phase39 runtime source handoff.
#
# The handoff contains source code, contracts and state schemas only.
# It contains no model weights, labels, predictions, embeddings,
# voxel arrays, patient rows or UIDs.

from pathlib import Path
import ast
import inspect
import json
import tempfile
import time
import zipfile

import numpy as np


phase39_source_handoff_started = (
    time.perf_counter()
)

PHASE39_SOURCE_PREFIXES = (
    "phase31_",
    "phase32_",
    "phase33_",
    "phase36_",
    "Phase31",
    "Phase32",
    "Phase33",
    "Phase36",
)

PHASE39_INITIAL_SOURCE_NAMES = [
    "phase31_case_features",
    "phase31_localize_bilateral",
    "phase31_preprocess_highres",
    "phase32_apply_p3ca",
    "phase32_dense_tokens",
    "phase32_predict_phase12_fold_matrix",
    "phase32_preprocess_legacy_crops",
    "PHASE32_MAKE_BILATERAL_VIEWS",
    "phase33_fit_preprocessor",
    "phase33_transform",
    "phase33_predict_logits",
    "Phase33DomainInvariantFusionNet",
    "phase36_build_features",
    "phase36_predict_residual",
]

phase39_source_handoff_directory = Path(
    tempfile.mkdtemp(
        prefix="phase39_source_handoff_",
        dir="/kaggle/working",
    )
)


# --------------------------------------------------------------
# 1. Discover exact source dependency closure
# --------------------------------------------------------------

def phase39_is_notebook_definition(value):
    return (
        inspect.isfunction(value)
        or inspect.isclass(value)
    )


def phase39_source_for_object(value):
    try:
        source = inspect.getsource(value)
        return source, "inspect"
    except Exception:
        return None, None


def phase39_ast_referenced_names(source):
    try:
        tree = ast.parse(source)
    except Exception:
        return set()

    return {
        node.id
        for node in ast.walk(tree)
        if isinstance(node, ast.Name)
    }


phase39_source_objects = {}
phase39_source_queue = []

for name in PHASE39_INITIAL_SOURCE_NAMES:
    value = globals().get(name)

    if (
        value is not None
        and phase39_is_notebook_definition(
            value
        )
    ):
        phase39_source_queue.append(name)


# Also include every locally defined Phase31–36 function/class.
for name, value in list(globals().items()):
    if not name.startswith(
        PHASE39_SOURCE_PREFIXES
    ):
        continue

    if phase39_is_notebook_definition(
        value
    ):
        phase39_source_queue.append(name)


phase39_processed_names = set()

while phase39_source_queue:
    name = phase39_source_queue.pop(0)

    if name in phase39_processed_names:
        continue

    phase39_processed_names.add(name)

    value = globals().get(name)

    if (
        value is None
        or not phase39_is_notebook_definition(
            value
        )
    ):
        continue

    source, source_method = (
        phase39_source_for_object(value)
    )

    if source is None:
        phase39_source_objects[name] = {
            "name": name,
            "source": None,
            "source_method": None,
            "error": "source_unavailable",
            "dependencies": [],
        }
        continue

    referenced_names = (
        phase39_ast_referenced_names(
            source
        )
    )

    dependencies = []

    for referenced_name in sorted(
        referenced_names
    ):
        referenced_value = globals().get(
            referenced_name
        )

        if (
            referenced_value is not None
            and phase39_is_notebook_definition(
                referenced_value
            )
        ):
            referenced_source, _ = (
                phase39_source_for_object(
                    referenced_value
                )
            )

            if referenced_source is not None:
                dependencies.append(
                    referenced_name
                )

                if (
                    referenced_name
                    not in phase39_processed_names
                ):
                    phase39_source_queue.append(
                        referenced_name
                    )

    try:
        first_line = int(
            inspect.getsourcelines(
                value
            )[1]
        )
    except Exception:
        first_line = -1

    phase39_source_objects[name] = {
        "name": name,
        "source": source,
        "source_method": source_method,
        "line_count": len(
            source.splitlines()
        ),
        "first_line": first_line,
        "dependencies": dependencies,
    }


available_source_records = [
    record
    for record
    in phase39_source_objects.values()
    if record.get("source") is not None
]

missing_source_records = [
    record
    for record
    in phase39_source_objects.values()
    if record.get("source") is None
]

assert available_source_records, (
    "No Phase39 runtime sources recovered."
)


# Deduplicate identical source blocks.
phase39_unique_source_blocks = []
phase39_seen_sources = set()

for record in sorted(
    available_source_records,
    key=lambda item: (
        item.get("first_line", -1),
        item["name"],
    ),
):
    normalized_source = (
        record["source"].strip()
    )

    if normalized_source in phase39_seen_sources:
        continue

    phase39_seen_sources.add(
        normalized_source
    )

    phase39_unique_source_blocks.append(
        record["source"].rstrip()
    )


phase39_combined_source = (
    "# Exact notebook source recovered for Phase39 runtime integration.\n"
    "# This file is a source handoff, not a standalone entrypoint.\n\n"
    + "\n\n\n".join(
        phase39_unique_source_blocks
    )
    + "\n"
)

(
    phase39_source_handoff_directory
    / "phase31_36_exact_notebook_source.py.txt"
).write_text(
    phase39_combined_source,
    encoding="utf-8",
)


# --------------------------------------------------------------
# 2. Export safe configuration values
# --------------------------------------------------------------

def phase39_json_safe(value, depth=0):
    if depth > 8:
        return {
            "type": type(value).__name__,
            "truncated": True,
        }

    if value is None or isinstance(
        value,
        (str, int, float, bool),
    ):
        return value

    if isinstance(
        value,
        (np.integer, np.floating),
    ):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {
            str(key): phase39_json_safe(
                child,
                depth + 1,
            )
            for key, child in value.items()
            if not any(
                forbidden
                in str(key).lower()
                for forbidden in [
                    "label",
                    "prediction",
                    "oof",
                    "patient",
                    "uid",
                    "embedding",
                    "voxel",
                    "hash",
                ]
            )
        }

    if isinstance(value, (list, tuple)):
        if len(value) <= 500:
            return [
                phase39_json_safe(
                    child,
                    depth + 1,
                )
                for child in value
            ]

        return {
            "type": type(value).__name__,
            "length": len(value),
        }

    if isinstance(value, np.ndarray):
        if (
            value.size <= 500
            and np.issubdtype(
                value.dtype,
                np.number,
            )
            and np.isfinite(value).all()
        ):
            return {
                "type": "ndarray",
                "shape": list(value.shape),
                "dtype": str(value.dtype),
                "values": value.tolist(),
            }

        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "values_exported": False,
        }

    return {
        "type": type(value).__name__,
        "module": type(value).__module__,
    }


phase39_configuration_candidates = {}

for name in [
    "PHASE31_CONFIG",
    "PHASE32_CONFIG",
    "PHASE33_CONFIG",
    "PHASE33_TRAINING_CONFIG",
    "PHASE36_CONFIG",
    "PHASE36_MODEL_CONFIG",
    "PHASE39_CONFIG",
    "PHASE39_SELECTED_SPECIFICATION_PRIVATE",
]:
    if name in globals():
        phase39_configuration_candidates[
            name
        ] = phase39_json_safe(
            globals()[name]
        )

(
    phase39_source_handoff_directory
    / "runtime_configurations.json"
).write_text(
    json.dumps(
        phase39_configuration_candidates,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------
# 3. Export deployment-state schemas without values
# --------------------------------------------------------------

def phase39_state_schema(
    value,
    depth=0,
):
    if depth > 8:
        return {
            "type": type(value).__name__,
            "truncated": True,
        }

    if isinstance(value, dict):
        result = {
            "type": "dict",
            "key_count": len(value),
            "keys": [
                str(key)
                for key in value.keys()
            ],
        }

        fields = {}

        for key, child in value.items():
            lowered_key = str(key).lower()

            if any(
                forbidden
                in lowered_key
                for forbidden in [
                    "label",
                    "prediction",
                    "oof",
                    "patient",
                    "uid",
                ]
            ):
                fields[str(key)] = {
                    "type": (
                        type(child).__name__
                    ),
                    "values_exported": False,
                }
                continue

            fields[str(key)] = (
                phase39_state_schema(
                    child,
                    depth + 1,
                )
            )

        result["fields"] = fields
        return result

    if isinstance(value, (list, tuple)):
        result = {
            "type": type(value).__name__,
            "length": len(value),
        }

        if len(value) <= 20:
            result["elements"] = [
                phase39_state_schema(
                    child,
                    depth + 1,
                )
                for child in value
            ]

        return result

    if isinstance(value, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "values_exported": False,
        }

    if (
        hasattr(value, "detach")
        and hasattr(value, "shape")
    ):
        return {
            "type": "tensor",
            "shape": [
                int(dimension)
                for dimension in value.shape
            ],
            "dtype": str(value.dtype),
            "values_exported": False,
        }

    if (
        type(value).__module__.startswith(
            "xgboost"
        )
        and type(value).__name__
        == "Booster"
    ):
        return {
            "type": "xgboost.Booster",
            "values_exported": False,
        }

    if isinstance(
        value,
        (str, int, float, bool),
    ) or value is None:
        return {
            "type": type(value).__name__,
        }

    return {
        "type": type(value).__name__,
        "module": type(value).__module__,
    }


phase39_state_schemas = {}

for name in [
    "PHASE33_DEPLOYMENT_STATES_PRIVATE",
    "PHASE33_SHARED_GATE_DEPLOYMENT_PRIVATE",
    "PHASE36_DEPLOYMENT_STATES_PRIVATE",
    "PHASE36_DEPLOYMENT_STATES_RELOADED_PRIVATE",
]:
    if name in globals():
        phase39_state_schemas[name] = (
            phase39_state_schema(
                globals()[name]
            )
        )

(
    phase39_source_handoff_directory
    / "deployment_state_schemas.json"
).write_text(
    json.dumps(
        phase39_state_schemas,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------
# 4. Export source inventory and unresolved globals
# --------------------------------------------------------------

defined_source_names = set(
    phase39_source_objects.keys()
)

all_referenced_names = set()

for record in available_source_records:
    all_referenced_names.update(
        phase39_ast_referenced_names(
            record["source"]
        )
    )

unresolved_global_records = []

for name in sorted(
    all_referenced_names
    - defined_source_names
):
    if name not in globals():
        continue

    value = globals()[name]

    if inspect.ismodule(value):
        continue

    if name.startswith("__"):
        continue

    if isinstance(
        value,
        (str, int, float, bool),
    ) or value is None:
        schema = {
            "type": type(value).__name__,
            "value": value,
        }

    elif isinstance(value, np.ndarray):
        schema = {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "values_exported": False,
        }

    elif isinstance(value, (dict, list, tuple)):
        schema = {
            "type": type(value).__name__,
            "length": len(value),
            "values_exported": False,
        }

    else:
        schema = {
            "type": type(value).__name__,
            "module": type(value).__module__,
        }

    unresolved_global_records.append({
        "name": name,
        "schema": schema,
    })


source_inventory = {
    "phase": (
        "phase39_runtime_source_handoff"
    ),
    "recovered_source_count": len(
        available_source_records
    ),
    "unique_source_block_count": len(
        phase39_unique_source_blocks
    ),
    "missing_source_count": len(
        missing_source_records
    ),
    "recovered_sources": [
        {
            "name": record["name"],
            "line_count": record[
                "line_count"
            ],
            "source_method": record[
                "source_method"
            ],
            "dependencies": record[
                "dependencies"
            ],
        }
        for record in sorted(
            available_source_records,
            key=lambda item: item["name"],
        )
    ],
    "missing_sources": [
        record["name"]
        for record
        in missing_source_records
    ],
    "unresolved_global_count": len(
        unresolved_global_records
    ),
    "unresolved_globals": (
        unresolved_global_records
    ),
}

(
    phase39_source_handoff_directory
    / "source_inventory.json"
).write_text(
    json.dumps(
        source_inventory,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------
# 5. Include the accepted Phase30 runtime source, if present
# --------------------------------------------------------------

phase30_main_candidates = [
    Path(
        "/kaggle/working/"
        "phase30_submission/main.py"
    ),
    Path(
        "/kaggle/working/"
        "phase30_zip_rehearsal/main.py"
    ),
]

phase30_main_source = next(
    (
        path
        for path in phase30_main_candidates
        if path.is_file()
    ),
    None,
)

assert phase30_main_source is not None, (
    "Accepted Phase30 main.py not found."
)

(
    phase39_source_handoff_directory
    / "phase30_accepted_main.py.txt"
).write_text(
    phase30_main_source.read_text(
        encoding="utf-8"
    ),
    encoding="utf-8",
)


# Include only licenses/manifests, never assets or weights.
phase30_runtime_directory = (
    phase30_main_source.parent
)

for file_name in [
    "LICENSE",
    "README.md",
    "THIRD_PARTY_LICENSES.md",
    "asset_license_registry.json",
    "manifest.json",
    "runtime_contract.json",
]:
    source_path = (
        phase30_runtime_directory
        / file_name
    )

    if source_path.is_file():
        (
            phase39_source_handoff_directory
            / f"phase30_{file_name}"
        ).write_bytes(
            source_path.read_bytes()
        )


# --------------------------------------------------------------
# 6. Include sanitized checkpoint metadata only
# --------------------------------------------------------------

sanitized_checkpoint_metadata = {}

for phase_name, metadata_path in {
    "phase33": Path(
        "/kaggle/working/"
        "phase33_private_checkpoint/"
        "phase33_checkpoint_metadata.json"
    ),
    "phase36": Path(
        "/kaggle/working/"
        "phase36_private_checkpoint/"
        "phase36_metadata.json"
    ),
    "phase39": Path(
        "/kaggle/working/"
        "phase39_private_checkpoint/"
        "phase39_metadata.json"
    ),
}.items():
    if not metadata_path.is_file():
        continue

    metadata = json.loads(
        metadata_path.read_text(
            encoding="utf-8"
        )
    )

    for forbidden_key in [
        "sha256",
        "hash",
        "hashes",
        "model_hashes",
    ]:
        metadata.pop(
            forbidden_key,
            None,
        )

    sanitized_checkpoint_metadata[
        phase_name
    ] = metadata

(
    phase39_source_handoff_directory
    / "sanitized_checkpoint_metadata.json"
).write_text(
    json.dumps(
        sanitized_checkpoint_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------
# 7. Runtime integration contract
# --------------------------------------------------------------

runtime_integration_contract = {
    "phase": (
        "phase39_experimental_runtime_"
        "integration_contract"
    ),
    "base_runtime": (
        "accepted Phase30 offline runtime"
    ),
    "deployment_formula": (
        "z_final=z_phase12c+clip("
        "0.75*r_phase36+0.25*r_phase33,"
        "-2.0,2.0)"
    ),
    "phase33_component": (
        "mean fold-local outer-train "
        "standalone component logit"
    ),
    "phase36_component": (
        "mean fold-local outer-train "
        "XGBoost base-margin residual"
    ),
    "required_external_assets": [
        "Phase12c 21-model ensemble",
        "DINOv3 ViT-S/16 checkpoint",
        "Phase32 fold-local P3CA bases",
        "Phase33 preprocessing and model states",
        "Phase36 preprocessing and XGBoost boosters",
    ],
    "runtime_tests_required": [
        "synthetic initialization",
        "Phase33 component parity",
        "Phase36 residual parity",
        "complete Phase39 OOF parity",
        "real smoke test",
        "extracted-archive smoke test",
    ],
    "separate_experimental_archive": True,
    "replace_existing_archive": False,
    "contains_model_weights": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
}

(
    phase39_source_handoff_directory
    / "runtime_integration_contract.json"
).write_text(
    json.dumps(
        runtime_integration_contract,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------
# 8. Zip the handoff
# --------------------------------------------------------------

zip_base_path = Path(
    "/kaggle/working/"
    "phase39_runtime_source_handoff.zip"
)

zip_path = zip_base_path

if zip_path.exists():
    version = 2

    while True:
        candidate = Path(
            "/kaggle/working/"
            f"phase39_runtime_source_handoff_v"
            f"{version}.zip"
        )

        if not candidate.exists():
            zip_path = candidate
            break

        version += 1


with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for path in sorted(
        phase39_source_handoff_directory.iterdir()
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.name,
            )


with zipfile.ZipFile(
    zip_path,
    mode="r",
) as archive:
    bad_file = archive.testzip()
    archive_names = archive.namelist()

assert bad_file is None


phase39_handoff_report = {
    "phase": (
        "phase39_runtime_source_handoff"
    ),
    "status": "complete",
    "zip_name": zip_path.name,
    "zip_size_kb": round(
        zip_path.stat().st_size
        / 1e3,
        2,
    ),
    "included_files": sorted(
        archive_names
    ),
    "recovered_source_count": len(
        available_source_records
    ),
    "unique_source_block_count": len(
        phase39_unique_source_blocks
    ),
    "missing_source_count": len(
        missing_source_records
    ),
    "unresolved_global_count": len(
        unresolved_global_records
    ),
    "contains_model_weights": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "model_hashes_displayed": False,
    "test_or_smoke_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase39_source_handoff_started,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE39_SOURCE_HANDOFF"
)
print(
    json.dumps(
        phase39_handoff_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE39_SOURCE_HANDOFF"
)

print(
    f"{zip_path}"
)

BEGIN SANITIZED_PHASE39_SOURCE_HANDOFF
{
  "phase": "phase39_runtime_source_handoff",
  "status": "complete",
  "zip_name": "phase39_runtime_source_handoff.zip",
  "zip_size_kb": 100.93,
  "included_files": [
    "deployment_state_schemas.json",
    "phase30_LICENSE",
    "phase30_README.md",
    "phase30_THIRD_PARTY_LICENSES.md",
    "phase30_accepted_main.py.txt",
    "phase30_asset_license_registry.json",
    "phase30_manifest.json",
    "phase31_36_exact_notebook_source.py.txt",
    "runtime_configurations.json",
    "runtime_integration_contract.json",
    "sanitized_checkpoint_metadata.json",
    "source_inventory.json"
  ],
  "recovered_source_count": 118,
  "unique_source_block_count": 117,
  "missing_source_count": 3,
  "unresolved_global_count": 156,
  "contains_model_weights": false,
  "contains_labels": false,
  "contains_predictions": false,
  "contains_embeddings": false,
  "contains_voxel_data": false,
  "contains_patient_rows": false,
  "model_hashes_displayed": false,


In [95]:
# Phase39 Cell 124
# Recover missing exact Phase33 source from notebook history and package
# the remaining runtime dependencies for inspection/integration.
#
# This does NOT modify the accepted Phase30 archive.

from pathlib import Path
import ast
import hashlib
import json
import shutil
import tempfile
import time
import zipfile

import numpy as np
import torch


PHASE39_PATCH_OUTPUT = Path(
    "/kaggle/working/phase39_runtime_dependency_patch.zip"
)

phase39_patch_started = time.perf_counter()


# ---------------------------------------------------------
# 1. Notebook-history recovery
# ---------------------------------------------------------

def phase39_history_records_containing(needle):
    ipython = get_ipython()
    manager = ipython.history_manager

    records = []
    seen = set()

    # Current in-memory session.
    for line_number, source in enumerate(
        manager.input_hist_raw
    ):
        if not isinstance(source, str):
            continue

        if needle not in source:
            continue

        key = (
            int(getattr(manager, "session_number", 0)),
            int(line_number),
            source,
        )

        if key not in seen:
            seen.add(key)
            records.append(key)

    # Persistent IPython history, including earlier sessions.
    try:
        iterator = manager.search(
            f"*{needle}*",
            raw=True,
            output=False,
            n=None,
            unique=False,
        )

        for session, line_number, source in iterator:
            if not isinstance(source, str):
                continue

            if needle not in source:
                continue

            key = (
                int(session),
                int(line_number),
                source,
            )

            if key not in seen:
                seen.add(key)
                records.append(key)

    except Exception:
        pass

    return sorted(
        records,
        key=lambda item: (
            item[0],
            item[1],
        ),
    )


def phase39_extract_named_class(source, class_name):
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return None

    source_lines = source.splitlines()

    for node in tree.body:
        if not (
            isinstance(node, ast.ClassDef)
            and node.name == class_name
        ):
            continue

        start_line = node.lineno

        if node.decorator_list:
            start_line = min(
                decorator.lineno
                for decorator
                in node.decorator_list
            )

        end_line = node.end_lineno

        return "\n".join(
            source_lines[
                start_line - 1:end_line
            ]
        ).strip()

    return None


# Construct strings at runtime so this recovery cell does not
# accidentally match itself in notebook history.
phase39_missing_class_names = [
    "Phase33" + "GradientReverseFunction",
    "Phase33" + "TemporalBlock",
    "Phase33" + "DomainInvariantFusionNet",
]

phase39_class_candidates = {}

for class_name in phase39_missing_class_names:
    marker = "class " + class_name

    candidates = []

    for (
        session,
        line_number,
        source,
    ) in phase39_history_records_containing(
        marker
    ):
        extracted = phase39_extract_named_class(
            source,
            class_name,
        )

        if extracted is not None:
            candidates.append({
                "session": session,
                "line": line_number,
                "source": extracted,
            })

    # Remove duplicate definitions while retaining chronology.
    unique_candidates = []
    seen_sources = set()

    for candidate in candidates:
        normalized = candidate["source"].strip()

        if normalized in seen_sources:
            continue

        seen_sources.add(normalized)
        unique_candidates.append(candidate)

    assert unique_candidates, {
        "message": (
            "Could not recover an exact class definition "
            "from IPython history."
        ),
        "class_name": class_name,
    }

    # The latest executed definition corresponds to the live
    # notebook definition if a class was corrected/redefined.
    phase39_class_candidates[
        class_name
    ] = unique_candidates[-1]


phase39_architecture_source = "\n\n".join([
    "# Exact Phase33 architecture source recovered from IPython history.",
    "import torch",
    "from torch import nn",
    "import torch.nn.functional as F",
    "",
    phase39_class_candidates[
        "Phase33GradientReverseFunction"
    ]["source"],
    "",
    phase39_class_candidates[
        "Phase33TemporalBlock"
    ]["source"],
    "",
    phase39_class_candidates[
        "Phase33DomainInvariantFusionNet"
    ]["source"],
    "",
])

compile(
    phase39_architecture_source,
    "phase33_exact_architecture.py",
    "exec",
)


# ---------------------------------------------------------
# 2. Recover the exact P3CA projection/cache-construction cell
# ---------------------------------------------------------

phase39_summary_symbol = (
    "PHASE32_" + "P3CA_SUMMARY_PRIVATE"
)
phase39_sequence_symbol = (
    "PHASE32_" + "P3CA_SEQUENCE_PRIVATE"
)

phase39_projection_candidates = []

for record in (
    phase39_history_records_containing(
        phase39_summary_symbol
    )
):
    session, line_number, source = record

    if phase39_sequence_symbol not in source:
        continue

    projection_score = sum([
        "phase32_apply_p3ca" in source,
        "open_memmap" in source,
        "summary_writer" in source,
        "sequence_writer" in source,
        "projected" in source,
    ])

    if projection_score >= 3:
        phase39_projection_candidates.append({
            "session": session,
            "line": line_number,
            "score": projection_score,
            "source": source,
        })

assert phase39_projection_candidates, {
    "message": (
        "Could not recover the exact Phase32 projected-cache "
        "construction cell from IPython history."
    ),
}

phase39_projection_candidates.sort(
    key=lambda item: (
        item["score"],
        item["session"],
        item["line"],
    )
)

phase39_projection_source = (
    phase39_projection_candidates[-1][
        "source"
    ]
)


# ---------------------------------------------------------
# 3. Resolve and validate the fold-local P3CA basis archive
# ---------------------------------------------------------

PHASE39_EXPECTED_P3CA_SHAPES = {
    "mean": (2, 3, 6, 384),
    "standard_deviation": (2, 3, 6, 384),
    "components": (2, 3, 6, 16, 384),
    "explained_ratio": (2, 3, 6, 16),
    "eigenvalues": (2, 3, 6, 16),
    "token_count": (2, 3),
}


def phase39_valid_p3ca_state(path):
    try:
        with np.load(
            path,
            allow_pickle=False,
        ) as state:
            for key, expected_shape in (
                PHASE39_EXPECTED_P3CA_SHAPES.items()
            ):
                if key not in state.files:
                    return False

                array = np.asarray(state[key])

                if array.shape != expected_shape:
                    return False

                if not np.isfinite(array).all():
                    return False

        return True

    except Exception:
        return False


phase39_p3ca_candidates = []

explicit_p3ca_path = globals().get(
    "PHASE32_P3CA_STATE_PATH"
)

if explicit_p3ca_path is not None:
    explicit_p3ca_path = Path(
        explicit_p3ca_path
    )

    if explicit_p3ca_path.is_file():
        phase39_p3ca_candidates.append(
            explicit_p3ca_path
        )

for candidate in Path(
    "/kaggle/working"
).rglob("*.npz"):
    if candidate not in phase39_p3ca_candidates:
        phase39_p3ca_candidates.append(
            candidate
        )

phase39_valid_p3ca_candidates = [
    candidate
    for candidate in phase39_p3ca_candidates
    if phase39_valid_p3ca_state(candidate)
]

assert len(
    phase39_valid_p3ca_candidates
) == 1, {
    "message": (
        "Expected exactly one valid fold-local "
        "Phase32 P3CA state archive."
    ),
    "candidate_count": len(
        phase39_valid_p3ca_candidates
    ),
    "candidate_names": [
        path.name
        for path in phase39_valid_p3ca_candidates
    ],
}

phase39_p3ca_state_path = (
    phase39_valid_p3ca_candidates[0]
)


# ---------------------------------------------------------
# 4. Resolve Phase33 deployment-only state
# ---------------------------------------------------------

phase39_phase33_directories = []

phase33_directory_global = globals().get(
    "PHASE33_PRIVATE_CHECKPOINT_DIRECTORY"
)

if phase33_directory_global is not None:
    phase39_phase33_directories.append(
        Path(phase33_directory_global)
    )

phase39_phase33_directories.append(
    Path(
        "/kaggle/working/"
        "phase33_private_checkpoint"
    )
)

phase39_phase33_state_candidates = []

for directory in phase39_phase33_directories:
    candidate = (
        directory
        / "phase33_deployment_states.pt"
    )

    if (
        candidate.is_file()
        and candidate
        not in phase39_phase33_state_candidates
    ):
        phase39_phase33_state_candidates.append(
            candidate
        )

assert len(
    phase39_phase33_state_candidates
) == 1, {
    "message": (
        "Expected exactly one Phase33 deployment-state file."
    ),
    "candidate_count": len(
        phase39_phase33_state_candidates
    ),
}

phase39_phase33_state_path = (
    phase39_phase33_state_candidates[0]
)

phase39_phase33_payload = torch.load(
    phase39_phase33_state_path,
    map_location="cpu",
    weights_only=False,
)


def phase39_is_phase33_state_list(value):
    if not (
        isinstance(value, (list, tuple))
        and len(value) == 3
    ):
        return False

    for fold_state in value:
        if not isinstance(fold_state, dict):
            return False

        if not {
            "fold",
            "preprocessor",
            "model_states",
        }.issubset(fold_state):
            return False

        model_states = fold_state[
            "model_states"
        ]

        if not (
            isinstance(
                model_states,
                (list, tuple),
            )
            and len(model_states) == 3
        ):
            return False

        if not all(
            isinstance(model_state, dict)
            for model_state in model_states
        ):
            return False

    return True


def phase39_find_phase33_states(
    value,
    route="root",
    visited=None,
):
    if visited is None:
        visited = set()

    object_id = id(value)

    if object_id in visited:
        return []

    visited.add(object_id)

    if phase39_is_phase33_state_list(value):
        return [(route, list(value))]

    matches = []

    if isinstance(value, dict):
        # Check likely wrapper keys first.
        preferred_keys = [
            "deployment_states",
            "states",
            "fold_states",
            "phase33_deployment_states",
        ]

        ordered_keys = [
            key
            for key in preferred_keys
            if key in value
        ] + [
            key
            for key in value
            if key not in preferred_keys
        ]

        for key in ordered_keys:
            child = value[key]

            if isinstance(
                child,
                (dict, list, tuple),
            ):
                matches.extend(
                    phase39_find_phase33_states(
                        child,
                        route=f"{route}.{key}",
                        visited=visited,
                    )
                )

    elif isinstance(value, (list, tuple)):
        for index, child in enumerate(value):
            if isinstance(
                child,
                (dict, list, tuple),
            ):
                matches.extend(
                    phase39_find_phase33_states(
                        child,
                        route=f"{route}[{index}]",
                        visited=visited,
                    )
                )

    return matches


phase39_phase33_state_matches = (
    phase39_find_phase33_states(
        phase39_phase33_payload
    )
)

assert len(
    phase39_phase33_state_matches
) == 1, {
    "message": (
        "Could not uniquely unwrap the Phase33 "
        "deployment-state list."
    ),
    "checkpoint_container_type": type(
        phase39_phase33_payload
    ).__name__,
    "match_count": len(
        phase39_phase33_state_matches
    ),
    "match_routes": [
        route
        for route, _
        in phase39_phase33_state_matches
    ],
}

(
    phase39_phase33_state_route,
    phase39_phase33_states,
) = phase39_phase33_state_matches[0]

assert len(phase39_phase33_states) == 3

print(
    "Phase33 deployment state accepted:",
    {
        "checkpoint_container_type": type(
            phase39_phase33_payload
        ).__name__,
        "resolved_route": (
            phase39_phase33_state_route
        ),
        "fold_count": len(
            phase39_phase33_states
        ),
        "model_count": sum(
            len(state["model_states"])
            for state
            in phase39_phase33_states
        ),
    },
)

phase39_phase33_tensor_count = 0
phase39_phase33_tensor_parameters = 0

for fold_state in phase39_phase33_states:
    assert set([
        "fold",
        "preprocessor",
        "model_states",
    ]).issubset(fold_state)

    assert len(
        fold_state["model_states"]
    ) == 3

    for model_state in fold_state[
        "model_states"
    ]:
        for tensor in model_state.values():
            if torch.is_tensor(tensor):
                assert torch.isfinite(
                    tensor
                ).all()

                phase39_phase33_tensor_count += 1
                phase39_phase33_tensor_parameters += (
                    tensor.numel()
                )


# ---------------------------------------------------------
# 5. Resolve Phase36 deployment-only checkpoint
# ---------------------------------------------------------

phase39_phase36_directory = Path(
    "/kaggle/working/"
    "phase36_private_checkpoint"
)

assert phase39_phase36_directory.is_dir(), {
    "message": "Phase36 checkpoint directory is missing.",
    "directory": str(phase39_phase36_directory),
}

phase39_phase36_deployment_files = sorted(
    [
        path for path in phase39_phase36_directory.iterdir()
        if path.is_file()
    ],
    key=lambda path: path.name,
)

phase39_phase36_state_path = (
    phase39_phase36_directory
    / "phase36_deployment_state.joblib"
)

phase39_phase36_metadata_path = (
    phase39_phase36_directory
    / "phase36_metadata.json"
)

phase39_phase36_booster_files = sorted(
    phase39_phase36_directory.glob("*.ubj"),
    key=lambda path: path.name,
)

assert phase39_phase36_state_path.is_file(), {
    "message": "Phase36 portable joblib state is missing.",
    "expected_name": phase39_phase36_state_path.name,
}

assert phase39_phase36_metadata_path.is_file(), {
    "message": "Phase36 metadata is missing.",
    "expected_name": phase39_phase36_metadata_path.name,
}

phase39_expected_booster_names = [
    (
        "phase36_deployment_states_"
        f"{fold}_boosters_{seed_index}_booster.ubj"
    )
    for fold in range(3)
    for seed_index in range(2)
]

phase39_actual_booster_names = [
    path.name
    for path in phase39_phase36_booster_files
]

assert (
    phase39_actual_booster_names
    == phase39_expected_booster_names
), {
    "message": (
        "Phase36 boosters do not match the expected "
        "3-fold by 2-seed contract."
    ),
    "actual": phase39_actual_booster_names,
    "expected": phase39_expected_booster_names,
}

phase39_phase36_deployment_files = [
    phase39_phase36_state_path,
    phase39_phase36_metadata_path,
    *phase39_phase36_booster_files,
]

assert len(phase39_phase36_deployment_files) == 8
assert len(phase39_phase36_booster_files) == 6

actual_directory_files = sorted([
    path.name
    for path in phase39_phase36_directory.iterdir()
    if path.is_file() and not path.name.startswith(".")
])

expected_directory_files = sorted([
    phase39_phase36_state_path.name,
    phase39_phase36_metadata_path.name,
    *[
        path.name
        for path in phase39_phase36_booster_files
    ],
])

assert actual_directory_files == expected_directory_files, {
    "message": (
        "Unexpected or missing file in the "
        "Phase36 deployment directory."
    ),
    "actual": actual_directory_files,
    "expected": expected_directory_files,
}

phase39_phase36_metadata = json.loads(
    phase39_phase36_metadata_path.read_text(
        encoding="utf-8"
    )
)

assert int(
    phase39_phase36_metadata[
        "fold_count"
    ]
) == 3

assert int(
    phase39_phase36_metadata[
        "booster_count"
    ]
) == 6

assert (
    phase39_phase36_metadata[
        "contains_labels"
    ]
    is False
)

assert (
    phase39_phase36_metadata[
        "contains_oof_predictions"
    ]
    is False
)

assert (
    phase39_phase36_metadata[
        "contains_case_rows"
    ]
    is False
)

assert (
    phase39_phase36_metadata[
        "contains_voxel_data"
    ]
    is False
)

for path in phase39_phase36_deployment_files:
    lowered = path.name.lower()

    assert not any(
        forbidden in lowered
        for forbidden in [
            "oof",
            "prediction",
            "label",
            "patient",
            "uid",
            "voxel",
            "embedding",
        ]
    ), path.name

print(
    "Phase36 deployment checkpoint accepted:",
    {
        "directory": (
            phase39_phase36_directory.name
        ),
        "file_count": len(
            phase39_phase36_deployment_files
        ),
        "booster_count": len(
            phase39_phase36_booster_files
        ),
        "portable_state": (
            phase39_phase36_state_path.name
        ),
        "metadata": (
            phase39_phase36_metadata_path.name
        ),
    },
)

# ---------------------------------------------------------
# 6. Export small runtime constants
# ---------------------------------------------------------

phase39_small_constant_names = [
    "PHASE31_CONFIG",
    "PHASE31_PROFILE_NAMES",
    "PHASE31_SHAPE_STAT_NAMES",
    "PHASE32_P3CA_COMPONENT_COUNT",
    "PHASE32_PROXY_CONFIG",
    "PHASE33_CONFIG",
    "PHASE36_CONFIG",
    "PHASE39_SELECTED_SPECIFICATION_PRIVATE",
]


def phase39_json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (str, int, float, bool),
    ):
        return value

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, tuple):
        return [
            phase39_json_safe(child)
            for child in value
        ]

    if isinstance(value, list):
        return [
            phase39_json_safe(child)
            for child in value
        ]

    if isinstance(value, dict):
        return {
            str(key): phase39_json_safe(child)
            for key, child in value.items()
        }

    raise TypeError(
        f"Unsupported small constant: {type(value)}"
    )


phase39_small_constants = {}

for name in phase39_small_constant_names:
    assert name in globals(), {
        "message": (
            "Required runtime constant is missing."
        ),
        "name": name,
    }

    phase39_small_constants[name] = (
        phase39_json_safe(
            globals()[name]
        )
    )

assert (
    phase39_small_constants[
        "PHASE32_P3CA_COMPONENT_COUNT"
    ]
    == 16
)

phase39_prompt_mask = np.asarray(
    globals()["PHASE32_REAL_PROMPT_MASK"],
    dtype=np.bool_,
)

assert phase39_prompt_mask.shape == (
    14,
    14,
)


# ---------------------------------------------------------
# 7. Build the dependency patch
# ---------------------------------------------------------

phase39_stage_directory = Path(
    tempfile.mkdtemp(
        prefix="phase39_dependency_patch_",
        dir="/kaggle/working",
    )
)

(
    phase39_stage_directory
    / "phase33_exact_architecture.py.txt"
).write_text(
    phase39_architecture_source,
    encoding="utf-8",
)

(
    phase39_stage_directory
    / "phase32_projection_history_cell.py.txt"
).write_text(
    phase39_projection_source,
    encoding="utf-8",
)

(
    phase39_stage_directory
    / "runtime_small_constants.json"
).write_text(
    json.dumps(
        phase39_small_constants,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

np.savez_compressed(
    phase39_stage_directory
    / "runtime_small_arrays.npz",
    phase32_real_prompt_mask=(
        phase39_prompt_mask
    ),
)

shutil.copy2(
    phase39_p3ca_state_path,
    phase39_stage_directory
    / "phase32_fold_local_p3ca_states.npz",
)

phase39_phase33_stage = (
    phase39_stage_directory
    / "phase33"
)
phase39_phase33_stage.mkdir()

shutil.copy2(
    phase39_phase33_state_path,
    phase39_phase33_stage
    / "phase33_deployment_states.pt",
)

phase39_phase36_stage = (
    phase39_stage_directory
    / "phase36"
)
phase39_phase36_stage.mkdir()

for source_path in (
    phase39_phase36_deployment_files
):
    shutil.copy2(
        source_path,
        phase39_phase36_stage
        / source_path.name,
    )


phase39_patch_manifest = {
    "schema_version": 1,
    "phase": (
        "phase39_runtime_dependency_patch"
    ),
    "missing_architecture_classes_recovered": (
        phase39_missing_class_names
    ),
    "class_candidate_counts": {
        name: len(
            {
                candidate["source"]
                for candidate
                in [
                    phase39_class_candidates[name]
                ]
            }
        )
        for name in phase39_missing_class_names
    },
    "projection_history_cell_recovered": True,
    "p3ca_state_shapes": {
        key: list(shape)
        for key, shape
        in PHASE39_EXPECTED_P3CA_SHAPES.items()
    },
    "phase33_fold_count": 3,
    "phase33_model_count": 9,
    "phase33_tensor_count": (
        phase39_phase33_tensor_count
    ),
    "phase33_tensor_parameter_count": (
        phase39_phase33_tensor_parameters
    ),
    "phase36_booster_count": 6,
    "phase36_checkpoint_file_count": len(
        phase39_phase36_deployment_files
    ),
    "contains_labels": False,
    "contains_oof_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
    "test_or_smoke_data_read": False,
}

(
    phase39_stage_directory
    / "dependency_patch_manifest.json"
).write_text(
    json.dumps(
        phase39_patch_manifest,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


if PHASE39_PATCH_OUTPUT.exists():
    PHASE39_PATCH_OUTPUT.unlink()

with zipfile.ZipFile(
    PHASE39_PATCH_OUTPUT,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(
        phase39_stage_directory.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(
                    phase39_stage_directory
                ).as_posix(),
            )

with zipfile.ZipFile(
    PHASE39_PATCH_OUTPUT,
    mode="r",
) as archive:
    bad_member = archive.testzip()
    archive_names = archive.namelist()

assert bad_member is None
assert (
    "phase33_exact_architecture.py.txt"
    in archive_names
)
assert (
    "phase32_fold_local_p3ca_states.npz"
    in archive_names
)
assert (
    "phase33/phase33_deployment_states.pt"
    in archive_names
)
assert len([
    name
    for name in archive_names
    if name.endswith(".ubj")
]) == 6


phase39_patch_elapsed = (
    time.perf_counter()
    - phase39_patch_started
)

phase39_report = {
    "phase": (
        "phase39_runtime_dependency_patch"
    ),
    "status": "complete",
    "zip_name": PHASE39_PATCH_OUTPUT.name,
    "zip_size_mb": round(
        PHASE39_PATCH_OUTPUT.stat().st_size
        / 1e6,
        3,
    ),
    "archive_file_count": len(
        archive_names
    ),
    "recovered_architecture_class_count": (
        len(phase39_missing_class_names)
    ),
    "recovered_architecture_classes": (
        phase39_missing_class_names
    ),
    "projection_history_cell_recovered": True,
    "p3ca_basis_count": 36,
    "p3ca_component_count_per_basis": 16,
    "phase33_fold_count": 3,
    "phase33_model_count": 9,
    "phase36_booster_count": 6,
    "archive_integrity_test_passed": True,
    "accepted_phase30_archive_modified": False,
    "contains_labels": False,
    "contains_oof_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        phase39_patch_elapsed,
        2,
    ),
}

print(
    "BEGIN SANITIZED_PHASE39_DEPENDENCY_PATCH"
)
print(
    json.dumps(
        phase39_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE39_DEPENDENCY_PATCH"
)
print(PHASE39_PATCH_OUTPUT)

Phase33 deployment state accepted: {'checkpoint_container_type': 'dict', 'resolved_route': 'root.deployment_states', 'fold_count': 3, 'model_count': 9}
Phase36 deployment checkpoint accepted: {'directory': 'phase36_private_checkpoint', 'file_count': 8, 'booster_count': 6, 'portable_state': 'phase36_deployment_state.joblib', 'metadata': 'phase36_metadata.json'}
BEGIN SANITIZED_PHASE39_DEPENDENCY_PATCH
{
  "phase": "phase39_runtime_dependency_patch",
  "status": "complete",
  "zip_name": "phase39_runtime_dependency_patch.zip",
  "zip_size_mb": 8.003,
  "archive_file_count": 15,
  "recovered_architecture_class_count": 3,
  "recovered_architecture_classes": [
    "Phase33GradientReverseFunction",
    "Phase33TemporalBlock",
    "Phase33DomainInvariantFusionNet"
  ],
  "projection_history_cell_recovered": true,
  "p3ca_basis_count": 36,
  "p3ca_component_count_per_basis": 16,
  "phase33_fold_count": 3,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "archive_integrity_test_passe

> Phase 39

In [96]:
# Phase39 Cell 125
# Create a separate experimental Phase39 staging package and export
# version-stable portable deployment assets.
#
# This cell does not modify either accepted Phase30 ZIP.

from pathlib import Path
import hashlib
import json
import os
import shutil
import tempfile
import time
import warnings
import zipfile

import joblib
import numpy as np
import torch


PHASE39_BASE_ARCHIVE = Path(
    "/kaggle/working/phase30_full_ensemble_submission.zip"
)
PHASE39_DEPENDENCY_ARCHIVE = Path(
    "/kaggle/working/phase39_runtime_dependency_patch.zip"
)
PHASE39_STAGING_DIRECTORY = Path(
    "/kaggle/working/phase39_experimental_submission"
)

phase39_stage_started = time.perf_counter()


def phase39_safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    resolved_destination = destination.resolve()

    with zipfile.ZipFile(zip_path, "r") as archive:
        assert archive.testzip() is None, {
            "message": "Archive integrity test failed.",
            "archive": zip_path.name,
        }

        for member in archive.infolist():
            target = (
                destination / member.filename
            ).resolve()

            assert (
                target == resolved_destination
                or resolved_destination in target.parents
            ), {
                "message": "Unsafe archive member.",
                "archive": zip_path.name,
                "member": member.filename,
            }

        archive.extractall(destination)


def phase39_file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def phase39_preprocessor_arrays(
    preprocessor,
    prefix,
    destination,
):
    assert int(preprocessor["stage_index"]) == 1
    assert int(preprocessor["summary_dimension"]) == 48
    assert int(preprocessor["radiomics_dimension"]) == 48

    summary_scaler = preprocessor["summary_scaler"]
    summary_projector = preprocessor["summary_projector"]
    radiomics_scaler = preprocessor["radiomics_scaler"]
    radiomics_projector = preprocessor["radiomics_projector"]

    assert summary_projector.whiten is False
    assert radiomics_projector.whiten is False

    summary_components = np.asarray(
        summary_projector.components_,
        dtype=np.float32,
    )
    radiomics_components = np.asarray(
        radiomics_projector.components_,
        dtype=np.float32,
    )

    # sklearn PCA.transform computes X @ components.T and then
    # subtracts mean @ components.T. Persist the exact offset so
    # runtime inference does not depend on the sklearn version.
    summary_offset = np.asarray(
        summary_projector.mean_,
        dtype=np.float32,
    ) @ summary_components.T

    radiomics_offset = np.asarray(
        radiomics_projector.mean_,
        dtype=np.float32,
    ) @ radiomics_components.T

    values = {
        f"{prefix}_p3ca_token_mean": np.asarray(
            preprocessor["p3ca_token_mean"],
            dtype=np.float32,
        ),
        f"{prefix}_p3ca_token_standard_deviation": np.asarray(
            preprocessor[
                "p3ca_token_standard_deviation"
            ],
            dtype=np.float32,
        ),
        f"{prefix}_geometry_token_mean": np.asarray(
            preprocessor["geometry_token_mean"],
            dtype=np.float32,
        ),
        f"{prefix}_geometry_token_standard_deviation": np.asarray(
            preprocessor[
                "geometry_token_standard_deviation"
            ],
            dtype=np.float32,
        ),
        f"{prefix}_summary_scaler_mean": np.asarray(
            summary_scaler.mean_,
            dtype=np.float64,
        ),
        f"{prefix}_summary_scaler_scale": np.asarray(
            summary_scaler.scale_,
            dtype=np.float64,
        ),
        f"{prefix}_summary_pca_components": summary_components,
        f"{prefix}_summary_pca_offset": np.asarray(
            summary_offset,
            dtype=np.float32,
        ),
        f"{prefix}_radiomics_scaler_mean": np.asarray(
            radiomics_scaler.mean_,
            dtype=np.float64,
        ),
        f"{prefix}_radiomics_scaler_scale": np.asarray(
            radiomics_scaler.scale_,
            dtype=np.float64,
        ),
        f"{prefix}_radiomics_pca_components": radiomics_components,
        f"{prefix}_radiomics_pca_offset": np.asarray(
            radiomics_offset,
            dtype=np.float32,
        ),
    }

    expected_shapes = {
        f"{prefix}_p3ca_token_mean": (48,),
        f"{prefix}_p3ca_token_standard_deviation": (48,),
        f"{prefix}_geometry_token_mean": (52,),
        f"{prefix}_geometry_token_standard_deviation": (52,),
        f"{prefix}_summary_scaler_mean": (768,),
        f"{prefix}_summary_scaler_scale": (768,),
        f"{prefix}_summary_pca_components": (48, 768),
        f"{prefix}_summary_pca_offset": (48,),
        f"{prefix}_radiomics_scaler_mean": (1081,),
        f"{prefix}_radiomics_scaler_scale": (1081,),
        f"{prefix}_radiomics_pca_components": (48, 1081),
        f"{prefix}_radiomics_pca_offset": (48,),
    }

    for name, expected_shape in expected_shapes.items():
        assert values[name].shape == expected_shape, {
            "name": name,
            "shape": list(values[name].shape),
            "expected": list(expected_shape),
        }
        assert np.isfinite(values[name]).all(), name

    destination.update(values)


def phase39_manual_project(
    matrix,
    scaler_mean,
    scaler_scale,
    components,
    offset,
):
    scaled = np.asarray(
        matrix,
        dtype=np.float32,
    ).copy()

    np.subtract(
        scaled,
        scaler_mean,
        out=scaled,
        casting="unsafe",
    )
    np.divide(
        scaled,
        scaler_scale,
        out=scaled,
        casting="unsafe",
    )

    projected = scaled @ components.T
    projected -= offset[None]

    return np.asarray(projected, dtype=np.float32)


# ---------------------------------------------------------
# 1. Create a clean staging directory from the submitted
#    full-ensemble Phase30 archive.
# ---------------------------------------------------------

assert PHASE39_BASE_ARCHIVE.is_file()
assert PHASE39_DEPENDENCY_ARCHIVE.is_file()

temporary_stage = Path(
    tempfile.mkdtemp(
        prefix="phase39_experimental_stage_",
        dir="/kaggle/working",
    )
)

phase39_safe_extract(
    PHASE39_BASE_ARCHIVE,
    temporary_stage,
)

assert (temporary_stage / "main.py").is_file()
assert (temporary_stage / "phase12c_main.py").is_file()
assert (temporary_stage / "models").is_dir()

base_archive_names = []
with zipfile.ZipFile(PHASE39_BASE_ARCHIVE, "r") as archive:
    base_archive_names = archive.namelist()

assert "main.py" in base_archive_names
assert not any(
    name.startswith("phase30_submission/")
    for name in base_archive_names
)

# Preserve the exact source that produced the latest submitted
# full-ensemble archive before the Phase39 entrypoint is written.
shutil.copy2(
    temporary_stage / "main.py",
    temporary_stage / "phase30_accepted_main.py",
)

if PHASE39_STAGING_DIRECTORY.exists():
    assert (
        PHASE39_STAGING_DIRECTORY.parent
        == Path("/kaggle/working")
    )
    shutil.rmtree(PHASE39_STAGING_DIRECTORY)

os.replace(
    temporary_stage,
    PHASE39_STAGING_DIRECTORY,
)


# ---------------------------------------------------------
# 2. Extract the dependency patch privately for conversion.
# ---------------------------------------------------------

dependency_directory = Path(
    tempfile.mkdtemp(
        prefix="phase39_dependency_extract_",
        dir="/kaggle/working",
    )
)

phase39_safe_extract(
    PHASE39_DEPENDENCY_ARCHIVE,
    dependency_directory,
)

assets_directory = (
    PHASE39_STAGING_DIRECTORY
    / "phase39_assets"
)
assets_directory.mkdir()

model_directory = assets_directory / "phase33_models"
model_directory.mkdir()

booster_directory = assets_directory / "phase36_boosters"
booster_directory.mkdir()


# ---------------------------------------------------------
# 3. Copy label-free P3CA states and runtime constants.
# ---------------------------------------------------------

for source_name, destination_name in [
    (
        "phase32_fold_local_p3ca_states.npz",
        "phase32_fold_local_p3ca_states.npz",
    ),
    (
        "runtime_small_arrays.npz",
        "runtime_small_arrays.npz",
    ),
    (
        "runtime_small_constants.json",
        "runtime_small_constants.json",
    ),
]:
    source = dependency_directory / source_name
    assert source.is_file(), source_name
    shutil.copy2(source, assets_directory / destination_name)

with np.load(
    assets_directory / "phase32_fold_local_p3ca_states.npz",
    allow_pickle=False,
) as p3ca_state:
    assert p3ca_state["mean"].shape == (2, 3, 6, 384)
    assert p3ca_state["components"].shape == (
        2, 3, 6, 16, 384
    )
    assert np.isfinite(p3ca_state["components"]).all()


# ---------------------------------------------------------
# 4. Export Phase33 model states and portable preprocessors.
# ---------------------------------------------------------

phase33_payload = torch.load(
    dependency_directory
    / "phase33"
    / "phase33_deployment_states.pt",
    map_location="cpu",
    weights_only=False,
)

assert isinstance(phase33_payload, dict)
assert "deployment_states" in phase33_payload

phase33_states = phase33_payload["deployment_states"]
assert len(phase33_states) == 3

phase33_preprocessor_arrays = {}
phase33_fold_metadata = []
phase33_model_files = []

for expected_fold, fold_state in enumerate(phase33_states):
    fold = int(fold_state["fold"])
    assert fold == expected_fold
    assert len(fold_state["model_states"]) == 3

    prefix = f"fold{fold}"
    phase39_preprocessor_arrays(
        fold_state["preprocessor"],
        prefix,
        phase33_preprocessor_arrays,
    )

    fold_model_files = []

    for seed_index, state_dict in enumerate(
        fold_state["model_states"]
    ):
        assert isinstance(state_dict, dict)

        cpu_state = {}
        for name, tensor in state_dict.items():
            assert torch.is_tensor(tensor), name
            tensor = tensor.detach().cpu().contiguous()
            assert torch.isfinite(tensor).all(), name
            cpu_state[name] = tensor

        model_name = (
            f"phase33_fold{fold}_seed{seed_index}.pt"
        )
        model_path = model_directory / model_name
        torch.save(cpu_state, model_path)

        reconstructed = torch.load(
            model_path,
            map_location="cpu",
            weights_only=True,
        )
        assert list(reconstructed) == list(cpu_state)
        assert all(
            torch.equal(reconstructed[name], tensor)
            for name, tensor in cpu_state.items()
        )

        relative_name = (
            f"phase39_assets/phase33_models/{model_name}"
        )
        fold_model_files.append(relative_name)
        phase33_model_files.append(relative_name)

    phase33_fold_metadata.append({
        "fold": fold,
        "stage_index": int(
            fold_state["preprocessor"]["stage_index"]
        ),
        "domain_strength": float(
            fold_state["domain_strength"]
        ),
        "best_epochs": [
            int(value)
            for value in fold_state["best_epochs"]
        ],
        "model_files": fold_model_files,
    })

np.savez_compressed(
    assets_directory / "phase33_preprocessors.npz",
    **phase33_preprocessor_arrays,
)


# ---------------------------------------------------------
# 5. Export Phase36 portable preprocessors and boosters.
# ---------------------------------------------------------

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    phase36_payload = joblib.load(
        dependency_directory
        / "phase36"
        / "phase36_deployment_state.joblib"
    )

assert isinstance(phase36_payload, dict)
assert len(phase36_payload["states"]) == 3
assert int(phase36_payload["fold_count"]) == 3
assert int(phase36_payload["seed_count"]) == 2

phase36_preprocessor_arrays = {}
phase36_fold_metadata = []

for expected_fold, fold_state in enumerate(
    phase36_payload["states"]
):
    fold = int(fold_state["fold"])
    assert fold == expected_fold
    assert len(fold_state["boosters"]) == 2

    prefix = f"fold{fold}"
    phase39_preprocessor_arrays(
        fold_state["preprocessor"],
        prefix,
        phase36_preprocessor_arrays,
    )

    booster_records = []

    for seed_index, booster_record in enumerate(
        fold_state["boosters"]
    ):
        reference = booster_record["booster"]
        assert set(reference) == {
            "__phase36_xgboost_booster__"
        }

        source_name = reference[
            "__phase36_xgboost_booster__"
        ]
        source_path = (
            dependency_directory
            / "phase36"
            / source_name
        )
        assert source_path.is_file(), source_name

        destination_path = booster_directory / source_name
        shutil.copy2(source_path, destination_path)

        booster_records.append({
            "seed_index": seed_index,
            "seed": int(booster_record["seed"]),
            "round_count": int(
                booster_record["round_count"]
            ),
            "file": (
                "phase39_assets/phase36_boosters/"
                + source_name
            ),
        })

    phase36_fold_metadata.append({
        "fold": fold,
        "stage_index": int(
            fold_state["preprocessor"]["stage_index"]
        ),
        "scale": float(fold_state["scale"]),
        "specification": {
            key: (
                int(value)
                if key == "maximum_depth"
                else float(value)
            )
            for key, value
            in fold_state["specification"].items()
        },
        "boosters": booster_records,
    })

np.savez_compressed(
    assets_directory / "phase36_preprocessors.npz",
    **phase36_preprocessor_arrays,
)


# ---------------------------------------------------------
# 6. Verify portable preprocessing against sklearn exactly.
# ---------------------------------------------------------

summary_cache = np.load(
    "/kaggle/working/phase32_p3ca_summary_float16.npy",
    mmap_mode="r",
    allow_pickle=False,
)
radiomics_cache = np.load(
    "/kaggle/working/phase31_radiomics_float32.npy",
    mmap_mode="r",
    allow_pickle=False,
)

audit_indices = np.linspace(
    0,
    1361,
    24,
    dtype=np.int64,
)

portable_errors = {
    "phase33": [],
    "phase36": [],
}

for family_name, states, exported in [
    (
        "phase33",
        phase33_states,
        phase33_preprocessor_arrays,
    ),
    (
        "phase36",
        phase36_payload["states"],
        phase36_preprocessor_arrays,
    ),
]:
    for fold_state in states:
        fold = int(fold_state["fold"])
        prefix = f"fold{fold}"
        preprocessor = fold_state["preprocessor"]

        summary_input = np.asarray(
            summary_cache[1, fold, audit_indices],
            dtype=np.float32,
        )
        radiomics_input = np.asarray(
            radiomics_cache[audit_indices],
            dtype=np.float32,
        )

        reference_summary = (
            preprocessor["summary_projector"].transform(
                preprocessor["summary_scaler"].transform(
                    summary_input.copy()
                )
            )
        ).astype(np.float32)

        portable_summary = phase39_manual_project(
            summary_input,
            exported[f"{prefix}_summary_scaler_mean"],
            exported[f"{prefix}_summary_scaler_scale"],
            exported[f"{prefix}_summary_pca_components"],
            exported[f"{prefix}_summary_pca_offset"],
        )

        reference_radiomics = (
            preprocessor["radiomics_projector"].transform(
                preprocessor["radiomics_scaler"].transform(
                    radiomics_input.copy()
                )
            )
        ).astype(np.float32)

        portable_radiomics = phase39_manual_project(
            radiomics_input,
            exported[f"{prefix}_radiomics_scaler_mean"],
            exported[f"{prefix}_radiomics_scaler_scale"],
            exported[f"{prefix}_radiomics_pca_components"],
            exported[f"{prefix}_radiomics_pca_offset"],
        )

        portable_errors[family_name].append({
            "fold": fold,
            "summary_maximum_error": float(
                np.max(
                    np.abs(
                        reference_summary
                        - portable_summary
                    )
                )
            ),
            "radiomics_maximum_error": float(
                np.max(
                    np.abs(
                        reference_radiomics
                        - portable_radiomics
                    )
                )
            ),
        })

maximum_portable_error = max(
    value
    for records in portable_errors.values()
    for record in records
    for key, value in record.items()
    if key.endswith("_error")
)

assert maximum_portable_error <= 5e-5, {
    "message": "Portable preprocessing parity failed.",
    "maximum_error": maximum_portable_error,
    "records": portable_errors,
}


# ---------------------------------------------------------
# 7. Persist runtime metadata and audit hashes.
# ---------------------------------------------------------

runtime_state = {
    "schema_version": 1,
    "phase": "phase39_phase33_phase36_residual_stack",
    "p3ca_stage_index": 1,
    "phase33_folds": phase33_fold_metadata,
    "phase36_folds": phase36_fold_metadata,
    "phase36_selected_specification": (
        phase36_payload["selected_specification"]
    ),
    "phase36_training_shared_scale": float(
        phase36_payload["shared_scale"]
    ),
    "phase39_formula": {
        "phase36_weight": 0.75,
        "phase33_weight": 0.25,
        "alpha": 1.0,
        "uncertainty_exponent": 0.0,
        "delta_cap": 2.0,
    },
    "deployment_fold_aggregation": (
        "mean fold-local residual across folds"
    ),
    "phase33_seed_aggregation": "mean logit",
    "phase36_seed_aggregation": "mean residual",
    "float16_dense_cache_quantization": True,
    "float16_projected_cache_quantization": True,
}

(assets_directory / "phase39_runtime_state.json").write_text(
    json.dumps(runtime_state, indent=2),
    encoding="utf-8",
)

# Preserve the recovered source contracts for the runtime-builder cell.
for source_name in [
    "phase33_exact_architecture.py.txt",
    "phase32_projection_history_cell.py.txt",
]:
    shutil.copy2(
        dependency_directory / source_name,
        assets_directory / source_name,
    )

new_asset_paths = sorted(
    path
    for path in assets_directory.rglob("*")
    if path.is_file()
)

asset_hashes = {
    path.relative_to(
        PHASE39_STAGING_DIRECTORY
    ).as_posix(): phase39_file_sha256(path)
    for path in new_asset_paths
}

asset_manifest = {
    "schema_version": 1,
    "phase": "phase39_portable_assets",
    "assets": asset_hashes,
    "contains_labels": False,
    "contains_oof_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
}

(assets_directory / "phase39_asset_manifest.json").write_text(
    json.dumps(asset_manifest, indent=2),
    encoding="utf-8",
)

package_files = sorted(
    path
    for path in PHASE39_STAGING_DIRECTORY.rglob("*")
    if path.is_file()
)

report = {
    "phase": "phase39_portable_runtime_staging",
    "status": "accepted",
    "base_archive": PHASE39_BASE_ARCHIVE.name,
    "staging_directory": PHASE39_STAGING_DIRECTORY.name,
    "package_file_count": len(package_files),
    "package_size_mb": round(
        sum(path.stat().st_size for path in package_files)
        / 1e6,
        3,
    ),
    "phase33_model_count": len(phase33_model_files),
    "phase33_preprocessor_count": 3,
    "phase36_booster_count": sum(
        len(record["boosters"])
        for record in phase36_fold_metadata
    ),
    "phase36_preprocessor_count": 3,
    "p3ca_basis_count": 36,
    "maximum_portable_preprocessing_error": round(
        maximum_portable_error,
        10,
    ),
    "accepted_phase30_main_preserved": True,
    "new_runtime_entrypoint_written": False,
    "accepted_phase30_archives_modified": False,
    "contains_labels": False,
    "contains_oof_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase39_stage_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE39_PORTABLE_STAGING")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE39_PORTABLE_STAGING")
print(PHASE39_STAGING_DIRECTORY)

BEGIN SANITIZED_PHASE39_PORTABLE_STAGING
{
  "phase": "phase39_portable_runtime_staging",
  "status": "accepted",
  "base_archive": "phase30_full_ensemble_submission.zip",
  "staging_directory": "phase39_experimental_submission",
  "package_file_count": 85,
  "package_size_mb": 202.661,
  "phase33_model_count": 9,
  "phase33_preprocessor_count": 3,
  "phase36_booster_count": 6,
  "phase36_preprocessor_count": 3,
  "p3ca_basis_count": 36,
  "maximum_portable_preprocessing_error": 0.0,
  "accepted_phase30_main_preserved": true,
  "new_runtime_entrypoint_written": false,
  "accepted_phase30_archives_modified": false,
  "contains_labels": false,
  "contains_oof_predictions": false,
  "contains_case_rows": false,
  "contains_voxel_data": false,
  "contains_embeddings": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "model_hashes_displayed": false,
  "elapsed_seconds": 3.04
}
END SANITIZED_PHASE39_PORTABLE_STAGING
/kaggle/working/phase39_experimental_submission


In [97]:
# Phase39 Cell 126
# Write and validate the complete offline Phase39 runtime entrypoint.
#
# Prerequisite: accepted SANITIZED_PHASE39_PORTABLE_STAGING from Cell 125.
# This cell uses synthetic inputs only. It does not read challenge, smoke,
# test, patient, label, prediction, or training-cache rows.

from pathlib import Path
import hashlib
import importlib
import json
import sys
import time

import numpy as np
import torch
import xgboost as xgb


PHASE39_STAGING_DIRECTORY = Path(
    "/kaggle/working/phase39_experimental_submission"
)
PHASE39_RUNTIME_PATH = (
    PHASE39_STAGING_DIRECTORY / "phase39_runtime.py"
)
PHASE39_MAIN_PATH = (
    PHASE39_STAGING_DIRECTORY / "main.py"
)
PHASE39_SOURCE_MANIFEST_PATH = (
    PHASE39_STAGING_DIRECTORY
    / "phase39_runtime_source_manifest.json"
)

phase39_runtime_started = time.perf_counter()


def phase39_source_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


assert PHASE39_STAGING_DIRECTORY.is_dir()
assert (
    PHASE39_STAGING_DIRECTORY
    / "phase30_accepted_main.py"
).is_file()
assert (
    PHASE39_STAGING_DIRECTORY
    / "phase39_assets"
    / "phase39_runtime_state.json"
).is_file()
assert (
    PHASE39_STAGING_DIRECTORY
    / "phase39_assets"
    / "phase39_asset_manifest.json"
).is_file()

PHASE39_RUNTIME_SOURCE = r'''
from __future__ import annotations

import hashlib
import json
import math
import os
from collections.abc import Mapping
from pathlib import Path

import numpy as np
import torch
from scipy import ndimage as ndi
from torch import nn
import xgboost as xgb

import phase30_accepted_main as legacy


ROOT = Path(__file__).resolve().parent
ASSET_DIRECTORY = ROOT / "phase39_assets"
PROBABILITY_EPSILON = 1e-5


def _read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


SMALL_CONSTANTS = _read_json(
    ASSET_DIRECTORY / "runtime_small_constants.json"
)
PHASE31_CONFIG = SMALL_CONSTANTS["PHASE31_CONFIG"]
PHASE31_PROFILE_NAMES = SMALL_CONSTANTS[
    "PHASE31_PROFILE_NAMES"
]
PHASE31_SHAPE_STAT_NAMES = SMALL_CONSTANTS[
    "PHASE31_SHAPE_STAT_NAMES"
]
PHASE36_CONFIG = SMALL_CONSTANTS["PHASE36_CONFIG"]


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def resolve_asset(relative_path):
    path = (ROOT / str(relative_path)).resolve()
    if path != ROOT and ROOT not in path.parents:
        raise RuntimeError("Invalid Phase39 asset path")
    if not path.is_file():
        raise RuntimeError("Missing Phase39 runtime asset")
    return path


def verify_phase39_assets():
    manifest_path = (
        ASSET_DIRECTORY / "phase39_asset_manifest.json"
    )
    manifest = _read_json(manifest_path)
    if manifest.get("schema_version") != 1:
        raise RuntimeError("Unsupported Phase39 asset manifest")
    if manifest.get("phase") != "phase39_portable_assets":
        raise RuntimeError("Unexpected Phase39 asset phase")

    hashes = manifest.get("assets")
    if not isinstance(hashes, dict) or not hashes:
        raise RuntimeError("Missing Phase39 integrity records")

    for relative_path, expected_hash in hashes.items():
        path = resolve_asset(relative_path)
        if file_sha256(path) != expected_hash:
            raise RuntimeError("Phase39 asset integrity check failed")

    return manifest


def probability_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )
    return np.log(probability / (1.0 - probability))


def probability_sigmoid(logit):
    logit = np.clip(
        np.asarray(logit, dtype=np.float64),
        -40.0,
        40.0,
    )
    return 1.0 / (1.0 + np.exp(-logit))


# ------------------------------------------------------------------
# Exact Phase31 localized multithreshold geometry/radiomics features.
# ------------------------------------------------------------------


def phase31_top_fraction_mean(array, fraction=0.10):
    flat = np.asarray(array, dtype=np.float32).ravel()
    if flat.size == 0:
        return 0.0
    count = max(1, int(math.ceil(flat.size * fraction)))
    boundary = flat.size - count
    selected = np.partition(flat, boundary)[boundary:]
    return float(np.mean(selected))


def phase31_background_level(volume):
    positive = np.asarray(
        volume[volume > 0],
        dtype=np.float32,
    )
    if positive.size < 128:
        return 1e-3
    upper = float(np.quantile(positive, 0.70))
    reference = positive[positive <= upper]
    if reference.size < 64:
        reference = positive
    return max(float(np.median(reference)), 1e-4)


def phase31_weighted_center(subvolume, offset_x):
    values = np.asarray(subvolume, dtype=np.float32)
    positive = values[values > 0]

    if positive.size < 32:
        local = np.asarray(
            np.unravel_index(
                int(np.argmax(values)),
                values.shape,
            ),
            dtype=np.float64,
        )
    else:
        threshold = float(
            np.quantile(
                positive,
                PHASE31_CONFIG["localization_quantile"],
            )
        )
        weights = np.square(
            np.clip(values - threshold, 0.0, None)
        )
        total = float(weights.sum())

        if total <= 1e-8:
            local = np.asarray(
                np.unravel_index(
                    int(np.argmax(values)),
                    values.shape,
                ),
                dtype=np.float64,
            )
        else:
            coordinates = np.indices(
                values.shape,
                dtype=np.float64,
            )
            local = np.asarray([
                np.sum(coordinates[axis] * weights) / total
                for axis in range(3)
            ])

    local[0] += float(offset_x)
    return local


def phase31_localize_bilateral(volume):
    if tuple(volume.shape) != (80, 80, 80):
        raise ValueError("Unexpected Phase31 volume shape")
    midpoint = volume.shape[0] // 2
    left_center = phase31_weighted_center(
        volume[:midpoint],
        offset_x=0,
    )
    right_center = phase31_weighted_center(
        volume[midpoint:],
        offset_x=midpoint,
    )
    return left_center, right_center


def phase31_extract_integer_patch(volume, center, size):
    if size % 2 != 1:
        raise ValueError("Phase31 patch size must be odd")

    center_index = np.rint(center).astype(np.int64)
    radius = size // 2
    start = center_index - radius
    stop = start + size

    patch = np.zeros(
        (size, size, size),
        dtype=np.float32,
    )

    source_start = np.maximum(start, 0)
    source_stop = np.minimum(
        stop,
        np.asarray(volume.shape),
    )
    destination_start = source_start - start
    destination_stop = (
        destination_start + source_stop - source_start
    )

    source_slices = tuple(
        slice(int(a), int(b))
        for a, b in zip(source_start, source_stop)
    )
    destination_slices = tuple(
        slice(int(a), int(b))
        for a, b in zip(
            destination_start,
            destination_stop,
        )
    )
    patch[destination_slices] = np.asarray(
        volume[source_slices],
        dtype=np.float32,
    )
    return patch


def phase31_shape_statistics(normalized_signal, threshold):
    mask = normalized_signal >= float(threshold)
    voxel_count = int(mask.sum())
    total_voxels = int(mask.size)

    if voxel_count < 3:
        return np.zeros(11, dtype=np.float32)

    structure_3d = ndi.generate_binary_structure(3, 1)
    labels, component_count = ndi.label(
        mask,
        structure=structure_3d,
    )
    counts = np.bincount(labels.ravel())[1:]
    largest_fraction = (
        float(counts.max()) / voxel_count
        if counts.size
        else 0.0
    )

    eroded = ndi.binary_erosion(
        mask,
        structure=structure_3d,
        border_value=0,
    )
    surface_count = int(np.sum(mask & ~eroded))
    surface_fraction = surface_count / max(voxel_count, 1)
    if surface_count > 0:
        sphericity = (
            (math.pi ** (1.0 / 3.0))
            * ((6.0 * voxel_count) ** (2.0 / 3.0))
            / surface_count
        )
    else:
        sphericity = 0.0

    coordinates = np.argwhere(mask).astype(np.float64)
    if len(coordinates) >= 4:
        covariance = np.cov(coordinates, rowvar=False)
        eigenvalues = np.sort(
            np.linalg.eigvalsh(covariance)
        )[::-1]
        eigenvalues = np.clip(eigenvalues, 0.0, None)
        denominator = max(float(eigenvalues[0]), 1e-8)
        eigen_middle_ratio = float(
            eigenvalues[1] / denominator
        )
        eigen_small_ratio = float(
            eigenvalues[2] / denominator
        )
    else:
        eigen_middle_ratio = 0.0
        eigen_small_ratio = 0.0

    projection = normalized_signal.max(axis=2)
    mask_2d = projection >= float(threshold)
    area_2d = int(mask_2d.sum())
    structure_2d = ndi.generate_binary_structure(2, 1)
    eroded_2d = ndi.binary_erosion(
        mask_2d,
        structure=structure_2d,
        border_value=0,
    )
    perimeter = int(np.sum(mask_2d & ~eroded_2d))
    circularity = (
        4.0 * math.pi * area_2d / (perimeter ** 2)
        if perimeter > 0
        else 0.0
    )

    coordinates_2d = np.argwhere(mask_2d).astype(np.float64)
    if len(coordinates_2d) >= 3:
        covariance_2d = np.cov(coordinates_2d, rowvar=False)
        eigenvalues_2d = np.sort(
            np.linalg.eigvalsh(covariance_2d)
        )[::-1]
        eigenvalues_2d = np.clip(
            eigenvalues_2d,
            0.0,
            None,
        )
        eccentricity = math.sqrt(
            max(
                0.0,
                1.0
                - float(eigenvalues_2d[1])
                / max(float(eigenvalues_2d[0]), 1e-8),
            )
        )
    else:
        eccentricity = 0.0

    return np.asarray([
        voxel_count / total_voxels,
        float(normalized_signal[mask].mean()),
        float(component_count),
        largest_fraction,
        surface_fraction,
        float(np.clip(sphericity, 0.0, 2.0)),
        eigen_middle_ratio,
        eigen_small_ratio,
        area_2d / mask_2d.size,
        float(np.clip(circularity, 0.0, 2.0)),
        eccentricity,
    ], dtype=np.float32)


def phase31_side_features(patch, background, scale_name):
    raw = np.asarray(patch, dtype=np.float32)
    signal = np.clip(raw - background, 0.0, None)
    positive = signal[signal > 0]
    if positive.size:
        scale = max(
            float(np.quantile(positive, 0.995)),
            1e-5,
        )
    else:
        scale = 1e-5

    normalized = np.clip(signal / scale, 0.0, 2.0)
    quantiles = [
        float(np.quantile(raw, q))
        for q in (0.50, 0.75, 0.90, 0.95, 0.99)
    ]
    values = [
        float(raw.mean() / background),
        float(raw.std() / background),
        *[value / background for value in quantiles],
        phase31_top_fraction_mean(raw, 0.05) / background,
        float(np.mean(raw > 0)),
        float(scale / background),
        float(normalized.mean()),
        float(normalized.std()),
    ]
    names = [
        f"{scale_name}_raw_mean_to_background",
        f"{scale_name}_raw_std_to_background",
        f"{scale_name}_raw_q50_to_background",
        f"{scale_name}_raw_q75_to_background",
        f"{scale_name}_raw_q90_to_background",
        f"{scale_name}_raw_q95_to_background",
        f"{scale_name}_raw_q99_to_background",
        f"{scale_name}_top05_to_background",
        f"{scale_name}_positive_fraction",
        f"{scale_name}_signal_scale_to_background",
        f"{scale_name}_normalized_mean",
        f"{scale_name}_normalized_std",
    ]

    for threshold in PHASE31_CONFIG["thresholds"]:
        statistics = phase31_shape_statistics(
            normalized,
            threshold,
        )
        threshold_name = (
            f"{scale_name}_t{int(round(threshold * 100)):02d}"
        )
        values.extend(statistics.tolist())
        names.extend([
            f"{threshold_name}_{name}"
            for name in PHASE31_SHAPE_STAT_NAMES
        ])

    return (
        np.asarray(values, dtype=np.float32),
        names,
        scale,
    )


def phase31_profile_tokens(normalized_patch):
    bin_indices = np.array_split(
        np.arange(normalized_patch.shape[1]),
        PHASE31_CONFIG["profile_bins"],
    )
    tokens = []
    for indices in bin_indices:
        slab = normalized_patch[:, indices, :]
        flat = slab.ravel()
        mask = slab >= 0.35
        coordinates = np.argwhere(mask)

        if coordinates.size:
            width_lr = (
                coordinates[:, 0].max()
                - coordinates[:, 0].min()
                + 1
            ) / slab.shape[0]
            width_si = (
                coordinates[:, 2].max()
                - coordinates[:, 2].min()
                + 1
            ) / slab.shape[2]
            centroid_lr = (
                2.0
                * float(coordinates[:, 0].mean())
                / max(slab.shape[0] - 1, 1)
                - 1.0
            )
            centroid_si = (
                2.0
                * float(coordinates[:, 2].mean())
                / max(slab.shape[2] - 1, 1)
                - 1.0
            )
        else:
            width_lr = 0.0
            width_si = 0.0
            centroid_lr = 0.0
            centroid_si = 0.0

        tokens.append([
            float(flat.mean()),
            float(flat.std()),
            float(np.quantile(flat, 0.75)),
            float(np.quantile(flat, 0.90)),
            float(np.quantile(flat, 0.99)),
            phase31_top_fraction_mean(flat, 0.10),
            float(np.mean(flat >= 0.35)),
            float(np.mean(flat >= 0.50)),
            float(np.mean(flat >= 0.65)),
            float(width_lr),
            float(width_si),
            float(centroid_lr),
            float(centroid_si),
        ])
    return np.asarray(tokens, dtype=np.float32)


def phase31_case_features(volume):
    volume = np.asarray(volume, dtype=np.float32)
    if volume.shape != (80, 80, 80):
        raise ValueError("Unexpected Phase31 high-resolution shape")
    if not np.isfinite(volume).all():
        raise ValueError("Non-finite Phase31 volume")

    background = phase31_background_level(volume)
    left_center, right_center = phase31_localize_bilateral(volume)
    feature_values = []
    feature_names = []
    profile_left = None
    profile_right = None
    profile_common_scale = None

    for patch_size in PHASE31_CONFIG["patch_sizes_voxels"]:
        left_patch = phase31_extract_integer_patch(
            volume,
            left_center,
            patch_size,
        )
        right_patch = phase31_extract_integer_patch(
            volume,
            right_center,
            patch_size,
        )
        right_patch = np.flip(right_patch, axis=0).copy()
        scale_name = f"patch_{patch_size:02d}"

        left_features, left_names, _ = phase31_side_features(
            left_patch,
            background,
            scale_name,
        )
        right_features, right_names, _ = phase31_side_features(
            right_patch,
            background,
            scale_name,
        )
        if left_names != right_names:
            raise RuntimeError("Phase31 side feature schema mismatch")

        operations = {
            "bilateral_mean": 0.5 * (
                left_features + right_features
            ),
            "bilateral_absdiff": np.abs(
                left_features - right_features
            ),
            "bilateral_minimum": np.minimum(
                left_features,
                right_features,
            ),
            "bilateral_maximum": np.maximum(
                left_features,
                right_features,
            ),
        }
        for operation_name, operation_values in operations.items():
            feature_values.extend(operation_values.tolist())
            feature_names.extend([
                f"{operation_name}_{name}"
                for name in left_names
            ])

        if patch_size == 41:
            left_signal = np.clip(
                left_patch - background,
                0.0,
                None,
            )
            right_signal = np.clip(
                right_patch - background,
                0.0,
                None,
            )
            combined_positive = np.concatenate([
                left_signal[left_signal > 0],
                right_signal[right_signal > 0],
            ])
            if combined_positive.size:
                common_scale = max(
                    float(
                        np.quantile(combined_positive, 0.995)
                    ),
                    1e-5,
                )
            else:
                common_scale = 1e-5

            profile_left = phase31_profile_tokens(
                np.clip(
                    left_signal / common_scale,
                    0.0,
                    2.0,
                )
            )
            profile_right = phase31_profile_tokens(
                np.clip(
                    right_signal / common_scale,
                    0.0,
                    2.0,
                )
            )
            profile_common_scale = common_scale

    if profile_left is None or profile_right is None:
        raise RuntimeError("Phase31 profile construction failed")

    sequence = np.concatenate([
        0.5 * (profile_left + profile_right),
        np.abs(profile_left - profile_right),
        np.minimum(profile_left, profile_right),
        np.maximum(profile_left, profile_right),
    ], axis=1)
    sequence_names = (
        [f"bilateral_mean_{name}" for name in PHASE31_PROFILE_NAMES]
        + [
            f"bilateral_absdiff_{name}"
            for name in PHASE31_PROFILE_NAMES
        ]
        + [
            f"bilateral_minimum_{name}"
            for name in PHASE31_PROFILE_NAMES
        ]
        + [
            f"bilateral_maximum_{name}"
            for name in PHASE31_PROFILE_NAMES
        ]
    )

    positive = volume[volume > 0]
    if positive.size:
        global_quantiles = [
            float(np.quantile(positive, q))
            for q in (0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)
        ]
    else:
        global_quantiles = [0.0] * 7

    center_separation = float(
        np.linalg.norm(left_center - right_center)
    )
    global_values = [
        background,
        float(volume.mean()),
        float(volume.std()),
        float(np.mean(volume > 0)),
        *global_quantiles,
        center_separation,
        float(profile_common_scale / background),
    ]
    global_names = [
        "global_background",
        "global_mean",
        "global_standard_deviation",
        "global_positive_fraction",
        "global_positive_q10",
        "global_positive_q25",
        "global_positive_q50",
        "global_positive_q75",
        "global_positive_q90",
        "global_positive_q95",
        "global_positive_q99",
        "bilateral_center_separation_voxels",
        "profile_common_scale_to_background",
    ]
    feature_values.extend(global_values)
    feature_names.extend(global_names)

    diagnostics = {
        "background": background,
        "center_separation_voxels": center_separation,
        "left_center": left_center,
        "right_center": right_center,
        "profile_common_scale": profile_common_scale,
    }
    features = np.asarray(feature_values, dtype=np.float32)
    sequence = sequence.astype(np.float32)
    if features.shape != (1081,):
        raise RuntimeError("Phase31 radiomics dimension mismatch")
    if sequence.shape != (16, 52):
        raise RuntimeError("Phase31 sequence dimension mismatch")

    return (
        features,
        feature_names,
        sequence,
        sequence_names,
        diagnostics,
    )


# ------------------------------------------------------------------
# Exact Phase33 inference architecture.
# ------------------------------------------------------------------


class Phase33GradientReverseFunction(torch.autograd.Function):
    @staticmethod
    def forward(context, input_tensor, coefficient):
        context.coefficient = float(coefficient)
        return input_tensor.view_as(input_tensor)

    @staticmethod
    def backward(context, gradient_output):
        return (
            -context.coefficient * gradient_output,
            None,
        )


def phase33_gradient_reverse(input_tensor, coefficient):
    return Phase33GradientReverseFunction.apply(
        input_tensor,
        coefficient,
    )


class Phase33TemporalBlock(nn.Module):
    def __init__(self, dimension, dilation, dropout=0.08):
        super().__init__()
        self.depthwise = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=3,
            padding=int(dilation),
            dilation=int(dilation),
            groups=dimension,
            bias=False,
        )
        self.normalization = nn.GroupNorm(
            num_groups=8,
            num_channels=dimension,
        )
        self.pointwise = nn.Conv1d(
            dimension,
            2 * dimension,
            kernel_size=1,
        )
        self.output = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=1,
        )
        self.dropout = nn.Dropout(p=float(dropout))

    def forward(self, sequence):
        residual = sequence
        hidden = self.depthwise(sequence)
        hidden = self.normalization(hidden)
        value, gate = self.pointwise(hidden).chunk(2, dim=1)
        hidden = value * torch.sigmoid(gate)
        hidden = self.output(hidden)
        hidden = self.dropout(hidden)
        return residual + hidden


class Phase33DomainInvariantFusionNet(nn.Module):
    def __init__(self, domain_class_count):
        super().__init__()
        model_dimension = 64
        geometry_dimension = 48

        self.p3ca_input = nn.Sequential(
            nn.Linear(48, model_dimension),
            nn.LayerNorm(model_dimension),
            nn.GELU(),
        )
        self.p3ca_view_embedding = nn.Parameter(
            torch.zeros(1, 6, 1, model_dimension)
        )
        self.p3ca_position_embedding = nn.Parameter(
            torch.zeros(1, 1, 14, model_dimension)
        )
        self.p3ca_temporal = nn.Sequential(
            Phase33TemporalBlock(model_dimension, dilation=1),
            Phase33TemporalBlock(model_dimension, dilation=2),
            Phase33TemporalBlock(model_dimension, dilation=3),
        )
        self.position_attention = nn.Linear(model_dimension, 1)
        self.view_attention = nn.Linear(model_dimension, 1)
        self.p3ca_projection = nn.Sequential(
            nn.Linear(3 * model_dimension, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.geometry_input = nn.Sequential(
            nn.Linear(52, geometry_dimension),
            nn.LayerNorm(geometry_dimension),
            nn.GELU(),
        )
        self.geometry_temporal = nn.Sequential(
            Phase33TemporalBlock(geometry_dimension, dilation=1),
            Phase33TemporalBlock(geometry_dimension, dilation=2),
        )
        self.geometry_attention = nn.Linear(geometry_dimension, 1)
        self.geometry_projection = nn.Sequential(
            nn.Linear(2 * geometry_dimension, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.static_projection = nn.Sequential(
            nn.Linear(96, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(96 + 64 + 64),
            nn.Linear(96 + 64 + 64, 128),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        self.classifier = nn.Linear(128, 1)
        self.domain_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(64, int(domain_class_count)),
        )
        nn.init.trunc_normal_(self.p3ca_view_embedding, std=0.02)
        nn.init.trunc_normal_(self.p3ca_position_embedding, std=0.02)

    def encode_p3ca(self, sequence):
        batch_size = sequence.shape[0]
        sequence = sequence.reshape(batch_size, 6, 14, 48)
        hidden = self.p3ca_input(sequence)
        hidden = (
            hidden
            + self.p3ca_view_embedding
            + self.p3ca_position_embedding
        )
        hidden = hidden.reshape(
            batch_size * 6,
            14,
            64,
        ).transpose(1, 2)
        hidden = self.p3ca_temporal(hidden).transpose(1, 2)
        position_weight = torch.softmax(
            self.position_attention(hidden).squeeze(-1),
            dim=1,
        )
        view_embedding = torch.sum(
            hidden * position_weight[:, :, None],
            dim=1,
        ).reshape(batch_size, 6, 64)
        view_weight = torch.softmax(
            self.view_attention(view_embedding).squeeze(-1),
            dim=1,
        )
        attentive_view = torch.sum(
            view_embedding * view_weight[:, :, None],
            dim=1,
        )
        mean_view = view_embedding.mean(dim=1)
        maximum_view = view_embedding.amax(dim=1)
        return self.p3ca_projection(
            torch.cat(
                [attentive_view, mean_view, maximum_view],
                dim=1,
            )
        )

    def encode_geometry(self, sequence):
        hidden = self.geometry_input(sequence).transpose(1, 2)
        hidden = self.geometry_temporal(hidden).transpose(1, 2)
        weight = torch.softmax(
            self.geometry_attention(hidden).squeeze(-1),
            dim=1,
        )
        attentive = torch.sum(
            hidden * weight[:, :, None],
            dim=1,
        )
        mean = hidden.mean(dim=1)
        return self.geometry_projection(
            torch.cat([attentive, mean], dim=1)
        )

    def forward(
        self,
        p3ca_sequence,
        geometry_sequence,
        summary_features,
        radiomics_features,
        domain_coefficient=0.0,
    ):
        p3ca_embedding = self.encode_p3ca(p3ca_sequence)
        geometry_embedding = self.encode_geometry(
            geometry_sequence
        )
        static_embedding = self.static_projection(
            torch.cat(
                [summary_features, radiomics_features],
                dim=1,
            )
        )
        fused = self.fusion(
            torch.cat(
                [
                    p3ca_embedding,
                    geometry_embedding,
                    static_embedding,
                ],
                dim=1,
            )
        )
        logit = self.classifier(fused).squeeze(1)
        reversed_embedding = phase33_gradient_reverse(
            fused,
            domain_coefficient,
        )
        domain_logit = self.domain_head(reversed_embedding)
        return {
            "logit": logit,
            "domain_logit": domain_logit,
            "embedding": fused,
        }


# ------------------------------------------------------------------
# Portable preprocessing and Phase36 feature construction.
# ------------------------------------------------------------------


def _load_npz_dictionary(path):
    with np.load(path, allow_pickle=False) as asset:
        return {
            key: np.asarray(asset[key]).copy()
            for key in asset.files
        }


def portable_project(
    matrix,
    scaler_mean,
    scaler_scale,
    components,
    offset,
):
    scaled = np.asarray(matrix, dtype=np.float32).copy()
    np.subtract(
        scaled,
        scaler_mean,
        out=scaled,
        casting="unsafe",
    )
    np.divide(
        scaled,
        scaler_scale,
        out=scaled,
        casting="unsafe",
    )
    projected = scaled @ components.T
    projected -= offset[None]
    return np.asarray(projected, dtype=np.float32)


def transform_fold_features(
    arrays,
    fold,
    p3ca_sequence,
    geometry_sequence,
    p3ca_summary,
    radiomics,
):
    prefix = f"fold{int(fold)}"
    p3ca_sequence = np.asarray(
        p3ca_sequence,
        dtype=np.float32,
    )
    geometry_sequence = np.asarray(
        geometry_sequence,
        dtype=np.float32,
    )

    p3ca_sequence = (
        p3ca_sequence
        - arrays[f"{prefix}_p3ca_token_mean"][None, None]
    ) / arrays[
        f"{prefix}_p3ca_token_standard_deviation"
    ][None, None]
    geometry_sequence = (
        geometry_sequence
        - arrays[f"{prefix}_geometry_token_mean"][None, None]
    ) / arrays[
        f"{prefix}_geometry_token_standard_deviation"
    ][None, None]

    summary_features = portable_project(
        p3ca_summary,
        arrays[f"{prefix}_summary_scaler_mean"],
        arrays[f"{prefix}_summary_scaler_scale"],
        arrays[f"{prefix}_summary_pca_components"],
        arrays[f"{prefix}_summary_pca_offset"],
    )
    radiomics_features = portable_project(
        radiomics,
        arrays[f"{prefix}_radiomics_scaler_mean"],
        arrays[f"{prefix}_radiomics_scaler_scale"],
        arrays[f"{prefix}_radiomics_pca_components"],
        arrays[f"{prefix}_radiomics_pca_offset"],
    )
    output = {
        "p3ca_sequence": np.asarray(
            p3ca_sequence,
            dtype=np.float32,
        ),
        "geometry_sequence": np.asarray(
            geometry_sequence,
            dtype=np.float32,
        ),
        "summary_features": summary_features,
        "radiomics_features": radiomics_features,
    }
    if not all(np.isfinite(value).all() for value in output.values()):
        raise RuntimeError("Non-finite transformed Phase39 feature")
    return output


def phase36_sequence_statistics(sequence, frequency_count):
    sequence = np.asarray(sequence, dtype=np.float32)
    if sequence.ndim != 3:
        raise ValueError("Expected a sequence feature batch")
    mean = np.mean(sequence, axis=1)
    standard_deviation = np.std(sequence, axis=1)
    q25 = np.quantile(sequence, 0.25, axis=1).astype(np.float32)
    q75 = np.quantile(sequence, 0.75, axis=1).astype(np.float32)
    dynamic_range = (
        np.max(sequence, axis=1)
        - np.min(sequence, axis=1)
    )
    frequency = np.fft.rfft(sequence, axis=1)
    frequency_magnitude = np.abs(
        frequency[
            :,
            1:1 + int(frequency_count),
            :,
        ]
    ).astype(np.float32)
    frequency_magnitude = frequency_magnitude.reshape(
        sequence.shape[0],
        -1,
    )
    return np.concatenate(
        [
            mean,
            standard_deviation,
            q25,
            q75,
            dynamic_range,
            frequency_magnitude,
        ],
        axis=1,
    ).astype(np.float32, copy=False)


def phase36_build_features(transformed_features, baseline_probability):
    p3ca_sequence = np.asarray(
        transformed_features["p3ca_sequence"],
        dtype=np.float32,
    )
    geometry_sequence = np.asarray(
        transformed_features["geometry_sequence"],
        dtype=np.float32,
    )
    summary_features = np.asarray(
        transformed_features["summary_features"],
        dtype=np.float32,
    )
    radiomics_features = np.asarray(
        transformed_features["radiomics_features"],
        dtype=np.float32,
    )
    baseline_probability = np.clip(
        np.asarray(baseline_probability, dtype=np.float64),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )
    baseline_logit = np.clip(
        probability_logit(baseline_probability),
        -PHASE36_CONFIG["base_margin_clip"],
        PHASE36_CONFIG["base_margin_clip"],
    )
    uncertainty = (
        4.0
        * baseline_probability
        * (1.0 - baseline_probability)
    )
    p3ca_statistics = phase36_sequence_statistics(
        p3ca_sequence,
        PHASE36_CONFIG["p3ca_frequency_count"],
    )
    geometry_statistics = phase36_sequence_statistics(
        geometry_sequence,
        PHASE36_CONFIG["geometry_frequency_count"],
    )
    if p3ca_sequence.shape[1:] != (84, 48):
        raise RuntimeError("Phase36 P3CA sequence shape mismatch")
    p3ca_view_means = np.mean(
        p3ca_sequence.reshape(
            p3ca_sequence.shape[0],
            PHASE36_CONFIG["p3ca_view_count"],
            PHASE36_CONFIG["p3ca_tokens_per_view"],
            p3ca_sequence.shape[2],
        ),
        axis=2,
    ).reshape(p3ca_sequence.shape[0], -1)
    anchor_features = np.stack(
        [
            baseline_logit,
            np.abs(baseline_logit),
            baseline_probability - 0.5,
            uncertainty,
        ],
        axis=1,
    ).astype(np.float32)
    feature_matrix = np.concatenate(
        [
            summary_features,
            radiomics_features,
            p3ca_statistics,
            p3ca_view_means,
            geometry_statistics,
            anchor_features,
        ],
        axis=1,
    ).astype(np.float32, copy=False)
    if feature_matrix.shape[1] != 1240:
        raise RuntimeError("Phase36 feature dimension mismatch")
    if not np.isfinite(feature_matrix).all():
        raise RuntimeError("Non-finite Phase36 features")
    return feature_matrix, baseline_logit


def phase36_predict_residual(
    booster,
    features,
    baseline_margin,
    round_count,
):
    matrix = xgb.DMatrix(
        np.asarray(features, dtype=np.float32),
        base_margin=np.asarray(
            baseline_margin,
            dtype=np.float32,
        ),
    )
    total_margin = booster.predict(
        matrix,
        output_margin=True,
        iteration_range=(0, int(round_count)),
    ).astype(np.float64)
    residual = total_margin - np.asarray(
        baseline_margin,
        dtype=np.float64,
    )
    if not np.isfinite(residual).all():
        raise RuntimeError("Non-finite Phase36 residual")
    return residual


class Phase39Runtime:
    def __init__(self, device):
        verify_phase39_assets()
        self.device = torch.device(device)
        self.runtime_state = _read_json(
            ASSET_DIRECTORY / "phase39_runtime_state.json"
        )
        if self.runtime_state.get("schema_version") != 1:
            raise RuntimeError("Unsupported Phase39 runtime state")

        self.formula = self.runtime_state["phase39_formula"]
        expected_formula = {
            "phase36_weight": 0.75,
            "phase33_weight": 0.25,
            "alpha": 1.0,
            "uncertainty_exponent": 0.0,
            "delta_cap": 2.0,
        }
        if self.formula != expected_formula:
            raise RuntimeError("Unexpected Phase39 deployment formula")

        self.prompt_mask = _load_npz_dictionary(
            ASSET_DIRECTORY / "runtime_small_arrays.npz"
        )["phase32_real_prompt_mask"].astype(bool)
        if self.prompt_mask.shape != (14, 14):
            raise RuntimeError("Invalid P3CA prompt mask")
        if int(self.prompt_mask.sum()) != 64:
            raise RuntimeError("Invalid P3CA prompt size")

        self.p3ca = _load_npz_dictionary(
            ASSET_DIRECTORY
            / "phase32_fold_local_p3ca_states.npz"
        )
        if self.p3ca["components"].shape != (
            2,
            3,
            6,
            16,
            384,
        ):
            raise RuntimeError("Invalid P3CA component state")

        self.phase33_preprocessors = _load_npz_dictionary(
            ASSET_DIRECTORY / "phase33_preprocessors.npz"
        )
        self.phase36_preprocessors = _load_npz_dictionary(
            ASSET_DIRECTORY / "phase36_preprocessors.npz"
        )
        self.phase33_models = []
        self.phase36_boosters = []

        for fold_record in self.runtime_state["phase33_folds"]:
            fold = int(fold_record["fold"])
            models = []
            for relative_path in fold_record["model_files"]:
                model = Phase33DomainInvariantFusionNet(
                    domain_class_count=15
                )
                state = torch.load(
                    resolve_asset(relative_path),
                    map_location="cpu",
                    weights_only=True,
                )
                model.load_state_dict(state, strict=True)
                model.eval().to(self.device)
                for parameter in model.parameters():
                    parameter.requires_grad_(False)
                models.append(model)
            if len(models) != 3:
                raise RuntimeError("Incomplete Phase33 fold ensemble")
            if fold != len(self.phase33_models):
                raise RuntimeError("Nonsequential Phase33 fold state")
            self.phase33_models.append(models)

        for fold_record in self.runtime_state["phase36_folds"]:
            fold = int(fold_record["fold"])
            boosters = []
            for booster_record in fold_record["boosters"]:
                booster = xgb.Booster()
                booster.load_model(
                    str(resolve_asset(booster_record["file"]))
                )
                boosters.append({
                    "booster": booster,
                    "round_count": int(
                        booster_record["round_count"]
                    ),
                })
            if len(boosters) != 2:
                raise RuntimeError("Incomplete Phase36 fold ensemble")
            if fold != len(self.phase36_boosters):
                raise RuntimeError("Nonsequential Phase36 fold state")
            self.phase36_boosters.append(boosters)

        if len(self.phase33_models) != 3:
            raise RuntimeError("Incomplete Phase33 deployment state")
        if len(self.phase36_boosters) != 3:
            raise RuntimeError("Incomplete Phase36 deployment state")

    @torch.inference_mode()
    def extract_dense_tokens(self, highres_volumes, backbone):
        array = np.asarray(highres_volumes, dtype=np.float32)
        if array.ndim != 4 or array.shape[1:] != (80, 80, 80):
            raise ValueError("Unexpected Phase39 high-resolution batch")
        volume = torch.from_numpy(array).to(self.device)
        views = legacy.phase30_make_bilateral_views(volume)
        batch_size = views.shape[0]
        flat_views = views.reshape(-1, 3, 224, 224)
        view_batch_size = max(
            1,
            min(
                24,
                int(os.environ.get("DAT_DINO_VIEW_BATCH", "12")),
            ),
        )
        chunks = []
        for start in range(0, flat_views.shape[0], view_batch_size):
            stop = min(start + view_batch_size, flat_views.shape[0])
            output = backbone.forward_features(flat_views[start:stop])
            if not isinstance(output, Mapping):
                raise RuntimeError("Unexpected DINOv3 output")
            patch_tokens = None
            for key in (
                "x_norm_patchtokens",
                "x_norm_patch_tokens",
                "patch_tokens",
            ):
                candidate = output.get(key)
                if torch.is_tensor(candidate) and candidate.ndim == 3:
                    patch_tokens = candidate
                    break
            if patch_tokens is None:
                raise RuntimeError("DINOv3 dense tokens unavailable")
            if patch_tokens.shape[1:] != (196, 384):
                raise RuntimeError("Unexpected DINOv3 token shape")
            chunks.append(patch_tokens.float().cpu())

        dense = torch.cat(chunks, dim=0).reshape(
            batch_size,
            6,
            14,
            14,
            384,
        ).numpy()
        # Exact training dense cache contract.
        return dense.astype(np.float16).astype(np.float32)

    def extract_geometry(self, highres_volumes):
        radiomics = []
        geometry = []
        for volume in np.asarray(highres_volumes, dtype=np.float32):
            features, _, sequence, _, _ = phase31_case_features(volume)
            radiomics.append(features)
            geometry.append(sequence)
        radiomics = np.stack(radiomics).astype(np.float32)
        geometry = np.stack(geometry).astype(np.float32)
        if radiomics.shape[1:] != (1081,):
            raise RuntimeError("Phase31 radiomics batch mismatch")
        if geometry.shape[1:] != (16, 52):
            raise RuntimeError("Phase31 geometry batch mismatch")
        return radiomics, geometry

    @torch.inference_mode()
    def project_p3ca(self, dense_features, fold):
        dense = np.asarray(dense_features, dtype=np.float32)
        if dense.ndim != 5 or dense.shape[1:] != (
            6,
            14,
            14,
            384,
        ):
            raise ValueError("Unexpected dense feature batch")
        stage_index = 1
        mean = torch.from_numpy(
            self.p3ca["mean"][stage_index, fold]
        ).to(self.device)
        standard_deviation = torch.from_numpy(
            self.p3ca["standard_deviation"][stage_index, fold]
        ).to(self.device)
        components = torch.from_numpy(
            self.p3ca["components"][stage_index, fold]
        ).to(self.device)
        dense_tensor = torch.from_numpy(dense).to(self.device)
        normalized = (
            dense_tensor - mean[None, :, None, None, :]
        ) / standard_deviation[None, :, None, None, :]
        projected = torch.einsum(
            "bvhwc,vkc->bvhwk",
            normalized,
            components,
        )
        batch_size = projected.shape[0]
        projected_flat = projected.reshape(batch_size, 6, 196, 16)
        prompt_mask = torch.from_numpy(
            self.prompt_mask.reshape(-1)
        ).to(self.device)
        outside_mask = ~prompt_mask
        prompt_projected = projected_flat[:, :, prompt_mask, :]
        outside_projected = projected_flat[:, :, outside_mask, :]

        full_mean = projected_flat.mean(dim=2)
        full_std = projected_flat.std(dim=2, unbiased=False)
        prompt_mean = prompt_projected.mean(dim=2)
        prompt_std = prompt_projected.std(dim=2, unbiased=False)
        outside_mean = outside_projected.mean(dim=2)
        outside_std = outside_projected.std(dim=2, unbiased=False)
        prompt_contrast = prompt_mean - outside_mean
        view_summary = torch.cat(
            [
                full_mean,
                full_std,
                prompt_mean,
                prompt_std,
                outside_mean,
                outside_std,
                prompt_contrast,
                torch.abs(prompt_contrast),
            ],
            dim=-1,
        )
        case_summary = view_summary.reshape(batch_size, 768)
        row_mean = projected.mean(dim=2)
        row_std = projected.std(dim=2, unbiased=False)
        center_row_mean = projected[:, :, 3:11, :, :].mean(dim=2)
        sequence = torch.cat(
            [row_mean, row_std, center_row_mean],
            dim=-1,
        ).reshape(batch_size, 84, 48)

        if not torch.isfinite(case_summary).all():
            raise RuntimeError("Non-finite P3CA summary")
        if not torch.isfinite(sequence).all():
            raise RuntimeError("Non-finite P3CA sequence")
        # Exact projected cache quantization contract.
        summary_numpy = (
            case_summary.cpu().to(torch.float16).numpy().astype(np.float32)
        )
        sequence_numpy = (
            sequence.cpu().to(torch.float16).numpy().astype(np.float32)
        )
        return summary_numpy, sequence_numpy

    @torch.inference_mode()
    def predict_from_dense(
        self,
        dense_features,
        radiomics,
        geometry_sequence,
        baseline_probability,
    ):
        baseline_probability = np.clip(
            np.asarray(baseline_probability, dtype=np.float64),
            PROBABILITY_EPSILON,
            1.0 - PROBABILITY_EPSILON,
        )
        batch_size = len(baseline_probability)
        if np.asarray(dense_features).shape[0] != batch_size:
            raise RuntimeError("Phase39 dense batch mismatch")
        if np.asarray(radiomics).shape != (batch_size, 1081):
            raise RuntimeError("Phase39 radiomics batch mismatch")
        if np.asarray(geometry_sequence).shape != (
            batch_size,
            16,
            52,
        ):
            raise RuntimeError("Phase39 geometry batch mismatch")

        phase33_fold_logits = []
        phase36_fold_residuals = []
        for fold in range(3):
            summary, p3ca_sequence = self.project_p3ca(
                dense_features,
                fold,
            )
            transformed33 = transform_fold_features(
                self.phase33_preprocessors,
                fold,
                p3ca_sequence,
                geometry_sequence,
                summary,
                radiomics,
            )
            model_arguments = {
                key: torch.from_numpy(value).to(self.device)
                for key, value in transformed33.items()
            }
            seed_logits = []
            for model in self.phase33_models[fold]:
                output = model(
                    p3ca_sequence=model_arguments["p3ca_sequence"],
                    geometry_sequence=model_arguments[
                        "geometry_sequence"
                    ],
                    summary_features=model_arguments["summary_features"],
                    radiomics_features=model_arguments[
                        "radiomics_features"
                    ],
                    domain_coefficient=0.0,
                )
                seed_logits.append(output["logit"].float().cpu().numpy())
            phase33_fold_logits.append(
                np.mean(np.stack(seed_logits, axis=0), axis=0)
            )

            transformed36 = transform_fold_features(
                self.phase36_preprocessors,
                fold,
                p3ca_sequence,
                geometry_sequence,
                summary,
                radiomics,
            )
            phase36_features, baseline_margin = phase36_build_features(
                transformed36,
                baseline_probability,
            )
            seed_residuals = []
            for booster_record in self.phase36_boosters[fold]:
                seed_residuals.append(
                    phase36_predict_residual(
                        booster_record["booster"],
                        phase36_features,
                        baseline_margin,
                        booster_record["round_count"],
                    )
                )
            phase36_fold_residuals.append(
                np.mean(np.stack(seed_residuals, axis=0), axis=0)
            )

        phase33_component_logit = np.mean(
            np.stack(phase33_fold_logits, axis=0),
            axis=0,
        ).astype(np.float64)
        raw_phase36_residual = np.mean(
            np.stack(phase36_fold_residuals, axis=0),
            axis=0,
        ).astype(np.float64)
        baseline_logit = probability_logit(baseline_probability)
        phase33_residual = phase33_component_logit - baseline_logit

        # The Phase36 residual is raw here. The selected 0.75 scale is
        # applied exactly once as the Phase39 phase36_weight.
        combined_residual = (
            float(self.formula["phase36_weight"])
            * raw_phase36_residual
            + float(self.formula["phase33_weight"])
            * phase33_residual
        )
        combined_residual = np.clip(
            combined_residual,
            -float(self.formula["delta_cap"]),
            float(self.formula["delta_cap"]),
        )
        final_logit = baseline_logit + combined_residual
        probability = np.clip(
            probability_sigmoid(final_logit),
            PROBABILITY_EPSILON,
            1.0 - PROBABILITY_EPSILON,
        )
        if not np.isfinite(probability).all():
            raise RuntimeError("Non-finite Phase39 probability")
        return probability, {
            "phase33_component_logit": phase33_component_logit,
            "raw_phase36_residual": raw_phase36_residual,
            "phase33_residual": phase33_residual,
            "combined_residual": combined_residual,
        }

    def predict(self, highres_volumes, baseline_probability, backbone):
        # Both Phase31 and Phase32 training caches stored the exact 128hr
        # crop as float16. Reproduce that boundary before either branch.
        highres = np.asarray(
            highres_volumes,
            dtype=np.float32,
        ).astype(np.float16).astype(np.float32)
        radiomics, geometry = self.extract_geometry(highres)
        dense = self.extract_dense_tokens(highres, backbone)
        return self.predict_from_dense(
            dense,
            radiomics,
            geometry,
            baseline_probability,
        )


def synthetic_highres_batch(batch_size=2):
    coordinates = np.indices((80, 80, 80), dtype=np.float32)
    output = []
    for index in range(int(batch_size)):
        volume = np.full((80, 80, 80), 0.20, dtype=np.float32)
        for center_x in (27.0, 52.0):
            exponent = (
                ((coordinates[0] - center_x) / 5.0) ** 2
                + ((coordinates[1] - (39.0 + index)) / 8.0) ** 2
                + ((coordinates[2] - 38.0) / 7.0) ** 2
            )
            volume += np.exp(-0.5 * exponent).astype(np.float32)
        output.append(np.clip(volume, 0.0, 1.5))
    return np.stack(output).astype(np.float32)
'''


PHASE39_MAIN_SOURCE = r'''
from __future__ import annotations

import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch

import phase30_accepted_main as legacy
from phase39_runtime import Phase39Runtime


DATA_ROOT = Path(
    os.environ.get("DAT_DATA_ROOT", "/code_execution/data")
)
OUTPUT_CSV = Path(
    os.environ.get(
        "DAT_OUTPUT_CSV",
        "/code_execution/submission.csv",
    )
)


def prepare_case(path, crop_config):
    return legacy.base.preprocess_safely(
        path,
        crop_config,
    )


def predict_phase39_safe(
    runtime,
    backbone,
    highres_volumes,
    baseline_probability,
):
    baseline_probability = np.asarray(
        baseline_probability,
        dtype=np.float64,
    )
    probability = baseline_probability.copy()
    failure_count = 0

    try:
        probability[:] = runtime.predict(
            highres_volumes,
            baseline_probability,
            backbone,
        )[0]
    except Exception:
        for index in range(len(baseline_probability)):
            try:
                probability[index] = float(
                    runtime.predict(
                        [highres_volumes[index]],
                        baseline_probability[index:index + 1],
                        backbone,
                    )[0][0]
                )
            except Exception:
                probability[index] = float(
                    baseline_probability[index]
                )
                failure_count += 1

    return probability, failure_count


def main():
    if torch.cuda.is_available():
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_math_sdp(True)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

    runtime_manifest = legacy.load_runtime_manifest()
    legacy.verify_runtime_assets(runtime_manifest)

    (
        base_manifest,
        base_models,
        family_manifest,
        anatomy_models,
        device,
    ) = legacy.load_phase12c_bundle()

    epsilon = float(
        family_manifest["probability_epsilon"]
    )
    fallback = float(
        np.clip(
            family_manifest["fallback_probability"],
            epsilon,
            1.0 - epsilon,
        )
    )

    legacy.DATA_ROOT = DATA_ROOT
    legacy.OUTPUT_CSV = OUTPUT_CSV
    legacy.base.DATA_ROOT = DATA_ROOT
    legacy.base.IMAGE_DIR = DATA_ROOT / "niftis"
    legacy.base.SUBMISSION_FORMAT = (
        DATA_ROOT / "submission_format.csv"
    )
    legacy.base.OUTPUT_CSV = OUTPUT_CSV

    uids, paths = legacy.base.read_submission_contract()
    probabilities = np.full(
        len(uids),
        fallback,
        dtype=np.float64,
    )

    workers = max(
        1,
        min(
            8,
            int(os.environ.get("DAT_PREPROCESS_WORKERS", "6")),
        ),
    )
    batch_size = max(
        1,
        min(
            12,
            int(os.environ.get("DAT_BATCH_SIZE", "6")),
        ),
    )

    phase39_runtime = None
    backbone = None
    initialization_fallbacks = 0
    try:
        phase39_runtime = Phase39Runtime(device)
        backbone = legacy.load_dinov3(
            runtime_manifest,
            device,
        )
    except Exception:
        phase39_runtime = None
        backbone = None
        initialization_fallbacks = 1

    preprocessing_failures = 0
    phase39_fallbacks = 0
    next_progress = 100

    print("Built with DINOv3.", flush=True)
    print("Initialization complete.", flush=True)
    print("Inference started.", flush=True)

    crop_config = base_manifest[
        "preprocessing"
    ]["crops"]

    with ThreadPoolExecutor(max_workers=workers) as executor:
        for start in range(0, len(uids), batch_size):
            stop = min(start + batch_size, len(uids))
            prepared = list(
                executor.map(
                    lambda path: prepare_case(path, crop_config),
                    paths[start:stop],
                )
            )
            valid_local = [
                index
                for index, crops in enumerate(prepared)
                if crops is not None
            ]
            preprocessing_failures += (
                len(prepared) - len(valid_local)
            )

            if valid_local:
                crop_batch = {
                    crop_name: [
                        prepared[index][crop_name]
                        for index in valid_local
                    ]
                    for crop_name in ("160", "192", "128hr")
                }
                full_baseline = legacy.predict_phase12c_safe(
                    crops=crop_batch,
                    selected_family_manifest=family_manifest,
                    selected_base_manifest=base_manifest,
                    base_models=base_models,
                    anatomy_models=anatomy_models,
                    device=device,
                    fallback=fallback,
                )
                final_batch = full_baseline.copy()

                if phase39_runtime is not None and backbone is not None:
                    final_batch, failures = predict_phase39_safe(
                        phase39_runtime,
                        backbone,
                        crop_batch["128hr"],
                        full_baseline,
                    )
                    phase39_fallbacks += int(failures)
                else:
                    phase39_fallbacks += len(valid_local)

                for local_index, probability in zip(
                    valid_local,
                    final_batch,
                ):
                    probabilities[start + local_index] = float(
                        probability
                    )

            while stop >= next_progress:
                print(
                    f"Progress: {next_progress}/{len(uids)}",
                    flush=True,
                )
                next_progress += 100

    values = np.asarray(probabilities, dtype=np.float64)
    if not np.isfinite(values).all():
        raise RuntimeError("Non-finite output probability")
    values = np.clip(values, epsilon, 1.0 - epsilon)
    legacy.base.write_submission(uids, values)

    print(
        "Inference completed: "
        f"cases={len(uids)}, "
        f"preprocessing_fallbacks={preprocessing_failures}, "
        f"phase39_initialization_fallbacks="
        f"{initialization_fallbacks}, "
        f"phase39_case_fallbacks={phase39_fallbacks}.",
        flush=True,
    )


if __name__ == "__main__":
    try:
        main()
    except Exception as error:
        message = str(error).replace("\n", " ")[:240]
        print(
            "Execution failed: "
            f"{type(error).__name__}: {message}",
            flush=True,
        )
        raise SystemExit(1) from None
'''


# Compile before mutating the staged entrypoint.
compile(
    PHASE39_RUNTIME_SOURCE,
    str(PHASE39_RUNTIME_PATH),
    "exec",
)
compile(
    PHASE39_MAIN_SOURCE,
    str(PHASE39_MAIN_PATH),
    "exec",
)

PHASE39_RUNTIME_PATH.write_text(
    PHASE39_RUNTIME_SOURCE,
    encoding="utf-8",
)
PHASE39_MAIN_PATH.write_text(
    PHASE39_MAIN_SOURCE,
    encoding="utf-8",
)

# Static source contracts.
assert "class Phase39Runtime" in PHASE39_RUNTIME_SOURCE
assert "def phase31_case_features" in PHASE39_RUNTIME_SOURCE
assert "def phase36_build_features" in PHASE39_RUNTIME_SOURCE
assert "phase36_weight" in PHASE39_RUNTIME_SOURCE
assert "phase33_weight" in PHASE39_RUNTIME_SOURCE
assert "delta_cap" in PHASE39_RUNTIME_SOURCE
assert "astype(np.float16).astype(np.float32)" in (
    PHASE39_RUNTIME_SOURCE
)
assert "import phase30_accepted_main as legacy" in (
    PHASE39_MAIN_SOURCE
)

# Import the exact staged runtime and load every portable deployment state.
stage_string = str(PHASE39_STAGING_DIRECTORY)
if stage_string not in sys.path:
    sys.path.insert(0, stage_string)

for module_name in [
    "phase39_runtime",
    "phase30_accepted_main",
]:
    sys.modules.pop(module_name, None)

runtime_module = importlib.import_module(
    "phase39_runtime"
)
runtime_module.verify_phase39_assets()

for required_legacy_helper in [
    "load_runtime_manifest",
    "verify_runtime_assets",
    "load_phase12c_bundle",
    "predict_phase12c_safe",
    "load_dinov3",
    "phase30_make_bilateral_views",
]:
    assert callable(
        getattr(
            runtime_module.legacy,
            required_legacy_helper,
            None,
        )
    ), required_legacy_helper

contract_device = torch.device("cpu")
runtime = runtime_module.Phase39Runtime(
    contract_device
)

assert len(runtime.phase33_models) == 3
assert all(
    len(models) == 3
    for models in runtime.phase33_models
)
assert len(runtime.phase36_boosters) == 3
assert all(
    len(boosters) == 2
    for boosters in runtime.phase36_boosters
)

parameter_count = sum(
    parameter.numel()
    for parameter in runtime.phase33_models[0][0].parameters()
)
assert parameter_count == 130355, parameter_count

# Exact Phase31 synthetic geometry and reflection contract.
synthetic_highres = runtime_module.synthetic_highres_batch(2)
synthetic_highres = (
    synthetic_highres
    .astype(np.float16)
    .astype(np.float32)
)

feature_a, _, sequence_a, _, _ = (
    runtime_module.phase31_case_features(
        synthetic_highres[0]
    )
)
feature_b, _, sequence_b, _, _ = (
    runtime_module.phase31_case_features(
        np.flip(
            synthetic_highres[0],
            axis=0,
        ).copy()
    )
)

reflection_feature_error = float(
    np.max(np.abs(feature_a - feature_b))
)
reflection_sequence_error = float(
    np.max(np.abs(sequence_a - sequence_b))
)

assert feature_a.shape == (1081,)
assert sequence_a.shape == (16, 52)
assert reflection_feature_error <= 5e-5, (
    reflection_feature_error
)
assert reflection_sequence_error <= 5e-5, (
    reflection_sequence_error
)

# Synthetic end-to-end Phase33 + Phase36 contract. Dense values are
# synthetic DINO-like tensors; the actual DINO checkpoint is loaded in the
# next offline initialization/parity cell.
rng = np.random.default_rng(390126)
synthetic_dense = rng.normal(
    loc=0.0,
    scale=0.35,
    size=(2, 6, 14, 14, 384),
).astype(np.float32)
synthetic_dense = (
    synthetic_dense
    .astype(np.float16)
    .astype(np.float32)
)

synthetic_radiomics, synthetic_geometry = (
    runtime.extract_geometry(synthetic_highres)
)
synthetic_baseline = np.asarray(
    [0.35, 0.65],
    dtype=np.float64,
)

synthetic_probability, synthetic_details = (
    runtime.predict_from_dense(
        synthetic_dense,
        synthetic_radiomics,
        synthetic_geometry,
        synthetic_baseline,
    )
)

assert synthetic_probability.shape == (2,)
assert np.isfinite(synthetic_probability).all()
assert np.all(synthetic_probability >= 1e-5)
assert np.all(synthetic_probability <= 1.0 - 1e-5)
assert synthetic_details[
    "raw_phase36_residual"
].shape == (2,)
assert synthetic_details[
    "phase33_residual"
].shape == (2,)
assert synthetic_details[
    "combined_residual"
].shape == (2,)
assert np.max(
    np.abs(
        synthetic_details["combined_residual"]
    )
) <= 2.0 + 1e-12

source_manifest = {
    "schema_version": 1,
    "phase": "phase39_offline_runtime_source",
    "files": {
        "main.py": phase39_source_sha256(
            PHASE39_MAIN_PATH
        ),
        "phase39_runtime.py": phase39_source_sha256(
            PHASE39_RUNTIME_PATH
        ),
        "phase30_accepted_main.py": phase39_source_sha256(
            PHASE39_STAGING_DIRECTORY
            / "phase30_accepted_main.py"
        ),
    },
    "phase39_formula": {
        "phase36_weight": 0.75,
        "phase33_weight": 0.25,
        "alpha": 1.0,
        "uncertainty_exponent": 0.0,
        "delta_cap": 2.0,
    },
    "contains_labels": False,
    "contains_oof_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
}

PHASE39_SOURCE_MANIFEST_PATH.write_text(
    json.dumps(source_manifest, indent=2),
    encoding="utf-8",
)

package_files = sorted(
    path
    for path in PHASE39_STAGING_DIRECTORY.rglob("*")
    if path.is_file()
)

report = {
    "phase": "phase39_offline_runtime_source_contract",
    "status": "accepted",
    "main_source_line_count": len(
        PHASE39_MAIN_SOURCE.splitlines()
    ),
    "runtime_source_line_count": len(
        PHASE39_RUNTIME_SOURCE.splitlines()
    ),
    "package_file_count": len(package_files),
    "package_size_mb": round(
        sum(path.stat().st_size for path in package_files)
        / 1e6,
        3,
    ),
    "phase33_model_count": 9,
    "phase33_parameter_count_per_model": parameter_count,
    "phase36_booster_count": 6,
    "xgboost_version": str(xgb.__version__),
    "synthetic": {
        "radiomics_shape": list(
            synthetic_radiomics.shape
        ),
        "geometry_shape": list(
            synthetic_geometry.shape
        ),
        "dense_shape": list(synthetic_dense.shape),
        "probability_shape": list(
            synthetic_probability.shape
        ),
        "all_probabilities_finite": bool(
            np.isfinite(synthetic_probability).all()
        ),
        "maximum_absolute_combined_residual": round(
            float(
                np.max(
                    np.abs(
                        synthetic_details[
                            "combined_residual"
                        ]
                    )
                )
            ),
            8,
        ),
        "reflection_feature_maximum_error": round(
            reflection_feature_error,
            8,
        ),
        "reflection_sequence_maximum_error": round(
            reflection_sequence_error,
            8,
        ),
    },
    "phase36_raw_residual_scaled_exactly_once": True,
    "phase12c_full_ensemble_anchor_present": True,
    "float16_highres_cache_contract_present": True,
    "float16_dense_cache_contract_present": True,
    "float16_projected_cache_contract_present": True,
    "new_runtime_entrypoint_written": True,
    "accepted_phase30_main_preserved": True,
    "dinov3_checkpoint_loaded": False,
    "challenge_voxel_data_read": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase39_runtime_started,
        2,
    ),
}

print("BEGIN SANITIZED_PHASE39_RUNTIME_SOURCE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE39_RUNTIME_SOURCE")
print(PHASE39_MAIN_PATH)

BEGIN SANITIZED_PHASE39_RUNTIME_SOURCE
{
  "phase": "phase39_offline_runtime_source_contract",
  "status": "accepted",
  "main_source_line_count": 251,
  "runtime_source_line_count": 1527,
  "package_file_count": 89,
  "package_size_mb": 202.781,
  "phase33_model_count": 9,
  "phase33_parameter_count_per_model": 130355,
  "phase36_booster_count": 6,
  "xgboost_version": "3.2.0",
  "synthetic": {
    "radiomics_shape": [
      2,
      1081
    ],
    "geometry_shape": [
      2,
      16,
      52
    ],
    "dense_shape": [
      2,
      6,
      14,
      14,
      384
    ],
    "probability_shape": [
      2
    ],
    "all_probabilities_finite": true,
    "maximum_absolute_combined_residual": 0.86099322,
    "reflection_feature_maximum_error": 0.0,
    "reflection_sequence_maximum_error": 0.0
  },
  "phase36_raw_residual_scaled_exactly_once": true,
  "phase12c_full_ensemble_anchor_present": true,
  "float16_highres_cache_contract_present": true,
  "float16_dense_cache_contract_pr

In [98]:
# Phase39 Cell 127
# Complete offline initialization and synthetic end-to-end runtime contract.
#
# Prerequisite: accepted SANITIZED_PHASE39_RUNTIME_SOURCE from Cell 126.
# This cell does not read training, challenge, smoke, or test images.

from pathlib import Path
import gc
import importlib
import json
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F


PHASE39_STAGING_DIRECTORY = Path(
    "/kaggle/working/phase39_experimental_submission"
)
PHASE39_OFFLINE_CONTRACT_PATH = (
    PHASE39_STAGING_DIRECTORY
    / "phase39_offline_initialization_contract.json"
)

phase39_initialization_started = time.perf_counter()

assert PHASE39_STAGING_DIRECTORY.is_dir()
assert (
    PHASE39_STAGING_DIRECTORY / "main.py"
).is_file()
assert (
    PHASE39_STAGING_DIRECTORY / "phase39_runtime.py"
).is_file()
assert (
    PHASE39_STAGING_DIRECTORY
    / "phase30_accepted_main.py"
).is_file()


# Release obsolete notebook model objects while preserving all persistent
# feature caches and private deployment checkpoints needed by later parity.
for variable_name in [
    "runtime",
    "runtime_module",
    "PHASE39_RUNTIME_PRIVATE",
    "PHASE39_BACKBONE_PRIVATE",
    "PHASE39_BASE_MODELS_PRIVATE",
    "PHASE39_ANATOMY_MODELS_PRIVATE",
    "PHASE32_DINOV3_BACKBONE",
    "PHASE32_PHASE12_BASE_MODELS",
    "PHASE32_PHASE12_ANATOMY_MODELS",
    "PHASE30_DINOV3_BACKBONE",
]:
    globals().pop(variable_name, None)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

torch.use_deterministic_algorithms(
    True,
    warn_only=True,
)


stage_string = str(PHASE39_STAGING_DIRECTORY)
if stage_string not in sys.path:
    sys.path.insert(0, stage_string)

importlib.invalidate_caches()
phase39_runtime_module = importlib.import_module(
    "phase39_runtime"
)
phase39_runtime_module = importlib.reload(
    phase39_runtime_module
)
legacy = phase39_runtime_module.legacy


# Verify the inherited accepted Phase30 assets and every new Phase39 asset.
phase30_runtime_manifest = legacy.load_runtime_manifest()
legacy.verify_runtime_assets(
    phase30_runtime_manifest
)
phase39_runtime_module.verify_phase39_assets()


# Load the complete 21-model Phase12c anchor.
(
    phase39_base_manifest,
    phase39_base_models,
    phase39_family_manifest,
    phase39_anatomy_models,
    PHASE39_DEVICE,
) = legacy.load_phase12c_bundle()

assert len(phase39_base_models) == 15
assert len(phase39_anatomy_models) == 6
assert len(phase39_base_manifest["folds"]) == 3
assert len(phase39_family_manifest["folds"]) == 3


# Load the new residual deployment states and DINOv3 entirely offline.
PHASE39_RUNTIME_PRIVATE = (
    phase39_runtime_module.Phase39Runtime(
        PHASE39_DEVICE
    )
)
PHASE39_BACKBONE_PRIVATE = legacy.load_dinov3(
    phase30_runtime_manifest,
    PHASE39_DEVICE,
)

dinov3_parameter_count = sum(
    parameter.numel()
    for parameter in PHASE39_BACKBONE_PRIVATE.parameters()
)
dinov3_frozen = all(
    not parameter.requires_grad
    for parameter in PHASE39_BACKBONE_PRIVATE.parameters()
)

assert dinov3_parameter_count == 21601152
assert dinov3_frozen
assert not PHASE39_BACKBONE_PRIVATE.training


# Build label-free synthetic crops for all three exact Phase12c branches.
synthetic_highres = (
    phase39_runtime_module.synthetic_highres_batch(2)
)

highres_tensor = torch.from_numpy(
    synthetic_highres[:, None]
)
synthetic_64 = F.interpolate(
    highres_tensor,
    size=(64, 64, 64),
    mode="trilinear",
    align_corners=False,
).squeeze(1).numpy().astype(np.float32)

synthetic_crops = {
    "128hr": [
        synthetic_highres[index].copy()
        for index in range(2)
    ],
    "160": [
        synthetic_64[index].copy()
        for index in range(2)
    ],
    "192": [
        np.clip(
            0.92 * synthetic_64[index],
            0.0,
            1.5,
        ).astype(np.float32)
        for index in range(2)
    ],
}

synthetic_anchor_probability = (
    legacy.predict_phase12c_safe(
        crops=synthetic_crops,
        selected_family_manifest=(
            phase39_family_manifest
        ),
        selected_base_manifest=(
            phase39_base_manifest
        ),
        base_models=phase39_base_models,
        anatomy_models=phase39_anatomy_models,
        device=PHASE39_DEVICE,
        fallback=float(
            phase39_family_manifest[
                "fallback_probability"
            ]
        ),
    )
)

synthetic_anchor_direct = (
    legacy.family.predict_family_batch(
        synthetic_crops,
        phase39_family_manifest,
        phase39_base_manifest,
        phase39_base_models,
        phase39_anatomy_models,
        PHASE39_DEVICE,
    )
)

phase12c_safe_wrapper_error = float(
    np.max(
        np.abs(
            synthetic_anchor_probability
            - synthetic_anchor_direct
        )
    )
)

assert phase12c_safe_wrapper_error <= 1e-12, (
    phase12c_safe_wrapper_error
)

assert synthetic_anchor_probability.shape == (2,)
assert np.isfinite(
    synthetic_anchor_probability
).all()
assert np.all(synthetic_anchor_probability >= 1e-5)
assert np.all(
    synthetic_anchor_probability <= 1.0 - 1e-5
)


# Complete end-to-end path: high-resolution crop -> clinical views ->
# DINOv3 dense tokens -> P3CA -> Phase33/36 -> Phase39 probability.
(
    synthetic_final_probability,
    synthetic_end_to_end_details,
) = PHASE39_RUNTIME_PRIVATE.predict(
    synthetic_highres,
    synthetic_anchor_probability,
    PHASE39_BACKBONE_PRIVATE,
)

assert synthetic_final_probability.shape == (2,)
assert np.isfinite(
    synthetic_final_probability
).all()
assert np.all(synthetic_final_probability >= 1e-5)
assert np.all(
    synthetic_final_probability <= 1.0 - 1e-5
)


# Independently compose the same result from the explicit intermediate
# tensors. This catches cache-quantization or residual-scaling drift.
quantized_highres = (
    synthetic_highres
    .astype(np.float16)
    .astype(np.float32)
)
synthetic_radiomics, synthetic_geometry = (
    PHASE39_RUNTIME_PRIVATE.extract_geometry(
        quantized_highres
    )
)
synthetic_dense = (
    PHASE39_RUNTIME_PRIVATE.extract_dense_tokens(
        quantized_highres,
        PHASE39_BACKBONE_PRIVATE,
    )
)
(
    synthetic_composed_probability,
    synthetic_composed_details,
) = PHASE39_RUNTIME_PRIVATE.predict_from_dense(
    synthetic_dense,
    synthetic_radiomics,
    synthetic_geometry,
    synthetic_anchor_probability,
)

end_to_end_composition_error = float(
    np.max(
        np.abs(
            synthetic_final_probability
            - synthetic_composed_probability
        )
    )
)

assert end_to_end_composition_error <= 2e-7, (
    end_to_end_composition_error
)


# Verify the persisted Phase39 formula from its explicit residuals.
baseline_logit = (
    phase39_runtime_module.probability_logit(
        synthetic_anchor_probability
    )
)
reconstructed_combined_residual = np.clip(
    0.75
    * synthetic_composed_details[
        "raw_phase36_residual"
    ]
    + 0.25
    * synthetic_composed_details[
        "phase33_residual"
    ],
    -2.0,
    2.0,
)
reconstructed_probability = np.clip(
    phase39_runtime_module.probability_sigmoid(
        baseline_logit
        + reconstructed_combined_residual
    ),
    1e-5,
    1.0 - 1e-5,
)

formula_reconstruction_error = float(
    np.max(
        np.abs(
            synthetic_final_probability
            - reconstructed_probability
        )
    )
)

assert formula_reconstruction_error <= 2e-7, (
    formula_reconstruction_error
)
assert np.max(
    np.abs(
        synthetic_end_to_end_details[
            "combined_residual"
        ]
    )
) <= 2.0 + 1e-12


if PHASE39_DEVICE.type == "cuda":
    peak_vram_mb = float(
        torch.cuda.max_memory_allocated(
            PHASE39_DEVICE
        )
        / (1024 ** 2)
    )
else:
    peak_vram_mb = 0.0


report = {
    "phase": "phase39_complete_offline_initialization",
    "status": "accepted",
    "device": str(PHASE39_DEVICE),
    "phase12c": {
        "base_model_count": len(
            phase39_base_models
        ),
        "anatomy_model_count": len(
            phase39_anatomy_models
        ),
        "total_model_count": (
            len(phase39_base_models)
            + len(phase39_anatomy_models)
        ),
        "synthetic_probability_shape": list(
            synthetic_anchor_probability.shape
        ),
        "all_probabilities_finite": bool(
            np.isfinite(
                synthetic_anchor_probability
            ).all()
        ),
        "safe_wrapper_maximum_error": round(
            phase12c_safe_wrapper_error,
            12,
        ),
    },
    "dinov3": {
        "parameter_count": (
            dinov3_parameter_count
        ),
        "backbone_frozen": dinov3_frozen,
        "dense_feature_shape": list(
            synthetic_dense.shape
        ),
        "checkpoint_loaded_offline": True,
    },
    "phase33_model_count": sum(
        len(models)
        for models in (
            PHASE39_RUNTIME_PRIVATE.phase33_models
        )
    ),
    "phase36_booster_count": sum(
        len(boosters)
        for boosters in (
            PHASE39_RUNTIME_PRIVATE.phase36_boosters
        )
    ),
    "synthetic_phase39": {
        "radiomics_shape": list(
            synthetic_radiomics.shape
        ),
        "geometry_shape": list(
            synthetic_geometry.shape
        ),
        "probability_shape": list(
            synthetic_final_probability.shape
        ),
        "all_probabilities_finite": bool(
            np.isfinite(
                synthetic_final_probability
            ).all()
        ),
        "all_probabilities_in_contract": bool(
            np.all(
                synthetic_final_probability >= 1e-5
            )
            and np.all(
                synthetic_final_probability
                <= 1.0 - 1e-5
            )
        ),
        "maximum_absolute_combined_residual": round(
            float(
                np.max(
                    np.abs(
                        synthetic_composed_details[
                            "combined_residual"
                        ]
                    )
                )
            ),
            8,
        ),
        "end_to_end_composition_error": round(
            end_to_end_composition_error,
            10,
        ),
        "formula_reconstruction_error": round(
            formula_reconstruction_error,
            10,
        ),
    },
    "peak_vram_mb": round(peak_vram_mb, 2),
    "phase36_raw_residual_scaled_exactly_once": True,
    "phase12c_full_ensemble_anchor_used": True,
    "asset_hash_verification_passed": True,
    "deployment_objects_retained_privately": True,
    "challenge_voxel_data_read": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase39_initialization_started,
        2,
    ),
}

PHASE39_OFFLINE_CONTRACT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)


# Retain the complete initialized objects for the next component-parity cell.
PHASE39_BASE_MODELS_PRIVATE = phase39_base_models
PHASE39_ANATOMY_MODELS_PRIVATE = phase39_anatomy_models
PHASE39_BASE_MANIFEST_PRIVATE = phase39_base_manifest
PHASE39_FAMILY_MANIFEST_PRIVATE = phase39_family_manifest


print("BEGIN SANITIZED_PHASE39_OFFLINE_INITIALIZATION")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE39_OFFLINE_INITIALIZATION")

/kaggle/working/phase30_submission/phase12c_main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


BEGIN SANITIZED_PHASE39_OFFLINE_INITIALIZATION
{
  "phase": "phase39_complete_offline_initialization",
  "status": "accepted",
  "device": "cuda:0",
  "phase12c": {
    "base_model_count": 15,
    "anatomy_model_count": 6,
    "total_model_count": 21,
    "synthetic_probability_shape": [
      2
    ],
    "all_probabilities_finite": true,
    "safe_wrapper_maximum_error": 0.0
  },
  "dinov3": {
    "parameter_count": 21601152,
    "backbone_frozen": true,
    "dense_feature_shape": [
      2,
      6,
      14,
      14,
      384
    ],
    "checkpoint_loaded_offline": true
  },
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "synthetic_phase39": {
    "radiomics_shape": [
      2,
      1081
    ],
    "geometry_shape": [
      2,
      16,
      52
    ],
    "probability_shape": [
      2
    ],
    "all_probabilities_finite": true,
    "all_probabilities_in_contract": true,
    "maximum_absolute_combined_residual": 1.4327301,
    "end_to_end_composition_error": 0.0,
 

In [100]:
# Phase39 Cell 128
# Exact leakage-free packaged-component and complete OOF parity.

from pathlib import Path
import gc
import json
import time

import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score


PHASE39_CASE_COUNT = 1362
PHASE39_PARITY_BATCH_SIZE = 48
PHASE39_PARITY_DIRECTORY = Path(
    "/kaggle/working/phase39_experimental_submission"
)
PHASE39_PARITY_CONTRACT_PATH = (
    PHASE39_PARITY_DIRECTORY
    / "phase39_complete_oof_parity_contract.json"
)

phase39_parity_started = time.perf_counter()


# ------------------------------------------------------------------
# Preconditions retained by Cell 127.
# ------------------------------------------------------------------

required_runtime_names = [
    "PHASE39_RUNTIME_PRIVATE",
    "PHASE39_BACKBONE_PRIVATE",
    "PHASE39_BASE_MODELS_PRIVATE",
    "PHASE39_ANATOMY_MODELS_PRIVATE",
    "PHASE39_BASE_MANIFEST_PRIVATE",
    "PHASE39_FAMILY_MANIFEST_PRIVATE",
    "PHASE39_DEVICE",
    "phase39_runtime_module",
]
missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]
assert not missing_runtime_names, {
    "message": (
        "Cell 128 requires the accepted Cell 127 runtime objects."
    ),
    "missing": missing_runtime_names,
}

assert PHASE39_PARITY_DIRECTORY.is_dir()
assert PHASE39_RUNTIME_PRIVATE.formula == {
    "phase36_weight": 0.75,
    "phase33_weight": 0.25,
    "alpha": 1.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 2.0,
}
assert len(PHASE39_RUNTIME_PRIVATE.phase33_models) == 3
assert all(
    len(models) == 3
    for models in PHASE39_RUNTIME_PRIVATE.phase33_models
)
assert len(PHASE39_RUNTIME_PRIVATE.phase36_boosters) == 3
assert all(
    len(boosters) == 2
    for boosters in PHASE39_RUNTIME_PRIVATE.phase36_boosters
)

runtime_module = phase39_runtime_module
runtime = PHASE39_RUNTIME_PRIVATE

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(
        PHASE39_DEVICE
    )


# ------------------------------------------------------------------
# Strict private-cache resolution.
# ------------------------------------------------------------------

cache_paths = {
    "highres": Path(
        "/kaggle/working/phase31_highres_float16.npy"
    ),
    "radiomics": Path(
        "/kaggle/working/phase31_radiomics_float32.npy"
    ),
    "geometry": Path(
        "/kaggle/working/phase31_sequence_float32.npy"
    ),
    "dense": Path(
        "/kaggle/working/phase32_bilateral_dense_float16.npy"
    ),
    "p3ca_summary": Path(
        "/kaggle/working/phase32_p3ca_summary_float16.npy"
    ),
    "p3ca_sequence": Path(
        "/kaggle/working/phase32_p3ca_sequence_float16.npy"
    ),
    "baseline_oof": Path(
        "/kaggle/working/phase32_phase12c_oof_float64.npy"
    ),
}

missing_cache_paths = [
    str(path)
    for path in cache_paths.values()
    if not path.is_file()
]
assert not missing_cache_paths, {
    "message": "A required private parity cache is missing.",
    "missing": missing_cache_paths,
}

highres_cache = np.load(
    cache_paths["highres"],
    mmap_mode="r",
    allow_pickle=False,
)
radiomics_cache = np.load(
    cache_paths["radiomics"],
    mmap_mode="r",
    allow_pickle=False,
)
geometry_cache = np.load(
    cache_paths["geometry"],
    mmap_mode="r",
    allow_pickle=False,
)
dense_cache = np.load(
    cache_paths["dense"],
    mmap_mode="r",
    allow_pickle=False,
)
p3ca_summary_cache = np.load(
    cache_paths["p3ca_summary"],
    mmap_mode="r",
    allow_pickle=False,
)
p3ca_sequence_cache = np.load(
    cache_paths["p3ca_sequence"],
    mmap_mode="r",
    allow_pickle=False,
)
baseline_oof = np.asarray(
    np.load(
        cache_paths["baseline_oof"],
        mmap_mode="r",
        allow_pickle=False,
    ),
    dtype=np.float64,
)

assert highres_cache.shape == (
    PHASE39_CASE_COUNT,
    80,
    80,
    80,
)
assert highres_cache.dtype == np.float16
assert radiomics_cache.shape == (
    PHASE39_CASE_COUNT,
    1081,
)
assert radiomics_cache.dtype == np.float32
assert geometry_cache.shape == (
    PHASE39_CASE_COUNT,
    16,
    52,
)
assert geometry_cache.dtype == np.float32
assert dense_cache.shape == (
    PHASE39_CASE_COUNT,
    6,
    14,
    14,
    384,
)
assert dense_cache.dtype == np.float16
assert p3ca_summary_cache.shape == (
    2,
    3,
    PHASE39_CASE_COUNT,
    768,
)
assert p3ca_summary_cache.dtype == np.float16
assert p3ca_sequence_cache.shape == (
    2,
    3,
    PHASE39_CASE_COUNT,
    84,
    48,
)
assert p3ca_sequence_cache.dtype == np.float16
assert baseline_oof.shape == (PHASE39_CASE_COUNT,)
assert np.isfinite(baseline_oof).all()
assert np.all(baseline_oof >= 1e-5)
assert np.all(baseline_oof <= 1.0 - 1e-5)


def phase39_resolve_partitions():
    candidate_names = [
        "PHASE19_PARTITIONS",
        "PHASE31_PARTITIONS",
        "phase19_partitions",
        "phase31_partitions",
    ]

    for name in candidate_names:
        candidate = globals().get(name)
        if not isinstance(candidate, (list, tuple)):
            continue
        if len(candidate) != 3:
            continue

        resolved = []
        valid = True
        for expected_fold, record in enumerate(candidate):
            if not isinstance(record, dict):
                valid = False
                break
            valid_key = next(
                (
                    key
                    for key in (
                        "outer_valid",
                        "outer_valid_indices",
                        "valid",
                    )
                    if key in record
                ),
                None,
            )
            train_key = next(
                (
                    key
                    for key in (
                        "outer_train",
                        "outer_train_indices",
                        "train",
                    )
                    if key in record
                ),
                None,
            )
            if valid_key is None or train_key is None:
                valid = False
                break

            outer_valid = np.asarray(
                record[valid_key],
                dtype=np.int64,
            )
            outer_train = np.asarray(
                record[train_key],
                dtype=np.int64,
            )
            if outer_valid.ndim != 1 or outer_train.ndim != 1:
                valid = False
                break
            if not (
                np.all(outer_valid >= 0)
                and np.all(outer_valid < PHASE39_CASE_COUNT)
                and np.all(outer_train >= 0)
                and np.all(outer_train < PHASE39_CASE_COUNT)
            ):
                valid = False
                break
            if len(np.unique(outer_valid)) != len(outer_valid):
                valid = False
                break
            if len(np.intersect1d(outer_train, outer_valid)):
                valid = False
                break

            resolved.append({
                "fold": int(record.get("fold", expected_fold)),
                "outer_train": outer_train,
                "outer_valid": outer_valid,
            })

        if not valid:
            continue
        if [record["fold"] for record in resolved] != [0, 1, 2]:
            continue

        coverage = np.zeros(
            PHASE39_CASE_COUNT,
            dtype=np.int64,
        )
        for record in resolved:
            coverage[record["outer_valid"]] += 1
        if np.all(coverage == 1):
            return resolved, name

    raise AssertionError({
        "message": "Could not resolve exact three-fold partitions.",
        "candidate_names": candidate_names,
    })


def phase39_binary_vector(value):
    try:
        array = np.asarray(value)
    except Exception:
        return None
    if array.shape != (PHASE39_CASE_COUNT,):
        return None
    if not np.isfinite(array).all():
        return None
    unique = np.unique(array)
    if not set(unique.tolist()).issubset({0, 1, 0.0, 1.0}):
        return None
    return array.astype(np.int64)


def phase39_resolve_labels():
    for name in (
        "PHASE19_LABELS",
        "PHASE31_LABELS",
        "label",
        "labels",
        "y",
    ):
        result = phase39_binary_vector(
            globals().get(name)
        )
        if result is not None:
            return result, name

    dataframe = globals().get("case_df")
    if dataframe is not None:
        for column in (
            "label",
            "y",
            "is_pathologic",
        ):
            if hasattr(dataframe, "columns") and column in dataframe.columns:
                result = phase39_binary_vector(
                    dataframe[column].to_numpy()
                )
                if result is not None:
                    return result, f"case_df.{column}"

    raise AssertionError(
        "Could not resolve training labels for final parity metrics."
    )


partitions, partition_source = (
    phase39_resolve_partitions()
)
labels, label_source = phase39_resolve_labels()


def phase39_probability_vector(value):
    try:
        array = np.asarray(value, dtype=np.float64)
    except Exception:
        return None
    if array.shape != (PHASE39_CASE_COUNT,):
        return None
    if not np.isfinite(array).all():
        return None
    if np.any(array < 0.0) or np.any(array > 1.0):
        return None
    return np.clip(array, 1e-5, 1.0 - 1e-5)


def phase39_finite_vector(value):
    try:
        array = np.asarray(value, dtype=np.float64)
    except Exception:
        return None
    if array.shape != (PHASE39_CASE_COUNT,):
        return None
    if not np.isfinite(array).all():
        return None
    return array


def phase39_load_probability_reference(
    global_names,
    file_paths,
    description,
):
    for name in global_names:
        result = phase39_probability_vector(
            globals().get(name)
        )
        if result is not None:
            return result, name
    for path in file_paths:
        if not path.is_file():
            continue
        result = phase39_probability_vector(
            np.load(
                path,
                mmap_mode="r",
                allow_pickle=False,
            )
        )
        if result is not None:
            return result, str(path)
    raise AssertionError(
        f"Could not resolve {description}."
    )


def phase39_load_finite_reference(
    global_names,
    file_paths,
):
    for name in global_names:
        result = phase39_finite_vector(
            globals().get(name)
        )
        if result is not None:
            return result, name
    for path in file_paths:
        if not path.is_file():
            continue
        result = phase39_finite_vector(
            np.load(
                path,
                mmap_mode="r",
                allow_pickle=False,
            )
        )
        if result is not None:
            return result, str(path)
    return None, None


phase33_reference, phase33_reference_source = (
    phase39_load_probability_reference(
        global_names=[
            "PHASE33_OOF_PRIVATE",
            "PHASE33_COMPONENT_OOF_PRIVATE",
        ],
        file_paths=[
            Path(
                "/kaggle/working/phase33_private_checkpoint/"
                "phase33_component_oof_float64.npy"
            ),
            Path(
                "/kaggle/working/phase33_component_oof_float64.npy"
            ),
        ],
        description="Phase33 component OOF probability",
    )
)

phase39_reference, phase39_reference_source = (
    phase39_load_probability_reference(
        global_names=[
            "PHASE39_OOF_PRIVATE",
        ],
        file_paths=[
            Path(
                "/kaggle/working/phase39_private_checkpoint/"
                "phase39_oof_float64.npy"
            ),
            Path(
                "/kaggle/working/phase39_oof_float64.npy"
            ),
        ],
        description="Phase39 OOF probability",
    )
)

phase39_combined_reference, phase39_combined_reference_source = (
    phase39_load_finite_reference(
        global_names=[
            "PHASE39_COMBINED_RESIDUAL_PRIVATE",
            "PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE",
        ],
        file_paths=[
            Path(
                "/kaggle/working/phase39_private_checkpoint/"
                "phase39_combined_residual_float64.npy"
            ),
            Path(
                "/kaggle/working/phase39_combined_residual_float64.npy"
            ),
        ],
    )
)

phase36_raw_reference, phase36_raw_reference_source = (
    phase39_load_finite_reference(
        global_names=[
            "PHASE36_RAW_RESIDUAL_OOF_PRIVATE",
        ],
        file_paths=[],
    )
)

phase36_reference, phase36_reference_source = (
    phase39_load_finite_reference(
        global_names=[
            "PHASE36_OOF_PRIVATE",
        ],
        file_paths=[],
    )
)
if phase36_reference is not None:
    phase36_reference = phase39_probability_vector(
        phase36_reference
    )
    if phase36_reference is None:
        phase36_reference_source = None

baseline_logit = runtime_module.probability_logit(
    baseline_oof
)

# Phase36 persistence deliberately excluded OOF predictions. In a restarted
# kernel, reconstruct the raw reference from the retained Phase36 probability
# if available. This inverse applies the historical selected scale exactly
# once and is used only as a parity reference.
if phase36_raw_reference is None and phase36_reference is not None:
    phase36_raw_reference = (
        runtime_module.probability_logit(
            phase36_reference
        )
        - baseline_logit
    ) / 0.75
    phase36_raw_reference_source = (
        f"derived_from_{phase36_reference_source}"
    )

assert phase36_raw_reference is not None, {
    "message": (
        "Phase36 raw residual OOF is unavailable. Rerun the accepted "
        "Phase36 reconstruction/persistence cell so either "
        "PHASE36_RAW_RESIDUAL_OOF_PRIVATE or PHASE36_OOF_PRIVATE "
        "exists; no outer labels are needed."
    )
}

if phase36_reference is None:
    phase36_reference = np.clip(
        runtime_module.probability_sigmoid(
            baseline_logit
            + 0.75 * phase36_raw_reference
        ),
        1e-5,
        1.0 - 1e-5,
    )
    phase36_reference_source = (
        f"derived_from_{phase36_raw_reference_source}"
    )


# ------------------------------------------------------------------
# Runtime feature extraction audits on a deterministic, label-free sample.
# ------------------------------------------------------------------

audit_indices = np.unique(
    np.concatenate([
        np.linspace(
            0,
            PHASE39_CASE_COUNT - 1,
            12,
            dtype=np.int64,
        ),
        np.asarray(
            [
                record["outer_valid"][0]
                for record in partitions
            ],
            dtype=np.int64,
        ),
    ])
)

audit_highres = np.asarray(
    highres_cache[audit_indices],
    dtype=np.float32,
)
audit_runtime_radiomics, audit_runtime_geometry = (
    runtime.extract_geometry(audit_highres)
)
audit_reference_radiomics = np.asarray(
    radiomics_cache[audit_indices],
    dtype=np.float32,
)
audit_reference_geometry = np.asarray(
    geometry_cache[audit_indices],
    dtype=np.float32,
)

radiomics_feature_difference = np.abs(
    audit_runtime_radiomics
    - audit_reference_radiomics
)
geometry_feature_difference = np.abs(
    audit_runtime_geometry
    - audit_reference_geometry
)

radiomics_feature_maximum_error = float(
    np.max(radiomics_feature_difference)
)
radiomics_feature_mean_error = float(
    np.mean(radiomics_feature_difference)
)
geometry_feature_maximum_error = float(
    np.max(geometry_feature_difference)
)
geometry_feature_mean_error = float(
    np.mean(geometry_feature_difference)
)

assert radiomics_feature_maximum_error <= 1e-5, {
    "radiomics_feature_maximum_error": (
        radiomics_feature_maximum_error
    )
}
assert geometry_feature_maximum_error <= 1e-5, {
    "geometry_feature_maximum_error": (
        geometry_feature_maximum_error
    )
}

del (
    audit_highres,
    audit_runtime_radiomics,
    audit_runtime_geometry,
    audit_reference_radiomics,
    audit_reference_geometry,
    radiomics_feature_difference,
    geometry_feature_difference,
)
gc.collect()


# ------------------------------------------------------------------
# Exact held-out-fold component reconstruction.
# ------------------------------------------------------------------

phase33_runtime_logit = np.full(
    PHASE39_CASE_COUNT,
    np.nan,
    dtype=np.float64,
)
phase36_runtime_raw_residual = np.full(
    PHASE39_CASE_COUNT,
    np.nan,
    dtype=np.float64,
)

p3ca_summary_maximum_error = 0.0
p3ca_sequence_maximum_error = 0.0
completed_case_count = 0
next_progress = 100

for partition in partitions:
    fold = int(partition["fold"])
    outer_valid_indices = np.asarray(
        partition["outer_valid"],
        dtype=np.int64,
    )

    for start in range(
        0,
        len(outer_valid_indices),
        PHASE39_PARITY_BATCH_SIZE,
    ):
        stop = min(
            start + PHASE39_PARITY_BATCH_SIZE,
            len(outer_valid_indices),
        )
        indices = outer_valid_indices[start:stop]

        dense = np.asarray(
            dense_cache[indices],
            dtype=np.float32,
        )
        radiomics = np.asarray(
            radiomics_cache[indices],
            dtype=np.float32,
        )
        geometry = np.asarray(
            geometry_cache[indices],
            dtype=np.float32,
        )

        p3ca_summary, p3ca_sequence = (
            runtime.project_p3ca(
                dense,
                fold,
            )
        )
        reference_summary = np.asarray(
            p3ca_summary_cache[
                1,
                fold,
                indices,
            ],
            dtype=np.float32,
        )
        reference_sequence = np.asarray(
            p3ca_sequence_cache[
                1,
                fold,
                indices,
            ],
            dtype=np.float32,
        )
        p3ca_summary_maximum_error = max(
            p3ca_summary_maximum_error,
            float(
                np.max(
                    np.abs(
                        p3ca_summary
                        - reference_summary
                    )
                )
            ),
        )
        p3ca_sequence_maximum_error = max(
            p3ca_sequence_maximum_error,
            float(
                np.max(
                    np.abs(
                        p3ca_sequence
                        - reference_sequence
                    )
                )
            ),
        )

        transformed33 = (
            runtime_module.transform_fold_features(
                runtime.phase33_preprocessors,
                fold,
                p3ca_sequence,
                geometry,
                p3ca_summary,
                radiomics,
            )
        )
        model_arguments = {
            key: torch.from_numpy(value).to(
                PHASE39_DEVICE
            )
            for key, value in transformed33.items()
        }
        seed_logits = []
        with torch.inference_mode():
            for model in runtime.phase33_models[fold]:
                output = model(
                    p3ca_sequence=model_arguments[
                        "p3ca_sequence"
                    ],
                    geometry_sequence=model_arguments[
                        "geometry_sequence"
                    ],
                    summary_features=model_arguments[
                        "summary_features"
                    ],
                    radiomics_features=model_arguments[
                        "radiomics_features"
                    ],
                    domain_coefficient=0.0,
                )
                seed_logits.append(
                    output["logit"]
                    .float()
                    .cpu()
                    .numpy()
                )
        phase33_runtime_logit[indices] = np.mean(
            np.stack(seed_logits, axis=0),
            axis=0,
        ).astype(np.float64)

        transformed36 = (
            runtime_module.transform_fold_features(
                runtime.phase36_preprocessors,
                fold,
                p3ca_sequence,
                geometry,
                p3ca_summary,
                radiomics,
            )
        )
        phase36_features, phase36_baseline_margin = (
            runtime_module.phase36_build_features(
                transformed36,
                baseline_oof[indices],
            )
        )
        assert phase36_features.shape == (
            len(indices),
            1240,
        )
        seed_residuals = []
        for booster_record in (
            runtime.phase36_boosters[fold]
        ):
            seed_residuals.append(
                runtime_module.phase36_predict_residual(
                    booster_record["booster"],
                    phase36_features,
                    phase36_baseline_margin,
                    booster_record["round_count"],
                )
            )
        phase36_runtime_raw_residual[indices] = np.mean(
            np.stack(seed_residuals, axis=0),
            axis=0,
        ).astype(np.float64)

        completed_case_count += len(indices)
        while completed_case_count >= next_progress:
            print(
                "Phase39 complete parity: "
                f"{next_progress}/{PHASE39_CASE_COUNT}"
            )
            next_progress += 100

        del (
            dense,
            radiomics,
            geometry,
            p3ca_summary,
            p3ca_sequence,
            reference_summary,
            reference_sequence,
            transformed33,
            model_arguments,
            seed_logits,
            transformed36,
            phase36_features,
            phase36_baseline_margin,
            seed_residuals,
        )

assert completed_case_count == PHASE39_CASE_COUNT
print(
    "Phase39 complete parity: "
    f"{PHASE39_CASE_COUNT}/{PHASE39_CASE_COUNT}"
)

assert np.isfinite(phase33_runtime_logit).all()
assert np.isfinite(
    phase36_runtime_raw_residual
).all()
assert p3ca_summary_maximum_error <= 5e-4, {
    "p3ca_summary_maximum_error": (
        p3ca_summary_maximum_error
    )
}
assert p3ca_sequence_maximum_error <= 5e-4, {
    "p3ca_sequence_maximum_error": (
        p3ca_sequence_maximum_error
    )
}


# ------------------------------------------------------------------
# Independent formula reconstruction and exact references.
# ------------------------------------------------------------------

phase33_runtime_probability = np.clip(
    runtime_module.probability_sigmoid(
        phase33_runtime_logit
    ),
    1e-5,
    1.0 - 1e-5,
)
phase33_runtime_residual = (
    phase33_runtime_logit
    - baseline_logit
)
phase36_runtime_probability = np.clip(
    runtime_module.probability_sigmoid(
        baseline_logit
        + 0.75 * phase36_runtime_raw_residual
    ),
    1e-5,
    1.0 - 1e-5,
)
phase39_runtime_raw_combined_residual = (
    0.75 * phase36_runtime_raw_residual
    + 0.25 * phase33_runtime_residual
)
phase39_runtime_combined_residual = np.clip(
    phase39_runtime_raw_combined_residual,
    -2.0,
    2.0,
)
phase39_runtime_probability = np.clip(
    runtime_module.probability_sigmoid(
        baseline_logit
        + phase39_runtime_combined_residual
    ),
    1e-5,
    1.0 - 1e-5,
)


def phase39_error_summary(actual, reference):
    difference = np.abs(
        np.asarray(actual, dtype=np.float64)
        - np.asarray(reference, dtype=np.float64)
    )
    return {
        "maximum": float(np.max(difference)),
        "mean": float(np.mean(difference)),
        "q99": float(np.quantile(difference, 0.99)),
    }


phase33_probability_error = phase39_error_summary(
    phase33_runtime_probability,
    phase33_reference,
)
phase36_raw_residual_error = phase39_error_summary(
    phase36_runtime_raw_residual,
    phase36_raw_reference,
)
phase36_probability_error = phase39_error_summary(
    phase36_runtime_probability,
    phase36_reference,
)
phase39_probability_error = phase39_error_summary(
    phase39_runtime_probability,
    phase39_reference,
)

if phase39_combined_reference is not None:
    phase39_combined_residual_error = (
        phase39_error_summary(
            # Persistence retained the uncapped diagnostic residual. The
            # cap is applied only when composing the final Phase39 logit.
            phase39_runtime_raw_combined_residual,
            phase39_combined_reference,
        )
    )
else:
    phase39_combined_residual_error = None
    phase39_combined_reference_source = "not_available"

assert phase33_probability_error["maximum"] <= 5e-5, (
    phase33_probability_error
)
assert phase36_raw_residual_error["maximum"] <= 5e-5, (
    phase36_raw_residual_error
)
assert phase36_probability_error["maximum"] <= 5e-6, (
    phase36_probability_error
)
assert phase39_probability_error["maximum"] <= 5e-5, (
    phase39_probability_error
)
if phase39_combined_residual_error is not None:
    assert (
        phase39_combined_residual_error["maximum"]
        <= 5e-5
    ), phase39_combined_residual_error


def phase39_metrics(probabilities):
    probability = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-5,
        1.0 - 1e-5,
    )
    return {
        "log_loss": float(
            log_loss(labels, probability)
        ),
        "auroc": float(
            roc_auc_score(labels, probability)
        ),
        "mean_probability": float(
            np.mean(probability)
        ),
    }


baseline_metrics = phase39_metrics(baseline_oof)
phase33_runtime_metrics = phase39_metrics(
    phase33_runtime_probability
)
phase36_runtime_metrics = phase39_metrics(
    phase36_runtime_probability
)
phase39_runtime_metrics = phase39_metrics(
    phase39_runtime_probability
)
phase39_reference_metrics = phase39_metrics(
    phase39_reference
)

historical_metrics = {
    "baseline": {
        "log_loss": 0.307616,
        "auroc": 0.938914,
    },
    "phase33_component": {
        "log_loss": 0.358791,
        "auroc": 0.923527,
    },
    "phase36": {
        "log_loss": 0.295657,
        "auroc": 0.944143,
    },
    "phase39": {
        "log_loss": 0.293027,
        "auroc": 0.945575,
    },
}

metric_pairs = [
    ("baseline", baseline_metrics),
    ("phase33_component", phase33_runtime_metrics),
    ("phase36", phase36_runtime_metrics),
    ("phase39", phase39_runtime_metrics),
]
historical_metric_deltas = {}
for name, metrics in metric_pairs:
    historical_metric_deltas[name] = {
        metric_name: abs(
            float(metrics[metric_name])
            - float(
                historical_metrics[name][metric_name]
            )
        )
        for metric_name in (
            "log_loss",
            "auroc",
        )
    }
    assert (
        historical_metric_deltas[name]["log_loss"]
        <= 5e-6
    ), {
        "name": name,
        "metrics": metrics,
        "historical": historical_metrics[name],
    }
    assert (
        historical_metric_deltas[name]["auroc"]
        <= 5e-5
    ), {
        "name": name,
        "metrics": metrics,
        "historical": historical_metrics[name],
    }

assert abs(
    phase39_runtime_metrics["log_loss"]
    - phase39_reference_metrics["log_loss"]
) <= 5e-6
assert abs(
    phase39_runtime_metrics["auroc"]
    - phase39_reference_metrics["auroc"]
) <= 5e-6


fold_metrics = []
for partition in partitions:
    fold = int(partition["fold"])
    indices = partition["outer_valid"]
    fold_metrics.append({
        "fold": fold,
        "n": int(len(indices)),
        "baseline_log_loss": float(
            log_loss(
                labels[indices],
                baseline_oof[indices],
            )
        ),
        "phase39_log_loss": float(
            log_loss(
                labels[indices],
                phase39_runtime_probability[indices],
            )
        ),
        "baseline_auroc": float(
            roc_auc_score(
                labels[indices],
                baseline_oof[indices],
            )
        ),
        "phase39_auroc": float(
            roc_auc_score(
                labels[indices],
                phase39_runtime_probability[indices],
            )
        ),
    })


elapsed_seconds = float(
    time.perf_counter() - phase39_parity_started
)
peak_vram_mb = (
    float(
        torch.cuda.max_memory_allocated(
            PHASE39_DEVICE
        )
        / (1024 ** 2)
    )
    if torch.cuda.is_available()
    else 0.0
)

report = {
    "phase": "phase39_complete_packaged_oof_parity",
    "status": "accepted",
    "case_count": PHASE39_CASE_COUNT,
    "fold_count": len(partitions),
    "partition_source": partition_source,
    "private_cache_contract": {
        "highres_shape": list(highres_cache.shape),
        "radiomics_shape": list(radiomics_cache.shape),
        "geometry_shape": list(geometry_cache.shape),
        "dense_shape": list(dense_cache.shape),
        "p3ca_summary_shape": list(
            p3ca_summary_cache.shape
        ),
        "p3ca_sequence_shape": list(
            p3ca_sequence_cache.shape
        ),
        "highres_float16_boundary": True,
        "dense_float16_boundary": True,
        "projected_float16_boundary": True,
    },
    "feature_extraction_parity": {
        "sampled_case_count": int(
            len(audit_indices)
        ),
        "radiomics_maximum_error": (
            radiomics_feature_maximum_error
        ),
        "radiomics_mean_error": (
            radiomics_feature_mean_error
        ),
        "geometry_maximum_error": (
            geometry_feature_maximum_error
        ),
        "geometry_mean_error": (
            geometry_feature_mean_error
        ),
        "p3ca_summary_maximum_error": (
            p3ca_summary_maximum_error
        ),
        "p3ca_sequence_maximum_error": (
            p3ca_sequence_maximum_error
        ),
    },
    "component_parity": {
        "phase33_probability_error": (
            phase33_probability_error
        ),
        "phase36_raw_residual_error": (
            phase36_raw_residual_error
        ),
        "phase36_probability_error": (
            phase36_probability_error
        ),
        "combined_residual_error": (
            phase39_combined_residual_error
        ),
        "combined_residual_reference_semantics": (
            "uncapped_before_delta_cap"
        ),
        "delta_cap_activation_fraction": float(
            np.mean(
                np.abs(
                    phase39_runtime_raw_combined_residual
                ) > 2.0
            )
        ),
        "phase39_probability_error": (
            phase39_probability_error
        ),
    },
    "reference_sources": {
        "baseline": str(
            cache_paths["baseline_oof"]
        ),
        "phase33_component": (
            phase33_reference_source
        ),
        "phase36_raw_residual": (
            phase36_raw_reference_source
        ),
        "phase36_probability": (
            phase36_reference_source
        ),
        "combined_residual": (
            phase39_combined_reference_source
        ),
        "phase39_probability": (
            phase39_reference_source
        ),
    },
    "runtime_metrics": {
        "baseline": baseline_metrics,
        "phase33_component": (
            phase33_runtime_metrics
        ),
        "phase36": phase36_runtime_metrics,
        "phase39": phase39_runtime_metrics,
    },
    "reference_phase39_metrics": (
        phase39_reference_metrics
    ),
    "historical_metric_absolute_deltas": (
        historical_metric_deltas
    ),
    "fold_metrics": fold_metrics,
    "deployment_formula": (
        "z_phase39 = z_full_phase12c + clip("
        "0.75*r_phase36_raw + "
        "0.25*(z_phase33-z_full_phase12c), -2, 2)"
    ),
    "phase36_raw_residual_scaled_exactly_once": True,
    "held_out_fold_model_used_for_each_oof_case": True,
    "three_fold_deployment_average_used_for_oof": False,
    "outer_validation_labels_used_only_for_final_parity_metrics": True,
    "training_feature_caches_read": True,
    "case_level_predictions_exported": False,
    "challenge_voxel_data_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "model_hashes_displayed": False,
    "peak_vram_mb": peak_vram_mb,
    "elapsed_seconds": elapsed_seconds,
}


def phase39_json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    raise TypeError(type(value).__name__)


PHASE39_PARITY_CONTRACT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
        default=phase39_json_default,
    ) + "\n",
    encoding="utf-8",
)

# Retain only aggregate diagnostics plus the accepted runtime objects.
PHASE39_COMPLETE_PARITY_REPORT = report

print("BEGIN SANITIZED_PHASE39_COMPLETE_PARITY")
print(
    json.dumps(
        report,
        indent=2,
        default=phase39_json_default,
    )
)
print("END SANITIZED_PHASE39_COMPLETE_PARITY")
print("Cell 128 accepted. Continue with Cell 129 real packaged smoke test.")

Phase39 complete parity: 100/1362
Phase39 complete parity: 200/1362
Phase39 complete parity: 300/1362
Phase39 complete parity: 400/1362
Phase39 complete parity: 500/1362
Phase39 complete parity: 600/1362
Phase39 complete parity: 700/1362
Phase39 complete parity: 800/1362
Phase39 complete parity: 900/1362
Phase39 complete parity: 1000/1362
Phase39 complete parity: 1100/1362
Phase39 complete parity: 1200/1362
Phase39 complete parity: 1300/1362
Phase39 complete parity: 1362/1362
BEGIN SANITIZED_PHASE39_COMPLETE_PARITY
{
  "phase": "phase39_complete_packaged_oof_parity",
  "status": "accepted",
  "case_count": 1362,
  "fold_count": 3,
  "partition_source": "PHASE19_PARTITIONS",
  "private_cache_contract": {
    "highres_shape": [
      1362,
      80,
      80,
      80
    ],
    "radiomics_shape": [
      1362,
      1081
    ],
    "geometry_shape": [
      1362,
      16,
      52
    ],
    "dense_shape": [
      1362,
      6,
      14,
      14,
      384
    ],
    "p3ca_summary_sh

In [102]:
# Phase39 Cell 129
# Real packaged smoke test in a separate offline subprocess.
#
# Prerequisite: accepted SANITIZED_PHASE39_COMPLETE_PARITY from Cell 128.
# This cell reads the private 20-case smoke images and submission format, but
# never reads smoke labels or challenge/test data. The temporary submission
# rows are deleted immediately after aggregate validation.

from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

import numpy as np
import pandas as pd


PHASE39_SMOKE_PACKAGE = Path(
    "/kaggle/working/phase39_experimental_submission"
)
PHASE39_SMOKE_MAXIMUM_SECONDS = 360.0
PHASE39_SMOKE_EXPECTED_CASES = 20
PHASE39_SMOKE_CONTRACT_PATH = (
    PHASE39_SMOKE_PACKAGE
    / "phase39_real_smoke_contract.json"
)

phase39_smoke_started = time.perf_counter()


# ------------------------------------------------------------------
# Preconditions and private smoke-root resolution.
# ------------------------------------------------------------------

assert isinstance(
    globals().get("PHASE39_COMPLETE_PARITY_REPORT"),
    dict,
), {
    "message": (
        "Cell 129 requires the accepted Cell 128 parity report "
        "in the current kernel."
    )
}
assert (
    PHASE39_COMPLETE_PARITY_REPORT.get("status")
    == "accepted"
)

required_package_files = [
    Path("main.py"),
    Path("phase39_runtime.py"),
    Path("phase30_accepted_main.py"),
    Path("phase39_assets") / "phase39_runtime_state.json",
    Path("phase39_assets") / "phase39_asset_manifest.json",
]
missing_package_files = [
    str(relative_path)
    for relative_path in required_package_files
    if not (
        PHASE39_SMOKE_PACKAGE / relative_path
    ).is_file()
]
assert not missing_package_files, {
    "message": "Phase39 staged package is incomplete.",
    "missing": missing_package_files,
}


def phase39_resolve_smoke_root():
    candidates = []

    for environment_name in (
        "DAT_PRIVATE_SMOKE_ROOT",
        "DAT_SMOKE_ROOT",
    ):
        value = os.environ.get(environment_name)
        if value:
            candidates.append(
                (
                    Path(value),
                    f"environment:{environment_name}",
                )
            )

    candidates.extend([
        (
            Path(
                "/kaggle/input/datasets/nahinalam/"
                "drivendata-dataset/smoke_test_data"
            ),
            "private_dataset_default",
        ),
        (
            Path(
                "/kaggle/input/drivendata-dataset/"
                "smoke_test_data"
            ),
            "private_dataset_short_path",
        ),
    ])

    seen = set()
    audit = []
    for root, source in candidates:
        normalized = str(root)
        if normalized in seen:
            continue
        seen.add(normalized)

        image_directory = root / "niftis"
        format_path = root / "submission_format.csv"
        valid = (
            image_directory.is_dir()
            and format_path.is_file()
        )
        audit.append({
            "source": source,
            "valid": bool(valid),
        })
        if valid:
            return root, source, audit

    raise AssertionError({
        "message": "Could not resolve the private smoke-test root.",
        "candidate_audit": audit,
    })


smoke_root, smoke_root_source, smoke_root_audit = (
    phase39_resolve_smoke_root()
)
smoke_image_directory = smoke_root / "niftis"
smoke_format_path = smoke_root / "submission_format.csv"
smoke_labels_do_not_read = smoke_root / "test_labels.csv"


def phase39_nifti_identifier(path):
    name = path.name
    lower = name.lower()
    if lower.endswith(".nii.gz"):
        return name[:-7]
    if lower.endswith(".nii"):
        return name[:-4]
    raise ValueError("Unexpected NIfTI extension")


smoke_paths = sorted(
    path
    for path in smoke_image_directory.iterdir()
    if path.is_file()
    and (
        path.name.lower().endswith(".nii")
        or path.name.lower().endswith(".nii.gz")
    )
)
assert len(smoke_paths) == PHASE39_SMOKE_EXPECTED_CASES, {
    "expected": PHASE39_SMOKE_EXPECTED_CASES,
    "observed": len(smoke_paths),
}

smoke_format = pd.read_csv(
    smoke_format_path,
    dtype={"uid": str},
)
assert list(smoke_format.columns) == [
    "uid",
    "is_pathologic",
]
assert len(smoke_format) == PHASE39_SMOKE_EXPECTED_CASES
assert smoke_format["uid"].notna().all()
assert smoke_format["uid"].is_unique

expected_uids = smoke_format["uid"].astype(str).tolist()
image_uids = [
    phase39_nifti_identifier(path)
    for path in smoke_paths
]
assert set(expected_uids) == set(image_uids)


# ------------------------------------------------------------------
# Independent subprocess with offline-only environment settings.
# ------------------------------------------------------------------

temporary_root = Path(
    tempfile.mkdtemp(
        prefix="phase39_real_smoke_",
        dir="/kaggle/working",
    )
)
temporary_output = temporary_root / "submission.csv"
temporary_torch_home = temporary_root / "torch_hub"

subprocess_environment = os.environ.copy()
subprocess_environment.update({
    "DAT_DATA_ROOT": str(smoke_root),
    "DAT_OUTPUT_CSV": str(temporary_output),
    "DAT_PREPROCESS_WORKERS": "6",
    "DAT_BATCH_SIZE": "4",
    "DAT_DINO_VIEW_BATCH": "12",
    "TORCH_HOME": str(temporary_torch_home),
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "PYTHONHASHSEED": "390129",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
    "CUDA_VISIBLE_DEVICES": "0",
})

process = None
smoke_submission = None
smoke_probability = None
combined_log = ""
safe_log_lines = []
output_deleted_after_validation = False

try:
    process_started = time.perf_counter()
    process = subprocess.run(
        [
            sys.executable,
            str(PHASE39_SMOKE_PACKAGE / "main.py"),
        ],
        cwd=str(PHASE39_SMOKE_PACKAGE),
        env=subprocess_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=PHASE39_SMOKE_MAXIMUM_SECONDS,
        check=False,
    )
    subprocess_seconds = float(
        time.perf_counter() - process_started
    )

    combined_log = (
        (process.stdout or "")
        + "\n"
        + (process.stderr or "")
    )

    # The package must never print identifiers or case-level values.
    uid_in_log = any(
        uid and uid in combined_log
        for uid in expected_uids
    )
    assert not uid_in_log, {
        "message": "The runtime log exposed a smoke UID."
    }

    # Only local file:// checkpoint activity is permitted. HTTP(S) would
    # indicate an invalid online dependency.
    network_url_present = bool(
        re.search(
            r"https?://",
            combined_log,
            flags=re.IGNORECASE,
        )
    )
    assert not network_url_present, {
        "message": "The offline smoke log contained a network URL."
    }

    relevant_fragments = (
        "Built with DINOv3.",
        "Initialization complete.",
        "Inference started.",
        "Inference completed:",
        "Execution failed:",
        "UserWarning:",
    )
    for raw_line in combined_log.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if not any(
            fragment in line
            for fragment in relevant_fragments
        ):
            continue
        line = line.replace(
            str(PHASE39_SMOKE_PACKAGE),
            "<phase39_package>",
        )
        line = line.replace(
            str(temporary_root),
            "<temporary_smoke_root>",
        )
        line = line.replace(
            str(smoke_root),
            "<private_smoke_root>",
        )
        safe_log_lines.append(line[:500])

    assert process.returncode == 0, {
        "message": "Phase39 smoke subprocess failed.",
        "return_code": process.returncode,
        "safe_log_tail": safe_log_lines[-8:],
    }
    assert subprocess_seconds <= (
        PHASE39_SMOKE_MAXIMUM_SECONDS
    )
    assert temporary_output.is_file(), {
        "message": "Runtime did not create submission.csv."
    }

    required_log_fragments = [
        "Built with DINOv3.",
        "Initialization complete.",
        "Inference started.",
        "Inference completed:",
    ]
    assert all(
        fragment in combined_log
        for fragment in required_log_fragments
    ), {
        "message": "Required runtime log contract is incomplete.",
        "safe_log": safe_log_lines,
    }

    completion_match = re.search(
        r"Inference completed:\s*"
        r"cases=(\d+),\s*"
        r"preprocessing_fallbacks=(\d+),\s*"
        r"phase39_initialization_fallbacks=(\d+),\s*"
        r"phase39_case_fallbacks=(\d+)\.",
        combined_log,
    )
    assert completion_match is not None, {
        "message": "Could not parse Phase39 completion counters.",
        "safe_log": safe_log_lines,
    }
    (
        logged_case_count,
        preprocessing_fallbacks,
        initialization_fallbacks,
        phase39_case_fallbacks,
    ) = [
        int(value)
        for value in completion_match.groups()
    ]
    assert logged_case_count == PHASE39_SMOKE_EXPECTED_CASES
    assert preprocessing_fallbacks == 0
    assert initialization_fallbacks == 0
    assert phase39_case_fallbacks == 0

    smoke_submission = pd.read_csv(
        temporary_output,
        dtype={"uid": str},
    )
    assert list(smoke_submission.columns) == [
        "uid",
        "is_pathologic",
    ]
    assert len(smoke_submission) == (
        PHASE39_SMOKE_EXPECTED_CASES
    )
    assert smoke_submission["uid"].notna().all()
    assert smoke_submission["uid"].is_unique
    assert (
        smoke_submission["uid"].astype(str).tolist()
        == expected_uids
    )

    smoke_probability = smoke_submission[
        "is_pathologic"
    ].to_numpy(dtype=np.float64)
    assert smoke_probability.shape == (
        PHASE39_SMOKE_EXPECTED_CASES,
    )
    assert np.isfinite(smoke_probability).all()
    assert np.all(smoke_probability >= 1e-5)
    assert np.all(smoke_probability <= 1.0 - 1e-5)
    assert float(np.ptp(smoke_probability)) > 0.0

finally:
    # Do not retain private smoke UIDs, rows, predictions, or the temporary
    # torch-hub checkpoint copy as a notebook artifact.
    smoke_submission = None
    smoke_probability = None
    if temporary_root.exists():
        shutil.rmtree(temporary_root)
    output_deleted_after_validation = (
        not temporary_root.exists()
    )


assert process is not None
assert output_deleted_after_validation

package_files = [
    path
    for path in PHASE39_SMOKE_PACKAGE.rglob("*")
    if path.is_file()
]
package_size_bytes = sum(
    path.stat().st_size
    for path in package_files
)

report = {
    "phase": "phase39_real_packaged_smoke_test",
    "status": "accepted",
    "package_directory": (
        PHASE39_SMOKE_PACKAGE.name
    ),
    "package_file_count": int(len(package_files)),
    "package_size_mb": float(
        package_size_bytes / (1024 ** 2)
    ),
    "case_count": PHASE39_SMOKE_EXPECTED_CASES,
    "output_row_count": PHASE39_SMOKE_EXPECTED_CASES,
    "schema_valid": True,
    "uid_order_exact": True,
    "all_probabilities_finite": True,
    "all_probabilities_clipped": True,
    "output_nonconstant": True,
    "fallbacks": {
        "preprocessing": int(
            preprocessing_fallbacks
        ),
        "phase39_initialization": int(
            initialization_fallbacks
        ),
        "phase39_case": int(
            phase39_case_fallbacks
        ),
        "total": int(
            preprocessing_fallbacks
            + initialization_fallbacks
            + phase39_case_fallbacks
        ),
    },
    "return_code": int(process.returncode),
    "elapsed_seconds": float(subprocess_seconds),
    "maximum_allowed_seconds": (
        PHASE39_SMOKE_MAXIMUM_SECONDS
    ),
    "log_line_count": int(
        len([
            line
            for line in combined_log.splitlines()
            if line.strip()
        ])
    ),
    "sanitized_log_line_count": int(
        len(safe_log_lines)
    ),
    "dinov3_attribution_logged": True,
    "offline_environment_enforced": True,
    "network_url_present": False,
    "local_file_checkpoint_copy_allowed": True,
    "separate_subprocess": True,
    "phase39_path_exercised_without_fallback": True,
    "temporary_submission_deleted": (
        output_deleted_after_validation
    ),
    "temporary_torch_cache_deleted": (
        output_deleted_after_validation
    ),
    "smoke_root_source": smoke_root_source,
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "training_voxel_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "model_hashes_displayed": False,
    "total_cell_seconds": float(
        time.perf_counter() - phase39_smoke_started
    ),
}

PHASE39_SMOKE_CONTRACT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

PHASE39_REAL_SMOKE_REPORT = report

print("BEGIN PHASE39_SMOKE_LOG")
for line in safe_log_lines:
    print(line)
print("END PHASE39_SMOKE_LOG")
print("BEGIN SANITIZED_PHASE39_REAL_SMOKE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE39_REAL_SMOKE")
print("Cell 129 accepted. Continue with Cell 130 final archive rehearsal.")

BEGIN PHASE39_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed: cases=20, preprocessing_fallbacks=0, phase39_initialization_fallbacks=0, phase39_case_fallbacks=0.
<phase39_package>/phase12c_main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
END PHASE39_SMOKE_LOG
BEGIN SANITIZED_PHASE39_REAL_SMOKE
{
  "phase": "phase39_real_packaged_smoke_test",
  "status": "accepted",
  "package_directory": "phase39_experimental_submission",
  "package_file_count": 113,
  "package_size_mb": 193.5492124557495,
  "case_count": 20,
  "output_row_count": 20,
  "schema_valid": true,
  "uid_order_exact": true,
  "all_probabilities_finite": true,
  "all_probabilities_clipped": true,
  "output_nonconstant": true,
  "fallbacks": {
    "preprocessing": 0,
    "phase39_initialization": 0,
    "phase39_case": 0,
    "total": 0
  },
  "return_code": 0,
  "elapsed_seconds": 20.222949452003377

In [104]:
# Phase39 Cell 130
# Build a clean deterministic submission archive, extract it into a fresh
# directory, and run the real 20-case smoke test from the extracted archive.
#
# Prerequisite: accepted SANITIZED_PHASE39_REAL_SMOKE from Cell 129.
# The accepted staging directory is not modified. A final ZIP is published
# atomically only after archive integrity and extracted-runtime smoke pass.

from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

import numpy as np
import pandas as pd


PHASE39_FINAL_STAGE = Path(
    "/kaggle/working/phase39_experimental_submission"
)
PHASE39_FINAL_ZIP = Path(
    "/kaggle/working/phase39_submission.zip"
)
# Cell 126 reported 89 files after importing the staged modules; two of
# those were generated .pyc files. The immutable deployable source/assets
# set contains exactly 87 files.
PHASE39_FINAL_EXPECTED_FILES = 87
PHASE39_FINAL_EXPECTED_SMOKE_CASES = 20
PHASE39_FINAL_SMOKE_LIMIT_SECONDS = 360.0

phase39_final_started = time.perf_counter()


# ------------------------------------------------------------------
# Preconditions retained from the accepted real staged smoke.
# ------------------------------------------------------------------

assert isinstance(
    globals().get("PHASE39_REAL_SMOKE_REPORT"),
    dict,
), {
    "message": (
        "Cell 130 requires the accepted Cell 129 report in the "
        "current kernel."
    )
}
assert PHASE39_REAL_SMOKE_REPORT.get("status") == "accepted"
assert PHASE39_REAL_SMOKE_REPORT.get("fallbacks", {}).get(
    "total"
) == 0
assert PHASE39_FINAL_STAGE.is_dir()

required_relative_paths = [
    Path("main.py"),
    Path("phase39_runtime.py"),
    Path("phase30_accepted_main.py"),
    Path("phase39_runtime_source_manifest.json"),
    Path("phase39_assets") / "phase39_runtime_state.json",
    Path("phase39_assets") / "phase39_asset_manifest.json",
]
missing_required_paths = [
    path.as_posix()
    for path in required_relative_paths
    if not (PHASE39_FINAL_STAGE / path).is_file()
]
assert not missing_required_paths, {
    "message": "Required Phase39 deployment files are missing.",
    "missing": missing_required_paths,
}


# ------------------------------------------------------------------
# Select only deployable package files; preserve staging unchanged.
# ------------------------------------------------------------------

excluded_contract_names = {
    "phase39_offline_initialization_contract.json",
    "phase39_complete_oof_parity_contract.json",
    "phase39_real_smoke_contract.json",
}


def phase39_should_archive(relative_path):
    parts = relative_path.parts
    if not parts:
        return False
    if "__pycache__" in parts:
        return False
    if relative_path.name in excluded_contract_names:
        return False
    if relative_path.suffix.lower() in {
        ".pyc",
        ".pyo",
        ".ipynb",
    }:
        return False
    if any(part.startswith(".") for part in parts):
        return False
    return True


all_stage_files = sorted(
    path
    for path in PHASE39_FINAL_STAGE.rglob("*")
    if path.is_file()
)
assert all(not path.is_symlink() for path in all_stage_files)

archive_records = []
excluded_stage_files = []
for path in all_stage_files:
    relative_path = path.relative_to(
        PHASE39_FINAL_STAGE
    )
    if phase39_should_archive(relative_path):
        archive_records.append((relative_path, path))
    else:
        excluded_stage_files.append(relative_path)

assert len(archive_records) == PHASE39_FINAL_EXPECTED_FILES, {
    "message": (
        "Clean Phase39 deployable file count changed. Audit before "
        "creating a submission."
    ),
    "observed": len(archive_records),
    "expected": PHASE39_FINAL_EXPECTED_FILES,
    "excluded_count": len(excluded_stage_files),
}

archive_names = [
    relative_path.as_posix()
    for relative_path, _ in archive_records
]
assert len(archive_names) == len(set(archive_names))
assert "main.py" in archive_names
assert all(
    not name.startswith(
        f"{PHASE39_FINAL_STAGE.name}/"
    )
    for name in archive_names
)


def phase39_safe_archive_name(name):
    pure = PurePosixPath(name)
    if pure.is_absolute():
        return False
    if ".." in pure.parts:
        return False
    if not pure.parts:
        return False
    return True


assert all(
    phase39_safe_archive_name(name)
    for name in archive_names
)

lower_archive_names = [
    name.lower()
    for name in archive_names
]
forbidden_name_fragments = [
    "__pycache__",
    "submission.csv",
    "test_labels",
    "train_labels",
    "patient_rows",
    "component_oof",
    "gated_oof",
    "raw_residual_oof",
    "phase31_highres",
    "phase31_radiomics",
    "phase31_sequence",
    "bilateral_dense",
    "p3ca_summary_float16",
    "p3ca_sequence_float16",
    "smoke_contract",
    "parity_contract",
]
for fragment in forbidden_name_fragments:
    assert not any(
        fragment in name
        for name in lower_archive_names
    ), {
        "message": "Private or transient artifact selected for archive.",
        "fragment": fragment,
    }

phase12_model_files = [
    name
    for name in archive_names
    if name.startswith("models/")
    and name.lower().endswith(".pt")
]
dinov3_checkpoint_files = [
    name
    for name in archive_names
    if name.startswith("models/")
    and name.lower().endswith(".pth")
    and "dinov3" in name.lower()
]
phase33_model_files = [
    name
    for name in archive_names
    if name.startswith(
        "phase39_assets/phase33_models/"
    )
    and name.lower().endswith(".pt")
]
phase36_booster_files = [
    name
    for name in archive_names
    if name.startswith(
        "phase39_assets/phase36_boosters/"
    )
    and name.lower().endswith(".ubj")
]
p3ca_state_files = [
    name
    for name in archive_names
    if name.endswith(
        "phase32_fold_local_p3ca_states.npz"
    )
]

assert len(phase12_model_files) == 21
assert len(dinov3_checkpoint_files) == 1
assert len(phase33_model_files) == 9
assert len(phase36_booster_files) == 6
assert len(p3ca_state_files) == 1
assert any(
    name.startswith("dinov3_repo/")
    for name in archive_names
)

license_names = [
    name
    for name in archive_names
    if "license" in Path(name).name.lower()
]
assert license_names
assert any(
    "dinov3" in name.lower()
    for name in license_names
)

attribution_present = False
for candidate_name in (
    "README.md",
    "THIRD_PARTY_LICENSES.md",
    "main.py",
):
    candidate_path = PHASE39_FINAL_STAGE / candidate_name
    if not candidate_path.is_file():
        continue
    text = candidate_path.read_text(
        encoding="utf-8",
        errors="replace",
    )
    if "Built with DINOv3" in text:
        attribution_present = True
        break
assert attribution_present


# ------------------------------------------------------------------
# Create a deterministic temporary ZIP. It becomes final only after smoke.
# ------------------------------------------------------------------

temporary_zip_handle, temporary_zip_name = tempfile.mkstemp(
    prefix="phase39_submission_",
    suffix=".partial.zip",
    dir="/kaggle/working",
)
os.close(temporary_zip_handle)
temporary_zip_path = Path(temporary_zip_name)


def phase39_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


try:
    with zipfile.ZipFile(
        temporary_zip_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
        allowZip64=True,
    ) as archive:
        for relative_path, source_path in archive_records:
            information = zipfile.ZipInfo(
                filename=relative_path.as_posix(),
                date_time=(2026, 8, 17, 0, 0, 0),
            )
            information.compress_type = zipfile.ZIP_DEFLATED
            information.create_system = 3
            information.external_attr = (
                0o100644 << 16
            )
            information.flag_bits |= 0x800
            with source_path.open("rb") as source_handle:
                with archive.open(
                    information,
                    mode="w",
                    force_zip64=True,
                ) as destination_handle:
                    shutil.copyfileobj(
                        source_handle,
                        destination_handle,
                        length=1024 * 1024,
                    )

    assert temporary_zip_path.is_file()
    assert temporary_zip_path.stat().st_size > 0
    assert temporary_zip_path.stat().st_size < 500 * 10**6

    with zipfile.ZipFile(
        temporary_zip_path,
        mode="r",
    ) as archive:
        archive_members = archive.namelist()
        assert archive.testzip() is None
        assert archive_members == archive_names
        assert len(archive_members) == (
            PHASE39_FINAL_EXPECTED_FILES
        )
        assert "main.py" in archive_members
        assert all(
            phase39_safe_archive_name(name)
            for name in archive_members
        )

    # --------------------------------------------------------------
    # Extract the exact temporary archive and smoke its root main.py.
    # --------------------------------------------------------------

    rehearsal_root = Path(
        tempfile.mkdtemp(
            prefix="phase39_archive_rehearsal_",
            dir="/kaggle/working",
        )
    )
    rehearsal_output = rehearsal_root / "submission.csv"
    rehearsal_torch_home = rehearsal_root / "torch_hub"

    try:
        with zipfile.ZipFile(
            temporary_zip_path,
            mode="r",
        ) as archive:
            for member in archive.infolist():
                assert phase39_safe_archive_name(
                    member.filename
                )
            archive.extractall(rehearsal_root)

        assert (rehearsal_root / "main.py").is_file()
        assert not (
            rehearsal_root
            / PHASE39_FINAL_STAGE.name
        ).exists()

        # Reuse only the already accepted Cell129 smoke-root contract.
        assert "smoke_root" in globals()
        assert "expected_uids" in globals()
        archive_smoke_root = Path(smoke_root)
        archive_expected_uids = [
            str(value)
            for value in expected_uids
        ]
        assert len(archive_expected_uids) == (
            PHASE39_FINAL_EXPECTED_SMOKE_CASES
        )
        assert (
            archive_smoke_root / "niftis"
        ).is_dir()
        assert (
            archive_smoke_root
            / "submission_format.csv"
        ).is_file()

        archive_environment = os.environ.copy()
        archive_environment.update({
            "DAT_DATA_ROOT": str(
                archive_smoke_root
            ),
            "DAT_OUTPUT_CSV": str(
                rehearsal_output
            ),
            "DAT_PREPROCESS_WORKERS": "6",
            "DAT_BATCH_SIZE": "4",
            "DAT_DINO_VIEW_BATCH": "12",
            "TORCH_HOME": str(
                rehearsal_torch_home
            ),
            "HF_HUB_OFFLINE": "1",
            "TRANSFORMERS_OFFLINE": "1",
            "HF_DATASETS_OFFLINE": "1",
            "PYTHONHASHSEED": "390130",
            "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
            "CUDA_VISIBLE_DEVICES": "0",
        })

        archive_smoke_started = time.perf_counter()
        archive_process = subprocess.run(
            [
                sys.executable,
                str(rehearsal_root / "main.py"),
            ],
            cwd=str(rehearsal_root),
            env=archive_environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=PHASE39_FINAL_SMOKE_LIMIT_SECONDS,
            check=False,
        )
        archive_smoke_seconds = float(
            time.perf_counter()
            - archive_smoke_started
        )
        archive_log = (
            (archive_process.stdout or "")
            + "\n"
            + (archive_process.stderr or "")
        )

        uid_in_log = any(
            uid and uid in archive_log
            for uid in archive_expected_uids
        )
        assert not uid_in_log
        assert not re.search(
            r"https?://",
            archive_log,
            flags=re.IGNORECASE,
        )

        safe_archive_log_lines = []
        relevant_fragments = (
            "Built with DINOv3.",
            "Initialization complete.",
            "Inference started.",
            "Inference completed:",
            "Execution failed:",
            "UserWarning:",
        )
        for raw_line in archive_log.splitlines():
            line = raw_line.strip()
            if not line:
                continue
            if not any(
                fragment in line
                for fragment in relevant_fragments
            ):
                continue
            line = line.replace(
                str(rehearsal_root),
                "<extracted_phase39_archive>",
            )
            line = line.replace(
                str(archive_smoke_root),
                "<private_smoke_root>",
            )
            safe_archive_log_lines.append(
                line[:500]
            )

        assert archive_process.returncode == 0, {
            "message": (
                "Extracted Phase39 archive smoke failed."
            ),
            "return_code": archive_process.returncode,
            "safe_log_tail": (
                safe_archive_log_lines[-8:]
            ),
        }
        assert archive_smoke_seconds <= (
            PHASE39_FINAL_SMOKE_LIMIT_SECONDS
        )
        assert rehearsal_output.is_file()

        for fragment in (
            "Built with DINOv3.",
            "Initialization complete.",
            "Inference started.",
            "Inference completed:",
        ):
            assert fragment in archive_log

        completion_match = re.search(
            r"Inference completed:\s*"
            r"cases=(\d+),\s*"
            r"preprocessing_fallbacks=(\d+),\s*"
            r"phase39_initialization_fallbacks=(\d+),\s*"
            r"phase39_case_fallbacks=(\d+)\.",
            archive_log,
        )
        assert completion_match is not None, {
            "safe_log": safe_archive_log_lines,
        }
        (
            archive_case_count,
            archive_preprocessing_fallbacks,
            archive_initialization_fallbacks,
            archive_case_fallbacks,
        ) = [
            int(value)
            for value in completion_match.groups()
        ]
        assert archive_case_count == (
            PHASE39_FINAL_EXPECTED_SMOKE_CASES
        )
        assert archive_preprocessing_fallbacks == 0
        assert archive_initialization_fallbacks == 0
        assert archive_case_fallbacks == 0

        extracted_submission = pd.read_csv(
            rehearsal_output,
            dtype={"uid": str},
        )
        assert list(extracted_submission.columns) == [
            "uid",
            "is_pathologic",
        ]
        assert len(extracted_submission) == (
            PHASE39_FINAL_EXPECTED_SMOKE_CASES
        )
        assert extracted_submission[
            "uid"
        ].astype(str).tolist() == archive_expected_uids
        archive_probabilities = extracted_submission[
            "is_pathologic"
        ].to_numpy(dtype=np.float64)
        assert np.isfinite(
            archive_probabilities
        ).all()
        assert np.all(
            archive_probabilities >= 1e-5
        )
        assert np.all(
            archive_probabilities <= 1.0 - 1e-5
        )
        assert float(
            np.ptp(archive_probabilities)
        ) > 0.0

        # Do not retain case-level smoke rows or probabilities.
        extracted_submission = None
        archive_probabilities = None

    finally:
        if rehearsal_root.exists():
            shutil.rmtree(rehearsal_root)

    assert not rehearsal_root.exists()

    # Atomic publication only after extracted-archive smoke succeeds.
    os.replace(
        temporary_zip_path,
        PHASE39_FINAL_ZIP,
    )

finally:
    if temporary_zip_path.exists():
        temporary_zip_path.unlink()


# ------------------------------------------------------------------
# Final immutable archive audit and sanitized handoff.
# ------------------------------------------------------------------

assert PHASE39_FINAL_ZIP.is_file()
with zipfile.ZipFile(
    PHASE39_FINAL_ZIP,
    mode="r",
) as final_archive:
    final_members = final_archive.namelist()
    assert final_archive.testzip() is None
    assert final_members == archive_names

submission_sha256 = phase39_file_sha256(
    PHASE39_FINAL_ZIP
)
compressed_size_bytes = PHASE39_FINAL_ZIP.stat().st_size
uncompressed_size_bytes = sum(
    source_path.stat().st_size
    for _, source_path in archive_records
)

report = {
    "phase": "phase39_final_submission_archive",
    "status": "accepted",
    "zip_name": PHASE39_FINAL_ZIP.name,
    "archive_file_count": len(final_members),
    "compressed_size_mb": float(
        compressed_size_bytes / 1e6
    ),
    "uncompressed_size_mb": float(
        uncompressed_size_bytes / 1e6
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "deterministic_archive_metadata": True,
    "excluded_transient_file_count": int(
        len(excluded_stage_files)
    ),
    "phase12_model_count": len(
        phase12_model_files
    ),
    "dinov3_checkpoint_count": len(
        dinov3_checkpoint_files
    ),
    "phase33_model_count": len(
        phase33_model_files
    ),
    "phase36_booster_count": len(
        phase36_booster_files
    ),
    "p3ca_state_count": len(
        p3ca_state_files
    ),
    "dinov3_source_present": True,
    "required_attribution_present": (
        attribution_present
    ),
    "dinov3_license_present": True,
    "extracted_archive_smoke": {
        "status": "accepted",
        "case_count": int(
            archive_case_count
        ),
        "return_code": int(
            archive_process.returncode
        ),
        "elapsed_seconds": float(
            archive_smoke_seconds
        ),
        "maximum_allowed_seconds": (
            PHASE39_FINAL_SMOKE_LIMIT_SECONDS
        ),
        "all_probabilities_finite": True,
        "all_probabilities_clipped": True,
        "output_nonconstant": True,
        "fallback_count": int(
            archive_preprocessing_fallbacks
            + archive_initialization_fallbacks
            + archive_case_fallbacks
        ),
        "network_url_present": False,
        "temporary_rows_deleted": True,
    },
    "submission_sha256": submission_sha256,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_oof_arrays": False,
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "model_hashes_displayed": False,
    "total_finalization_seconds": float(
        time.perf_counter() - phase39_final_started
    ),
}

PHASE39_FINAL_ARCHIVE_REPORT = report

print("BEGIN PHASE39_EXTRACTED_ARCHIVE_SMOKE_LOG")
for line in safe_archive_log_lines:
    print(line)
print("END PHASE39_EXTRACTED_ARCHIVE_SMOKE_LOG")
print("BEGIN SANITIZED_PHASE39_FINAL_ARCHIVE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE39_FINAL_ARCHIVE")
print("Final Phase39 submission archive:")
print(PHASE39_FINAL_ZIP)

BEGIN PHASE39_EXTRACTED_ARCHIVE_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed: cases=20, preprocessing_fallbacks=0, phase39_initialization_fallbacks=0, phase39_case_fallbacks=0.
<extracted_phase39_archive>/phase12c_main.py:88: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
END PHASE39_EXTRACTED_ARCHIVE_SMOKE_LOG
BEGIN SANITIZED_PHASE39_FINAL_ARCHIVE
{
  "phase": "phase39_final_submission_archive",
  "status": "accepted",
  "zip_name": "phase39_submission.zip",
  "archive_file_count": 87,
  "compressed_size_mb": 185.821099,
  "uncompressed_size_mb": 202.682363,
  "root_main_present": true,
  "wrapping_directory_present": false,
  "archive_integrity_test_passed": true,
  "deterministic_archive_metadata": true,
  "excluded_transient_file_count": 27,
  "phase12_model_count": 21,
  "dinov3_checkpoint_count": 1,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "

> Phase 40

In [106]:
# Phase40 Cell 131A — restore UID-aligned training path order.
# No voxel arrays, labels, smoke data, or test data are read.

from pathlib import Path

phase40_frame = PHASE40_CASE_DF_PRIVATE

phase40_uid_column = next(
    column
    for column in ("uid", "case_id", "id")
    if column in phase40_frame.columns
)


def phase40_nifti_uid(path):
    name = Path(path).name

    if name.endswith(".nii.gz"):
        return name[:-7]

    if name.endswith(".nii"):
        return name[:-4]

    raise ValueError(name)


phase40_raw_paths = [
    Path(path)
    for path in PHASE40_NIFTI_PATHS_PRIVATE
]

phase40_path_by_uid = {
    phase40_nifti_uid(path): path
    for path in phase40_raw_paths
}

phase40_case_uids = [
    str(value)
    for value in phase40_frame[
        phase40_uid_column
    ].tolist()
]

assert len(phase40_path_by_uid) == 1362
assert len(set(phase40_case_uids)) == 1362
assert set(phase40_path_by_uid) == set(
    phase40_case_uids
)

nifti_paths = [
    phase40_path_by_uid[uid]
    for uid in phase40_case_uids
]

assert len(nifti_paths) == 1362
assert len(set(nifti_paths)) == 1362
assert all(path.is_file() for path in nifti_paths)

print(
    "Phase40 UID path alignment restored: "
    f"cases={len(nifti_paths)}"
)

Phase40 UID path alignment restored: cases=1362


In [107]:
# Phase40 Cell 131
# Recover the training-only acquisition router and prove that its deployable
# header route is identical to the held-out-fold route used by OOF evaluation.
# This cell reads NIfTI headers only; it does not read voxel arrays, smoke data,
# or challenge test data.

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import json
import time

import nibabel as nib
import numpy as np


PHASE40_CASE_COUNT = 1362
PHASE40_GROUP_COUNT = 15
PHASE40_FOLD_COUNT = 3

PHASE40_CONFIG = {
    "phase": "phase40_exact_header_routed_residual",
    "route_requires_exact_prototype": True,
    "unknown_route_policy": "phase12c_anchor_only",
    "shared_parameters_across_folds": True,
    "fold_specific_group_parameters": False,
    "phase36_weights": [
        round(float(value), 2)
        for value in np.linspace(0.0, 1.0, 21)
    ],
    "alphas": [
        0.0, 0.05, 0.10, 0.15, 0.20, 0.25,
        0.30, 0.40, 0.50, 0.60, 0.75,
    ],
    "uncertainty_exponents": [0.0, 0.5, 1.0, 2.0],
    "delta_caps": [0.50, 0.75, 1.00, 1.50, 2.00],
    "maximum_monitor_group_harm": 0.010,
    "maximum_q95_probability_update": 0.150,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "minimum_bootstrap_lcb_gain": 0.0,
    "minimum_bootstrap_probability_positive": 0.95,
    "auroc_log_loss_band": 0.001,
    "production_logging": "generic_status_lines_only",
}


def phase40_resolve_case_dataframe():
    for name in (
        "case_df",
        "PHASE31_CASE_DF",
        "PHASE19_CASE_DF",
    ):
        value = globals().get(name)
        if value is None:
            continue
        if not hasattr(value, "columns"):
            continue
        if len(value) != PHASE40_CASE_COUNT:
            continue
        if "acquisition_group" not in value.columns:
            continue
        return value.reset_index(drop=True), name

    raise AssertionError({
        "message": "Could not resolve the aligned 1362-row case dataframe.",
        "required_column": "acquisition_group",
    })


def phase40_resolve_partitions():
    candidate_names = (
        "PHASE19_PARTITIONS",
        "PHASE31_PARTITIONS",
        "phase19_partitions",
        "phase31_partitions",
    )

    for name in candidate_names:
        candidate = globals().get(name)
        if not isinstance(candidate, (list, tuple)):
            continue
        if len(candidate) != PHASE40_FOLD_COUNT:
            continue

        normalized = []
        valid = True
        for expected_fold, record in enumerate(candidate):
            if not isinstance(record, dict):
                valid = False
                break

            def resolve_key(names, required=True):
                key = next((item for item in names if item in record), None)
                if key is None:
                    if required:
                        raise KeyError(tuple(names))
                    return np.empty(0, dtype=np.int64)
                return np.asarray(record[key], dtype=np.int64)

            try:
                outer_valid = resolve_key((
                    "outer_valid", "outer_valid_indices", "valid",
                ))
                outer_train = resolve_key((
                    "outer_train", "outer_train_indices", "train",
                ))
                fit = resolve_key((
                    "fit", "fit_indices", "inner_train",
                ), required=False)
                monitor = resolve_key((
                    "monitor", "monitor_indices", "inner_valid",
                ), required=False)
            except Exception:
                valid = False
                break

            arrays = (outer_valid, outer_train, fit, monitor)
            if any(array.ndim != 1 for array in arrays):
                valid = False
                break
            if any(
                len(array)
                and (
                    np.any(array < 0)
                    or np.any(array >= PHASE40_CASE_COUNT)
                    or len(np.unique(array)) != len(array)
                )
                for array in arrays
            ):
                valid = False
                break
            if len(np.intersect1d(outer_train, outer_valid)):
                valid = False
                break
            if len(fit) and len(monitor):
                if len(np.intersect1d(fit, monitor)):
                    valid = False
                    break
                if not np.array_equal(
                    np.sort(np.concatenate([fit, monitor])),
                    np.sort(outer_train),
                ):
                    valid = False
                    break

            normalized.append({
                "fold": int(record.get("fold", expected_fold)),
                "outer_train": outer_train,
                "outer_valid": outer_valid,
                "fit": fit,
                "monitor": monitor,
            })

        if not valid:
            continue
        if [item["fold"] for item in normalized] != [0, 1, 2]:
            continue

        coverage = np.zeros(PHASE40_CASE_COUNT, dtype=np.int64)
        for item in normalized:
            coverage[item["outer_valid"]] += 1
        if np.all(coverage == 1):
            return normalized, name

    raise AssertionError({
        "message": "Could not resolve the exact three-fold partitions.",
        "candidate_names": list(candidate_names),
    })


def phase40_resolve_paths(case_dataframe):
    global_names = (
        "nifti_paths",
        "PHASE31_NIFTI_PATHS",
        "PHASE19_NIFTI_PATHS",
        "training_nifti_paths",
    )
    for name in global_names:
        value = globals().get(name)
        if value is None:
            continue
        try:
            paths = [Path(item) for item in list(value)]
        except Exception:
            continue
        if len(paths) != PHASE40_CASE_COUNT:
            continue
        if all(path.is_file() for path in paths):
            return paths, name

    for column in (
        "nifti_path",
        "path",
        "file_path",
        "image_path",
    ):
        if column not in case_dataframe.columns:
            continue
        paths = [Path(item) for item in case_dataframe[column].tolist()]
        if len(paths) == PHASE40_CASE_COUNT and all(
            path.is_file() for path in paths
        ):
            return paths, f"case_df.{column}"

    raise AssertionError({
        "message": "Could not resolve aligned training NIfTI paths.",
        "voxel_arrays_read": False,
    })


def phase40_router_candidates():
    roots = [
        Path("/kaggle/working/phase39_experimental_submission"),
        Path("/kaggle/working/phase39_zip_rehearsal"),
        Path("/kaggle/working/phase30_full_ensemble_rehearsal"),
        Path("/kaggle/working/phase30_submission"),
        Path("/kaggle/working/phase30_zip_rehearsal"),
    ]
    candidates = []
    for root in roots:
        path = root / "models" / "phase30_acquisition_router.npz"
        if path.is_file():
            candidates.append(path)

    working_root = Path("/kaggle/working")
    if working_root.is_dir():
        candidates.extend(
            working_root.glob(
                "*/models/phase30_acquisition_router.npz"
            )
        )

    unique = []
    seen = set()
    for path in candidates:
        resolved = str(path.resolve())
        if resolved not in seen:
            seen.add(resolved)
            unique.append(path)
    return unique


def phase40_load_router(path):
    with np.load(path, allow_pickle=False) as asset:
        required = {
            "header_columns",
            "prototypes",
            "prototype_groups",
            "feature_mean",
            "feature_scale",
            "fold_for_group",
            "corrected_groups",
            "rounding_decimals",
        }
        missing = sorted(required.difference(asset.files))
        assert not missing, {
            "message": "Router asset is incomplete.",
            "missing": missing,
        }
        router = {
            "header_columns": [
                str(value)
                for value in asset["header_columns"].tolist()
            ],
            "prototypes": np.asarray(
                asset["prototypes"], dtype=np.float64
            ),
            "prototype_groups": np.asarray(
                asset["prototype_groups"], dtype=np.int64
            ),
            "feature_mean": np.asarray(
                asset["feature_mean"], dtype=np.float64
            ),
            "feature_scale": np.asarray(
                asset["feature_scale"], dtype=np.float64
            ),
            "fold_for_group": np.asarray(
                asset["fold_for_group"], dtype=np.int64
            ),
            "corrected_groups": np.asarray(
                asset["corrected_groups"], dtype=np.int64
            ),
            "rounding_decimals": int(
                np.asarray(asset["rounding_decimals"]).reshape(-1)[0]
            ),
        }

    assert router["prototypes"].ndim == 2
    assert router["prototype_groups"].shape == (
        router["prototypes"].shape[0],
    )
    assert router["prototypes"].shape[1] == len(
        router["header_columns"]
    )
    assert router["feature_scale"].shape == (
        len(router["header_columns"]),
    )
    assert np.all(router["feature_scale"] > 0)
    assert router["fold_for_group"].shape == (
        PHASE40_GROUP_COUNT,
    )
    assert np.all(
        np.isin(router["fold_for_group"], [0, 1, 2])
    )
    return router


def phase40_router_signature(router):
    return (
        tuple(router["header_columns"]),
        router["prototypes"].shape,
        router["prototypes"].tobytes(),
        router["prototype_groups"].tobytes(),
        router["feature_scale"].tobytes(),
        router["fold_for_group"].tobytes(),
        int(router["rounding_decimals"]),
    )


def phase40_header_vector(path, header_columns):
    image = nib.load(str(path), mmap=True)
    assert len(image.shape) == 3
    shape = np.asarray(image.shape, dtype=np.float64)
    spacing = np.asarray(
        image.header.get_zooms()[:3], dtype=np.float64
    )
    affine = np.asarray(image.affine, dtype=np.float64)
    assert np.isfinite(shape).all() and np.all(shape > 0)
    assert np.isfinite(spacing).all() and np.all(spacing > 0)
    assert np.isfinite(affine).all()
    assert abs(float(np.linalg.det(affine[:3, :3]))) > 1e-8
    fov = shape * spacing
    features = {
        "shape_x": float(shape[0]),
        "shape_y": float(shape[1]),
        "shape_z": float(shape[2]),
        "spacing_x": float(spacing[0]),
        "spacing_y": float(spacing[1]),
        "spacing_z": float(spacing[2]),
        "fov_x": float(fov[0]),
        "fov_y": float(fov[1]),
        "fov_z": float(fov[2]),
        "dtype_is_int16": int(
            str(image.get_data_dtype()) == "int16"
        ),
    }
    return np.asarray(
        [features[column] for column in header_columns],
        dtype=np.float64,
    )


def phase40_route_header(path, router):
    vector = phase40_header_vector(
        path, router["header_columns"]
    )
    rounded = np.round(
        vector, decimals=router["rounding_decimals"]
    )
    exact_indices = np.flatnonzero(
        np.all(
            router["prototypes"] == rounded[None, :],
            axis=1,
        )
    )

    difference = (
        router["prototypes"] - rounded[None, :]
    ) / router["feature_scale"][None, :]
    squared_distance = np.sum(difference ** 2, axis=1)
    prototype_index = (
        int(exact_indices[0])
        if exact_indices.size
        else int(np.argmin(squared_distance))
    )
    group = int(router["prototype_groups"][prototype_index])
    fold = int(router["fold_for_group"][group])
    return (
        group,
        fold,
        bool(exact_indices.size),
        float(np.sqrt(squared_distance[prototype_index])),
    )


phase40_started = time.perf_counter()

PHASE40_CASE_DF_PRIVATE, phase40_case_df_source = (
    phase40_resolve_case_dataframe()
)
PHASE40_PARTITIONS_PRIVATE, phase40_partition_source = (
    phase40_resolve_partitions()
)
PHASE40_NIFTI_PATHS_PRIVATE, phase40_path_source = (
    phase40_resolve_paths(PHASE40_CASE_DF_PRIVATE)
)

phase40_router_paths = phase40_router_candidates()
assert phase40_router_paths, {
    "message": "No Phase30 acquisition router asset was found.",
}

phase40_loaded_routers = [
    phase40_load_router(path)
    for path in phase40_router_paths
]
phase40_router_signatures = {
    phase40_router_signature(router)
    for router in phase40_loaded_routers
}
assert len(phase40_router_signatures) == 1, {
    "message": "Conflicting acquisition router assets were found.",
    "candidate_count": len(phase40_router_paths),
    "distinct_payload_count": len(phase40_router_signatures),
}

PHASE40_ROUTER_PRIVATE = phase40_loaded_routers[0]
phase40_router_source = phase40_router_paths[0]

phase40_groups = np.asarray(
    PHASE40_CASE_DF_PRIVATE["acquisition_group"],
    dtype=np.int64,
)
assert phase40_groups.shape == (PHASE40_CASE_COUNT,)
assert np.all(
    (phase40_groups >= 0)
    & (phase40_groups < PHASE40_GROUP_COUNT)
)

phase40_oof_fold = np.full(
    PHASE40_CASE_COUNT, -1, dtype=np.int64
)
for partition in PHASE40_PARTITIONS_PRIVATE:
    phase40_oof_fold[partition["outer_valid"]] = int(
        partition["fold"]
    )
assert np.all(phase40_oof_fold >= 0)

phase40_fold_from_dataframe_group = (
    PHASE40_ROUTER_PRIVATE["fold_for_group"][phase40_groups]
)
phase40_dataframe_route_alignment = float(np.mean(
    phase40_fold_from_dataframe_group == phase40_oof_fold
))
assert phase40_dataframe_route_alignment == 1.0, {
    "message": (
        "Router group-to-fold mapping does not reproduce the exact "
        "held-out OOF route."
    ),
    "alignment_fraction": phase40_dataframe_route_alignment,
}

with ThreadPoolExecutor(max_workers=8) as executor:
    phase40_header_routes = list(executor.map(
        lambda path: phase40_route_header(
            path, PHASE40_ROUTER_PRIVATE
        ),
        PHASE40_NIFTI_PATHS_PRIVATE,
    ))

PHASE40_ROUTED_GROUP_PRIVATE = np.asarray(
    [item[0] for item in phase40_header_routes],
    dtype=np.int64,
)
PHASE40_ROUTED_FOLD_PRIVATE = np.asarray(
    [item[1] for item in phase40_header_routes],
    dtype=np.int64,
)
PHASE40_ROUTE_EXACT_PRIVATE = np.asarray(
    [item[2] for item in phase40_header_routes],
    dtype=bool,
)
PHASE40_ROUTE_DISTANCE_PRIVATE = np.asarray(
    [item[3] for item in phase40_header_routes],
    dtype=np.float64,
)

phase40_header_group_alignment = float(np.mean(
    PHASE40_ROUTED_GROUP_PRIVATE == phase40_groups
))
phase40_header_fold_alignment = float(np.mean(
    PHASE40_ROUTED_FOLD_PRIVATE == phase40_oof_fold
))
phase40_exact_fraction = float(np.mean(
    PHASE40_ROUTE_EXACT_PRIVATE
))

assert phase40_header_group_alignment == 1.0
assert phase40_header_fold_alignment == 1.0
assert phase40_exact_fraction == 1.0
assert np.max(PHASE40_ROUTE_DISTANCE_PRIVATE) == 0.0

phase40_group_records = []
for group in range(PHASE40_GROUP_COUNT):
    indices = np.flatnonzero(phase40_groups == group)
    monitor_count = int(sum(
        np.sum(phase40_groups[item["monitor"]] == group)
        for item in PHASE40_PARTITIONS_PRIVATE
        if len(item["monitor"])
    ))
    phase40_group_records.append({
        "group": int(group),
        "n": int(len(indices)),
        "held_out_fold": int(
            PHASE40_ROUTER_PRIVATE["fold_for_group"][group]
        ),
        "monitor_observation_count": monitor_count,
        "exact_header_count": int(
            np.sum(PHASE40_ROUTE_EXACT_PRIVATE[indices])
        ),
    })

PHASE40_ROUTE_CONTRACT = {
    "phase": "phase40_exact_header_routed_residual_contract",
    "status": "accepted",
    "case_count": PHASE40_CASE_COUNT,
    "acquisition_group_count": PHASE40_GROUP_COUNT,
    "fold_count": PHASE40_FOLD_COUNT,
    "router": {
        "candidate_file_count": len(phase40_router_paths),
        "distinct_payload_count": len(phase40_router_signatures),
        "prototype_count": int(
            PHASE40_ROUTER_PRIVATE["prototypes"].shape[0]
        ),
        "header_feature_count": len(
            PHASE40_ROUTER_PRIVATE["header_columns"]
        ),
        "rounding_decimals": int(
            PHASE40_ROUTER_PRIVATE["rounding_decimals"]
        ),
        "corrected_groups": sorted(
            int(value)
            for value in PHASE40_ROUTER_PRIVATE[
                "corrected_groups"
            ].tolist()
        ),
    },
    "route_parity": {
        "dataframe_group_to_oof_fold_alignment_fraction": (
            phase40_dataframe_route_alignment
        ),
        "header_to_dataframe_group_alignment_fraction": (
            phase40_header_group_alignment
        ),
        "header_to_oof_fold_alignment_fraction": (
            phase40_header_fold_alignment
        ),
        "exact_prototype_match_fraction": phase40_exact_fraction,
        "maximum_training_route_distance": float(
            np.max(PHASE40_ROUTE_DISTANCE_PRIVATE)
        ),
    },
    "deployment_policy": {
        "known_header_prototype": (
            "route_to_the_single_model_fold_that_held_out_its_"
            "acquisition_group"
        ),
        "unknown_header_prototype": "phase12c_anchor_only",
        "three_fold_residual_average": False,
        "phase12c_anchor": "complete_three_fold_ensemble",
        "generic_production_logging_only": True,
    },
    "groups": phase40_group_records,
    "planned_gate_candidate_count": int(
        len(PHASE40_CONFIG["phase36_weights"])
        * len(PHASE40_CONFIG["alphas"])
        * len(PHASE40_CONFIG["uncertainty_exponents"])
        * len(PHASE40_CONFIG["delta_caps"])
    ),
    "partition_source": phase40_partition_source,
    "case_dataframe_source": phase40_case_df_source,
    "path_source": phase40_path_source,
    "elapsed_seconds": round(
        time.perf_counter() - phase40_started, 2
    ),
    "training_nifti_headers_read": True,
    "training_voxel_arrays_read": False,
    "labels_used": False,
    "outer_validation_labels_used": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "model_hashes_displayed": False,
}

print("BEGIN SANITIZED_PHASE40_ROUTING_CONTRACT")
print(json.dumps(PHASE40_ROUTE_CONTRACT, indent=2))
print("END SANITIZED_PHASE40_ROUTING_CONTRACT")

BEGIN SANITIZED_PHASE40_ROUTING_CONTRACT
{
  "phase": "phase40_exact_header_routed_residual_contract",
  "status": "accepted",
  "case_count": 1362,
  "acquisition_group_count": 15,
  "fold_count": 3,
  "router": {
    "candidate_file_count": 4,
    "distinct_payload_count": 1,
    "prototype_count": 137,
    "header_feature_count": 10,
    "rounding_decimals": 6,
    "corrected_groups": [
      1,
      5,
      9
    ]
  },
  "route_parity": {
    "dataframe_group_to_oof_fold_alignment_fraction": 1.0,
    "header_to_dataframe_group_alignment_fraction": 1.0,
    "header_to_oof_fold_alignment_fraction": 1.0,
    "exact_prototype_match_fraction": 1.0,
    "maximum_training_route_distance": 0.0
  },
  "deployment_policy": {
    "known_header_prototype": "route_to_the_single_model_fold_that_held_out_its_acquisition_group",
    "unknown_header_prototype": "phase12c_anchor_only",
    "three_fold_residual_average": false,
    "phase12c_anchor": "complete_three_fold_ensemble",
    "generic_pr

In [108]:
# Phase40 Cell 132A
# Recover the exact held-out-fold component vectors and freeze Phase39's
# formula for a controlled routing-only experiment. No labels are read.

from pathlib import Path
import json
import time

import numpy as np


PHASE40_CASE_COUNT = 1362
PHASE40_EPSILON = 1e-5
phase40_132a_started = time.perf_counter()

assert PHASE40_ROUTE_CONTRACT["status"] == "accepted"
assert PHASE40_ROUTE_CONTRACT["route_parity"][
    "header_to_oof_fold_alignment_fraction"
] == 1.0


def phase40_probability_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        PHASE40_EPSILON,
        1.0 - PHASE40_EPSILON,
    )
    return np.log(probability) - np.log1p(-probability)


def phase40_probability_sigmoid(logit):
    logit = np.asarray(logit, dtype=np.float64)
    output = np.empty_like(logit)
    positive = logit >= 0
    output[positive] = 1.0 / (1.0 + np.exp(-logit[positive]))
    exponential = np.exp(logit[~positive])
    output[~positive] = exponential / (1.0 + exponential)
    return np.clip(
        output,
        PHASE40_EPSILON,
        1.0 - PHASE40_EPSILON,
    )


def phase40_vector(value, probability=False):
    try:
        array = np.asarray(value, dtype=np.float64)
    except Exception:
        return None
    if array.shape != (PHASE40_CASE_COUNT,):
        return None
    if not np.isfinite(array).all():
        return None
    if probability and not np.all(
        (array >= PHASE40_EPSILON)
        & (array <= 1.0 - PHASE40_EPSILON)
    ):
        return None
    return array


baseline_path = Path(
    "/kaggle/working/phase32_phase12c_oof_float64.npy"
)
assert baseline_path.is_file(), baseline_path
PHASE40_BASELINE_OOF_PRIVATE = phase40_vector(
    np.load(baseline_path, allow_pickle=False),
    probability=True,
)
assert PHASE40_BASELINE_OOF_PRIVATE is not None

PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE = None
phase40_phase33_source = None

for name in (
    "phase33_runtime_logit",
    "PHASE39_PHASE33_ROUTED_LOGIT_PRIVATE",
):
    candidate = phase40_vector(globals().get(name))
    if candidate is not None:
        PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE = candidate.copy()
        phase40_phase33_source = name
        break

if PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE is None:
    for name in (
        "PHASE33_OOF_PRIVATE",
        "phase33_oof",
    ):
        candidate = phase40_vector(
            globals().get(name), probability=True
        )
        if candidate is not None:
            PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE = (
                phase40_probability_logit(candidate)
            )
            phase40_phase33_source = f"logit({name})"
            break

assert PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE is not None, {
    "message": "Could not resolve the exact Phase33 routed OOF component.",
}

PHASE40_PHASE36_ROUTED_RAW_RESIDUAL_PRIVATE = None
phase40_phase36_source = None
for name in (
    "phase36_runtime_raw_residual",
    "PHASE36_RAW_RESIDUAL_OOF_PRIVATE",
    "phase36_raw_residual_oof",
):
    candidate = phase40_vector(globals().get(name))
    if candidate is not None:
        PHASE40_PHASE36_ROUTED_RAW_RESIDUAL_PRIVATE = candidate.copy()
        phase40_phase36_source = name
        break


assert PHASE40_PHASE36_ROUTED_RAW_RESIDUAL_PRIVATE is not None, {
    "message": "Could not resolve the exact Phase36 routed raw residual.",
}

PHASE40_PHASE39_REFERENCE_OOF_PRIVATE = None
phase40_phase39_source = None
for name in (
    "PHASE39_OOF_PRIVATE",
    "phase39_runtime_probability",
):
    candidate = phase40_vector(
        globals().get(name), probability=True
    )
    if candidate is not None:
        PHASE40_PHASE39_REFERENCE_OOF_PRIVATE = candidate.copy()
        phase40_phase39_source = name
        break

if PHASE40_PHASE39_REFERENCE_OOF_PRIVATE is None:
    for path in (
        Path(
            "/kaggle/working/phase39_private_checkpoint/"
            "phase39_oof_float64.npy"
        ),
        Path("/kaggle/working/phase39_oof_float64.npy"),
    ):
        if not path.is_file():
            continue
        candidate = phase40_vector(
            np.load(path, allow_pickle=False),
            probability=True,
        )
        if candidate is not None:
            PHASE40_PHASE39_REFERENCE_OOF_PRIVATE = candidate.copy()
            phase40_phase39_source = str(path)
            break

assert PHASE40_PHASE39_REFERENCE_OOF_PRIVATE is not None, {
    "message": "Could not resolve the accepted Phase39 OOF reference.",
}

# Freeze the Phase39 formula. Phase40 changes only fold routing, so a public
# score difference cannot be attributed to another hyperparameter search.
PHASE40_FROZEN_FORMULA_PRIVATE = {
    "phase36_weight": 0.75,
    "phase33_weight": 0.25,
    "alpha": 1.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 2.0,
}

phase40_baseline_logit = phase40_probability_logit(
    PHASE40_BASELINE_OOF_PRIVATE
)
phase40_phase33_residual = (
    PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE
    - phase40_baseline_logit
)
phase40_uncapped_combined_residual = (
    PHASE40_FROZEN_FORMULA_PRIVATE["phase36_weight"]
    * PHASE40_PHASE36_ROUTED_RAW_RESIDUAL_PRIVATE
    + PHASE40_FROZEN_FORMULA_PRIVATE["phase33_weight"]
    * phase40_phase33_residual
)
PHASE40_ROUTED_COMBINED_RESIDUAL_PRIVATE = np.clip(
    phase40_uncapped_combined_residual,
    -PHASE40_FROZEN_FORMULA_PRIVATE["delta_cap"],
    PHASE40_FROZEN_FORMULA_PRIVATE["delta_cap"],
)
phase40_reconstructed_probability = phase40_probability_sigmoid(
    phase40_baseline_logit
    + PHASE40_ROUTED_COMBINED_RESIDUAL_PRIVATE
)

phase40_reference_difference = np.abs(
    phase40_reconstructed_probability
    - PHASE40_PHASE39_REFERENCE_OOF_PRIVATE
)
phase40_reference_error = {
    "maximum": float(np.max(phase40_reference_difference)),
    "mean": float(np.mean(phase40_reference_difference)),
    "q99": float(np.quantile(phase40_reference_difference, 0.99)),
}
assert phase40_reference_error["maximum"] <= 5e-7, {
    "message": "Frozen Phase39 formula reconstruction failed.",
    "error": phase40_reference_error,
}

PHASE40_COMPONENT_CONTRACT = {
    "phase": "phase40_routed_component_state_contract",
    "status": "accepted",
    "case_count": PHASE40_CASE_COUNT,
    "formula": PHASE40_FROZEN_FORMULA_PRIVATE,
    "controlled_change": "three_fold_average_to_exact_header_routed_fold",
    "unchanged": [
        "phase12c_complete_three_fold_anchor",
        "phase33_models",
        "phase36_boosters",
        "p3ca_bases",
        "residual_formula",
    ],
    "sources": {
        "baseline": str(baseline_path),
        "phase33_routed_logit": phase40_phase33_source,
        "phase36_routed_raw_residual": phase40_phase36_source,
        "phase39_reference": phase40_phase39_source,
    },
    "formula_reconstruction_error": phase40_reference_error,
    "cap_activation_fraction": float(np.mean(
        np.abs(phase40_uncapped_combined_residual)
        > PHASE40_FROZEN_FORMULA_PRIVATE["delta_cap"]
    )),
    "all_values_finite": bool(
        np.isfinite(PHASE40_ROUTED_COMBINED_RESIDUAL_PRIVATE).all()
    ),
    "labels_used": False,
    "outer_validation_labels_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase40_132a_started, 3
    ),
}

print("BEGIN SANITIZED_PHASE40_COMPONENT_CONTRACT")
print(json.dumps(PHASE40_COMPONENT_CONTRACT, indent=2))
print("END SANITIZED_PHASE40_COMPONENT_CONTRACT")

BEGIN SANITIZED_PHASE40_COMPONENT_CONTRACT
{
  "phase": "phase40_routed_component_state_contract",
  "status": "accepted",
  "case_count": 1362,
  "formula": {
    "phase36_weight": 0.75,
    "phase33_weight": 0.25,
    "alpha": 1.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 2.0
  },
  "controlled_change": "three_fold_average_to_exact_header_routed_fold",
  "unchanged": [
    "phase12c_complete_three_fold_anchor",
    "phase33_models",
    "phase36_boosters",
    "p3ca_bases",
    "residual_formula"
  ],
  "sources": {
    "baseline": "/kaggle/working/phase32_phase12c_oof_float64.npy",
    "phase33_routed_logit": "phase33_runtime_logit",
    "phase36_routed_raw_residual": "phase36_runtime_raw_residual",
    "phase39_reference": "PHASE39_OOF_PRIVATE"
  },
  "formula_reconstruction_error": {
    "maximum": 2.5391631741644716e-08,
    "mean": 2.38062791421943e-09,
    "q99": 1.446864485332711e-08
  },
  "cap_activation_fraction": 0.010279001468428781,
  "all_values_finite": true,


In [109]:
# Phase40 Cell 132B
# Audit whether the old monitor predictions used the same fold route that
# Phase40 will deploy. No labels or predictions are displayed.

import json
import time

import numpy as np


phase40_132b_started = time.perf_counter()
assert PHASE40_COMPONENT_CONTRACT["status"] == "accepted"


def phase40_monitor_array(name, dictionary_key=None, dtype=None):
    value = globals().get(name)
    if value is None:
        pool = globals().get("PHASE37_MONITOR_POOL_PRIVATE")
        if isinstance(pool, dict) and dictionary_key in pool:
            value = pool[dictionary_key]
    assert value is not None, {
        "message": "Required Phase37 monitor state is missing.",
        "name": name,
        "dictionary_key": dictionary_key,
    }
    array = np.asarray(value, dtype=dtype)
    assert array.ndim == 1
    assert np.isfinite(array).all()
    return array


phase40_monitor_indices = phase40_monitor_array(
    "phase37_monitor_indices", "indices", np.int64
)
phase40_monitor_folds = phase40_monitor_array(
    "phase37_monitor_folds", "folds", np.int64
)
phase40_monitor_groups = phase40_monitor_array(
    "phase37_monitor_groups", "groups", np.int64
)

phase40_monitor_n = len(phase40_monitor_indices)
assert phase40_monitor_n > 0
assert phase40_monitor_folds.shape == (phase40_monitor_n,)
assert phase40_monitor_groups.shape == (phase40_monitor_n,)
assert np.all(
    (phase40_monitor_indices >= 0)
    & (phase40_monitor_indices < PHASE40_CASE_COUNT)
)
assert np.all(np.isin(phase40_monitor_folds, [0, 1, 2]))
assert np.all(
    (phase40_monitor_groups >= 0)
    & (phase40_monitor_groups < PHASE40_GROUP_COUNT)
)

phase40_deployment_folds_for_monitor = (
    PHASE40_ROUTER_PRIVATE["fold_for_group"][
        phase40_monitor_groups
    ]
)
phase40_monitor_route_match = (
    phase40_monitor_folds
    == phase40_deployment_folds_for_monitor
)
phase40_monitor_route_alignment = float(np.mean(
    phase40_monitor_route_match
))

phase40_monitor_multiplicity = np.bincount(
    phase40_monitor_indices,
    minlength=PHASE40_CASE_COUNT,
)
phase40_unique_monitor_indices = np.flatnonzero(
    phase40_monitor_multiplicity > 0
)

phase40_monitor_group_records = []
for group in sorted(np.unique(phase40_monitor_groups).tolist()):
    mask = phase40_monitor_groups == group
    phase40_monitor_group_records.append({
        "group": int(group),
        "observation_count": int(np.sum(mask)),
        "unique_case_count": int(len(np.unique(
            phase40_monitor_indices[mask]
        ))),
        "monitor_model_folds": sorted(
            int(value)
            for value in np.unique(
                phase40_monitor_folds[mask]
            ).tolist()
        ),
        "deployment_held_out_fold": int(
            PHASE40_ROUTER_PRIVATE["fold_for_group"][group]
        ),
        "route_consistent_observation_count": int(np.sum(
            phase40_monitor_route_match[mask]
        )),
    })

phase40_unrepresented_groups = sorted(
    set(range(PHASE40_GROUP_COUNT)).difference(
        set(np.unique(phase40_monitor_groups).tolist())
    )
)

# A hyperparameter search on a monitor model fold different from deployment
# would repeat Phase39's validation/deployment mismatch. Freeze the old formula
# and test routing alone instead.
PHASE40_ROUTE_ONLY_ABLATION_PRIVATE = True

PHASE40_MONITOR_ROUTE_AUDIT = {
    "phase": "phase40_monitor_deployment_route_audit",
    "status": "accepted_route_only_ablation_required",
    "monitor_observation_count": phase40_monitor_n,
    "unique_monitor_case_count": int(
        len(phase40_unique_monitor_indices)
    ),
    "maximum_case_multiplicity": int(
        np.max(phase40_monitor_multiplicity)
    ),
    "monitor_group_count": int(
        len(np.unique(phase40_monitor_groups))
    ),
    "unrepresented_acquisition_groups": (
        phase40_unrepresented_groups
    ),
    "monitor_model_fold_equals_deployment_fold_fraction": (
        phase40_monitor_route_alignment
    ),
    "route_consistent_monitor_observation_count": int(
        np.sum(phase40_monitor_route_match)
    ),
    "group_records": phase40_monitor_group_records,
    "decision": {
        "search_4620_gate_candidates_now": False,
        "reason": (
            "old monitor components were generated by nondeployment "
            "folds for the routed acquisition groups"
        ),
        "phase40_experiment": (
            "keep the Phase39 formula fixed and change only runtime "
            "fold aggregation to exact header routing"
        ),
        "future_gate_search_requirement": (
            "new group-held-out cross-fitted monitor components"
        ),
    },
    "labels_used": False,
    "outer_validation_labels_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_values_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase40_132b_started, 3
    ),
}

print("BEGIN SANITIZED_PHASE40_MONITOR_ROUTE_AUDIT")
print(json.dumps(PHASE40_MONITOR_ROUTE_AUDIT, indent=2))
print("END SANITIZED_PHASE40_MONITOR_ROUTE_AUDIT")


BEGIN SANITIZED_PHASE40_MONITOR_ROUTE_AUDIT
{
  "phase": "phase40_monitor_deployment_route_audit",
  "status": "accepted_route_only_ablation_required",
  "monitor_observation_count": 493,
  "unique_monitor_case_count": 407,
  "maximum_case_multiplicity": 2,
  "monitor_group_count": 10,
  "unrepresented_acquisition_groups": [
    1,
    3,
    7,
    8,
    9
  ],
  "monitor_model_fold_equals_deployment_fold_fraction": 0.0,
  "route_consistent_monitor_observation_count": 0,
  "group_records": [
    {
      "group": 0,
      "observation_count": 39,
      "unique_case_count": 39,
      "monitor_model_folds": [
        0
      ],
      "deployment_held_out_fold": 1,
      "route_consistent_observation_count": 0
    },
    {
      "group": 2,
      "observation_count": 145,
      "unique_case_count": 145,
      "monitor_model_folds": [
        2
      ],
      "deployment_held_out_fold": 1,
      "route_consistent_observation_count": 0
    },
    {
      "group": 4,
      "observation_coun

In [110]:
# Phase40 Cell 132C
# Exercise all packaged Phase33/Phase36 fold states, select exactly one fold
# per case using the header router, and prove parity with held-out-fold OOF.
# Reads private cached training features only; no labels are used.

from pathlib import Path
import gc
import json
import time

import numpy as np
import torch


PHASE40_PARITY_BATCH_SIZE = 48
phase40_132c_started = time.perf_counter()

assert PHASE40_MONITOR_ROUTE_AUDIT[
    "status"
] == "accepted_route_only_ablation_required"

required_names = (
    "PHASE39_RUNTIME_PRIVATE",
    "PHASE39_DEVICE",
    "phase39_runtime_module",
)
missing_names = [
    name for name in required_names if name not in globals()
]
assert not missing_names, {
    "message": "Accepted Phase39 runtime objects are missing.",
    "missing": missing_names,
}

phase40_runtime = PHASE39_RUNTIME_PRIVATE
phase40_module = phase39_runtime_module

dense_path = Path(
    "/kaggle/working/phase32_bilateral_dense_float16.npy"
)
radiomics_path = Path(
    "/kaggle/working/phase31_radiomics_float32.npy"
)
geometry_path = Path(
    "/kaggle/working/phase31_sequence_float32.npy"
)
assert dense_path.is_file()
assert radiomics_path.is_file()
assert geometry_path.is_file()

phase40_dense_cache = np.load(
    dense_path, mmap_mode="r", allow_pickle=False
)
phase40_radiomics_cache = np.load(
    radiomics_path, mmap_mode="r", allow_pickle=False
)
phase40_geometry_cache = np.load(
    geometry_path, mmap_mode="r", allow_pickle=False
)
assert phase40_dense_cache.shape == (
    PHASE40_CASE_COUNT, 6, 14, 14, 384
)
assert phase40_radiomics_cache.shape == (
    PHASE40_CASE_COUNT, 1081
)
assert phase40_geometry_cache.shape == (
    PHASE40_CASE_COUNT, 16, 52
)


@torch.inference_mode()
def phase40_predict_routed_dense(
    runtime,
    dense_features,
    radiomics,
    geometry_sequence,
    baseline_probability,
    route_folds,
    route_exact,
):
    baseline_probability = np.clip(
        np.asarray(baseline_probability, dtype=np.float64),
        PHASE40_EPSILON,
        1.0 - PHASE40_EPSILON,
    )
    route_folds = np.asarray(route_folds, dtype=np.int64)
    route_exact = np.asarray(route_exact, dtype=bool)
    batch_size = len(baseline_probability)
    assert route_folds.shape == (batch_size,)
    assert route_exact.shape == (batch_size,)
    assert np.all(np.isin(route_folds, [0, 1, 2]))

    fold_phase33_logits = []
    fold_phase36_residuals = []

    for fold in range(3):
        summary, p3ca_sequence = runtime.project_p3ca(
            dense_features, fold
        )

        transformed33 = phase40_module.transform_fold_features(
            runtime.phase33_preprocessors,
            fold,
            p3ca_sequence,
            geometry_sequence,
            summary,
            radiomics,
        )
        arguments = {
            key: torch.from_numpy(value).to(PHASE39_DEVICE)
            for key, value in transformed33.items()
        }
        seed_logits = []
        for model in runtime.phase33_models[fold]:
            output = model(
                p3ca_sequence=arguments["p3ca_sequence"],
                geometry_sequence=arguments["geometry_sequence"],
                summary_features=arguments["summary_features"],
                radiomics_features=arguments["radiomics_features"],
                domain_coefficient=0.0,
            )
            seed_logits.append(
                output["logit"].float().cpu().numpy()
            )
        fold_phase33_logits.append(np.mean(
            np.stack(seed_logits, axis=0), axis=0
        ).astype(np.float64))

        transformed36 = phase40_module.transform_fold_features(
            runtime.phase36_preprocessors,
            fold,
            p3ca_sequence,
            geometry_sequence,
            summary,
            radiomics,
        )
        features36, baseline_margin = (
            phase40_module.phase36_build_features(
                transformed36, baseline_probability
            )
        )
        seed_residuals = []
        for record in runtime.phase36_boosters[fold]:
            seed_residuals.append(
                phase40_module.phase36_predict_residual(
                    record["booster"],
                    features36,
                    baseline_margin,
                    record["round_count"],
                )
            )
        fold_phase36_residuals.append(np.mean(
            np.stack(seed_residuals, axis=0), axis=0
        ).astype(np.float64))

    phase33_matrix = np.stack(
        fold_phase33_logits, axis=0
    )
    phase36_matrix = np.stack(
        fold_phase36_residuals, axis=0
    )
    column = np.arange(batch_size, dtype=np.int64)
    selected_phase33_logit = phase33_matrix[
        route_folds, column
    ]
    selected_phase36_residual = phase36_matrix[
        route_folds, column
    ]

    baseline_logit = phase40_probability_logit(
        baseline_probability
    )
    phase33_residual = selected_phase33_logit - baseline_logit
    formula = PHASE40_FROZEN_FORMULA_PRIVATE
    uncapped = (
        formula["phase36_weight"] * selected_phase36_residual
        + formula["phase33_weight"] * phase33_residual
    )
    bounded = np.clip(
        uncapped, -formula["delta_cap"], formula["delta_cap"]
    )
    uncertainty = (
        4.0 * baseline_probability * (1.0 - baseline_probability)
    ) ** formula["uncertainty_exponent"]
    update = formula["alpha"] * uncertainty * bounded
    update = np.where(route_exact, update, 0.0)
    probability = phase40_probability_sigmoid(
        baseline_logit + update
    )
    return probability, {
        "phase33_logit": selected_phase33_logit,
        "phase36_raw_residual": selected_phase36_residual,
        "uncapped_combined_residual": uncapped,
        "bounded_combined_residual": bounded,
        "applied_update": update,
    }


PHASE40_RUNTIME_ROUTED_OOF_PRIVATE = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)
phase40_runtime_phase33_logit = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)
phase40_runtime_phase36_residual = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)
phase40_runtime_combined_residual = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)

for start in range(
    0, PHASE40_CASE_COUNT, PHASE40_PARITY_BATCH_SIZE
):
    stop = min(
        start + PHASE40_PARITY_BATCH_SIZE,
        PHASE40_CASE_COUNT,
    )
    indices = np.arange(start, stop, dtype=np.int64)
    probability, details = phase40_predict_routed_dense(
        phase40_runtime,
        np.asarray(phase40_dense_cache[indices], dtype=np.float32),
        np.asarray(phase40_radiomics_cache[indices], dtype=np.float32),
        np.asarray(phase40_geometry_cache[indices], dtype=np.float32),
        PHASE40_BASELINE_OOF_PRIVATE[indices],
        PHASE40_ROUTED_FOLD_PRIVATE[indices],
        PHASE40_ROUTE_EXACT_PRIVATE[indices],
    )
    PHASE40_RUNTIME_ROUTED_OOF_PRIVATE[indices] = probability
    phase40_runtime_phase33_logit[indices] = details[
        "phase33_logit"
    ]
    phase40_runtime_phase36_residual[indices] = details[
        "phase36_raw_residual"
    ]
    phase40_runtime_combined_residual[indices] = details[
        "bounded_combined_residual"
    ]
    if stop % 192 == 0 or stop == PHASE40_CASE_COUNT:
        print(
            f"Phase40 routed parity: {stop}/{PHASE40_CASE_COUNT}"
        )


def phase40_error_summary(actual, reference):
    difference = np.abs(
        np.asarray(actual, dtype=np.float64)
        - np.asarray(reference, dtype=np.float64)
    )
    return {
        "maximum": float(np.max(difference)),
        "mean": float(np.mean(difference)),
        "q99": float(np.quantile(difference, 0.99)),
    }


phase40_phase33_error = phase40_error_summary(
    phase40_runtime_phase33_logit,
    PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE,
)
phase40_phase36_error = phase40_error_summary(
    phase40_runtime_phase36_residual,
    PHASE40_PHASE36_ROUTED_RAW_RESIDUAL_PRIVATE,
)
phase40_combined_error = phase40_error_summary(
    phase40_runtime_combined_residual,
    PHASE40_ROUTED_COMBINED_RESIDUAL_PRIVATE,
)
phase40_probability_error = phase40_error_summary(
    PHASE40_RUNTIME_ROUTED_OOF_PRIVATE,
    PHASE40_PHASE39_REFERENCE_OOF_PRIVATE,
)

assert phase40_phase33_error["maximum"] <= 5e-5
assert phase40_phase36_error["maximum"] <= 5e-5
assert phase40_combined_error["maximum"] <= 5e-5
assert phase40_probability_error["maximum"] <= 5e-6

PHASE40_ROUTED_RUNTIME_PARITY = {
    "phase": "phase40_exact_header_routed_runtime_parity",
    "status": "accepted",
    "case_count": PHASE40_CASE_COUNT,
    "route": {
        "exact_prototype_fraction": float(np.mean(
            PHASE40_ROUTE_EXACT_PRIVATE
        )),
        "fold_route_counts": {
            str(fold): int(np.sum(
                PHASE40_ROUTED_FOLD_PRIVATE == fold
            ))
            for fold in range(3)
        },
        "three_fold_residual_average_used": False,
        "unknown_prototype_policy_exercised": False,
    },
    "component_errors": {
        "phase33_logit": phase40_phase33_error,
        "phase36_raw_residual": phase40_phase36_error,
        "bounded_combined_residual": phase40_combined_error,
        "final_probability": phase40_probability_error,
    },
    "formula": PHASE40_FROZEN_FORMULA_PRIVATE,
    "phase12c_anchor": "exact_cross_fitted_oof_for_parity",
    "deployment_anchor_planned": "complete_three_fold_ensemble",
    "all_values_finite": bool(
        np.isfinite(PHASE40_RUNTIME_ROUTED_OOF_PRIVATE).all()
    ),
    "labels_used": False,
    "outer_validation_labels_used": False,
    "training_cached_features_read": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "peak_vram_mb": (
        float(torch.cuda.max_memory_allocated(PHASE39_DEVICE))
        / (1024.0 ** 2)
        if torch.cuda.is_available()
        else 0.0
    ),
    "elapsed_seconds": round(
        time.perf_counter() - phase40_132c_started, 3
    ),
}

print("BEGIN SANITIZED_PHASE40_ROUTED_RUNTIME_PARITY")
print(json.dumps(PHASE40_ROUTED_RUNTIME_PARITY, indent=2))
print("END SANITIZED_PHASE40_ROUTED_RUNTIME_PARITY")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Phase40 routed parity: 192/1362
Phase40 routed parity: 384/1362
Phase40 routed parity: 576/1362
Phase40 routed parity: 768/1362
Phase40 routed parity: 960/1362
Phase40 routed parity: 1152/1362
Phase40 routed parity: 1344/1362
Phase40 routed parity: 1362/1362
BEGIN SANITIZED_PHASE40_ROUTED_RUNTIME_PARITY
{
  "phase": "phase40_exact_header_routed_runtime_parity",
  "status": "accepted",
  "case_count": 1362,
  "route": {
    "exact_prototype_fraction": 1.0,
    "fold_route_counts": {
      "0": 467,
      "1": 443,
      "2": 452
    },
    "three_fold_residual_average_used": false,
    "unknown_prototype_policy_exercised": false
  },
  "component_errors": {
    "phase33_logit": {
      "maximum": 9.5367431640625e-07,
      "mean": 4.737300144593271e-09,
      "q99": 2.384185791015625e-07
    },
    "phase36_raw_residual": {
      "maximum": 0.0,
      "mean": 0.0,
      "q99": 0.0
    },
    "bounded_combined_residual": {
      "maximum": 2.384185791015625e-07,
      "mean": 1.184325036

In [111]:
# Phase40 Cell 133A
# Build a separate route-aware staging directory from the accepted Phase39
# archive. The accepted Phase39 directory and ZIP are not modified.

from pathlib import Path
import hashlib
import json
import os
import py_compile
import tempfile
import time
import zipfile


PHASE40_RUNTIME_SOURCE = 'from __future__ import annotations\n\nimport numpy as np\nimport torch\n\nfrom phase39_runtime import (\n    PHASE36_CONFIG,\n    PROBABILITY_EPSILON,\n    Phase39Runtime,\n    phase36_build_features,\n    phase36_predict_residual,\n    probability_logit,\n    probability_sigmoid,\n    transform_fold_features,\n)\n\n\nclass Phase40Runtime(Phase39Runtime):\n    """Phase39 components with exact header-routed fold selection."""\n\n    @torch.inference_mode()\n    def predict_routed_from_dense(\n        self,\n        dense_features,\n        radiomics,\n        geometry_sequence,\n        baseline_probability,\n        route_folds,\n        route_exact,\n    ):\n        baseline_probability = np.clip(\n            np.asarray(baseline_probability, dtype=np.float64),\n            PROBABILITY_EPSILON,\n            1.0 - PROBABILITY_EPSILON,\n        )\n        dense_features = np.asarray(dense_features, dtype=np.float32)\n        radiomics = np.asarray(radiomics, dtype=np.float32)\n        geometry_sequence = np.asarray(\n            geometry_sequence, dtype=np.float32\n        )\n        route_folds = np.asarray(route_folds, dtype=np.int64)\n        route_exact = np.asarray(route_exact, dtype=bool)\n\n        batch_size = len(baseline_probability)\n        if dense_features.shape != (\n            batch_size, 6, 14, 14, 384\n        ):\n            raise RuntimeError("Phase40 dense batch mismatch")\n        if radiomics.shape != (batch_size, 1081):\n            raise RuntimeError("Phase40 radiomics batch mismatch")\n        if geometry_sequence.shape != (batch_size, 16, 52):\n            raise RuntimeError("Phase40 geometry batch mismatch")\n        if route_folds.shape != (batch_size,):\n            raise RuntimeError("Phase40 fold route mismatch")\n        if route_exact.shape != (batch_size,):\n            raise RuntimeError("Phase40 route mask mismatch")\n        if not np.all(np.isin(route_folds, [0, 1, 2])):\n            raise RuntimeError("Invalid Phase40 fold route")\n\n        baseline_logit = probability_logit(baseline_probability)\n        phase33_component_logit = baseline_logit.copy()\n        raw_phase36_residual = np.zeros(\n            batch_size, dtype=np.float64\n        )\n\n        # Compute only the one fold assigned to each known header prototype.\n        # Unknown prototypes never enter a residual model.\n        for fold in range(3):\n            local = np.flatnonzero(\n                route_exact & (route_folds == fold)\n            )\n            if local.size == 0:\n                continue\n\n            summary, p3ca_sequence = self.project_p3ca(\n                dense_features[local], fold\n            )\n            transformed33 = transform_fold_features(\n                self.phase33_preprocessors,\n                fold,\n                p3ca_sequence,\n                geometry_sequence[local],\n                summary,\n                radiomics[local],\n            )\n            arguments = {\n                key: torch.from_numpy(value).to(self.device)\n                for key, value in transformed33.items()\n            }\n            seed_logits = []\n            for model in self.phase33_models[fold]:\n                output = model(\n                    p3ca_sequence=arguments["p3ca_sequence"],\n                    geometry_sequence=arguments[\n                        "geometry_sequence"\n                    ],\n                    summary_features=arguments["summary_features"],\n                    radiomics_features=arguments[\n                        "radiomics_features"\n                    ],\n                    domain_coefficient=0.0,\n                )\n                seed_logits.append(\n                    output["logit"].float().cpu().numpy()\n                )\n            phase33_component_logit[local] = np.mean(\n                np.stack(seed_logits, axis=0), axis=0\n            ).astype(np.float64)\n\n            transformed36 = transform_fold_features(\n                self.phase36_preprocessors,\n                fold,\n                p3ca_sequence,\n                geometry_sequence[local],\n                summary,\n                radiomics[local],\n            )\n            features36, baseline_margin = phase36_build_features(\n                transformed36, baseline_probability[local]\n            )\n            seed_residuals = []\n            for record in self.phase36_boosters[fold]:\n                seed_residuals.append(\n                    phase36_predict_residual(\n                        record["booster"],\n                        features36,\n                        baseline_margin,\n                        record["round_count"],\n                    )\n                )\n            raw_phase36_residual[local] = np.mean(\n                np.stack(seed_residuals, axis=0), axis=0\n            ).astype(np.float64)\n\n        phase33_residual = (\n            phase33_component_logit - baseline_logit\n        )\n        combined_uncapped = (\n            float(self.formula["phase36_weight"])\n            * raw_phase36_residual\n            + float(self.formula["phase33_weight"])\n            * phase33_residual\n        )\n        combined_bounded = np.clip(\n            combined_uncapped,\n            -float(self.formula["delta_cap"]),\n            float(self.formula["delta_cap"]),\n        )\n        uncertainty = (\n            4.0\n            * baseline_probability\n            * (1.0 - baseline_probability)\n        ) ** float(self.formula["uncertainty_exponent"])\n        applied_update = (\n            float(self.formula["alpha"])\n            * uncertainty\n            * combined_bounded\n        )\n        applied_update = np.where(\n            route_exact, applied_update, 0.0\n        )\n        probability = np.clip(\n            probability_sigmoid(\n                baseline_logit + applied_update\n            ),\n            PROBABILITY_EPSILON,\n            1.0 - PROBABILITY_EPSILON,\n        )\n        if not np.isfinite(probability).all():\n            raise RuntimeError("Non-finite Phase40 probability")\n\n        return probability, {\n            "phase33_component_logit": phase33_component_logit,\n            "raw_phase36_residual": raw_phase36_residual,\n            "phase33_residual": phase33_residual,\n            "combined_uncapped": combined_uncapped,\n            "combined_bounded": combined_bounded,\n            "applied_update": applied_update,\n            "route_folds": route_folds,\n            "route_exact": route_exact,\n        }\n\n    def predict_routed(\n        self,\n        highres_volumes,\n        baseline_probability,\n        backbone,\n        route_folds,\n        route_exact,\n    ):\n        # Reproduce Phase31/32 float16 training cache boundaries.\n        highres = np.asarray(\n            highres_volumes, dtype=np.float32\n        ).astype(np.float16).astype(np.float32)\n        radiomics, geometry = self.extract_geometry(highres)\n        dense = self.extract_dense_tokens(highres, backbone)\n        return self.predict_routed_from_dense(\n            dense,\n            radiomics,\n            geometry,\n            baseline_probability,\n            route_folds,\n            route_exact,\n        )\n\n'
PHASE40_MAIN_SOURCE = 'from __future__ import annotations\n\nimport os\nimport warnings\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nwarnings.filterwarnings(\n    "ignore",\n    message="enable_nested_tensor is True.*",\n    category=UserWarning,\n)\n\nimport numpy as np\nimport torch\n\nimport phase30_accepted_main as legacy\nfrom phase40_runtime import Phase40Runtime\n\n\nDATA_ROOT = Path(\n    os.environ.get("DAT_DATA_ROOT", "/code_execution/data")\n)\nOUTPUT_CSV = Path(\n    os.environ.get(\n        "DAT_OUTPUT_CSV", "/code_execution/submission.csv"\n    )\n)\n\n\ndef exact_header_route(path, router):\n    try:\n        features = legacy.header_features(path)\n        vector = np.asarray(\n            [features[column] for column in router["header_columns"]],\n            dtype=np.float64,\n        )\n        if not np.isfinite(vector).all():\n            return -1, 0, False\n        rounded = np.round(\n            vector, decimals=router["rounding_decimals"]\n        )\n        exact = np.flatnonzero(np.all(\n            router["prototypes"] == rounded[None, :], axis=1\n        ))\n        if exact.size == 0:\n            return -1, 0, False\n        groups = np.unique(\n            router["prototype_groups"][exact]\n        ).astype(np.int64)\n        if groups.size != 1:\n            return -1, 0, False\n        group = int(groups[0])\n        if group < 0 or group >= len(router["fold_for_group"]):\n            return -1, 0, False\n        fold = int(router["fold_for_group"][group])\n        if fold not in (0, 1, 2):\n            return -1, 0, False\n        return group, fold, True\n    except Exception:\n        return -1, 0, False\n\n\ndef prepare_case(path, crop_config, router):\n    group, fold, exact = exact_header_route(path, router)\n    crops = legacy.base.preprocess_safely(path, crop_config)\n    return crops, group, fold, exact\n\n\ndef predict_phase40_safe(\n    runtime,\n    backbone,\n    highres_volumes,\n    baseline_probability,\n    route_folds,\n    route_exact,\n):\n    baseline_probability = np.asarray(\n        baseline_probability, dtype=np.float64\n    )\n    route_folds = np.asarray(route_folds, dtype=np.int64)\n    route_exact = np.asarray(route_exact, dtype=bool)\n    probability = baseline_probability.copy()\n    failure_count = 0\n\n    try:\n        probability[:] = runtime.predict_routed(\n            highres_volumes,\n            baseline_probability,\n            backbone,\n            route_folds,\n            route_exact,\n        )[0]\n    except Exception:\n        for index in range(len(baseline_probability)):\n            try:\n                probability[index] = float(\n                    runtime.predict_routed(\n                        [highres_volumes[index]],\n                        baseline_probability[index:index + 1],\n                        backbone,\n                        route_folds[index:index + 1],\n                        route_exact[index:index + 1],\n                    )[0][0]\n                )\n            except Exception:\n                probability[index] = float(\n                    baseline_probability[index]\n                )\n                failure_count += 1\n    return probability, failure_count\n\n\ndef main():\n    if torch.cuda.is_available():\n        torch.backends.cuda.enable_flash_sdp(False)\n        torch.backends.cuda.enable_mem_efficient_sdp(False)\n        torch.backends.cuda.enable_math_sdp(True)\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n    runtime_manifest = legacy.load_runtime_manifest()\n    legacy.verify_runtime_assets(runtime_manifest)\n    router = legacy.load_router(runtime_manifest)\n\n    (\n        base_manifest,\n        base_models,\n        family_manifest,\n        anatomy_models,\n        device,\n    ) = legacy.load_phase12c_bundle()\n\n    epsilon = float(family_manifest["probability_epsilon"])\n    fallback = float(np.clip(\n        family_manifest["fallback_probability"],\n        epsilon,\n        1.0 - epsilon,\n    ))\n\n    legacy.DATA_ROOT = DATA_ROOT\n    legacy.OUTPUT_CSV = OUTPUT_CSV\n    legacy.base.DATA_ROOT = DATA_ROOT\n    legacy.base.IMAGE_DIR = DATA_ROOT / "niftis"\n    legacy.base.SUBMISSION_FORMAT = (\n        DATA_ROOT / "submission_format.csv"\n    )\n    legacy.base.OUTPUT_CSV = OUTPUT_CSV\n\n    uids, paths = legacy.base.read_submission_contract()\n    probabilities = np.full(\n        len(uids), fallback, dtype=np.float64\n    )\n    workers = max(1, min(\n        8, int(os.environ.get("DAT_PREPROCESS_WORKERS", "6"))\n    ))\n    batch_size = max(1, min(\n        12, int(os.environ.get("DAT_BATCH_SIZE", "6"))\n    ))\n\n    runtime = None\n    backbone = None\n    initialization_fallbacks = 0\n    try:\n        runtime = Phase40Runtime(device)\n        backbone = legacy.load_dinov3(runtime_manifest, device)\n    except Exception:\n        initialization_fallbacks = 1\n\n    preprocessing_failures = 0\n    route_unknowns = 0\n    case_fallbacks = 0\n\n    print("Built with DINOv3.", flush=True)\n    print("Initialization complete.", flush=True)\n    print("Inference started.", flush=True)\n\n    crop_config = base_manifest["preprocessing"]["crops"]\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for start in range(0, len(uids), batch_size):\n            stop = min(start + batch_size, len(uids))\n            prepared = list(executor.map(\n                lambda path: prepare_case(\n                    path, crop_config, router\n                ),\n                paths[start:stop],\n            ))\n            valid_local = [\n                index\n                for index, item in enumerate(prepared)\n                if item[0] is not None\n            ]\n            preprocessing_failures += (\n                len(prepared) - len(valid_local)\n            )\n            if not valid_local:\n                continue\n\n            crop_batch = {\n                crop_name: [\n                    prepared[index][0][crop_name]\n                    for index in valid_local\n                ]\n                for crop_name in ("160", "192", "128hr")\n            }\n            route_folds = np.asarray(\n                [prepared[index][2] for index in valid_local],\n                dtype=np.int64,\n            )\n            route_exact = np.asarray(\n                [prepared[index][3] for index in valid_local],\n                dtype=bool,\n            )\n            route_unknowns += int(np.sum(~route_exact))\n\n            full_baseline = legacy.predict_phase12c_safe(\n                crops=crop_batch,\n                selected_family_manifest=family_manifest,\n                selected_base_manifest=base_manifest,\n                base_models=base_models,\n                anatomy_models=anatomy_models,\n                device=device,\n                fallback=fallback,\n            )\n            final_batch = full_baseline.copy()\n            if runtime is not None and backbone is not None:\n                final_batch, failures = predict_phase40_safe(\n                    runtime,\n                    backbone,\n                    crop_batch["128hr"],\n                    full_baseline,\n                    route_folds,\n                    route_exact,\n                )\n                case_fallbacks += int(failures)\n            else:\n                case_fallbacks += len(valid_local)\n\n            for local_index, probability in zip(\n                valid_local, final_batch\n            ):\n                probabilities[start + local_index] = float(probability)\n\n    values = np.asarray(probabilities, dtype=np.float64)\n    if not np.isfinite(values).all():\n        raise RuntimeError("Non-finite output probability")\n    values = np.clip(values, epsilon, 1.0 - epsilon)\n    legacy.base.write_submission(uids, values)\n\n    if os.environ.get("DAT_PRIVATE_DIAGNOSTICS") == "1":\n        print(\n            "Private diagnostics: "\n            f"preprocessing={preprocessing_failures}, "\n            f"unknown_routes={route_unknowns}, "\n            f"initialization={initialization_fallbacks}, "\n            f"case_fallbacks={case_fallbacks}.",\n            flush=True,\n        )\n    print("Inference completed.", flush=True)\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except Exception:\n        print("Execution failed.", flush=True)\n        raise SystemExit(1) from None\n\n'

PHASE40_BASE_ARCHIVE = Path(
    "/kaggle/working/phase39_submission.zip"
)
PHASE40_STAGE_DIRECTORY = Path(
    "/kaggle/working/phase40_experimental_submission"
)
PHASE40_SOURCE_CONTRACT_PATH = (
    PHASE40_STAGE_DIRECTORY / "phase40_runtime_contract.json"
)


def phase40_safe_extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=False)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert (
                target == destination_resolved
                or destination_resolved in target.parents
            ), {
                "message": "Unsafe archive member.",
                "member": member.filename,
            }
        archive.extractall(destination)


def phase40_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


phase40_133a_started = time.perf_counter()
assert PHASE40_ROUTED_RUNTIME_PARITY["status"] == "accepted"
assert PHASE40_BASE_ARCHIVE.is_file(), PHASE40_BASE_ARCHIVE

if not PHASE40_STAGE_DIRECTORY.exists():
    temporary_directory = Path(tempfile.mkdtemp(
        prefix="phase40_stage_",
        dir="/kaggle/working",
    ))
    # tempfile created the directory; extraction requires a new child.
    extracted = temporary_directory / "package"
    phase40_safe_extract(PHASE40_BASE_ARCHIVE, extracted)
    os.replace(extracted, PHASE40_STAGE_DIRECTORY)
else:
    assert PHASE40_STAGE_DIRECTORY.is_dir()
    assert (PHASE40_STAGE_DIRECTORY / "phase39_runtime.py").is_file()
    assert (PHASE40_STAGE_DIRECTORY / "phase39_assets").is_dir()

assert (PHASE40_STAGE_DIRECTORY / "main.py").is_file()
assert (PHASE40_STAGE_DIRECTORY / "phase39_runtime.py").is_file()
assert (PHASE40_STAGE_DIRECTORY / "phase39_assets").is_dir()

(PHASE40_STAGE_DIRECTORY / "phase40_runtime.py").write_text(
    PHASE40_RUNTIME_SOURCE,
    encoding="utf-8",
)
(PHASE40_STAGE_DIRECTORY / "main.py").write_text(
    PHASE40_MAIN_SOURCE,
    encoding="utf-8",
)

py_compile.compile(
    str(PHASE40_STAGE_DIRECTORY / "phase40_runtime.py"),
    doraise=True,
)
py_compile.compile(
    str(PHASE40_STAGE_DIRECTORY / "main.py"),
    doraise=True,
)

phase40_source_contract = {
    "schema_version": 1,
    "phase": "phase40_exact_header_routed_runtime",
    "base_archive": PHASE40_BASE_ARCHIVE.name,
    "runtime_entrypoint": "phase40_runtime.py",
    "formula": PHASE40_FROZEN_FORMULA_PRIVATE,
    "known_prototype_policy": (
        "single_acquisition_group_held_out_fold"
    ),
    "unknown_prototype_policy": "phase12c_anchor_only",
    "residual_fold_aggregation": "single_routed_fold",
    "production_logging": [
        "Built with DINOv3.",
        "Initialization complete.",
        "Inference started.",
        "Inference completed.",
    ],
    "private_diagnostic_environment_variable": (
        "DAT_PRIVATE_DIAGNOSTICS"
    ),
}
PHASE40_SOURCE_CONTRACT_PATH.write_text(
    json.dumps(phase40_source_contract, indent=2) + "\n",
    encoding="utf-8",
)

phase40_files = sorted(
    path
    for path in PHASE40_STAGE_DIRECTORY.rglob("*")
    if path.is_file()
)
phase40_runtime_text = (
    PHASE40_STAGE_DIRECTORY / "phase40_runtime.py"
).read_text(encoding="utf-8")
phase40_main_text = (
    PHASE40_STAGE_DIRECTORY / "main.py"
).read_text(encoding="utf-8")

assert "class Phase40Runtime" in phase40_runtime_text
assert "predict_routed_from_dense" in phase40_runtime_text
assert "three_fold" not in phase40_runtime_text.casefold()
assert 'print("Progress:' not in phase40_main_text
assert 'print("Inference completed: ' not in phase40_main_text
assert 'print("Inference completed."' in phase40_main_text
assert "Execution failed: " not in phase40_main_text

PHASE40_STAGING_CONTRACT = {
    "phase": "phase40_route_aware_runtime_staging",
    "status": "accepted",
    "base_archive": PHASE40_BASE_ARCHIVE.name,
    "staging_directory": PHASE40_STAGE_DIRECTORY.name,
    "package_file_count": len(phase40_files),
    "package_size_mb": round(sum(
        path.stat().st_size for path in phase40_files
    ) / (1024.0 ** 2), 3),
    "runtime_source_line_count": len(
        phase40_runtime_text.splitlines()
    ),
    "main_source_line_count": len(
        phase40_main_text.splitlines()
    ),
    "phase39_runtime_preserved": bool(
        (PHASE40_STAGE_DIRECTORY / "phase39_runtime.py").is_file()
    ),
    "phase40_runtime_written": True,
    "main_entrypoint_replaced_in_new_staging_only": True,
    "accepted_phase39_archive_modified": False,
    "accepted_phase39_staging_modified": False,
    "route_policy": {
        "known_prototype": "single_held_out_fold",
        "unknown_prototype": "phase12c_anchor_only",
        "three_fold_residual_average": False,
    },
    "logging_policy": {
        "progress_lines_present": False,
        "numeric_completion_line_present": False,
        "detailed_exception_line_present": False,
        "generic_completion_line_present": True,
    },
    "source_hashes_verified_privately": bool(
        phase40_sha256(
            PHASE40_STAGE_DIRECTORY / "phase40_runtime.py"
        )
        and phase40_sha256(
            PHASE40_STAGE_DIRECTORY / "main.py"
        )
    ),
    "model_hashes_displayed": False,
    "challenge_voxel_data_read": False,
    "training_voxel_data_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase40_133a_started, 3
    ),
}

print("BEGIN SANITIZED_PHASE40_STAGING")
print(json.dumps(PHASE40_STAGING_CONTRACT, indent=2))
print("END SANITIZED_PHASE40_STAGING")
print(PHASE40_STAGE_DIRECTORY)

BEGIN SANITIZED_PHASE40_STAGING
{
  "phase": "phase40_route_aware_runtime_staging",
  "status": "accepted",
  "base_archive": "phase39_submission.zip",
  "staging_directory": "phase40_experimental_submission",
  "package_file_count": 91,
  "package_size_mb": 193.319,
  "runtime_source_line_count": 205,
  "main_source_line_count": 267,
  "phase39_runtime_preserved": true,
  "phase40_runtime_written": true,
  "main_entrypoint_replaced_in_new_staging_only": true,
  "accepted_phase39_archive_modified": false,
  "accepted_phase39_staging_modified": false,
  "route_policy": {
    "known_prototype": "single_held_out_fold",
    "unknown_prototype": "phase12c_anchor_only",
    "three_fold_residual_average": false
  },
  "logging_policy": {
    "progress_lines_present": false,
    "numeric_completion_line_present": false,
    "detailed_exception_line_present": false,
    "generic_completion_line_present": true
  },
  "source_hashes_verified_privately": true,
  "model_hashes_displayed": false,
  

In [112]:
# Phase40 Cell 133B
# Import the staged Phase40 source, initialize all residual assets, and test
# known/unknown routes against the already accepted independent implementation.

import importlib
import importlib.util
import json
import sys
import time

import numpy as np


phase40_133b_started = time.perf_counter()
assert PHASE40_STAGING_CONTRACT["status"] == "accepted"

if "phase39_runtime" not in sys.modules:
    sys.path.insert(0, str(PHASE40_STAGE_DIRECTORY))
    importlib.import_module("phase39_runtime")

phase40_runtime_source_path = (
    PHASE40_STAGE_DIRECTORY / "phase40_runtime.py"
)
specification = importlib.util.spec_from_file_location(
    "phase40_staged_runtime_validation",
    phase40_runtime_source_path,
)
assert specification is not None
assert specification.loader is not None
phase40_staged_module = importlib.util.module_from_spec(
    specification
)
sys.modules[specification.name] = phase40_staged_module
specification.loader.exec_module(phase40_staged_module)

PHASE40_STAGED_RUNTIME_PRIVATE = (
    phase40_staged_module.Phase40Runtime(PHASE39_DEVICE)
)
PHASE40_STAGED_RUNTIME_MODULE_PRIVATE = phase40_staged_module

phase40_audit_indices = np.asarray(
    [0, 467, 910, 1], dtype=np.int64
)
phase40_audit_dense = np.asarray(
    phase40_dense_cache[phase40_audit_indices],
    dtype=np.float32,
)
phase40_audit_radiomics = np.asarray(
    phase40_radiomics_cache[phase40_audit_indices],
    dtype=np.float32,
)
phase40_audit_geometry = np.asarray(
    phase40_geometry_cache[phase40_audit_indices],
    dtype=np.float32,
)
phase40_audit_baseline = PHASE40_BASELINE_OOF_PRIVATE[
    phase40_audit_indices
]
phase40_audit_folds = PHASE40_ROUTED_FOLD_PRIVATE[
    phase40_audit_indices
].copy()
phase40_audit_exact = np.asarray(
    [True, True, True, False], dtype=bool
)

phase40_staged_probability, phase40_staged_details = (
    PHASE40_STAGED_RUNTIME_PRIVATE.predict_routed_from_dense(
        phase40_audit_dense,
        phase40_audit_radiomics,
        phase40_audit_geometry,
        phase40_audit_baseline,
        phase40_audit_folds,
        phase40_audit_exact,
    )
)
phase40_reference_probability, phase40_reference_details = (
    phase40_predict_routed_dense(
        PHASE39_RUNTIME_PRIVATE,
        phase40_audit_dense,
        phase40_audit_radiomics,
        phase40_audit_geometry,
        phase40_audit_baseline,
        phase40_audit_folds,
        phase40_audit_exact,
    )
)

phase40_known_mask = phase40_audit_exact
phase40_unknown_mask = ~phase40_audit_exact
phase40_known_error = float(np.max(np.abs(
    phase40_staged_probability[phase40_known_mask]
    - phase40_reference_probability[phase40_known_mask]
)))
phase40_unknown_anchor_error = float(np.max(np.abs(
    phase40_staged_probability[phase40_unknown_mask]
    - phase40_audit_baseline[phase40_unknown_mask]
)))
phase40_complete_error = float(np.max(np.abs(
    phase40_staged_probability
    - phase40_reference_probability
)))
phase40_unknown_update_error = float(np.max(np.abs(
    phase40_staged_details["applied_update"][
        phase40_unknown_mask
    ]
)))

assert phase40_known_error <= 5e-6
assert phase40_unknown_anchor_error <= 1e-12
assert phase40_complete_error <= 5e-6
assert phase40_unknown_update_error == 0.0
assert np.isfinite(phase40_staged_probability).all()

PHASE40_OFFLINE_SOURCE_CONTRACT = {
    "phase": "phase40_staged_offline_source_contract",
    "status": "accepted",
    "audit_case_count": len(phase40_audit_indices),
    "known_route_case_count": int(np.sum(phase40_known_mask)),
    "unknown_route_case_count": int(np.sum(phase40_unknown_mask)),
    "known_route_probability_error": phase40_known_error,
    "unknown_route_anchor_error": phase40_unknown_anchor_error,
    "complete_probability_error": phase40_complete_error,
    "unknown_route_update_error": phase40_unknown_update_error,
    "phase33_model_count": int(sum(
        len(models)
        for models in PHASE40_STAGED_RUNTIME_PRIVATE.phase33_models
    )),
    "phase36_booster_count": int(sum(
        len(boosters)
        for boosters in PHASE40_STAGED_RUNTIME_PRIVATE.phase36_boosters
    )),
    "formula": PHASE40_STAGED_RUNTIME_PRIVATE.formula,
    "single_routed_fold_source_exercised": True,
    "three_fold_average_source_exercised": False,
    "unknown_protocol_anchor_only_exercised": True,
    "all_outputs_finite": True,
    "labels_used": False,
    "training_cached_features_read": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase40_133b_started, 3
    ),
}

print("BEGIN SANITIZED_PHASE40_OFFLINE_SOURCE")
print(json.dumps(PHASE40_OFFLINE_SOURCE_CONTRACT, indent=2))
print("END SANITIZED_PHASE40_OFFLINE_SOURCE")


BEGIN SANITIZED_PHASE40_OFFLINE_SOURCE
{
  "phase": "phase40_staged_offline_source_contract",
  "status": "accepted",
  "audit_case_count": 4,
  "known_route_case_count": 3,
  "unknown_route_case_count": 1,
  "known_route_probability_error": 6.122035947631055e-09,
  "unknown_route_anchor_error": 3.469446951953614e-18,
  "complete_probability_error": 6.122035947631055e-09,
  "unknown_route_update_error": 0.0,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "formula": {
    "phase36_weight": 0.75,
    "phase33_weight": 0.25,
    "alpha": 1.0,
    "uncertainty_exponent": 0.0,
    "delta_cap": 2.0
  },
  "single_routed_fold_source_exercised": true,
  "three_fold_average_source_exercised": false,
  "unknown_protocol_anchor_only_exercised": true,
  "all_outputs_finite": true,
  "labels_used": false,
  "training_cached_features_read": true,
  "training_voxel_arrays_read": false,
  "smoke_data_read": false,
  "test_data_read": false,
  "case_level_predictions_exported": false,
  "e

In [113]:
# Phase40 Cell 133C
# Full cached parity using the exact source now staged for deployment.

import gc
import json
import time

import numpy as np
import torch


phase40_133c_started = time.perf_counter()
assert PHASE40_OFFLINE_SOURCE_CONTRACT["status"] == "accepted"

PHASE40_STAGED_ROUTE_OOF_PRIVATE = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)
phase40_staged_phase33_logit = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)
phase40_staged_phase36_residual = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)
phase40_staged_combined_residual = np.full(
    PHASE40_CASE_COUNT, np.nan, dtype=np.float64
)

for start in range(0, PHASE40_CASE_COUNT, 48):
    stop = min(start + 48, PHASE40_CASE_COUNT)
    indices = np.arange(start, stop, dtype=np.int64)
    probability, details = (
        PHASE40_STAGED_RUNTIME_PRIVATE.predict_routed_from_dense(
            np.asarray(
                phase40_dense_cache[indices], dtype=np.float32
            ),
            np.asarray(
                phase40_radiomics_cache[indices], dtype=np.float32
            ),
            np.asarray(
                phase40_geometry_cache[indices], dtype=np.float32
            ),
            PHASE40_BASELINE_OOF_PRIVATE[indices],
            PHASE40_ROUTED_FOLD_PRIVATE[indices],
            PHASE40_ROUTE_EXACT_PRIVATE[indices],
        )
    )
    PHASE40_STAGED_ROUTE_OOF_PRIVATE[indices] = probability
    phase40_staged_phase33_logit[indices] = details[
        "phase33_component_logit"
    ]
    phase40_staged_phase36_residual[indices] = details[
        "raw_phase36_residual"
    ]
    phase40_staged_combined_residual[indices] = details[
        "combined_bounded"
    ]
    if stop % 192 == 0 or stop == PHASE40_CASE_COUNT:
        print(
            f"Phase40 staged parity: {stop}/{PHASE40_CASE_COUNT}"
        )


def phase40_staged_error(actual, reference):
    difference = np.abs(
        np.asarray(actual, dtype=np.float64)
        - np.asarray(reference, dtype=np.float64)
    )
    return {
        "maximum": float(np.max(difference)),
        "mean": float(np.mean(difference)),
        "q99": float(np.quantile(difference, 0.99)),
    }


phase40_staged_errors = {
    "phase33_logit": phase40_staged_error(
        phase40_staged_phase33_logit,
        PHASE40_PHASE33_ROUTED_LOGIT_PRIVATE,
    ),
    "phase36_raw_residual": phase40_staged_error(
        phase40_staged_phase36_residual,
        PHASE40_PHASE36_ROUTED_RAW_RESIDUAL_PRIVATE,
    ),
    "bounded_combined_residual": phase40_staged_error(
        phase40_staged_combined_residual,
        PHASE40_ROUTED_COMBINED_RESIDUAL_PRIVATE,
    ),
    "final_probability": phase40_staged_error(
        PHASE40_STAGED_ROUTE_OOF_PRIVATE,
        PHASE40_PHASE39_REFERENCE_OOF_PRIVATE,
    ),
}

assert phase40_staged_errors["phase33_logit"]["maximum"] <= 5e-5
assert phase40_staged_errors[
    "phase36_raw_residual"
]["maximum"] <= 5e-5
assert phase40_staged_errors[
    "bounded_combined_residual"
]["maximum"] <= 5e-5
assert phase40_staged_errors[
    "final_probability"
]["maximum"] <= 5e-6

PHASE40_STAGED_PARITY_CONTRACT = {
    "phase": "phase40_staged_exact_route_complete_parity",
    "status": "accepted",
    "case_count": PHASE40_CASE_COUNT,
    "component_errors": phase40_staged_errors,
    "fold_route_counts": {
        str(fold): int(np.sum(
            PHASE40_ROUTED_FOLD_PRIVATE == fold
        ))
        for fold in range(3)
    },
    "exact_route_fraction": float(np.mean(
        PHASE40_ROUTE_EXACT_PRIVATE
    )),
    "single_fold_computation_per_known_case": True,
    "three_fold_residual_average_used": False,
    "unknown_protocol_policy": "phase12c_anchor_only",
    "phase12c_anchor": "cross_fitted_oof_for_parity",
    "deployment_anchor_planned": "complete_three_fold_ensemble",
    "all_values_finite": bool(
        np.isfinite(PHASE40_STAGED_ROUTE_OOF_PRIVATE).all()
    ),
    "labels_used": False,
    "outer_validation_labels_used": False,
    "training_cached_features_read": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "peak_vram_mb": (
        float(torch.cuda.max_memory_allocated(PHASE39_DEVICE))
        / (1024.0 ** 2)
        if torch.cuda.is_available()
        else 0.0
    ),
    "elapsed_seconds": round(
        time.perf_counter() - phase40_133c_started, 3
    ),
}

print("BEGIN SANITIZED_PHASE40_STAGED_PARITY")
print(json.dumps(PHASE40_STAGED_PARITY_CONTRACT, indent=2))
print("END SANITIZED_PHASE40_STAGED_PARITY")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Phase40 staged parity: 192/1362
Phase40 staged parity: 384/1362
Phase40 staged parity: 576/1362
Phase40 staged parity: 768/1362
Phase40 staged parity: 960/1362
Phase40 staged parity: 1152/1362
Phase40 staged parity: 1344/1362
Phase40 staged parity: 1362/1362
BEGIN SANITIZED_PHASE40_STAGED_PARITY
{
  "phase": "phase40_staged_exact_route_complete_parity",
  "status": "accepted",
  "case_count": 1362,
  "component_errors": {
    "phase33_logit": {
      "maximum": 1.1920928955078125e-06,
      "mean": 4.783797894280387e-08,
      "q99": 9.5367431640625e-07
    },
    "phase36_raw_residual": {
      "maximum": 0.0,
      "mean": 0.0,
      "q99": 0.0
    },
    "bounded_combined_residual": {
      "maximum": 2.980232238769531e-07,
      "mean": 1.1959494762376481e-08,
      "q99": 2.384185790338389e-07
    },
    "final_probability": {
      "maximum": 5.6261858227824035e-08,
      "mean": 2.681477594208647e-09,
      "q99": 1.7304309297694355e-08
    }
  },
  "fold_route_counts": {
    "0

In [114]:
# Phase40 Cell 134A
# Real 20-case staged smoke test in two independent offline subprocesses.
#
# The private-diagnostic run proves that preprocessing, initialization, and
# case-level Phase40 inference all succeed.  The production run proves that
# the submitted entrypoint emits only generic, organizer-safe lifecycle logs.
# No smoke labels are read and no case-level values are printed or exported.

from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

import numpy as np
import pandas as pd


PHASE40_SMOKE_PACKAGE = Path(
    "/kaggle/working/phase40_experimental_submission"
)
PHASE40_SMOKE_EXPECTED_CASES = 20
PHASE40_SMOKE_MAXIMUM_SECONDS_PER_RUN = 360.0
PHASE40_SMOKE_CONTRACT_PATH = Path(
    "/kaggle/working/phase40_real_smoke_contract.json"
)

phase40_134a_started = time.perf_counter()


# ------------------------------------------------------------------
# Preconditions and private smoke-root resolution.
# ------------------------------------------------------------------

assert isinstance(
    globals().get("PHASE40_STAGED_PARITY_CONTRACT"),
    dict,
), {
    "message": (
        "Cell 134A requires the accepted Cell 133C staged parity "
        "contract in the current kernel."
    )
}
assert PHASE40_STAGED_PARITY_CONTRACT.get("status") == "accepted"

required_package_files = [
    Path("main.py"),
    Path("phase40_runtime.py"),
    Path("phase40_runtime_contract.json"),
    Path("phase39_runtime.py"),
    Path("phase30_accepted_main.py"),
    Path("phase39_assets") / "phase39_runtime_state.json",
    Path("phase39_assets") / "phase39_asset_manifest.json",
]
missing_package_files = [
    relative_path.as_posix()
    for relative_path in required_package_files
    if not (PHASE40_SMOKE_PACKAGE / relative_path).is_file()
]
assert not missing_package_files, {
    "message": "Phase40 staged package is incomplete.",
    "missing": missing_package_files,
}


def phase40_resolve_smoke_root():
    candidates = []
    for environment_name in (
        "DAT_PRIVATE_SMOKE_ROOT",
        "DAT_SMOKE_ROOT",
    ):
        value = os.environ.get(environment_name)
        if value:
            candidates.append((
                Path(value),
                f"environment:{environment_name}",
            ))

    candidates.extend([
        (
            Path(
                "/kaggle/input/datasets/nahinalam/"
                "drivendata-dataset/smoke_test_data"
            ),
            "private_dataset_default",
        ),
        (
            Path(
                "/kaggle/input/drivendata-dataset/"
                "smoke_test_data"
            ),
            "private_dataset_short_path",
        ),
    ])

    audit = []
    seen = set()
    for root, source in candidates:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)
        valid = (
            (root / "niftis").is_dir()
            and (root / "submission_format.csv").is_file()
        )
        audit.append({
            "source": source,
            "valid": bool(valid),
        })
        if valid:
            return root, source, audit

    raise AssertionError({
        "message": "Could not resolve the private smoke-test root.",
        "candidate_audit": audit,
    })


def phase40_nifti_identifier(path):
    name = path.name
    lower = name.lower()
    if lower.endswith(".nii.gz"):
        return name[:-7]
    if lower.endswith(".nii"):
        return name[:-4]
    raise ValueError("Unexpected NIfTI extension")


smoke_root, smoke_root_source, smoke_root_audit = (
    phase40_resolve_smoke_root()
)
smoke_paths = sorted(
    path
    for path in (smoke_root / "niftis").iterdir()
    if path.is_file()
    and (
        path.name.lower().endswith(".nii")
        or path.name.lower().endswith(".nii.gz")
    )
)
assert len(smoke_paths) == PHASE40_SMOKE_EXPECTED_CASES

smoke_format = pd.read_csv(
    smoke_root / "submission_format.csv",
    dtype={"uid": str},
)
assert list(smoke_format.columns) == [
    "uid",
    "is_pathologic",
]
assert len(smoke_format) == PHASE40_SMOKE_EXPECTED_CASES
assert smoke_format["uid"].notna().all()
assert smoke_format["uid"].is_unique

expected_uids = smoke_format["uid"].astype(str).tolist()
image_uids = [
    phase40_nifti_identifier(path)
    for path in smoke_paths
]
assert set(expected_uids) == set(image_uids)


# ------------------------------------------------------------------
# Subprocess helper.  Diagnostic and production modes are isolated.
# ------------------------------------------------------------------

generic_required_lines = (
    "Built with DINOv3.",
    "Initialization complete.",
    "Inference started.",
    "Inference completed.",
)


def phase40_run_smoke(mode, private_diagnostics):
    temporary_root = Path(tempfile.mkdtemp(
        prefix=f"phase40_{mode}_smoke_",
        dir="/kaggle/working",
    ))
    temporary_output = temporary_root / "submission.csv"
    temporary_torch_home = temporary_root / "torch_hub"

    environment = os.environ.copy()
    environment.update({
        "DAT_DATA_ROOT": str(smoke_root),
        "DAT_OUTPUT_CSV": str(temporary_output),
        "DAT_PREPROCESS_WORKERS": "6",
        "DAT_BATCH_SIZE": "4",
        "DAT_DINO_VIEW_BATCH": "12",
        "TORCH_HOME": str(temporary_torch_home),
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "HF_DATASETS_OFFLINE": "1",
        "PYTHONHASHSEED": "400134",
        "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
        "CUDA_VISIBLE_DEVICES": "0",
    })
    if private_diagnostics:
        environment["DAT_PRIVATE_DIAGNOSTICS"] = "1"
    else:
        environment.pop("DAT_PRIVATE_DIAGNOSTICS", None)

    process = None
    probability = None
    combined_log = ""
    elapsed_seconds = None
    try:
        started = time.perf_counter()
        process = subprocess.run(
            [
                sys.executable,
                str(PHASE40_SMOKE_PACKAGE / "main.py"),
            ],
            cwd=str(PHASE40_SMOKE_PACKAGE),
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=PHASE40_SMOKE_MAXIMUM_SECONDS_PER_RUN,
            check=False,
        )
        elapsed_seconds = float(time.perf_counter() - started)
        combined_log = (
            (process.stdout or "")
            + "\n"
            + (process.stderr or "")
        )

        assert process.returncode == 0, {
            "message": f"Phase40 {mode} smoke subprocess failed.",
            "return_code": process.returncode,
            "log_tail": [
                line[:300]
                for line in combined_log.splitlines()[-8:]
            ],
        }
        assert elapsed_seconds <= PHASE40_SMOKE_MAXIMUM_SECONDS_PER_RUN
        assert temporary_output.is_file()
        assert not re.search(
            r"https?://",
            combined_log,
            flags=re.IGNORECASE,
        )
        assert not any(
            uid and uid in combined_log
            for uid in expected_uids
        )
        for required_line in generic_required_lines:
            assert combined_log.count(required_line) == 1, {
                "message": f"Missing or repeated {mode} lifecycle line.",
                "line": required_line,
            }

        submission = pd.read_csv(
            temporary_output,
            dtype={"uid": str},
        )
        assert list(submission.columns) == [
            "uid",
            "is_pathologic",
        ]
        assert len(submission) == PHASE40_SMOKE_EXPECTED_CASES
        assert submission["uid"].astype(str).tolist() == expected_uids
        probability = submission[
            "is_pathologic"
        ].to_numpy(dtype=np.float64)
        assert probability.shape == (PHASE40_SMOKE_EXPECTED_CASES,)
        assert np.isfinite(probability).all()
        assert np.all(probability >= 1e-5)
        assert np.all(probability <= 1.0 - 1e-5)
        assert float(np.ptp(probability)) > 0.0

        return {
            "mode": mode,
            "process": process,
            "elapsed_seconds": elapsed_seconds,
            "combined_log": combined_log,
            "probability": probability.copy(),
            "log_line_count": len([
                line
                for line in combined_log.splitlines()
                if line.strip()
            ]),
        }
    finally:
        if temporary_root.exists():
            shutil.rmtree(temporary_root)


diagnostic_result = phase40_run_smoke(
    mode="diagnostic",
    private_diagnostics=True,
)
production_result = phase40_run_smoke(
    mode="production",
    private_diagnostics=False,
)


# ------------------------------------------------------------------
# Diagnostic counters and production-log compliance.
# ------------------------------------------------------------------

diagnostic_match = re.search(
    r"Private diagnostics:\s*"
    r"preprocessing=(\d+),\s*"
    r"unknown_routes=(\d+),\s*"
    r"initialization=(\d+),\s*"
    r"case_fallbacks=(\d+)\.",
    diagnostic_result["combined_log"],
)
assert diagnostic_match is not None, {
    "message": "Could not parse Phase40 private diagnostics."
}
(
    preprocessing_failures,
    unknown_routes,
    initialization_fallbacks,
    case_fallbacks,
) = [int(value) for value in diagnostic_match.groups()]

assert preprocessing_failures == 0
assert initialization_fallbacks == 0
assert case_fallbacks == 0
assert 0 <= unknown_routes < PHASE40_SMOKE_EXPECTED_CASES, {
    "message": (
        "The real smoke did not exercise any exact Phase40 route."
    ),
    "unknown_routes": unknown_routes,
}
known_routes = PHASE40_SMOKE_EXPECTED_CASES - unknown_routes

production_log = production_result["combined_log"]
forbidden_production_patterns = {
    "progress_counter": r"\bprogress\b",
    "case_count_counter": r"\bcases\s*=",
    "fallback_counter": r"\bfallback(?:s)?\s*=",
    "unknown_route_counter": r"\bunknown_routes\s*=",
    "private_diagnostics": r"\bprivate diagnostics\b",
    "detailed_failure": r"\bexecution failed\s*:",
    "uid_field": r"\buid\b",
    "prediction_field": r"\bis_pathologic\b",
}
production_forbidden_hits = {
    name: bool(re.search(pattern, production_log, re.IGNORECASE))
    for name, pattern in forbidden_production_patterns.items()
}
assert not any(production_forbidden_hits.values()), {
    "message": "Production log contains a forbidden detailed pattern.",
    "hits": production_forbidden_hits,
}

diagnostic_production_probability_error = float(np.max(np.abs(
    diagnostic_result["probability"]
    - production_result["probability"]
)))
assert diagnostic_production_probability_error <= 1e-12

production_safe_lines = [
    line.strip()
    for line in production_log.splitlines()
    if line.strip() in generic_required_lines
]
assert tuple(production_safe_lines) == generic_required_lines

# Private in-memory state used only by Cell 134C to prove that the extracted
# archive produces exactly the same probabilities.  It is never serialized.
PHASE40_STAGED_SMOKE_PROBABILITY_PRIVATE = production_result[
    "probability"
].copy()
PHASE40_SMOKE_UID_ORDER_PRIVATE = tuple(expected_uids)
PHASE40_SMOKE_ROOT_PRIVATE = smoke_root
PHASE40_SMOKE_ROOT_SOURCE_PRIVATE = smoke_root_source

package_files = [
    path
    for path in PHASE40_SMOKE_PACKAGE.rglob("*")
    if path.is_file()
]
package_size_bytes = sum(
    path.stat().st_size
    for path in package_files
)

report = {
    "phase": "phase40_dual_mode_real_packaged_smoke_test",
    "status": "accepted",
    "package_directory": PHASE40_SMOKE_PACKAGE.name,
    "package_file_count": len(package_files),
    "package_size_mb": round(package_size_bytes / (1024 ** 2), 6),
    "case_count": PHASE40_SMOKE_EXPECTED_CASES,
    "diagnostic_run": {
        "return_code": int(diagnostic_result["process"].returncode),
        "elapsed_seconds": diagnostic_result["elapsed_seconds"],
        "preprocessing_failures": preprocessing_failures,
        "known_routes": known_routes,
        "unknown_routes": unknown_routes,
        "initialization_fallbacks": initialization_fallbacks,
        "case_fallbacks": case_fallbacks,
        "routed_branch_exercised": bool(known_routes > 0),
    },
    "production_run": {
        "return_code": int(production_result["process"].returncode),
        "elapsed_seconds": production_result["elapsed_seconds"],
        "log_line_count": int(production_result["log_line_count"]),
        "generic_lifecycle_line_count": len(production_safe_lines),
        "forbidden_pattern_count": int(sum(
            production_forbidden_hits.values()
        )),
        "generic_logging_only": True,
        "network_url_present": False,
    },
    "diagnostic_production_probability_error": (
        diagnostic_production_probability_error
    ),
    "schema_valid": True,
    "uid_order_exact": True,
    "all_probabilities_finite": True,
    "all_probabilities_clipped": True,
    "output_nonconstant": True,
    "offline_environment_enforced": True,
    "separate_subprocesses": True,
    "temporary_outputs_deleted": True,
    "temporary_torch_caches_deleted": True,
    "production_logging_contract_passed": True,
    "smoke_root_source": smoke_root_source,
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "training_voxel_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "model_hashes_displayed": False,
    "total_cell_seconds": round(
        time.perf_counter() - phase40_134a_started,
        3,
    ),
}

PHASE40_SMOKE_CONTRACT_PATH.write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
PHASE40_REAL_SMOKE_REPORT = report

print("BEGIN PHASE40_PRODUCTION_SMOKE_LOG")
for line in production_safe_lines:
    print(line)
print("END PHASE40_PRODUCTION_SMOKE_LOG")
print("BEGIN SANITIZED_PHASE40_REAL_SMOKE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE40_REAL_SMOKE")
print("Cell 134A accepted. Continue with Cell 134B archive construction.")

BEGIN PHASE40_PRODUCTION_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed.
END PHASE40_PRODUCTION_SMOKE_LOG
BEGIN SANITIZED_PHASE40_REAL_SMOKE
{
  "phase": "phase40_dual_mode_real_packaged_smoke_test",
  "status": "accepted",
  "package_directory": "phase40_experimental_submission",
  "package_file_count": 115,
  "package_size_mb": 193.569074,
  "case_count": 20,
  "diagnostic_run": {
    "return_code": 0,
    "elapsed_seconds": 16.1914908990002,
    "preprocessing_failures": 0,
    "known_routes": 20,
    "unknown_routes": 0,
    "initialization_fallbacks": 0,
    "case_fallbacks": 0,
    "routed_branch_exercised": true
  },
  "production_run": {
    "return_code": 0,
    "elapsed_seconds": 16.25961943600123,
    "log_line_count": 4,
    "generic_lifecycle_line_count": 4,
    "forbidden_pattern_count": 0,
    "generic_logging_only": true,
    "network_url_present": false
  },
  "diagnostic_production_probability_error": 0.0,
  "schema_valid"

In [115]:
# Phase40 Cell 134B
# Build and audit a clean deterministic archive candidate.
#
# The exact expected manifest is derived from the already accepted Phase39
# archive plus the two new Phase40 deployment files.  This avoids the brittle
# hard-coded file-count assertion that previously failed after harmless cache
# files appeared in staging.  The final path is not published until Cell 134C
# passes an extracted-archive smoke test.

from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import tempfile
import time
import zipfile


PHASE40_ARCHIVE_STAGE = Path(
    "/kaggle/working/phase40_experimental_submission"
)
PHASE40_ACCEPTED_BASE_ARCHIVE = Path(
    "/kaggle/working/phase39_submission.zip"
)
PHASE40_ARCHIVE_CANDIDATE = Path(
    "/kaggle/working/phase40_submission.candidate.zip"
)
PHASE40_ARCHIVE_BUILD_CONTRACT_PATH = Path(
    "/kaggle/working/phase40_archive_build_contract.json"
)

phase40_134b_started = time.perf_counter()


assert isinstance(
    globals().get("PHASE40_REAL_SMOKE_REPORT"),
    dict,
), {
    "message": "Cell 134B requires the accepted Cell 134A smoke report."
}
assert PHASE40_REAL_SMOKE_REPORT.get("status") == "accepted"
assert PHASE40_REAL_SMOKE_REPORT.get(
    "production_logging_contract_passed"
) is True
assert PHASE40_ARCHIVE_STAGE.is_dir()
assert PHASE40_ACCEPTED_BASE_ARCHIVE.is_file()


def phase40_safe_archive_name(name):
    pure = PurePosixPath(name)
    return bool(
        pure.parts
        and not pure.is_absolute()
        and ".." not in pure.parts
    )


def phase40_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


# ------------------------------------------------------------------
# Exact manifest contract: accepted Phase39 + two Phase40 files.
# ------------------------------------------------------------------

with zipfile.ZipFile(
    PHASE40_ACCEPTED_BASE_ARCHIVE,
    mode="r",
) as base_archive:
    assert base_archive.testzip() is None
    phase39_member_names = base_archive.namelist()

assert phase39_member_names
assert len(phase39_member_names) == len(set(phase39_member_names))
assert "main.py" in phase39_member_names
assert all(
    phase40_safe_archive_name(name)
    for name in phase39_member_names
)

phase40_added_names = {
    "phase40_runtime.py",
    "phase40_runtime_contract.json",
}
expected_archive_names = sorted(
    set(phase39_member_names) | phase40_added_names
)

excluded_contract_names = {
    "phase39_offline_initialization_contract.json",
    "phase39_complete_oof_parity_contract.json",
    "phase39_real_smoke_contract.json",
    "phase40_real_smoke_contract.json",
    "phase40_archive_build_contract.json",
}


def phase40_should_archive(relative_path):
    parts = relative_path.parts
    if not parts:
        return False
    if "__pycache__" in parts:
        return False
    if relative_path.name in excluded_contract_names:
        return False
    if relative_path.suffix.lower() in {
        ".pyc",
        ".pyo",
        ".ipynb",
    }:
        return False
    if any(part.startswith(".") for part in parts):
        return False
    return True


all_stage_files = sorted(
    path
    for path in PHASE40_ARCHIVE_STAGE.rglob("*")
    if path.is_file()
)
assert all(not path.is_symlink() for path in all_stage_files)

archive_record_by_name = {}
excluded_stage_files = []
for source_path in all_stage_files:
    relative_path = source_path.relative_to(PHASE40_ARCHIVE_STAGE)
    if phase40_should_archive(relative_path):
        archive_record_by_name[relative_path.as_posix()] = source_path
    else:
        excluded_stage_files.append(relative_path.as_posix())

observed_archive_names = sorted(archive_record_by_name)
assert observed_archive_names == expected_archive_names, {
    "message": (
        "Phase40 deployable manifest differs from the accepted Phase39 "
        "manifest plus the two controlled Phase40 additions."
    ),
    "missing": sorted(
        set(expected_archive_names) - set(observed_archive_names)
    ),
    "unexpected": sorted(
        set(observed_archive_names) - set(expected_archive_names)
    ),
    "excluded_count": len(excluded_stage_files),
}

required_relative_paths = [
    "main.py",
    "phase40_runtime.py",
    "phase40_runtime_contract.json",
    "phase39_runtime.py",
    "phase30_accepted_main.py",
    "phase39_assets/phase39_runtime_state.json",
    "phase39_assets/phase39_asset_manifest.json",
]
assert all(
    name in observed_archive_names
    for name in required_relative_paths
)
assert all(
    phase40_safe_archive_name(name)
    for name in observed_archive_names
)
assert all(
    not name.startswith(f"{PHASE40_ARCHIVE_STAGE.name}/")
    for name in observed_archive_names
)

lower_archive_names = [
    name.lower()
    for name in observed_archive_names
]
for forbidden_fragment in (
    "__pycache__",
    "submission.csv",
    "test_labels",
    "train_labels",
    "patient_rows",
    "component_oof",
    "gated_oof",
    "raw_residual_oof",
    "phase31_highres",
    "phase31_radiomics",
    "phase31_sequence",
    "bilateral_dense",
    "p3ca_summary_float16",
    "p3ca_sequence_float16",
    "smoke_contract",
    "parity_contract",
):
    assert not any(
        forbidden_fragment in name
        for name in lower_archive_names
    ), {
        "message": "Private/transient artifact selected for archive.",
        "fragment": forbidden_fragment,
    }

phase12_model_files = [
    name
    for name in observed_archive_names
    if name.startswith("models/")
    and name.lower().endswith(".pt")
]
dinov3_checkpoint_files = [
    name
    for name in observed_archive_names
    if name.startswith("models/")
    and name.lower().endswith(".pth")
    and "dinov3" in name.lower()
]
phase33_model_files = [
    name
    for name in observed_archive_names
    if name.startswith("phase39_assets/phase33_models/")
    and name.lower().endswith(".pt")
]
phase36_booster_files = [
    name
    for name in observed_archive_names
    if name.startswith("phase39_assets/phase36_boosters/")
    and name.lower().endswith(".ubj")
]
p3ca_state_files = [
    name
    for name in observed_archive_names
    if name.endswith("phase32_fold_local_p3ca_states.npz")
]

assert len(phase12_model_files) == 21
assert len(dinov3_checkpoint_files) == 1
assert len(phase33_model_files) == 9
assert len(phase36_booster_files) == 6
assert len(p3ca_state_files) == 1
assert any(name.startswith("dinov3_repo/") for name in observed_archive_names)
assert any(
    "dinov3" in Path(name).name.lower()
    and "license" in Path(name).name.lower()
    for name in observed_archive_names
)

attribution_present = False
for candidate_name in (
    "README.md",
    "THIRD_PARTY_LICENSES.md",
    "main.py",
):
    candidate_path = PHASE40_ARCHIVE_STAGE / candidate_name
    if candidate_path.is_file() and "Built with DINOv3" in (
        candidate_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    ):
        attribution_present = True
        break
assert attribution_present


# ------------------------------------------------------------------
# Deterministic candidate ZIP.  Cell 134C publishes it only after smoke.
# ------------------------------------------------------------------

temporary_handle, temporary_name = tempfile.mkstemp(
    prefix="phase40_submission_",
    suffix=".partial.zip",
    dir="/kaggle/working",
)
os.close(temporary_handle)
temporary_zip_path = Path(temporary_name)

try:
    with zipfile.ZipFile(
        temporary_zip_path,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
        allowZip64=True,
    ) as archive:
        for archive_name in observed_archive_names:
            source_path = archive_record_by_name[archive_name]
            information = zipfile.ZipInfo(
                filename=archive_name,
                date_time=(2026, 8, 18, 0, 0, 0),
            )
            information.compress_type = zipfile.ZIP_DEFLATED
            information.create_system = 3
            information.external_attr = 0o100644 << 16
            information.flag_bits |= 0x800
            with source_path.open("rb") as source_handle:
                with archive.open(
                    information,
                    mode="w",
                    force_zip64=True,
                ) as destination_handle:
                    shutil.copyfileobj(
                        source_handle,
                        destination_handle,
                        length=1024 * 1024,
                    )

    assert temporary_zip_path.is_file()
    assert 0 < temporary_zip_path.stat().st_size < 500 * 10**6
    with zipfile.ZipFile(temporary_zip_path, mode="r") as archive:
        assert archive.testzip() is None
        assert archive.namelist() == observed_archive_names

    os.replace(temporary_zip_path, PHASE40_ARCHIVE_CANDIDATE)
finally:
    if temporary_zip_path.exists():
        temporary_zip_path.unlink()

candidate_sha256 = phase40_file_sha256(PHASE40_ARCHIVE_CANDIDATE)
PHASE40_ARCHIVE_CANDIDATE_SHA256_PRIVATE = candidate_sha256
PHASE40_ARCHIVE_MANIFEST_PRIVATE = tuple(observed_archive_names)

report = {
    "phase": "phase40_deterministic_archive_candidate_build",
    "status": "accepted_candidate_not_yet_published",
    "candidate_name": PHASE40_ARCHIVE_CANDIDATE.name,
    "manifest_contract": (
        "accepted_phase39_archive_members_plus_two_phase40_files"
    ),
    "phase39_member_count": len(phase39_member_names),
    "phase40_added_file_count": len(phase40_added_names),
    "archive_file_count": len(observed_archive_names),
    "excluded_transient_file_count": len(excluded_stage_files),
    "compressed_size_mb": round(
        PHASE40_ARCHIVE_CANDIDATE.stat().st_size / 1e6,
        6,
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "deterministic_archive_metadata": True,
    "phase12_model_count": len(phase12_model_files),
    "dinov3_checkpoint_count": len(dinov3_checkpoint_files),
    "phase33_model_count": len(phase33_model_files),
    "phase36_booster_count": len(phase36_booster_files),
    "p3ca_state_count": len(p3ca_state_files),
    "dinov3_source_present": True,
    "required_attribution_present": attribution_present,
    "dinov3_license_present": True,
    "final_archive_published": False,
    "accepted_phase39_archive_modified": False,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_oof_arrays": False,
    "smoke_data_read": False,
    "challenge_test_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase40_134b_started,
        3,
    ),
}

PHASE40_ARCHIVE_BUILD_CONTRACT_PATH.write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
PHASE40_ARCHIVE_BUILD_REPORT = report

print("BEGIN SANITIZED_PHASE40_ARCHIVE_BUILD")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE40_ARCHIVE_BUILD")
print("Cell 134B accepted. Continue with Cell 134C extracted-archive smoke.")

BEGIN SANITIZED_PHASE40_ARCHIVE_BUILD
{
  "phase": "phase40_deterministic_archive_candidate_build",
  "status": "accepted_candidate_not_yet_published",
  "candidate_name": "phase40_submission.candidate.zip",
  "manifest_contract": "accepted_phase39_archive_members_plus_two_phase40_files",
  "phase39_member_count": 87,
  "phase40_added_file_count": 2,
  "archive_file_count": 89,
  "excluded_transient_file_count": 26,
  "compressed_size_mb": 185.823937,
  "root_main_present": true,
  "wrapping_directory_present": false,
  "archive_integrity_test_passed": true,
  "deterministic_archive_metadata": true,
  "phase12_model_count": 21,
  "dinov3_checkpoint_count": 1,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "p3ca_state_count": 1,
  "dinov3_source_present": true,
  "required_attribution_present": true,
  "dinov3_license_present": true,
  "final_archive_published": false,
  "accepted_phase39_archive_modified": false,
  "contains_voxel_data": false,
  "contains_patient_rows": f

In [116]:
# Phase40 Cell 134C
# Extract the exact candidate archive, run a production-mode real smoke test,
# compare its probabilities with Cell 134A, and publish the final ZIP atomically.

from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

import numpy as np
import pandas as pd


PHASE40_FINAL_CANDIDATE = Path(
    "/kaggle/working/phase40_submission.candidate.zip"
)
PHASE40_FINAL_ZIP = Path(
    "/kaggle/working/phase40_submission.zip"
)
PHASE40_FINAL_SMOKE_LIMIT_SECONDS = 360.0
PHASE40_FINAL_EXPECTED_SMOKE_CASES = 20
PHASE40_FINAL_CONTRACT_PATH = Path(
    "/kaggle/working/phase40_final_archive_contract.json"
)

phase40_134c_started = time.perf_counter()


assert isinstance(
    globals().get("PHASE40_ARCHIVE_BUILD_REPORT"),
    dict,
), {
    "message": "Cell 134C requires the accepted Cell 134B build report."
}
assert PHASE40_ARCHIVE_BUILD_REPORT.get("status") == (
    "accepted_candidate_not_yet_published"
)
assert PHASE40_FINAL_CANDIDATE.is_file()
assert isinstance(
    globals().get("PHASE40_STAGED_SMOKE_PROBABILITY_PRIVATE"),
    np.ndarray,
), {
    "message": "Cell 134C requires Cell 134A staged smoke probabilities."
}
assert PHASE40_STAGED_SMOKE_PROBABILITY_PRIVATE.shape == (
    PHASE40_FINAL_EXPECTED_SMOKE_CASES,
)
assert len(PHASE40_SMOKE_UID_ORDER_PRIVATE) == (
    PHASE40_FINAL_EXPECTED_SMOKE_CASES
)
assert Path(PHASE40_SMOKE_ROOT_PRIVATE).is_dir()


def phase40_final_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def phase40_final_safe_name(name):
    pure = PurePosixPath(name)
    return bool(
        pure.parts
        and not pure.is_absolute()
        and ".." not in pure.parts
    )


assert phase40_final_sha256(PHASE40_FINAL_CANDIDATE) == (
    PHASE40_ARCHIVE_CANDIDATE_SHA256_PRIVATE
)

with zipfile.ZipFile(PHASE40_FINAL_CANDIDATE, mode="r") as archive:
    candidate_members = archive.namelist()
    assert archive.testzip() is None
    assert tuple(candidate_members) == PHASE40_ARCHIVE_MANIFEST_PRIVATE
    assert all(phase40_final_safe_name(name) for name in candidate_members)
    assert "main.py" in candidate_members
    assert "phase40_runtime.py" in candidate_members


rehearsal_root = Path(tempfile.mkdtemp(
    prefix="phase40_archive_rehearsal_",
    dir="/kaggle/working",
))
rehearsal_output = rehearsal_root / "submission.csv"
rehearsal_torch_home = rehearsal_root / "torch_hub"

archive_process = None
archive_smoke_seconds = None
archive_log = ""
archive_probabilities = None
safe_archive_log_lines = []
extracted_staged_probability_error = None

try:
    with zipfile.ZipFile(PHASE40_FINAL_CANDIDATE, mode="r") as archive:
        for member in archive.infolist():
            assert phase40_final_safe_name(member.filename)
        archive.extractall(rehearsal_root)

    assert (rehearsal_root / "main.py").is_file()
    assert (rehearsal_root / "phase40_runtime.py").is_file()
    assert not (
        rehearsal_root / "phase40_experimental_submission"
    ).exists()

    environment = os.environ.copy()
    environment.update({
        "DAT_DATA_ROOT": str(PHASE40_SMOKE_ROOT_PRIVATE),
        "DAT_OUTPUT_CSV": str(rehearsal_output),
        "DAT_PREPROCESS_WORKERS": "6",
        "DAT_BATCH_SIZE": "4",
        "DAT_DINO_VIEW_BATCH": "12",
        "TORCH_HOME": str(rehearsal_torch_home),
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "HF_DATASETS_OFFLINE": "1",
        "PYTHONHASHSEED": "400134",
        "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
        "CUDA_VISIBLE_DEVICES": "0",
    })
    environment.pop("DAT_PRIVATE_DIAGNOSTICS", None)

    smoke_started = time.perf_counter()
    archive_process = subprocess.run(
        [sys.executable, str(rehearsal_root / "main.py")],
        cwd=str(rehearsal_root),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=PHASE40_FINAL_SMOKE_LIMIT_SECONDS,
        check=False,
    )
    archive_smoke_seconds = float(
        time.perf_counter() - smoke_started
    )
    archive_log = (
        (archive_process.stdout or "")
        + "\n"
        + (archive_process.stderr or "")
    )

    assert archive_process.returncode == 0, {
        "message": "Extracted Phase40 archive smoke failed.",
        "return_code": archive_process.returncode,
        "log_tail": [
            line[:300]
            for line in archive_log.splitlines()[-8:]
        ],
    }
    assert archive_smoke_seconds <= PHASE40_FINAL_SMOKE_LIMIT_SECONDS
    assert rehearsal_output.is_file()
    assert not re.search(r"https?://", archive_log, re.IGNORECASE)
    assert not any(
        uid and uid in archive_log
        for uid in PHASE40_SMOKE_UID_ORDER_PRIVATE
    )

    required_lines = (
        "Built with DINOv3.",
        "Initialization complete.",
        "Inference started.",
        "Inference completed.",
    )
    for line in required_lines:
        assert archive_log.count(line) == 1

    forbidden_patterns = {
        "progress_counter": r"\bprogress\b",
        "case_count_counter": r"\bcases\s*=",
        "fallback_counter": r"\bfallback(?:s)?\s*=",
        "unknown_route_counter": r"\bunknown_routes\s*=",
        "private_diagnostics": r"\bprivate diagnostics\b",
        "detailed_failure": r"\bexecution failed\s*:",
        "uid_field": r"\buid\b",
        "prediction_field": r"\bis_pathologic\b",
    }
    forbidden_hits = {
        name: bool(re.search(pattern, archive_log, re.IGNORECASE))
        for name, pattern in forbidden_patterns.items()
    }
    assert not any(forbidden_hits.values()), forbidden_hits

    safe_archive_log_lines = [
        line.strip()
        for line in archive_log.splitlines()
        if line.strip() in required_lines
    ]
    assert tuple(safe_archive_log_lines) == required_lines

    extracted_submission = pd.read_csv(
        rehearsal_output,
        dtype={"uid": str},
    )
    assert list(extracted_submission.columns) == [
        "uid",
        "is_pathologic",
    ]
    assert len(extracted_submission) == PHASE40_FINAL_EXPECTED_SMOKE_CASES
    assert tuple(
        extracted_submission["uid"].astype(str).tolist()
    ) == PHASE40_SMOKE_UID_ORDER_PRIVATE

    archive_probabilities = extracted_submission[
        "is_pathologic"
    ].to_numpy(dtype=np.float64)
    assert np.isfinite(archive_probabilities).all()
    assert np.all(archive_probabilities >= 1e-5)
    assert np.all(archive_probabilities <= 1.0 - 1e-5)
    assert float(np.ptp(archive_probabilities)) > 0.0

    extracted_staged_probability_error = float(np.max(np.abs(
        archive_probabilities
        - PHASE40_STAGED_SMOKE_PROBABILITY_PRIVATE
    )))
    assert extracted_staged_probability_error <= 1e-12

finally:
    if rehearsal_root.exists():
        shutil.rmtree(rehearsal_root)

assert archive_process is not None
assert archive_probabilities is not None
assert not rehearsal_root.exists()


# Publish atomically only after the exact archive passed the real smoke test.
os.replace(PHASE40_FINAL_CANDIDATE, PHASE40_FINAL_ZIP)
assert PHASE40_FINAL_ZIP.is_file()

with zipfile.ZipFile(PHASE40_FINAL_ZIP, mode="r") as final_archive:
    final_members = final_archive.namelist()
    assert final_archive.testzip() is None
    assert tuple(final_members) == PHASE40_ARCHIVE_MANIFEST_PRIVATE

submission_sha256 = phase40_final_sha256(PHASE40_FINAL_ZIP)
assert submission_sha256 == PHASE40_ARCHIVE_CANDIDATE_SHA256_PRIVATE

uncompressed_size_bytes = sum(
    (Path("/kaggle/working/phase40_experimental_submission") / name)
    .stat().st_size
    for name in final_members
)

report = {
    "phase": "phase40_final_submission_archive",
    "status": "accepted",
    "zip_name": PHASE40_FINAL_ZIP.name,
    "archive_file_count": len(final_members),
    "compressed_size_mb": round(
        PHASE40_FINAL_ZIP.stat().st_size / 1e6,
        6,
    ),
    "uncompressed_size_mb": round(
        uncompressed_size_bytes / 1e6,
        6,
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "deterministic_archive_metadata": True,
    "manifest_contract": (
        "accepted_phase39_archive_members_plus_two_phase40_files"
    ),
    "phase12_model_count": 21,
    "dinov3_checkpoint_count": 1,
    "phase33_model_count": 9,
    "phase36_booster_count": 6,
    "p3ca_state_count": 1,
    "route_policy": {
        "known_prototype": "single_held_out_fold",
        "unknown_prototype": "phase12c_anchor_only",
        "three_fold_residual_average": False,
    },
    "extracted_archive_smoke": {
        "status": "accepted",
        "case_count": PHASE40_FINAL_EXPECTED_SMOKE_CASES,
        "return_code": int(archive_process.returncode),
        "elapsed_seconds": archive_smoke_seconds,
        "maximum_allowed_seconds": PHASE40_FINAL_SMOKE_LIMIT_SECONDS,
        "all_probabilities_finite": True,
        "all_probabilities_clipped": True,
        "output_nonconstant": True,
        "staged_probability_maximum_error": (
            extracted_staged_probability_error
        ),
        "generic_logging_only": True,
        "forbidden_pattern_count": 0,
        "network_url_present": False,
        "temporary_rows_deleted": True,
    },
    "submission_sha256": submission_sha256,
    "dinov3_source_present": True,
    "required_attribution_present": True,
    "dinov3_license_present": True,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_oof_arrays": False,
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "model_hashes_displayed": False,
    "total_finalization_seconds": round(
        time.perf_counter() - phase40_134c_started,
        3,
    ),
}

PHASE40_FINAL_CONTRACT_PATH.write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
PHASE40_FINAL_ARCHIVE_REPORT = report

print("BEGIN PHASE40_EXTRACTED_ARCHIVE_SMOKE_LOG")
for line in safe_archive_log_lines:
    print(line)
print("END PHASE40_EXTRACTED_ARCHIVE_SMOKE_LOG")
print("BEGIN SANITIZED_PHASE40_FINAL_ARCHIVE")
print(json.dumps(report, indent=2))
print("END SANITIZED_PHASE40_FINAL_ARCHIVE")
print("Final Phase40 submission archive:")
print(PHASE40_FINAL_ZIP)

BEGIN PHASE40_EXTRACTED_ARCHIVE_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed.
END PHASE40_EXTRACTED_ARCHIVE_SMOKE_LOG
BEGIN SANITIZED_PHASE40_FINAL_ARCHIVE
{
  "phase": "phase40_final_submission_archive",
  "status": "accepted",
  "zip_name": "phase40_submission.zip",
  "archive_file_count": 89,
  "compressed_size_mb": 185.823937,
  "uncompressed_size_mb": 202.691422,
  "root_main_present": true,
  "wrapping_directory_present": false,
  "archive_integrity_test_passed": true,
  "deterministic_archive_metadata": true,
  "manifest_contract": "accepted_phase39_archive_members_plus_two_phase40_files",
  "phase12_model_count": 21,
  "dinov3_checkpoint_count": 1,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "p3ca_state_count": 1,
  "route_policy": {
    "known_prototype": "single_held_out_fold",
    "unknown_prototype": "phase12c_anchor_only",
    "three_fold_residual_average": false
  },
  "extracted_archive_smoke": {
    "status

> Phase 41

In [10]:
# Phase41 Cell 135A
# Restore the exact training index, acquisition groups, and nested partitions
# from persisted training data. No voxel arrays, smoke data, or test data read.

from __future__ import annotations

import itertools
import json
import os
import time
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler


PHASE41_STARTED = time.perf_counter()
PHASE41_SEED = 20260727
EXPECTED_CASES = 1362

PHASE41_EXPECTED_GROUP_SIZES = {
    0: 39,
    1: 456,
    2: 145,
    3: 255,
    4: 76,
    5: 7,
    6: 49,
    7: 208,
    8: 32,
    9: 4,
    10: 35,
    11: 10,
    12: 32,
    13: 9,
    14: 5,
}

# Frozen acquisition-group-to-held-out-fold assignment accepted in Phase40.
PHASE41_GROUP_TO_OUTER_FOLD = {
    0: 1,
    1: 0,
    2: 1,
    3: 2,
    4: 2,
    5: 0,
    6: 2,
    7: 1,
    8: 2,
    9: 0,
    10: 2,
    11: 1,
    12: 1,
    13: 1,
    14: 2,
}

PHASE41_EXPECTED_PARTITIONS = {
    0: {
        "fit_n": 721,
        "monitor_n": 174,
        "outer_train_n": 895,
        "outer_valid_n": 467,
        "monitor_positive": 91,
    },
    1: {
        "fit_n": 796,
        "monitor_n": 123,
        "outer_train_n": 919,
        "outer_valid_n": 443,
        "monitor_positive": 69,
    },
    2: {
        "fit_n": 714,
        "monitor_n": 196,
        "outer_train_n": 910,
        "outer_valid_n": 452,
        "monitor_positive": 102,
    },
}


def phase41_nifti_uid(path):
    name = Path(path).name

    if name.endswith(".nii.gz"):
        return name[:-7]

    if name.endswith(".nii"):
        return name[:-4]

    raise ValueError(f"Unexpected NIfTI filename: {name}")


def phase41_discover_training_dataset():
    preferred_roots = [
        Path(
            os.environ.get(
                "DAT_PRIVATE_ROOT",
                "/kaggle/input/datasets/nahinalam/"
                "drivendata-dataset",
            )
        ),
        Path("/kaggle/input/drivendata-dataset"),
    ]

    label_candidates = []

    for root in preferred_roots:
        candidate = root / "train_labels.csv"
        if candidate.is_file():
            label_candidates.append(candidate)

    kaggle_input = Path("/kaggle/input")

    if kaggle_input.is_dir():
        for candidate in kaggle_input.rglob(
            "train_labels.csv"
        ):
            candidate_text = str(candidate).lower()

            if (
                "smoke" in candidate_text
                or "test" in candidate.parent.name.lower()
            ):
                continue

            if candidate not in label_candidates:
                label_candidates.append(candidate)

    attempts = []

    for label_path in label_candidates:
        try:
            frame = pd.read_csv(label_path)

            uid_column = next(
                (
                    column
                    for column in [
                        "uid",
                        "case_id",
                        "id",
                    ]
                    if column in frame.columns
                ),
                None,
            )

            label_column = next(
                (
                    column
                    for column in [
                        "is_pathologic",
                        "label",
                        "target",
                        "y",
                    ]
                    if column in frame.columns
                ),
                None,
            )

            if (
                len(frame) != EXPECTED_CASES
                or uid_column is None
                or label_column is None
            ):
                attempts.append({
                    "labels": str(label_path),
                    "reason": "schema_or_case_count",
                })
                continue

            image_directories = [
                label_path.parent / "niftis",
                label_path.parent / "train_niftis",
                label_path.parent / "train_images",
                label_path.parent / "images",
            ]

            for image_directory in image_directories:
                if not image_directory.is_dir():
                    continue

                paths = sorted([
                    path
                    for path in image_directory.iterdir()
                    if path.is_file()
                    and (
                        path.name.endswith(".nii")
                        or path.name.endswith(".nii.gz")
                    )
                ])

                if len(paths) != EXPECTED_CASES:
                    continue

                path_by_uid = {
                    phase41_nifti_uid(path): path
                    for path in paths
                }

                label_uids = (
                    frame[uid_column]
                    .astype(str)
                    .tolist()
                )

                if (
                    len(path_by_uid) != EXPECTED_CASES
                    or set(label_uids)
                    != set(path_by_uid)
                ):
                    continue

                labels = np.asarray(
                    frame[label_column],
                    dtype=np.int64,
                )

                if not set(np.unique(labels)).issubset(
                    {0, 1}
                ):
                    continue

                ordered_paths = [
                    path_by_uid[uid]
                    for uid in label_uids
                ]

                return {
                    "labels_path": label_path,
                    "image_directory": image_directory,
                    "uids": label_uids,
                    "labels": labels,
                    "paths": ordered_paths,
                    "uid_column": uid_column,
                    "label_column": label_column,
                    "attempts": attempts,
                }

        except Exception as exc:
            attempts.append({
                "labels": str(label_path),
                "reason": type(exc).__name__,
            })

    raise AssertionError({
        "message": (
            "Could not resolve the 1362-case training "
            "dataset."
        ),
        "candidate_count": len(label_candidates),
        "attempts": attempts,
    })


phase41_dataset = phase41_discover_training_dataset()

nifti_paths = list(phase41_dataset["paths"])
PHASE19_LABELS = np.asarray(
    phase41_dataset["labels"],
    dtype=np.int64,
)
label = PHASE19_LABELS
y = PHASE19_LABELS

assert len(nifti_paths) == EXPECTED_CASES
assert PHASE19_LABELS.shape == (EXPECTED_CASES,)
assert int(np.sum(PHASE19_LABELS == 0)) == 615
assert int(np.sum(PHASE19_LABELS == 1)) == 747


def phase41_header_features(path):
    image = nib.load(str(path), mmap=True)

    assert len(image.shape) == 3

    shape = np.asarray(
        image.shape,
        dtype=np.float64,
    )
    spacing = np.asarray(
        image.header.get_zooms()[:3],
        dtype=np.float64,
    )
    affine = np.asarray(
        image.affine,
        dtype=np.float64,
    )

    assert np.isfinite(shape).all()
    assert np.isfinite(spacing).all()
    assert np.isfinite(affine).all()
    assert np.all(shape > 0)
    assert np.all(spacing > 0)
    assert abs(np.linalg.det(affine[:3, :3])) > 1e-8

    field_of_view = shape * spacing

    return {
        "shape_x": float(shape[0]),
        "shape_y": float(shape[1]),
        "shape_z": float(shape[2]),
        "spacing_x": float(spacing[0]),
        "spacing_y": float(spacing[1]),
        "spacing_z": float(spacing[2]),
        "fov_x": float(field_of_view[0]),
        "fov_y": float(field_of_view[1]),
        "fov_z": float(field_of_view[2]),
        "dtype_is_int16": int(
            str(image.get_data_dtype()) == "int16"
        ),
        "orientation": "".join(
            nib.aff2axcodes(affine)
        ),
    }


phase41_header_rows = [
    phase41_header_features(path)
    for path in nifti_paths
]

phase41_header_df = pd.DataFrame(
    phase41_header_rows
)

assert (
    phase41_header_df["orientation"] == "RAS"
).all()

PHASE41_HEADER_COLUMNS = [
    "shape_x",
    "shape_y",
    "shape_z",
    "spacing_x",
    "spacing_y",
    "spacing_z",
    "fov_x",
    "fov_y",
    "fov_z",
    "dtype_is_int16",
]

phase41_header_matrix = phase41_header_df[
    PHASE41_HEADER_COLUMNS
].to_numpy(dtype=np.float64)

phase41_cluster_matrix = (
    StandardScaler().fit_transform(
        np.log1p(phase41_header_matrix)
    )
)

phase41_cluster_model = KMeans(
    n_clusters=15,
    random_state=PHASE41_SEED,
    n_init=25,
)

phase20_acquisition_groups = (
    phase41_cluster_model.fit_predict(
        phase41_cluster_matrix
    ).astype(np.int64)
)

acquisition_group = phase20_acquisition_groups
groups = phase20_acquisition_groups

phase41_observed_group_sizes = {
    int(group): int(
        np.sum(
            phase20_acquisition_groups == group
        )
    )
    for group in np.unique(
        phase20_acquisition_groups
    )
}

assert phase41_observed_group_sizes == (
    PHASE41_EXPECTED_GROUP_SIZES
), {
    "message": (
        "Frozen acquisition-group identities were not "
        "reproduced."
    ),
    "observed": phase41_observed_group_sizes,
    "expected": PHASE41_EXPECTED_GROUP_SIZES,
}

phase41_outer_fold = np.asarray([
    PHASE41_GROUP_TO_OUTER_FOLD[int(group)]
    for group in phase20_acquisition_groups
], dtype=np.int64)

case_df = pd.DataFrame({
    "uid": phase41_dataset["uids"],
    "label": PHASE19_LABELS,
    "y": PHASE19_LABELS,
    "path": nifti_paths,
    "acquisition_group": (
        phase20_acquisition_groups
    ),
    "fold": phase41_outer_fold,
})

for column in PHASE41_HEADER_COLUMNS:
    case_df[column] = phase41_header_df[
        column
    ].to_numpy()


def phase41_select_monitor_partition(
    outer_train_indices,
):
    available_groups = np.asarray(
        sorted(
            np.unique(
                phase20_acquisition_groups[
                    outer_train_indices
                ]
            ).tolist()
        ),
        dtype=np.int64,
    )

    target_n = (
        len(outer_train_indices) * 0.20
    )
    target_prevalence = float(
        PHASE19_LABELS[
            outer_train_indices
        ].mean()
    )

    best = None

    for group_count in range(
        1,
        len(available_groups),
    ):
        for group_combination in (
            itertools.combinations(
                available_groups.tolist(),
                group_count,
            )
        ):
            monitor_mask = np.isin(
                phase20_acquisition_groups[
                    outer_train_indices
                ],
                np.asarray(
                    group_combination,
                    dtype=np.int64,
                ),
            )

            monitor_indices = (
                outer_train_indices[
                    monitor_mask
                ]
            )
            fit_indices = (
                outer_train_indices[
                    ~monitor_mask
                ]
            )

            if len(monitor_indices) < 80:
                continue

            if (
                len(fit_indices)
                < 0.60 * len(outer_train_indices)
            ):
                continue

            if (
                np.unique(
                    PHASE19_LABELS[
                        monitor_indices
                    ]
                ).size != 2
                or np.unique(
                    PHASE19_LABELS[
                        fit_indices
                    ]
                ).size != 2
            ):
                continue

            size_term = (
                (
                    len(monitor_indices)
                    - target_n
                )
                / target_n
            ) ** 2

            prevalence_term = (
                (
                    float(
                        PHASE19_LABELS[
                            monitor_indices
                        ].mean()
                    )
                    - target_prevalence
                )
                / 0.10
            ) ** 2

            group_term = (
                0.002
                * (len(group_combination) - 2.0)
                ** 2
            )

            candidate = (
                size_term
                + prevalence_term
                + group_term,
                tuple(group_combination),
            )

            if (
                best is None
                or candidate < best
            ):
                best = candidate

    assert best is not None

    monitor_groups = np.asarray(
        best[1],
        dtype=np.int64,
    )

    monitor_mask = np.isin(
        phase20_acquisition_groups[
            outer_train_indices
        ],
        monitor_groups,
    )

    monitor_indices = (
        outer_train_indices[monitor_mask]
    )
    fit_indices = (
        outer_train_indices[~monitor_mask]
    )

    return (
        fit_indices.astype(np.int64),
        monitor_indices.astype(np.int64),
        monitor_groups,
    )


PHASE19_PARTITIONS = []
phase41_partition_records = []

for fold in range(3):
    outer_valid = np.flatnonzero(
        phase41_outer_fold == fold
    ).astype(np.int64)

    outer_train = np.flatnonzero(
        phase41_outer_fold != fold
    ).astype(np.int64)

    (
        fit_indices,
        monitor_indices,
        monitor_groups,
    ) = phase41_select_monitor_partition(
        outer_train
    )

    partition = {
        "fold": int(fold),
        "fit": fit_indices,
        "monitor": monitor_indices,
        "outer_train": outer_train,
        "outer_valid": outer_valid,
    }

    expected = (
        PHASE41_EXPECTED_PARTITIONS[fold]
    )

    assert len(fit_indices) == expected["fit_n"]
    assert (
        len(monitor_indices)
        == expected["monitor_n"]
    )
    assert (
        len(outer_train)
        == expected["outer_train_n"]
    )
    assert (
        len(outer_valid)
        == expected["outer_valid_n"]
    )
    assert int(
        PHASE19_LABELS[
            monitor_indices
        ].sum()
    ) == expected["monitor_positive"]

    assert len(
        np.intersect1d(
            fit_indices,
            monitor_indices,
        )
    ) == 0

    assert len(
        np.intersect1d(
            outer_train,
            outer_valid,
        )
    ) == 0

    PHASE19_PARTITIONS.append(partition)

    phase41_partition_records.append({
        "fold": int(fold),
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(
            len(monitor_indices)
        ),
        "outer_train_n": int(
            len(outer_train)
        ),
        "outer_valid_n": int(
            len(outer_valid)
        ),
        "monitor_group_count": int(
            len(monitor_groups)
        ),
        "monitor_prevalence": round(
            float(
                PHASE19_LABELS[
                    monitor_indices
                ].mean()
            ),
            6,
        ),
    })

phase41_validation_coverage = np.zeros(
    EXPECTED_CASES,
    dtype=np.int64,
)

for partition in PHASE19_PARTITIONS:
    phase41_validation_coverage[
        partition["outer_valid"]
    ] += 1

assert np.all(
    phase41_validation_coverage == 1
)

# Compatibility aliases used by earlier notebook cells.
PHASE31_PARTITIONS = PHASE19_PARTITIONS
phase19_partitions = PHASE19_PARTITIONS

phase41_report = {
    "phase": (
        "phase41_fresh_kernel_state_restore"
    ),
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "normal_count": int(
        np.sum(PHASE19_LABELS == 0)
    ),
    "pathologic_count": int(
        np.sum(PHASE19_LABELS == 1)
    ),
    "training_nifti_count": len(
        nifti_paths
    ),
    "header_feature_count": len(
        PHASE41_HEADER_COLUMNS
    ),
    "acquisition_group_count": int(
        np.unique(
            phase20_acquisition_groups
        ).size
    ),
    "group_sizes_exact": (
        phase41_observed_group_sizes
        == PHASE41_EXPECTED_GROUP_SIZES
    ),
    "outer_fold_counts": {
        str(fold): int(
            np.sum(
                phase41_outer_fold == fold
            )
        )
        for fold in range(3)
    },
    "partitions": phase41_partition_records,
    "training_nifti_headers_read": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE41_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE41_FRESH_STATE"
)
print(
    json.dumps(
        phase41_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE41_FRESH_STATE"
)

BEGIN SANITIZED_PHASE41_FRESH_STATE
{
  "phase": "phase41_fresh_kernel_state_restore",
  "status": "accepted",
  "case_count": 1362,
  "normal_count": 615,
  "pathologic_count": 747,
  "training_nifti_count": 1362,
  "header_feature_count": 10,
  "acquisition_group_count": 15,
  "group_sizes_exact": true,
  "outer_fold_counts": {
    "0": 467,
    "1": 443,
    "2": 452
  },
  "partitions": [
    {
      "fold": 0,
      "fit_n": 721,
      "monitor_n": 174,
      "outer_train_n": 895,
      "outer_valid_n": 467,
      "monitor_group_count": 4,
      "monitor_prevalence": 0.522989
    },
    {
      "fold": 1,
      "fit_n": 796,
      "monitor_n": 123,
      "outer_train_n": 919,
      "outer_valid_n": 443,
      "monitor_group_count": 4,
      "monitor_prevalence": 0.560976
    },
    {
      "fold": 2,
      "fit_n": 714,
      "monitor_n": 196,
      "outer_train_n": 910,
      "outer_valid_n": 452,
      "monitor_group_count": 4,
      "monitor_prevalence": 0.520408
    }
  ],
  "

In [2]:
# Phase41 Cell 135B
# Recover the persisted cross-fitted components needed for Phase41.
# Phase36 is reconstructed exactly from the uncapped Phase39 residual.

from scipy.special import expit
from sklearn.metrics import log_loss, roc_auc_score


PHASE41_COMPONENT_STARTED = (
    time.perf_counter()
)
PHASE41_WORKING = Path("/kaggle/working")


def phase41_resolve_file(
    preferred_paths,
    filename,
):
    attempts = []

    for path in preferred_paths:
        path = Path(path)
        attempts.append(str(path))

        if path.is_file():
            return path, attempts

    matches = sorted(
        PHASE41_WORKING.rglob(filename)
    )

    # Resolve duplicate references to the same physical file.
    unique_matches = {}

    for match in matches:
        unique_matches[
            str(match.resolve())
        ] = match

    matches = list(unique_matches.values())

    assert len(matches) == 1, {
        "message": (
            f"Could not uniquely resolve {filename}."
        ),
        "candidate_count": len(matches),
        "attempts": attempts,
        "matches": [
            str(path)
            for path in matches
        ],
    }

    return matches[0], attempts


def phase41_load_vector(path, kind):
    vector = np.asarray(
        np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        ),
        dtype=np.float64,
    ).reshape(-1)

    assert vector.shape == (
        EXPECTED_CASES,
    ), {
        "path": str(path),
        "shape": list(vector.shape),
    }

    assert np.isfinite(vector).all(), {
        "path": str(path),
        "message": "Non-finite values.",
    }

    if kind == "probability":
        assert np.all(vector >= 0.0)
        assert np.all(vector <= 1.0)

    return vector


def phase41_logit(probability):
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        1e-7,
        1.0 - 1e-7,
    )

    return np.log(
        probability / (1.0 - probability)
    )


def phase41_metrics(probability):
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        1e-7,
        1.0 - 1e-7,
    )

    return {
        "log_loss": float(
            log_loss(
                PHASE19_LABELS,
                probability,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                PHASE19_LABELS,
                probability,
            )
        ),
        "mean_probability": float(
            probability.mean()
        ),
    }


phase41_baseline_path, _ = (
    phase41_resolve_file(
        [
            PHASE41_WORKING
            / "phase32_phase12c_oof_float64.npy",
        ],
        "phase32_phase12c_oof_float64.npy",
    )
)

phase41_phase33_path, _ = (
    phase41_resolve_file(
        [
            PHASE41_WORKING
            / "phase33_private_checkpoint"
            / "phase33_component_oof_float64.npy",
        ],
        "phase33_component_oof_float64.npy",
    )
)

phase41_combined_residual_path, _ = (
    phase41_resolve_file(
        [
            PHASE41_WORKING
            / "phase39_private_checkpoint"
            / "phase39_combined_residual_float64.npy",
        ],
        "phase39_combined_residual_float64.npy",
    )
)

phase41_phase39_path, _ = (
    phase41_resolve_file(
        [
            PHASE41_WORKING
            / "phase39_private_checkpoint"
            / "phase39_oof_float64.npy",
        ],
        "phase39_oof_float64.npy",
    )
)

PHASE41_BASELINE_OOF_PRIVATE = (
    phase41_load_vector(
        phase41_baseline_path,
        "probability",
    )
)

PHASE41_PHASE33_COMPONENT_OOF_PRIVATE = (
    phase41_load_vector(
        phase41_phase33_path,
        "probability",
    )
)

PHASE41_PHASE39_COMBINED_RESIDUAL_PRIVATE = (
    phase41_load_vector(
        phase41_combined_residual_path,
        "residual",
    )
)

PHASE41_PHASE39_OOF_PRIVATE = (
    phase41_load_vector(
        phase41_phase39_path,
        "probability",
    )
)

phase41_baseline_logit = phase41_logit(
    PHASE41_BASELINE_OOF_PRIVATE
)

phase41_phase33_logit = phase41_logit(
    PHASE41_PHASE33_COMPONENT_OOF_PRIVATE
)

PHASE41_PHASE33_RESIDUAL_PRIVATE = (
    phase41_phase33_logit
    - phase41_baseline_logit
)

# Phase39 persisted the uncapped residual:
#
# r_combined = 0.75*r36_raw + 0.25*r33
#
# Therefore r36_raw is exactly recoverable.
PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE = (
    (
        PHASE41_PHASE39_COMBINED_RESIDUAL_PRIVATE
        - 0.25
        * PHASE41_PHASE33_RESIDUAL_PRIVATE
    )
    / 0.75
)

PHASE41_PHASE36_OOF_PRIVATE = expit(
    phase41_baseline_logit
    + 0.75
    * PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE
)

phase41_phase39_reconstructed = expit(
    phase41_baseline_logit
    + np.clip(
        PHASE41_PHASE39_COMBINED_RESIDUAL_PRIVATE,
        -2.0,
        2.0,
    )
)

phase41_phase39_reconstruction_error = (
    np.abs(
        phase41_phase39_reconstructed
        - PHASE41_PHASE39_OOF_PRIVATE
    )
)

assert (
    float(
        phase41_phase39_reconstruction_error.max()
    )
    <= 5e-7
), {
    "message": (
        "Persisted Phase39 residual semantics "
        "did not reconstruct Phase39."
    ),
    "maximum_error": float(
        phase41_phase39_reconstruction_error.max()
    ),
}

phase41_component_metrics = {
    "phase12c": phase41_metrics(
        PHASE41_BASELINE_OOF_PRIVATE
    ),
    "phase33_component": phase41_metrics(
        PHASE41_PHASE33_COMPONENT_OOF_PRIVATE
    ),
    "phase36": phase41_metrics(
        PHASE41_PHASE36_OOF_PRIVATE
    ),
    "phase39": phase41_metrics(
        PHASE41_PHASE39_OOF_PRIVATE
    ),
}

phase41_expected_metrics = {
    "phase12c": {
        "log_loss": 0.307616,
        "auroc": 0.938914,
    },
    "phase33_component": {
        "log_loss": 0.358791,
        "auroc": 0.923527,
    },
    "phase36": {
        "log_loss": 0.295657,
        "auroc": 0.944143,
    },
    "phase39": {
        "log_loss": 0.293027,
        "auroc": 0.945575,
    },
}

for component, expected in (
    phase41_expected_metrics.items()
):
    observed = (
        phase41_component_metrics[component]
    )

    assert abs(
        observed["log_loss"]
        - expected["log_loss"]
    ) <= 5e-5, {
        "component": component,
        "observed": observed,
        "expected": expected,
    }

    assert abs(
        observed["auroc"]
        - expected["auroc"]
    ) <= 5e-5, {
        "component": component,
        "observed": observed,
        "expected": expected,
    }


def phase41_optional_oof(filename):
    matches = sorted(
        PHASE41_WORKING.rglob(filename)
    )

    unique_matches = {
        str(path.resolve()): path
        for path in matches
    }

    if len(unique_matches) != 1:
        return None, len(unique_matches)

    path = next(
        iter(unique_matches.values())
    )

    try:
        return (
            phase41_load_vector(
                path,
                "probability",
            ),
            1,
        )
    except Exception:
        return None, 1


PHASE41_PHASE31_OOF_PRIVATE, (
    phase41_phase31_candidate_count
) = phase41_optional_oof(
    "phase31_classical_oof_private.npy"
)

PHASE41_PHASE38_OOF_PRIVATE, (
    phase41_phase38_candidate_count
) = phase41_optional_oof(
    "phase38_oof.npy"
)

# Phase30 OOF was often retained only in notebook memory.
# Search only explicitly named private OOF files; never infer it
# from leaderboard scores or unrelated predictions.
phase41_phase30_candidates = []

for pattern in [
    "*phase30*oof*.npy",
    "*PHASE30*OOF*.npy",
]:
    phase41_phase30_candidates.extend(
        PHASE41_WORKING.rglob(pattern)
    )

phase41_phase30_candidates = {
    str(path.resolve()): path
    for path in phase41_phase30_candidates
}

PHASE41_PHASE30_OOF_PRIVATE = None

for path in phase41_phase30_candidates.values():
    try:
        candidate = phase41_load_vector(
            path,
            "probability",
        )
        candidate_metrics = phase41_metrics(
            candidate
        )

        if (
            abs(
                candidate_metrics["log_loss"]
                - 0.298375
            ) <= 5e-4
            and abs(
                candidate_metrics["auroc"]
                - 0.941627
            ) <= 5e-4
        ):
            PHASE41_PHASE30_OOF_PRIVATE = (
                candidate
            )
            break

    except Exception:
        pass

# Compatibility aliases for later Phase41 cells.
PHASE31_BASELINE_OOF_PRIVATE = (
    PHASE41_BASELINE_OOF_PRIVATE
)
PHASE30_PHASE12C_OOF_PRIVATE = (
    PHASE41_BASELINE_OOF_PRIVATE
)
PHASE33_OOF_PRIVATE = (
    PHASE41_PHASE33_COMPONENT_OOF_PRIVATE
)
PHASE36_RAW_RESIDUAL_OOF_PRIVATE = (
    PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE
)
PHASE36_OOF_PRIVATE = (
    PHASE41_PHASE36_OOF_PRIVATE
)
PHASE39_COMBINED_RESIDUAL_OOF_PRIVATE = (
    PHASE41_PHASE39_COMBINED_RESIDUAL_PRIVATE
)
PHASE39_OOF_PRIVATE = (
    PHASE41_PHASE39_OOF_PRIVATE
)

# For training OOF evaluation, exact Phase40 routing selects the
# held-out component fold and therefore has the Phase39 OOF vector.
PHASE40_OOF_PRIVATE = (
    PHASE41_PHASE39_OOF_PRIVATE.copy()
)

phase41_optional_metrics = {}

if PHASE41_PHASE31_OOF_PRIVATE is not None:
    phase41_optional_metrics["phase31"] = (
        phase41_metrics(
            PHASE41_PHASE31_OOF_PRIVATE
        )
    )

if PHASE41_PHASE38_OOF_PRIVATE is not None:
    phase41_optional_metrics["phase38"] = (
        phase41_metrics(
            PHASE41_PHASE38_OOF_PRIVATE
        )
    )

if PHASE41_PHASE30_OOF_PRIVATE is not None:
    phase41_optional_metrics["phase30"] = (
        phase41_metrics(
            PHASE41_PHASE30_OOF_PRIVATE
        )
    )

phase41_component_report = {
    "phase": (
        "phase41_cross_fitted_component_restore"
    ),
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "required_components": (
        phase41_component_metrics
    ),
    "optional_components": (
        phase41_optional_metrics
    ),
    "phase30_oof_available": (
        PHASE41_PHASE30_OOF_PRIVATE
        is not None
    ),
    "phase31_oof_available": (
        PHASE41_PHASE31_OOF_PRIVATE
        is not None
    ),
    "phase38_oof_available": (
        PHASE41_PHASE38_OOF_PRIVATE
        is not None
    ),
    "phase39_reconstruction_error": {
        "maximum": float(
            phase41_phase39_reconstruction_error.max()
        ),
        "mean": float(
            phase41_phase39_reconstruction_error.mean()
        ),
    },
    "phase36_reconstruction": (
        "exactly_solved_from_uncapped_phase39_"
        "combined_residual_and_phase33_residual"
    ),
    "phase36_raw_residual_scaled_exactly_once": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE41_COMPONENT_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE41_COMPONENT_RESTORE"
)
print(
    json.dumps(
        phase41_component_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE41_COMPONENT_RESTORE"
)

BEGIN SANITIZED_PHASE41_COMPONENT_RESTORE
{
  "phase": "phase41_cross_fitted_component_restore",
  "status": "accepted",
  "case_count": 1362,
  "required_components": {
    "phase12c": {
      "log_loss": 0.30761581113729136,
      "auroc": 0.9389144654498752,
      "mean_probability": 0.5306978306863394
    },
    "phase33_component": {
      "log_loss": 0.35879075331189647,
      "auroc": 0.9235271710146822,
      "mean_probability": 0.5589990863124595
    },
    "phase36": {
      "log_loss": 0.295657193497576,
      "auroc": 0.9441429675340931,
      "mean_probability": 0.5534928323061861
    },
    "phase39": {
      "log_loss": 0.29302734886541737,
      "auroc": 0.9455752549493367,
      "mean_probability": 0.5609995096990729
    }
  },
  "optional_components": {
    "phase31": {
      "log_loss": 0.3758200472357325,
      "auroc": 0.9147963126217606,
      "mean_probability": 0.5505495878735149
    },
    "phase38": {
      "log_loss": 0.30904196751134105,
      "auroc": 0.937

In [3]:
# Phase41 Cell 135C
# Create:
#   1. Same-domain IID development split
#   2. 15-fold leave-one-acquisition-group-out evaluation
#   3. Label-free prospective shadow-group split
#
# The prospective shadow labels must not be inspected during model selection.

import hashlib
import json
import itertools
from pathlib import Path

PHASE41_SPLIT_STARTED = time.perf_counter()

PHASE41_SPLIT_SEED = 410701
PHASE41_SPLIT_PATH = Path(
    "/kaggle/working/"
    "phase41_validation_partitions_private.npz"
)
PHASE41_SPLIT_METADATA_PATH = Path(
    "/kaggle/working/"
    "phase41_validation_partitions_metadata.json"
)


def phase41_stable_hash(value):
    payload = (
        f"{PHASE41_SPLIT_SEED}|{value}"
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


# ---------------------------------------------------------
# 1. Same-domain IID split
# ---------------------------------------------------------
# Cases are split within acquisition group and label.
# Every selected development case remains unseen at the
# future meta-model level, while acquisition domains overlap.

phase41_iid_dev_mask = np.zeros(
    EXPECTED_CASES,
    dtype=bool,
)

for group in sorted(
    np.unique(
        phase20_acquisition_groups
    ).tolist()
):
    for target in [0, 1]:
        stratum = np.flatnonzero(
            (
                phase20_acquisition_groups
                == group
            )
            & (PHASE19_LABELS == target)
        )

        if len(stratum) <= 1:
            continue

        ordered = sorted(
            stratum.tolist(),
            key=lambda index: phase41_stable_hash(
                case_df.iloc[index]["uid"]
            ),
        )

        development_count = max(
            1,
            int(round(0.20 * len(ordered))),
        )

        development_count = min(
            development_count,
            len(ordered) - 1,
        )

        phase41_iid_dev_mask[
            np.asarray(
                ordered[:development_count],
                dtype=np.int64,
            )
        ] = True

PHASE41_IID_DEV_PRIVATE = np.flatnonzero(
    phase41_iid_dev_mask
).astype(np.int64)

PHASE41_IID_FIT_PRIVATE = np.flatnonzero(
    ~phase41_iid_dev_mask
).astype(np.int64)

assert len(
    np.intersect1d(
        PHASE41_IID_FIT_PRIVATE,
        PHASE41_IID_DEV_PRIVATE,
    )
) == 0

assert (
    len(PHASE41_IID_FIT_PRIVATE)
    + len(PHASE41_IID_DEV_PRIVATE)
    == EXPECTED_CASES
)

phase41_iid_fit_groups = set(
    phase20_acquisition_groups[
        PHASE41_IID_FIT_PRIVATE
    ].tolist()
)

phase41_iid_dev_groups = set(
    phase20_acquisition_groups[
        PHASE41_IID_DEV_PRIVATE
    ].tolist()
)

assert (
    phase41_iid_fit_groups
    == phase41_iid_dev_groups
    == set(range(15))
)


# ---------------------------------------------------------
# 2. Complete 15-fold LOGO protocol
# ---------------------------------------------------------

PHASE41_LOGO_PARTITIONS = []

for held_out_group in range(15):
    validation_indices = np.flatnonzero(
        phase20_acquisition_groups
        == held_out_group
    ).astype(np.int64)

    training_indices = np.flatnonzero(
        phase20_acquisition_groups
        != held_out_group
    ).astype(np.int64)

    assert len(validation_indices) == (
        PHASE41_EXPECTED_GROUP_SIZES[
            held_out_group
        ]
    )

    assert len(
        np.intersect1d(
            training_indices,
            validation_indices,
        )
    ) == 0

    PHASE41_LOGO_PARTITIONS.append({
        "held_out_group": int(
            held_out_group
        ),
        "train": training_indices,
        "valid": validation_indices,
    })


# ---------------------------------------------------------
# 3. Prospective label-free shadow split
# ---------------------------------------------------------
# Selection uses only case counts and header features.
# Labels are deliberately absent from the objective.
#
# This is prospective rather than historically pristine:
# aggregate results from these cases were seen in prior phases.
# We will nevertheless stop using its labels for Phase41
# model/hyperparameter selection.

phase41_group_ids = np.arange(
    15,
    dtype=np.int64,
)

phase41_group_counts = np.asarray([
    PHASE41_EXPECTED_GROUP_SIZES[
        int(group)
    ]
    for group in phase41_group_ids
], dtype=np.float64)

phase41_header_standardized = (
    StandardScaler().fit_transform(
        np.log1p(
            phase41_header_matrix
        )
    )
)

phase41_global_header_mean = np.average(
    phase41_header_standardized,
    axis=0,
)

phase41_global_header_std = np.sqrt(
    np.average(
        (
            phase41_header_standardized
            - phase41_global_header_mean
        ) ** 2,
        axis=0,
    )
)

phase41_global_header_std = np.maximum(
    phase41_global_header_std,
    1e-6,
)

phase41_shadow_target_n = (
    0.20 * EXPECTED_CASES
)

phase41_shadow_candidates = []

for shadow_group_count in [3, 4, 5]:
    for shadow_group_tuple in (
        itertools.combinations(
            phase41_group_ids.tolist(),
            shadow_group_count,
        )
    ):
        shadow_group_array = np.asarray(
            shadow_group_tuple,
            dtype=np.int64,
        )

        shadow_mask = np.isin(
            phase20_acquisition_groups,
            shadow_group_array,
        )

        shadow_n = int(
            shadow_mask.sum()
        )

        if not (
            0.14 * EXPECTED_CASES
            <= shadow_n
            <= 0.28 * EXPECTED_CASES
        ):
            continue

        shadow_header = (
            phase41_header_standardized[
                shadow_mask
            ]
        )

        shadow_mean = np.average(
            shadow_header,
            axis=0,
        )

        shadow_std = np.sqrt(
            np.average(
                (
                    shadow_header
                    - shadow_mean
                ) ** 2,
                axis=0,
            )
        )

        size_penalty = (
            (
                shadow_n
                - phase41_shadow_target_n
            )
            / phase41_shadow_target_n
        ) ** 2

        mean_penalty = float(
            np.mean(
                (
                    (
                        shadow_mean
                        - phase41_global_header_mean
                    )
                    / phase41_global_header_std
                ) ** 2
            )
        )

        std_penalty = float(
            np.mean(
                (
                    (
                        shadow_std
                        - phase41_global_header_std
                    )
                    / phase41_global_header_std
                ) ** 2
            )
        )

        group_count_penalty = (
            0.01
            * (
                shadow_group_count - 4
            ) ** 2
        )

        score = (
            size_penalty
            + 0.20 * mean_penalty
            + 0.05 * std_penalty
            + group_count_penalty
        )

        deterministic_tie_break = (
            phase41_stable_hash(
                ",".join(
                    map(
                        str,
                        shadow_group_tuple,
                    )
                )
            )
        )

        phase41_shadow_candidates.append({
            "score": float(score),
            "tie_break": (
                deterministic_tie_break
            ),
            "groups": shadow_group_tuple,
            "case_count": shadow_n,
        })

assert phase41_shadow_candidates

phase41_shadow_selection = sorted(
    phase41_shadow_candidates,
    key=lambda record: (
        record["score"],
        record["tie_break"],
    ),
)[0]

PHASE41_SHADOW_GROUPS_PRIVATE = (
    np.asarray(
        phase41_shadow_selection[
            "groups"
        ],
        dtype=np.int64,
    )
)

phase41_shadow_mask = np.isin(
    phase20_acquisition_groups,
    PHASE41_SHADOW_GROUPS_PRIVATE,
)

PHASE41_SHADOW_INDICES_PRIVATE = (
    np.flatnonzero(
        phase41_shadow_mask
    ).astype(np.int64)
)

PHASE41_DEVELOPMENT_INDICES_PRIVATE = (
    np.flatnonzero(
        ~phase41_shadow_mask
    ).astype(np.int64)
)

assert len(
    np.intersect1d(
        PHASE41_SHADOW_INDICES_PRIVATE,
        PHASE41_DEVELOPMENT_INDICES_PRIVATE,
    )
) == 0

assert (
    len(PHASE41_SHADOW_INDICES_PRIVATE)
    + len(
        PHASE41_DEVELOPMENT_INDICES_PRIVATE
    )
    == EXPECTED_CASES
)

assert len(
    PHASE41_SHADOW_GROUPS_PRIVATE
) in {3, 4, 5}

# The hash lets us verify that the prospective shadow
# partition never changes in later cells.
phase41_shadow_contract_bytes = (
    PHASE41_SHADOW_INDICES_PRIVATE
    .astype("<i8")
    .tobytes()
)

PHASE41_SHADOW_CONTRACT_SHA256 = (
    hashlib.sha256(
        phase41_shadow_contract_bytes
    ).hexdigest()
)

phase41_logo_fold_id = (
    phase20_acquisition_groups.copy()
)

np.savez_compressed(
    PHASE41_SPLIT_PATH,
    iid_fit=PHASE41_IID_FIT_PRIVATE,
    iid_dev=PHASE41_IID_DEV_PRIVATE,
    logo_fold_id=phase41_logo_fold_id,
    development=(
        PHASE41_DEVELOPMENT_INDICES_PRIVATE
    ),
    shadow=PHASE41_SHADOW_INDICES_PRIVATE,
    shadow_groups=(
        PHASE41_SHADOW_GROUPS_PRIVATE
    ),
)

phase41_split_metadata = {
    "schema_version": 1,
    "phase": (
        "phase41_validation_protocol"
    ),
    "case_count": EXPECTED_CASES,
    "iid_fit_count": int(
        len(PHASE41_IID_FIT_PRIVATE)
    ),
    "iid_dev_count": int(
        len(PHASE41_IID_DEV_PRIVATE)
    ),
    "logo_fold_count": 15,
    "prospective_shadow_count": int(
        len(
            PHASE41_SHADOW_INDICES_PRIVATE
        )
    ),
    "prospective_development_count": int(
        len(
            PHASE41_DEVELOPMENT_INDICES_PRIVATE
        )
    ),
    "prospective_shadow_group_count": int(
        len(
            PHASE41_SHADOW_GROUPS_PRIVATE
        )
    ),
    "shadow_contract_sha256": (
        PHASE41_SHADOW_CONTRACT_SHA256
    ),
    "shadow_selection_used_labels": False,
}

PHASE41_SPLIT_METADATA_PATH.write_text(
    json.dumps(
        phase41_split_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

phase41_split_report = {
    "phase": (
        "phase41_coursera_style_"
        "validation_contract"
    ),
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "same_domain_iid": {
        "fit_n": int(
            len(PHASE41_IID_FIT_PRIVATE)
        ),
        "dev_n": int(
            len(PHASE41_IID_DEV_PRIVATE)
        ),
        "shared_acquisition_group_count": int(
            len(
                phase41_iid_fit_groups
                & phase41_iid_dev_groups
            )
        ),
        "split_within_group_and_label": True,
        "purpose": (
            "estimate case-level variance "
            "when acquisition domains overlap"
        ),
    },
    "leave_one_acquisition_group_out": {
        "fold_count": len(
            PHASE41_LOGO_PARTITIONS
        ),
        "minimum_validation_n": int(
            min(
                len(partition["valid"])
                for partition
                in PHASE41_LOGO_PARTITIONS
            )
        ),
        "maximum_validation_n": int(
            max(
                len(partition["valid"])
                for partition
                in PHASE41_LOGO_PARTITIONS
            )
        ),
        "exact_case_coverage": True,
        "purpose": (
            "estimate acquisition-domain "
            "transport failure"
        ),
    },
    "prospective_shadow": {
        "case_count": int(
            len(
                PHASE41_SHADOW_INDICES_PRIVATE
            )
        ),
        "group_count": int(
            len(
                PHASE41_SHADOW_GROUPS_PRIVATE
            )
        ),
        "development_case_count": int(
            len(
                PHASE41_DEVELOPMENT_INDICES_PRIVATE
            )
        ),
        "selection_label_free": True,
        "contract_sha256": (
            PHASE41_SHADOW_CONTRACT_SHA256
        ),
        "historically_pristine": False,
        "prospectively_locked": True,
        "labels_evaluated_now": False,
    },
    "persistent_private_partition_written": (
        PHASE41_SPLIT_PATH.is_file()
    ),
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE41_SPLIT_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE41_VALIDATION_CONTRACT"
)
print(
    json.dumps(
        phase41_split_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE41_VALIDATION_CONTRACT"
)

BEGIN SANITIZED_PHASE41_VALIDATION_CONTRACT
{
  "phase": "phase41_coursera_style_validation_contract",
  "status": "accepted",
  "case_count": 1362,
  "same_domain_iid": {
    "fit_n": 1088,
    "dev_n": 274,
    "shared_acquisition_group_count": 15,
    "split_within_group_and_label": true,
    "purpose": "estimate case-level variance when acquisition domains overlap"
  },
  "leave_one_acquisition_group_out": {
    "fold_count": 15,
    "minimum_validation_n": 4,
    "maximum_validation_n": 456,
    "exact_case_coverage": true,
    "purpose": "estimate acquisition-domain transport failure"
  },
  "prospective_shadow": {
    "case_count": 257,
    "group_count": 4,
    "development_case_count": 1105,
    "selection_label_free": true,
    "contract_sha256": "6f11fdf646f7468d8e641faa10f8b7b01d54c6850bc732449ab8852cf3d017f2",
    "historically_pristine": false,
    "prospectively_locked": true,
    "labels_evaluated_now": false
  },
  "persistent_private_partition_written": true,
  "train

In [4]:
# Phase41 Cell 135D
# Diagnose existing components before searching new weights.
#
# This cell does not evaluate the newly locked prospective
# shadow partition separately. It uses the historical complete
# OOF only for aggregate model diagnosis.

from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression

PHASE41_DIAGNOSTIC_STARTED = (
    time.perf_counter()
)


def phase41_safe_probability(values):
    return np.clip(
        np.asarray(
            values,
            dtype=np.float64,
        ).reshape(-1),
        1e-7,
        1.0 - 1e-7,
    )


def phase41_subset_metrics(
    probability,
    indices,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    labels_subset = PHASE19_LABELS[
        indices
    ]

    probability_subset = (
        phase41_safe_probability(
            probability
        )[indices]
    )

    record = {
        "n": int(len(indices)),
        "log_loss": float(
            log_loss(
                labels_subset,
                probability_subset,
                labels=[0, 1],
            )
        ),
        "mean_probability": float(
            probability_subset.mean()
        ),
        "prevalence": float(
            labels_subset.mean()
        ),
    }

    if np.unique(labels_subset).size == 2:
        record["auroc"] = float(
            roc_auc_score(
                labels_subset,
                probability_subset,
            )
        )
    else:
        record["auroc"] = None

    return record


phase41_components = {
    "phase12c": (
        PHASE41_BASELINE_OOF_PRIVATE
    ),
    "phase31": (
        PHASE41_PHASE31_OOF_PRIVATE
    ),
    "phase33": (
        PHASE41_PHASE33_COMPONENT_OOF_PRIVATE
    ),
    "phase36": (
        PHASE41_PHASE36_OOF_PRIVATE
    ),
    "phase38": (
        PHASE41_PHASE38_OOF_PRIVATE
    ),
    "phase39": (
        PHASE41_PHASE39_OOF_PRIVATE
    ),
}

phase41_components = {
    name: phase41_safe_probability(
        probability
    )
    for name, probability
    in phase41_components.items()
    if probability is not None
}

phase41_baseline_probability = (
    phase41_components["phase12c"]
)

phase41_baseline_logit = phase41_logit(
    phase41_baseline_probability
)

phase41_baseline_group_loss = {}

for group in range(15):
    group_indices = np.flatnonzero(
        phase20_acquisition_groups
        == group
    )

    phase41_baseline_group_loss[group] = (
        phase41_subset_metrics(
            phase41_baseline_probability,
            group_indices,
        )["log_loss"]
    )


def phase41_global_platt_diagnostic(
    probability,
):
    score = phase41_logit(
        probability
    ).reshape(-1, 1)

    calibrator = LogisticRegression(
        C=1000.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=410702,
    )

    calibrator.fit(
        score,
        PHASE19_LABELS,
    )

    calibrated = (
        calibrator.predict_proba(
            score
        )[:, 1]
    )

    raw_metrics = phase41_metrics(
        probability
    )

    calibrated_metrics = (
        phase41_metrics(
            calibrated
        )
    )

    # A single strictly increasing Platt map must
    # preserve AUROC, except for numerical precision.
    assert abs(
        raw_metrics["auroc"]
        - calibrated_metrics["auroc"]
    ) <= 1e-10

    return {
        "intercept": float(
            calibrator.intercept_[0]
        ),
        "slope": float(
            calibrator.coef_[0, 0]
        ),
        "probability": calibrated,
        "metrics": calibrated_metrics,
        "log_loss_gain": float(
            raw_metrics["log_loss"]
            - calibrated_metrics["log_loss"]
        ),
    }


def phase41_logo_platt(
    probability,
):
    score = phase41_logit(
        probability
    )

    calibrated = np.full(
        EXPECTED_CASES,
        np.nan,
        dtype=np.float64,
    )

    fold_records = []

    for partition in (
        PHASE41_LOGO_PARTITIONS
    ):
        training_indices = (
            partition["train"]
        )
        validation_indices = (
            partition["valid"]
        )

        calibrator = LogisticRegression(
            C=100.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=(
                410800
                + partition[
                    "held_out_group"
                ]
            ),
        )

        calibrator.fit(
            score[
                training_indices
            ].reshape(-1, 1),
            PHASE19_LABELS[
                training_indices
            ],
        )

        calibrated[
            validation_indices
        ] = calibrator.predict_proba(
            score[
                validation_indices
            ].reshape(-1, 1)
        )[:, 1]

        fold_records.append({
            "held_out_group": int(
                partition[
                    "held_out_group"
                ]
            ),
            "intercept": float(
                calibrator.intercept_[0]
            ),
            "slope": float(
                calibrator.coef_[0, 0]
            ),
        })

    assert np.isfinite(calibrated).all()

    return {
        "probability": calibrated,
        "metrics": phase41_metrics(
            calibrated
        ),
        "folds": fold_records,
    }


phase41_component_records = {}
phase41_component_updates = {}

for component_name, probability in (
    phase41_components.items()
):
    raw_metrics = phase41_metrics(
        probability
    )

    global_platt = (
        phase41_global_platt_diagnostic(
            probability
        )
    )

    logo_platt = phase41_logo_platt(
        probability
    )

    group_records = []

    for group in range(15):
        group_indices = np.flatnonzero(
            phase20_acquisition_groups
            == group
        )

        metrics = phase41_subset_metrics(
            probability,
            group_indices,
        )

        metrics["group"] = int(group)
        metrics["log_loss_change_vs_phase12c"] = (
            float(
                metrics["log_loss"]
                - phase41_baseline_group_loss[
                    group
                ]
            )
        )

        group_records.append(metrics)

    group_log_losses = np.asarray([
        record["log_loss"]
        for record in group_records
    ])

    major_group_records = [
        record
        for record in group_records
        if record["n"] >= 30
    ]

    major_group_harms = np.asarray([
        record[
            "log_loss_change_vs_phase12c"
        ]
        for record in major_group_records
    ])

    fold_records = []

    for partition in PHASE19_PARTITIONS:
        fold = int(partition["fold"])
        outer_valid = (
            partition["outer_valid"]
        )

        component_fold_metrics = (
            phase41_subset_metrics(
                probability,
                outer_valid,
            )
        )

        baseline_fold_metrics = (
            phase41_subset_metrics(
                phase41_baseline_probability,
                outer_valid,
            )
        )

        fold_records.append({
            "fold": fold,
            "n": int(
                len(outer_valid)
            ),
            "log_loss": float(
                component_fold_metrics[
                    "log_loss"
                ]
            ),
            "auroc": float(
                component_fold_metrics[
                    "auroc"
                ]
            ),
            "log_loss_gain_vs_phase12c": (
                float(
                    baseline_fold_metrics[
                        "log_loss"
                    ]
                    - component_fold_metrics[
                        "log_loss"
                    ]
                )
            ),
            "auroc_gain_vs_phase12c": (
                float(
                    component_fold_metrics[
                        "auroc"
                    ]
                    - baseline_fold_metrics[
                        "auroc"
                    ]
                )
            ),
        })

    ranking_correlation = float(
        spearmanr(
            phase41_baseline_probability,
            probability,
        ).statistic
    )

    if component_name == "phase12c":
        phase41_component_updates[
            component_name
        ] = np.zeros(
            EXPECTED_CASES,
            dtype=np.float64,
        )
    else:
        phase41_component_updates[
            component_name
        ] = (
            phase41_logit(probability)
            - phase41_baseline_logit
        )

    harmful_groups = sorted(
        group_records,
        key=lambda record: (
            record[
                "log_loss_change_vs_phase12c"
            ]
        ),
        reverse=True,
    )[:5]

    phase41_component_records[
        component_name
    ] = {
        "raw": {
            key: round(value, 6)
            for key, value
            in raw_metrics.items()
        },
        "global_platt_diagnostic_only": {
            "log_loss": round(
                global_platt[
                    "metrics"
                ]["log_loss"],
                6,
            ),
            "auroc": round(
                global_platt[
                    "metrics"
                ]["auroc"],
                6,
            ),
            "log_loss_gain": round(
                global_platt[
                    "log_loss_gain"
                ],
                6,
            ),
            "intercept": round(
                global_platt[
                    "intercept"
                ],
                6,
            ),
            "slope": round(
                global_platt[
                    "slope"
                ],
                6,
            ),
        },
        "leave_one_group_out_platt": {
            "log_loss": round(
                logo_platt[
                    "metrics"
                ]["log_loss"],
                6,
            ),
            "auroc": round(
                logo_platt[
                    "metrics"
                ]["auroc"],
                6,
            ),
        },
        "group_robustness": {
            "equal_group_mean_log_loss": round(
                float(
                    group_log_losses.mean()
                ),
                6,
            ),
            "worst_group_log_loss": round(
                float(
                    group_log_losses.max()
                ),
                6,
            ),
            "maximum_major_group_harm": round(
                float(
                    major_group_harms.max()
                ),
                6,
            ),
            "major_group_win_count": int(
                np.sum(
                    major_group_harms < 0
                )
            ),
            "major_group_count": int(
                len(
                    major_group_records
                )
            ),
        },
        "ranking_correlation_to_phase12c": round(
            ranking_correlation,
            6,
        ),
        "folds": fold_records,
        "five_most_harmed_groups": [
            {
                "group": int(
                    record["group"]
                ),
                "n": int(
                    record["n"]
                ),
                "log_loss_change": round(
                    record[
                        "log_loss_change_vs_phase12c"
                    ],
                    6,
                ),
                "auroc": (
                    None
                    if record["auroc"] is None
                    else round(
                        record["auroc"],
                        6,
                    )
                ),
            }
            for record in harmful_groups
        ],
    }


# Residual complementarity.
phase41_update_names = [
    name
    for name in phase41_components
    if name != "phase12c"
]

phase41_update_correlation = {}

for first_name in phase41_update_names:
    phase41_update_correlation[
        first_name
    ] = {}

    for second_name in (
        phase41_update_names
    ):
        correlation = np.corrcoef(
            phase41_component_updates[
                first_name
            ],
            phase41_component_updates[
                second_name
            ],
        )[0, 1]

        phase41_update_correlation[
            first_name
        ][second_name] = round(
            float(correlation),
            6,
        )


# Correlation between each update and the Phase12c
# signed probability residual. This is descriptive only.
phase41_anchor_probability_residual = (
    PHASE19_LABELS.astype(np.float64)
    - phase41_baseline_probability
)

phase41_residual_signal = {}

for component_name in (
    phase41_update_names
):
    correlation = np.corrcoef(
        phase41_component_updates[
            component_name
        ],
        phase41_anchor_probability_residual,
    )[0, 1]

    phase41_residual_signal[
        component_name
    ] = round(
        float(correlation),
        6,
    )


# Public score is an aggregate supplied by the user.
# It is not used for any fitting or parameter selection.
phase41_public_phase40 = {
    "log_loss": 0.3259,
    "auroc": 0.9243,
}

phase41_phase40_internal = (
    phase41_metrics(
        PHASE40_OOF_PRIVATE
    )
)

phase41_public_gap = {
    "log_loss_public_minus_oof": round(
        phase41_public_phase40[
            "log_loss"
        ]
        - phase41_phase40_internal[
            "log_loss"
        ],
        6,
    ),
    "auroc_oof_minus_public": round(
        phase41_phase40_internal[
            "auroc"
        ]
        - phase41_public_phase40[
            "auroc"
        ],
        6,
    ),
}

PHASE41_DIAGNOSTIC_PRIVATE = {
    "components": (
        phase41_component_records
    ),
    "update_correlation": (
        phase41_update_correlation
    ),
    "residual_signal": (
        phase41_residual_signal
    ),
    "public_gap": phase41_public_gap,
}

phase41_diagnostic_report = {
    "phase": (
        "phase41_bias_variance_"
        "acquisition_shift_diagnostic"
    ),
    "status": "complete",
    "component_count": int(
        len(phase41_components)
    ),
    "components": (
        phase41_component_records
    ),
    "component_update_correlation": (
        phase41_update_correlation
    ),
    "anchor_residual_update_correlation": (
        phase41_residual_signal
    ),
    "phase40_public_transport_gap": (
        phase41_public_gap
    ),
    "interpretation_contract": {
        "global_platt": (
            "same-data diagnostic only; "
            "not deployable"
        ),
        "logo_platt": (
            "each acquisition group's "
            "calibrator excludes that group"
        ),
        "auroc_requires_ranking_change": True,
        "monotonic_global_calibration_preserves_auroc": True,
        "iid_split_not_evaluated_now": (
            "requires newly fitted models "
            "to estimate true same-domain error"
        ),
        "prospective_shadow_evaluated": False,
    },
    "outer_validation_labels_used_for_diagnostic": True,
    "model_parameters_selected": False,
    "prospective_shadow_labels_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE41_DIAGNOSTIC_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE41_DIAGNOSTIC"
)
print(
    json.dumps(
        phase41_diagnostic_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE41_DIAGNOSTIC"
)

BEGIN SANITIZED_PHASE41_DIAGNOSTIC
{
  "phase": "phase41_bias_variance_acquisition_shift_diagnostic",
  "status": "complete",
  "component_count": 6,
  "components": {
    "phase12c": {
      "raw": {
        "log_loss": 0.307616,
        "auroc": 0.938914,
        "mean_probability": 0.530698
      },
      "global_platt_diagnostic_only": {
        "log_loss": 0.305181,
        "auroc": 0.938914,
        "log_loss_gain": 0.002435,
        "intercept": 0.167523,
        "slope": 0.93554
      },
      "leave_one_group_out_platt": {
        "log_loss": 0.309648,
        "auroc": 0.936742
      },
      "group_robustness": {
        "equal_group_mean_log_loss": 0.261241,
        "worst_group_log_loss": 0.361193,
        "maximum_major_group_harm": 0.0,
        "major_group_win_count": 0,
        "major_group_count": 10
      },
      "ranking_correlation_to_phase12c": 1.0,
      "folds": [
        {
          "fold": 0,
          "n": 467,
          "log_loss": 0.3584956553253375,
      

In [5]:
# Phase41 Cell 136A
# Construct compact meta-features for an anchored residual ranker.
# No model fitting and no prospective-shadow label access.

PHASE41_RANK_FEATURE_STARTED = (
    time.perf_counter()
)


def phase41_component_residual(
    probability,
):
    return (
        phase41_logit(probability)
        - phase41_baseline_logit
    )


phase41_r31 = phase41_component_residual(
    PHASE41_PHASE31_OOF_PRIVATE
)

phase41_r33 = (
    PHASE41_PHASE33_RESIDUAL_PRIVATE
)

phase41_r36 = (
    PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE
)

phase41_r38 = phase41_component_residual(
    PHASE41_PHASE38_OOF_PRIVATE
)

phase41_residual_stack = np.column_stack([
    phase41_r31,
    phase41_r33,
    phase41_r36,
    phase41_r38,
]).astype(np.float64)

phase41_component_probability_stack = (
    np.column_stack([
        PHASE41_PHASE31_OOF_PRIVATE,
        PHASE41_PHASE33_COMPONENT_OOF_PRIVATE,
        PHASE41_PHASE36_OOF_PRIVATE,
        PHASE41_PHASE38_OOF_PRIVATE,
    ]).astype(np.float64)
)

phase41_anchor_uncertainty = (
    4.0
    * PHASE41_BASELINE_OOF_PRIVATE
    * (
        1.0
        - PHASE41_BASELINE_OOF_PRIVATE
    )
)

phase41_meta_feature_blocks = [
    phase41_residual_stack,
    np.abs(phase41_residual_stack),
    np.column_stack([
        phase41_r36 * phase41_r33,
        phase41_r36 * phase41_r31,
        phase41_r36 * phase41_r38,
    ]),
    np.column_stack([
        phase41_residual_stack.mean(axis=1),
        phase41_residual_stack.std(axis=1),
        phase41_residual_stack.min(axis=1),
        phase41_residual_stack.max(axis=1),
    ]),
    np.column_stack([
        np.clip(
            phase41_baseline_logit,
            -7.0,
            7.0,
        ),
        phase41_anchor_uncertainty,
        phase41_component_probability_stack.std(
            axis=1
        ),
    ]),
    np.log1p(
        np.asarray(
            phase41_header_matrix,
            dtype=np.float64,
        )
    ),
]

PHASE41_META_RAW_MATRIX_PRIVATE = (
    np.column_stack(
        phase41_meta_feature_blocks
    ).astype(np.float64)
)

PHASE41_META_FEATURE_NAMES = [
    "r31",
    "r33",
    "r36_raw",
    "r38",
    "abs_r31",
    "abs_r33",
    "abs_r36_raw",
    "abs_r38",
    "r36_x_r33",
    "r36_x_r31",
    "r36_x_r38",
    "residual_mean",
    "residual_std",
    "residual_min",
    "residual_max",
    "anchor_logit",
    "anchor_uncertainty",
    "component_probability_std",
    *[
        f"header_log1p_{column}"
        for column
        in PHASE41_HEADER_COLUMNS
    ],
]

assert (
    PHASE41_META_RAW_MATRIX_PRIVATE.shape
    == (
        EXPECTED_CASES,
        len(PHASE41_META_FEATURE_NAMES),
    )
)

assert np.isfinite(
    PHASE41_META_RAW_MATRIX_PRIVATE
).all()

assert len(
    set(PHASE41_META_FEATURE_NAMES)
) == len(PHASE41_META_FEATURE_NAMES)


def phase41_fit_rank_preprocessor(
    training_indices,
):
    matrix = np.asarray(
        PHASE41_META_RAW_MATRIX_PRIVATE[
            training_indices
        ],
        dtype=np.float64,
    )

    lower = np.quantile(
        matrix,
        0.005,
        axis=0,
    )

    upper = np.quantile(
        matrix,
        0.995,
        axis=0,
    )

    winsorized = np.clip(
        matrix,
        lower,
        upper,
    )

    median = np.median(
        winsorized,
        axis=0,
    )

    q25 = np.quantile(
        winsorized,
        0.25,
        axis=0,
    )

    q75 = np.quantile(
        winsorized,
        0.75,
        axis=0,
    )

    scale = q75 - q25

    standard_deviation = (
        winsorized.std(axis=0)
    )

    keep = (
        (scale > 1e-7)
        | (standard_deviation > 1e-7)
    )

    fallback_scale = np.maximum(
        standard_deviation,
        1e-6,
    )

    scale = np.where(
        scale > 1e-7,
        scale,
        fallback_scale,
    )

    assert keep.any()

    return {
        "lower": lower,
        "upper": upper,
        "median": median,
        "scale": scale,
        "keep": keep,
    }


def phase41_apply_rank_preprocessor(
    state,
    indices,
):
    matrix = np.asarray(
        PHASE41_META_RAW_MATRIX_PRIVATE[
            indices
        ],
        dtype=np.float64,
    )

    matrix = np.clip(
        matrix,
        state["lower"],
        state["upper"],
    )

    matrix = (
        matrix - state["median"]
    ) / state["scale"]

    matrix = matrix[:, state["keep"]]

    matrix = np.clip(
        matrix,
        -8.0,
        8.0,
    )

    assert np.isfinite(matrix).all()

    return matrix.astype(np.float64)


phase41_development_groups = np.unique(
    phase20_acquisition_groups[
        PHASE41_DEVELOPMENT_INDICES_PRIVATE
    ]
)

phase41_contract_group = int(
    phase41_development_groups[0]
)

phase41_contract_fit = np.flatnonzero(
    np.isin(
        np.arange(EXPECTED_CASES),
        PHASE41_DEVELOPMENT_INDICES_PRIVATE,
    )
    & (
        phase20_acquisition_groups
        != phase41_contract_group
    )
)

phase41_contract_valid = np.flatnonzero(
    np.isin(
        np.arange(EXPECTED_CASES),
        PHASE41_DEVELOPMENT_INDICES_PRIVATE,
    )
    & (
        phase20_acquisition_groups
        == phase41_contract_group
    )
)

phase41_contract_preprocessor = (
    phase41_fit_rank_preprocessor(
        phase41_contract_fit
    )
)

phase41_contract_fit_matrix = (
    phase41_apply_rank_preprocessor(
        phase41_contract_preprocessor,
        phase41_contract_fit,
    )
)

phase41_contract_valid_matrix = (
    phase41_apply_rank_preprocessor(
        phase41_contract_preprocessor,
        phase41_contract_valid,
    )
)

phase41_rank_feature_report = {
    "phase": (
        "phase41_anchored_pairwise_ranker_"
        "feature_contract"
    ),
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "raw_feature_shape": list(
        PHASE41_META_RAW_MATRIX_PRIVATE.shape
    ),
    "raw_feature_count": len(
        PHASE41_META_FEATURE_NAMES
    ),
    "feature_families": [
        "cross_fitted_component_residuals",
        "absolute_residual_magnitudes",
        "component_interactions",
        "residual_distribution_statistics",
        "phase12c_anchor_uncertainty",
        "component_disagreement",
        "header_acquisition_features",
    ],
    "contract_fold": {
        "fit_n": int(
            len(phase41_contract_fit)
        ),
        "valid_n": int(
            len(phase41_contract_valid)
        ),
        "transformed_feature_count": int(
            phase41_contract_fit_matrix.shape[1]
        ),
    },
    "preprocessing": {
        "winsorization_quantiles": [
            0.005,
            0.995,
        ],
        "robust_scaling": (
            "training_partition_median_and_IQR"
        ),
        "collapsed_coordinate_removal": (
            "training_partition_only"
        ),
        "outer_group_features_used_for_fit": False,
    },
    "all_values_finite": True,
    "prospective_shadow_labels_used": False,
    "labels_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE41_RANK_FEATURE_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE41_RANK_FEATURES"
)
print(
    json.dumps(
        phase41_rank_feature_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE41_RANK_FEATURES"
)

BEGIN SANITIZED_PHASE41_RANK_FEATURES
{
  "phase": "phase41_anchored_pairwise_ranker_feature_contract",
  "status": "accepted",
  "case_count": 1362,
  "raw_feature_shape": [
    1362,
    28
  ],
  "raw_feature_count": 28,
  "feature_families": [
    "cross_fitted_component_residuals",
    "absolute_residual_magnitudes",
    "component_interactions",
    "residual_distribution_statistics",
    "phase12c_anchor_uncertainty",
    "component_disagreement",
    "header_acquisition_features"
  ],
  "contract_fold": {
    "fit_n": 1066,
    "valid_n": 39,
    "transformed_feature_count": 28
  },
  "preprocessing": {
    "winsorization_quantiles": [
      0.005,
      0.995
    ],
    "robust_scaling": "training_partition_median_and_IQR",
    "collapsed_coordinate_removal": "training_partition_only",
    "outer_group_features_used_for_fit": false
  },
  "all_values_finite": true,
  "prospective_shadow_labels_used": false,
  "labels_used": false,
  "training_voxel_arrays_read": false,
  "smok

In [6]:
# Phase41 Cell 136B
# Nested leave-one-development-group-out training and exhaustive
# bounded-gate selection.
#
# Prospective shadow groups are completely excluded from fitting,
# hyperparameter selection, metrics, and bootstrap evaluation.

from scipy.optimize import minimize
from scipy.special import expit

PHASE41_RANK_SEARCH_STARTED = (
    time.perf_counter()
)

PHASE41_RANK_CONFIG = {
    "rank_lambdas": [
        0.0,
        0.03,
        0.07,
        0.12,
        0.20,
        0.35,
        0.50,
    ],
    "l2_values": [
        1e-5,
        3e-5,
        1e-4,
        3e-4,
        1e-3,
        3e-3,
    ],
    "alpha_values": [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
        1.00,
    ],
    "uncertainty_exponents": [
        0.0,
        0.25,
        0.50,
        0.75,
        1.0,
    ],
    "residual_caps": [
        0.50,
        0.75,
        1.00,
        1.25,
        1.50,
        2.00,
    ],
    "maximum_optimizer_iterations": 140,
    "global_pair_count": 6000,
    "within_group_pair_count": 500,
    "group_weight_exponent": 0.35,
    "minimum_log_loss_gain": 0.002,
    "minimum_auroc_gain": 0.001,
    "maximum_major_group_harm": 0.015,
    "log_loss_selection_band": 0.001,
    "bootstrap_replicates": 1500,
    "bootstrap_minimum_q05_gain": 0.0,
    "bootstrap_minimum_probability_positive": 0.90,
}

phase41_development_mask = np.zeros(
    EXPECTED_CASES,
    dtype=bool,
)

phase41_development_mask[
    PHASE41_DEVELOPMENT_INDICES_PRIVATE
] = True

phase41_rank_development_groups = (
    np.unique(
        phase20_acquisition_groups[
            PHASE41_DEVELOPMENT_INDICES_PRIVATE
        ]
    )
)

assert len(
    np.intersect1d(
        PHASE41_DEVELOPMENT_INDICES_PRIVATE,
        PHASE41_SHADOW_INDICES_PRIVATE,
    )
) == 0


def phase41_group_balanced_weights(
    indices,
):
    local_groups = (
        phase20_acquisition_groups[
            indices
        ]
    )

    counts = {
        int(group): int(
            np.sum(local_groups == group)
        )
        for group in np.unique(
            local_groups
        )
    }

    weights = np.asarray([
        (
            len(indices)
            / counts[int(group)]
        ) ** PHASE41_RANK_CONFIG[
            "group_weight_exponent"
        ]
        for group in local_groups
    ], dtype=np.float64)

    weights /= weights.mean()

    return weights


def phase41_make_rank_pairs(
    indices,
    seed,
):
    rng = np.random.default_rng(seed)

    local_labels = PHASE19_LABELS[
        indices
    ]

    local_groups = (
        phase20_acquisition_groups[
            indices
        ]
    )

    positive_global = np.flatnonzero(
        local_labels == 1
    )

    negative_global = np.flatnonzero(
        local_labels == 0
    )

    global_pair_count = (
        PHASE41_RANK_CONFIG[
            "global_pair_count"
        ]
    )

    positive_pairs = [
        rng.choice(
            positive_global,
            size=global_pair_count,
            replace=True,
        )
    ]

    negative_pairs = [
        rng.choice(
            negative_global,
            size=global_pair_count,
            replace=True,
        )
    ]

    for group in np.unique(
        local_groups
    ):
        group_positive = np.flatnonzero(
            (local_groups == group)
            & (local_labels == 1)
        )

        group_negative = np.flatnonzero(
            (local_groups == group)
            & (local_labels == 0)
        )

        if (
            len(group_positive) == 0
            or len(group_negative) == 0
        ):
            continue

        pair_count = min(
            PHASE41_RANK_CONFIG[
                "within_group_pair_count"
            ],
            max(
                50,
                len(group_positive)
                * len(group_negative),
            ),
        )

        positive_pairs.append(
            rng.choice(
                group_positive,
                size=pair_count,
                replace=True,
            )
        )

        negative_pairs.append(
            rng.choice(
                group_negative,
                size=pair_count,
                replace=True,
            )
        )

    positive_pairs = np.concatenate(
        positive_pairs
    ).astype(np.int64)

    negative_pairs = np.concatenate(
        negative_pairs
    ).astype(np.int64)

    assert len(positive_pairs) == len(
        negative_pairs
    )

    return positive_pairs, negative_pairs


def phase41_fit_anchored_linear_ranker(
    training_matrix,
    training_indices,
    rank_lambda,
    l2_value,
    seed,
):
    labels_train = (
        PHASE19_LABELS[
            training_indices
        ].astype(np.float64)
    )

    baseline_train = (
        phase41_baseline_logit[
            training_indices
        ]
    )

    sample_weight = (
        phase41_group_balanced_weights(
            training_indices
        )
    )

    pair_positive, pair_negative = (
        phase41_make_rank_pairs(
            training_indices,
            seed,
        )
    )

    feature_count = (
        training_matrix.shape[1]
    )

    def objective(parameters):
        weights = parameters[
            :feature_count
        ]
        intercept = parameters[
            feature_count
        ]

        raw_residual = (
            training_matrix @ weights
            + intercept
        )

        final_logit = (
            baseline_train
            + raw_residual
        )

        probability = expit(
            final_logit
        )

        bce_terms = (
            np.logaddexp(
                0.0,
                final_logit,
            )
            - labels_train
            * final_logit
        )

        bce_loss = float(
            np.sum(
                sample_weight
                * bce_terms
            )
            / np.sum(sample_weight)
        )

        gradient_logit = (
            sample_weight
            * (
                probability
                - labels_train
            )
            / np.sum(sample_weight)
        )

        if rank_lambda > 0:
            pair_difference = (
                final_logit[
                    pair_positive
                ]
                - final_logit[
                    pair_negative
                ]
            )

            rank_loss = float(
                np.mean(
                    np.logaddexp(
                        0.0,
                        -pair_difference,
                    )
                )
            )

            pair_gradient = (
                expit(pair_difference)
                - 1.0
            ) / len(pair_difference)

            np.add.at(
                gradient_logit,
                pair_positive,
                rank_lambda
                * pair_gradient,
            )

            np.add.at(
                gradient_logit,
                pair_negative,
                -rank_lambda
                * pair_gradient,
            )

        else:
            rank_loss = 0.0

        regularization = (
            0.5
            * l2_value
            * float(
                np.dot(
                    weights,
                    weights,
                )
            )
        )

        total_loss = (
            bce_loss
            + rank_lambda * rank_loss
            + regularization
        )

        gradient_weights = (
            training_matrix.T
            @ gradient_logit
            + l2_value * weights
        )

        gradient_intercept = float(
            gradient_logit.sum()
        )

        gradient = np.concatenate([
            gradient_weights,
            np.asarray([
                gradient_intercept
            ]),
        ])

        return total_loss, gradient

    initial = np.zeros(
        feature_count + 1,
        dtype=np.float64,
    )

    result = minimize(
        fun=lambda parameters: objective(
            parameters
        ),
        x0=initial,
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": (
                PHASE41_RANK_CONFIG[
                    "maximum_optimizer_iterations"
                ]
            ),
            "ftol": 1e-10,
            "gtol": 1e-7,
            "maxls": 30,
        },
    )

    parameters = np.asarray(
        result.x,
        dtype=np.float64,
    )

    assert np.isfinite(
        parameters
    ).all()

    return {
        "weights": parameters[
            :feature_count
        ],
        "intercept": float(
            parameters[feature_count]
        ),
        "optimizer_success": bool(
            result.success
        ),
        "optimizer_iterations": int(
            result.nit
        ),
        "objective": float(
            result.fun
        ),
    }


# Precompute each development LOGO context once.
phase41_rank_contexts = []

for held_out_position, held_out_group in enumerate(
    phase41_rank_development_groups
):
    validation_indices = np.flatnonzero(
        phase41_development_mask
        & (
            phase20_acquisition_groups
            == held_out_group
        )
    ).astype(np.int64)

    training_indices = np.flatnonzero(
        phase41_development_mask
        & (
            phase20_acquisition_groups
            != held_out_group
        )
    ).astype(np.int64)

    preprocessor = (
        phase41_fit_rank_preprocessor(
            training_indices
        )
    )

    training_matrix = (
        phase41_apply_rank_preprocessor(
            preprocessor,
            training_indices,
        )
    )

    validation_matrix = (
        phase41_apply_rank_preprocessor(
            preprocessor,
            validation_indices,
        )
    )

    phase41_rank_contexts.append({
        "held_out_position": int(
            held_out_position
        ),
        "train_indices": training_indices,
        "valid_indices": validation_indices,
        "training_matrix": training_matrix,
        "validation_matrix": validation_matrix,
    })


phase41_model_specifications = [
    {
        "rank_lambda": float(
            rank_lambda
        ),
        "l2_value": float(l2_value),
    }
    for rank_lambda in (
        PHASE41_RANK_CONFIG[
            "rank_lambdas"
        ]
    )
    for l2_value in (
        PHASE41_RANK_CONFIG[
            "l2_values"
        ]
    )
]

phase41_raw_oof_by_model = []
phase41_model_fit_records = []

for specification_index, specification in enumerate(
    phase41_model_specifications
):
    raw_oof = np.full(
        EXPECTED_CASES,
        np.nan,
        dtype=np.float64,
    )

    fit_records = []

    for context in phase41_rank_contexts:
        model = (
            phase41_fit_anchored_linear_ranker(
                training_matrix=(
                    context[
                        "training_matrix"
                    ]
                ),
                training_indices=(
                    context[
                        "train_indices"
                    ]
                ),
                rank_lambda=(
                    specification[
                        "rank_lambda"
                    ]
                ),
                l2_value=(
                    specification[
                        "l2_value"
                    ]
                ),
                seed=(
                    411000
                    + 100
                    * specification_index
                    + context[
                        "held_out_position"
                    ]
                ),
            )
        )

        validation_raw = (
            context[
                "validation_matrix"
            ]
            @ model["weights"]
            + model["intercept"]
        )

        raw_oof[
            context["valid_indices"]
        ] = validation_raw

        fit_records.append({
            "optimizer_success": (
                model[
                    "optimizer_success"
                ]
            ),
            "optimizer_iterations": (
                model[
                    "optimizer_iterations"
                ]
            ),
        })

    assert np.isfinite(
        raw_oof[
            PHASE41_DEVELOPMENT_INDICES_PRIVATE
        ]
    ).all()

    assert np.isnan(
        raw_oof[
            PHASE41_SHADOW_INDICES_PRIVATE
        ]
    ).all()

    phase41_raw_oof_by_model.append(
        raw_oof
    )

    phase41_model_fit_records.append(
        fit_records
    )

    print(
        "Phase41 rank model "
        f"{specification_index + 1}/"
        f"{len(phase41_model_specifications)} "
        "complete"
    )


phase41_development_indices = (
    PHASE41_DEVELOPMENT_INDICES_PRIVATE
)

phase41_development_labels = (
    PHASE19_LABELS[
        phase41_development_indices
    ]
)

phase41_development_anchor = (
    PHASE41_BASELINE_OOF_PRIVATE[
        phase41_development_indices
    ]
)

phase41_development_anchor_logit = (
    phase41_baseline_logit[
        phase41_development_indices
    ]
)

phase41_development_uncertainty = (
    phase41_anchor_uncertainty[
        phase41_development_indices
    ]
)

phase41_development_group_values = (
    phase20_acquisition_groups[
        phase41_development_indices
    ]
)

phase41_baseline_development_metrics = {
    "log_loss": float(
        log_loss(
            phase41_development_labels,
            phase41_development_anchor,
            labels=[0, 1],
        )
    ),
    "auroc": float(
        roc_auc_score(
            phase41_development_labels,
            phase41_development_anchor,
        )
    ),
}

phase41_development_group_indices = {
    int(group): np.flatnonzero(
        phase41_development_group_values
        == group
    )
    for group in np.unique(
        phase41_development_group_values
    )
}

phase41_development_group_baseline_loss = {
    group: float(
        log_loss(
            phase41_development_labels[
                local_indices
            ],
            phase41_development_anchor[
                local_indices
            ],
            labels=[0, 1],
        )
    )
    for group, local_indices
    in phase41_development_group_indices.items()
}

phase41_major_development_groups = [
    group
    for group, local_indices
    in phase41_development_group_indices.items()
    if len(local_indices) >= 30
]


def phase41_evaluate_development_candidate(
    probability,
):
    probability = phase41_safe_probability(
        probability
    )

    overall_log_loss = float(
        log_loss(
            phase41_development_labels,
            probability,
            labels=[0, 1],
        )
    )

    overall_auroc = float(
        roc_auc_score(
            phase41_development_labels,
            probability,
        )
    )

    group_changes = {}
    group_wins = 0

    for group, local_indices in (
        phase41_development_group_indices.items()
    ):
        candidate_loss = float(
            log_loss(
                phase41_development_labels[
                    local_indices
                ],
                probability[
                    local_indices
                ],
                labels=[0, 1],
            )
        )

        change = (
            candidate_loss
            - phase41_development_group_baseline_loss[
                group
            ]
        )

        group_changes[group] = float(
            change
        )

        if change < 0:
            group_wins += 1

    major_harm = max(
        group_changes[group]
        for group
        in phase41_major_development_groups
    )

    major_wins = sum(
        group_changes[group] < 0
        for group
        in phase41_major_development_groups
    )

    return {
        "log_loss": overall_log_loss,
        "auroc": overall_auroc,
        "log_loss_gain": (
            phase41_baseline_development_metrics[
                "log_loss"
            ]
            - overall_log_loss
        ),
        "auroc_gain": (
            overall_auroc
            - phase41_baseline_development_metrics[
                "auroc"
            ]
        ),
        "maximum_major_group_harm": float(
            major_harm
        ),
        "group_wins": int(
            group_wins
        ),
        "major_group_wins": int(
            major_wins
        ),
        "group_changes": group_changes,
    }


phase41_gate_records = []

for model_index, raw_oof in enumerate(
    phase41_raw_oof_by_model
):
    raw_development = raw_oof[
        phase41_development_indices
    ]

    for alpha in (
        PHASE41_RANK_CONFIG[
            "alpha_values"
        ]
    ):
        for uncertainty_exponent in (
            PHASE41_RANK_CONFIG[
                "uncertainty_exponents"
            ]
        ):
            effective_alpha = (
                alpha
                * np.power(
                    phase41_development_uncertainty,
                    uncertainty_exponent,
                )
            )

            for residual_cap in (
                PHASE41_RANK_CONFIG[
                    "residual_caps"
                ]
            ):
                bounded_residual = np.clip(
                    raw_development,
                    -residual_cap,
                    residual_cap,
                )

                probability = expit(
                    phase41_development_anchor_logit
                    + effective_alpha
                    * bounded_residual
                )

                metrics = (
                    phase41_evaluate_development_candidate(
                        probability
                    )
                )

                eligible = (
                    metrics["log_loss_gain"]
                    >= PHASE41_RANK_CONFIG[
                        "minimum_log_loss_gain"
                    ]
                    and metrics["auroc_gain"]
                    >= PHASE41_RANK_CONFIG[
                        "minimum_auroc_gain"
                    ]
                    and metrics[
                        "maximum_major_group_harm"
                    ]
                    <= PHASE41_RANK_CONFIG[
                        "maximum_major_group_harm"
                    ]
                    and metrics[
                        "major_group_wins"
                    ]
                    >= int(
                        np.ceil(
                            len(
                                phase41_major_development_groups
                            )
                            / 2
                        )
                    )
                )

                phase41_gate_records.append({
                    "model_index": int(
                        model_index
                    ),
                    "rank_lambda": (
                        phase41_model_specifications[
                            model_index
                        ]["rank_lambda"]
                    ),
                    "l2_value": (
                        phase41_model_specifications[
                            model_index
                        ]["l2_value"]
                    ),
                    "alpha": float(alpha),
                    "uncertainty_exponent": float(
                        uncertainty_exponent
                    ),
                    "residual_cap": float(
                        residual_cap
                    ),
                    **metrics,
                    "eligible": bool(
                        eligible
                    ),
                })


phase41_eligible_records = [
    record
    for record in phase41_gate_records
    if record["eligible"]
]

phase41_gate_advanced = bool(
    phase41_eligible_records
)

if phase41_gate_advanced:
    phase41_raw_best = min(
        phase41_eligible_records,
        key=lambda record: (
            record["log_loss"],
            -record["auroc"],
        ),
    )

    phase41_bootstrap_pool = [
        record
        for record in phase41_eligible_records
        if record["log_loss"]
        <= (
            phase41_raw_best["log_loss"]
            + 0.002
        )
    ]

    phase41_bootstrap_pool = sorted(
        phase41_bootstrap_pool,
        key=lambda record: (
            record["log_loss"],
            -record["auroc"],
        ),
    )[:24]

    bootstrap_rng = np.random.default_rng(
        411901
    )

    phase41_bootstrap_indices = []

    for _ in range(
        PHASE41_RANK_CONFIG[
            "bootstrap_replicates"
        ]
    ):
        sampled_parts = []

        for local_indices in (
            phase41_development_group_indices.values()
        ):
            sampled_parts.append(
                bootstrap_rng.choice(
                    local_indices,
                    size=len(local_indices),
                    replace=True,
                )
            )

        phase41_bootstrap_indices.append(
            np.concatenate(sampled_parts)
        )

    phase41_bootstrap_indices = (
        np.stack(
            phase41_bootstrap_indices,
            axis=0,
        )
    )

    baseline_case_loss = (
        -phase41_development_labels
        * np.log(
            phase41_development_anchor
        )
        - (
            1
            - phase41_development_labels
        )
        * np.log(
            1.0
            - phase41_development_anchor
        )
    )

    for record in (
        phase41_bootstrap_pool
    ):
        raw_development = (
            phase41_raw_oof_by_model[
                record["model_index"]
            ][phase41_development_indices]
        )

        effective_alpha = (
            record["alpha"]
            * np.power(
                phase41_development_uncertainty,
                record[
                    "uncertainty_exponent"
                ],
            )
        )

        probability = expit(
            phase41_development_anchor_logit
            + effective_alpha
            * np.clip(
                raw_development,
                -record["residual_cap"],
                record["residual_cap"],
            )
        )

        candidate_case_loss = (
            -phase41_development_labels
            * np.log(
                phase41_safe_probability(
                    probability
                )
            )
            - (
                1
                - phase41_development_labels
            )
            * np.log(
                1.0
                - phase41_safe_probability(
                    probability
                )
            )
        )

        case_gain = (
            baseline_case_loss
            - candidate_case_loss
        )

        bootstrap_gain = (
            case_gain[
                phase41_bootstrap_indices
            ].mean(axis=1)
        )

        record["bootstrap_mean_gain"] = (
            float(
                bootstrap_gain.mean()
            )
        )

        record["bootstrap_q05_gain"] = (
            float(
                np.quantile(
                    bootstrap_gain,
                    0.05,
                )
            )
        )

        record[
            "bootstrap_probability_positive"
        ] = float(
            np.mean(
                bootstrap_gain > 0
            )
        )

        record["bootstrap_eligible"] = (
            record[
                "bootstrap_q05_gain"
            ]
            >= PHASE41_RANK_CONFIG[
                "bootstrap_minimum_q05_gain"
            ]
            and record[
                "bootstrap_probability_positive"
            ]
            >= PHASE41_RANK_CONFIG[
                "bootstrap_minimum_probability_positive"
            ]
        )

    phase41_robust_records = [
        record
        for record
        in phase41_bootstrap_pool
        if record.get(
            "bootstrap_eligible",
            False,
        )
    ]

    if not phase41_robust_records:
        phase41_robust_records = (
            phase41_bootstrap_pool
        )

    phase41_best_robust_loss = min(
        record["log_loss"]
        for record
        in phase41_robust_records
    )

    phase41_final_band = [
        record
        for record
        in phase41_robust_records
        if record["log_loss"]
        <= (
            phase41_best_robust_loss
            + PHASE41_RANK_CONFIG[
                "log_loss_selection_band"
            ]
        )
    ]

    PHASE41_SELECTED_RANK_SPEC_PRIVATE = max(
        phase41_final_band,
        key=lambda record: (
            record["auroc"],
            record.get(
                "bootstrap_q05_gain",
                -np.inf,
            ),
            -record["log_loss"],
        ),
    )

else:
    phase41_raw_best = None

    PHASE41_SELECTED_RANK_SPEC_PRIVATE = {
        "model_index": None,
        "rank_lambda": 0.0,
        "l2_value": 0.0,
        "alpha": 0.0,
        "uncertainty_exponent": 0.0,
        "residual_cap": 0.0,
        "log_loss": (
            phase41_baseline_development_metrics[
                "log_loss"
            ]
        ),
        "auroc": (
            phase41_baseline_development_metrics[
                "auroc"
            ]
        ),
        "log_loss_gain": 0.0,
        "auroc_gain": 0.0,
        "maximum_major_group_harm": 0.0,
        "group_wins": 0,
        "major_group_wins": 0,
        "bootstrap_q05_gain": 0.0,
        "bootstrap_probability_positive": 0.0,
        "eligible": False,
    }


if phase41_gate_advanced:
    selected_model_index = (
        PHASE41_SELECTED_RANK_SPEC_PRIVATE[
            "model_index"
        ]
    )

    PHASE41_SELECTED_DEVELOPMENT_RAW_OOF_PRIVATE = (
        phase41_raw_oof_by_model[
            selected_model_index
        ].copy()
    )

    np.save(
        "/kaggle/working/"
        "phase41_selected_development_"
        "raw_oof_float32.npy",
        PHASE41_SELECTED_DEVELOPMENT_RAW_OOF_PRIVATE.astype(
            np.float32
        ),
        allow_pickle=False,
    )

phase41_selected_sanitized = {
    key: value
    for key, value
    in PHASE41_SELECTED_RANK_SPEC_PRIVATE.items()
    if key not in {
        "group_changes",
    }
}

phase41_selection_metadata = {
    "schema_version": 1,
    "phase": (
        "phase41_anchored_pairwise_ranker"
    ),
    "selected": (
        phase41_selected_sanitized
    ),
    "prospective_shadow_contract_sha256": (
        PHASE41_SHADOW_CONTRACT_SHA256
    ),
    "shadow_labels_used": False,
}

Path(
    "/kaggle/working/"
    "phase41_ranker_selection_private.json"
).write_text(
    json.dumps(
        phase41_selection_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

phase41_rank_search_report = {
    "phase": (
        "phase41_nested_logo_anchored_"
        "pairwise_ranker_selection"
    ),
    "status": (
        "ranker_frozen"
        if phase41_gate_advanced
        else "anchor_retained"
    ),
    "development": {
        "case_count": int(
            len(
                PHASE41_DEVELOPMENT_INDICES_PRIVATE
            )
        ),
        "acquisition_group_count": int(
            len(
                phase41_rank_development_groups
            )
        ),
        "baseline_log_loss": round(
            phase41_baseline_development_metrics[
                "log_loss"
            ],
            6,
        ),
        "baseline_auroc": round(
            phase41_baseline_development_metrics[
                "auroc"
            ],
            6,
        ),
    },
    "search": {
        "model_specification_count": int(
            len(
                phase41_model_specifications
            )
        ),
        "gate_candidate_count": int(
            len(phase41_gate_records)
        ),
        "eligible_candidate_count": int(
            len(
                phase41_eligible_records
            )
        ),
        "development_logo_fold_count": int(
            len(phase41_rank_contexts)
        ),
        "bootstrap_replicates": (
            PHASE41_RANK_CONFIG[
                "bootstrap_replicates"
            ]
        ),
    },
    "raw_best": (
        None
        if phase41_raw_best is None
        else {
            key: value
            for key, value
            in phase41_raw_best.items()
            if key not in {
                "group_changes",
            }
        }
    ),
    "selected": (
        phase41_selected_sanitized
    ),
    "selection_rule": (
        "minimum robust development LOGO "
        "log loss, then maximum AUROC within "
        "the 0.001 log-loss band"
    ),
    "prospective_shadow": {
        "case_count": int(
            len(
                PHASE41_SHADOW_INDICES_PRIVATE
            )
        ),
        "evaluated": False,
        "labels_used": False,
        "contract_sha256": (
            PHASE41_SHADOW_CONTRACT_SHA256
        ),
    },
    "shared_hyperparameters_across_groups": True,
    "explicit_group_identity_feature_used": False,
    "header_features_used": True,
    "pairwise_ranking_loss_used": True,
    "outer_validation_labels_used": (
        "development_groups_only"
    ),
    "prospective_shadow_labels_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE41_RANK_SEARCH_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE41_RANK_SELECTION"
)
print(
    json.dumps(
        phase41_rank_search_report,
        indent=2,
        default=lambda value: (
            value.item()
            if isinstance(
                value,
                np.generic,
            )
            else value
        ),
    )
)
print(
    "END SANITIZED_PHASE41_RANK_SELECTION"
)

Phase41 rank model 1/42 complete
Phase41 rank model 2/42 complete
Phase41 rank model 3/42 complete
Phase41 rank model 4/42 complete
Phase41 rank model 5/42 complete
Phase41 rank model 6/42 complete
Phase41 rank model 7/42 complete
Phase41 rank model 8/42 complete
Phase41 rank model 9/42 complete
Phase41 rank model 10/42 complete
Phase41 rank model 11/42 complete
Phase41 rank model 12/42 complete
Phase41 rank model 13/42 complete
Phase41 rank model 14/42 complete
Phase41 rank model 15/42 complete
Phase41 rank model 16/42 complete
Phase41 rank model 17/42 complete
Phase41 rank model 18/42 complete
Phase41 rank model 19/42 complete
Phase41 rank model 20/42 complete
Phase41 rank model 21/42 complete
Phase41 rank model 22/42 complete
Phase41 rank model 23/42 complete
Phase41 rank model 24/42 complete
Phase41 rank model 25/42 complete
Phase41 rank model 26/42 complete
Phase41 rank model 27/42 complete
Phase41 rank model 28/42 complete
Phase41 rank model 29/42 complete
Phase41 rank model 30/4

In [7]:
# Phase41 Cell 136C
# One-time evaluation of the already frozen Phase41 specification
# on the prospectively locked shadow acquisition groups.
#
# No hyperparameters are selected or modified in this cell.

from sklearn.linear_model import LogisticRegression

PHASE41_SHADOW_EVALUATION_STARTED = (
    time.perf_counter()
)

assert (
    PHASE41_SELECTED_RANK_SPEC_PRIVATE[
        "model_index"
    ]
    is not None
), "The Phase41 ranker did not advance."

phase41_current_shadow_hash = (
    hashlib.sha256(
        PHASE41_SHADOW_INDICES_PRIVATE
        .astype("<i8")
        .tobytes()
    ).hexdigest()
)

assert (
    phase41_current_shadow_hash
    == PHASE41_SHADOW_CONTRACT_SHA256
), {
    "message": (
        "Prospective shadow partition changed."
    ),
    "expected": (
        PHASE41_SHADOW_CONTRACT_SHA256
    ),
    "observed": (
        phase41_current_shadow_hash
    ),
}

phase41_frozen_specification = dict(
    PHASE41_SELECTED_RANK_SPEC_PRIVATE
)

phase41_shadow_fit_indices = (
    PHASE41_DEVELOPMENT_INDICES_PRIVATE
)

phase41_shadow_valid_indices = (
    PHASE41_SHADOW_INDICES_PRIVATE
)

assert len(
    np.intersect1d(
        phase41_shadow_fit_indices,
        phase41_shadow_valid_indices,
    )
) == 0

# ---------------------------------------------------------
# Fit the frozen model once on all development groups.
# ---------------------------------------------------------

PHASE41_SHADOW_PREPROCESSOR_PRIVATE = (
    phase41_fit_rank_preprocessor(
        phase41_shadow_fit_indices
    )
)

phase41_shadow_fit_matrix = (
    phase41_apply_rank_preprocessor(
        PHASE41_SHADOW_PREPROCESSOR_PRIVATE,
        phase41_shadow_fit_indices,
    )
)

phase41_shadow_valid_matrix = (
    phase41_apply_rank_preprocessor(
        PHASE41_SHADOW_PREPROCESSOR_PRIVATE,
        phase41_shadow_valid_indices,
    )
)

PHASE41_SHADOW_MODEL_PRIVATE = (
    phase41_fit_anchored_linear_ranker(
        training_matrix=(
            phase41_shadow_fit_matrix
        ),
        training_indices=(
            phase41_shadow_fit_indices
        ),
        rank_lambda=(
            phase41_frozen_specification[
                "rank_lambda"
            ]
        ),
        l2_value=(
            phase41_frozen_specification[
                "l2_value"
            ]
        ),
        seed=412001,
    )
)

PHASE41_SHADOW_RAW_RESIDUAL_PRIVATE = (
    phase41_shadow_valid_matrix
    @ PHASE41_SHADOW_MODEL_PRIVATE[
        "weights"
    ]
    + PHASE41_SHADOW_MODEL_PRIVATE[
        "intercept"
    ]
)

phase41_shadow_anchor_probability = (
    PHASE41_BASELINE_OOF_PRIVATE[
        phase41_shadow_valid_indices
    ]
)

phase41_shadow_anchor_logit = (
    phase41_baseline_logit[
        phase41_shadow_valid_indices
    ]
)

phase41_shadow_uncertainty = (
    phase41_anchor_uncertainty[
        phase41_shadow_valid_indices
    ]
)

phase41_shadow_effective_alpha = (
    phase41_frozen_specification[
        "alpha"
    ]
    * np.power(
        phase41_shadow_uncertainty,
        phase41_frozen_specification[
            "uncertainty_exponent"
        ],
    )
)

phase41_shadow_bounded_residual = (
    np.clip(
        PHASE41_SHADOW_RAW_RESIDUAL_PRIVATE,
        -phase41_frozen_specification[
            "residual_cap"
        ],
        phase41_frozen_specification[
            "residual_cap"
        ],
    )
)

PHASE41_SHADOW_PROBABILITY_PRIVATE = (
    expit(
        phase41_shadow_anchor_logit
        + phase41_shadow_effective_alpha
        * phase41_shadow_bounded_residual
    )
)

assert np.isfinite(
    PHASE41_SHADOW_PROBABILITY_PRIVATE
).all()

assert np.all(
    PHASE41_SHADOW_PROBABILITY_PRIVATE
    > 0
)

assert np.all(
    PHASE41_SHADOW_PROBABILITY_PRIVATE
    < 1
)


# ---------------------------------------------------------
# The shadow labels are accessed for the first and only
# Phase41 evaluation here.
# ---------------------------------------------------------

phase41_shadow_labels = PHASE19_LABELS[
    phase41_shadow_valid_indices
]

phase41_shadow_groups = (
    phase20_acquisition_groups[
        phase41_shadow_valid_indices
    ]
)


def phase41_local_metrics(
    labels_local,
    probability_local,
):
    labels_local = np.asarray(
        labels_local,
        dtype=np.int64,
    )

    probability_local = (
        phase41_safe_probability(
            probability_local
        )
    )

    record = {
        "n": int(len(labels_local)),
        "log_loss": float(
            log_loss(
                labels_local,
                probability_local,
                labels=[0, 1],
            )
        ),
        "mean_probability": float(
            probability_local.mean()
        ),
        "prevalence": float(
            labels_local.mean()
        ),
    }

    if np.unique(labels_local).size == 2:
        record["auroc"] = float(
            roc_auc_score(
                labels_local,
                probability_local,
            )
        )
    else:
        record["auroc"] = None

    return record


phase41_shadow_component_probabilities = {
    "phase12c": (
        PHASE41_BASELINE_OOF_PRIVATE[
            phase41_shadow_valid_indices
        ]
    ),
    "phase36": (
        PHASE41_PHASE36_OOF_PRIVATE[
            phase41_shadow_valid_indices
        ]
    ),
    "phase39": (
        PHASE41_PHASE39_OOF_PRIVATE[
            phase41_shadow_valid_indices
        ]
    ),
    "phase41": (
        PHASE41_SHADOW_PROBABILITY_PRIVATE
    ),
}

phase41_shadow_component_metrics = {
    name: phase41_local_metrics(
        phase41_shadow_labels,
        probability,
    )
    for name, probability
    in phase41_shadow_component_probabilities.items()
}

phase41_shadow_improvements = {
    "log_loss_gain_over_phase12c": float(
        phase41_shadow_component_metrics[
            "phase12c"
        ]["log_loss"]
        - phase41_shadow_component_metrics[
            "phase41"
        ]["log_loss"]
    ),
    "auroc_gain_over_phase12c": float(
        phase41_shadow_component_metrics[
            "phase41"
        ]["auroc"]
        - phase41_shadow_component_metrics[
            "phase12c"
        ]["auroc"]
    ),
    "log_loss_gain_over_phase36": float(
        phase41_shadow_component_metrics[
            "phase36"
        ]["log_loss"]
        - phase41_shadow_component_metrics[
            "phase41"
        ]["log_loss"]
    ),
    "auroc_gain_over_phase36": float(
        phase41_shadow_component_metrics[
            "phase41"
        ]["auroc"]
        - phase41_shadow_component_metrics[
            "phase36"
        ]["auroc"]
    ),
    "log_loss_gain_over_phase39": float(
        phase41_shadow_component_metrics[
            "phase39"
        ]["log_loss"]
        - phase41_shadow_component_metrics[
            "phase41"
        ]["log_loss"]
    ),
    "auroc_gain_over_phase39": float(
        phase41_shadow_component_metrics[
            "phase41"
        ]["auroc"]
        - phase41_shadow_component_metrics[
            "phase39"
        ]["auroc"]
    ),
}


# ---------------------------------------------------------
# Anonymous per-shadow-group robustness metrics.
# Group identities are deliberately omitted from the report
# to discourage post-hoc group-specific tuning.
# ---------------------------------------------------------

phase41_shadow_group_private_records = []

for group in np.unique(
    phase41_shadow_groups
):
    local_indices = np.flatnonzero(
        phase41_shadow_groups == group
    )

    baseline_metrics = (
        phase41_local_metrics(
            phase41_shadow_labels[
                local_indices
            ],
            phase41_shadow_component_probabilities[
                "phase12c"
            ][local_indices],
        )
    )

    candidate_metrics = (
        phase41_local_metrics(
            phase41_shadow_labels[
                local_indices
            ],
            phase41_shadow_component_probabilities[
                "phase41"
            ][local_indices],
        )
    )

    phase39_metrics = (
        phase41_local_metrics(
            phase41_shadow_labels[
                local_indices
            ],
            phase41_shadow_component_probabilities[
                "phase39"
            ][local_indices],
        )
    )

    phase41_shadow_group_private_records.append({
        "group": int(group),
        "n": int(len(local_indices)),
        "baseline_log_loss": float(
            baseline_metrics["log_loss"]
        ),
        "phase39_log_loss": float(
            phase39_metrics["log_loss"]
        ),
        "phase41_log_loss": float(
            candidate_metrics["log_loss"]
        ),
        "phase41_gain_over_phase12c": float(
            baseline_metrics["log_loss"]
            - candidate_metrics["log_loss"]
        ),
        "phase41_gain_over_phase39": float(
            phase39_metrics["log_loss"]
            - candidate_metrics["log_loss"]
        ),
        "baseline_auroc": (
            baseline_metrics["auroc"]
        ),
        "phase41_auroc": (
            candidate_metrics["auroc"]
        ),
    })

phase41_shadow_major_records = [
    record
    for record
    in phase41_shadow_group_private_records
    if record["n"] >= 30
]

phase41_shadow_group_win_count = sum(
    record["phase41_gain_over_phase12c"]
    > 0
    for record
    in phase41_shadow_group_private_records
)

phase41_shadow_maximum_major_harm = max(
    -record["phase41_gain_over_phase12c"]
    for record
    in phase41_shadow_major_records
)

phase41_shadow_anonymous_groups = []

for anonymous_index, record in enumerate(
    sorted(
        phase41_shadow_group_private_records,
        key=lambda item: (
            -item["n"],
            item["phase41_log_loss"],
        ),
    ),
    start=1,
):
    phase41_shadow_anonymous_groups.append({
        "shadow_group_slot": int(
            anonymous_index
        ),
        "n": int(record["n"]),
        "phase41_gain_over_phase12c": round(
            record[
                "phase41_gain_over_phase12c"
            ],
            6,
        ),
        "phase41_gain_over_phase39": round(
            record[
                "phase41_gain_over_phase39"
            ],
            6,
        ),
        "baseline_auroc": (
            None
            if record["baseline_auroc"]
            is None
            else round(
                record["baseline_auroc"],
                6,
            )
        ),
        "phase41_auroc": (
            None
            if record["phase41_auroc"]
            is None
            else round(
                record["phase41_auroc"],
                6,
            )
        ),
    })


# ---------------------------------------------------------
# Within-group log-loss bootstrap.
# ---------------------------------------------------------

phase41_shadow_baseline_case_loss = (
    -phase41_shadow_labels
    * np.log(
        phase41_safe_probability(
            phase41_shadow_component_probabilities[
                "phase12c"
            ]
        )
    )
    - (
        1 - phase41_shadow_labels
    )
    * np.log(
        1.0
        - phase41_safe_probability(
            phase41_shadow_component_probabilities[
                "phase12c"
            ]
        )
    )
)

phase41_shadow_candidate_case_loss = (
    -phase41_shadow_labels
    * np.log(
        phase41_safe_probability(
            PHASE41_SHADOW_PROBABILITY_PRIVATE
        )
    )
    - (
        1 - phase41_shadow_labels
    )
    * np.log(
        1.0
        - phase41_safe_probability(
            PHASE41_SHADOW_PROBABILITY_PRIVATE
        )
    )
)

phase41_shadow_case_gain = (
    phase41_shadow_baseline_case_loss
    - phase41_shadow_candidate_case_loss
)

phase41_shadow_bootstrap_rng = (
    np.random.default_rng(412101)
)

phase41_shadow_bootstrap_gains = []

for _ in range(5000):
    sampled_parts = []

    for group in np.unique(
        phase41_shadow_groups
    ):
        local_indices = np.flatnonzero(
            phase41_shadow_groups == group
        )

        sampled_parts.append(
            phase41_shadow_bootstrap_rng.choice(
                local_indices,
                size=len(local_indices),
                replace=True,
            )
        )

    sampled_indices = np.concatenate(
        sampled_parts
    )

    phase41_shadow_bootstrap_gains.append(
        float(
            phase41_shadow_case_gain[
                sampled_indices
            ].mean()
        )
    )

phase41_shadow_bootstrap_gains = (
    np.asarray(
        phase41_shadow_bootstrap_gains,
        dtype=np.float64,
    )
)


# ---------------------------------------------------------
# Group-and-label-stratified AUROC bootstrap.
# ---------------------------------------------------------

phase41_shadow_auc_gains = []

for _ in range(3000):
    sampled_parts = []

    for group in np.unique(
        phase41_shadow_groups
    ):
        for target in [0, 1]:
            local_indices = np.flatnonzero(
                (
                    phase41_shadow_groups
                    == group
                )
                & (
                    phase41_shadow_labels
                    == target
                )
            )

            if len(local_indices) == 0:
                continue

            sampled_parts.append(
                phase41_shadow_bootstrap_rng.choice(
                    local_indices,
                    size=len(local_indices),
                    replace=True,
                )
            )

    sampled_indices = np.concatenate(
        sampled_parts
    )

    sampled_labels = (
        phase41_shadow_labels[
            sampled_indices
        ]
    )

    baseline_auc = roc_auc_score(
        sampled_labels,
        phase41_shadow_component_probabilities[
            "phase12c"
        ][sampled_indices],
    )

    candidate_auc = roc_auc_score(
        sampled_labels,
        PHASE41_SHADOW_PROBABILITY_PRIVATE[
            sampled_indices
        ],
    )

    phase41_shadow_auc_gains.append(
        float(
            candidate_auc
            - baseline_auc
        )
    )

phase41_shadow_auc_gains = np.asarray(
    phase41_shadow_auc_gains,
    dtype=np.float64,
)


# ---------------------------------------------------------
# Frozen pass/fail contract.
# ---------------------------------------------------------

PHASE41_SHADOW_THRESHOLDS = {
    "minimum_log_loss_gain_over_phase12c": 0.003,
    "minimum_auroc_gain_over_phase12c": 0.002,
    "minimum_log_loss_bootstrap_probability_positive": 0.80,
    "minimum_auroc_bootstrap_probability_positive": 0.80,
    "maximum_major_group_harm": 0.020,
    "minimum_shadow_group_wins": 2,
    "maximum_log_loss_excess_over_phase39": 0.002,
    "maximum_auroc_deficit_vs_phase39": 0.0005,
}

phase41_shadow_gate_passed = (
    phase41_shadow_improvements[
        "log_loss_gain_over_phase12c"
    ]
    >= PHASE41_SHADOW_THRESHOLDS[
        "minimum_log_loss_gain_over_phase12c"
    ]
    and phase41_shadow_improvements[
        "auroc_gain_over_phase12c"
    ]
    >= PHASE41_SHADOW_THRESHOLDS[
        "minimum_auroc_gain_over_phase12c"
    ]
    and float(
        np.mean(
            phase41_shadow_bootstrap_gains
            > 0
        )
    )
    >= PHASE41_SHADOW_THRESHOLDS[
        "minimum_log_loss_bootstrap_probability_positive"
    ]
    and float(
        np.mean(
            phase41_shadow_auc_gains > 0
        )
    )
    >= PHASE41_SHADOW_THRESHOLDS[
        "minimum_auroc_bootstrap_probability_positive"
    ]
    and phase41_shadow_maximum_major_harm
    <= PHASE41_SHADOW_THRESHOLDS[
        "maximum_major_group_harm"
    ]
    and phase41_shadow_group_win_count
    >= PHASE41_SHADOW_THRESHOLDS[
        "minimum_shadow_group_wins"
    ]
    and phase41_shadow_improvements[
        "log_loss_gain_over_phase39"
    ]
    >= -PHASE41_SHADOW_THRESHOLDS[
        "maximum_log_loss_excess_over_phase39"
    ]
    and phase41_shadow_improvements[
        "auroc_gain_over_phase39"
    ]
    >= -PHASE41_SHADOW_THRESHOLDS[
        "maximum_auroc_deficit_vs_phase39"
    ]
)

PHASE41_SHADOW_EVALUATION_COMPLETE = True
PHASE41_SHADOW_GATE_PASSED = bool(
    phase41_shadow_gate_passed
)

phase41_shadow_report = {
    "phase": (
        "phase41_frozen_prospective_"
        "shadow_evaluation"
    ),
    "status": (
        "shadow_gate_passed"
        if phase41_shadow_gate_passed
        else (
            "shadow_gate_failed_"
            "do_not_tune_on_shadow"
        )
    ),
    "frozen_specification": {
        "rank_lambda": (
            phase41_frozen_specification[
                "rank_lambda"
            ]
        ),
        "l2_value": (
            phase41_frozen_specification[
                "l2_value"
            ]
        ),
        "alpha": (
            phase41_frozen_specification[
                "alpha"
            ]
        ),
        "uncertainty_exponent": (
            phase41_frozen_specification[
                "uncertainty_exponent"
            ]
        ),
        "residual_cap": (
            phase41_frozen_specification[
                "residual_cap"
            ]
        ),
    },
    "shadow": {
        "case_count": int(
            len(
                phase41_shadow_valid_indices
            )
        ),
        "group_count": int(
            np.unique(
                phase41_shadow_groups
            ).size
        ),
        "contract_sha256": (
            PHASE41_SHADOW_CONTRACT_SHA256
        ),
    },
    "metrics": {
        name: {
            key: (
                None
                if value is None
                else round(value, 6)
            )
            for key, value
            in metrics.items()
        }
        for name, metrics
        in phase41_shadow_component_metrics.items()
    },
    "improvements": {
        key: round(value, 6)
        for key, value
        in phase41_shadow_improvements.items()
    },
    "anonymous_group_metrics": (
        phase41_shadow_anonymous_groups
    ),
    "robustness": {
        "shadow_group_wins": int(
            phase41_shadow_group_win_count
        ),
        "shadow_group_count": int(
            len(
                phase41_shadow_group_private_records
            )
        ),
        "maximum_major_group_harm": round(
            float(
                phase41_shadow_maximum_major_harm
            ),
            6,
        ),
        "log_loss_bootstrap": {
            "replicate_count": 5000,
            "mean_gain": round(
                float(
                    phase41_shadow_bootstrap_gains.mean()
                ),
                6,
            ),
            "q05_gain": round(
                float(
                    np.quantile(
                        phase41_shadow_bootstrap_gains,
                        0.05,
                    )
                ),
                6,
            ),
            "probability_positive": round(
                float(
                    np.mean(
                        phase41_shadow_bootstrap_gains
                        > 0
                    )
                ),
                6,
            ),
        },
        "auroc_bootstrap": {
            "replicate_count": 3000,
            "mean_gain": round(
                float(
                    phase41_shadow_auc_gains.mean()
                ),
                6,
            ),
            "q05_gain": round(
                float(
                    np.quantile(
                        phase41_shadow_auc_gains,
                        0.05,
                    )
                ),
                6,
            ),
            "probability_positive": round(
                float(
                    np.mean(
                        phase41_shadow_auc_gains
                        > 0
                    )
                ),
                6,
            ),
        },
    },
    "thresholds": (
        PHASE41_SHADOW_THRESHOLDS
    ),
    "shadow_gate_passed": bool(
        phase41_shadow_gate_passed
    ),
    "specification_modified_after_shadow": False,
    "shadow_labels_used_once": True,
    "shadow_labels_used_for_selection": False,
    "model_refit_used_development_only": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE41_SHADOW_EVALUATION_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE41_SHADOW"
)
print(
    json.dumps(
        phase41_shadow_report,
        indent=2,
        default=lambda value: (
            value.item()
            if isinstance(value, np.generic)
            else value
        ),
    )
)
print(
    "END SANITIZED_PHASE41_SHADOW"
)

BEGIN SANITIZED_PHASE41_SHADOW
{
  "phase": "phase41_frozen_prospective_shadow_evaluation",
  "status": "shadow_gate_failed_do_not_tune_on_shadow",
  "frozen_specification": {
    "rank_lambda": 0.2,
    "l2_value": 0.0001,
    "alpha": 0.6,
    "uncertainty_exponent": 0.0,
    "residual_cap": 1.25
  },
  "shadow": {
    "case_count": 257,
    "group_count": 4,
    "contract_sha256": "6f11fdf646f7468d8e641faa10f8b7b01d54c6850bc732449ab8852cf3d017f2"
  },
  "metrics": {
    "phase12c": {
      "n": 257,
      "log_loss": 0.204037,
      "mean_probability": 0.558616,
      "prevalence": 0.529183,
      "auroc": 0.97314
    },
    "phase36": {
      "n": 257,
      "log_loss": 0.218145,
      "mean_probability": 0.578847,
      "prevalence": 0.529183,
      "auroc": 0.9715
    },
    "phase39": {
      "n": 257,
      "log_loss": 0.225321,
      "mean_probability": 0.596774,
      "prevalence": 0.529183,
      "auroc": 0.974234
    },
    "phase41": {
      "n": 257,
      "log_loss": 0.2

In [8]:
# Phase42 Cell 137A
# Define a fully nested, within-acquisition-protocol validation contract
# for a hierarchically shrunk safety gate.
#
# This cell creates folds and residual candidates only. It does not fit,
# select, or evaluate a Phase42 candidate.

from __future__ import annotations

import hashlib
import json
import time
from pathlib import Path

import numpy as np

PHASE42_CONTRACT_STARTED = (
    time.perf_counter()
)

assert (
    PHASE41_SHADOW_EVALUATION_COMPLETE
    is True
)

assert (
    PHASE41_SHADOW_GATE_PASSED
    is False
), {
    "message": (
        "Phase42 is intended only after the "
        "Phase41 universal ranker is rejected."
    )
}

PHASE42_CONFIG = {
    "phase": (
        "phase42_hierarchical_known_protocol_gate"
    ),
    "random_seed": 420701,
    "outer_fold_count": 5,
    "inner_fold_count": 4,
    "phase39_blend_weights": [
        0.0,
        0.25,
        0.50,
        0.75,
        1.0,
    ],
    "residual_caps": [
        0.50,
        1.00,
        1.50,
        2.00,
    ],
    "uncertainty_exponents": [
        0.0,
        0.50,
        1.0,
    ],
    "shrinkage_strengths": [
        20.0,
        50.0,
        100.0,
        200.0,
    ],
    "alpha_grid": [
        round(value, 4)
        for value in np.linspace(
            0.0,
            1.0,
            21,
        )
    ],
    "log_loss_auroc_tie_band": 0.001,
    "minimum_group_size_for_auroc_tie_break": 30,
    "unknown_protocol_policy": "phase12c_anchor_only",
}

PHASE42_MODEL_CANDIDATE_COUNT = (
    len(
        PHASE42_CONFIG[
            "phase39_blend_weights"
        ]
    )
    * len(
        PHASE42_CONFIG[
            "residual_caps"
        ]
    )
    * len(
        PHASE42_CONFIG[
            "uncertainty_exponents"
        ]
    )
    * len(
        PHASE42_CONFIG[
            "shrinkage_strengths"
        ]
    )
)

assert PHASE42_MODEL_CANDIDATE_COUNT == 240


def phase42_stable_hash(
    uid,
    group,
    target,
):
    payload = (
        f"{PHASE42_CONFIG['random_seed']}|"
        f"{group}|{target}|{uid}"
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


# ---------------------------------------------------------
# Five folds stratified within each acquisition group and
# class. These are not group-held-out folds:
#
# - training and validation share known acquisition protocols;
# - individual validation cases remain held out;
# - this directly simulates Phase40's known-prototype branch.
# ---------------------------------------------------------

PHASE42_OUTER_FOLD_ID_PRIVATE = np.full(
    EXPECTED_CASES,
    -1,
    dtype=np.int64,
)

for group in sorted(
    np.unique(
        phase20_acquisition_groups
    ).tolist()
):
    for target in [0, 1]:
        stratum_indices = np.flatnonzero(
            (
                phase20_acquisition_groups
                == group
            )
            & (
                PHASE19_LABELS
                == target
            )
        )

        ordered_indices = sorted(
            stratum_indices.tolist(),
            key=lambda index: (
                phase42_stable_hash(
                    case_df.iloc[index]["uid"],
                    group,
                    target,
                )
            ),
        )

        for position, index in enumerate(
            ordered_indices
        ):
            PHASE42_OUTER_FOLD_ID_PRIVATE[
                index
            ] = (
                position
                % PHASE42_CONFIG[
                    "outer_fold_count"
                ]
            )

assert np.all(
    PHASE42_OUTER_FOLD_ID_PRIVATE >= 0
)

assert np.all(
    PHASE42_OUTER_FOLD_ID_PRIVATE
    < PHASE42_CONFIG[
        "outer_fold_count"
    ]
)


PHASE42_OUTER_PARTITIONS = []
phase42_outer_records = []
phase42_outer_coverage = np.zeros(
    EXPECTED_CASES,
    dtype=np.int64,
)

for fold in range(
    PHASE42_CONFIG[
        "outer_fold_count"
    ]
):
    validation_indices = np.flatnonzero(
        PHASE42_OUTER_FOLD_ID_PRIVATE
        == fold
    ).astype(np.int64)

    training_indices = np.flatnonzero(
        PHASE42_OUTER_FOLD_ID_PRIVATE
        != fold
    ).astype(np.int64)

    assert len(
        np.intersect1d(
            training_indices,
            validation_indices,
        )
    ) == 0

    assert np.unique(
        PHASE19_LABELS[
            training_indices
        ]
    ).size == 2

    assert np.unique(
        PHASE19_LABELS[
            validation_indices
        ]
    ).size == 2

    training_groups = set(
        phase20_acquisition_groups[
            training_indices
        ].tolist()
    )

    validation_groups = set(
        phase20_acquisition_groups[
            validation_indices
        ].tolist()
    )

    shared_groups = (
        training_groups
        & validation_groups
    )

    # Tiny groups may not appear in every validation fold,
    # but every validation group must be known in training.
    assert validation_groups.issubset(
        training_groups
    )

    phase42_outer_coverage[
        validation_indices
    ] += 1

    PHASE42_OUTER_PARTITIONS.append({
        "fold": int(fold),
        "train": training_indices,
        "valid": validation_indices,
    })

    phase42_outer_records.append({
        "fold": int(fold),
        "train_n": int(
            len(training_indices)
        ),
        "valid_n": int(
            len(validation_indices)
        ),
        "train_group_count": int(
            len(training_groups)
        ),
        "valid_group_count": int(
            len(validation_groups)
        ),
        "shared_group_count": int(
            len(shared_groups)
        ),
        "valid_prevalence": round(
            float(
                PHASE19_LABELS[
                    validation_indices
                ].mean()
            ),
            6,
        ),
    })

assert np.all(
    phase42_outer_coverage == 1
)


# ---------------------------------------------------------
# Residual source contract
# ---------------------------------------------------------
#
# Phase36 final update:
#     r36 = 0.75 * r36_raw
#
# Phase39 uncapped update:
#     r39 = 0.75*r36_raw + 0.25*r33
#
# Phase42 searches a convex interpolation between these two.
# It does not reintroduce Phase31 or Phase38.

PHASE42_R36_RESIDUAL_PRIVATE = (
    0.75
    * PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE
)

PHASE42_R39_RESIDUAL_PRIVATE = (
    PHASE41_PHASE39_COMBINED_RESIDUAL_PRIVATE.copy()
)

assert PHASE42_R36_RESIDUAL_PRIVATE.shape == (
    EXPECTED_CASES,
)

assert PHASE42_R39_RESIDUAL_PRIVATE.shape == (
    EXPECTED_CASES,
)

assert np.isfinite(
    PHASE42_R36_RESIDUAL_PRIVATE
).all()

assert np.isfinite(
    PHASE42_R39_RESIDUAL_PRIVATE
).all()

# Verify the residual definitions reproduce their historical
# component probabilities exactly.

phase42_phase36_reconstruction = expit(
    phase41_baseline_logit
    + PHASE42_R36_RESIDUAL_PRIVATE
)

phase42_phase39_reconstruction = expit(
    phase41_baseline_logit
    + np.clip(
        PHASE42_R39_RESIDUAL_PRIVATE,
        -2.0,
        2.0,
    )
)

phase42_phase36_reconstruction_error = (
    np.abs(
        phase42_phase36_reconstruction
        - PHASE41_PHASE36_OOF_PRIVATE
    )
)

phase42_phase39_reconstruction_error = (
    np.abs(
        phase42_phase39_reconstruction
        - PHASE41_PHASE39_OOF_PRIVATE
    )
)

assert float(
    phase42_phase36_reconstruction_error.max()
) <= 5e-7

assert float(
    phase42_phase39_reconstruction_error.max()
) <= 5e-7


# ---------------------------------------------------------
# Persistent private contract
# ---------------------------------------------------------

PHASE42_PARTITION_PATH = Path(
    "/kaggle/working/"
    "phase42_known_protocol_folds_private.npz"
)

np.savez_compressed(
    PHASE42_PARTITION_PATH,
    outer_fold_id=(
        PHASE42_OUTER_FOLD_ID_PRIVATE
    ),
)

phase42_partition_sha256 = (
    hashlib.sha256(
        PHASE42_OUTER_FOLD_ID_PRIVATE
        .astype("<i8")
        .tobytes()
    ).hexdigest()
)

PHASE42_CONTRACT_PRIVATE = {
    "configuration": PHASE42_CONFIG,
    "partition_sha256": (
        phase42_partition_sha256
    ),
    "phase41_shadow_candidate_rejected": True,
    "phase41_shadow_reused_as_holdout": False,
}

phase42_contract_report = {
    "phase": (
        "phase42_hierarchical_known_"
        "protocol_gate_contract"
    ),
    "status": "accepted",
    "case_count": EXPECTED_CASES,
    "validation_design": {
        "outer_fold_count": int(
            len(
                PHASE42_OUTER_PARTITIONS
            )
        ),
        "folds": phase42_outer_records,
        "exact_case_coverage": True,
        "stratification": (
            "within_acquisition_group_and_label"
        ),
        "known_protocol_simulation": True,
        "group_held_out_simulation": False,
    },
    "candidate_grid": {
        "phase39_blend_weight_count": len(
            PHASE42_CONFIG[
                "phase39_blend_weights"
            ]
        ),
        "residual_cap_count": len(
            PHASE42_CONFIG[
                "residual_caps"
            ]
        ),
        "uncertainty_exponent_count": len(
            PHASE42_CONFIG[
                "uncertainty_exponents"
            ]
        ),
        "shrinkage_strength_count": len(
            PHASE42_CONFIG[
                "shrinkage_strengths"
            ]
        ),
        "candidate_count": int(
            PHASE42_MODEL_CANDIDATE_COUNT
        ),
        "group_alpha_candidate_count": len(
            PHASE42_CONFIG[
                "alpha_grid"
            ]
        ),
    },
    "residual_sources": {
        "phase36": (
            "0.75 * phase36_raw_residual"
        ),
        "phase39": (
            "0.75 * phase36_raw_residual "
            "+ 0.25 * phase33_residual"
        ),
        "planned_interpolation": (
            "(1-beta)*r36 + beta*r39"
        ),
    },
    "historical_reconstruction_error": {
        "phase36_maximum": float(
            phase42_phase36_reconstruction_error.max()
        ),
        "phase39_maximum": float(
            phase42_phase39_reconstruction_error.max()
        ),
    },
    "deployment_policy": {
        "known_protocol": (
            "hierarchically_shrunk_"
            "protocol_specific_alpha"
        ),
        "unknown_protocol": (
            "phase12c_anchor_only"
        ),
        "explicit_probability_intercept": False,
        "test_batch_statistics": False,
    },
    "epistemic_status": {
        "new_pristine_holdout_available": False,
        "reason": (
            "all training labels participated in "
            "earlier aggregate analysis"
        ),
        "planned_estimator": (
            "fully_nested_cross_validation"
        ),
        "phase41_shadow_used_for_phase42_selection": False,
    },
    "partition_sha256": (
        phase42_partition_sha256
    ),
    "partition_written": (
        PHASE42_PARTITION_PATH.is_file()
    ),
    "labels_used_for_fold_stratification": True,
    "model_parameters_selected": False,
    "phase41_shadow_labels_reused": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE42_CONTRACT_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_CONTRACT"
)
print(
    json.dumps(
        phase42_contract_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_CONTRACT"
)

BEGIN SANITIZED_PHASE42_CONTRACT
{
  "phase": "phase42_hierarchical_known_protocol_gate_contract",
  "status": "accepted",
  "case_count": 1362,
  "validation_design": {
    "outer_fold_count": 5,
    "folds": [
      {
        "fold": 0,
        "train_n": 1076,
        "valid_n": 286,
        "train_group_count": 15,
        "valid_group_count": 15,
        "shared_group_count": 15,
        "valid_prevalence": 0.545455
      },
      {
        "fold": 1,
        "train_n": 1083,
        "valid_n": 279,
        "train_group_count": 15,
        "valid_group_count": 15,
        "shared_group_count": 15,
        "valid_prevalence": 0.548387
      },
      {
        "fold": 2,
        "train_n": 1089,
        "valid_n": 273,
        "train_group_count": 15,
        "valid_group_count": 15,
        "shared_group_count": 15,
        "valid_prevalence": 0.545788
      },
      {
        "fold": 3,
        "train_n": 1098,
        "valid_n": 264,
        "train_group_count": 15,
        "vali

In [9]:
# Phase42 Cell 137B
# Fully nested known-protocol validation.
#
# For every outer fold:
#   1. Use only outer-training cases for four-fold inner selection.
#   2. Freeze one specification.
#   3. Refit protocol-specific alphas on the complete outer-training set.
#   4. Evaluate the outer-validation cases exactly once.
#
# No test or smoke data are read.

from collections import Counter
from scipy.special import expit

PHASE42_NESTED_STARTED = (
    time.perf_counter()
)

PHASE42_INNER_THRESHOLDS = {
    "minimum_log_loss_gain": 0.001,
    "minimum_auroc_gain": 0.0003,
    "maximum_major_group_harm": 0.012,
    "log_loss_selection_band": 0.00075,
}

PHASE42_OUTER_THRESHOLDS = {
    "minimum_log_loss_gain_over_phase12c": 0.003,
    "minimum_auroc_gain_over_phase12c": 0.0015,
    "minimum_log_loss_gain_over_phase36": 0.001,
    "minimum_auroc_gain_over_phase36": 0.0005,
    "minimum_outer_fold_wins": 3,
    "minimum_major_group_wins": 7,
    "maximum_major_group_harm": 0.015,
    "minimum_log_loss_bootstrap_probability_positive": 0.90,
    "minimum_auroc_bootstrap_probability_positive": 0.80,
}

phase42_alpha_grid = np.asarray(
    PHASE42_CONFIG["alpha_grid"],
    dtype=np.float64,
)

phase42_baseline_probability = (
    PHASE41_BASELINE_OOF_PRIVATE
)

phase42_baseline_logit = (
    phase41_baseline_logit
)

phase42_anchor_uncertainty = (
    4.0
    * phase42_baseline_probability
    * (
        1.0
        - phase42_baseline_probability
    )
)


def phase42_binary_metrics(
    labels_local,
    probability_local,
):
    labels_local = np.asarray(
        labels_local,
        dtype=np.int64,
    ).reshape(-1)

    probability_local = np.clip(
        np.asarray(
            probability_local,
            dtype=np.float64,
        ).reshape(-1),
        1e-7,
        1.0 - 1e-7,
    )

    result = {
        "n": int(len(labels_local)),
        "log_loss": float(
            log_loss(
                labels_local,
                probability_local,
                labels=[0, 1],
            )
        ),
        "mean_probability": float(
            probability_local.mean()
        ),
    }

    if np.unique(labels_local).size == 2:
        result["auroc"] = float(
            roc_auc_score(
                labels_local,
                probability_local,
            )
        )
    else:
        result["auroc"] = None

    return result


def phase42_case_log_loss(
    labels_local,
    probability_local,
):
    labels_local = np.asarray(
        labels_local,
        dtype=np.float64,
    )

    probability_local = np.clip(
        np.asarray(
            probability_local,
            dtype=np.float64,
        ),
        1e-7,
        1.0 - 1e-7,
    )

    return (
        -labels_local
        * np.log(probability_local)
        - (
            1.0 - labels_local
        )
        * np.log(
            1.0 - probability_local
        )
    )


def phase42_make_unit_update(
    phase39_blend_weight,
    residual_cap,
    uncertainty_exponent,
):
    blend_weight = float(
        phase39_blend_weight
    )

    residual = (
        (
            1.0 - blend_weight
        )
        * PHASE42_R36_RESIDUAL_PRIVATE
        + blend_weight
        * PHASE42_R39_RESIDUAL_PRIVATE
    )

    bounded_residual = np.clip(
        residual,
        -float(residual_cap),
        float(residual_cap),
    )

    uncertainty_gate = np.power(
        phase42_anchor_uncertainty,
        float(uncertainty_exponent),
    )

    unit_update = (
        uncertainty_gate
        * bounded_residual
    )

    assert unit_update.shape == (
        EXPECTED_CASES,
    )

    assert np.isfinite(
        unit_update
    ).all()

    return unit_update.astype(
        np.float64
    )


# Only 60 distinct update vectors are required;
# shrinkage strength is applied afterward.
PHASE42_UNIT_UPDATES_PRIVATE = {}

for phase39_blend_weight in (
    PHASE42_CONFIG[
        "phase39_blend_weights"
    ]
):
    for residual_cap in (
        PHASE42_CONFIG[
            "residual_caps"
        ]
    ):
        for uncertainty_exponent in (
            PHASE42_CONFIG[
                "uncertainty_exponents"
            ]
        ):
            key = (
                float(
                    phase39_blend_weight
                ),
                float(residual_cap),
                float(
                    uncertainty_exponent
                ),
            )

            PHASE42_UNIT_UPDATES_PRIVATE[
                key
            ] = phase42_make_unit_update(
                phase39_blend_weight=(
                    phase39_blend_weight
                ),
                residual_cap=residual_cap,
                uncertainty_exponent=(
                    uncertainty_exponent
                ),
            )

assert len(
    PHASE42_UNIT_UPDATES_PRIVATE
) == 60


def phase42_fit_unshrunk_alpha_state(
    training_indices,
    unit_update,
):
    training_indices = np.asarray(
        training_indices,
        dtype=np.int64,
    )

    labels_train = PHASE19_LABELS[
        training_indices
    ].astype(np.float64)

    baseline_logit_train = (
        phase42_baseline_logit[
            training_indices
        ]
    )

    update_train = unit_update[
        training_indices
    ]

    candidate_logits = (
        baseline_logit_train[:, None]
        + update_train[:, None]
        * phase42_alpha_grid[None, :]
    )

    candidate_case_losses = (
        np.logaddexp(
            0.0,
            candidate_logits,
        )
        - labels_train[:, None]
        * candidate_logits
    )

    global_losses = (
        candidate_case_losses.mean(
            axis=0
        )
    )

    global_best_index = int(
        np.argmin(global_losses)
    )

    global_alpha = float(
        phase42_alpha_grid[
            global_best_index
        ]
    )

    local_alpha = {}
    group_counts = {}

    training_groups = (
        phase20_acquisition_groups[
            training_indices
        ]
    )

    for group in np.unique(
        training_groups
    ):
        group = int(group)

        group_mask = (
            training_groups == group
        )

        group_count = int(
            group_mask.sum()
        )

        group_losses = (
            candidate_case_losses[
                group_mask
            ].mean(axis=0)
        )

        group_best_index = int(
            np.argmin(group_losses)
        )

        local_alpha[group] = float(
            phase42_alpha_grid[
                group_best_index
            ]
        )

        group_counts[group] = group_count

    return {
        "global_alpha": global_alpha,
        "local_alpha": local_alpha,
        "group_counts": group_counts,
    }


def phase42_shrunk_alpha_for_group(
    alpha_state,
    group,
    shrinkage_strength,
):
    group = int(group)

    # Unknown acquisition protocols always retain the anchor.
    if group not in alpha_state[
        "local_alpha"
    ]:
        return 0.0

    group_count = float(
        alpha_state[
            "group_counts"
        ][group]
    )

    shrinkage_strength = float(
        shrinkage_strength
    )

    local_weight = (
        group_count
        / (
            group_count
            + shrinkage_strength
        )
    )

    alpha = (
        local_weight
        * alpha_state[
            "local_alpha"
        ][group]
        + (
            1.0 - local_weight
        )
        * alpha_state[
            "global_alpha"
        ]
    )

    return float(
        np.clip(
            alpha,
            0.0,
            1.0,
        )
    )


def phase42_predict_with_alpha_state(
    prediction_indices,
    unit_update,
    alpha_state,
    shrinkage_strength,
):
    prediction_indices = np.asarray(
        prediction_indices,
        dtype=np.int64,
    )

    prediction_groups = (
        phase20_acquisition_groups[
            prediction_indices
        ]
    )

    alpha_vector = np.asarray([
        phase42_shrunk_alpha_for_group(
            alpha_state=alpha_state,
            group=group,
            shrinkage_strength=(
                shrinkage_strength
            ),
        )
        for group in prediction_groups
    ], dtype=np.float64)

    final_logit = (
        phase42_baseline_logit[
            prediction_indices
        ]
        + alpha_vector
        * unit_update[
            prediction_indices
        ]
    )

    probability = expit(
        final_logit
    )

    assert np.isfinite(
        probability
    ).all()

    return (
        probability.astype(
            np.float64
        ),
        alpha_vector,
    )


def phase42_candidate_metrics(
    evaluation_indices,
    candidate_probability,
):
    evaluation_indices = np.asarray(
        evaluation_indices,
        dtype=np.int64,
    )

    candidate_probability = np.asarray(
        candidate_probability,
        dtype=np.float64,
    )

    labels_local = PHASE19_LABELS[
        evaluation_indices
    ]

    baseline_local = (
        phase42_baseline_probability[
            evaluation_indices
        ]
    )

    groups_local = (
        phase20_acquisition_groups[
            evaluation_indices
        ]
    )

    baseline_metrics = (
        phase42_binary_metrics(
            labels_local,
            baseline_local,
        )
    )

    candidate_metrics = (
        phase42_binary_metrics(
            labels_local,
            candidate_probability,
        )
    )

    group_changes = {}
    major_group_changes = {}

    for group in np.unique(
        groups_local
    ):
        group = int(group)

        group_mask = (
            groups_local == group
        )

        baseline_group_loss = float(
            log_loss(
                labels_local[
                    group_mask
                ],
                baseline_local[
                    group_mask
                ],
                labels=[0, 1],
            )
        )

        candidate_group_loss = float(
            log_loss(
                labels_local[
                    group_mask
                ],
                candidate_probability[
                    group_mask
                ],
                labels=[0, 1],
            )
        )

        change = float(
            candidate_group_loss
            - baseline_group_loss
        )

        group_changes[group] = change

        if int(group_mask.sum()) >= 30:
            major_group_changes[
                group
            ] = change

    if major_group_changes:
        maximum_major_group_harm = max(
            major_group_changes.values()
        )

        major_group_wins = sum(
            change < 0
            for change
            in major_group_changes.values()
        )
    else:
        maximum_major_group_harm = (
            float("inf")
        )
        major_group_wins = 0

    return {
        "log_loss": float(
            candidate_metrics[
                "log_loss"
            ]
        ),
        "auroc": float(
            candidate_metrics[
                "auroc"
            ]
        ),
        "log_loss_gain": float(
            baseline_metrics[
                "log_loss"
            ]
            - candidate_metrics[
                "log_loss"
            ]
        ),
        "auroc_gain": float(
            candidate_metrics[
                "auroc"
            ]
            - baseline_metrics[
                "auroc"
            ]
        ),
        "group_wins": int(
            sum(
                change < 0
                for change
                in group_changes.values()
            )
        ),
        "group_count": int(
            len(group_changes)
        ),
        "major_group_wins": int(
            major_group_wins
        ),
        "major_group_count": int(
            len(
                major_group_changes
            )
        ),
        "maximum_major_group_harm": float(
            maximum_major_group_harm
        ),
        "group_changes": group_changes,
    }


phase42_candidate_specifications = []

for update_key in (
    PHASE42_UNIT_UPDATES_PRIVATE
):
    (
        phase39_blend_weight,
        residual_cap,
        uncertainty_exponent,
    ) = update_key

    for shrinkage_strength in (
        PHASE42_CONFIG[
            "shrinkage_strengths"
        ]
    ):
        phase42_candidate_specifications.append({
            "phase39_blend_weight": float(
                phase39_blend_weight
            ),
            "residual_cap": float(
                residual_cap
            ),
            "uncertainty_exponent": float(
                uncertainty_exponent
            ),
            "shrinkage_strength": float(
                shrinkage_strength
            ),
            "update_key": update_key,
        })

assert len(
    phase42_candidate_specifications
) == PHASE42_MODEL_CANDIDATE_COUNT


# ---------------------------------------------------------
# Fully nested evaluation
# ---------------------------------------------------------

PHASE42_NESTED_OOF_PRIVATE = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

PHASE42_NESTED_ALPHA_PRIVATE = np.full(
    EXPECTED_CASES,
    np.nan,
    dtype=np.float64,
)

PHASE42_OUTER_SELECTION_PRIVATE = []

phase42_outer_reports = []

for outer_partition in (
    PHASE42_OUTER_PARTITIONS
):
    outer_fold = int(
        outer_partition["fold"]
    )

    outer_train = np.asarray(
        outer_partition["train"],
        dtype=np.int64,
    )

    outer_valid = np.asarray(
        outer_partition["valid"],
        dtype=np.int64,
    )

    inner_validation_fold_values = [
        fold
        for fold in range(
            PHASE42_CONFIG[
                "outer_fold_count"
            ]
        )
        if fold != outer_fold
    ]

    assert len(
        inner_validation_fold_values
    ) == PHASE42_CONFIG[
        "inner_fold_count"
    ]

    inner_partitions = []

    inner_coverage = np.zeros(
        EXPECTED_CASES,
        dtype=np.int64,
    )

    for inner_validation_fold in (
        inner_validation_fold_values
    ):
        inner_valid = outer_train[
            PHASE42_OUTER_FOLD_ID_PRIVATE[
                outer_train
            ]
            == inner_validation_fold
        ]

        inner_fit = outer_train[
            PHASE42_OUTER_FOLD_ID_PRIVATE[
                outer_train
            ]
            != inner_validation_fold
        ]

        assert len(inner_fit) > 0
        assert len(inner_valid) > 0

        assert len(
            np.intersect1d(
                inner_fit,
                inner_valid,
            )
        ) == 0

        inner_coverage[
            inner_valid
        ] += 1

        inner_partitions.append({
            "fit": inner_fit,
            "valid": inner_valid,
        })

    assert np.all(
        inner_coverage[outer_train] == 1
    )

    outer_candidate_records = []

    # Search each of 60 update vectors.
    for update_key, unit_update in (
        PHASE42_UNIT_UPDATES_PRIVATE.items()
    ):
        # Alpha optima do not depend on shrinkage strength,
        # so fit these states only once per inner split.
        inner_alpha_states = [
            phase42_fit_unshrunk_alpha_state(
                training_indices=(
                    inner_partition["fit"]
                ),
                unit_update=unit_update,
            )
            for inner_partition
            in inner_partitions
        ]

        for shrinkage_strength in (
            PHASE42_CONFIG[
                "shrinkage_strengths"
            ]
        ):
            inner_oof = np.full(
                EXPECTED_CASES,
                np.nan,
                dtype=np.float64,
            )

            for (
                inner_partition,
                alpha_state,
            ) in zip(
                inner_partitions,
                inner_alpha_states,
            ):
                inner_probability, _ = (
                    phase42_predict_with_alpha_state(
                        prediction_indices=(
                            inner_partition[
                                "valid"
                            ]
                        ),
                        unit_update=unit_update,
                        alpha_state=alpha_state,
                        shrinkage_strength=(
                            shrinkage_strength
                        ),
                    )
                )

                inner_oof[
                    inner_partition["valid"]
                ] = inner_probability

            assert np.isfinite(
                inner_oof[
                    outer_train
                ]
            ).all()

            metrics = (
                phase42_candidate_metrics(
                    evaluation_indices=(
                        outer_train
                    ),
                    candidate_probability=(
                        inner_oof[
                            outer_train
                        ]
                    ),
                )
            )

            eligible = (
                metrics["log_loss_gain"]
                >= PHASE42_INNER_THRESHOLDS[
                    "minimum_log_loss_gain"
                ]
                and metrics["auroc_gain"]
                >= PHASE42_INNER_THRESHOLDS[
                    "minimum_auroc_gain"
                ]
                and metrics[
                    "maximum_major_group_harm"
                ]
                <= PHASE42_INNER_THRESHOLDS[
                    "maximum_major_group_harm"
                ]
                and metrics[
                    "major_group_wins"
                ]
                >= int(
                    np.ceil(
                        metrics[
                            "major_group_count"
                        ]
                        / 2
                    )
                )
            )

            (
                phase39_blend_weight,
                residual_cap,
                uncertainty_exponent,
            ) = update_key

            outer_candidate_records.append({
                "phase39_blend_weight": float(
                    phase39_blend_weight
                ),
                "residual_cap": float(
                    residual_cap
                ),
                "uncertainty_exponent": float(
                    uncertainty_exponent
                ),
                "shrinkage_strength": float(
                    shrinkage_strength
                ),
                **metrics,
                "eligible": bool(
                    eligible
                ),
            })

    assert len(
        outer_candidate_records
    ) == PHASE42_MODEL_CANDIDATE_COUNT

    eligible_records = [
        record
        for record
        in outer_candidate_records
        if record["eligible"]
    ]

    if eligible_records:
        raw_best = min(
            eligible_records,
            key=lambda record: (
                record["log_loss"],
                -record["auroc"],
            ),
        )

        selection_band = [
            record
            for record
            in eligible_records
            if record["log_loss"]
            <= (
                raw_best["log_loss"]
                + PHASE42_INNER_THRESHOLDS[
                    "log_loss_selection_band"
                ]
            )
        ]

        selected = max(
            selection_band,
            key=lambda record: (
                record["auroc"],
                -record[
                    "maximum_major_group_harm"
                ],
                -record["log_loss"],
            ),
        )

        selected_specification = {
            "phase39_blend_weight": float(
                selected[
                    "phase39_blend_weight"
                ]
            ),
            "residual_cap": float(
                selected["residual_cap"]
            ),
            "uncertainty_exponent": float(
                selected[
                    "uncertainty_exponent"
                ]
            ),
            "shrinkage_strength": float(
                selected[
                    "shrinkage_strength"
                ]
            ),
        }

        selected_update_key = (
            selected_specification[
                "phase39_blend_weight"
            ],
            selected_specification[
                "residual_cap"
            ],
            selected_specification[
                "uncertainty_exponent"
            ],
        )

        selected_unit_update = (
            PHASE42_UNIT_UPDATES_PRIVATE[
                selected_update_key
            ]
        )

        final_alpha_state = (
            phase42_fit_unshrunk_alpha_state(
                training_indices=outer_train,
                unit_update=(
                    selected_unit_update
                ),
            )
        )

        (
            outer_probability,
            outer_alpha,
        ) = phase42_predict_with_alpha_state(
            prediction_indices=outer_valid,
            unit_update=(
                selected_unit_update
            ),
            alpha_state=final_alpha_state,
            shrinkage_strength=(
                selected_specification[
                    "shrinkage_strength"
                ]
            ),
        )

        inner_selected_metrics = {
            key: value
            for key, value
            in selected.items()
            if key not in {
                "group_changes",
                "eligible",
            }
        }

        model_advanced = True

    else:
        selected_specification = {
            "phase39_blend_weight": 0.0,
            "residual_cap": 0.0,
            "uncertainty_exponent": 0.0,
            "shrinkage_strength": 0.0,
        }

        outer_probability = (
            phase42_baseline_probability[
                outer_valid
            ].copy()
        )

        outer_alpha = np.zeros(
            len(outer_valid),
            dtype=np.float64,
        )

        inner_selected_metrics = {
            "log_loss_gain": 0.0,
            "auroc_gain": 0.0,
        }

        final_alpha_state = None
        model_advanced = False

    PHASE42_NESTED_OOF_PRIVATE[
        outer_valid
    ] = outer_probability

    PHASE42_NESTED_ALPHA_PRIVATE[
        outer_valid
    ] = outer_alpha

    outer_candidate_metrics = (
        phase42_candidate_metrics(
            evaluation_indices=(
                outer_valid
            ),
            candidate_probability=(
                outer_probability
            ),
        )
    )

    PHASE42_OUTER_SELECTION_PRIVATE.append({
        "fold": outer_fold,
        "model_advanced": (
            model_advanced
        ),
        "selected_specification": (
            selected_specification
        ),
        "inner_selected_metrics": (
            inner_selected_metrics
        ),
        "alpha_state": (
            final_alpha_state
        ),
        "outer_metrics": (
            outer_candidate_metrics
        ),
    })

    phase42_outer_reports.append({
        "fold": outer_fold,
        "train_n": int(
            len(outer_train)
        ),
        "valid_n": int(
            len(outer_valid)
        ),
        "eligible_candidate_count": int(
            len(eligible_records)
        ),
        "model_advanced": bool(
            model_advanced
        ),
        "selected_specification": (
            selected_specification
        ),
        "inner_log_loss_gain": round(
            float(
                inner_selected_metrics[
                    "log_loss_gain"
                ]
            ),
            6,
        ),
        "inner_auroc_gain": round(
            float(
                inner_selected_metrics[
                    "auroc_gain"
                ]
            ),
            6,
        ),
        "outer_log_loss": round(
            float(
                outer_candidate_metrics[
                    "log_loss"
                ]
            ),
            6,
        ),
        "outer_auroc": round(
            float(
                outer_candidate_metrics[
                    "auroc"
                ]
            ),
            6,
        ),
        "outer_log_loss_gain": round(
            float(
                outer_candidate_metrics[
                    "log_loss_gain"
                ]
            ),
            6,
        ),
        "outer_auroc_gain": round(
            float(
                outer_candidate_metrics[
                    "auroc_gain"
                ]
            ),
            6,
        ),
        "mean_effective_alpha": round(
            float(
                outer_alpha.mean()
            ),
            6,
        ),
        "maximum_effective_alpha": round(
            float(
                outer_alpha.max()
            ),
            6,
        ),
    })

    print(
        "Phase42 outer fold "
        f"{outer_fold}: "
        f"eligible={len(eligible_records)}, "
        f"advanced={model_advanced}, "
        "log_loss_gain="
        f"{outer_candidate_metrics['log_loss_gain']:.6f}, "
        "auroc_gain="
        f"{outer_candidate_metrics['auroc_gain']:.6f}"
    )


assert np.isfinite(
    PHASE42_NESTED_OOF_PRIVATE
).all()

assert np.isfinite(
    PHASE42_NESTED_ALPHA_PRIVATE
).all()


# ---------------------------------------------------------
# Aggregate nested metrics
# ---------------------------------------------------------

phase42_baseline_metrics = (
    phase42_binary_metrics(
        PHASE19_LABELS,
        phase42_baseline_probability,
    )
)

phase42_phase36_metrics = (
    phase42_binary_metrics(
        PHASE19_LABELS,
        PHASE41_PHASE36_OOF_PRIVATE,
    )
)

phase42_phase39_metrics = (
    phase42_binary_metrics(
        PHASE19_LABELS,
        PHASE41_PHASE39_OOF_PRIVATE,
    )
)

phase42_nested_metrics = (
    phase42_binary_metrics(
        PHASE19_LABELS,
        PHASE42_NESTED_OOF_PRIVATE,
    )
)

phase42_improvements = {
    "log_loss_gain_over_phase12c": float(
        phase42_baseline_metrics[
            "log_loss"
        ]
        - phase42_nested_metrics[
            "log_loss"
        ]
    ),
    "auroc_gain_over_phase12c": float(
        phase42_nested_metrics["auroc"]
        - phase42_baseline_metrics["auroc"]
    ),
    "log_loss_gain_over_phase36": float(
        phase42_phase36_metrics[
            "log_loss"
        ]
        - phase42_nested_metrics[
            "log_loss"
        ]
    ),
    "auroc_gain_over_phase36": float(
        phase42_nested_metrics["auroc"]
        - phase42_phase36_metrics["auroc"]
    ),
    "log_loss_gain_over_phase39": float(
        phase42_phase39_metrics[
            "log_loss"
        ]
        - phase42_nested_metrics[
            "log_loss"
        ]
    ),
    "auroc_gain_over_phase39": float(
        phase42_nested_metrics["auroc"]
        - phase42_phase39_metrics["auroc"]
    ),
}


# Acquisition-group robustness.
phase42_group_records = []

for group in range(15):
    group_indices = np.flatnonzero(
        phase20_acquisition_groups
        == group
    )

    baseline_group_metrics = (
        phase42_binary_metrics(
            PHASE19_LABELS[
                group_indices
            ],
            phase42_baseline_probability[
                group_indices
            ],
        )
    )

    candidate_group_metrics = (
        phase42_binary_metrics(
            PHASE19_LABELS[
                group_indices
            ],
            PHASE42_NESTED_OOF_PRIVATE[
                group_indices
            ],
        )
    )

    phase42_group_records.append({
        "group": int(group),
        "n": int(
            len(group_indices)
        ),
        "baseline_log_loss": float(
            baseline_group_metrics[
                "log_loss"
            ]
        ),
        "phase42_log_loss": float(
            candidate_group_metrics[
                "log_loss"
            ]
        ),
        "log_loss_gain": float(
            baseline_group_metrics[
                "log_loss"
            ]
            - candidate_group_metrics[
                "log_loss"
            ]
        ),
        "baseline_auroc": (
            baseline_group_metrics[
                "auroc"
            ]
        ),
        "phase42_auroc": (
            candidate_group_metrics[
                "auroc"
            ]
        ),
    })

phase42_major_group_records = [
    record
    for record
    in phase42_group_records
    if record["n"] >= 30
]

phase42_major_group_wins = sum(
    record["log_loss_gain"] > 0
    for record
    in phase42_major_group_records
)

phase42_maximum_major_group_harm = max(
    -record["log_loss_gain"]
    for record
    in phase42_major_group_records
)

phase42_outer_fold_wins = sum(
    record["outer_log_loss_gain"] > 0
    for record
    in phase42_outer_reports
)


# ---------------------------------------------------------
# Stratified robustness bootstrap
# ---------------------------------------------------------

phase42_baseline_case_loss = (
    phase42_case_log_loss(
        PHASE19_LABELS,
        phase42_baseline_probability,
    )
)

phase42_candidate_case_loss = (
    phase42_case_log_loss(
        PHASE19_LABELS,
        PHASE42_NESTED_OOF_PRIVATE,
    )
)

phase42_case_gain = (
    phase42_baseline_case_loss
    - phase42_candidate_case_loss
)

phase42_bootstrap_rng = (
    np.random.default_rng(420901)
)

phase42_log_loss_bootstrap_gain = []

for _ in range(5000):
    sampled_parts = []

    for group in range(15):
        group_indices = np.flatnonzero(
            phase20_acquisition_groups
            == group
        )

        sampled_parts.append(
            phase42_bootstrap_rng.choice(
                group_indices,
                size=len(group_indices),
                replace=True,
            )
        )

    sampled_indices = np.concatenate(
        sampled_parts
    )

    phase42_log_loss_bootstrap_gain.append(
        float(
            phase42_case_gain[
                sampled_indices
            ].mean()
        )
    )

phase42_log_loss_bootstrap_gain = (
    np.asarray(
        phase42_log_loss_bootstrap_gain,
        dtype=np.float64,
    )
)

phase42_auroc_bootstrap_gain = []

for _ in range(3000):
    sampled_parts = []

    for group in range(15):
        for target in [0, 1]:
            stratum_indices = np.flatnonzero(
                (
                    phase20_acquisition_groups
                    == group
                )
                & (
                    PHASE19_LABELS
                    == target
                )
            )

            if len(stratum_indices) == 0:
                continue

            sampled_parts.append(
                phase42_bootstrap_rng.choice(
                    stratum_indices,
                    size=len(stratum_indices),
                    replace=True,
                )
            )

    sampled_indices = np.concatenate(
        sampled_parts
    )

    sampled_labels = PHASE19_LABELS[
        sampled_indices
    ]

    baseline_auc = roc_auc_score(
        sampled_labels,
        phase42_baseline_probability[
            sampled_indices
        ],
    )

    candidate_auc = roc_auc_score(
        sampled_labels,
        PHASE42_NESTED_OOF_PRIVATE[
            sampled_indices
        ],
    )

    phase42_auroc_bootstrap_gain.append(
        float(
            candidate_auc
            - baseline_auc
        )
    )

phase42_auroc_bootstrap_gain = np.asarray(
    phase42_auroc_bootstrap_gain,
    dtype=np.float64,
)


# ---------------------------------------------------------
# Final nested gate
# ---------------------------------------------------------

phase42_nested_gate_passed = (
    phase42_improvements[
        "log_loss_gain_over_phase12c"
    ]
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_log_loss_gain_over_phase12c"
    ]
    and phase42_improvements[
        "auroc_gain_over_phase12c"
    ]
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_auroc_gain_over_phase12c"
    ]
    and phase42_improvements[
        "log_loss_gain_over_phase36"
    ]
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_log_loss_gain_over_phase36"
    ]
    and phase42_improvements[
        "auroc_gain_over_phase36"
    ]
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_auroc_gain_over_phase36"
    ]
    and phase42_outer_fold_wins
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_outer_fold_wins"
    ]
    and phase42_major_group_wins
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_major_group_wins"
    ]
    and phase42_maximum_major_group_harm
    <= PHASE42_OUTER_THRESHOLDS[
        "maximum_major_group_harm"
    ]
    and float(
        np.mean(
            phase42_log_loss_bootstrap_gain
            > 0
        )
    )
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_log_loss_bootstrap_probability_positive"
    ]
    and float(
        np.mean(
            phase42_auroc_bootstrap_gain
            > 0
        )
    )
    >= PHASE42_OUTER_THRESHOLDS[
        "minimum_auroc_bootstrap_probability_positive"
    ]
)

PHASE42_NESTED_GATE_PASSED = bool(
    phase42_nested_gate_passed
)


# Selected-specification stability.
phase42_selected_specification_strings = [
    json.dumps(
        record["selected_specification"],
        sort_keys=True,
    )
    for record
    in PHASE42_OUTER_SELECTION_PRIVATE
    if record["model_advanced"]
]

phase42_specification_counts = Counter(
    phase42_selected_specification_strings
)

phase42_specification_stability = [
    {
        "specification": json.loads(
            specification
        ),
        "outer_fold_count": int(count),
    }
    for specification, count
    in phase42_specification_counts.most_common()
]


# Persist private nested results.
np.save(
    "/kaggle/working/"
    "phase42_nested_oof_float64.npy",
    PHASE42_NESTED_OOF_PRIVATE,
    allow_pickle=False,
)

np.save(
    "/kaggle/working/"
    "phase42_nested_alpha_float64.npy",
    PHASE42_NESTED_ALPHA_PRIVATE,
    allow_pickle=False,
)

phase42_private_selection_metadata = {
    "schema_version": 1,
    "phase": PHASE42_CONFIG["phase"],
    "nested_gate_passed": bool(
        PHASE42_NESTED_GATE_PASSED
    ),
    "partition_sha256": (
        PHASE42_CONTRACT_PRIVATE[
            "partition_sha256"
        ]
    ),
    "outer_selections": [
        {
            "fold": int(
                record["fold"]
            ),
            "model_advanced": bool(
                record["model_advanced"]
            ),
            "selected_specification": (
                record[
                    "selected_specification"
                ]
            ),
        }
        for record
        in PHASE42_OUTER_SELECTION_PRIVATE
    ],
}

Path(
    "/kaggle/working/"
    "phase42_nested_selection_private.json"
).write_text(
    json.dumps(
        phase42_private_selection_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


phase42_nested_report = {
    "phase": (
        "phase42_fully_nested_hierarchical_"
        "known_protocol_gate"
    ),
    "status": (
        "nested_gate_passed"
        if phase42_nested_gate_passed
        else "nested_gate_failed"
    ),
    "candidate_count_per_outer_fold": int(
        PHASE42_MODEL_CANDIDATE_COUNT
    ),
    "outer_fold_count": int(
        len(PHASE42_OUTER_PARTITIONS)
    ),
    "inner_fold_count_per_outer_fold": int(
        PHASE42_CONFIG[
            "inner_fold_count"
        ]
    ),
    "components": {
        "phase12c": {
            key: round(value, 6)
            for key, value
            in phase42_baseline_metrics.items()
        },
        "phase36": {
            key: round(value, 6)
            for key, value
            in phase42_phase36_metrics.items()
        },
        "phase39": {
            key: round(value, 6)
            for key, value
            in phase42_phase39_metrics.items()
        },
        "phase42_nested": {
            key: round(value, 6)
            for key, value
            in phase42_nested_metrics.items()
        },
    },
    "improvements": {
        key: round(value, 6)
        for key, value
        in phase42_improvements.items()
    },
    "outer_folds": (
        phase42_outer_reports
    ),
    "selected_specification_stability": (
        phase42_specification_stability
    ),
    "group_robustness": {
        "major_group_count": int(
            len(
                phase42_major_group_records
            )
        ),
        "major_group_wins": int(
            phase42_major_group_wins
        ),
        "maximum_major_group_harm": round(
            float(
                phase42_maximum_major_group_harm
            ),
            6,
        ),
        "all_group_win_count": int(
            sum(
                record["log_loss_gain"] > 0
                for record
                in phase42_group_records
            )
        ),
    },
    "bootstrap": {
        "log_loss": {
            "replicate_count": 5000,
            "mean_gain": round(
                float(
                    phase42_log_loss_bootstrap_gain.mean()
                ),
                6,
            ),
            "q05_gain": round(
                float(
                    np.quantile(
                        phase42_log_loss_bootstrap_gain,
                        0.05,
                    )
                ),
                6,
            ),
            "probability_positive": round(
                float(
                    np.mean(
                        phase42_log_loss_bootstrap_gain
                        > 0
                    )
                ),
                6,
            ),
        },
        "auroc": {
            "replicate_count": 3000,
            "mean_gain": round(
                float(
                    phase42_auroc_bootstrap_gain.mean()
                ),
                6,
            ),
            "q05_gain": round(
                float(
                    np.quantile(
                        phase42_auroc_bootstrap_gain,
                        0.05,
                    )
                ),
                6,
            ),
            "probability_positive": round(
                float(
                    np.mean(
                        phase42_auroc_bootstrap_gain
                        > 0
                    )
                ),
                6,
            ),
        },
    },
    "thresholds": PHASE42_OUTER_THRESHOLDS,
    "nested_gate_passed": bool(
        phase42_nested_gate_passed
    ),
    "validation_semantics": {
        "known_protocol_case_holdout": True,
        "fully_nested_selection": True,
        "outer_validation_case_used_for_its_model_selection": False,
        "unknown_protocol_policy_evaluated": False,
        "phase41_shadow_status_as_holdout_retired": True,
        "previously_unblinded_cases_included_in_nested_cv": True,
        "new_pristine_holdout_claimed": False,
    },
    "group_specific_alpha_used": True,
    "probability_intercept_used": False,
    "test_batch_statistics_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE42_NESTED_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_NESTED"
)
print(
    json.dumps(
        phase42_nested_report,
        indent=2,
        default=lambda value: (
            value.item()
            if isinstance(value, np.generic)
            else value
        ),
    )
)
print(
    "END SANITIZED_PHASE42_NESTED"
)

Phase42 outer fold 0: eligible=176, advanced=True, log_loss_gain=0.018487, auroc_gain=0.008925
Phase42 outer fold 1: eligible=129, advanced=True, log_loss_gain=0.019523, auroc_gain=0.005602
Phase42 outer fold 2: eligible=123, advanced=True, log_loss_gain=-0.014434, auroc_gain=-0.004925
Phase42 outer fold 3: eligible=175, advanced=True, log_loss_gain=0.029170, auroc_gain=0.009678
Phase42 outer fold 4: eligible=208, advanced=True, log_loss_gain=0.018116, auroc_gain=0.009339
BEGIN SANITIZED_PHASE42_NESTED
{
  "phase": "phase42_fully_nested_hierarchical_known_protocol_gate",
  "status": "nested_gate_passed",
  "candidate_count_per_outer_fold": 240,
  "outer_fold_count": 5,
  "inner_fold_count_per_outer_fold": 4,
  "components": {
    "phase12c": {
      "n": 1362,
      "log_loss": 0.307616,
      "mean_probability": 0.530698,
      "auroc": 0.938914
    },
    "phase36": {
      "n": 1362,
      "log_loss": 0.295657,
      "mean_probability": 0.553493,
      "auroc": 0.944143
    },
    "

In [10]:
# Phase42 Cell 137C
# Select one shared Phase42 specification across all five
# known-protocol folds, enforce fold/group robustness, and persist
# a portable scalar gate state.
#
# The nested Phase42 result remains the less-biased performance
# estimate. Metrics from this cell are selection metrics.

import os
from pathlib import Path

PHASE42_SHARED_STARTED = (
    time.perf_counter()
)

PHASE42_SHARED_THRESHOLDS = {
    "minimum_log_loss_gain_over_phase12c": 0.010,
    "minimum_auroc_gain_over_phase12c": 0.004,
    "minimum_log_loss_gain_over_phase36": 0.001,
    "minimum_auroc_gain_over_phase36": 0.000,
    "maximum_log_loss_excess_vs_phase39": 0.001,
    "maximum_auroc_deficit_vs_phase39": 0.001,
    "minimum_outer_fold_wins": 4,
    "maximum_worst_outer_fold_regret": 0.005,
    "minimum_major_group_wins": 7,
    "maximum_major_group_harm": 0.015,
    "log_loss_selection_band": 0.001,
    "minimum_bootstrap_probability_positive": 0.95,
    "minimum_bootstrap_q05_gain": 0.002,
    "minimum_auroc_bootstrap_probability_positive": 0.90,
}

phase42_shared_records = []
phase42_shared_oof_by_candidate = []
phase42_shared_alpha_by_candidate = []


for candidate_index, specification in enumerate(
    phase42_candidate_specifications
):
    update_key = (
        specification["update_key"]
    )

    unit_update = (
        PHASE42_UNIT_UPDATES_PRIVATE[
            update_key
        ]
    )

    shrinkage_strength = float(
        specification[
            "shrinkage_strength"
        ]
    )

    candidate_oof = np.full(
        EXPECTED_CASES,
        np.nan,
        dtype=np.float64,
    )

    candidate_alpha = np.full(
        EXPECTED_CASES,
        np.nan,
        dtype=np.float64,
    )

    fold_records = []

    for outer_partition in (
        PHASE42_OUTER_PARTITIONS
    ):
        fold = int(
            outer_partition["fold"]
        )

        training_indices = np.asarray(
            outer_partition["train"],
            dtype=np.int64,
        )

        validation_indices = np.asarray(
            outer_partition["valid"],
            dtype=np.int64,
        )

        alpha_state = (
            phase42_fit_unshrunk_alpha_state(
                training_indices=(
                    training_indices
                ),
                unit_update=unit_update,
            )
        )

        (
            validation_probability,
            validation_alpha,
        ) = phase42_predict_with_alpha_state(
            prediction_indices=(
                validation_indices
            ),
            unit_update=unit_update,
            alpha_state=alpha_state,
            shrinkage_strength=(
                shrinkage_strength
            ),
        )

        candidate_oof[
            validation_indices
        ] = validation_probability

        candidate_alpha[
            validation_indices
        ] = validation_alpha

        fold_metrics = (
            phase42_candidate_metrics(
                evaluation_indices=(
                    validation_indices
                ),
                candidate_probability=(
                    validation_probability
                ),
            )
        )

        fold_records.append({
            "fold": fold,
            "n": int(
                len(validation_indices)
            ),
            "log_loss": float(
                fold_metrics["log_loss"]
            ),
            "auroc": float(
                fold_metrics["auroc"]
            ),
            "log_loss_gain": float(
                fold_metrics[
                    "log_loss_gain"
                ]
            ),
            "auroc_gain": float(
                fold_metrics[
                    "auroc_gain"
                ]
            ),
        })

    assert np.isfinite(
        candidate_oof
    ).all()

    assert np.isfinite(
        candidate_alpha
    ).all()

    complete_metrics = (
        phase42_candidate_metrics(
            evaluation_indices=(
                np.arange(
                    EXPECTED_CASES,
                    dtype=np.int64,
                )
            ),
            candidate_probability=(
                candidate_oof
            ),
        )
    )

    fold_log_loss_gains = np.asarray([
        record["log_loss_gain"]
        for record in fold_records
    ], dtype=np.float64)

    fold_auroc_gains = np.asarray([
        record["auroc_gain"]
        for record in fold_records
    ], dtype=np.float64)

    outer_fold_wins = int(
        np.sum(
            fold_log_loss_gains > 0
        )
    )

    worst_outer_fold_regret = float(
        max(
            0.0,
            -float(
                fold_log_loss_gains.min()
            ),
        )
    )

    log_loss_gain_over_phase36 = float(
        phase42_phase36_metrics[
            "log_loss"
        ]
        - complete_metrics[
            "log_loss"
        ]
    )

    auroc_gain_over_phase36 = float(
        complete_metrics["auroc"]
        - phase42_phase36_metrics[
            "auroc"
        ]
    )

    log_loss_gain_over_phase39 = float(
        phase42_phase39_metrics[
            "log_loss"
        ]
        - complete_metrics[
            "log_loss"
        ]
    )

    auroc_gain_over_phase39 = float(
        complete_metrics["auroc"]
        - phase42_phase39_metrics[
            "auroc"
        ]
    )

    eligible = (
        complete_metrics[
            "log_loss_gain"
        ]
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_log_loss_gain_over_phase12c"
        ]
        and complete_metrics[
            "auroc_gain"
        ]
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_auroc_gain_over_phase12c"
        ]
        and log_loss_gain_over_phase36
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_log_loss_gain_over_phase36"
        ]
        and auroc_gain_over_phase36
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_auroc_gain_over_phase36"
        ]
        and log_loss_gain_over_phase39
        >= -PHASE42_SHARED_THRESHOLDS[
            "maximum_log_loss_excess_vs_phase39"
        ]
        and auroc_gain_over_phase39
        >= -PHASE42_SHARED_THRESHOLDS[
            "maximum_auroc_deficit_vs_phase39"
        ]
        and outer_fold_wins
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_outer_fold_wins"
        ]
        and worst_outer_fold_regret
        <= PHASE42_SHARED_THRESHOLDS[
            "maximum_worst_outer_fold_regret"
        ]
        and complete_metrics[
            "major_group_wins"
        ]
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_major_group_wins"
        ]
        and complete_metrics[
            "maximum_major_group_harm"
        ]
        <= PHASE42_SHARED_THRESHOLDS[
            "maximum_major_group_harm"
        ]
    )

    record = {
        "candidate_index": int(
            candidate_index
        ),
        "phase39_blend_weight": float(
            specification[
                "phase39_blend_weight"
            ]
        ),
        "residual_cap": float(
            specification[
                "residual_cap"
            ]
        ),
        "uncertainty_exponent": float(
            specification[
                "uncertainty_exponent"
            ]
        ),
        "shrinkage_strength": float(
            specification[
                "shrinkage_strength"
            ]
        ),
        "log_loss": float(
            complete_metrics[
                "log_loss"
            ]
        ),
        "auroc": float(
            complete_metrics[
                "auroc"
            ]
        ),
        "log_loss_gain_over_phase12c": float(
            complete_metrics[
                "log_loss_gain"
            ]
        ),
        "auroc_gain_over_phase12c": float(
            complete_metrics[
                "auroc_gain"
            ]
        ),
        "log_loss_gain_over_phase36": (
            log_loss_gain_over_phase36
        ),
        "auroc_gain_over_phase36": (
            auroc_gain_over_phase36
        ),
        "log_loss_gain_over_phase39": (
            log_loss_gain_over_phase39
        ),
        "auroc_gain_over_phase39": (
            auroc_gain_over_phase39
        ),
        "outer_fold_wins": int(
            outer_fold_wins
        ),
        "worst_outer_fold_regret": float(
            worst_outer_fold_regret
        ),
        "major_group_wins": int(
            complete_metrics[
                "major_group_wins"
            ]
        ),
        "major_group_count": int(
            complete_metrics[
                "major_group_count"
            ]
        ),
        "maximum_major_group_harm": float(
            complete_metrics[
                "maximum_major_group_harm"
            ]
        ),
        "mean_effective_alpha": float(
            candidate_alpha.mean()
        ),
        "minimum_effective_alpha": float(
            candidate_alpha.min()
        ),
        "maximum_effective_alpha": float(
            candidate_alpha.max()
        ),
        "fold_records": fold_records,
        "eligible": bool(eligible),
    }

    phase42_shared_records.append(
        record
    )

    phase42_shared_oof_by_candidate.append(
        candidate_oof
    )

    phase42_shared_alpha_by_candidate.append(
        candidate_alpha
    )


assert len(
    phase42_shared_records
) == PHASE42_MODEL_CANDIDATE_COUNT

phase42_shared_eligible = [
    record
    for record
    in phase42_shared_records
    if record["eligible"]
]

phase42_shared_advanced = bool(
    phase42_shared_eligible
)

if phase42_shared_advanced:
    phase42_shared_raw_best = min(
        phase42_shared_eligible,
        key=lambda record: (
            record["log_loss"],
            -record["auroc"],
            record[
                "worst_outer_fold_regret"
            ],
        ),
    )

    phase42_shared_band = [
        record
        for record
        in phase42_shared_eligible
        if record["log_loss"]
        <= (
            phase42_shared_raw_best[
                "log_loss"
            ]
            + PHASE42_SHARED_THRESHOLDS[
                "log_loss_selection_band"
            ]
        )
    ]

    PHASE42_SHARED_SELECTED_PRIVATE = max(
        phase42_shared_band,
        key=lambda record: (
            record["auroc"],
            -record[
                "worst_outer_fold_regret"
            ],
            -record[
                "maximum_major_group_harm"
            ],
            -record["log_loss"],
        ),
    )

else:
    phase42_shared_raw_best = None
    PHASE42_SHARED_SELECTED_PRIVATE = None


# ---------------------------------------------------------
# Fixed-candidate bootstrap
# ---------------------------------------------------------

if phase42_shared_advanced:
    phase42_selected_candidate_index = int(
        PHASE42_SHARED_SELECTED_PRIVATE[
            "candidate_index"
        ]
    )

    PHASE42_SHARED_OOF_PRIVATE = (
        phase42_shared_oof_by_candidate[
            phase42_selected_candidate_index
        ].copy()
    )

    PHASE42_SHARED_ALPHA_OOF_PRIVATE = (
        phase42_shared_alpha_by_candidate[
            phase42_selected_candidate_index
        ].copy()
    )

    phase42_shared_baseline_case_loss = (
        phase42_case_log_loss(
            PHASE19_LABELS,
            phase42_baseline_probability,
        )
    )

    phase42_shared_candidate_case_loss = (
        phase42_case_log_loss(
            PHASE19_LABELS,
            PHASE42_SHARED_OOF_PRIVATE,
        )
    )

    phase42_shared_case_gain = (
        phase42_shared_baseline_case_loss
        - phase42_shared_candidate_case_loss
    )

    phase42_shared_bootstrap_rng = (
        np.random.default_rng(421101)
    )

    phase42_shared_log_loss_bootstrap = []

    for _ in range(5000):
        sampled_parts = []

        for group in range(15):
            group_indices = np.flatnonzero(
                phase20_acquisition_groups
                == group
            )

            sampled_parts.append(
                phase42_shared_bootstrap_rng.choice(
                    group_indices,
                    size=len(group_indices),
                    replace=True,
                )
            )

        sampled_indices = np.concatenate(
            sampled_parts
        )

        phase42_shared_log_loss_bootstrap.append(
            float(
                phase42_shared_case_gain[
                    sampled_indices
                ].mean()
            )
        )

    phase42_shared_log_loss_bootstrap = (
        np.asarray(
            phase42_shared_log_loss_bootstrap,
            dtype=np.float64,
        )
    )

    phase42_shared_auroc_bootstrap = []

    for _ in range(3000):
        sampled_parts = []

        for group in range(15):
            for target in [0, 1]:
                stratum_indices = np.flatnonzero(
                    (
                        phase20_acquisition_groups
                        == group
                    )
                    & (
                        PHASE19_LABELS
                        == target
                    )
                )

                if len(stratum_indices) == 0:
                    continue

                sampled_parts.append(
                    phase42_shared_bootstrap_rng.choice(
                        stratum_indices,
                        size=len(stratum_indices),
                        replace=True,
                    )
                )

        sampled_indices = np.concatenate(
            sampled_parts
        )

        sampled_labels = PHASE19_LABELS[
            sampled_indices
        ]

        baseline_auc = roc_auc_score(
            sampled_labels,
            phase42_baseline_probability[
                sampled_indices
            ],
        )

        candidate_auc = roc_auc_score(
            sampled_labels,
            PHASE42_SHARED_OOF_PRIVATE[
                sampled_indices
            ],
        )

        phase42_shared_auroc_bootstrap.append(
            float(
                candidate_auc
                - baseline_auc
            )
        )

    phase42_shared_auroc_bootstrap = (
        np.asarray(
            phase42_shared_auroc_bootstrap,
            dtype=np.float64,
        )
    )

    phase42_shared_bootstrap_passed = (
        float(
            np.mean(
                phase42_shared_log_loss_bootstrap
                > 0
            )
        )
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_bootstrap_probability_positive"
        ]
        and float(
            np.quantile(
                phase42_shared_log_loss_bootstrap,
                0.05,
            )
        )
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_bootstrap_q05_gain"
        ]
        and float(
            np.mean(
                phase42_shared_auroc_bootstrap
                > 0
            )
        )
        >= PHASE42_SHARED_THRESHOLDS[
            "minimum_auroc_bootstrap_probability_positive"
        ]
    )

else:
    phase42_shared_bootstrap_passed = False


PHASE42_SHARED_GATE_PASSED = bool(
    phase42_shared_advanced
    and phase42_shared_bootstrap_passed
)


# ---------------------------------------------------------
# Fit the final portable gate state on every training case.
# This state contains no labels, rows, or OOF predictions.
# ---------------------------------------------------------

if PHASE42_SHARED_GATE_PASSED:
    phase42_final_specification = {
        "phase39_blend_weight": float(
            PHASE42_SHARED_SELECTED_PRIVATE[
                "phase39_blend_weight"
            ]
        ),
        "residual_cap": float(
            PHASE42_SHARED_SELECTED_PRIVATE[
                "residual_cap"
            ]
        ),
        "uncertainty_exponent": float(
            PHASE42_SHARED_SELECTED_PRIVATE[
                "uncertainty_exponent"
            ]
        ),
        "shrinkage_strength": float(
            PHASE42_SHARED_SELECTED_PRIVATE[
                "shrinkage_strength"
            ]
        ),
    }

    phase42_final_update_key = (
        phase42_final_specification[
            "phase39_blend_weight"
        ],
        phase42_final_specification[
            "residual_cap"
        ],
        phase42_final_specification[
            "uncertainty_exponent"
        ],
    )

    PHASE42_FINAL_UNIT_UPDATE_PRIVATE = (
        PHASE42_UNIT_UPDATES_PRIVATE[
            phase42_final_update_key
        ]
    )

    PHASE42_FINAL_ALPHA_STATE_PRIVATE = (
        phase42_fit_unshrunk_alpha_state(
            training_indices=np.arange(
                EXPECTED_CASES,
                dtype=np.int64,
            ),
            unit_update=(
                PHASE42_FINAL_UNIT_UPDATE_PRIVATE
            ),
        )
    )

    phase42_final_group_alpha = {
        str(group): float(
            phase42_shrunk_alpha_for_group(
                alpha_state=(
                    PHASE42_FINAL_ALPHA_STATE_PRIVATE
                ),
                group=group,
                shrinkage_strength=(
                    phase42_final_specification[
                        "shrinkage_strength"
                    ]
                ),
            )
        )
        for group in range(15)
    }

    PHASE42_FINAL_STATE_PRIVATE = {
        "schema_version": 1,
        "phase": (
            "phase42_hierarchical_"
            "known_protocol_gate"
        ),
        "specification": (
            phase42_final_specification
        ),
        "phase36_weight": float(
            0.75
        ),
        "phase33_weight": float(
            0.25
            * phase42_final_specification[
                "phase39_blend_weight"
            ]
        ),
        "phase36_effective_weight": float(
            1.0
            - 0.25
            * phase42_final_specification[
                "phase39_blend_weight"
            ]
        ),
        "global_alpha_before_shrinkage": float(
            PHASE42_FINAL_ALPHA_STATE_PRIVATE[
                "global_alpha"
            ]
        ),
        "known_group_alpha": (
            phase42_final_group_alpha
        ),
        "unknown_protocol_alpha": 0.0,
        "unknown_protocol_policy": (
            "phase12c_anchor_only"
        ),
        "header_rounding_decimals": 6,
        "test_batch_statistics_used": False,
    }

    PHASE42_PRIVATE_CHECKPOINT_DIRECTORY = Path(
        "/kaggle/working/"
        "phase42_private_checkpoint"
    )

    PHASE42_PRIVATE_CHECKPOINT_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True,
    )

    phase42_final_state_path = (
        PHASE42_PRIVATE_CHECKPOINT_DIRECTORY
        / "phase42_hierarchical_gate.json"
    )

    phase42_temporary_state_path = (
        PHASE42_PRIVATE_CHECKPOINT_DIRECTORY
        / "phase42_hierarchical_gate.json.tmp"
    )

    phase42_temporary_state_path.write_text(
        json.dumps(
            PHASE42_FINAL_STATE_PRIVATE,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    os.replace(
        phase42_temporary_state_path,
        phase42_final_state_path,
    )

    np.save(
        "/kaggle/working/"
        "phase42_shared_oof_float64.npy",
        PHASE42_SHARED_OOF_PRIVATE,
        allow_pickle=False,
    )

    np.save(
        "/kaggle/working/"
        "phase42_shared_alpha_oof_float64.npy",
        PHASE42_SHARED_ALPHA_OOF_PRIVATE,
        allow_pickle=False,
    )

    phase42_portable_alpha_values = (
        np.asarray(
            list(
                phase42_final_group_alpha.values()
            ),
            dtype=np.float64,
        )
    )

else:
    phase42_final_specification = None
    phase42_final_state_path = None
    phase42_portable_alpha_values = None


def phase42_sanitize_shared_record(
    record,
):
    if record is None:
        return None

    return {
        key: value
        for key, value in record.items()
        if key not in {
            "fold_records",
        }
    }


phase42_shared_report = {
    "phase": (
        "phase42_shared_stable_"
        "hierarchical_gate"
    ),
    "status": (
        "portable_state_frozen"
        if PHASE42_SHARED_GATE_PASSED
        else "shared_gate_not_promoted"
    ),
    "search": {
        "candidate_count": int(
            len(phase42_shared_records)
        ),
        "eligible_candidate_count": int(
            len(phase42_shared_eligible)
        ),
        "shared_specification_across_folds": True,
        "outer_fold_count": int(
            len(
                PHASE42_OUTER_PARTITIONS
            )
        ),
    },
    "nested_reference": {
        "log_loss": round(
            float(
                phase42_nested_metrics[
                    "log_loss"
                ]
            ),
            6,
        ),
        "auroc": round(
            float(
                phase42_nested_metrics[
                    "auroc"
                ]
            ),
            6,
        ),
        "less_biased_than_shared_selection_metrics": True,
    },
    "raw_best": (
        phase42_sanitize_shared_record(
            phase42_shared_raw_best
        )
    ),
    "selected": (
        phase42_sanitize_shared_record(
            PHASE42_SHARED_SELECTED_PRIVATE
        )
    ),
    "bootstrap": (
        None
        if not phase42_shared_advanced
        else {
            "log_loss_replicates": 5000,
            "log_loss_mean_gain": round(
                float(
                    phase42_shared_log_loss_bootstrap.mean()
                ),
                6,
            ),
            "log_loss_q05_gain": round(
                float(
                    np.quantile(
                        phase42_shared_log_loss_bootstrap,
                        0.05,
                    )
                ),
                6,
            ),
            "log_loss_probability_positive": round(
                float(
                    np.mean(
                        phase42_shared_log_loss_bootstrap
                        > 0
                    )
                ),
                6,
            ),
            "auroc_replicates": 3000,
            "auroc_mean_gain": round(
                float(
                    phase42_shared_auroc_bootstrap.mean()
                ),
                6,
            ),
            "auroc_q05_gain": round(
                float(
                    np.quantile(
                        phase42_shared_auroc_bootstrap,
                        0.05,
                    )
                ),
                6,
            ),
            "auroc_probability_positive": round(
                float(
                    np.mean(
                        phase42_shared_auroc_bootstrap
                        > 0
                    )
                ),
                6,
            ),
            "bootstrap_gate_passed": bool(
                phase42_shared_bootstrap_passed
            ),
        }
    ),
    "portable_state": (
        None
        if not PHASE42_SHARED_GATE_PASSED
        else {
            "specification": (
                phase42_final_specification
            ),
            "known_group_count": 15,
            "group_alpha_quantiles": {
                "minimum": round(
                    float(
                        phase42_portable_alpha_values.min()
                    ),
                    6,
                ),
                "q25": round(
                    float(
                        np.quantile(
                            phase42_portable_alpha_values,
                            0.25,
                        )
                    ),
                    6,
                ),
                "median": round(
                    float(
                        np.median(
                            phase42_portable_alpha_values
                        )
                    ),
                    6,
                ),
                "q75": round(
                    float(
                        np.quantile(
                            phase42_portable_alpha_values,
                            0.75,
                        )
                    ),
                    6,
                ),
                "maximum": round(
                    float(
                        phase42_portable_alpha_values.max()
                    ),
                    6,
                ),
            },
            "unknown_protocol_alpha": 0.0,
            "checkpoint_file": (
                phase42_final_state_path.name
            ),
        }
    ),
    "thresholds": (
        PHASE42_SHARED_THRESHOLDS
    ),
    "shared_gate_passed": bool(
        PHASE42_SHARED_GATE_PASSED
    ),
    "performance_interpretation": {
        "nested_metrics_are_primary": True,
        "shared_selection_metrics_are_optimistic": True,
        "new_pristine_holdout_claimed": False,
        "public_score_used_for_selection": False,
    },
    "contains_labels": False,
    "portable_state_contains_predictions": False,
    "portable_state_contains_case_rows": False,
    "portable_state_contains_voxel_data": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE42_SHARED_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_SHARED_GATE"
)
print(
    json.dumps(
        phase42_shared_report,
        indent=2,
        default=lambda value: (
            value.item()
            if isinstance(value, np.generic)
            else value
        ),
    )
)
print(
    "END SANITIZED_PHASE42_SHARED_GATE"
)

BEGIN SANITIZED_PHASE42_SHARED_GATE
{
  "phase": "phase42_shared_stable_hierarchical_gate",
  "status": "portable_state_frozen",
  "search": {
    "candidate_count": 240,
    "eligible_candidate_count": 49,
    "shared_specification_across_folds": true,
    "outer_fold_count": 5
  },
  "nested_reference": {
    "log_loss": 0.293516,
    "auroc": 0.9448,
    "less_biased_than_shared_selection_metrics": true
  },
  "raw_best": {
    "candidate_index": 136,
    "phase39_blend_weight": 0.5,
    "residual_cap": 2.0,
    "uncertainty_exponent": 0.5,
    "shrinkage_strength": 20.0,
    "log_loss": 0.2907981379652894,
    "auroc": 0.9457015052078231,
    "log_loss_gain_over_phase12c": 0.016817673172001946,
    "auroc_gain_over_phase12c": 0.006787039757947921,
    "log_loss_gain_over_phase36": 0.004859055532286605,
    "auroc_gain_over_phase36": 0.0015585376737300427,
    "log_loss_gain_over_phase39": 0.0022292109001279536,
    "auroc_gain_over_phase39": 0.00012625025848644889,
    "outer_fold_

In [11]:
# Phase42 Cell 138A
# Correct the redundant Phase36 weight metadata and verify the exact
# Phase42 operation order before any runtime/package modification.

PHASE42_FORMULA_STARTED = (
    time.perf_counter()
)

assert PHASE42_SHARED_GATE_PASSED
assert PHASE42_FINAL_STATE_PRIVATE is not None

phase42_state_path = (
    Path("/kaggle/working/")
    / "phase42_private_checkpoint"
    / "phase42_hierarchical_gate.json"
)

assert phase42_state_path.is_file()

phase42_state = json.loads(
    phase42_state_path.read_text(
        encoding="utf-8"
    )
)

phase42_specification = (
    phase42_state["specification"]
)

assert abs(
    float(
        phase42_specification[
            "phase39_blend_weight"
        ]
    )
    - 0.5
) <= 1e-12

assert abs(
    float(
        phase42_specification[
            "residual_cap"
        ]
    )
    - 1.0
) <= 1e-12

assert abs(
    float(
        phase42_specification[
            "uncertainty_exponent"
        ]
    )
    - 0.0
) <= 1e-12

assert abs(
    float(
        phase42_specification[
            "shrinkage_strength"
        ]
    )
    - 20.0
) <= 1e-12

phase42_beta = float(
    phase42_specification[
        "phase39_blend_weight"
    ]
)

# Exact algebra:
#
# (1-beta)*(0.75*r36_raw)
# + beta*(0.75*r36_raw + 0.25*r33)
#
# = 0.75*r36_raw + 0.25*beta*r33

phase42_r36_weight = 0.75

phase42_r33_weight = (
    0.25 * phase42_beta
)

assert abs(
    phase42_r36_weight - 0.75
) <= 1e-12

assert abs(
    phase42_r33_weight - 0.125
) <= 1e-12

# Remove the incorrect redundant value written by Cell 137C.
phase42_removed_incorrect_value = (
    phase42_state.pop(
        "phase36_effective_weight",
        None,
    )
)

phase42_state.pop(
    "phase36_weight",
    None,
)

phase42_state.pop(
    "phase33_weight",
    None,
)

phase42_state.update({
    "phase36_raw_residual_weight": float(
        phase42_r36_weight
    ),
    "phase33_logit_residual_weight": float(
        phase42_r33_weight
    ),
    "residual_cap": float(
        phase42_specification[
            "residual_cap"
        ]
    ),
    "uncertainty_exponent": float(
        phase42_specification[
            "uncertainty_exponent"
        ]
    ),
    "operation_order": [
        "compute_phase36_raw_residual",
        "compute_phase33_logit_residual",
        "weighted_residual_sum",
        "symmetric_residual_clip",
        "multiply_by_known_group_alpha",
        "add_to_complete_phase12c_anchor_logit",
        "sigmoid",
    ],
    "deployment_formula": (
        "z42 = z_phase12c_full + alpha_group * "
        "clip(0.75*r36_raw + 0.125*r33, -1, 1)"
    ),
    "known_protocol_policy": (
        "exact_header_group_alpha"
    ),
    "unknown_protocol_policy": (
        "phase12c_anchor_only"
    ),
})

assert len(
    phase42_state[
        "known_group_alpha"
    ]
) == 15

phase42_state_alpha_values = (
    np.asarray(
        [
            float(
                phase42_state[
                    "known_group_alpha"
                ][str(group)]
            )
            for group in range(15)
        ],
        dtype=np.float64,
    )
)

assert np.isfinite(
    phase42_state_alpha_values
).all()

assert np.all(
    phase42_state_alpha_values >= 0.0
)

assert np.all(
    phase42_state_alpha_values <= 1.0
)

assert float(
    phase42_state[
        "unknown_protocol_alpha"
    ]
) == 0.0


# ---------------------------------------------------------
# Cross-fitted shared-selection parity
# ---------------------------------------------------------

phase42_exact_combined_residual = (
    phase42_r36_weight
    * PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE
    + phase42_r33_weight
    * PHASE41_PHASE33_RESIDUAL_PRIVATE
)

phase42_exact_bounded_residual = (
    np.clip(
        phase42_exact_combined_residual,
        -float(
            phase42_state[
                "residual_cap"
            ]
        ),
        float(
            phase42_state[
                "residual_cap"
            ]
        ),
    )
)

phase42_shared_reconstructed = expit(
    phase42_baseline_logit
    + PHASE42_SHARED_ALPHA_OOF_PRIVATE
    * phase42_exact_bounded_residual
)

phase42_shared_formula_error = np.abs(
    phase42_shared_reconstructed
    - PHASE42_SHARED_OOF_PRIVATE
)

assert float(
    phase42_shared_formula_error.max()
) <= 5e-12, {
    "message": (
        "Corrected Phase42 formula failed "
        "cross-fitted parity."
    ),
    "maximum_error": float(
        phase42_shared_formula_error.max()
    ),
}


# ---------------------------------------------------------
# Full-fit state self-consistency diagnostic
# ---------------------------------------------------------

phase42_full_fit_alpha_vector = (
    np.asarray(
        [
            phase42_state_alpha_values[
                int(group)
            ]
            for group
            in phase20_acquisition_groups
        ],
        dtype=np.float64,
    )
)

PHASE42_FULL_FIT_DIAGNOSTIC_PRIVATE = (
    expit(
        phase42_baseline_logit
        + phase42_full_fit_alpha_vector
        * phase42_exact_bounded_residual
    )
)

assert np.isfinite(
    PHASE42_FULL_FIT_DIAGNOSTIC_PRIVATE
).all()

phase42_full_fit_diagnostic_metrics = (
    phase42_binary_metrics(
        PHASE19_LABELS,
        PHASE42_FULL_FIT_DIAGNOSTIC_PRIVATE,
    )
)

# Atomically overwrite only the private Phase42 state.
phase42_temporary_state_path = (
    phase42_state_path.with_suffix(
        ".json.tmp"
    )
)

phase42_temporary_state_path.write_text(
    json.dumps(
        phase42_state,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

os.replace(
    phase42_temporary_state_path,
    phase42_state_path,
)

PHASE42_CORRECTED_FINAL_STATE_PRIVATE = (
    phase42_state
)

phase42_formula_report = {
    "phase": (
        "phase42_corrected_portable_"
        "formula_contract"
    ),
    "status": "accepted",
    "corrected_formula": (
        phase42_state[
            "deployment_formula"
        ]
    ),
    "weights": {
        "phase36_raw_residual": round(
            phase42_r36_weight,
            6,
        ),
        "phase33_logit_residual": round(
            phase42_r33_weight,
            6,
        ),
    },
    "residual_cap": float(
        phase42_state[
            "residual_cap"
        ]
    ),
    "uncertainty_exponent": float(
        phase42_state[
            "uncertainty_exponent"
        ]
    ),
    "operation_order": (
        phase42_state[
            "operation_order"
        ]
    ),
    "removed_incorrect_redundant_value": (
        None
        if phase42_removed_incorrect_value
        is None
        else float(
            phase42_removed_incorrect_value
        )
    ),
    "cross_fitted_formula_error": {
        "maximum": float(
            phase42_shared_formula_error.max()
        ),
        "mean": float(
            phase42_shared_formula_error.mean()
        ),
    },
    "reference_metrics": {
        "nested": {
            "log_loss": round(
                float(
                    phase42_nested_metrics[
                        "log_loss"
                    ]
                ),
                6,
            ),
            "auroc": round(
                float(
                    phase42_nested_metrics[
                        "auroc"
                    ]
                ),
                6,
            ),
        },
        "shared_selection": {
            "log_loss": round(
                float(
                    PHASE42_SHARED_SELECTED_PRIVATE[
                        "log_loss"
                    ]
                ),
                6,
            ),
            "auroc": round(
                float(
                    PHASE42_SHARED_SELECTED_PRIVATE[
                        "auroc"
                    ]
                ),
                6,
            ),
        },
        "full_fit_diagnostic_only": {
            "log_loss": round(
                float(
                    phase42_full_fit_diagnostic_metrics[
                        "log_loss"
                    ]
                ),
                6,
            ),
            "auroc": round(
                float(
                    phase42_full_fit_diagnostic_metrics[
                        "auroc"
                    ]
                ),
                6,
            ),
        },
    },
    "known_group_count": 15,
    "known_group_alpha_quantiles": {
        "minimum": round(
            float(
                phase42_state_alpha_values.min()
            ),
            6,
        ),
        "median": round(
            float(
                np.median(
                    phase42_state_alpha_values
                )
            ),
            6,
        ),
        "maximum": round(
            float(
                phase42_state_alpha_values.max()
            ),
            6,
        ),
    },
    "unknown_protocol_alpha": 0.0,
    "private_state_rewritten_atomically": True,
    "accepted_phase40_archive_modified": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE42_FORMULA_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_FORMULA"
)
print(
    json.dumps(
        phase42_formula_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_FORMULA"
)

BEGIN SANITIZED_PHASE42_FORMULA
{
  "phase": "phase42_corrected_portable_formula_contract",
  "status": "accepted",
  "corrected_formula": "z42 = z_phase12c_full + alpha_group * clip(0.75*r36_raw + 0.125*r33, -1, 1)",
  "weights": {
    "phase36_raw_residual": 0.75,
    "phase33_logit_residual": 0.125
  },
  "residual_cap": 1.0,
  "uncertainty_exponent": 0.0,
  "operation_order": [
    "compute_phase36_raw_residual",
    "compute_phase33_logit_residual",
    "weighted_residual_sum",
    "symmetric_residual_clip",
    "multiply_by_known_group_alpha",
    "add_to_complete_phase12c_anchor_logit",
    "sigmoid"
  ],
  "removed_incorrect_redundant_value": 0.875,
  "cross_fitted_formula_error": {
    "maximum": 5.551115123125783e-17,
    "mean": 8.151417214575305e-20
  },
  "reference_metrics": {
    "nested": {
      "log_loss": 0.293516,
      "auroc": 0.9448
    },
    "shared_selection": {
      "log_loss": 0.291221,
      "auroc": 0.946032
    },
    "full_fit_diagnostic_only": {
      

In [12]:
# Phase42 Cell 138B
# Audit the accepted Phase40 archive and discover its runtime symbols.
# No extraction and no package modification.

import ast
import hashlib
import zipfile

PHASE42_SOURCE_AUDIT_STARTED = (
    time.perf_counter()
)

PHASE42_ACCEPTED_PHASE40_ZIP = Path(
    "/kaggle/working/"
    "phase40_submission.zip"
)

PHASE42_EXPECTED_PHASE40_SHA256 = (
    "b3420f2ca9e7de7b452765c3f9607c665901a90c63"
    "fed6ee19c9fadbd5f73f12"
)

assert (
    PHASE42_ACCEPTED_PHASE40_ZIP.is_file()
), {
    "message": (
        "Accepted Phase40 archive is missing."
    ),
    "path": str(
        PHASE42_ACCEPTED_PHASE40_ZIP
    ),
}

phase42_phase40_sha256 = hashlib.sha256(
    PHASE42_ACCEPTED_PHASE40_ZIP.read_bytes()
).hexdigest()

assert (
    phase42_phase40_sha256
    == PHASE42_EXPECTED_PHASE40_SHA256
), {
    "message": (
        "Phase40 archive hash differs from "
        "the accepted archive."
    )
}

phase42_python_inventory = []
phase42_json_members = []
phase42_relevant_symbols = []
phase42_keyword_records = []

phase42_relevant_terms = [
    "phase39",
    "phase40",
    "route",
    "residual",
    "predict",
    "inference",
    "initialize",
    "header",
    "prototype",
]

with zipfile.ZipFile(
    PHASE42_ACCEPTED_PHASE40_ZIP,
    "r",
) as archive:
    bad_member = archive.testzip()

    assert bad_member is None, {
        "message": (
            "Phase40 archive integrity "
            "test failed."
        ),
        "member": bad_member,
    }

    phase42_archive_members = [
        member.filename
        for member
        in archive.infolist()
        if not member.is_dir()
    ]

    assert "main.py" in (
        phase42_archive_members
    )

    for member_name in (
        phase42_archive_members
    ):
        if member_name.endswith(".json"):
            phase42_json_members.append(
                member_name
            )

        if not member_name.endswith(".py"):
            continue

        source_bytes = archive.read(
            member_name
        )

        source_text = source_bytes.decode(
            "utf-8"
        )

        try:
            tree = ast.parse(
                source_text,
                filename=member_name,
            )
            syntax_valid = True
        except SyntaxError:
            tree = None
            syntax_valid = False

        symbols = []

        if tree is not None:
            for node in ast.walk(tree):
                if isinstance(
                    node,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                        ast.ClassDef,
                    ),
                ):
                    symbols.append({
                        "name": node.name,
                        "kind": (
                            "class"
                            if isinstance(
                                node,
                                ast.ClassDef,
                            )
                            else "function"
                        ),
                    })

                    lowered = (
                        node.name.lower()
                    )

                    if any(
                        term in lowered
                        for term
                        in phase42_relevant_terms
                    ):
                        phase42_relevant_symbols.append({
                            "file": member_name,
                            "name": node.name,
                            "kind": (
                                "class"
                                if isinstance(
                                    node,
                                    ast.ClassDef,
                                )
                                else "function"
                            ),
                        })

        keyword_hits = {
            term: int(
                source_text.lower().count(
                    term
                )
            )
            for term in (
                phase42_relevant_terms
            )
            if term in source_text.lower()
        }

        if keyword_hits:
            phase42_keyword_records.append({
                "file": member_name,
                "keyword_hits": (
                    keyword_hits
                ),
            })

        phase42_python_inventory.append({
            "file": member_name,
            "line_count": int(
                len(
                    source_text.splitlines()
                )
            ),
            "syntax_valid": bool(
                syntax_valid
            ),
            "symbol_count": int(
                len(symbols)
            ),
        })


assert len(
    phase42_archive_members
) == 89, {
    "message": (
        "Accepted Phase40 archive member "
        "count changed."
    ),
    "observed": len(
        phase42_archive_members
    ),
    "expected": 89,
}

assert all(
    record["syntax_valid"]
    for record
    in phase42_python_inventory
)

phase42_relevant_symbols = sorted(
    phase42_relevant_symbols,
    key=lambda record: (
        record["file"],
        record["name"],
    ),
)

PHASE42_PHASE40_SOURCE_INVENTORY_PRIVATE = {
    "archive": str(
        PHASE42_ACCEPTED_PHASE40_ZIP
    ),
    "members": (
        phase42_archive_members
    ),
    "python_inventory": (
        phase42_python_inventory
    ),
    "json_members": sorted(
        phase42_json_members
    ),
    "relevant_symbols": (
        phase42_relevant_symbols
    ),
    "keyword_records": (
        phase42_keyword_records
    ),
}

phase42_source_inventory_path = Path(
    "/kaggle/working/"
    "phase42_phase40_source_inventory_private.json"
)

phase42_source_inventory_path.write_text(
    json.dumps(
        PHASE42_PHASE40_SOURCE_INVENTORY_PRIVATE,
        indent=2,
    ),
    encoding="utf-8",
)

phase42_source_audit_report = {
    "phase": (
        "phase42_accepted_phase40_"
        "runtime_source_audit"
    ),
    "status": "accepted",
    "archive_name": (
        PHASE42_ACCEPTED_PHASE40_ZIP.name
    ),
    "archive_file_count": int(
        len(
            phase42_archive_members
        )
    ),
    "archive_hash_verified": True,
    "archive_integrity_test_passed": True,
    "root_main_present": True,
    "python_files": (
        phase42_python_inventory
    ),
    "json_members": sorted(
        phase42_json_members
    ),
    "relevant_runtime_symbols": (
        phase42_relevant_symbols
    ),
    "keyword_records": (
        phase42_keyword_records
    ),
    "source_inventory_written": (
        phase42_source_inventory_path.is_file()
    ),
    "archive_extracted": False,
    "accepted_phase40_archive_modified": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE42_SOURCE_AUDIT_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_SOURCE_AUDIT"
)
print(
    json.dumps(
        phase42_source_audit_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_SOURCE_AUDIT"
)

BEGIN SANITIZED_PHASE42_SOURCE_AUDIT
{
  "phase": "phase42_accepted_phase40_runtime_source_audit",
  "status": "accepted",
  "archive_name": "phase40_submission.zip",
  "archive_file_count": 89,
  "archive_hash_verified": true,
  "archive_integrity_test_passed": true,
  "root_main_present": true,
  "python_files": [
    {
      "file": "base_main.py",
      "line_count": 384,
      "syntax_valid": true,
      "symbol_count": 19
    },
    {
      "file": "dinov3_repo/dinov3/__init__.py",
      "line_count": 6,
      "syntax_valid": true,
      "symbol_count": 0
    },
    {
      "file": "dinov3_repo/dinov3/checkpointer/__init__.py",
      "line_count": 18,
      "syntax_valid": true,
      "symbol_count": 0
    },
    {
      "file": "dinov3_repo/dinov3/checkpointer/checkpointer.py",
      "line_count": 352,
      "syntax_valid": true,
      "symbol_count": 17
    },
    {
      "file": "dinov3_repo/dinov3/fsdp/__init__.py",
      "line_count": 0,
      "syntax_valid": true,
      "sy

In [13]:
# Phase42 Cell 138C
# Export only the exact accepted runtime sources and corrected Phase42
# scalar gate state required to construct the integration patch.
#
# No model weights, predictions, labels, case rows, voxel data,
# embeddings, or training caches are included.

import hashlib
import io
import zipfile

PHASE42_HANDOFF_STARTED = (
    time.perf_counter()
)

PHASE42_RUNTIME_SOURCE_HANDOFF = Path(
    "/kaggle/working/"
    "phase42_runtime_source_handoff.zip"
)

assert (
    PHASE42_ACCEPTED_PHASE40_ZIP.is_file()
)

assert phase42_state_path.is_file()

phase42_required_archive_members = [
    "main.py",
    "phase40_runtime.py",
    "phase39_runtime.py",
    "phase30_accepted_main.py",
    "phase40_runtime_contract.json",
    "phase39_runtime_source_manifest.json",
    "phase39_assets/phase39_runtime_state.json",
    "phase39_assets/phase39_asset_manifest.json",
    "phase39_assets/runtime_small_constants.json",
]

phase42_integration_contract = {
    "schema_version": 1,
    "phase": (
        "phase42_runtime_integration_handoff"
    ),
    "accepted_base_archive": (
        "phase40_submission.zip"
    ),
    "accepted_base_archive_sha256": (
        PHASE42_EXPECTED_PHASE40_SHA256
    ),
    "target_formula": (
        "z42 = z_phase12c_full + alpha_group * "
        "clip(0.75*r36_raw + 0.125*r33, -1, 1)"
    ),
    "operation_order": [
        "compute_phase36_raw_residual",
        "compute_phase33_logit_residual",
        "weighted_residual_sum",
        "symmetric_residual_clip",
        "multiply_by_known_group_alpha",
        "add_to_complete_phase12c_anchor_logit",
        "sigmoid",
    ],
    "route_policy": {
        "known_header_prototype": (
            "use exact acquisition group, its "
            "held-out component fold, and its "
            "portable Phase42 alpha"
        ),
        "unknown_header_prototype": (
            "phase12c complete ensemble anchor only"
        ),
        "three_fold_residual_average": False,
    },
    "weights": {
        "phase36_raw_residual": 0.75,
        "phase33_logit_residual": 0.125,
    },
    "residual_cap": 1.0,
    "uncertainty_exponent": 0.0,
    "known_group_count": 15,
    "unknown_protocol_alpha": 0.0,
    "required_runtime_entrypoints": {
        "main": [
            "exact_header_route",
            "predict_phase40_safe",
        ],
        "phase40_runtime": [
            "Phase40Runtime",
            "predict_routed",
            "predict_routed_from_dense",
        ],
        "phase39_runtime": [
            "Phase39Runtime",
            "phase36_predict_residual",
            "predict",
            "predict_from_dense",
        ],
    },
    "logging_policy": (
        "generic lifecycle messages only"
    ),
    "network_allowed": False,
    "accepted_phase40_archive_must_remain_unchanged": True,
}

phase42_handoff_files = {}

with zipfile.ZipFile(
    PHASE42_ACCEPTED_PHASE40_ZIP,
    "r",
) as accepted_archive:
    accepted_names = set(
        accepted_archive.namelist()
    )

    missing_members = [
        member
        for member
        in phase42_required_archive_members
        if member not in accepted_names
    ]

    assert not missing_members, {
        "message": (
            "Accepted Phase40 archive is missing "
            "required runtime sources."
        ),
        "missing": missing_members,
    }

    for member in (
        phase42_required_archive_members
    ):
        phase42_handoff_files[
            f"phase40/{member}"
        ] = accepted_archive.read(
            member
        )

phase42_corrected_state_bytes = (
    phase42_state_path.read_bytes()
)

phase42_reloaded_state = json.loads(
    phase42_corrected_state_bytes.decode(
        "utf-8"
    )
)

assert (
    phase42_reloaded_state[
        "deployment_formula"
    ]
    == phase42_integration_contract[
        "target_formula"
    ]
)

assert (
    float(
        phase42_reloaded_state[
            "phase36_raw_residual_weight"
        ]
    )
    == 0.75
)

assert (
    float(
        phase42_reloaded_state[
            "phase33_logit_residual_weight"
        ]
    )
    == 0.125
)

assert (
    float(
        phase42_reloaded_state[
            "residual_cap"
        ]
    )
    == 1.0
)

assert (
    "phase36_effective_weight"
    not in phase42_reloaded_state
)

phase42_handoff_files[
    "phase42/phase42_hierarchical_gate.json"
] = phase42_corrected_state_bytes

phase42_handoff_files[
    "phase42/phase42_integration_contract.json"
] = json.dumps(
    phase42_integration_contract,
    indent=2,
    sort_keys=True,
).encode("utf-8")

phase42_source_inventory = []

for relative_name, payload in sorted(
    phase42_handoff_files.items()
):
    phase42_source_inventory.append({
        "file": relative_name,
        "size_bytes": int(
            len(payload)
        ),
        "sha256": hashlib.sha256(
            payload
        ).hexdigest(),
    })

phase42_handoff_files[
    "source_inventory.json"
] = json.dumps(
    {
        "schema_version": 1,
        "file_count_excluding_inventory": int(
            len(
                phase42_source_inventory
            )
        ),
        "files": phase42_source_inventory,
    },
    indent=2,
    sort_keys=True,
).encode("utf-8")

phase42_allowed_suffixes = {
    ".py",
    ".json",
}

for relative_name in (
    phase42_handoff_files
):
    suffix = Path(
        relative_name
    ).suffix.lower()

    assert suffix in (
        phase42_allowed_suffixes
    ), {
        "message": (
            "Unexpected handoff file type."
        ),
        "file": relative_name,
    }

# Deterministic ZIP metadata.
phase42_zip_timestamp = (
    2026,
    8,
    22,
    0,
    0,
    0,
)

with zipfile.ZipFile(
    PHASE42_RUNTIME_SOURCE_HANDOFF,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as handoff_archive:
    for relative_name, payload in sorted(
        phase42_handoff_files.items()
    ):
        information = zipfile.ZipInfo(
            filename=relative_name,
            date_time=(
                phase42_zip_timestamp
            ),
        )

        information.compress_type = (
            zipfile.ZIP_DEFLATED
        )

        information.external_attr = (
            0o100644 << 16
        )

        handoff_archive.writestr(
            information,
            payload,
        )

with zipfile.ZipFile(
    PHASE42_RUNTIME_SOURCE_HANDOFF,
    "r",
) as verification_archive:
    assert (
        verification_archive.testzip()
        is None
    )

    phase42_written_members = [
        member.filename
        for member
        in verification_archive.infolist()
        if not member.is_dir()
    ]

    assert set(
        phase42_written_members
    ) == set(
        phase42_handoff_files
    )

phase42_handoff_size_kb = (
    PHASE42_RUNTIME_SOURCE_HANDOFF
    .stat()
    .st_size
    / 1024.0
)

phase42_handoff_report = {
    "phase": (
        "phase42_exact_runtime_source_handoff"
    ),
    "status": "complete",
    "zip_name": (
        PHASE42_RUNTIME_SOURCE_HANDOFF.name
    ),
    "zip_size_kb": round(
        float(
            phase42_handoff_size_kb
        ),
        2,
    ),
    "file_count": int(
        len(
            phase42_handoff_files
        )
    ),
    "included_files": sorted(
        phase42_handoff_files
    ),
    "target_formula": (
        phase42_integration_contract[
            "target_formula"
        ]
    ),
    "accepted_phase40_hash_verified": True,
    "archive_integrity_test_passed": True,
    "contains_model_weights": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_oof_arrays": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
    "accepted_phase40_archive_modified": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "model_hashes_displayed": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - PHASE42_HANDOFF_STARTED,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_SOURCE_HANDOFF"
)
print(
    json.dumps(
        phase42_handoff_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_SOURCE_HANDOFF"
)
print(
    PHASE42_RUNTIME_SOURCE_HANDOFF
)

BEGIN SANITIZED_PHASE42_SOURCE_HANDOFF
{
  "phase": "phase42_exact_runtime_source_handoff",
  "status": "complete",
  "zip_name": "phase42_runtime_source_handoff.zip",
  "zip_size_kb": 28.52,
  "file_count": 12,
  "included_files": [
    "phase40/main.py",
    "phase40/phase30_accepted_main.py",
    "phase40/phase39_assets/phase39_asset_manifest.json",
    "phase40/phase39_assets/phase39_runtime_state.json",
    "phase40/phase39_assets/runtime_small_constants.json",
    "phase40/phase39_runtime.py",
    "phase40/phase39_runtime_source_manifest.json",
    "phase40/phase40_runtime.py",
    "phase40/phase40_runtime_contract.json",
    "phase42/phase42_hierarchical_gate.json",
    "phase42/phase42_integration_contract.json",
    "source_inventory.json"
  ],
  "target_formula": "z42 = z_phase12c_full + alpha_group * clip(0.75*r36_raw + 0.125*r33, -1, 1)",
  "accepted_phase40_hash_verified": true,
  "archive_integrity_test_passed": true,
  "contains_model_weights": false,
  "contains_labels"

In [14]:
# Phase42 Cell 138D
# Stage the corrected hierarchical gate over the accepted Phase40 runtime.
# This cell does not read training, smoke, or challenge voxel arrays.

from pathlib import Path
import ast
import hashlib
import json
import os
import shutil
import tempfile
import time
import zipfile

phase42_stage_started = time.perf_counter()

PHASE42_ACCEPTED_PHASE40_ARCHIVE = Path(
    "/kaggle/working/phase40_submission.zip"
)
PHASE42_SOURCE_HANDOFF_PATH = Path(
    "/kaggle/working/phase42_runtime_source_handoff.zip"
)
PHASE42_STAGE_DIRECTORY = Path(
    "/kaggle/working/phase42_experimental_submission"
)

PHASE42_EXPECTED_PHASE40_SHA256 = (
    "b3420f2ca9e7de7b452765c3f9607c665901a90c63fed6ee19c9fadbd5f73f12"
)
PHASE42_EXPECTED_PHASE40_FILE_COUNT = 89
PHASE42_EXPECTED_STAGE_FILE_COUNT = 94


def phase42_file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def phase42_json_bytes(value):
    return (
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
        )
        + "\n"
    ).encode("utf-8")


def phase42_safe_archive_members(archive):
    members = archive.namelist()

    assert members, "Phase40 archive is empty."

    for member in members:
        path = Path(member)

        assert not path.is_absolute(), {
            "message": "Absolute archive member rejected.",
            "member": member,
        }
        assert ".." not in path.parts, {
            "message": "Parent traversal archive member rejected.",
            "member": member,
        }

    return members


def phase42_replace_exact(
    source,
    old,
    new,
    expected_count=1,
):
    observed_count = source.count(old)

    assert observed_count == expected_count, {
        "message": "Phase42 exact source replacement failed.",
        "expected_count": expected_count,
        "observed_count": observed_count,
        "old_prefix": old[:120],
    }

    return source.replace(old, new)


assert PHASE42_ACCEPTED_PHASE40_ARCHIVE.is_file()
assert PHASE42_SOURCE_HANDOFF_PATH.is_file()

phase40_hash_before = phase42_file_sha256(
    PHASE42_ACCEPTED_PHASE40_ARCHIVE
)

assert (
    phase40_hash_before
    == PHASE42_EXPECTED_PHASE40_SHA256
), {
    "message": "Accepted Phase40 archive hash changed.",
    "observed": phase40_hash_before,
    "expected": PHASE42_EXPECTED_PHASE40_SHA256,
}

with zipfile.ZipFile(
    PHASE42_ACCEPTED_PHASE40_ARCHIVE,
    mode="r",
) as archive:
    phase40_members = phase42_safe_archive_members(
        archive
    )

    assert archive.testzip() is None

    phase40_files = [
        member
        for member in phase40_members
        if not member.endswith("/")
    ]

    assert len(phase40_files) == (
        PHASE42_EXPECTED_PHASE40_FILE_COUNT
    ), {
        "message": "Unexpected accepted Phase40 file count.",
        "observed": len(phase40_files),
        "expected": PHASE42_EXPECTED_PHASE40_FILE_COUNT,
    }

with zipfile.ZipFile(
    PHASE42_SOURCE_HANDOFF_PATH,
    mode="r",
) as handoff_archive:
    assert handoff_archive.testzip() is None

    phase42_state_bytes = handoff_archive.read(
        "phase42/phase42_hierarchical_gate.json"
    )
    phase42_integration_bytes = handoff_archive.read(
        "phase42/phase42_integration_contract.json"
    )

phase42_state = json.loads(
    phase42_state_bytes.decode("utf-8")
)
phase42_integration_contract = json.loads(
    phase42_integration_bytes.decode("utf-8")
)

PHASE42_OPERATION_ORDER = [
    "compute_phase36_raw_residual",
    "compute_phase33_logit_residual",
    "weighted_residual_sum",
    "symmetric_residual_clip",
    "multiply_by_known_group_alpha",
    "add_to_complete_phase12c_anchor_logit",
    "sigmoid",
]

assert phase42_state["schema_version"] == 1
assert phase42_state["operation_order"] == (
    PHASE42_OPERATION_ORDER
)
assert (
    float(
        phase42_state[
            "phase36_raw_residual_weight"
        ]
    )
    == 0.75
)
assert (
    float(
        phase42_state[
            "phase33_logit_residual_weight"
        ]
    )
    == 0.125
)
assert float(phase42_state["residual_cap"]) == 1.0
assert (
    float(
        phase42_state[
            "uncertainty_exponent"
        ]
    )
    == 0.0
)
assert (
    float(
        phase42_state[
            "unknown_protocol_alpha"
        ]
    )
    == 0.0
)
assert set(
    phase42_state["known_group_alpha"]
) == {
    str(group)
    for group in range(15)
}

PHASE42_RUNTIME_SOURCE = r'''from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np

from phase39_runtime import (
    PROBABILITY_EPSILON,
    probability_logit,
    probability_sigmoid,
)
from phase40_runtime import Phase40Runtime


ROOT = Path(__file__).resolve().parent
PHASE42_ASSET_DIRECTORY = ROOT / "phase42_assets"

EXPECTED_OPERATION_ORDER = [
    "compute_phase36_raw_residual",
    "compute_phase33_logit_residual",
    "weighted_residual_sum",
    "symmetric_residual_clip",
    "multiply_by_known_group_alpha",
    "add_to_complete_phase12c_anchor_logit",
    "sigmoid",
]


def _phase42_read_json(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )


def _phase42_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def load_phase42_state():
    manifest_path = (
        PHASE42_ASSET_DIRECTORY
        / "phase42_asset_manifest.json"
    )
    state_path = (
        PHASE42_ASSET_DIRECTORY
        / "phase42_hierarchical_gate.json"
    )

    manifest = _phase42_read_json(manifest_path)

    if manifest.get("schema_version") != 1:
        raise RuntimeError(
            "Unsupported Phase42 asset manifest"
        )

    if (
        manifest.get("phase")
        != "phase42_portable_assets"
    ):
        raise RuntimeError(
            "Unexpected Phase42 asset manifest"
        )

    expected_hash = manifest.get(
        "assets",
        {},
    ).get(
        "phase42_assets/"
        "phase42_hierarchical_gate.json"
    )

    if not isinstance(expected_hash, str):
        raise RuntimeError(
            "Missing Phase42 state integrity record"
        )

    if _phase42_sha256(state_path) != expected_hash:
        raise RuntimeError(
            "Phase42 state integrity check failed"
        )

    state = _phase42_read_json(state_path)

    if state.get("schema_version") != 1:
        raise RuntimeError(
            "Unsupported Phase42 runtime state"
        )

    if (
        state.get("phase")
        != "phase42_hierarchical_known_protocol_gate"
    ):
        raise RuntimeError(
            "Unexpected Phase42 runtime state"
        )

    if (
        float(
            state.get(
                "phase36_raw_residual_weight",
                np.nan,
            )
        )
        != 0.75
    ):
        raise RuntimeError(
            "Unexpected Phase42 Phase36 weight"
        )

    if (
        float(
            state.get(
                "phase33_logit_residual_weight",
                np.nan,
            )
        )
        != 0.125
    ):
        raise RuntimeError(
            "Unexpected Phase42 Phase33 weight"
        )

    if (
        float(
            state.get(
                "residual_cap",
                np.nan,
            )
        )
        != 1.0
    ):
        raise RuntimeError(
            "Unexpected Phase42 residual cap"
        )

    if (
        float(
            state.get(
                "uncertainty_exponent",
                np.nan,
            )
        )
        != 0.0
    ):
        raise RuntimeError(
            "Unexpected Phase42 uncertainty exponent"
        )

    if (
        float(
            state.get(
                "unknown_protocol_alpha",
                np.nan,
            )
        )
        != 0.0
    ):
        raise RuntimeError(
            "Unexpected unknown-protocol alpha"
        )

    if (
        state.get("operation_order")
        != EXPECTED_OPERATION_ORDER
    ):
        raise RuntimeError(
            "Unexpected Phase42 operation order"
        )

    alpha_map = state.get("known_group_alpha")
    expected_keys = {
        str(group)
        for group in range(15)
    }

    if (
        not isinstance(alpha_map, dict)
        or set(alpha_map) != expected_keys
    ):
        raise RuntimeError(
            "Incomplete Phase42 group-alpha state"
        )

    alpha_values = np.asarray(
        [
            float(alpha_map[str(group)])
            for group in range(15)
        ],
        dtype=np.float64,
    )

    if not np.isfinite(alpha_values).all():
        raise RuntimeError(
            "Non-finite Phase42 group alpha"
        )

    if np.any(
        (alpha_values < 0.0)
        | (alpha_values > 1.0)
    ):
        raise RuntimeError(
            "Phase42 group alpha outside [0, 1]"
        )

    state["known_group_alpha_array"] = (
        alpha_values
    )

    return state


def phase42_compose_probability(
    baseline_probability,
    raw_phase36_residual,
    phase33_residual,
    route_groups,
    route_exact,
    state,
):
    baseline_probability = np.clip(
        np.asarray(
            baseline_probability,
            dtype=np.float64,
        ),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )
    raw_phase36_residual = np.asarray(
        raw_phase36_residual,
        dtype=np.float64,
    )
    phase33_residual = np.asarray(
        phase33_residual,
        dtype=np.float64,
    )
    route_groups = np.asarray(
        route_groups,
        dtype=np.int64,
    )
    route_exact = np.asarray(
        route_exact,
        dtype=bool,
    )

    expected_shape = baseline_probability.shape

    for name, array in (
        (
            "raw Phase36 residual",
            raw_phase36_residual,
        ),
        (
            "Phase33 residual",
            phase33_residual,
        ),
        (
            "route groups",
            route_groups,
        ),
        (
            "route mask",
            route_exact,
        ),
    ):
        if array.shape != expected_shape:
            raise RuntimeError(
                f"Phase42 {name} shape mismatch"
            )

    if baseline_probability.ndim != 1:
        raise RuntimeError(
            "Phase42 expects one-dimensional batches"
        )

    if not np.isfinite(
        raw_phase36_residual
    ).all():
        raise RuntimeError(
            "Non-finite Phase36 residual"
        )

    if not np.isfinite(
        phase33_residual
    ).all():
        raise RuntimeError(
            "Non-finite Phase33 residual"
        )

    if np.any(
        route_exact
        & (
            (route_groups < 0)
            | (route_groups >= 15)
        )
    ):
        raise RuntimeError(
            "Invalid exact Phase42 group route"
        )

    group_alpha = np.full(
        baseline_probability.shape,
        float(
            state["unknown_protocol_alpha"]
        ),
        dtype=np.float64,
    )

    known = np.flatnonzero(route_exact)

    if known.size:
        group_alpha[known] = state[
            "known_group_alpha_array"
        ][route_groups[known]]

    combined_uncapped = (
        float(
            state[
                "phase36_raw_residual_weight"
            ]
        )
        * raw_phase36_residual
        + float(
            state[
                "phase33_logit_residual_weight"
            ]
        )
        * phase33_residual
    )

    combined_bounded = np.clip(
        combined_uncapped,
        -float(state["residual_cap"]),
        float(state["residual_cap"]),
    )

    # The protocol alpha is deliberately applied
    # after the symmetric residual clip.
    applied_update = (
        group_alpha
        * combined_bounded
    )

    baseline_logit = probability_logit(
        baseline_probability
    )

    probability = np.clip(
        probability_sigmoid(
            baseline_logit
            + applied_update
        ),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )

    if not np.isfinite(probability).all():
        raise RuntimeError(
            "Non-finite Phase42 probability"
        )

    return probability, {
        "combined_uncapped": (
            combined_uncapped
        ),
        "combined_bounded": (
            combined_bounded
        ),
        "group_alpha": group_alpha,
        "applied_update": applied_update,
    }


class Phase42Runtime(Phase40Runtime):
    """
    Exact-routed Phase33/36 components with
    a hierarchically shrunk Phase42 group gate.
    """

    def __init__(self, device):
        super().__init__(device)

        self.phase42_state = (
            load_phase42_state()
        )

    def predict_routed_from_dense(
        self,
        dense_features,
        radiomics,
        geometry_sequence,
        baseline_probability,
        route_groups,
        route_folds,
        route_exact,
    ):
        # Reuse the parity-tested Phase40 implementation
        # only to compute the two raw routed components.
        # Its old composed probability is discarded.
        _, components = (
            super().predict_routed_from_dense(
                dense_features=(
                    dense_features
                ),
                radiomics=radiomics,
                geometry_sequence=(
                    geometry_sequence
                ),
                baseline_probability=(
                    baseline_probability
                ),
                route_folds=route_folds,
                route_exact=route_exact,
            )
        )

        probability, phase42 = (
            phase42_compose_probability(
                baseline_probability=(
                    baseline_probability
                ),
                raw_phase36_residual=(
                    components[
                        "raw_phase36_residual"
                    ]
                ),
                phase33_residual=(
                    components[
                        "phase33_residual"
                    ]
                ),
                route_groups=route_groups,
                route_exact=route_exact,
                state=self.phase42_state,
            )
        )

        diagnostics = dict(components)
        diagnostics.update(phase42)
        diagnostics["route_groups"] = np.asarray(
            route_groups,
            dtype=np.int64,
        )

        return probability, diagnostics

    def predict_routed(
        self,
        highres_volumes,
        baseline_probability,
        backbone,
        route_groups,
        route_folds,
        route_exact,
    ):
        # Preserve the exact Phase31/32 float16
        # training-cache boundary.
        highres = np.asarray(
            highres_volumes,
            dtype=np.float32,
        ).astype(
            np.float16
        ).astype(
            np.float32
        )

        radiomics, geometry = (
            self.extract_geometry(highres)
        )
        dense = self.extract_dense_tokens(
            highres,
            backbone,
        )

        return self.predict_routed_from_dense(
            dense_features=dense,
            radiomics=radiomics,
            geometry_sequence=geometry,
            baseline_probability=(
                baseline_probability
            ),
            route_groups=route_groups,
            route_folds=route_folds,
            route_exact=route_exact,
        )
'''

# Parse before writing anything.
ast.parse(
    PHASE42_RUNTIME_SOURCE,
    filename="phase42_runtime.py",
)

temporary_stage = Path(
    tempfile.mkdtemp(
        prefix="phase42_stage_",
        dir="/kaggle/working",
    )
)

try:
    with zipfile.ZipFile(
        PHASE42_ACCEPTED_PHASE40_ARCHIVE,
        mode="r",
    ) as archive:
        phase42_safe_archive_members(archive)
        archive.extractall(temporary_stage)

    phase40_main_path = (
        temporary_stage / "main.py"
    )
    phase40_main_source = (
        phase40_main_path.read_text(
            encoding="utf-8"
        )
    )

    phase42_main_source = phase40_main_source

    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        "from phase40_runtime import "
        "Phase40Runtime",
        "from phase42_runtime import "
        "Phase42Runtime",
    )
    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        "predict_phase40_safe",
        "predict_phase42_safe",
        expected_count=2,
    )
    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        "Phase40Runtime(device)",
        "Phase42Runtime(device)",
    )

    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        """    baseline_probability,
    route_folds,
    route_exact,
):
    baseline_probability = np.asarray(""",
        """    baseline_probability,
    route_groups,
    route_folds,
    route_exact,
):
    baseline_probability = np.asarray(""",
    )

    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        """    route_folds = np.asarray(route_folds, dtype=np.int64)
    route_exact = np.asarray(route_exact, dtype=bool)""",
        """    route_groups = np.asarray(route_groups, dtype=np.int64)
    route_folds = np.asarray(route_folds, dtype=np.int64)
    route_exact = np.asarray(route_exact, dtype=bool)""",
    )

    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        """            backbone,
            route_folds,
            route_exact,
        )[0]""",
        """            backbone,
            route_groups,
            route_folds,
            route_exact,
        )[0]""",
    )

    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        """                        backbone,
                        route_folds[index:index + 1],
                        route_exact[index:index + 1],
                    )[0][0]""",
        """                        backbone,
                        route_groups[index:index + 1],
                        route_folds[index:index + 1],
                        route_exact[index:index + 1],
                    )[0][0]""",
    )

    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        """            route_folds = np.asarray(
                [prepared[index][2] for index in valid_local],
                dtype=np.int64,
            )""",
        """            route_groups = np.asarray(
                [prepared[index][1] for index in valid_local],
                dtype=np.int64,
            )
            route_folds = np.asarray(
                [prepared[index][2] for index in valid_local],
                dtype=np.int64,
            )""",
    )

    phase42_main_source = phase42_replace_exact(
        phase42_main_source,
        """                    full_baseline,
                    route_folds,
                    route_exact,
                )""",
        """                    full_baseline,
                    route_groups,
                    route_folds,
                    route_exact,
                )""",
    )

    assert "Phase40Runtime" not in (
        phase42_main_source
    )
    assert "predict_phase40_safe" not in (
        phase42_main_source
    )
    assert (
        phase42_main_source.count(
            "route_groups"
        )
        == 7
    )

    ast.parse(
        phase42_main_source,
        filename="main.py",
    )

    phase42_asset_directory = (
        temporary_stage / "phase42_assets"
    )
    phase42_asset_directory.mkdir(
        parents=True,
        exist_ok=False,
    )

    phase42_state_path = (
        phase42_asset_directory
        / "phase42_hierarchical_gate.json"
    )
    phase42_state_path.write_bytes(
        phase42_state_bytes
    )

    phase42_state_hash = (
        phase42_file_sha256(
            phase42_state_path
        )
    )

    phase42_asset_manifest = {
        "schema_version": 1,
        "phase": "phase42_portable_assets",
        "assets": {
            (
                "phase42_assets/"
                "phase42_hierarchical_gate.json"
            ): phase42_state_hash,
        },
        "contains_labels": False,
        "contains_predictions": False,
        "contains_case_rows": False,
        "contains_voxel_data": False,
        "contains_embeddings": False,
    }

    phase42_asset_manifest_path = (
        phase42_asset_directory
        / "phase42_asset_manifest.json"
    )
    phase42_asset_manifest_path.write_bytes(
        phase42_json_bytes(
            phase42_asset_manifest
        )
    )

    phase42_runtime_path = (
        temporary_stage
        / "phase42_runtime.py"
    )
    phase42_runtime_path.write_text(
        PHASE42_RUNTIME_SOURCE,
        encoding="utf-8",
    )

    phase40_main_path.write_text(
        phase42_main_source,
        encoding="utf-8",
    )

    phase42_runtime_contract = dict(
        phase42_integration_contract
    )
    phase42_runtime_contract.update({
        "schema_version": 1,
        "phase": (
            "phase42_hierarchical_"
            "known_protocol_runtime"
        ),
        "runtime_entrypoint": (
            "phase42_runtime.py"
        ),
        "accepted_phase40_archive_sha256": (
            PHASE42_EXPECTED_PHASE40_SHA256
        ),
        "component_runtime": (
            "phase40_runtime.py"
        ),
        "unknown_protocol_computation": (
            "anchor_probability_retained"
        ),
        "test_batch_statistics_used": False,
    })

    phase42_runtime_contract_path = (
        temporary_stage
        / "phase42_runtime_contract.json"
    )
    phase42_runtime_contract_path.write_bytes(
        phase42_json_bytes(
            phase42_runtime_contract
        )
    )

    phase42_source_manifest = {
        "schema_version": 1,
        "phase": (
            "phase42_offline_runtime_source"
        ),
        "files": {
            "main.py": phase42_file_sha256(
                phase40_main_path
            ),
            "phase42_runtime.py": (
                phase42_file_sha256(
                    phase42_runtime_path
                )
            ),
            (
                "phase42_assets/"
                "phase42_hierarchical_gate.json"
            ): phase42_state_hash,
            (
                "phase42_assets/"
                "phase42_asset_manifest.json"
            ): phase42_file_sha256(
                phase42_asset_manifest_path
            ),
            "phase42_runtime_contract.json": (
                phase42_file_sha256(
                    phase42_runtime_contract_path
                )
            ),
        },
        "deployment_formula": (
            "z42 = z_phase12c_full + "
            "alpha_group * clip("
            "0.75*r36_raw + 0.125*r33, "
            "-1, 1)"
        ),
        "contains_labels": False,
        "contains_predictions": False,
        "contains_case_rows": False,
        "contains_voxel_data": False,
        "contains_embeddings": False,
    }

    (
        temporary_stage
        / "phase42_runtime_source_manifest.json"
    ).write_bytes(
        phase42_json_bytes(
            phase42_source_manifest
        )
    )

    stage_python_files = sorted(
        temporary_stage.rglob("*.py")
    )

    for path in stage_python_files:
        ast.parse(
            path.read_text(encoding="utf-8"),
            filename=str(path),
        )

    stage_files = sorted(
        path
        for path in temporary_stage.rglob("*")
        if path.is_file()
    )

    assert len(stage_files) == (
        PHASE42_EXPECTED_STAGE_FILE_COUNT
    ), {
        "message": (
            "Unexpected Phase42 staged file count."
        ),
        "observed": len(stage_files),
        "expected": (
            PHASE42_EXPECTED_STAGE_FILE_COUNT
        ),
    }

    if PHASE42_STAGE_DIRECTORY.exists():
        shutil.rmtree(
            PHASE42_STAGE_DIRECTORY
        )

    os.replace(
        temporary_stage,
        PHASE42_STAGE_DIRECTORY,
    )

except Exception:
    if temporary_stage.exists():
        shutil.rmtree(
            temporary_stage,
            ignore_errors=True,
        )
    raise

phase40_hash_after = phase42_file_sha256(
    PHASE42_ACCEPTED_PHASE40_ARCHIVE
)

assert phase40_hash_after == phase40_hash_before

phase42_stage_files = sorted(
    path
    for path in PHASE42_STAGE_DIRECTORY.rglob("*")
    if path.is_file()
)

phase42_stage_size_mb = (
    sum(
        path.stat().st_size
        for path in phase42_stage_files
    )
    / (1024 ** 2)
)

phase42_stage_report = {
    "phase": (
        "phase42_hierarchical_runtime_staging"
    ),
    "status": "accepted",
    "base_archive": (
        PHASE42_ACCEPTED_PHASE40_ARCHIVE.name
    ),
    "staging_directory": (
        PHASE42_STAGE_DIRECTORY.name
    ),
    "package_file_count": len(
        phase42_stage_files
    ),
    "package_size_mb": round(
        phase42_stage_size_mb,
        6,
    ),
    "new_files": [
        "phase42_runtime.py",
        "phase42_runtime_contract.json",
        "phase42_runtime_source_manifest.json",
        (
            "phase42_assets/"
            "phase42_asset_manifest.json"
        ),
        (
            "phase42_assets/"
            "phase42_hierarchical_gate.json"
        ),
    ],
    "runtime_source_line_count": len(
        PHASE42_RUNTIME_SOURCE.splitlines()
    ),
    "main_source_line_count": len(
        phase42_main_source.splitlines()
    ),
    "known_group_count": len(
        phase42_state["known_group_alpha"]
    ),
    "formula": {
        "phase36_raw_residual_weight": 0.75,
        "phase33_logit_residual_weight": 0.125,
        "residual_cap": 1.0,
        "uncertainty_exponent": 0.0,
    },
    "operation_order": (
        PHASE42_OPERATION_ORDER
    ),
    "unknown_protocol_alpha": 0.0,
    "phase40_component_runtime_preserved": True,
    "phase40_archive_hash_verified": True,
    "accepted_phase40_archive_modified": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
    "contains_embeddings": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase42_stage_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_STAGING"
)
print(
    json.dumps(
        phase42_stage_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_STAGING"
)
print(PHASE42_STAGE_DIRECTORY)

BEGIN SANITIZED_PHASE42_STAGING
{
  "phase": "phase42_hierarchical_runtime_staging",
  "status": "accepted",
  "base_archive": "phase40_submission.zip",
  "staging_directory": "phase42_experimental_submission",
  "package_file_count": 94,
  "package_size_mb": 193.316714,
  "new_files": [
    "phase42_runtime.py",
    "phase42_runtime_contract.json",
    "phase42_runtime_source_manifest.json",
    "phase42_assets/phase42_asset_manifest.json",
    "phase42_assets/phase42_hierarchical_gate.json"
  ],
  "runtime_source_line_count": 495,
  "main_source_line_count": 276,
  "known_group_count": 15,
  "formula": {
    "phase36_raw_residual_weight": 0.75,
    "phase33_logit_residual_weight": 0.125,
    "residual_cap": 1.0,
    "uncertainty_exponent": 0.0
  },
  "operation_order": [
    "compute_phase36_raw_residual",
    "compute_phase33_logit_residual",
    "weighted_residual_sum",
    "symmetric_residual_clip",
    "multiply_by_known_group_alpha",
    "add_to_complete_phase12c_anchor_logit",


In [15]:
# Phase42 Cell 138E
# Audit the staged source in an isolated subprocess.
# No model initialization and no voxel-array access.

from pathlib import Path
import ast
import hashlib
import inspect
import json
import os
import subprocess
import sys
import time

phase42_source_audit_started = (
    time.perf_counter()
)

PHASE42_STAGE_DIRECTORY = Path(
    "/kaggle/working/phase42_experimental_submission"
)
PHASE42_ACCEPTED_PHASE40_ARCHIVE = Path(
    "/kaggle/working/phase40_submission.zip"
)
PHASE42_EXPECTED_PHASE40_SHA256 = (
    "b3420f2ca9e7de7b452765c3f9607c665901a90c63fed6ee19c9fadbd5f73f12"
)


def phase42_audit_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


assert PHASE42_STAGE_DIRECTORY.is_dir()
assert (
    phase42_audit_sha256(
        PHASE42_ACCEPTED_PHASE40_ARCHIVE
    )
    == PHASE42_EXPECTED_PHASE40_SHA256
)

required_paths = [
    PHASE42_STAGE_DIRECTORY / "main.py",
    (
        PHASE42_STAGE_DIRECTORY
        / "phase39_runtime.py"
    ),
    (
        PHASE42_STAGE_DIRECTORY
        / "phase40_runtime.py"
    ),
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_runtime.py"
    ),
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_runtime_contract.json"
    ),
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_runtime_source_manifest.json"
    ),
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_assets"
        / "phase42_asset_manifest.json"
    ),
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_assets"
        / "phase42_hierarchical_gate.json"
    ),
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

assert not missing_paths, {
    "message": "Phase42 staged files missing.",
    "missing": missing_paths,
}

for path in (
    PHASE42_STAGE_DIRECTORY / "main.py",
    PHASE42_STAGE_DIRECTORY
    / "phase42_runtime.py",
    PHASE42_STAGE_DIRECTORY
    / "phase40_runtime.py",
    PHASE42_STAGE_DIRECTORY
    / "phase39_runtime.py",
):
    ast.parse(
        path.read_text(encoding="utf-8"),
        filename=str(path),
    )

phase42_subprocess_source = r'''
import inspect
import json
from pathlib import Path

import numpy as np

import phase42_runtime as runtime


state = runtime.load_phase42_state()

baseline = np.asarray(
    [0.20, 0.35, 0.80, 0.40],
    dtype=np.float64,
)
raw36 = np.asarray(
    [4.0, 4.0, -2.0, 5.0],
    dtype=np.float64,
)
residual33 = np.asarray(
    [0.0, 2.0, 8.0, -4.0],
    dtype=np.float64,
)
groups = np.asarray(
    [0, 2, 10, -1],
    dtype=np.int64,
)
exact = np.asarray(
    [True, True, True, False],
    dtype=bool,
)

probability, diagnostics = (
    runtime.phase42_compose_probability(
        baseline_probability=baseline,
        raw_phase36_residual=raw36,
        phase33_residual=residual33,
        route_groups=groups,
        route_exact=exact,
        state=state,
    )
)

alpha_array = state[
    "known_group_alpha_array"
]
expected_alpha = np.asarray(
    [
        alpha_array[0],
        alpha_array[2],
        alpha_array[10],
        0.0,
    ],
    dtype=np.float64,
)

expected_uncapped = (
    0.75 * raw36
    + 0.125 * residual33
)
expected_bounded = np.clip(
    expected_uncapped,
    -1.0,
    1.0,
)
expected_update = (
    expected_alpha
    * expected_bounded
)
baseline_logit = np.log(
    baseline / (1.0 - baseline)
)
expected_probability = (
    1.0
    / (
        1.0
        + np.exp(
            -(
                baseline_logit
                + expected_update
            )
        )
    )
)

formula_error = float(
    np.max(
        np.abs(
            probability
            - expected_probability
        )
    )
)
uncapped_error = float(
    np.max(
        np.abs(
            diagnostics[
                "combined_uncapped"
            ]
            - expected_uncapped
        )
    )
)
bounded_error = float(
    np.max(
        np.abs(
            diagnostics[
                "combined_bounded"
            ]
            - expected_bounded
        )
    )
)
alpha_error = float(
    np.max(
        np.abs(
            diagnostics["group_alpha"]
            - expected_alpha
        )
    )
)
update_error = float(
    np.max(
        np.abs(
            diagnostics["applied_update"]
            - expected_update
        )
    )
)
unknown_anchor_error = float(
    abs(probability[3] - baseline[3])
)

# Group 2 deliberately has an uncapped update > 1.
# This distinguishes:
# alpha * clip(residual)  [correct]
# clip(alpha * residual)  [incorrect]
group2_correct_update = float(
    expected_alpha[1]
    * expected_bounded[1]
)
group2_wrong_update = float(
    np.clip(
        expected_alpha[1]
        * expected_uncapped[1],
        -1.0,
        1.0,
    )
)
cap_order_separation = float(
    abs(
        group2_correct_update
        - group2_wrong_update
    )
)

assert formula_error <= 1e-15
assert uncapped_error <= 1e-15
assert bounded_error <= 1e-15
assert alpha_error <= 1e-15
assert update_error <= 1e-15
assert unknown_anchor_error <= 1e-15
assert cap_order_separation >= 0.20

runtime_signature = str(
    inspect.signature(
        runtime.Phase42Runtime.predict_routed
    )
)
dense_signature = str(
    inspect.signature(
        runtime.Phase42Runtime
        .predict_routed_from_dense
    )
)

assert "route_groups" in runtime_signature
assert "route_folds" in runtime_signature
assert "route_exact" in runtime_signature
assert "route_groups" in dense_signature

print(json.dumps({
    "module_path": str(
        Path(runtime.__file__).resolve()
    ),
    "formula_error": formula_error,
    "uncapped_error": uncapped_error,
    "bounded_error": bounded_error,
    "alpha_error": alpha_error,
    "update_error": update_error,
    "unknown_anchor_error": (
        unknown_anchor_error
    ),
    "cap_order_separation": (
        cap_order_separation
    ),
    "known_group_count": int(
        len(alpha_array)
    ),
    "group_alpha_minimum": float(
        np.min(alpha_array)
    ),
    "group_alpha_median": float(
        np.median(alpha_array)
    ),
    "group_alpha_maximum": float(
        np.max(alpha_array)
    ),
    "runtime_signature": runtime_signature,
    "dense_signature": dense_signature,
    "all_probabilities_finite": bool(
        np.isfinite(probability).all()
    ),
    "all_probabilities_in_contract": bool(
        np.all(
            (probability >= 1e-5)
            & (probability <= 1.0 - 1e-5)
        )
    ),
}, sort_keys=True))
'''

phase42_subprocess_environment = dict(
    os.environ
)
phase42_subprocess_environment.update({
    "PYTHONPATH": str(
        PHASE42_STAGE_DIRECTORY
    ),
    "NO_PROXY": "*",
    "no_proxy": "*",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "TORCH_HOME": (
        "/kaggle/working/"
        "phase42_source_audit_torch_cache"
    ),
})

phase42_subprocess_result = subprocess.run(
    [
        sys.executable,
        "-c",
        phase42_subprocess_source,
    ],
    cwd=str(PHASE42_STAGE_DIRECTORY),
    env=phase42_subprocess_environment,
    capture_output=True,
    text=True,
    timeout=120,
    check=False,
)

assert phase42_subprocess_result.returncode == 0, {
    "message": (
        "Phase42 isolated source audit failed."
    ),
    "stderr_tail": (
        phase42_subprocess_result.stderr[-2000:]
    ),
    "stdout_tail": (
        phase42_subprocess_result.stdout[-2000:]
    ),
}

phase42_subprocess_lines = [
    line.strip()
    for line in (
        phase42_subprocess_result.stdout
        .splitlines()
    )
    if line.strip()
]

assert phase42_subprocess_lines

phase42_formula_audit = json.loads(
    phase42_subprocess_lines[-1]
)

assert Path(
    phase42_formula_audit["module_path"]
) == (
    PHASE42_STAGE_DIRECTORY
    / "phase42_runtime.py"
).resolve()

phase42_source_manifest = json.loads(
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_runtime_source_manifest.json"
    ).read_text(encoding="utf-8")
)

for relative_path, expected_hash in (
    phase42_source_manifest["files"].items()
):
    path = (
        PHASE42_STAGE_DIRECTORY
        / relative_path
    )

    assert path.is_file()
    assert (
        phase42_audit_sha256(path)
        == expected_hash
    ), {
        "message": (
            "Phase42 source-manifest "
            "hash mismatch."
        ),
        "path": relative_path,
    }

phase42_main_source = (
    PHASE42_STAGE_DIRECTORY
    / "main.py"
).read_text(encoding="utf-8")

phase42_runtime_source = (
    PHASE42_STAGE_DIRECTORY
    / "phase42_runtime.py"
).read_text(encoding="utf-8")

assert (
    "from phase42_runtime import "
    "Phase42Runtime"
) in phase42_main_source
assert (
    "runtime = Phase42Runtime(device)"
) in phase42_main_source
assert "route_groups" in phase42_main_source

for line in (
    'print("Built with DINOv3.", flush=True)',
    'print("Initialization complete.", flush=True)',
    'print("Inference started.", flush=True)',
    'print("Inference completed.", flush=True)',
):
    assert line in phase42_main_source

assert (
    "0.75\n"
    "        * raw_phase36_residual"
) not in phase42_runtime_source

# The exact coefficients appear through state lookup,
# not as an additional second scaling operation.
assert phase42_runtime_source.count(
    '"phase36_raw_residual_weight"'
) >= 2
assert phase42_runtime_source.count(
    '"phase33_logit_residual_weight"'
) >= 2
assert (
    "group_alpha\n"
    "        * combined_bounded"
) in phase42_runtime_source

phase42_contract = json.loads(
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_runtime_contract.json"
    ).read_text(encoding="utf-8")
)

assert (
    phase42_contract["target_formula"]
    == (
        "z42 = z_phase12c_full + "
        "alpha_group * clip("
        "0.75*r36_raw + 0.125*r33, "
        "-1, 1)"
    )
)
assert (
    phase42_contract[
        "test_batch_statistics_used"
    ]
    is False
)

phase42_audit_report = {
    "phase": (
        "phase42_staged_offline_"
        "source_and_formula_contract"
    ),
    "status": "accepted",
    "formula": (
        "z42 = z_phase12c_full + "
        "alpha_group * clip("
        "0.75*r36_raw + 0.125*r33, "
        "-1, 1)"
    ),
    "formula_errors": {
        "probability": (
            phase42_formula_audit[
                "formula_error"
            ]
        ),
        "uncapped_residual": (
            phase42_formula_audit[
                "uncapped_error"
            ]
        ),
        "bounded_residual": (
            phase42_formula_audit[
                "bounded_error"
            ]
        ),
        "group_alpha": (
            phase42_formula_audit[
                "alpha_error"
            ]
        ),
        "applied_update": (
            phase42_formula_audit[
                "update_error"
            ]
        ),
    },
    "cap_before_alpha_contract": True,
    "cap_order_test_separation": (
        phase42_formula_audit[
            "cap_order_separation"
        ]
    ),
    "unknown_protocol_anchor_error": (
        phase42_formula_audit[
            "unknown_anchor_error"
        ]
    ),
    "known_group_count": (
        phase42_formula_audit[
            "known_group_count"
        ]
    ),
    "group_alpha_quantiles": {
        "minimum": (
            phase42_formula_audit[
                "group_alpha_minimum"
            ]
        ),
        "median": (
            phase42_formula_audit[
                "group_alpha_median"
            ]
        ),
        "maximum": (
            phase42_formula_audit[
                "group_alpha_maximum"
            ]
        ),
    },
    "single_routed_fold_interface": True,
    "phase36_raw_residual_scaled_once": True,
    "phase40_component_runtime_reused": True,
    "old_phase40_probability_discarded": True,
    "all_probabilities_finite": (
        phase42_formula_audit[
            "all_probabilities_finite"
        ]
    ),
    "all_probabilities_in_contract": (
        phase42_formula_audit[
            "all_probabilities_in_contract"
        ]
    ),
    "source_manifest_verified": True,
    "accepted_phase40_hash_verified": True,
    "generic_production_logging_preserved": True,
    "model_initialization_performed": False,
    "network_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase42_source_audit_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_SOURCE_FORMULA_AUDIT"
)
print(
    json.dumps(
        phase42_audit_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_SOURCE_FORMULA_AUDIT"
)

BEGIN SANITIZED_PHASE42_SOURCE_FORMULA_AUDIT
{
  "phase": "phase42_staged_offline_source_and_formula_contract",
  "status": "accepted",
  "formula": "z42 = z_phase12c_full + alpha_group * clip(0.75*r36_raw + 0.125*r33, -1, 1)",
  "formula_errors": {
    "probability": 0.0,
    "uncapped_residual": 0.0,
    "bounded_residual": 0.0,
    "group_alpha": 0.0,
    "applied_update": 0.0
  },
  "cap_before_alpha_contract": true,
  "cap_order_test_separation": 0.2727272727272727,
  "unknown_protocol_anchor_error": 0.0,
  "known_group_count": 15,
  "group_alpha_quantiles": {
    "minimum": 0.12121212121212122,
    "median": 1.0,
    "maximum": 1.0
  },
  "single_routed_fold_interface": true,
  "phase36_raw_residual_scaled_once": true,
  "phase40_component_runtime_reused": true,
  "old_phase40_probability_discarded": true,
  "all_probabilities_finite": true,
  "all_probabilities_in_contract": true,
  "source_manifest_verified": true,
  "accepted_phase40_hash_verified": true,
  "generic_production

In [17]:
# Phase42 Cell 138F
# Complete staged-runtime OOF parity using cached representations.
#
# This validates:
# - exact routed fold for every training case;
# - Phase33 residual parity;
# - raw Phase36 residual parity;
# - group-alpha parity;
# - cap-before-alpha formula parity;
# - final probability parity;
# - unknown-protocol anchor-only behavior.
#
# The resulting full-fit metrics are diagnostic/optimistic.
# Phase42 nested metrics remain the primary performance estimate.

from pathlib import Path
import gc
import importlib
import json
import os
import sys
import time
import warnings

import numpy as np
import torch
from sklearn.metrics import log_loss, roc_auc_score

phase42_parity_started = time.perf_counter()

warnings.filterwarnings(
    "ignore",
    message="enable_nested_tensor is True.*",
    category=UserWarning,
)

PHASE42_STAGE_DIRECTORY = Path(
    "/kaggle/working/phase42_experimental_submission"
)
PHASE42_STATE_PATH = (
    PHASE42_STAGE_DIRECTORY
    / "phase42_assets"
    / "phase42_hierarchical_gate.json"
)

PHASE42_DENSE_CACHE_PATH = Path(
    "/kaggle/working/"
    "phase32_bilateral_dense_float16.npy"
)
PHASE42_RADIOMICS_CACHE_PATH = Path(
    "/kaggle/working/"
    "phase31_radiomics_float32.npy"
)
PHASE42_GEOMETRY_CACHE_PATH = Path(
    "/kaggle/working/"
    "phase31_sequence_float32.npy"
)

PHASE42_PARITY_BATCH_SIZE = 128
PHASE42_CASE_COUNT = 1362
PHASE42_PROBABILITY_EPSILON = 1e-5

required_globals = [
    "PHASE19_LABELS",
    "phase20_acquisition_groups",
    "PHASE41_BASELINE_OOF_PRIVATE",
    "PHASE41_PHASE33_RESIDUAL_PRIVATE",
    "PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

assert not missing_globals, {
    "message": (
        "Phase42 parity requires restored "
        "Phase41 component state."
    ),
    "missing_globals": missing_globals,
}

for path in (
    PHASE42_STAGE_DIRECTORY,
    PHASE42_STATE_PATH,
    PHASE42_DENSE_CACHE_PATH,
    PHASE42_RADIOMICS_CACHE_PATH,
    PHASE42_GEOMETRY_CACHE_PATH,
):
    assert path.exists(), {
        "message": "Phase42 parity dependency missing.",
        "path": str(path),
    }

phase42_labels = np.asarray(
    PHASE19_LABELS,
    dtype=np.int64,
)
phase42_groups = np.asarray(
    phase20_acquisition_groups,
    dtype=np.int64,
)
phase42_baseline_probability = np.clip(
    np.asarray(
        PHASE41_BASELINE_OOF_PRIVATE,
        dtype=np.float64,
    ),
    PHASE42_PROBABILITY_EPSILON,
    1.0 - PHASE42_PROBABILITY_EPSILON,
)
phase42_reference_r33 = np.asarray(
    PHASE41_PHASE33_RESIDUAL_PRIVATE,
    dtype=np.float64,
)
phase42_reference_r36_raw = np.asarray(
    PHASE41_PHASE36_RAW_RESIDUAL_PRIVATE,
    dtype=np.float64,
)

for name, array in (
    ("labels", phase42_labels),
    ("groups", phase42_groups),
    (
        "baseline probabilities",
        phase42_baseline_probability,
    ),
    (
        "Phase33 residual",
        phase42_reference_r33,
    ),
    (
        "raw Phase36 residual",
        phase42_reference_r36_raw,
    ),
):
    assert array.shape == (
        PHASE42_CASE_COUNT,
    ), {
        "message": (
            f"Phase42 {name} shape mismatch."
        ),
        "shape": list(array.shape),
    }

assert set(np.unique(phase42_labels)) == {0, 1}
assert set(np.unique(phase42_groups)) == set(
    range(15)
)

phase42_state = json.loads(
    PHASE42_STATE_PATH.read_text(
        encoding="utf-8"
    )
)

phase42_group_alpha_array = np.asarray(
    [
        float(
            phase42_state[
                "known_group_alpha"
            ][str(group)]
        )
        for group in range(15)
    ],
    dtype=np.float64,
)

assert phase42_group_alpha_array.shape == (
    15,
)
assert np.isfinite(
    phase42_group_alpha_array
).all()
assert np.all(
    (phase42_group_alpha_array >= 0.0)
    & (phase42_group_alpha_array <= 1.0)
)

# Exact held-out component fold for each acquisition group.
# This mapping is the accepted Phase40 routing contract.
PHASE42_GROUP_TO_COMPONENT_FOLD = np.asarray(
    [
        1,  # group 0
        0,  # group 1
        1,  # group 2
        2,  # group 3
        2,  # group 4
        0,  # group 5
        2,  # group 6
        1,  # group 7
        2,  # group 8
        0,  # group 9
        2,  # group 10
        1,  # group 11
        1,  # group 12
        1,  # group 13
        2,  # group 14
    ],
    dtype=np.int64,
)

phase42_route_folds = (
    PHASE42_GROUP_TO_COMPONENT_FOLD[
        phase42_groups
    ]
)
phase42_route_exact = np.ones(
    PHASE42_CASE_COUNT,
    dtype=bool,
)

phase42_fold_route_counts = {
    str(fold): int(
        np.sum(
            phase42_route_folds == fold
        )
    )
    for fold in range(3)
}

assert phase42_fold_route_counts == {
    "0": 467,
    "1": 443,
    "2": 452,
}, {
    "message": (
        "Phase42 routed fold counts changed."
    ),
    "observed": phase42_fold_route_counts,
}

# Validate the hard-coded runtime mapping against
# the restored Phase19 outer-validation partitions.
if "PHASE19_PARTITIONS" in globals():
    phase42_partition_fold = np.full(
        PHASE42_CASE_COUNT,
        -1,
        dtype=np.int64,
    )

    for fold, partition in enumerate(
        PHASE19_PARTITIONS
    ):
        outer_valid_indices = None

        for key in (
            "outer_valid_indices",
            "outer_valid",
            "valid_indices",
            "validation_indices",
        ):
            if key in partition:
                outer_valid_indices = np.asarray(
                    partition[key],
                    dtype=np.int64,
                )
                break

        assert outer_valid_indices is not None, {
            "message": (
                "Could not resolve Phase19 "
                "outer-validation indices."
            ),
            "fold": fold,
            "available_keys": sorted(
                partition.keys()
            ),
        }

        assert np.all(
            phase42_partition_fold[
                outer_valid_indices
            ]
            == -1
        )

        phase42_partition_fold[
            outer_valid_indices
        ] = fold

    assert np.all(
        phase42_partition_fold >= 0
    )
    assert np.array_equal(
        phase42_partition_fold,
        phase42_route_folds,
    ), {
        "message": (
            "Phase42 header route and "
            "Phase19 held-out fold disagree."
        ),
        "alignment_fraction": float(
            np.mean(
                phase42_partition_fold
                == phase42_route_folds
            )
        ),
    }

phase42_dense_cache = np.load(
    PHASE42_DENSE_CACHE_PATH,
    mmap_mode="r",
    allow_pickle=False,
)
phase42_radiomics_cache = np.load(
    PHASE42_RADIOMICS_CACHE_PATH,
    mmap_mode="r",
    allow_pickle=False,
)
phase42_geometry_cache = np.load(
    PHASE42_GEOMETRY_CACHE_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

assert phase42_dense_cache.shape == (
    PHASE42_CASE_COUNT,
    6,
    14,
    14,
    384,
)
assert phase42_dense_cache.dtype == np.float16

assert phase42_radiomics_cache.shape == (
    PHASE42_CASE_COUNT,
    1081,
)
assert (
    phase42_radiomics_cache.dtype
    == np.float32
)

assert phase42_geometry_cache.shape == (
    PHASE42_CASE_COUNT,
    16,
    52,
)
assert (
    phase42_geometry_cache.dtype
    == np.float32
)

phase42_baseline_logit = np.log(
    phase42_baseline_probability
    / (
        1.0
        - phase42_baseline_probability
    )
)

phase42_reference_uncapped = (
    0.75
    * phase42_reference_r36_raw
    + 0.125
    * phase42_reference_r33
)
phase42_reference_bounded = np.clip(
    phase42_reference_uncapped,
    -1.0,
    1.0,
)
phase42_reference_alpha = (
    phase42_group_alpha_array[
        phase42_groups
    ]
)
phase42_reference_update = (
    phase42_reference_alpha
    * phase42_reference_bounded
)
phase42_reference_probability = np.clip(
    1.0
    / (
        1.0
        + np.exp(
            -(
                phase42_baseline_logit
                + phase42_reference_update
            )
        )
    ),
    PHASE42_PROBABILITY_EPSILON,
    1.0 - PHASE42_PROBABILITY_EPSILON,
)

assert np.isfinite(
    phase42_reference_probability
).all()

# Force imports to come from the staged package,
# avoiding stale notebook module objects.
for module_name in (
    "phase42_runtime",
    "phase40_runtime",
    "phase39_runtime",
    "phase30_accepted_main",
    "base_main",
):
    sys.modules.pop(
        module_name,
        None,
    )

phase42_stage_string = str(
    PHASE42_STAGE_DIRECTORY
)

sys.path = [
    item
    for item in sys.path
    if item != phase42_stage_string
]
sys.path.insert(
    0,
    phase42_stage_string,
)

importlib.invalidate_caches()
import phase42_runtime as phase42_staged_runtime_module

assert Path(
    phase42_staged_runtime_module.__file__
).resolve() == (
    PHASE42_STAGE_DIRECTORY
    / "phase42_runtime.py"
).resolve()

Phase42StagedRuntime = (
    phase42_staged_runtime_module
    .Phase42Runtime
)

if torch.cuda.is_available():
    PHASE42_RUNTIME_DEVICE = "cuda:0"
else:
    PHASE42_RUNTIME_DEVICE = "cpu"

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

phase42_runtime = Phase42StagedRuntime(
    PHASE42_RUNTIME_DEVICE
)

assert len(
    phase42_runtime.phase33_models
) == 3
assert sum(
    len(models)
    for models in (
        phase42_runtime.phase33_models
    )
) == 9

assert len(
    phase42_runtime.phase36_boosters
) == 3
assert sum(
    len(records)
    for records in (
        phase42_runtime.phase36_boosters
    )
) == 6

phase42_runtime_probability = np.zeros(
    PHASE42_CASE_COUNT,
    dtype=np.float64,
)
phase42_runtime_r33 = np.zeros(
    PHASE42_CASE_COUNT,
    dtype=np.float64,
)
phase42_runtime_r36_raw = np.zeros(
    PHASE42_CASE_COUNT,
    dtype=np.float64,
)
phase42_runtime_uncapped = np.zeros(
    PHASE42_CASE_COUNT,
    dtype=np.float64,
)
phase42_runtime_bounded = np.zeros(
    PHASE42_CASE_COUNT,
    dtype=np.float64,
)
phase42_runtime_alpha = np.zeros(
    PHASE42_CASE_COUNT,
    dtype=np.float64,
)
phase42_runtime_update = np.zeros(
    PHASE42_CASE_COUNT,
    dtype=np.float64,
)

with torch.inference_mode():
    for start in range(
        0,
        PHASE42_CASE_COUNT,
        PHASE42_PARITY_BATCH_SIZE,
    ):
        stop = min(
            start
            + PHASE42_PARITY_BATCH_SIZE,
            PHASE42_CASE_COUNT,
        )

        batch_probability, batch_diagnostics = (
            phase42_runtime
            .predict_routed_from_dense(
                dense_features=(
                    phase42_dense_cache[
                        start:stop
                    ]
                ),
                radiomics=(
                    phase42_radiomics_cache[
                        start:stop
                    ]
                ),
                geometry_sequence=(
                    phase42_geometry_cache[
                        start:stop
                    ]
                ),
                baseline_probability=(
                    phase42_baseline_probability[
                        start:stop
                    ]
                ),
                route_groups=(
                    phase42_groups[
                        start:stop
                    ]
                ),
                route_folds=(
                    phase42_route_folds[
                        start:stop
                    ]
                ),
                route_exact=(
                    phase42_route_exact[
                        start:stop
                    ]
                ),
            )
        )

        phase42_runtime_probability[
            start:stop
        ] = np.asarray(
            batch_probability,
            dtype=np.float64,
        )
        phase42_runtime_r33[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "phase33_residual"
            ],
            dtype=np.float64,
        )
        phase42_runtime_r36_raw[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "raw_phase36_residual"
            ],
            dtype=np.float64,
        )
        phase42_runtime_uncapped[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "combined_uncapped"
            ],
            dtype=np.float64,
        )
        phase42_runtime_bounded[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "combined_bounded"
            ],
            dtype=np.float64,
        )
        phase42_runtime_alpha[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "group_alpha"
            ],
            dtype=np.float64,
        )
        phase42_runtime_update[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "applied_update"
            ],
            dtype=np.float64,
        )

        if (
            stop % 192 == 0
            or stop == PHASE42_CASE_COUNT
        ):
            print(
                "Phase42 staged parity: "
                f"{stop}/{PHASE42_CASE_COUNT}"
            )

assert np.isfinite(
    phase42_runtime_probability
).all()
assert np.all(
    (
        phase42_runtime_probability
        >= PHASE42_PROBABILITY_EPSILON
    )
    & (
        phase42_runtime_probability
        <= (
            1.0
            - PHASE42_PROBABILITY_EPSILON
        )
    )
)

# Exercise unknown-header routing through the
# complete staged runtime class.
phase42_unknown_indices = np.asarray(
    [0, 1, 2, 3],
    dtype=np.int64,
)

with torch.inference_mode():
    (
        phase42_unknown_probability,
        phase42_unknown_diagnostics,
    ) = (
        phase42_runtime
        .predict_routed_from_dense(
            dense_features=(
                phase42_dense_cache[
                    phase42_unknown_indices
                ]
            ),
            radiomics=(
                phase42_radiomics_cache[
                    phase42_unknown_indices
                ]
            ),
            geometry_sequence=(
                phase42_geometry_cache[
                    phase42_unknown_indices
                ]
            ),
            baseline_probability=(
                phase42_baseline_probability[
                    phase42_unknown_indices
                ]
            ),
            route_groups=np.full(
                len(
                    phase42_unknown_indices
                ),
                -1,
                dtype=np.int64,
            ),
            route_folds=np.zeros(
                len(
                    phase42_unknown_indices
                ),
                dtype=np.int64,
            ),
            route_exact=np.zeros(
                len(
                    phase42_unknown_indices
                ),
                dtype=bool,
            ),
        )
    )

phase42_unknown_anchor_error = float(
    np.max(
        np.abs(
            np.asarray(
                phase42_unknown_probability,
                dtype=np.float64,
            )
            - phase42_baseline_probability[
                phase42_unknown_indices
            ]
        )
    )
)
phase42_unknown_update_error = float(
    np.max(
        np.abs(
            np.asarray(
                phase42_unknown_diagnostics[
                    "applied_update"
                ],
                dtype=np.float64,
            )
        )
    )
)

def phase42_error_summary(
    observed,
    expected,
):
    difference = np.abs(
        np.asarray(
            observed,
            dtype=np.float64,
        )
        - np.asarray(
            expected,
            dtype=np.float64,
        )
    )

    return {
        "maximum": float(
            np.max(difference)
        ),
        "mean": float(
            np.mean(difference)
        ),
        "q99": float(
            np.quantile(
                difference,
                0.99,
            )
        ),
    }


phase42_component_errors = {
    "phase33_logit_residual": (
        phase42_error_summary(
            phase42_runtime_r33,
            phase42_reference_r33,
        )
    ),
    "phase36_raw_residual": (
        phase42_error_summary(
            phase42_runtime_r36_raw,
            phase42_reference_r36_raw,
        )
    ),
    "combined_uncapped": (
        phase42_error_summary(
            phase42_runtime_uncapped,
            phase42_reference_uncapped,
        )
    ),
    "combined_bounded": (
        phase42_error_summary(
            phase42_runtime_bounded,
            phase42_reference_bounded,
        )
    ),
    "group_alpha": (
        phase42_error_summary(
            phase42_runtime_alpha,
            phase42_reference_alpha,
        )
    ),
    "applied_update": (
        phase42_error_summary(
            phase42_runtime_update,
            phase42_reference_update,
        )
    ),
    "final_probability": (
        phase42_error_summary(
            phase42_runtime_probability,
            phase42_reference_probability,
        )
    ),
}

assert (
    phase42_component_errors[
        "phase33_logit_residual"
    ]["maximum"]
    <= 2e-6
), phase42_component_errors[
    "phase33_logit_residual"
]

assert (
    phase42_component_errors[
        "phase36_raw_residual"
    ]["maximum"]
    <= 1e-10
), phase42_component_errors[
    "phase36_raw_residual"
]

assert (
    phase42_component_errors[
        "combined_uncapped"
    ]["maximum"]
    <= 2e-6
), phase42_component_errors[
    "combined_uncapped"
]

assert (
    phase42_component_errors[
        "combined_bounded"
    ]["maximum"]
    <= 2e-6
), phase42_component_errors[
    "combined_bounded"
]

assert (
    phase42_component_errors[
        "group_alpha"
    ]["maximum"]
    == 0.0
), phase42_component_errors[
    "group_alpha"
]

assert (
    phase42_component_errors[
        "applied_update"
    ]["maximum"]
    <= 2e-6
), phase42_component_errors[
    "applied_update"
]

assert (
    phase42_component_errors[
        "final_probability"
    ]["maximum"]
    <= 2e-7
), phase42_component_errors[
    "final_probability"
]

assert phase42_unknown_anchor_error <= 1e-15
assert phase42_unknown_update_error == 0.0

phase42_runtime_metrics = {
    "log_loss": float(
        log_loss(
            phase42_labels,
            phase42_runtime_probability,
            labels=[0, 1],
        )
    ),
    "auroc": float(
        roc_auc_score(
            phase42_labels,
            phase42_runtime_probability,
        )
    ),
    "mean_probability": float(
        np.mean(
            phase42_runtime_probability
        )
    ),
}

phase42_reference_metrics = {
    "log_loss": float(
        log_loss(
            phase42_labels,
            phase42_reference_probability,
            labels=[0, 1],
        )
    ),
    "auroc": float(
        roc_auc_score(
            phase42_labels,
            phase42_reference_probability,
        )
    ),
    "mean_probability": float(
        np.mean(
            phase42_reference_probability
        )
    ),
}

assert abs(
    phase42_runtime_metrics["log_loss"]
    - phase42_reference_metrics["log_loss"]
) <= 1e-9

assert abs(
    phase42_runtime_metrics["auroc"]
    - phase42_reference_metrics["auroc"]
) <= 1e-10

# Confirm the corrected full-fit diagnostic
# previously recorded in Cell 138A.
assert abs(
    phase42_runtime_metrics["log_loss"]
    - 0.290166
) <= 1e-5, phase42_runtime_metrics

assert abs(
    phase42_runtime_metrics["auroc"]
    - 0.946474
) <= 1e-5, phase42_runtime_metrics

phase42_baseline_metrics = {
    "log_loss": float(
        log_loss(
            phase42_labels,
            phase42_baseline_probability,
            labels=[0, 1],
        )
    ),
    "auroc": float(
        roc_auc_score(
            phase42_labels,
            phase42_baseline_probability,
        )
    ),
    "mean_probability": float(
        np.mean(
            phase42_baseline_probability
        )
    ),
}

phase42_fold_metrics = []

for fold in range(3):
    local = np.flatnonzero(
        phase42_route_folds == fold
    )

    baseline_fold_log_loss = float(
        log_loss(
            phase42_labels[local],
            phase42_baseline_probability[
                local
            ],
            labels=[0, 1],
        )
    )
    runtime_fold_log_loss = float(
        log_loss(
            phase42_labels[local],
            phase42_runtime_probability[
                local
            ],
            labels=[0, 1],
        )
    )

    phase42_fold_metrics.append({
        "fold": fold,
        "n": int(len(local)),
        "baseline_log_loss": (
            baseline_fold_log_loss
        ),
        "phase42_log_loss": (
            runtime_fold_log_loss
        ),
        "log_loss_gain": (
            baseline_fold_log_loss
            - runtime_fold_log_loss
        ),
        "baseline_auroc": float(
            roc_auc_score(
                phase42_labels[local],
                phase42_baseline_probability[
                    local
                ],
            )
        ),
        "phase42_auroc": float(
            roc_auc_score(
                phase42_labels[local],
                phase42_runtime_probability[
                    local
                ],
            )
        ),
    })

phase42_major_group_metrics = []

for group in sorted(
    np.unique(phase42_groups)
):
    local = np.flatnonzero(
        phase42_groups == group
    )

    if len(local) < 30:
        continue

    baseline_group_log_loss = float(
        log_loss(
            phase42_labels[local],
            phase42_baseline_probability[
                local
            ],
            labels=[0, 1],
        )
    )
    runtime_group_log_loss = float(
        log_loss(
            phase42_labels[local],
            phase42_runtime_probability[
                local
            ],
            labels=[0, 1],
        )
    )

    group_record = {
        "group": int(group),
        "n": int(len(local)),
        "alpha": float(
            phase42_group_alpha_array[
                group
            ]
        ),
        "baseline_log_loss": (
            baseline_group_log_loss
        ),
        "phase42_log_loss": (
            runtime_group_log_loss
        ),
        "log_loss_gain": (
            baseline_group_log_loss
            - runtime_group_log_loss
        ),
    }

    if len(
        np.unique(
            phase42_labels[local]
        )
    ) == 2:
        group_record.update({
            "baseline_auroc": float(
                roc_auc_score(
                    phase42_labels[local],
                    phase42_baseline_probability[
                        local
                    ],
                )
            ),
            "phase42_auroc": float(
                roc_auc_score(
                    phase42_labels[local],
                    phase42_runtime_probability[
                        local
                    ],
                )
            ),
        })

    phase42_major_group_metrics.append(
        group_record
    )

phase42_major_group_harms = [
    -record["log_loss_gain"]
    for record in phase42_major_group_metrics
    if record["log_loss_gain"] < 0.0
]

phase42_maximum_major_group_harm = (
    max(phase42_major_group_harms)
    if phase42_major_group_harms
    else 0.0
)

phase42_major_group_wins = sum(
    record["log_loss_gain"] > 0.0
    for record in (
        phase42_major_group_metrics
    )
)

phase42_fold_wins = sum(
    record["log_loss_gain"] > 0.0
    for record in phase42_fold_metrics
)

if torch.cuda.is_available():
    phase42_peak_vram_mb = float(
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )
else:
    phase42_peak_vram_mb = 0.0

# Retain only private notebook arrays needed for
# later smoke/archive parity.
PHASE42_RUNTIME_OOF_PRIVATE = (
    phase42_runtime_probability.copy()
)
PHASE42_FULL_FIT_REFERENCE_OOF_PRIVATE = (
    phase42_reference_probability.copy()
)
PHASE42_RUNTIME_COMPONENTS_PRIVATE = {
    "phase33_residual": (
        phase42_runtime_r33.copy()
    ),
    "phase36_raw_residual": (
        phase42_runtime_r36_raw.copy()
    ),
    "combined_uncapped": (
        phase42_runtime_uncapped.copy()
    ),
    "combined_bounded": (
        phase42_runtime_bounded.copy()
    ),
    "group_alpha": (
        phase42_runtime_alpha.copy()
    ),
    "applied_update": (
        phase42_runtime_update.copy()
    ),
}

phase42_parity_report = {
    "phase": (
        "phase42_complete_staged_"
        "runtime_oof_parity"
    ),
    "status": "accepted",
    "case_count": PHASE42_CASE_COUNT,
    "route": {
        "known_route_fraction": float(
            np.mean(
                phase42_route_exact
            )
        ),
        "fold_route_counts": (
            phase42_fold_route_counts
        ),
        "single_routed_fold_per_case": True,
        "three_fold_residual_average": False,
    },
    "component_errors": (
        phase42_component_errors
    ),
    "unknown_protocol_contract": {
        "tested_case_count": int(
            len(
                phase42_unknown_indices
            )
        ),
        "anchor_probability_error": (
            phase42_unknown_anchor_error
        ),
        "applied_update_error": (
            phase42_unknown_update_error
        ),
        "policy": (
            "complete_phase12c_anchor_only"
        ),
    },
    "formula": {
        "phase36_raw_residual_weight": 0.75,
        "phase33_logit_residual_weight": 0.125,
        "residual_cap": 1.0,
        "uncertainty_exponent": 0.0,
        "cap_before_alpha": True,
    },
    "runtime_metrics": (
        phase42_runtime_metrics
    ),
    "reference_metrics": (
        phase42_reference_metrics
    ),
    "honest_nested_reference": {
        "log_loss": 0.293516,
        "auroc": 0.9448,
        "primary_estimate": True,
    },
    "full_fit_metric_interpretation": (
        "diagnostic_only_optimistic"
    ),
    "baseline": (
        phase42_baseline_metrics
    ),
    "diagnostic_improvements": {
        "log_loss_gain_over_phase12c": (
            phase42_baseline_metrics[
                "log_loss"
            ]
            - phase42_runtime_metrics[
                "log_loss"
            ]
        ),
        "auroc_gain_over_phase12c": (
            phase42_runtime_metrics[
                "auroc"
            ]
            - phase42_baseline_metrics[
                "auroc"
            ]
        ),
        "fold_wins": int(
            phase42_fold_wins
        ),
        "major_group_wins": int(
            phase42_major_group_wins
        ),
        "major_group_count": int(
            len(
                phase42_major_group_metrics
            )
        ),
        "maximum_major_group_harm": (
            phase42_maximum_major_group_harm
        ),
    },
    "fold_metrics": phase42_fold_metrics,
    "major_group_metrics": (
        phase42_major_group_metrics
    ),
    "runtime": {
        "device": (
            PHASE42_RUNTIME_DEVICE
        ),
        "phase33_model_count": 9,
        "phase36_booster_count": 6,
        "dinov3_loaded": False,
        "peak_vram_mb": (
            phase42_peak_vram_mb
        ),
    },
    "phase12c_anchor": (
        "exact_cross_fitted_oof_for_parity"
    ),
    "deployment_anchor": (
        "complete_three_fold_phase12c_ensemble"
    ),
    "labels_used_only_for_aggregate_metrics": True,
    "training_cached_features_read": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase42_parity_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_STAGED_PARITY"
)
print(
    json.dumps(
        phase42_parity_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_STAGED_PARITY"
)

Phase42 staged parity: 384/1362
Phase42 staged parity: 768/1362
Phase42 staged parity: 1152/1362
Phase42 staged parity: 1362/1362
BEGIN SANITIZED_PHASE42_STAGED_PARITY
{
  "phase": "phase42_complete_staged_runtime_oof_parity",
  "status": "accepted",
  "case_count": 1362,
  "route": {
    "known_route_fraction": 1.0,
    "fold_route_counts": {
      "0": 467,
      "1": 443,
      "2": 452
    },
    "single_routed_fold_per_case": true,
    "three_fold_residual_average": false
  },
  "component_errors": {
    "phase33_logit_residual": {
      "maximum": 9.536743474924947e-07,
      "mean": 1.2453556156372392e-07,
      "q99": 4.76837182441514e-07
    },
    "phase36_raw_residual": {
      "maximum": 8.881784197001252e-16,
      "mean": 1.4954013691189605e-16,
      "q99": 8.881784197001252e-16
    },
    "combined_uncapped": {
      "maximum": 1.1920929354758414e-07,
      "mean": 1.5566945211298343e-08,
      "q99": 5.960464792731379e-08
    },
    "combined_bounded": {
      "maximum

In [19]:
# Phase42 Cell 138G
# Fixes smoke-path resolution:
# - accepts exactly one of .nii.gz or .nii;
# - preserves UID strings exactly;
# - follows the accepted base_main.py contract.

from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

import numpy as np
import pandas as pd

phase42_smoke_started = time.perf_counter()

PHASE42_SMOKE_PACKAGE = Path(
    "/kaggle/working/phase42_experimental_submission"
)
PHASE42_EXPECTED_SMOKE_CASES = 20
PHASE42_SMOKE_TIMEOUT_SECONDS = 360.0

PHASE42_ALLOWED_PRODUCTION_LINES = [
    "Built with DINOv3.",
    "Initialization complete.",
    "Inference started.",
    "Inference completed.",
]

assert PHASE42_SMOKE_PACKAGE.is_dir()

for relative_path in (
    "main.py",
    "phase39_runtime.py",
    "phase40_runtime.py",
    "phase42_runtime.py",
    "phase42_assets/phase42_hierarchical_gate.json",
    "phase42_assets/phase42_asset_manifest.json",
):
    assert (
        PHASE42_SMOKE_PACKAGE
        / relative_path
    ).is_file(), relative_path


def phase42_read_submission_contract(
    root,
):
    submission_path = (
        Path(root)
        / "submission_format.csv"
    )
    nifti_directory = (
        Path(root)
        / "niftis"
    )

    if (
        not submission_path.is_file()
        or not nifti_directory.is_dir()
    ):
        return None

    try:
        with submission_path.open(
            "r",
            newline="",
            encoding="utf-8-sig",
        ) as handle:
            reader = csv.DictReader(handle)

            if reader.fieldnames != [
                "uid",
                "is_pathologic",
            ]:
                return None

            rows = list(reader)
    except Exception:
        return None

    if len(rows) != (
        PHASE42_EXPECTED_SMOKE_CASES
    ):
        return None

    uids = [
        str(row["uid"])
        for row in rows
    ]

    if (
        not uids
        or len(uids) != len(set(uids))
        or any(
            not uid
            or uid != uid.strip()
            or Path(uid).name != uid
            for uid in uids
        )
    ):
        return None

    resolved_paths = []

    for uid in uids:
        candidates = [
            nifti_directory
            / f"{uid}.nii.gz",
            nifti_directory
            / f"{uid}.nii",
        ]

        existing = [
            path
            for path in candidates
            if path.is_file()
        ]

        # This exactly matches base_main.py:
        # each UID must map to one and only one file.
        if len(existing) != 1:
            return None

        resolved_paths.append(
            existing[0]
        )

    return {
        "root": Path(root).resolve(),
        "uids": uids,
        "paths": resolved_paths,
        "columns": [
            "uid",
            "is_pathologic",
        ],
        "compressed_nifti_count": int(
            sum(
                path.name.endswith(
                    ".nii.gz"
                )
                for path in resolved_paths
            )
        ),
        "uncompressed_nifti_count": int(
            sum(
                path.name.endswith(
                    ".nii"
                )
                and not path.name.endswith(
                    ".nii.gz"
                )
                for path in resolved_paths
            )
        ),
    }


def phase42_resolve_smoke_root():
    raw_candidates = []

    for global_name in (
        "PHASE40_SMOKE_ROOT_PRIVATE",
        "PHASE39_SMOKE_ROOT_PRIVATE",
        "PHASE30_SMOKE_ROOT_PRIVATE",
        "PHASE40_SMOKE_ROOT",
        "PHASE39_SMOKE_ROOT",
        "PHASE30_SMOKE_ROOT",
        "SMOKE_DATA_ROOT",
        "SMOKE_ROOT",
    ):
        value = globals().get(
            global_name
        )

        if value is None:
            continue

        candidate = Path(value)

        if candidate.is_dir():
            raw_candidates.append({
                "source": (
                    "retained_private_root"
                ),
                "root": candidate.resolve(),
            })

    kaggle_input_root = Path(
        "/kaggle/input"
    )

    if kaggle_input_root.is_dir():
        for submission_path in (
            kaggle_input_root.rglob(
                "submission_format.csv"
            )
        ):
            raw_candidates.append({
                "source": (
                    "private_dataset_discovery"
                ),
                "root": (
                    submission_path
                    .parent
                    .resolve()
                ),
            })

    unique_candidates = {}

    for record in raw_candidates:
        unique_candidates[
            str(record["root"])
        ] = record

    accepted = []

    for record in (
        unique_candidates.values()
    ):
        contract = (
            phase42_read_submission_contract(
                record["root"]
            )
        )

        if contract is None:
            continue

        accepted.append({
            **record,
            "contract": contract,
            "smoke_named": (
                "smoke"
                in str(
                    record["root"]
                ).lower()
            ),
        })

    assert accepted, {
        "message": (
            "Could not resolve a valid "
            "20-case private smoke dataset."
        ),
        "candidate_root_count": len(
            unique_candidates
        ),
        "resolver_contract": (
            "exactly one of "
            "<uid>.nii.gz or <uid>.nii"
        ),
    }

    accepted.sort(
        key=lambda record: (
            not record["smoke_named"],
            record["source"]
            != "retained_private_root",
            len(record["root"].parts),
            str(record["root"]),
        )
    )

    selected = accepted[0]
    selected_uids = selected[
        "contract"
    ]["uids"]

    # Ensure any duplicate valid mounts refer
    # to the same smoke case contract.
    for record in accepted:
        if (
            record["smoke_named"]
            == selected["smoke_named"]
            and record["contract"]["uids"]
            != selected_uids
        ):
            raise AssertionError({
                "message": (
                    "Multiple distinct private "
                    "smoke contracts discovered."
                ),
                "valid_candidate_count": len(
                    accepted
                ),
            })

    return (
        selected["contract"],
        selected["source"],
        len(unique_candidates),
        len(accepted),
    )


(
    phase42_smoke_contract,
    phase42_smoke_root_source,
    phase42_candidate_root_count,
    phase42_valid_root_count,
) = phase42_resolve_smoke_root()

phase42_smoke_root = (
    phase42_smoke_contract["root"]
)
phase42_expected_uids = (
    phase42_smoke_contract["uids"]
)

assert len(
    phase42_expected_uids
) == PHASE42_EXPECTED_SMOKE_CASES


def phase42_run_package(
    output_path,
    diagnostics,
    torch_cache,
):
    environment = dict(os.environ)

    environment.update({
        "DAT_DATA_ROOT": str(
            phase42_smoke_root
        ),
        "DAT_OUTPUT_CSV": str(
            output_path
        ),
        "DAT_PREPROCESS_WORKERS": "6",
        "DAT_BATCH_SIZE": "6",
        "DAT_DINO_VIEW_BATCH": "12",
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "TORCH_HOME": str(
            torch_cache
        ),
        "PYTHONHASHSEED": "0",
        "http_proxy": (
            "http://127.0.0.1:9"
        ),
        "https_proxy": (
            "http://127.0.0.1:9"
        ),
        "HTTP_PROXY": (
            "http://127.0.0.1:9"
        ),
        "HTTPS_PROXY": (
            "http://127.0.0.1:9"
        ),
        "NO_PROXY": "",
        "no_proxy": "",
    })

    if diagnostics:
        environment[
            "DAT_PRIVATE_DIAGNOSTICS"
        ] = "1"
    else:
        environment.pop(
            "DAT_PRIVATE_DIAGNOSTICS",
            None,
        )

    started = time.perf_counter()

    result = subprocess.run(
        [
            sys.executable,
            "main.py",
        ],
        cwd=str(
            PHASE42_SMOKE_PACKAGE
        ),
        env=environment,
        capture_output=True,
        text=True,
        timeout=(
            PHASE42_SMOKE_TIMEOUT_SECONDS
        ),
        check=False,
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    combined_log = (
        result.stdout
        + "\n"
        + result.stderr
    )

    return result, combined_log, elapsed


phase42_temporary_root = Path(
    tempfile.mkdtemp(
        prefix="phase42_smoke_",
        dir="/kaggle/working",
    )
)
phase42_diagnostic_output = (
    phase42_temporary_root
    / "diagnostic_submission.csv"
)
phase42_production_output = (
    phase42_temporary_root
    / "production_submission.csv"
)
phase42_torch_cache = (
    phase42_temporary_root
    / "torch_cache"
)

(
    phase42_diagnostic_result,
    phase42_diagnostic_log,
    phase42_diagnostic_seconds,
) = phase42_run_package(
    output_path=(
        phase42_diagnostic_output
    ),
    diagnostics=True,
    torch_cache=phase42_torch_cache,
)

assert (
    phase42_diagnostic_result.returncode
    == 0
), {
    "message": (
        "Phase42 diagnostic smoke failed."
    ),
    "return_code": (
        phase42_diagnostic_result.returncode
    ),
    "stdout_tail": (
        phase42_diagnostic_result.stdout[
            -2000:
        ]
    ),
    "stderr_tail": (
        phase42_diagnostic_result.stderr[
            -2000:
        ]
    ),
}

assert phase42_diagnostic_output.is_file()

phase42_diagnostic_match = re.search(
    (
        r"Private diagnostics:\s*"
        r"preprocessing=(\d+),\s*"
        r"unknown_routes=(\d+),\s*"
        r"initialization=(\d+),\s*"
        r"case_fallbacks=(\d+)\."
    ),
    phase42_diagnostic_log,
)

assert (
    phase42_diagnostic_match
    is not None
), {
    "message": (
        "Phase42 private diagnostic "
        "counter line missing."
    ),
}

(
    phase42_preprocessing_failures,
    phase42_unknown_routes,
    phase42_initialization_fallbacks,
    phase42_case_fallbacks,
) = [
    int(value)
    for value in (
        phase42_diagnostic_match.groups()
    )
]

phase42_known_routes = (
    PHASE42_EXPECTED_SMOKE_CASES
    - phase42_unknown_routes
)

assert phase42_preprocessing_failures == 0
assert phase42_initialization_fallbacks == 0
assert phase42_case_fallbacks == 0
assert phase42_known_routes > 0

(
    phase42_production_result,
    phase42_production_log,
    phase42_production_seconds,
) = phase42_run_package(
    output_path=(
        phase42_production_output
    ),
    diagnostics=False,
    torch_cache=phase42_torch_cache,
)

assert (
    phase42_production_result.returncode
    == 0
), {
    "message": (
        "Phase42 production smoke failed."
    ),
    "return_code": (
        phase42_production_result.returncode
    ),
    "stdout_tail": (
        phase42_production_result.stdout[
            -2000:
        ]
    ),
    "stderr_tail": (
        phase42_production_result.stderr[
            -2000:
        ]
    ),
}

assert phase42_production_output.is_file()

phase42_production_lines = [
    line.strip()
    for line in (
        phase42_production_log
        .splitlines()
    )
    if line.strip()
]

assert phase42_production_lines == (
    PHASE42_ALLOWED_PRODUCTION_LINES
), {
    "message": (
        "Phase42 production logs are "
        "not generic-only."
    ),
    "observed_lines": (
        phase42_production_lines
    ),
}

phase42_forbidden_patterns = [
    r"\buid\b",
    r"\bpatient\b",
    r"\bcase[_\s-]*id\b",
    r"\bprobabilit(?:y|ies)\b",
    r"\bpreprocessing\s*=",
    r"\bunknown_routes\s*=",
    r"\binitialization\s*=",
    r"\bcase_fallbacks\s*=",
    r"\b\d+\s*/\s*\d+\b",
    r"https?://",
]

phase42_forbidden_hits = [
    pattern
    for pattern in (
        phase42_forbidden_patterns
    )
    if re.search(
        pattern,
        phase42_production_log,
        flags=re.IGNORECASE,
    )
]

assert not phase42_forbidden_hits, {
    "message": (
        "Forbidden Phase42 production "
        "logging pattern detected."
    ),
    "patterns": (
        phase42_forbidden_hits
    ),
}

phase42_diagnostic_frame = pd.read_csv(
    phase42_diagnostic_output,
    dtype={"uid": str},
)
phase42_production_frame = pd.read_csv(
    phase42_production_output,
    dtype={"uid": str},
)

expected_columns = [
    "uid",
    "is_pathologic",
]

assert list(
    phase42_diagnostic_frame.columns
) == expected_columns
assert list(
    phase42_production_frame.columns
) == expected_columns

assert len(
    phase42_diagnostic_frame
) == PHASE42_EXPECTED_SMOKE_CASES
assert len(
    phase42_production_frame
) == PHASE42_EXPECTED_SMOKE_CASES

assert (
    phase42_diagnostic_frame["uid"]
    .astype(str)
    .tolist()
    == phase42_expected_uids
)
assert (
    phase42_production_frame["uid"]
    .astype(str)
    .tolist()
    == phase42_expected_uids
)

phase42_diagnostic_probability = (
    phase42_diagnostic_frame[
        "is_pathologic"
    ].to_numpy(
        dtype=np.float64
    )
)
phase42_production_probability = (
    phase42_production_frame[
        "is_pathologic"
    ].to_numpy(
        dtype=np.float64
    )
)

assert np.isfinite(
    phase42_diagnostic_probability
).all()
assert np.isfinite(
    phase42_production_probability
).all()

assert np.all(
    (
        phase42_production_probability
        >= 1e-5
    )
    & (
        phase42_production_probability
        <= 1.0 - 1e-5
    )
)

phase42_probability_error = float(
    np.max(
        np.abs(
            phase42_diagnostic_probability
            - phase42_production_probability
        )
    )
)

assert phase42_probability_error == 0.0

phase42_output_nonconstant = bool(
    np.ptp(
        phase42_production_probability
    )
    > 1e-8
)

assert phase42_output_nonconstant

phase42_package_files = [
    path
    for path in (
        PHASE42_SMOKE_PACKAGE
        .rglob("*")
    )
    if path.is_file()
]

phase42_smoke_report = {
    "phase": (
        "phase42_dual_mode_real_"
        "packaged_smoke_test"
    ),
    "status": "accepted",
    "package_directory": (
        PHASE42_SMOKE_PACKAGE.name
    ),
    "package_file_count": len(
        phase42_package_files
    ),
    "package_size_mb": float(
        sum(
            path.stat().st_size
            for path in (
                phase42_package_files
            )
        )
        / (1024 ** 2)
    ),
    "case_count": (
        PHASE42_EXPECTED_SMOKE_CASES
    ),
    "input_contract": {
        "compressed_nifti_count": (
            phase42_smoke_contract[
                "compressed_nifti_count"
            ]
        ),
        "uncompressed_nifti_count": (
            phase42_smoke_contract[
                "uncompressed_nifti_count"
            ]
        ),
        "exactly_one_file_per_uid": True,
    },
    "diagnostic_run": {
        "return_code": 0,
        "elapsed_seconds": float(
            phase42_diagnostic_seconds
        ),
        "preprocessing_failures": (
            phase42_preprocessing_failures
        ),
        "known_routes": (
            phase42_known_routes
        ),
        "unknown_routes": (
            phase42_unknown_routes
        ),
        "initialization_fallbacks": (
            phase42_initialization_fallbacks
        ),
        "case_fallbacks": (
            phase42_case_fallbacks
        ),
        "known_route_branch_exercised": bool(
            phase42_known_routes > 0
        ),
        "unknown_anchor_branch_exercised": bool(
            phase42_unknown_routes > 0
        ),
    },
    "production_run": {
        "return_code": 0,
        "elapsed_seconds": float(
            phase42_production_seconds
        ),
        "log_line_count": len(
            phase42_production_lines
        ),
        "generic_lifecycle_line_count": 4,
        "forbidden_pattern_count": len(
            phase42_forbidden_hits
        ),
        "generic_logging_only": True,
        "network_url_present": False,
    },
    "diagnostic_production_probability_error": (
        phase42_probability_error
    ),
    "schema_valid": True,
    "uid_order_exact": True,
    "all_probabilities_finite": True,
    "all_probabilities_clipped": True,
    "output_nonconstant": (
        phase42_output_nonconstant
    ),
    "offline_environment_enforced": True,
    "separate_subprocesses": True,
    "production_logging_contract_passed": True,
    "smoke_root_source": (
        phase42_smoke_root_source
    ),
    "candidate_root_count": (
        phase42_candidate_root_count
    ),
    "valid_smoke_root_count": (
        phase42_valid_root_count
    ),
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "training_voxel_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "model_hashes_displayed": False,
    "total_cell_seconds": float(
        time.perf_counter()
        - phase42_smoke_started
    ),
}

shutil.rmtree(
    phase42_temporary_root,
    ignore_errors=False,
)

assert not (
    phase42_temporary_root.exists()
)

print(
    "BEGIN PHASE42_PRODUCTION_SMOKE_LOG"
)
for line in phase42_production_lines:
    print(line)
print(
    "END PHASE42_PRODUCTION_SMOKE_LOG"
)

print(
    "BEGIN SANITIZED_PHASE42_REAL_SMOKE"
)
print(
    json.dumps(
        phase42_smoke_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_REAL_SMOKE"
)

BEGIN PHASE42_PRODUCTION_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed.
END PHASE42_PRODUCTION_SMOKE_LOG
BEGIN SANITIZED_PHASE42_REAL_SMOKE
{
  "phase": "phase42_dual_mode_real_packaged_smoke_test",
  "status": "accepted",
  "package_directory": "phase42_experimental_submission",
  "package_file_count": 120,
  "package_size_mb": 193.58465480804443,
  "case_count": 20,
  "input_contract": {
    "compressed_nifti_count": 0,
    "uncompressed_nifti_count": 20,
    "exactly_one_file_per_uid": true
  },
  "diagnostic_run": {
    "return_code": 0,
    "elapsed_seconds": 19.600949571000456,
    "preprocessing_failures": 0,
    "known_routes": 20,
    "unknown_routes": 0,
    "initialization_fallbacks": 0,
    "case_fallbacks": 0,
    "known_route_branch_exercised": true,
    "unknown_anchor_branch_exercised": false
  },
  "production_run": {
    "return_code": 0,
    "elapsed_seconds": 17.09747445299945,
    "log_line_count": 4,
    "generic_life

In [21]:
# Phase42 Cell 138H
# Construct a deterministic clean candidate archive.
#
# Archive contract:
# - exactly the accepted Phase40 member names;
# - main.py replaced by the tested Phase42 entrypoint;
# - five new Phase42 runtime/state files;
# - notebook-created __pycache__ and other transient
#   files are excluded by construction.

from pathlib import Path
import hashlib
import json
import os
import shutil
import time
import zipfile

phase42_archive_build_started = (
    time.perf_counter()
)

PHASE42_ACCEPTED_PHASE40_ARCHIVE = Path(
    "/kaggle/working/phase40_submission.zip"
)
PHASE42_STAGE_DIRECTORY = Path(
    "/kaggle/working/phase42_experimental_submission"
)
PHASE42_CANDIDATE_ARCHIVE = Path(
    "/kaggle/working/phase42_submission.candidate.zip"
)

PHASE42_EXPECTED_PHASE40_SHA256 = (
    "b3420f2ca9e7de7b452765c3f9607c665901a90c63fed6ee19c9fadbd5f73f12"
)
PHASE42_EXPECTED_BASE_MEMBER_COUNT = 89
PHASE42_EXPECTED_ARCHIVE_MEMBER_COUNT = 94

PHASE42_NEW_ARCHIVE_MEMBERS = [
    "phase42_runtime.py",
    "phase42_runtime_contract.json",
    "phase42_runtime_source_manifest.json",
    (
        "phase42_assets/"
        "phase42_asset_manifest.json"
    ),
    (
        "phase42_assets/"
        "phase42_hierarchical_gate.json"
    ),
]


def phase42_archive_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def phase42_safe_member_name(name):
    path = Path(name)

    return (
        bool(name)
        and not name.endswith("/")
        and not path.is_absolute()
        and ".." not in path.parts
        and "\\" not in name
    )


assert (
    PHASE42_ACCEPTED_PHASE40_ARCHIVE
    .is_file()
)
assert PHASE42_STAGE_DIRECTORY.is_dir()

phase40_hash_before = phase42_archive_sha256(
    PHASE42_ACCEPTED_PHASE40_ARCHIVE
)

assert (
    phase40_hash_before
    == PHASE42_EXPECTED_PHASE40_SHA256
), {
    "message": (
        "Accepted Phase40 archive hash changed."
    ),
    "observed": phase40_hash_before,
}

with zipfile.ZipFile(
    PHASE42_ACCEPTED_PHASE40_ARCHIVE,
    mode="r",
) as phase40_archive:
    assert phase40_archive.testzip() is None

    phase40_members = sorted(
        member
        for member in (
            phase40_archive.namelist()
        )
        if not member.endswith("/")
    )

    assert len(phase40_members) == (
        PHASE42_EXPECTED_BASE_MEMBER_COUNT
    )

    assert all(
        phase42_safe_member_name(member)
        for member in phase40_members
    )

    phase40_main_bytes = (
        phase40_archive.read("main.py")
    )

assert not (
    set(PHASE42_NEW_ARCHIVE_MEMBERS)
    & set(phase40_members)
)

phase42_archive_members = sorted(
    phase40_members
    + PHASE42_NEW_ARCHIVE_MEMBERS
)

assert len(
    phase42_archive_members
) == PHASE42_EXPECTED_ARCHIVE_MEMBER_COUNT
assert len(
    phase42_archive_members
) == len(
    set(phase42_archive_members)
)

for member in phase42_archive_members:
    assert phase42_safe_member_name(
        member
    )

    staged_path = (
        PHASE42_STAGE_DIRECTORY
        / member
    )

    assert staged_path.is_file(), {
        "message": (
            "Required staged archive member "
            "is missing."
        ),
        "member": member,
    }

phase42_main_bytes = (
    PHASE42_STAGE_DIRECTORY
    / "main.py"
).read_bytes()

assert (
    phase42_main_bytes
    != phase40_main_bytes
), {
    "message": (
        "Phase42 main.py did not replace "
        "the accepted Phase40 entrypoint."
    ),
}

# Every inherited Phase40 file except main.py must
# remain byte-identical to the accepted archive.
phase42_unchanged_member_count = 0

with zipfile.ZipFile(
    PHASE42_ACCEPTED_PHASE40_ARCHIVE,
    mode="r",
) as phase40_archive:
    for member in phase40_members:
        if member == "main.py":
            continue

        staged_bytes = (
            PHASE42_STAGE_DIRECTORY
            / member
        ).read_bytes()

        assert staged_bytes == (
            phase40_archive.read(member)
        ), {
            "message": (
                "An inherited Phase40 member "
                "was modified."
            ),
            "member": member,
        }

        phase42_unchanged_member_count += 1

assert phase42_unchanged_member_count == 88

phase42_source_manifest = json.loads(
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_runtime_source_manifest.json"
    ).read_text(encoding="utf-8")
)

for relative_path, expected_hash in (
    phase42_source_manifest[
        "files"
    ].items()
):
    path = (
        PHASE42_STAGE_DIRECTORY
        / relative_path
    )

    assert path.is_file()
    assert (
        phase42_archive_sha256(path)
        == expected_hash
    ), {
        "message": (
            "Phase42 source-manifest "
            "verification failed."
        ),
        "file": relative_path,
    }

phase42_asset_manifest = json.loads(
    (
        PHASE42_STAGE_DIRECTORY
        / "phase42_assets"
        / "phase42_asset_manifest.json"
    ).read_text(encoding="utf-8")
)

for relative_path, expected_hash in (
    phase42_asset_manifest[
        "assets"
    ].items()
):
    path = (
        PHASE42_STAGE_DIRECTORY
        / relative_path
    )

    assert path.is_file()
    assert (
        phase42_archive_sha256(path)
        == expected_hash
    )

phase42_all_stage_files = sorted(
    path
    for path in (
        PHASE42_STAGE_DIRECTORY.rglob("*")
    )
    if path.is_file()
)

phase42_all_stage_relative = {
    path.relative_to(
        PHASE42_STAGE_DIRECTORY
    ).as_posix()
    for path in phase42_all_stage_files
}

phase42_excluded_stage_files = sorted(
    phase42_all_stage_relative
    - set(phase42_archive_members)
)

# Transient files must never enter the archive.
for relative_path in (
    phase42_excluded_stage_files
):
    lowered = relative_path.lower()

    assert (
        "__pycache__" in lowered
        or lowered.endswith(".pyc")
        or ".pytest_cache" in lowered
        or lowered.endswith(".tmp")
    ), {
        "message": (
            "Unexpected non-manifest staged "
            "file requires manual audit."
        ),
        "file": relative_path,
    }

if PHASE42_CANDIDATE_ARCHIVE.exists():
    PHASE42_CANDIDATE_ARCHIVE.unlink()

with zipfile.ZipFile(
    PHASE42_CANDIDATE_ARCHIVE,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
    allowZip64=True,
) as output_archive:
    for member in phase42_archive_members:
        source_path = (
            PHASE42_STAGE_DIRECTORY
            / member
        )

        archive_info = zipfile.ZipInfo(
            filename=member,
            date_time=(
                2020,
                1,
                1,
                0,
                0,
                0,
            ),
        )
        archive_info.compress_type = (
            zipfile.ZIP_DEFLATED
        )
        archive_info.create_system = 3
        archive_info.external_attr = (
            0o100644 << 16
        )
        archive_info.flag_bits |= 0x800

        with source_path.open(
            "rb"
        ) as source_handle:
            with output_archive.open(
                archive_info,
                mode="w",
                force_zip64=True,
            ) as target_handle:
                shutil.copyfileobj(
                    source_handle,
                    target_handle,
                    length=1024 * 1024,
                )

assert PHASE42_CANDIDATE_ARCHIVE.is_file()

with zipfile.ZipFile(
    PHASE42_CANDIDATE_ARCHIVE,
    mode="r",
) as candidate_archive:
    assert candidate_archive.testzip() is None

    candidate_members = sorted(
        member
        for member in (
            candidate_archive.namelist()
        )
        if not member.endswith("/")
    )

    assert candidate_members == (
        phase42_archive_members
    )
    assert len(candidate_members) == (
        PHASE42_EXPECTED_ARCHIVE_MEMBER_COUNT
    )
    assert "main.py" in candidate_members
    assert not any(
        member.startswith(
            "phase42_experimental_submission/"
        )
        for member in candidate_members
    )

    for information in (
        candidate_archive.infolist()
    ):
        if information.is_dir():
            continue

        assert information.date_time == (
            2020,
            1,
            1,
            0,
            0,
            0,
        )

for forbidden_fragment in (
    "__pycache__",
    ".pyc",
    "submission.csv",
    "smoke",
    "oof",
    "prediction",
    "patient",
):
    assert not any(
        forbidden_fragment
        in member.lower()
        for member in (
            phase42_archive_members
        )
    ), forbidden_fragment

phase42_model_members = [
    member
    for member in phase42_archive_members
    if (
        member.startswith("models/")
        and member.endswith(".pt")
    )
]
phase42_dinov3_checkpoints = [
    member
    for member in phase42_archive_members
    if (
        "dinov3" in member.lower()
        and member.endswith(".pth")
    )
]
phase42_phase33_models = [
    member
    for member in phase42_archive_members
    if (
        member.startswith(
            "phase39_assets/"
            "phase33_models/"
        )
        and member.endswith(".pt")
    )
]
phase42_phase36_boosters = [
    member
    for member in phase42_archive_members
    if (
        member.startswith(
            "phase39_assets/"
            "phase36_boosters/"
        )
        and member.endswith(".ubj")
    )
]
phase42_p3ca_states = [
    member
    for member in phase42_archive_members
    if member.endswith(
        "phase32_fold_local_p3ca_states.npz"
    )
]

assert len(phase42_model_members) == 21
assert len(phase42_dinov3_checkpoints) == 1
assert len(phase42_phase33_models) == 9
assert len(phase42_phase36_boosters) == 6
assert len(phase42_p3ca_states) == 1

required_attribution_present = any(
    member.endswith("README.md")
    for member in phase42_archive_members
)
dinov3_license_present = any(
    (
        "dinov3" in member.lower()
        and "license" in member.lower()
    )
    for member in phase42_archive_members
)
dinov3_source_present = any(
    member.startswith("dinov3_repo/")
    and member.endswith(".py")
    for member in phase42_archive_members
)

assert required_attribution_present
assert dinov3_license_present
assert dinov3_source_present

phase40_hash_after = phase42_archive_sha256(
    PHASE42_ACCEPTED_PHASE40_ARCHIVE
)

assert phase40_hash_after == phase40_hash_before

phase42_candidate_hash = (
    phase42_archive_sha256(
        PHASE42_CANDIDATE_ARCHIVE
    )
)

phase42_archive_build_report = {
    "phase": (
        "phase42_deterministic_"
        "archive_candidate_build"
    ),
    "status": (
        "accepted_candidate_not_yet_published"
    ),
    "candidate_name": (
        PHASE42_CANDIDATE_ARCHIVE.name
    ),
    "manifest_contract": (
        "accepted_phase40_archive_members_"
        "plus_five_phase42_files"
    ),
    "phase40_member_count": len(
        phase40_members
    ),
    "phase40_unchanged_member_count": (
        phase42_unchanged_member_count
    ),
    "phase42_added_file_count": len(
        PHASE42_NEW_ARCHIVE_MEMBERS
    ),
    "archive_file_count": len(
        phase42_archive_members
    ),
    "excluded_transient_file_count": len(
        phase42_excluded_stage_files
    ),
    "compressed_size_mb": float(
        PHASE42_CANDIDATE_ARCHIVE
        .stat()
        .st_size
        / (1024 ** 2)
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "deterministic_archive_metadata": True,
    "phase12_model_count": len(
        phase42_model_members
    ),
    "dinov3_checkpoint_count": len(
        phase42_dinov3_checkpoints
    ),
    "phase33_model_count": len(
        phase42_phase33_models
    ),
    "phase36_booster_count": len(
        phase42_phase36_boosters
    ),
    "p3ca_state_count": len(
        phase42_p3ca_states
    ),
    "phase42_state_count": 1,
    "dinov3_source_present": (
        dinov3_source_present
    ),
    "required_attribution_present": (
        required_attribution_present
    ),
    "dinov3_license_present": (
        dinov3_license_present
    ),
    "candidate_sha256": (
        phase42_candidate_hash
    ),
    "final_archive_published": False,
    "accepted_phase40_archive_modified": False,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_oof_arrays": False,
    "smoke_data_read": False,
    "challenge_test_data_read": False,
    "elapsed_seconds": float(
        time.perf_counter()
        - phase42_archive_build_started
    ),
}

print(
    "BEGIN SANITIZED_PHASE42_ARCHIVE_BUILD"
)
print(
    json.dumps(
        phase42_archive_build_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_ARCHIVE_BUILD"
)
print(PHASE42_CANDIDATE_ARCHIVE)

BEGIN SANITIZED_PHASE42_ARCHIVE_BUILD
{
  "phase": "phase42_deterministic_archive_candidate_build",
  "status": "accepted_candidate_not_yet_published",
  "candidate_name": "phase42_submission.candidate.zip",
  "manifest_contract": "accepted_phase40_archive_members_plus_five_phase42_files",
  "phase40_member_count": 89,
  "phase40_unchanged_member_count": 88,
  "phase42_added_file_count": 5,
  "archive_file_count": 94,
  "excluded_transient_file_count": 26,
  "compressed_size_mb": 177.2205638885498,
  "root_main_present": true,
  "wrapping_directory_present": false,
  "archive_integrity_test_passed": true,
  "deterministic_archive_metadata": true,
  "phase12_model_count": 21,
  "dinov3_checkpoint_count": 1,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "p3ca_state_count": 1,
  "phase42_state_count": 1,
  "dinov3_source_present": true,
  "required_attribution_present": true,
  "dinov3_license_present": true,
  "candidate_sha256": "48cbe2ea01970b073e5d72ca5801882bba27c0385c0

In [22]:
# Phase42 Cell 138I
# Extract candidate archive into a clean temporary directory,
# run production inference, compare against the accepted staged
# smoke predictions, and publish only after every check passes.

from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

import numpy as np
import pandas as pd

phase42_finalization_started = (
    time.perf_counter()
)

PHASE42_CANDIDATE_ARCHIVE = Path(
    "/kaggle/working/phase42_submission.candidate.zip"
)
PHASE42_FINAL_ARCHIVE = Path(
    "/kaggle/working/phase42_submission.zip"
)
PHASE42_EXPECTED_ARCHIVE_FILES = 94
PHASE42_ARCHIVE_SMOKE_TIMEOUT = 360.0

PHASE42_ALLOWED_PRODUCTION_LINES = [
    "Built with DINOv3.",
    "Initialization complete.",
    "Inference started.",
    "Inference completed.",
]


def phase42_final_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


required_private_globals = [
    "phase42_smoke_root",
    "phase42_expected_uids",
    "phase42_production_probability",
]

missing_private_globals = [
    name
    for name in required_private_globals
    if name not in globals()
]

assert not missing_private_globals, {
    "message": (
        "Run corrected Cell 138G before "
        "the extracted-archive smoke."
    ),
    "missing": missing_private_globals,
}

assert PHASE42_CANDIDATE_ARCHIVE.is_file()
assert Path(phase42_smoke_root).is_dir()

phase42_reference_smoke_probability = (
    np.asarray(
        phase42_production_probability,
        dtype=np.float64,
    )
)
phase42_reference_smoke_uids = [
    str(uid)
    for uid in phase42_expected_uids
]

assert (
    phase42_reference_smoke_probability
    .shape
    == (20,)
)
assert len(
    phase42_reference_smoke_uids
) == 20

phase42_candidate_hash_before = (
    phase42_final_sha256(
        PHASE42_CANDIDATE_ARCHIVE
    )
)

with zipfile.ZipFile(
    PHASE42_CANDIDATE_ARCHIVE,
    mode="r",
) as archive:
    assert archive.testzip() is None

    phase42_candidate_members = [
        member
        for member in (
            archive.namelist()
        )
        if not member.endswith("/")
    ]

    assert len(
        phase42_candidate_members
    ) == PHASE42_EXPECTED_ARCHIVE_FILES

    assert "main.py" in (
        phase42_candidate_members
    )
    assert "phase42_runtime.py" in (
        phase42_candidate_members
    )
    assert (
        "phase42_assets/"
        "phase42_hierarchical_gate.json"
    ) in phase42_candidate_members

phase42_extract_parent = Path(
    tempfile.mkdtemp(
        prefix="phase42_extract_",
        dir="/kaggle/working",
    )
)
phase42_extracted_package = (
    phase42_extract_parent
    / "package"
)
phase42_extracted_package.mkdir()

phase42_extracted_output = (
    phase42_extract_parent
    / "submission.csv"
)
phase42_extracted_torch_cache = (
    phase42_extract_parent
    / "torch_cache"
)

try:
    with zipfile.ZipFile(
        PHASE42_CANDIDATE_ARCHIVE,
        mode="r",
    ) as archive:
        for member in archive.namelist():
            path = Path(member)

            assert not path.is_absolute()
            assert ".." not in path.parts

        archive.extractall(
            phase42_extracted_package
        )

    assert (
        phase42_extracted_package
        / "main.py"
    ).is_file()

    environment = dict(os.environ)

    environment.update({
        "DAT_DATA_ROOT": str(
            phase42_smoke_root
        ),
        "DAT_OUTPUT_CSV": str(
            phase42_extracted_output
        ),
        "DAT_PREPROCESS_WORKERS": "6",
        "DAT_BATCH_SIZE": "6",
        "DAT_DINO_VIEW_BATCH": "12",
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "TORCH_HOME": str(
            phase42_extracted_torch_cache
        ),
        "PYTHONHASHSEED": "0",
        "http_proxy": (
            "http://127.0.0.1:9"
        ),
        "https_proxy": (
            "http://127.0.0.1:9"
        ),
        "HTTP_PROXY": (
            "http://127.0.0.1:9"
        ),
        "HTTPS_PROXY": (
            "http://127.0.0.1:9"
        ),
        "NO_PROXY": "",
        "no_proxy": "",
    })

    environment.pop(
        "DAT_PRIVATE_DIAGNOSTICS",
        None,
    )

    extracted_smoke_started = (
        time.perf_counter()
    )

    phase42_extracted_result = (
        subprocess.run(
            [
                sys.executable,
                "main.py",
            ],
            cwd=str(
                phase42_extracted_package
            ),
            env=environment,
            capture_output=True,
            text=True,
            timeout=(
                PHASE42_ARCHIVE_SMOKE_TIMEOUT
            ),
            check=False,
        )
    )

    phase42_extracted_seconds = (
        time.perf_counter()
        - extracted_smoke_started
    )

    assert (
        phase42_extracted_result.returncode
        == 0
    ), {
        "message": (
            "Extracted Phase42 archive "
            "smoke failed."
        ),
        "return_code": (
            phase42_extracted_result.returncode
        ),
        "stdout_tail": (
            phase42_extracted_result.stdout[
                -2000:
            ]
        ),
        "stderr_tail": (
            phase42_extracted_result.stderr[
                -2000:
            ]
        ),
    }

    assert (
        phase42_extracted_output.is_file()
    )

    phase42_extracted_log = (
        phase42_extracted_result.stdout
        + "\n"
        + phase42_extracted_result.stderr
    )

    phase42_extracted_log_lines = [
        line.strip()
        for line in (
            phase42_extracted_log
            .splitlines()
        )
        if line.strip()
    ]

    assert (
        phase42_extracted_log_lines
        == PHASE42_ALLOWED_PRODUCTION_LINES
    ), {
        "message": (
            "Extracted Phase42 archive "
            "emitted non-generic logs."
        ),
        "observed": (
            phase42_extracted_log_lines
        ),
    }

    phase42_forbidden_patterns = [
        r"\buid\b",
        r"\bpatient\b",
        r"\bcase[_\s-]*id\b",
        r"\bprobabilit(?:y|ies)\b",
        r"\bpreprocessing\s*=",
        r"\bunknown_routes\s*=",
        r"\binitialization\s*=",
        r"\bcase_fallbacks\s*=",
        r"\b\d+\s*/\s*\d+\b",
        r"https?://",
    ]

    phase42_forbidden_hits = [
        pattern
        for pattern in (
            phase42_forbidden_patterns
        )
        if re.search(
            pattern,
            phase42_extracted_log,
            flags=re.IGNORECASE,
        )
    ]

    assert not phase42_forbidden_hits

    phase42_extracted_frame = pd.read_csv(
        phase42_extracted_output,
        dtype={"uid": str},
    )

    assert list(
        phase42_extracted_frame.columns
    ) == [
        "uid",
        "is_pathologic",
    ]
    assert len(
        phase42_extracted_frame
    ) == 20
    assert (
        phase42_extracted_frame["uid"]
        .astype(str)
        .tolist()
        == phase42_reference_smoke_uids
    )

    phase42_extracted_probability = (
        phase42_extracted_frame[
            "is_pathologic"
        ].to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        phase42_extracted_probability
    ).all()
    assert np.all(
        (
            phase42_extracted_probability
            >= 1e-5
        )
        & (
            phase42_extracted_probability
            <= 1.0 - 1e-5
        )
    )
    assert np.ptp(
        phase42_extracted_probability
    ) > 1e-8

    phase42_staged_probability_error = float(
        np.max(
            np.abs(
                phase42_extracted_probability
                - phase42_reference_smoke_probability
            )
        )
    )

    assert (
        phase42_staged_probability_error
        == 0.0
    )

    # Recheck archive after inference.
    with zipfile.ZipFile(
        PHASE42_CANDIDATE_ARCHIVE,
        mode="r",
    ) as archive:
        assert archive.testzip() is None

    phase42_candidate_hash_after = (
        phase42_final_sha256(
            PHASE42_CANDIDATE_ARCHIVE
        )
    )

    assert (
        phase42_candidate_hash_after
        == phase42_candidate_hash_before
    )

    # Publish atomically only after every test passes.
    os.replace(
        PHASE42_CANDIDATE_ARCHIVE,
        PHASE42_FINAL_ARCHIVE,
    )

    assert PHASE42_FINAL_ARCHIVE.is_file()

    phase42_final_hash = (
        phase42_final_sha256(
            PHASE42_FINAL_ARCHIVE
        )
    )

    assert (
        phase42_final_hash
        == phase42_candidate_hash_before
    )

    with zipfile.ZipFile(
        PHASE42_FINAL_ARCHIVE,
        mode="r",
    ) as final_archive:
        assert final_archive.testzip() is None

        final_information = [
            information
            for information in (
                final_archive.infolist()
            )
            if not information.is_dir()
        ]

        assert len(
            final_information
        ) == (
            PHASE42_EXPECTED_ARCHIVE_FILES
        )

        phase42_uncompressed_size_mb = (
            sum(
                information.file_size
                for information in (
                    final_information
                )
            )
            / (1024 ** 2)
        )

finally:
    shutil.rmtree(
        phase42_extract_parent,
        ignore_errors=True,
    )

assert not phase42_extract_parent.exists()
assert not PHASE42_CANDIDATE_ARCHIVE.exists()

phase42_final_report = {
    "phase": (
        "phase42_final_submission_archive"
    ),
    "status": "accepted",
    "zip_name": (
        PHASE42_FINAL_ARCHIVE.name
    ),
    "archive_file_count": (
        PHASE42_EXPECTED_ARCHIVE_FILES
    ),
    "compressed_size_mb": float(
        PHASE42_FINAL_ARCHIVE
        .stat()
        .st_size
        / (1024 ** 2)
    ),
    "uncompressed_size_mb": float(
        phase42_uncompressed_size_mb
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "deterministic_archive_metadata": True,
    "manifest_contract": (
        "accepted_phase40_archive_members_"
        "plus_five_phase42_files"
    ),
    "phase12_model_count": 21,
    "dinov3_checkpoint_count": 1,
    "phase33_model_count": 9,
    "phase36_booster_count": 6,
    "p3ca_state_count": 1,
    "phase42_state_count": 1,
    "route_policy": {
        "known_protocol": (
            "single_held_out_component_fold_"
            "plus_hierarchical_group_alpha"
        ),
        "unknown_protocol": (
            "complete_phase12c_anchor_only"
        ),
        "three_fold_residual_average": False,
    },
    "deployment_formula": (
        "z42 = z_phase12c_full + "
        "alpha_group * clip("
        "0.75*r36_raw + 0.125*r33, "
        "-1, 1)"
    ),
    "extracted_archive_smoke": {
        "status": "accepted",
        "case_count": 20,
        "return_code": 0,
        "elapsed_seconds": float(
            phase42_extracted_seconds
        ),
        "maximum_allowed_seconds": (
            PHASE42_ARCHIVE_SMOKE_TIMEOUT
        ),
        "all_probabilities_finite": True,
        "all_probabilities_clipped": True,
        "output_nonconstant": True,
        "staged_probability_maximum_error": (
            phase42_staged_probability_error
        ),
        "generic_logging_only": True,
        "forbidden_pattern_count": len(
            phase42_forbidden_hits
        ),
        "network_url_present": False,
        "temporary_rows_deleted": True,
    },
    "submission_sha256": (
        phase42_final_hash
    ),
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_oof_arrays": False,
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "model_hashes_displayed": False,
    "total_finalization_seconds": float(
        time.perf_counter()
        - phase42_finalization_started
    ),
}

print(
    "BEGIN PHASE42_EXTRACTED_ARCHIVE_SMOKE_LOG"
)
for line in phase42_extracted_log_lines:
    print(line)
print(
    "END PHASE42_EXTRACTED_ARCHIVE_SMOKE_LOG"
)

print(
    "BEGIN SANITIZED_PHASE42_FINAL_ARCHIVE"
)
print(
    json.dumps(
        phase42_final_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE42_FINAL_ARCHIVE"
)

print(
    "Final Phase42 submission archive:"
)
print(PHASE42_FINAL_ARCHIVE)

BEGIN PHASE42_EXTRACTED_ARCHIVE_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed.
END PHASE42_EXTRACTED_ARCHIVE_SMOKE_LOG
BEGIN SANITIZED_PHASE42_FINAL_ARCHIVE
{
  "phase": "phase42_final_submission_archive",
  "status": "accepted",
  "zip_name": "phase42_submission.zip",
  "archive_file_count": 94,
  "compressed_size_mb": 177.2205638885498,
  "uncompressed_size_mb": 193.3167142868042,
  "root_main_present": true,
  "wrapping_directory_present": false,
  "archive_integrity_test_passed": true,
  "deterministic_archive_metadata": true,
  "manifest_contract": "accepted_phase40_archive_members_plus_five_phase42_files",
  "phase12_model_count": 21,
  "dinov3_checkpoint_count": 1,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "p3ca_state_count": 1,
  "phase42_state_count": 1,
  "route_policy": {
    "known_protocol": "single_held_out_component_fold_plus_hierarchical_group_alpha",
    "unknown_protocol": "complete_phase12c_anchor_only"

In [23]:
# Phase43 Cell 139A
# Diagnose the cross-fitted-anchor versus deployment-anchor mismatch.
#
# The three-fold mean includes models that trained on each case,
# so its label metrics are contaminated and diagnostic only.
# It must not be used for model selection.
#
# No voxel arrays, smoke data, or challenge test data are read.

from itertools import permutations
from pathlib import Path
import json
import time

import numpy as np
import torch
from scipy.stats import spearmanr
from sklearn.metrics import log_loss, roc_auc_score

phase43_anchor_audit_started = (
    time.perf_counter()
)

PHASE43_FOLD_PROBABILITY_PATH = Path(
    "/kaggle/working/"
    "phase32_phase12c_fold_probabilities_float64.npy"
)
PHASE43_CASE_COUNT = 1362
PHASE43_BATCH_SIZE = 128
PHASE43_EPSILON = 1e-5

required_globals = [
    "phase42_runtime",
    "phase42_labels",
    "phase42_groups",
    "phase42_route_folds",
    "phase42_route_exact",
    "phase42_baseline_probability",
    "phase42_dense_cache",
    "phase42_radiomics_cache",
    "phase42_geometry_cache",
    "phase42_runtime_probability",
    "phase42_runtime_r33",
    "phase42_runtime_r36_raw",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

assert not missing_globals, {
    "message": (
        "Run Phase42 Cell 138F before "
        "the Phase43 anchor audit."
    ),
    "missing": missing_globals,
}

assert PHASE43_FOLD_PROBABILITY_PATH.is_file()

phase43_fold_probability = np.load(
    PHASE43_FOLD_PROBABILITY_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

assert phase43_fold_probability.shape == (
    3,
    PHASE43_CASE_COUNT,
)
assert np.isfinite(
    phase43_fold_probability
).all()
assert np.all(
    (phase43_fold_probability > 0.0)
    & (phase43_fold_probability < 1.0)
)

phase43_labels = np.asarray(
    phase42_labels,
    dtype=np.int64,
)
phase43_groups = np.asarray(
    phase42_groups,
    dtype=np.int64,
)
phase43_route_folds = np.asarray(
    phase42_route_folds,
    dtype=np.int64,
)
phase43_cross_fitted_anchor = np.clip(
    np.asarray(
        phase42_baseline_probability,
        dtype=np.float64,
    ),
    PHASE43_EPSILON,
    1.0 - PHASE43_EPSILON,
)

# Resolve how stored probability rows map to
# the accepted routed component-fold numbering.
phase43_permutation_records = []

for permutation in permutations(
    range(3)
):
    reconstructed = np.empty(
        PHASE43_CASE_COUNT,
        dtype=np.float64,
    )

    for route_fold in range(3):
        local = np.flatnonzero(
            phase43_route_folds
            == route_fold
        )

        reconstructed[local] = (
            phase43_fold_probability[
                permutation[route_fold],
                local,
            ]
        )

    difference = np.abs(
        reconstructed
        - phase43_cross_fitted_anchor
    )

    phase43_permutation_records.append({
        "permutation": tuple(
            int(value)
            for value in permutation
        ),
        "maximum_error": float(
            np.max(difference)
        ),
        "mean_error": float(
            np.mean(difference)
        ),
    })

phase43_permutation_records.sort(
    key=lambda record: (
        record["maximum_error"],
        record["mean_error"],
        record["permutation"],
    )
)

phase43_selected_permutation = (
    phase43_permutation_records[0][
        "permutation"
    ]
)
phase43_oof_reconstruction_error = {
    "maximum": (
        phase43_permutation_records[0][
            "maximum_error"
        ]
    ),
    "mean": (
        phase43_permutation_records[0][
            "mean_error"
        ]
    ),
}

assert (
    phase43_oof_reconstruction_error[
        "maximum"
    ]
    <= 1e-12
), {
    "message": (
        "Could not reconstruct the exact "
        "Phase12c cross-fitted anchor."
    ),
    "best_record": (
        phase43_permutation_records[0]
    ),
}

# The arithmetic mean is permutation-invariant.
# It represents the complete three-fold anchor
# used by deployment.
phase43_full_ensemble_anchor = np.clip(
    np.mean(
        np.asarray(
            phase43_fold_probability,
            dtype=np.float64,
        ),
        axis=0,
    ),
    PHASE43_EPSILON,
    1.0 - PHASE43_EPSILON,
)

phase43_cross_fitted_logit = np.log(
    phase43_cross_fitted_anchor
    / (
        1.0
        - phase43_cross_fitted_anchor
    )
)
phase43_full_ensemble_logit = np.log(
    phase43_full_ensemble_anchor
    / (
        1.0
        - phase43_full_ensemble_anchor
    )
)
phase43_anchor_logit_shift = (
    phase43_full_ensemble_logit
    - phase43_cross_fitted_logit
)

# Re-run the exact Phase42 staged components using
# the deployment-style full anchor. This reproduces
# the runtime contract but is evaluated on training
# cases only as a transport diagnostic.
phase43_full_anchor_probability = np.zeros(
    PHASE43_CASE_COUNT,
    dtype=np.float64,
)
phase43_full_anchor_r33 = np.zeros(
    PHASE43_CASE_COUNT,
    dtype=np.float64,
)
phase43_full_anchor_r36 = np.zeros(
    PHASE43_CASE_COUNT,
    dtype=np.float64,
)
phase43_full_anchor_update = np.zeros(
    PHASE43_CASE_COUNT,
    dtype=np.float64,
)
phase43_full_anchor_alpha = np.zeros(
    PHASE43_CASE_COUNT,
    dtype=np.float64,
)

with torch.inference_mode():
    for start in range(
        0,
        PHASE43_CASE_COUNT,
        PHASE43_BATCH_SIZE,
    ):
        stop = min(
            start + PHASE43_BATCH_SIZE,
            PHASE43_CASE_COUNT,
        )

        (
            batch_probability,
            batch_diagnostics,
        ) = (
            phase42_runtime
            .predict_routed_from_dense(
                dense_features=(
                    phase42_dense_cache[
                        start:stop
                    ]
                ),
                radiomics=(
                    phase42_radiomics_cache[
                        start:stop
                    ]
                ),
                geometry_sequence=(
                    phase42_geometry_cache[
                        start:stop
                    ]
                ),
                baseline_probability=(
                    phase43_full_ensemble_anchor[
                        start:stop
                    ]
                ),
                route_groups=(
                    phase43_groups[
                        start:stop
                    ]
                ),
                route_folds=(
                    phase43_route_folds[
                        start:stop
                    ]
                ),
                route_exact=(
                    phase42_route_exact[
                        start:stop
                    ]
                ),
            )
        )

        phase43_full_anchor_probability[
            start:stop
        ] = np.asarray(
            batch_probability,
            dtype=np.float64,
        )
        phase43_full_anchor_r33[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "phase33_residual"
            ],
            dtype=np.float64,
        )
        phase43_full_anchor_r36[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "raw_phase36_residual"
            ],
            dtype=np.float64,
        )
        phase43_full_anchor_update[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "applied_update"
            ],
            dtype=np.float64,
        )
        phase43_full_anchor_alpha[
            start:stop
        ] = np.asarray(
            batch_diagnostics[
                "group_alpha"
            ],
            dtype=np.float64,
        )

        if (
            stop % 384 == 0
            or stop == PHASE43_CASE_COUNT
        ):
            print(
                "Phase43 anchor audit: "
                f"{stop}/{PHASE43_CASE_COUNT}"
            )

assert np.isfinite(
    phase43_full_anchor_probability
).all()

phase43_cross_fitted_component_logit = (
    phase43_cross_fitted_logit
    + np.asarray(
        phase42_runtime_r33,
        dtype=np.float64,
    )
)
phase43_full_anchor_component_logit = (
    phase43_full_ensemble_logit
    + phase43_full_anchor_r33
)

# The Phase33 model output itself must be invariant
# to the anchor; only its residual definition changes.
phase43_phase33_absolute_logit_error = (
    np.abs(
        phase43_full_anchor_component_logit
        - phase43_cross_fitted_component_logit
    )
)

phase43_phase36_residual_shift = (
    phase43_full_anchor_r36
    - np.asarray(
        phase42_runtime_r36_raw,
        dtype=np.float64,
    )
)
phase43_final_probability_shift = (
    phase43_full_anchor_probability
    - np.asarray(
        phase42_runtime_probability,
        dtype=np.float64,
    )
)

assert (
    np.max(
        phase43_phase33_absolute_logit_error
    )
    <= 2e-6
), {
    "message": (
        "Phase33 absolute component logit "
        "changed with the anchor."
    ),
    "maximum_error": float(
        np.max(
            phase43_phase33_absolute_logit_error
        )
    ),
}

assert np.array_equal(
    phase43_full_anchor_alpha,
    np.asarray(
        phase42_runtime_alpha,
        dtype=np.float64,
    ),
)

def phase43_quantile_summary(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "mean": float(
            np.mean(values)
        ),
        "standard_deviation": float(
            np.std(values)
        ),
        "q01": float(
            np.quantile(values, 0.01)
        ),
        "q05": float(
            np.quantile(values, 0.05)
        ),
        "q50": float(
            np.quantile(values, 0.50)
        ),
        "q95": float(
            np.quantile(values, 0.95)
        ),
        "q99": float(
            np.quantile(values, 0.99)
        ),
        "maximum_absolute": float(
            np.max(
                np.abs(values)
            )
        ),
    }


def phase43_binary_metrics(
    probability,
):
    probability = np.asarray(
        probability,
        dtype=np.float64,
    )

    return {
        "log_loss": float(
            log_loss(
                phase43_labels,
                probability,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                phase43_labels,
                probability,
            )
        ),
        "mean_probability": float(
            np.mean(probability)
        ),
    }


phase43_anchor_spearman = float(
    spearmanr(
        phase43_cross_fitted_anchor,
        phase43_full_ensemble_anchor,
    ).statistic
)
phase43_final_spearman = float(
    spearmanr(
        phase42_runtime_probability,
        phase43_full_anchor_probability,
    ).statistic
)

phase43_group_records = []

for group in range(15):
    local = np.flatnonzero(
        phase43_groups == group
    )

    phase43_group_records.append({
        "group": group,
        "n": int(len(local)),
        "mean_absolute_anchor_logit_shift": float(
            np.mean(
                np.abs(
                    phase43_anchor_logit_shift[
                        local
                    ]
                )
            )
        ),
        "q95_absolute_anchor_logit_shift": float(
            np.quantile(
                np.abs(
                    phase43_anchor_logit_shift[
                        local
                    ]
                ),
                0.95,
            )
        ),
        "mean_absolute_phase36_residual_shift": float(
            np.mean(
                np.abs(
                    phase43_phase36_residual_shift[
                        local
                    ]
                )
            )
        ),
        "mean_absolute_final_probability_shift": float(
            np.mean(
                np.abs(
                    phase43_final_probability_shift[
                        local
                    ]
                )
            )
        ),
    })

phase43_anchor_shift_material = bool(
    np.mean(
        np.abs(
            phase43_anchor_logit_shift
        )
    ) >= 0.05
    or np.quantile(
        np.abs(
            phase43_anchor_logit_shift
        ),
        0.95,
    ) >= 0.20
    or np.mean(
        np.abs(
            phase43_phase36_residual_shift
        )
    ) >= 0.03
)

# Private notebook state for the next controlled
# anchor-routing experiment.
PHASE43_FULL_ENSEMBLE_ANCHOR_PRIVATE = (
    phase43_full_ensemble_anchor.copy()
)
PHASE43_FULL_ANCHOR_PHASE42_PRIVATE = (
    phase43_full_anchor_probability.copy()
)
PHASE43_FULL_ANCHOR_R36_PRIVATE = (
    phase43_full_anchor_r36.copy()
)
PHASE43_ANCHOR_LOGIT_SHIFT_PRIVATE = (
    phase43_anchor_logit_shift.copy()
)

phase43_anchor_report = {
    "phase": (
        "phase43_phase12c_deployment_"
        "anchor_transport_audit"
    ),
    "status": "complete",
    "case_count": PHASE43_CASE_COUNT,
    "fold_probability_shape": list(
        phase43_fold_probability.shape
    ),
    "fold_row_to_route_fold_permutation": list(
        phase43_selected_permutation
    ),
    "cross_fitted_reconstruction_error": (
        phase43_oof_reconstruction_error
    ),
    "anchor_shift": {
        "probability_difference": (
            phase43_quantile_summary(
                phase43_full_ensemble_anchor
                - phase43_cross_fitted_anchor
            )
        ),
        "absolute_logit_difference": (
            phase43_quantile_summary(
                np.abs(
                    phase43_anchor_logit_shift
                )
            )
        ),
        "spearman_correlation": (
            phase43_anchor_spearman
        ),
    },
    "component_transport": {
        "phase33_absolute_logit_maximum_error": float(
            np.max(
                phase43_phase33_absolute_logit_error
            )
        ),
        "phase36_raw_residual_difference": (
            phase43_quantile_summary(
                phase43_phase36_residual_shift
            )
        ),
        "final_probability_difference": (
            phase43_quantile_summary(
                phase43_final_probability_shift
            )
        ),
        "final_probability_spearman_correlation": (
            phase43_final_spearman
        ),
    },
    "cross_fitted_reference_metrics": (
        phase43_binary_metrics(
            phase42_runtime_probability
        )
    ),
    "full_ensemble_training_diagnostic": {
        **phase43_binary_metrics(
            phase43_full_anchor_probability
        ),
        "label_leakage_contaminated": True,
        "eligible_for_selection": False,
    },
    "group_transport_records": (
        phase43_group_records
    ),
    "decision": {
        "anchor_shift_material": (
            phase43_anchor_shift_material
        ),
        "next_experiment_if_material": (
            "deployment_matched_anchor_"
            "routing_ablation"
        ),
        "tune_phase42_group_alphas_now": False,
    },
    "interpretation": {
        "full_ensemble_training_metrics_are_honest": False,
        "full_ensemble_predictions_include_in_sample_models": True,
        "public_leaderboard_used_for_parameter_selection": False,
    },
    "labels_used_only_for_contaminated_aggregate_diagnostic": True,
    "training_cached_features_read": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": float(
        time.perf_counter()
        - phase43_anchor_audit_started
    ),
}

print(
    "BEGIN SANITIZED_PHASE43_ANCHOR_AUDIT"
)
print(
    json.dumps(
        phase43_anchor_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE43_ANCHOR_AUDIT"
)

Phase43 anchor audit: 384/1362
Phase43 anchor audit: 768/1362
Phase43 anchor audit: 1152/1362
Phase43 anchor audit: 1362/1362
BEGIN SANITIZED_PHASE43_ANCHOR_AUDIT
{
  "phase": "phase43_phase12c_deployment_anchor_transport_audit",
  "status": "complete",
  "case_count": 1362,
  "fold_probability_shape": [
    3,
    1362
  ],
  "fold_row_to_route_fold_permutation": [
    0,
    1,
    2
  ],
  "cross_fitted_reconstruction_error": {
    "maximum": 0.0,
    "mean": 0.0
  },
  "anchor_shift": {
    "probability_difference": {
      "mean": -0.01018683488788093,
      "standard_deviation": 0.07416893536257416,
      "q01": -0.2468591233574363,
      "q05": -0.15804886820887262,
      "q50": 0.0007685061989067266,
      "q95": 0.08161640153108678,
      "q99": 0.1977326828035248,
      "maximum_absolute": 0.3581929485350849
    },
    "absolute_logit_difference": {
      "mean": 0.531322002994722,
      "standard_deviation": 0.3828046307341335,
      "q01": 0.008876819211298815,
      "q05":

In [24]:
# Phase43 Cell 139B
# Validate that the accepted runtime already contains
# an exact routed Phase12c predictor.
#
# No models, labels, or voxel arrays are read.

from pathlib import Path
import ast
import hashlib
import inspect
import json
import time

import numpy as np

phase43_contract_started = (
    time.perf_counter()
)

PHASE43_PHASE42_ARCHIVE = Path(
    "/kaggle/working/phase42_submission.zip"
)
PHASE43_PHASE42_STAGE = Path(
    "/kaggle/working/phase42_experimental_submission"
)
PHASE43_FOLD_PROBABILITY_PATH = Path(
    "/kaggle/working/"
    "phase32_phase12c_fold_probabilities_float64.npy"
)
PHASE43_EXPECTED_PHASE42_SHA256 = (
    "48cbe2ea01970b073e5d72ca5801882bba27c0385c04713a30c8fad0351fc79e"
)


def phase43_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


assert PHASE43_PHASE42_ARCHIVE.is_file()
assert PHASE43_PHASE42_STAGE.is_dir()
assert PHASE43_FOLD_PROBABILITY_PATH.is_file()

assert (
    phase43_sha256(
        PHASE43_PHASE42_ARCHIVE
    )
    == PHASE43_EXPECTED_PHASE42_SHA256
)

phase43_legacy_path = (
    PHASE43_PHASE42_STAGE
    / "phase30_accepted_main.py"
)
phase43_main_path = (
    PHASE43_PHASE42_STAGE
    / "main.py"
)

assert phase43_legacy_path.is_file()
assert phase43_main_path.is_file()

phase43_legacy_source = (
    phase43_legacy_path.read_text(
        encoding="utf-8"
    )
)
phase43_main_source = (
    phase43_main_path.read_text(
        encoding="utf-8"
    )
)

phase43_legacy_tree = ast.parse(
    phase43_legacy_source,
    filename=str(phase43_legacy_path),
)
phase43_main_tree = ast.parse(
    phase43_main_source,
    filename=str(phase43_main_path),
)

phase43_function_signatures = {}

for node in phase43_legacy_tree.body:
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):
        if node.name in {
            "manifest_for_fold",
            "subset_crops",
            "predict_phase12c_safe",
            "predict_routed_phase12c",
        }:
            phase43_function_signatures[
                node.name
            ] = [
                argument.arg
                for argument in (
                    node.args.args
                )
            ]

assert set(
    phase43_function_signatures
) == {
    "manifest_for_fold",
    "subset_crops",
    "predict_phase12c_safe",
    "predict_routed_phase12c",
}

assert (
    phase43_function_signatures[
        "predict_routed_phase12c"
    ]
    == [
        "crops",
        "groups",
        "router",
        "base_manifest",
        "base_models",
        "family_manifest",
        "anatomy_models",
        "device",
        "fallback",
    ]
)

required_source_fragments = [
    'router["fold_for_group"]',
    "manifest_for_fold(",
    "local_base_manifest",
    "local_family_manifest",
    "family.predict_family_batch(",
    (
        "Header-routing fallback: use "
        "the original complete"
    ),
]

for fragment in required_source_fragments:
    assert fragment in phase43_legacy_source, {
        "message": (
            "Routed Phase12c source "
            "contract changed."
        ),
        "missing_fragment": fragment,
    }

# Phase42 currently uses the complete ensemble.
assert (
    "legacy.predict_phase12c_safe("
    in phase43_main_source
)
assert (
    "legacy.predict_routed_phase12c("
    not in phase43_main_source
)

phase43_fold_probability = np.load(
    PHASE43_FOLD_PROBABILITY_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

assert phase43_fold_probability.shape == (
    3,
    1362,
)

phase43_route_groups = np.asarray(
    phase20_acquisition_groups,
    dtype=np.int64,
)
phase43_route_folds = np.asarray(
    [
        1, 0, 1, 2, 2,
        0, 2, 1, 2, 0,
        2, 1, 1, 1, 2,
    ],
    dtype=np.int64,
)[phase43_route_groups]

phase43_routed_anchor = np.empty(
    1362,
    dtype=np.float64,
)

for fold in range(3):
    local = np.flatnonzero(
        phase43_route_folds == fold
    )

    phase43_routed_anchor[local] = (
        phase43_fold_probability[
            fold,
            local,
        ]
    )

phase43_reference_oof = np.asarray(
    PHASE41_BASELINE_OOF_PRIVATE,
    dtype=np.float64,
)

phase43_routed_anchor_error = {
    "maximum": float(
        np.max(
            np.abs(
                phase43_routed_anchor
                - phase43_reference_oof
            )
        )
    ),
    "mean": float(
        np.mean(
            np.abs(
                phase43_routed_anchor
                - phase43_reference_oof
            )
        )
    ),
}

assert (
    phase43_routed_anchor_error[
        "maximum"
    ]
    == 0.0
)

phase43_full_anchor = np.mean(
    np.asarray(
        phase43_fold_probability,
        dtype=np.float64,
    ),
    axis=0,
)

phase43_full_logit = np.log(
    phase43_full_anchor
    / (1.0 - phase43_full_anchor)
)
phase43_routed_logit = np.log(
    phase43_routed_anchor
    / (1.0 - phase43_routed_anchor)
)

phase43_contract_report = {
    "phase": (
        "phase43_exact_routed_"
        "phase12c_anchor_contract"
    ),
    "status": "accepted",
    "controlled_change": (
        "complete_three_fold_anchor_to_"
        "exact_group_held_out_fold_anchor"
    ),
    "unchanged": [
        "phase33_models",
        "phase36_boosters",
        "p3ca_bases",
        "phase42_group_alphas",
        "phase42_residual_formula",
        "phase42_residual_cap",
        "exact_header_router",
    ],
    "routed_predictor": {
        "function": (
            "predict_routed_phase12c"
        ),
        "signature": (
            phase43_function_signatures[
                "predict_routed_phase12c"
            ]
        ),
        "base_and_anatomy_manifests_filtered_by_fold": True,
        "unknown_group_uses_complete_ensemble": True,
    },
    "training_route_parity": {
        "routed_anchor_error": (
            phase43_routed_anchor_error
        ),
        "fold_route_counts": {
            str(fold): int(
                np.sum(
                    phase43_route_folds
                    == fold
                )
            )
            for fold in range(3)
        },
    },
    "anchor_mismatch": {
        "mean_absolute_logit_shift": float(
            np.mean(
                np.abs(
                    phase43_full_logit
                    - phase43_routed_logit
                )
            )
        ),
        "q95_absolute_logit_shift": float(
            np.quantile(
                np.abs(
                    phase43_full_logit
                    - phase43_routed_logit
                ),
                0.95,
            )
        ),
    },
    "deployment_policy": {
        "known_protocol": (
            "single_group_held_out_"
            "phase12c_fold"
        ),
        "unknown_protocol": (
            "complete_phase12c_ensemble_"
            "and_zero_residual_update"
        ),
    },
    "experiment_interpretation": (
        "single_causal_deployment_ablation"
    ),
    "public_score_used_for_parameter_selection": False,
    "labels_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": float(
        time.perf_counter()
        - phase43_contract_started
    ),
}

print(
    "BEGIN SANITIZED_PHASE43_ROUTED_ANCHOR_CONTRACT"
)
print(
    json.dumps(
        phase43_contract_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE43_ROUTED_ANCHOR_CONTRACT"
)

BEGIN SANITIZED_PHASE43_ROUTED_ANCHOR_CONTRACT
{
  "phase": "phase43_exact_routed_phase12c_anchor_contract",
  "status": "accepted",
  "controlled_change": "complete_three_fold_anchor_to_exact_group_held_out_fold_anchor",
  "unchanged": [
    "phase33_models",
    "phase36_boosters",
    "p3ca_bases",
    "phase42_group_alphas",
    "phase42_residual_formula",
    "phase42_residual_cap",
    "exact_header_router"
  ],
  "routed_predictor": {
    "function": "predict_routed_phase12c",
    "signature": [
      "crops",
      "groups",
      "router",
      "base_manifest",
      "base_models",
      "family_manifest",
      "anatomy_models",
      "device",
      "fallback"
    ],
    "base_and_anatomy_manifests_filtered_by_fold": true,
    "unknown_group_uses_complete_ensemble": true
  },
  "training_route_parity": {
    "routed_anchor_error": {
      "maximum": 0.0,
      "mean": 0.0
    },
    "fold_route_counts": {
      "0": 467,
      "1": 443,
      "2": 452
    }
  },
  "anchor_m

In [25]:
# Phase43 Cell 139C
# Create a new staging directory from the accepted
# Phase42 archive and change only the Phase12c anchor.
#
# Accepted Phase42 archive remains untouched.

from pathlib import Path
import ast
import hashlib
import json
import os
import shutil
import tempfile
import time
import zipfile

phase43_staging_started = (
    time.perf_counter()
)

PHASE43_BASE_ARCHIVE = Path(
    "/kaggle/working/phase42_submission.zip"
)
PHASE43_STAGE_DIRECTORY = Path(
    "/kaggle/working/phase43_experimental_submission"
)
PHASE43_EXPECTED_BASE_SHA256 = (
    "48cbe2ea01970b073e5d72ca5801882bba27c0385c04713a30c8fad0351fc79e"
)
PHASE43_EXPECTED_BASE_FILES = 94
PHASE43_EXPECTED_STAGE_FILES = 96


def phase43_file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def phase43_json_bytes(value):
    return (
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
        )
        + "\n"
    ).encode("utf-8")


def phase43_replace_once(
    source,
    old,
    new,
):
    observed = source.count(old)

    assert observed == 1, {
        "message": (
            "Phase43 exact source "
            "replacement failed."
        ),
        "observed_count": observed,
        "old_prefix": old[:160],
    }

    return source.replace(old, new)


assert PHASE43_BASE_ARCHIVE.is_file()

phase43_base_hash_before = (
    phase43_file_sha256(
        PHASE43_BASE_ARCHIVE
    )
)

assert (
    phase43_base_hash_before
    == PHASE43_EXPECTED_BASE_SHA256
)

with zipfile.ZipFile(
    PHASE43_BASE_ARCHIVE,
    mode="r",
) as archive:
    assert archive.testzip() is None

    phase43_base_members = [
        member
        for member in archive.namelist()
        if not member.endswith("/")
    ]

    assert len(
        phase43_base_members
    ) == PHASE43_EXPECTED_BASE_FILES

phase43_temporary_stage = Path(
    tempfile.mkdtemp(
        prefix="phase43_stage_",
        dir="/kaggle/working",
    )
)

try:
    with zipfile.ZipFile(
        PHASE43_BASE_ARCHIVE,
        mode="r",
    ) as archive:
        for member in archive.namelist():
            path = Path(member)

            assert not path.is_absolute()
            assert ".." not in path.parts

        archive.extractall(
            phase43_temporary_stage
        )

    phase43_main_path = (
        phase43_temporary_stage
        / "main.py"
    )
    phase43_main_source = (
        phase43_main_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        phase43_main_source.count(
            "full_baseline"
        )
        == 3
    )

    phase43_main_source = (
        phase43_main_source.replace(
            "full_baseline",
            "deployment_anchor",
        )
    )

    phase43_old_anchor_call = '''legacy.predict_phase12c_safe(
                crops=crop_batch,
                selected_family_manifest=family_manifest,
                selected_base_manifest=base_manifest,
                base_models=base_models,
                anatomy_models=anatomy_models,
                device=device,
                fallback=fallback,
            )'''

    phase43_new_anchor_call = '''legacy.predict_routed_phase12c(
                crops=crop_batch,
                groups=route_groups,
                router=router,
                base_manifest=base_manifest,
                base_models=base_models,
                family_manifest=family_manifest,
                anatomy_models=anatomy_models,
                device=device,
                fallback=fallback,
            )'''

    phase43_main_source = (
        phase43_replace_once(
            phase43_main_source,
            phase43_old_anchor_call,
            phase43_new_anchor_call,
        )
    )

    assert (
        "legacy.predict_phase12c_safe("
        not in phase43_main_source
    )
    assert (
        phase43_main_source.count(
            "legacy.predict_routed_phase12c("
        )
        == 1
    )
    assert (
        phase43_main_source.count(
            "deployment_anchor"
        )
        == 3
    )
    assert (
        "route_groups" in phase43_main_source
    )

    ast.parse(
        phase43_main_source,
        filename="main.py",
    )

    phase43_main_path.write_text(
        phase43_main_source,
        encoding="utf-8",
    )

    phase43_runtime_contract = {
        "schema_version": 1,
        "phase": (
            "phase43_exact_routed_"
            "phase12c_anchor_runtime"
        ),
        "base_archive": (
            PHASE43_BASE_ARCHIVE.name
        ),
        "base_archive_sha256": (
            PHASE43_EXPECTED_BASE_SHA256
        ),
        "controlled_change": (
            "phase12c_anchor_aggregation_only"
        ),
        "known_protocol_anchor": (
            "single_acquisition_group_"
            "held_out_phase12c_fold"
        ),
        "unknown_protocol_anchor": (
            "complete_three_fold_"
            "phase12c_ensemble"
        ),
        "known_protocol_residual_route": (
            "single_acquisition_group_"
            "held_out_component_fold"
        ),
        "unknown_protocol_residual_update": 0.0,
        "phase42_formula_unchanged": (
            "z = z_anchor + alpha_group * "
            "clip(0.75*r36_raw + "
            "0.125*r33, -1, 1)"
        ),
        "phase42_group_alpha_state_unchanged": True,
        "test_batch_statistics_used": False,
        "network_allowed": False,
        "production_logging": [
            "Built with DINOv3.",
            "Initialization complete.",
            "Inference started.",
            "Inference completed.",
        ],
    }

    phase43_contract_path = (
        phase43_temporary_stage
        / "phase43_runtime_contract.json"
    )
    phase43_contract_path.write_bytes(
        phase43_json_bytes(
            phase43_runtime_contract
        )
    )

    phase43_source_manifest = {
        "schema_version": 1,
        "phase": (
            "phase43_routed_anchor_"
            "runtime_source"
        ),
        "files": {
            "main.py": (
                phase43_file_sha256(
                    phase43_main_path
                )
            ),
            "phase30_accepted_main.py": (
                phase43_file_sha256(
                    phase43_temporary_stage
                    / "phase30_accepted_main.py"
                )
            ),
            "phase42_runtime.py": (
                phase43_file_sha256(
                    phase43_temporary_stage
                    / "phase42_runtime.py"
                )
            ),
            "phase43_runtime_contract.json": (
                phase43_file_sha256(
                    phase43_contract_path
                )
            ),
        },
        "contains_labels": False,
        "contains_predictions": False,
        "contains_case_rows": False,
        "contains_voxel_data": False,
        "contains_embeddings": False,
    }

    (
        phase43_temporary_stage
        / "phase43_runtime_source_manifest.json"
    ).write_bytes(
        phase43_json_bytes(
            phase43_source_manifest
        )
    )

    for python_path in (
        phase43_temporary_stage
        .rglob("*.py")
    ):
        ast.parse(
            python_path.read_text(
                encoding="utf-8"
            ),
            filename=str(python_path),
        )

    phase43_stage_files = [
        path
        for path in (
            phase43_temporary_stage
            .rglob("*")
        )
        if path.is_file()
    ]

    assert len(
        phase43_stage_files
    ) == PHASE43_EXPECTED_STAGE_FILES, {
        "message": (
            "Unexpected clean Phase43 "
            "staging file count."
        ),
        "observed": len(
            phase43_stage_files
        ),
        "expected": (
            PHASE43_EXPECTED_STAGE_FILES
        ),
    }

    if PHASE43_STAGE_DIRECTORY.exists():
        shutil.rmtree(
            PHASE43_STAGE_DIRECTORY
        )

    os.replace(
        phase43_temporary_stage,
        PHASE43_STAGE_DIRECTORY,
    )

except Exception:
    if phase43_temporary_stage.exists():
        shutil.rmtree(
            phase43_temporary_stage,
            ignore_errors=True,
        )
    raise

phase43_base_hash_after = (
    phase43_file_sha256(
        PHASE43_BASE_ARCHIVE
    )
)

assert (
    phase43_base_hash_after
    == phase43_base_hash_before
)

phase43_final_stage_files = [
    path
    for path in (
        PHASE43_STAGE_DIRECTORY
        .rglob("*")
    )
    if path.is_file()
]

phase43_staging_report = {
    "phase": (
        "phase43_exact_routed_"
        "anchor_runtime_staging"
    ),
    "status": "accepted",
    "base_archive": (
        PHASE43_BASE_ARCHIVE.name
    ),
    "staging_directory": (
        PHASE43_STAGE_DIRECTORY.name
    ),
    "package_file_count": len(
        phase43_final_stage_files
    ),
    "package_size_mb": float(
        sum(
            path.stat().st_size
            for path in (
                phase43_final_stage_files
            )
        )
        / (1024 ** 2)
    ),
    "new_files": [
        "phase43_runtime_contract.json",
        "phase43_runtime_source_manifest.json",
    ],
    "controlled_change": {
        "old_anchor": (
            "complete_three_fold_ensemble"
        ),
        "new_known_protocol_anchor": (
            "single_group_held_out_fold"
        ),
        "unknown_protocol_anchor": (
            "complete_three_fold_ensemble"
        ),
    },
    "unchanged": [
        "phase42_runtime",
        "phase42_hierarchical_gate",
        "phase33_models",
        "phase36_boosters",
        "p3ca_bases",
        "dinov3_backbone",
        "preprocessing",
        "production_logging",
    ],
    "accepted_phase42_hash_verified": True,
    "accepted_phase42_archive_modified": False,
    "labels_used": False,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": float(
        time.perf_counter()
        - phase43_staging_started
    ),
}

print(
    "BEGIN SANITIZED_PHASE43_STAGING"
)
print(
    json.dumps(
        phase43_staging_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE43_STAGING"
)
print(PHASE43_STAGE_DIRECTORY)

BEGIN SANITIZED_PHASE43_STAGING
{
  "phase": "phase43_exact_routed_anchor_runtime_staging",
  "status": "accepted",
  "base_archive": "phase42_submission.zip",
  "staging_directory": "phase43_experimental_submission",
  "package_file_count": 96,
  "package_size_mb": 193.31825160980225,
  "new_files": [
    "phase43_runtime_contract.json",
    "phase43_runtime_source_manifest.json"
  ],
  "controlled_change": {
    "old_anchor": "complete_three_fold_ensemble",
    "new_known_protocol_anchor": "single_group_held_out_fold",
    "unknown_protocol_anchor": "complete_three_fold_ensemble"
  },
  "unchanged": [
    "phase42_runtime",
    "phase42_hierarchical_gate",
    "phase33_models",
    "phase36_boosters",
    "p3ca_bases",
    "dinov3_backbone",
    "preprocessing",
    "production_logging"
  ],
  "accepted_phase42_hash_verified": true,
  "accepted_phase42_archive_modified": false,
  "labels_used": false,
  "training_voxel_arrays_read": false,
  "smoke_data_read": false,
  "test_data_rea

In [26]:
# Phase43 Cell 139D
# Real smoke comparison between:
# - Phase42: complete three-fold Phase12c anchor;
# - Phase43: exact group-held-out Phase12c anchor.
#
# Phase43 is run twice:
# 1. private diagnostic mode;
# 2. production generic-logging mode.
#
# No smoke labels or challenge test data are read.

from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

phase43_smoke_started = time.perf_counter()

PHASE43_PACKAGE_DIRECTORY = Path(
    "/kaggle/working/phase43_experimental_submission"
)
PHASE43_SMOKE_TIMEOUT_SECONDS = 360.0
PHASE43_EXPECTED_SMOKE_CASES = 20

PHASE43_ALLOWED_PRODUCTION_LINES = [
    "Built with DINOv3.",
    "Initialization complete.",
    "Inference started.",
    "Inference completed.",
]

required_private_globals = [
    "phase42_smoke_root",
    "phase42_expected_uids",
    "phase42_production_probability",
]

missing_private_globals = [
    name
    for name in required_private_globals
    if name not in globals()
]

assert not missing_private_globals, {
    "message": (
        "Run corrected Phase42 Cell 138G "
        "before Phase43 smoke comparison."
    ),
    "missing": missing_private_globals,
}

assert PHASE43_PACKAGE_DIRECTORY.is_dir()

for relative_path in (
    "main.py",
    "phase30_accepted_main.py",
    "phase42_runtime.py",
    "phase43_runtime_contract.json",
    "phase43_runtime_source_manifest.json",
):
    assert (
        PHASE43_PACKAGE_DIRECTORY
        / relative_path
    ).is_file(), relative_path

phase43_smoke_root = Path(
    phase42_smoke_root
)
phase43_expected_uids = [
    str(uid)
    for uid in phase42_expected_uids
]
phase43_phase42_reference = np.asarray(
    phase42_production_probability,
    dtype=np.float64,
)

assert phase43_smoke_root.is_dir()
assert len(
    phase43_expected_uids
) == PHASE43_EXPECTED_SMOKE_CASES
assert phase43_phase42_reference.shape == (
    PHASE43_EXPECTED_SMOKE_CASES,
)
assert np.isfinite(
    phase43_phase42_reference
).all()

phase43_temporary_root = Path(
    tempfile.mkdtemp(
        prefix="phase43_smoke_",
        dir="/kaggle/working",
    )
)
phase43_diagnostic_output = (
    phase43_temporary_root
    / "diagnostic_submission.csv"
)
phase43_production_output = (
    phase43_temporary_root
    / "production_submission.csv"
)
phase43_torch_cache = (
    phase43_temporary_root
    / "torch_cache"
)


def phase43_run_package(
    output_path,
    diagnostics,
):
    environment = dict(os.environ)

    environment.update({
        "DAT_DATA_ROOT": str(
            phase43_smoke_root
        ),
        "DAT_OUTPUT_CSV": str(
            output_path
        ),
        "DAT_PREPROCESS_WORKERS": "6",
        "DAT_BATCH_SIZE": "6",
        "DAT_DINO_VIEW_BATCH": "12",
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "TORCH_HOME": str(
            phase43_torch_cache
        ),
        "PYTHONHASHSEED": "0",
        "http_proxy": (
            "http://127.0.0.1:9"
        ),
        "https_proxy": (
            "http://127.0.0.1:9"
        ),
        "HTTP_PROXY": (
            "http://127.0.0.1:9"
        ),
        "HTTPS_PROXY": (
            "http://127.0.0.1:9"
        ),
        "NO_PROXY": "",
        "no_proxy": "",
    })

    if diagnostics:
        environment[
            "DAT_PRIVATE_DIAGNOSTICS"
        ] = "1"
    else:
        environment.pop(
            "DAT_PRIVATE_DIAGNOSTICS",
            None,
        )

    started = time.perf_counter()

    result = subprocess.run(
        [
            sys.executable,
            "main.py",
        ],
        cwd=str(
            PHASE43_PACKAGE_DIRECTORY
        ),
        env=environment,
        capture_output=True,
        text=True,
        timeout=(
            PHASE43_SMOKE_TIMEOUT_SECONDS
        ),
        check=False,
    )

    elapsed = (
        time.perf_counter()
        - started
    )
    combined_log = (
        result.stdout
        + "\n"
        + result.stderr
    )

    return result, combined_log, elapsed


(
    phase43_diagnostic_result,
    phase43_diagnostic_log,
    phase43_diagnostic_seconds,
) = phase43_run_package(
    output_path=(
        phase43_diagnostic_output
    ),
    diagnostics=True,
)

assert (
    phase43_diagnostic_result.returncode
    == 0
), {
    "message": (
        "Phase43 diagnostic smoke failed."
    ),
    "return_code": (
        phase43_diagnostic_result.returncode
    ),
    "stdout_tail": (
        phase43_diagnostic_result.stdout[
            -2000:
        ]
    ),
    "stderr_tail": (
        phase43_diagnostic_result.stderr[
            -2000:
        ]
    ),
}

assert phase43_diagnostic_output.is_file()

phase43_diagnostic_match = re.search(
    (
        r"Private diagnostics:\s*"
        r"preprocessing=(\d+),\s*"
        r"unknown_routes=(\d+),\s*"
        r"initialization=(\d+),\s*"
        r"case_fallbacks=(\d+)\."
    ),
    phase43_diagnostic_log,
)

assert phase43_diagnostic_match is not None

(
    phase43_preprocessing_failures,
    phase43_unknown_routes,
    phase43_initialization_fallbacks,
    phase43_case_fallbacks,
) = [
    int(value)
    for value in (
        phase43_diagnostic_match.groups()
    )
]

phase43_known_routes = (
    PHASE43_EXPECTED_SMOKE_CASES
    - phase43_unknown_routes
)

assert phase43_preprocessing_failures == 0
assert phase43_initialization_fallbacks == 0
assert phase43_case_fallbacks == 0
assert phase43_known_routes > 0

(
    phase43_production_result,
    phase43_production_log,
    phase43_production_seconds,
) = phase43_run_package(
    output_path=(
        phase43_production_output
    ),
    diagnostics=False,
)

assert (
    phase43_production_result.returncode
    == 0
), {
    "message": (
        "Phase43 production smoke failed."
    ),
    "return_code": (
        phase43_production_result.returncode
    ),
    "stdout_tail": (
        phase43_production_result.stdout[
            -2000:
        ]
    ),
    "stderr_tail": (
        phase43_production_result.stderr[
            -2000:
        ]
    ),
}

assert phase43_production_output.is_file()

phase43_production_lines = [
    line.strip()
    for line in (
        phase43_production_log
        .splitlines()
    )
    if line.strip()
]

assert phase43_production_lines == (
    PHASE43_ALLOWED_PRODUCTION_LINES
), {
    "message": (
        "Phase43 production log contains "
        "non-generic output."
    ),
    "observed": phase43_production_lines,
}

phase43_forbidden_patterns = [
    r"\buid\b",
    r"\bpatient\b",
    r"\bcase[_\s-]*id\b",
    r"\bprobabilit(?:y|ies)\b",
    r"\bpreprocessing\s*=",
    r"\bunknown_routes\s*=",
    r"\binitialization\s*=",
    r"\bcase_fallbacks\s*=",
    r"\b\d+\s*/\s*\d+\b",
    r"https?://",
]

phase43_forbidden_hits = [
    pattern
    for pattern in (
        phase43_forbidden_patterns
    )
    if re.search(
        pattern,
        phase43_production_log,
        flags=re.IGNORECASE,
    )
]

assert not phase43_forbidden_hits

phase43_diagnostic_frame = pd.read_csv(
    phase43_diagnostic_output,
    dtype={"uid": str},
)
phase43_production_frame = pd.read_csv(
    phase43_production_output,
    dtype={"uid": str},
)

for frame in (
    phase43_diagnostic_frame,
    phase43_production_frame,
):
    assert list(frame.columns) == [
        "uid",
        "is_pathologic",
    ]
    assert len(frame) == (
        PHASE43_EXPECTED_SMOKE_CASES
    )
    assert (
        frame["uid"]
        .astype(str)
        .tolist()
        == phase43_expected_uids
    )

phase43_diagnostic_probability = (
    phase43_diagnostic_frame[
        "is_pathologic"
    ].to_numpy(
        dtype=np.float64
    )
)
phase43_production_probability = (
    phase43_production_frame[
        "is_pathologic"
    ].to_numpy(
        dtype=np.float64
    )
)

assert np.isfinite(
    phase43_diagnostic_probability
).all()
assert np.isfinite(
    phase43_production_probability
).all()
assert np.all(
    (
        phase43_production_probability
        >= 1e-5
    )
    & (
        phase43_production_probability
        <= 1.0 - 1e-5
    )
)
assert np.ptp(
    phase43_production_probability
) > 1e-8

phase43_diagnostic_production_error = float(
    np.max(
        np.abs(
            phase43_diagnostic_probability
            - phase43_production_probability
        )
    )
)

assert (
    phase43_diagnostic_production_error
    == 0.0
)

phase43_vs_phase42_difference = (
    phase43_production_probability
    - phase43_phase42_reference
)
phase43_absolute_difference = np.abs(
    phase43_vs_phase42_difference
)

phase43_phase42_maximum_difference = float(
    np.max(
        phase43_absolute_difference
    )
)
phase43_phase42_mean_difference = float(
    np.mean(
        phase43_absolute_difference
    )
)
phase43_phase42_q50_difference = float(
    np.quantile(
        phase43_absolute_difference,
        0.50,
    )
)
phase43_phase42_q95_difference = float(
    np.quantile(
        phase43_absolute_difference,
        0.95,
    )
)
phase43_phase42_spearman = float(
    spearmanr(
        phase43_phase42_reference,
        phase43_production_probability,
    ).statistic
)

# This must be a real anchor ablation rather
# than an accidental reconstruction of Phase42.
assert (
    phase43_phase42_maximum_difference
    > 1e-4
), {
    "message": (
        "Phase43 did not materially change "
        "the Phase42 smoke predictions."
    ),
    "maximum_difference": (
        phase43_phase42_maximum_difference
    ),
}

phase43_package_files = [
    path
    for path in (
        PHASE43_PACKAGE_DIRECTORY
        .rglob("*")
    )
    if path.is_file()
]

# Retain for extracted-archive parity only.
PHASE43_SMOKE_PROBABILITY_PRIVATE = (
    phase43_production_probability.copy()
)
PHASE43_SMOKE_UIDS_PRIVATE = list(
    phase43_expected_uids
)

phase43_smoke_report = {
    "phase": (
        "phase43_exact_routed_anchor_"
        "real_smoke_ablation"
    ),
    "status": "accepted",
    "package_directory": (
        PHASE43_PACKAGE_DIRECTORY.name
    ),
    "package_file_count": len(
        phase43_package_files
    ),
    "package_size_mb": float(
        sum(
            path.stat().st_size
            for path in (
                phase43_package_files
            )
        )
        / (1024 ** 2)
    ),
    "case_count": (
        PHASE43_EXPECTED_SMOKE_CASES
    ),
    "diagnostic_run": {
        "return_code": 0,
        "elapsed_seconds": float(
            phase43_diagnostic_seconds
        ),
        "preprocessing_failures": (
            phase43_preprocessing_failures
        ),
        "known_routes": (
            phase43_known_routes
        ),
        "unknown_routes": (
            phase43_unknown_routes
        ),
        "initialization_fallbacks": (
            phase43_initialization_fallbacks
        ),
        "case_fallbacks": (
            phase43_case_fallbacks
        ),
        "known_route_branch_exercised": bool(
            phase43_known_routes > 0
        ),
        "unknown_route_source_contract_only": bool(
            phase43_unknown_routes == 0
        ),
    },
    "production_run": {
        "return_code": 0,
        "elapsed_seconds": float(
            phase43_production_seconds
        ),
        "log_line_count": len(
            phase43_production_lines
        ),
        "generic_lifecycle_line_count": 4,
        "forbidden_pattern_count": len(
            phase43_forbidden_hits
        ),
        "generic_logging_only": True,
        "network_url_present": False,
    },
    "diagnostic_production_probability_error": (
        phase43_diagnostic_production_error
    ),
    "phase43_vs_phase42": {
        "maximum_absolute_probability_difference": (
            phase43_phase42_maximum_difference
        ),
        "mean_absolute_probability_difference": (
            phase43_phase42_mean_difference
        ),
        "q50_absolute_probability_difference": (
            phase43_phase42_q50_difference
        ),
        "q95_absolute_probability_difference": (
            phase43_phase42_q95_difference
        ),
        "spearman_correlation": (
            phase43_phase42_spearman
        ),
        "controlled_ablation_exercised": True,
    },
    "schema_valid": True,
    "uid_order_exact": True,
    "all_probabilities_finite": True,
    "all_probabilities_clipped": True,
    "output_nonconstant": True,
    "offline_environment_enforced": True,
    "separate_subprocesses": True,
    "production_logging_contract_passed": True,
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "training_voxel_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "total_cell_seconds": float(
        time.perf_counter()
        - phase43_smoke_started
    ),
}

shutil.rmtree(
    phase43_temporary_root,
    ignore_errors=False,
)

assert not phase43_temporary_root.exists()

print(
    "BEGIN PHASE43_PRODUCTION_SMOKE_LOG"
)
for line in phase43_production_lines:
    print(line)
print(
    "END PHASE43_PRODUCTION_SMOKE_LOG"
)

print(
    "BEGIN SANITIZED_PHASE43_REAL_SMOKE"
)
print(
    json.dumps(
        phase43_smoke_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE43_REAL_SMOKE"
)

BEGIN PHASE43_PRODUCTION_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed.
END PHASE43_PRODUCTION_SMOKE_LOG
BEGIN SANITIZED_PHASE43_REAL_SMOKE
{
  "phase": "phase43_exact_routed_anchor_real_smoke_ablation",
  "status": "accepted",
  "package_directory": "phase43_experimental_submission",
  "package_file_count": 122,
  "package_size_mb": 193.58619213104248,
  "case_count": 20,
  "diagnostic_run": {
    "return_code": 0,
    "elapsed_seconds": 17.244441160999486,
    "preprocessing_failures": 0,
    "known_routes": 20,
    "unknown_routes": 0,
    "initialization_fallbacks": 0,
    "case_fallbacks": 0,
    "known_route_branch_exercised": true,
    "unknown_route_source_contract_only": true
  },
  "production_run": {
    "return_code": 0,
    "elapsed_seconds": 17.176857275999282,
    "log_line_count": 4,
    "generic_lifecycle_line_count": 4,
    "forbidden_pattern_count": 0,
    "generic_logging_only": true,
    "network_url_present": false
  

In [28]:
# Phase43 Cell 139E
# Build a clean deterministic Phase43 archive candidate.
#
# Contract:
# - accepted Phase42 member set;
# - replace only main.py;
# - add two Phase43 contract/manifest files;
# - exclude all smoke-created transient files.

from pathlib import Path
import hashlib
import json
import shutil
import time
import zipfile

phase43_archive_started = time.perf_counter()

PHASE43_BASE_ARCHIVE = Path(
    "/kaggle/working/phase42_submission.zip"
)
PHASE43_STAGE_DIRECTORY = Path(
    "/kaggle/working/phase43_experimental_submission"
)
PHASE43_CANDIDATE_ARCHIVE = Path(
    "/kaggle/working/phase43_submission.candidate.zip"
)

PHASE43_EXPECTED_BASE_SHA256 = (
    "48cbe2ea01970b073e5d72ca5801882bba27c0385c04713a30c8fad0351fc79e"
)
PHASE43_EXPECTED_BASE_FILES = 94
PHASE43_EXPECTED_ARCHIVE_FILES = 96

PHASE43_NEW_MEMBERS = [
    "phase43_runtime_contract.json",
    "phase43_runtime_source_manifest.json",
]


def phase43_archive_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def phase43_member_is_safe(member):
    path = Path(member)

    return (
        bool(member)
        and not member.endswith("/")
        and not path.is_absolute()
        and ".." not in path.parts
        and "\\" not in member
    )


assert PHASE43_BASE_ARCHIVE.is_file()
assert PHASE43_STAGE_DIRECTORY.is_dir()

phase43_base_hash_before = (
    phase43_archive_sha256(
        PHASE43_BASE_ARCHIVE
    )
)

assert (
    phase43_base_hash_before
    == PHASE43_EXPECTED_BASE_SHA256
)

with zipfile.ZipFile(
    PHASE43_BASE_ARCHIVE,
    mode="r",
) as base_archive:
    assert base_archive.testzip() is None

    phase43_base_members = sorted(
        member
        for member in (
            base_archive.namelist()
        )
        if not member.endswith("/")
    )

    assert len(
        phase43_base_members
    ) == PHASE43_EXPECTED_BASE_FILES

    assert all(
        phase43_member_is_safe(member)
        for member in phase43_base_members
    )

    phase43_base_main_bytes = (
        base_archive.read("main.py")
    )

assert not (
    set(PHASE43_NEW_MEMBERS)
    & set(phase43_base_members)
)

phase43_archive_members = sorted(
    phase43_base_members
    + PHASE43_NEW_MEMBERS
)

assert len(
    phase43_archive_members
) == PHASE43_EXPECTED_ARCHIVE_FILES
assert len(
    phase43_archive_members
) == len(
    set(phase43_archive_members)
)

for member in phase43_archive_members:
    assert phase43_member_is_safe(
        member
    )

    assert (
        PHASE43_STAGE_DIRECTORY
        / member
    ).is_file(), member

phase43_new_main_bytes = (
    PHASE43_STAGE_DIRECTORY
    / "main.py"
).read_bytes()

assert (
    phase43_new_main_bytes
    != phase43_base_main_bytes
)

# All inherited files except main.py must remain
# byte-identical to the accepted Phase42 archive.
phase43_unchanged_member_count = 0

with zipfile.ZipFile(
    PHASE43_BASE_ARCHIVE,
    mode="r",
) as base_archive:
    for member in phase43_base_members:
        if member == "main.py":
            continue

        staged_bytes = (
            PHASE43_STAGE_DIRECTORY
            / member
        ).read_bytes()

        assert staged_bytes == (
            base_archive.read(member)
        ), {
            "message": (
                "Inherited Phase42 file changed."
            ),
            "member": member,
        }

        phase43_unchanged_member_count += 1

assert phase43_unchanged_member_count == 93

phase43_source_manifest = json.loads(
    (
        PHASE43_STAGE_DIRECTORY
        / "phase43_runtime_source_manifest.json"
    ).read_text(encoding="utf-8")
)

for relative_path, expected_hash in (
    phase43_source_manifest[
        "files"
    ].items()
):
    source_path = (
        PHASE43_STAGE_DIRECTORY
        / relative_path
    )

    assert source_path.is_file()
    assert (
        phase43_archive_sha256(
            source_path
        )
        == expected_hash
    ), relative_path

phase43_stage_files = [
    path
    for path in (
        PHASE43_STAGE_DIRECTORY.rglob("*")
    )
    if path.is_file()
]

phase43_stage_relative = {
    path.relative_to(
        PHASE43_STAGE_DIRECTORY
    ).as_posix()
    for path in phase43_stage_files
}

phase43_excluded_files = sorted(
    phase43_stage_relative
    - set(phase43_archive_members)
)

for relative_path in phase43_excluded_files:
    lowered = relative_path.lower()

    assert (
        "__pycache__" in lowered
        or lowered.endswith(".pyc")
        or ".pytest_cache" in lowered
        or lowered.endswith(".tmp")
    ), {
        "message": (
            "Unexpected non-manifest staged file."
        ),
        "file": relative_path,
    }

if PHASE43_CANDIDATE_ARCHIVE.exists():
    PHASE43_CANDIDATE_ARCHIVE.unlink()

with zipfile.ZipFile(
    PHASE43_CANDIDATE_ARCHIVE,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
    allowZip64=True,
) as output_archive:
    for member in phase43_archive_members:
        source_path = (
            PHASE43_STAGE_DIRECTORY
            / member
        )

        information = zipfile.ZipInfo(
            filename=member,
            date_time=(
                2020,
                1,
                1,
                0,
                0,
                0,
            ),
        )
        information.compress_type = (
            zipfile.ZIP_DEFLATED
        )
        information.create_system = 3
        information.external_attr = (
            0o100644 << 16
        )
        information.flag_bits |= 0x800

        with source_path.open(
            "rb"
        ) as source_handle:
            with output_archive.open(
                information,
                mode="w",
                force_zip64=True,
            ) as target_handle:
                shutil.copyfileobj(
                    source_handle,
                    target_handle,
                    length=1024 * 1024,
                )

assert PHASE43_CANDIDATE_ARCHIVE.is_file()

with zipfile.ZipFile(
    PHASE43_CANDIDATE_ARCHIVE,
    mode="r",
) as candidate_archive:
    assert candidate_archive.testzip() is None

    candidate_members = sorted(
        member
        for member in (
            candidate_archive.namelist()
        )
        if not member.endswith("/")
    )

    assert candidate_members == (
        phase43_archive_members
    )
    assert len(candidate_members) == (
        PHASE43_EXPECTED_ARCHIVE_FILES
    )
    assert "main.py" in candidate_members
    assert (
        "phase43_runtime_contract.json"
        in candidate_members
    )
    assert (
        "phase43_runtime_source_manifest.json"
        in candidate_members
    )
    assert not any(
        member.startswith(
            "phase43_experimental_submission/"
        )
        for member in candidate_members
    )

    for information in (
        candidate_archive.infolist()
    ):
        if information.is_dir():
            continue

        assert information.date_time == (
            2020,
            1,
            1,
            0,
            0,
            0,
        )

for forbidden_fragment in (
    "__pycache__",
    ".pyc",
    "submission.csv",
    "smoke",
    "oof",
    "prediction",
    "patient",
):
    assert not any(
        forbidden_fragment
        in member.lower()
        for member in (
            phase43_archive_members
        )
    ), forbidden_fragment

phase43_phase12_models = [
    member
    for member in phase43_archive_members
    if (
        member.startswith("models/")
        and member.endswith(".pt")
    )
]
phase43_dinov3_checkpoints = [
    member
    for member in phase43_archive_members
    if (
        "dinov3" in member.lower()
        and member.endswith(".pth")
    )
]
phase43_phase33_models = [
    member
    for member in phase43_archive_members
    if (
        member.startswith(
            "phase39_assets/"
            "phase33_models/"
        )
        and member.endswith(".pt")
    )
]
phase43_phase36_boosters = [
    member
    for member in phase43_archive_members
    if (
        member.startswith(
            "phase39_assets/"
            "phase36_boosters/"
        )
        and member.endswith(".ubj")
    )
]
phase43_p3ca_states = [
    member
    for member in phase43_archive_members
    if member.endswith(
        "phase32_fold_local_p3ca_states.npz"
    )
]

assert len(phase43_phase12_models) == 21
assert len(phase43_dinov3_checkpoints) == 1
assert len(phase43_phase33_models) == 9
assert len(phase43_phase36_boosters) == 6
assert len(phase43_p3ca_states) == 1

phase43_base_hash_after = (
    phase43_archive_sha256(
        PHASE43_BASE_ARCHIVE
    )
)

assert (
    phase43_base_hash_after
    == phase43_base_hash_before
)

phase43_candidate_hash = (
    phase43_archive_sha256(
        PHASE43_CANDIDATE_ARCHIVE
    )
)

phase43_archive_report = {
    "phase": (
        "phase43_deterministic_"
        "archive_candidate_build"
    ),
    "status": (
        "accepted_candidate_not_yet_published"
    ),
    "candidate_name": (
        PHASE43_CANDIDATE_ARCHIVE.name
    ),
    "manifest_contract": (
        "accepted_phase42_members_plus_"
        "two_phase43_files"
    ),
    "phase42_member_count": len(
        phase43_base_members
    ),
    "phase42_unchanged_member_count": (
        phase43_unchanged_member_count
    ),
    "phase43_added_file_count": len(
        PHASE43_NEW_MEMBERS
    ),
    "archive_file_count": len(
        phase43_archive_members
    ),
    "excluded_transient_file_count": len(
        phase43_excluded_files
    ),
    "compressed_size_mb": float(
        PHASE43_CANDIDATE_ARCHIVE
        .stat()
        .st_size
        / (1024 ** 2)
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "deterministic_archive_metadata": True,
    "phase12_model_count": 21,
    "dinov3_checkpoint_count": 1,
    "phase33_model_count": 9,
    "phase36_booster_count": 6,
    "p3ca_state_count": 1,
    "phase42_state_count": 1,
    "candidate_sha256": (
        phase43_candidate_hash
    ),
    "final_archive_published": False,
    "accepted_phase42_archive_modified": False,
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_oof_arrays": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": float(
        time.perf_counter()
        - phase43_archive_started
    ),
}

print(
    "BEGIN SANITIZED_PHASE43_ARCHIVE_BUILD"
)
print(
    json.dumps(
        phase43_archive_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE43_ARCHIVE_BUILD"
)
print(PHASE43_CANDIDATE_ARCHIVE)

BEGIN SANITIZED_PHASE43_ARCHIVE_BUILD
{
  "phase": "phase43_deterministic_archive_candidate_build",
  "status": "accepted_candidate_not_yet_published",
  "candidate_name": "phase43_submission.candidate.zip",
  "manifest_contract": "accepted_phase42_members_plus_two_phase43_files",
  "phase42_member_count": 94,
  "phase42_unchanged_member_count": 93,
  "phase43_added_file_count": 2,
  "archive_file_count": 96,
  "excluded_transient_file_count": 26,
  "compressed_size_mb": 177.221661567688,
  "root_main_present": true,
  "wrapping_directory_present": false,
  "archive_integrity_test_passed": true,
  "deterministic_archive_metadata": true,
  "phase12_model_count": 21,
  "dinov3_checkpoint_count": 1,
  "phase33_model_count": 9,
  "phase36_booster_count": 6,
  "p3ca_state_count": 1,
  "phase42_state_count": 1,
  "candidate_sha256": "36ff8a9d976db69c0b14ac9cbe4a3cac2230a6d42cb45eaecac19431ad94de83",
  "final_archive_published": false,
  "accepted_phase42_archive_modified": false,
  "contains

In [29]:
# Phase43 Cell 139F
# Extract, smoke-test, compare with staged Phase43,
# and publish only after exact parity.

from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

import numpy as np
import pandas as pd

phase43_finalization_started = (
    time.perf_counter()
)

PHASE43_CANDIDATE_ARCHIVE = Path(
    "/kaggle/working/phase43_submission.candidate.zip"
)
PHASE43_FINAL_ARCHIVE = Path(
    "/kaggle/working/phase43_submission.zip"
)
PHASE43_EXPECTED_ARCHIVE_FILES = 96
PHASE43_SMOKE_TIMEOUT_SECONDS = 360.0

PHASE43_ALLOWED_PRODUCTION_LINES = [
    "Built with DINOv3.",
    "Initialization complete.",
    "Inference started.",
    "Inference completed.",
]


def phase43_final_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


required_globals = [
    "phase43_smoke_root",
    "PHASE43_SMOKE_PROBABILITY_PRIVATE",
    "PHASE43_SMOKE_UIDS_PRIVATE",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

assert not missing_globals, {
    "message": (
        "Run Phase43 Cell 139D before "
        "archive publication."
    ),
    "missing": missing_globals,
}

assert PHASE43_CANDIDATE_ARCHIVE.is_file()
assert Path(phase43_smoke_root).is_dir()

phase43_reference_probability = np.asarray(
    PHASE43_SMOKE_PROBABILITY_PRIVATE,
    dtype=np.float64,
)
phase43_reference_uids = [
    str(uid)
    for uid in (
        PHASE43_SMOKE_UIDS_PRIVATE
    )
]

assert phase43_reference_probability.shape == (
    20,
)
assert len(phase43_reference_uids) == 20

phase43_candidate_hash_before = (
    phase43_final_sha256(
        PHASE43_CANDIDATE_ARCHIVE
    )
)

with zipfile.ZipFile(
    PHASE43_CANDIDATE_ARCHIVE,
    mode="r",
) as archive:
    assert archive.testzip() is None

    members = [
        member
        for member in archive.namelist()
        if not member.endswith("/")
    ]

    assert len(members) == (
        PHASE43_EXPECTED_ARCHIVE_FILES
    )
    assert "main.py" in members
    assert (
        "phase43_runtime_contract.json"
        in members
    )

phase43_extract_root = Path(
    tempfile.mkdtemp(
        prefix="phase43_extract_",
        dir="/kaggle/working",
    )
)
phase43_extracted_package = (
    phase43_extract_root / "package"
)
phase43_extracted_package.mkdir()

phase43_output_path = (
    phase43_extract_root
    / "submission.csv"
)
phase43_torch_cache = (
    phase43_extract_root
    / "torch_cache"
)

try:
    with zipfile.ZipFile(
        PHASE43_CANDIDATE_ARCHIVE,
        mode="r",
    ) as archive:
        for member in archive.namelist():
            path = Path(member)

            assert not path.is_absolute()
            assert ".." not in path.parts

        archive.extractall(
            phase43_extracted_package
        )

    environment = dict(os.environ)

    environment.update({
        "DAT_DATA_ROOT": str(
            phase43_smoke_root
        ),
        "DAT_OUTPUT_CSV": str(
            phase43_output_path
        ),
        "DAT_PREPROCESS_WORKERS": "6",
        "DAT_BATCH_SIZE": "6",
        "DAT_DINO_VIEW_BATCH": "12",
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "TORCH_HOME": str(
            phase43_torch_cache
        ),
        "PYTHONHASHSEED": "0",
        "http_proxy": (
            "http://127.0.0.1:9"
        ),
        "https_proxy": (
            "http://127.0.0.1:9"
        ),
        "HTTP_PROXY": (
            "http://127.0.0.1:9"
        ),
        "HTTPS_PROXY": (
            "http://127.0.0.1:9"
        ),
        "NO_PROXY": "",
        "no_proxy": "",
    })

    environment.pop(
        "DAT_PRIVATE_DIAGNOSTICS",
        None,
    )

    extracted_started = time.perf_counter()

    phase43_extracted_result = (
        subprocess.run(
            [
                sys.executable,
                "main.py",
            ],
            cwd=str(
                phase43_extracted_package
            ),
            env=environment,
            capture_output=True,
            text=True,
            timeout=(
                PHASE43_SMOKE_TIMEOUT_SECONDS
            ),
            check=False,
        )
    )

    phase43_extracted_seconds = (
        time.perf_counter()
        - extracted_started
    )

    assert (
        phase43_extracted_result.returncode
        == 0
    ), {
        "message": (
            "Extracted Phase43 smoke failed."
        ),
        "return_code": (
            phase43_extracted_result.returncode
        ),
        "stdout_tail": (
            phase43_extracted_result.stdout[
                -2000:
            ]
        ),
        "stderr_tail": (
            phase43_extracted_result.stderr[
                -2000:
            ]
        ),
    }

    assert phase43_output_path.is_file()

    phase43_extracted_log = (
        phase43_extracted_result.stdout
        + "\n"
        + phase43_extracted_result.stderr
    )

    phase43_extracted_log_lines = [
        line.strip()
        for line in (
            phase43_extracted_log
            .splitlines()
        )
        if line.strip()
    ]

    assert (
        phase43_extracted_log_lines
        == PHASE43_ALLOWED_PRODUCTION_LINES
    )

    forbidden_patterns = [
        r"\buid\b",
        r"\bpatient\b",
        r"\bcase[_\s-]*id\b",
        r"\bprobabilit(?:y|ies)\b",
        r"\bpreprocessing\s*=",
        r"\bunknown_routes\s*=",
        r"\binitialization\s*=",
        r"\bcase_fallbacks\s*=",
        r"\b\d+\s*/\s*\d+\b",
        r"https?://",
    ]

    phase43_forbidden_hits = [
        pattern
        for pattern in forbidden_patterns
        if re.search(
            pattern,
            phase43_extracted_log,
            flags=re.IGNORECASE,
        )
    ]

    assert not phase43_forbidden_hits

    phase43_extracted_frame = pd.read_csv(
        phase43_output_path,
        dtype={"uid": str},
    )

    assert list(
        phase43_extracted_frame.columns
    ) == [
        "uid",
        "is_pathologic",
    ]
    assert len(
        phase43_extracted_frame
    ) == 20
    assert (
        phase43_extracted_frame["uid"]
        .astype(str)
        .tolist()
        == phase43_reference_uids
    )

    phase43_extracted_probability = (
        phase43_extracted_frame[
            "is_pathologic"
        ].to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        phase43_extracted_probability
    ).all()
    assert np.all(
        (
            phase43_extracted_probability
            >= 1e-5
        )
        & (
            phase43_extracted_probability
            <= 1.0 - 1e-5
        )
    )
    assert np.ptp(
        phase43_extracted_probability
    ) > 1e-8

    phase43_staged_probability_error = float(
        np.max(
            np.abs(
                phase43_extracted_probability
                - phase43_reference_probability
            )
        )
    )

    assert (
        phase43_staged_probability_error
        == 0.0
    )

    with zipfile.ZipFile(
        PHASE43_CANDIDATE_ARCHIVE,
        mode="r",
    ) as archive:
        assert archive.testzip() is None

    assert (
        phase43_final_sha256(
            PHASE43_CANDIDATE_ARCHIVE
        )
        == phase43_candidate_hash_before
    )

    os.replace(
        PHASE43_CANDIDATE_ARCHIVE,
        PHASE43_FINAL_ARCHIVE,
    )

    assert PHASE43_FINAL_ARCHIVE.is_file()

    phase43_final_hash = (
        phase43_final_sha256(
            PHASE43_FINAL_ARCHIVE
        )
    )

    assert (
        phase43_final_hash
        == phase43_candidate_hash_before
    )

    with zipfile.ZipFile(
        PHASE43_FINAL_ARCHIVE,
        mode="r",
    ) as archive:
        assert archive.testzip() is None

        final_information = [
            information
            for information in (
                archive.infolist()
            )
            if not information.is_dir()
        ]

        assert len(
            final_information
        ) == (
            PHASE43_EXPECTED_ARCHIVE_FILES
        )

        phase43_uncompressed_size_mb = (
            sum(
                information.file_size
                for information in (
                    final_information
                )
            )
            / (1024 ** 2)
        )

finally:
    shutil.rmtree(
        phase43_extract_root,
        ignore_errors=True,
    )

assert not phase43_extract_root.exists()
assert not PHASE43_CANDIDATE_ARCHIVE.exists()

phase43_final_report = {
    "phase": (
        "phase43_final_submission_archive"
    ),
    "status": "accepted",
    "zip_name": (
        PHASE43_FINAL_ARCHIVE.name
    ),
    "archive_file_count": (
        PHASE43_EXPECTED_ARCHIVE_FILES
    ),
    "compressed_size_mb": float(
        PHASE43_FINAL_ARCHIVE
        .stat()
        .st_size
        / (1024 ** 2)
    ),
    "uncompressed_size_mb": float(
        phase43_uncompressed_size_mb
    ),
    "root_main_present": True,
    "wrapping_directory_present": False,
    "archive_integrity_test_passed": True,
    "deterministic_archive_metadata": True,
    "manifest_contract": (
        "accepted_phase42_members_plus_"
        "two_phase43_files"
    ),
    "controlled_change": (
        "full_phase12c_anchor_to_"
        "exact_routed_phase12c_anchor"
    ),
    "known_protocol_anchor": (
        "single_group_held_out_fold"
    ),
    "unknown_protocol_anchor": (
        "complete_three_fold_ensemble"
    ),
    "phase42_residual_formula_unchanged": True,
    "extracted_archive_smoke": {
        "status": "accepted",
        "case_count": 20,
        "return_code": 0,
        "elapsed_seconds": float(
            phase43_extracted_seconds
        ),
        "maximum_allowed_seconds": (
            PHASE43_SMOKE_TIMEOUT_SECONDS
        ),
        "all_probabilities_finite": True,
        "all_probabilities_clipped": True,
        "output_nonconstant": True,
        "staged_probability_maximum_error": (
            phase43_staged_probability_error
        ),
        "generic_logging_only": True,
        "forbidden_pattern_count": len(
            phase43_forbidden_hits
        ),
        "network_url_present": False,
        "temporary_rows_deleted": True,
    },
    "submission_sha256": (
        phase43_final_hash
    ),
    "contains_voxel_data": False,
    "contains_patient_rows": False,
    "contains_labels": False,
    "contains_predictions": False,
    "contains_embeddings": False,
    "contains_oof_arrays": False,
    "smoke_labels_read": False,
    "smoke_data_read": True,
    "challenge_test_data_read": False,
    "uids_displayed": False,
    "patient_rows_displayed": False,
    "case_level_predictions_displayed": False,
    "case_level_predictions_exported": False,
    "total_finalization_seconds": float(
        time.perf_counter()
        - phase43_finalization_started
    ),
}

print(
    "BEGIN PHASE43_EXTRACTED_ARCHIVE_SMOKE_LOG"
)
for line in phase43_extracted_log_lines:
    print(line)
print(
    "END PHASE43_EXTRACTED_ARCHIVE_SMOKE_LOG"
)

print(
    "BEGIN SANITIZED_PHASE43_FINAL_ARCHIVE"
)
print(
    json.dumps(
        phase43_final_report,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE43_FINAL_ARCHIVE"
)

print(
    "Final Phase43 submission archive:"
)
print(PHASE43_FINAL_ARCHIVE)

BEGIN PHASE43_EXTRACTED_ARCHIVE_SMOKE_LOG
Built with DINOv3.
Initialization complete.
Inference started.
Inference completed.
END PHASE43_EXTRACTED_ARCHIVE_SMOKE_LOG
BEGIN SANITIZED_PHASE43_FINAL_ARCHIVE
{
  "phase": "phase43_final_submission_archive",
  "status": "accepted",
  "zip_name": "phase43_submission.zip",
  "archive_file_count": 96,
  "compressed_size_mb": 177.221661567688,
  "uncompressed_size_mb": 193.31825160980225,
  "root_main_present": true,
  "wrapping_directory_present": false,
  "archive_integrity_test_passed": true,
  "deterministic_archive_metadata": true,
  "manifest_contract": "accepted_phase42_members_plus_two_phase43_files",
  "controlled_change": "full_phase12c_anchor_to_exact_routed_phase12c_anchor",
  "known_protocol_anchor": "single_group_held_out_fold",
  "unknown_protocol_anchor": "complete_three_fold_ensemble",
  "phase42_residual_formula_unchanged": true,
  "extracted_archive_smoke": {
    "status": "accepted",
    "case_count": 20,
    "return_code": 0

> Phase 44

In [32]:
# Phase 44 Cell 140A

import json
import math
import time
from collections import OrderedDict

import numpy as np
import pandas as pd

from scipy.special import expit, logit
from scipy.stats import spearmanr
from sklearn.metrics import log_loss, roc_auc_score


phase44_started = time.perf_counter()
phase44_namespace = globals()
PHASE44_CASE_COUNT = 1362
PHASE44_EPSILON = 1e-7


# ------------------------------------------------------------
# 1. Small compatibility helpers
# ------------------------------------------------------------

def phase44_to_numpy(value):
    if isinstance(value, np.ndarray):
        return value

    try:
        import torch

        if torch.is_tensor(value):
            return value.detach().cpu().numpy()
    except Exception:
        pass

    return np.asarray(value)


def phase44_extract_partition_indices(partition, field_names):
    for field_name in field_names:
        if isinstance(partition, dict) and field_name in partition:
            return np.asarray(
                partition[field_name],
                dtype=np.int64,
            ).reshape(-1)

        if hasattr(partition, field_name):
            return np.asarray(
                getattr(partition, field_name),
                dtype=np.int64,
            ).reshape(-1)

    return None


def phase44_binary_vector(value):
    try:
        array = phase44_to_numpy(value).reshape(-1)
    except Exception:
        return None

    if array.shape != (PHASE44_CASE_COUNT,):
        return None

    if not np.all(np.isfinite(array)):
        return None

    rounded = np.rint(array).astype(np.int64)

    if not np.allclose(array, rounded):
        return None

    if not set(np.unique(rounded)).issubset({0, 1}):
        return None

    if len(np.unique(rounded)) != 2:
        return None

    return rounded


def phase44_integer_group_vector(value):
    try:
        array = phase44_to_numpy(value).reshape(-1)
    except Exception:
        return None

    if array.shape != (PHASE44_CASE_COUNT,):
        return None

    if not np.all(np.isfinite(array)):
        return None

    rounded = np.rint(array).astype(np.int64)

    if not np.allclose(array, rounded):
        return None

    if len(np.unique(rounded)) != 15:
        return None

    return rounded


def phase44_probability_vector(value):
    try:
        shape = tuple(value.shape)
    except Exception:
        try:
            shape = tuple(np.asarray(value).shape)
        except Exception:
            return None

    if shape != (PHASE44_CASE_COUNT,):
        return None

    try:
        array = phase44_to_numpy(value).astype(
            np.float64,
            copy=False,
        ).reshape(-1)
    except Exception:
        return None

    if not np.all(np.isfinite(array)):
        return None

    if np.min(array) < 0.0 or np.max(array) > 1.0:
        return None

    if np.std(array) <= 1e-8:
        return None

    return np.clip(
        array,
        PHASE44_EPSILON,
        1.0 - PHASE44_EPSILON,
    )


# ------------------------------------------------------------
# 2. Resolve labels
# ------------------------------------------------------------

phase44_label_candidates = [
    "PHASE41_LABELS_PRIVATE",
    "PHASE42_LABELS_PRIVATE",
    "PHASE40_LABELS_PRIVATE",
    "labels_numpy",
    "y_numpy",
    "y",
]

phase44_y = None
phase44_label_source = None

for candidate_name in phase44_label_candidates:
    if candidate_name not in phase44_namespace:
        continue

    candidate = phase44_binary_vector(
        phase44_namespace[candidate_name]
    )

    if candidate is not None:
        phase44_y = candidate
        phase44_label_source = candidate_name
        break

if phase44_y is None and "case_df" in phase44_namespace:
    phase44_case_df = phase44_namespace["case_df"]

    for column_name in [
        "label",
        "y",
        "is_pathologic",
        "target",
    ]:
        if column_name not in phase44_case_df.columns:
            continue

        candidate = phase44_binary_vector(
            phase44_case_df[column_name].to_numpy()
        )

        if candidate is not None:
            phase44_y = candidate
            phase44_label_source = (
                f"case_df.{column_name}"
            )
            break

assert phase44_y is not None, {
    "message": "Could not resolve the 1362 binary labels.",
    "checked_globals": phase44_label_candidates,
}


# ------------------------------------------------------------
# 3. Resolve acquisition groups
# ------------------------------------------------------------

phase44_group_candidates = [
    "PHASE41_GROUPS_PRIVATE",
    "PHASE42_GROUPS_PRIVATE",
    "PHASE40_GROUPS_PRIVATE",
    "phase20_acquisition_groups",
    "acquisition_groups",
    "groups_numpy",
    "groups",
]

phase44_groups = None
phase44_group_source = None

for candidate_name in phase44_group_candidates:
    if candidate_name not in phase44_namespace:
        continue

    candidate = phase44_integer_group_vector(
        phase44_namespace[candidate_name]
    )

    if candidate is not None:
        phase44_groups = candidate
        phase44_group_source = candidate_name
        break

if (
    phase44_groups is None
    and "case_df" in phase44_namespace
    and "acquisition_group"
    in phase44_namespace["case_df"].columns
):
    candidate = phase44_integer_group_vector(
        phase44_namespace["case_df"][
            "acquisition_group"
        ].to_numpy()
    )

    if candidate is not None:
        phase44_groups = candidate
        phase44_group_source = (
            "case_df.acquisition_group"
        )

assert phase44_groups is not None, {
    "message": (
        "Could not resolve the 15 acquisition groups."
    ),
    "checked_globals": phase44_group_candidates,
}

phase44_group_sizes = {
    int(group_value): int(
        np.sum(phase44_groups == group_value)
    )
    for group_value in np.unique(phase44_groups)
}

assert sorted(
    phase44_group_sizes.values()
) == sorted([
    39, 456, 145, 255, 76,
    7, 49, 208, 32, 4,
    35, 10, 32, 9, 5,
]), {
    "message": "Acquisition-group sizes changed.",
    "observed": phase44_group_sizes,
}


# ------------------------------------------------------------
# 4. Resolve original Phase19 outer-fold assignment
# ------------------------------------------------------------

phase44_original_fold = None
phase44_fold_source = None

for candidate_name in [
    "PHASE41_OUTER_FOLD_PRIVATE",
    "PHASE40_OUTER_FOLD_PRIVATE",
    "phase19_fold_id",
    "outer_fold_id",
    "fold_id",
]:
    if candidate_name not in phase44_namespace:
        continue

    try:
        candidate = phase44_to_numpy(
            phase44_namespace[candidate_name]
        ).astype(np.int64).reshape(-1)
    except Exception:
        continue

    if (
        candidate.shape == (PHASE44_CASE_COUNT,)
        and set(np.unique(candidate)) == {0, 1, 2}
    ):
        phase44_original_fold = candidate
        phase44_fold_source = candidate_name
        break

if (
    phase44_original_fold is None
    and "case_df" in phase44_namespace
):
    for column_name in ["fold", "outer_fold"]:
        if column_name not in phase44_namespace[
            "case_df"
        ].columns:
            continue

        candidate = np.asarray(
            phase44_namespace["case_df"][
                column_name
            ],
            dtype=np.int64,
        ).reshape(-1)

        if (
            candidate.shape == (PHASE44_CASE_COUNT,)
            and set(np.unique(candidate)) == {0, 1, 2}
        ):
            phase44_original_fold = candidate
            phase44_fold_source = (
                f"case_df.{column_name}"
            )
            break

if (
    phase44_original_fold is None
    and "PHASE19_PARTITIONS" in phase44_namespace
):
    phase44_partitions = phase44_namespace[
        "PHASE19_PARTITIONS"
    ]

    candidate = np.full(
        PHASE44_CASE_COUNT,
        -1,
        dtype=np.int64,
    )

    for fold_index, partition in enumerate(
        phase44_partitions
    ):
        valid_indices = phase44_extract_partition_indices(
            partition,
            [
                "outer_valid",
                "outer_valid_indices",
                "valid",
                "valid_indices",
                "validation_indices",
            ],
        )

        assert valid_indices is not None, {
            "message": (
                "PHASE19_PARTITIONS was found, but an "
                "outer-validation field could not be resolved."
            ),
            "fold": int(fold_index),
        }

        assert np.all(
            (valid_indices >= 0)
            & (valid_indices < PHASE44_CASE_COUNT)
        )

        assert np.all(candidate[valid_indices] == -1), {
            "message": (
                "An outer-validation case appeared in "
                "more than one Phase19 fold."
            ),
            "fold": int(fold_index),
        }

        candidate[valid_indices] = int(fold_index)

    if np.all(candidate >= 0):
        phase44_original_fold = candidate
        phase44_fold_source = (
            "PHASE19_PARTITIONS.outer_valid"
        )

assert phase44_original_fold is not None, {
    "message": (
        "Could not reconstruct the original three "
        "cross-fitting folds."
    )
}

phase44_fold_counts = {
    int(fold_value): int(
        np.sum(phase44_original_fold == fold_value)
    )
    for fold_value in np.unique(
        phase44_original_fold
    )
}

assert phase44_fold_counts == {
    0: 467,
    1: 443,
    2: 452,
}, {
    "message": "Original fold counts changed.",
    "observed": phase44_fold_counts,
}


# ------------------------------------------------------------
# 5. Discover small probability vectors in notebook state
# ------------------------------------------------------------

# Take a frozen namespace snapshot before creating/populating
# the result dictionary. Exclude all variables created by this
# Phase44 cell so repeated execution remains deterministic.
phase44_namespace_snapshot = [
    (name, value)
    for name, value in list(globals().items())
    if not name.startswith("__")
    and not name.startswith("phase44_")
    and not name.startswith("PHASE44_")
]

phase44_probability_candidates = OrderedDict()

def phase44_register_probability(route, value):
    probability = phase44_probability_vector(value)

    if probability is None:
        return

    phase44_probability_candidates[
        str(route)
    ] = probability.copy()

for global_name, global_value in (
    phase44_namespace_snapshot
):
    phase44_register_probability(
        global_name,
        global_value,
    )

    if not isinstance(global_value, dict):
        continue

    # Snapshot every dictionary before inspecting it.
    for key_1, value_1 in list(
        global_value.items()
    ):
        route_1 = f"{global_name}.{key_1}"

        phase44_register_probability(
            route_1,
            value_1,
        )

        if not isinstance(value_1, dict):
            continue

        for key_2, value_2 in list(
            value_1.items()
        ):
            phase44_register_probability(
                f"{route_1}.{key_2}",
                value_2,
            )

assert phase44_probability_candidates, {
    "message": (
        "No finite 1362-case probability vectors "
        "were discovered in notebook state."
    )
}


# ------------------------------------------------------------
# 6. Resolve components by their exact historical metrics
# ------------------------------------------------------------

PHASE44_EXPECTED_COMPONENT_METRICS = OrderedDict([
    (
        "phase12c",
        {
            "log_loss": 0.30761581113729136,
            "auroc": 0.9389144654498752,
            "required": True,
        },
    ),
    (
        "phase31",
        {
            "log_loss": 0.3758200472357325,
            "auroc": 0.9147963126217606,
            "required": False,
        },
    ),
    (
        "phase33",
        {
            "log_loss": 0.35879075331189647,
            "auroc": 0.9235271710146822,
            "required": True,
        },
    ),
    (
        "phase36",
        {
            "log_loss": 0.295657193497576,
            "auroc": 0.9441429675340931,
            "required": True,
        },
    ),
    (
        "phase38",
        {
            "log_loss": 0.30904196751134105,
            "auroc": 0.9378478684385237,
            "required": False,
        },
    ),
    (
        "phase39",
        {
            "log_loss": 0.29302734886541737,
            "auroc": 0.9455752549493367,
            "required": True,
        },
    ),
    (
        "phase42_full_fit_diagnostic",
        {
            "log_loss": 0.29016628416289664,
            "auroc": 0.9464742438589044,
            "required": False,
        },
    ),
])

phase44_components = OrderedDict()
phase44_component_sources = OrderedDict()
phase44_component_metrics = OrderedDict()
phase44_match_diagnostics = OrderedDict()

for component_name, expected in (
    PHASE44_EXPECTED_COMPONENT_METRICS.items()
):
    matches = []

    for route, probability in (
        phase44_probability_candidates.items()
    ):
        candidate_log_loss = float(
            log_loss(
                phase44_y,
                probability,
                labels=[0, 1],
            )
        )
        candidate_auroc = float(
            roc_auc_score(
                phase44_y,
                probability,
            )
        )

        log_loss_error = abs(
            candidate_log_loss
            - expected["log_loss"]
        )
        auroc_error = abs(
            candidate_auroc
            - expected["auroc"]
        )

        if (
            log_loss_error <= 5e-6
            and auroc_error <= 5e-6
        ):
            matches.append({
                "route": route,
                "probability": probability,
                "log_loss": candidate_log_loss,
                "auroc": candidate_auroc,
                "log_loss_error": log_loss_error,
                "auroc_error": auroc_error,
            })

    matches.sort(
        key=lambda record: (
            record["log_loss_error"]
            + record["auroc_error"],
            len(record["route"]),
            record["route"],
        )
    )

    phase44_match_diagnostics[
        component_name
    ] = int(len(matches))

    if matches:
        selected_match = matches[0]

        phase44_components[
            component_name
        ] = selected_match["probability"].copy()

        phase44_component_sources[
            component_name
        ] = selected_match["route"]

        phase44_component_metrics[
            component_name
        ] = {
            "log_loss": selected_match["log_loss"],
            "auroc": selected_match["auroc"],
            "mean_probability": float(
                np.mean(
                    selected_match["probability"]
                )
            ),
        }

    elif expected["required"]:
        raise AssertionError({
            "message": (
                "A required cross-fitted component "
                "could not be recovered by metric parity."
            ),
            "component": component_name,
            "expected": expected,
            "probability_candidate_count": int(
                len(phase44_probability_candidates)
            ),
        })


# ------------------------------------------------------------
# 7. Verify Phase39 reconstruction from primitive residuals
# ------------------------------------------------------------

phase44_z12 = logit(
    np.clip(
        phase44_components["phase12c"],
        PHASE44_EPSILON,
        1.0 - PHASE44_EPSILON,
    )
)

phase44_z33 = logit(
    np.clip(
        phase44_components["phase33"],
        PHASE44_EPSILON,
        1.0 - PHASE44_EPSILON,
    )
)

phase44_z36 = logit(
    np.clip(
        phase44_components["phase36"],
        PHASE44_EPSILON,
        1.0 - PHASE44_EPSILON,
    )
)

phase44_z39 = logit(
    np.clip(
        phase44_components["phase39"],
        PHASE44_EPSILON,
        1.0 - PHASE44_EPSILON,
    )
)

phase44_r33 = phase44_z33 - phase44_z12
phase44_r36_scaled = phase44_z36 - phase44_z12

phase44_phase39_reconstructed = expit(
    phase44_z12
    + np.clip(
        phase44_r36_scaled
        + 0.25 * phase44_r33,
        -2.0,
        2.0,
    )
)

phase44_phase39_reconstruction_error = {
    "maximum": float(
        np.max(
            np.abs(
                phase44_phase39_reconstructed
                - phase44_components["phase39"]
            )
        )
    ),
    "mean": float(
        np.mean(
            np.abs(
                phase44_phase39_reconstructed
                - phase44_components["phase39"]
            )
        )
    ),
}

assert (
    phase44_phase39_reconstruction_error["maximum"]
    <= 5e-7
), {
    "message": "Phase39 formula reconstruction failed.",
    "error": phase44_phase39_reconstruction_error,
}


# ------------------------------------------------------------
# 8. Define candidates for the Phase44 ceiling search
# ------------------------------------------------------------

phase44_search_component_names = [
    component_name
    for component_name in [
        "phase12c",
        "phase31",
        "phase33",
        "phase36",
        "phase38",
        "phase39",
    ]
    if component_name in phase44_components
]

assert len(phase44_search_component_names) >= 4

PHASE44_SIMPLEX_STEP = 0.05
phase44_simplex_units = int(
    round(1.0 / PHASE44_SIMPLEX_STEP)
)

phase44_simplex_candidate_count = math.comb(
    phase44_simplex_units
    + len(phase44_search_component_names)
    - 1,
    len(phase44_search_component_names) - 1,
)


# ------------------------------------------------------------
# 9. Pairwise component diagnostics
# ------------------------------------------------------------

phase44_pairwise_spearman = {}

for left_index, left_name in enumerate(
    phase44_search_component_names
):
    for right_name in (
        phase44_search_component_names[
            left_index + 1:
        ]
    ):
        correlation = float(
            spearmanr(
                phase44_components[left_name],
                phase44_components[right_name],
            ).statistic
        )

        phase44_pairwise_spearman[
            f"{left_name}__{right_name}"
        ] = correlation


# ------------------------------------------------------------
# 10. Retain exact private state for subsequent Phase44 cells
# ------------------------------------------------------------

PHASE44_Y_PRIVATE = phase44_y.copy()
PHASE44_GROUPS_PRIVATE = phase44_groups.copy()
PHASE44_ORIGINAL_FOLD_PRIVATE = (
    phase44_original_fold.copy()
)
PHASE44_COMPONENTS_PRIVATE = OrderedDict(
    (
        name,
        values.copy(),
    )
    for name, values in phase44_components.items()
)
PHASE44_COMPONENT_SOURCES_PRIVATE = dict(
    phase44_component_sources
)
PHASE44_SEARCH_COMPONENT_NAMES_PRIVATE = list(
    phase44_search_component_names
)


# ------------------------------------------------------------
# 11. Sanitized contract
# ------------------------------------------------------------

phase44_contract = {
    "phase": (
        "phase44_existing_component_ceiling_audit_contract"
    ),
    "status": "accepted",
    "case_count": int(PHASE44_CASE_COUNT),
    "normal_count": int(
        np.sum(PHASE44_Y_PRIVATE == 0)
    ),
    "pathologic_count": int(
        np.sum(PHASE44_Y_PRIVATE == 1)
    ),
    "acquisition_group_count": int(
        len(np.unique(PHASE44_GROUPS_PRIVATE))
    ),
    "original_fold_counts": {
        str(key): value
        for key, value in phase44_fold_counts.items()
    },
    "resolved_components": list(
        PHASE44_COMPONENTS_PRIVATE.keys()
    ),
    "component_sources": dict(
        PHASE44_COMPONENT_SOURCES_PRIVATE
    ),
    "component_metrics": {
        name: {
            metric_name: round(metric_value, 9)
            for metric_name, metric_value
            in metrics.items()
        }
        for name, metrics
        in phase44_component_metrics.items()
    },
    "metric_match_counts": dict(
        phase44_match_diagnostics
    ),
    "phase39_reconstruction_error": (
        phase44_phase39_reconstruction_error
    ),
    "search_components": list(
        PHASE44_SEARCH_COMPONENT_NAMES_PRIVATE
    ),
    "planned_search": {
        "representation": (
            "convex_combination_of_component_logits"
        ),
        "simplex_step": PHASE44_SIMPLEX_STEP,
        "simplex_candidate_count": int(
            phase44_simplex_candidate_count
        ),
        "validation": (
            "new_five_fold_within_protocol_nested_holdout"
        ),
        "selection_tracks": [
            "minimum_log_loss_then_auroc_within_band",
            "maximum_auroc_under_log_loss_constraint",
        ],
        "public_leaderboard_used": False,
    },
    "pairwise_spearman_range": {
        "minimum": float(
            min(phase44_pairwise_spearman.values())
        ),
        "maximum": float(
            max(phase44_pairwise_spearman.values())
        ),
    },
    "interpretation": {
        "phase42_full_fit_is_selection_optimistic": (
            "phase42_full_fit_diagnostic"
            in PHASE44_COMPONENTS_PRIVATE
        ),
        "phase42_excluded_from_ceiling_search": True,
        "phase39_is_derived_but_retained_for_nonlinear_cap": (
            True
        ),
        "new_encoder_training_started": False,
    },
    "labels_used_for_metric_parity": True,
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase44_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE44_CEILING_CONTRACT")
print(json.dumps(phase44_contract, indent=2))
print("END SANITIZED_PHASE44_CEILING_CONTRACT")

BEGIN SANITIZED_PHASE44_CEILING_CONTRACT
{
  "phase": "phase44_existing_component_ceiling_audit_contract",
  "status": "accepted",
  "case_count": 1362,
  "normal_count": 615,
  "pathologic_count": 747,
  "acquisition_group_count": 15,
  "original_fold_counts": {
    "0": 467,
    "1": 443,
    "2": 452
  },
  "resolved_components": [
    "phase12c",
    "phase31",
    "phase33",
    "phase36",
    "phase38",
    "phase39",
    "phase42_full_fit_diagnostic"
  ],
  "component_sources": {
    "phase12c": "phase43_reference_oof",
    "phase31": "phase41_components.phase31",
    "phase33": "PHASE33_OOF_PRIVATE",
    "phase36": "PHASE36_OOF_PRIVATE",
    "phase38": "phase41_components.phase38",
    "phase39": "PHASE39_OOF_PRIVATE",
    "phase42_full_fit_diagnostic": "phase42_reference_probability"
  },
  "component_metrics": {
    "phase12c": {
      "log_loss": 0.307615811,
      "auroc": 0.938914465,
      "mean_probability": 0.530697831
    },
    "phase31": {
      "log_loss": 0.3758200

In [34]:
# Cell 140B — Phase44 

import json
import time
from collections import OrderedDict

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.metrics import log_loss, roc_auc_score


phase44_nested_started = time.perf_counter()

required_phase44_state = [
    "PHASE44_Y_PRIVATE",
    "PHASE44_GROUPS_PRIVATE",
    "PHASE44_COMPONENTS_PRIVATE",
    "PHASE44_SEARCH_COMPONENT_NAMES_PRIVATE",
]

missing_phase44_state = [
    name
    for name in required_phase44_state
    if name not in globals()
]

assert not missing_phase44_state, {
    "message": "Cell 140A state is incomplete.",
    "missing": missing_phase44_state,
}


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PHASE44_NESTED_CONFIG = {
    "random_seed": 440602,
    "outer_fold_count": 5,
    "simplex_step": 0.05,
    "candidate_block_size": 768,
    "log_loss_band": 0.001,
    "auc_track_maximum_log_loss_excess_vs_phase39": 0.003,
    "maximum_major_group_harm": 0.015,
    "major_group_minimum_n": 30,
    "exact_auc_shortlist_size": 512,
    "calibration_intercept_bound": 2.0,
    "calibration_slope_range": [0.5, 2.0],
    "calibration_regularization": 1e-4,
}

phase44_y = np.asarray(
    PHASE44_Y_PRIVATE,
    dtype=np.int64,
).reshape(-1)

phase44_groups = np.asarray(
    PHASE44_GROUPS_PRIVATE,
    dtype=np.int64,
).reshape(-1)

phase44_component_names = list(
    PHASE44_SEARCH_COMPONENT_NAMES_PRIVATE
)

phase44_component_probabilities = np.column_stack([
    np.asarray(
        PHASE44_COMPONENTS_PRIVATE[name],
        dtype=np.float64,
    )
    for name in phase44_component_names
])

phase44_component_logits = logit(
    np.clip(
        phase44_component_probabilities,
        1e-7,
        1.0 - 1e-7,
    )
)

phase44_case_count = len(phase44_y)
phase44_component_count = len(
    phase44_component_names
)

assert phase44_component_probabilities.shape == (
    1362,
    phase44_component_count,
)

assert np.all(
    np.isfinite(phase44_component_logits)
)

assert "phase12c" in phase44_component_names
assert "phase39" in phase44_component_names

phase44_phase12c_column = (
    phase44_component_names.index("phase12c")
)
phase44_phase39_column = (
    phase44_component_names.index("phase39")
)


# ------------------------------------------------------------
# 2. New deterministic within-protocol five-fold partition
# ------------------------------------------------------------

phase44_rng = np.random.default_rng(
    PHASE44_NESTED_CONFIG["random_seed"]
)

phase44_meta_fold = np.full(
    phase44_case_count,
    -1,
    dtype=np.int64,
)

phase44_strata = []

for group_value in sorted(
    np.unique(phase44_groups).tolist()
):
    for label_value in [0, 1]:
        indices = np.flatnonzero(
            (phase44_groups == group_value)
            & (phase44_y == label_value)
        )

        if len(indices):
            phase44_strata.append(
                (
                    -len(indices),
                    int(group_value),
                    int(label_value),
                    indices,
                )
            )

# Assign large strata first. Every stratum is distributed
# round-robin across a randomly permuted fold order.
for (
    _,
    group_value,
    label_value,
    indices,
) in sorted(
    phase44_strata,
    key=lambda item: (
        item[0],
        item[1],
        item[2],
    ),
):
    shuffled_indices = phase44_rng.permutation(
        indices
    )
    fold_order = phase44_rng.permutation(
        PHASE44_NESTED_CONFIG[
            "outer_fold_count"
        ]
    )

    assignments = fold_order[
        np.arange(len(shuffled_indices))
        % PHASE44_NESTED_CONFIG[
            "outer_fold_count"
        ]
    ]

    phase44_meta_fold[
        shuffled_indices
    ] = assignments

assert np.all(phase44_meta_fold >= 0)
assert set(np.unique(phase44_meta_fold)) == {
    0, 1, 2, 3, 4
}

phase44_partition_records = []

for fold_value in range(
    PHASE44_NESTED_CONFIG["outer_fold_count"]
):
    valid_mask = (
        phase44_meta_fold == fold_value
    )
    train_mask = ~valid_mask

    assert np.sum(valid_mask) > 0
    assert len(np.unique(
        phase44_y[valid_mask]
    )) == 2

    valid_groups = set(
        np.unique(
            phase44_groups[valid_mask]
        ).tolist()
    )
    train_groups = set(
        np.unique(
            phase44_groups[train_mask]
        ).tolist()
    )

    assert valid_groups.issubset(train_groups)

    phase44_partition_records.append({
        "fold": int(fold_value),
        "train_n": int(np.sum(train_mask)),
        "valid_n": int(np.sum(valid_mask)),
        "train_group_count": int(
            len(train_groups)
        ),
        "valid_group_count": int(
            len(valid_groups)
        ),
        "shared_group_count": int(
            len(valid_groups & train_groups)
        ),
        "valid_prevalence": float(
            np.mean(phase44_y[valid_mask])
        ),
    })


# ------------------------------------------------------------
# 3. Generate every simplex weight at 0.05 resolution
# ------------------------------------------------------------

phase44_simplex_units = int(
    round(
        1.0
        / PHASE44_NESTED_CONFIG["simplex_step"]
    )
)

assert np.isclose(
    phase44_simplex_units
    * PHASE44_NESTED_CONFIG["simplex_step"],
    1.0,
)


def phase44_integer_compositions(
    total,
    part_count,
    prefix=(),
):
    if part_count == 1:
        yield prefix + (total,)
        return

    for first_value in range(total + 1):
        yield from phase44_integer_compositions(
            total=total - first_value,
            part_count=part_count - 1,
            prefix=prefix + (first_value,),
        )


phase44_weight_units = np.asarray(
    list(
        phase44_integer_compositions(
            phase44_simplex_units,
            phase44_component_count,
        )
    ),
    dtype=np.int16,
)

phase44_weight_grid = (
    phase44_weight_units.astype(np.float64)
    / float(phase44_simplex_units)
)

del phase44_weight_units

assert phase44_weight_grid.shape == (
    53130,
    phase44_component_count,
), {
    "message": "Unexpected simplex-grid shape.",
    "observed": list(
        phase44_weight_grid.shape
    ),
    "component_names": phase44_component_names,
}

assert np.allclose(
    np.sum(phase44_weight_grid, axis=1),
    1.0,
)

assert np.all(phase44_weight_grid >= 0.0)


# ------------------------------------------------------------
# 4. Metric helpers
# ------------------------------------------------------------

def phase44_metrics(y_true, probability):
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        1e-7,
        1.0 - 1e-7,
    )

    return {
        "n": int(len(y_true)),
        "log_loss": float(
            log_loss(
                y_true,
                probability,
                labels=[0, 1],
            )
        ),
        "auroc": float(
            roc_auc_score(
                y_true,
                probability,
            )
        ),
        "mean_probability": float(
            np.mean(probability)
        ),
        "prevalence": float(
            np.mean(y_true)
        ),
    }


def phase44_exact_auc_for_weight(
    y_true,
    logit_matrix,
    weight,
):
    score = (
        logit_matrix
        @ np.asarray(weight, dtype=np.float64)
    )

    return float(
        roc_auc_score(y_true, score)
    )


def phase44_fast_auc_columns(
    y_true,
    score_matrix,
):
    """
    Fast AUC for continuous score columns.

    The component logits are continuous. Exact sklearn AUC is
    recomputed for all shortlisted/selected candidates.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    observation_count = len(y_true)
    positive_count = int(np.sum(y_true == 1))
    negative_count = int(np.sum(y_true == 0))

    assert positive_count > 0
    assert negative_count > 0

    order = np.argsort(
        score_matrix,
        axis=0,
        kind="mergesort",
    )

    ranks = np.empty(
        order.shape,
        dtype=np.int32,
    )

    rank_values = np.broadcast_to(
        np.arange(
            observation_count,
            dtype=np.int32,
        )[:, None],
        order.shape,
    )

    np.put_along_axis(
        ranks,
        order,
        rank_values,
        axis=0,
    )

    positive_rank_sum = np.sum(
        ranks[y_true == 1],
        axis=0,
        dtype=np.float64,
    )

    return (
        positive_rank_sum
        - positive_count
        * (positive_count - 1)
        / 2.0
    ) / (
        positive_count * negative_count
    )


def phase44_fit_monotonic_calibrator(
    score,
    y_true,
):
    score = np.asarray(
        score,
        dtype=np.float64,
    )
    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    )

    regularization = PHASE44_NESTED_CONFIG[
        "calibration_regularization"
    ]

    minimum_slope = PHASE44_NESTED_CONFIG[
        "calibration_slope_range"
    ][0]

    maximum_slope = PHASE44_NESTED_CONFIG[
        "calibration_slope_range"
    ][1]

    def objective(parameters):
        intercept = parameters[0]
        log_slope = parameters[1]
        slope = np.exp(log_slope)

        calibrated_score = (
            intercept + slope * score
        )

        negative_log_likelihood = np.mean(
            np.logaddexp(
                0.0,
                calibrated_score,
            )
            - y_true * calibrated_score
        )

        penalty = regularization * (
            intercept ** 2
            + log_slope ** 2
        )

        return float(
            negative_log_likelihood + penalty
        )

    result = minimize(
        objective,
        x0=np.asarray(
            [0.0, 0.0],
            dtype=np.float64,
        ),
        method="L-BFGS-B",
        bounds=[
            (
                -PHASE44_NESTED_CONFIG[
                    "calibration_intercept_bound"
                ],
                PHASE44_NESTED_CONFIG[
                    "calibration_intercept_bound"
                ],
            ),
            (
                float(np.log(minimum_slope)),
                float(np.log(maximum_slope)),
            ),
        ],
    )

    if not result.success:
        return {
            "intercept": 0.0,
            "slope": 1.0,
            "optimizer_success": False,
        }

    return {
        "intercept": float(result.x[0]),
        "slope": float(np.exp(result.x[1])),
        "optimizer_success": True,
    }


# ------------------------------------------------------------
# 5. Evaluate the complete grid on one training partition
# ------------------------------------------------------------

phase44_major_groups = np.asarray([
    group_value
    for group_value in np.unique(
        phase44_groups
    )
    if np.sum(
        phase44_groups == group_value
    ) >= PHASE44_NESTED_CONFIG[
        "major_group_minimum_n"
    ]
], dtype=np.int64)


def phase44_evaluate_complete_grid(
    train_indices,
):
    train_indices = np.asarray(
        train_indices,
        dtype=np.int64,
    )

    y_train = phase44_y[train_indices]
    groups_train = phase44_groups[
        train_indices
    ]
    logits_train = phase44_component_logits[
        train_indices
    ]

    candidate_count = len(
        phase44_weight_grid
    )

    overall_log_loss = np.empty(
        candidate_count,
        dtype=np.float64,
    )
    approximate_auroc = np.empty(
        candidate_count,
        dtype=np.float64,
    )
    maximum_major_group_harm = np.full(
        candidate_count,
        -np.inf,
        dtype=np.float64,
    )

    phase12_train_score = logits_train[
        :,
        phase44_phase12c_column,
    ]

    group_baseline_losses = {}

    for group_value in phase44_major_groups:
        group_mask = (
            groups_train == group_value
        )

        if np.sum(group_mask) == 0:
            continue

        group_baseline_losses[
            int(group_value)
        ] = float(
            np.mean(
                np.logaddexp(
                    0.0,
                    phase12_train_score[
                        group_mask
                    ],
                )
                - y_train[group_mask]
                * phase12_train_score[
                    group_mask
                ]
            )
        )

    block_size = PHASE44_NESTED_CONFIG[
        "candidate_block_size"
    ]

    for block_start in range(
        0,
        candidate_count,
        block_size,
    ):
        block_end = min(
            block_start + block_size,
            candidate_count,
        )

        block_weights = phase44_weight_grid[
            block_start:block_end
        ]

        block_scores = (
            logits_train
            @ block_weights.T
        )

        block_case_losses = (
            np.logaddexp(0.0, block_scores)
            - y_train[:, None] * block_scores
        )

        overall_log_loss[
            block_start:block_end
        ] = np.mean(
            block_case_losses,
            axis=0,
        )

        approximate_auroc[
            block_start:block_end
        ] = phase44_fast_auc_columns(
            y_train,
            block_scores,
        )

        block_maximum_harm = np.full(
            block_end - block_start,
            -np.inf,
            dtype=np.float64,
        )

        for (
            group_value,
            baseline_group_loss,
        ) in group_baseline_losses.items():
            group_mask = (
                groups_train == group_value
            )

            candidate_group_loss = np.mean(
                block_case_losses[group_mask],
                axis=0,
            )

            block_maximum_harm = np.maximum(
                block_maximum_harm,
                candidate_group_loss
                - baseline_group_loss,
            )

        maximum_major_group_harm[
            block_start:block_end
        ] = block_maximum_harm

    return {
        "overall_log_loss": overall_log_loss,
        "approximate_auroc": approximate_auroc,
        "maximum_major_group_harm": (
            maximum_major_group_harm
        ),
    }


# ------------------------------------------------------------
# 6. Exact shortlist selection
# ------------------------------------------------------------

def phase44_select_exact_auc_best(
    eligible_mask,
    grid_metrics,
    train_indices,
):
    eligible_indices = np.flatnonzero(
        eligible_mask
    )

    assert len(eligible_indices) > 0

    approximate_auc = grid_metrics[
        "approximate_auroc"
    ]

    shortlist_size = min(
        PHASE44_NESTED_CONFIG[
            "exact_auc_shortlist_size"
        ],
        len(eligible_indices),
    )

    shortlist_order = np.argsort(
        -approximate_auc[eligible_indices],
        kind="mergesort",
    )[:shortlist_size]

    shortlist_indices = eligible_indices[
        shortlist_order
    ]

    y_train = phase44_y[train_indices]
    logits_train = phase44_component_logits[
        train_indices
    ]

    exact_records = []

    for candidate_index in shortlist_indices:
        exact_auc = (
            phase44_exact_auc_for_weight(
                y_true=y_train,
                logit_matrix=logits_train,
                weight=phase44_weight_grid[
                    candidate_index
                ],
            )
        )

        exact_records.append((
            -exact_auc,
            grid_metrics[
                "overall_log_loss"
            ][candidate_index],
            int(candidate_index),
            exact_auc,
        ))

    exact_records.sort()

    selected_record = exact_records[0]

    return {
        "candidate_index": int(
            selected_record[2]
        ),
        "exact_auroc": float(
            selected_record[3]
        ),
        "eligible_candidate_count": int(
            len(eligible_indices)
        ),
    }


def phase44_weight_dictionary(weight):
    return {
        component_name: float(weight[index])
        for index, component_name
        in enumerate(phase44_component_names)
        if weight[index] > 0.0
    }


# ------------------------------------------------------------
# 7. Fully nested five-fold evaluation
# ------------------------------------------------------------

phase44_track_names = [
    "log_loss_first",
    "auroc_constrained",
    "pure_auroc_diagnostic",
]

phase44_oof_raw = {
    track_name: np.full(
        phase44_case_count,
        np.nan,
        dtype=np.float64,
    )
    for track_name in phase44_track_names
}

phase44_oof_calibrated = {
    track_name: np.full(
        phase44_case_count,
        np.nan,
        dtype=np.float64,
    )
    for track_name in phase44_track_names
}

phase44_fold_records = []

for fold_value in range(
    PHASE44_NESTED_CONFIG["outer_fold_count"]
):
    fold_started = time.perf_counter()

    valid_indices = np.flatnonzero(
        phase44_meta_fold == fold_value
    )
    train_indices = np.flatnonzero(
        phase44_meta_fold != fold_value
    )

    grid_metrics = (
        phase44_evaluate_complete_grid(
            train_indices
        )
    )

    train_phase39_metrics = phase44_metrics(
        phase44_y[train_indices],
        phase44_component_probabilities[
            train_indices,
            phase44_phase39_column,
        ],
    )

    # First establish deterministic group safety.
    group_safe_mask = (
        grid_metrics[
            "maximum_major_group_harm"
        ]
        <= PHASE44_NESTED_CONFIG[
            "maximum_major_group_harm"
        ]
    )

    assert np.any(group_safe_mask), {
        "message": (
            "No group-safe simplex candidate exists."
        ),
        "fold": int(fold_value),
        "minimum_observed_group_harm": float(
            np.min(
                grid_metrics[
                    "maximum_major_group_harm"
                ]
            )
        ),
    }

    # The log-loss band must begin at the best eligible
    # candidate—not at the unconstrained global minimum.
    minimum_grid_log_loss = float(
        np.min(
            grid_metrics[
                "overall_log_loss"
            ][group_safe_mask]
        )
    )

    log_loss_band_mask = (
        group_safe_mask
        & (
            grid_metrics[
                "overall_log_loss"
            ]
            <= minimum_grid_log_loss
            + PHASE44_NESTED_CONFIG[
                "log_loss_band"
            ]
        )
    )

    # Preserve the requested Phase39 constraint whenever
    # feasible. If it is stricter than the best group-safe
    # candidate, minimally expand it to include the safe
    # log-loss band.
    requested_auc_log_loss_limit = (
        train_phase39_metrics["log_loss"]
        + PHASE44_NESTED_CONFIG[
            "auc_track_maximum_log_loss_excess_vs_phase39"
        ]
    )

    effective_auc_log_loss_limit = max(
        requested_auc_log_loss_limit,
        minimum_grid_log_loss
        + PHASE44_NESTED_CONFIG[
            "log_loss_band"
        ],
    )

    auroc_constraint_mask = (
        group_safe_mask
        & (
            grid_metrics[
                "overall_log_loss"
            ]
            <= effective_auc_log_loss_limit
        )
    )

    pure_auroc_mask = np.ones(
        len(phase44_weight_grid),
        dtype=bool,
    )

    assert np.any(log_loss_band_mask), {
        "message": (
            "No group-safe candidate remained in "
            "the log-loss band."
        ),
        "fold": int(fold_value),
    }

    assert np.any(auroc_constraint_mask), {
        "message": (
            "No candidate remained for the "
            "AUROC-constrained track."
        ),
        "fold": int(fold_value),
    }

    selection_masks = {
        "log_loss_first": log_loss_band_mask,
        "auroc_constrained": auroc_constraint_mask,
        "pure_auroc_diagnostic": pure_auroc_mask,
    }

    fold_track_records = {}

    for track_name, eligible_mask in (
        selection_masks.items()
    ):
        selection = (
            phase44_select_exact_auc_best(
                eligible_mask=eligible_mask,
                grid_metrics=grid_metrics,
                train_indices=train_indices,
            )
        )

        candidate_index = selection[
            "candidate_index"
        ]

        selected_weight = (
            phase44_weight_grid[
                candidate_index
            ]
        )

        train_score = (
            phase44_component_logits[
                train_indices
            ]
            @ selected_weight
        )

        valid_score = (
            phase44_component_logits[
                valid_indices
            ]
            @ selected_weight
        )

        calibrator = (
            phase44_fit_monotonic_calibrator(
                train_score,
                phase44_y[train_indices],
            )
        )

        train_raw_probability = expit(
            train_score
        )
        valid_raw_probability = expit(
            valid_score
        )

        valid_calibrated_probability = expit(
            calibrator["intercept"]
            + calibrator["slope"]
            * valid_score
        )

        phase44_oof_raw[track_name][
            valid_indices
        ] = valid_raw_probability

        phase44_oof_calibrated[track_name][
            valid_indices
        ] = valid_calibrated_probability

        fold_track_records[track_name] = {
            "candidate_index": int(
                candidate_index
            ),
            "weights": (
                phase44_weight_dictionary(
                    selected_weight
                )
            ),
            "eligible_candidate_count": (
                selection[
                    "eligible_candidate_count"
                ]
            ),
            "train": {
                "log_loss": float(
                    grid_metrics[
                        "overall_log_loss"
                    ][candidate_index]
                ),
                "auroc": float(
                    selection["exact_auroc"]
                ),
                "maximum_major_group_harm": (
                    float(
                        grid_metrics[
                            "maximum_major_group_harm"
                        ][candidate_index]
                    )
                ),
            },
            "calibrator": calibrator,
            "outer_valid_raw": (
                phase44_metrics(
                    phase44_y[valid_indices],
                    valid_raw_probability,
                )
            ),
            "outer_valid_calibrated": (
                phase44_metrics(
                    phase44_y[valid_indices],
                    valid_calibrated_probability,
                )
            ),
        }

    phase44_fold_records.append({
        "fold": int(fold_value),
        "train_n": int(len(train_indices)),
        "valid_n": int(len(valid_indices)),
        "tracks": fold_track_records,
        "elapsed_seconds": round(
            time.perf_counter() - fold_started,
            3,
        ),
    })

    print(
        "Phase44 nested fold "
        f"{fold_value}: "
        f"{len(phase44_weight_grid)}/"
        f"{len(phase44_weight_grid)} "
        "weights evaluated"
    )


# ------------------------------------------------------------
# 8. Aggregate honest nested metrics
# ------------------------------------------------------------

for track_name in phase44_track_names:
    assert np.all(
        np.isfinite(
            phase44_oof_raw[track_name]
        )
    )
    assert np.all(
        np.isfinite(
            phase44_oof_calibrated[
                track_name
            ]
        )
    )

phase44_reference_metrics = {
    component_name: phase44_metrics(
        phase44_y,
        PHASE44_COMPONENTS_PRIVATE[
            component_name
        ],
    )
    for component_name in [
        "phase12c",
        "phase36",
        "phase39",
    ]
}

phase44_nested_track_metrics = OrderedDict()

for track_name in phase44_track_names:
    raw_metrics = phase44_metrics(
        phase44_y,
        phase44_oof_raw[track_name],
    )

    calibrated_metrics = phase44_metrics(
        phase44_y,
        phase44_oof_calibrated[
            track_name
        ],
    )

    major_group_records = []

    for group_value in phase44_major_groups:
        group_mask = (
            phase44_groups == group_value
        )

        baseline_group_metrics = (
            phase44_metrics(
                phase44_y[group_mask],
                PHASE44_COMPONENTS_PRIVATE[
                    "phase12c"
                ][group_mask],
            )
        )

        candidate_group_metrics = (
            phase44_metrics(
                phase44_y[group_mask],
                phase44_oof_calibrated[
                    track_name
                ][group_mask],
            )
        )

        major_group_records.append({
            "group": int(group_value),
            "n": int(np.sum(group_mask)),
            "log_loss_gain": float(
                baseline_group_metrics[
                    "log_loss"
                ]
                - candidate_group_metrics[
                    "log_loss"
                ]
            ),
            "auroc_gain": float(
                candidate_group_metrics[
                    "auroc"
                ]
                - baseline_group_metrics[
                    "auroc"
                ]
            ),
        })

    phase44_nested_track_metrics[
        track_name
    ] = {
        "raw": raw_metrics,
        "calibrated": calibrated_metrics,
        "improvements": {
            "log_loss_gain_over_phase12c": (
                phase44_reference_metrics[
                    "phase12c"
                ]["log_loss"]
                - calibrated_metrics[
                    "log_loss"
                ]
            ),
            "auroc_gain_over_phase12c": (
                calibrated_metrics["auroc"]
                - phase44_reference_metrics[
                    "phase12c"
                ]["auroc"]
            ),
            "log_loss_gain_over_phase39": (
                phase44_reference_metrics[
                    "phase39"
                ]["log_loss"]
                - calibrated_metrics[
                    "log_loss"
                ]
            ),
            "auroc_gain_over_phase39": (
                calibrated_metrics["auroc"]
                - phase44_reference_metrics[
                    "phase39"
                ]["auroc"]
            ),
        },
        "major_group_wins": int(
            sum(
                record["log_loss_gain"] > 0.0
                for record
                in major_group_records
            )
        ),
        "maximum_major_group_harm": float(
            max(
                max(
                    0.0,
                    -record["log_loss_gain"],
                )
                for record
                in major_group_records
            )
        ),
        "major_group_records": (
            major_group_records
        ),
    }


# ------------------------------------------------------------
# 9. Full-data grid diagnostic — explicitly optimistic
# ------------------------------------------------------------

phase44_full_grid_metrics = (
    phase44_evaluate_complete_grid(
        np.arange(
            phase44_case_count,
            dtype=np.int64,
        )
    )
)

phase44_full_group_safe_mask = (
    phase44_full_grid_metrics[
        "maximum_major_group_harm"
    ]
    <= PHASE44_NESTED_CONFIG[
        "maximum_major_group_harm"
    ]
)

assert np.any(phase44_full_group_safe_mask), {
    "message": (
        "No group-safe full-data candidate exists."
    ),
    "minimum_observed_group_harm": float(
        np.min(
            phase44_full_grid_metrics[
                "maximum_major_group_harm"
            ]
        )
    ),
}

phase44_full_minimum_log_loss = float(
    np.min(
        phase44_full_grid_metrics[
            "overall_log_loss"
        ][phase44_full_group_safe_mask]
    )
)

phase44_full_phase39_metrics = (
    phase44_reference_metrics["phase39"]
)

phase44_full_requested_auc_limit = (
    phase44_full_phase39_metrics["log_loss"]
    + PHASE44_NESTED_CONFIG[
        "auc_track_maximum_log_loss_excess_vs_phase39"
    ]
)

phase44_full_effective_auc_limit = max(
    phase44_full_requested_auc_limit,
    phase44_full_minimum_log_loss
    + PHASE44_NESTED_CONFIG[
        "log_loss_band"
    ],
)

phase44_full_masks = {
    "log_loss_first": (
        phase44_full_group_safe_mask
        & (
            phase44_full_grid_metrics[
                "overall_log_loss"
            ]
            <= phase44_full_minimum_log_loss
            + PHASE44_NESTED_CONFIG[
                "log_loss_band"
            ]
        )
    ),
    "auroc_constrained": (
        phase44_full_group_safe_mask
        & (
            phase44_full_grid_metrics[
                "overall_log_loss"
            ]
            <= phase44_full_effective_auc_limit
        )
    ),
    "pure_auroc_diagnostic": np.ones(
        len(phase44_weight_grid),
        dtype=bool,
    ),
}

phase44_full_diagnostics = OrderedDict()
phase44_all_indices = np.arange(
    phase44_case_count,
    dtype=np.int64,
)

for track_name, eligible_mask in (
    phase44_full_masks.items()
):
    selection = phase44_select_exact_auc_best(
        eligible_mask=eligible_mask,
        grid_metrics=phase44_full_grid_metrics,
        train_indices=phase44_all_indices,
    )

    candidate_index = selection[
        "candidate_index"
    ]
    selected_weight = phase44_weight_grid[
        candidate_index
    ]
    selected_score = (
        phase44_component_logits
        @ selected_weight
    )

    calibrator = (
        phase44_fit_monotonic_calibrator(
            selected_score,
            phase44_y,
        )
    )

    raw_probability = expit(
        selected_score
    )
    calibrated_probability = expit(
        calibrator["intercept"]
        + calibrator["slope"]
        * selected_score
    )

    phase44_full_diagnostics[
        track_name
    ] = {
        "candidate_index": int(
            candidate_index
        ),
        "weights": (
            phase44_weight_dictionary(
                selected_weight
            )
        ),
        "eligible_candidate_count": int(
            selection[
                "eligible_candidate_count"
            ]
        ),
        "raw": phase44_metrics(
            phase44_y,
            raw_probability,
        ),
        "calibrated": phase44_metrics(
            phase44_y,
            calibrated_probability,
        ),
        "maximum_major_group_harm": float(
            phase44_full_grid_metrics[
                "maximum_major_group_harm"
            ][candidate_index]
        ),
        "calibrator": calibrator,
    }


# ------------------------------------------------------------
# 10. Freeze private outputs
# ------------------------------------------------------------

PHASE44_META_FOLD_PRIVATE = (
    phase44_meta_fold.copy()
)
PHASE44_WEIGHT_GRID_PRIVATE = (
    phase44_weight_grid.copy()
)
PHASE44_NESTED_RAW_OOF_PRIVATE = {
    name: values.copy()
    for name, values
    in phase44_oof_raw.items()
}
PHASE44_NESTED_CALIBRATED_OOF_PRIVATE = {
    name: values.copy()
    for name, values
    in phase44_oof_calibrated.items()
}
PHASE44_NESTED_FOLD_RECORDS_PRIVATE = (
    phase44_fold_records
)
PHASE44_FULL_DIAGNOSTICS_PRIVATE = (
    phase44_full_diagnostics
)


# ------------------------------------------------------------
# 11. Sanitized result
# ------------------------------------------------------------

phase44_best_honest_auroc = max(
    record["calibrated"]["auroc"]
    for record
    in phase44_nested_track_metrics.values()
)

phase44_best_honest_log_loss = min(
    record["calibrated"]["log_loss"]
    for record
    in phase44_nested_track_metrics.values()
)

phase44_best_optimistic_auroc = max(
    record["calibrated"]["auroc"]
    for record
    in phase44_full_diagnostics.values()
)

phase44_result = {
    "phase": (
        "phase44_exhaustive_nested_component_ceiling_audit"
    ),
    "status": (
        "existing_components_support_0p96"
        if phase44_best_honest_auroc >= 0.96
        else "existing_component_ceiling_below_0p96"
    ),
    "case_count": int(
        phase44_case_count
    ),
    "component_names": (
        phase44_component_names
    ),
    "weight_search": {
        "simplex_step": (
            PHASE44_NESTED_CONFIG[
                "simplex_step"
            ]
        ),
        "candidate_count": int(
            len(phase44_weight_grid)
        ),
        "all_candidates_evaluated": True,
        "approximate_auc_used_for_shortlisting": True,
        "exact_sklearn_auc_used_for_selection": True,
    },
    "validation": {
        "outer_fold_count": (
            PHASE44_NESTED_CONFIG[
                "outer_fold_count"
            ]
        ),
        "partition_records": (
            phase44_partition_records
        ),
        "within_protocol_case_holdout": True,
        "outer_valid_labels_used_for_selection": False,
        "new_pristine_holdout_claimed": False,
    },
    "references": (
        phase44_reference_metrics
    ),
    "honest_nested_tracks": (
        phase44_nested_track_metrics
    ),
    "fold_selections": (
        phase44_fold_records
    ),
    "optimistic_full_data_diagnostic_only": (
        phase44_full_diagnostics
    ),
    "ceiling_summary": {
        "best_honest_nested_log_loss": float(
            phase44_best_honest_log_loss
        ),
        "best_honest_nested_auroc": float(
            phase44_best_honest_auroc
        ),
        "best_optimistic_full_data_auroc": float(
            phase44_best_optimistic_auroc
        ),
        "honest_auroc_at_least_0p96": bool(
            phase44_best_honest_auroc >= 0.96
        ),
        "optimistic_auroc_at_least_0p96": bool(
            phase44_best_optimistic_auroc
            >= 0.96
        ),
        "log_loss_at_most_0p22": bool(
            phase44_best_honest_log_loss
            <= 0.22
        ),
    },
    "interpretation": {
        "phase42_excluded_from_search": True,
        "phase39_derived_component_retained": True,
        "calibration_is_monotonic": True,
        "calibration_can_change_auroc": False,
        "public_leaderboard_used": False,
        "test_data_used": False,
    },
    "training_voxel_arrays_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase44_nested_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE44_NESTED_CEILING"
)
print(
    json.dumps(
        phase44_result,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE44_NESTED_CEILING"
)

Phase44 nested fold 0: 53130/53130 weights evaluated
Phase44 nested fold 1: 53130/53130 weights evaluated
Phase44 nested fold 2: 53130/53130 weights evaluated
Phase44 nested fold 3: 53130/53130 weights evaluated
Phase44 nested fold 4: 53130/53130 weights evaluated
BEGIN SANITIZED_PHASE44_NESTED_CEILING
{
  "phase": "phase44_exhaustive_nested_component_ceiling_audit",
  "status": "existing_component_ceiling_below_0p96",
  "case_count": 1362,
  "component_names": [
    "phase12c",
    "phase31",
    "phase33",
    "phase36",
    "phase38",
    "phase39"
  ],
  "weight_search": {
    "simplex_step": 0.05,
    "candidate_count": 53130,
    "all_candidates_evaluated": true,
    "approximate_auc_used_for_shortlisting": true,
    "exact_sklearn_auc_used_for_selection": true
  },
  "validation": {
    "outer_fold_count": 5,
    "partition_records": [
      {
        "fold": 0,
        "train_n": 1086,
        "valid_n": 276,
        "train_group_count": 15,
        "valid_group_count": 15,
   

In [35]:
# Cell 141A — Phase45

import json
import time
from pathlib import Path

import numpy as np


phase45_started = time.perf_counter()

assert (
    "PHASE44_NESTED_RAW_OOF_PRIVATE"
    in globals()
), {
    "message": "Phase44 raw nested results are unavailable."
}

assert (
    "phase44_result" in globals()
), {
    "message": "Phase44 sanitized result is unavailable."
}


# ------------------------------------------------------------
# 1. Resolve the existing standardized 80^3 training cache
# ------------------------------------------------------------

phase45_candidate_paths = []

explicit_paths = [
    Path(
        "/kaggle/working/"
        "phase31_highres_float16.npy"
    ),
]

for candidate_path in explicit_paths:
    if candidate_path.is_file():
        phase45_candidate_paths.append(
            candidate_path
        )

for candidate_path in Path(
    "/kaggle/working"
).glob("*highres*float16*.npy"):
    if candidate_path.is_file():
        phase45_candidate_paths.append(
            candidate_path
        )

# Remove duplicate routes without resolving symlinks.
phase45_unique_paths = {
    str(path.absolute()): path
    for path in phase45_candidate_paths
}

phase45_valid_cache_records = []

for candidate_path in (
    phase45_unique_paths.values()
):
    try:
        candidate_array = np.load(
            candidate_path,
            mmap_mode="r",
            allow_pickle=False,
        )
    except Exception:
        continue

    if (
        candidate_array.shape
        == (1362, 80, 80, 80)
        and np.issubdtype(
            candidate_array.dtype,
            np.floating,
        )
    ):
        phase45_valid_cache_records.append({
            "path": candidate_path,
            "shape": tuple(
                candidate_array.shape
            ),
            "dtype": str(
                candidate_array.dtype
            ),
        })

assert len(
    phase45_valid_cache_records
) == 1, {
    "message": (
        "Could not uniquely resolve the standardized "
        "1362 x 80 x 80 x 80 cache."
    ),
    "valid_candidates": [
        {
            "file": record["path"].name,
            "shape": list(record["shape"]),
            "dtype": record["dtype"],
        }
        for record in phase45_valid_cache_records
    ],
}

PHASE45_HIGHRES_CACHE_PATH_PRIVATE = (
    phase45_valid_cache_records[0]["path"]
)

PHASE45_HIGHRES_CACHE_PRIVATE = np.load(
    PHASE45_HIGHRES_CACHE_PATH_PRIVATE,
    mmap_mode="r",
    allow_pickle=False,
)

assert PHASE45_HIGHRES_CACHE_PRIVATE.shape == (
    1362,
    80,
    80,
    80,
)


# ------------------------------------------------------------
# 2. Deterministic label-free real-volume sample
# ------------------------------------------------------------

PHASE45_AUDIT_CONFIG = {
    "random_seed": 450701,
    "sample_case_count": 32,
    "spectral_case_count": 8,
    "candidate_patch_sizes": [5, 8, 10],
    "selected_patch_size": 10,
    "mask_ratio": 0.75,
    "encoder_dimension": 192,
    "encoder_depth": 6,
    "encoder_head_count": 6,
    "decoder_dimension": 96,
    "decoder_depth": 2,
    "decoder_head_count": 4,
    "dropout": 0.05,
    "spatial_loss_weight": 1.0,
    "spectral_loss_weight": 0.25,
    "reflection_loss_weight": 0.10,
}

phase45_rng = np.random.default_rng(
    PHASE45_AUDIT_CONFIG["random_seed"]
)

phase45_sample_indices = np.sort(
    phase45_rng.choice(
        len(PHASE45_HIGHRES_CACHE_PRIVATE),
        size=PHASE45_AUDIT_CONFIG[
            "sample_case_count"
        ],
        replace=False,
    )
)

phase45_sample = np.asarray(
    PHASE45_HIGHRES_CACHE_PRIVATE[
        phase45_sample_indices
    ],
    dtype=np.float32,
)

assert phase45_sample.shape == (
    PHASE45_AUDIT_CONFIG[
        "sample_case_count"
    ],
    80,
    80,
    80,
)

assert np.all(np.isfinite(phase45_sample))


# ------------------------------------------------------------
# 3. Intensity and non-degeneracy audit
# ------------------------------------------------------------

phase45_per_volume_mean = np.mean(
    phase45_sample,
    axis=(1, 2, 3),
)

phase45_per_volume_std = np.std(
    phase45_sample,
    axis=(1, 2, 3),
)

phase45_per_volume_minimum = np.min(
    phase45_sample,
    axis=(1, 2, 3),
)

phase45_per_volume_maximum = np.max(
    phase45_sample,
    axis=(1, 2, 3),
)

phase45_nonzero_fraction = np.mean(
    np.abs(phase45_sample) > 1e-7,
    axis=(1, 2, 3),
)

assert np.min(phase45_per_volume_std) > 1e-6
assert np.max(phase45_per_volume_maximum) > 0.0


def phase45_quantile_record(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "minimum": float(np.min(values)),
        "q05": float(np.quantile(values, 0.05)),
        "q25": float(np.quantile(values, 0.25)),
        "q50": float(np.quantile(values, 0.50)),
        "q75": float(np.quantile(values, 0.75)),
        "q95": float(np.quantile(values, 0.95)),
        "maximum": float(np.max(values)),
    }


phase45_global_intensity_quantiles = {
    quantile_name: float(quantile_value)
    for quantile_name, quantile_value in zip(
        [
            "q001",
            "q01",
            "q05",
            "q50",
            "q95",
            "q99",
            "q999",
        ],
        np.quantile(
            phase45_sample,
            [
                0.001,
                0.01,
                0.05,
                0.50,
                0.95,
                0.99,
                0.999,
            ],
        ),
    )
}


# ------------------------------------------------------------
# 4. Spectral energy audit
# ------------------------------------------------------------

phase45_spectral_sample = phase45_sample[
    :PHASE45_AUDIT_CONFIG[
        "spectral_case_count"
    ]
]

phase45_frequency_axis_0 = np.fft.fftfreq(80)
phase45_frequency_axis_1 = np.fft.fftfreq(80)
phase45_frequency_axis_2 = np.fft.rfftfreq(80)

(
    phase45_frequency_grid_0,
    phase45_frequency_grid_1,
    phase45_frequency_grid_2,
) = np.meshgrid(
    phase45_frequency_axis_0,
    phase45_frequency_axis_1,
    phase45_frequency_axis_2,
    indexing="ij",
)

phase45_radial_frequency = np.sqrt(
    phase45_frequency_grid_0 ** 2
    + phase45_frequency_grid_1 ** 2
    + phase45_frequency_grid_2 ** 2
)

phase45_frequency_masks = {
    "low_0_to_0p10": (
        phase45_radial_frequency <= 0.10
    ),
    "middle_0p10_to_0p25": (
        (phase45_radial_frequency > 0.10)
        & (phase45_radial_frequency <= 0.25)
    ),
    "high_above_0p25": (
        phase45_radial_frequency > 0.25
    ),
}

phase45_spectral_fraction_records = []

for sampled_volume in phase45_spectral_sample:
    centered_volume = (
        sampled_volume
        - np.mean(sampled_volume)
    )

    spectrum = np.fft.rfftn(
        centered_volume,
        axes=(0, 1, 2),
    )

    spectral_energy = (
        np.abs(spectrum) ** 2
    )

    total_energy = float(
        np.sum(spectral_energy)
    )

    assert total_energy > 0.0

    phase45_spectral_fraction_records.append({
        band_name: float(
            np.sum(
                spectral_energy[band_mask]
            )
            / total_energy
        )
        for band_name, band_mask
        in phase45_frequency_masks.items()
    })

phase45_spectral_summary = {
    band_name: {
        "mean": float(
            np.mean([
                record[band_name]
                for record
                in phase45_spectral_fraction_records
            ])
        ),
        "minimum": float(
            np.min([
                record[band_name]
                for record
                in phase45_spectral_fraction_records
            ])
        ),
        "maximum": float(
            np.max([
                record[band_name]
                for record
                in phase45_spectral_fraction_records
            ])
        ),
    }
    for band_name in phase45_frequency_masks
}


# ------------------------------------------------------------
# 5. Reflection-axis audit
# ------------------------------------------------------------

phase45_reflection_records = {}

for spatial_axis in [0, 1, 2]:
    normalized_errors = []

    for sampled_volume in phase45_spectral_sample:
        denominator = (
            float(np.std(sampled_volume))
            + 1e-6
        )

        normalized_error = float(
            np.mean(
                np.abs(
                    sampled_volume
                    - np.flip(
                        sampled_volume,
                        axis=spatial_axis,
                    )
                )
            )
            / denominator
        )

        normalized_errors.append(
            normalized_error
        )

    phase45_reflection_records[
        f"spatial_axis_{spatial_axis}"
    ] = {
        "mean_normalized_absolute_difference": (
            float(np.mean(normalized_errors))
        ),
        "minimum": float(
            np.min(normalized_errors)
        ),
        "maximum": float(
            np.max(normalized_errors)
        ),
    }


# ------------------------------------------------------------
# 6. Patch/token contract
# ------------------------------------------------------------

phase45_patch_contracts = {}

for patch_size in PHASE45_AUDIT_CONFIG[
    "candidate_patch_sizes"
]:
    assert 80 % patch_size == 0

    grid_size = 80 // patch_size
    token_count = grid_size ** 3
    visible_token_count = int(
        round(
            token_count
            * (
                1.0
                - PHASE45_AUDIT_CONFIG[
                    "mask_ratio"
                ]
            )
        )
    )

    phase45_patch_contracts[
        str(patch_size)
    ] = {
        "grid_size": [
            int(grid_size),
            int(grid_size),
            int(grid_size),
        ],
        "token_count": int(token_count),
        "visible_token_count_at_mask_ratio": (
            int(visible_token_count)
        ),
        "patch_target_dimension": int(
            patch_size ** 3
        ),
    }

phase45_selected_patch_size = (
    PHASE45_AUDIT_CONFIG[
        "selected_patch_size"
    ]
)

phase45_selected_token_count = (
    phase45_patch_contracts[
        str(phase45_selected_patch_size)
    ]["token_count"]
)

assert phase45_selected_token_count == 512


# ------------------------------------------------------------
# 7. Freeze private Phase45 data state
# ------------------------------------------------------------

PHASE45_SAMPLE_INDICES_PRIVATE = (
    phase45_sample_indices.copy()
)

PHASE45_DATA_AUDIT_PRIVATE = {
    "per_volume_mean": (
        phase45_per_volume_mean.copy()
    ),
    "per_volume_std": (
        phase45_per_volume_std.copy()
    ),
    "nonzero_fraction": (
        phase45_nonzero_fraction.copy()
    ),
    "spectral_fraction_records": (
        phase45_spectral_fraction_records
    ),
    "reflection_records": (
        phase45_reflection_records
    ),
}


# ------------------------------------------------------------
# 8. Sanitized contract
# ------------------------------------------------------------

phase44_raw_nested_aurocs = {
    track_name: float(
        roc_auc_score(
            PHASE44_Y_PRIVATE,
            prediction,
        )
    )
    for track_name, prediction
    in PHASE44_NESTED_RAW_OOF_PRIVATE.items()
}

phase45_contract = {
    "phase": (
        "phase45_task_aligned_3d_spectral_mae_data_contract"
    ),
    "status": "accepted",
    "motivation": {
        "phase44_best_honest_raw_auroc": float(
            max(
                phase44_raw_nested_aurocs.values()
            )
        ),
        "phase44_best_optimistic_auroc": float(
            phase44_result[
                "ceiling_summary"
            ][
                "best_optimistic_full_data_auroc"
            ]
        ),
        "existing_component_support_for_0p96": False,
        "next_required_change": (
            "new_task_specific_3d_representation"
        ),
    },
    "cache": {
        "source_file": (
            PHASE45_HIGHRES_CACHE_PATH_PRIVATE.name
        ),
        "shape": list(
            PHASE45_HIGHRES_CACHE_PRIVATE.shape
        ),
        "dtype": str(
            PHASE45_HIGHRES_CACHE_PRIVATE.dtype
        ),
        "memory_mapped": True,
    },
    "sample": {
        "case_count": int(
            len(phase45_sample_indices)
        ),
        "shape": list(
            phase45_sample.shape
        ),
        "all_values_finite": True,
        "global_intensity_quantiles": (
            phase45_global_intensity_quantiles
        ),
        "per_volume_mean": (
            phase45_quantile_record(
                phase45_per_volume_mean
            )
        ),
        "per_volume_standard_deviation": (
            phase45_quantile_record(
                phase45_per_volume_std
            )
        ),
        "per_volume_minimum": (
            phase45_quantile_record(
                phase45_per_volume_minimum
            )
        ),
        "per_volume_maximum": (
            phase45_quantile_record(
                phase45_per_volume_maximum
            )
        ),
        "nonzero_fraction": (
            phase45_quantile_record(
                phase45_nonzero_fraction
            )
        ),
    },
    "spectral_energy_fractions": (
        phase45_spectral_summary
    ),
    "reflection_audit": (
        phase45_reflection_records
    ),
    "architecture_plan": {
        "model": (
            "Phase45SpectralMaskedAutoencoder3D"
        ),
        "input_shape": [1, 80, 80, 80],
        "patch_contracts": (
            phase45_patch_contracts
        ),
        "selected_patch_size": int(
            phase45_selected_patch_size
        ),
        "selected_token_count": int(
            phase45_selected_token_count
        ),
        "mask_ratio": (
            PHASE45_AUDIT_CONFIG[
                "mask_ratio"
            ]
        ),
        "encoder_dimension": (
            PHASE45_AUDIT_CONFIG[
                "encoder_dimension"
            ]
        ),
        "encoder_depth": (
            PHASE45_AUDIT_CONFIG[
                "encoder_depth"
            ]
        ),
        "encoder_head_count": (
            PHASE45_AUDIT_CONFIG[
                "encoder_head_count"
            ]
        ),
        "decoder_dimension": (
            PHASE45_AUDIT_CONFIG[
                "decoder_dimension"
            ]
        ),
        "decoder_depth": (
            PHASE45_AUDIT_CONFIG[
                "decoder_depth"
            ]
        ),
        "decoder_head_count": (
            PHASE45_AUDIT_CONFIG[
                "decoder_head_count"
            ]
        ),
    },
    "planned_pretraining_objective": {
        "spatial_masked_reconstruction_weight": (
            PHASE45_AUDIT_CONFIG[
                "spatial_loss_weight"
            ]
        ),
        "spectral_reconstruction_weight": (
            PHASE45_AUDIT_CONFIG[
                "spectral_loss_weight"
            ]
        ),
        "reflection_consistency_weight": (
            PHASE45_AUDIT_CONFIG[
                "reflection_loss_weight"
            ]
        ),
    },
    "validation_policy": {
        "cross_fitted_encoder_pretraining": True,
        "pretrain_images": "outer_train_only",
        "pretrain_labels_used": False,
        "downstream_selection": (
            "nested_within_protocol_and_group_shift"
        ),
        "public_score_used": False,
    },
    "labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_voxels_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_DATA_CONTRACT"
)
print(
    json.dumps(
        phase45_contract,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_DATA_CONTRACT"
)

BEGIN SANITIZED_PHASE45_DATA_CONTRACT
{
  "phase": "phase45_task_aligned_3d_spectral_mae_data_contract",
  "status": "accepted",
  "motivation": {
    "phase44_best_honest_raw_auroc": 0.946863878277337,
    "phase44_best_optimistic_auroc": 0.947821638858959,
    "existing_component_support_for_0p96": false,
    "next_required_change": "new_task_specific_3d_representation"
  },
  "cache": {
    "source_file": "phase31_highres_float16.npy",
    "shape": [
      1362,
      80,
      80,
      80
    ],
    "dtype": "float16",
    "memory_mapped": true
  },
  "sample": {
    "case_count": 32,
    "shape": [
      32,
      80,
      80,
      80
    ],
    "all_values_finite": true,
    "global_intensity_quantiles": {
      "q001": 0.0,
      "q01": 0.0,
      "q05": 0.0,
      "q50": 0.31298828125,
      "q95": 0.61083984375,
      "q99": 0.83740234375,
      "q999": 1.3388671875
    },
    "per_volume_mean": {
      "minimum": 0.20699022710323334,
      "q05": 0.22583226040005683,
     

In [37]:
# Cell 141B — Phase45 

import json
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


phase45_architecture_started = time.perf_counter()

assert "PHASE45_HIGHRES_CACHE_PRIVATE" in globals()
assert "PHASE45_SAMPLE_INDICES_PRIVATE" in globals()

torch.set_float32_matmul_precision("high")

PHASE45_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PHASE45_MODEL_CONFIG = {
    "input_channels": 1,
    "input_size": 80,
    "patch_size": 8,
    "grid_size": 10,
    "token_count": 1000,
    "mask_ratio": 0.75,
    "encoder_dimension": 192,
    "encoder_depth": 6,
    "encoder_head_count": 6,
    "encoder_mlp_ratio": 4,
    "decoder_dimension": 96,
    "decoder_depth": 2,
    "decoder_head_count": 4,
    "decoder_mlp_ratio": 4,
    "representation_dimension": 192,
    "symmetry_dimension": 64,
    "dropout": 0.05,
    "reflection_axis": 2,
    "spatial_loss_weight": 1.0,
    "spectral_loss_weight": 0.25,
    "reflection_loss_weight": 0.10,
    "spectral_bands": [
        [0.0, 0.10],
        [0.10, 0.25],
        [0.25, None],
    ],
}

# Tensor layout is [B, C, D, H, W]. Cache spatial axis 0
# becomes tensor dimension 2.
assert (
    PHASE45_MODEL_CONFIG["reflection_axis"]
    == 2
)

assert (
    PHASE45_MODEL_CONFIG["input_size"]
    % PHASE45_MODEL_CONFIG["patch_size"]
    == 0
)

assert (
    PHASE45_MODEL_CONFIG["grid_size"]
    == PHASE45_MODEL_CONFIG["input_size"]
    // PHASE45_MODEL_CONFIG["patch_size"]
)

assert (
    PHASE45_MODEL_CONFIG["token_count"]
    == PHASE45_MODEL_CONFIG["grid_size"] ** 3
)


# ------------------------------------------------------------
# 1. Initialization helper
# ------------------------------------------------------------

def phase45_initialize_module(module):
    if isinstance(module, nn.Linear):
        nn.init.trunc_normal_(
            module.weight,
            std=0.02,
        )

        if module.bias is not None:
            nn.init.zeros_(module.bias)

    elif isinstance(module, nn.LayerNorm):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


# ------------------------------------------------------------
# 2. Balanced log-amplitude spectral loss
# ------------------------------------------------------------

class Phase45BalancedSpectralLoss3D(nn.Module):
    def __init__(self, spatial_size=80):
        super().__init__()

        frequency_0 = torch.fft.fftfreq(
            spatial_size
        )
        frequency_1 = torch.fft.fftfreq(
            spatial_size
        )
        frequency_2 = torch.fft.rfftfreq(
            spatial_size
        )

        grid_0, grid_1, grid_2 = (
            torch.meshgrid(
                frequency_0,
                frequency_1,
                frequency_2,
                indexing="ij",
            )
        )

        radial_frequency = torch.sqrt(
            grid_0.square()
            + grid_1.square()
            + grid_2.square()
        )

        self.register_buffer(
            "low_frequency_mask",
            radial_frequency <= 0.10,
            persistent=False,
        )

        self.register_buffer(
            "middle_frequency_mask",
            (
                (radial_frequency > 0.10)
                & (radial_frequency <= 0.25)
            ),
            persistent=False,
        )

        self.register_buffer(
            "high_frequency_mask",
            radial_frequency > 0.25,
            persistent=False,
        )

    def forward(self, reconstruction, target):
        assert reconstruction.shape == target.shape
        assert reconstruction.ndim == 5

        spatial_dimensions = (-3, -2, -1)

        reconstruction_centered = (
            reconstruction
            - reconstruction.mean(
                dim=spatial_dimensions,
                keepdim=True,
            )
        )

        target_centered = (
            target
            - target.mean(
                dim=spatial_dimensions,
                keepdim=True,
            )
        )

        reconstruction_spectrum = torch.fft.rfftn(
            reconstruction_centered,
            dim=spatial_dimensions,
            norm="ortho",
        )

        target_spectrum = torch.fft.rfftn(
            target_centered,
            dim=spatial_dimensions,
            norm="ortho",
        )

        reconstruction_log_amplitude = torch.log1p(
            torch.abs(reconstruction_spectrum)
        )

        target_log_amplitude = torch.log1p(
            torch.abs(target_spectrum)
        )

        absolute_difference = torch.abs(
            reconstruction_log_amplitude
            - target_log_amplitude
        )

        band_losses = []

        for band_mask in [
            self.low_frequency_mask,
            self.middle_frequency_mask,
            self.high_frequency_mask,
        ]:
            selected_difference = (
                absolute_difference[..., band_mask]
            )

            assert selected_difference.numel() > 0

            band_losses.append(
                selected_difference.mean()
            )

        # Each band has equal influence regardless of the number
        # of Fourier coordinates or its raw energy magnitude.
        return torch.stack(
            band_losses
        ).mean(), torch.stack(band_losses)


# ------------------------------------------------------------
# 3. Spectral masked autoencoder
# ------------------------------------------------------------

class Phase45SpectralMaskedAutoencoder3D(
    nn.Module
):
    def __init__(
        self,
        input_channels=1,
        input_size=80,
        patch_size=8,
        encoder_dimension=192,
        encoder_depth=6,
        encoder_head_count=6,
        decoder_dimension=96,
        decoder_depth=2,
        decoder_head_count=4,
        dropout=0.05,
        symmetry_dimension=64,
    ):
        super().__init__()

        assert input_size % patch_size == 0
        assert (
            encoder_dimension
            % encoder_head_count
            == 0
        )
        assert (
            decoder_dimension
            % decoder_head_count
            == 0
        )

        self.input_channels = int(
            input_channels
        )
        self.input_size = int(input_size)
        self.patch_size = int(patch_size)
        self.grid_size = (
            self.input_size // self.patch_size
        )
        self.token_count = (
            self.grid_size ** 3
        )
        self.patch_target_dimension = (
            self.input_channels
            * self.patch_size ** 3
        )
        self.encoder_dimension = int(
            encoder_dimension
        )
        self.decoder_dimension = int(
            decoder_dimension
        )

        self.patch_embedding = nn.Conv3d(
            in_channels=self.input_channels,
            out_channels=self.encoder_dimension,
            kernel_size=self.patch_size,
            stride=self.patch_size,
            bias=True,
        )

        self.encoder_position = nn.Parameter(
            torch.zeros(
                1,
                self.token_count,
                self.encoder_dimension,
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=self.encoder_dimension,
                nhead=encoder_head_count,
                dim_feedforward=(
                    self.encoder_dimension * 4
                ),
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=encoder_depth,
            enable_nested_tensor=False,
        )

        self.encoder_norm = nn.LayerNorm(
            self.encoder_dimension
        )

        self.encoder_to_decoder = nn.Linear(
            self.encoder_dimension,
            self.decoder_dimension,
        )

        self.mask_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                self.decoder_dimension,
            )
        )

        self.decoder_position = nn.Parameter(
            torch.zeros(
                1,
                self.token_count,
                self.decoder_dimension,
            )
        )

        decoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=self.decoder_dimension,
                nhead=decoder_head_count,
                dim_feedforward=(
                    self.decoder_dimension * 4
                ),
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.decoder = nn.TransformerEncoder(
            decoder_layer,
            num_layers=decoder_depth,
            enable_nested_tensor=False,
        )

        self.decoder_norm = nn.LayerNorm(
            self.decoder_dimension
        )

        self.patch_prediction = nn.Linear(
            self.decoder_dimension,
            self.patch_target_dimension,
        )

        # Symmetric and antisymmetric projection branches retain
        # different information under left-right reflection.
        self.symmetric_projector = nn.Sequential(
            nn.LayerNorm(
                self.encoder_dimension
            ),
            nn.Linear(
                self.encoder_dimension,
                symmetry_dimension,
            ),
            nn.GELU(),
            nn.Linear(
                symmetry_dimension,
                symmetry_dimension,
            ),
        )

        self.antisymmetric_projector = nn.Sequential(
            nn.LayerNorm(
                self.encoder_dimension
            ),
            nn.Linear(
                self.encoder_dimension,
                symmetry_dimension,
            ),
            nn.GELU(),
            nn.Linear(
                symmetry_dimension,
                symmetry_dimension,
            ),
        )

        self.apply(
            phase45_initialize_module
        )

        nn.init.trunc_normal_(
            self.encoder_position,
            std=0.02,
        )
        nn.init.trunc_normal_(
            self.decoder_position,
            std=0.02,
        )
        nn.init.trunc_normal_(
            self.mask_token,
            std=0.02,
        )

    def patchify(self, volume):
        batch_size, channel_count, depth, height, width = (
            volume.shape
        )

        assert channel_count == self.input_channels
        assert depth == self.input_size
        assert height == self.input_size
        assert width == self.input_size

        patch_size = self.patch_size
        grid_size = self.grid_size

        patches = volume.reshape(
            batch_size,
            channel_count,
            grid_size,
            patch_size,
            grid_size,
            patch_size,
            grid_size,
            patch_size,
        )

        patches = patches.permute(
            0, 2, 4, 6, 1, 3, 5, 7
        ).contiguous()

        patches = patches.reshape(
            batch_size,
            self.token_count,
            self.patch_target_dimension,
        )

        return patches

    def unpatchify(self, patches):
        batch_size = patches.shape[0]
        grid_size = self.grid_size
        patch_size = self.patch_size

        assert patches.shape[1:] == (
            self.token_count,
            self.patch_target_dimension,
        )

        volume = patches.reshape(
            batch_size,
            grid_size,
            grid_size,
            grid_size,
            self.input_channels,
            patch_size,
            patch_size,
            patch_size,
        )

        volume = volume.permute(
            0, 4, 1, 5, 2, 6, 3, 7
        ).contiguous()

        return volume.reshape(
            batch_size,
            self.input_channels,
            self.input_size,
            self.input_size,
            self.input_size,
        )

    def embed_patches(self, volume):
        embedded = self.patch_embedding(
            volume
        )

        embedded = embedded.flatten(
            start_dim=2
        ).transpose(1, 2)

        assert embedded.shape[1:] == (
            self.token_count,
            self.encoder_dimension,
        )

        return embedded

    def random_masking(
        self,
        embedded_tokens,
        mask_ratio,
    ):
        batch_size, token_count, dimension = (
            embedded_tokens.shape
        )

        keep_count = max(
            1,
            int(
                round(
                    token_count
                    * (1.0 - mask_ratio)
                )
            ),
        )

        noise = torch.rand(
            batch_size,
            token_count,
            device=embedded_tokens.device,
        )

        shuffle_indices = torch.argsort(
            noise,
            dim=1,
        )

        keep_indices = shuffle_indices[
            :, :keep_count
        ]

        expanded_keep_indices = (
            keep_indices.unsqueeze(-1).expand(
                -1,
                -1,
                dimension,
            )
        )

        visible_tokens = torch.gather(
            embedded_tokens,
            dim=1,
            index=expanded_keep_indices,
        )

        mask = torch.ones(
            batch_size,
            token_count,
            dtype=torch.bool,
            device=embedded_tokens.device,
        )

        mask.scatter_(
            dim=1,
            index=keep_indices,
            value=False,
        )

        return (
            visible_tokens,
            keep_indices,
            mask,
        )

    def encode_visible(
        self,
        volume,
        mask_ratio,
    ):
        embedded_tokens = self.embed_patches(
            volume
        )

        positioned_tokens = (
            embedded_tokens
            + self.encoder_position
        )

        (
            visible_tokens,
            keep_indices,
            mask,
        ) = self.random_masking(
            positioned_tokens,
            mask_ratio,
        )

        encoded_visible = self.encoder(
            visible_tokens
        )

        encoded_visible = self.encoder_norm(
            encoded_visible
        )

        embedding = encoded_visible.mean(
            dim=1
        )

        return {
            "encoded_visible": encoded_visible,
            "embedding": embedding,
            "keep_indices": keep_indices,
            "mask": mask,
        }

    def encode_full(self, volume):
        embedded_tokens = self.embed_patches(
            volume
        )

        positioned_tokens = (
            embedded_tokens
            + self.encoder_position
        )

        encoded = self.encoder(
            positioned_tokens
        )

        encoded = self.encoder_norm(
            encoded
        )

        embedding = encoded.mean(dim=1)

        return {
            "tokens": encoded,
            "embedding": embedding,
        }

    def decode(
        self,
        encoded_visible,
        keep_indices,
    ):
        batch_size = (
            encoded_visible.shape[0]
        )

        decoded_visible = (
            self.encoder_to_decoder(
                encoded_visible
            )
        )

        full_decoder_tokens = (
            self.mask_token.expand(
                batch_size,
                self.token_count,
                self.decoder_dimension,
            ).clone()
        )

        expanded_keep_indices = (
            keep_indices.unsqueeze(-1).expand(
                -1,
                -1,
                self.decoder_dimension,
            )
        )

        full_decoder_tokens.scatter_(
            dim=1,
            index=expanded_keep_indices,
            src=decoded_visible,
        )

        full_decoder_tokens = (
            full_decoder_tokens
            + self.decoder_position
        )

        decoded_tokens = self.decoder(
            full_decoder_tokens
        )

        decoded_tokens = self.decoder_norm(
            decoded_tokens
        )

        return self.patch_prediction(
            decoded_tokens
        )

    def forward(
        self,
        volume,
        mask_ratio=0.75,
    ):
        encoded = self.encode_visible(
            volume,
            mask_ratio,
        )

        predicted_patches = self.decode(
            encoded_visible=encoded[
                "encoded_visible"
            ],
            keep_indices=encoded[
                "keep_indices"
            ],
        )

        return {
            "predicted_patches": (
                predicted_patches
            ),
            "mask": encoded["mask"],
            "embedding": encoded["embedding"],
            "keep_indices": (
                encoded["keep_indices"]
            ),
        }

    def reflection_representation(
        self,
        volume,
        reflection_dimension=2,
    ):
        original_embedding = self.encode_full(
            volume
        )["embedding"]

        reflected_volume = torch.flip(
            volume,
            dims=[reflection_dimension],
        )

        reflected_embedding = self.encode_full(
            reflected_volume
        )["embedding"]

        original_symmetric = (
            self.symmetric_projector(
                original_embedding
            )
        )

        reflected_symmetric = (
            self.symmetric_projector(
                reflected_embedding
            )
        )

        original_antisymmetric = (
            self.antisymmetric_projector(
                original_embedding
            )
        )

        reflected_antisymmetric = (
            self.antisymmetric_projector(
                reflected_embedding
            )
        )

        return {
            "original_embedding": (
                original_embedding
            ),
            "reflected_embedding": (
                reflected_embedding
            ),
            "original_symmetric": (
                original_symmetric
            ),
            "reflected_symmetric": (
                reflected_symmetric
            ),
            "original_antisymmetric": (
                original_antisymmetric
            ),
            "reflected_antisymmetric": (
                reflected_antisymmetric
            ),
        }


# ------------------------------------------------------------
# 4. Loss contract
# ------------------------------------------------------------

def phase45_masked_spatial_loss(
    predicted_patches,
    target_patches,
    mask,
):
    assert predicted_patches.shape == (
        target_patches.shape
    )
    assert mask.shape == (
        predicted_patches.shape[:2]
    )
    assert torch.any(mask)

    return F.smooth_l1_loss(
        predicted_patches[mask],
        target_patches[mask],
        beta=0.05,
        reduction="mean",
    )


def phase45_reflection_equivariance_loss(
    reflection_outputs,
):
    original_symmetric = F.normalize(
        reflection_outputs[
            "original_symmetric"
        ],
        dim=-1,
    )

    reflected_symmetric = F.normalize(
        reflection_outputs[
            "reflected_symmetric"
        ],
        dim=-1,
    )

    original_antisymmetric = F.normalize(
        reflection_outputs[
            "original_antisymmetric"
        ],
        dim=-1,
    )

    reflected_antisymmetric = F.normalize(
        reflection_outputs[
            "reflected_antisymmetric"
        ],
        dim=-1,
    )

    symmetric_loss = F.mse_loss(
        original_symmetric,
        reflected_symmetric,
    )

    antisymmetric_loss = F.mse_loss(
        original_antisymmetric,
        -reflected_antisymmetric,
    )

    return (
        0.5 * symmetric_loss
        + 0.5 * antisymmetric_loss
    ), {
        "symmetric": symmetric_loss,
        "antisymmetric": antisymmetric_loss,
    }


phase45_spectral_loss_module = (
    Phase45BalancedSpectralLoss3D(
        spatial_size=80
    ).to(PHASE45_DEVICE)
)

# ------------------------------------------------------------
# 5. Real cached-volume forward/backward contract
# ------------------------------------------------------------

phase45_contract_indices = (
    PHASE45_SAMPLE_INDICES_PRIVATE[:2]
)

phase45_contract_input = torch.from_numpy(
    np.asarray(
        PHASE45_HIGHRES_CACHE_PRIVATE[
            phase45_contract_indices
        ],
        dtype=np.float32,
    )
).unsqueeze(1)

assert phase45_contract_input.shape == (
    2,
    1,
    80,
    80,
    80,
)

if PHASE45_DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

torch.manual_seed(450702)

if PHASE45_DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(450702)

phase45_contract_model = (
    Phase45SpectralMaskedAutoencoder3D(
        input_channels=(
            PHASE45_MODEL_CONFIG[
                "input_channels"
            ]
        ),
        input_size=(
            PHASE45_MODEL_CONFIG[
                "input_size"
            ]
        ),
        patch_size=(
            PHASE45_MODEL_CONFIG[
                "patch_size"
            ]
        ),
        encoder_dimension=(
            PHASE45_MODEL_CONFIG[
                "encoder_dimension"
            ]
        ),
        encoder_depth=(
            PHASE45_MODEL_CONFIG[
                "encoder_depth"
            ]
        ),
        encoder_head_count=(
            PHASE45_MODEL_CONFIG[
                "encoder_head_count"
            ]
        ),
        decoder_dimension=(
            PHASE45_MODEL_CONFIG[
                "decoder_dimension"
            ]
        ),
        decoder_depth=(
            PHASE45_MODEL_CONFIG[
                "decoder_depth"
            ]
        ),
        decoder_head_count=(
            PHASE45_MODEL_CONFIG[
                "decoder_head_count"
            ]
        ),
        dropout=(
            PHASE45_MODEL_CONFIG[
                "dropout"
            ]
        ),
        symmetry_dimension=(
            PHASE45_MODEL_CONFIG[
                "symmetry_dimension"
            ]
        ),
    ).to(PHASE45_DEVICE)
)

phase45_contract_model.train()

phase45_contract_input = (
    phase45_contract_input.to(
        PHASE45_DEVICE,
        non_blocking=True,
    )
)

phase45_forward = phase45_contract_model(
    phase45_contract_input,
    mask_ratio=(
        PHASE45_MODEL_CONFIG["mask_ratio"]
    ),
)

phase45_target_patches = (
    phase45_contract_model.patchify(
        phase45_contract_input
    )
)

phase45_spatial_loss = (
    phase45_masked_spatial_loss(
        predicted_patches=phase45_forward[
            "predicted_patches"
        ],
        target_patches=phase45_target_patches,
        mask=phase45_forward["mask"],
    )
)

# Preserve visible patches exactly. The spectral objective
# therefore measures reconstruction of masked regions.
phase45_completed_patches = torch.where(
    phase45_forward[
        "mask"
    ].unsqueeze(-1),
    phase45_forward[
        "predicted_patches"
    ],
    phase45_target_patches,
)

phase45_reconstruction = (
    phase45_contract_model.unpatchify(
        phase45_completed_patches
    )
)

(
    phase45_spectral_loss,
    phase45_spectral_band_losses,
) = phase45_spectral_loss_module(
    phase45_reconstruction,
    phase45_contract_input,
)

phase45_reflection_outputs = (
    phase45_contract_model
    .reflection_representation(
        phase45_contract_input,
        reflection_dimension=(
            PHASE45_MODEL_CONFIG[
                "reflection_axis"
            ]
        ),
    )
)

(
    phase45_reflection_loss,
    phase45_reflection_loss_parts,
) = phase45_reflection_equivariance_loss(
    phase45_reflection_outputs
)

phase45_total_loss = (
    PHASE45_MODEL_CONFIG[
        "spatial_loss_weight"
    ]
    * phase45_spatial_loss
    + PHASE45_MODEL_CONFIG[
        "spectral_loss_weight"
    ]
    * phase45_spectral_loss
    + PHASE45_MODEL_CONFIG[
        "reflection_loss_weight"
    ]
    * phase45_reflection_loss
)

assert torch.isfinite(phase45_total_loss)

phase45_contract_model.zero_grad(
    set_to_none=True
)

phase45_total_loss.backward()

phase45_squared_gradient_norm = 0.0
phase45_gradient_tensor_count = 0
phase45_all_gradients_finite = True

for parameter in (
    phase45_contract_model.parameters()
):
    if parameter.grad is None:
        continue

    phase45_gradient_tensor_count += 1

    if not torch.all(
        torch.isfinite(parameter.grad)
    ):
        phase45_all_gradients_finite = False

    phase45_squared_gradient_norm += float(
        parameter.grad.detach()
        .float()
        .square()
        .sum()
        .cpu()
    )

phase45_gradient_norm = math.sqrt(
    phase45_squared_gradient_norm
)

assert phase45_gradient_tensor_count > 0
assert phase45_all_gradients_finite
assert phase45_gradient_norm > 0.0

phase45_parameter_count = sum(
    parameter.numel()
    for parameter
    in phase45_contract_model.parameters()
)

phase45_trainable_parameter_count = sum(
    parameter.numel()
    for parameter
    in phase45_contract_model.parameters()
    if parameter.requires_grad
)

phase45_mask_fraction = float(
    phase45_forward[
        "mask"
    ].float().mean().detach().cpu()
)

assert abs(
    phase45_mask_fraction
    - PHASE45_MODEL_CONFIG["mask_ratio"]
) <= 1e-6

phase45_patch_roundtrip_error = float(
    torch.max(
        torch.abs(
            phase45_contract_model.unpatchify(
                phase45_target_patches
            )
            - phase45_contract_input
        )
    ).detach().cpu()
)

assert phase45_patch_roundtrip_error == 0.0

phase45_peak_vram_mb = (
    float(
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )
    if PHASE45_DEVICE.type == "cuda"
    else 0.0
)


# ------------------------------------------------------------
# 6. Retain reusable architecture state
# ------------------------------------------------------------

PHASE45_MODEL_CLASS_PRIVATE = (
    Phase45SpectralMaskedAutoencoder3D
)

PHASE45_SPECTRAL_LOSS_CLASS_PRIVATE = (
    Phase45BalancedSpectralLoss3D
)


# ------------------------------------------------------------
# 7. Sanitized architecture result
# ------------------------------------------------------------

phase45_architecture_result = {
    "phase": (
        "phase45_task_aligned_spectral_mae_architecture"
    ),
    "status": "accepted",
    "model": (
        "Phase45SpectralMaskedAutoencoder3D"
    ),
    "device": str(PHASE45_DEVICE),
    "parameter_count": int(
        phase45_parameter_count
    ),
    "trainable_parameter_count": int(
        phase45_trainable_parameter_count
    ),
    "input_shape": list(
        phase45_contract_input.shape
    ),
    "patch_contract": {
        "patch_size": int(
            PHASE45_MODEL_CONFIG[
                "patch_size"
            ]
        ),
        "grid_size": [
            int(
                PHASE45_MODEL_CONFIG[
                    "grid_size"
                ]
            )
        ] * 3,
        "token_count": int(
            PHASE45_MODEL_CONFIG[
                "token_count"
            ]
        ),
        "visible_token_count": int(
            phase45_forward[
                "keep_indices"
            ].shape[1]
        ),
        "mask_fraction": (
            phase45_mask_fraction
        ),
        "patch_target_dimension": int(
            phase45_contract_model
            .patch_target_dimension
        ),
        "patch_roundtrip_maximum_error": (
            phase45_patch_roundtrip_error
        ),
    },
    "output_shapes": {
        "predicted_patches": list(
            phase45_forward[
                "predicted_patches"
            ].shape
        ),
        "mask": list(
            phase45_forward["mask"].shape
        ),
        "pretraining_embedding": list(
            phase45_forward[
                "embedding"
            ].shape
        ),
        "reconstruction": list(
            phase45_reconstruction.shape
        ),
        "full_embedding": list(
            phase45_reflection_outputs[
                "original_embedding"
            ].shape
        ),
        "symmetric_projection": list(
            phase45_reflection_outputs[
                "original_symmetric"
            ].shape
        ),
        "antisymmetric_projection": list(
            phase45_reflection_outputs[
                "original_antisymmetric"
            ].shape
        ),
    },
    "loss_contract": {
        "spatial_masked_smooth_l1": float(
            phase45_spatial_loss
            .detach().cpu()
        ),
        "spectral_balanced_log_amplitude": (
            float(
                phase45_spectral_loss
                .detach().cpu()
            )
        ),
        "spectral_band_losses": {
            "low": float(
                phase45_spectral_band_losses[
                    0
                ].detach().cpu()
            ),
            "middle": float(
                phase45_spectral_band_losses[
                    1
                ].detach().cpu()
            ),
            "high": float(
                phase45_spectral_band_losses[
                    2
                ].detach().cpu()
            ),
        },
        "reflection_equivariance": float(
            phase45_reflection_loss
            .detach().cpu()
        ),
        "reflection_parts": {
            "symmetric": float(
                phase45_reflection_loss_parts[
                    "symmetric"
                ].detach().cpu()
            ),
            "antisymmetric": float(
                phase45_reflection_loss_parts[
                    "antisymmetric"
                ].detach().cpu()
            ),
        },
        "total": float(
            phase45_total_loss
            .detach().cpu()
        ),
    },
    "reflection_contract": {
        "cache_spatial_axis": 0,
        "tensor_dimension": int(
            PHASE45_MODEL_CONFIG[
                "reflection_axis"
            ]
        ),
        "symmetric_branch_invariant": True,
        "antisymmetric_branch_sign_equivariant": True,
        "complete_embedding_forced_invariant": False,
    },
    "backward_contract": {
        "gradient_tensor_count": int(
            phase45_gradient_tensor_count
        ),
        "gradient_norm": float(
            phase45_gradient_norm
        ),
        "all_gradients_finite": bool(
            phase45_all_gradients_finite
        ),
        "passed": True,
    },
    "peak_vram_mb": (
        phase45_peak_vram_mb
    ),
    "real_cached_inputs_used": True,
    "labels_used": False,
    "outer_validation_labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_architecture_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_ARCHITECTURE"
)
print(
    json.dumps(
        phase45_architecture_result,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_ARCHITECTURE"
)

BEGIN SANITIZED_PHASE45_ARCHITECTURE
{
  "phase": "phase45_task_aligned_spectral_mae_architecture",
  "status": "accepted",
  "model": "Phase45SpectralMaskedAutoencoder3D",
  "device": "cuda",
  "parameter_count": 3382016,
  "trainable_parameter_count": 3382016,
  "input_shape": [
    2,
    1,
    80,
    80,
    80
  ],
  "patch_contract": {
    "patch_size": 8,
    "grid_size": [
      10,
      10,
      10
    ],
    "token_count": 1000,
    "visible_token_count": 250,
    "mask_fraction": 0.7500000596046448,
    "patch_target_dimension": 512,
    "patch_roundtrip_maximum_error": 0.0
  },
  "output_shapes": {
    "predicted_patches": [
      2,
      1000,
      512
    ],
    "mask": [
      2,
      1000
    ],
    "pretraining_embedding": [
      2,
      192
    ],
    "reconstruction": [
      2,
      1,
      80,
      80,
      80
    ],
    "full_embedding": [
      2,
      192
    ],
    "symmetric_projection": [
      2,
      64
    ],
    "antisymmetric_projection": 

In [38]:
# Cell 141C — Phase45 

import json
import time

import numpy as np
import torch


phase45_reflection_started = time.perf_counter()

assert "phase45_contract_model" in globals()
assert "phase45_contract_input" in globals()
assert "PHASE45_MODEL_CONFIG" in globals()

phase45_contract_model.eval()

with torch.no_grad():
    phase45_original_embedding = (
        phase45_contract_model.encode_full(
            phase45_contract_input
        )["embedding"]
    )

    phase45_reflected_input = torch.flip(
        phase45_contract_input,
        dims=[
            PHASE45_MODEL_CONFIG[
                "reflection_axis"
            ]
        ],
    )

    phase45_reflected_embedding = (
        phase45_contract_model.encode_full(
            phase45_reflected_input
        )["embedding"]
    )

    phase45_symmetric_embedding = 0.5 * (
        phase45_original_embedding
        + phase45_reflected_embedding
    )

    phase45_antisymmetric_embedding = 0.5 * (
        phase45_original_embedding
        - phase45_reflected_embedding
    )

    # Under reflection, original/reflected embeddings exchange.
    phase45_reflected_symmetric_embedding = 0.5 * (
        phase45_reflected_embedding
        + phase45_original_embedding
    )

    phase45_reflected_antisymmetric_embedding = 0.5 * (
        phase45_reflected_embedding
        - phase45_original_embedding
    )

    phase45_invariant_representation = torch.cat(
        [
            phase45_symmetric_embedding,
            torch.abs(
                phase45_antisymmetric_embedding
            ),
        ],
        dim=-1,
    )

    phase45_reflected_invariant_representation = (
        torch.cat(
            [
                phase45_reflected_symmetric_embedding,
                torch.abs(
                    phase45_reflected_antisymmetric_embedding
                ),
            ],
            dim=-1,
        )
    )

phase45_symmetric_equivariance_error = float(
    torch.max(
        torch.abs(
            phase45_symmetric_embedding
            - phase45_reflected_symmetric_embedding
        )
    ).cpu()
)

phase45_antisymmetric_equivariance_error = float(
    torch.max(
        torch.abs(
            phase45_antisymmetric_embedding
            + phase45_reflected_antisymmetric_embedding
        )
    ).cpu()
)

phase45_invariant_representation_error = float(
    torch.max(
        torch.abs(
            phase45_invariant_representation
            - phase45_reflected_invariant_representation
        )
    ).cpu()
)

assert phase45_symmetric_equivariance_error == 0.0
assert phase45_antisymmetric_equivariance_error == 0.0
assert phase45_invariant_representation_error == 0.0

assert phase45_invariant_representation.shape == (
    phase45_contract_input.shape[0],
    2
    * PHASE45_MODEL_CONFIG[
        "encoder_dimension"
    ],
)

# Learned reflection projectors will not participate in
# pretraining. They remain in the prototype class only to avoid
# unnecessarily redefining the already-tested architecture.
for parameter_name, parameter in (
    phase45_contract_model.named_parameters()
):
    if (
        parameter_name.startswith(
            "symmetric_projector."
        )
        or parameter_name.startswith(
            "antisymmetric_projector."
        )
    ):
        parameter.requires_grad_(False)

PHASE45_PRETRAINING_CONFIG_PRIVATE = {
    "mask_ratio": 0.75,
    "spatial_loss_weight": 1.0,
    "spectral_loss_weight": 0.25,
    "learned_reflection_loss_weight": 0.0,
    "reflection_augmentation_probability": 0.5,
    "downstream_representation": (
        "concatenate_analytic_symmetric_"
        "embedding_and_absolute_analytic_"
        "antisymmetric_embedding"
    ),
    "downstream_dimension": (
        2
        * PHASE45_MODEL_CONFIG[
            "encoder_dimension"
        ]
    ),
    "reflection_axis": (
        PHASE45_MODEL_CONFIG[
            "reflection_axis"
        ]
    ),
    "pretraining_uses_labels": False,
}

PHASE45_MODEL_CONFIG[
    "reflection_loss_weight"
] = 0.0

phase45_trainable_after_correction = sum(
    parameter.numel()
    for parameter
    in phase45_contract_model.parameters()
    if parameter.requires_grad
)

phase45_frozen_projector_parameters = sum(
    parameter.numel()
    for parameter_name, parameter
    in phase45_contract_model.named_parameters()
    if (
        parameter_name.startswith(
            "symmetric_projector."
        )
        or parameter_name.startswith(
            "antisymmetric_projector."
        )
    )
)

phase45_reflection_result = {
    "phase": (
        "phase45_analytic_reflection_"
        "equivariance_correction"
    ),
    "status": "accepted",
    "reason": (
        "avoid_trivial_collapse_of_learned_"
        "reflection_projectors"
    ),
    "representation": {
        "encoder_dimension": int(
            PHASE45_MODEL_CONFIG[
                "encoder_dimension"
            ]
        ),
        "symmetric_dimension": int(
            phase45_symmetric_embedding.shape[1]
        ),
        "absolute_antisymmetric_dimension": int(
            phase45_antisymmetric_embedding.shape[1]
        ),
        "downstream_dimension": int(
            phase45_invariant_representation.shape[1]
        ),
    },
    "equivariance_errors": {
        "symmetric": (
            phase45_symmetric_equivariance_error
        ),
        "antisymmetric_sign": (
            phase45_antisymmetric_equivariance_error
        ),
        "final_invariant_representation": (
            phase45_invariant_representation_error
        ),
    },
    "pretraining_objective": {
        "masked_spatial_weight": (
            PHASE45_PRETRAINING_CONFIG_PRIVATE[
                "spatial_loss_weight"
            ]
        ),
        "balanced_spectral_weight": (
            PHASE45_PRETRAINING_CONFIG_PRIVATE[
                "spectral_loss_weight"
            ]
        ),
        "learned_reflection_loss_weight": 0.0,
        "random_reflection_augmentation": True,
    },
    "parameters": {
        "prototype_total": int(
            sum(
                parameter.numel()
                for parameter
                in phase45_contract_model.parameters()
            )
        ),
        "frozen_unused_projector_parameters": int(
            phase45_frozen_projector_parameters
        ),
        "effective_trainable_parameters": int(
            phase45_trainable_after_correction
        ),
    },
    "interpretation": {
        "complete_embedding_forced_invariant": False,
        "directional_asymmetry_retained": True,
        "asymmetry_magnitude_flip_invariant": True,
        "projector_collapse_possible": False,
    },
    "labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_reflection_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_REFLECTION_CORRECTION"
)
print(
    json.dumps(
        phase45_reflection_result,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_REFLECTION_CORRECTION"
)

BEGIN SANITIZED_PHASE45_REFLECTION_CORRECTION
{
  "phase": "phase45_analytic_reflection_equivariance_correction",
  "status": "accepted",
  "reason": "avoid_trivial_collapse_of_learned_reflection_projectors",
  "representation": {
    "encoder_dimension": 192,
    "symmetric_dimension": 192,
    "absolute_antisymmetric_dimension": 192,
    "downstream_dimension": 384
  },
  "equivariance_errors": {
    "symmetric": 0.0,
    "antisymmetric_sign": 0.0,
    "final_invariant_representation": 0.0
  },
  "pretraining_objective": {
    "masked_spatial_weight": 1.0,
    "balanced_spectral_weight": 0.25,
    "learned_reflection_loss_weight": 0.0,
    "random_reflection_augmentation": true
  },
  "parameters": {
    "prototype_total": 3382016,
    "frozen_unused_projector_parameters": 33792,
    "effective_trainable_parameters": 3348224
  },
  "interpretation": {
    "complete_embedding_forced_invariant": false,
    "directional_asymmetry_retained": true,
    "asymmetry_magnitude_flip_invariant"

In [9]:
# Phase45 AMP-safe decoder patch + verified forward contract

import json
import time

import numpy as np
import torch


phase45_amp_patch_started = time.perf_counter()

required_names = [
    "Phase45SpectralMaskedAutoencoder3D",
    "PHASE45_DEVICE",
    "PHASE45_MODEL_CONFIG",
    "PHASE45_HIGHRES_CACHE_PRIVATE",
    "PHASE45_SAMPLE_INDICES_PRIVATE",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, {
    "message": "AMP patch dependencies are missing.",
    "missing": missing_names,
}


def phase45_decode_amp_safe(
    self,
    encoded_visible,
    keep_indices,
):
    decoded_visible = self.encoder_to_decoder(
        encoded_visible
    )

    batch_size = decoded_visible.shape[0]

    assert decoded_visible.ndim == 3
    assert keep_indices.ndim == 2
    assert keep_indices.dtype == torch.int64
    assert (
        decoded_visible.shape[:2]
        == keep_indices.shape
    )

    # Explicitly match the decoder-visible tensor's dtype and
    # device before scatter_. This is required under BF16 AMP.
    typed_mask_token = self.mask_token.to(
        device=decoded_visible.device,
        dtype=decoded_visible.dtype,
    )

    full_decoder_tokens = (
        typed_mask_token.expand(
            batch_size,
            self.token_count,
            self.decoder_dimension,
        ).clone()
    )

    expanded_keep_indices = (
        keep_indices.unsqueeze(-1).expand(
            -1,
            -1,
            self.decoder_dimension,
        )
    )

    assert (
        full_decoder_tokens.dtype
        == decoded_visible.dtype
    )
    assert (
        full_decoder_tokens.device
        == decoded_visible.device
    )

    full_decoder_tokens.scatter_(
        dim=1,
        index=expanded_keep_indices,
        src=decoded_visible,
    )

    typed_decoder_position = (
        self.decoder_position.to(
            device=full_decoder_tokens.device,
            dtype=full_decoder_tokens.dtype,
        )
    )

    full_decoder_tokens = (
        full_decoder_tokens
        + typed_decoder_position
    )

    decoded_tokens = self.decoder(
        full_decoder_tokens
    )

    decoded_tokens = self.decoder_norm(
        decoded_tokens
    )

    predicted_patches = self.patch_prediction(
        decoded_tokens
    )

    assert predicted_patches.shape == (
        batch_size,
        self.token_count,
        self.patch_target_dimension,
    )

    return predicted_patches


# Patch the class. Existing and future model instances now use
# the corrected implementation.
Phase45SpectralMaskedAutoencoder3D.decode = (
    phase45_decode_amp_safe
)


# ------------------------------------------------------------
# Verify the exact failing BF16 execution path
# ------------------------------------------------------------

if "phase45_pilot_model" in globals():
    phase45_patch_test_model = (
        phase45_pilot_model
    )
elif "phase45_contract_model" in globals():
    phase45_patch_test_model = (
        phase45_contract_model
    )
else:
    raise AssertionError({
        "message": (
            "No Phase45 model instance is available "
            "for the BF16 verification."
        )
    })

phase45_patch_test_model = (
    phase45_patch_test_model.to(
        PHASE45_DEVICE
    )
)
phase45_patch_test_model.eval()

phase45_patch_test_index = int(
    PHASE45_SAMPLE_INDICES_PRIVATE[0]
)

phase45_patch_test_input = torch.from_numpy(
    np.asarray(
        PHASE45_HIGHRES_CACHE_PRIVATE[
            phase45_patch_test_index
        ],
        dtype=np.float32,
    ).copy()
).unsqueeze(0).unsqueeze(0).to(
    PHASE45_DEVICE
)

assert phase45_patch_test_input.shape == (
    1,
    1,
    80,
    80,
    80,
)

phase45_amp_enabled = (
    PHASE45_DEVICE.type == "cuda"
)

with torch.no_grad():
    with torch.autocast(
        device_type=PHASE45_DEVICE.type,
        dtype=torch.bfloat16,
        enabled=phase45_amp_enabled,
    ):
        phase45_patch_test_output = (
            phase45_patch_test_model(
                phase45_patch_test_input,
                mask_ratio=0.75,
            )
        )

phase45_patch_prediction = (
    phase45_patch_test_output[
        "predicted_patches"
    ]
)

assert phase45_patch_prediction.shape == (
    1,
    1000,
    512,
)

assert torch.all(
    torch.isfinite(
        phase45_patch_prediction.float()
    )
)

phase45_patch_result = {
    "phase": (
        "phase45_amp_safe_decoder_patch"
    ),
    "status": "accepted",
    "device": str(PHASE45_DEVICE),
    "autocast_enabled": bool(
        phase45_amp_enabled
    ),
    "encoded_execution_dtype": str(
        phase45_patch_prediction.dtype
    ),
    "prediction_shape": list(
        phase45_patch_prediction.shape
    ),
    "all_predictions_finite": True,
    "patched_operations": [
        "cast_mask_token_to_encoded_visible_dtype",
        "cast_decoder_position_to_decoder_token_dtype",
        "verify_scatter_source_destination_dtype",
    ],
    "labels_used": False,
    "training_voxel_cache_read": True,
    "test_data_read": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_amp_patch_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_AMP_PATCH"
)
print(
    json.dumps(
        phase45_patch_result,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_AMP_PATCH"
)

AssertionError: {'message': 'AMP patch dependencies are missing.', 'missing': ['Phase45SpectralMaskedAutoencoder3D', 'PHASE45_DEVICE', 'PHASE45_MODEL_CONFIG', 'PHASE45_HIGHRES_CACHE_PRIVATE', 'PHASE45_SAMPLE_INDICES_PRIVATE']}

In [8]:
# Cell 141D — Phase45 

import copy
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


phase45_pilot_started = time.perf_counter()

required_phase45_pilot_state = [
    "PHASE45_HIGHRES_CACHE_PATH_PRIVATE",
    "PHASE44_ORIGINAL_FOLD_PRIVATE",
    "Phase45SpectralMaskedAutoencoder3D",
    "Phase45BalancedSpectralLoss3D",
    "phase45_masked_spatial_loss",
    "PHASE45_MODEL_CONFIG",
    "PHASE45_PRETRAINING_CONFIG_PRIVATE",
    "PHASE45_DEVICE",
]

missing_phase45_pilot_state = [
    name
    for name in required_phase45_pilot_state
    if name not in globals()
]

assert not missing_phase45_pilot_state, {
    "message": "Phase45 pilot dependencies are missing.",
    "missing": missing_phase45_pilot_state,
}


# ------------------------------------------------------------
# 1. Pilot configuration
# ------------------------------------------------------------

PHASE45_PILOT_CONFIG = {
    "random_seed": 450703,
    "original_outer_fold": 0,
    "training_case_count": 256,
    "validation_case_count": 64,
    "batch_size": 4,
    "worker_count": 2,
    "epoch_count": 4,
    "learning_rate": 1.5e-4,
    "minimum_learning_rate": 1.5e-5,
    "warmup_fraction": 0.10,
    "weight_decay": 0.05,
    "gradient_clip": 1.0,
    "mask_ratio": 0.75,
    "reflection_probability": 0.5,
    "intensity_scale_range": [0.98, 1.02],
    "noise_standard_deviation": 0.003,
    "spatial_loss_weight": 1.0,
    "spectral_loss_weight": 0.25,
    "minimum_total_validation_gain": 0.03,
    "minimum_spatial_validation_gain": 0.03,
    "minimum_improving_spectral_band_count": 2,
}

phase45_pilot_rng = np.random.default_rng(
    PHASE45_PILOT_CONFIG["random_seed"]
)

phase45_original_outer_fold = np.asarray(
    PHASE44_ORIGINAL_FOLD_PRIVATE,
    dtype=np.int64,
)

phase45_available_pilot_indices = np.flatnonzero(
    phase45_original_outer_fold
    != PHASE45_PILOT_CONFIG[
        "original_outer_fold"
    ]
)

assert len(phase45_available_pilot_indices) == 895

phase45_shuffled_pilot_indices = (
    phase45_pilot_rng.permutation(
        phase45_available_pilot_indices
    )
)

phase45_pilot_validation_indices = np.sort(
    phase45_shuffled_pilot_indices[
        :PHASE45_PILOT_CONFIG[
            "validation_case_count"
        ]
    ]
)

phase45_pilot_training_indices = np.sort(
    phase45_shuffled_pilot_indices[
        PHASE45_PILOT_CONFIG[
            "validation_case_count"
        ]:
        PHASE45_PILOT_CONFIG[
            "validation_case_count"
        ]
        + PHASE45_PILOT_CONFIG[
            "training_case_count"
        ]
    ]
)

assert len(
    np.intersect1d(
        phase45_pilot_training_indices,
        phase45_pilot_validation_indices,
    )
) == 0

assert np.all(
    phase45_original_outer_fold[
        phase45_pilot_training_indices
    ] != 0
)

assert np.all(
    phase45_original_outer_fold[
        phase45_pilot_validation_indices
    ] != 0
)


# ------------------------------------------------------------
# 2. Memory-mapped label-free dataset
# ------------------------------------------------------------

class Phase45PilotDataset(Dataset):
    def __init__(
        self,
        cache_path,
        indices,
        augment,
        reflection_probability,
        intensity_scale_range,
        noise_standard_deviation,
    ):
        self.cache_path = str(cache_path)
        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )
        self.augment = bool(augment)
        self.reflection_probability = float(
            reflection_probability
        )
        self.intensity_scale_range = tuple(
            intensity_scale_range
        )
        self.noise_standard_deviation = float(
            noise_standard_deviation
        )
        self._cache = None

    def __len__(self):
        return len(self.indices)

    def _resolve_cache(self):
        if self._cache is None:
            self._cache = np.load(
                self.cache_path,
                mmap_mode="r",
                allow_pickle=False,
            )

        return self._cache

    def __getitem__(self, position):
        cache = self._resolve_cache()

        volume = np.asarray(
            cache[
                int(self.indices[position])
            ],
            dtype=np.float32,
        ).copy()

        volume = torch.from_numpy(
            volume
        ).unsqueeze(0)

        if self.augment:
            if (
                torch.rand(()).item()
                < self.reflection_probability
            ):
                # Unbatched tensor layout is [C, D, H, W].
                volume = torch.flip(
                    volume,
                    dims=[1],
                )

            minimum_scale, maximum_scale = (
                self.intensity_scale_range
            )

            intensity_scale = (
                minimum_scale
                + (
                    maximum_scale
                    - minimum_scale
                )
                * torch.rand(()).item()
            )

            volume = volume * intensity_scale

            if self.noise_standard_deviation > 0:
                foreground = (
                    volume > 1e-7
                ).to(volume.dtype)

                volume = (
                    volume
                    + foreground
                    * torch.randn_like(volume)
                    * self.noise_standard_deviation
                )

            volume = torch.clamp(
                volume,
                min=0.0,
                max=1.5,
            )

        return volume


phase45_pilot_training_dataset = (
    Phase45PilotDataset(
        cache_path=(
            PHASE45_HIGHRES_CACHE_PATH_PRIVATE
        ),
        indices=(
            phase45_pilot_training_indices
        ),
        augment=True,
        reflection_probability=(
            PHASE45_PILOT_CONFIG[
                "reflection_probability"
            ]
        ),
        intensity_scale_range=(
            PHASE45_PILOT_CONFIG[
                "intensity_scale_range"
            ]
        ),
        noise_standard_deviation=(
            PHASE45_PILOT_CONFIG[
                "noise_standard_deviation"
            ]
        ),
    )
)

phase45_pilot_validation_dataset = (
    Phase45PilotDataset(
        cache_path=(
            PHASE45_HIGHRES_CACHE_PATH_PRIVATE
        ),
        indices=(
            phase45_pilot_validation_indices
        ),
        augment=False,
        reflection_probability=0.0,
        intensity_scale_range=[1.0, 1.0],
        noise_standard_deviation=0.0,
    )
)


def phase45_worker_seed(worker_id):
    worker_seed = (
        PHASE45_PILOT_CONFIG[
            "random_seed"
        ]
        + 1000
        + worker_id
    )

    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)


phase45_training_generator = (
    torch.Generator()
)
phase45_training_generator.manual_seed(
    PHASE45_PILOT_CONFIG["random_seed"]
)

phase45_pilot_training_loader = DataLoader(
    phase45_pilot_training_dataset,
    batch_size=PHASE45_PILOT_CONFIG[
        "batch_size"
    ],
    shuffle=True,
    drop_last=True,
    num_workers=PHASE45_PILOT_CONFIG[
        "worker_count"
    ],
    pin_memory=(
        PHASE45_DEVICE.type == "cuda"
    ),
    persistent_workers=False,
    worker_init_fn=phase45_worker_seed,
    generator=phase45_training_generator,
)

phase45_pilot_validation_loader = DataLoader(
    phase45_pilot_validation_dataset,
    batch_size=PHASE45_PILOT_CONFIG[
        "batch_size"
    ],
    shuffle=False,
    drop_last=False,
    num_workers=PHASE45_PILOT_CONFIG[
        "worker_count"
    ],
    pin_memory=(
        PHASE45_DEVICE.type == "cuda"
    ),
    persistent_workers=False,
    worker_init_fn=phase45_worker_seed,
)


# ------------------------------------------------------------
# 3. Fresh pilot model
# ------------------------------------------------------------

def phase45_create_pretraining_model():
    model = (
        Phase45SpectralMaskedAutoencoder3D(
            input_channels=(
                PHASE45_MODEL_CONFIG[
                    "input_channels"
                ]
            ),
            input_size=(
                PHASE45_MODEL_CONFIG[
                    "input_size"
                ]
            ),
            patch_size=(
                PHASE45_MODEL_CONFIG[
                    "patch_size"
                ]
            ),
            encoder_dimension=(
                PHASE45_MODEL_CONFIG[
                    "encoder_dimension"
                ]
            ),
            encoder_depth=(
                PHASE45_MODEL_CONFIG[
                    "encoder_depth"
                ]
            ),
            encoder_head_count=(
                PHASE45_MODEL_CONFIG[
                    "encoder_head_count"
                ]
            ),
            decoder_dimension=(
                PHASE45_MODEL_CONFIG[
                    "decoder_dimension"
                ]
            ),
            decoder_depth=(
                PHASE45_MODEL_CONFIG[
                    "decoder_depth"
                ]
            ),
            decoder_head_count=(
                PHASE45_MODEL_CONFIG[
                    "decoder_head_count"
                ]
            ),
            dropout=(
                PHASE45_MODEL_CONFIG[
                    "dropout"
                ]
            ),
            symmetry_dimension=(
                PHASE45_MODEL_CONFIG[
                    "symmetry_dimension"
                ]
            ),
        )
    )

    # Learned reflection projectors are deliberately unused.
    for parameter_name, parameter in (
        model.named_parameters()
    ):
        if (
            parameter_name.startswith(
                "symmetric_projector."
            )
            or parameter_name.startswith(
                "antisymmetric_projector."
            )
        ):
            parameter.requires_grad_(False)

    return model


torch.manual_seed(
    PHASE45_PILOT_CONFIG["random_seed"]
)

if PHASE45_DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(
        PHASE45_PILOT_CONFIG[
            "random_seed"
        ]
    )
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

phase45_pilot_model = (
    phase45_create_pretraining_model()
    .to(PHASE45_DEVICE)
)

phase45_pilot_spectral_loss = (
    Phase45BalancedSpectralLoss3D(
        spatial_size=PHASE45_MODEL_CONFIG[
            "input_size"
        ]
    ).to(PHASE45_DEVICE)
)

phase45_pilot_parameters = [
    parameter
    for parameter
    in phase45_pilot_model.parameters()
    if parameter.requires_grad
]

phase45_pilot_optimizer = (
    torch.optim.AdamW(
        phase45_pilot_parameters,
        lr=PHASE45_PILOT_CONFIG[
            "learning_rate"
        ],
        weight_decay=PHASE45_PILOT_CONFIG[
            "weight_decay"
        ],
        betas=(0.9, 0.95),
    )
)

phase45_total_training_steps = (
    len(phase45_pilot_training_loader)
    * PHASE45_PILOT_CONFIG[
        "epoch_count"
    ]
)

phase45_warmup_steps = max(
    1,
    int(
        round(
            phase45_total_training_steps
            * PHASE45_PILOT_CONFIG[
                "warmup_fraction"
            ]
        )
    ),
)


def phase45_learning_rate(step):
    maximum_lr = PHASE45_PILOT_CONFIG[
        "learning_rate"
    ]
    minimum_lr = PHASE45_PILOT_CONFIG[
        "minimum_learning_rate"
    ]

    if step < phase45_warmup_steps:
        return maximum_lr * (
            (step + 1)
            / phase45_warmup_steps
        )

    denominator = max(
        1,
        phase45_total_training_steps
        - phase45_warmup_steps,
    )

    progress = min(
        1.0,
        (
            step - phase45_warmup_steps
        ) / denominator,
    )

    cosine_value = 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

    return (
        minimum_lr
        + (
            maximum_lr - minimum_lr
        )
        * cosine_value
    )


# ------------------------------------------------------------
# 4. Loss computation
# ------------------------------------------------------------

def phase45_pretraining_loss(
    model,
    spectral_loss_module,
    batch,
):
    amp_enabled = (
        PHASE45_DEVICE.type == "cuda"
    )

    with torch.autocast(
        device_type=PHASE45_DEVICE.type,
        dtype=torch.bfloat16,
        enabled=amp_enabled,
    ):
        outputs = model(
            batch,
            mask_ratio=(
                PHASE45_PILOT_CONFIG[
                    "mask_ratio"
                ]
            ),
        )

        target_patches = model.patchify(
            batch
        )

        spatial_loss = (
            phase45_masked_spatial_loss(
                predicted_patches=outputs[
                    "predicted_patches"
                ],
                target_patches=target_patches,
                mask=outputs["mask"],
            )
        )

        completed_patches = torch.where(
            outputs["mask"].unsqueeze(-1),
            outputs["predicted_patches"],
            target_patches,
        )

        reconstruction = model.unpatchify(
            completed_patches
        )

    # CUDA FFT does not reliably support bfloat16 for this
    # spatial size, so spectral loss explicitly uses float32.
    with torch.autocast(
        device_type=PHASE45_DEVICE.type,
        enabled=False,
    ):
        spectral_loss, spectral_bands = (
            spectral_loss_module(
                reconstruction.float(),
                batch.float(),
            )
        )

        total_loss = (
            PHASE45_PILOT_CONFIG[
                "spatial_loss_weight"
            ]
            * spatial_loss.float()
            + PHASE45_PILOT_CONFIG[
                "spectral_loss_weight"
            ]
            * spectral_loss
        )

    return {
        "total": total_loss,
        "spatial": spatial_loss.float(),
        "spectral": spectral_loss,
        "spectral_bands": (
            spectral_bands
        ),
    }


# ------------------------------------------------------------
# 5. Deterministic label-free validation
# ------------------------------------------------------------

@torch.no_grad()
def phase45_evaluate_pilot():
    phase45_pilot_model.eval()

    # Reset masking RNG so every validation pass uses the same
    # masks and is directly comparable across epochs.
    torch.manual_seed(450799)

    if PHASE45_DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(450799)

    totals = {
        "total": 0.0,
        "spatial": 0.0,
        "spectral": 0.0,
        "low": 0.0,
        "middle": 0.0,
        "high": 0.0,
        "n": 0,
    }

    for batch in (
        phase45_pilot_validation_loader
    ):
        batch = batch.to(
            PHASE45_DEVICE,
            non_blocking=True,
        )

        losses = phase45_pretraining_loss(
            model=phase45_pilot_model,
            spectral_loss_module=(
                phase45_pilot_spectral_loss
            ),
            batch=batch,
        )

        batch_size = len(batch)

        totals["total"] += (
            float(
                losses["total"]
                .detach().cpu()
            )
            * batch_size
        )
        totals["spatial"] += (
            float(
                losses["spatial"]
                .detach().cpu()
            )
            * batch_size
        )
        totals["spectral"] += (
            float(
                losses["spectral"]
                .detach().cpu()
            )
            * batch_size
        )
        totals["low"] += (
            float(
                losses["spectral_bands"][0]
                .detach().cpu()
            )
            * batch_size
        )
        totals["middle"] += (
            float(
                losses["spectral_bands"][1]
                .detach().cpu()
            )
            * batch_size
        )
        totals["high"] += (
            float(
                losses["spectral_bands"][2]
                .detach().cpu()
            )
            * batch_size
        )
        totals["n"] += batch_size

    assert totals["n"] == len(
        phase45_pilot_validation_dataset
    )

    return {
        key: float(value / totals["n"])
        for key, value in totals.items()
        if key != "n"
    }


# ------------------------------------------------------------
# 6. Pilot training
# ------------------------------------------------------------

phase45_initial_validation = (
    phase45_evaluate_pilot()
)

phase45_pilot_history = []
phase45_global_step = 0
phase45_best_validation_loss = float(
    phase45_initial_validation["total"]
)
phase45_best_epoch = -1
phase45_best_state = {
    key: value.detach().cpu().clone()
    for key, value
    in phase45_pilot_model.state_dict().items()
}

for epoch_index in range(
    PHASE45_PILOT_CONFIG["epoch_count"]
):
    epoch_started = time.perf_counter()

    torch.manual_seed(
        PHASE45_PILOT_CONFIG[
            "random_seed"
        ]
        + epoch_index
    )

    if PHASE45_DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(
            PHASE45_PILOT_CONFIG[
                "random_seed"
            ]
            + epoch_index
        )

    phase45_pilot_model.train()

    epoch_total_loss = 0.0
    epoch_spatial_loss = 0.0
    epoch_spectral_loss = 0.0
    epoch_case_count = 0
    epoch_gradient_norms = []

    for batch in (
        phase45_pilot_training_loader
    ):
        batch = batch.to(
            PHASE45_DEVICE,
            non_blocking=True,
        )

        current_learning_rate = (
            phase45_learning_rate(
                phase45_global_step
            )
        )

        for parameter_group in (
            phase45_pilot_optimizer
            .param_groups
        ):
            parameter_group["lr"] = (
                current_learning_rate
            )

        phase45_pilot_optimizer.zero_grad(
            set_to_none=True
        )

        losses = phase45_pretraining_loss(
            model=phase45_pilot_model,
            spectral_loss_module=(
                phase45_pilot_spectral_loss
            ),
            batch=batch,
        )

        assert torch.isfinite(
            losses["total"]
        )

        losses["total"].backward()

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                phase45_pilot_parameters,
                max_norm=(
                    PHASE45_PILOT_CONFIG[
                        "gradient_clip"
                    ]
                ),
            )
        )

        assert torch.isfinite(
            gradient_norm
        )

        phase45_pilot_optimizer.step()

        batch_size = len(batch)

        epoch_total_loss += (
            float(
                losses["total"]
                .detach().cpu()
            )
            * batch_size
        )
        epoch_spatial_loss += (
            float(
                losses["spatial"]
                .detach().cpu()
            )
            * batch_size
        )
        epoch_spectral_loss += (
            float(
                losses["spectral"]
                .detach().cpu()
            )
            * batch_size
        )
        epoch_case_count += batch_size

        epoch_gradient_norms.append(
            float(
                gradient_norm.detach().cpu()
            )
        )

        phase45_global_step += 1

    validation_metrics = (
        phase45_evaluate_pilot()
    )

    if (
        validation_metrics["total"]
        < phase45_best_validation_loss
    ):
        phase45_best_validation_loss = float(
            validation_metrics["total"]
        )
        phase45_best_epoch = int(
            epoch_index
        )
        phase45_best_state = {
            key: value.detach().cpu().clone()
            for key, value
            in phase45_pilot_model
            .state_dict().items()
        }

    epoch_record = {
        "epoch": int(epoch_index + 1),
        "train": {
            "total": float(
                epoch_total_loss
                / epoch_case_count
            ),
            "spatial": float(
                epoch_spatial_loss
                / epoch_case_count
            ),
            "spectral": float(
                epoch_spectral_loss
                / epoch_case_count
            ),
        },
        "validation": validation_metrics,
        "learning_rate_end": float(
            current_learning_rate
        ),
        "mean_gradient_norm": float(
            np.mean(
                epoch_gradient_norms
            )
        ),
        "maximum_gradient_norm": float(
            np.max(
                epoch_gradient_norms
            )
        ),
        "elapsed_seconds": round(
            time.perf_counter()
            - epoch_started,
            3,
        ),
    }

    phase45_pilot_history.append(
        epoch_record
    )

    print(
        "Phase45 pilot epoch "
        f"{epoch_index + 1}/"
        f"{PHASE45_PILOT_CONFIG['epoch_count']}: "
        f"train={epoch_record['train']['total']:.6f}, "
        f"valid={validation_metrics['total']:.6f}"
    )


# ------------------------------------------------------------
# 7. Pilot gate
# ------------------------------------------------------------

phase45_best_validation_record = min(
    [
        {
            "epoch": 0,
            **phase45_initial_validation,
        }
    ]
    + [
        {
            "epoch": record["epoch"],
            **record["validation"],
        }
        for record in phase45_pilot_history
    ],
    key=lambda record: record["total"],
)

phase45_total_validation_gain = (
    phase45_initial_validation["total"]
    - phase45_best_validation_record[
        "total"
    ]
) / phase45_initial_validation["total"]

phase45_spatial_validation_gain = (
    phase45_initial_validation["spatial"]
    - phase45_best_validation_record[
        "spatial"
    ]
) / phase45_initial_validation["spatial"]

phase45_spectral_band_gains = {
    band_name: (
        phase45_initial_validation[
            band_name
        ]
        - phase45_best_validation_record[
            band_name
        ]
    ) / phase45_initial_validation[
        band_name
    ]
    for band_name in [
        "low",
        "middle",
        "high",
    ]
}

phase45_improving_band_count = int(
    sum(
        gain > 0.0
        for gain
        in phase45_spectral_band_gains.values()
    )
)

phase45_pilot_gate_passed = bool(
    phase45_total_validation_gain
    >= PHASE45_PILOT_CONFIG[
        "minimum_total_validation_gain"
    ]
    and phase45_spatial_validation_gain
    >= PHASE45_PILOT_CONFIG[
        "minimum_spatial_validation_gain"
    ]
    and phase45_improving_band_count
    >= PHASE45_PILOT_CONFIG[
        "minimum_improving_spectral_band_count"
    ]
)

PHASE45_PILOT_BEST_STATE_PRIVATE = (
    phase45_best_state
)
PHASE45_PILOT_HISTORY_PRIVATE = (
    phase45_pilot_history
)
PHASE45_PILOT_SPLIT_PRIVATE = {
    "training_indices": (
        phase45_pilot_training_indices.copy()
    ),
    "validation_indices": (
        phase45_pilot_validation_indices.copy()
    ),
}


# ------------------------------------------------------------
# 8. Sanitized pilot result
# ------------------------------------------------------------

phase45_pilot_result = {
    "phase": (
        "phase45_label_free_spectral_mae_pilot"
    ),
    "status": (
        "pilot_gate_passed"
        if phase45_pilot_gate_passed
        else "pilot_gate_failed"
    ),
    "partition": {
        "source": (
            "original_outer_fold_0_train_only"
        ),
        "available_n": int(
            len(
                phase45_available_pilot_indices
            )
        ),
        "pilot_train_n": int(
            len(
                phase45_pilot_training_indices
            )
        ),
        "pilot_validation_n": int(
            len(
                phase45_pilot_validation_indices
            )
        ),
        "outer_validation_images_used": False,
    },
    "configuration": (
        PHASE45_PILOT_CONFIG
    ),
    "model": {
        "effective_trainable_parameter_count": int(
            sum(
                parameter.numel()
                for parameter
                in phase45_pilot_model.parameters()
                if parameter.requires_grad
            )
        ),
        "mixed_precision": (
            "bfloat16_except_float32_fft"
            if PHASE45_DEVICE.type == "cuda"
            else "float32"
        ),
    },
    "initial_validation": (
        phase45_initial_validation
    ),
    "best_validation": (
        phase45_best_validation_record
    ),
    "gains": {
        "relative_total": float(
            phase45_total_validation_gain
        ),
        "relative_spatial": float(
            phase45_spatial_validation_gain
        ),
        "relative_spectral_bands": (
            phase45_spectral_band_gains
        ),
        "improving_spectral_band_count": int(
            phase45_improving_band_count
        ),
    },
    "history": (
        phase45_pilot_history
    ),
    "gate_thresholds": {
        "minimum_total_validation_gain": (
            PHASE45_PILOT_CONFIG[
                "minimum_total_validation_gain"
            ]
        ),
        "minimum_spatial_validation_gain": (
            PHASE45_PILOT_CONFIG[
                "minimum_spatial_validation_gain"
            ]
        ),
        "minimum_improving_spectral_band_count": (
            PHASE45_PILOT_CONFIG[
                "minimum_improving_spectral_band_count"
            ]
        ),
    },
    "pilot_gate_passed": bool(
        phase45_pilot_gate_passed
    ),
    "peak_vram_mb": (
        float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )
        if PHASE45_DEVICE.type == "cuda"
        else 0.0
    ),
    "labels_used": False,
    "outer_validation_images_used": False,
    "outer_validation_labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_pilot_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_PILOT"
)
print(
    json.dumps(
        phase45_pilot_result,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_PILOT"
)

AssertionError: {'message': 'Phase45 pilot dependencies are missing.', 'missing': ['PHASE45_HIGHRES_CACHE_PATH_PRIVATE', 'PHASE44_ORIGINAL_FOLD_PRIVATE', 'Phase45SpectralMaskedAutoencoder3D', 'Phase45BalancedSpectralLoss3D', 'phase45_masked_spatial_loss', 'PHASE45_MODEL_CONFIG', 'PHASE45_PRETRAINING_CONFIG_PRIVATE', 'PHASE45_DEVICE']}

In [43]:
# Cell 142A — Phase45 

import json
import time

import numpy as np
import torch


phase45_training_contract_started = (
    time.perf_counter()
)

required_state = [
    "PHASE44_ORIGINAL_FOLD_PRIVATE",
    "PHASE44_GROUPS_PRIVATE",
    "PHASE45_HIGHRES_CACHE_PRIVATE",
    "PHASE45_MODEL_CONFIG",
    "PHASE45_PRETRAINING_CONFIG_PRIVATE",
    "phase45_pilot_result",
]

missing_state = [
    name
    for name in required_state
    if name not in globals()
]

assert not missing_state, {
    "message": (
        "Phase45 training-contract dependencies "
        "are missing."
    ),
    "missing": missing_state,
}

assert phase45_pilot_result[
    "pilot_gate_passed"
] is True


# ------------------------------------------------------------
# 1. Frozen full-pretraining configuration
# ------------------------------------------------------------

PHASE45_FULL_PRETRAINING_CONFIG = {
    "fold_count": 3,
    "fold_seeds": [
        451101,
        451102,
        451103,
    ],
    "selection_validation_fraction": 0.12,
    "batch_size": 8,
    "worker_count": 4,
    "maximum_selection_epochs": 16,
    "minimum_selection_epochs": 6,
    "early_stopping_patience": 3,
    "minimum_relative_improvement": 0.002,
    "learning_rate": 1.5e-4,
    "minimum_learning_rate": 1.5e-5,
    "warmup_fraction": 0.10,
    "weight_decay": 0.05,
    "optimizer_betas": [0.9, 0.95],
    "gradient_clip": 1.0,
    "mask_ratio": 0.75,
    "reflection_probability": 0.5,
    "intensity_scale_range": [
        0.98,
        1.02,
    ],
    "noise_standard_deviation": 0.003,
    "spatial_loss_weight": 1.0,
    "spectral_loss_weight": 0.25,
    "learned_reflection_loss_weight": 0.0,
    "mixed_precision": (
        "bfloat16_except_float32_fft"
        if torch.cuda.is_available()
        else "float32"
    ),
    "model_selection": (
        "label_free_group_balanced_"
        "reconstruction_validation"
    ),
    "final_refit": (
        "fresh_initialization_on_complete_"
        "outer_train_for_selected_epoch_count"
    ),
}


# ------------------------------------------------------------
# 2. Construct group-balanced label-free monitor splits
# ------------------------------------------------------------

phase45_original_fold = np.asarray(
    PHASE44_ORIGINAL_FOLD_PRIVATE,
    dtype=np.int64,
)

phase45_groups = np.asarray(
    PHASE44_GROUPS_PRIVATE,
    dtype=np.int64,
)

assert phase45_original_fold.shape == (
    1362,
)
assert phase45_groups.shape == (1362,)

PHASE45_SSL_PARTITIONS_PRIVATE = []
phase45_partition_records = []

for fold_value in range(
    PHASE45_FULL_PRETRAINING_CONFIG[
        "fold_count"
    ]
):
    fold_seed = (
        PHASE45_FULL_PRETRAINING_CONFIG[
            "fold_seeds"
        ][fold_value]
    )

    fold_rng = np.random.default_rng(
        fold_seed
    )

    outer_valid_indices = np.flatnonzero(
        phase45_original_fold == fold_value
    )

    outer_train_indices = np.flatnonzero(
        phase45_original_fold != fold_value
    )

    selection_validation_parts = []

    for group_value in sorted(
        np.unique(
            phase45_groups[
                outer_train_indices
            ]
        ).tolist()
    ):
        group_indices = outer_train_indices[
            phase45_groups[
                outer_train_indices
            ] == group_value
        ]

        shuffled_group_indices = (
            fold_rng.permutation(
                group_indices
            )
        )

        group_validation_count = max(
            1,
            int(
                round(
                    len(group_indices)
                    * PHASE45_FULL_PRETRAINING_CONFIG[
                        "selection_validation_fraction"
                    ]
                )
            ),
        )

        # Every represented group retains at least one
        # optimization image.
        group_validation_count = min(
            group_validation_count,
            len(group_indices) - 1,
        )

        assert group_validation_count >= 1

        selection_validation_parts.append(
            shuffled_group_indices[
                :group_validation_count
            ]
        )

    selection_validation_indices = np.sort(
        np.concatenate(
            selection_validation_parts
        )
    )

    selection_training_indices = np.setdiff1d(
        outer_train_indices,
        selection_validation_indices,
        assume_unique=True,
    )

    assert len(
        np.intersect1d(
            selection_training_indices,
            selection_validation_indices,
        )
    ) == 0

    assert len(
        np.intersect1d(
            outer_train_indices,
            outer_valid_indices,
        )
    ) == 0

    assert np.array_equal(
        np.sort(
            np.concatenate([
                selection_training_indices,
                selection_validation_indices,
            ])
        ),
        outer_train_indices,
    )

    training_groups = set(
        np.unique(
            phase45_groups[
                selection_training_indices
            ]
        ).tolist()
    )

    monitor_groups = set(
        np.unique(
            phase45_groups[
                selection_validation_indices
            ]
        ).tolist()
    )

    outer_valid_groups = set(
        np.unique(
            phase45_groups[
                outer_valid_indices
            ]
        ).tolist()
    )

    assert training_groups == monitor_groups
    assert not (
        outer_valid_groups
        & training_groups
    ), {
        "message": (
            "The original outer fold is not "
            "acquisition-group held out."
        ),
        "fold": int(fold_value),
        "shared_groups": sorted(
            outer_valid_groups
            & training_groups
        ),
    }

    partition = {
        "fold": int(fold_value),
        "seed": int(fold_seed),
        "selection_train_indices": (
            selection_training_indices.copy()
        ),
        "selection_valid_indices": (
            selection_validation_indices.copy()
        ),
        "outer_train_indices": (
            outer_train_indices.copy()
        ),
        "outer_valid_indices": (
            outer_valid_indices.copy()
        ),
    }

    PHASE45_SSL_PARTITIONS_PRIVATE.append(
        partition
    )

    phase45_partition_records.append({
        "fold": int(fold_value),
        "seed": int(fold_seed),
        "selection_train_n": int(
            len(selection_training_indices)
        ),
        "selection_valid_n": int(
            len(selection_validation_indices)
        ),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
        "selection_group_count": int(
            len(training_groups)
        ),
        "outer_valid_group_count": int(
            len(outer_valid_groups)
        ),
        "shared_group_count_between_outer_"
        "train_and_valid": 0,
    })


# ------------------------------------------------------------
# 3. Compute planned training volume
# ------------------------------------------------------------

phase45_maximum_selection_updates = int(
    sum(
        math.ceil(
            record["selection_train_n"]
            / PHASE45_FULL_PRETRAINING_CONFIG[
                "batch_size"
            ]
        )
        * PHASE45_FULL_PRETRAINING_CONFIG[
            "maximum_selection_epochs"
        ]
        for record in phase45_partition_records
    )
)

phase45_expected_sequential_model_count = 6


# ------------------------------------------------------------
# 4. Sanitized contract
# ------------------------------------------------------------

phase45_training_contract = {
    "phase": (
        "phase45_cross_fitted_spectral_mae_"
        "pretraining_contract"
    ),
    "status": "accepted",
    "pilot_evidence": {
        "total_validation_relative_gain": (
            phase45_pilot_result[
                "gains"
            ]["relative_total"]
        ),
        "spatial_validation_relative_gain": (
            phase45_pilot_result[
                "gains"
            ]["relative_spatial"]
        ),
        "spectral_band_relative_gains": (
            phase45_pilot_result[
                "gains"
            ][
                "relative_spectral_bands"
            ]
        ),
        "all_spectral_bands_improved": True,
    },
    "configuration": (
        PHASE45_FULL_PRETRAINING_CONFIG
    ),
    "partitions": (
        phase45_partition_records
    ),
    "architecture": {
        "patch_size": int(
            PHASE45_MODEL_CONFIG[
                "patch_size"
            ]
        ),
        "token_count": int(
            PHASE45_MODEL_CONFIG[
                "token_count"
            ]
        ),
        "visible_token_count": int(
            round(
                PHASE45_MODEL_CONFIG[
                    "token_count"
                ]
                * (
                    1.0
                    - PHASE45_FULL_PRETRAINING_CONFIG[
                        "mask_ratio"
                    ]
                )
            )
        ),
        "encoder_dimension": int(
            PHASE45_MODEL_CONFIG[
                "encoder_dimension"
            ]
        ),
        "downstream_reflection_invariant_"
        "dimension": int(
            PHASE45_PRETRAINING_CONFIG_PRIVATE[
                "downstream_dimension"
            ]
        ),
    },
    "planned_execution": {
        "selection_model_count": 3,
        "final_refit_model_count": 3,
        "maximum_sequential_model_count": int(
            phase45_expected_sequential_model_count
        ),
        "maximum_selection_optimizer_updates": (
            phase45_maximum_selection_updates
        ),
        "models_resident_on_gpu_simultaneously": 1,
        "selection_checkpoint": (
            "minimum_label_free_validation_loss"
        ),
        "refit_epoch_source": (
            "fold_specific_selected_epoch_count"
        ),
    },
    "reflection_policy": {
        "pretraining_random_reflection": True,
        "learned_projector_loss": False,
        "downstream_decomposition": (
            "analytic_symmetric_plus_absolute_"
            "antisymmetric"
        ),
    },
    "validation_semantics": {
        "outer_validation_images_used_for_"
        "pretraining_selection": False,
        "outer_validation_labels_used": False,
        "selection_labels_used": False,
        "group_balanced_label_free_monitor": True,
        "final_refit_uses_complete_outer_train": True,
        "held_out_group_encoder_for_each_oof_case": True,
    },
    "labels_used": False,
    "training_voxel_cache_read": False,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_embeddings_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_training_contract_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_TRAINING_CONTRACT"
)
print(
    json.dumps(
        phase45_training_contract,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_TRAINING_CONTRACT"
)

BEGIN SANITIZED_PHASE45_TRAINING_CONTRACT
{
  "phase": "phase45_cross_fitted_spectral_mae_pretraining_contract",
  "status": "accepted",
  "pilot_evidence": {
    "total_validation_relative_gain": 0.7302690356950637,
    "spatial_validation_relative_gain": 0.7808558732585635,
    "spectral_band_relative_gains": {
      "low": 0.3063962890132304,
      "middle": 0.5956728782855804,
      "high": 0.8335410917512588
    },
    "all_spectral_bands_improved": true
  },
  "configuration": {
    "fold_count": 3,
    "fold_seeds": [
      451101,
      451102,
      451103
    ],
    "selection_validation_fraction": 0.12,
    "batch_size": 8,
    "worker_count": 4,
    "maximum_selection_epochs": 16,
    "minimum_selection_epochs": 6,
    "early_stopping_patience": 3,
    "minimum_relative_improvement": 0.002,
    "learning_rate": 0.00015,
    "minimum_learning_rate": 1.5e-05,
    "warmup_fraction": 0.1,
    "weight_decay": 0.05,
    "optimizer_betas": [
      0.9,
      0.95
    ],
    "gradi

In [44]:
# Cell 142B — Phase45 

import gc
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


phase45_selection_started = time.perf_counter()

required_names = [
    "PHASE45_SSL_PARTITIONS_PRIVATE",
    "PHASE45_FULL_PRETRAINING_CONFIG",
    "PHASE45_HIGHRES_CACHE_PATH_PRIVATE",
    "PHASE45_MODEL_CONFIG",
    "PHASE45_DEVICE",
    "Phase45SpectralMaskedAutoencoder3D",
    "Phase45BalancedSpectralLoss3D",
    "phase45_masked_spatial_loss",
    "phase45_decode_amp_safe",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, {
    "message": "Phase45 selection dependencies are missing.",
    "missing": missing_names,
}

# Reapply the verified BF16-safe decoder in case a class cell
# was rerun after the patch.
Phase45SpectralMaskedAutoencoder3D.decode = (
    phase45_decode_amp_safe
)

assert (
    Phase45SpectralMaskedAutoencoder3D
    .decode.__name__
    == "phase45_decode_amp_safe"
)


# ------------------------------------------------------------
# 1. Full-training dataset
# ------------------------------------------------------------

class Phase45FullPretrainingDataset(Dataset):
    def __init__(
        self,
        cache_path,
        indices,
        augment,
        reflection_probability=0.0,
        intensity_scale_range=(1.0, 1.0),
        noise_standard_deviation=0.0,
    ):
        self.cache_path = str(cache_path)
        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )
        self.augment = bool(augment)
        self.reflection_probability = float(
            reflection_probability
        )
        self.intensity_scale_range = tuple(
            float(value)
            for value
            in intensity_scale_range
        )
        self.noise_standard_deviation = float(
            noise_standard_deviation
        )
        self._cache = None

    def __len__(self):
        return len(self.indices)

    def _get_cache(self):
        if self._cache is None:
            self._cache = np.load(
                self.cache_path,
                mmap_mode="r",
                allow_pickle=False,
            )

        return self._cache

    def __getitem__(self, position):
        cache = self._get_cache()

        volume = np.asarray(
            cache[
                int(self.indices[position])
            ],
            dtype=np.float32,
        ).copy()

        volume = torch.from_numpy(
            volume
        ).unsqueeze(0)

        if self.augment:
            if (
                torch.rand(()).item()
                < self.reflection_probability
            ):
                # [C, D, H, W]; cache axis 0 is D.
                volume = torch.flip(
                    volume,
                    dims=[1],
                )

            minimum_scale, maximum_scale = (
                self.intensity_scale_range
            )

            scale = (
                minimum_scale
                + torch.rand(()).item()
                * (
                    maximum_scale
                    - minimum_scale
                )
            )

            volume = volume * scale

            if self.noise_standard_deviation > 0:
                foreground = (
                    volume > 1e-7
                ).to(volume.dtype)

                volume = (
                    volume
                    + foreground
                    * torch.randn_like(volume)
                    * self.noise_standard_deviation
                )

            volume = torch.clamp(
                volume,
                min=0.0,
                max=1.5,
            )

        return volume


def phase45_full_worker_init(worker_id):
    worker_seed = (
        torch.initial_seed()
        % (2 ** 32)
    )

    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)


def phase45_make_loader(
    indices,
    augment,
    seed,
    shuffle,
    drop_last,
):
    config = (
        PHASE45_FULL_PRETRAINING_CONFIG
    )

    dataset = (
        Phase45FullPretrainingDataset(
            cache_path=(
                PHASE45_HIGHRES_CACHE_PATH_PRIVATE
            ),
            indices=indices,
            augment=augment,
            reflection_probability=(
                config[
                    "reflection_probability"
                ]
                if augment
                else 0.0
            ),
            intensity_scale_range=(
                config[
                    "intensity_scale_range"
                ]
                if augment
                else (1.0, 1.0)
            ),
            noise_standard_deviation=(
                config[
                    "noise_standard_deviation"
                ]
                if augment
                else 0.0
            ),
        )
    )

    generator = torch.Generator()
    generator.manual_seed(int(seed))

    return DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=bool(shuffle),
        drop_last=bool(drop_last),
        num_workers=config["worker_count"],
        pin_memory=(
            PHASE45_DEVICE.type == "cuda"
        ),
        persistent_workers=False,
        worker_init_fn=(
            phase45_full_worker_init
        ),
        generator=generator,
    )


# ------------------------------------------------------------
# 2. Model factory
# ------------------------------------------------------------

def phase45_new_ssl_model():
    model = (
        Phase45SpectralMaskedAutoencoder3D(
            input_channels=(
                PHASE45_MODEL_CONFIG[
                    "input_channels"
                ]
            ),
            input_size=(
                PHASE45_MODEL_CONFIG[
                    "input_size"
                ]
            ),
            patch_size=(
                PHASE45_MODEL_CONFIG[
                    "patch_size"
                ]
            ),
            encoder_dimension=(
                PHASE45_MODEL_CONFIG[
                    "encoder_dimension"
                ]
            ),
            encoder_depth=(
                PHASE45_MODEL_CONFIG[
                    "encoder_depth"
                ]
            ),
            encoder_head_count=(
                PHASE45_MODEL_CONFIG[
                    "encoder_head_count"
                ]
            ),
            decoder_dimension=(
                PHASE45_MODEL_CONFIG[
                    "decoder_dimension"
                ]
            ),
            decoder_depth=(
                PHASE45_MODEL_CONFIG[
                    "decoder_depth"
                ]
            ),
            decoder_head_count=(
                PHASE45_MODEL_CONFIG[
                    "decoder_head_count"
                ]
            ),
            dropout=(
                PHASE45_MODEL_CONFIG[
                    "dropout"
                ]
            ),
            symmetry_dimension=(
                PHASE45_MODEL_CONFIG[
                    "symmetry_dimension"
                ]
            ),
        )
    )

    for parameter_name, parameter in (
        model.named_parameters()
    ):
        if (
            parameter_name.startswith(
                "symmetric_projector."
            )
            or parameter_name.startswith(
                "antisymmetric_projector."
            )
        ):
            parameter.requires_grad_(False)

    return model


# ------------------------------------------------------------
# 3. Loss function
# ------------------------------------------------------------

def phase45_full_ssl_loss(
    model,
    spectral_loss_module,
    batch,
):
    config = (
        PHASE45_FULL_PRETRAINING_CONFIG
    )

    amp_enabled = (
        PHASE45_DEVICE.type == "cuda"
    )

    with torch.autocast(
        device_type=PHASE45_DEVICE.type,
        dtype=torch.bfloat16,
        enabled=amp_enabled,
    ):
        outputs = model(
            batch,
            mask_ratio=config[
                "mask_ratio"
            ],
        )

        target_patches = model.patchify(
            batch
        )

        spatial_loss = (
            phase45_masked_spatial_loss(
                predicted_patches=outputs[
                    "predicted_patches"
                ],
                target_patches=target_patches,
                mask=outputs["mask"],
            )
        )

        completed_patches = torch.where(
            outputs["mask"].unsqueeze(-1),
            outputs["predicted_patches"],
            target_patches,
        )

        reconstruction = model.unpatchify(
            completed_patches
        )

    with torch.autocast(
        device_type=PHASE45_DEVICE.type,
        enabled=False,
    ):
        (
            spectral_loss,
            spectral_band_losses,
        ) = spectral_loss_module(
            reconstruction.float(),
            batch.float(),
        )

        total_loss = (
            config[
                "spatial_loss_weight"
            ]
            * spatial_loss.float()
            + config[
                "spectral_loss_weight"
            ]
            * spectral_loss
        )

    return {
        "total": total_loss,
        "spatial": spatial_loss.float(),
        "spectral": spectral_loss,
        "bands": spectral_band_losses,
    }


# ------------------------------------------------------------
# 4. Deterministic reconstruction evaluation
# ------------------------------------------------------------

@torch.no_grad()
def phase45_evaluate_ssl(
    model,
    loader,
    spectral_loss_module,
    mask_seed,
):
    model.eval()

    torch.manual_seed(int(mask_seed))

    if PHASE45_DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(
            int(mask_seed)
        )

    sums = {
        "total": 0.0,
        "spatial": 0.0,
        "spectral": 0.0,
        "low": 0.0,
        "middle": 0.0,
        "high": 0.0,
    }
    case_count = 0

    for batch in loader:
        batch = batch.to(
            PHASE45_DEVICE,
            non_blocking=True,
        )

        losses = phase45_full_ssl_loss(
            model=model,
            spectral_loss_module=(
                spectral_loss_module
            ),
            batch=batch,
        )

        current_batch_size = len(batch)

        values = {
            "total": losses["total"],
            "spatial": losses["spatial"],
            "spectral": losses["spectral"],
            "low": losses["bands"][0],
            "middle": losses["bands"][1],
            "high": losses["bands"][2],
        }

        for name, value in values.items():
            assert torch.isfinite(value)

            sums[name] += (
                float(
                    value.detach().cpu()
                )
                * current_batch_size
            )

        case_count += current_batch_size

    assert case_count == len(loader.dataset)

    return {
        name: float(value / case_count)
        for name, value in sums.items()
    }


# ------------------------------------------------------------
# 5. Fold-specific label-free selection run
# ------------------------------------------------------------

def phase45_select_ssl_epoch(partition):
    fold_value = int(partition["fold"])
    fold_seed = int(partition["seed"])
    config = (
        PHASE45_FULL_PRETRAINING_CONFIG
    )

    fold_started = time.perf_counter()

    train_loader = phase45_make_loader(
        indices=partition[
            "selection_train_indices"
        ],
        augment=True,
        seed=fold_seed,
        shuffle=True,
        drop_last=True,
    )

    valid_loader = phase45_make_loader(
        indices=partition[
            "selection_valid_indices"
        ],
        augment=False,
        seed=fold_seed + 1,
        shuffle=False,
        drop_last=False,
    )

    torch.manual_seed(fold_seed)

    if PHASE45_DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(
            fold_seed
        )
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = phase45_new_ssl_model().to(
        PHASE45_DEVICE
    )

    spectral_loss_module = (
        Phase45BalancedSpectralLoss3D(
            spatial_size=(
                PHASE45_MODEL_CONFIG[
                    "input_size"
                ]
            )
        ).to(PHASE45_DEVICE)
    )

    trainable_parameters = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=config["learning_rate"],
        weight_decay=config[
            "weight_decay"
        ],
        betas=tuple(
            config["optimizer_betas"]
        ),
    )

    total_steps = (
        len(train_loader)
        * config[
            "maximum_selection_epochs"
        ]
    )

    warmup_steps = max(
        1,
        int(
            round(
                total_steps
                * config[
                    "warmup_fraction"
                ]
            )
        ),
    )

    def learning_rate_at(step):
        if step < warmup_steps:
            return (
                config["learning_rate"]
                * (step + 1)
                / warmup_steps
            )

        denominator = max(
            1,
            total_steps - warmup_steps,
        )

        progress = min(
            1.0,
            (
                step - warmup_steps
            ) / denominator,
        )

        cosine_multiplier = 0.5 * (
            1.0
            + math.cos(
                math.pi * progress
            )
        )

        return (
            config[
                "minimum_learning_rate"
            ]
            + (
                config["learning_rate"]
                - config[
                    "minimum_learning_rate"
                ]
            )
            * cosine_multiplier
        )

    validation_seed = (
        fold_seed + 50000
    )

    initial_validation = (
        phase45_evaluate_ssl(
            model=model,
            loader=valid_loader,
            spectral_loss_module=(
                spectral_loss_module
            ),
            mask_seed=validation_seed,
        )
    )

    best_loss = float(
        initial_validation["total"]
    )
    best_epoch = 0

    patience_reference_loss = best_loss
    epochs_without_meaningful_improvement = 0
    global_step = 0
    history = []

    for epoch_index in range(
        config[
            "maximum_selection_epochs"
        ]
    ):
        epoch_started = time.perf_counter()

        epoch_seed = (
            fold_seed + epoch_index
        )

        torch.manual_seed(epoch_seed)

        if PHASE45_DEVICE.type == "cuda":
            torch.cuda.manual_seed_all(
                epoch_seed
            )

        model.train()

        training_sums = {
            "total": 0.0,
            "spatial": 0.0,
            "spectral": 0.0,
        }
        training_case_count = 0
        gradient_norms = []

        for batch in train_loader:
            batch = batch.to(
                PHASE45_DEVICE,
                non_blocking=True,
            )

            current_lr = learning_rate_at(
                global_step
            )

            for parameter_group in (
                optimizer.param_groups
            ):
                parameter_group["lr"] = (
                    current_lr
                )

            optimizer.zero_grad(
                set_to_none=True
            )

            losses = phase45_full_ssl_loss(
                model=model,
                spectral_loss_module=(
                    spectral_loss_module
                ),
                batch=batch,
            )

            assert torch.isfinite(
                losses["total"]
            )

            losses["total"].backward()

            gradient_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    trainable_parameters,
                    max_norm=config[
                        "gradient_clip"
                    ],
                )
            )

            assert torch.isfinite(
                gradient_norm
            )

            optimizer.step()

            current_batch_size = len(batch)

            for name in [
                "total",
                "spatial",
                "spectral",
            ]:
                training_sums[name] += (
                    float(
                        losses[name]
                        .detach().cpu()
                    )
                    * current_batch_size
                )

            training_case_count += (
                current_batch_size
            )

            gradient_norms.append(
                float(
                    gradient_norm
                    .detach().cpu()
                )
            )

            global_step += 1

        validation = phase45_evaluate_ssl(
            model=model,
            loader=valid_loader,
            spectral_loss_module=(
                spectral_loss_module
            ),
            mask_seed=validation_seed,
        )

        if validation["total"] < best_loss:
            best_loss = float(
                validation["total"]
            )
            best_epoch = int(
                epoch_index + 1
            )

        meaningful_threshold = (
            patience_reference_loss
            * (
                1.0
                - config[
                    "minimum_relative_improvement"
                ]
            )
        )

        if (
            validation["total"]
            < meaningful_threshold
        ):
            patience_reference_loss = float(
                validation["total"]
            )
            epochs_without_meaningful_improvement = 0
        else:
            epochs_without_meaningful_improvement += 1

        epoch_record = {
            "epoch": int(epoch_index + 1),
            "train": {
                name: float(
                    value
                    / training_case_count
                )
                for name, value
                in training_sums.items()
            },
            "validation": validation,
            "learning_rate_end": float(
                current_lr
            ),
            "mean_gradient_norm": float(
                np.mean(gradient_norms)
            ),
            "maximum_gradient_norm": float(
                np.max(gradient_norms)
            ),
            "elapsed_seconds": round(
                time.perf_counter()
                - epoch_started,
                3,
            ),
        }

        history.append(epoch_record)

        print(
            f"Phase45 selection fold {fold_value}, "
            f"epoch {epoch_index + 1}: "
            f"train={epoch_record['train']['total']:.6f}, "
            f"valid={validation['total']:.6f}"
        )

        minimum_epoch_reached = (
            epoch_index + 1
            >= config[
                "minimum_selection_epochs"
            ]
        )

        patience_exhausted = (
            epochs_without_meaningful_improvement
            >= config[
                "early_stopping_patience"
            ]
        )

        if (
            minimum_epoch_reached
            and patience_exhausted
        ):
            break

    assert best_epoch > 0

    peak_vram_mb = (
        float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )
        if PHASE45_DEVICE.type == "cuda"
        else 0.0
    )

    result = {
        "fold": fold_value,
        "seed": fold_seed,
        "selection_train_n": int(
            len(
                partition[
                    "selection_train_indices"
                ]
            )
        ),
        "selection_valid_n": int(
            len(
                partition[
                    "selection_valid_indices"
                ]
            )
        ),
        "initial_validation": (
            initial_validation
        ),
        "selected_epoch_count": int(
            best_epoch
        ),
        "best_validation_total": float(
            best_loss
        ),
        "relative_validation_gain": float(
            (
                initial_validation["total"]
                - best_loss
            )
            / initial_validation["total"]
        ),
        "epochs_executed": int(
            len(history)
        ),
        "history": history,
        "peak_vram_mb": peak_vram_mb,
        "elapsed_seconds": round(
            time.perf_counter()
            - fold_started,
            3,
        ),
    }

    model.to("cpu")

    del model
    del optimizer
    del spectral_loss_module
    del train_loader
    del valid_loader

    gc.collect()

    if PHASE45_DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result


# ------------------------------------------------------------
# 6. Execute the three selection runs
# ------------------------------------------------------------

phase45_selection_records = []

for partition in (
    PHASE45_SSL_PARTITIONS_PRIVATE
):
    selection_record = (
        phase45_select_ssl_epoch(
            partition
        )
    )

    phase45_selection_records.append(
        selection_record
    )

    print(
        "Phase45 fold "
        f"{selection_record['fold']} selected "
        f"{selection_record['selected_epoch_count']} epochs"
    )

PHASE45_SSL_SELECTION_RECORDS_PRIVATE = (
    phase45_selection_records
)

PHASE45_SELECTED_EPOCHS_PRIVATE = [
    int(
        record[
            "selected_epoch_count"
        ]
    )
    for record
    in phase45_selection_records
]


# ------------------------------------------------------------
# 7. Persist sanitized selection state
# ------------------------------------------------------------

phase45_selection_checkpoint = {
    "phase": (
        "phase45_label_free_ssl_epoch_selection"
    ),
    "configuration": (
        PHASE45_FULL_PRETRAINING_CONFIG
    ),
    "selected_epochs": (
        PHASE45_SELECTED_EPOCHS_PRIVATE
    ),
    "folds": [
        {
            key: value
            for key, value
            in record.items()
            if key != "history"
        }
        for record in phase45_selection_records
    ],
}

phase45_selection_path = Path(
    "/kaggle/working/"
    "phase45_ssl_epoch_selection.json"
)

phase45_selection_temporary_path = (
    phase45_selection_path.with_suffix(
        ".json.tmp"
    )
)

phase45_selection_temporary_path.write_text(
    json.dumps(
        phase45_selection_checkpoint,
        indent=2,
    ),
    encoding="utf-8",
)

phase45_selection_temporary_path.replace(
    phase45_selection_path
)


# ------------------------------------------------------------
# 8. Sanitized result
# ------------------------------------------------------------

phase45_selection_result = {
    "phase": (
        "phase45_fold_specific_label_free_"
        "pretraining_selection"
    ),
    "status": "selection_complete",
    "fold_count": int(
        len(phase45_selection_records)
    ),
    "selected_epoch_counts": (
        PHASE45_SELECTED_EPOCHS_PRIVATE
    ),
    "folds": (
        phase45_selection_records
    ),
    "selection_checkpoint": (
        phase45_selection_path.name
    ),
    "next_step": (
        "fresh_outer_train_refit_and_"
        "cross_fitted_embedding_extraction"
    ),
    "labels_used": False,
    "outer_validation_images_used_for_selection": False,
    "outer_validation_labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_embeddings_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_selection_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_SSL_SELECTION"
)
print(
    json.dumps(
        phase45_selection_result,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_SSL_SELECTION"
)

Phase45 selection fold 0, epoch 1: train=0.251314, valid=0.150288
Phase45 selection fold 0, epoch 2: train=0.128523, valid=0.098106
Phase45 selection fold 0, epoch 3: train=0.102262, valid=0.094254
Phase45 selection fold 0, epoch 4: train=0.100197, valid=0.094894
Phase45 selection fold 0, epoch 5: train=0.096365, valid=0.089194
Phase45 selection fold 0, epoch 6: train=0.094363, valid=0.087690
Phase45 selection fold 0, epoch 7: train=0.092679, valid=0.086717
Phase45 selection fold 0, epoch 8: train=0.091960, valid=0.087047
Phase45 selection fold 0, epoch 9: train=0.090998, valid=0.084723
Phase45 selection fold 0, epoch 10: train=0.089869, valid=0.084891
Phase45 selection fold 0, epoch 11: train=0.089322, valid=0.083820
Phase45 selection fold 0, epoch 12: train=0.088364, valid=0.082874
Phase45 selection fold 0, epoch 13: train=0.087702, valid=0.082387
Phase45 selection fold 0, epoch 14: train=0.087328, valid=0.082121
Phase45 selection fold 0, epoch 15: train=0.086561, valid=0.081739
Phas

In [46]:
# Cell 142C — Phase45

import gc
import json
import math
import time
from pathlib import Path

import numpy as np
import torch


phase45_refit_started = time.perf_counter()

required_names = [
    "PHASE45_SSL_PARTITIONS_PRIVATE",
    "PHASE45_SELECTED_EPOCHS_PRIVATE",
    "PHASE45_FULL_PRETRAINING_CONFIG",
    "PHASE45_MODEL_CONFIG",
    "PHASE45_DEVICE",
    "PHASE45_HIGHRES_CACHE_PATH_PRIVATE",
    "phase45_make_loader",
    "phase45_new_ssl_model",
    "phase45_full_ssl_loss",
    "phase45_evaluate_ssl",
    "phase45_decode_amp_safe",
    "Phase45SpectralMaskedAutoencoder3D",
    "Phase45BalancedSpectralLoss3D",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, {
    "message": "Phase45 refit dependencies are missing.",
    "missing": missing_names,
}

assert list(
    PHASE45_SELECTED_EPOCHS_PRIVATE
) == [16, 15, 15]

# Reapply verified BF16 patch.
Phase45SpectralMaskedAutoencoder3D.decode = (
    phase45_decode_amp_safe
)

PHASE45_REFIT_CONFIG = {
    "refit_seed_offset": 100000,
    "embedding_batch_size": 8,
    "representation_components": [
        "original_embedding",
        "analytic_symmetric_embedding",
        "absolute_analytic_antisymmetric_embedding",
    ],
    "component_dimension": 192,
    "bundle_dimension": 576,
    "primary_invariant_dimension": 384,
    "checkpoint_directory": (
        "/kaggle/working/"
        "phase45_private_checkpoint"
    ),
    "embedding_cache": (
        "/kaggle/working/"
        "phase45_embedding_bundle_float16.npy"
    ),
}


# ------------------------------------------------------------
# 1. Atomic checkpoint directory
# ------------------------------------------------------------

phase45_checkpoint_directory = Path(
    PHASE45_REFIT_CONFIG[
        "checkpoint_directory"
    ]
)

phase45_checkpoint_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Encoder-only checkpoint extraction
# ------------------------------------------------------------

def phase45_encoder_state_dict(model):
    accepted_prefixes = (
        "patch_embedding.",
        "encoder_position",
        "encoder.",
        "encoder_norm.",
    )

    encoder_state = {}

    for key, value in (
        model.state_dict().items()
    ):
        if key.startswith(
            accepted_prefixes
        ):
            encoder_state[key] = (
                value.detach()
                .cpu()
                .contiguous()
                .clone()
            )

    assert "encoder_position" in encoder_state
    assert any(
        key.startswith(
            "patch_embedding."
        )
        for key in encoder_state
    )
    assert any(
        key.startswith("encoder.")
        for key in encoder_state
    )
    assert any(
        key.startswith("encoder_norm.")
        for key in encoder_state
    )

    assert all(
        torch.all(torch.isfinite(value))
        for value in encoder_state.values()
    )

    return encoder_state


def phase45_atomic_torch_save(
    payload,
    destination,
):
    destination = Path(destination)
    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    temporary_path.replace(destination)

    assert destination.is_file()


# ------------------------------------------------------------
# 3. Refit learning-rate schedule
# ------------------------------------------------------------

def phase45_refit_learning_rate(
    step,
    total_steps,
):
    config = (
        PHASE45_FULL_PRETRAINING_CONFIG
    )

    warmup_steps = max(
        1,
        int(
            round(
                total_steps
                * config[
                    "warmup_fraction"
                ]
            )
        ),
    )

    if step < warmup_steps:
        return (
            config["learning_rate"]
            * (step + 1)
            / warmup_steps
        )

    denominator = max(
        1,
        total_steps - warmup_steps,
    )

    progress = min(
        1.0,
        (
            step - warmup_steps
        ) / denominator,
    )

    cosine_multiplier = 0.5 * (
        1.0
        + math.cos(
            math.pi * progress
        )
    )

    return (
        config["minimum_learning_rate"]
        + (
            config["learning_rate"]
            - config[
                "minimum_learning_rate"
            ]
        )
        * cosine_multiplier
    )


# ------------------------------------------------------------
# 4. Extract original, symmetric, and asymmetry features
# ------------------------------------------------------------

@torch.no_grad()
def phase45_extract_fold_bundle(model):
    model.eval()

    all_indices = np.arange(
        1362,
        dtype=np.int64,
    )

    extraction_loader = phase45_make_loader(
        indices=all_indices,
        augment=False,
        seed=459999,
        shuffle=False,
        drop_last=False,
    )

    bundle = np.empty(
        (
            1362,
            PHASE45_REFIT_CONFIG[
                "bundle_dimension"
            ],
        ),
        dtype=np.float32,
    )

    cursor = 0
    amp_enabled = (
        PHASE45_DEVICE.type == "cuda"
    )

    for batch in extraction_loader:
        batch = batch.to(
            PHASE45_DEVICE,
            non_blocking=True,
        )

        reflected_batch = torch.flip(
            batch,
            dims=[
                PHASE45_MODEL_CONFIG[
                    "reflection_axis"
                ]
            ],
        )

        with torch.autocast(
            device_type=PHASE45_DEVICE.type,
            dtype=torch.bfloat16,
            enabled=amp_enabled,
        ):
            original_embedding = (
                model.encode_full(
                    batch
                )["embedding"]
            )

            reflected_embedding = (
                model.encode_full(
                    reflected_batch
                )["embedding"]
            )

        original_embedding = (
            original_embedding.float()
        )
        reflected_embedding = (
            reflected_embedding.float()
        )

        symmetric_embedding = 0.5 * (
            original_embedding
            + reflected_embedding
        )

        antisymmetric_embedding = 0.5 * (
            original_embedding
            - reflected_embedding
        )

        representation = torch.cat(
            [
                original_embedding,
                symmetric_embedding,
                torch.abs(
                    antisymmetric_embedding
                ),
            ],
            dim=-1,
        )

        assert representation.shape[1] == (
            PHASE45_REFIT_CONFIG[
                "bundle_dimension"
            ]
        )

        current_batch_size = len(batch)

        bundle[
            cursor:
            cursor + current_batch_size
        ] = (
            representation
            .detach()
            .cpu()
            .numpy()
        )

        cursor += current_batch_size

    assert cursor == 1362
    assert np.all(np.isfinite(bundle))

    return bundle


# ------------------------------------------------------------
# 5. Fresh fold refit
# ------------------------------------------------------------

def phase45_refit_one_fold(
    partition,
    selected_epoch_count,
):
    fold_value = int(partition["fold"])
    selection_seed = int(
        partition["seed"]
    )
    refit_seed = (
        selection_seed
        + PHASE45_REFIT_CONFIG[
            "refit_seed_offset"
        ]
    )

    fold_started = time.perf_counter()

    torch.manual_seed(refit_seed)

    if PHASE45_DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(
            refit_seed
        )
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    training_loader = phase45_make_loader(
        indices=partition[
            "outer_train_indices"
        ],
        augment=True,
        seed=refit_seed,
        shuffle=True,
        drop_last=True,
    )

    held_out_loader = phase45_make_loader(
        indices=partition[
            "outer_valid_indices"
        ],
        augment=False,
        seed=refit_seed + 1,
        shuffle=False,
        drop_last=False,
    )

    model = phase45_new_ssl_model().to(
        PHASE45_DEVICE
    )

    spectral_loss_module = (
        Phase45BalancedSpectralLoss3D(
            spatial_size=(
                PHASE45_MODEL_CONFIG[
                    "input_size"
                ]
            )
        ).to(PHASE45_DEVICE)
    )

    trainable_parameters = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=(
            PHASE45_FULL_PRETRAINING_CONFIG[
                "learning_rate"
            ]
        ),
        weight_decay=(
            PHASE45_FULL_PRETRAINING_CONFIG[
                "weight_decay"
            ]
        ),
        betas=tuple(
            PHASE45_FULL_PRETRAINING_CONFIG[
                "optimizer_betas"
            ]
        ),
    )

    total_steps = (
        len(training_loader)
        * int(selected_epoch_count)
    )

    global_step = 0
    epoch_records = []

    for epoch_index in range(
        int(selected_epoch_count)
    ):
        epoch_started = time.perf_counter()

        epoch_seed = (
            refit_seed + epoch_index
        )

        torch.manual_seed(epoch_seed)

        if PHASE45_DEVICE.type == "cuda":
            torch.cuda.manual_seed_all(
                epoch_seed
            )

        model.train()

        sums = {
            "total": 0.0,
            "spatial": 0.0,
            "spectral": 0.0,
        }
        case_count = 0
        gradient_norms = []

        for batch in training_loader:
            batch = batch.to(
                PHASE45_DEVICE,
                non_blocking=True,
            )

            current_learning_rate = (
                phase45_refit_learning_rate(
                    global_step,
                    total_steps,
                )
            )

            for parameter_group in (
                optimizer.param_groups
            ):
                parameter_group["lr"] = (
                    current_learning_rate
                )

            optimizer.zero_grad(
                set_to_none=True
            )

            losses = phase45_full_ssl_loss(
                model=model,
                spectral_loss_module=(
                    spectral_loss_module
                ),
                batch=batch,
            )

            assert torch.isfinite(
                losses["total"]
            )

            losses["total"].backward()

            gradient_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    trainable_parameters,
                    max_norm=(
                        PHASE45_FULL_PRETRAINING_CONFIG[
                            "gradient_clip"
                        ]
                    ),
                )
            )

            assert torch.isfinite(
                gradient_norm
            )

            optimizer.step()

            current_batch_size = len(batch)

            for metric_name in sums:
                sums[metric_name] += (
                    float(
                        losses[metric_name]
                        .detach().cpu()
                    )
                    * current_batch_size
                )

            case_count += current_batch_size

            gradient_norms.append(
                float(
                    gradient_norm
                    .detach().cpu()
                )
            )

            global_step += 1

        epoch_record = {
            "epoch": int(epoch_index + 1),
            "train_total": float(
                sums["total"] / case_count
            ),
            "train_spatial": float(
                sums["spatial"] / case_count
            ),
            "train_spectral": float(
                sums["spectral"] / case_count
            ),
            "learning_rate_end": float(
                current_learning_rate
            ),
            "mean_gradient_norm": float(
                np.mean(gradient_norms)
            ),
            "maximum_gradient_norm": float(
                np.max(gradient_norms)
            ),
            "elapsed_seconds": round(
                time.perf_counter()
                - epoch_started,
                3,
            ),
        }

        epoch_records.append(
            epoch_record
        )

        print(
            f"Phase45 refit fold {fold_value}, "
            f"epoch {epoch_index + 1}/"
            f"{selected_epoch_count}: "
            f"loss={epoch_record['train_total']:.6f}"
        )

    # Outer-validation images are evaluated only after the
    # refit is frozen; no labels are read.
    held_out_reconstruction = (
        phase45_evaluate_ssl(
            model=model,
            loader=held_out_loader,
            spectral_loss_module=(
                spectral_loss_module
            ),
            mask_seed=(
                refit_seed + 50000
            ),
        )
    )

    fold_bundle = (
        phase45_extract_fold_bundle(model)
    )

    encoder_state = (
        phase45_encoder_state_dict(
            model
        )
    )

    checkpoint_path = (
        phase45_checkpoint_directory
        / (
            f"phase45_fold_{fold_value}_"
            "encoder.pt"
        )
    )

    checkpoint_payload = {
        "phase": (
            "phase45_spectral_mae_encoder"
        ),
        "fold": fold_value,
        "selection_seed": selection_seed,
        "refit_seed": refit_seed,
        "selected_epoch_count": int(
            selected_epoch_count
        ),
        "model_configuration": dict(
            PHASE45_MODEL_CONFIG
        ),
        "representation_contract": {
            "original_slice": [0, 192],
            "symmetric_slice": [192, 384],
            "absolute_antisymmetric_slice": [
                384,
                576,
            ],
            "primary_invariant_slices": [
                [192, 384],
                [384, 576],
            ],
        },
        "encoder_state_dict": (
            encoder_state
        ),
    }

    phase45_atomic_torch_save(
        checkpoint_payload,
        checkpoint_path,
    )

    checkpoint_size_mb = float(
        checkpoint_path.stat().st_size
        / (1024 ** 2)
    )

    peak_vram_mb = (
        float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )
        if PHASE45_DEVICE.type == "cuda"
        else 0.0
    )

    fold_record = {
        "fold": fold_value,
        "selection_seed": selection_seed,
        "refit_seed": refit_seed,
        "outer_train_n": int(
            len(
                partition[
                    "outer_train_indices"
                ]
            )
        ),
        "outer_valid_n": int(
            len(
                partition[
                    "outer_valid_indices"
                ]
            )
        ),
        "selected_epoch_count": int(
            selected_epoch_count
        ),
        "final_training": (
            epoch_records[-1]
        ),
        "held_out_reconstruction": (
            held_out_reconstruction
        ),
        "checkpoint_file": (
            checkpoint_path.name
        ),
        "checkpoint_size_mb": (
            checkpoint_size_mb
        ),
        "encoder_tensor_count": int(
            len(encoder_state)
        ),
        "bundle_shape": list(
            fold_bundle.shape
        ),
        "peak_vram_mb": (
            peak_vram_mb
        ),
        "elapsed_seconds": round(
            time.perf_counter()
            - fold_started,
            3,
        ),
    }

    model.to("cpu")

    del model
    del optimizer
    del spectral_loss_module
    del training_loader
    del held_out_loader
    del encoder_state
    del checkpoint_payload

    gc.collect()

    if PHASE45_DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return fold_bundle, fold_record


# ------------------------------------------------------------
# 6. Execute three refits
# ------------------------------------------------------------

phase45_embedding_bundle_by_fold = (
    np.empty(
        (
            3,
            1362,
            PHASE45_REFIT_CONFIG[
                "bundle_dimension"
            ],
        ),
        dtype=np.float32,
    )
)

phase45_refit_records = []

for partition, selected_epoch_count in zip(
    PHASE45_SSL_PARTITIONS_PRIVATE,
    PHASE45_SELECTED_EPOCHS_PRIVATE,
):
    fold_value = int(partition["fold"])

    (
        fold_bundle,
        fold_record,
    ) = phase45_refit_one_fold(
        partition=partition,
        selected_epoch_count=(
            selected_epoch_count
        ),
    )

    phase45_embedding_bundle_by_fold[
        fold_value
    ] = fold_bundle

    phase45_refit_records.append(
        fold_record
    )

    del fold_bundle
    gc.collect()


# ------------------------------------------------------------
# 7. Construct cross-fitted OOF representation
# ------------------------------------------------------------

phase45_original_fold = np.asarray(
    PHASE44_ORIGINAL_FOLD_PRIVATE,
    dtype=np.int64,
)

phase45_embedding_oof = np.empty(
    (
        1362,
        PHASE45_REFIT_CONFIG[
            "bundle_dimension"
        ],
    ),
    dtype=np.float32,
)

for case_index in range(1362):
    fold_value = int(
        phase45_original_fold[
            case_index
        ]
    )

    phase45_embedding_oof[
        case_index
    ] = (
        phase45_embedding_bundle_by_fold[
            fold_value,
            case_index,
        ]
    )

assert np.all(
    np.isfinite(
        phase45_embedding_bundle_by_fold
    )
)

assert np.all(
    np.isfinite(
        phase45_embedding_oof
    )
)

phase45_embedding_coordinate_std = (
    np.std(
        phase45_embedding_oof,
        axis=0,
        dtype=np.float64,
    )
)

phase45_collapsed_coordinate_count = int(
    np.sum(
        phase45_embedding_coordinate_std
        < 1e-6
    )
)


# ------------------------------------------------------------
# 8. Persist private embeddings atomically
# ------------------------------------------------------------

phase45_embedding_cache_path = Path(
    PHASE45_REFIT_CONFIG[
        "embedding_cache"
    ]
)

phase45_embedding_temporary_path = (
    phase45_embedding_cache_path.with_suffix(
        ".npy.tmp"
    )
)

phase45_embedding_float16 = (
    phase45_embedding_bundle_by_fold.astype(
        np.float16
    )
)

with phase45_embedding_temporary_path.open(
    "wb"
) as output_file:
    np.save(
        output_file,
        phase45_embedding_float16,
        allow_pickle=False,
    )

phase45_embedding_temporary_path.replace(
    phase45_embedding_cache_path
)

phase45_reloaded_embedding = np.load(
    phase45_embedding_cache_path,
    mmap_mode="r",
    allow_pickle=False,
)

assert phase45_reloaded_embedding.shape == (
    3,
    1362,
    576,
)

phase45_float16_roundtrip_maximum_error = float(
    np.max(
        np.abs(
            np.asarray(
                phase45_reloaded_embedding,
                dtype=np.float32,
            )
            - phase45_embedding_bundle_by_fold
        )
    )
)

PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE = (
    phase45_embedding_bundle_by_fold
)

PHASE45_EMBEDDING_OOF_PRIVATE = (
    phase45_embedding_oof
)

PHASE45_EMBEDDING_FAMILIES_PRIVATE = {
    "original": slice(0, 192),
    "symmetric": slice(192, 384),
    "absolute_antisymmetric": slice(
        384,
        576,
    ),
    "invariant_combined": np.r_[
        192:384,
        384:576,
    ],
}


# ------------------------------------------------------------
# 9. Sanitized result
# ------------------------------------------------------------

phase45_checkpoint_files = sorted(
    phase45_checkpoint_directory.glob(
        "phase45_fold_*_encoder.pt"
    )
)

assert len(phase45_checkpoint_files) == 3

phase45_refit_result = {
    "phase": (
        "phase45_cross_fitted_spectral_mae_"
        "refit_and_embedding_extraction"
    ),
    "status": "complete",
    "selected_epoch_counts": list(
        PHASE45_SELECTED_EPOCHS_PRIVATE
    ),
    "folds": phase45_refit_records,
    "embedding_contract": {
        "by_fold_shape": list(
            phase45_embedding_bundle_by_fold.shape
        ),
        "cross_fitted_oof_shape": list(
            phase45_embedding_oof.shape
        ),
        "component_slices": {
            "original": [0, 192],
            "symmetric": [192, 384],
            "absolute_antisymmetric": [
                384,
                576,
            ],
            "primary_invariant": [
                [192, 384],
                [384, 576],
            ],
        },
        "collapsed_coordinate_count": int(
            phase45_collapsed_coordinate_count
        ),
        "all_values_finite": True,
        "float16_cache_file": (
            phase45_embedding_cache_path.name
        ),
        "float16_cache_size_mb": float(
            phase45_embedding_cache_path
            .stat().st_size
            / (1024 ** 2)
        ),
        "float16_roundtrip_maximum_error": (
            phase45_float16_roundtrip_maximum_error
        ),
    },
    "checkpoint_contract": {
        "directory": (
            phase45_checkpoint_directory.name
        ),
        "file_count": int(
            len(phase45_checkpoint_files)
        ),
        "total_size_mb": float(
            sum(
                path.stat().st_size
                for path
                in phase45_checkpoint_files
            )
            / (1024 ** 2)
        ),
        "encoder_only": True,
        "decoder_excluded": True,
        "learned_reflection_projectors_excluded": True,
    },
    "validation_semantics": {
        "outer_validation_images_used_for_training": False,
        "outer_validation_images_used_for_selection": False,
        "outer_validation_images_used_only_after_freeze": True,
        "outer_validation_labels_used": False,
        "held_out_group_encoder_used_for_each_oof_case": True,
    },
    "labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_embeddings_exported": False,
    "persistent_private_cache_only": True,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_refit_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_REFIT"
)
print(
    json.dumps(
        phase45_refit_result,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_REFIT"
)

Phase45 refit fold 0, epoch 1/16: loss=0.250018
Phase45 refit fold 0, epoch 2/16: loss=0.118964
Phase45 refit fold 0, epoch 3/16: loss=0.098907
Phase45 refit fold 0, epoch 4/16: loss=0.096035
Phase45 refit fold 0, epoch 5/16: loss=0.093878
Phase45 refit fold 0, epoch 6/16: loss=0.092499
Phase45 refit fold 0, epoch 7/16: loss=0.090978
Phase45 refit fold 0, epoch 8/16: loss=0.089979
Phase45 refit fold 0, epoch 9/16: loss=0.088663
Phase45 refit fold 0, epoch 10/16: loss=0.087335
Phase45 refit fold 0, epoch 11/16: loss=0.086396
Phase45 refit fold 0, epoch 12/16: loss=0.085443
Phase45 refit fold 0, epoch 13/16: loss=0.085180
Phase45 refit fold 0, epoch 14/16: loss=0.084587
Phase45 refit fold 0, epoch 15/16: loss=0.084283
Phase45 refit fold 0, epoch 16/16: loss=0.083937
Phase45 refit fold 1, epoch 1/15: loss=0.229815
Phase45 refit fold 1, epoch 2/15: loss=0.115956
Phase45 refit fold 1, epoch 3/15: loss=0.098343
Phase45 refit fold 1, epoch 4/15: loss=0.092994
Phase45 refit fold 1, epoch 5/15:

In [6]:
# Cell 143A — Phase45 

import json
import math
import time

import numpy as np


phase45_probe_contract_started = (
    time.perf_counter()
)

required_names = [
    "PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE",
    "PHASE45_EMBEDDING_OOF_PRIVATE",
    "PHASE44_ORIGINAL_FOLD_PRIVATE",
    "PHASE19_PARTITIONS",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, {
    "message": (
        "Phase45 downstream-probe dependencies "
        "are missing."
    ),
    "missing": missing_names,
}


# ------------------------------------------------------------
# 1. Validate embedding state
# ------------------------------------------------------------

phase45_embedding_by_fold = np.asarray(
    PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE,
    dtype=np.float32,
)

phase45_embedding_oof = np.asarray(
    PHASE45_EMBEDDING_OOF_PRIVATE,
    dtype=np.float32,
)

phase45_original_fold = np.asarray(
    PHASE44_ORIGINAL_FOLD_PRIVATE,
    dtype=np.int64,
)

assert phase45_embedding_by_fold.shape == (
    3,
    1362,
    576,
)

assert phase45_embedding_oof.shape == (
    1362,
    576,
)

assert np.all(
    np.isfinite(phase45_embedding_by_fold)
)

assert np.all(
    np.isfinite(phase45_embedding_oof)
)

phase45_oof_reconstruction = np.empty_like(
    phase45_embedding_oof
)

for case_index in range(1362):
    phase45_oof_reconstruction[
        case_index
    ] = phase45_embedding_by_fold[
        phase45_original_fold[
            case_index
        ],
        case_index,
    ]

phase45_oof_embedding_error = float(
    np.max(
        np.abs(
            phase45_oof_reconstruction
            - phase45_embedding_oof
        )
    )
)

assert phase45_oof_embedding_error == 0.0


# ------------------------------------------------------------
# 2. Resolve exact Phase19 partitions
# ------------------------------------------------------------

def phase45_partition_field(
    partition,
    candidate_names,
):
    for candidate_name in candidate_names:
        if (
            isinstance(partition, dict)
            and candidate_name in partition
        ):
            return np.asarray(
                partition[candidate_name],
                dtype=np.int64,
            ).reshape(-1)

        if hasattr(
            partition,
            candidate_name,
        ):
            return np.asarray(
                getattr(
                    partition,
                    candidate_name,
                ),
                dtype=np.int64,
            ).reshape(-1)

    return None


PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE = []
phase45_partition_records = []

phase45_expected_counts = [
    {
        "fit_n": 721,
        "monitor_n": 174,
        "outer_train_n": 895,
        "outer_valid_n": 467,
    },
    {
        "fit_n": 796,
        "monitor_n": 123,
        "outer_train_n": 919,
        "outer_valid_n": 443,
    },
    {
        "fit_n": 714,
        "monitor_n": 196,
        "outer_train_n": 910,
        "outer_valid_n": 452,
    },
]

assert len(PHASE19_PARTITIONS) == 3

for fold_value, partition in enumerate(
    PHASE19_PARTITIONS
):
    fit_indices = phase45_partition_field(
        partition,
        [
            "fit",
            "fit_indices",
            "inner_fit",
            "inner_fit_indices",
        ],
    )

    monitor_indices = phase45_partition_field(
        partition,
        [
            "monitor",
            "monitor_indices",
            "inner_monitor",
            "inner_monitor_indices",
        ],
    )

    outer_train_indices = (
        phase45_partition_field(
            partition,
            [
                "outer_train",
                "outer_train_indices",
                "train",
                "train_indices",
            ],
        )
    )

    outer_valid_indices = (
        phase45_partition_field(
            partition,
            [
                "outer_valid",
                "outer_valid_indices",
                "valid",
                "valid_indices",
                "validation_indices",
            ],
        )
    )

    assert fit_indices is not None, {
        "message": (
            "Could not resolve fit indices."
        ),
        "fold": int(fold_value),
    }

    assert monitor_indices is not None, {
        "message": (
            "Could not resolve monitor indices."
        ),
        "fold": int(fold_value),
    }

    assert outer_train_indices is not None
    assert outer_valid_indices is not None

    expected = phase45_expected_counts[
        fold_value
    ]

    observed = {
        "fit_n": int(len(fit_indices)),
        "monitor_n": int(
            len(monitor_indices)
        ),
        "outer_train_n": int(
            len(outer_train_indices)
        ),
        "outer_valid_n": int(
            len(outer_valid_indices)
        ),
    }

    assert observed == expected, {
        "message": (
            "Phase19 partition sizes changed."
        ),
        "fold": int(fold_value),
        "observed": observed,
        "expected": expected,
    }

    assert len(
        np.intersect1d(
            fit_indices,
            monitor_indices,
        )
    ) == 0

    assert len(
        np.intersect1d(
            outer_train_indices,
            outer_valid_indices,
        )
    ) == 0

    assert np.array_equal(
        np.sort(
            np.concatenate([
                fit_indices,
                monitor_indices,
            ])
        ),
        np.sort(outer_train_indices),
    )

    expected_outer_valid = np.flatnonzero(
        phase45_original_fold
        == fold_value
    )

    assert np.array_equal(
        np.sort(outer_valid_indices),
        expected_outer_valid,
    )

    private_partition = {
        "fold": int(fold_value),
        "fit_indices": (
            fit_indices.copy()
        ),
        "monitor_indices": (
            monitor_indices.copy()
        ),
        "outer_train_indices": (
            outer_train_indices.copy()
        ),
        "outer_valid_indices": (
            outer_valid_indices.copy()
        ),
    }

    PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE.append(
        private_partition
    )

    phase45_partition_records.append({
        "fold": int(fold_value),
        **observed,
    })


# ------------------------------------------------------------
# 3. Representation-family definitions
# ------------------------------------------------------------

PHASE45_REPRESENTATION_FAMILIES_PRIVATE = {
    "original": np.arange(
        0,
        192,
        dtype=np.int64,
    ),
    "symmetric": np.arange(
        192,
        384,
        dtype=np.int64,
    ),
    "absolute_antisymmetric": np.arange(
        384,
        576,
        dtype=np.int64,
    ),
    "invariant_combined": np.arange(
        192,
        576,
        dtype=np.int64,
    ),
    "complete_bundle": np.arange(
        0,
        576,
        dtype=np.int64,
    ),
}

phase45_family_audits = {}

for family_name, columns in (
    PHASE45_REPRESENTATION_FAMILIES_PRIVATE
    .items()
):
    family_matrix = (
        phase45_embedding_oof[:, columns]
    ).astype(np.float64)

    coordinate_std = np.std(
        family_matrix,
        axis=0,
    )

    collapsed_coordinate_count = int(
        np.sum(coordinate_std < 1e-6)
    )

    centered_matrix = (
        family_matrix
        - np.mean(
            family_matrix,
            axis=0,
            keepdims=True,
        )
    )

    singular_values = np.linalg.svd(
        centered_matrix,
        compute_uv=False,
        full_matrices=False,
    )

    variance_values = (
        singular_values ** 2
    )

    total_variance = float(
        np.sum(variance_values)
    )

    assert total_variance > 0.0

    cumulative_variance = np.cumsum(
        variance_values
    ) / total_variance

    effective_rank_95 = int(
        np.searchsorted(
            cumulative_variance,
            0.95,
        )
        + 1
    )

    effective_rank_99 = int(
        np.searchsorted(
            cumulative_variance,
            0.99,
        )
        + 1
    )

    phase45_family_audits[
        family_name
    ] = {
        "dimension": int(
            family_matrix.shape[1]
        ),
        "collapsed_coordinate_count": (
            collapsed_coordinate_count
        ),
        "median_coordinate_std": float(
            np.median(coordinate_std)
        ),
        "minimum_coordinate_std": float(
            np.min(coordinate_std)
        ),
        "maximum_coordinate_std": float(
            np.max(coordinate_std)
        ),
        "effective_rank_95_percent": (
            effective_rank_95
        ),
        "effective_rank_99_percent": (
            effective_rank_99
        ),
    }


# ------------------------------------------------------------
# 4. Label-free acquisition-transport audit
# ------------------------------------------------------------

phase45_transport_records = []

for partition in (
    PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE
):
    fold_value = partition["fold"]

    outer_train_indices = partition[
        "outer_train_indices"
    ]
    outer_valid_indices = partition[
        "outer_valid_indices"
    ]

    # Use the fold-specific encoder for both partitions.
    matrix = phase45_embedding_by_fold[
        fold_value
    ]

    family_transport = {}

    for family_name, columns in (
        PHASE45_REPRESENTATION_FAMILIES_PRIVATE
        .items()
    ):
        train_matrix = matrix[
            outer_train_indices
        ][:, columns].astype(np.float64)

        valid_matrix = matrix[
            outer_valid_indices
        ][:, columns].astype(np.float64)

        train_mean = np.mean(
            train_matrix,
            axis=0,
        )
        valid_mean = np.mean(
            valid_matrix,
            axis=0,
        )

        train_std = np.std(
            train_matrix,
            axis=0,
        )

        standardized_mean_difference = (
            np.abs(
                valid_mean - train_mean
            )
            / np.maximum(
                train_std,
                1e-6,
            )
        )

        family_transport[
            family_name
        ] = {
            "median_absolute_standardized_"
            "mean_shift": float(
                np.median(
                    standardized_mean_difference
                )
            ),
            "q95_absolute_standardized_"
            "mean_shift": float(
                np.quantile(
                    standardized_mean_difference,
                    0.95,
                )
            ),
            "maximum_absolute_standardized_"
            "mean_shift": float(
                np.max(
                    standardized_mean_difference
                )
            ),
        }

    phase45_transport_records.append({
        "fold": int(fold_value),
        "families": family_transport,
    })


# ------------------------------------------------------------
# 5. Frozen probe-search plan
# ------------------------------------------------------------

PHASE45_PROBE_GRID_PRIVATE = {
    "representations": list(
        PHASE45_REPRESENTATION_FAMILIES_PRIVATE
        .keys()
    ),
    "pca_dimensions": [
        16,
        32,
        64,
        128,
    ],
    "rank_loss_weights": [
        0.0,
        0.1,
        0.3,
    ],
    "l2_regularization_values": [
        1e-4,
        1e-3,
        1e-2,
    ],
}

phase45_probe_candidate_count = (
    len(
        PHASE45_PROBE_GRID_PRIVATE[
            "representations"
        ]
    )
    * len(
        PHASE45_PROBE_GRID_PRIVATE[
            "pca_dimensions"
        ]
    )
    * len(
        PHASE45_PROBE_GRID_PRIVATE[
            "rank_loss_weights"
        ]
    )
    * len(
        PHASE45_PROBE_GRID_PRIVATE[
            "l2_regularization_values"
        ]
    )
)

assert phase45_probe_candidate_count == 180


# ------------------------------------------------------------
# 6. Sanitized contract
# ------------------------------------------------------------

phase45_probe_contract = {
    "phase": (
        "phase45_frozen_embedding_"
        "downstream_probe_contract"
    ),
    "status": "accepted",
    "embedding_source": {
        "by_fold_shape": list(
            phase45_embedding_by_fold.shape
        ),
        "cross_fitted_oof_shape": list(
            phase45_embedding_oof.shape
        ),
        "oof_reconstruction_error": (
            phase45_oof_embedding_error
        ),
        "float32_private_state_used": True,
        "float16_cache_used_for_selection": False,
    },
    "partitions": (
        phase45_partition_records
    ),
    "representation_families": (
        phase45_family_audits
    ),
    "label_free_transport_audit": (
        phase45_transport_records
    ),
    "probe_grid": {
        **PHASE45_PROBE_GRID_PRIVATE,
        "candidate_count": int(
            phase45_probe_candidate_count
        ),
        "fold_count": 3,
        "fit_count_during_selection": int(
            phase45_probe_candidate_count
            * 3
        ),
    },
    "preprocessing": {
        "robust_scaling": (
            "fit_partition_only"
        ),
        "pca": "fit_partition_only",
        "outer_validation_features_used_for_fit": False,
    },
    "model": {
        "type": (
            "linear_binary_probe_with_optional_"
            "within_group_pairwise_rank_loss"
        ),
        "selection": (
            "one_shared_specification_across_folds"
        ),
        "primary_metric": "monitor_log_loss",
        "secondary_metric": (
            "monitor_auroc_within_log_loss_band"
        ),
    },
    "labels_used": False,
    "outer_validation_labels_used": False,
    "training_voxel_cache_read": False,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_embeddings_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_probe_contract_started,
        3,
    ),
}

print(
    "BEGIN SANITIZED_PHASE45_PROBE_CONTRACT"
)
print(
    json.dumps(
        phase45_probe_contract,
        indent=2,
    )
)
print(
    "END SANITIZED_PHASE45_PROBE_CONTRACT"
)

AssertionError: {'message': 'Phase45 downstream-probe dependencies are missing.', 'missing': ['PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE', 'PHASE45_EMBEDDING_OOF_PRIVATE', 'PHASE44_ORIGINAL_FOLD_PRIVATE', 'PHASE19_PARTITIONS']}

In [48]:
# phase45_cell_143B

import itertools
import json
import math
import time

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import RobustScaler


phase45_probe_selection_started = time.perf_counter()


PHASE45_PROBE_SELECTION_CONFIG = {
    "random_seed": 451403,
    "pca_whiten": True,
    "pca_iterated_power": 5,
    "transformed_clip": 10.0,
    "maximum_pairs_per_group": 256,
    "optimizer_maximum_iterations": 160,
    "optimizer_ftol": 1e-10,
    "optimizer_gtol": 1e-6,
    "log_loss_selection_band": 0.003,
}


def phase45_as_index_array(value, name):
    array = np.asarray(value, dtype=np.int64).reshape(-1)
    assert array.size > 0, f"{name} is empty."
    assert np.all((array >= 0) & (array < 1362)), {
        "message": f"{name} contains invalid case indices.",
        "minimum": int(array.min()),
        "maximum": int(array.max()),
    }
    assert np.unique(array).size == array.size, {
        "message": f"{name} contains duplicate case indices."
    }
    return array


def phase45_partition_indices(partition, semantic_name):
    aliases = {
        "fit": (
            "fit_indices",
            "fit_idx",
            "fit",
        ),
        "monitor": (
            "monitor_indices",
            "monitor_idx",
            "monitor",
        ),
        "outer_train": (
            "outer_train_indices",
            "outer_train_idx",
            "outer_train",
            "train_indices",
        ),
        "outer_valid": (
            "outer_valid_indices",
            "outer_valid_idx",
            "outer_valid",
            "valid_indices",
        ),
    }
    for key in aliases[semantic_name]:
        if key in partition:
            return phase45_as_index_array(
                partition[key],
                f"partition[{semantic_name}]",
            )
    raise KeyError({
        "message": "Could not resolve downstream partition indices.",
        "semantic_name": semantic_name,
        "available_keys": sorted(partition.keys()),
    })


phase45_y = np.asarray(
    PHASE44_Y_PRIVATE,
    dtype=np.float64,
).reshape(-1)
phase45_groups = np.asarray(
    PHASE44_GROUPS_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase45_embedding_by_fold = np.asarray(
    PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE,
    dtype=np.float32,
)

assert phase45_y.shape == (1362,)
assert phase45_groups.shape == (1362,)
assert phase45_embedding_by_fold.shape == (
    3,
    1362,
    576,
)
assert set(np.unique(phase45_y).tolist()) == {0.0, 1.0}
assert np.all(np.isfinite(phase45_embedding_by_fold))


phase45_partitions = []
for fold, partition in enumerate(
    PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE
):
    fit_indices = phase45_partition_indices(
        partition,
        "fit",
    )
    monitor_indices = phase45_partition_indices(
        partition,
        "monitor",
    )
    outer_train_indices = phase45_partition_indices(
        partition,
        "outer_train",
    )
    outer_valid_indices = phase45_partition_indices(
        partition,
        "outer_valid",
    )

    assert np.intersect1d(
        fit_indices,
        monitor_indices,
    ).size == 0
    assert np.array_equal(
        np.sort(
            np.concatenate([
                fit_indices,
                monitor_indices,
            ])
        ),
        np.sort(outer_train_indices),
    )
    assert np.intersect1d(
        outer_train_indices,
        outer_valid_indices,
    ).size == 0
    assert (
        outer_train_indices.size
        + outer_valid_indices.size
        == 1362
    )

    phase45_partitions.append({
        "fold": int(fold),
        "fit_indices": fit_indices,
        "monitor_indices": monitor_indices,
        "outer_train_indices": outer_train_indices,
        "outer_valid_indices": outer_valid_indices,
    })

assert len(phase45_partitions) == 3


PHASE45_REPRESENTATION_SLICES_PRIVATE = {
    "original": ((0, 192),),
    "symmetric": ((192, 384),),
    "absolute_antisymmetric": ((384, 576),),
    "invariant_combined": (
        (192, 384),
        (384, 576),
    ),
    "complete_bundle": ((0, 576),),
}


def phase45_select_representation(matrix, representation):
    slices = PHASE45_REPRESENTATION_SLICES_PRIVATE[
        representation
    ]
    parts = [
        matrix[:, start:stop]
        for start, stop in slices
    ]
    output = (
        parts[0]
        if len(parts) == 1
        else np.concatenate(parts, axis=1)
    )
    output = np.asarray(output, dtype=np.float64)
    assert output.ndim == 2
    assert np.all(np.isfinite(output))
    return output


def phase45_make_pair_contract(
    labels,
    groups,
    maximum_pairs_per_group,
    seed,
):
    labels = np.asarray(labels, dtype=np.float64)
    groups = np.asarray(groups, dtype=np.int64)
    rng = np.random.default_rng(seed)

    positive_records = []
    negative_records = []
    weight_records = []
    group_pair_counts = {}

    eligible_groups = []
    for group in np.unique(groups):
        local = np.flatnonzero(groups == group)
        positive = local[labels[local] == 1.0]
        negative = local[labels[local] == 0.0]
        if positive.size and negative.size:
            eligible_groups.append((
                int(group),
                positive,
                negative,
            ))

    if not eligible_groups:
        return {
            "positive_indices": np.empty(0, dtype=np.int64),
            "negative_indices": np.empty(0, dtype=np.int64),
            "weights": np.empty(0, dtype=np.float64),
            "group_pair_counts": {},
        }

    group_weight = 1.0 / len(eligible_groups)
    for group, positive, negative in eligible_groups:
        pair_count = int(positive.size * negative.size)
        retained_count = min(
            pair_count,
            int(maximum_pairs_per_group),
        )

        if retained_count == pair_count:
            flat = np.arange(pair_count, dtype=np.int64)
        else:
            flat = np.sort(
                rng.choice(
                    pair_count,
                    size=retained_count,
                    replace=False,
                )
            )

        positive_local = flat // negative.size
        negative_local = flat % negative.size
        positive_records.append(positive[positive_local])
        negative_records.append(negative[negative_local])
        weight_records.append(
            np.full(
                retained_count,
                group_weight / retained_count,
                dtype=np.float64,
            )
        )
        group_pair_counts[str(group)] = int(retained_count)

    positive_indices = np.concatenate(positive_records)
    negative_indices = np.concatenate(negative_records)
    weights = np.concatenate(weight_records)
    weights /= weights.sum()

    return {
        "positive_indices": positive_indices,
        "negative_indices": negative_indices,
        "weights": weights,
        "group_pair_counts": group_pair_counts,
    }


def phase45_fit_convex_linear_probe(
    fit_features,
    fit_labels,
    pair_contract,
    rank_loss_weight,
    l2_regularization,
    initial_parameters=None,
):
    x = np.asarray(fit_features, dtype=np.float64)
    y = np.asarray(fit_labels, dtype=np.float64)
    n, dimension = x.shape
    assert y.shape == (n,)
    assert np.all(np.isfinite(x))

    positive_indices = pair_contract["positive_indices"]
    negative_indices = pair_contract["negative_indices"]
    pair_weights = pair_contract["weights"]
    if positive_indices.size:
        pair_differences = (
            x[positive_indices]
            - x[negative_indices]
        )
    else:
        pair_differences = np.empty(
            (0, dimension),
            dtype=np.float64,
        )

    if initial_parameters is None:
        prevalence = float(np.clip(y.mean(), 1e-6, 1.0 - 1e-6))
        parameters_0 = np.zeros(dimension + 1, dtype=np.float64)
        parameters_0[-1] = math.log(
            prevalence / (1.0 - prevalence)
        )
    else:
        parameters_0 = np.asarray(
            initial_parameters,
            dtype=np.float64,
        ).copy()
        assert parameters_0.shape == (dimension + 1,)

    rank_loss_weight = float(rank_loss_weight)
    l2_regularization = float(l2_regularization)

    def objective(parameters):
        weights = parameters[:-1]
        intercept = parameters[-1]
        scores = x @ weights + intercept

        logistic_loss = np.mean(
            np.logaddexp(0.0, scores)
            - y * scores
        )
        probabilities = expit(scores)
        score_gradient = (probabilities - y) / n
        weight_gradient = x.T @ score_gradient
        intercept_gradient = float(score_gradient.sum())

        regularization_loss = (
            0.5
            * l2_regularization
            * float(weights @ weights)
        )
        weight_gradient += l2_regularization * weights

        ranking_loss = 0.0
        if rank_loss_weight > 0.0 and pair_differences.shape[0]:
            margins = pair_differences @ weights
            ranking_loss = float(
                np.sum(
                    pair_weights
                    * np.logaddexp(0.0, -margins)
                )
            )
            margin_derivative = -expit(-margins)
            weight_gradient += (
                rank_loss_weight
                * pair_differences.T
                @ (pair_weights * margin_derivative)
            )

        total_loss = (
            logistic_loss
            + regularization_loss
            + rank_loss_weight * ranking_loss
        )
        gradient = np.concatenate([
            weight_gradient,
            np.asarray([intercept_gradient]),
        ])
        return float(total_loss), gradient

    result = minimize(
        objective,
        parameters_0,
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": int(
                PHASE45_PROBE_SELECTION_CONFIG[
                    "optimizer_maximum_iterations"
                ]
            ),
            "ftol": float(
                PHASE45_PROBE_SELECTION_CONFIG[
                    "optimizer_ftol"
                ]
            ),
            "gtol": float(
                PHASE45_PROBE_SELECTION_CONFIG[
                    "optimizer_gtol"
                ]
            ),
            "maxls": 30,
        },
    )

    parameters = np.asarray(result.x, dtype=np.float64)
    objective_value, gradient = objective(parameters)
    finite = bool(
        np.all(np.isfinite(parameters))
        and np.isfinite(objective_value)
        and np.all(np.isfinite(gradient))
    )
    assert finite, {
        "message": "Non-finite linear-probe optimization result.",
        "rank_loss_weight": rank_loss_weight,
        "l2_regularization": l2_regularization,
    }

    return {
        "parameters": parameters,
        "objective": float(objective_value),
        "success": bool(result.success),
        "status": int(result.status),
        "iterations": int(result.nit),
        "gradient_maximum_absolute": float(
            np.max(np.abs(gradient))
        ),
    }


def phase45_predict_linear_probe(features, parameters):
    features = np.asarray(features, dtype=np.float64)
    parameters = np.asarray(parameters, dtype=np.float64)
    probabilities = expit(
        features @ parameters[:-1]
        + parameters[-1]
    )
    probabilities = np.clip(
        probabilities,
        1e-7,
        1.0 - 1e-7,
    )
    assert np.all(np.isfinite(probabilities))
    return probabilities


def phase45_weighted_log_loss(labels, probabilities, weights=None):
    labels = np.asarray(labels, dtype=np.float64)
    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        1e-7,
        1.0 - 1e-7,
    )
    losses = -(
        labels * np.log(probabilities)
        + (1.0 - labels) * np.log1p(-probabilities)
    )
    if weights is None:
        return float(losses.mean())
    weights = np.asarray(weights, dtype=np.float64)
    return float(np.sum(weights * losses) / np.sum(weights))


def phase45_safe_weighted_auroc(labels, probabilities, weights=None):
    labels = np.asarray(labels, dtype=np.float64)
    if np.unique(labels).size != 2:
        return None
    return float(
        roc_auc_score(
            labels,
            probabilities,
            sample_weight=weights,
        )
    )


phase45_candidate_specifications = []
for candidate_index, values in enumerate(
    itertools.product(
        PHASE45_PROBE_GRID_PRIVATE["representations"],
        PHASE45_PROBE_GRID_PRIVATE["pca_dimensions"],
        PHASE45_PROBE_GRID_PRIVATE["rank_loss_weights"],
        PHASE45_PROBE_GRID_PRIVATE[
            "l2_regularization_values"
        ],
    )
):
    (
        representation,
        pca_dimension,
        rank_loss_weight,
        l2_regularization,
    ) = values
    phase45_candidate_specifications.append({
        "candidate_index": int(candidate_index),
        "representation": str(representation),
        "pca_dimension": int(pca_dimension),
        "rank_loss_weight": float(rank_loss_weight),
        "l2_regularization": float(l2_regularization),
    })

assert len(phase45_candidate_specifications) == 180

phase45_candidate_lookup = {
    (
        record["representation"],
        record["pca_dimension"],
        record["rank_loss_weight"],
        record["l2_regularization"],
    ): record["candidate_index"]
    for record in phase45_candidate_specifications
}

phase45_monitor_indices = np.concatenate([
    partition["monitor_indices"]
    for partition in phase45_partitions
])
phase45_monitor_fold = np.concatenate([
    np.full(
        partition["monitor_indices"].size,
        partition["fold"],
        dtype=np.int64,
    )
    for partition in phase45_partitions
])
phase45_monitor_labels = phase45_y[phase45_monitor_indices]
phase45_monitor_groups = phase45_groups[phase45_monitor_indices]

phase45_monitor_multiplicity = np.bincount(
    phase45_monitor_indices,
    minlength=1362,
)
phase45_monitor_weights = (
    1.0
    / phase45_monitor_multiplicity[
        phase45_monitor_indices
    ]
)
phase45_monitor_weights = np.asarray(
    phase45_monitor_weights,
    dtype=np.float64,
)

phase45_candidate_monitor_probability = np.full(
    (
        len(phase45_candidate_specifications),
        phase45_monitor_indices.size,
    ),
    np.nan,
    dtype=np.float64,
)
phase45_optimizer_records = []
phase45_preprocessing_records = []

phase45_monitor_offset = 0
for partition in phase45_partitions:
    fold = partition["fold"]
    fit_indices = partition["fit_indices"]
    monitor_indices = partition["monitor_indices"]
    monitor_slice = slice(
        phase45_monitor_offset,
        phase45_monitor_offset + monitor_indices.size,
    )
    fold_embedding = phase45_embedding_by_fold[fold]

    fit_labels = phase45_y[fit_indices]
    fit_groups = phase45_groups[fit_indices]
    pair_contract = phase45_make_pair_contract(
        labels=fit_labels,
        groups=fit_groups,
        maximum_pairs_per_group=(
            PHASE45_PROBE_SELECTION_CONFIG[
                "maximum_pairs_per_group"
            ]
        ),
        seed=(
            PHASE45_PROBE_SELECTION_CONFIG["random_seed"]
            + 1000 * fold
        ),
    )

    for representation in PHASE45_PROBE_GRID_PRIVATE[
        "representations"
    ]:
        fit_raw = phase45_select_representation(
            fold_embedding[fit_indices],
            representation,
        )
        monitor_raw = phase45_select_representation(
            fold_embedding[monitor_indices],
            representation,
        )

        scaler = RobustScaler(
            with_centering=True,
            with_scaling=True,
            quantile_range=(25.0, 75.0),
            unit_variance=False,
        )
        fit_scaled = scaler.fit_transform(fit_raw)
        monitor_scaled = scaler.transform(monitor_raw)

        for pca_dimension in PHASE45_PROBE_GRID_PRIVATE[
            "pca_dimensions"
        ]:
            assert pca_dimension <= min(fit_scaled.shape)
            pca = PCA(
                n_components=int(pca_dimension),
                whiten=bool(
                    PHASE45_PROBE_SELECTION_CONFIG[
                        "pca_whiten"
                    ]
                ),
                svd_solver="randomized",
                iterated_power=int(
                    PHASE45_PROBE_SELECTION_CONFIG[
                        "pca_iterated_power"
                    ]
                ),
                random_state=(
                    PHASE45_PROBE_SELECTION_CONFIG[
                        "random_seed"
                    ]
                    + 1000 * fold
                    + int(pca_dimension)
                ),
            )
            fit_transformed = pca.fit_transform(fit_scaled)
            monitor_transformed = pca.transform(monitor_scaled)

            transformed_clip = float(
                PHASE45_PROBE_SELECTION_CONFIG[
                    "transformed_clip"
                ]
            )
            fit_transformed = np.clip(
                np.nan_to_num(
                    fit_transformed,
                    nan=0.0,
                    posinf=transformed_clip,
                    neginf=-transformed_clip,
                ),
                -transformed_clip,
                transformed_clip,
            )
            monitor_transformed = np.clip(
                np.nan_to_num(
                    monitor_transformed,
                    nan=0.0,
                    posinf=transformed_clip,
                    neginf=-transformed_clip,
                ),
                -transformed_clip,
                transformed_clip,
            )

            phase45_preprocessing_records.append({
                "fold": int(fold),
                "representation": str(representation),
                "pca_dimension": int(pca_dimension),
                "explained_variance_fraction": float(
                    np.sum(pca.explained_variance_ratio_)
                ),
                "pair_count": int(
                    pair_contract["positive_indices"].size
                ),
            })

            initial_by_l2 = {}
            for rank_loss_weight in PHASE45_PROBE_GRID_PRIVATE[
                "rank_loss_weights"
            ]:
                for l2_regularization in PHASE45_PROBE_GRID_PRIVATE[
                    "l2_regularization_values"
                ]:
                    candidate_index = phase45_candidate_lookup[(
                        representation,
                        int(pca_dimension),
                        float(rank_loss_weight),
                        float(l2_regularization),
                    )]

                    fit_result = phase45_fit_convex_linear_probe(
                        fit_features=fit_transformed,
                        fit_labels=fit_labels,
                        pair_contract=pair_contract,
                        rank_loss_weight=rank_loss_weight,
                        l2_regularization=l2_regularization,
                        initial_parameters=initial_by_l2.get(
                            float(l2_regularization)
                        ),
                    )
                    if float(rank_loss_weight) == 0.0:
                        initial_by_l2[
                            float(l2_regularization)
                        ] = fit_result["parameters"].copy()

                    probability = phase45_predict_linear_probe(
                        monitor_transformed,
                        fit_result["parameters"],
                    )
                    phase45_candidate_monitor_probability[
                        candidate_index,
                        monitor_slice,
                    ] = probability

                    phase45_optimizer_records.append({
                        "fold": int(fold),
                        "candidate_index": int(candidate_index),
                        "success": bool(fit_result["success"]),
                        "status": int(fit_result["status"]),
                        "iterations": int(fit_result["iterations"]),
                        "gradient_maximum_absolute": float(
                            fit_result[
                                "gradient_maximum_absolute"
                            ]
                        ),
                    })

    phase45_monitor_offset += monitor_indices.size
    print(
        f"Phase45 supervised probe selection: fold {fold + 1}/3 complete"
    )

assert phase45_monitor_offset == phase45_monitor_indices.size
assert np.all(np.isfinite(
    phase45_candidate_monitor_probability
))
assert np.all(
    (phase45_candidate_monitor_probability > 0.0)
    & (phase45_candidate_monitor_probability < 1.0)
)


phase45_candidate_records = []
for specification in phase45_candidate_specifications:
    candidate_index = specification["candidate_index"]
    probability = phase45_candidate_monitor_probability[
        candidate_index
    ]

    fold_metrics = []
    fold_aurocs = []
    for fold in range(3):
        mask = phase45_monitor_fold == fold
        fold_log_loss = phase45_weighted_log_loss(
            phase45_monitor_labels[mask],
            probability[mask],
        )
        fold_auroc = phase45_safe_weighted_auroc(
            phase45_monitor_labels[mask],
            probability[mask],
        )
        fold_metrics.append({
            "fold": int(fold),
            "n": int(mask.sum()),
            "log_loss": float(fold_log_loss),
            "auroc": (
                None
                if fold_auroc is None
                else float(fold_auroc)
            ),
        })
        if fold_auroc is not None:
            fold_aurocs.append(fold_auroc)

    group_metrics = []
    for group in np.unique(phase45_monitor_groups):
        mask = phase45_monitor_groups == group
        group_weights = phase45_monitor_weights[mask]
        group_log_loss = phase45_weighted_log_loss(
            phase45_monitor_labels[mask],
            probability[mask],
            group_weights,
        )
        group_auroc = phase45_safe_weighted_auroc(
            phase45_monitor_labels[mask],
            probability[mask],
            group_weights,
        )
        group_metrics.append({
            "group": int(group),
            "unique_n": int(
                np.unique(
                    phase45_monitor_indices[mask]
                ).size
            ),
            "log_loss": float(group_log_loss),
            "auroc": (
                None
                if group_auroc is None
                else float(group_auroc)
            ),
        })

    pooled_log_loss = phase45_weighted_log_loss(
        phase45_monitor_labels,
        probability,
        phase45_monitor_weights,
    )
    pooled_auroc = phase45_safe_weighted_auroc(
        phase45_monitor_labels,
        probability,
        phase45_monitor_weights,
    )
    assert pooled_auroc is not None

    optimizer_subset = [
        record
        for record in phase45_optimizer_records
        if record["candidate_index"] == candidate_index
    ]
    assert len(optimizer_subset) == 3

    phase45_candidate_records.append({
        **specification,
        "monitor_log_loss": float(pooled_log_loss),
        "monitor_auroc": float(pooled_auroc),
        "worst_fold_log_loss": float(max(
            record["log_loss"]
            for record in fold_metrics
        )),
        "minimum_fold_auroc": float(min(fold_aurocs)),
        "worst_group_log_loss": float(max(
            record["log_loss"]
            for record in group_metrics
        )),
        "minimum_group_auroc": (
            None
            if not any(
                record["auroc"] is not None
                for record in group_metrics
            )
            else float(min(
                record["auroc"]
                for record in group_metrics
                if record["auroc"] is not None
            ))
        ),
        "optimizer_success_count": int(sum(
            record["success"]
            for record in optimizer_subset
        )),
        "optimizer_iteration_maximum": int(max(
            record["iterations"]
            for record in optimizer_subset
        )),
        "fold_metrics": fold_metrics,
        "group_metrics": group_metrics,
    })


phase45_raw_best = min(
    phase45_candidate_records,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["candidate_index"],
    ),
)
phase45_log_loss_band_limit = (
    phase45_raw_best["monitor_log_loss"]
    + PHASE45_PROBE_SELECTION_CONFIG[
        "log_loss_selection_band"
    ]
)
phase45_log_loss_band = [
    record
    for record in phase45_candidate_records
    if record["monitor_log_loss"]
    <= phase45_log_loss_band_limit + 1e-12
]
assert phase45_log_loss_band

phase45_selected = max(
    phase45_log_loss_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["minimum_fold_auroc"],
        -record["candidate_index"],
    ),
)

phase45_auroc_best = max(
    phase45_candidate_records,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["candidate_index"],
    ),
)


def phase45_sanitize_candidate(record):
    return {
        "candidate_index": int(record["candidate_index"]),
        "specification": {
            "representation": record["representation"],
            "pca_dimension": int(record["pca_dimension"]),
            "rank_loss_weight": float(
                record["rank_loss_weight"]
            ),
            "l2_regularization": float(
                record["l2_regularization"]
            ),
        },
        "monitor_log_loss": float(
            record["monitor_log_loss"]
        ),
        "monitor_auroc": float(record["monitor_auroc"]),
        "worst_fold_log_loss": float(
            record["worst_fold_log_loss"]
        ),
        "minimum_fold_auroc": float(
            record["minimum_fold_auroc"]
        ),
        "worst_group_log_loss": float(
            record["worst_group_log_loss"]
        ),
        "minimum_group_auroc": record[
            "minimum_group_auroc"
        ],
        "optimizer_success_count": int(
            record["optimizer_success_count"]
        ),
        "optimizer_iteration_maximum": int(
            record["optimizer_iteration_maximum"]
        ),
        "fold_metrics": record["fold_metrics"],
    }


PHASE45_PROBE_CANDIDATE_RECORDS_PRIVATE = (
    phase45_candidate_records
)
PHASE45_PROBE_MONITOR_PREDICTIONS_PRIVATE = (
    phase45_candidate_monitor_probability
)
PHASE45_PROBE_MONITOR_INDICES_PRIVATE = (
    phase45_monitor_indices
)
PHASE45_PROBE_MONITOR_WEIGHTS_PRIVATE = (
    phase45_monitor_weights
)
PHASE45_PROBE_SELECTED_SPECIFICATION_PRIVATE = {
    "representation": phase45_selected["representation"],
    "pca_dimension": int(
        phase45_selected["pca_dimension"]
    ),
    "rank_loss_weight": float(
        phase45_selected["rank_loss_weight"]
    ),
    "l2_regularization": float(
        phase45_selected["l2_regularization"]
    ),
}
PHASE45_PROBE_SELECTED_MONITOR_PROBABILITY_PRIVATE = (
    phase45_candidate_monitor_probability[
        phase45_selected["candidate_index"]
    ].copy()
)

phase45_probe_selection_report = {
    "phase": (
        "phase45_frozen_embedding_shared_linear_probe_monitor_selection"
    ),
    "status": "probe_specification_frozen_before_outer_evaluation",
    "selection_data": {
        "monitor_observation_count": int(
            phase45_monitor_indices.size
        ),
        "unique_monitor_case_count": int(
            np.unique(phase45_monitor_indices).size
        ),
        "maximum_case_multiplicity": int(
            phase45_monitor_multiplicity.max()
        ),
        "inverse_multiplicity_weighting": True,
        "monitor_group_count": int(
            np.unique(phase45_monitor_groups).size
        ),
    },
    "search": {
        "candidate_count": int(
            len(phase45_candidate_records)
        ),
        "fold_count": 3,
        "supervised_fit_count": int(
            len(phase45_optimizer_records)
        ),
        "preprocessing_fit_count": int(
            len(phase45_preprocessing_records)
        ),
        "all_predictions_finite": bool(np.all(np.isfinite(
            phase45_candidate_monitor_probability
        ))),
        "optimizer_success_fraction": float(np.mean([
            record["success"]
            for record in phase45_optimizer_records
        ])),
        "optimizer_iteration_limit_hit_count": int(sum(
            record["iterations"]
            >= PHASE45_PROBE_SELECTION_CONFIG[
                "optimizer_maximum_iterations"
            ]
            for record in phase45_optimizer_records
        )),
    },
    "raw_log_loss_best": phase45_sanitize_candidate(
        phase45_raw_best
    ),
    "raw_auroc_best": phase45_sanitize_candidate(
        phase45_auroc_best
    ),
    "selected": phase45_sanitize_candidate(
        phase45_selected
    ),
    "selection_rule": (
        "maximum pooled monitor AUROC within 0.003 absolute "
        "log-loss of the pooled monitor log-loss minimum"
    ),
    "preprocessing": {
        "robust_scaling": "fit_partition_only",
        "pca": "fit_partition_only_randomized_svd",
        "pca_whiten": bool(
            PHASE45_PROBE_SELECTION_CONFIG["pca_whiten"]
        ),
        "transformed_absolute_clip": float(
            PHASE45_PROBE_SELECTION_CONFIG[
                "transformed_clip"
            ]
        ),
    },
    "ranking": {
        "within_acquisition_group_pairs_only": True,
        "maximum_pairs_per_fit_group": int(
            PHASE45_PROBE_SELECTION_CONFIG[
                "maximum_pairs_per_group"
            ]
        ),
        "equal_total_weight_per_eligible_fit_group": True,
        "pair_sampling_deterministic": True,
    },
    "shared_specification_across_folds": True,
    "fold_specific_probe_routing_planned": True,
    "fit_labels_used_for_probe_training": True,
    "monitor_labels_used_for_selection": True,
    "outer_validation_features_used_for_fit": False,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter()
        - phase45_probe_selection_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE45_PROBE_SELECTION")
print(json.dumps(
    phase45_probe_selection_report,
    indent=2,
))
print("END SANITIZED_PHASE45_PROBE_SELECTION")

Phase45 supervised probe selection: fold 1/3 complete
Phase45 supervised probe selection: fold 2/3 complete
Phase45 supervised probe selection: fold 3/3 complete
BEGIN SANITIZED_PHASE45_PROBE_SELECTION
{
  "phase": "phase45_frozen_embedding_shared_linear_probe_monitor_selection",
  "status": "probe_specification_frozen_before_outer_evaluation",
  "selection_data": {
    "monitor_observation_count": 493,
    "unique_monitor_case_count": 407,
    "maximum_case_multiplicity": 2,
    "inverse_multiplicity_weighting": true,
    "monitor_group_count": 10
  },
  "search": {
    "candidate_count": 180,
    "fold_count": 3,
    "supervised_fit_count": 540,
    "preprocessing_fit_count": 60,
    "all_predictions_finite": true,
    "optimizer_success_fraction": 1.0,
    "optimizer_iteration_limit_hit_count": 0
  },
  "raw_log_loss_best": {
    "candidate_index": 161,
    "specification": {
      "representation": "complete_bundle",
      "pca_dimension": 32,
      "rank_loss_weight": 0.3,
      "

In [49]:
# phase 45_cell_143C

import itertools
import json
import time

import numpy as np
from scipy.special import expit
from sklearn.metrics import roc_auc_score


phase45_integration_started = time.perf_counter()

PHASE45_INTEGRATION_CONFIG = {
    "probe_weights": [0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30],
    "residual_caps": [0.25, 0.50, 0.75, 1.00, 1.50, 2.00],
    "uncertainty_exponents": [0.0, 0.5, 1.0],
    "log_loss_selection_band": 0.001,
    "maximum_pooled_log_loss_excess": 0.0005,
    "maximum_fold_log_loss_regret": 0.003,
    "maximum_group_log_loss_harm": 0.015,
    "maximum_auroc_deficit": 0.0005,
    "minimum_fold_wins": 2,
    "minimum_log_loss_gain_for_advance": 0.001,
    "minimum_auroc_gain_for_advance": 0.001,
}


def phase45_clip_probability(probability):
    probability = np.asarray(probability, dtype=np.float64).reshape(-1)
    assert probability.shape == (1362,)
    assert np.all(np.isfinite(probability))
    return np.clip(probability, 1e-7, 1.0 - 1e-7)


def phase45_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-7,
        1.0 - 1e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase45_weighted_log_loss_local(labels, probability, weights=None):
    labels = np.asarray(labels, dtype=np.float64)
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-7,
        1.0 - 1e-7,
    )
    losses = -(
        labels * np.log(probability)
        + (1.0 - labels) * np.log1p(-probability)
    )
    if weights is None:
        return float(losses.mean())
    weights = np.asarray(weights, dtype=np.float64)
    return float(np.sum(weights * losses) / np.sum(weights))


def phase45_weighted_auroc_local(labels, probability, weights=None):
    labels = np.asarray(labels, dtype=np.float64)
    if np.unique(labels).size != 2:
        return None
    return float(roc_auc_score(
        labels,
        probability,
        sample_weight=weights,
    ))


required_names = [
    "PHASE39_OOF_PRIVATE",
    "PHASE45_PROBE_SELECTED_MONITOR_PROBABILITY_PRIVATE",
    "PHASE45_PROBE_SELECTED_SPECIFICATION_PRIVATE",
    "PHASE45_PROBE_MONITOR_INDICES_PRIVATE",
    "PHASE45_PROBE_MONITOR_WEIGHTS_PRIVATE",
    "PHASE44_Y_PRIVATE",
    "PHASE44_GROUPS_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Cell 143C is missing required private state.",
    "missing": missing_names,
}

phase45_y = np.asarray(PHASE44_Y_PRIVATE, dtype=np.float64).reshape(-1)
phase45_groups = np.asarray(PHASE44_GROUPS_PRIVATE, dtype=np.int64).reshape(-1)
phase45_anchor_oof = phase45_clip_probability(PHASE39_OOF_PRIVATE)
phase45_monitor_indices = np.asarray(
    PHASE45_PROBE_MONITOR_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase45_monitor_weights = np.asarray(
    PHASE45_PROBE_MONITOR_WEIGHTS_PRIVATE,
    dtype=np.float64,
).reshape(-1)
phase45_probe_monitor = np.clip(
    np.asarray(
        PHASE45_PROBE_SELECTED_MONITOR_PROBABILITY_PRIVATE,
        dtype=np.float64,
    ).reshape(-1),
    1e-7,
    1.0 - 1e-7,
)

assert phase45_y.shape == (1362,)
assert phase45_groups.shape == (1362,)
assert phase45_monitor_indices.size == 493
assert phase45_monitor_weights.shape == phase45_monitor_indices.shape
assert phase45_probe_monitor.shape == phase45_monitor_indices.shape
assert np.all(np.isfinite(phase45_probe_monitor))

phase45_monitor_y = phase45_y[phase45_monitor_indices]
phase45_monitor_groups = phase45_groups[phase45_monitor_indices]
phase45_anchor_monitor = phase45_anchor_oof[phase45_monitor_indices]
phase45_anchor_monitor_logit = phase45_logit(phase45_anchor_monitor)
phase45_probe_monitor_logit = phase45_logit(phase45_probe_monitor)
phase45_probe_residual = (
    phase45_probe_monitor_logit - phase45_anchor_monitor_logit
)
phase45_anchor_uncertainty = np.clip(
    4.0 * phase45_anchor_monitor * (1.0 - phase45_anchor_monitor),
    0.0,
    1.0,
)

phase45_monitor_fold = np.concatenate([
    np.full(
        len(partition["monitor_indices"]),
        int(partition["fold"]),
        dtype=np.int64,
    )
    for partition in PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE
])
assert phase45_monitor_fold.shape == phase45_monitor_indices.shape


def phase45_metrics(probability):
    probability = np.asarray(probability, dtype=np.float64)
    pooled_log_loss = phase45_weighted_log_loss_local(
        phase45_monitor_y,
        probability,
        phase45_monitor_weights,
    )
    pooled_auroc = phase45_weighted_auroc_local(
        phase45_monitor_y,
        probability,
        phase45_monitor_weights,
    )
    assert pooled_auroc is not None

    folds = []
    for fold in range(3):
        mask = phase45_monitor_fold == fold
        folds.append({
            "fold": int(fold),
            "n": int(mask.sum()),
            "log_loss": phase45_weighted_log_loss_local(
                phase45_monitor_y[mask], probability[mask]
            ),
            "auroc": phase45_weighted_auroc_local(
                phase45_monitor_y[mask], probability[mask]
            ),
        })

    groups = []
    for group in np.unique(phase45_monitor_groups):
        mask = phase45_monitor_groups == group
        groups.append({
            "group": int(group),
            "unique_n": int(np.unique(
                phase45_monitor_indices[mask]
            ).size),
            "log_loss": phase45_weighted_log_loss_local(
                phase45_monitor_y[mask],
                probability[mask],
                phase45_monitor_weights[mask],
            ),
            "auroc": phase45_weighted_auroc_local(
                phase45_monitor_y[mask],
                probability[mask],
                phase45_monitor_weights[mask],
            ),
        })
    return {
        "log_loss": float(pooled_log_loss),
        "auroc": float(pooled_auroc),
        "folds": folds,
        "groups": groups,
    }


phase45_anchor_metrics = phase45_metrics(phase45_anchor_monitor)
phase45_anchor_fold_loss = {
    record["fold"]: record["log_loss"]
    for record in phase45_anchor_metrics["folds"]
}
phase45_anchor_group_loss = {
    record["group"]: record["log_loss"]
    for record in phase45_anchor_metrics["groups"]
}

phase45_integration_candidates = [{
    "candidate_index": 0,
    "probe_weight": 0.0,
    "residual_cap": 0.0,
    "uncertainty_exponent": 0.0,
    "probability": phase45_anchor_monitor.copy(),
}]

for values in itertools.product(
    PHASE45_INTEGRATION_CONFIG["probe_weights"],
    PHASE45_INTEGRATION_CONFIG["residual_caps"],
    PHASE45_INTEGRATION_CONFIG["uncertainty_exponents"],
):
    probe_weight, residual_cap, uncertainty_exponent = values
    bounded_residual = np.clip(
        phase45_probe_residual,
        -float(residual_cap),
        float(residual_cap),
    )
    effective_weight = (
        float(probe_weight)
        * np.power(
            phase45_anchor_uncertainty,
            float(uncertainty_exponent),
        )
    )
    probability = expit(
        phase45_anchor_monitor_logit
        + effective_weight * bounded_residual
    )
    phase45_integration_candidates.append({
        "candidate_index": int(len(phase45_integration_candidates)),
        "probe_weight": float(probe_weight),
        "residual_cap": float(residual_cap),
        "uncertainty_exponent": float(uncertainty_exponent),
        "probability": np.asarray(probability, dtype=np.float64),
    })

assert len(phase45_integration_candidates) == 127

phase45_integration_records = []
for candidate in phase45_integration_candidates:
    metrics = phase45_metrics(candidate["probability"])
    fold_changes = [
        record["log_loss"]
        - phase45_anchor_fold_loss[record["fold"]]
        for record in metrics["folds"]
    ]
    group_changes = [
        record["log_loss"]
        - phase45_anchor_group_loss[record["group"]]
        for record in metrics["groups"]
    ]
    fold_wins = int(sum(change < -1e-12 for change in fold_changes))
    log_loss_gain = float(
        phase45_anchor_metrics["log_loss"] - metrics["log_loss"]
    )
    auroc_gain = float(
        metrics["auroc"] - phase45_anchor_metrics["auroc"]
    )
    safe = bool(
        metrics["log_loss"]
        <= phase45_anchor_metrics["log_loss"]
        + PHASE45_INTEGRATION_CONFIG[
            "maximum_pooled_log_loss_excess"
        ]
        + 1e-12
        and max(fold_changes)
        <= PHASE45_INTEGRATION_CONFIG[
            "maximum_fold_log_loss_regret"
        ]
        + 1e-12
        and max(group_changes)
        <= PHASE45_INTEGRATION_CONFIG[
            "maximum_group_log_loss_harm"
        ]
        + 1e-12
        and auroc_gain
        >= -PHASE45_INTEGRATION_CONFIG[
            "maximum_auroc_deficit"
        ]
        - 1e-12
        and (
            candidate["probe_weight"] == 0.0
            or fold_wins
            >= PHASE45_INTEGRATION_CONFIG["minimum_fold_wins"]
        )
    )
    phase45_integration_records.append({
        **{k: v for k, v in candidate.items() if k != "probability"},
        "log_loss": float(metrics["log_loss"]),
        "auroc": float(metrics["auroc"]),
        "log_loss_gain": log_loss_gain,
        "auroc_gain": auroc_gain,
        "fold_wins": fold_wins,
        "worst_fold_regret": float(max(fold_changes)),
        "maximum_group_harm": float(max(group_changes)),
        "safe": safe,
        "fold_metrics": metrics["folds"],
    })

phase45_safe_records = [
    record for record in phase45_integration_records if record["safe"]
]
assert phase45_safe_records

phase45_raw_best = min(
    phase45_safe_records,
    key=lambda record: (
        record["log_loss"],
        -record["auroc"],
        record["candidate_index"],
    ),
)
phase45_band_limit = (
    phase45_raw_best["log_loss"]
    + PHASE45_INTEGRATION_CONFIG["log_loss_selection_band"]
)
phase45_band_records = [
    record for record in phase45_safe_records
    if record["log_loss"] <= phase45_band_limit + 1e-12
]
phase45_provisional = max(
    phase45_band_records,
    key=lambda record: (
        record["auroc"],
        -record["log_loss"],
        -record["candidate_index"],
    ),
)
phase45_auroc_best = max(
    phase45_safe_records,
    key=lambda record: (
        record["auroc"],
        -record["log_loss"],
        -record["candidate_index"],
    ),
)

phase45_advance_by_log_loss = bool(
    phase45_provisional["log_loss_gain"]
    >= PHASE45_INTEGRATION_CONFIG[
        "minimum_log_loss_gain_for_advance"
    ]
    and phase45_provisional["auroc_gain"] >= 0.0
)
phase45_advance_by_auroc = bool(
    phase45_provisional["auroc_gain"]
    >= PHASE45_INTEGRATION_CONFIG[
        "minimum_auroc_gain_for_advance"
    ]
    and phase45_provisional["log_loss_gain"]
    >= -PHASE45_INTEGRATION_CONFIG[
        "maximum_pooled_log_loss_excess"
    ]
)
phase45_gate_advanced = bool(
    phase45_provisional["probe_weight"] > 0.0
    and (
        phase45_advance_by_log_loss
        or phase45_advance_by_auroc
    )
)
phase45_selected_integration = (
    phase45_provisional
    if phase45_gate_advanced
    else phase45_integration_records[0]
)


def phase45_public_record(record):
    return {
        "candidate_index": int(record["candidate_index"]),
        "specification": {
            "probe_weight": float(record["probe_weight"]),
            "residual_cap": float(record["residual_cap"]),
            "uncertainty_exponent": float(
                record["uncertainty_exponent"]
            ),
        },
        "log_loss": float(record["log_loss"]),
        "auroc": float(record["auroc"]),
        "log_loss_gain": float(record["log_loss_gain"]),
        "auroc_gain": float(record["auroc_gain"]),
        "fold_wins": int(record["fold_wins"]),
        "worst_fold_regret": float(record["worst_fold_regret"]),
        "maximum_group_harm": float(record["maximum_group_harm"]),
        "safe": bool(record["safe"]),
        "fold_metrics": record["fold_metrics"],
    }


PHASE45_INTEGRATION_RECORDS_PRIVATE = phase45_integration_records
PHASE45_INTEGRATION_SELECTED_PRIVATE = {
    "probe_weight": float(phase45_selected_integration["probe_weight"]),
    "residual_cap": float(phase45_selected_integration["residual_cap"]),
    "uncertainty_exponent": float(
        phase45_selected_integration["uncertainty_exponent"]
    ),
    "gate_advanced": bool(phase45_gate_advanced),
}

phase45_integration_report = {
    "phase": "phase45_frozen_probe_bounded_phase39_residual_integration",
    "status": (
        "integration_gate_frozen_for_outer_evaluation"
        if phase45_gate_advanced
        else "no_update_selected_stop_before_outer_evaluation"
    ),
    "frozen_probe": dict(PHASE45_PROBE_SELECTED_SPECIFICATION_PRIVATE),
    "anchor": {
        "component": "exact_cross_fitted_phase39_oof",
        "monitor_log_loss": float(phase45_anchor_metrics["log_loss"]),
        "monitor_auroc": float(phase45_anchor_metrics["auroc"]),
    },
    "search": {
        "candidate_count": int(len(phase45_integration_records)),
        "safe_candidate_count": int(len(phase45_safe_records)),
        "nonzero_candidate_count": 126,
        "selection_log_loss_band": float(
            PHASE45_INTEGRATION_CONFIG["log_loss_selection_band"]
        ),
    },
    "raw_log_loss_best": phase45_public_record(phase45_raw_best),
    "raw_auroc_best": phase45_public_record(phase45_auroc_best),
    "provisional_nonzero_or_zero_selection": phase45_public_record(
        phase45_provisional
    ),
    "selected": phase45_public_record(phase45_selected_integration),
    "gate_advanced": bool(phase45_gate_advanced),
    "advance_rule": (
        "gain_log_loss_by_0p001_without_auroc_loss_or_gain_auroc_by_0p001_"
        "with_at_most_0p0005_log_loss_excess"
    ),
    "formula_if_advanced": (
        "z45 = z39 + beta * uncertainty(z39)^gamma * "
        "clip(z_probe-z39, -cap, cap)"
    ),
    "monitor_labels_used_for_integration_selection": True,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase45_integration_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE45_INTEGRATION_SELECTION")
print(json.dumps(phase45_integration_report, indent=2))
print("END SANITIZED_PHASE45_INTEGRATION_SELECTION")

BEGIN SANITIZED_PHASE45_INTEGRATION_SELECTION
{
  "phase": "phase45_frozen_probe_bounded_phase39_residual_integration",
  "status": "no_update_selected_stop_before_outer_evaluation",
  "frozen_probe": {
    "representation": "complete_bundle",
    "pca_dimension": 32,
    "rank_loss_weight": 0.3,
    "l2_regularization": 0.01
  },
  "anchor": {
    "component": "exact_cross_fitted_phase39_oof",
    "monitor_log_loss": 0.24746871522270958,
    "monitor_auroc": 0.9577955039883974
  },
  "search": {
    "candidate_count": 127,
    "safe_candidate_count": 16,
    "nonzero_candidate_count": 126,
    "selection_log_loss_band": 0.001
  },
  "raw_log_loss_best": {
    "candidate_index": 0,
    "specification": {
      "probe_weight": 0.0,
      "residual_cap": 0.0,
      "uncertainty_exponent": 0.0
    },
    "log_loss": 0.24746871522270958,
    "auroc": 0.9577955039883974,
    "log_loss_gain": 0.0,
    "auroc_gain": 0.0,
    "fold_wins": 0,
    "worst_fold_regret": 0.0,
    "maximum_group_har

In [51]:
# phase46_cell_144A

import itertools
import json
import time

import numpy as np
import xgboost as xgb


phase46_contract_started = time.perf_counter()

PHASE46_CONFIG = {
    "seeds": [460701, 460702],
    "learning_rate": 0.03,
    "maximum_rounds": 600,
    "early_stopping_rounds": 50,
    "subsample": 0.85,
    "maximum_bin_count": 128,
    "maximum_delta_step": 1.0,
    "nthread": 8,
    "representations": [
        "symmetric",
        "absolute_antisymmetric",
        "invariant_combined",
        "complete_bundle",
    ],
    "maximum_depth_values": [1, 2],
    "minimum_child_weight_values": [8.0, 16.0, 32.0],
    "l2_regularization_values": [10.0, 30.0],
    "column_subsample_values": [0.5, 1.0],
    "gate_scales": [0.0, 0.10, 0.25, 0.50, 0.75, 1.0],
    "gate_caps": [0.50, 1.00, 1.50, 2.00],
    "gate_uncertainty_exponents": [0.0, 0.5, 1.0],
    "log_loss_selection_band": 0.001,
    "minimum_monitor_fold_wins": 2,
    "maximum_monitor_fold_regret": 0.005,
    "maximum_monitor_group_harm": 0.015,
}


required_names = [
    "PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE",
    "PHASE45_EMBEDDING_OOF_PRIVATE",
    "PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE",
    "PHASE39_OOF_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase46 requires the accepted Phase39 and Phase45 state.",
    "missing": missing_names,
}

phase46_embedding_by_fold = np.asarray(
    PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE,
    dtype=np.float32,
)
phase46_embedding_oof = np.asarray(
    PHASE45_EMBEDDING_OOF_PRIVATE,
    dtype=np.float32,
)
phase46_anchor_probability = np.clip(
    np.asarray(PHASE39_OOF_PRIVATE, dtype=np.float64).reshape(-1),
    1e-7,
    1.0 - 1e-7,
)
phase46_anchor_logit = (
    np.log(phase46_anchor_probability)
    - np.log1p(-phase46_anchor_probability)
)

assert phase46_embedding_by_fold.shape == (3, 1362, 576)
assert phase46_embedding_oof.shape == (1362, 576)
assert phase46_anchor_probability.shape == (1362,)
assert np.all(np.isfinite(phase46_embedding_by_fold))
assert np.all(np.isfinite(phase46_embedding_oof))
assert np.all(np.isfinite(phase46_anchor_logit))

PHASE46_REPRESENTATION_SLICES_PRIVATE = {
    "symmetric": ((192, 384),),
    "absolute_antisymmetric": ((384, 576),),
    "invariant_combined": ((192, 384), (384, 576)),
    "complete_bundle": ((0, 576),),
}


def phase46_representation(matrix, name):
    parts = [
        matrix[:, start:stop]
        for start, stop in PHASE46_REPRESENTATION_SLICES_PRIVATE[name]
    ]
    output = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=1)
    output = np.asarray(output, dtype=np.float32)
    assert output.ndim == 2
    assert np.all(np.isfinite(output))
    return output


phase46_representation_contracts = {}
for name in PHASE46_CONFIG["representations"]:
    matrix = phase46_representation(phase46_embedding_oof, name)
    standard_deviation = np.std(matrix.astype(np.float64), axis=0)
    phase46_representation_contracts[name] = {
        "dimension": int(matrix.shape[1]),
        "collapsed_coordinate_count": int(np.sum(standard_deviation <= 1e-10)),
        "median_coordinate_standard_deviation": float(np.median(standard_deviation)),
        "all_values_finite": bool(np.all(np.isfinite(matrix))),
    }

phase46_model_grid = []
for candidate_index, values in enumerate(itertools.product(
    PHASE46_CONFIG["representations"],
    PHASE46_CONFIG["maximum_depth_values"],
    PHASE46_CONFIG["minimum_child_weight_values"],
    PHASE46_CONFIG["l2_regularization_values"],
    PHASE46_CONFIG["column_subsample_values"],
)):
    representation, depth, child_weight, l2_value, column_subsample = values
    phase46_model_grid.append({
        "candidate_index": int(candidate_index),
        "representation": str(representation),
        "maximum_depth": int(depth),
        "minimum_child_weight": float(child_weight),
        "l2_regularization": float(l2_value),
        "column_subsample": float(column_subsample),
    })

assert len(phase46_model_grid) == 96

phase46_gate_grid = [{
    "scale": 0.0,
    "residual_cap": 0.0,
    "uncertainty_exponent": 0.0,
}]
for scale, cap, exponent in itertools.product(
    PHASE46_CONFIG["gate_scales"][1:],
    PHASE46_CONFIG["gate_caps"],
    PHASE46_CONFIG["gate_uncertainty_exponents"],
):
    phase46_gate_grid.append({
        "scale": float(scale),
        "residual_cap": float(cap),
        "uncertainty_exponent": float(exponent),
    })
assert len(phase46_gate_grid) == 61

# A booster containing zero trees must reproduce the supplied logit base margin.
phase46_contract_features = np.zeros((128, 8), dtype=np.float32)
phase46_contract_margin = np.linspace(-3.0, 3.0, 128, dtype=np.float32)
phase46_contract_labels = np.tile(
    np.asarray([0.0, 1.0], dtype=np.float32),
    64,
)
phase46_contract_matrix = xgb.DMatrix(
    phase46_contract_features,
    label=phase46_contract_labels,
)
phase46_contract_matrix.set_base_margin(phase46_contract_margin)
phase46_zero_tree_booster = xgb.train(
    params={
        "objective": "binary:logistic",
        "nthread": PHASE46_CONFIG["nthread"],
    },
    dtrain=phase46_contract_matrix,
    num_boost_round=0,
)
phase46_zero_tree_margin = phase46_zero_tree_booster.predict(
    phase46_contract_matrix,
    output_margin=True,
)
phase46_zero_tree_error = float(np.max(np.abs(
    phase46_zero_tree_margin.astype(np.float64)
    - phase46_contract_margin.astype(np.float64)
)))
assert phase46_zero_tree_error <= 1e-6, phase46_zero_tree_error

phase46_partition_records = []
for fold, partition in enumerate(PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE):
    fit_indices = np.asarray(partition["fit_indices"], dtype=np.int64)
    monitor_indices = np.asarray(partition["monitor_indices"], dtype=np.int64)
    outer_train_indices = np.asarray(partition["outer_train_indices"], dtype=np.int64)
    outer_valid_indices = np.asarray(partition["outer_valid_indices"], dtype=np.int64)
    assert np.intersect1d(fit_indices, monitor_indices).size == 0
    assert np.array_equal(
        np.sort(np.concatenate([fit_indices, monitor_indices])),
        np.sort(outer_train_indices),
    )
    assert np.intersect1d(outer_train_indices, outer_valid_indices).size == 0
    phase46_partition_records.append({
        "fold": int(fold),
        "fit_n": int(fit_indices.size),
        "monitor_n": int(monitor_indices.size),
        "outer_train_n": int(outer_train_indices.size),
        "outer_valid_n": int(outer_valid_indices.size),
    })

PHASE46_MODEL_GRID_PRIVATE = phase46_model_grid
PHASE46_GATE_GRID_PRIVATE = phase46_gate_grid

phase46_contract_report = {
    "phase": "phase46_phase39_anchored_ssl_embedding_boosting_contract",
    "status": "accepted",
    "backend": "xgboost",
    "backend_version": xgb.__version__,
    "motivation": {
        "phase45_standalone_monitor_log_loss": 0.4338636078185849,
        "phase45_standalone_monitor_auroc": 0.88311578438482,
        "phase45_direct_integration_advanced": False,
        "hypothesis": "conditional_nonlinear_signal_may_remain_in_frozen_ssl_embedding",
    },
    "embedding": {
        "by_fold_shape": list(phase46_embedding_by_fold.shape),
        "oof_shape": list(phase46_embedding_oof.shape),
        "representations": phase46_representation_contracts,
    },
    "base_margin": {
        "source": "exact_cross_fitted_phase39_logit",
        "shape": list(phase46_anchor_logit.shape),
        "zero_tree_maximum_absolute_error": phase46_zero_tree_error,
    },
    "model_grid": {
        "candidate_count": len(phase46_model_grid),
        "seed_count": len(PHASE46_CONFIG["seeds"]),
        "fold_count": 3,
        "planned_model_fit_count": (
            len(phase46_model_grid)
            * len(PHASE46_CONFIG["seeds"])
            * 3
        ),
        "specifications": {
            "representations": PHASE46_CONFIG["representations"],
            "maximum_depth_values": PHASE46_CONFIG["maximum_depth_values"],
            "minimum_child_weight_values": PHASE46_CONFIG["minimum_child_weight_values"],
            "l2_regularization_values": PHASE46_CONFIG["l2_regularization_values"],
            "column_subsample_values": PHASE46_CONFIG["column_subsample_values"],
        },
    },
    "gate_grid": {
        "candidate_count": len(phase46_gate_grid),
        "searched_only_after_one_shared_model_specification_is_frozen": True,
    },
    "partitions": phase46_partition_records,
    "selection_strategy": [
        "fit_each_candidate_on_fit_partition",
        "select_one_shared_model_specification_on_pooled_monitor",
        "freeze_model_specification",
        "search_one_shared_bounded_residual_gate_on_monitor",
        "freeze_gate_before_outer_evaluation",
    ],
    "primary_metric": "inverse_multiplicity_weighted_monitor_log_loss",
    "secondary_metric": "monitor_auroc_within_log_loss_band",
    "fit_labels_used": False,
    "monitor_labels_used": False,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_embeddings_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase46_contract_started, 3),
}

print("BEGIN SANITIZED_PHASE46_CONTRACT")
print(json.dumps(phase46_contract_report, indent=2))
print("END SANITIZED_PHASE46_CONTRACT")

BEGIN SANITIZED_PHASE46_CONTRACT
{
  "phase": "phase46_phase39_anchored_ssl_embedding_boosting_contract",
  "status": "accepted",
  "backend": "xgboost",
  "backend_version": "3.2.0",
  "motivation": {
    "phase45_standalone_monitor_log_loss": 0.4338636078185849,
    "phase45_standalone_monitor_auroc": 0.88311578438482,
    "phase45_direct_integration_advanced": false,
    "hypothesis": "conditional_nonlinear_signal_may_remain_in_frozen_ssl_embedding"
  },
  "embedding": {
    "by_fold_shape": [
      3,
      1362,
      576
    ],
    "oof_shape": [
      1362,
      576
    ],
    "representations": {
      "symmetric": {
        "dimension": 192,
        "collapsed_coordinate_count": 0,
        "median_coordinate_standard_deviation": 0.623753675877053,
        "all_values_finite": true
      },
      "absolute_antisymmetric": {
        "dimension": 192,
        "collapsed_coordinate_count": 0,
        "median_coordinate_standard_deviation": 0.00522084828645567,
        "all_values

In [52]:
# phase 46 cell 144B

import json
import time

import numpy as np
import xgboost as xgb
from scipy.special import expit
from sklearn.metrics import roc_auc_score


phase46_selection_started = time.perf_counter()

PHASE46_SELECTION_THRESHOLDS = {
    "model_screen_log_loss_band": 0.002,
    "model_screen_maximum_fold_regret": 0.010,
    "model_screen_maximum_group_harm": 0.030,
    "model_screen_maximum_auroc_deficit": 0.005,
    "gate_log_loss_band": 0.001,
    "gate_maximum_pooled_log_loss_excess": 0.0005,
    "gate_maximum_fold_regret": 0.005,
    "gate_maximum_group_harm": 0.015,
    "gate_maximum_auroc_deficit": 0.0005,
    "gate_minimum_fold_wins": 2,
    "minimum_log_loss_gain_for_advance": 0.002,
    "minimum_auroc_gain_for_advance": 0.001,
}


required_names = [
    "PHASE46_CONFIG",
    "PHASE46_MODEL_GRID_PRIVATE",
    "PHASE46_GATE_GRID_PRIVATE",
    "PHASE46_REPRESENTATION_SLICES_PRIVATE",
    "PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE",
    "PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE",
    "PHASE39_OOF_PRIVATE",
    "PHASE44_Y_PRIVATE",
    "PHASE44_GROUPS_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase46 training is missing required accepted state.",
    "missing": missing_names,
}


def phase46_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-7,
        1.0 - 1e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase46_log_loss(labels, probability, weights=None):
    labels = np.asarray(labels, dtype=np.float64)
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-7,
        1.0 - 1e-7,
    )
    losses = -(
        labels * np.log(probability)
        + (1.0 - labels) * np.log1p(-probability)
    )
    if weights is None:
        return float(losses.mean())
    weights = np.asarray(weights, dtype=np.float64)
    return float(np.sum(weights * losses) / np.sum(weights))


def phase46_auroc(labels, probability, weights=None):
    labels = np.asarray(labels, dtype=np.float64)
    if np.unique(labels).size != 2:
        return None
    return float(roc_auc_score(
        labels,
        probability,
        sample_weight=weights,
    ))


def phase46_group_balanced_weights(groups):
    groups = np.asarray(groups, dtype=np.int64)
    unique_groups, counts = np.unique(groups, return_counts=True)
    count_by_group = {
        int(group): int(count)
        for group, count in zip(unique_groups, counts)
    }
    weights = np.asarray([
        1.0 / count_by_group[int(group)]
        for group in groups
    ], dtype=np.float64)
    weights *= weights.size / weights.sum()
    assert np.isclose(weights.mean(), 1.0)
    return weights


def phase46_features(matrix, representation):
    parts = [
        matrix[:, start:stop]
        for start, stop in PHASE46_REPRESENTATION_SLICES_PRIVATE[
            representation
        ]
    ]
    output = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=1)
    output = np.asarray(output, dtype=np.float32)
    assert output.ndim == 2
    assert np.all(np.isfinite(output))
    return output


phase46_y = np.asarray(PHASE44_Y_PRIVATE, dtype=np.float64).reshape(-1)
phase46_groups = np.asarray(PHASE44_GROUPS_PRIVATE, dtype=np.int64).reshape(-1)
phase46_embedding_by_fold = np.asarray(
    PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE,
    dtype=np.float32,
)
phase46_anchor_probability = np.clip(
    np.asarray(PHASE39_OOF_PRIVATE, dtype=np.float64).reshape(-1),
    1e-7,
    1.0 - 1e-7,
)
phase46_anchor_logit = phase46_logit(phase46_anchor_probability)

assert phase46_y.shape == (1362,)
assert phase46_groups.shape == (1362,)
assert phase46_embedding_by_fold.shape == (3, 1362, 576)
assert phase46_anchor_probability.shape == (1362,)
assert set(np.unique(phase46_y).tolist()) == {0.0, 1.0}

phase46_partitions = []
for fold, partition in enumerate(PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE):
    record = {
        "fold": int(fold),
        "fit_indices": np.asarray(partition["fit_indices"], dtype=np.int64),
        "monitor_indices": np.asarray(partition["monitor_indices"], dtype=np.int64),
        "outer_train_indices": np.asarray(partition["outer_train_indices"], dtype=np.int64),
        "outer_valid_indices": np.asarray(partition["outer_valid_indices"], dtype=np.int64),
    }
    assert np.intersect1d(
        record["fit_indices"], record["monitor_indices"]
    ).size == 0
    assert np.array_equal(
        np.sort(np.concatenate([
            record["fit_indices"], record["monitor_indices"]
        ])),
        np.sort(record["outer_train_indices"]),
    )
    phase46_partitions.append(record)

phase46_monitor_indices = np.concatenate([
    partition["monitor_indices"] for partition in phase46_partitions
])
phase46_monitor_fold = np.concatenate([
    np.full(
        partition["monitor_indices"].size,
        partition["fold"],
        dtype=np.int64,
    )
    for partition in phase46_partitions
])
phase46_monitor_y = phase46_y[phase46_monitor_indices]
phase46_monitor_groups = phase46_groups[phase46_monitor_indices]
phase46_monitor_multiplicity = np.bincount(
    phase46_monitor_indices,
    minlength=1362,
)
phase46_monitor_weights = (
    1.0 / phase46_monitor_multiplicity[phase46_monitor_indices]
)
phase46_monitor_anchor_probability = phase46_anchor_probability[
    phase46_monitor_indices
]
phase46_monitor_anchor_logit = phase46_anchor_logit[
    phase46_monitor_indices
]

assert phase46_monitor_indices.size == 493
assert np.unique(phase46_monitor_indices).size == 407
assert phase46_monitor_multiplicity.max() == 2


def phase46_complete_metrics(probability):
    probability = np.asarray(probability, dtype=np.float64)
    assert probability.shape == phase46_monitor_y.shape
    assert np.all(np.isfinite(probability))
    pooled_log_loss = phase46_log_loss(
        phase46_monitor_y,
        probability,
        phase46_monitor_weights,
    )
    pooled_auroc = phase46_auroc(
        phase46_monitor_y,
        probability,
        phase46_monitor_weights,
    )
    assert pooled_auroc is not None

    fold_metrics = []
    for fold in range(3):
        mask = phase46_monitor_fold == fold
        fold_metrics.append({
            "fold": int(fold),
            "n": int(mask.sum()),
            "log_loss": phase46_log_loss(
                phase46_monitor_y[mask], probability[mask]
            ),
            "auroc": phase46_auroc(
                phase46_monitor_y[mask], probability[mask]
            ),
        })

    group_metrics = []
    for group in np.unique(phase46_monitor_groups):
        mask = phase46_monitor_groups == group
        group_metrics.append({
            "group": int(group),
            "unique_n": int(np.unique(
                phase46_monitor_indices[mask]
            ).size),
            "log_loss": phase46_log_loss(
                phase46_monitor_y[mask],
                probability[mask],
                phase46_monitor_weights[mask],
            ),
            "auroc": phase46_auroc(
                phase46_monitor_y[mask],
                probability[mask],
                phase46_monitor_weights[mask],
            ),
        })
    return {
        "log_loss": float(pooled_log_loss),
        "auroc": float(pooled_auroc),
        "fold_metrics": fold_metrics,
        "group_metrics": group_metrics,
    }


phase46_anchor_metrics = phase46_complete_metrics(
    phase46_monitor_anchor_probability
)
phase46_anchor_fold_loss = {
    record["fold"]: record["log_loss"]
    for record in phase46_anchor_metrics["fold_metrics"]
}
phase46_anchor_group_loss = {
    record["group"]: record["log_loss"]
    for record in phase46_anchor_metrics["group_metrics"]
}

candidate_count = len(PHASE46_MODEL_GRID_PRIVATE)
phase46_model_monitor_raw_residual = np.full(
    (candidate_count, phase46_monitor_indices.size),
    np.nan,
    dtype=np.float64,
)
phase46_round_counts = {
    int(specification["candidate_index"]): {
        fold: [] for fold in range(3)
    }
    for specification in PHASE46_MODEL_GRID_PRIVATE
}
phase46_fit_records = []

monitor_offset = 0
for partition in phase46_partitions:
    fold = partition["fold"]
    fit_indices = partition["fit_indices"]
    monitor_indices = partition["monitor_indices"]
    monitor_slice = slice(
        monitor_offset,
        monitor_offset + monitor_indices.size,
    )
    fold_embedding = phase46_embedding_by_fold[fold]
    fit_groups = phase46_groups[fit_indices]
    monitor_groups = phase46_groups[monitor_indices]
    fit_weights = phase46_group_balanced_weights(fit_groups)
    monitor_training_weights = phase46_group_balanced_weights(
        monitor_groups
    )

    specifications_by_representation = {
        representation: [
            specification
            for specification in PHASE46_MODEL_GRID_PRIVATE
            if specification["representation"] == representation
        ]
        for representation in PHASE46_CONFIG["representations"]
    }

    completed_in_fold = 0
    for representation in PHASE46_CONFIG["representations"]:
        fit_features = phase46_features(
            fold_embedding[fit_indices], representation
        )
        monitor_features = phase46_features(
            fold_embedding[monitor_indices], representation
        )
        dfit = xgb.DMatrix(
            fit_features,
            label=phase46_y[fit_indices],
            weight=fit_weights,
            base_margin=phase46_anchor_logit[fit_indices],
        )
        dmonitor = xgb.DMatrix(
            monitor_features,
            label=phase46_y[monitor_indices],
            weight=monitor_training_weights,
            base_margin=phase46_anchor_logit[monitor_indices],
        )

        for specification in specifications_by_representation[representation]:
            candidate_index = specification["candidate_index"]
            seed_residuals = []

            for seed in PHASE46_CONFIG["seeds"]:
                parameters = {
                    "objective": "binary:logistic",
                    "eval_metric": "logloss",
                    "tree_method": "hist",
                    "eta": PHASE46_CONFIG["learning_rate"],
                    "max_depth": specification["maximum_depth"],
                    "min_child_weight": specification[
                        "minimum_child_weight"
                    ],
                    "lambda": specification["l2_regularization"],
                    "alpha": 0.0,
                    "subsample": PHASE46_CONFIG["subsample"],
                    "colsample_bytree": specification[
                        "column_subsample"
                    ],
                    "max_bin": PHASE46_CONFIG["maximum_bin_count"],
                    "max_delta_step": PHASE46_CONFIG[
                        "maximum_delta_step"
                    ],
                    "seed": int(seed),
                    "nthread": PHASE46_CONFIG["nthread"],
                    "verbosity": 0,
                }
                booster = xgb.train(
                    params=parameters,
                    dtrain=dfit,
                    num_boost_round=PHASE46_CONFIG["maximum_rounds"],
                    evals=[(dmonitor, "monitor")],
                    early_stopping_rounds=PHASE46_CONFIG[
                        "early_stopping_rounds"
                    ],
                    verbose_eval=False,
                )
                best_iteration = int(booster.best_iteration)
                round_count = best_iteration + 1
                predicted_margin = booster.predict(
                    dmonitor,
                    output_margin=True,
                    iteration_range=(0, round_count),
                ).astype(np.float64)
                residual = (
                    predicted_margin
                    - phase46_anchor_logit[monitor_indices]
                )
                assert residual.shape == (monitor_indices.size,)
                assert np.all(np.isfinite(residual))
                seed_residuals.append(residual)
                phase46_round_counts[candidate_index][fold].append(
                    int(round_count)
                )
                phase46_fit_records.append({
                    "candidate_index": int(candidate_index),
                    "fold": int(fold),
                    "seed": int(seed),
                    "round_count": int(round_count),
                    "best_score": float(booster.best_score),
                })

            mean_residual = np.mean(
                np.stack(seed_residuals, axis=0),
                axis=0,
            )
            phase46_model_monitor_raw_residual[
                candidate_index,
                monitor_slice,
            ] = mean_residual
            completed_in_fold += 1
            if completed_in_fold % 12 == 0:
                print(
                    "Phase46 model selection: "
                    f"fold {fold}, {completed_in_fold}/{candidate_count} "
                    "specifications complete"
                )

    assert completed_in_fold == candidate_count
    monitor_offset += monitor_indices.size

assert monitor_offset == phase46_monitor_indices.size
assert len(phase46_fit_records) == candidate_count * 3 * 2
assert np.all(np.isfinite(phase46_model_monitor_raw_residual))


def phase46_changes_from_anchor(metrics):
    fold_changes = [
        record["log_loss"]
        - phase46_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ]
    group_changes = [
        record["log_loss"]
        - phase46_anchor_group_loss[record["group"]]
        for record in metrics["group_metrics"]
    ]
    return {
        "log_loss_gain": float(
            phase46_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": float(
            metrics["auroc"] - phase46_anchor_metrics["auroc"]
        ),
        "fold_wins": int(sum(change < -1e-12 for change in fold_changes)),
        "worst_fold_regret": float(max(fold_changes)),
        "maximum_group_harm": float(max(group_changes)),
    }


phase46_model_records = []
for specification in PHASE46_MODEL_GRID_PRIVATE:
    candidate_index = specification["candidate_index"]
    residual = phase46_model_monitor_raw_residual[candidate_index]
    probability = expit(phase46_monitor_anchor_logit + residual)
    metrics = phase46_complete_metrics(probability)
    changes = phase46_changes_from_anchor(metrics)
    screen_safe = bool(
        changes["worst_fold_regret"]
        <= PHASE46_SELECTION_THRESHOLDS[
            "model_screen_maximum_fold_regret"
        ] + 1e-12
        and changes["maximum_group_harm"]
        <= PHASE46_SELECTION_THRESHOLDS[
            "model_screen_maximum_group_harm"
        ] + 1e-12
        and changes["auroc_gain"]
        >= -PHASE46_SELECTION_THRESHOLDS[
            "model_screen_maximum_auroc_deficit"
        ] - 1e-12
    )
    phase46_model_records.append({
        **specification,
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        **changes,
        "screen_safe": screen_safe,
        "fold_metrics": metrics["fold_metrics"],
    })

phase46_screen_safe_records = [
    record for record in phase46_model_records if record["screen_safe"]
]
phase46_model_selection_pool = (
    phase46_screen_safe_records
    if phase46_screen_safe_records
    else phase46_model_records
)
phase46_model_raw_best = min(
    phase46_model_selection_pool,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["candidate_index"],
    ),
)
phase46_model_band_limit = (
    phase46_model_raw_best["monitor_log_loss"]
    + PHASE46_SELECTION_THRESHOLDS["model_screen_log_loss_band"]
)
phase46_model_band = [
    record for record in phase46_model_selection_pool
    if record["monitor_log_loss"] <= phase46_model_band_limit + 1e-12
]
phase46_selected_model = max(
    phase46_model_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["candidate_index"],
    ),
)

selected_model_index = phase46_selected_model["candidate_index"]
phase46_selected_raw_residual = (
    phase46_model_monitor_raw_residual[selected_model_index].copy()
)
phase46_uncertainty = np.clip(
    4.0
    * phase46_monitor_anchor_probability
    * (1.0 - phase46_monitor_anchor_probability),
    0.0,
    1.0,
)

phase46_gate_records = []
for gate_index, gate in enumerate(PHASE46_GATE_GRID_PRIVATE):
    if gate["scale"] == 0.0:
        applied_residual = np.zeros_like(phase46_selected_raw_residual)
    else:
        applied_residual = (
            gate["scale"]
            * np.power(
                phase46_uncertainty,
                gate["uncertainty_exponent"],
            )
            * np.clip(
                phase46_selected_raw_residual,
                -gate["residual_cap"],
                gate["residual_cap"],
            )
        )
    probability = expit(
        phase46_monitor_anchor_logit + applied_residual
    )
    metrics = phase46_complete_metrics(probability)
    changes = phase46_changes_from_anchor(metrics)
    safe = bool(
        metrics["log_loss"]
        <= phase46_anchor_metrics["log_loss"]
        + PHASE46_SELECTION_THRESHOLDS[
            "gate_maximum_pooled_log_loss_excess"
        ] + 1e-12
        and changes["worst_fold_regret"]
        <= PHASE46_SELECTION_THRESHOLDS[
            "gate_maximum_fold_regret"
        ] + 1e-12
        and changes["maximum_group_harm"]
        <= PHASE46_SELECTION_THRESHOLDS[
            "gate_maximum_group_harm"
        ] + 1e-12
        and changes["auroc_gain"]
        >= -PHASE46_SELECTION_THRESHOLDS[
            "gate_maximum_auroc_deficit"
        ] - 1e-12
        and (
            gate["scale"] == 0.0
            or changes["fold_wins"]
            >= PHASE46_SELECTION_THRESHOLDS[
                "gate_minimum_fold_wins"
            ]
        )
    )
    phase46_gate_records.append({
        "gate_index": int(gate_index),
        **gate,
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        **changes,
        "safe": safe,
        "fold_metrics": metrics["fold_metrics"],
    })

phase46_safe_gates = [record for record in phase46_gate_records if record["safe"]]
assert phase46_safe_gates
phase46_gate_raw_best = min(
    phase46_safe_gates,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["gate_index"],
    ),
)
phase46_gate_band_limit = (
    phase46_gate_raw_best["monitor_log_loss"]
    + PHASE46_SELECTION_THRESHOLDS["gate_log_loss_band"]
)
phase46_gate_band = [
    record for record in phase46_safe_gates
    if record["monitor_log_loss"] <= phase46_gate_band_limit + 1e-12
]
phase46_gate_provisional = max(
    phase46_gate_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["gate_index"],
    ),
)

advance_by_log_loss = bool(
    phase46_gate_provisional["log_loss_gain"]
    >= PHASE46_SELECTION_THRESHOLDS[
        "minimum_log_loss_gain_for_advance"
    ]
    and phase46_gate_provisional["auroc_gain"] >= 0.0
)
advance_by_auroc = bool(
    phase46_gate_provisional["auroc_gain"]
    >= PHASE46_SELECTION_THRESHOLDS[
        "minimum_auroc_gain_for_advance"
    ]
    and phase46_gate_provisional["log_loss_gain"]
    >= -PHASE46_SELECTION_THRESHOLDS[
        "gate_maximum_pooled_log_loss_excess"
    ]
)
phase46_gate_advanced = bool(
    phase46_gate_provisional["scale"] > 0.0
    and (advance_by_log_loss or advance_by_auroc)
)
phase46_selected_gate = (
    phase46_gate_provisional
    if phase46_gate_advanced
    else phase46_gate_records[0]
)


def phase46_model_public(record):
    return {
        "candidate_index": int(record["candidate_index"]),
        "specification": {
            "representation": record["representation"],
            "maximum_depth": int(record["maximum_depth"]),
            "minimum_child_weight": float(record["minimum_child_weight"]),
            "l2_regularization": float(record["l2_regularization"]),
            "column_subsample": float(record["column_subsample"]),
        },
        "monitor_log_loss": float(record["monitor_log_loss"]),
        "monitor_auroc": float(record["monitor_auroc"]),
        "log_loss_gain": float(record["log_loss_gain"]),
        "auroc_gain": float(record["auroc_gain"]),
        "fold_wins": int(record["fold_wins"]),
        "worst_fold_regret": float(record["worst_fold_regret"]),
        "maximum_group_harm": float(record["maximum_group_harm"]),
        "screen_safe": bool(record["screen_safe"]),
        "fold_metrics": record["fold_metrics"],
    }


def phase46_gate_public(record):
    return {
        "gate_index": int(record["gate_index"]),
        "specification": {
            "scale": float(record["scale"]),
            "residual_cap": float(record["residual_cap"]),
            "uncertainty_exponent": float(record["uncertainty_exponent"]),
        },
        "monitor_log_loss": float(record["monitor_log_loss"]),
        "monitor_auroc": float(record["monitor_auroc"]),
        "log_loss_gain": float(record["log_loss_gain"]),
        "auroc_gain": float(record["auroc_gain"]),
        "fold_wins": int(record["fold_wins"]),
        "worst_fold_regret": float(record["worst_fold_regret"]),
        "maximum_group_harm": float(record["maximum_group_harm"]),
        "safe": bool(record["safe"]),
        "fold_metrics": record["fold_metrics"],
    }


PHASE46_MODEL_RECORDS_PRIVATE = phase46_model_records
PHASE46_MODEL_MONITOR_RAW_RESIDUAL_PRIVATE = (
    phase46_model_monitor_raw_residual
)
PHASE46_SELECTED_MODEL_PRIVATE = {
    key: phase46_selected_model[key]
    for key in [
        "candidate_index",
        "representation",
        "maximum_depth",
        "minimum_child_weight",
        "l2_regularization",
        "column_subsample",
    ]
}
PHASE46_SELECTED_ROUND_COUNTS_PRIVATE = {
    int(fold): [int(value) for value in values]
    for fold, values in phase46_round_counts[
        selected_model_index
    ].items()
}
PHASE46_SELECTED_GATE_PRIVATE = {
    "scale": float(phase46_selected_gate["scale"]),
    "residual_cap": float(phase46_selected_gate["residual_cap"]),
    "uncertainty_exponent": float(
        phase46_selected_gate["uncertainty_exponent"]
    ),
    "gate_advanced": bool(phase46_gate_advanced),
}

phase46_selection_report = {
    "phase": "phase46_phase39_anchored_ssl_embedding_boosting_selection",
    "status": (
        "model_and_gate_frozen_for_outer_evaluation"
        if phase46_gate_advanced
        else "no_update_selected_stop_before_outer_evaluation"
    ),
    "anchor_monitor": {
        "log_loss": float(phase46_anchor_metrics["log_loss"]),
        "auroc": float(phase46_anchor_metrics["auroc"]),
    },
    "training": {
        "model_specification_count": int(candidate_count),
        "seed_count": len(PHASE46_CONFIG["seeds"]),
        "fold_count": 3,
        "trained_model_count": len(phase46_fit_records),
        "group_balanced_fit_weights": True,
        "group_balanced_early_stopping_weights": True,
    },
    "model_selection": {
        "screen_safe_candidate_count": len(phase46_screen_safe_records),
        "fallback_to_all_candidates": not bool(phase46_screen_safe_records),
        "raw_log_loss_best": phase46_model_public(phase46_model_raw_best),
        "selected": phase46_model_public(phase46_selected_model),
        "selected_round_counts": PHASE46_SELECTED_ROUND_COUNTS_PRIVATE,
    },
    "gate_selection": {
        "candidate_count": len(phase46_gate_records),
        "safe_candidate_count": len(phase46_safe_gates),
        "raw_log_loss_best": phase46_gate_public(phase46_gate_raw_best),
        "provisional": phase46_gate_public(phase46_gate_provisional),
        "selected": phase46_gate_public(phase46_selected_gate),
        "gate_advanced": bool(phase46_gate_advanced),
    },
    "selection_rule": (
        "freeze maximum AUROC model within 0.002 of safe raw model log-loss "
        "minimum; then freeze maximum AUROC gate within 0.001 of safe gate "
        "log-loss minimum"
    ),
    "formula_if_advanced": (
        "z46 = z39 + scale * uncertainty(z39)^gamma * "
        "clip(r46_raw, -cap, cap)"
    ),
    "shared_model_specification_across_folds": True,
    "shared_gate_across_folds": True,
    "fit_labels_used_for_training": True,
    "monitor_labels_used_for_selection": True,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase46_selection_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE46_SELECTION")
print(json.dumps(phase46_selection_report, indent=2))
print("END SANITIZED_PHASE46_SELECTION")

Phase46 model selection: fold 0, 12/96 specifications complete
Phase46 model selection: fold 0, 24/96 specifications complete
Phase46 model selection: fold 0, 36/96 specifications complete
Phase46 model selection: fold 0, 48/96 specifications complete
Phase46 model selection: fold 0, 60/96 specifications complete
Phase46 model selection: fold 0, 72/96 specifications complete
Phase46 model selection: fold 0, 84/96 specifications complete
Phase46 model selection: fold 0, 96/96 specifications complete
Phase46 model selection: fold 1, 12/96 specifications complete
Phase46 model selection: fold 1, 24/96 specifications complete
Phase46 model selection: fold 1, 36/96 specifications complete
Phase46 model selection: fold 1, 48/96 specifications complete
Phase46 model selection: fold 1, 60/96 specifications complete
Phase46 model selection: fold 1, 72/96 specifications complete
Phase46 model selection: fold 1, 84/96 specifications complete
Phase46 model selection: fold 1, 96/96 specifications c

In [53]:
# phase 46 cell144C

import json
import time

import numpy as np
import xgboost as xgb
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import roc_auc_score


phase46_outer_started = time.perf_counter()

PHASE46_OUTER_THRESHOLDS = {
    "minimum_log_loss_gain_over_phase39": 0.001,
    "minimum_auroc_gain_over_phase39": 0.0005,
    "minimum_fold_wins": 2,
    "maximum_worst_fold_excess": 0.003,
    "maximum_major_group_harm": 0.015,
    "calibration_slope_minimum": 0.80,
    "calibration_slope_maximum": 1.20,
}


required_names = [
    "PHASE46_CONFIG",
    "PHASE46_SELECTED_MODEL_PRIVATE",
    "PHASE46_SELECTED_GATE_PRIVATE",
    "PHASE46_SELECTED_ROUND_COUNTS_PRIVATE",
    "PHASE46_REPRESENTATION_SLICES_PRIVATE",
    "PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE",
    "PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE",
    "PHASE39_OOF_PRIVATE",
    "PHASE44_Y_PRIVATE",
    "PHASE44_GROUPS_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase46 outer evaluation is missing frozen state.",
    "missing": missing_names,
}
assert bool(PHASE46_SELECTED_GATE_PRIVATE["gate_advanced"]), {
    "message": "Phase46 monitor gate did not advance."
}


def phase46_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-7,
        1.0 - 1e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase46_log_loss(labels, probability):
    labels = np.asarray(labels, dtype=np.float64)
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-7,
        1.0 - 1e-7,
    )
    return float(np.mean(-(
        labels * np.log(probability)
        + (1.0 - labels) * np.log1p(-probability)
    )))


def phase46_auroc(labels, probability):
    labels = np.asarray(labels, dtype=np.float64)
    if np.unique(labels).size != 2:
        return None
    return float(roc_auc_score(labels, probability))


def phase46_group_balanced_weights(groups):
    groups = np.asarray(groups, dtype=np.int64)
    unique_groups, counts = np.unique(groups, return_counts=True)
    count_by_group = {
        int(group): int(count)
        for group, count in zip(unique_groups, counts)
    }
    weights = np.asarray([
        1.0 / count_by_group[int(group)]
        for group in groups
    ], dtype=np.float64)
    weights *= weights.size / weights.sum()
    assert np.isclose(weights.mean(), 1.0)
    return weights


def phase46_features(matrix, representation):
    parts = [
        matrix[:, start:stop]
        for start, stop in PHASE46_REPRESENTATION_SLICES_PRIVATE[
            representation
        ]
    ]
    output = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=1)
    output = np.asarray(output, dtype=np.float32)
    assert output.ndim == 2
    assert np.all(np.isfinite(output))
    return output


def phase46_calibration_parameters(labels, probability):
    labels = np.asarray(labels, dtype=np.float64)
    predictor = phase46_logit(probability)

    def objective(parameters):
        intercept, slope = parameters
        score = intercept + slope * predictor
        loss = np.mean(
            np.logaddexp(0.0, score) - labels * score
        )
        residual = (expit(score) - labels) / labels.size
        gradient = np.asarray([
            residual.sum(),
            np.sum(residual * predictor),
        ])
        return float(loss), gradient

    result = minimize(
        objective,
        np.asarray([0.0, 1.0], dtype=np.float64),
        method="L-BFGS-B",
        jac=True,
        options={"maxiter": 200, "ftol": 1e-12, "gtol": 1e-8},
    )
    assert np.all(np.isfinite(result.x))
    return {
        "intercept": float(result.x[0]),
        "slope": float(result.x[1]),
        "optimizer_success": bool(result.success),
    }


phase46_y = np.asarray(PHASE44_Y_PRIVATE, dtype=np.float64).reshape(-1)
phase46_groups = np.asarray(PHASE44_GROUPS_PRIVATE, dtype=np.int64).reshape(-1)
phase46_embedding_by_fold = np.asarray(
    PHASE45_EMBEDDING_BUNDLE_BY_FOLD_PRIVATE,
    dtype=np.float32,
)
phase46_anchor_probability = np.clip(
    np.asarray(PHASE39_OOF_PRIVATE, dtype=np.float64).reshape(-1),
    1e-7,
    1.0 - 1e-7,
)
phase46_anchor_logit = phase46_logit(phase46_anchor_probability)

assert phase46_y.shape == (1362,)
assert phase46_groups.shape == (1362,)
assert phase46_embedding_by_fold.shape == (3, 1362, 576)
assert phase46_anchor_probability.shape == (1362,)

selected_model = dict(PHASE46_SELECTED_MODEL_PRIVATE)
selected_gate = dict(PHASE46_SELECTED_GATE_PRIVATE)
selected_round_counts = {
    int(fold): [int(value) for value in values]
    for fold, values in PHASE46_SELECTED_ROUND_COUNTS_PRIVATE.items()
}

assert selected_model["representation"] in PHASE46_CONFIG["representations"]
assert set(selected_round_counts) == {0, 1, 2}
assert all(len(selected_round_counts[fold]) == 2 for fold in range(3))
assert all(value >= 1 for values in selected_round_counts.values() for value in values)
assert 0.0 < float(selected_gate["scale"]) <= 1.0
assert float(selected_gate["residual_cap"]) > 0.0

phase46_raw_residual_oof = np.full(1362, np.nan, dtype=np.float64)
phase46_probability_oof = np.full(1362, np.nan, dtype=np.float64)
phase46_fold_assignment = np.full(1362, -1, dtype=np.int64)
phase46_deployment_states = []
phase46_fold_records = []

for fold, partition in enumerate(PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE):
    fold_started = time.perf_counter()
    outer_train_indices = np.asarray(
        partition["outer_train_indices"], dtype=np.int64
    )
    outer_valid_indices = np.asarray(
        partition["outer_valid_indices"], dtype=np.int64
    )
    assert np.intersect1d(outer_train_indices, outer_valid_indices).size == 0
    assert outer_train_indices.size + outer_valid_indices.size == 1362
    assert np.all(phase46_fold_assignment[outer_valid_indices] == -1)

    fold_embedding = phase46_embedding_by_fold[fold]
    train_features = phase46_features(
        fold_embedding[outer_train_indices],
        selected_model["representation"],
    )
    valid_features = phase46_features(
        fold_embedding[outer_valid_indices],
        selected_model["representation"],
    )
    train_weights = phase46_group_balanced_weights(
        phase46_groups[outer_train_indices]
    )
    dtrain = xgb.DMatrix(
        train_features,
        label=phase46_y[outer_train_indices],
        weight=train_weights,
        base_margin=phase46_anchor_logit[outer_train_indices],
    )
    dvalid = xgb.DMatrix(
        valid_features,
        base_margin=phase46_anchor_logit[outer_valid_indices],
    )

    fold_boosters = []
    seed_residuals = []
    for seed_index, seed in enumerate(PHASE46_CONFIG["seeds"]):
        round_count = selected_round_counts[fold][seed_index]
        parameters = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "eta": PHASE46_CONFIG["learning_rate"],
            "max_depth": int(selected_model["maximum_depth"]),
            "min_child_weight": float(
                selected_model["minimum_child_weight"]
            ),
            "lambda": float(selected_model["l2_regularization"]),
            "alpha": 0.0,
            "subsample": PHASE46_CONFIG["subsample"],
            "colsample_bytree": float(
                selected_model["column_subsample"]
            ),
            "max_bin": PHASE46_CONFIG["maximum_bin_count"],
            "max_delta_step": PHASE46_CONFIG["maximum_delta_step"],
            "seed": int(seed),
            "nthread": PHASE46_CONFIG["nthread"],
            "verbosity": 0,
        }
        booster = xgb.train(
            params=parameters,
            dtrain=dtrain,
            num_boost_round=round_count,
            verbose_eval=False,
        )
        predicted_margin = booster.predict(
            dvalid,
            output_margin=True,
        ).astype(np.float64)
        residual = (
            predicted_margin
            - phase46_anchor_logit[outer_valid_indices]
        )
        assert residual.shape == (outer_valid_indices.size,)
        assert np.all(np.isfinite(residual))
        seed_residuals.append(residual)
        fold_boosters.append(booster)

    raw_residual = np.mean(np.stack(seed_residuals, axis=0), axis=0)
    anchor_probability = phase46_anchor_probability[outer_valid_indices]
    uncertainty = np.clip(
        4.0 * anchor_probability * (1.0 - anchor_probability),
        0.0,
        1.0,
    )
    applied_update = (
        float(selected_gate["scale"])
        * np.power(
            uncertainty,
            float(selected_gate["uncertainty_exponent"]),
        )
        * np.clip(
            raw_residual,
            -float(selected_gate["residual_cap"]),
            float(selected_gate["residual_cap"]),
        )
    )
    probability = expit(
        phase46_anchor_logit[outer_valid_indices] + applied_update
    )
    assert np.all(np.isfinite(probability))
    assert np.all((probability > 0.0) & (probability < 1.0))

    phase46_raw_residual_oof[outer_valid_indices] = raw_residual
    phase46_probability_oof[outer_valid_indices] = probability
    phase46_fold_assignment[outer_valid_indices] = fold

    fold_anchor_log_loss = phase46_log_loss(
        phase46_y[outer_valid_indices],
        phase46_anchor_probability[outer_valid_indices],
    )
    fold_candidate_log_loss = phase46_log_loss(
        phase46_y[outer_valid_indices], probability
    )
    fold_anchor_auroc = phase46_auroc(
        phase46_y[outer_valid_indices],
        phase46_anchor_probability[outer_valid_indices],
    )
    fold_candidate_auroc = phase46_auroc(
        phase46_y[outer_valid_indices], probability
    )
    assert fold_anchor_auroc is not None and fold_candidate_auroc is not None

    phase46_fold_records.append({
        "fold": int(fold),
        "n": int(outer_valid_indices.size),
        "round_counts": selected_round_counts[fold],
        "anchor_log_loss": float(fold_anchor_log_loss),
        "phase46_log_loss": float(fold_candidate_log_loss),
        "log_loss_gain": float(
            fold_anchor_log_loss - fold_candidate_log_loss
        ),
        "anchor_auroc": float(fold_anchor_auroc),
        "phase46_auroc": float(fold_candidate_auroc),
        "auroc_gain": float(
            fold_candidate_auroc - fold_anchor_auroc
        ),
        "mean_absolute_raw_residual": float(np.mean(np.abs(raw_residual))),
        "mean_absolute_applied_update": float(np.mean(np.abs(applied_update))),
        "elapsed_seconds": round(time.perf_counter() - fold_started, 3),
    })
    phase46_deployment_states.append({
        "fold": int(fold),
        "representation": selected_model["representation"],
        "model_specification": dict(selected_model),
        "gate": dict(selected_gate),
        "round_counts": selected_round_counts[fold],
        "seeds": [int(seed) for seed in PHASE46_CONFIG["seeds"]],
        "boosters": fold_boosters,
    })
    print(
        f"Phase46 outer fold {fold}: "
        f"log_loss={fold_candidate_log_loss:.6f}, "
        f"gain={fold_anchor_log_loss - fold_candidate_log_loss:.6f}, "
        f"auroc={fold_candidate_auroc:.6f}"
    )

assert np.all(phase46_fold_assignment >= 0)
assert np.all(np.isfinite(phase46_raw_residual_oof))
assert np.all(np.isfinite(phase46_probability_oof))
assert sorted(np.bincount(phase46_fold_assignment).tolist()) == sorted([467, 443, 452])

phase46_anchor_log_loss = phase46_log_loss(
    phase46_y, phase46_anchor_probability
)
phase46_log_loss_value = phase46_log_loss(
    phase46_y, phase46_probability_oof
)
phase46_anchor_auroc = phase46_auroc(
    phase46_y, phase46_anchor_probability
)
phase46_auroc_value = phase46_auroc(
    phase46_y, phase46_probability_oof
)
assert phase46_anchor_auroc is not None and phase46_auroc_value is not None

phase46_log_loss_gain = float(
    phase46_anchor_log_loss - phase46_log_loss_value
)
phase46_auroc_gain = float(
    phase46_auroc_value - phase46_anchor_auroc
)
phase46_fold_wins = int(sum(
    record["log_loss_gain"] > 0.0
    for record in phase46_fold_records
))
phase46_worst_fold_excess = float(max(
    -record["log_loss_gain"]
    for record in phase46_fold_records
))

phase46_group_records = []
for group in np.unique(phase46_groups):
    mask = phase46_groups == group
    anchor_group_log_loss = phase46_log_loss(
        phase46_y[mask], phase46_anchor_probability[mask]
    )
    candidate_group_log_loss = phase46_log_loss(
        phase46_y[mask], phase46_probability_oof[mask]
    )
    anchor_group_auroc = phase46_auroc(
        phase46_y[mask], phase46_anchor_probability[mask]
    )
    candidate_group_auroc = phase46_auroc(
        phase46_y[mask], phase46_probability_oof[mask]
    )
    phase46_group_records.append({
        "group": int(group),
        "n": int(mask.sum()),
        "anchor_log_loss": float(anchor_group_log_loss),
        "phase46_log_loss": float(candidate_group_log_loss),
        "log_loss_change": float(
            candidate_group_log_loss - anchor_group_log_loss
        ),
        "anchor_auroc": anchor_group_auroc,
        "phase46_auroc": candidate_group_auroc,
    })

phase46_major_group_records = [
    record for record in phase46_group_records if record["n"] >= 30
]
phase46_maximum_major_group_harm = float(max(
    record["log_loss_change"]
    for record in phase46_major_group_records
))
phase46_major_group_wins = int(sum(
    record["log_loss_change"] < 0.0
    for record in phase46_major_group_records
))
phase46_calibration = phase46_calibration_parameters(
    phase46_y, phase46_probability_oof
)

phase46_component_gate_passed = bool(
    phase46_log_loss_gain
    >= PHASE46_OUTER_THRESHOLDS[
        "minimum_log_loss_gain_over_phase39"
    ]
    and phase46_auroc_gain
    >= PHASE46_OUTER_THRESHOLDS[
        "minimum_auroc_gain_over_phase39"
    ]
    and phase46_fold_wins
    >= PHASE46_OUTER_THRESHOLDS["minimum_fold_wins"]
    and phase46_worst_fold_excess
    <= PHASE46_OUTER_THRESHOLDS[
        "maximum_worst_fold_excess"
    ]
    and phase46_maximum_major_group_harm
    <= PHASE46_OUTER_THRESHOLDS[
        "maximum_major_group_harm"
    ]
    and PHASE46_OUTER_THRESHOLDS["calibration_slope_minimum"]
    <= phase46_calibration["slope"]
    <= PHASE46_OUTER_THRESHOLDS["calibration_slope_maximum"]
)

PHASE46_RAW_RESIDUAL_OOF_PRIVATE = phase46_raw_residual_oof
PHASE46_OOF_PRIVATE = phase46_probability_oof
PHASE46_DEPLOYMENT_STATES_PRIVATE = phase46_deployment_states

phase46_outer_report = {
    "phase": "phase46_phase39_anchored_ssl_embedding_boosting_outer_oof",
    "status": (
        "eligible_for_fixed_candidate_shift_stress"
        if phase46_component_gate_passed
        else "not_promoted"
    ),
    "frozen_model_specification": {
        "representation": selected_model["representation"],
        "maximum_depth": int(selected_model["maximum_depth"]),
        "minimum_child_weight": float(selected_model["minimum_child_weight"]),
        "l2_regularization": float(selected_model["l2_regularization"]),
        "column_subsample": float(selected_model["column_subsample"]),
    },
    "frozen_gate": {
        "scale": float(selected_gate["scale"]),
        "residual_cap": float(selected_gate["residual_cap"]),
        "uncertainty_exponent": float(
            selected_gate["uncertainty_exponent"]
        ),
    },
    "anchor_phase39": {
        "log_loss": float(phase46_anchor_log_loss),
        "auroc": float(phase46_anchor_auroc),
        "mean_probability": float(np.mean(phase46_anchor_probability)),
    },
    "phase46": {
        "log_loss": float(phase46_log_loss_value),
        "auroc": float(phase46_auroc_value),
        "mean_probability": float(np.mean(phase46_probability_oof)),
        "calibration_intercept": float(phase46_calibration["intercept"]),
        "calibration_slope": float(phase46_calibration["slope"]),
    },
    "improvements": {
        "log_loss_gain_over_phase39": phase46_log_loss_gain,
        "auroc_gain_over_phase39": phase46_auroc_gain,
        "fold_wins": phase46_fold_wins,
        "worst_fold_excess": phase46_worst_fold_excess,
        "major_group_wins": phase46_major_group_wins,
        "major_group_count": len(phase46_major_group_records),
        "maximum_major_group_harm": phase46_maximum_major_group_harm,
    },
    "fold_metrics": phase46_fold_records,
    "major_group_metrics": phase46_major_group_records,
    "update_magnitude": {
        "mean_absolute_raw_residual": float(np.mean(np.abs(
            phase46_raw_residual_oof
        ))),
        "q95_absolute_raw_residual": float(np.quantile(
            np.abs(phase46_raw_residual_oof), 0.95
        )),
        "cap_activation_fraction": float(np.mean(
            np.abs(phase46_raw_residual_oof)
            >= float(selected_gate["residual_cap"])
        )),
        "mean_absolute_probability_change": float(np.mean(np.abs(
            phase46_probability_oof - phase46_anchor_probability
        ))),
        "q95_absolute_probability_change": float(np.quantile(
            np.abs(phase46_probability_oof - phase46_anchor_probability),
            0.95,
        )),
    },
    "promotion_thresholds": PHASE46_OUTER_THRESHOLDS,
    "component_gate_passed": bool(phase46_component_gate_passed),
    "selection_frozen_before_outer_evaluation": True,
    "outer_validation_labels_used_only_for_final_evaluation": True,
    "shared_model_specification_across_folds": True,
    "shared_gate_across_folds": True,
    "fold_specific_deployment_models_retained_privately": True,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase46_outer_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE46_OUTER_OOF")
print(json.dumps(phase46_outer_report, indent=2))
print("END SANITIZED_PHASE46_OUTER_OOF")

Phase46 outer fold 0: log_loss=0.326452, gain=-0.003060, auroc=0.938902
Phase46 outer fold 1: log_loss=0.258111, gain=-0.000395, auroc=0.953182
Phase46 outer fold 2: log_loss=0.295259, gain=0.001004, auroc=0.948002
BEGIN SANITIZED_PHASE46_OUTER_OOF
{
  "phase": "phase46_phase39_anchored_ssl_embedding_boosting_outer_oof",
  "status": "not_promoted",
  "frozen_model_specification": {
    "representation": "absolute_antisymmetric",
    "maximum_depth": 1,
    "minimum_child_weight": 8.0,
    "l2_regularization": 10.0,
    "column_subsample": 1.0
  },
  "frozen_gate": {
    "scale": 1.0,
    "residual_cap": 0.5,
    "uncertainty_exponent": 0.0
  },
  "anchor_phase39": {
    "log_loss": 0.29302734886541737,
    "auroc": 0.9455752549493367,
    "mean_probability": 0.5609995096990729
  },
  "phase46": {
    "log_loss": 0.2938717972896911,
    "auroc": 0.945115965215877,
    "mean_probability": 0.5613385361630711,
    "calibration_intercept": -0.17627883505869613,
    "calibration_slope": 0.88

In [5]:
# phase47_cell_145A

import inspect
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch


phase47_audit_started = time.perf_counter()

required_names = [
    "Phase45SpectralMaskedAutoencoder3D",
    "PHASE45_MODEL_CONFIG",
    "PHASE45_DEVICE",
    "PHASE45_HIGHRES_CACHE_PATH_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase47 interface audit is missing Phase45 state.",
    "missing": missing_names,
}


def phase47_signature(value):
    try:
        return str(inspect.signature(value))
    except (TypeError, ValueError):
        return None


def phase47_shape_tree(value):
    if torch.is_tensor(value):
        return {
            "type": "tensor",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "finite": bool(torch.isfinite(value).all().item()),
        }
    if isinstance(value, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "finite": bool(np.isfinite(value).all()),
        }
    if isinstance(value, dict):
        return {
            str(key): phase47_shape_tree(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [phase47_shape_tree(item) for item in value]
    if value is None:
        return None
    return {"type": type(value).__name__}


def phase47_resolve_tensor_state(payload):
    if isinstance(payload, dict):
        if payload and all(torch.is_tensor(value) for value in payload.values()):
            return "root", payload
        for key in (
            "encoder_state_dict",
            "model_state_dict",
            "state_dict",
            "encoder",
            "model",
        ):
            value = payload.get(key)
            if (
                isinstance(value, dict)
                and value
                and all(torch.is_tensor(item) for item in value.values())
            ):
                return f"root.{key}", value
    raise AssertionError({
        "message": "Could not resolve the encoder tensor state.",
        "payload_type": type(payload).__name__,
        "top_level_keys": (
            sorted(str(key) for key in payload.keys())
            if isinstance(payload, dict)
            else None
        ),
    })


phase47_class = Phase45SpectralMaskedAutoencoder3D
phase47_class_methods = []
for name, value in phase47_class.__dict__.items():
    if name.startswith("__") or not callable(value):
        continue
    try:
        source_line_count = len(inspect.getsource(value).splitlines())
    except (OSError, TypeError):
        source_line_count = None
    phase47_class_methods.append({
        "name": str(name),
        "signature": phase47_signature(value),
        "source_line_count": source_line_count,
    })

phase47_class_methods.sort(key=lambda record: record["name"])

if "phase45_create_pretraining_model" in globals():
    phase47_model = phase45_create_pretraining_model()
    phase47_constructor_route = "phase45_create_pretraining_model"
else:
    phase47_model = Phase45SpectralMaskedAutoencoder3D(
        input_channels=PHASE45_MODEL_CONFIG["input_channels"],
        input_size=PHASE45_MODEL_CONFIG["input_size"],
        patch_size=PHASE45_MODEL_CONFIG["patch_size"],
        encoder_dimension=PHASE45_MODEL_CONFIG["encoder_dimension"],
        encoder_depth=PHASE45_MODEL_CONFIG["encoder_depth"],
        encoder_head_count=PHASE45_MODEL_CONFIG["encoder_head_count"],
        decoder_dimension=PHASE45_MODEL_CONFIG["decoder_dimension"],
        decoder_depth=PHASE45_MODEL_CONFIG["decoder_depth"],
        decoder_head_count=PHASE45_MODEL_CONFIG["decoder_head_count"],
        dropout=PHASE45_MODEL_CONFIG["dropout"],
        symmetry_dimension=PHASE45_MODEL_CONFIG["symmetry_dimension"],
    )
    phase47_constructor_route = "direct_exact_phase45_constructor"

phase47_module_records = []
for name, module in phase47_model.named_modules():
    if not name:
        continue
    direct_parameters = list(module.parameters(recurse=False))
    direct_parameter_count = int(sum(
        parameter.numel() for parameter in direct_parameters
    ))
    direct_trainable_count = int(sum(
        parameter.numel()
        for parameter in direct_parameters
        if parameter.requires_grad
    ))
    if direct_parameter_count > 0:
        phase47_module_records.append({
            "name": str(name),
            "type": module.__class__.__name__,
            "direct_parameter_count": direct_parameter_count,
            "direct_trainable_parameter_count": direct_trainable_count,
        })

phase47_checkpoint_directory = Path(
    "/kaggle/working/phase45_private_checkpoint"
)
phase47_checkpoint_paths = [
    phase47_checkpoint_directory / f"phase45_fold_{fold}_encoder.pt"
    for fold in range(3)
]
missing_checkpoints = [
    str(path) for path in phase47_checkpoint_paths if not path.is_file()
]
assert not missing_checkpoints, {
    "message": "Phase45 encoder checkpoints are missing.",
    "missing": missing_checkpoints,
}

phase47_checkpoint_payload = torch.load(
    phase47_checkpoint_paths[0],
    map_location="cpu",
    weights_only=True,
)
phase47_state_route, phase47_encoder_state = phase47_resolve_tensor_state(
    phase47_checkpoint_payload
)

phase47_state_prefix_1 = Counter(
    key.split(".", 1)[0] for key in phase47_encoder_state
)
phase47_state_prefix_2 = Counter(
    ".".join(key.split(".")[:2]) for key in phase47_encoder_state
)

phase47_load_result = phase47_model.load_state_dict(
    phase47_encoder_state,
    strict=False,
)
phase47_missing_keys = list(phase47_load_result.missing_keys)
phase47_unexpected_keys = list(phase47_load_result.unexpected_keys)

phase47_model = phase47_model.to(PHASE45_DEVICE).eval()
phase47_cache = np.load(
    PHASE45_HIGHRES_CACHE_PATH_PRIVATE,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase47_cache.shape == (1362, 80, 80, 80)
phase47_contract_input = torch.from_numpy(
    np.asarray(phase47_cache[0], dtype=np.float32).copy()
).unsqueeze(0).unsqueeze(0).to(PHASE45_DEVICE)

with torch.inference_mode():
    phase47_forward_output = phase47_model(
        phase47_contract_input,
        mask_ratio=0.0,
    )

phase47_forward_tree = phase47_shape_tree(phase47_forward_output)

phase47_candidate_method_outputs = {}
for method_name in (
    "encode_full",
    "forward_encoder",
    "extract_embedding",
    "encode_embedding",
):
    if not hasattr(phase47_model, method_name):
        continue
    method = getattr(phase47_model, method_name)
    signature = inspect.signature(method)
    required_parameters = [
        parameter
        for parameter in signature.parameters.values()
        if parameter.default is inspect.Parameter.empty
        and parameter.kind
        in (
            inspect.Parameter.POSITIONAL_ONLY,
            inspect.Parameter.POSITIONAL_OR_KEYWORD,
        )
    ]
    if len(required_parameters) == 1:
        with torch.inference_mode():
            output = method(phase47_contract_input)
        phase47_candidate_method_outputs[method_name] = {
            "signature": str(signature),
            "output": phase47_shape_tree(output),
        }

phase47_report = {
    "phase": "phase47_phase45_encoder_interface_and_checkpoint_audit",
    "status": "accepted",
    "class": {
        "name": phase47_class.__name__,
        "constructor_signature": phase47_signature(phase47_class),
        "constructor_route": phase47_constructor_route,
        "class_defined_methods": phase47_class_methods,
    },
    "model": {
        "total_parameter_count": int(sum(
            parameter.numel() for parameter in phase47_model.parameters()
        )),
        "trainable_parameter_count_after_constructor": int(sum(
            parameter.numel()
            for parameter in phase47_model.parameters()
            if parameter.requires_grad
        )),
        "parameterized_modules": phase47_module_records,
    },
    "checkpoint": {
        "directory": phase47_checkpoint_directory.name,
        "file_count": len(phase47_checkpoint_paths),
        "file_sizes_mb": [
            float(path.stat().st_size / (1024 ** 2))
            for path in phase47_checkpoint_paths
        ],
        "payload_type": type(phase47_checkpoint_payload).__name__,
        "resolved_state_route": phase47_state_route,
        "tensor_count": len(phase47_encoder_state),
        "first_level_prefix_counts": dict(sorted(
            phase47_state_prefix_1.items()
        )),
        "second_level_prefix_counts": dict(sorted(
            phase47_state_prefix_2.items()
        )),
        "strict_false_missing_key_count": len(phase47_missing_keys),
        "strict_false_missing_key_prefixes": sorted(set(
            key.split(".", 1)[0] for key in phase47_missing_keys
        )),
        "strict_false_unexpected_key_count": len(phase47_unexpected_keys),
        "strict_false_unexpected_key_prefixes": sorted(set(
            key.split(".", 1)[0] for key in phase47_unexpected_keys
        )),
    },
    "execution": {
        "device": str(PHASE45_DEVICE),
        "input_shape": list(phase47_contract_input.shape),
        "forward_mask_ratio": 0.0,
        "forward_output": phase47_forward_tree,
        "single_argument_embedding_methods": phase47_candidate_method_outputs,
    },
    "labels_used": False,
    "outer_validation_labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_embeddings_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase47_audit_started, 3),
}

PHASE47_PHASE45_INTERFACE_PRIVATE = phase47_report

print("BEGIN SANITIZED_PHASE47_ENCODER_AUDIT")
print(json.dumps(phase47_report, indent=2))
print("END SANITIZED_PHASE47_ENCODER_AUDIT")

del phase47_model
if PHASE45_DEVICE.type == "cuda":
    torch.cuda.empty_cache()

AssertionError: {'message': 'Phase47 interface audit is missing Phase45 state.', 'missing': ['Phase45SpectralMaskedAutoencoder3D', 'PHASE45_MODEL_CONFIG', 'PHASE45_DEVICE', 'PHASE45_HIGHRES_CACHE_PATH_PRIVATE']}

In [55]:
# phase47_cell_145B

import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


phase47_contract_started = time.perf_counter()

PHASE47_ADAPTATION_CONFIG = {
    "seed": 470701,
    "reflection_axis": 2,
    "unfrozen_encoder_block_count": 1,
    "head_hidden_dimension": 64,
    "head_dropout": 0.10,
    "residual_cap": 0.50,
    "encoder_learning_rate": 1.0e-5,
    "head_learning_rate": 2.0e-4,
    "weight_decay": 0.02,
    "contract_batch_size": 2,
}

required_names = [
    "Phase45SpectralMaskedAutoencoder3D",
    "phase45_create_pretraining_model",
    "PHASE45_DEVICE",
    "PHASE45_HIGHRES_CACHE_PATH_PRIVATE",
    "PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE",
    "PHASE39_OOF_PRIVATE",
    "PHASE44_Y_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase47 adaptation contract is missing notebook state.",
    "missing": missing_names,
}


def phase47_set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def phase47_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase47_resolve_encoder_state(payload):
    if isinstance(payload, dict):
        if payload and all(torch.is_tensor(value) for value in payload.values()):
            return "root", payload
        for key in (
            "encoder_state_dict",
            "model_state_dict",
            "state_dict",
            "encoder",
            "model",
        ):
            candidate = payload.get(key)
            if (
                isinstance(candidate, dict)
                and candidate
                and all(torch.is_tensor(value) for value in candidate.values())
            ):
                return f"root.{key}", candidate
    raise AssertionError({
        "message": "Could not resolve a tensor state from the Phase45 checkpoint.",
        "payload_type": type(payload).__name__,
        "top_level_keys": (
            sorted(str(key) for key in payload.keys())
            if isinstance(payload, dict)
            else None
        ),
    })


def phase47_parameter_gradient_norm(parameters):
    squared_norm = 0.0
    tensor_count = 0
    finite = True
    for parameter in parameters:
        if parameter.grad is None:
            continue
        gradient = parameter.grad.detach().float()
        tensor_count += 1
        finite = finite and bool(torch.isfinite(gradient).all().item())
        squared_norm += float(torch.sum(gradient * gradient).item())
    return {
        "tensor_count": int(tensor_count),
        "norm": float(squared_norm ** 0.5),
        "all_finite": bool(finite),
    }


class Phase47AnchoredEncoderAdapter(nn.Module):
    """Phase39 identity at initialization, then a bounded learned logit update."""

    def __init__(
        self,
        encoder_checkpoint_path,
        unfrozen_encoder_block_count=1,
        hidden_dimension=64,
        dropout=0.10,
        residual_cap=0.50,
    ):
        super().__init__()
        self.backbone = phase45_create_pretraining_model()

        checkpoint_payload = torch.load(
            Path(encoder_checkpoint_path),
            map_location="cpu",
            weights_only=True,
        )
        self.checkpoint_state_route, encoder_state = (
            phase47_resolve_encoder_state(checkpoint_payload)
        )
        load_result = self.backbone.load_state_dict(
            encoder_state,
            strict=False,
        )
        self.checkpoint_tensor_count = int(len(encoder_state))
        self.missing_checkpoint_keys = tuple(load_result.missing_keys)
        self.unexpected_checkpoint_keys = tuple(load_result.unexpected_keys)

        assert self.checkpoint_tensor_count == 77
        assert not self.unexpected_checkpoint_keys
        assert hasattr(self.backbone, "encoder")
        assert hasattr(self.backbone.encoder, "layers")
        assert hasattr(self.backbone, "encoder_norm")

        for parameter in self.backbone.parameters():
            parameter.requires_grad_(False)

        block_count = len(self.backbone.encoder.layers)
        unfrozen_encoder_block_count = int(unfrozen_encoder_block_count)
        assert 0 <= unfrozen_encoder_block_count <= block_count
        if unfrozen_encoder_block_count > 0:
            for block in self.backbone.encoder.layers[
                block_count - unfrozen_encoder_block_count:
            ]:
                for parameter in block.parameters():
                    parameter.requires_grad_(True)
            for parameter in self.backbone.encoder_norm.parameters():
                parameter.requires_grad_(True)

        encoder_dimension = int(self.backbone.encoder_position.shape[-1])
        self.encoder_dimension = encoder_dimension
        self.representation_dimension = 2 * encoder_dimension
        self.residual_cap = float(residual_cap)
        assert self.residual_cap > 0.0

        self.head = nn.Sequential(
            nn.LayerNorm(self.representation_dimension),
            nn.Linear(self.representation_dimension, int(hidden_dimension)),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(int(hidden_dimension), 1),
        )

        # This is the identity guarantee: before supervised updates, the
        # residual is exactly zero for every image and every anchor logit.
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def encoder_embedding(self, volume):
        encoded = self.backbone.encode_full(volume)
        assert isinstance(encoded, dict) and "embedding" in encoded
        embedding = encoded["embedding"]
        assert embedding.ndim == 2
        assert embedding.shape[1] == self.encoder_dimension
        return embedding

    def invariant_representation(self, volume):
        original = self.encoder_embedding(volume)
        reflected = self.encoder_embedding(
            torch.flip(volume, dims=(PHASE47_ADAPTATION_CONFIG["reflection_axis"],))
        )
        symmetric = 0.5 * (original + reflected)
        absolute_antisymmetric = torch.abs(0.5 * (original - reflected))
        return torch.cat(
            [symmetric, absolute_antisymmetric],
            dim=1,
        )

    def forward(self, volume, anchor_logit):
        representation = self.invariant_representation(volume)
        # Keep the small supervised head and the bounded composition in fp32;
        # encoder execution may still use bfloat16 autocast.
        raw_residual = self.head(representation.float()).squeeze(1)
        bounded_residual = self.residual_cap * torch.tanh(
            raw_residual / self.residual_cap
        )
        updated_logit = anchor_logit.float().reshape(-1) + bounded_residual
        probability = torch.sigmoid(updated_logit)
        return {
            "probability": probability,
            "updated_logit": updated_logit,
            "raw_residual": raw_residual,
            "bounded_residual": bounded_residual,
            "representation": representation,
        }


phase47_set_seed(PHASE47_ADAPTATION_CONFIG["seed"])
phase47_device = torch.device(PHASE45_DEVICE)
phase47_amp_enabled = phase47_device.type == "cuda"
phase47_amp_dtype = (
    torch.bfloat16
    if (
        phase47_amp_enabled
        and torch.cuda.is_bf16_supported()
    )
    else torch.float16
)

phase47_checkpoint_directory = Path(
    "/kaggle/working/phase45_private_checkpoint"
)
phase47_checkpoint_paths = [
    phase47_checkpoint_directory / f"phase45_fold_{fold}_encoder.pt"
    for fold in range(3)
]
missing_checkpoints = [
    str(path) for path in phase47_checkpoint_paths if not path.is_file()
]
assert not missing_checkpoints, {
    "message": "Phase47 is missing Phase45 encoder checkpoints.",
    "missing": missing_checkpoints,
}

# Load all three checkpoints once on CPU. This catches a fold-specific payload
# or key-layout problem before any expensive supervised training begins.
phase47_checkpoint_records = []
for fold, checkpoint_path in enumerate(phase47_checkpoint_paths):
    audit_model = Phase47AnchoredEncoderAdapter(
        encoder_checkpoint_path=checkpoint_path,
        unfrozen_encoder_block_count=(
            PHASE47_ADAPTATION_CONFIG["unfrozen_encoder_block_count"]
        ),
        hidden_dimension=PHASE47_ADAPTATION_CONFIG["head_hidden_dimension"],
        dropout=PHASE47_ADAPTATION_CONFIG["head_dropout"],
        residual_cap=PHASE47_ADAPTATION_CONFIG["residual_cap"],
    )
    phase47_checkpoint_records.append({
        "fold": int(fold),
        "file": checkpoint_path.name,
        "state_route": audit_model.checkpoint_state_route,
        "tensor_count": audit_model.checkpoint_tensor_count,
        "missing_key_count": len(audit_model.missing_checkpoint_keys),
        "unexpected_key_count": len(audit_model.unexpected_checkpoint_keys),
    })
    del audit_model

phase47_partition = PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE[0]
phase47_fit_indices = np.asarray(
    phase47_partition["fit_indices"],
    dtype=np.int64,
)
phase47_contract_indices = phase47_fit_indices[
    :PHASE47_ADAPTATION_CONFIG["contract_batch_size"]
]
assert phase47_contract_indices.size == 2

phase47_cache = np.load(
    PHASE45_HIGHRES_CACHE_PATH_PRIVATE,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase47_cache.shape == (1362, 80, 80, 80)
phase47_contract_volume = torch.from_numpy(
    np.asarray(
        phase47_cache[phase47_contract_indices],
        dtype=np.float32,
    ).copy()
).unsqueeze(1).to(phase47_device)

phase47_anchor_logit_all = phase47_logit(PHASE39_OOF_PRIVATE)
phase47_contract_anchor = torch.from_numpy(
    phase47_anchor_logit_all[phase47_contract_indices].astype(np.float32)
).to(phase47_device)
phase47_contract_target = torch.from_numpy(
    np.asarray(PHASE44_Y_PRIVATE, dtype=np.float32).reshape(-1)[
        phase47_contract_indices
    ].copy()
).to(phase47_device)

if phase47_device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase47_device)

phase47_contract_model = Phase47AnchoredEncoderAdapter(
    encoder_checkpoint_path=phase47_checkpoint_paths[0],
    unfrozen_encoder_block_count=(
        PHASE47_ADAPTATION_CONFIG["unfrozen_encoder_block_count"]
    ),
    hidden_dimension=PHASE47_ADAPTATION_CONFIG["head_hidden_dimension"],
    dropout=PHASE47_ADAPTATION_CONFIG["head_dropout"],
    residual_cap=PHASE47_ADAPTATION_CONFIG["residual_cap"],
).to(phase47_device)

trainable_named_parameters = [
    (name, parameter)
    for name, parameter in phase47_contract_model.named_parameters()
    if parameter.requires_grad
]
frozen_named_parameters = [
    (name, parameter)
    for name, parameter in phase47_contract_model.named_parameters()
    if not parameter.requires_grad
]
phase47_trainable_prefixes = sorted({
    ".".join(name.split(".")[:4])
    if name.startswith("backbone.encoder.layers.")
    else name.split(".", 1)[0] + (
        ".encoder_norm" if name.startswith("backbone.encoder_norm") else ""
    )
    for name, _ in trainable_named_parameters
})
assert all(
    name.startswith("head.")
    or name.startswith("backbone.encoder.layers.5.")
    or name.startswith("backbone.encoder_norm.")
    for name, _ in trainable_named_parameters
), [name for name, _ in trainable_named_parameters]

phase47_contract_model.train()
with torch.autocast(
    device_type=phase47_device.type,
    dtype=phase47_amp_dtype,
    enabled=phase47_amp_enabled,
):
    phase47_initial_output = phase47_contract_model(
        phase47_contract_volume,
        phase47_contract_anchor,
    )

phase47_identity_logit_error = float(torch.max(torch.abs(
    phase47_initial_output["updated_logit"].float()
    - phase47_contract_anchor.float()
)).item())
phase47_identity_probability_error = float(torch.max(torch.abs(
    phase47_initial_output["probability"].float()
    - torch.sigmoid(phase47_contract_anchor.float())
)).item())
phase47_initial_residual_maximum = float(torch.max(torch.abs(
    phase47_initial_output["bounded_residual"].float()
)).item())
assert phase47_identity_logit_error == 0.0
assert phase47_identity_probability_error == 0.0
assert phase47_initial_residual_maximum == 0.0
assert phase47_initial_output["representation"].shape == (2, 384)

phase47_encoder_parameters = [
    parameter
    for name, parameter in trainable_named_parameters
    if name.startswith("backbone.")
]
phase47_head_parameters = [
    parameter
    for name, parameter in trainable_named_parameters
    if name.startswith("head.")
]
assert phase47_encoder_parameters and phase47_head_parameters

phase47_optimizer = torch.optim.AdamW(
    [
        {
            "params": phase47_encoder_parameters,
            "lr": PHASE47_ADAPTATION_CONFIG["encoder_learning_rate"],
        },
        {
            "params": phase47_head_parameters,
            "lr": PHASE47_ADAPTATION_CONFIG["head_learning_rate"],
        },
    ],
    weight_decay=PHASE47_ADAPTATION_CONFIG["weight_decay"],
    betas=(0.9, 0.95),
)

# Step 1: exact-zero final layer means the head's final layer learns first.
phase47_optimizer.zero_grad(set_to_none=True)
phase47_first_loss = F.binary_cross_entropy_with_logits(
    phase47_initial_output["updated_logit"].float(),
    phase47_contract_target.float(),
)
phase47_first_loss.backward()
phase47_first_encoder_gradient = phase47_parameter_gradient_norm(
    phase47_encoder_parameters
)
phase47_first_head_gradient = phase47_parameter_gradient_norm(
    phase47_head_parameters
)
assert phase47_first_head_gradient["all_finite"]
assert phase47_first_head_gradient["norm"] > 0.0
phase47_optimizer.step()

# Step 2: after the zero-initialized output layer has moved, supervised
# gradients must reach the unfrozen encoder block.
phase47_optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=phase47_device.type,
    dtype=phase47_amp_dtype,
    enabled=phase47_amp_enabled,
):
    phase47_second_output = phase47_contract_model(
        phase47_contract_volume,
        phase47_contract_anchor,
    )
phase47_second_loss = F.binary_cross_entropy_with_logits(
    phase47_second_output["updated_logit"].float(),
    phase47_contract_target.float(),
)
phase47_second_loss.backward()
phase47_second_encoder_gradient = phase47_parameter_gradient_norm(
    phase47_encoder_parameters
)
phase47_second_head_gradient = phase47_parameter_gradient_norm(
    phase47_head_parameters
)
assert phase47_second_encoder_gradient["all_finite"]
assert phase47_second_head_gradient["all_finite"]
assert phase47_second_encoder_gradient["norm"] > 0.0
assert phase47_second_head_gradient["norm"] > 0.0

phase47_second_residual_maximum = float(torch.max(torch.abs(
    phase47_second_output["bounded_residual"].float()
)).item())
assert phase47_second_residual_maximum <= (
    PHASE47_ADAPTATION_CONFIG["residual_cap"] + 1.0e-6
)
assert torch.isfinite(phase47_second_output["probability"]).all()

if phase47_device.type == "cuda":
    torch.cuda.synchronize(phase47_device)
    phase47_peak_vram_mb = float(
        torch.cuda.max_memory_allocated(phase47_device) / (1024 ** 2)
    )
else:
    phase47_peak_vram_mb = 0.0

phase47_trainable_parameter_count = int(sum(
    parameter.numel() for _, parameter in trainable_named_parameters
))
phase47_frozen_parameter_count = int(sum(
    parameter.numel() for _, parameter in frozen_named_parameters
))

PHASE47_ADAPTER_CLASS_PRIVATE = Phase47AnchoredEncoderAdapter
PHASE47_ADAPTATION_CONFIG_PRIVATE = dict(PHASE47_ADAPTATION_CONFIG)

phase47_contract_report = {
    "phase": "phase47_anchor_preserving_supervised_encoder_adaptation_contract",
    "status": "accepted",
    "controlled_change": (
        "fine_tune_last_pretrained_encoder_block_with_zero_initialized_"
        "bounded_phase39_residual_head"
    ),
    "checkpoint_contract": {
        "fold_count": len(phase47_checkpoint_records),
        "records": phase47_checkpoint_records,
    },
    "architecture": {
        "encoder_embedding_dimension": int(
            phase47_contract_model.encoder_dimension
        ),
        "reflection_invariant_representation_dimension": int(
            phase47_contract_model.representation_dimension
        ),
        "representation": "symmetric_plus_absolute_antisymmetric",
        "reflection_axis": int(
            PHASE47_ADAPTATION_CONFIG["reflection_axis"]
        ),
        "unfrozen_encoder_block_count": int(
            PHASE47_ADAPTATION_CONFIG["unfrozen_encoder_block_count"]
        ),
        "head_hidden_dimension": int(
            PHASE47_ADAPTATION_CONFIG["head_hidden_dimension"]
        ),
        "residual_cap": float(
            PHASE47_ADAPTATION_CONFIG["residual_cap"]
        ),
    },
    "parameters": {
        "total": int(
            phase47_trainable_parameter_count + phase47_frozen_parameter_count
        ),
        "trainable": phase47_trainable_parameter_count,
        "frozen": phase47_frozen_parameter_count,
        "trainable_prefixes": phase47_trainable_prefixes,
    },
    "initial_identity": {
        "maximum_logit_error": phase47_identity_logit_error,
        "maximum_probability_error": phase47_identity_probability_error,
        "maximum_absolute_residual": phase47_initial_residual_maximum,
        "exact": bool(
            phase47_identity_logit_error == 0.0
            and phase47_identity_probability_error == 0.0
        ),
    },
    "two_step_backward_contract": {
        "first_loss": float(phase47_first_loss.detach().item()),
        "second_loss": float(phase47_second_loss.detach().item()),
        "first_encoder_gradient": phase47_first_encoder_gradient,
        "first_head_gradient": phase47_first_head_gradient,
        "second_encoder_gradient": phase47_second_encoder_gradient,
        "second_head_gradient": phase47_second_head_gradient,
        "second_step_maximum_absolute_residual": (
            phase47_second_residual_maximum
        ),
        "all_finite": True,
    },
    "optimization_contract": {
        "encoder_learning_rate": float(
            PHASE47_ADAPTATION_CONFIG["encoder_learning_rate"]
        ),
        "head_learning_rate": float(
            PHASE47_ADAPTATION_CONFIG["head_learning_rate"]
        ),
        "weight_decay": float(
            PHASE47_ADAPTATION_CONFIG["weight_decay"]
        ),
        "mixed_precision": (
            str(phase47_amp_dtype).replace("torch.", "")
            if phase47_amp_enabled
            else "disabled"
        ),
    },
    "planned_selection": {
        "fit_partition": "fit_indices_only",
        "monitor_partition": "monitor_indices_only",
        "one_shared_specification_across_folds": True,
        "anchor": "exact_cross_fitted_phase39_logit",
        "outer_evaluation_after_freeze": True,
    },
    "peak_vram_mb": phase47_peak_vram_mb,
    "fit_labels_used_for_backward_contract": True,
    "monitor_labels_used": False,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase47_contract_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE47_ADAPTATION_CONTRACT")
print(json.dumps(phase47_contract_report, indent=2))
print("END SANITIZED_PHASE47_ADAPTATION_CONTRACT")

del phase47_optimizer
del phase47_contract_model
del phase47_contract_volume
del phase47_contract_anchor
del phase47_contract_target
if phase47_device.type == "cuda":
    torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE47_ADAPTATION_CONTRACT
{
  "phase": "phase47_anchor_preserving_supervised_encoder_adaptation_contract",
  "status": "accepted",
  "controlled_change": "fine_tune_last_pretrained_encoder_block_with_zero_initialized_bounded_phase39_residual_head",
  "checkpoint_contract": {
    "fold_count": 3,
    "records": [
      {
        "fold": 0,
        "file": "phase45_fold_0_encoder.pt",
        "state_route": "root.encoder_state_dict",
        "tensor_count": 77,
        "missing_key_count": 44,
        "unexpected_key_count": 0
      },
      {
        "fold": 1,
        "file": "phase45_fold_1_encoder.pt",
        "state_route": "root.encoder_state_dict",
        "tensor_count": 77,
        "missing_key_count": 44,
        "unexpected_key_count": 0
      },
      {
        "fold": 2,
        "file": "phase45_fold_2_encoder.pt",
        "state_route": "root.encoder_state_dict",
        "tensor_count": 77,
        "missing_key_count": 44,
        "unexpected_key_count": 0

In [56]:
# phase47_cell_145C

import gc
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import torch


phase47_cache_started = time.perf_counter()

PHASE47_PREFIX_CACHE_CONFIG = {
    "batch_size": 8,
    "worker_count": 0,
    "view_names": ["original", "reflected"],
    "reflection_axis": 2,
    "storage_dtype": "float16",
    "progress_interval": 128,
    "parity_sample_count_per_fold": 6,
    "maximum_direct_tail_error": 5.0e-5,
    "maximum_float16_tail_error": 2.0e-2,
}

required_names = [
    "PHASE47_ADAPTER_CLASS_PRIVATE",
    "PHASE47_ADAPTATION_CONFIG_PRIVATE",
    "PHASE45_DEVICE",
    "PHASE45_HIGHRES_CACHE_PATH_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase47 prefix caching requires the accepted adaptation contract.",
    "missing": missing_names,
}

phase47_device = torch.device(PHASE45_DEVICE)
phase47_amp_enabled = phase47_device.type == "cuda"
phase47_amp_dtype = (
    torch.bfloat16
    if phase47_amp_enabled and torch.cuda.is_bf16_supported()
    else torch.float16
)
phase47_highres_cache = np.load(
    PHASE45_HIGHRES_CACHE_PATH_PRIVATE,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase47_highres_cache.shape == (1362, 80, 80, 80)
assert phase47_highres_cache.dtype == np.float16

phase47_checkpoint_directory = Path(
    "/kaggle/working/phase45_private_checkpoint"
)
phase47_checkpoint_paths = [
    phase47_checkpoint_directory / f"phase45_fold_{fold}_encoder.pt"
    for fold in range(3)
]
assert all(path.is_file() for path in phase47_checkpoint_paths)

phase47_prefix_directory = Path(
    "/kaggle/working/phase47_private_prefix_cache"
)
phase47_prefix_directory.mkdir(parents=True, exist_ok=True)
phase47_prefix_paths = [
    phase47_prefix_directory
    / f"phase47_fold_{fold}_penultimate_tokens_float16.npy"
    for fold in range(3)
]
phase47_metadata_path = (
    phase47_prefix_directory / "phase47_prefix_cache_metadata.json"
)

phase47_expected_shape = (2, 1362, 1000, 192)
phase47_expected_bytes_per_file = int(
    np.prod(phase47_expected_shape, dtype=np.int64)
    * np.dtype(np.float16).itemsize
)
phase47_required_new_bytes = int(sum(
    0 if path.is_file() else phase47_expected_bytes_per_file
    for path in phase47_prefix_paths
))
phase47_free_disk_bytes = int(
    shutil.disk_usage(phase47_prefix_directory).free
)
assert phase47_free_disk_bytes >= (
    phase47_required_new_bytes + 512 * 1024 ** 2
), {
    "message": "Insufficient working-disk space for the Phase47 prefix cache.",
    "required_new_gb": phase47_required_new_bytes / (1024 ** 3),
    "free_gb": phase47_free_disk_bytes / (1024 ** 3),
}


def phase47_volume_batch(indices):
    array = np.asarray(
        phase47_highres_cache[np.asarray(indices, dtype=np.int64)],
        dtype=np.float32,
    ).copy()
    tensor = torch.from_numpy(array).unsqueeze(1)
    assert tensor.ndim == 5 and tensor.shape[1:] == (1, 80, 80, 80)
    return tensor.to(phase47_device, non_blocking=False)


def phase47_capture_penultimate(backbone, volume):
    captured = []

    def capture_hook(module, arguments):
        del module
        assert isinstance(arguments, tuple) and len(arguments) >= 1
        captured.append(arguments[0].detach())

    handle = backbone.encoder.layers[-1].register_forward_pre_hook(
        capture_hook
    )
    try:
        encoded = backbone.encode_full(volume)
    finally:
        handle.remove()
    assert len(captured) == 1
    tokens = captured[0]
    assert tokens.ndim == 3
    assert tokens.shape[1:] == (1000, 192)
    assert isinstance(encoded, dict) and "embedding" in encoded
    return tokens, encoded["embedding"]


def phase47_tail_embedding(backbone, penultimate_tokens):
    tokens = backbone.encoder.layers[-1](penultimate_tokens)
    tokens = backbone.encoder_norm(tokens)
    embedding = torch.mean(tokens, dim=1)
    assert embedding.ndim == 2 and embedding.shape[1] == 192
    return embedding


def phase47_existing_cache_valid(path):
    if not path.is_file():
        return False
    try:
        matrix = np.load(path, mmap_mode="r", allow_pickle=False)
        valid = (
            matrix.shape == phase47_expected_shape
            and matrix.dtype == np.float16
        )
        if valid:
            probe = np.asarray(
                matrix[:, [0, 681, 1361], :2, :4],
                dtype=np.float32,
            )
            valid = bool(np.all(np.isfinite(probe)))
        del matrix
        return bool(valid)
    except (OSError, ValueError):
        return False


phase47_cache_records = []
phase47_direct_tail_errors = []
phase47_float16_tail_errors = []

if phase47_device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase47_device)

for fold, (checkpoint_path, cache_path) in enumerate(zip(
    phase47_checkpoint_paths,
    phase47_prefix_paths,
)):
    fold_started = time.perf_counter()
    cache_reused = phase47_existing_cache_valid(cache_path)

    adapter = PHASE47_ADAPTER_CLASS_PRIVATE(
        encoder_checkpoint_path=checkpoint_path,
        unfrozen_encoder_block_count=(
            PHASE47_ADAPTATION_CONFIG_PRIVATE[
                "unfrozen_encoder_block_count"
            ]
        ),
        hidden_dimension=PHASE47_ADAPTATION_CONFIG_PRIVATE[
            "head_hidden_dimension"
        ],
        dropout=PHASE47_ADAPTATION_CONFIG_PRIVATE["head_dropout"],
        residual_cap=PHASE47_ADAPTATION_CONFIG_PRIVATE["residual_cap"],
    ).to(phase47_device).eval()
    backbone = adapter.backbone

    # Validate the exact decomposition before trusting a large cache. The
    # direct path must match encode_full; the float16 path measures only the
    # intentional cache-boundary error.
    parity_indices = np.linspace(
        fold,
        1361 - fold,
        PHASE47_PREFIX_CACHE_CONFIG["parity_sample_count_per_fold"],
        dtype=np.int64,
    )
    parity_volume = phase47_volume_batch(parity_indices)
    with torch.inference_mode(), torch.autocast(
        device_type=phase47_device.type,
        dtype=phase47_amp_dtype,
        enabled=phase47_amp_enabled,
    ):
        direct_tokens, direct_embedding = phase47_capture_penultimate(
            backbone,
            parity_volume,
        )
        reconstructed_embedding = phase47_tail_embedding(
            backbone,
            direct_tokens,
        )
        quantized_tokens = direct_tokens.to(torch.float16).to(
            direct_tokens.dtype
        )
        quantized_embedding = phase47_tail_embedding(
            backbone,
            quantized_tokens,
        )

    direct_tail_error = float(torch.max(torch.abs(
        reconstructed_embedding.float() - direct_embedding.float()
    )).item())
    float16_tail_error = float(torch.max(torch.abs(
        quantized_embedding.float() - direct_embedding.float()
    )).item())
    assert direct_tail_error <= PHASE47_PREFIX_CACHE_CONFIG[
        "maximum_direct_tail_error"
    ], {
        "message": "Penultimate-token tail reconstruction is not exact.",
        "fold": fold,
        "maximum_error": direct_tail_error,
    }
    assert float16_tail_error <= PHASE47_PREFIX_CACHE_CONFIG[
        "maximum_float16_tail_error"
    ], {
        "message": "Float16 prefix caching introduces excessive error.",
        "fold": fold,
        "maximum_error": float16_tail_error,
    }
    phase47_direct_tail_errors.append(direct_tail_error)
    phase47_float16_tail_errors.append(float16_tail_error)
    del parity_volume
    del direct_tokens
    del direct_embedding
    del reconstructed_embedding
    del quantized_tokens
    del quantized_embedding

    if not cache_reused:
        building_path = cache_path.with_suffix(".building.npy")
        if building_path.exists():
            building_path.unlink()
        output = np.lib.format.open_memmap(
            building_path,
            mode="w+",
            dtype=np.float16,
            shape=phase47_expected_shape,
        )

        completed = 0
        batch_size = int(PHASE47_PREFIX_CACHE_CONFIG["batch_size"])
        for start in range(0, 1362, batch_size):
            stop = min(start + batch_size, 1362)
            indices = np.arange(start, stop, dtype=np.int64)
            volume = phase47_volume_batch(indices)

            with torch.inference_mode(), torch.autocast(
                device_type=phase47_device.type,
                dtype=phase47_amp_dtype,
                enabled=phase47_amp_enabled,
            ):
                original_tokens, _ = phase47_capture_penultimate(
                    backbone,
                    volume,
                )
                reflected_tokens, _ = phase47_capture_penultimate(
                    backbone,
                    torch.flip(
                        volume,
                        dims=(
                            PHASE47_PREFIX_CACHE_CONFIG["reflection_axis"],
                        ),
                    ),
                )

            output[0, start:stop] = (
                original_tokens.detach().float().cpu().numpy().astype(
                    np.float16,
                    copy=False,
                )
            )
            output[1, start:stop] = (
                reflected_tokens.detach().float().cpu().numpy().astype(
                    np.float16,
                    copy=False,
                )
            )

            del volume
            del original_tokens
            del reflected_tokens
            completed = stop
            if (
                completed == 1362
                or completed
                // PHASE47_PREFIX_CACHE_CONFIG["progress_interval"]
                > (start)
                // PHASE47_PREFIX_CACHE_CONFIG["progress_interval"]
            ):
                print(
                    f"Phase47 prefix cache fold {fold}: "
                    f"{completed}/1362"
                )

        output.flush()
        del output
        os.replace(building_path, cache_path)
        cache_reused = False

    cached = np.load(cache_path, mmap_mode="r", allow_pickle=False)
    assert cached.shape == phase47_expected_shape
    assert cached.dtype == np.float16
    sampled = np.asarray(
        cached[:, parity_indices, ::127, ::31],
        dtype=np.float32,
    )
    assert np.all(np.isfinite(sampled))
    phase47_cache_records.append({
        "fold": int(fold),
        "file": cache_path.name,
        "shape": list(cached.shape),
        "dtype": str(cached.dtype),
        "size_gb": float(cache_path.stat().st_size / (1024 ** 3)),
        "cache_reused": bool(cache_reused),
        "direct_tail_maximum_error": direct_tail_error,
        "float16_tail_maximum_error": float16_tail_error,
        "sampled_minimum": float(np.min(sampled)),
        "sampled_maximum": float(np.max(sampled)),
        "sampled_all_finite": bool(np.all(np.isfinite(sampled))),
        "elapsed_seconds": round(time.perf_counter() - fold_started, 3),
    })
    del sampled
    del cached
    del backbone
    del adapter
    gc.collect()
    if phase47_device.type == "cuda":
        torch.cuda.empty_cache()

phase47_metadata = {
    "phase": "phase47_fold_specific_penultimate_token_cache",
    "status": "complete",
    "shape_per_fold": list(phase47_expected_shape),
    "dtype": "float16",
    "view_order": PHASE47_PREFIX_CACHE_CONFIG["view_names"],
    "reflection_axis": int(
        PHASE47_PREFIX_CACHE_CONFIG["reflection_axis"]
    ),
    "checkpoint_files": [path.name for path in phase47_checkpoint_paths],
    "cache_files": [path.name for path in phase47_prefix_paths],
    "encoder_tail": "encoder_layer_5_then_encoder_norm_then_token_mean",
    "complete": True,
}
metadata_building_path = phase47_metadata_path.with_suffix(".building.json")
with open(metadata_building_path, "w", encoding="utf-8") as handle:
    json.dump(phase47_metadata, handle, indent=2, sort_keys=True)
os.replace(metadata_building_path, phase47_metadata_path)

PHASE47_PREFIX_CACHE_PATHS_PRIVATE = tuple(phase47_prefix_paths)
PHASE47_PREFIX_CACHE_METADATA_PRIVATE = dict(phase47_metadata)

if phase47_device.type == "cuda":
    phase47_peak_vram_mb = float(
        torch.cuda.max_memory_allocated(phase47_device) / (1024 ** 2)
    )
else:
    phase47_peak_vram_mb = 0.0

phase47_cache_report = {
    "phase": "phase47_fold_specific_penultimate_token_cache",
    "status": "accepted",
    "fold_count": 3,
    "case_count": 1362,
    "view_count": 2,
    "view_order": PHASE47_PREFIX_CACHE_CONFIG["view_names"],
    "shape_per_fold": list(phase47_expected_shape),
    "storage_dtype": "float16",
    "records": phase47_cache_records,
    "parity": {
        "maximum_direct_tail_error": float(max(
            phase47_direct_tail_errors
        )),
        "maximum_float16_tail_error": float(max(
            phase47_float16_tail_errors
        )),
        "direct_tail_threshold": float(
            PHASE47_PREFIX_CACHE_CONFIG["maximum_direct_tail_error"]
        ),
        "float16_tail_threshold": float(
            PHASE47_PREFIX_CACHE_CONFIG["maximum_float16_tail_error"]
        ),
    },
    "storage": {
        "total_size_gb": float(sum(
            path.stat().st_size for path in phase47_prefix_paths
        ) / (1024 ** 3)),
        "free_disk_gb_after": float(
            shutil.disk_usage(phase47_prefix_directory).free / (1024 ** 3)
        ),
        "persistent_notebook_cache_only": True,
        "include_in_submission": False,
    },
    "execution": {
        "device": str(phase47_device),
        "mixed_precision": (
            str(phase47_amp_dtype).replace("torch.", "")
            if phase47_amp_enabled
            else "disabled"
        ),
        "batch_size": int(PHASE47_PREFIX_CACHE_CONFIG["batch_size"]),
        "peak_vram_mb": phase47_peak_vram_mb,
    },
    "labels_used": False,
    "outer_validation_images_used_for_training": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_embeddings_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase47_cache_started, 3),
}

print("BEGIN SANITIZED_PHASE47_PREFIX_CACHE")
print(json.dumps(phase47_cache_report, indent=2))
print("END SANITIZED_PHASE47_PREFIX_CACHE")

Phase47 prefix cache fold 0: 128/1362
Phase47 prefix cache fold 0: 256/1362
Phase47 prefix cache fold 0: 384/1362
Phase47 prefix cache fold 0: 512/1362
Phase47 prefix cache fold 0: 640/1362
Phase47 prefix cache fold 0: 768/1362
Phase47 prefix cache fold 0: 896/1362
Phase47 prefix cache fold 0: 1024/1362
Phase47 prefix cache fold 0: 1152/1362
Phase47 prefix cache fold 0: 1280/1362
Phase47 prefix cache fold 0: 1362/1362
Phase47 prefix cache fold 1: 128/1362
Phase47 prefix cache fold 1: 256/1362
Phase47 prefix cache fold 1: 384/1362
Phase47 prefix cache fold 1: 512/1362
Phase47 prefix cache fold 1: 640/1362
Phase47 prefix cache fold 1: 768/1362
Phase47 prefix cache fold 1: 896/1362
Phase47 prefix cache fold 1: 1024/1362
Phase47 prefix cache fold 1: 1152/1362
Phase47 prefix cache fold 1: 1280/1362
Phase47 prefix cache fold 1: 1362/1362
Phase47 prefix cache fold 2: 128/1362
Phase47 prefix cache fold 2: 256/1362
Phase47 prefix cache fold 2: 384/1362
Phase47 prefix cache fold 2: 512/1362
Phas

In [57]:
# phase47_cell_145D

import copy
import gc
import itertools
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset


phase47_selection_started = time.perf_counter()

PHASE47_SELECTION_CONFIG = {
    "batch_size": 24,
    "evaluation_batch_size": 32,
    "worker_count": 0,
    "maximum_epochs": 10,
    "minimum_epochs": 4,
    "early_stopping_patience": 3,
    "early_stopping_minimum_gain": 1.0e-4,
    "gradient_clip": 1.0,
    "weight_decay": 0.02,
    "warmup_fraction": 0.10,
    "minimum_learning_rate_fraction": 0.10,
    "maximum_training_case_weight": 4.0,
    "model_log_loss_band": 0.002,
    "gate_log_loss_band": 0.001,
    "maximum_model_fold_regret": 0.006,
    "maximum_model_group_harm": 0.015,
    "minimum_gate_fold_wins": 2,
    "maximum_gate_fold_regret": 0.003,
    "maximum_gate_group_harm": 0.010,
    "minimum_advance_log_loss_gain": 0.001,
    "minimum_advance_auroc_gain": 0.001,
    "maximum_advance_auroc_deficit": 0.0002,
    "maximum_advance_log_loss_excess": 0.0005,
}

required_names = [
    "PHASE47_ADAPTER_CLASS_PRIVATE",
    "PHASE47_PREFIX_CACHE_PATHS_PRIVATE",
    "PHASE47_PREFIX_CACHE_METADATA_PRIVATE",
    "PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE",
    "PHASE45_DEVICE",
    "PHASE39_OOF_PRIVATE",
    "PHASE44_Y_PRIVATE",
    "PHASE44_GROUPS_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase47 selection is missing accepted prior state.",
    "missing": missing_names,
}


def phase47_selection_set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def phase47_selection_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase47_selection_sigmoid(logit):
    logit = np.asarray(logit, dtype=np.float64)
    output = np.empty_like(logit)
    nonnegative = logit >= 0.0
    output[nonnegative] = 1.0 / (1.0 + np.exp(-logit[nonnegative]))
    exponential = np.exp(logit[~nonnegative])
    output[~nonnegative] = exponential / (1.0 + exponential)
    return np.clip(output, 1.0e-7, 1.0 - 1.0e-7)


def phase47_selection_log_loss(y_true, probability, sample_weight=None):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    probability = np.clip(
        np.asarray(probability, dtype=np.float64).reshape(-1),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    losses = -(
        y_true * np.log(probability)
        + (1.0 - y_true) * np.log1p(-probability)
    )
    if sample_weight is None:
        return float(np.mean(losses))
    sample_weight = np.asarray(sample_weight, dtype=np.float64).reshape(-1)
    assert sample_weight.shape == losses.shape
    return float(np.sum(sample_weight * losses) / np.sum(sample_weight))


def phase47_selection_auroc(y_true, probability, sample_weight=None):
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    probability = np.asarray(probability, dtype=np.float64).reshape(-1)
    if np.unique(y_true).size < 2:
        return None
    return float(roc_auc_score(
        y_true,
        probability,
        sample_weight=sample_weight,
    ))


phase47_y = np.asarray(PHASE44_Y_PRIVATE, dtype=np.float64).reshape(-1)
phase47_groups = np.asarray(
    PHASE44_GROUPS_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase47_anchor_probability = np.clip(
    np.asarray(PHASE39_OOF_PRIVATE, dtype=np.float64).reshape(-1),
    1.0e-7,
    1.0 - 1.0e-7,
)
phase47_anchor_logit = phase47_selection_logit(
    phase47_anchor_probability
)
assert phase47_y.shape == (1362,)
assert phase47_groups.shape == (1362,)
assert phase47_anchor_probability.shape == (1362,)
assert set(np.unique(phase47_y)).issubset({0.0, 1.0})

phase47_device = torch.device(PHASE45_DEVICE)
phase47_amp_enabled = phase47_device.type == "cuda"
phase47_amp_dtype = (
    torch.bfloat16
    if phase47_amp_enabled and torch.cuda.is_bf16_supported()
    else torch.float16
)
if phase47_device.type == "cuda":
    assert phase47_amp_dtype == torch.bfloat16, {
        "message": "Phase47 selection expects the accepted bfloat16 contract.",
        "resolved_dtype": str(phase47_amp_dtype),
    }

phase47_prefix_paths = tuple(
    Path(path) for path in PHASE47_PREFIX_CACHE_PATHS_PRIVATE
)
assert len(phase47_prefix_paths) == 3
for path in phase47_prefix_paths:
    matrix = np.load(path, mmap_mode="r", allow_pickle=False)
    assert matrix.shape == (2, 1362, 1000, 192)
    assert matrix.dtype == np.float16
    del matrix


class Phase47PrefixDataset(Dataset):
    def __init__(
        self,
        cache_path,
        indices,
        anchor_logit,
        labels,
        groups,
        case_weights=None,
    ):
        self.cache_path = str(cache_path)
        self.indices = np.asarray(indices, dtype=np.int64).reshape(-1)
        self.anchor_logit = np.asarray(
            anchor_logit,
            dtype=np.float32,
        ).reshape(-1)
        self.labels = np.asarray(labels, dtype=np.float32).reshape(-1)
        self.groups = np.asarray(groups, dtype=np.int64).reshape(-1)
        if case_weights is None:
            self.case_weights = np.ones(self.indices.size, dtype=np.float32)
        else:
            self.case_weights = np.asarray(
                case_weights,
                dtype=np.float32,
            ).reshape(-1)
        assert self.indices.size == self.anchor_logit.size
        assert self.indices.size == self.labels.size
        assert self.indices.size == self.groups.size
        assert self.indices.size == self.case_weights.size
        self._matrix = None

    def __len__(self):
        return int(self.indices.size)

    def _get_matrix(self):
        if self._matrix is None:
            self._matrix = np.load(
                self.cache_path,
                mmap_mode="r",
                allow_pickle=False,
            )
        return self._matrix

    def __getitem__(self, position):
        case_index = int(self.indices[position])
        # A private copy avoids non-writable memmap tensors and makes DataLoader
        # collation behavior independent of NumPy's view semantics.
        prefix = np.array(
            self._get_matrix()[:, case_index],
            dtype=np.float16,
            copy=True,
        )
        return {
            "prefix": torch.from_numpy(prefix),
            "anchor_logit": torch.tensor(
                self.anchor_logit[position],
                dtype=torch.float32,
            ),
            "label": torch.tensor(
                self.labels[position],
                dtype=torch.float32,
            ),
            "group": torch.tensor(
                self.groups[position],
                dtype=torch.int64,
            ),
            "case_weight": torch.tensor(
                self.case_weights[position],
                dtype=torch.float32,
            ),
            "case_index": torch.tensor(case_index, dtype=torch.int64),
        }


class Phase47CachedTailAdapter(nn.Module):
    def __init__(
        self,
        checkpoint_path,
        hidden_dimension,
        dropout,
        residual_cap,
        train_tail,
    ):
        super().__init__()
        source = PHASE47_ADAPTER_CLASS_PRIVATE(
            encoder_checkpoint_path=checkpoint_path,
            unfrozen_encoder_block_count=1,
            hidden_dimension=int(hidden_dimension),
            dropout=float(dropout),
            residual_cap=float(residual_cap),
        )
        self.tail_block = copy.deepcopy(source.backbone.encoder.layers[-1])
        self.encoder_norm = copy.deepcopy(source.backbone.encoder_norm)
        self.head = copy.deepcopy(source.head)
        self.residual_cap = float(residual_cap)
        self.train_tail = bool(train_tail)
        del source

        for parameter in self.tail_block.parameters():
            parameter.requires_grad_(self.train_tail)
        for parameter in self.encoder_norm.parameters():
            parameter.requires_grad_(self.train_tail)
        for parameter in self.head.parameters():
            parameter.requires_grad_(True)

    def set_training_mode(self):
        self.train()
        if not self.train_tail:
            self.tail_block.eval()
            self.encoder_norm.eval()

    def tail_embedding(self, tokens):
        if self.train_tail:
            output = self.tail_block(tokens)
            output = self.encoder_norm(output)
        else:
            with torch.no_grad():
                output = self.tail_block(tokens)
                output = self.encoder_norm(output)
        return torch.mean(output, dim=1)

    def forward(self, prefix, anchor_logit):
        assert prefix.ndim == 4 and prefix.shape[1:] == (2, 1000, 192)
        original = self.tail_embedding(prefix[:, 0])
        reflected = self.tail_embedding(prefix[:, 1])
        symmetric = 0.5 * (original + reflected)
        absolute_antisymmetric = torch.abs(0.5 * (original - reflected))
        representation = torch.cat(
            [symmetric, absolute_antisymmetric],
            dim=1,
        )
        raw_residual = self.head(representation.float()).squeeze(1)
        bounded_residual = self.residual_cap * torch.tanh(
            raw_residual / self.residual_cap
        )
        updated_logit = anchor_logit.float().reshape(-1) + bounded_residual
        return {
            "raw_residual": raw_residual,
            "bounded_residual": bounded_residual,
            "updated_logit": updated_logit,
            "probability": torch.sigmoid(updated_logit),
        }


def phase47_training_weights(groups):
    groups = np.asarray(groups, dtype=np.int64).reshape(-1)
    unique_groups, counts = np.unique(groups, return_counts=True)
    count_by_group = {
        int(group): int(count)
        for group, count in zip(unique_groups, counts)
    }
    maximum_count = float(np.max(counts))
    weights = np.asarray([
        math.sqrt(maximum_count / count_by_group[int(group)])
        for group in groups
    ], dtype=np.float64)
    weights = np.minimum(
        weights,
        PHASE47_SELECTION_CONFIG["maximum_training_case_weight"],
    )
    weights /= np.mean(weights)
    return weights.astype(np.float32)


def phase47_within_group_rank_loss(logits, labels, groups):
    group_losses = []
    for group in torch.unique(groups):
        mask = groups == group
        positive = logits[mask & (labels > 0.5)]
        negative = logits[mask & (labels <= 0.5)]
        if positive.numel() == 0 or negative.numel() == 0:
            continue
        differences = positive[:, None] - negative[None, :]
        group_losses.append(F.softplus(-differences).mean())
    if not group_losses:
        return logits.sum() * 0.0
    return torch.stack(group_losses).mean()


def phase47_learning_rate_factor(step, total_steps, warmup_steps):
    step = int(step)
    total_steps = max(int(total_steps), 1)
    warmup_steps = max(int(warmup_steps), 1)
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(
        max(total_steps - warmup_steps, 1)
    )
    progress = min(max(progress, 0.0), 1.0)
    minimum = PHASE47_SELECTION_CONFIG[
        "minimum_learning_rate_fraction"
    ]
    return float(
        minimum
        + (1.0 - minimum) * 0.5 * (1.0 + math.cos(math.pi * progress))
    )


@torch.no_grad()
def phase47_predict_cached(model, loader, expected_indices):
    model.eval()
    raw_parts = []
    bounded_parts = []
    probability_parts = []
    index_parts = []
    for batch in loader:
        prefix = batch["prefix"].to(
            phase47_device,
            non_blocking=False,
        ).float()
        anchor = batch["anchor_logit"].to(
            phase47_device,
            non_blocking=False,
        )
        with torch.autocast(
            device_type=phase47_device.type,
            dtype=phase47_amp_dtype,
            enabled=phase47_amp_enabled,
        ):
            output = model(prefix, anchor)
        raw_parts.append(
            output["raw_residual"].float().cpu().numpy()
        )
        bounded_parts.append(
            output["bounded_residual"].float().cpu().numpy()
        )
        probability_parts.append(
            output["probability"].float().cpu().numpy()
        )
        index_parts.append(batch["case_index"].numpy())

    indices = np.concatenate(index_parts).astype(np.int64, copy=False)
    expected_indices = np.asarray(expected_indices, dtype=np.int64)
    assert np.array_equal(indices, expected_indices)
    raw = np.concatenate(raw_parts).astype(np.float64, copy=False)
    bounded = np.concatenate(bounded_parts).astype(np.float64, copy=False)
    probability = np.concatenate(probability_parts).astype(
        np.float64,
        copy=False,
    )
    assert raw.shape == expected_indices.shape
    assert bounded.shape == expected_indices.shape
    assert probability.shape == expected_indices.shape
    assert np.all(np.isfinite(raw))
    assert np.all(np.isfinite(bounded))
    assert np.all(np.isfinite(probability))
    return {
        "raw_residual": raw,
        "bounded_residual": bounded,
        "probability": np.clip(probability, 1.0e-7, 1.0 - 1.0e-7),
    }


phase47_model_grid = [
    {
        "candidate_index": 0,
        "train_tail": False,
        "hidden_dimension": 32,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": 0.0,
        "head_learning_rate": 3.0e-4,
        "rank_loss_weight": 0.0,
    },
    {
        "candidate_index": 1,
        "train_tail": False,
        "hidden_dimension": 64,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": 0.0,
        "head_learning_rate": 3.0e-4,
        "rank_loss_weight": 0.05,
    },
    {
        "candidate_index": 2,
        "train_tail": True,
        "hidden_dimension": 32,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": 3.0e-6,
        "head_learning_rate": 2.0e-4,
        "rank_loss_weight": 0.0,
    },
    {
        "candidate_index": 3,
        "train_tail": True,
        "hidden_dimension": 64,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": 3.0e-6,
        "head_learning_rate": 2.0e-4,
        "rank_loss_weight": 0.05,
    },
    {
        "candidate_index": 4,
        "train_tail": True,
        "hidden_dimension": 64,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": 1.0e-5,
        "head_learning_rate": 1.0e-4,
        "rank_loss_weight": 0.0,
    },
    {
        "candidate_index": 5,
        "train_tail": True,
        "hidden_dimension": 64,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": 1.0e-5,
        "head_learning_rate": 1.0e-4,
        "rank_loss_weight": 0.05,
    },
]
assert len(phase47_model_grid) == 6

phase47_gate_grid = [{
    "gate_index": 0,
    "scale": 0.0,
    "residual_cap": 0.0,
    "uncertainty_exponent": 0.0,
}]
for scale, cap, exponent in itertools.product(
    [0.10, 0.25, 0.50, 0.75, 1.0],
    [0.10, 0.25, 0.50, 1.0],
    [0.0, 0.5, 1.0],
):
    phase47_gate_grid.append({
        "gate_index": int(len(phase47_gate_grid)),
        "scale": float(scale),
        "residual_cap": float(cap),
        "uncertainty_exponent": float(exponent),
    })
assert len(phase47_gate_grid) == 61

phase47_partitions = []
for fold, partition in enumerate(PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE):
    record = {
        "fold": int(fold),
        "fit_indices": np.asarray(
            partition["fit_indices"],
            dtype=np.int64,
        ),
        "monitor_indices": np.asarray(
            partition["monitor_indices"],
            dtype=np.int64,
        ),
        "outer_train_indices": np.asarray(
            partition["outer_train_indices"],
            dtype=np.int64,
        ),
        "outer_valid_indices": np.asarray(
            partition["outer_valid_indices"],
            dtype=np.int64,
        ),
    }
    assert np.intersect1d(
        record["fit_indices"],
        record["monitor_indices"],
    ).size == 0
    assert np.array_equal(
        np.sort(np.concatenate([
            record["fit_indices"],
            record["monitor_indices"],
        ])),
        np.sort(record["outer_train_indices"]),
    )
    assert np.intersect1d(
        record["outer_train_indices"],
        record["outer_valid_indices"],
    ).size == 0
    phase47_partitions.append(record)

phase47_monitor_indices = np.concatenate([
    partition["monitor_indices"] for partition in phase47_partitions
])
phase47_monitor_folds = np.concatenate([
    np.full(
        partition["monitor_indices"].size,
        partition["fold"],
        dtype=np.int64,
    )
    for partition in phase47_partitions
])
phase47_monitor_y = phase47_y[phase47_monitor_indices]
phase47_monitor_groups = phase47_groups[phase47_monitor_indices]
phase47_monitor_anchor_probability = phase47_anchor_probability[
    phase47_monitor_indices
]
phase47_monitor_anchor_logit = phase47_anchor_logit[
    phase47_monitor_indices
]
phase47_monitor_multiplicity = np.bincount(
    phase47_monitor_indices,
    minlength=1362,
)
phase47_monitor_weights = (
    1.0 / phase47_monitor_multiplicity[phase47_monitor_indices]
)
assert phase47_monitor_indices.size == 493
assert np.unique(phase47_monitor_indices).size == 407
assert phase47_monitor_multiplicity.max() == 2

phase47_monitor_slices = []
offset = 0
for partition in phase47_partitions:
    stop = offset + partition["monitor_indices"].size
    phase47_monitor_slices.append(slice(offset, stop))
    offset = stop
assert offset == 493


def phase47_complete_monitor_metrics(probability):
    probability = np.asarray(probability, dtype=np.float64).reshape(-1)
    assert probability.shape == (493,)
    assert np.all(np.isfinite(probability))
    fold_metrics = []
    for fold in range(3):
        mask = phase47_monitor_folds == fold
        fold_metrics.append({
            "fold": int(fold),
            "n": int(mask.sum()),
            "log_loss": phase47_selection_log_loss(
                phase47_monitor_y[mask],
                probability[mask],
            ),
            "auroc": phase47_selection_auroc(
                phase47_monitor_y[mask],
                probability[mask],
            ),
        })
    group_metrics = []
    for group in np.unique(phase47_monitor_groups):
        mask = phase47_monitor_groups == group
        group_metrics.append({
            "group": int(group),
            "unique_n": int(np.unique(
                phase47_monitor_indices[mask]
            ).size),
            "log_loss": phase47_selection_log_loss(
                phase47_monitor_y[mask],
                probability[mask],
                phase47_monitor_weights[mask],
            ),
            "auroc": phase47_selection_auroc(
                phase47_monitor_y[mask],
                probability[mask],
                phase47_monitor_weights[mask],
            ),
        })
    return {
        "log_loss": phase47_selection_log_loss(
            phase47_monitor_y,
            probability,
            phase47_monitor_weights,
        ),
        "auroc": phase47_selection_auroc(
            phase47_monitor_y,
            probability,
            phase47_monitor_weights,
        ),
        "fold_metrics": fold_metrics,
        "group_metrics": group_metrics,
    }


phase47_anchor_metrics = phase47_complete_monitor_metrics(
    phase47_monitor_anchor_probability
)
phase47_anchor_fold_loss = {
    record["fold"]: record["log_loss"]
    for record in phase47_anchor_metrics["fold_metrics"]
}
phase47_anchor_group_loss = {
    record["group"]: record["log_loss"]
    for record in phase47_anchor_metrics["group_metrics"]
}

candidate_count = len(phase47_model_grid)
phase47_candidate_monitor_raw = np.zeros(
    (candidate_count, 493),
    dtype=np.float64,
)
phase47_candidate_monitor_probability = np.tile(
    phase47_monitor_anchor_probability[None, :],
    (candidate_count, 1),
)
phase47_candidate_epoch_counts = np.zeros(
    (candidate_count, 3),
    dtype=np.int64,
)
phase47_training_records = []

if phase47_device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase47_device)

for specification in phase47_model_grid:
    candidate_index = int(specification["candidate_index"])
    for partition, monitor_slice in zip(
        phase47_partitions,
        phase47_monitor_slices,
    ):
        fold = int(partition["fold"])
        fit_indices = partition["fit_indices"]
        monitor_indices = partition["monitor_indices"]
        run_seed = 470900 + 100 * candidate_index + fold
        phase47_selection_set_seed(run_seed)
        run_started = time.perf_counter()

        fit_case_weights = phase47_training_weights(
            phase47_groups[fit_indices]
        )
        fit_dataset = Phase47PrefixDataset(
            cache_path=phase47_prefix_paths[fold],
            indices=fit_indices,
            anchor_logit=phase47_anchor_logit[fit_indices],
            labels=phase47_y[fit_indices],
            groups=phase47_groups[fit_indices],
            case_weights=fit_case_weights,
        )
        monitor_dataset = Phase47PrefixDataset(
            cache_path=phase47_prefix_paths[fold],
            indices=monitor_indices,
            anchor_logit=phase47_anchor_logit[monitor_indices],
            labels=phase47_y[monitor_indices],
            groups=phase47_groups[monitor_indices],
        )
        generator = torch.Generator()
        generator.manual_seed(run_seed)
        fit_loader = DataLoader(
            fit_dataset,
            batch_size=PHASE47_SELECTION_CONFIG["batch_size"],
            shuffle=True,
            num_workers=PHASE47_SELECTION_CONFIG["worker_count"],
            pin_memory=(phase47_device.type == "cuda"),
            drop_last=False,
            generator=generator,
        )
        monitor_loader = DataLoader(
            monitor_dataset,
            batch_size=PHASE47_SELECTION_CONFIG[
                "evaluation_batch_size"
            ],
            shuffle=False,
            num_workers=PHASE47_SELECTION_CONFIG["worker_count"],
            pin_memory=(phase47_device.type == "cuda"),
            drop_last=False,
        )

        checkpoint_path = (
            Path("/kaggle/working/phase45_private_checkpoint")
            / f"phase45_fold_{fold}_encoder.pt"
        )
        model = Phase47CachedTailAdapter(
            checkpoint_path=checkpoint_path,
            hidden_dimension=specification["hidden_dimension"],
            dropout=specification["dropout"],
            residual_cap=specification["residual_cap"],
            train_tail=specification["train_tail"],
        ).to(phase47_device)

        tail_parameters = [
            parameter
            for name, parameter in model.named_parameters()
            if parameter.requires_grad
            and (
                name.startswith("tail_block.")
                or name.startswith("encoder_norm.")
            )
        ]
        head_parameters = [
            parameter
            for name, parameter in model.named_parameters()
            if parameter.requires_grad and name.startswith("head.")
        ]
        optimizer_groups = [{
            "params": head_parameters,
            "lr": float(specification["head_learning_rate"]),
        }]
        if specification["train_tail"]:
            assert tail_parameters
            optimizer_groups.insert(0, {
                "params": tail_parameters,
                "lr": float(specification["tail_learning_rate"]),
            })
        else:
            assert not tail_parameters
        assert head_parameters

        optimizer = torch.optim.AdamW(
            optimizer_groups,
            weight_decay=PHASE47_SELECTION_CONFIG["weight_decay"],
            betas=(0.9, 0.95),
        )
        total_steps = (
            PHASE47_SELECTION_CONFIG["maximum_epochs"] * len(fit_loader)
        )
        warmup_steps = max(
            1,
            int(round(
                PHASE47_SELECTION_CONFIG["warmup_fraction"] * total_steps
            )),
        )
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lambda step, total_steps=total_steps,
            warmup_steps=warmup_steps: phase47_learning_rate_factor(
                step,
                total_steps,
                warmup_steps,
            ),
        )

        # Epoch 0 is the exact Phase39 identity. A candidate must genuinely
        # improve its monitor loss to replace this safe checkpoint.
        anchor_fold_probability = phase47_anchor_probability[monitor_indices]
        best_log_loss = phase47_selection_log_loss(
            phase47_y[monitor_indices],
            anchor_fold_probability,
        )
        best_auroc = phase47_selection_auroc(
            phase47_y[monitor_indices],
            anchor_fold_probability,
        )
        best_epoch = 0
        best_raw_residual = np.zeros(monitor_indices.size, dtype=np.float64)
        best_probability = anchor_fold_probability.copy()
        epochs_without_improvement = 0
        epoch_history = []

        for epoch in range(1, PHASE47_SELECTION_CONFIG["maximum_epochs"] + 1):
            model.set_training_mode()
            total_bce = 0.0
            total_rank = 0.0
            total_weight = 0.0
            maximum_gradient_norm = 0.0

            for batch in fit_loader:
                prefix = batch["prefix"].to(
                    phase47_device,
                    non_blocking=False,
                ).float()
                anchor = batch["anchor_logit"].to(
                    phase47_device,
                    non_blocking=False,
                )
                labels = batch["label"].to(
                    phase47_device,
                    non_blocking=False,
                )
                groups = batch["group"].to(
                    phase47_device,
                    non_blocking=False,
                )
                case_weight = batch["case_weight"].to(
                    phase47_device,
                    non_blocking=False,
                )

                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=phase47_device.type,
                    dtype=phase47_amp_dtype,
                    enabled=phase47_amp_enabled,
                ):
                    output = model(prefix, anchor)
                element_bce = F.binary_cross_entropy_with_logits(
                    output["updated_logit"].float(),
                    labels.float(),
                    reduction="none",
                )
                bce_loss = torch.sum(
                    element_bce * case_weight.float()
                ) / torch.sum(case_weight.float())
                rank_loss = phase47_within_group_rank_loss(
                    output["updated_logit"].float(),
                    labels.float(),
                    groups,
                )
                loss = (
                    bce_loss
                    + float(specification["rank_loss_weight"]) * rank_loss
                )
                assert torch.isfinite(loss)
                loss.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for parameter in model.parameters()
                        if parameter.requires_grad
                    ],
                    PHASE47_SELECTION_CONFIG["gradient_clip"],
                )
                assert torch.isfinite(gradient_norm)
                optimizer.step()
                scheduler.step()

                batch_n = int(labels.numel())
                total_bce += float(bce_loss.detach().item()) * batch_n
                total_rank += float(rank_loss.detach().item()) * batch_n
                total_weight += batch_n
                maximum_gradient_norm = max(
                    maximum_gradient_norm,
                    float(gradient_norm.detach().item()),
                )

            prediction = phase47_predict_cached(
                model,
                monitor_loader,
                monitor_indices,
            )
            monitor_log_loss = phase47_selection_log_loss(
                phase47_y[monitor_indices],
                prediction["probability"],
            )
            monitor_auroc = phase47_selection_auroc(
                phase47_y[monitor_indices],
                prediction["probability"],
            )
            improved = monitor_log_loss < (
                best_log_loss
                - PHASE47_SELECTION_CONFIG[
                    "early_stopping_minimum_gain"
                ]
            )
            if improved:
                best_log_loss = float(monitor_log_loss)
                best_auroc = float(monitor_auroc)
                best_epoch = int(epoch)
                best_raw_residual = prediction["raw_residual"].copy()
                best_probability = prediction["probability"].copy()
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            epoch_history.append({
                "epoch": int(epoch),
                "train_bce": float(total_bce / total_weight),
                "train_rank": float(total_rank / total_weight),
                "monitor_log_loss": float(monitor_log_loss),
                "monitor_auroc": float(monitor_auroc),
                "improved": bool(improved),
                "maximum_gradient_norm": float(maximum_gradient_norm),
            })

            if (
                epoch >= PHASE47_SELECTION_CONFIG["minimum_epochs"]
                and epochs_without_improvement
                >= PHASE47_SELECTION_CONFIG["early_stopping_patience"]
            ):
                break

        phase47_candidate_monitor_raw[
            candidate_index,
            monitor_slice,
        ] = best_raw_residual
        phase47_candidate_monitor_probability[
            candidate_index,
            monitor_slice,
        ] = best_probability
        phase47_candidate_epoch_counts[candidate_index, fold] = best_epoch
        phase47_training_records.append({
            "candidate_index": int(candidate_index),
            "fold": int(fold),
            "seed": int(run_seed),
            "best_epoch": int(best_epoch),
            "best_monitor_log_loss": float(best_log_loss),
            "best_monitor_auroc": float(best_auroc),
            "epochs_executed": int(len(epoch_history)),
            "final_epoch": epoch_history[-1],
            "elapsed_seconds": round(time.perf_counter() - run_started, 3),
        })
        print(
            f"Phase47 candidate {candidate_index + 1}/{candidate_count}, "
            f"fold {fold + 1}/3: best_epoch={best_epoch}, "
            f"log_loss={best_log_loss:.6f}, auroc={best_auroc:.6f}"
        )

        del scheduler
        del optimizer
        del model
        del fit_loader
        del monitor_loader
        del fit_dataset
        del monitor_dataset
        gc.collect()
        if phase47_device.type == "cuda":
            torch.cuda.empty_cache()

assert np.all(np.isfinite(phase47_candidate_monitor_raw))
assert np.all(np.isfinite(phase47_candidate_monitor_probability))


def phase47_candidate_summary(candidate_index, probability):
    metrics = phase47_complete_monitor_metrics(probability)
    fold_wins = int(sum(
        record["log_loss"] < phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    worst_fold_regret = float(max(
        record["log_loss"] - phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    maximum_group_harm = float(max(
        record["log_loss"] - phase47_anchor_group_loss[record["group"]]
        for record in metrics["group_metrics"]
    ))
    screen_safe = bool(
        worst_fold_regret
        <= PHASE47_SELECTION_CONFIG["maximum_model_fold_regret"]
        and maximum_group_harm
        <= PHASE47_SELECTION_CONFIG["maximum_model_group_harm"]
    )
    return {
        "candidate_index": int(candidate_index),
        "specification": dict(phase47_model_grid[candidate_index]),
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        "log_loss_gain": float(
            phase47_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": float(
            metrics["auroc"] - phase47_anchor_metrics["auroc"]
        ),
        "fold_wins": fold_wins,
        "worst_fold_regret": worst_fold_regret,
        "maximum_group_harm": maximum_group_harm,
        "screen_safe": screen_safe,
        "selected_epoch_counts": [
            int(value)
            for value in phase47_candidate_epoch_counts[candidate_index]
        ],
        "fold_metrics": metrics["fold_metrics"],
    }


phase47_model_summaries = [
    phase47_candidate_summary(
        candidate_index,
        phase47_candidate_monitor_probability[candidate_index],
    )
    for candidate_index in range(candidate_count)
]
phase47_safe_model_summaries = [
    record for record in phase47_model_summaries if record["screen_safe"]
]
phase47_model_screen_fallback = not bool(phase47_safe_model_summaries)
phase47_model_selection_pool = (
    phase47_safe_model_summaries
    if phase47_safe_model_summaries
    else phase47_model_summaries
)
phase47_raw_model_best = min(
    phase47_model_selection_pool,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["candidate_index"],
    ),
)
phase47_model_band = [
    record
    for record in phase47_model_selection_pool
    if record["monitor_log_loss"]
    <= phase47_raw_model_best["monitor_log_loss"]
    + PHASE47_SELECTION_CONFIG["model_log_loss_band"]
]
phase47_selected_model = max(
    phase47_model_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["candidate_index"],
    ),
)
phase47_selected_candidate_index = int(
    phase47_selected_model["candidate_index"]
)
phase47_selected_raw_residual = phase47_candidate_monitor_raw[
    phase47_selected_candidate_index
].copy()


def phase47_gate_summary(gate_specification):
    if gate_specification["scale"] == 0.0:
        update = np.zeros_like(phase47_selected_raw_residual)
    else:
        uncertainty = np.power(
            4.0
            * phase47_monitor_anchor_probability
            * (1.0 - phase47_monitor_anchor_probability),
            gate_specification["uncertainty_exponent"],
        )
        update = (
            gate_specification["scale"]
            * uncertainty
            * np.clip(
                phase47_selected_raw_residual,
                -gate_specification["residual_cap"],
                gate_specification["residual_cap"],
            )
        )
    probability = phase47_selection_sigmoid(
        phase47_monitor_anchor_logit + update
    )
    metrics = phase47_complete_monitor_metrics(probability)
    fold_wins = int(sum(
        record["log_loss"] < phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    worst_fold_regret = float(max(
        record["log_loss"] - phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    maximum_group_harm = float(max(
        record["log_loss"] - phase47_anchor_group_loss[record["group"]]
        for record in metrics["group_metrics"]
    ))
    is_zero = gate_specification["scale"] == 0.0
    safe = bool(
        is_zero
        or (
            fold_wins >= PHASE47_SELECTION_CONFIG["minimum_gate_fold_wins"]
            and worst_fold_regret
            <= PHASE47_SELECTION_CONFIG["maximum_gate_fold_regret"]
            and maximum_group_harm
            <= PHASE47_SELECTION_CONFIG["maximum_gate_group_harm"]
        )
    )
    return {
        "gate_index": int(gate_specification["gate_index"]),
        "specification": dict(gate_specification),
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        "log_loss_gain": float(
            phase47_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": float(
            metrics["auroc"] - phase47_anchor_metrics["auroc"]
        ),
        "fold_wins": fold_wins,
        "worst_fold_regret": worst_fold_regret,
        "maximum_group_harm": maximum_group_harm,
        "mean_absolute_update": float(np.mean(np.abs(update))),
        "q95_absolute_update": float(np.quantile(np.abs(update), 0.95)),
        "safe": safe,
        "fold_metrics": metrics["fold_metrics"],
    }


phase47_gate_summaries = [
    phase47_gate_summary(specification)
    for specification in phase47_gate_grid
]
phase47_safe_gate_summaries = [
    record for record in phase47_gate_summaries if record["safe"]
]
assert phase47_safe_gate_summaries
phase47_raw_gate_best = min(
    phase47_safe_gate_summaries,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["gate_index"],
    ),
)
phase47_gate_band = [
    record
    for record in phase47_safe_gate_summaries
    if record["monitor_log_loss"]
    <= phase47_raw_gate_best["monitor_log_loss"]
    + PHASE47_SELECTION_CONFIG["gate_log_loss_band"]
]
phase47_provisional_gate = max(
    phase47_gate_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["gate_index"],
    ),
)

phase47_advance_by_log_loss = bool(
    phase47_provisional_gate["log_loss_gain"]
    >= PHASE47_SELECTION_CONFIG["minimum_advance_log_loss_gain"]
    and phase47_provisional_gate["auroc_gain"]
    >= -PHASE47_SELECTION_CONFIG["maximum_advance_auroc_deficit"]
)
phase47_advance_by_auroc = bool(
    phase47_provisional_gate["auroc_gain"]
    >= PHASE47_SELECTION_CONFIG["minimum_advance_auroc_gain"]
    and phase47_provisional_gate["log_loss_gain"]
    >= -PHASE47_SELECTION_CONFIG["maximum_advance_log_loss_excess"]
)
phase47_gate_advanced = bool(
    phase47_provisional_gate["gate_index"] != 0
    and (phase47_advance_by_log_loss or phase47_advance_by_auroc)
)
phase47_selected_gate = (
    phase47_provisional_gate
    if phase47_gate_advanced
    else phase47_gate_summaries[0]
)

if phase47_device.type == "cuda":
    phase47_peak_vram_mb = float(
        torch.cuda.max_memory_allocated(phase47_device) / (1024 ** 2)
    )
else:
    phase47_peak_vram_mb = 0.0

PHASE47_MODEL_GRID_PRIVATE = copy.deepcopy(phase47_model_grid)
PHASE47_SELECTED_MODEL_PRIVATE = copy.deepcopy(phase47_selected_model)
PHASE47_SELECTED_GATE_PRIVATE = copy.deepcopy(phase47_selected_gate)
PHASE47_SELECTED_EPOCH_COUNTS_PRIVATE = tuple(
    phase47_selected_model["selected_epoch_counts"]
)
PHASE47_GATE_ADVANCED_PRIVATE = bool(phase47_gate_advanced)

phase47_selection_report = {
    "phase": "phase47_cached_tail_anchor_preserving_supervised_selection",
    "status": (
        "model_and_gate_frozen_for_outer_evaluation"
        if phase47_gate_advanced
        else "no_update_selected_stop_before_outer_evaluation"
    ),
    "anchor_monitor": {
        "log_loss": float(phase47_anchor_metrics["log_loss"]),
        "auroc": float(phase47_anchor_metrics["auroc"]),
    },
    "training": {
        "model_specification_count": candidate_count,
        "fold_count": 3,
        "trained_model_count": int(candidate_count * 3),
        "maximum_epochs": int(
            PHASE47_SELECTION_CONFIG["maximum_epochs"]
        ),
        "batch_size": int(PHASE47_SELECTION_CONFIG["batch_size"]),
        "group_balanced_case_weights": True,
        "epoch_zero_phase39_identity_allowed": True,
        "records": phase47_training_records,
    },
    "model_selection": {
        "screen_safe_candidate_count": len(
            phase47_safe_model_summaries
        ),
        "fallback_to_all_candidates": bool(
            phase47_model_screen_fallback
        ),
        "raw_log_loss_best": phase47_raw_model_best,
        "selected": phase47_selected_model,
    },
    "gate_selection": {
        "candidate_count": len(phase47_gate_summaries),
        "safe_candidate_count": len(phase47_safe_gate_summaries),
        "raw_log_loss_best": phase47_raw_gate_best,
        "provisional": phase47_provisional_gate,
        "selected": phase47_selected_gate,
        "advance_by_log_loss": phase47_advance_by_log_loss,
        "advance_by_auroc": phase47_advance_by_auroc,
        "gate_advanced": phase47_gate_advanced,
    },
    "selection_rule": (
        "choose maximum AUROC within 0.002 monitor log-loss of the safe "
        "shared-model minimum; then choose maximum AUROC within 0.001 "
        "monitor log-loss of the safe bounded-gate minimum"
    ),
    "formula_if_advanced": (
        "z47 = z39 + scale * uncertainty(z39)^gamma * "
        "clip(r47_raw, -cap, cap)"
    ),
    "compute": {
        "prefix_cache_used": True,
        "frozen_encoder_blocks_recomputed_during_search": False,
        "mixed_precision": (
            str(phase47_amp_dtype).replace("torch.", "")
            if phase47_amp_enabled
            else "disabled"
        ),
        "peak_vram_mb": phase47_peak_vram_mb,
    },
    "shared_model_specification_across_folds": True,
    "shared_gate_across_folds": True,
    "fit_labels_used_for_training": True,
    "monitor_labels_used_for_selection": True,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase47_selection_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE47_SELECTION")
print(json.dumps(phase47_selection_report, indent=2))
print("END SANITIZED_PHASE47_SELECTION")

Phase47 candidate 1/6, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661
Phase47 candidate 1/6, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867
Phase47 candidate 1/6, fold 3/3: best_epoch=2, log_loss=0.235181, auroc=0.971944
Phase47 candidate 2/6, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661
Phase47 candidate 2/6, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867
Phase47 candidate 2/6, fold 3/3: best_epoch=1, log_loss=0.235360, auroc=0.972048
Phase47 candidate 3/6, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661
Phase47 candidate 3/6, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867
Phase47 candidate 3/6, fold 3/3: best_epoch=3, log_loss=0.234508, auroc=0.971944
Phase47 candidate 4/6, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661
Phase47 candidate 4/6, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867
Phase47 candidate 4/6, fold 3/3: best_epoch=2, log_loss=0.235462, auroc=0.971944
Phase47 candidate 5/6, fold 

In [58]:
# Phase48 Cell 146A 

import copy
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


phase48_contract_started = time.perf_counter()

PHASE48_ATTENTION_CONFIG = {
    "seed": 480701,
    "attention_head_count": 2,
    "encoder_dimension": 192,
    "token_count": 1000,
    "head_hidden_dimension": 64,
    "head_dropout": 0.10,
    "residual_cap": 0.50,
    "train_tail": True,
    "tail_learning_rate": 3.0e-6,
    "pooling_learning_rate": 2.0e-4,
    "head_learning_rate": 2.0e-4,
    "weight_decay": 0.02,
    "contract_batch_size": 2,
}

required_names = [
    "PHASE47_ADAPTER_CLASS_PRIVATE",
    "PHASE47_PREFIX_CACHE_PATHS_PRIVATE",
    "PHASE45_DEVICE",
    "PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE",
    "PHASE39_OOF_PRIVATE",
    "PHASE44_Y_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase48 attention contract is missing Phase47 state.",
    "missing": missing_names,
}


def phase48_set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def phase48_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase48_gradient_contract(parameters):
    tensor_count = 0
    squared_norm = 0.0
    all_finite = True
    for parameter in parameters:
        if parameter.grad is None:
            continue
        gradient = parameter.grad.detach().float()
        tensor_count += 1
        all_finite = all_finite and bool(
            torch.isfinite(gradient).all().item()
        )
        squared_norm += float(torch.sum(gradient * gradient).item())
    return {
        "tensor_count": int(tensor_count),
        "norm": float(squared_norm ** 0.5),
        "all_finite": bool(all_finite),
    }


class Phase48SpatialAttentionAdapter(nn.Module):
    """Coordinate-aware token pooling with a zero-initialized Phase39 update."""

    def __init__(
        self,
        checkpoint_path,
        attention_head_count=2,
        hidden_dimension=64,
        dropout=0.10,
        residual_cap=0.50,
        train_tail=True,
        initialization_seed=480701,
    ):
        super().__init__()
        attention_head_count = int(attention_head_count)
        assert attention_head_count >= 1
        self.attention_head_count = attention_head_count
        self.encoder_dimension = int(
            PHASE48_ATTENTION_CONFIG["encoder_dimension"]
        )
        self.token_count = int(PHASE48_ATTENTION_CONFIG["token_count"])
        self.residual_cap = float(residual_cap)
        self.train_tail = bool(train_tail)
        assert self.residual_cap > 0.0

        source = PHASE47_ADAPTER_CLASS_PRIVATE(
            encoder_checkpoint_path=checkpoint_path,
            unfrozen_encoder_block_count=1,
            hidden_dimension=hidden_dimension,
            dropout=dropout,
            residual_cap=residual_cap,
        )
        self.tail_block = copy.deepcopy(
            source.backbone.encoder.layers[-1]
        )
        self.encoder_norm = copy.deepcopy(source.backbone.encoder_norm)
        del source

        for parameter in self.tail_block.parameters():
            parameter.requires_grad_(self.train_tail)
        for parameter in self.encoder_norm.parameters():
            parameter.requires_grad_(self.train_tail)

        # Small distinct queries break multi-head symmetry. Spatial biases begin
        # at zero, so no acquisition protocol or diagnosis-specific coordinate
        # is hard-coded.
        query_generator = torch.Generator(device="cpu")
        query_generator.manual_seed(int(initialization_seed))
        initial_queries = torch.randn(
            attention_head_count,
            self.encoder_dimension,
            generator=query_generator,
            dtype=torch.float32,
        ) * 0.01
        self.attention_queries = nn.Parameter(initial_queries)
        self.spatial_bias = nn.Parameter(torch.zeros(
            attention_head_count,
            self.token_count,
            dtype=torch.float32,
        ))

        pooled_dimension = attention_head_count * self.encoder_dimension
        self.pooled_dimension = pooled_dimension
        self.representation_dimension = 2 * pooled_dimension
        self.residual_head = nn.Sequential(
            nn.LayerNorm(self.representation_dimension),
            nn.Linear(self.representation_dimension, int(hidden_dimension)),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(int(hidden_dimension), 1),
        )
        nn.init.zeros_(self.residual_head[-1].weight)
        nn.init.zeros_(self.residual_head[-1].bias)

    def set_training_mode(self):
        self.train()
        if not self.train_tail:
            self.tail_block.eval()
            self.encoder_norm.eval()

    def encoded_tokens(self, penultimate_tokens):
        if self.train_tail:
            output = self.tail_block(penultimate_tokens)
            output = self.encoder_norm(output)
        else:
            with torch.no_grad():
                output = self.tail_block(penultimate_tokens)
                output = self.encoder_norm(output)
        assert output.ndim == 3
        assert output.shape[1:] == (
            self.token_count,
            self.encoder_dimension,
        )
        return output

    def attention_pool(self, tokens):
        query = self.attention_queries.to(dtype=tokens.dtype)
        bias = self.spatial_bias.to(dtype=tokens.dtype)
        attention_logit = torch.einsum(
            "bnd,hd->bhn",
            tokens,
            query,
        ) / math.sqrt(float(self.encoder_dimension))
        attention_logit = attention_logit + bias.unsqueeze(0)
        attention_weight = torch.softmax(attention_logit.float(), dim=-1)
        pooled = torch.einsum(
            "bhn,bnd->bhd",
            attention_weight.to(dtype=tokens.dtype),
            tokens,
        )
        return pooled.flatten(1), attention_weight

    def invariant_representation(self, prefix):
        assert prefix.ndim == 4
        assert prefix.shape[1:] == (
            2,
            self.token_count,
            self.encoder_dimension,
        )
        original_tokens = self.encoded_tokens(prefix[:, 0])
        reflected_tokens = self.encoded_tokens(prefix[:, 1])
        original_pool, original_attention = self.attention_pool(
            original_tokens
        )
        reflected_pool, reflected_attention = self.attention_pool(
            reflected_tokens
        )
        symmetric = 0.5 * (original_pool + reflected_pool)
        absolute_antisymmetric = torch.abs(
            0.5 * (original_pool - reflected_pool)
        )
        representation = torch.cat(
            [symmetric, absolute_antisymmetric],
            dim=1,
        )
        return {
            "representation": representation,
            "original_attention": original_attention,
            "reflected_attention": reflected_attention,
        }

    def forward(self, prefix, anchor_logit):
        pooled = self.invariant_representation(prefix)
        raw_residual = self.residual_head(
            pooled["representation"].float()
        ).squeeze(1)
        bounded_residual = self.residual_cap * torch.tanh(
            raw_residual / self.residual_cap
        )
        updated_logit = anchor_logit.float().reshape(-1) + bounded_residual
        return {
            "probability": torch.sigmoid(updated_logit),
            "updated_logit": updated_logit,
            "raw_residual": raw_residual,
            "bounded_residual": bounded_residual,
            **pooled,
        }


phase48_set_seed(PHASE48_ATTENTION_CONFIG["seed"])
phase48_device = torch.device(PHASE45_DEVICE)
phase48_amp_enabled = phase48_device.type == "cuda"
phase48_amp_dtype = (
    torch.bfloat16
    if phase48_amp_enabled and torch.cuda.is_bf16_supported()
    else torch.float16
)
if phase48_device.type == "cuda":
    assert phase48_amp_dtype == torch.bfloat16

phase48_checkpoint_path = Path(
    "/kaggle/working/phase45_private_checkpoint/phase45_fold_0_encoder.pt"
)
assert phase48_checkpoint_path.is_file()
phase48_prefix_path = Path(PHASE47_PREFIX_CACHE_PATHS_PRIVATE[0])
assert phase48_prefix_path.is_file()
phase48_prefix_matrix = np.load(
    phase48_prefix_path,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase48_prefix_matrix.shape == (2, 1362, 1000, 192)
assert phase48_prefix_matrix.dtype == np.float16

phase48_fit_indices = np.asarray(
    PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE[0]["fit_indices"],
    dtype=np.int64,
)
phase48_contract_indices = phase48_fit_indices[
    :PHASE48_ATTENTION_CONFIG["contract_batch_size"]
]
assert phase48_contract_indices.size == 2
phase48_contract_prefix = torch.from_numpy(np.array(
    phase48_prefix_matrix[:, phase48_contract_indices],
    dtype=np.float32,
    copy=True,
)).permute(1, 0, 2, 3).contiguous().to(phase48_device)
assert phase48_contract_prefix.shape == (2, 2, 1000, 192)

phase48_anchor_logit_all = phase48_logit(PHASE39_OOF_PRIVATE)
phase48_contract_anchor = torch.from_numpy(
    phase48_anchor_logit_all[phase48_contract_indices].astype(np.float32)
).to(phase48_device)
phase48_contract_target = torch.from_numpy(
    np.asarray(PHASE44_Y_PRIVATE, dtype=np.float32).reshape(-1)[
        phase48_contract_indices
    ].copy()
).to(phase48_device)

if phase48_device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase48_device)

phase48_model = Phase48SpatialAttentionAdapter(
    checkpoint_path=phase48_checkpoint_path,
    attention_head_count=PHASE48_ATTENTION_CONFIG[
        "attention_head_count"
    ],
    hidden_dimension=PHASE48_ATTENTION_CONFIG["head_hidden_dimension"],
    dropout=PHASE48_ATTENTION_CONFIG["head_dropout"],
    residual_cap=PHASE48_ATTENTION_CONFIG["residual_cap"],
    train_tail=PHASE48_ATTENTION_CONFIG["train_tail"],
    initialization_seed=PHASE48_ATTENTION_CONFIG["seed"],
).to(phase48_device)

tail_parameters = list(phase48_model.tail_block.parameters()) + list(
    phase48_model.encoder_norm.parameters()
)
pooling_parameters = [
    phase48_model.attention_queries,
    phase48_model.spatial_bias,
]
head_parameters = list(phase48_model.residual_head.parameters())
assert tail_parameters and pooling_parameters and head_parameters

# Identity and view-swap invariance are tested in evaluation mode, eliminating
# dropout as a confounder.
phase48_model.eval()
with torch.inference_mode(), torch.autocast(
    device_type=phase48_device.type,
    dtype=phase48_amp_dtype,
    enabled=phase48_amp_enabled,
):
    phase48_initial_output = phase48_model(
        phase48_contract_prefix,
        phase48_contract_anchor,
    )
    phase48_swapped_output = phase48_model(
        phase48_contract_prefix[:, [1, 0]],
        phase48_contract_anchor,
    )

phase48_identity_logit_error = float(torch.max(torch.abs(
    phase48_initial_output["updated_logit"].float()
    - phase48_contract_anchor.float()
)).item())
phase48_identity_probability_error = float(torch.max(torch.abs(
    phase48_initial_output["probability"].float()
    - torch.sigmoid(phase48_contract_anchor.float())
)).item())
phase48_swap_representation_error = float(torch.max(torch.abs(
    phase48_initial_output["representation"].float()
    - phase48_swapped_output["representation"].float()
)).item())
phase48_swap_probability_error = float(torch.max(torch.abs(
    phase48_initial_output["probability"].float()
    - phase48_swapped_output["probability"].float()
)).item())
assert phase48_identity_logit_error == 0.0
assert phase48_identity_probability_error == 0.0
assert phase48_swap_representation_error == 0.0
assert phase48_swap_probability_error == 0.0
assert phase48_initial_output["representation"].shape == (2, 768)
assert phase48_initial_output["original_attention"].shape == (2, 2, 1000)

attention_weight = phase48_initial_output["original_attention"].float()
phase48_attention_sum_error = float(torch.max(torch.abs(
    attention_weight.sum(dim=-1) - 1.0
)).item())
phase48_attention_entropy = -torch.sum(
    attention_weight * torch.log(torch.clamp(attention_weight, min=1.0e-12)),
    dim=-1,
) / math.log(float(PHASE48_ATTENTION_CONFIG["token_count"]))
phase48_mean_normalized_entropy = float(
    phase48_attention_entropy.mean().item()
)
assert phase48_attention_sum_error <= 1.0e-6
assert 0.0 <= phase48_mean_normalized_entropy <= 1.0 + 1.0e-6

phase48_optimizer = torch.optim.AdamW(
    [
        {
            "params": tail_parameters,
            "lr": PHASE48_ATTENTION_CONFIG["tail_learning_rate"],
        },
        {
            "params": pooling_parameters,
            "lr": PHASE48_ATTENTION_CONFIG["pooling_learning_rate"],
        },
        {
            "params": head_parameters,
            "lr": PHASE48_ATTENTION_CONFIG["head_learning_rate"],
        },
    ],
    weight_decay=PHASE48_ATTENTION_CONFIG["weight_decay"],
    betas=(0.9, 0.95),
)

# First step: only the zero-initialized output layer is expected to receive a
# useful supervised signal. Second step: the signal must reach spatial pooling
# and the unfrozen transformer tail.
phase48_model.set_training_mode()
phase48_optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=phase48_device.type,
    dtype=phase48_amp_dtype,
    enabled=phase48_amp_enabled,
):
    phase48_first_output = phase48_model(
        phase48_contract_prefix,
        phase48_contract_anchor,
    )
phase48_first_loss = F.binary_cross_entropy_with_logits(
    phase48_first_output["updated_logit"].float(),
    phase48_contract_target.float(),
)
phase48_first_loss.backward()
phase48_first_tail_gradient = phase48_gradient_contract(tail_parameters)
phase48_first_pooling_gradient = phase48_gradient_contract(
    pooling_parameters
)
phase48_first_head_gradient = phase48_gradient_contract(head_parameters)
assert phase48_first_head_gradient["all_finite"]
assert phase48_first_head_gradient["norm"] > 0.0
phase48_optimizer.step()

phase48_optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=phase48_device.type,
    dtype=phase48_amp_dtype,
    enabled=phase48_amp_enabled,
):
    phase48_second_output = phase48_model(
        phase48_contract_prefix,
        phase48_contract_anchor,
    )
phase48_second_loss = F.binary_cross_entropy_with_logits(
    phase48_second_output["updated_logit"].float(),
    phase48_contract_target.float(),
)
phase48_second_loss.backward()
phase48_second_tail_gradient = phase48_gradient_contract(tail_parameters)
phase48_second_pooling_gradient = phase48_gradient_contract(
    pooling_parameters
)
phase48_second_head_gradient = phase48_gradient_contract(head_parameters)
for record in (
    phase48_second_tail_gradient,
    phase48_second_pooling_gradient,
    phase48_second_head_gradient,
):
    assert record["all_finite"]
    assert record["norm"] > 0.0

phase48_second_residual_maximum = float(torch.max(torch.abs(
    phase48_second_output["bounded_residual"].float()
)).item())
assert phase48_second_residual_maximum <= (
    PHASE48_ATTENTION_CONFIG["residual_cap"] + 1.0e-6
)
assert torch.isfinite(phase48_second_output["probability"]).all()

phase48_trainable_names = [
    name
    for name, parameter in phase48_model.named_parameters()
    if parameter.requires_grad
]
assert all(
    name.startswith("tail_block.")
    or name.startswith("encoder_norm.")
    or name in {"attention_queries", "spatial_bias"}
    or name.startswith("residual_head.")
    for name in phase48_trainable_names
)

if phase48_device.type == "cuda":
    torch.cuda.synchronize(phase48_device)
    phase48_peak_vram_mb = float(
        torch.cuda.max_memory_allocated(phase48_device) / (1024 ** 2)
    )
else:
    phase48_peak_vram_mb = 0.0

PHASE48_ADAPTER_CLASS_PRIVATE = Phase48SpatialAttentionAdapter
PHASE48_ATTENTION_CONFIG_PRIVATE = dict(PHASE48_ATTENTION_CONFIG)

phase48_contract_report = {
    "phase": "phase48_reflection_consistent_spatial_token_attention_contract",
    "status": "accepted",
    "motivation": {
        "phase47_gate_advanced": False,
        "phase47_monitor_log_loss_gain": 0.0006584433855790772,
        "phase47_monitor_auroc_gain": 0.0002356780275563075,
        "phase47_improving_fold_count": 1,
        "identified_bottleneck": (
            "uniform_mean_pooling_of_1000_spatial_tokens"
        ),
    },
    "architecture": {
        "input_shape": [2, 2, 1000, 192],
        "attention_head_count": int(
            PHASE48_ATTENTION_CONFIG["attention_head_count"]
        ),
        "token_count": int(PHASE48_ATTENTION_CONFIG["token_count"]),
        "encoder_dimension": int(
            PHASE48_ATTENTION_CONFIG["encoder_dimension"]
        ),
        "pooled_dimension_per_view": int(phase48_model.pooled_dimension),
        "reflection_invariant_representation_dimension": int(
            phase48_model.representation_dimension
        ),
        "representation": (
            "symmetric_plus_absolute_antisymmetric_attention_pool"
        ),
        "spatial_bias_initialized_uniformly": True,
        "attention_queries_initialized_distinctly": True,
        "residual_cap": float(PHASE48_ATTENTION_CONFIG["residual_cap"]),
    },
    "initial_contract": {
        "phase39_logit_error": phase48_identity_logit_error,
        "phase39_probability_error": phase48_identity_probability_error,
        "view_swap_representation_error": (
            phase48_swap_representation_error
        ),
        "view_swap_probability_error": phase48_swap_probability_error,
        "attention_sum_error": phase48_attention_sum_error,
        "mean_normalized_attention_entropy": (
            phase48_mean_normalized_entropy
        ),
    },
    "two_step_backward_contract": {
        "first_loss": float(phase48_first_loss.detach().item()),
        "second_loss": float(phase48_second_loss.detach().item()),
        "first_tail_gradient": phase48_first_tail_gradient,
        "first_pooling_gradient": phase48_first_pooling_gradient,
        "first_head_gradient": phase48_first_head_gradient,
        "second_tail_gradient": phase48_second_tail_gradient,
        "second_pooling_gradient": phase48_second_pooling_gradient,
        "second_head_gradient": phase48_second_head_gradient,
        "second_step_maximum_absolute_residual": (
            phase48_second_residual_maximum
        ),
    },
    "parameters": {
        "total": int(sum(
            parameter.numel() for parameter in phase48_model.parameters()
        )),
        "trainable": int(sum(
            parameter.numel()
            for parameter in phase48_model.parameters()
            if parameter.requires_grad
        )),
        "attention_query": int(
            phase48_model.attention_queries.numel()
        ),
        "spatial_bias": int(phase48_model.spatial_bias.numel()),
    },
    "planned_selection": {
        "attention_head_counts": [1, 2, 4],
        "tail_modes": ["frozen", "trainable"],
        "ranking_loss_weights": [0.0, 0.05],
        "shared_specification_across_folds": True,
        "epoch_zero_phase39_identity_allowed": True,
        "outer_evaluation_only_after_gate_freeze": True,
    },
    "execution": {
        "device": str(phase48_device),
        "mixed_precision": (
            str(phase48_amp_dtype).replace("torch.", "")
            if phase48_amp_enabled
            else "disabled"
        ),
        "peak_vram_mb": phase48_peak_vram_mb,
    },
    "fit_labels_used_for_backward_contract": True,
    "monitor_labels_used": False,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_prefix_cache_read": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase48_contract_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE48_ATTENTION_CONTRACT")
print(json.dumps(phase48_contract_report, indent=2))
print("END SANITIZED_PHASE48_ATTENTION_CONTRACT")

del phase48_optimizer
del phase48_model
del phase48_contract_prefix
del phase48_contract_anchor
del phase48_contract_target
if phase48_device.type == "cuda":
    torch.cuda.empty_cache()

BEGIN SANITIZED_PHASE48_ATTENTION_CONTRACT
{
  "phase": "phase48_reflection_consistent_spatial_token_attention_contract",
  "status": "accepted",
  "motivation": {
    "phase47_gate_advanced": false,
    "phase47_monitor_log_loss_gain": 0.0006584433855790772,
    "phase47_monitor_auroc_gain": 0.0002356780275563075,
    "phase47_improving_fold_count": 1,
    "identified_bottleneck": "uniform_mean_pooling_of_1000_spatial_tokens"
  },
  "architecture": {
    "input_shape": [
      2,
      2,
      1000,
      192
    ],
    "attention_head_count": 2,
    "token_count": 1000,
    "encoder_dimension": 192,
    "pooled_dimension_per_view": 384,
    "reflection_invariant_representation_dimension": 768,
    "representation": "symmetric_plus_absolute_antisymmetric_attention_pool",
    "spatial_bias_initialized_uniformly": true,
    "attention_queries_initialized_distinctly": true,
    "residual_cap": 0.5
  },
  "initial_contract": {
    "phase39_logit_error": 0.0,
    "phase39_probability_erro

In [59]:
# Phase48 Cell 146B 

import copy
import gc
import itertools
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


phase48_selection_started = time.perf_counter()

PHASE48_SELECTION_CONFIG = {
    "batch_size": 24,
    "evaluation_batch_size": 32,
    "worker_count": 0,
    "maximum_epochs": 8,
    "minimum_epochs": 4,
    "early_stopping_patience": 3,
    "early_stopping_minimum_gain": 1.0e-4,
    "gradient_clip": 1.0,
    "weight_decay": 0.02,
    "warmup_fraction": 0.10,
    "minimum_learning_rate_fraction": 0.10,
    "model_log_loss_band": 0.002,
    "gate_log_loss_band": 0.001,
    "maximum_model_fold_regret": 0.006,
    "maximum_model_group_harm": 0.015,
    "minimum_gate_fold_wins": 2,
    "maximum_gate_fold_regret": 0.003,
    "maximum_gate_group_harm": 0.010,
    "minimum_advance_log_loss_gain": 0.001,
    "minimum_advance_auroc_gain": 0.001,
    "maximum_advance_auroc_deficit": 0.0002,
    "maximum_advance_log_loss_excess": 0.0005,
}

required_names = [
    "PHASE48_ADAPTER_CLASS_PRIVATE",
    "PHASE47_PREFIX_CACHE_PATHS_PRIVATE",
    "Phase47PrefixDataset",
    "phase47_training_weights",
    "phase47_within_group_rank_loss",
    "phase47_complete_monitor_metrics",
    "phase47_selection_log_loss",
    "phase47_selection_auroc",
    "phase47_selection_sigmoid",
    "phase47_partitions",
    "phase47_y",
    "phase47_groups",
    "phase47_anchor_probability",
    "phase47_anchor_logit",
    "phase47_monitor_indices",
    "phase47_monitor_folds",
    "phase47_monitor_y",
    "phase47_monitor_groups",
    "phase47_monitor_weights",
    "phase47_monitor_anchor_probability",
    "phase47_monitor_anchor_logit",
    "phase47_monitor_slices",
    "phase47_anchor_metrics",
    "phase47_anchor_fold_loss",
    "phase47_anchor_group_loss",
    "PHASE45_DEVICE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase48 selection requires the accepted Phase47/48 state.",
    "missing": missing_names,
}


def phase48_selection_set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def phase48_learning_rate_factor(step, total_steps, warmup_steps):
    step = int(step)
    total_steps = max(int(total_steps), 1)
    warmup_steps = max(int(warmup_steps), 1)
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(
        max(total_steps - warmup_steps, 1)
    )
    progress = min(max(progress, 0.0), 1.0)
    minimum = PHASE48_SELECTION_CONFIG[
        "minimum_learning_rate_fraction"
    ]
    return float(
        minimum
        + (1.0 - minimum)
        * 0.5
        * (1.0 + math.cos(math.pi * progress))
    )


phase48_device = torch.device(PHASE45_DEVICE)
phase48_amp_enabled = phase48_device.type == "cuda"
phase48_amp_dtype = (
    torch.bfloat16
    if phase48_amp_enabled and torch.cuda.is_bf16_supported()
    else torch.float16
)
if phase48_device.type == "cuda":
    assert phase48_amp_dtype == torch.bfloat16

phase48_prefix_paths = tuple(
    Path(path) for path in PHASE47_PREFIX_CACHE_PATHS_PRIVATE
)
assert len(phase48_prefix_paths) == 3
assert all(path.is_file() for path in phase48_prefix_paths)
phase48_checkpoint_paths = tuple(
    Path("/kaggle/working/phase45_private_checkpoint")
    / f"phase45_fold_{fold}_encoder.pt"
    for fold in range(3)
)
assert all(path.is_file() for path in phase48_checkpoint_paths)


@torch.no_grad()
def phase48_predict_attention(model, loader, expected_indices):
    model.eval()
    raw_parts = []
    bounded_parts = []
    probability_parts = []
    index_parts = []
    entropy_sum = 0.0
    maximum_weight_sum = 0.0
    observation_count = 0

    for batch in loader:
        prefix = batch["prefix"].to(
            phase48_device,
            non_blocking=False,
        ).float()
        anchor = batch["anchor_logit"].to(
            phase48_device,
            non_blocking=False,
        )
        with torch.autocast(
            device_type=phase48_device.type,
            dtype=phase48_amp_dtype,
            enabled=phase48_amp_enabled,
        ):
            output = model(prefix, anchor)

        raw_parts.append(
            output["raw_residual"].float().cpu().numpy()
        )
        bounded_parts.append(
            output["bounded_residual"].float().cpu().numpy()
        )
        probability_parts.append(
            output["probability"].float().cpu().numpy()
        )
        index_parts.append(batch["case_index"].numpy())

        attention = torch.cat([
            output["original_attention"].float(),
            output["reflected_attention"].float(),
        ], dim=1)
        entropy = -torch.sum(
            attention * torch.log(torch.clamp(attention, min=1.0e-12)),
            dim=-1,
        ) / math.log(float(attention.shape[-1]))
        batch_n = int(attention.shape[0])
        entropy_sum += float(entropy.mean().item()) * batch_n
        maximum_weight_sum += float(
            attention.max(dim=-1).values.mean().item()
        ) * batch_n
        observation_count += batch_n

    indices = np.concatenate(index_parts).astype(np.int64, copy=False)
    expected_indices = np.asarray(expected_indices, dtype=np.int64)
    assert np.array_equal(indices, expected_indices)
    raw = np.concatenate(raw_parts).astype(np.float64, copy=False)
    bounded = np.concatenate(bounded_parts).astype(np.float64, copy=False)
    probability = np.concatenate(probability_parts).astype(
        np.float64,
        copy=False,
    )
    assert raw.shape == expected_indices.shape
    assert bounded.shape == expected_indices.shape
    assert probability.shape == expected_indices.shape
    assert observation_count == expected_indices.size
    assert np.all(np.isfinite(raw))
    assert np.all(np.isfinite(bounded))
    assert np.all(np.isfinite(probability))
    return {
        "raw_residual": raw,
        "bounded_residual": bounded,
        "probability": np.clip(probability, 1.0e-7, 1.0 - 1.0e-7),
        "mean_normalized_attention_entropy": float(
            entropy_sum / observation_count
        ),
        "mean_maximum_attention_weight": float(
            maximum_weight_sum / observation_count
        ),
    }


phase48_model_grid = []
for candidate_index, values in enumerate(itertools.product(
    [1, 2, 4],
    [False, True],
    [0.0, 0.05],
)):
    attention_head_count, train_tail, rank_loss_weight = values
    phase48_model_grid.append({
        "candidate_index": int(candidate_index),
        "attention_head_count": int(attention_head_count),
        "train_tail": bool(train_tail),
        "hidden_dimension": 64,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": (3.0e-6 if train_tail else 0.0),
        "pooling_learning_rate": 2.0e-4,
        "head_learning_rate": 2.0e-4,
        "rank_loss_weight": float(rank_loss_weight),
    })
assert len(phase48_model_grid) == 12

phase48_gate_grid = [{
    "gate_index": 0,
    "scale": 0.0,
    "residual_cap": 0.0,
    "uncertainty_exponent": 0.0,
}]
for scale, cap, exponent in itertools.product(
    [0.10, 0.25, 0.50, 0.75, 1.0],
    [0.10, 0.25, 0.50, 1.0],
    [0.0, 0.5, 1.0],
):
    phase48_gate_grid.append({
        "gate_index": int(len(phase48_gate_grid)),
        "scale": float(scale),
        "residual_cap": float(cap),
        "uncertainty_exponent": float(exponent),
    })
assert len(phase48_gate_grid) == 61

candidate_count = len(phase48_model_grid)
phase48_candidate_monitor_raw = np.zeros(
    (candidate_count, 493),
    dtype=np.float64,
)
phase48_candidate_monitor_probability = np.tile(
    phase47_monitor_anchor_probability[None, :],
    (candidate_count, 1),
)
phase48_candidate_epoch_counts = np.zeros(
    (candidate_count, 3),
    dtype=np.int64,
)
phase48_candidate_attention_entropy = np.ones(
    (candidate_count, 3),
    dtype=np.float64,
)
phase48_candidate_maximum_attention = np.full(
    (candidate_count, 3),
    1.0 / 1000.0,
    dtype=np.float64,
)
phase48_training_records = []

if phase48_device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase48_device)

for specification in phase48_model_grid:
    candidate_index = int(specification["candidate_index"])
    for partition, monitor_slice in zip(
        phase47_partitions,
        phase47_monitor_slices,
    ):
        fold = int(partition["fold"])
        fit_indices = partition["fit_indices"]
        monitor_indices = partition["monitor_indices"]
        run_seed = 480900 + 100 * candidate_index + fold
        phase48_selection_set_seed(run_seed)
        run_started = time.perf_counter()

        fit_weights = phase47_training_weights(
            phase47_groups[fit_indices]
        )
        fit_dataset = Phase47PrefixDataset(
            cache_path=phase48_prefix_paths[fold],
            indices=fit_indices,
            anchor_logit=phase47_anchor_logit[fit_indices],
            labels=phase47_y[fit_indices],
            groups=phase47_groups[fit_indices],
            case_weights=fit_weights,
        )
        monitor_dataset = Phase47PrefixDataset(
            cache_path=phase48_prefix_paths[fold],
            indices=monitor_indices,
            anchor_logit=phase47_anchor_logit[monitor_indices],
            labels=phase47_y[monitor_indices],
            groups=phase47_groups[monitor_indices],
        )
        generator = torch.Generator()
        generator.manual_seed(run_seed)
        fit_loader = DataLoader(
            fit_dataset,
            batch_size=PHASE48_SELECTION_CONFIG["batch_size"],
            shuffle=True,
            num_workers=PHASE48_SELECTION_CONFIG["worker_count"],
            pin_memory=(phase48_device.type == "cuda"),
            drop_last=False,
            generator=generator,
        )
        monitor_loader = DataLoader(
            monitor_dataset,
            batch_size=PHASE48_SELECTION_CONFIG[
                "evaluation_batch_size"
            ],
            shuffle=False,
            num_workers=PHASE48_SELECTION_CONFIG["worker_count"],
            pin_memory=(phase48_device.type == "cuda"),
            drop_last=False,
        )

        model = PHASE48_ADAPTER_CLASS_PRIVATE(
            checkpoint_path=phase48_checkpoint_paths[fold],
            attention_head_count=specification["attention_head_count"],
            hidden_dimension=specification["hidden_dimension"],
            dropout=specification["dropout"],
            residual_cap=specification["residual_cap"],
            train_tail=specification["train_tail"],
            initialization_seed=run_seed,
        ).to(phase48_device)

        tail_parameters = [
            parameter
            for name, parameter in model.named_parameters()
            if parameter.requires_grad
            and (
                name.startswith("tail_block.")
                or name.startswith("encoder_norm.")
            )
        ]
        pooling_parameters = [
            model.attention_queries,
            model.spatial_bias,
        ]
        head_parameters = [
            parameter
            for name, parameter in model.named_parameters()
            if parameter.requires_grad and name.startswith("residual_head.")
        ]
        assert pooling_parameters and head_parameters
        optimizer_groups = [
            {
                "params": pooling_parameters,
                "lr": specification["pooling_learning_rate"],
            },
            {
                "params": head_parameters,
                "lr": specification["head_learning_rate"],
            },
        ]
        if specification["train_tail"]:
            assert tail_parameters
            optimizer_groups.insert(0, {
                "params": tail_parameters,
                "lr": specification["tail_learning_rate"],
            })
        else:
            assert not tail_parameters

        optimizer = torch.optim.AdamW(
            optimizer_groups,
            weight_decay=PHASE48_SELECTION_CONFIG["weight_decay"],
            betas=(0.9, 0.95),
        )
        total_steps = (
            PHASE48_SELECTION_CONFIG["maximum_epochs"] * len(fit_loader)
        )
        warmup_steps = max(
            1,
            int(round(
                PHASE48_SELECTION_CONFIG["warmup_fraction"] * total_steps
            )),
        )
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lambda step, total_steps=total_steps,
            warmup_steps=warmup_steps: phase48_learning_rate_factor(
                step,
                total_steps,
                warmup_steps,
            ),
        )

        anchor_probability = phase47_anchor_probability[monitor_indices]
        best_log_loss = phase47_selection_log_loss(
            phase47_y[monitor_indices],
            anchor_probability,
        )
        best_auroc = phase47_selection_auroc(
            phase47_y[monitor_indices],
            anchor_probability,
        )
        best_epoch = 0
        best_raw = np.zeros(monitor_indices.size, dtype=np.float64)
        best_probability = anchor_probability.copy()
        best_entropy = 1.0
        best_maximum_attention = 1.0 / 1000.0
        epochs_without_improvement = 0
        epoch_history = []

        for epoch in range(1, PHASE48_SELECTION_CONFIG["maximum_epochs"] + 1):
            model.set_training_mode()
            total_bce = 0.0
            total_rank = 0.0
            total_n = 0
            maximum_gradient_norm = 0.0

            for batch in fit_loader:
                prefix = batch["prefix"].to(
                    phase48_device,
                    non_blocking=False,
                ).float()
                anchor = batch["anchor_logit"].to(
                    phase48_device,
                    non_blocking=False,
                )
                labels = batch["label"].to(
                    phase48_device,
                    non_blocking=False,
                )
                groups = batch["group"].to(
                    phase48_device,
                    non_blocking=False,
                )
                case_weight = batch["case_weight"].to(
                    phase48_device,
                    non_blocking=False,
                )

                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=phase48_device.type,
                    dtype=phase48_amp_dtype,
                    enabled=phase48_amp_enabled,
                ):
                    output = model(prefix, anchor)
                element_bce = F.binary_cross_entropy_with_logits(
                    output["updated_logit"].float(),
                    labels.float(),
                    reduction="none",
                )
                bce_loss = torch.sum(
                    element_bce * case_weight.float()
                ) / torch.sum(case_weight.float())
                rank_loss = phase47_within_group_rank_loss(
                    output["updated_logit"].float(),
                    labels.float(),
                    groups,
                )
                loss = (
                    bce_loss
                    + specification["rank_loss_weight"] * rank_loss
                )
                assert torch.isfinite(loss)
                loss.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for parameter in model.parameters()
                        if parameter.requires_grad
                    ],
                    PHASE48_SELECTION_CONFIG["gradient_clip"],
                )
                assert torch.isfinite(gradient_norm)
                optimizer.step()
                scheduler.step()

                batch_n = int(labels.numel())
                total_bce += float(bce_loss.detach().item()) * batch_n
                total_rank += float(rank_loss.detach().item()) * batch_n
                total_n += batch_n
                maximum_gradient_norm = max(
                    maximum_gradient_norm,
                    float(gradient_norm.detach().item()),
                )

            prediction = phase48_predict_attention(
                model,
                monitor_loader,
                monitor_indices,
            )
            monitor_log_loss = phase47_selection_log_loss(
                phase47_y[monitor_indices],
                prediction["probability"],
            )
            monitor_auroc = phase47_selection_auroc(
                phase47_y[monitor_indices],
                prediction["probability"],
            )
            improved = monitor_log_loss < (
                best_log_loss
                - PHASE48_SELECTION_CONFIG[
                    "early_stopping_minimum_gain"
                ]
            )
            if improved:
                best_log_loss = float(monitor_log_loss)
                best_auroc = float(monitor_auroc)
                best_epoch = int(epoch)
                best_raw = prediction["raw_residual"].copy()
                best_probability = prediction["probability"].copy()
                best_entropy = float(
                    prediction["mean_normalized_attention_entropy"]
                )
                best_maximum_attention = float(
                    prediction["mean_maximum_attention_weight"]
                )
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            epoch_history.append({
                "epoch": int(epoch),
                "train_bce": float(total_bce / total_n),
                "train_rank": float(total_rank / total_n),
                "monitor_log_loss": float(monitor_log_loss),
                "monitor_auroc": float(monitor_auroc),
                "attention_entropy": float(
                    prediction["mean_normalized_attention_entropy"]
                ),
                "maximum_attention_weight": float(
                    prediction["mean_maximum_attention_weight"]
                ),
                "improved": bool(improved),
                "maximum_gradient_norm": float(maximum_gradient_norm),
            })
            if (
                epoch >= PHASE48_SELECTION_CONFIG["minimum_epochs"]
                and epochs_without_improvement
                >= PHASE48_SELECTION_CONFIG["early_stopping_patience"]
            ):
                break

        phase48_candidate_monitor_raw[
            candidate_index,
            monitor_slice,
        ] = best_raw
        phase48_candidate_monitor_probability[
            candidate_index,
            monitor_slice,
        ] = best_probability
        phase48_candidate_epoch_counts[candidate_index, fold] = best_epoch
        phase48_candidate_attention_entropy[
            candidate_index,
            fold,
        ] = best_entropy
        phase48_candidate_maximum_attention[
            candidate_index,
            fold,
        ] = best_maximum_attention
        phase48_training_records.append({
            "candidate_index": int(candidate_index),
            "fold": int(fold),
            "seed": int(run_seed),
            "best_epoch": int(best_epoch),
            "best_monitor_log_loss": float(best_log_loss),
            "best_monitor_auroc": float(best_auroc),
            "best_attention_entropy": float(best_entropy),
            "best_maximum_attention_weight": float(
                best_maximum_attention
            ),
            "epochs_executed": int(len(epoch_history)),
            "final_epoch": epoch_history[-1],
            "elapsed_seconds": round(time.perf_counter() - run_started, 3),
        })
        print(
            f"Phase48 candidate {candidate_index + 1}/{candidate_count}, "
            f"fold {fold + 1}/3: best_epoch={best_epoch}, "
            f"log_loss={best_log_loss:.6f}, auroc={best_auroc:.6f}, "
            f"entropy={best_entropy:.6f}"
        )

        del scheduler
        del optimizer
        del model
        del fit_loader
        del monitor_loader
        del fit_dataset
        del monitor_dataset
        gc.collect()
        if phase48_device.type == "cuda":
            torch.cuda.empty_cache()

assert np.all(np.isfinite(phase48_candidate_monitor_raw))
assert np.all(np.isfinite(phase48_candidate_monitor_probability))


def phase48_model_summary(candidate_index):
    probability = phase48_candidate_monitor_probability[candidate_index]
    metrics = phase47_complete_monitor_metrics(probability)
    fold_wins = int(sum(
        record["log_loss"] < phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    worst_fold_regret = float(max(
        record["log_loss"] - phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    maximum_group_harm = float(max(
        record["log_loss"] - phase47_anchor_group_loss[record["group"]]
        for record in metrics["group_metrics"]
    ))
    screen_safe = bool(
        worst_fold_regret
        <= PHASE48_SELECTION_CONFIG["maximum_model_fold_regret"]
        and maximum_group_harm
        <= PHASE48_SELECTION_CONFIG["maximum_model_group_harm"]
    )
    return {
        "candidate_index": int(candidate_index),
        "specification": dict(phase48_model_grid[candidate_index]),
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        "log_loss_gain": float(
            phase47_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": float(
            metrics["auroc"] - phase47_anchor_metrics["auroc"]
        ),
        "fold_wins": fold_wins,
        "worst_fold_regret": worst_fold_regret,
        "maximum_group_harm": maximum_group_harm,
        "screen_safe": screen_safe,
        "selected_epoch_counts": [
            int(value)
            for value in phase48_candidate_epoch_counts[candidate_index]
        ],
        "attention_entropy_by_fold": [
            float(value)
            for value in phase48_candidate_attention_entropy[candidate_index]
        ],
        "maximum_attention_weight_by_fold": [
            float(value)
            for value in phase48_candidate_maximum_attention[candidate_index]
        ],
        "fold_metrics": metrics["fold_metrics"],
    }


phase48_model_summaries = [
    phase48_model_summary(candidate_index)
    for candidate_index in range(candidate_count)
]
phase48_safe_model_summaries = [
    record for record in phase48_model_summaries if record["screen_safe"]
]
phase48_model_screen_fallback = not bool(phase48_safe_model_summaries)
phase48_model_selection_pool = (
    phase48_safe_model_summaries
    if phase48_safe_model_summaries
    else phase48_model_summaries
)
phase48_raw_model_best = min(
    phase48_model_selection_pool,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["candidate_index"],
    ),
)
phase48_model_band = [
    record
    for record in phase48_model_selection_pool
    if record["monitor_log_loss"]
    <= phase48_raw_model_best["monitor_log_loss"]
    + PHASE48_SELECTION_CONFIG["model_log_loss_band"]
]
phase48_selected_model = max(
    phase48_model_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["candidate_index"],
    ),
)
phase48_selected_candidate_index = int(
    phase48_selected_model["candidate_index"]
)
phase48_selected_raw = phase48_candidate_monitor_raw[
    phase48_selected_candidate_index
].copy()


def phase48_gate_summary(specification):
    if specification["scale"] == 0.0:
        update = np.zeros_like(phase48_selected_raw)
    else:
        uncertainty = np.power(
            4.0
            * phase47_monitor_anchor_probability
            * (1.0 - phase47_monitor_anchor_probability),
            specification["uncertainty_exponent"],
        )
        update = (
            specification["scale"]
            * uncertainty
            * np.clip(
                phase48_selected_raw,
                -specification["residual_cap"],
                specification["residual_cap"],
            )
        )
    probability = phase47_selection_sigmoid(
        phase47_monitor_anchor_logit + update
    )
    metrics = phase47_complete_monitor_metrics(probability)
    fold_wins = int(sum(
        record["log_loss"] < phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    worst_fold_regret = float(max(
        record["log_loss"] - phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    maximum_group_harm = float(max(
        record["log_loss"] - phase47_anchor_group_loss[record["group"]]
        for record in metrics["group_metrics"]
    ))
    is_zero = specification["scale"] == 0.0
    safe = bool(
        is_zero
        or (
            fold_wins >= PHASE48_SELECTION_CONFIG["minimum_gate_fold_wins"]
            and worst_fold_regret
            <= PHASE48_SELECTION_CONFIG["maximum_gate_fold_regret"]
            and maximum_group_harm
            <= PHASE48_SELECTION_CONFIG["maximum_gate_group_harm"]
        )
    )
    return {
        "gate_index": int(specification["gate_index"]),
        "specification": dict(specification),
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        "log_loss_gain": float(
            phase47_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": float(
            metrics["auroc"] - phase47_anchor_metrics["auroc"]
        ),
        "fold_wins": fold_wins,
        "worst_fold_regret": worst_fold_regret,
        "maximum_group_harm": maximum_group_harm,
        "mean_absolute_update": float(np.mean(np.abs(update))),
        "q95_absolute_update": float(np.quantile(np.abs(update), 0.95)),
        "safe": safe,
        "fold_metrics": metrics["fold_metrics"],
    }


phase48_gate_summaries = [
    phase48_gate_summary(specification)
    for specification in phase48_gate_grid
]
phase48_safe_gate_summaries = [
    record for record in phase48_gate_summaries if record["safe"]
]
assert phase48_safe_gate_summaries
phase48_raw_gate_best = min(
    phase48_safe_gate_summaries,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["gate_index"],
    ),
)
phase48_gate_band = [
    record
    for record in phase48_safe_gate_summaries
    if record["monitor_log_loss"]
    <= phase48_raw_gate_best["monitor_log_loss"]
    + PHASE48_SELECTION_CONFIG["gate_log_loss_band"]
]
phase48_provisional_gate = max(
    phase48_gate_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["gate_index"],
    ),
)

phase48_advance_by_log_loss = bool(
    phase48_provisional_gate["log_loss_gain"]
    >= PHASE48_SELECTION_CONFIG["minimum_advance_log_loss_gain"]
    and phase48_provisional_gate["auroc_gain"]
    >= -PHASE48_SELECTION_CONFIG["maximum_advance_auroc_deficit"]
)
phase48_advance_by_auroc = bool(
    phase48_provisional_gate["auroc_gain"]
    >= PHASE48_SELECTION_CONFIG["minimum_advance_auroc_gain"]
    and phase48_provisional_gate["log_loss_gain"]
    >= -PHASE48_SELECTION_CONFIG["maximum_advance_log_loss_excess"]
)
phase48_gate_advanced = bool(
    phase48_provisional_gate["gate_index"] != 0
    and (phase48_advance_by_log_loss or phase48_advance_by_auroc)
)
phase48_selected_gate = (
    phase48_provisional_gate
    if phase48_gate_advanced
    else phase48_gate_summaries[0]
)

if phase48_device.type == "cuda":
    phase48_peak_vram_mb = float(
        torch.cuda.max_memory_allocated(phase48_device) / (1024 ** 2)
    )
else:
    phase48_peak_vram_mb = 0.0

PHASE48_MODEL_GRID_PRIVATE = copy.deepcopy(phase48_model_grid)
PHASE48_SELECTED_MODEL_PRIVATE = copy.deepcopy(phase48_selected_model)
PHASE48_SELECTED_GATE_PRIVATE = copy.deepcopy(phase48_selected_gate)
PHASE48_SELECTED_EPOCH_COUNTS_PRIVATE = tuple(
    phase48_selected_model["selected_epoch_counts"]
)
PHASE48_GATE_ADVANCED_PRIVATE = bool(phase48_gate_advanced)

phase48_selection_report = {
    "phase": "phase48_spatial_attention_anchor_preserving_selection",
    "status": (
        "model_and_gate_frozen_for_outer_evaluation"
        if phase48_gate_advanced
        else "no_update_selected_stop_before_outer_evaluation"
    ),
    "anchor_monitor": {
        "log_loss": float(phase47_anchor_metrics["log_loss"]),
        "auroc": float(phase47_anchor_metrics["auroc"]),
    },
    "training": {
        "model_specification_count": candidate_count,
        "fold_count": 3,
        "trained_model_count": int(candidate_count * 3),
        "maximum_epochs": int(
            PHASE48_SELECTION_CONFIG["maximum_epochs"]
        ),
        "batch_size": int(PHASE48_SELECTION_CONFIG["batch_size"]),
        "complete_factorial_grid": {
            "attention_head_counts": [1, 2, 4],
            "tail_modes": ["frozen", "trainable"],
            "rank_loss_weights": [0.0, 0.05],
        },
        "group_balanced_case_weights": True,
        "epoch_zero_phase39_identity_allowed": True,
        "records": phase48_training_records,
    },
    "model_selection": {
        "screen_safe_candidate_count": len(
            phase48_safe_model_summaries
        ),
        "fallback_to_all_candidates": bool(
            phase48_model_screen_fallback
        ),
        "raw_log_loss_best": phase48_raw_model_best,
        "selected": phase48_selected_model,
    },
    "gate_selection": {
        "candidate_count": len(phase48_gate_summaries),
        "safe_candidate_count": len(phase48_safe_gate_summaries),
        "raw_log_loss_best": phase48_raw_gate_best,
        "provisional": phase48_provisional_gate,
        "selected": phase48_selected_gate,
        "advance_by_log_loss": phase48_advance_by_log_loss,
        "advance_by_auroc": phase48_advance_by_auroc,
        "gate_advanced": phase48_gate_advanced,
    },
    "selection_rule": (
        "maximum monitor AUROC within 0.002 log-loss of the safe shared "
        "model minimum, followed by maximum AUROC within 0.001 log-loss "
        "of the safe bounded-gate minimum"
    ),
    "formula_if_advanced": (
        "z48 = z39 + scale * uncertainty(z39)^gamma * "
        "clip(r48_raw, -cap, cap)"
    ),
    "shared_model_specification_across_folds": True,
    "shared_gate_across_folds": True,
    "fit_labels_used_for_training": True,
    "monitor_labels_used_for_selection": True,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_prefix_cache_read": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "peak_vram_mb": phase48_peak_vram_mb,
    "elapsed_seconds": round(
        time.perf_counter() - phase48_selection_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE48_SELECTION")
print(json.dumps(phase48_selection_report, indent=2))
print("END SANITIZED_PHASE48_SELECTION")

Phase48 candidate 1/12, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, entropy=1.000000
Phase48 candidate 1/12, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867, entropy=1.000000
Phase48 candidate 1/12, fold 3/3: best_epoch=4, log_loss=0.234122, auroc=0.971736, entropy=0.999991
Phase48 candidate 2/12, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, entropy=1.000000
Phase48 candidate 2/12, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867, entropy=1.000000
Phase48 candidate 2/12, fold 3/3: best_epoch=1, log_loss=0.235558, auroc=0.972257, entropy=0.999994
Phase48 candidate 3/12, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, entropy=1.000000
Phase48 candidate 3/12, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867, entropy=1.000000
Phase48 candidate 3/12, fold 3/3: best_epoch=3, log_loss=0.235435, auroc=0.971631, entropy=0.999998
Phase48 candidate 4/12, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, entropy=1.000000


In [60]:
# Phase49 Cell 147A 

import copy
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


phase49_contract_started = time.perf_counter()

PHASE49_CONFIG = {
    "seed": 490701,
    "patch_size": 8,
    "grid_size": 10,
    "token_count": 1000,
    "encoder_dimension": 192,
    "feature_batch_size": 8,
    "patch_q90_index": 459,
    "mean_weight": 0.5,
    "q90_weight": 0.5,
    "contract_pool_mode": "topk",
    "contract_top_k": 64,
    "contract_temperature": 0.25,
    "train_tail": True,
    "head_hidden_dimension": 64,
    "head_dropout": 0.10,
    "residual_cap": 0.50,
    "tail_learning_rate": 3.0e-6,
    "head_learning_rate": 2.0e-4,
    "weight_decay": 0.02,
    "contract_batch_size": 2,
}

required_names = [
    "PHASE47_ADAPTER_CLASS_PRIVATE",
    "PHASE47_PREFIX_CACHE_PATHS_PRIVATE",
    "PHASE45_HIGHRES_CACHE_PATH_PRIVATE",
    "PHASE45_DEVICE",
    "PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE",
    "PHASE39_OOF_PRIVATE",
    "PHASE44_Y_PRIVATE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase49 uptake-guided contract is missing prior state.",
    "missing": missing_names,
}


def phase49_set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def phase49_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase49_gradient_contract(parameters):
    tensor_count = 0
    squared_norm = 0.0
    all_finite = True
    for parameter in parameters:
        if parameter.grad is None:
            continue
        gradient = parameter.grad.detach().float()
        tensor_count += 1
        all_finite = all_finite and bool(
            torch.isfinite(gradient).all().item()
        )
        squared_norm += float(torch.sum(gradient * gradient).item())
    return {
        "tensor_count": int(tensor_count),
        "norm": float(squared_norm ** 0.5),
        "all_finite": bool(all_finite),
    }


phase49_highres = np.load(
    PHASE45_HIGHRES_CACHE_PATH_PRIVATE,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase49_highres.shape == (1362, 80, 80, 80)
assert phase49_highres.dtype == np.float16

phase49_uptake_cache_path = Path(
    "/kaggle/working/phase49_patch_uptake_score_float32.npy"
)
phase49_uptake_metadata_path = Path(
    "/kaggle/working/phase49_patch_uptake_metadata.json"
)


def phase49_existing_uptake_cache_valid(path):
    if not path.is_file():
        return False
    try:
        matrix = np.load(path, mmap_mode="r", allow_pickle=False)
        valid = (
            matrix.shape == (1362, 1000)
            and matrix.dtype == np.float32
        )
        if valid:
            sample = np.asarray(
                matrix[[0, 681, 1361], ::113],
                dtype=np.float64,
            )
            valid = bool(np.all(np.isfinite(sample)))
        del matrix
        return bool(valid)
    except (OSError, ValueError):
        return False


phase49_uptake_cache_reused = phase49_existing_uptake_cache_valid(
    phase49_uptake_cache_path
)
if not phase49_uptake_cache_reused:
    phase49_building_path = phase49_uptake_cache_path.with_suffix(
        ".building.npy"
    )
    if phase49_building_path.exists():
        phase49_building_path.unlink()
    phase49_uptake_output = np.lib.format.open_memmap(
        phase49_building_path,
        mode="w+",
        dtype=np.float32,
        shape=(1362, 1000),
    )
    batch_size = int(PHASE49_CONFIG["feature_batch_size"])
    for start in range(0, 1362, batch_size):
        stop = min(start + batch_size, 1362)
        volume = np.asarray(
            phase49_highres[start:stop],
            dtype=np.float32,
        ).copy()
        patches = volume.reshape(
            stop - start,
            10,
            8,
            10,
            8,
            10,
            8,
        ).transpose(0, 1, 3, 5, 2, 4, 6).reshape(
            stop - start,
            1000,
            512,
        )
        patch_mean = np.mean(patches, axis=-1, dtype=np.float64)
        patch_q90 = np.partition(
            patches,
            PHASE49_CONFIG["patch_q90_index"],
            axis=-1,
        )[:, :, PHASE49_CONFIG["patch_q90_index"]].astype(
            np.float64,
            copy=False,
        )
        raw_score = (
            PHASE49_CONFIG["mean_weight"] * patch_mean
            + PHASE49_CONFIG["q90_weight"] * patch_q90
        )
        median = np.median(raw_score, axis=1, keepdims=True)
        q25 = np.quantile(raw_score, 0.25, axis=1, keepdims=True)
        q75 = np.quantile(raw_score, 0.75, axis=1, keepdims=True)
        scale = np.maximum(q75 - q25, 1.0e-6)
        normalized_score = np.clip(
            (raw_score - median) / scale,
            -10.0,
            10.0,
        )
        assert np.all(np.isfinite(normalized_score))
        phase49_uptake_output[start:stop] = normalized_score.astype(
            np.float32,
            copy=False,
        )
        if stop == 1362 or stop // 128 > start // 128:
            print(f"Phase49 uptake cache: {stop}/1362")
        del volume
        del patches
        del patch_mean
        del patch_q90
        del raw_score
        del median
        del q25
        del q75
        del scale
        del normalized_score

    phase49_uptake_output.flush()
    del phase49_uptake_output
    os.replace(phase49_building_path, phase49_uptake_cache_path)

phase49_uptake_score = np.load(
    phase49_uptake_cache_path,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase49_uptake_score.shape == (1362, 1000)
assert phase49_uptake_score.dtype == np.float32
phase49_sampled_score = np.asarray(
    phase49_uptake_score[::37],
    dtype=np.float64,
)
assert np.all(np.isfinite(phase49_sampled_score))

phase49_metadata = {
    "phase": "phase49_label_free_patch_uptake_score",
    "status": "complete",
    "shape": [1362, 1000],
    "dtype": "float32",
    "patch_size": 8,
    "grid_size": [10, 10, 10],
    "score": "per_case_robust_scaled_0p5_patch_mean_plus_0p5_patch_q90",
    "cache_file": phase49_uptake_cache_path.name,
    "complete": True,
}
phase49_metadata_building = phase49_uptake_metadata_path.with_suffix(
    ".building.json"
)
with open(phase49_metadata_building, "w", encoding="utf-8") as handle:
    json.dump(phase49_metadata, handle, indent=2, sort_keys=True)
os.replace(phase49_metadata_building, phase49_uptake_metadata_path)


class Phase49UptakeGuidedAdapter(nn.Module):
    """High-uptake, global, and uptake-contrast token pooling."""

    def __init__(
        self,
        checkpoint_path,
        pool_mode="topk",
        top_k=64,
        temperature=0.25,
        hidden_dimension=64,
        dropout=0.10,
        residual_cap=0.50,
        train_tail=True,
    ):
        super().__init__()
        assert pool_mode in {"topk", "softmax"}
        self.pool_mode = str(pool_mode)
        self.top_k = int(top_k)
        self.temperature = float(temperature)
        self.residual_cap = float(residual_cap)
        self.train_tail = bool(train_tail)
        assert 1 <= self.top_k <= 1000
        assert self.temperature > 0.0
        assert self.residual_cap > 0.0

        source = PHASE47_ADAPTER_CLASS_PRIVATE(
            encoder_checkpoint_path=checkpoint_path,
            unfrozen_encoder_block_count=1,
            hidden_dimension=hidden_dimension,
            dropout=dropout,
            residual_cap=residual_cap,
        )
        self.tail_block = copy.deepcopy(
            source.backbone.encoder.layers[-1]
        )
        self.encoder_norm = copy.deepcopy(source.backbone.encoder_norm)
        del source

        for parameter in self.tail_block.parameters():
            parameter.requires_grad_(self.train_tail)
        for parameter in self.encoder_norm.parameters():
            parameter.requires_grad_(self.train_tail)

        # Per view: high-uptake pool, global pool, and their contrast.
        self.pooled_dimension = 3 * 192
        self.representation_dimension = 2 * self.pooled_dimension
        self.residual_head = nn.Sequential(
            nn.LayerNorm(self.representation_dimension),
            nn.Linear(self.representation_dimension, int(hidden_dimension)),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(int(hidden_dimension), 1),
        )
        nn.init.zeros_(self.residual_head[-1].weight)
        nn.init.zeros_(self.residual_head[-1].bias)

    def set_training_mode(self):
        self.train()
        if not self.train_tail:
            self.tail_block.eval()
            self.encoder_norm.eval()

    def encoded_tokens(self, penultimate_tokens):
        if self.train_tail:
            tokens = self.tail_block(penultimate_tokens)
            tokens = self.encoder_norm(tokens)
        else:
            with torch.no_grad():
                tokens = self.tail_block(penultimate_tokens)
                tokens = self.encoder_norm(tokens)
        assert tokens.ndim == 3 and tokens.shape[1:] == (1000, 192)
        return tokens

    def uptake_weights(self, score):
        assert score.ndim == 2 and score.shape[1] == 1000
        if self.pool_mode == "topk":
            indices = torch.topk(
                score,
                k=self.top_k,
                dim=1,
                largest=True,
                sorted=False,
            ).indices
            weight = torch.zeros_like(score, dtype=torch.float32)
            weight.scatter_(
                1,
                indices,
                1.0 / float(self.top_k),
            )
        else:
            weight = torch.softmax(
                score.float() / self.temperature,
                dim=1,
            )
        return weight

    def pool_view(self, tokens, score):
        weight = self.uptake_weights(score)
        high_uptake = torch.sum(
            tokens * weight.to(tokens.dtype).unsqueeze(-1),
            dim=1,
        )
        global_mean = torch.mean(tokens, dim=1)
        uptake_contrast = high_uptake - global_mean
        pooled = torch.cat(
            [high_uptake, global_mean, uptake_contrast],
            dim=1,
        )
        return pooled, weight

    @staticmethod
    def reflect_score(score):
        return torch.flip(
            score.reshape(-1, 10, 10, 10),
            dims=(1,),
        ).reshape(-1, 1000)

    def invariant_representation(self, prefix, original_score):
        assert prefix.ndim == 4 and prefix.shape[1:] == (2, 1000, 192)
        assert original_score.ndim == 2 and original_score.shape[1] == 1000
        original_tokens = self.encoded_tokens(prefix[:, 0])
        reflected_tokens = self.encoded_tokens(prefix[:, 1])
        original_pool, original_weight = self.pool_view(
            original_tokens,
            original_score,
        )
        reflected_pool, reflected_weight = self.pool_view(
            reflected_tokens,
            self.reflect_score(original_score),
        )
        symmetric = 0.5 * (original_pool + reflected_pool)
        absolute_antisymmetric = torch.abs(
            0.5 * (original_pool - reflected_pool)
        )
        representation = torch.cat(
            [symmetric, absolute_antisymmetric],
            dim=1,
        )
        return {
            "representation": representation,
            "original_weight": original_weight,
            "reflected_weight": reflected_weight,
        }

    def forward(self, prefix, original_score, anchor_logit):
        pooled = self.invariant_representation(prefix, original_score)
        raw_residual = self.residual_head(
            pooled["representation"].float()
        ).squeeze(1)
        bounded_residual = self.residual_cap * torch.tanh(
            raw_residual / self.residual_cap
        )
        updated_logit = anchor_logit.float().reshape(-1) + bounded_residual
        return {
            "probability": torch.sigmoid(updated_logit),
            "updated_logit": updated_logit,
            "raw_residual": raw_residual,
            "bounded_residual": bounded_residual,
            **pooled,
        }


phase49_set_seed(PHASE49_CONFIG["seed"])
phase49_device = torch.device(PHASE45_DEVICE)
phase49_amp_enabled = phase49_device.type == "cuda"
phase49_amp_dtype = (
    torch.bfloat16
    if phase49_amp_enabled and torch.cuda.is_bf16_supported()
    else torch.float16
)
if phase49_device.type == "cuda":
    assert phase49_amp_dtype == torch.bfloat16

phase49_checkpoint_path = Path(
    "/kaggle/working/phase45_private_checkpoint/phase45_fold_0_encoder.pt"
)
phase49_prefix_path = Path(PHASE47_PREFIX_CACHE_PATHS_PRIVATE[0])
assert phase49_checkpoint_path.is_file()
assert phase49_prefix_path.is_file()
phase49_prefix = np.load(
    phase49_prefix_path,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase49_prefix.shape == (2, 1362, 1000, 192)

phase49_fit_indices = np.asarray(
    PHASE45_DOWNSTREAM_PARTITIONS_PRIVATE[0]["fit_indices"],
    dtype=np.int64,
)
phase49_contract_indices = phase49_fit_indices[
    :PHASE49_CONFIG["contract_batch_size"]
]
phase49_contract_prefix = torch.from_numpy(np.array(
    phase49_prefix[:, phase49_contract_indices],
    dtype=np.float32,
    copy=True,
)).permute(1, 0, 2, 3).contiguous().to(phase49_device)
phase49_contract_score = torch.from_numpy(np.array(
    phase49_uptake_score[phase49_contract_indices],
    dtype=np.float32,
    copy=True,
)).to(phase49_device)
phase49_anchor_logit_all = phase49_logit(PHASE39_OOF_PRIVATE)
phase49_contract_anchor = torch.from_numpy(
    phase49_anchor_logit_all[phase49_contract_indices].astype(np.float32)
).to(phase49_device)
phase49_contract_target = torch.from_numpy(
    np.asarray(PHASE44_Y_PRIVATE, dtype=np.float32).reshape(-1)[
        phase49_contract_indices
    ].copy()
).to(phase49_device)
assert phase49_contract_prefix.shape == (2, 2, 1000, 192)
assert phase49_contract_score.shape == (2, 1000)

if phase49_device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase49_device)

phase49_model = Phase49UptakeGuidedAdapter(
    checkpoint_path=phase49_checkpoint_path,
    pool_mode=PHASE49_CONFIG["contract_pool_mode"],
    top_k=PHASE49_CONFIG["contract_top_k"],
    temperature=PHASE49_CONFIG["contract_temperature"],
    hidden_dimension=PHASE49_CONFIG["head_hidden_dimension"],
    dropout=PHASE49_CONFIG["head_dropout"],
    residual_cap=PHASE49_CONFIG["residual_cap"],
    train_tail=PHASE49_CONFIG["train_tail"],
).to(phase49_device)

phase49_model.eval()
with torch.inference_mode(), torch.autocast(
    device_type=phase49_device.type,
    dtype=phase49_amp_dtype,
    enabled=phase49_amp_enabled,
):
    phase49_initial_output = phase49_model(
        phase49_contract_prefix,
        phase49_contract_score,
        phase49_contract_anchor,
    )
    phase49_swapped_output = phase49_model(
        phase49_contract_prefix[:, [1, 0]],
        Phase49UptakeGuidedAdapter.reflect_score(
            phase49_contract_score
        ),
        phase49_contract_anchor,
    )

phase49_identity_logit_error = float(torch.max(torch.abs(
    phase49_initial_output["updated_logit"].float()
    - phase49_contract_anchor.float()
)).item())
phase49_identity_probability_error = float(torch.max(torch.abs(
    phase49_initial_output["probability"].float()
    - torch.sigmoid(phase49_contract_anchor.float())
)).item())
phase49_swap_representation_error = float(torch.max(torch.abs(
    phase49_initial_output["representation"].float()
    - phase49_swapped_output["representation"].float()
)).item())
phase49_swap_probability_error = float(torch.max(torch.abs(
    phase49_initial_output["probability"].float()
    - phase49_swapped_output["probability"].float()
)).item())
assert phase49_identity_logit_error == 0.0
assert phase49_identity_probability_error == 0.0
assert phase49_swap_representation_error == 0.0
assert phase49_swap_probability_error == 0.0
assert phase49_initial_output["representation"].shape == (2, 1152)

phase49_weight = phase49_initial_output["original_weight"].float()
phase49_weight_sum_error = float(torch.max(torch.abs(
    phase49_weight.sum(dim=1) - 1.0
)).item())
phase49_support_count = torch.sum(phase49_weight > 0.0, dim=1)
phase49_weight_entropy = -torch.sum(
    phase49_weight
    * torch.log(torch.clamp(phase49_weight, min=1.0e-12)),
    dim=1,
) / math.log(1000.0)
assert phase49_weight_sum_error <= 1.0e-7
assert torch.all(
    phase49_support_count == PHASE49_CONFIG["contract_top_k"]
)

tail_parameters = list(phase49_model.tail_block.parameters()) + list(
    phase49_model.encoder_norm.parameters()
)
head_parameters = list(phase49_model.residual_head.parameters())
phase49_optimizer = torch.optim.AdamW(
    [
        {
            "params": tail_parameters,
            "lr": PHASE49_CONFIG["tail_learning_rate"],
        },
        {
            "params": head_parameters,
            "lr": PHASE49_CONFIG["head_learning_rate"],
        },
    ],
    weight_decay=PHASE49_CONFIG["weight_decay"],
    betas=(0.9, 0.95),
)

phase49_model.set_training_mode()
phase49_optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=phase49_device.type,
    dtype=phase49_amp_dtype,
    enabled=phase49_amp_enabled,
):
    phase49_first_output = phase49_model(
        phase49_contract_prefix,
        phase49_contract_score,
        phase49_contract_anchor,
    )
phase49_first_loss = F.binary_cross_entropy_with_logits(
    phase49_first_output["updated_logit"].float(),
    phase49_contract_target.float(),
)
phase49_first_loss.backward()
phase49_first_tail_gradient = phase49_gradient_contract(tail_parameters)
phase49_first_head_gradient = phase49_gradient_contract(head_parameters)
assert phase49_first_head_gradient["all_finite"]
assert phase49_first_head_gradient["norm"] > 0.0
phase49_optimizer.step()

phase49_optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=phase49_device.type,
    dtype=phase49_amp_dtype,
    enabled=phase49_amp_enabled,
):
    phase49_second_output = phase49_model(
        phase49_contract_prefix,
        phase49_contract_score,
        phase49_contract_anchor,
    )
phase49_second_loss = F.binary_cross_entropy_with_logits(
    phase49_second_output["updated_logit"].float(),
    phase49_contract_target.float(),
)
phase49_second_loss.backward()
phase49_second_tail_gradient = phase49_gradient_contract(tail_parameters)
phase49_second_head_gradient = phase49_gradient_contract(head_parameters)
assert phase49_second_tail_gradient["all_finite"]
assert phase49_second_head_gradient["all_finite"]
assert phase49_second_tail_gradient["norm"] > 0.0
assert phase49_second_head_gradient["norm"] > 0.0

if phase49_device.type == "cuda":
    torch.cuda.synchronize(phase49_device)
    phase49_peak_vram_mb = float(
        torch.cuda.max_memory_allocated(phase49_device) / (1024 ** 2)
    )
else:
    phase49_peak_vram_mb = 0.0

PHASE49_ADAPTER_CLASS_PRIVATE = Phase49UptakeGuidedAdapter
PHASE49_UPTAKE_CACHE_PATH_PRIVATE = phase49_uptake_cache_path
PHASE49_CONFIG_PRIVATE = dict(PHASE49_CONFIG)

phase49_contract_report = {
    "phase": "phase49_label_free_uptake_guided_token_pooling_contract",
    "status": "accepted",
    "motivation": {
        "phase48_gate_advanced": False,
        "phase48_attention_entropy": "approximately_1p0",
        "phase48_maximum_attention_weight": "approximately_1_over_1000",
        "controlled_change": (
            "learned_near_uniform_attention_to_label_free_uptake_guidance"
        ),
    },
    "uptake_score_cache": {
        "file": phase49_uptake_cache_path.name,
        "shape": list(phase49_uptake_score.shape),
        "dtype": str(phase49_uptake_score.dtype),
        "cache_reused": bool(phase49_uptake_cache_reused),
        "size_mb": float(
            phase49_uptake_cache_path.stat().st_size / (1024 ** 2)
        ),
        "sampled_quantiles": {
            "q01": float(np.quantile(phase49_sampled_score, 0.01)),
            "q05": float(np.quantile(phase49_sampled_score, 0.05)),
            "q50": float(np.quantile(phase49_sampled_score, 0.50)),
            "q95": float(np.quantile(phase49_sampled_score, 0.95)),
            "q99": float(np.quantile(phase49_sampled_score, 0.99)),
        },
        "all_sampled_values_finite": bool(np.all(np.isfinite(
            phase49_sampled_score
        ))),
    },
    "pooling": {
        "contract_mode": PHASE49_CONFIG["contract_pool_mode"],
        "top_k": int(PHASE49_CONFIG["contract_top_k"]),
        "weight_support_count": [
            int(value) for value in phase49_support_count.cpu().tolist()
        ],
        "weight_sum_maximum_error": phase49_weight_sum_error,
        "mean_normalized_weight_entropy": float(
            phase49_weight_entropy.mean().item()
        ),
        "per_view_components": [
            "high_uptake_pool",
            "global_mean_pool",
            "high_minus_global_contrast",
        ],
        "reflection_invariant_representation_dimension": 1152,
    },
    "identity_and_symmetry": {
        "phase39_logit_error": phase49_identity_logit_error,
        "phase39_probability_error": phase49_identity_probability_error,
        "view_swap_representation_error": (
            phase49_swap_representation_error
        ),
        "view_swap_probability_error": phase49_swap_probability_error,
    },
    "two_step_backward_contract": {
        "first_loss": float(phase49_first_loss.detach().item()),
        "second_loss": float(phase49_second_loss.detach().item()),
        "first_tail_gradient": phase49_first_tail_gradient,
        "first_head_gradient": phase49_first_head_gradient,
        "second_tail_gradient": phase49_second_tail_gradient,
        "second_head_gradient": phase49_second_head_gradient,
    },
    "planned_selection": {
        "top_k_values": [16, 32, 64, 128],
        "softmax_temperatures": [0.10, 0.25, 0.50],
        "tail_modes": ["frozen", "trainable"],
        "epoch_zero_phase39_identity_allowed": True,
        "shared_specification_across_folds": True,
    },
    "execution": {
        "device": str(phase49_device),
        "mixed_precision": (
            str(phase49_amp_dtype).replace("torch.", "")
            if phase49_amp_enabled
            else "disabled"
        ),
        "peak_vram_mb": phase49_peak_vram_mb,
    },
    "labels_used_for_uptake_score": False,
    "fit_labels_used_for_backward_contract": True,
    "monitor_labels_used": False,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_features_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase49_contract_started, 3),
}

print("BEGIN SANITIZED_PHASE49_UPTAKE_CONTRACT")
print(json.dumps(phase49_contract_report, indent=2))
print("END SANITIZED_PHASE49_UPTAKE_CONTRACT")

del phase49_optimizer
del phase49_model
del phase49_contract_prefix
del phase49_contract_score
del phase49_contract_anchor
del phase49_contract_target
if phase49_device.type == "cuda":
    torch.cuda.empty_cache()

Phase49 uptake cache: 128/1362
Phase49 uptake cache: 256/1362
Phase49 uptake cache: 384/1362
Phase49 uptake cache: 512/1362
Phase49 uptake cache: 640/1362
Phase49 uptake cache: 768/1362
Phase49 uptake cache: 896/1362
Phase49 uptake cache: 1024/1362
Phase49 uptake cache: 1152/1362
Phase49 uptake cache: 1280/1362
Phase49 uptake cache: 1362/1362
BEGIN SANITIZED_PHASE49_UPTAKE_CONTRACT
{
  "phase": "phase49_label_free_uptake_guided_token_pooling_contract",
  "status": "accepted",
  "motivation": {
    "phase48_gate_advanced": false,
    "phase48_attention_entropy": "approximately_1p0",
    "phase48_maximum_attention_weight": "approximately_1_over_1000",
    "controlled_change": "learned_near_uniform_attention_to_label_free_uptake_guidance"
  },
  "uptake_score_cache": {
    "file": "phase49_patch_uptake_score_float32.npy",
    "shape": [
      1362,
      1000
    ],
    "dtype": "float32",
    "cache_reused": false,
    "size_mb": 5.19573974609375,
    "sampled_quantiles": {
      "q01": 

In [61]:
# Phase49 Cell 147B

import copy
import gc
import itertools
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


phase49_selection_started = time.perf_counter()

PHASE49_SELECTION_CONFIG = {
    "batch_size": 24,
    "evaluation_batch_size": 32,
    "worker_count": 0,
    "maximum_epochs": 8,
    "minimum_epochs": 4,
    "early_stopping_patience": 3,
    "early_stopping_minimum_gain": 1.0e-4,
    "gradient_clip": 1.0,
    "weight_decay": 0.02,
    "warmup_fraction": 0.10,
    "minimum_learning_rate_fraction": 0.10,
    "model_log_loss_band": 0.002,
    "gate_log_loss_band": 0.001,
    "maximum_model_fold_regret": 0.006,
    "maximum_model_group_harm": 0.015,
    "minimum_gate_fold_wins": 2,
    "maximum_gate_fold_regret": 0.003,
    "maximum_gate_group_harm": 0.010,
    "minimum_advance_log_loss_gain": 0.001,
    "minimum_advance_auroc_gain": 0.001,
    "maximum_advance_auroc_deficit": 0.0002,
    "maximum_advance_log_loss_excess": 0.0005,
}

required_names = [
    "PHASE49_ADAPTER_CLASS_PRIVATE",
    "PHASE49_UPTAKE_CACHE_PATH_PRIVATE",
    "PHASE47_PREFIX_CACHE_PATHS_PRIVATE",
    "phase47_training_weights",
    "phase47_complete_monitor_metrics",
    "phase47_selection_log_loss",
    "phase47_selection_auroc",
    "phase47_selection_sigmoid",
    "phase47_partitions",
    "phase47_y",
    "phase47_groups",
    "phase47_anchor_probability",
    "phase47_anchor_logit",
    "phase47_monitor_indices",
    "phase47_monitor_anchor_probability",
    "phase47_monitor_anchor_logit",
    "phase47_monitor_slices",
    "phase47_anchor_metrics",
    "phase47_anchor_fold_loss",
    "phase47_anchor_group_loss",
    "PHASE45_DEVICE",
]
missing_names = [name for name in required_names if name not in globals()]
assert not missing_names, {
    "message": "Phase49 selection requires the accepted Phase49 contract.",
    "missing": missing_names,
}


def phase49_selection_set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def phase49_learning_rate_factor(step, total_steps, warmup_steps):
    step = int(step)
    total_steps = max(int(total_steps), 1)
    warmup_steps = max(int(warmup_steps), 1)
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(
        max(total_steps - warmup_steps, 1)
    )
    progress = min(max(progress, 0.0), 1.0)
    minimum = PHASE49_SELECTION_CONFIG[
        "minimum_learning_rate_fraction"
    ]
    return float(
        minimum
        + (1.0 - minimum)
        * 0.5
        * (1.0 + math.cos(math.pi * progress))
    )


phase49_device = torch.device(PHASE45_DEVICE)
phase49_amp_enabled = phase49_device.type == "cuda"
phase49_amp_dtype = (
    torch.bfloat16
    if phase49_amp_enabled and torch.cuda.is_bf16_supported()
    else torch.float16
)
if phase49_device.type == "cuda":
    assert phase49_amp_dtype == torch.bfloat16

phase49_prefix_paths = tuple(
    Path(path) for path in PHASE47_PREFIX_CACHE_PATHS_PRIVATE
)
phase49_checkpoint_paths = tuple(
    Path("/kaggle/working/phase45_private_checkpoint")
    / f"phase45_fold_{fold}_encoder.pt"
    for fold in range(3)
)
phase49_uptake_path = Path(PHASE49_UPTAKE_CACHE_PATH_PRIVATE)
assert len(phase49_prefix_paths) == 3
assert all(path.is_file() for path in phase49_prefix_paths)
assert all(path.is_file() for path in phase49_checkpoint_paths)
assert phase49_uptake_path.is_file()

# Fail before launching the 42 training runs if a cache or restored monitor
# contract is inconsistent with the accepted Phase47/Phase49 state.
for phase49_prefix_path in phase49_prefix_paths:
    phase49_prefix_contract = np.load(
        phase49_prefix_path,
        mmap_mode="r",
        allow_pickle=False,
    )
    assert phase49_prefix_contract.shape == (2, 1362, 1000, 192), {
        "message": "Unexpected Phase47 prefix-cache shape.",
        "file": phase49_prefix_path.name,
        "shape": list(phase49_prefix_contract.shape),
    }
    assert phase49_prefix_contract.dtype == np.float16
    del phase49_prefix_contract

phase49_uptake_contract = np.load(
    phase49_uptake_path,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase49_uptake_contract.shape == (1362, 1000), {
    "message": "Unexpected Phase49 uptake-cache shape.",
    "shape": list(phase49_uptake_contract.shape),
}
assert phase49_uptake_contract.dtype == np.float32
del phase49_uptake_contract

assert np.asarray(phase47_y).shape == (1362,)
assert np.asarray(phase47_groups).shape == (1362,)
assert np.asarray(phase47_anchor_probability).shape == (1362,)
assert np.asarray(phase47_anchor_logit).shape == (1362,)
assert np.asarray(phase47_monitor_indices).shape == (493,)
assert np.asarray(phase47_monitor_anchor_probability).shape == (493,)
assert np.asarray(phase47_monitor_anchor_logit).shape == (493,)
assert len(phase47_partitions) == 3
assert len(phase47_monitor_slices) == 3
for phase49_expected_fold, (
    phase49_partition,
    phase49_monitor_slice,
) in enumerate(zip(phase47_partitions, phase47_monitor_slices)):
    assert int(phase49_partition["fold"]) == phase49_expected_fold
    phase49_partition_monitor = np.asarray(
        phase49_partition["monitor_indices"],
        dtype=np.int64,
    )
    assert (
        phase49_monitor_slice.stop - phase49_monitor_slice.start
        == phase49_partition_monitor.size
    )
    assert np.array_equal(
        phase47_monitor_indices[phase49_monitor_slice],
        phase49_partition_monitor,
    )


class Phase49PrefixUptakeDataset(Dataset):
    def __init__(
        self,
        prefix_path,
        uptake_path,
        indices,
        anchor_logit,
        labels,
        groups,
        case_weights=None,
    ):
        self.prefix_path = str(prefix_path)
        self.uptake_path = str(uptake_path)
        self.indices = np.asarray(indices, dtype=np.int64).reshape(-1)
        self.anchor_logit = np.asarray(
            anchor_logit,
            dtype=np.float32,
        ).reshape(-1)
        self.labels = np.asarray(labels, dtype=np.float32).reshape(-1)
        self.groups = np.asarray(groups, dtype=np.int64).reshape(-1)
        if case_weights is None:
            self.case_weights = np.ones(self.indices.size, dtype=np.float32)
        else:
            self.case_weights = np.asarray(
                case_weights,
                dtype=np.float32,
            ).reshape(-1)
        assert self.indices.size == self.anchor_logit.size
        assert self.indices.size == self.labels.size
        assert self.indices.size == self.groups.size
        assert self.indices.size == self.case_weights.size
        self._prefix = None
        self._uptake = None

    def __len__(self):
        return int(self.indices.size)

    def _get_prefix(self):
        if self._prefix is None:
            self._prefix = np.load(
                self.prefix_path,
                mmap_mode="r",
                allow_pickle=False,
            )
        return self._prefix

    def _get_uptake(self):
        if self._uptake is None:
            self._uptake = np.load(
                self.uptake_path,
                mmap_mode="r",
                allow_pickle=False,
            )
        return self._uptake

    def __getitem__(self, position):
        case_index = int(self.indices[position])
        prefix = np.array(
            self._get_prefix()[:, case_index],
            dtype=np.float16,
            copy=True,
        )
        uptake = np.array(
            self._get_uptake()[case_index],
            dtype=np.float32,
            copy=True,
        )
        return {
            "prefix": torch.from_numpy(prefix),
            "uptake": torch.from_numpy(uptake),
            "anchor_logit": torch.tensor(
                self.anchor_logit[position],
                dtype=torch.float32,
            ),
            "label": torch.tensor(
                self.labels[position],
                dtype=torch.float32,
            ),
            "group": torch.tensor(
                self.groups[position],
                dtype=torch.int64,
            ),
            "case_weight": torch.tensor(
                self.case_weights[position],
                dtype=torch.float32,
            ),
            "case_index": torch.tensor(case_index, dtype=torch.int64),
        }


@torch.no_grad()
def phase49_predict_uptake(model, loader, expected_indices):
    model.eval()
    raw_parts = []
    bounded_parts = []
    probability_parts = []
    index_parts = []
    entropy_sum = 0.0
    maximum_weight_sum = 0.0
    support_sum = 0.0
    observation_count = 0

    for batch in loader:
        prefix = batch["prefix"].to(
            phase49_device,
            non_blocking=False,
        ).float()
        uptake = batch["uptake"].to(
            phase49_device,
            non_blocking=False,
        ).float()
        anchor = batch["anchor_logit"].to(
            phase49_device,
            non_blocking=False,
        )
        with torch.autocast(
            device_type=phase49_device.type,
            dtype=phase49_amp_dtype,
            enabled=phase49_amp_enabled,
        ):
            output = model(prefix, uptake, anchor)

        raw_parts.append(
            output["raw_residual"].float().cpu().numpy()
        )
        bounded_parts.append(
            output["bounded_residual"].float().cpu().numpy()
        )
        probability_parts.append(
            output["probability"].float().cpu().numpy()
        )
        index_parts.append(batch["case_index"].numpy())

        weight = torch.cat([
            output["original_weight"].float(),
            output["reflected_weight"].float(),
        ], dim=0)
        entropy = -torch.sum(
            weight * torch.log(torch.clamp(weight, min=1.0e-12)),
            dim=1,
        ) / math.log(float(weight.shape[1]))
        batch_n = int(prefix.shape[0])
        entropy_sum += float(entropy.mean().item()) * batch_n
        maximum_weight_sum += float(
            weight.max(dim=1).values.mean().item()
        ) * batch_n
        support_sum += float(
            torch.sum(weight > 0.0, dim=1).float().mean().item()
        ) * batch_n
        observation_count += batch_n

    indices = np.concatenate(index_parts).astype(np.int64, copy=False)
    expected_indices = np.asarray(expected_indices, dtype=np.int64)
    assert np.array_equal(indices, expected_indices)
    raw = np.concatenate(raw_parts).astype(np.float64, copy=False)
    bounded = np.concatenate(bounded_parts).astype(np.float64, copy=False)
    probability = np.concatenate(probability_parts).astype(
        np.float64,
        copy=False,
    )
    assert raw.shape == expected_indices.shape
    assert bounded.shape == expected_indices.shape
    assert probability.shape == expected_indices.shape
    assert observation_count == expected_indices.size
    assert np.all(np.isfinite(raw))
    assert np.all(np.isfinite(bounded))
    assert np.all(np.isfinite(probability))
    return {
        "raw_residual": raw,
        "bounded_residual": bounded,
        "probability": np.clip(probability, 1.0e-7, 1.0 - 1.0e-7),
        "mean_normalized_weight_entropy": float(
            entropy_sum / observation_count
        ),
        "mean_maximum_weight": float(
            maximum_weight_sum / observation_count
        ),
        "mean_support_count": float(support_sum / observation_count),
    }


phase49_pool_specifications = [
    {
        "pool_mode": "topk",
        "top_k": int(top_k),
        "temperature": 0.25,
        "pool_name": f"topk_{top_k}",
    }
    for top_k in [16, 32, 64, 128]
]
phase49_pool_specifications.extend([
    {
        "pool_mode": "softmax",
        "top_k": 64,
        "temperature": float(temperature),
        "pool_name": f"softmax_{str(temperature).replace('.', 'p')}",
    }
    for temperature in [0.10, 0.25, 0.50]
])
assert len(phase49_pool_specifications) == 7

phase49_model_grid = []
for candidate_index, (pool_specification, train_tail) in enumerate(
    itertools.product(phase49_pool_specifications, [False, True])
):
    phase49_model_grid.append({
        "candidate_index": int(candidate_index),
        **dict(pool_specification),
        "train_tail": bool(train_tail),
        "hidden_dimension": 64,
        "dropout": 0.10,
        "residual_cap": 0.50,
        "tail_learning_rate": (3.0e-6 if train_tail else 0.0),
        "head_learning_rate": 2.0e-4,
        "rank_loss_weight": 0.0,
    })
assert len(phase49_model_grid) == 14

phase49_gate_grid = [{
    "gate_index": 0,
    "scale": 0.0,
    "residual_cap": 0.0,
    "uncertainty_exponent": 0.0,
}]
for scale, cap, exponent in itertools.product(
    [0.10, 0.25, 0.50, 0.75, 1.0],
    [0.10, 0.25, 0.50, 1.0],
    [0.0, 0.5, 1.0],
):
    phase49_gate_grid.append({
        "gate_index": int(len(phase49_gate_grid)),
        "scale": float(scale),
        "residual_cap": float(cap),
        "uncertainty_exponent": float(exponent),
    })
assert len(phase49_gate_grid) == 61

candidate_count = len(phase49_model_grid)
phase49_candidate_monitor_raw = np.zeros(
    (candidate_count, 493),
    dtype=np.float64,
)
phase49_candidate_monitor_probability = np.tile(
    phase47_monitor_anchor_probability[None, :],
    (candidate_count, 1),
)
phase49_candidate_epoch_counts = np.zeros(
    (candidate_count, 3),
    dtype=np.int64,
)
phase49_candidate_entropy = np.ones(
    (candidate_count, 3),
    dtype=np.float64,
)
phase49_candidate_maximum_weight = np.full(
    (candidate_count, 3),
    1.0 / 1000.0,
    dtype=np.float64,
)
phase49_candidate_support = np.full(
    (candidate_count, 3),
    1000.0,
    dtype=np.float64,
)
phase49_training_records = []

if phase49_device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase49_device)

for specification in phase49_model_grid:
    candidate_index = int(specification["candidate_index"])
    for partition, monitor_slice in zip(
        phase47_partitions,
        phase47_monitor_slices,
    ):
        fold = int(partition["fold"])
        fit_indices = partition["fit_indices"]
        monitor_indices = partition["monitor_indices"]
        run_seed = 490900 + 100 * candidate_index + fold
        phase49_selection_set_seed(run_seed)
        run_started = time.perf_counter()

        fit_weights = phase47_training_weights(
            phase47_groups[fit_indices]
        )
        fit_dataset = Phase49PrefixUptakeDataset(
            prefix_path=phase49_prefix_paths[fold],
            uptake_path=phase49_uptake_path,
            indices=fit_indices,
            anchor_logit=phase47_anchor_logit[fit_indices],
            labels=phase47_y[fit_indices],
            groups=phase47_groups[fit_indices],
            case_weights=fit_weights,
        )
        monitor_dataset = Phase49PrefixUptakeDataset(
            prefix_path=phase49_prefix_paths[fold],
            uptake_path=phase49_uptake_path,
            indices=monitor_indices,
            anchor_logit=phase47_anchor_logit[monitor_indices],
            labels=phase47_y[monitor_indices],
            groups=phase47_groups[monitor_indices],
        )
        generator = torch.Generator()
        generator.manual_seed(run_seed)
        fit_loader = DataLoader(
            fit_dataset,
            batch_size=PHASE49_SELECTION_CONFIG["batch_size"],
            shuffle=True,
            num_workers=PHASE49_SELECTION_CONFIG["worker_count"],
            pin_memory=(phase49_device.type == "cuda"),
            drop_last=False,
            generator=generator,
        )
        monitor_loader = DataLoader(
            monitor_dataset,
            batch_size=PHASE49_SELECTION_CONFIG[
                "evaluation_batch_size"
            ],
            shuffle=False,
            num_workers=PHASE49_SELECTION_CONFIG["worker_count"],
            pin_memory=(phase49_device.type == "cuda"),
            drop_last=False,
        )

        model = PHASE49_ADAPTER_CLASS_PRIVATE(
            checkpoint_path=phase49_checkpoint_paths[fold],
            pool_mode=specification["pool_mode"],
            top_k=specification["top_k"],
            temperature=specification["temperature"],
            hidden_dimension=specification["hidden_dimension"],
            dropout=specification["dropout"],
            residual_cap=specification["residual_cap"],
            train_tail=specification["train_tail"],
        ).to(phase49_device)

        tail_parameters = [
            parameter
            for name, parameter in model.named_parameters()
            if parameter.requires_grad
            and (
                name.startswith("tail_block.")
                or name.startswith("encoder_norm.")
            )
        ]
        head_parameters = [
            parameter
            for name, parameter in model.named_parameters()
            if parameter.requires_grad and name.startswith("residual_head.")
        ]
        optimizer_groups = [{
            "params": head_parameters,
            "lr": specification["head_learning_rate"],
        }]
        if specification["train_tail"]:
            assert tail_parameters
            optimizer_groups.insert(0, {
                "params": tail_parameters,
                "lr": specification["tail_learning_rate"],
            })
        else:
            assert not tail_parameters
        assert head_parameters

        optimizer = torch.optim.AdamW(
            optimizer_groups,
            weight_decay=PHASE49_SELECTION_CONFIG["weight_decay"],
            betas=(0.9, 0.95),
        )
        total_steps = (
            PHASE49_SELECTION_CONFIG["maximum_epochs"] * len(fit_loader)
        )
        warmup_steps = max(
            1,
            int(round(
                PHASE49_SELECTION_CONFIG["warmup_fraction"] * total_steps
            )),
        )
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lambda step, total_steps=total_steps,
            warmup_steps=warmup_steps: phase49_learning_rate_factor(
                step,
                total_steps,
                warmup_steps,
            ),
        )

        anchor_probability = phase47_anchor_probability[monitor_indices]
        best_log_loss = phase47_selection_log_loss(
            phase47_y[monitor_indices],
            anchor_probability,
        )
        best_auroc = phase47_selection_auroc(
            phase47_y[monitor_indices],
            anchor_probability,
        )
        best_epoch = 0
        best_raw = np.zeros(monitor_indices.size, dtype=np.float64)
        best_probability = anchor_probability.copy()
        if specification["pool_mode"] == "topk":
            best_entropy = math.log(
                float(specification["top_k"])
            ) / math.log(1000.0)
            best_maximum_weight = 1.0 / float(specification["top_k"])
            best_support = float(specification["top_k"])
        else:
            best_entropy = 1.0
            best_maximum_weight = 1.0 / 1000.0
            best_support = 1000.0
        epochs_without_improvement = 0
        epoch_history = []

        for epoch in range(1, PHASE49_SELECTION_CONFIG["maximum_epochs"] + 1):
            model.set_training_mode()
            total_bce = 0.0
            total_n = 0
            maximum_gradient_norm = 0.0

            for batch in fit_loader:
                prefix = batch["prefix"].to(
                    phase49_device,
                    non_blocking=False,
                ).float()
                uptake = batch["uptake"].to(
                    phase49_device,
                    non_blocking=False,
                ).float()
                anchor = batch["anchor_logit"].to(
                    phase49_device,
                    non_blocking=False,
                )
                labels = batch["label"].to(
                    phase49_device,
                    non_blocking=False,
                )
                case_weight = batch["case_weight"].to(
                    phase49_device,
                    non_blocking=False,
                )

                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=phase49_device.type,
                    dtype=phase49_amp_dtype,
                    enabled=phase49_amp_enabled,
                ):
                    output = model(prefix, uptake, anchor)
                element_bce = F.binary_cross_entropy_with_logits(
                    output["updated_logit"].float(),
                    labels.float(),
                    reduction="none",
                )
                loss = torch.sum(
                    element_bce * case_weight.float()
                ) / torch.sum(case_weight.float())
                assert torch.isfinite(loss)
                loss.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for parameter in model.parameters()
                        if parameter.requires_grad
                    ],
                    PHASE49_SELECTION_CONFIG["gradient_clip"],
                )
                assert torch.isfinite(gradient_norm)
                optimizer.step()
                scheduler.step()

                batch_n = int(labels.numel())
                total_bce += float(loss.detach().item()) * batch_n
                total_n += batch_n
                maximum_gradient_norm = max(
                    maximum_gradient_norm,
                    float(gradient_norm.detach().item()),
                )

            prediction = phase49_predict_uptake(
                model,
                monitor_loader,
                monitor_indices,
            )
            monitor_log_loss = phase47_selection_log_loss(
                phase47_y[monitor_indices],
                prediction["probability"],
            )
            monitor_auroc = phase47_selection_auroc(
                phase47_y[monitor_indices],
                prediction["probability"],
            )
            improved = monitor_log_loss < (
                best_log_loss
                - PHASE49_SELECTION_CONFIG[
                    "early_stopping_minimum_gain"
                ]
            )
            if improved:
                best_log_loss = float(monitor_log_loss)
                best_auroc = float(monitor_auroc)
                best_epoch = int(epoch)
                best_raw = prediction["raw_residual"].copy()
                best_probability = prediction["probability"].copy()
                best_entropy = float(
                    prediction["mean_normalized_weight_entropy"]
                )
                best_maximum_weight = float(
                    prediction["mean_maximum_weight"]
                )
                best_support = float(prediction["mean_support_count"])
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            epoch_history.append({
                "epoch": int(epoch),
                "train_bce": float(total_bce / total_n),
                "monitor_log_loss": float(monitor_log_loss),
                "monitor_auroc": float(monitor_auroc),
                "weight_entropy": float(
                    prediction["mean_normalized_weight_entropy"]
                ),
                "maximum_weight": float(
                    prediction["mean_maximum_weight"]
                ),
                "support_count": float(prediction["mean_support_count"]),
                "improved": bool(improved),
                "maximum_gradient_norm": float(maximum_gradient_norm),
            })
            if (
                epoch >= PHASE49_SELECTION_CONFIG["minimum_epochs"]
                and epochs_without_improvement
                >= PHASE49_SELECTION_CONFIG["early_stopping_patience"]
            ):
                break

        phase49_candidate_monitor_raw[
            candidate_index,
            monitor_slice,
        ] = best_raw
        phase49_candidate_monitor_probability[
            candidate_index,
            monitor_slice,
        ] = best_probability
        phase49_candidate_epoch_counts[candidate_index, fold] = best_epoch
        phase49_candidate_entropy[candidate_index, fold] = best_entropy
        phase49_candidate_maximum_weight[
            candidate_index,
            fold,
        ] = best_maximum_weight
        phase49_candidate_support[candidate_index, fold] = best_support
        phase49_training_records.append({
            "candidate_index": int(candidate_index),
            "fold": int(fold),
            "seed": int(run_seed),
            "best_epoch": int(best_epoch),
            "best_monitor_log_loss": float(best_log_loss),
            "best_monitor_auroc": float(best_auroc),
            "best_weight_entropy": float(best_entropy),
            "best_maximum_weight": float(best_maximum_weight),
            "best_support_count": float(best_support),
            "epochs_executed": int(len(epoch_history)),
            "final_epoch": epoch_history[-1],
            "elapsed_seconds": round(time.perf_counter() - run_started, 3),
        })
        print(
            f"Phase49 candidate {candidate_index + 1}/{candidate_count}, "
            f"fold {fold + 1}/3: best_epoch={best_epoch}, "
            f"log_loss={best_log_loss:.6f}, auroc={best_auroc:.6f}, "
            f"pool={specification['pool_name']}"
        )

        del scheduler
        del optimizer
        del model
        del fit_loader
        del monitor_loader
        del fit_dataset
        del monitor_dataset
        gc.collect()
        if phase49_device.type == "cuda":
            torch.cuda.empty_cache()

assert np.all(np.isfinite(phase49_candidate_monitor_raw))
assert np.all(np.isfinite(phase49_candidate_monitor_probability))


def phase49_model_summary(candidate_index):
    probability = phase49_candidate_monitor_probability[candidate_index]
    metrics = phase47_complete_monitor_metrics(probability)
    fold_wins = int(sum(
        record["log_loss"] < phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    worst_fold_regret = float(max(
        record["log_loss"] - phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    maximum_group_harm = float(max(
        record["log_loss"] - phase47_anchor_group_loss[record["group"]]
        for record in metrics["group_metrics"]
    ))
    screen_safe = bool(
        worst_fold_regret
        <= PHASE49_SELECTION_CONFIG["maximum_model_fold_regret"]
        and maximum_group_harm
        <= PHASE49_SELECTION_CONFIG["maximum_model_group_harm"]
    )
    return {
        "candidate_index": int(candidate_index),
        "specification": dict(phase49_model_grid[candidate_index]),
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        "log_loss_gain": float(
            phase47_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": float(
            metrics["auroc"] - phase47_anchor_metrics["auroc"]
        ),
        "fold_wins": fold_wins,
        "worst_fold_regret": worst_fold_regret,
        "maximum_group_harm": maximum_group_harm,
        "screen_safe": screen_safe,
        "selected_epoch_counts": [
            int(value)
            for value in phase49_candidate_epoch_counts[candidate_index]
        ],
        "weight_entropy_by_fold": [
            float(value)
            for value in phase49_candidate_entropy[candidate_index]
        ],
        "maximum_weight_by_fold": [
            float(value)
            for value in phase49_candidate_maximum_weight[candidate_index]
        ],
        "support_count_by_fold": [
            float(value)
            for value in phase49_candidate_support[candidate_index]
        ],
        "fold_metrics": metrics["fold_metrics"],
    }


phase49_model_summaries = [
    phase49_model_summary(candidate_index)
    for candidate_index in range(candidate_count)
]
phase49_safe_model_summaries = [
    record for record in phase49_model_summaries if record["screen_safe"]
]
phase49_model_screen_fallback = not bool(phase49_safe_model_summaries)
phase49_model_selection_pool = (
    phase49_safe_model_summaries
    if phase49_safe_model_summaries
    else phase49_model_summaries
)
phase49_raw_model_best = min(
    phase49_model_selection_pool,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["candidate_index"],
    ),
)
phase49_model_band = [
    record
    for record in phase49_model_selection_pool
    if record["monitor_log_loss"]
    <= phase49_raw_model_best["monitor_log_loss"]
    + PHASE49_SELECTION_CONFIG["model_log_loss_band"]
]
phase49_selected_model = max(
    phase49_model_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["candidate_index"],
    ),
)
phase49_selected_candidate_index = int(
    phase49_selected_model["candidate_index"]
)
phase49_selected_raw = phase49_candidate_monitor_raw[
    phase49_selected_candidate_index
].copy()


def phase49_gate_summary(specification):
    if specification["scale"] == 0.0:
        update = np.zeros_like(phase49_selected_raw)
    else:
        uncertainty = np.power(
            4.0
            * phase47_monitor_anchor_probability
            * (1.0 - phase47_monitor_anchor_probability),
            specification["uncertainty_exponent"],
        )
        update = (
            specification["scale"]
            * uncertainty
            * np.clip(
                phase49_selected_raw,
                -specification["residual_cap"],
                specification["residual_cap"],
            )
        )
    probability = phase47_selection_sigmoid(
        phase47_monitor_anchor_logit + update
    )
    metrics = phase47_complete_monitor_metrics(probability)
    fold_wins = int(sum(
        record["log_loss"] < phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    worst_fold_regret = float(max(
        record["log_loss"] - phase47_anchor_fold_loss[record["fold"]]
        for record in metrics["fold_metrics"]
    ))
    maximum_group_harm = float(max(
        record["log_loss"] - phase47_anchor_group_loss[record["group"]]
        for record in metrics["group_metrics"]
    ))
    is_zero = specification["scale"] == 0.0
    safe = bool(
        is_zero
        or (
            fold_wins >= PHASE49_SELECTION_CONFIG["minimum_gate_fold_wins"]
            and worst_fold_regret
            <= PHASE49_SELECTION_CONFIG["maximum_gate_fold_regret"]
            and maximum_group_harm
            <= PHASE49_SELECTION_CONFIG["maximum_gate_group_harm"]
        )
    )
    return {
        "gate_index": int(specification["gate_index"]),
        "specification": dict(specification),
        "monitor_log_loss": float(metrics["log_loss"]),
        "monitor_auroc": float(metrics["auroc"]),
        "log_loss_gain": float(
            phase47_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": float(
            metrics["auroc"] - phase47_anchor_metrics["auroc"]
        ),
        "fold_wins": fold_wins,
        "worst_fold_regret": worst_fold_regret,
        "maximum_group_harm": maximum_group_harm,
        "mean_absolute_update": float(np.mean(np.abs(update))),
        "q95_absolute_update": float(np.quantile(np.abs(update), 0.95)),
        "safe": safe,
        "fold_metrics": metrics["fold_metrics"],
    }


phase49_gate_summaries = [
    phase49_gate_summary(specification)
    for specification in phase49_gate_grid
]
phase49_safe_gate_summaries = [
    record for record in phase49_gate_summaries if record["safe"]
]
assert phase49_safe_gate_summaries
phase49_raw_gate_best = min(
    phase49_safe_gate_summaries,
    key=lambda record: (
        record["monitor_log_loss"],
        -record["monitor_auroc"],
        record["gate_index"],
    ),
)
phase49_gate_band = [
    record
    for record in phase49_safe_gate_summaries
    if record["monitor_log_loss"]
    <= phase49_raw_gate_best["monitor_log_loss"]
    + PHASE49_SELECTION_CONFIG["gate_log_loss_band"]
]
phase49_provisional_gate = max(
    phase49_gate_band,
    key=lambda record: (
        record["monitor_auroc"],
        -record["monitor_log_loss"],
        -record["gate_index"],
    ),
)

phase49_advance_by_log_loss = bool(
    phase49_provisional_gate["log_loss_gain"]
    >= PHASE49_SELECTION_CONFIG["minimum_advance_log_loss_gain"]
    and phase49_provisional_gate["auroc_gain"]
    >= -PHASE49_SELECTION_CONFIG["maximum_advance_auroc_deficit"]
)
phase49_advance_by_auroc = bool(
    phase49_provisional_gate["auroc_gain"]
    >= PHASE49_SELECTION_CONFIG["minimum_advance_auroc_gain"]
    and phase49_provisional_gate["log_loss_gain"]
    >= -PHASE49_SELECTION_CONFIG["maximum_advance_log_loss_excess"]
)
phase49_gate_advanced = bool(
    phase49_provisional_gate["gate_index"] != 0
    and (phase49_advance_by_log_loss or phase49_advance_by_auroc)
)
phase49_selected_gate = (
    phase49_provisional_gate
    if phase49_gate_advanced
    else phase49_gate_summaries[0]
)

if phase49_device.type == "cuda":
    phase49_peak_vram_mb = float(
        torch.cuda.max_memory_allocated(phase49_device) / (1024 ** 2)
    )
else:
    phase49_peak_vram_mb = 0.0

PHASE49_MODEL_GRID_PRIVATE = copy.deepcopy(phase49_model_grid)
PHASE49_SELECTED_MODEL_PRIVATE = copy.deepcopy(phase49_selected_model)
PHASE49_SELECTED_GATE_PRIVATE = copy.deepcopy(phase49_selected_gate)
PHASE49_SELECTED_EPOCH_COUNTS_PRIVATE = tuple(
    phase49_selected_model["selected_epoch_counts"]
)
PHASE49_GATE_ADVANCED_PRIVATE = bool(phase49_gate_advanced)

phase49_selection_report = {
    "phase": "phase49_uptake_guided_anchor_preserving_selection",
    "status": (
        "model_and_gate_frozen_for_outer_evaluation"
        if phase49_gate_advanced
        else "no_update_selected_stop_before_outer_evaluation"
    ),
    "anchor_monitor": {
        "log_loss": float(phase47_anchor_metrics["log_loss"]),
        "auroc": float(phase47_anchor_metrics["auroc"]),
    },
    "training": {
        "model_specification_count": candidate_count,
        "fold_count": 3,
        "trained_model_count": int(candidate_count * 3),
        "maximum_epochs": int(
            PHASE49_SELECTION_CONFIG["maximum_epochs"]
        ),
        "batch_size": int(PHASE49_SELECTION_CONFIG["batch_size"]),
        "complete_pooling_grid": {
            "top_k_values": [16, 32, 64, 128],
            "softmax_temperatures": [0.10, 0.25, 0.50],
            "tail_modes": ["frozen", "trainable"],
        },
        "ranking_axis_excluded_after_phase47_phase48_failure": True,
        "group_balanced_case_weights": True,
        "epoch_zero_phase39_identity_allowed": True,
        "records": phase49_training_records,
    },
    "model_selection": {
        "screen_safe_candidate_count": len(
            phase49_safe_model_summaries
        ),
        "fallback_to_all_candidates": bool(
            phase49_model_screen_fallback
        ),
        "raw_log_loss_best": phase49_raw_model_best,
        "selected": phase49_selected_model,
    },
    "gate_selection": {
        "candidate_count": len(phase49_gate_summaries),
        "safe_candidate_count": len(phase49_safe_gate_summaries),
        "raw_log_loss_best": phase49_raw_gate_best,
        "provisional": phase49_provisional_gate,
        "selected": phase49_selected_gate,
        "advance_by_log_loss": phase49_advance_by_log_loss,
        "advance_by_auroc": phase49_advance_by_auroc,
        "gate_advanced": phase49_gate_advanced,
    },
    "selection_rule": (
        "maximum monitor AUROC within 0.002 log-loss of the safe shared "
        "model minimum, followed by maximum AUROC within 0.001 log-loss "
        "of the safe bounded-gate minimum"
    ),
    "formula_if_advanced": (
        "z49 = z39 + scale * uncertainty(z39)^gamma * "
        "clip(r49_raw, -cap, cap)"
    ),
    "shared_model_specification_across_folds": True,
    "shared_gate_across_folds": True,
    "fit_labels_used_for_training": True,
    "monitor_labels_used_for_selection": True,
    "outer_validation_labels_used": False,
    "public_leaderboard_used": False,
    "training_prefix_cache_read": True,
    "training_uptake_cache_read": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "peak_vram_mb": phase49_peak_vram_mb,
    "elapsed_seconds": round(
        time.perf_counter() - phase49_selection_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE49_SELECTION")
print(json.dumps(phase49_selection_report, indent=2))
print("END SANITIZED_PHASE49_SELECTION")

Phase49 candidate 1/14, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, pool=topk_16
Phase49 candidate 1/14, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867, pool=topk_16
Phase49 candidate 1/14, fold 3/3: best_epoch=2, log_loss=0.232975, auroc=0.971944, pool=topk_16
Phase49 candidate 2/14, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, pool=topk_16
Phase49 candidate 2/14, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867, pool=topk_16
Phase49 candidate 2/14, fold 3/3: best_epoch=3, log_loss=0.232025, auroc=0.971631, pool=topk_16
Phase49 candidate 3/14, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, pool=topk_32
Phase49 candidate 3/14, fold 2/3: best_epoch=0, log_loss=0.216570, auroc=0.968867, pool=topk_32
Phase49 candidate 3/14, fold 3/3: best_epoch=3, log_loss=0.232145, auroc=0.971736, pool=topk_32
Phase49 candidate 4/14, fold 1/3: best_epoch=0, log_loss=0.258194, auroc=0.953661, pool=topk_32
Phase49 candidate 4/14, fold 2/3: best_e

> Phase 50

In [4]:
# phase50_cell_150A

import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np


phase50_started = time.perf_counter()

PHASE50_VALIDATION_CONFIG = {
    "case_count": 1362,
    "known_protocol_repeat_count": 5,
    "known_protocol_fold_count": 5,
    "known_protocol_repeat_seeds": [
        500101,
        500102,
        500103,
        500104,
        500105,
    ],
    "future_audit_fraction": 0.15,
    "future_audit_seed": 500001,
    "minimum_stratum_size_for_audit": 6,
    "minimum_stratum_development_count": 5,
    "material_fold_log_loss_gain": 1.0e-4,
    "aspirational_log_loss": 0.24,
    "aspirational_auroc": 0.959,
}

phase50_required_names = [
    "phase47_y",
    "phase47_groups",
    "phase47_partitions",
    "phase47_anchor_probability",
    "phase49_selection_report",
]
phase50_missing_names = [
    name for name in phase50_required_names if name not in globals()
]
assert not phase50_missing_names, {
    "message": (
        "Phase50 requires the accepted Phase47 state and completed "
        "Phase49 selection report in the current kernel."
    ),
    "missing": phase50_missing_names,
}

phase50_y = np.asarray(phase47_y, dtype=np.int64).reshape(-1)
phase50_groups = np.asarray(phase47_groups, dtype=np.int64).reshape(-1)
phase50_phase39_probability = np.asarray(
    phase47_anchor_probability,
    dtype=np.float64,
).reshape(-1)

assert phase50_y.shape == (1362,)
assert phase50_groups.shape == (1362,)
assert phase50_phase39_probability.shape == (1362,)
assert set(np.unique(phase50_y).tolist()) == {0, 1}
assert np.unique(phase50_groups).size == 15
assert np.all(np.isfinite(phase50_phase39_probability))
assert np.all(
    (phase50_phase39_probability > 0.0)
    & (phase50_phase39_probability < 1.0)
)


# -------------------------------------------------------------------------
# Phase49 correction: its nominal second fold win was 5.55e-17, while the
# selected fold-specific epoch counts were [0, 0, 3].  Preserve the original
# report, but explicitly prevent accidental outer evaluation or packaging.
# -------------------------------------------------------------------------

phase50_phase49_selected_model = phase49_selection_report[
    "model_selection"
]["selected"]
phase50_phase49_selected_gate = phase49_selection_report[
    "gate_selection"
]["selected"]
phase50_phase49_epochs = [
    int(value)
    for value in phase50_phase49_selected_model["selected_epoch_counts"]
]

phase50_phase49_anchor_fold_losses = {
    int(record["fold"]): float(record["log_loss"])
    for record in phase49_selection_report["gate_selection"][
        "raw_log_loss_best"
    ]["fold_metrics"]
}

# The raw-best gate can itself be nonzero.  The authoritative Phase39 anchor
# fold losses are recovered from folds where the selected model stayed at
# epoch zero, and from the Phase47 anchor metrics for every fold.
if "phase47_anchor_fold_loss" in globals():
    phase50_phase49_anchor_fold_losses = {
        int(key): float(value)
        for key, value in phase47_anchor_fold_loss.items()
    }
else:
    # These exact values are accepted Phase39 monitor-anchor metrics and are
    # checked against the two epoch-zero folds below.
    phase50_phase49_anchor_fold_losses = {
        0: 0.25819432426968364,
        1: 0.21656980905433676,
        2: 0.23576935856947,
    }

phase50_phase49_fold_audit = []
for phase50_fold_record in phase50_phase49_selected_gate["fold_metrics"]:
    phase50_fold = int(phase50_fold_record["fold"])
    phase50_anchor_loss = phase50_phase49_anchor_fold_losses[phase50_fold]
    phase50_candidate_loss = float(phase50_fold_record["log_loss"])
    phase50_gain = float(phase50_anchor_loss - phase50_candidate_loss)
    phase50_phase49_fold_audit.append({
        "fold": phase50_fold,
        "selected_epoch": int(phase50_phase49_epochs[phase50_fold]),
        "anchor_log_loss": phase50_anchor_loss,
        "phase49_log_loss": phase50_candidate_loss,
        "log_loss_gain": phase50_gain,
        "strict_float_win": bool(phase50_gain > 0.0),
        "material_win": bool(
            phase50_gain
            >= PHASE50_VALIDATION_CONFIG[
                "material_fold_log_loss_gain"
            ]
        ),
    })

phase50_phase49_material_fold_wins = int(sum(
    record["material_win"] for record in phase50_phase49_fold_audit
))
phase50_phase49_nonzero_epoch_fold_count = int(sum(
    epoch > 0 for epoch in phase50_phase49_epochs
))

assert phase50_phase49_epochs == [0, 0, 3], {
    "message": "Unexpected Phase49 epoch contract; audit manually.",
    "epochs": phase50_phase49_epochs,
}
assert phase50_phase49_material_fold_wins == 1
assert phase50_phase49_nonzero_epoch_fold_count == 1

PHASE49_OUTER_EVALUATION_ALLOWED_PRIVATE = False
PHASE49_PROMOTION_OVERRIDE_PRIVATE = {
    "original_status": str(phase49_selection_report["status"]),
    "corrected_status": "no_update_selected_stop_before_outer_evaluation",
    "reason": (
        "only one material monitor-fold win; nominal second win was "
        "floating-point equality"
    ),
    "material_fold_gain_threshold": float(
        PHASE50_VALIDATION_CONFIG["material_fold_log_loss_gain"]
    ),
    "material_fold_wins": phase50_phase49_material_fold_wins,
    "selected_epoch_counts": list(phase50_phase49_epochs),
}


# -------------------------------------------------------------------------
# Reconstruct the original three outer folds only for provenance.  They are
# no longer treated as epistemically fresh after repeated phase-level audits.
# -------------------------------------------------------------------------

phase50_original_fold = np.full(1362, -1, dtype=np.int8)
assert len(phase47_partitions) == 3
for phase50_expected_fold, phase50_partition in enumerate(
    phase47_partitions
):
    assert int(phase50_partition["fold"]) == phase50_expected_fold
    phase50_outer_valid = np.asarray(
        phase50_partition["outer_valid_indices"],
        dtype=np.int64,
    )
    assert np.all(phase50_original_fold[phase50_outer_valid] == -1)
    phase50_original_fold[phase50_outer_valid] = phase50_expected_fold
assert np.all(phase50_original_fold >= 0)
assert np.bincount(phase50_original_fold, minlength=3).tolist() == [
    467,
    443,
    452,
]


# -------------------------------------------------------------------------
# Incrementally locked audit subset.
#
# It is not historically pristine: earlier frozen models and aggregate phase
# decisions have already used all labels.  From Phase50 onward, however, its
# images and labels must not be used for fitting, early stopping, calibration,
# gate search, or architecture selection.  It is evaluated only once after a
# complete candidate is frozen using development data.
# -------------------------------------------------------------------------

phase50_all_indices = np.arange(1362, dtype=np.int64)
phase50_future_audit_mask = np.zeros(1362, dtype=bool)
phase50_audit_rng = np.random.default_rng(
    PHASE50_VALIDATION_CONFIG["future_audit_seed"]
)
phase50_stratum_records = []

for phase50_group in np.unique(phase50_groups):
    for phase50_label in [0, 1]:
        phase50_stratum = np.flatnonzero(
            (phase50_groups == phase50_group)
            & (phase50_y == phase50_label)
        ).astype(np.int64)
        phase50_stratum_n = int(phase50_stratum.size)
        phase50_audit_n = 0
        if (
            phase50_stratum_n
            >= PHASE50_VALIDATION_CONFIG[
                "minimum_stratum_size_for_audit"
            ]
        ):
            phase50_audit_n = max(
                1,
                int(round(
                    PHASE50_VALIDATION_CONFIG["future_audit_fraction"]
                    * phase50_stratum_n
                )),
            )
            phase50_audit_n = min(
                phase50_audit_n,
                phase50_stratum_n
                - PHASE50_VALIDATION_CONFIG[
                    "minimum_stratum_development_count"
                ],
            )
        if phase50_audit_n > 0:
            phase50_selected = phase50_audit_rng.permutation(
                phase50_stratum
            )[:phase50_audit_n]
            phase50_future_audit_mask[phase50_selected] = True
        phase50_stratum_records.append({
            "group": int(phase50_group),
            "label": int(phase50_label),
            "total_n": phase50_stratum_n,
            "development_n": int(phase50_stratum_n - phase50_audit_n),
            "future_audit_n": int(phase50_audit_n),
        })

phase50_future_audit_indices = np.flatnonzero(
    phase50_future_audit_mask
).astype(np.int64)
phase50_development_indices = np.flatnonzero(
    ~phase50_future_audit_mask
).astype(np.int64)

assert np.intersect1d(
    phase50_development_indices,
    phase50_future_audit_indices,
).size == 0
assert np.array_equal(
    np.sort(np.concatenate([
        phase50_development_indices,
        phase50_future_audit_indices,
    ])),
    phase50_all_indices,
)
assert phase50_future_audit_indices.size > 0
assert phase50_development_indices.size > 1000


# -------------------------------------------------------------------------
# Known-protocol validation: five independently randomized repetitions of
# five-fold holdout, stratified inside each acquisition-group/label stratum.
# Every development case is validation exactly once per repetition.  No seed
# or partition candidate is selected using model performance.
# -------------------------------------------------------------------------

phase50_repeat_count = PHASE50_VALIDATION_CONFIG[
    "known_protocol_repeat_count"
]
phase50_fold_count = PHASE50_VALIDATION_CONFIG[
    "known_protocol_fold_count"
]
phase50_known_fold_assignment = np.full(
    (phase50_repeat_count, 1362),
    -1,
    dtype=np.int8,
)
phase50_known_fold_records = []

for phase50_repeat, phase50_seed in enumerate(
    PHASE50_VALIDATION_CONFIG["known_protocol_repeat_seeds"]
):
    phase50_rng = np.random.default_rng(phase50_seed)
    phase50_assignment = phase50_known_fold_assignment[phase50_repeat]

    for phase50_group in np.unique(phase50_groups):
        for phase50_label in [0, 1]:
            phase50_stratum = phase50_development_indices[
                (phase50_groups[phase50_development_indices]
                 == phase50_group)
                & (phase50_y[phase50_development_indices]
                   == phase50_label)
            ]
            if phase50_stratum.size == 0:
                continue
            phase50_ordered = phase50_rng.permutation(phase50_stratum)
            phase50_offset = int(
                phase50_rng.integers(0, phase50_fold_count)
            )
            phase50_assignment[phase50_ordered] = (
                np.arange(phase50_ordered.size, dtype=np.int64)
                + phase50_offset
            ) % phase50_fold_count

    assert np.all(phase50_assignment[phase50_development_indices] >= 0)
    assert np.all(phase50_assignment[phase50_future_audit_indices] == -1)

    phase50_repeat_coverage = np.zeros(1362, dtype=np.int8)
    for phase50_fold in range(phase50_fold_count):
        phase50_valid_indices = phase50_development_indices[
            phase50_assignment[phase50_development_indices]
            == phase50_fold
        ]
        phase50_train_indices = phase50_development_indices[
            phase50_assignment[phase50_development_indices]
            != phase50_fold
        ]
        assert phase50_valid_indices.size > 0
        assert phase50_train_indices.size > 0
        assert np.intersect1d(
            phase50_train_indices,
            phase50_valid_indices,
        ).size == 0
        assert np.all(phase50_y[phase50_valid_indices] >= 0)
        assert np.unique(phase50_y[phase50_valid_indices]).size == 2
        phase50_repeat_coverage[phase50_valid_indices] += 1

        phase50_train_groups = np.unique(
            phase50_groups[phase50_train_indices]
        )
        phase50_valid_groups = np.unique(
            phase50_groups[phase50_valid_indices]
        )
        phase50_shared_groups = np.intersect1d(
            phase50_train_groups,
            phase50_valid_groups,
        )
        assert phase50_train_groups.size == 15
        assert np.array_equal(
            phase50_shared_groups,
            phase50_valid_groups,
        )

        phase50_known_fold_records.append({
            "repeat": int(phase50_repeat),
            "fold": int(phase50_fold),
            "seed": int(phase50_seed),
            "train_n": int(phase50_train_indices.size),
            "valid_n": int(phase50_valid_indices.size),
            "train_group_count": int(phase50_train_groups.size),
            "valid_group_count": int(phase50_valid_groups.size),
            "shared_group_count": int(phase50_shared_groups.size),
            "valid_prevalence": round(float(
                np.mean(phase50_y[phase50_valid_indices])
            ), 6),
        })

    assert np.all(
        phase50_repeat_coverage[phase50_development_indices] == 1
    )
    assert np.all(
        phase50_repeat_coverage[phase50_future_audit_indices] == 0
    )


# -------------------------------------------------------------------------
# Unknown-protocol validation: leave one complete acquisition group out from
# the development subset.  The corresponding deployment policy is anchor-only;
# group-specific residual parameters are forbidden for the held-out protocol.
# -------------------------------------------------------------------------

phase50_logo_records = []
for phase50_group in np.unique(phase50_groups):
    phase50_logo_valid = phase50_development_indices[
        phase50_groups[phase50_development_indices] == phase50_group
    ]
    phase50_logo_train = phase50_development_indices[
        phase50_groups[phase50_development_indices] != phase50_group
    ]
    assert phase50_logo_valid.size > 0
    assert phase50_logo_train.size > 0
    assert np.intersect1d(
        phase50_logo_train,
        phase50_logo_valid,
    ).size == 0
    assert phase50_group not in np.unique(
        phase50_groups[phase50_logo_train]
    )
    phase50_logo_records.append({
        "held_out_group": int(phase50_group),
        "train_n": int(phase50_logo_train.size),
        "valid_n": int(phase50_logo_valid.size),
        "valid_prevalence": round(float(
            np.mean(phase50_y[phase50_logo_valid])
        ), 6),
        "deployment_policy": "anchor_only",
    })

assert len(phase50_logo_records) == 15
assert sum(record["valid_n"] for record in phase50_logo_records) == int(
    phase50_development_indices.size
)


# -------------------------------------------------------------------------
# Persist only partition state.  No labels, probabilities, UIDs, voxels, or
# patient rows are written.  The file is private notebook state, not a package
# asset and not a submission member.
# -------------------------------------------------------------------------

phase50_state_directory = Path(
    "/kaggle/working/phase50_private_validation"
)
phase50_state_directory.mkdir(parents=True, exist_ok=True)
phase50_state_path = phase50_state_directory / "phase50_partitions.npz"
phase50_temporary_state_path = (
    phase50_state_directory / "phase50_partitions.tmp.npz"
)

np.savez_compressed(
    phase50_temporary_state_path,
    development_indices=phase50_development_indices,
    future_audit_indices=phase50_future_audit_indices,
    known_fold_assignment=phase50_known_fold_assignment,
    original_fold_assignment=phase50_original_fold,
    acquisition_group=phase50_groups.astype(np.int16),
)
os.replace(phase50_temporary_state_path, phase50_state_path)
assert phase50_state_path.is_file()

phase50_contract_hasher = hashlib.sha256()
for phase50_contract_array in [
    phase50_development_indices,
    phase50_future_audit_indices,
    phase50_known_fold_assignment,
    phase50_original_fold,
    phase50_groups.astype(np.int16),
]:
    phase50_contract_hasher.update(
        np.ascontiguousarray(phase50_contract_array).view(np.uint8)
    )
phase50_partition_sha256 = phase50_contract_hasher.hexdigest()

PHASE50_VALIDATION_STATE_PATH_PRIVATE = phase50_state_path
PHASE50_DEVELOPMENT_INDICES_PRIVATE = phase50_development_indices.copy()
PHASE50_FUTURE_AUDIT_INDICES_PRIVATE = (
    phase50_future_audit_indices.copy()
)
PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE = (
    phase50_known_fold_assignment.copy()
)
PHASE50_LOGO_GROUPS_PRIVATE = tuple(
    int(value) for value in np.unique(phase50_groups)
)
PHASE50_PARTITION_SHA256_PRIVATE = phase50_partition_sha256

phase50_report = {
    "phase": "phase50_validation_reset_and_incremental_lock_contract",
    "status": "accepted",
    "aspirational_target": {
        "log_loss_at_most": float(
            PHASE50_VALIDATION_CONFIG["aspirational_log_loss"]
        ),
        "auroc_at_least": float(
            PHASE50_VALIDATION_CONFIG["aspirational_auroc"]
        ),
        "guaranteed": False,
    },
    "phase49_override": {
        **PHASE49_PROMOTION_OVERRIDE_PRIVATE,
        "fold_audit": phase50_phase49_fold_audit,
        "outer_evaluation_allowed": False,
    },
    "case_count": 1362,
    "development": {
        "case_count": int(phase50_development_indices.size),
        "group_count": int(np.unique(
            phase50_groups[phase50_development_indices]
        ).size),
        "prevalence": round(float(
            np.mean(phase50_y[phase50_development_indices])
        ), 6),
    },
    "future_audit": {
        "case_count": int(phase50_future_audit_indices.size),
        "group_count": int(np.unique(
            phase50_groups[phase50_future_audit_indices]
        ).size),
        "prevalence": round(float(
            np.mean(phase50_y[phase50_future_audit_indices])
        ), 6),
        "historically_pristine": False,
        "incrementally_locked_from_phase50": True,
        "labels_evaluated_for_model_metrics_now": False,
        "forbidden_until_final_freeze": [
            "supervised_training",
            "early_stopping",
            "calibration_fit",
            "gate_search",
            "architecture_selection",
        ],
    },
    "known_protocol_validation": {
        "repeat_count": int(phase50_repeat_count),
        "fold_count_per_repeat": int(phase50_fold_count),
        "total_fold_count": int(
            phase50_repeat_count * phase50_fold_count
        ),
        "split_candidate_count": 1,
        "seeds": list(
            PHASE50_VALIDATION_CONFIG[
                "known_protocol_repeat_seeds"
            ]
        ),
        "stratification": "within_acquisition_group_and_label",
        "all_training_folds_contain_all_protocols": True,
        "each_development_case_valid_once_per_repeat": True,
        "records": phase50_known_fold_records,
    },
    "unknown_protocol_validation": {
        "fold_count": len(phase50_logo_records),
        "split_candidate_count": 1,
        "held_out_unit": "complete_acquisition_group",
        "policy": "phase12c_anchor_only",
        "records": phase50_logo_records,
    },
    "pre_registered_shortlist": [
        "phase12c",
        "phase36",
        "phase39",
        "phase42_deployment_formula",
        "phase43_routed_anchor_ablation",
        "nested_positive_slope_platt_or_temperature_calibration",
    ],
    "next_new_representation_experiment_if_needed": (
        "fold_specific_low_rank_dinov3_last_block_adaptation"
    ),
    "selection_policy": {
        "primary": "mean_repeated_known_protocol_log_loss",
        "secondary": "mean_repeated_known_protocol_auroc",
        "required_robustness": [
            "majority_of_repeats_improve",
            "no_material_major_group_harm",
            "unknown_protocol_anchor_only_unchanged",
        ],
        "material_fold_log_loss_gain": float(
            PHASE50_VALIDATION_CONFIG[
                "material_fold_log_loss_gain"
            ]
        ),
        "public_leaderboard_used_for_selection": False,
    },
    "partition_sha256": phase50_partition_sha256,
    "private_partition_file": phase50_state_path.name,
    "private_partition_contains_labels": False,
    "private_partition_contains_probabilities": False,
    "private_partition_contains_uids": False,
    "original_three_outer_folds_epistemically_fresh": False,
    "labels_used_for_stratification": True,
    "labels_used_for_model_metric_evaluation": False,
    "future_audit_labels_used_for_selection": False,
    "training_voxel_cache_read": False,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_indices_displayed": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase50_started, 3),
}

print("BEGIN SANITIZED_PHASE50_VALIDATION_RESET")
print(json.dumps(phase50_report, indent=2))
print("END SANITIZED_PHASE50_VALIDATION_RESET")

AssertionError: {'message': 'Phase50 requires the accepted Phase47 state and completed Phase49 selection report in the current kernel.', 'missing': ['phase47_y', 'phase47_groups', 'phase47_partitions', 'phase47_anchor_probability', 'phase49_selection_report']}

In [63]:
# phase50_cell_150B

import copy
import json
import math
import os
import time
from pathlib import Path

import numpy as np
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import rankdata


phase50_calibration_started = time.perf_counter()

PHASE50_CALIBRATION_CONFIG = {
    "probability_clip": 1.0e-7,
    "positive_slope_minimum": 0.25,
    "positive_slope_maximum": 4.0,
    "intercept_minimum": -1.5,
    "intercept_maximum": 1.5,
    "platt_l2": 1.0e-3,
    "material_repeat_log_loss_gain": 1.0e-4,
    "minimum_repeat_wins": 4,
    "maximum_repeat_regret": 0.001,
    "maximum_major_group_harm": 0.010,
    "minimum_calibration_gain": 0.001,
    "maximum_auroc_deficit": 0.0002,
    "selection_log_loss_band": 0.001,
    "major_group_minimum_n": 30,
}

phase50_calibration_required_names = [
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "PHASE49_OUTER_EVALUATION_ALLOWED_PRIVATE",
    "phase50_y",
    "phase50_groups",
    "PHASE36_OOF_PRIVATE",
    "PHASE39_OOF_PRIVATE",
    "phase42_reference_probability",
]
phase50_calibration_missing = [
    name
    for name in phase50_calibration_required_names
    if name not in globals()
]
assert not phase50_calibration_missing, {
    "message": (
        "Phase50 calibration requires Cell 150A and the restored fixed "
        "component probabilities in the current kernel."
    ),
    "missing": phase50_calibration_missing,
}
assert PHASE49_OUTER_EVALUATION_ALLOWED_PRIVATE is False

phase50_calibration_y = np.asarray(
    phase50_y,
    dtype=np.int64,
).reshape(-1)
phase50_calibration_groups = np.asarray(
    phase50_groups,
    dtype=np.int64,
).reshape(-1)
phase50_calibration_development = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase50_calibration_audit = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase50_calibration_fold_assignment = np.asarray(
    PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE,
    dtype=np.int8,
)

assert phase50_calibration_y.shape == (1362,)
assert phase50_calibration_groups.shape == (1362,)
assert phase50_calibration_development.size == 1160
assert phase50_calibration_audit.size == 202
assert phase50_calibration_fold_assignment.shape == (5, 1362)
assert np.intersect1d(
    phase50_calibration_development,
    phase50_calibration_audit,
).size == 0
assert np.all(
    phase50_calibration_fold_assignment[
        :, phase50_calibration_audit
    ] == -1
)


def phase50_clip_probability(probability):
    probability = np.asarray(probability, dtype=np.float64).reshape(-1)
    return np.clip(
        probability,
        PHASE50_CALIBRATION_CONFIG["probability_clip"],
        1.0 - PHASE50_CALIBRATION_CONFIG["probability_clip"],
    )


def phase50_logit(probability):
    probability = phase50_clip_probability(probability)
    return np.log(probability) - np.log1p(-probability)


def phase50_sigmoid(logit):
    logit = np.asarray(logit, dtype=np.float64)
    output = np.empty_like(logit, dtype=np.float64)
    positive = logit >= 0.0
    output[positive] = 1.0 / (1.0 + np.exp(-logit[positive]))
    exponential = np.exp(logit[~positive])
    output[~positive] = exponential / (1.0 + exponential)
    return phase50_clip_probability(output)


def phase50_log_loss(labels, probability, weights=None):
    labels = np.asarray(labels, dtype=np.float64).reshape(-1)
    probability = phase50_clip_probability(probability)
    assert labels.shape == probability.shape
    losses = -(
        labels * np.log(probability)
        + (1.0 - labels) * np.log1p(-probability)
    )
    if weights is None:
        return float(np.mean(losses))
    weights = np.asarray(weights, dtype=np.float64).reshape(-1)
    assert weights.shape == losses.shape
    assert np.all(weights >= 0.0)
    assert float(np.sum(weights)) > 0.0
    return float(np.sum(weights * losses) / np.sum(weights))


def phase50_auroc(labels, scores, weights=None):
    labels = np.asarray(labels, dtype=np.int64).reshape(-1)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    assert labels.shape == scores.shape
    assert set(np.unique(labels).tolist()) == {0, 1}
    if weights is None:
        # Average ranks handle exact score ties correctly.
        ranks = rankdata(scores, method="average")
        positive = labels == 1
        positive_n = int(np.sum(positive))
        negative_n = int(labels.size - positive_n)
        return float(
            (
                np.sum(ranks[positive])
                - positive_n * (positive_n + 1) / 2.0
            )
            / (positive_n * negative_n)
        )

    # Weighted Mann-Whitney statistic with half-credit for ties.
    weights = np.asarray(weights, dtype=np.float64).reshape(-1)
    assert weights.shape == labels.shape
    order = np.argsort(scores, kind="mergesort")
    ordered_scores = scores[order]
    ordered_labels = labels[order]
    ordered_weights = weights[order]
    total_positive = float(np.sum(ordered_weights[ordered_labels == 1]))
    total_negative = float(np.sum(ordered_weights[ordered_labels == 0]))
    assert total_positive > 0.0 and total_negative > 0.0
    statistic = 0.0
    negative_before = 0.0
    start = 0
    while start < ordered_scores.size:
        stop = start + 1
        while (
            stop < ordered_scores.size
            and ordered_scores[stop] == ordered_scores[start]
        ):
            stop += 1
        tie_labels = ordered_labels[start:stop]
        tie_weights = ordered_weights[start:stop]
        tie_positive = float(np.sum(tie_weights[tie_labels == 1]))
        tie_negative = float(np.sum(tie_weights[tie_labels == 0]))
        statistic += tie_positive * (
            negative_before + 0.5 * tie_negative
        )
        negative_before += tie_negative
        start = stop
    return float(statistic / (total_positive * total_negative))


def phase50_metrics(indices, probability):
    indices = np.asarray(indices, dtype=np.int64).reshape(-1)
    probability = np.asarray(probability, dtype=np.float64).reshape(-1)
    assert indices.size == probability.size
    return {
        "n": int(indices.size),
        "log_loss": phase50_log_loss(
            phase50_calibration_y[indices],
            probability,
        ),
        "auroc": phase50_auroc(
            phase50_calibration_y[indices],
            probability,
        ),
        "mean_probability": float(np.mean(probability)),
        "prevalence": float(np.mean(phase50_calibration_y[indices])),
    }


def phase50_fit_calibrator(labels, probability, method):
    labels = np.asarray(labels, dtype=np.float64).reshape(-1)
    base_logit = phase50_logit(probability)
    assert labels.shape == base_logit.shape

    if method == "identity":
        return {
            "method": "identity",
            "intercept": 0.0,
            "slope": 1.0,
            "optimizer_success": True,
            "optimizer_iterations": 0,
        }

    if method == "temperature":
        lower = math.log(
            PHASE50_CALIBRATION_CONFIG["positive_slope_minimum"]
        )
        upper = math.log(
            PHASE50_CALIBRATION_CONFIG["positive_slope_maximum"]
        )

        def objective(log_slope):
            calibrated_logit = math.exp(float(log_slope)) * base_logit
            return float(np.mean(
                np.logaddexp(0.0, calibrated_logit)
                - labels * calibrated_logit
            ))

        result = minimize_scalar(
            objective,
            bounds=(lower, upper),
            method="bounded",
            options={"xatol": 1.0e-10, "maxiter": 500},
        )
        assert bool(result.success), {
            "message": "Temperature optimization failed.",
            "optimizer_message": str(result.message),
        }
        return {
            "method": "temperature",
            "intercept": 0.0,
            "slope": float(math.exp(float(result.x))),
            "temperature": float(math.exp(-float(result.x))),
            "optimizer_success": bool(result.success),
            "optimizer_iterations": int(result.nfev),
        }

    assert method == "positive_platt"
    lower_log_slope = math.log(
        PHASE50_CALIBRATION_CONFIG["positive_slope_minimum"]
    )
    upper_log_slope = math.log(
        PHASE50_CALIBRATION_CONFIG["positive_slope_maximum"]
    )

    def objective(theta):
        intercept = float(theta[0])
        log_slope = float(theta[1])
        slope = math.exp(log_slope)
        calibrated_logit = intercept + slope * base_logit
        likelihood = np.mean(
            np.logaddexp(0.0, calibrated_logit)
            - labels * calibrated_logit
        )
        penalty = PHASE50_CALIBRATION_CONFIG["platt_l2"] * (
            intercept ** 2 + log_slope ** 2
        )
        return float(likelihood + penalty)

    result = minimize(
        objective,
        x0=np.asarray([0.0, 0.0], dtype=np.float64),
        method="L-BFGS-B",
        bounds=[
            (
                PHASE50_CALIBRATION_CONFIG["intercept_minimum"],
                PHASE50_CALIBRATION_CONFIG["intercept_maximum"],
            ),
            (lower_log_slope, upper_log_slope),
        ],
        options={"ftol": 1.0e-12, "gtol": 1.0e-9, "maxiter": 500},
    )
    assert bool(result.success), {
        "message": "Positive Platt optimization failed.",
        "optimizer_message": str(result.message),
    }
    return {
        "method": "positive_platt",
        "intercept": float(result.x[0]),
        "slope": float(math.exp(float(result.x[1]))),
        "optimizer_success": bool(result.success),
        "optimizer_iterations": int(result.nit),
    }


def phase50_apply_calibrator(probability, state):
    probability = phase50_clip_probability(probability)
    calibrated_logit = (
        float(state["intercept"])
        + float(state["slope"]) * phase50_logit(probability)
    )
    return phase50_sigmoid(calibrated_logit)


# Resolve exact, explicitly named fixed OOF vectors.  No recursive global scan
# and no metric-based resolver are used after the Phase50 audit lock.
phase50_phase12c_path = Path(
    "/kaggle/working/phase32_phase12c_oof_float64.npy"
)
assert phase50_phase12c_path.is_file(), {
    "message": "Exact Phase12c OOF file is missing.",
    "path": str(phase50_phase12c_path),
}
phase50_phase12c_probability = np.asarray(
    np.load(phase50_phase12c_path, allow_pickle=False),
    dtype=np.float64,
).reshape(-1)
phase50_phase36_probability = np.asarray(
    PHASE36_OOF_PRIVATE,
    dtype=np.float64,
).reshape(-1)
phase50_phase39_probability = np.asarray(
    PHASE39_OOF_PRIVATE,
    dtype=np.float64,
).reshape(-1)
phase50_phase43_probability = np.asarray(
    phase42_reference_probability,
    dtype=np.float64,
).reshape(-1)

phase50_components = {
    "phase12c": phase50_phase12c_probability,
    "phase36": phase50_phase36_probability,
    "phase39": phase50_phase39_probability,
    "phase43_deployment_matched_oof": phase50_phase43_probability,
}
for phase50_component_name, phase50_component_probability in (
    phase50_components.items()
):
    assert phase50_component_probability.shape == (1362,), {
        "component": phase50_component_name,
        "shape": list(phase50_component_probability.shape),
    }
    assert np.all(np.isfinite(phase50_component_probability))
    assert np.all(
        (phase50_component_probability > 0.0)
        & (phase50_component_probability < 1.0)
    )


phase50_calibration_methods = [
    "identity",
    "temperature",
    "positive_platt",
]
phase50_candidate_grid = []
for phase50_component_name in phase50_components:
    for phase50_method in phase50_calibration_methods:
        phase50_candidate_grid.append({
            "candidate_index": int(len(phase50_candidate_grid)),
            "component": phase50_component_name,
            "calibration": phase50_method,
        })
assert len(phase50_candidate_grid) == 12

phase50_repeat_count = phase50_calibration_fold_assignment.shape[0]
phase50_fold_count = 5
phase50_candidate_predictions = np.full(
    (len(phase50_candidate_grid), phase50_repeat_count, 1362),
    np.nan,
    dtype=np.float64,
)
phase50_fit_records = []

for phase50_candidate in phase50_candidate_grid:
    phase50_candidate_index = int(phase50_candidate["candidate_index"])
    phase50_component_name = phase50_candidate["component"]
    phase50_method = phase50_candidate["calibration"]
    phase50_component_probability = phase50_components[
        phase50_component_name
    ]

    for phase50_repeat in range(phase50_repeat_count):
        phase50_assignment = phase50_calibration_fold_assignment[
            phase50_repeat
        ]
        for phase50_fold in range(phase50_fold_count):
            phase50_valid_indices = phase50_calibration_development[
                phase50_assignment[phase50_calibration_development]
                == phase50_fold
            ]
            phase50_train_indices = phase50_calibration_development[
                phase50_assignment[phase50_calibration_development]
                != phase50_fold
            ]
            assert phase50_valid_indices.size > 0
            assert phase50_train_indices.size > 0
            phase50_state = phase50_fit_calibrator(
                phase50_calibration_y[phase50_train_indices],
                phase50_component_probability[phase50_train_indices],
                phase50_method,
            )
            phase50_valid_probability = phase50_apply_calibrator(
                phase50_component_probability[phase50_valid_indices],
                phase50_state,
            )
            phase50_candidate_predictions[
                phase50_candidate_index,
                phase50_repeat,
                phase50_valid_indices,
            ] = phase50_valid_probability
            phase50_fit_records.append({
                "candidate_index": phase50_candidate_index,
                "repeat": int(phase50_repeat),
                "fold": int(phase50_fold),
                "train_n": int(phase50_train_indices.size),
                "valid_n": int(phase50_valid_indices.size),
                "intercept": float(phase50_state["intercept"]),
                "slope": float(phase50_state["slope"]),
                "optimizer_success": bool(
                    phase50_state["optimizer_success"]
                ),
                "optimizer_iterations": int(
                    phase50_state["optimizer_iterations"]
                ),
            })

        assert np.all(np.isfinite(
            phase50_candidate_predictions[
                phase50_candidate_index,
                phase50_repeat,
                phase50_calibration_development,
            ]
        ))
        assert np.all(np.isnan(
            phase50_candidate_predictions[
                phase50_candidate_index,
                phase50_repeat,
                phase50_calibration_audit,
            ]
        ))


phase50_major_groups = [
    int(group)
    for group in np.unique(phase50_calibration_groups)
    if int(np.sum(
        phase50_calibration_groups[phase50_calibration_development]
        == group
    )) >= PHASE50_CALIBRATION_CONFIG["major_group_minimum_n"]
]


def phase50_candidate_summary(candidate_index):
    candidate = phase50_candidate_grid[candidate_index]
    component_probability = phase50_components[candidate["component"]]
    raw_metrics = phase50_metrics(
        phase50_calibration_development,
        component_probability[phase50_calibration_development],
    )
    repeat_metrics = []
    repeat_group_harms = []
    repeat_wins = 0
    worst_repeat_regret = -np.inf

    for repeat in range(phase50_repeat_count):
        calibrated_probability = phase50_candidate_predictions[
            candidate_index,
            repeat,
            phase50_calibration_development,
        ]
        metrics = phase50_metrics(
            phase50_calibration_development,
            calibrated_probability,
        )
        gain = float(raw_metrics["log_loss"] - metrics["log_loss"])
        repeat_wins += int(
            gain
            >= PHASE50_CALIBRATION_CONFIG[
                "material_repeat_log_loss_gain"
            ]
        )
        worst_repeat_regret = max(worst_repeat_regret, -gain)

        maximum_group_harm = -np.inf
        major_group_wins = 0
        for group in phase50_major_groups:
            group_indices = phase50_calibration_development[
                phase50_calibration_groups[
                    phase50_calibration_development
                ] == group
            ]
            raw_group_loss = phase50_log_loss(
                phase50_calibration_y[group_indices],
                component_probability[group_indices],
            )
            calibrated_group_loss = phase50_log_loss(
                phase50_calibration_y[group_indices],
                phase50_candidate_predictions[
                    candidate_index,
                    repeat,
                    group_indices,
                ],
            )
            harm = float(calibrated_group_loss - raw_group_loss)
            maximum_group_harm = max(maximum_group_harm, harm)
            major_group_wins += int(
                raw_group_loss - calibrated_group_loss
                >= PHASE50_CALIBRATION_CONFIG[
                    "material_repeat_log_loss_gain"
                ]
            )
        repeat_group_harms.append(float(maximum_group_harm))
        repeat_metrics.append({
            "repeat": int(repeat),
            "log_loss": float(metrics["log_loss"]),
            "auroc": float(metrics["auroc"]),
            "log_loss_gain_vs_uncalibrated_component": gain,
            "maximum_major_group_harm": float(maximum_group_harm),
            "major_group_wins": int(major_group_wins),
        })

    mean_log_loss = float(np.mean([
        record["log_loss"] for record in repeat_metrics
    ]))
    mean_auroc = float(np.mean([
        record["auroc"] for record in repeat_metrics
    ]))
    mean_gain = float(raw_metrics["log_loss"] - mean_log_loss)
    maximum_group_harm = float(np.max(repeat_group_harms))
    is_identity = candidate["calibration"] == "identity"
    safe = bool(
        is_identity
        or (
            repeat_wins
            >= PHASE50_CALIBRATION_CONFIG["minimum_repeat_wins"]
            and worst_repeat_regret
            <= PHASE50_CALIBRATION_CONFIG["maximum_repeat_regret"]
            and maximum_group_harm
            <= PHASE50_CALIBRATION_CONFIG[
                "maximum_major_group_harm"
            ]
        )
    )
    return {
        "candidate_index": int(candidate_index),
        "specification": dict(candidate),
        "mean_repeated_log_loss": mean_log_loss,
        "standard_deviation_repeated_log_loss": float(np.std([
            record["log_loss"] for record in repeat_metrics
        ], ddof=1)),
        "mean_repeated_auroc": mean_auroc,
        "standard_deviation_repeated_auroc": float(np.std([
            record["auroc"] for record in repeat_metrics
        ], ddof=1)),
        "uncalibrated_component_log_loss": float(raw_metrics["log_loss"]),
        "uncalibrated_component_auroc": float(raw_metrics["auroc"]),
        "mean_log_loss_gain_vs_uncalibrated_component": mean_gain,
        "mean_auroc_change_vs_uncalibrated_component": float(
            mean_auroc - raw_metrics["auroc"]
        ),
        "repeat_wins": int(repeat_wins),
        "repeat_count": int(phase50_repeat_count),
        "worst_repeat_regret": float(worst_repeat_regret),
        "major_group_count": int(len(phase50_major_groups)),
        "maximum_major_group_harm": maximum_group_harm,
        "safe": safe,
        "repeat_metrics": repeat_metrics,
    }


phase50_candidate_summaries = [
    phase50_candidate_summary(candidate_index)
    for candidate_index in range(len(phase50_candidate_grid))
]
phase50_safe_candidates = [
    record for record in phase50_candidate_summaries if record["safe"]
]
assert phase50_safe_candidates

phase50_raw_component_records = [
    record
    for record in phase50_candidate_summaries
    if record["specification"]["calibration"] == "identity"
]
phase50_best_raw_component = min(
    phase50_raw_component_records,
    key=lambda record: (
        record["mean_repeated_log_loss"],
        -record["mean_repeated_auroc"],
        record["candidate_index"],
    ),
)
phase50_safe_minimum = min(
    record["mean_repeated_log_loss"]
    for record in phase50_safe_candidates
)
phase50_selection_band = [
    record
    for record in phase50_safe_candidates
    if record["mean_repeated_log_loss"]
    <= phase50_safe_minimum
    + PHASE50_CALIBRATION_CONFIG["selection_log_loss_band"]
]
phase50_provisional_selection = max(
    phase50_selection_band,
    key=lambda record: (
        record["mean_repeated_auroc"],
        -record["mean_repeated_log_loss"],
        -record["candidate_index"],
    ),
)

phase50_gain_over_best_raw = float(
    phase50_best_raw_component["mean_repeated_log_loss"]
    - phase50_provisional_selection["mean_repeated_log_loss"]
)
phase50_auroc_change_vs_best_raw = float(
    phase50_provisional_selection["mean_repeated_auroc"]
    - phase50_best_raw_component["mean_repeated_auroc"]
)
phase50_calibration_advanced = bool(
    phase50_provisional_selection["specification"]["calibration"]
    != "identity"
    and phase50_gain_over_best_raw
    >= PHASE50_CALIBRATION_CONFIG["minimum_calibration_gain"]
    and phase50_auroc_change_vs_best_raw
    >= -PHASE50_CALIBRATION_CONFIG["maximum_auroc_deficit"]
)
phase50_selected_candidate = (
    phase50_provisional_selection
    if phase50_calibration_advanced
    else phase50_best_raw_component
)

phase50_selected_component_name = phase50_selected_candidate[
    "specification"
]["component"]
phase50_selected_method = phase50_selected_candidate[
    "specification"
]["calibration"]
phase50_full_development_state = phase50_fit_calibrator(
    phase50_calibration_y[phase50_calibration_development],
    phase50_components[phase50_selected_component_name][
        phase50_calibration_development
    ],
    phase50_selected_method,
)

phase50_state_directory = Path(
    "/kaggle/working/phase50_private_validation"
)
phase50_state_directory.mkdir(parents=True, exist_ok=True)
phase50_calibration_state_path = (
    phase50_state_directory / "phase50_calibration_state.json"
)
phase50_calibration_temporary_path = (
    phase50_state_directory / "phase50_calibration_state.tmp.json"
)
phase50_calibration_state = {
    "phase": "phase50_development_only_monotonic_calibration",
    "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
    "component": phase50_selected_component_name,
    "method": phase50_selected_method,
    "intercept": float(phase50_full_development_state["intercept"]),
    "slope": float(phase50_full_development_state["slope"]),
    "known_protocol_policy": "apply_selected_monotonic_calibrator",
    "unknown_protocol_policy": "phase12c_anchor_only_unchanged",
    "future_audit_used": False,
}
with phase50_calibration_temporary_path.open("w", encoding="utf-8") as file:
    json.dump(phase50_calibration_state, file, indent=2, sort_keys=True)
    file.write("\n")
os.replace(
    phase50_calibration_temporary_path,
    phase50_calibration_state_path,
)
assert phase50_calibration_state_path.is_file()

PHASE50_COMPONENTS_PRIVATE = {
    key: value.copy() for key, value in phase50_components.items()
}
PHASE50_CANDIDATE_SUMMARIES_PRIVATE = copy.deepcopy(
    phase50_candidate_summaries
)
PHASE50_SELECTED_CALIBRATION_PRIVATE = copy.deepcopy(
    phase50_calibration_state
)
PHASE50_CALIBRATION_ADVANCED_PRIVATE = bool(
    phase50_calibration_advanced
)

phase50_calibration_report = {
    "phase": "phase50_repeated_known_protocol_component_calibration",
    "status": (
        "calibration_frozen_development_only"
        if phase50_calibration_advanced
        else "no_calibration_advanced"
    ),
    "development_case_count": int(
        phase50_calibration_development.size
    ),
    "future_audit": {
        "case_count": int(phase50_calibration_audit.size),
        "labels_used": False,
        "probabilities_scored": False,
        "still_locked": True,
    },
    "evaluation_contract": {
        "repeat_count": int(phase50_repeat_count),
        "fold_count_per_repeat": int(phase50_fold_count),
        "calibrator_fit_count": int(len(phase50_fit_records)),
        "raw_component_predictions_retrained_per_repeat": False,
        "interpretation": (
            "repeated folds estimate calibrator and metric stability; "
            "they do not retrain the frozen base components"
        ),
    },
    "component_sources": {
        "phase12c": str(phase50_phase12c_path),
        "phase36": "PHASE36_OOF_PRIVATE",
        "phase39": "PHASE39_OOF_PRIVATE",
        "phase43_deployment_matched_oof": (
            "phase42_reference_probability"
        ),
    },
    "candidate_search": {
        "component_count": int(len(phase50_components)),
        "calibration_method_count": int(
            len(phase50_calibration_methods)
        ),
        "candidate_count": int(len(phase50_candidate_grid)),
        "safe_candidate_count": int(len(phase50_safe_candidates)),
        "split_candidate_count": 1,
        "methods": list(phase50_calibration_methods),
    },
    "raw_components": phase50_raw_component_records,
    "best_raw_component": phase50_best_raw_component,
    "provisional_selection": phase50_provisional_selection,
    "selected": phase50_selected_candidate,
    "selection_comparison": {
        "log_loss_gain_over_best_raw": phase50_gain_over_best_raw,
        "auroc_change_vs_best_raw": phase50_auroc_change_vs_best_raw,
        "calibration_advanced": bool(phase50_calibration_advanced),
    },
    "portable_development_state": phase50_calibration_state,
    "selection_thresholds": {
        key: value
        for key, value in PHASE50_CALIBRATION_CONFIG.items()
        if key not in {"probability_clip"}
    },
    "auroc_interpretation": (
        "a single positive-slope deployment calibrator preserves ranking; "
        "cross-fitted pooled AUROC can vary because folds use different "
        "calibration parameters"
    ),
    "unknown_protocol_predictions_modified": False,
    "future_audit_labels_used": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase50_calibration_started,
        3,
    ),
}

print("BEGIN SANITIZED_PHASE50_CALIBRATION")
print(json.dumps(phase50_calibration_report, indent=2))
print("END SANITIZED_PHASE50_CALIBRATION")

BEGIN SANITIZED_PHASE50_CALIBRATION
{
  "phase": "phase50_repeated_known_protocol_component_calibration",
  "status": "no_calibration_advanced",
  "development_case_count": 1160,
  "future_audit": {
    "case_count": 202,
    "labels_used": false,
    "probabilities_scored": false,
    "still_locked": true
  },
  "evaluation_contract": {
    "repeat_count": 5,
    "fold_count_per_repeat": 5,
    "calibrator_fit_count": 300,
    "raw_component_predictions_retrained_per_repeat": false,
    "interpretation": "repeated folds estimate calibrator and metric stability; they do not retrain the frozen base components"
  },
  "component_sources": {
    "phase12c": "/kaggle/working/phase32_phase12c_oof_float64.npy",
    "phase36": "PHASE36_OOF_PRIVATE",
    "phase39": "PHASE39_OOF_PRIVATE",
    "phase43_deployment_matched_oof": "phase42_reference_probability"
  },
  "candidate_search": {
    "component_count": 4,
    "calibration_method_count": 3,
    "candidate_count": 12,
    "safe_candidate_co

In [3]:
# Phase 51 cell 151A

import gc
import importlib.util
import json
import sys
import time
import zipfile
from pathlib import Path

import numpy as np


phase51_started = time.perf_counter()

PHASE51_CONFIG = {
    "case_count": 1362,
    "view_count": 6,
    "patch_grid": [14, 14],
    "patch_token_count": 196,
    "embedding_dimension": 384,
    "sample_cases_per_group": 4,
    "candidate_lora_ranks": [4, 8],
    "candidate_rank_loss_weights": [0.0, 0.05],
    "residual_cap": 0.5,
    "maximum_epoch_count": 12,
    "minimum_epoch_count": 3,
    "early_stopping_patience": 2,
    "minimum_development_log_loss_gain": 0.005,
    "minimum_development_auroc_gain": 0.002,
    "minimum_repeat_wins": 4,
    "maximum_worst_repeat_regret": 0.003,
    "maximum_major_group_harm": 0.015,
}


# -------------------------------------------------------------------------
# Phase50 is now the only admissible selection harness.  In particular, this
# cell never indexes the incrementally locked audit rows and never reads a
# label vector.
# -------------------------------------------------------------------------

phase51_required_names = [
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase50_groups",
    "phase50_calibration_report",
]
phase51_missing_names = [
    name for name in phase51_required_names if name not in globals()
]
assert not phase51_missing_names, {
    "message": (
        "Phase51 requires accepted Cells 150A and 150B in the current "
        "kernel. Run those cells first."
    ),
    "missing": phase51_missing_names,
}

assert phase50_calibration_report["status"] == "no_calibration_advanced", {
    "message": "Unexpected Phase50 calibration status; audit manually.",
    "status": phase50_calibration_report.get("status"),
}
assert not bool(
    phase50_calibration_report["future_audit"]["labels_used"]
)
assert bool(
    phase50_calibration_report["future_audit"]["still_locked"]
)

phase51_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51_future_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51_known_fold_assignment = np.asarray(
    PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE,
    dtype=np.int8,
)
phase51_groups = np.asarray(phase50_groups, dtype=np.int64).reshape(-1)

assert phase51_groups.shape == (PHASE51_CONFIG["case_count"],)
assert phase51_known_fold_assignment.shape == (5, 1362)
assert np.intersect1d(
    phase51_development_indices,
    phase51_future_audit_indices,
).size == 0
assert np.all(
    phase51_known_fold_assignment[:, phase51_future_audit_indices] == -1
)


# -------------------------------------------------------------------------
# Resolve the accepted DINOv3 backbone without relying on one fragile alias.
# PHASE39_BACKBONE_PRIVATE is preferred when it survived in memory.  If all
# notebook aliases were intentionally released, reconstruct the same backbone
# offline from the accepted staged package (or accepted archive fallback).
# -------------------------------------------------------------------------

phase51_backbone_names = [
    "PHASE39_BACKBONE_PRIVATE",
    "PHASE32_DINOV3_BACKBONE",
    "PHASE30_DINOV3_BACKBONE",
]
phase51_backbone_records = []
for phase51_name in phase51_backbone_names:
    phase51_value = globals().get(phase51_name)
    if phase51_value is None:
        continue
    if not hasattr(phase51_value, "forward_features"):
        continue
    if not hasattr(phase51_value, "blocks"):
        continue
    phase51_backbone_records.append((phase51_name, phase51_value))

phase51_backbone_load_source = "existing_notebook_global"
phase51_runtime_stage = None
phase51_runtime_assets_verified = False
phase51_archive_extracted = False

if not phase51_backbone_records:
    try:
        import torch
    except ImportError as phase51_torch_error:
        raise AssertionError({
            "message": (
                "PyTorch is required to reload the accepted DINOv3 "
                "backbone offline."
            ),
            "error": str(phase51_torch_error),
        }) from None

    phase51_stage_candidates = [
        Path("/kaggle/working/phase43_experimental_submission"),
        Path("/kaggle/working/phase42_experimental_submission"),
        Path("/kaggle/working/phase40_experimental_submission"),
        Path("/kaggle/working/phase39_experimental_submission"),
        Path("/kaggle/working/phase30_submission"),
    ]

    phase51_valid_stages = [
        path
        for path in phase51_stage_candidates
        if path.is_dir()
        and (path / "phase30_accepted_main.py").is_file()
        and (path / "phase30_manifest.json").is_file()
    ]

    if phase51_valid_stages:
        phase51_runtime_stage = phase51_valid_stages[0]
        phase51_backbone_load_source = (
            "accepted_staging_directory"
        )
    else:
        phase51_archive_candidates = [
            Path("/kaggle/working/phase43_submission.zip"),
            Path("/kaggle/working/phase42_submission.zip"),
            Path("/kaggle/working/phase40_submission.zip"),
            Path("/kaggle/working/phase39_submission.zip"),
            Path("/kaggle/working/phase30_submission.zip"),
        ]
        phase51_valid_archives = [
            path for path in phase51_archive_candidates if path.is_file()
        ]
        assert phase51_valid_archives, {
            "message": (
                "No initialized DINOv3 backbone, accepted staging "
                "directory, or accepted submission archive was found."
            ),
            "checked_backbone_names": phase51_backbone_names,
            "checked_staging_directories": [
                str(path) for path in phase51_stage_candidates
            ],
            "checked_archives": [
                str(path) for path in phase51_archive_candidates
            ],
        }

        phase51_runtime_archive = phase51_valid_archives[0]
        phase51_runtime_stage = Path(
            "/kaggle/working/phase51_offline_runtime_extract"
        )
        phase51_runtime_stage.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(phase51_runtime_archive, "r") as archive:
            assert archive.testzip() is None, {
                "message": "Accepted runtime archive integrity failed.",
                "archive": phase51_runtime_archive.name,
            }
            phase51_stage_resolved = phase51_runtime_stage.resolve()
            for phase51_member in archive.infolist():
                phase51_target = (
                    phase51_runtime_stage / phase51_member.filename
                ).resolve()
                assert (
                    phase51_target == phase51_stage_resolved
                    or phase51_stage_resolved in phase51_target.parents
                ), {
                    "message": "Unsafe path in accepted runtime archive.",
                    "member": phase51_member.filename,
                }
            archive.extractall(phase51_runtime_stage)

        phase51_archive_extracted = True
        phase51_backbone_load_source = (
            "accepted_archive_offline_extraction"
        )

    phase51_legacy_path = (
        phase51_runtime_stage / "phase30_accepted_main.py"
    )
    assert phase51_legacy_path.is_file()
    assert (
        phase51_runtime_stage / "phase30_manifest.json"
    ).is_file()

    phase51_stage_string = str(phase51_runtime_stage)
    if phase51_stage_string not in sys.path:
        sys.path.insert(0, phase51_stage_string)

    phase51_module_name = "phase51_accepted_phase30_runtime"
    phase51_module_specification = (
        importlib.util.spec_from_file_location(
            phase51_module_name,
            phase51_legacy_path,
        )
    )
    assert (
        phase51_module_specification is not None
        and phase51_module_specification.loader is not None
    ), {
        "message": "Could not create the accepted runtime module spec."
    }
    phase51_legacy = importlib.util.module_from_spec(
        phase51_module_specification
    )
    sys.modules[phase51_module_name] = phase51_legacy
    phase51_module_specification.loader.exec_module(phase51_legacy)

    phase51_runtime_manifest = phase51_legacy.load_runtime_manifest()
    phase51_legacy.verify_runtime_assets(phase51_runtime_manifest)
    phase51_runtime_assets_verified = True

    phase51_device = globals().get("PHASE39_DEVICE")
    if phase51_device is None:
        phase51_device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

    PHASE51_BACKBONE_PRIVATE = phase51_legacy.load_dinov3(
        phase51_runtime_manifest,
        phase51_device,
    )
    PHASE39_BACKBONE_PRIVATE = PHASE51_BACKBONE_PRIVATE
    phase51_backbone_records.append((
        "PHASE51_OFFLINE_RELOAD",
        PHASE51_BACKBONE_PRIVATE,
    ))
    gc.collect()

phase51_backbone_name, PHASE51_BACKBONE_PRIVATE = (
    phase51_backbone_records[0]
)
phase51_backbone_aliases = [
    name for name, value in phase51_backbone_records
    if value is PHASE51_BACKBONE_PRIVATE
]

phase51_blocks = list(PHASE51_BACKBONE_PRIVATE.blocks)
assert len(phase51_blocks) >= 2, {
    "message": "DINOv3 backbone has no separable final transformer block.",
    "block_count": len(phase51_blocks),
}
phase51_final_block = phase51_blocks[-1]

phase51_embedding_dimension = None
for phase51_attribute in ["embed_dim", "num_features"]:
    phase51_candidate = getattr(
        PHASE51_BACKBONE_PRIVATE,
        phase51_attribute,
        None,
    )
    if phase51_candidate is not None:
        phase51_embedding_dimension = int(phase51_candidate)
        break
if phase51_embedding_dimension is None:
    phase51_norm = getattr(PHASE51_BACKBONE_PRIVATE, "norm", None)
    phase51_normalized_shape = getattr(
        phase51_norm,
        "normalized_shape",
        None,
    )
    if isinstance(phase51_normalized_shape, (tuple, list)):
        phase51_embedding_dimension = int(phase51_normalized_shape[-1])
    elif phase51_normalized_shape is not None:
        phase51_embedding_dimension = int(phase51_normalized_shape)

assert phase51_embedding_dimension == PHASE51_CONFIG[
    "embedding_dimension"
], {
    "message": "Unexpected DINOv3 embedding dimension.",
    "observed": phase51_embedding_dimension,
    "expected": PHASE51_CONFIG["embedding_dimension"],
}

phase51_prepare_method = None
for phase51_method_name in [
    "prepare_tokens_with_masks",
    "prepare_tokens",
]:
    if callable(getattr(PHASE51_BACKBONE_PRIVATE, phase51_method_name, None)):
        phase51_prepare_method = phase51_method_name
        break
assert phase51_prepare_method is not None, {
    "message": (
        "Could not resolve the DINOv3 token-preparation method needed "
        "for exact penultimate-token extraction."
    ),
}


def phase51_parameter_count(module):
    return int(sum(
        int(parameter.numel()) for parameter in module.parameters()
    ))


phase51_total_parameter_count = phase51_parameter_count(
    PHASE51_BACKBONE_PRIVATE
)
assert phase51_total_parameter_count == 21601152, {
    "message": "Accepted DINOv3 parameter-count contract failed.",
    "observed": phase51_total_parameter_count,
    "expected": 21601152,
}
phase51_final_block_parameter_count = phase51_parameter_count(
    phase51_final_block
)
phase51_final_norm_parameter_count = phase51_parameter_count(
    PHASE51_BACKBONE_PRIVATE.norm
)

phase51_linear_module_names = [
    name
    for name, module in phase51_final_block.named_modules()
    if module.__class__.__name__.lower() == "linear"
]
assert phase51_linear_module_names, {
    "message": "No linear modules were found in the final DINOv3 block."
}

# DINOv3 may expose its extra prefix tokens under different release-specific
# names.  Resolve the count from tensors first, then scalar attributes.
phase51_storage_token_count = 0
for phase51_attribute in [
    "storage_tokens",
    "register_tokens",
]:
    phase51_token_value = getattr(
        PHASE51_BACKBONE_PRIVATE,
        phase51_attribute,
        None,
    )
    phase51_token_shape = getattr(phase51_token_value, "shape", None)
    if phase51_token_shape is not None and len(phase51_token_shape) >= 2:
        phase51_storage_token_count = int(phase51_token_shape[-2])
        break
if phase51_storage_token_count == 0:
    for phase51_attribute in [
        "num_storage_tokens",
        "num_register_tokens",
    ]:
        phase51_value = getattr(
            PHASE51_BACKBONE_PRIVATE,
            phase51_attribute,
            None,
        )
        if phase51_value is not None:
            phase51_storage_token_count = int(phase51_value)
            break

phase51_prefix_token_count = 1 + phase51_storage_token_count
phase51_penultimate_token_count = (
    PHASE51_CONFIG["patch_token_count"] + phase51_prefix_token_count
)


# -------------------------------------------------------------------------
# Resolve and inspect the accepted final normalized patch-token cache.  This
# cache is useful for data-contract checks, but it is explicitly not treated
# as the input to the last block.  Cell 151B will create a separate cache from
# the output of blocks[:-1].
# -------------------------------------------------------------------------

phase51_dense_candidates = []
phase51_explicit_dense_path = globals().get(
    "PHASE51_DENSE_CACHE_PATH_PRIVATE"
)
if phase51_explicit_dense_path is not None:
    phase51_dense_candidates.append(Path(phase51_explicit_dense_path))
phase51_dense_candidates.extend([
    Path("/kaggle/working/phase32_bilateral_dense_float16.npy"),
])

phase51_unique_dense_candidates = []
phase51_seen_dense_paths = set()
for phase51_candidate in phase51_dense_candidates:
    phase51_key = str(phase51_candidate.expanduser().resolve(strict=False))
    if phase51_key in phase51_seen_dense_paths:
        continue
    phase51_seen_dense_paths.add(phase51_key)
    if phase51_candidate.is_file():
        phase51_unique_dense_candidates.append(phase51_candidate)

assert len(phase51_unique_dense_candidates) == 1, {
    "message": (
        "Could not uniquely resolve the accepted Phase32 bilateral dense "
        "token cache."
    ),
    "valid_candidate_count": len(phase51_unique_dense_candidates),
    "valid_candidates": [
        str(path) for path in phase51_unique_dense_candidates
    ],
}

PHASE51_DENSE_CACHE_PATH_PRIVATE = phase51_unique_dense_candidates[0]
PHASE51_DENSE_CACHE_PRIVATE = np.load(
    PHASE51_DENSE_CACHE_PATH_PRIVATE,
    mmap_mode="r",
    allow_pickle=False,
)
phase51_expected_dense_shape = (
    PHASE51_CONFIG["case_count"],
    PHASE51_CONFIG["view_count"],
    PHASE51_CONFIG["patch_grid"][0],
    PHASE51_CONFIG["patch_grid"][1],
    PHASE51_CONFIG["embedding_dimension"],
)
assert PHASE51_DENSE_CACHE_PRIVATE.shape == phase51_expected_dense_shape, {
    "message": "Unexpected Phase32 bilateral dense cache shape.",
    "observed": list(PHASE51_DENSE_CACHE_PRIVATE.shape),
    "expected": list(phase51_expected_dense_shape),
}
assert PHASE51_DENSE_CACHE_PRIVATE.dtype == np.float16, {
    "message": "The accepted dense cache must preserve float16 boundary.",
    "observed": str(PHASE51_DENSE_CACHE_PRIVATE.dtype),
}

phase51_metadata_path = Path(
    "/kaggle/working/phase32_bilateral_dense_metadata.json"
)
phase51_metadata = None
if phase51_metadata_path.is_file():
    with phase51_metadata_path.open("r", encoding="utf-8") as handle:
        phase51_metadata = json.load(handle)
    assert isinstance(phase51_metadata, dict)


# -------------------------------------------------------------------------
# Deterministic, label-free development-only sample.  This is an integrity
# audit, not model selection.  No future-audit row is indexed below.
# -------------------------------------------------------------------------

phase51_sample_indices_list = []
for phase51_group in np.unique(phase51_groups[phase51_development_indices]):
    phase51_group_indices = phase51_development_indices[
        phase51_groups[phase51_development_indices] == phase51_group
    ]
    phase51_take = min(
        PHASE51_CONFIG["sample_cases_per_group"],
        int(phase51_group_indices.size),
    )
    if phase51_take == 1:
        phase51_positions = np.asarray([0], dtype=np.int64)
    else:
        phase51_positions = np.linspace(
            0,
            phase51_group_indices.size - 1,
            num=phase51_take,
            dtype=np.int64,
        )
    phase51_sample_indices_list.extend(
        phase51_group_indices[phase51_positions].tolist()
    )

phase51_sample_indices = np.asarray(
    sorted(set(phase51_sample_indices_list)),
    dtype=np.int64,
)
assert phase51_sample_indices.size > 0
assert np.intersect1d(
    phase51_sample_indices,
    phase51_future_audit_indices,
).size == 0
assert np.all(np.isin(
    phase51_sample_indices,
    phase51_development_indices,
))

phase51_dense_sample = np.asarray(
    PHASE51_DENSE_CACHE_PRIVATE[phase51_sample_indices],
    dtype=np.float32,
)
assert phase51_dense_sample.shape[1:] == phase51_expected_dense_shape[1:]
assert np.all(np.isfinite(phase51_dense_sample))

phase51_prompt_mask = None
phase51_prompt_mask_source = "central_8_by_8_contract"
phase51_existing_prompt_mask = globals().get("PHASE32_REAL_PROMPT_MASK")
if phase51_existing_prompt_mask is not None:
    phase51_existing_prompt_mask = np.asarray(
        phase51_existing_prompt_mask,
        dtype=bool,
    )
    if (
        phase51_existing_prompt_mask.shape == (14, 14)
        and int(phase51_existing_prompt_mask.sum()) == 64
    ):
        phase51_prompt_mask = phase51_existing_prompt_mask.copy()
        phase51_prompt_mask_source = "PHASE32_REAL_PROMPT_MASK"
if phase51_prompt_mask is None:
    phase51_prompt_mask = np.zeros((14, 14), dtype=bool)
    phase51_prompt_mask[3:11, 3:11] = True
assert phase51_prompt_mask.shape == (14, 14)
assert int(phase51_prompt_mask.sum()) == 64

phase51_global_pool = np.mean(
    phase51_dense_sample,
    axis=(2, 3),
    dtype=np.float32,
)
phase51_prompt_pool = np.mean(
    phase51_dense_sample[:, :, phase51_prompt_mask, :],
    axis=2,
    dtype=np.float32,
)
phase51_prompt_contrast = phase51_prompt_pool - phase51_global_pool
phase51_pooled_contract = np.concatenate([
    phase51_global_pool,
    phase51_prompt_pool,
    phase51_prompt_contrast,
], axis=2)
assert phase51_pooled_contract.shape == (
    phase51_sample_indices.size,
    6,
    3 * PHASE51_CONFIG["embedding_dimension"],
)

phase51_flat_pooled = phase51_pooled_contract.reshape(
    phase51_sample_indices.size,
    -1,
).astype(np.float64)
phase51_centered_pooled = (
    phase51_flat_pooled
    - np.mean(phase51_flat_pooled, axis=0, keepdims=True)
)
phase51_singular_values = np.linalg.svd(
    phase51_centered_pooled,
    full_matrices=False,
    compute_uv=False,
)
phase51_variance = phase51_singular_values ** 2
phase51_variance_fraction = phase51_variance / max(
    float(np.sum(phase51_variance)),
    np.finfo(np.float64).tiny,
)
phase51_cumulative_variance = np.cumsum(phase51_variance_fraction)
phase51_effective_rank_95 = int(
    np.searchsorted(phase51_cumulative_variance, 0.95) + 1
)
phase51_effective_rank_99 = int(
    np.searchsorted(phase51_cumulative_variance, 0.99) + 1
)

phase51_coordinate_std = np.std(
    phase51_flat_pooled,
    axis=0,
    ddof=1,
)
phase51_token_norm = np.linalg.norm(
    phase51_dense_sample,
    axis=-1,
)
phase51_prompt_norm = np.linalg.norm(
    phase51_prompt_pool,
    axis=-1,
)
phase51_global_norm = np.linalg.norm(
    phase51_global_pool,
    axis=-1,
)
phase51_prompt_to_global_norm_ratio = (
    phase51_prompt_norm
    / np.maximum(phase51_global_norm, np.finfo(np.float32).tiny)
)
phase51_view_dispersion = np.mean(
    np.std(phase51_global_pool, axis=1, ddof=1),
    axis=1,
)

del phase51_dense_sample


# -------------------------------------------------------------------------
# Freeze the next experiment before labels are used.  The final DINO block is
# adapted from cached penultimate tokens; the rest of DINOv3 remains frozen.
# A zero-initialized bounded residual head makes epoch zero exactly Phase43.
# -------------------------------------------------------------------------

phase51_candidate_count = int(
    len(PHASE51_CONFIG["candidate_lora_ranks"])
    * len(PHASE51_CONFIG["candidate_rank_loss_weights"])
)
phase51_cross_validated_fit_count = int(
    phase51_candidate_count
    * phase51_known_fold_assignment.shape[0]
    * 5
)

phase51_penultimate_cache_shape = (
    PHASE51_CONFIG["case_count"],
    PHASE51_CONFIG["view_count"],
    phase51_penultimate_token_count,
    PHASE51_CONFIG["embedding_dimension"],
)
phase51_penultimate_cache_size_gb = float(
    np.prod(phase51_penultimate_cache_shape)
    * np.dtype(np.float16).itemsize
    / (1024 ** 3)
)

PHASE51_DINO_CONTRACT_PRIVATE = {
    "backbone_global": phase51_backbone_name,
    "prepare_method": phase51_prepare_method,
    "block_count": int(len(phase51_blocks)),
    "embedding_dimension": int(phase51_embedding_dimension),
    "patch_token_count": int(PHASE51_CONFIG["patch_token_count"]),
    "prefix_token_count": int(phase51_prefix_token_count),
    "penultimate_token_count": int(phase51_penultimate_token_count),
    "dense_cache_path": str(PHASE51_DENSE_CACHE_PATH_PRIVATE),
    "prompt_mask": phase51_prompt_mask.copy(),
    "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
}

phase51_report = {
    "phase": "phase51_fold_specific_low_rank_dinov3_contract",
    "status": "accepted",
    "phase50_decision": {
        "calibration_advanced": False,
        "development_leader": "phase43_deployment_matched_oof",
        "development_log_loss": float(
            phase50_calibration_report["selected"][
                "mean_repeated_log_loss"
            ]
        ),
        "development_auroc": float(
            phase50_calibration_report["selected"][
                "mean_repeated_auroc"
            ]
        ),
        "next_required_change": "new_task_specific_ranking_signal",
    },
    "validation_lock": {
        "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
        "development_case_count": int(phase51_development_indices.size),
        "future_audit_case_count": int(phase51_future_audit_indices.size),
        "future_audit_still_locked": True,
        "future_audit_rows_read": False,
        "future_audit_labels_used": False,
    },
    "dinov3_backbone": {
        "resolved_global": phase51_backbone_name,
        "same_object_aliases": phase51_backbone_aliases,
        "load_source": phase51_backbone_load_source,
        "runtime_stage": (
            phase51_runtime_stage.name
            if phase51_runtime_stage is not None else None
        ),
        "runtime_assets_verified_in_this_cell": bool(
            phase51_runtime_assets_verified
        ),
        "accepted_archive_extracted": bool(
            phase51_archive_extracted
        ),
        "total_parameter_count": int(phase51_total_parameter_count),
        "block_count": int(len(phase51_blocks)),
        "final_block_parameter_count": int(
            phase51_final_block_parameter_count
        ),
        "final_norm_parameter_count": int(
            phase51_final_norm_parameter_count
        ),
        "final_block_linear_module_count": int(
            len(phase51_linear_module_names)
        ),
        "token_preparation_method": phase51_prepare_method,
        "embedding_dimension": int(phase51_embedding_dimension),
        "storage_token_count": int(phase51_storage_token_count),
        "prefix_token_count": int(phase51_prefix_token_count),
        "patch_token_count": int(PHASE51_CONFIG["patch_token_count"]),
        "penultimate_token_count": int(phase51_penultimate_token_count),
        "currently_frozen": bool(all(
            not bool(parameter.requires_grad)
            for parameter in PHASE51_BACKBONE_PRIVATE.parameters()
        )),
    },
    "accepted_dense_cache": {
        "file": PHASE51_DENSE_CACHE_PATH_PRIVATE.name,
        "shape": list(PHASE51_DENSE_CACHE_PRIVATE.shape),
        "dtype": str(PHASE51_DENSE_CACHE_PRIVATE.dtype),
        "semantics": "final_normalized_patch_tokens_not_penultimate_tokens",
        "metadata_present": bool(phase51_metadata is not None),
        "metadata_top_level_keys": (
            sorted(str(key) for key in phase51_metadata.keys())
            if phase51_metadata is not None else []
        ),
    },
    "label_free_development_sample": {
        "case_count": int(phase51_sample_indices.size),
        "group_count": int(np.unique(
            phase51_groups[phase51_sample_indices]
        ).size),
        "prompt_mask_source": phase51_prompt_mask_source,
        "prompt_token_count": int(phase51_prompt_mask.sum()),
        "all_values_finite": True,
        "token_norm_quantiles": {
            "q01": float(np.quantile(phase51_token_norm, 0.01)),
            "q50": float(np.quantile(phase51_token_norm, 0.50)),
            "q99": float(np.quantile(phase51_token_norm, 0.99)),
        },
        "pooled_coordinate_std_quantiles": {
            "minimum": float(np.min(phase51_coordinate_std)),
            "q25": float(np.quantile(phase51_coordinate_std, 0.25)),
            "median": float(np.median(phase51_coordinate_std)),
            "q75": float(np.quantile(phase51_coordinate_std, 0.75)),
            "maximum": float(np.max(phase51_coordinate_std)),
        },
        "pooled_effective_rank_95_percent": int(
            phase51_effective_rank_95
        ),
        "pooled_effective_rank_99_percent": int(
            phase51_effective_rank_99
        ),
        "prompt_to_global_norm_ratio": {
            "mean": float(np.mean(
                phase51_prompt_to_global_norm_ratio
            )),
            "minimum": float(np.min(
                phase51_prompt_to_global_norm_ratio
            )),
            "maximum": float(np.max(
                phase51_prompt_to_global_norm_ratio
            )),
        },
        "mean_view_dispersion": float(np.mean(
            phase51_view_dispersion
        )),
    },
    "penultimate_cache_plan": {
        "source": "output_of_frozen_dinov3_blocks_except_final_block",
        "shape": list(phase51_penultimate_cache_shape),
        "storage_dtype": "float16",
        "estimated_size_gb": phase51_penultimate_cache_size_gb,
        "cache_contains_labels": False,
        "cache_contains_probabilities": False,
        "include_in_submission": False,
    },
    "adaptation_plan": {
        "anchor": "phase43_deployment_matched_probability",
        "trainable_backbone_scope": "lora_on_final_dinov3_block_only",
        "frozen_backbone_scope": "patch_embed_and_all_earlier_blocks",
        "pooling": (
            "shared_prompt_global_contrast_pooling_across_six_views"
        ),
        "head": "zero_initialized_bounded_logit_residual",
        "residual_cap": float(PHASE51_CONFIG["residual_cap"]),
        "lora_ranks": list(PHASE51_CONFIG["candidate_lora_ranks"]),
        "rank_loss_weights": list(
            PHASE51_CONFIG["candidate_rank_loss_weights"]
        ),
        "candidate_count": int(phase51_candidate_count),
        "repeated_fold_count": 25,
        "planned_supervised_fit_count": int(
            phase51_cross_validated_fit_count
        ),
        "explicit_acquisition_group_feature": False,
        "unknown_protocol_policy": "phase12c_anchor_only",
    },
    "promotion_thresholds": {
        "minimum_development_log_loss_gain": float(
            PHASE51_CONFIG["minimum_development_log_loss_gain"]
        ),
        "minimum_development_auroc_gain": float(
            PHASE51_CONFIG["minimum_development_auroc_gain"]
        ),
        "minimum_repeat_wins": int(
            PHASE51_CONFIG["minimum_repeat_wins"]
        ),
        "maximum_worst_repeat_regret": float(
            PHASE51_CONFIG["maximum_worst_repeat_regret"]
        ),
        "maximum_major_group_harm": float(
            PHASE51_CONFIG["maximum_major_group_harm"]
        ),
        "audit_evaluation": (
            "exactly_once_only_after_complete_development_freeze"
        ),
    },
    "labels_used": False,
    "future_audit_labels_used": False,
    "public_leaderboard_used": False,
    "training_dense_cache_read": True,
    "training_voxel_cache_read": False,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_indices_displayed": False,
    "case_level_features_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase51_started, 3),
}

print("BEGIN SANITIZED_PHASE51_DINO_CONTRACT")
print(json.dumps(phase51_report, indent=2))
print("END SANITIZED_PHASE51_DINO_CONTRACT")

AssertionError: {'message': 'Phase51 requires accepted Cells 150A and 150B in the current kernel. Run those cells first.', 'missing': ['PHASE50_DEVELOPMENT_INDICES_PRIVATE', 'PHASE50_FUTURE_AUDIT_INDICES_PRIVATE', 'PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE', 'PHASE50_PARTITION_SHA256_PRIVATE', 'phase50_groups', 'phase50_calibration_report']}

In [66]:
# Phase51 Cell 151B

import json
import os
import time
from pathlib import Path

import numpy as np
import torch


phase51b_started = time.perf_counter()

PHASE51_PENULTIMATE_CACHE_CONFIG = {
    "case_count": 1362,
    "view_count": 6,
    "patch_grid": [14, 14],
    "patch_token_count": 196,
    "embedding_dimension": 384,
    "case_batch_size": 2,
    "progress_interval": 128,
    "direct_float16_maximum_error": 0.005,
    "cached_roundtrip_maximum_error": 0.02,
}


# -------------------------------------------------------------------------
# Preconditions from accepted Cell 151A.
# -------------------------------------------------------------------------

phase51b_required_names = [
    "PHASE51_BACKBONE_PRIVATE",
    "PHASE51_DINO_CONTRACT_PRIVATE",
    "PHASE51_DENSE_CACHE_PRIVATE",
    "PHASE51_DENSE_CACHE_PATH_PRIVATE",
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase51_sample_indices",
    "phase51_report",
]
phase51b_missing_names = [
    name for name in phase51b_required_names if name not in globals()
]
assert not phase51b_missing_names, {
    "message": "Run the accepted corrected Cell 151A first.",
    "missing": phase51b_missing_names,
}

assert phase51_report["status"] == "accepted"
assert not bool(phase51_report["future_audit_labels_used"])
assert (
    str(PHASE51_DINO_CONTRACT_PRIVATE["partition_sha256"])
    == str(PHASE50_PARTITION_SHA256_PRIVATE)
)

phase51b_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51b_future_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51b_parity_indices = np.asarray(
    phase51_sample_indices,
    dtype=np.int64,
).reshape(-1)

assert phase51b_development_indices.size == 1160
assert phase51b_future_audit_indices.size == 202
assert np.intersect1d(
    phase51b_development_indices,
    phase51b_future_audit_indices,
).size == 0
assert np.all(np.isin(
    phase51b_parity_indices,
    phase51b_development_indices,
))
assert np.intersect1d(
    phase51b_parity_indices,
    phase51b_future_audit_indices,
).size == 0


# -------------------------------------------------------------------------
# Resolve the accepted runtime view builder.  Cell 151A creates phase51_legacy
# during offline reload.  The alternatives cover a still-resident older
# initialization without importing another package copy.
# -------------------------------------------------------------------------

phase51b_legacy_candidates = []
for phase51b_name in ["phase51_legacy", "legacy"]:
    phase51b_value = globals().get(phase51b_name)
    if (
        phase51b_value is not None
        and callable(getattr(
            phase51b_value,
            "phase30_make_bilateral_views",
            None,
        ))
    ):
        phase51b_legacy_candidates.append((phase51b_name, phase51b_value))

phase51b_runtime_module = globals().get("phase39_runtime_module")
if phase51b_runtime_module is not None:
    phase51b_value = getattr(phase51b_runtime_module, "legacy", None)
    if (
        phase51b_value is not None
        and callable(getattr(
            phase51b_value,
            "phase30_make_bilateral_views",
            None,
        ))
    ):
        phase51b_legacy_candidates.append((
            "phase39_runtime_module.legacy",
            phase51b_value,
        ))

assert phase51b_legacy_candidates, {
    "message": (
        "Accepted Phase30 bilateral-view builder is unavailable. "
        "Re-run corrected Cell 151A without deleting its globals."
    ),
}

phase51b_legacy_name, phase51b_legacy = phase51b_legacy_candidates[0]


# -------------------------------------------------------------------------
# Cache and architecture contracts.
# -------------------------------------------------------------------------

PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE = Path(
    "/kaggle/working/phase51_dinov3_penultimate_float16.npy"
)
phase51b_temporary_cache_path = Path(
    "/kaggle/working/phase51_dinov3_penultimate_float16.tmp.npy"
)
PHASE51_PENULTIMATE_METADATA_PATH_PRIVATE = Path(
    "/kaggle/working/phase51_dinov3_penultimate_metadata.json"
)
phase51b_temporary_metadata_path = Path(
    "/kaggle/working/phase51_dinov3_penultimate_metadata.tmp.json"
)
phase51b_highres_path = Path(
    "/kaggle/working/phase31_highres_float16.npy"
)

assert phase51b_highres_path.is_file(), {
    "message": "Accepted Phase31 high-resolution cache is missing.",
    "path": str(phase51b_highres_path),
}

phase51b_highres_cache = np.load(
    phase51b_highres_path,
    mmap_mode="r",
    allow_pickle=False,
)
assert phase51b_highres_cache.shape == (1362, 80, 80, 80)
assert phase51b_highres_cache.dtype == np.float16

phase51b_dense_cache = PHASE51_DENSE_CACHE_PRIVATE
assert phase51b_dense_cache.shape == (1362, 6, 14, 14, 384)
assert phase51b_dense_cache.dtype == np.float16

phase51b_backbone = PHASE51_BACKBONE_PRIVATE
phase51b_blocks = list(phase51b_backbone.blocks)
phase51b_prefix_token_count = int(
    PHASE51_DINO_CONTRACT_PRIVATE["prefix_token_count"]
)
phase51b_penultimate_token_count = int(
    PHASE51_DINO_CONTRACT_PRIVATE["penultimate_token_count"]
)

assert len(phase51b_blocks) == 12
assert phase51b_prefix_token_count == 5
assert phase51b_penultimate_token_count == 201
assert all(
    not bool(parameter.requires_grad)
    for parameter in phase51b_backbone.parameters()
)
assert not bool(phase51b_backbone.training)

phase51b_expected_cache_shape = (
    PHASE51_PENULTIMATE_CACHE_CONFIG["case_count"],
    PHASE51_PENULTIMATE_CACHE_CONFIG["view_count"],
    phase51b_penultimate_token_count,
    PHASE51_PENULTIMATE_CACHE_CONFIG["embedding_dimension"],
)

phase51b_device = next(phase51b_backbone.parameters()).device
phase51b_cuda = phase51b_device.type == "cuda"

phase51b_rope_embed = getattr(phase51b_backbone, "rope_embed", None)
phase51b_rope_sincos = (
    phase51b_rope_embed(H=14, W=14)
    if phase51b_rope_embed is not None else None
)
PHASE51_ROPE_SINCOS_PRIVATE = phase51b_rope_sincos

if phase51b_cuda:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase51b_device)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

torch.use_deterministic_algorithms(True, warn_only=True)


def phase51b_prepare_penultimate(flat_views):
    """Exact frozen DINO path up to, but excluding, transformer block 12."""
    assert flat_views.ndim == 4
    assert tuple(flat_views.shape[1:]) == (3, 224, 224)

    prepared = phase51b_backbone.prepare_tokens_with_masks(
        flat_views,
        masks=None,
    )
    assert isinstance(prepared, tuple) and len(prepared) == 2, {
        "message": "Unexpected DINOv3 token-preparation return contract.",
        "type": type(prepared).__name__,
    }
    tokens, spatial_grid = prepared
    assert tuple(int(value) for value in spatial_grid) == (14, 14), {
        "message": "Unexpected DINOv3 patch grid.",
        "observed": list(spatial_grid),
        "expected": [14, 14],
    }
    assert tokens.ndim == 3
    assert tuple(tokens.shape[1:]) == (
        phase51b_penultimate_token_count,
        384,
    )

    for block in phase51b_blocks[:-1]:
        tokens = block(tokens, phase51b_rope_sincos)

    assert tokens.ndim == 3
    assert tuple(tokens.shape[1:]) == (
        phase51b_penultimate_token_count,
        384,
    )
    return tokens


def phase51b_finish_from_penultimate(tokens):
    """Untouched final block and final norm; returns normalized patch tokens."""
    assert tokens.ndim == 3
    assert tuple(tokens.shape[1:]) == (
        phase51b_penultimate_token_count,
        384,
    )
    final_tokens = phase51b_blocks[-1](
        tokens,
        phase51b_rope_sincos,
    )
    final_tokens = phase51b_backbone.norm(final_tokens)
    patch_tokens = final_tokens[:, phase51b_prefix_token_count:, :]
    assert tuple(patch_tokens.shape[1:]) == (196, 384)
    return patch_tokens


def phase51b_build_views(case_indices):
    case_indices = np.asarray(case_indices, dtype=np.int64).reshape(-1)
    highres = np.asarray(
        phase51b_highres_cache[case_indices],
        dtype=np.float32,
    )
    assert highres.shape == (len(case_indices), 80, 80, 80)
    assert np.all(np.isfinite(highres))
    volume = torch.from_numpy(highres).to(
        phase51b_device,
        non_blocking=phase51b_cuda,
    )
    views = phase51b_legacy.phase30_make_bilateral_views(volume)
    assert tuple(views.shape) == (
        len(case_indices), 6, 3, 224, 224
    )
    return views.reshape(-1, 3, 224, 224)


# -------------------------------------------------------------------------
# Create the cache atomically when absent.  If a valid cache already exists,
# reuse it and still run an independent round-trip parity audit below.
# -------------------------------------------------------------------------

phase51b_cache_reused = False
phase51b_direct_float_error_maximum = None
phase51b_direct_float_error_sum = 0.0
phase51b_direct_float_error_count = 0
phase51b_direct_quantized_error_maximum = None
phase51b_direct_quantized_error_sum = 0.0
phase51b_direct_quantized_error_count = 0

if PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE.is_file():
    PHASE51_PENULTIMATE_CACHE_PRIVATE = np.load(
        PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE,
        mmap_mode="r",
        allow_pickle=False,
    )
    assert (
        PHASE51_PENULTIMATE_CACHE_PRIVATE.shape
        == phase51b_expected_cache_shape
    ), {
        "message": "Existing Phase51 cache has an unexpected shape.",
        "observed": list(PHASE51_PENULTIMATE_CACHE_PRIVATE.shape),
        "expected": list(phase51b_expected_cache_shape),
    }
    assert PHASE51_PENULTIMATE_CACHE_PRIVATE.dtype == np.float16
    phase51b_cache_reused = True
else:
    phase51b_cache_writer = np.lib.format.open_memmap(
        phase51b_temporary_cache_path,
        mode="w+",
        dtype=np.float16,
        shape=phase51b_expected_cache_shape,
    )

    phase51b_batch_size = int(
        PHASE51_PENULTIMATE_CACHE_CONFIG["case_batch_size"]
    )
    phase51b_next_progress = int(
        PHASE51_PENULTIMATE_CACHE_CONFIG["progress_interval"]
    )

    with torch.inference_mode():
        for phase51b_start in range(
            0,
            PHASE51_PENULTIMATE_CACHE_CONFIG["case_count"],
            phase51b_batch_size,
        ):
            phase51b_stop = min(
                phase51b_start + phase51b_batch_size,
                PHASE51_PENULTIMATE_CACHE_CONFIG["case_count"],
            )
            phase51b_indices = np.arange(
                phase51b_start,
                phase51b_stop,
                dtype=np.int64,
            )
            phase51b_flat_views = phase51b_build_views(phase51b_indices)
            phase51b_penultimate = phase51b_prepare_penultimate(
                phase51b_flat_views
            )
            phase51b_final_patch = phase51b_finish_from_penultimate(
                phase51b_penultimate
            )

            phase51b_penultimate_numpy = (
                phase51b_penultimate
                .reshape(
                    len(phase51b_indices),
                    6,
                    phase51b_penultimate_token_count,
                    384,
                )
                .float()
                .cpu()
                .numpy()
            )
            phase51b_cache_writer[phase51b_indices] = (
                phase51b_penultimate_numpy.astype(np.float16)
            )

            phase51b_final_numpy = (
                phase51b_final_patch
                .reshape(
                    len(phase51b_indices),
                    6,
                    14,
                    14,
                    384,
                )
                .float()
                .cpu()
                .numpy()
            )
            phase51b_reference = np.asarray(
                phase51b_dense_cache[phase51b_indices],
                dtype=np.float32,
            )

            phase51b_float_error = np.abs(
                phase51b_final_numpy - phase51b_reference
            )
            phase51b_quantized_error = np.abs(
                phase51b_final_numpy.astype(np.float16).astype(np.float32)
                - phase51b_reference
            )

            phase51b_float_batch_maximum = float(np.max(
                phase51b_float_error
            ))
            phase51b_quantized_batch_maximum = float(np.max(
                phase51b_quantized_error
            ))
            phase51b_direct_float_error_maximum = (
                phase51b_float_batch_maximum
                if phase51b_direct_float_error_maximum is None
                else max(
                    phase51b_direct_float_error_maximum,
                    phase51b_float_batch_maximum,
                )
            )
            phase51b_direct_quantized_error_maximum = (
                phase51b_quantized_batch_maximum
                if phase51b_direct_quantized_error_maximum is None
                else max(
                    phase51b_direct_quantized_error_maximum,
                    phase51b_quantized_batch_maximum,
                )
            )
            phase51b_direct_float_error_sum += float(np.sum(
                phase51b_float_error,
                dtype=np.float64,
            ))
            phase51b_direct_float_error_count += int(
                phase51b_float_error.size
            )
            phase51b_direct_quantized_error_sum += float(np.sum(
                phase51b_quantized_error,
                dtype=np.float64,
            ))
            phase51b_direct_quantized_error_count += int(
                phase51b_quantized_error.size
            )

            del phase51b_flat_views
            del phase51b_penultimate
            del phase51b_final_patch
            del phase51b_penultimate_numpy
            del phase51b_final_numpy
            del phase51b_reference
            del phase51b_float_error
            del phase51b_quantized_error

            if (
                phase51b_stop >= phase51b_next_progress
                or phase51b_stop
                == PHASE51_PENULTIMATE_CACHE_CONFIG["case_count"]
            ):
                print(
                    "Phase51 penultimate cache: "
                    f"{phase51b_stop}/"
                    f"{PHASE51_PENULTIMATE_CACHE_CONFIG['case_count']}",
                    flush=True,
                )
                while phase51b_next_progress <= phase51b_stop:
                    phase51b_next_progress += int(
                        PHASE51_PENULTIMATE_CACHE_CONFIG[
                            "progress_interval"
                        ]
                    )

    phase51b_cache_writer.flush()
    del phase51b_cache_writer
    os.replace(
        phase51b_temporary_cache_path,
        PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE,
    )

    PHASE51_PENULTIMATE_CACHE_PRIVATE = np.load(
        PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE,
        mmap_mode="r",
        allow_pickle=False,
    )

assert PHASE51_PENULTIMATE_CACHE_PRIVATE.shape == (
    phase51b_expected_cache_shape
)
assert PHASE51_PENULTIMATE_CACHE_PRIVATE.dtype == np.float16

if phase51b_direct_quantized_error_maximum is not None:
    assert (
        phase51b_direct_quantized_error_maximum
        <= PHASE51_PENULTIMATE_CACHE_CONFIG[
            "direct_float16_maximum_error"
        ]
    ), {
        "message": (
            "Manual DINO forward path does not reconstruct the accepted "
            "final dense-token cache."
        ),
        "maximum_error": phase51b_direct_quantized_error_maximum,
        "threshold": PHASE51_PENULTIMATE_CACHE_CONFIG[
            "direct_float16_maximum_error"
        ],
    }


# -------------------------------------------------------------------------
# Independent float16-cache round trip.  Only the fixed label-free development
# sample from Cell 151A is audited; locked audit rows are not inspected.
# -------------------------------------------------------------------------

phase51b_roundtrip_error_maximum = 0.0
phase51b_roundtrip_error_sum = 0.0
phase51b_roundtrip_error_count = 0
phase51b_roundtrip_batch_size = 8

with torch.inference_mode():
    for phase51b_start in range(
        0,
        phase51b_parity_indices.size,
        phase51b_roundtrip_batch_size,
    ):
        phase51b_indices = phase51b_parity_indices[
            phase51b_start:phase51b_start + phase51b_roundtrip_batch_size
        ]
        phase51b_cached = np.asarray(
            PHASE51_PENULTIMATE_CACHE_PRIVATE[phase51b_indices],
            dtype=np.float32,
        )
        phase51b_cached_tensor = torch.from_numpy(
            phase51b_cached.reshape(
                -1,
                phase51b_penultimate_token_count,
                384,
            )
        ).to(
            phase51b_device,
            non_blocking=phase51b_cuda,
        )
        phase51b_reconstructed = phase51b_finish_from_penultimate(
            phase51b_cached_tensor
        )
        phase51b_reconstructed = (
            phase51b_reconstructed
            .reshape(len(phase51b_indices), 6, 14, 14, 384)
            .float()
            .cpu()
            .numpy()
            .astype(np.float16)
            .astype(np.float32)
        )
        phase51b_reference = np.asarray(
            phase51b_dense_cache[phase51b_indices],
            dtype=np.float32,
        )
        phase51b_error = np.abs(
            phase51b_reconstructed - phase51b_reference
        )
        phase51b_roundtrip_error_maximum = max(
            phase51b_roundtrip_error_maximum,
            float(np.max(phase51b_error)),
        )
        phase51b_roundtrip_error_sum += float(np.sum(
            phase51b_error,
            dtype=np.float64,
        ))
        phase51b_roundtrip_error_count += int(phase51b_error.size)

assert (
    phase51b_roundtrip_error_maximum
    <= PHASE51_PENULTIMATE_CACHE_CONFIG[
        "cached_roundtrip_maximum_error"
    ]
), {
    "message": "Float16 penultimate-token cache parity failed.",
    "maximum_error": phase51b_roundtrip_error_maximum,
    "threshold": PHASE51_PENULTIMATE_CACHE_CONFIG[
        "cached_roundtrip_maximum_error"
    ],
}


# -------------------------------------------------------------------------
# Persist sanitized cache metadata atomically.  No labels, probabilities,
# UIDs, acquisition-group identities, or case rows are written.
# -------------------------------------------------------------------------

phase51b_cache_size_gb = float(
    PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE.stat().st_size
    / (1024 ** 3)
)
phase51b_direct_float_mean = (
    None
    if phase51b_direct_float_error_count == 0
    else float(
        phase51b_direct_float_error_sum
        / phase51b_direct_float_error_count
    )
)
phase51b_direct_quantized_mean = (
    None
    if phase51b_direct_quantized_error_count == 0
    else float(
        phase51b_direct_quantized_error_sum
        / phase51b_direct_quantized_error_count
    )
)
phase51b_roundtrip_error_mean = float(
    phase51b_roundtrip_error_sum
    / phase51b_roundtrip_error_count
)

phase51b_metadata = {
    "schema_version": 1,
    "phase": "phase51_dinov3_penultimate_token_cache",
    "status": "accepted",
    "shape": list(PHASE51_PENULTIMATE_CACHE_PRIVATE.shape),
    "dtype": str(PHASE51_PENULTIMATE_CACHE_PRIVATE.dtype),
    "backbone_parameter_count": 21601152,
    "block_count": 12,
    "cached_block_output_count": 11,
    "prefix_token_count": int(phase51b_prefix_token_count),
    "patch_token_count": 196,
    "embedding_dimension": 384,
    "source_highres_cache": phase51b_highres_path.name,
    "reference_dense_cache": Path(
        PHASE51_DENSE_CACHE_PATH_PRIVATE
    ).name,
    "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
    "contains_labels": False,
    "contains_probabilities": False,
    "contains_uids": False,
}

with phase51b_temporary_metadata_path.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(phase51b_metadata, handle, indent=2)
    handle.write("\n")
os.replace(
    phase51b_temporary_metadata_path,
    PHASE51_PENULTIMATE_METADATA_PATH_PRIVATE,
)

assert PHASE51_PENULTIMATE_METADATA_PATH_PRIVATE.is_file()

PHASE51_PENULTIMATE_CACHE_CONTRACT_PRIVATE = {
    **phase51b_metadata,
    "cache_path": str(PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE),
    "metadata_path": str(PHASE51_PENULTIMATE_METADATA_PATH_PRIVATE),
}

phase51b_peak_vram_mb = (
    float(torch.cuda.max_memory_allocated(phase51b_device) / (1024 ** 2))
    if phase51b_cuda else 0.0
)

phase51b_report = {
    "phase": "phase51_exact_dinov3_penultimate_token_cache",
    "status": "accepted",
    "source": {
        "highres_cache": phase51b_highres_path.name,
        "highres_shape": list(phase51b_highres_cache.shape),
        "highres_dtype": str(phase51b_highres_cache.dtype),
        "view_builder": phase51b_legacy_name,
        "backbone_parameter_count": int(sum(
            parameter.numel()
            for parameter in phase51b_backbone.parameters()
        )),
        "block_count": int(len(phase51b_blocks)),
        "frozen_block_count": 12,
        "rope_position_encoding_used": bool(
            phase51b_rope_sincos is not None
        ),
    },
    "cache": {
        "file": PHASE51_PENULTIMATE_CACHE_PATH_PRIVATE.name,
        "metadata_file": (
            PHASE51_PENULTIMATE_METADATA_PATH_PRIVATE.name
        ),
        "shape": list(PHASE51_PENULTIMATE_CACHE_PRIVATE.shape),
        "dtype": str(PHASE51_PENULTIMATE_CACHE_PRIVATE.dtype),
        "size_gb": phase51b_cache_size_gb,
        "cache_reused": bool(phase51b_cache_reused),
        "cached_semantics": (
            "output_after_dinov3_blocks_0_through_10_before_block_11"
        ),
        "include_in_submission": False,
    },
    "direct_forward_parity": {
        "evaluated_during_creation": bool(
            phase51b_direct_quantized_error_maximum is not None
        ),
        "float_vs_accepted_float16_reference": {
            "maximum": phase51b_direct_float_error_maximum,
            "mean": phase51b_direct_float_mean,
        },
        "float16_vs_accepted_float16_reference": {
            "maximum": phase51b_direct_quantized_error_maximum,
            "mean": phase51b_direct_quantized_mean,
        },
        "maximum_allowed_float16_error": float(
            PHASE51_PENULTIMATE_CACHE_CONFIG[
                "direct_float16_maximum_error"
            ]
        ),
    },
    "cached_roundtrip_parity": {
        "development_sample_case_count": int(
            phase51b_parity_indices.size
        ),
        "future_audit_case_count": 0,
        "maximum": float(phase51b_roundtrip_error_maximum),
        "mean": phase51b_roundtrip_error_mean,
        "maximum_allowed": float(
            PHASE51_PENULTIMATE_CACHE_CONFIG[
                "cached_roundtrip_maximum_error"
            ]
        ),
    },
    "validation_lock": {
        "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
        "future_audit_images_encoded_by_fixed_frozen_transform": True,
        "future_audit_features_inspected_for_selection": False,
        "future_audit_labels_used": False,
        "future_audit_metrics_computed": False,
    },
    "execution": {
        "device": str(phase51b_device),
        "case_batch_size": int(
            PHASE51_PENULTIMATE_CACHE_CONFIG["case_batch_size"]
        ),
        "mixed_precision": False,
        "peak_vram_mb": phase51b_peak_vram_mb,
    },
    "labels_used": False,
    "training_voxel_cache_read": True,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_tokens_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase51b_started, 3),
}

print("BEGIN SANITIZED_PHASE51_PENULTIMATE_CACHE")
print(json.dumps(phase51b_report, indent=2))
print("END SANITIZED_PHASE51_PENULTIMATE_CACHE")

Phase51 penultimate cache: 128/1362
Phase51 penultimate cache: 256/1362
Phase51 penultimate cache: 384/1362
Phase51 penultimate cache: 512/1362
Phase51 penultimate cache: 640/1362
Phase51 penultimate cache: 768/1362
Phase51 penultimate cache: 896/1362
Phase51 penultimate cache: 1024/1362
Phase51 penultimate cache: 1152/1362
Phase51 penultimate cache: 1280/1362
Phase51 penultimate cache: 1362/1362
BEGIN SANITIZED_PHASE51_PENULTIMATE_CACHE
{
  "phase": "phase51_exact_dinov3_penultimate_token_cache",
  "status": "accepted",
  "source": {
    "highres_cache": "phase31_highres_float16.npy",
    "highres_shape": [
      1362,
      80,
      80,
      80
    ],
    "highres_dtype": "float16",
    "view_builder": "phase51_legacy",
    "backbone_parameter_count": 21601152,
    "block_count": 12,
    "frozen_block_count": 12,
    "rope_position_encoding_used": true
  },
  "cache": {
    "file": "phase51_dinov3_penultimate_float16.npy",
    "metadata_file": "phase51_dinov3_penultimate_metadata.j

In [69]:
# Phase51 Cell 151C 

import copy
import json
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


phase51c_started = time.perf_counter()

PHASE51_ADAPTER_CONFIG = {
    "contract_seed": 511501,
    "contract_lora_rank": 4,
    "lora_alpha": 4.0,
    "view_projection_dimension": 96,
    "head_hidden_dimension": 128,
    "residual_cap": 0.5,
    "contract_batch_size": 8,
    "lora_learning_rate": 2.0e-5,
    "pooling_learning_rate": 1.0e-4,
    "head_learning_rate": 2.0e-4,
    "weight_decay": 0.02,
    "gradient_clip": 1.0,
    "candidate_lora_ranks": [4, 8],
    "candidate_rank_loss_weights": [0.0, 0.05],
    "maximum_epochs": 12,
    "minimum_epochs": 3,
    "early_stopping_patience": 2,
}


# -------------------------------------------------------------------------
# Accepted state and lock contract.
# -------------------------------------------------------------------------

phase51c_required_names = [
    "PHASE51_BACKBONE_PRIVATE",
    "PHASE51_DINO_CONTRACT_PRIVATE",
    "PHASE51_ROPE_SINCOS_PRIVATE",
    "PHASE51_PENULTIMATE_CACHE_PRIVATE",
    "PHASE51_PENULTIMATE_CACHE_CONTRACT_PRIVATE",
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase50_groups",
    "phase50_y",
    "phase50_phase43_probability",
    "phase51b_report",
]
phase51c_missing_names = [
    name for name in phase51c_required_names if name not in globals()
]
assert not phase51c_missing_names, {
    "message": "Run accepted Cells 151A and 151B first.",
    "missing": phase51c_missing_names,
}

assert phase51b_report["status"] == "accepted"
assert (
    str(PHASE51_PENULTIMATE_CACHE_CONTRACT_PRIVATE["partition_sha256"])
    == str(PHASE50_PARTITION_SHA256_PRIVATE)
)

phase51c_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51c_future_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51c_fold_assignment = np.asarray(
    PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE,
    dtype=np.int8,
)
phase51c_groups = np.asarray(phase50_groups, dtype=np.int64).reshape(-1)
phase51c_labels = np.asarray(phase50_y, dtype=np.float32).reshape(-1)
phase51c_anchor_probability = np.asarray(
    phase50_phase43_probability,
    dtype=np.float64,
).reshape(-1)

assert phase51c_development_indices.size == 1160
assert phase51c_future_audit_indices.size == 202
assert phase51c_fold_assignment.shape == (5, 1362)
assert phase51c_groups.shape == (1362,)
assert phase51c_labels.shape == (1362,)
assert phase51c_anchor_probability.shape == (1362,)
assert np.intersect1d(
    phase51c_development_indices,
    phase51c_future_audit_indices,
).size == 0
assert np.all(
    phase51c_fold_assignment[:, phase51c_future_audit_indices] == -1
)
assert np.all(np.isfinite(phase51c_anchor_probability))
assert np.all(
    (phase51c_anchor_probability > 0.0)
    & (phase51c_anchor_probability < 1.0)
)

phase51c_cache = PHASE51_PENULTIMATE_CACHE_PRIVATE
assert phase51c_cache.shape == (1362, 6, 201, 384)
assert phase51c_cache.dtype == np.float16

phase51c_backbone = PHASE51_BACKBONE_PRIVATE
phase51c_backbone_blocks = list(phase51c_backbone.blocks)
assert len(phase51c_backbone_blocks) == 12
assert all(
    not bool(parameter.requires_grad)
    for parameter in phase51c_backbone.parameters()
)
phase51c_device = next(phase51c_backbone.parameters()).device
phase51c_cuda = phase51c_device.type == "cuda"
phase51c_autocast_dtype = torch.bfloat16

if phase51c_cuda:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase51c_device)

torch.manual_seed(PHASE51_ADAPTER_CONFIG["contract_seed"])
if phase51c_cuda:
    torch.cuda.manual_seed_all(PHASE51_ADAPTER_CONFIG["contract_seed"])
torch.use_deterministic_algorithms(True, warn_only=True)


def phase51c_probability_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1.0e-6,
        1.0 - 1.0e-6,
    )
    return np.log(probability) - np.log1p(-probability)


class Phase51LoRALinear(nn.Module):
    """Frozen accepted linear layer plus a zero-initialized low-rank delta."""

    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        assert isinstance(base_linear, nn.Linear)
        assert int(rank) > 0

        self.base = base_linear
        # Preserve the public nn.Linear interface.  DINOv3's attention
        # implementation reads qkv.in_features after qkv(x), so a LoRA
        # replacement must be computationally and structurally transparent.
        self.in_features = int(base_linear.in_features)
        self.out_features = int(base_linear.out_features)
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank

        for parameter in self.base.parameters():
            parameter.requires_grad_(False)

        self.lora_a = nn.Parameter(torch.empty(
            self.rank,
            int(self.base.in_features),
        ))
        self.lora_b = nn.Parameter(torch.zeros(
            int(self.base.out_features),
            self.rank,
        ))
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5.0))

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def extra_repr(self):
        return (
            f"in_features={self.in_features}, "
            f"out_features={self.out_features}, "
            f"bias={self.bias is not None}, "
            f"rank={self.rank}, alpha={self.alpha}"
        )

    def forward(self, inputs):
        base_output = self.base(inputs)
        low_rank = F.linear(inputs, self.lora_a)
        low_rank = F.linear(low_rank, self.lora_b)
        return base_output + self.scaling * low_rank


def phase51c_replace_final_block_linears(block, rank, alpha):
    """Replace a snapshot of all nn.Linear leaves; never mutate while iterating."""
    linear_records = [
        (name, module)
        for name, module in list(block.named_modules())
        if name and isinstance(module, nn.Linear)
    ]
    assert linear_records, {
        "message": "No nn.Linear leaves found in accepted DINO block 12."
    }

    replaced_names = []
    for name, module in linear_records:
        if "." in name:
            parent_name, child_name = name.rsplit(".", 1)
            parent = block.get_submodule(parent_name)
        else:
            parent = block
            child_name = name
        setattr(
            parent,
            child_name,
            Phase51LoRALinear(module, rank=rank, alpha=alpha),
        )
        replaced_names.append(name)
    return replaced_names


class Phase51DinoLastBlockResidualAdapter(nn.Module):
    """LoRA-adapted DINO block 12 with ordered six-view residual pooling."""

    def __init__(
        self,
        accepted_final_block,
        accepted_final_norm,
        prompt_mask,
        lora_rank,
        lora_alpha,
        view_projection_dimension,
        head_hidden_dimension,
        residual_cap,
    ):
        super().__init__()
        self.prefix_token_count = 5
        self.patch_token_count = 196
        self.embedding_dimension = 384
        self.view_count = 6
        self.residual_cap = float(residual_cap)

        self.final_block = copy.deepcopy(accepted_final_block)
        self.final_norm = copy.deepcopy(accepted_final_norm)
        for parameter in self.final_block.parameters():
            parameter.requires_grad_(False)
        for parameter in self.final_norm.parameters():
            parameter.requires_grad_(False)

        self.lora_module_names = phase51c_replace_final_block_linears(
            self.final_block,
            rank=int(lora_rank),
            alpha=float(lora_alpha),
        )

        prompt_mask = torch.as_tensor(prompt_mask, dtype=torch.bool)
        assert tuple(prompt_mask.shape) == (14, 14)
        assert int(prompt_mask.sum().item()) == 64
        self.register_buffer(
            "prompt_mask_flat",
            prompt_mask.reshape(-1),
            persistent=True,
        )

        per_view_input_dimension = 3 * self.embedding_dimension
        self.view_projection = nn.Sequential(
            nn.LayerNorm(per_view_input_dimension),
            nn.Linear(
                per_view_input_dimension,
                int(view_projection_dimension),
            ),
            nn.GELU(),
        )

        complete_dimension = (
            self.view_count * int(view_projection_dimension)
            + 3 * int(view_projection_dimension)
        )
        self.head = nn.Sequential(
            nn.LayerNorm(complete_dimension),
            nn.Linear(complete_dimension, int(head_hidden_dimension)),
            nn.GELU(),
            nn.Linear(int(head_hidden_dimension), 1),
        )

        output_layer = self.head[-1]
        nn.init.zeros_(output_layer.weight)
        nn.init.zeros_(output_layer.bias)

    def forward(self, penultimate_tokens, anchor_logit, rope_sincos):
        assert penultimate_tokens.ndim == 4
        batch_size = int(penultimate_tokens.shape[0])
        assert tuple(penultimate_tokens.shape[1:]) == (6, 201, 384)
        assert anchor_logit.shape == (batch_size,)

        flat_tokens = penultimate_tokens.reshape(-1, 201, 384)
        final_tokens = self.final_block(flat_tokens, rope_sincos)
        final_tokens = self.final_norm(final_tokens)
        patch_tokens = final_tokens[:, self.prefix_token_count:, :]
        patch_tokens = patch_tokens.reshape(
            batch_size,
            self.view_count,
            self.patch_token_count,
            self.embedding_dimension,
        )

        global_pool = torch.mean(patch_tokens, dim=2)
        prompt_pool = torch.mean(
            patch_tokens[:, :, self.prompt_mask_flat, :],
            dim=2,
        )
        prompt_contrast = prompt_pool - global_pool
        per_view = torch.cat(
            [global_pool, prompt_pool, prompt_contrast],
            dim=-1,
        )
        projected = self.view_projection(per_view)

        ordered_views = projected.reshape(batch_size, -1)
        view_mean = torch.mean(projected, dim=1)
        view_std = torch.std(projected, dim=1, unbiased=False)
        view_maximum = torch.amax(projected, dim=1)
        representation = torch.cat(
            [ordered_views, view_mean, view_std, view_maximum],
            dim=-1,
        )

        raw_residual = self.head(representation).squeeze(-1)
        bounded_residual = self.residual_cap * torch.tanh(raw_residual)
        final_logit = anchor_logit + bounded_residual

        return {
            "logit": final_logit,
            "raw_residual": raw_residual,
            "bounded_residual": bounded_residual,
            "representation": representation,
            "patch_tokens": patch_tokens,
        }


def phase51c_gradient_record(model, prefix):
    gradients = []
    for name, parameter in model.named_parameters():
        if not name.startswith(prefix):
            continue
        if parameter.grad is None:
            continue
        gradients.append(parameter.grad.detach().float())
    if not gradients:
        return {
            "tensor_count": 0,
            "norm": 0.0,
            "all_finite": True,
        }
    squared_norm = sum(
        float(torch.sum(gradient * gradient).item())
        for gradient in gradients
    )
    return {
        "tensor_count": int(len(gradients)),
        "norm": float(math.sqrt(squared_norm)),
        "all_finite": bool(all(
            torch.isfinite(gradient).all().item()
            for gradient in gradients
        )),
    }


def phase51c_lora_gradient_record(model, suffix):
    gradients = []
    for name, parameter in model.named_parameters():
        if f".{suffix}" not in name:
            continue
        if parameter.grad is None:
            continue
        gradients.append(parameter.grad.detach().float())
    if not gradients:
        return {
            "tensor_count": 0,
            "norm": 0.0,
            "all_finite": True,
        }
    squared_norm = sum(
        float(torch.sum(gradient * gradient).item())
        for gradient in gradients
    )
    return {
        "tensor_count": int(len(gradients)),
        "norm": float(math.sqrt(squared_norm)),
        "all_finite": bool(all(
            torch.isfinite(gradient).all().item()
            for gradient in gradients
        )),
    }


# -------------------------------------------------------------------------
# Instantiate the rank-4 contract model.  Selection will rebuild fresh models
# for all four preregistered specifications; this object is never a candidate.
# -------------------------------------------------------------------------

phase51c_prompt_mask = np.asarray(
    PHASE51_DINO_CONTRACT_PRIVATE["prompt_mask"],
    dtype=bool,
)

PHASE51_ADAPTER_CLASS_PRIVATE = Phase51DinoLastBlockResidualAdapter
phase51c_expected_lora_names = [
    name
    for name, module in list(
        phase51c_backbone_blocks[-1].named_modules()
    )
    if name and isinstance(module, nn.Linear)
]
assert phase51c_expected_lora_names, {
    "message": "The accepted final DINO block contains no linear leaves."
}
phase51c_contract_model = Phase51DinoLastBlockResidualAdapter(
    accepted_final_block=phase51c_backbone_blocks[-1],
    accepted_final_norm=phase51c_backbone.norm,
    prompt_mask=phase51c_prompt_mask,
    lora_rank=PHASE51_ADAPTER_CONFIG["contract_lora_rank"],
    lora_alpha=PHASE51_ADAPTER_CONFIG["lora_alpha"],
    view_projection_dimension=(
        PHASE51_ADAPTER_CONFIG["view_projection_dimension"]
    ),
    head_hidden_dimension=(
        PHASE51_ADAPTER_CONFIG["head_hidden_dimension"]
    ),
    residual_cap=PHASE51_ADAPTER_CONFIG["residual_cap"],
).to(phase51c_device)

phase51c_lora_names = list(
    phase51c_contract_model.lora_module_names
)
assert phase51c_lora_names == phase51c_expected_lora_names, {
    "message": "The LoRA replacement did not cover every original linear leaf.",
    "expected": phase51c_expected_lora_names,
    "observed": phase51c_lora_names,
}

phase51c_linear_interface_records = []
for phase51c_linear_name in phase51c_lora_names:
    phase51c_original_linear = (
        phase51c_backbone_blocks[-1].get_submodule(
            phase51c_linear_name
        )
    )
    phase51c_adapted_linear = (
        phase51c_contract_model.final_block.get_submodule(
            phase51c_linear_name
        )
    )
    assert isinstance(phase51c_original_linear, nn.Linear)
    assert isinstance(phase51c_adapted_linear, Phase51LoRALinear)
    assert phase51c_adapted_linear.in_features == int(
        phase51c_original_linear.in_features
    )
    assert phase51c_adapted_linear.out_features == int(
        phase51c_original_linear.out_features
    )
    assert phase51c_adapted_linear.weight is (
        phase51c_adapted_linear.base.weight
    )
    assert phase51c_adapted_linear.bias is (
        phase51c_adapted_linear.base.bias
    )
    phase51c_linear_interface_records.append({
        "module": phase51c_linear_name,
        "in_features": int(phase51c_adapted_linear.in_features),
        "out_features": int(phase51c_adapted_linear.out_features),
        "bias": bool(phase51c_adapted_linear.bias is not None),
    })

phase51c_total_parameter_count = int(sum(
    parameter.numel()
    for parameter in phase51c_contract_model.parameters()
))
phase51c_trainable_parameter_count = int(sum(
    parameter.numel()
    for parameter in phase51c_contract_model.parameters()
    if parameter.requires_grad
))
phase51c_frozen_parameter_count = (
    phase51c_total_parameter_count - phase51c_trainable_parameter_count
)

phase51c_lora_parameter_count = int(sum(
    parameter.numel()
    for name, parameter in phase51c_contract_model.named_parameters()
    if ".lora_" in name
))
phase51c_pooling_parameter_count = int(sum(
    parameter.numel()
    for name, parameter in phase51c_contract_model.named_parameters()
    if name.startswith("view_projection.")
))
phase51c_head_parameter_count = int(sum(
    parameter.numel()
    for name, parameter in phase51c_contract_model.named_parameters()
    if name.startswith("head.")
))

assert phase51c_trainable_parameter_count == (
    phase51c_lora_parameter_count
    + phase51c_pooling_parameter_count
    + phase51c_head_parameter_count
)
assert all(
    not parameter.requires_grad
    for name, parameter in phase51c_contract_model.named_parameters()
    if ".base." in name or name.startswith("final_norm.")
)


# -------------------------------------------------------------------------
# Development-fit-only identity and three-step gradient-flow contract.
# The locked audit rows are neither indexed nor scored.
# -------------------------------------------------------------------------

phase51c_contract_fit_pool = phase51c_development_indices[
    phase51c_fold_assignment[0, phase51c_development_indices] != 0
]
phase51c_contract_indices = []
for phase51c_label in [0, 1]:
    phase51c_local = phase51c_contract_fit_pool[
        phase51c_labels[phase51c_contract_fit_pool] == phase51c_label
    ]
    phase51c_take = PHASE51_ADAPTER_CONFIG["contract_batch_size"] // 2
    assert phase51c_local.size >= phase51c_take
    phase51c_positions = np.linspace(
        0,
        phase51c_local.size - 1,
        num=phase51c_take,
        dtype=np.int64,
    )
    phase51c_contract_indices.extend(
        phase51c_local[phase51c_positions].tolist()
    )
phase51c_contract_indices = np.asarray(
    sorted(phase51c_contract_indices),
    dtype=np.int64,
)

assert phase51c_contract_indices.size == (
    PHASE51_ADAPTER_CONFIG["contract_batch_size"]
)
assert np.intersect1d(
    phase51c_contract_indices,
    phase51c_future_audit_indices,
).size == 0

phase51c_contract_tokens = torch.from_numpy(np.asarray(
    phase51c_cache[phase51c_contract_indices],
    dtype=np.float32,
)).to(
    phase51c_device,
    non_blocking=phase51c_cuda,
)
phase51c_contract_anchor_logit = torch.from_numpy(
    phase51c_probability_logit(
        phase51c_anchor_probability[phase51c_contract_indices]
    ).astype(np.float32)
).to(phase51c_device)
phase51c_contract_labels = torch.from_numpy(
    phase51c_labels[phase51c_contract_indices].astype(np.float32)
).to(phase51c_device)

phase51c_contract_model.eval()
with torch.inference_mode(), torch.autocast(
    device_type=phase51c_device.type,
    dtype=phase51c_autocast_dtype,
    enabled=phase51c_cuda,
):
    phase51c_initial_output = phase51c_contract_model(
        phase51c_contract_tokens,
        phase51c_contract_anchor_logit,
        PHASE51_ROPE_SINCOS_PRIVATE,
    )

phase51c_initial_logit_error = float(torch.max(torch.abs(
    phase51c_initial_output["logit"].float()
    - phase51c_contract_anchor_logit.float()
)).item())
phase51c_initial_residual_maximum = float(torch.max(torch.abs(
    phase51c_initial_output["bounded_residual"].float()
)).item())
phase51c_initial_probability_error = float(torch.max(torch.abs(
    torch.sigmoid(phase51c_initial_output["logit"].float())
    - torch.sigmoid(phase51c_contract_anchor_logit.float())
)).item())

assert phase51c_initial_logit_error == 0.0
assert phase51c_initial_residual_maximum == 0.0
assert phase51c_initial_probability_error == 0.0

phase51c_lora_parameters = []
phase51c_pooling_parameters = []
phase51c_head_parameters = []
for phase51c_name, phase51c_parameter in (
    phase51c_contract_model.named_parameters()
):
    if not phase51c_parameter.requires_grad:
        continue
    if ".lora_" in phase51c_name:
        phase51c_lora_parameters.append(phase51c_parameter)
    elif phase51c_name.startswith("view_projection."):
        phase51c_pooling_parameters.append(phase51c_parameter)
    elif phase51c_name.startswith("head."):
        phase51c_head_parameters.append(phase51c_parameter)
    else:
        raise AssertionError({
            "message": "Unexpected trainable parameter.",
            "name": phase51c_name,
        })

phase51c_optimizer = torch.optim.AdamW([
    {
        "params": phase51c_lora_parameters,
        "lr": PHASE51_ADAPTER_CONFIG["lora_learning_rate"],
    },
    {
        "params": phase51c_pooling_parameters,
        "lr": PHASE51_ADAPTER_CONFIG["pooling_learning_rate"],
    },
    {
        "params": phase51c_head_parameters,
        "lr": PHASE51_ADAPTER_CONFIG["head_learning_rate"],
    },
], weight_decay=PHASE51_ADAPTER_CONFIG["weight_decay"])

phase51c_step_records = []
phase51c_contract_model.train()
# Keep the accepted frozen computation deterministic.  LoRA parameters still
# receive gradients in eval mode because eval() changes module behavior, not
# autograd participation.
phase51c_contract_model.final_block.eval()
phase51c_contract_model.final_norm.eval()
for phase51c_step in range(1, 4):
    phase51c_optimizer.zero_grad(set_to_none=True)
    with torch.autocast(
        device_type=phase51c_device.type,
        dtype=phase51c_autocast_dtype,
        enabled=phase51c_cuda,
    ):
        phase51c_output = phase51c_contract_model(
            phase51c_contract_tokens,
            phase51c_contract_anchor_logit,
            PHASE51_ROPE_SINCOS_PRIVATE,
        )
        phase51c_loss = F.binary_cross_entropy_with_logits(
            phase51c_output["logit"].float(),
            phase51c_contract_labels.float(),
        )
    phase51c_loss.backward()

    phase51c_record = {
        "step": int(phase51c_step),
        "loss": float(phase51c_loss.detach().item()),
        "lora_a_gradient": phase51c_lora_gradient_record(
            phase51c_contract_model,
            "lora_a",
        ),
        "lora_b_gradient": phase51c_lora_gradient_record(
            phase51c_contract_model,
            "lora_b",
        ),
        "pooling_gradient": phase51c_gradient_record(
            phase51c_contract_model,
            "view_projection.",
        ),
        "head_gradient": phase51c_gradient_record(
            phase51c_contract_model,
            "head.",
        ),
        "maximum_absolute_residual": float(torch.max(torch.abs(
            phase51c_output["bounded_residual"].detach().float()
        )).item()),
    }
    phase51c_step_records.append(phase51c_record)

    torch.nn.utils.clip_grad_norm_(
        [
            parameter
            for parameter in phase51c_contract_model.parameters()
            if parameter.requires_grad
        ],
        max_norm=PHASE51_ADAPTER_CONFIG["gradient_clip"],
    )
    phase51c_optimizer.step()

assert phase51c_step_records[0]["head_gradient"]["norm"] > 0.0
assert phase51c_step_records[0]["pooling_gradient"]["norm"] == 0.0
assert phase51c_step_records[0]["lora_b_gradient"]["norm"] == 0.0
assert phase51c_step_records[1]["pooling_gradient"]["norm"] > 0.0
assert phase51c_step_records[1]["lora_b_gradient"]["norm"] > 0.0
assert phase51c_step_records[2]["lora_a_gradient"]["norm"] > 0.0
assert all(
    record[section]["all_finite"]
    for record in phase51c_step_records
    for section in [
        "lora_a_gradient",
        "lora_b_gradient",
        "pooling_gradient",
        "head_gradient",
    ]
)

PHASE51_ADAPTER_CONFIG_PRIVATE = copy.deepcopy(
    PHASE51_ADAPTER_CONFIG
)

phase51c_candidate_count = int(
    len(PHASE51_ADAPTER_CONFIG["candidate_lora_ranks"])
    * len(PHASE51_ADAPTER_CONFIG["candidate_rank_loss_weights"])
)
phase51c_planned_fit_count = int(
    phase51c_candidate_count * 5 * 5
)

phase51c_peak_vram_mb = (
    float(torch.cuda.max_memory_allocated(phase51c_device) / (1024 ** 2))
    if phase51c_cuda else 0.0
)

phase51c_report = {
    "phase": "phase51_low_rank_dinov3_last_block_adapter_contract",
    "status": "accepted",
    "architecture": {
        "input_shape": [
            PHASE51_ADAPTER_CONFIG["contract_batch_size"],
            6,
            201,
            384,
        ],
        "adapted_block": 11,
        "frozen_earlier_block_count": 11,
        "lora_target_module_count": int(len(phase51c_lora_names)),
        "lora_target_modules": phase51c_lora_names,
        "linear_interface_records": phase51c_linear_interface_records,
        "nn_linear_interface_preserved": True,
        "contract_lora_rank": int(
            PHASE51_ADAPTER_CONFIG["contract_lora_rank"]
        ),
        "lora_alpha": float(PHASE51_ADAPTER_CONFIG["lora_alpha"]),
        "rope_position_encoding_used": True,
        "pooling": (
            "ordered_six_view_prompt_global_contrast_plus_view_moments"
        ),
        "view_projection_dimension": int(
            PHASE51_ADAPTER_CONFIG["view_projection_dimension"]
        ),
        "head_hidden_dimension": int(
            PHASE51_ADAPTER_CONFIG["head_hidden_dimension"]
        ),
        "residual_cap": float(
            PHASE51_ADAPTER_CONFIG["residual_cap"]
        ),
        "anchor": "phase43_deployment_matched_logit",
        "frozen_final_block_eval_mode": True,
    },
    "parameters": {
        "total": int(phase51c_total_parameter_count),
        "trainable": int(phase51c_trainable_parameter_count),
        "frozen": int(phase51c_frozen_parameter_count),
        "lora": int(phase51c_lora_parameter_count),
        "pooling": int(phase51c_pooling_parameter_count),
        "head": int(phase51c_head_parameter_count),
    },
    "initial_identity": {
        "maximum_logit_error": phase51c_initial_logit_error,
        "maximum_probability_error": phase51c_initial_probability_error,
        "maximum_absolute_residual": phase51c_initial_residual_maximum,
        "exact": True,
    },
    "three_step_backward_contract": {
        "records": phase51c_step_records,
        "head_receives_step_1_gradient": True,
        "pooling_receives_step_2_gradient": True,
        "lora_b_receives_step_2_gradient": True,
        "lora_a_receives_step_3_gradient": True,
        "all_gradients_finite": True,
    },
    "frozen_search": {
        "lora_ranks": list(
            PHASE51_ADAPTER_CONFIG["candidate_lora_ranks"]
        ),
        "rank_loss_weights": list(
            PHASE51_ADAPTER_CONFIG["candidate_rank_loss_weights"]
        ),
        "candidate_count": int(phase51c_candidate_count),
        "repeat_count": 5,
        "fold_count_per_repeat": 5,
        "planned_fit_count": int(phase51c_planned_fit_count),
        "epoch_zero_identity_allowed": True,
        "shared_specification_across_all_25_folds": True,
    },
    "optimization": {
        "lora_learning_rate": float(
            PHASE51_ADAPTER_CONFIG["lora_learning_rate"]
        ),
        "pooling_learning_rate": float(
            PHASE51_ADAPTER_CONFIG["pooling_learning_rate"]
        ),
        "head_learning_rate": float(
            PHASE51_ADAPTER_CONFIG["head_learning_rate"]
        ),
        "weight_decay": float(
            PHASE51_ADAPTER_CONFIG["weight_decay"]
        ),
        "maximum_epochs": int(
            PHASE51_ADAPTER_CONFIG["maximum_epochs"]
        ),
        "minimum_epochs": int(
            PHASE51_ADAPTER_CONFIG["minimum_epochs"]
        ),
        "early_stopping_patience": int(
            PHASE51_ADAPTER_CONFIG["early_stopping_patience"]
        ),
        "mixed_precision": "bfloat16_on_cuda",
    },
    "validation_lock": {
        "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
        "contract_fit_case_count": int(phase51c_contract_indices.size),
        "contract_cases_from_development_fit_only": True,
        "future_audit_features_used": False,
        "future_audit_labels_used": False,
        "future_audit_metrics_computed": False,
    },
    "execution": {
        "device": str(phase51c_device),
        "peak_vram_mb": phase51c_peak_vram_mb,
    },
    "fit_labels_used_for_backward_contract": True,
    "monitor_labels_used": False,
    "future_audit_labels_used": False,
    "public_leaderboard_used": False,
    "training_penultimate_cache_read": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase51c_started, 3),
}

print("BEGIN SANITIZED_PHASE51_ADAPTER_CONTRACT")
print(json.dumps(phase51c_report, indent=2))
print("END SANITIZED_PHASE51_ADAPTER_CONTRACT")

BEGIN SANITIZED_PHASE51_ADAPTER_CONTRACT
{
  "phase": "phase51_low_rank_dinov3_last_block_adapter_contract",
  "status": "accepted",
  "architecture": {
    "input_shape": [
      8,
      6,
      201,
      384
    ],
    "adapted_block": 11,
    "frozen_earlier_block_count": 11,
    "lora_target_module_count": 4,
    "lora_target_modules": [
      "attn.qkv",
      "attn.proj",
      "mlp.fc1",
      "mlp.fc2"
    ],
    "linear_interface_records": [
      {
        "module": "attn.qkv",
        "in_features": 384,
        "out_features": 1152,
        "bias": true
      },
      {
        "module": "attn.proj",
        "in_features": 384,
        "out_features": 384,
        "bias": true
      },
      {
        "module": "mlp.fc1",
        "in_features": 384,
        "out_features": 1536,
        "bias": true
      },
      {
        "module": "mlp.fc2",
        "in_features": 1536,
        "out_features": 384,
        "bias": true
      }
    ],
    "nn_linear_interface_preserved

In [70]:
# Phase51 Cell 151D

import copy
import gc
import json
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset, Sampler


phase51d_started = time.perf_counter()

PHASE51_TRAINING_ENGINE_CONFIG = {
    "pilot_seed": 511601,
    "repeat": 0,
    "fold": 0,
    "pilot_train_case_count": 256,
    "pilot_valid_case_count": 96,
    "batch_size": 16,
    "evaluation_batch_size": 24,
    "worker_count": 0,
    "pilot_epoch_count": 2,
    "lora_rank": 8,
    "lora_alpha": 4.0,
    "rank_loss_weight": 0.05,
    "maximum_training_case_weight": 4.0,
    "gradient_clip": 1.0,
    "warmup_fraction": 0.10,
    "minimum_learning_rate_fraction": 0.10,
    "weight_decay": 0.02,
    "optimizer_betas": [0.9, 0.95],
    "maximum_allowed_restore_probability_error": 2.0e-6,
}


# -------------------------------------------------------------------------
# Accepted-state and audit-lock checks.
# -------------------------------------------------------------------------

phase51d_required_names = [
    "PHASE51_ADAPTER_CLASS_PRIVATE",
    "PHASE51_ADAPTER_CONFIG_PRIVATE",
    "PHASE51_BACKBONE_PRIVATE",
    "PHASE51_DINO_CONTRACT_PRIVATE",
    "PHASE51_ROPE_SINCOS_PRIVATE",
    "PHASE51_PENULTIMATE_CACHE_PRIVATE",
    "PHASE51_PENULTIMATE_CACHE_CONTRACT_PRIVATE",
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase50_groups",
    "phase50_y",
    "phase50_phase43_probability",
    "phase51c_report",
]
phase51d_missing_names = [
    name for name in phase51d_required_names if name not in globals()
]
assert not phase51d_missing_names, {
    "message": "Run accepted Phase51 Cells 151A through 151C first.",
    "missing": phase51d_missing_names,
}

assert phase51c_report["status"] == "accepted"
assert phase51c_report["initial_identity"]["exact"] is True
assert phase51c_report["architecture"][
    "nn_linear_interface_preserved"
] is True
assert phase51c_report["three_step_backward_contract"][
    "all_gradients_finite"
] is True
assert (
    str(PHASE51_DINO_CONTRACT_PRIVATE["partition_sha256"])
    == str(PHASE50_PARTITION_SHA256_PRIVATE)
)
assert (
    str(PHASE51_PENULTIMATE_CACHE_CONTRACT_PRIVATE[
        "partition_sha256"
    ])
    == str(PHASE50_PARTITION_SHA256_PRIVATE)
)

phase51d_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51d_future_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51d_fold_assignment = np.asarray(
    PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE,
    dtype=np.int8,
)
phase51d_groups = np.asarray(phase50_groups, dtype=np.int64).reshape(-1)
phase51d_labels = np.asarray(phase50_y, dtype=np.float32).reshape(-1)
phase51d_anchor_probability = np.clip(
    np.asarray(phase50_phase43_probability, dtype=np.float64).reshape(-1),
    1.0e-7,
    1.0 - 1.0e-7,
)
phase51d_anchor_logit = (
    np.log(phase51d_anchor_probability)
    - np.log1p(-phase51d_anchor_probability)
).astype(np.float32)
phase51d_cache = PHASE51_PENULTIMATE_CACHE_PRIVATE

assert phase51d_development_indices.size == 1160
assert phase51d_future_audit_indices.size == 202
assert phase51d_fold_assignment.shape == (5, 1362)
assert phase51d_groups.shape == (1362,)
assert phase51d_labels.shape == (1362,)
assert phase51d_anchor_probability.shape == (1362,)
assert phase51d_anchor_logit.shape == (1362,)
assert phase51d_cache.shape == (1362, 6, 201, 384)
assert phase51d_cache.dtype == np.float16
assert np.intersect1d(
    phase51d_development_indices,
    phase51d_future_audit_indices,
).size == 0
assert np.all(
    phase51d_fold_assignment[:, phase51d_future_audit_indices] == -1
)

phase51d_device = next(PHASE51_BACKBONE_PRIVATE.parameters()).device
phase51d_amp_enabled = phase51d_device.type == "cuda"
phase51d_amp_dtype = (
    torch.bfloat16
    if phase51d_amp_enabled and torch.cuda.is_bf16_supported()
    else torch.float16
)
if phase51d_amp_enabled:
    assert phase51d_amp_dtype == torch.bfloat16
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase51d_device)


def phase51d_set_seed(seed):
    seed = int(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def phase51d_log_loss(labels, probability, sample_weight=None):
    labels = np.asarray(labels, dtype=np.float64).reshape(-1)
    probability = np.clip(
        np.asarray(probability, dtype=np.float64).reshape(-1),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    losses = -(
        labels * np.log(probability)
        + (1.0 - labels) * np.log1p(-probability)
    )
    if sample_weight is None:
        return float(np.mean(losses))
    sample_weight = np.asarray(
        sample_weight,
        dtype=np.float64,
    ).reshape(-1)
    assert sample_weight.shape == losses.shape
    assert np.all(sample_weight > 0.0)
    return float(
        np.sum(sample_weight * losses) / np.sum(sample_weight)
    )


def phase51d_auroc(labels, probability, sample_weight=None):
    labels = np.asarray(labels, dtype=np.int64).reshape(-1)
    probability = np.asarray(probability, dtype=np.float64).reshape(-1)
    if np.unique(labels).size < 2:
        return None
    return float(roc_auc_score(
        labels,
        probability,
        sample_weight=sample_weight,
    ))


def phase51d_group_balanced_weights(groups):
    groups = np.asarray(groups, dtype=np.int64).reshape(-1)
    unique_groups, counts = np.unique(groups, return_counts=True)
    count_by_group = {
        int(group): int(count)
        for group, count in zip(unique_groups, counts)
    }
    maximum_count = float(np.max(counts))
    weights = np.asarray([
        math.sqrt(maximum_count / count_by_group[int(group)])
        for group in groups
    ], dtype=np.float64)
    weights = np.minimum(
        weights,
        PHASE51_TRAINING_ENGINE_CONFIG[
            "maximum_training_case_weight"
        ],
    )
    weights /= np.mean(weights)
    assert np.all(np.isfinite(weights))
    assert np.all(weights > 0.0)
    return weights.astype(np.float32)


def phase51d_stratified_subsample(indices, target_count, seed):
    """Deterministic round-robin sampling over acquisition-group/label strata."""
    indices = np.asarray(indices, dtype=np.int64).reshape(-1)
    target_count = min(int(target_count), int(indices.size))
    assert target_count > 0
    rng = np.random.default_rng(int(seed))

    stratum_queues = []
    for group in sorted(np.unique(phase51d_groups[indices]).tolist()):
        for label in [0, 1]:
            local = indices[
                (phase51d_groups[indices] == group)
                & (phase51d_labels[indices] == label)
            ].copy()
            if local.size == 0:
                continue
            rng.shuffle(local)
            stratum_queues.append(local.tolist())

    selected = []
    cursor = 0
    while len(selected) < target_count:
        progressed = False
        for queue in stratum_queues:
            if cursor < len(queue):
                selected.append(queue[cursor])
                progressed = True
                if len(selected) == target_count:
                    break
        assert progressed, {
            "message": "Stratified sampler exhausted before target count."
        }
        cursor += 1

    selected = np.asarray(sorted(selected), dtype=np.int64)
    assert selected.size == target_count
    assert np.unique(selected).size == target_count
    assert np.setdiff1d(selected, indices).size == 0
    return selected


class Phase51PenultimateDataset(Dataset):
    def __init__(self, cache, indices, case_weights=None):
        self.cache = cache
        self.indices = np.asarray(indices, dtype=np.int64).reshape(-1)
        if case_weights is None:
            self.case_weights = np.ones(
                self.indices.size,
                dtype=np.float32,
            )
        else:
            self.case_weights = np.asarray(
                case_weights,
                dtype=np.float32,
            ).reshape(-1)
        assert self.case_weights.shape == self.indices.shape
        assert np.all(np.isfinite(self.case_weights))
        assert np.all(self.case_weights > 0.0)

    def __len__(self):
        return int(self.indices.size)

    def __getitem__(self, position):
        case_index = int(self.indices[position])
        tokens = np.array(
            self.cache[case_index],
            dtype=np.float16,
            copy=True,
        )
        assert tokens.shape == (6, 201, 384)
        return {
            "tokens": torch.from_numpy(tokens),
            "anchor_logit": torch.tensor(
                phase51d_anchor_logit[case_index],
                dtype=torch.float32,
            ),
            "label": torch.tensor(
                phase51d_labels[case_index],
                dtype=torch.float32,
            ),
            "group": torch.tensor(
                phase51d_groups[case_index],
                dtype=torch.int64,
            ),
            "case_weight": torch.tensor(
                self.case_weights[position],
                dtype=torch.float32,
            ),
            "case_index": torch.tensor(case_index, dtype=torch.int64),
        }


class Phase51GroupPairBatchSampler(Sampler):
    """One-pass batches grouped and label-interleaved for usable rank pairs."""
    def __init__(self, labels, groups, batch_size, seed):
        self.labels = np.asarray(labels, dtype=np.int64).reshape(-1)
        self.groups = np.asarray(groups, dtype=np.int64).reshape(-1)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.epoch = 0
        assert self.labels.shape == self.groups.shape
        assert self.batch_size >= 2

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return int(sum(
            math.ceil(np.sum(self.groups == group) / self.batch_size)
            for group in np.unique(self.groups)
        ))

    def __iter__(self):
        rng = np.random.default_rng(self.seed + 1009 * self.epoch)
        batches = []
        for group in np.unique(self.groups):
            group_positions = np.flatnonzero(self.groups == group)
            positive = group_positions[
                self.labels[group_positions] == 1
            ].copy()
            negative = group_positions[
                self.labels[group_positions] == 0
            ].copy()
            rng.shuffle(positive)
            rng.shuffle(negative)

            interleaved = []
            common = min(positive.size, negative.size)
            for offset in range(common):
                interleaved.append(int(positive[offset]))
                interleaved.append(int(negative[offset]))
            remainder = np.concatenate([
                positive[common:],
                negative[common:],
            ]).astype(np.int64, copy=False)
            rng.shuffle(remainder)
            interleaved.extend(remainder.tolist())

            assert len(interleaved) == group_positions.size
            for start in range(0, len(interleaved), self.batch_size):
                batches.append(interleaved[start:start + self.batch_size])

        rng.shuffle(batches)
        flattened = [position for batch in batches for position in batch]
        assert len(flattened) == self.labels.size
        assert sorted(flattened) == list(range(self.labels.size))
        yield from batches


def phase51d_within_group_rank_loss(logits, labels, groups):
    group_losses = []
    for group in torch.unique(groups):
        mask = groups == group
        positive = logits[mask & (labels > 0.5)]
        negative = logits[mask & (labels <= 0.5)]
        if positive.numel() == 0 or negative.numel() == 0:
            continue
        differences = positive[:, None] - negative[None, :]
        group_losses.append(F.softplus(-differences).mean())
    if not group_losses:
        return logits.sum() * 0.0, 0
    return torch.stack(group_losses).mean(), len(group_losses)


def phase51d_learning_rate_factor(step, total_steps, warmup_steps):
    step = int(step)
    total_steps = max(int(total_steps), 1)
    warmup_steps = max(int(warmup_steps), 1)
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(
        max(total_steps - warmup_steps, 1)
    )
    progress = min(max(progress, 0.0), 1.0)
    minimum = PHASE51_TRAINING_ENGINE_CONFIG[
        "minimum_learning_rate_fraction"
    ]
    return float(
        minimum
        + (1.0 - minimum)
        * 0.5
        * (1.0 + math.cos(math.pi * progress))
    )


def phase51d_make_model(rank):
    model = PHASE51_ADAPTER_CLASS_PRIVATE(
        accepted_final_block=list(
            PHASE51_BACKBONE_PRIVATE.blocks
        )[-1],
        accepted_final_norm=PHASE51_BACKBONE_PRIVATE.norm,
        prompt_mask=PHASE51_DINO_CONTRACT_PRIVATE["prompt_mask"],
        lora_rank=int(rank),
        lora_alpha=float(
            PHASE51_TRAINING_ENGINE_CONFIG["lora_alpha"]
        ),
        view_projection_dimension=int(
            PHASE51_ADAPTER_CONFIG_PRIVATE[
                "view_projection_dimension"
            ]
        ),
        head_hidden_dimension=int(
            PHASE51_ADAPTER_CONFIG_PRIVATE[
                "head_hidden_dimension"
            ]
        ),
        residual_cap=float(
            PHASE51_ADAPTER_CONFIG_PRIVATE["residual_cap"]
        ),
    ).to(phase51d_device)
    assert model.lora_module_names == [
        "attn.qkv",
        "attn.proj",
        "mlp.fc1",
        "mlp.fc2",
    ]
    return model


def phase51d_set_training_mode(model):
    model.train()
    model.final_block.eval()
    model.final_norm.eval()


def phase51d_clone_state_dict(model):
    return {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
    }


@torch.no_grad()
def phase51d_predict(model, loader, expected_indices):
    model.eval()
    model.final_block.eval()
    model.final_norm.eval()
    probability_parts = []
    residual_parts = []
    index_parts = []

    for batch in loader:
        tokens = batch["tokens"].to(
            phase51d_device,
            non_blocking=phase51d_amp_enabled,
        ).float()
        anchor_logit = batch["anchor_logit"].to(
            phase51d_device,
            non_blocking=phase51d_amp_enabled,
        )
        with torch.autocast(
            device_type=phase51d_device.type,
            dtype=phase51d_amp_dtype,
            enabled=phase51d_amp_enabled,
        ):
            output = model(
                tokens,
                anchor_logit,
                PHASE51_ROPE_SINCOS_PRIVATE,
            )
        probability_parts.append(
            torch.sigmoid(output["logit"].float()).cpu().numpy()
        )
        residual_parts.append(
            output["bounded_residual"].float().cpu().numpy()
        )
        index_parts.append(batch["case_index"].cpu().numpy())

    observed_indices = np.concatenate(index_parts).astype(
        np.int64,
        copy=False,
    )
    expected_indices = np.asarray(expected_indices, dtype=np.int64)
    assert np.array_equal(observed_indices, expected_indices)
    probability = np.concatenate(probability_parts).astype(
        np.float64,
        copy=False,
    )
    residual = np.concatenate(residual_parts).astype(
        np.float64,
        copy=False,
    )
    assert probability.shape == expected_indices.shape
    assert residual.shape == expected_indices.shape
    assert np.all(np.isfinite(probability))
    assert np.all(np.isfinite(residual))
    assert np.all((probability > 0.0) & (probability < 1.0))
    return {
        "probability": np.clip(probability, 1.0e-7, 1.0 - 1.0e-7),
        "bounded_residual": residual,
    }


# -------------------------------------------------------------------------
# Pilot partition: repeat-0/fold-0 development rows only.  Subsampling is
# deterministic and is not a hyperparameter-selection result.
# -------------------------------------------------------------------------

phase51d_repeat = int(PHASE51_TRAINING_ENGINE_CONFIG["repeat"])
phase51d_fold = int(PHASE51_TRAINING_ENGINE_CONFIG["fold"])
phase51d_repeat_assignment = phase51d_fold_assignment[phase51d_repeat]

phase51d_complete_train_indices = phase51d_development_indices[
    phase51d_repeat_assignment[phase51d_development_indices]
    != phase51d_fold
]
phase51d_complete_valid_indices = phase51d_development_indices[
    phase51d_repeat_assignment[phase51d_development_indices]
    == phase51d_fold
]
assert np.intersect1d(
    phase51d_complete_train_indices,
    phase51d_complete_valid_indices,
).size == 0
assert np.union1d(
    phase51d_complete_train_indices,
    phase51d_complete_valid_indices,
).size == phase51d_development_indices.size

phase51d_train_indices = phase51d_stratified_subsample(
    phase51d_complete_train_indices,
    PHASE51_TRAINING_ENGINE_CONFIG["pilot_train_case_count"],
    PHASE51_TRAINING_ENGINE_CONFIG["pilot_seed"] + 1,
)
phase51d_valid_indices = phase51d_stratified_subsample(
    phase51d_complete_valid_indices,
    PHASE51_TRAINING_ENGINE_CONFIG["pilot_valid_case_count"],
    PHASE51_TRAINING_ENGINE_CONFIG["pilot_seed"] + 2,
)
assert np.intersect1d(
    phase51d_train_indices,
    phase51d_valid_indices,
).size == 0
assert np.intersect1d(
    phase51d_train_indices,
    phase51d_future_audit_indices,
).size == 0
assert np.intersect1d(
    phase51d_valid_indices,
    phase51d_future_audit_indices,
).size == 0
assert np.unique(phase51d_labels[phase51d_train_indices]).size == 2
assert np.unique(phase51d_labels[phase51d_valid_indices]).size == 2

phase51d_train_weights = phase51d_group_balanced_weights(
    phase51d_groups[phase51d_train_indices]
)
phase51d_train_dataset = Phase51PenultimateDataset(
    phase51d_cache,
    phase51d_train_indices,
    phase51d_train_weights,
)
phase51d_valid_dataset = Phase51PenultimateDataset(
    phase51d_cache,
    phase51d_valid_indices,
)
phase51d_batch_sampler = Phase51GroupPairBatchSampler(
    labels=phase51d_labels[phase51d_train_indices],
    groups=phase51d_groups[phase51d_train_indices],
    batch_size=PHASE51_TRAINING_ENGINE_CONFIG["batch_size"],
    seed=PHASE51_TRAINING_ENGINE_CONFIG["pilot_seed"] + 3,
)
phase51d_train_loader = DataLoader(
    phase51d_train_dataset,
    batch_sampler=phase51d_batch_sampler,
    num_workers=PHASE51_TRAINING_ENGINE_CONFIG["worker_count"],
    pin_memory=phase51d_amp_enabled,
)
phase51d_valid_loader = DataLoader(
    phase51d_valid_dataset,
    batch_size=PHASE51_TRAINING_ENGINE_CONFIG["evaluation_batch_size"],
    shuffle=False,
    num_workers=PHASE51_TRAINING_ENGINE_CONFIG["worker_count"],
    pin_memory=phase51d_amp_enabled,
    drop_last=False,
)


# -------------------------------------------------------------------------
# Superset candidate pilot: rank 8 and nonzero within-group ranking loss.
# -------------------------------------------------------------------------

phase51d_set_seed(PHASE51_TRAINING_ENGINE_CONFIG["pilot_seed"])
phase51d_model = phase51d_make_model(
    PHASE51_TRAINING_ENGINE_CONFIG["lora_rank"]
)

phase51d_lora_parameters = []
phase51d_pooling_parameters = []
phase51d_head_parameters = []
for phase51d_name, phase51d_parameter in phase51d_model.named_parameters():
    if not phase51d_parameter.requires_grad:
        continue
    if ".lora_" in phase51d_name:
        phase51d_lora_parameters.append(phase51d_parameter)
    elif phase51d_name.startswith("view_projection."):
        phase51d_pooling_parameters.append(phase51d_parameter)
    elif phase51d_name.startswith("head."):
        phase51d_head_parameters.append(phase51d_parameter)
    else:
        raise AssertionError({
            "message": "Unexpected Phase51 trainable parameter.",
            "name": phase51d_name,
        })
assert phase51d_lora_parameters
assert phase51d_pooling_parameters
assert phase51d_head_parameters

phase51d_optimizer = torch.optim.AdamW([
    {
        "params": phase51d_lora_parameters,
        "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
            "lora_learning_rate"
        ]),
    },
    {
        "params": phase51d_pooling_parameters,
        "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
            "pooling_learning_rate"
        ]),
    },
    {
        "params": phase51d_head_parameters,
        "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
            "head_learning_rate"
        ]),
    },
], betas=tuple(
    PHASE51_TRAINING_ENGINE_CONFIG["optimizer_betas"]
), weight_decay=PHASE51_TRAINING_ENGINE_CONFIG["weight_decay"])

phase51d_total_steps = int(
    PHASE51_TRAINING_ENGINE_CONFIG["pilot_epoch_count"]
    * len(phase51d_train_loader)
)
phase51d_warmup_steps = max(
    1,
    int(round(
        PHASE51_TRAINING_ENGINE_CONFIG["warmup_fraction"]
        * phase51d_total_steps
    )),
)
phase51d_scheduler = torch.optim.lr_scheduler.LambdaLR(
    phase51d_optimizer,
    lr_lambda=lambda step: phase51d_learning_rate_factor(
        step,
        phase51d_total_steps,
        phase51d_warmup_steps,
    ),
)

# Epoch zero must reproduce the anchor and remains an eligible fallback.
phase51d_epoch_zero_prediction = phase51d_predict(
    phase51d_model,
    phase51d_valid_loader,
    phase51d_valid_indices,
)
phase51d_epoch_zero_reference = 1.0 / (
    1.0 + np.exp(-phase51d_anchor_logit[phase51d_valid_indices].astype(
        np.float64
    ))
)
phase51d_epoch_zero_probability_error = float(np.max(np.abs(
    phase51d_epoch_zero_prediction["probability"]
    - phase51d_epoch_zero_reference
)))
phase51d_epoch_zero_residual_maximum = float(np.max(np.abs(
    phase51d_epoch_zero_prediction["bounded_residual"]
)))
assert phase51d_epoch_zero_probability_error <= 2.0e-7
assert phase51d_epoch_zero_residual_maximum == 0.0

phase51d_best_epoch = 0
phase51d_best_probability = phase51d_epoch_zero_prediction[
    "probability"
].copy()
phase51d_best_log_loss = phase51d_log_loss(
    phase51d_labels[phase51d_valid_indices],
    phase51d_best_probability,
)
phase51d_best_auroc = phase51d_auroc(
    phase51d_labels[phase51d_valid_indices],
    phase51d_best_probability,
)
phase51d_best_state = phase51d_clone_state_dict(phase51d_model)
phase51d_epoch_records = []
phase51d_total_rank_active_batches = 0

for phase51d_epoch in range(
    1,
    PHASE51_TRAINING_ENGINE_CONFIG["pilot_epoch_count"] + 1,
):
    phase51d_batch_sampler.set_epoch(phase51d_epoch)
    phase51d_set_training_mode(phase51d_model)
    phase51d_train_bce_sum = 0.0
    phase51d_train_rank_sum = 0.0
    phase51d_train_case_count = 0
    phase51d_rank_active_batch_count = 0
    phase51d_maximum_gradient_norm = 0.0

    for phase51d_batch in phase51d_train_loader:
        phase51d_tokens = phase51d_batch["tokens"].to(
            phase51d_device,
            non_blocking=phase51d_amp_enabled,
        ).float()
        phase51d_batch_anchor = phase51d_batch["anchor_logit"].to(
            phase51d_device,
            non_blocking=phase51d_amp_enabled,
        )
        phase51d_batch_labels = phase51d_batch["label"].to(
            phase51d_device,
            non_blocking=phase51d_amp_enabled,
        )
        phase51d_batch_groups = phase51d_batch["group"].to(
            phase51d_device,
            non_blocking=phase51d_amp_enabled,
        )
        phase51d_batch_weights = phase51d_batch["case_weight"].to(
            phase51d_device,
            non_blocking=phase51d_amp_enabled,
        )

        phase51d_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=phase51d_device.type,
            dtype=phase51d_amp_dtype,
            enabled=phase51d_amp_enabled,
        ):
            phase51d_output = phase51d_model(
                phase51d_tokens,
                phase51d_batch_anchor,
                PHASE51_ROPE_SINCOS_PRIVATE,
            )
        phase51d_element_bce = F.binary_cross_entropy_with_logits(
            phase51d_output["logit"].float(),
            phase51d_batch_labels.float(),
            reduction="none",
        )
        phase51d_bce_loss = torch.sum(
            phase51d_element_bce * phase51d_batch_weights.float()
        ) / torch.sum(phase51d_batch_weights.float())
        (
            phase51d_rank_loss,
            phase51d_rank_group_count,
        ) = phase51d_within_group_rank_loss(
            phase51d_output["logit"].float(),
            phase51d_batch_labels.float(),
            phase51d_batch_groups,
        )
        phase51d_loss = (
            phase51d_bce_loss
            + PHASE51_TRAINING_ENGINE_CONFIG["rank_loss_weight"]
            * phase51d_rank_loss
        )
        assert torch.isfinite(phase51d_loss)
        phase51d_loss.backward()
        phase51d_gradient_norm = torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter in phase51d_model.parameters()
                if parameter.requires_grad
            ],
            PHASE51_TRAINING_ENGINE_CONFIG["gradient_clip"],
        )
        assert torch.isfinite(phase51d_gradient_norm)
        phase51d_optimizer.step()
        phase51d_scheduler.step()

        phase51d_batch_n = int(phase51d_batch_labels.numel())
        phase51d_train_bce_sum += float(
            phase51d_bce_loss.detach().item()
        ) * phase51d_batch_n
        phase51d_train_rank_sum += float(
            phase51d_rank_loss.detach().item()
        ) * phase51d_batch_n
        phase51d_train_case_count += phase51d_batch_n
        phase51d_rank_active_batch_count += int(
            phase51d_rank_group_count > 0
        )
        phase51d_maximum_gradient_norm = max(
            phase51d_maximum_gradient_norm,
            float(phase51d_gradient_norm.detach().item()),
        )

    assert phase51d_train_case_count == phase51d_train_indices.size
    assert phase51d_rank_active_batch_count > 0
    phase51d_total_rank_active_batches += phase51d_rank_active_batch_count

    phase51d_validation_prediction = phase51d_predict(
        phase51d_model,
        phase51d_valid_loader,
        phase51d_valid_indices,
    )
    phase51d_validation_log_loss = phase51d_log_loss(
        phase51d_labels[phase51d_valid_indices],
        phase51d_validation_prediction["probability"],
    )
    phase51d_validation_auroc = phase51d_auroc(
        phase51d_labels[phase51d_valid_indices],
        phase51d_validation_prediction["probability"],
    )
    phase51d_improved = bool(
        phase51d_validation_log_loss < phase51d_best_log_loss
    )
    if phase51d_improved:
        phase51d_best_epoch = int(phase51d_epoch)
        phase51d_best_log_loss = float(phase51d_validation_log_loss)
        phase51d_best_auroc = float(phase51d_validation_auroc)
        phase51d_best_probability = phase51d_validation_prediction[
            "probability"
        ].copy()
        phase51d_best_state = phase51d_clone_state_dict(phase51d_model)

    phase51d_epoch_records.append({
        "epoch": int(phase51d_epoch),
        "train_bce": float(
            phase51d_train_bce_sum / phase51d_train_case_count
        ),
        "train_rank": float(
            phase51d_train_rank_sum / phase51d_train_case_count
        ),
        "rank_active_batch_count": int(
            phase51d_rank_active_batch_count
        ),
        "validation_log_loss": float(phase51d_validation_log_loss),
        "validation_auroc": float(phase51d_validation_auroc),
        "maximum_gradient_norm": float(
            phase51d_maximum_gradient_norm
        ),
        "improved_over_previous_best": phase51d_improved,
    })
    print(
        "Phase51 training-engine pilot "
        f"epoch {phase51d_epoch}/"
        f"{PHASE51_TRAINING_ENGINE_CONFIG['pilot_epoch_count']}: "
        f"valid_log_loss={phase51d_validation_log_loss:.6f}, "
        f"valid_auroc={phase51d_validation_auroc:.6f}"
    )


# Restore the selected checkpoint and prove that its predictions are exact.
phase51d_model.load_state_dict(phase51d_best_state, strict=True)
phase51d_restored_prediction = phase51d_predict(
    phase51d_model,
    phase51d_valid_loader,
    phase51d_valid_indices,
)
phase51d_restore_probability_error = float(np.max(np.abs(
    phase51d_restored_prediction["probability"]
    - phase51d_best_probability
)))
assert phase51d_restore_probability_error <= (
    PHASE51_TRAINING_ENGINE_CONFIG[
        "maximum_allowed_restore_probability_error"
    ]
)
assert phase51d_total_rank_active_batches > 0

phase51d_all_gradient_values_finite = bool(all(
    np.isfinite(record["maximum_gradient_norm"])
    for record in phase51d_epoch_records
))
assert phase51d_all_gradient_values_finite

phase51d_peak_vram_mb = (
    float(torch.cuda.max_memory_allocated(phase51d_device) / (1024 ** 2))
    if phase51d_amp_enabled else 0.0
)

PHASE51_TRAINING_ENGINE_CONFIG_PRIVATE = copy.deepcopy(
    PHASE51_TRAINING_ENGINE_CONFIG
)
PHASE51_DATASET_CLASS_PRIVATE = Phase51PenultimateDataset
PHASE51_BATCH_SAMPLER_CLASS_PRIVATE = Phase51GroupPairBatchSampler
PHASE51_TRAINING_FUNCTIONS_PRIVATE = {
    "set_seed": phase51d_set_seed,
    "log_loss": phase51d_log_loss,
    "auroc": phase51d_auroc,
    "group_balanced_weights": phase51d_group_balanced_weights,
    "within_group_rank_loss": phase51d_within_group_rank_loss,
    "learning_rate_factor": phase51d_learning_rate_factor,
    "make_model": phase51d_make_model,
    "set_training_mode": phase51d_set_training_mode,
    "clone_state_dict": phase51d_clone_state_dict,
    "predict": phase51d_predict,
}

phase51d_report = {
    "phase": "phase51_repeated_cv_training_engine_pilot",
    "status": "accepted_ready_for_full_search",
    "purpose": (
        "validate_the_complete_training_engine_before_100_model_fits"
    ),
    "partition": {
        "repeat": int(phase51d_repeat),
        "fold": int(phase51d_fold),
        "complete_train_n": int(phase51d_complete_train_indices.size),
        "complete_valid_n": int(phase51d_complete_valid_indices.size),
        "pilot_train_n": int(phase51d_train_indices.size),
        "pilot_valid_n": int(phase51d_valid_indices.size),
        "pilot_train_group_count": int(np.unique(
            phase51d_groups[phase51d_train_indices]
        ).size),
        "pilot_valid_group_count": int(np.unique(
            phase51d_groups[phase51d_valid_indices]
        ).size),
    },
    "superset_candidate": {
        "lora_rank": int(
            PHASE51_TRAINING_ENGINE_CONFIG["lora_rank"]
        ),
        "lora_alpha": float(
            PHASE51_TRAINING_ENGINE_CONFIG["lora_alpha"]
        ),
        "rank_loss_weight": float(
            PHASE51_TRAINING_ENGINE_CONFIG["rank_loss_weight"]
        ),
        "adapted_linear_count": int(len(
            phase51d_model.lora_module_names
        )),
        "adapted_linears": list(phase51d_model.lora_module_names),
    },
    "engine_contract": {
        "group_balanced_case_weights": True,
        "group_local_label_interleaved_batches": True,
        "each_training_case_used_once_per_epoch": True,
        "within_group_pairwise_rank_loss": True,
        "rank_active_batch_count": int(
            phase51d_total_rank_active_batches
        ),
        "mixed_precision": (
            "bfloat16" if phase51d_amp_enabled else "disabled"
        ),
        "frozen_final_block_eval_behavior": True,
        "epoch_zero_anchor_fallback": True,
        "checkpoint_restoration_verified": True,
    },
    "epoch_zero": {
        "maximum_probability_error": float(
            phase51d_epoch_zero_probability_error
        ),
        "maximum_absolute_residual": float(
            phase51d_epoch_zero_residual_maximum
        ),
        "validation_log_loss": float(phase51d_log_loss(
            phase51d_labels[phase51d_valid_indices],
            phase51d_epoch_zero_prediction["probability"],
        )),
        "validation_auroc": float(phase51d_auroc(
            phase51d_labels[phase51d_valid_indices],
            phase51d_epoch_zero_prediction["probability"],
        )),
    },
    "training": {
        "epoch_count": int(
            PHASE51_TRAINING_ENGINE_CONFIG["pilot_epoch_count"]
        ),
        "records": phase51d_epoch_records,
        "all_gradient_norms_finite": bool(
            phase51d_all_gradient_values_finite
        ),
    },
    "checkpoint_selection": {
        "selected_epoch": int(phase51d_best_epoch),
        "selected_validation_log_loss": float(
            phase51d_best_log_loss
        ),
        "selected_validation_auroc": float(phase51d_best_auroc),
        "restored_probability_maximum_error": float(
            phase51d_restore_probability_error
        ),
        "epoch_zero_remained_selected": bool(
            phase51d_best_epoch == 0
        ),
    },
    "full_search_plan": {
        "candidate_count": 4,
        "repeat_count": 5,
        "fold_count_per_repeat": 5,
        "planned_model_fit_count": 100,
        "shared_specification_across_all_folds": True,
        "future_audit_evaluated_only_after_complete_freeze": True,
    },
    "validation_lock": {
        "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
        "future_audit_case_count": int(
            phase51d_future_audit_indices.size
        ),
        "future_audit_features_used": False,
        "future_audit_labels_used": False,
        "future_audit_metrics_computed": False,
    },
    "execution": {
        "device": str(phase51d_device),
        "peak_vram_mb": float(phase51d_peak_vram_mb),
    },
    "fit_labels_used": True,
    "development_validation_labels_used_for_engine_test": True,
    "future_audit_labels_used": False,
    "public_leaderboard_used": False,
    "training_penultimate_cache_read": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase51d_started, 3),
}

PHASE51_TRAINING_ENGINE_REPORT_PRIVATE = copy.deepcopy(
    phase51d_report
)

assert phase51d_report["status"] == "accepted_ready_for_full_search"
assert phase51d_report["engine_contract"][
    "checkpoint_restoration_verified"
] is True
assert phase51d_report["validation_lock"][
    "future_audit_labels_used"
] is False

print("BEGIN SANITIZED_PHASE51_TRAINING_ENGINE")
print(json.dumps(phase51d_report, indent=2))
print("END SANITIZED_PHASE51_TRAINING_ENGINE")

del phase51d_optimizer
del phase51d_scheduler
del phase51d_model
del phase51d_best_state
gc.collect()
if phase51d_amp_enabled:
    torch.cuda.empty_cache()

Phase51 training-engine pilot epoch 1/2: valid_log_loss=0.352810, valid_auroc=0.929253
Phase51 training-engine pilot epoch 2/2: valid_log_loss=0.353568, valid_auroc=0.929253
BEGIN SANITIZED_PHASE51_TRAINING_ENGINE
{
  "phase": "phase51_repeated_cv_training_engine_pilot",
  "status": "accepted_ready_for_full_search",
  "purpose": "validate_the_complete_training_engine_before_100_model_fits",
  "partition": {
    "repeat": 0,
    "fold": 0,
    "complete_train_n": 929,
    "complete_valid_n": 231,
    "pilot_train_n": 256,
    "pilot_valid_n": 96,
    "pilot_train_group_count": 15,
    "pilot_valid_group_count": 15
  },
  "superset_candidate": {
    "lora_rank": 8,
    "lora_alpha": 4.0,
    "rank_loss_weight": 0.05,
    "adapted_linear_count": 4,
    "adapted_linears": [
      "attn.qkv",
      "attn.proj",
      "mlp.fc1",
      "mlp.fc2"
    ]
  },
  "engine_contract": {
    "group_balanced_case_weights": true,
    "group_local_label_interleaved_batches": true,
    "each_training_case

In [71]:
# Phase51 Cell 151E

import copy
import gc
import hashlib
import itertools
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


phase51e_started = time.perf_counter()

PHASE51_SELECTION_SCREEN_CONFIG = {
    "selection_repeats": [1, 2],
    "confirmation_repeats": [3, 4],
    "retired_pilot_repeat": 0,
    "fold_count": 5,
    "batch_size": 16,
    "evaluation_batch_size": 24,
    "worker_count": 0,
    "maximum_epochs": 12,
    "gradient_clip": 1.0,
    "warmup_fraction": 0.10,
    "minimum_learning_rate_fraction": 0.10,
    "weight_decay": 0.02,
    "optimizer_betas": [0.9, 0.95],
    "selection_log_loss_band": 0.002,
    "maximum_repeat_log_loss_regret": 0.003,
    "maximum_major_group_harm": 0.015,
    "maximum_auroc_deficit": 0.0005,
    "minimum_repeat_log_loss_wins": 2,
    "minimum_screen_log_loss_gain": 0.002,
    "minimum_screen_auroc_gain": 0.001,
    "maximum_screen_log_loss_excess": 0.0005,
    "maximum_screen_auroc_deficit": 0.0002,
    "major_group_minimum_n": 30,
    "progress_directory": "/kaggle/working/phase51_selection_screen_private",
}


# -------------------------------------------------------------------------
# Accepted engine and prospectively locked partition.
# -------------------------------------------------------------------------

phase51e_required_names = [
    "PHASE51_ADAPTER_CONFIG_PRIVATE",
    "PHASE51_ROPE_SINCOS_PRIVATE",
    "PHASE51_TRAINING_ENGINE_REPORT_PRIVATE",
    "PHASE51_DATASET_CLASS_PRIVATE",
    "PHASE51_BATCH_SAMPLER_CLASS_PRIVATE",
    "PHASE51_TRAINING_FUNCTIONS_PRIVATE",
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase51d_cache",
    "phase51d_groups",
    "phase51d_labels",
    "phase51d_anchor_probability",
    "phase51d_anchor_logit",
    "phase51d_device",
    "phase51d_amp_enabled",
    "phase51d_amp_dtype",
]
phase51e_missing_names = [
    name for name in phase51e_required_names if name not in globals()
]
assert not phase51e_missing_names, {
    "message": "Run accepted Phase51 Cell 151D before the selection screen.",
    "missing": phase51e_missing_names,
}

assert PHASE51_TRAINING_ENGINE_REPORT_PRIVATE[
    "status"
] == "accepted_ready_for_full_search"
assert PHASE51_TRAINING_ENGINE_REPORT_PRIVATE[
    "validation_lock"
]["future_audit_labels_used"] is False

phase51e_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51e_future_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51e_fold_assignment = np.asarray(
    PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE,
    dtype=np.int8,
)
phase51e_groups = np.asarray(phase51d_groups, dtype=np.int64).reshape(-1)
phase51e_labels = np.asarray(phase51d_labels, dtype=np.float32).reshape(-1)
phase51e_anchor_probability = np.asarray(
    phase51d_anchor_probability,
    dtype=np.float64,
).reshape(-1)
phase51e_anchor_logit = np.asarray(
    phase51d_anchor_logit,
    dtype=np.float32,
).reshape(-1)
phase51e_cache = phase51d_cache
phase51e_device = phase51d_device
phase51e_amp_enabled = bool(phase51d_amp_enabled)
phase51e_amp_dtype = phase51d_amp_dtype

assert phase51e_development_indices.size == 1160
assert phase51e_future_audit_indices.size == 202
assert phase51e_fold_assignment.shape == (5, 1362)
assert phase51e_cache.shape == (1362, 6, 201, 384)
assert phase51e_cache.dtype == np.float16
assert np.intersect1d(
    phase51e_development_indices,
    phase51e_future_audit_indices,
).size == 0
assert np.all(
    phase51e_fold_assignment[:, phase51e_future_audit_indices] == -1
)
assert set(PHASE51_SELECTION_SCREEN_CONFIG["selection_repeats"]) == {1, 2}
assert set(PHASE51_SELECTION_SCREEN_CONFIG["confirmation_repeats"]) == {3, 4}
assert PHASE51_SELECTION_SCREEN_CONFIG["retired_pilot_repeat"] == 0
assert not (
    set(PHASE51_SELECTION_SCREEN_CONFIG["selection_repeats"])
    & set(PHASE51_SELECTION_SCREEN_CONFIG["confirmation_repeats"])
)

phase51e_functions = PHASE51_TRAINING_FUNCTIONS_PRIVATE
phase51e_set_seed = phase51e_functions["set_seed"]
phase51e_log_loss = phase51e_functions["log_loss"]
phase51e_auroc = phase51e_functions["auroc"]
phase51e_training_weights = phase51e_functions[
    "group_balanced_weights"
]
phase51e_rank_loss = phase51e_functions["within_group_rank_loss"]
phase51e_make_model = phase51e_functions["make_model"]
phase51e_set_training_mode = phase51e_functions["set_training_mode"]
phase51e_predict = phase51e_functions["predict"]
Phase51Dataset = PHASE51_DATASET_CLASS_PRIVATE
Phase51BatchSampler = PHASE51_BATCH_SAMPLER_CLASS_PRIVATE


# -------------------------------------------------------------------------
# Frozen four-candidate grid.  Epoch is selected globally across the two
# selection repeats; fold-specific epoch selection is forbidden.
# -------------------------------------------------------------------------

phase51e_candidates = []
for phase51e_candidate_index, phase51e_values in enumerate(
    itertools.product(
        PHASE51_ADAPTER_CONFIG_PRIVATE["candidate_lora_ranks"],
        PHASE51_ADAPTER_CONFIG_PRIVATE["candidate_rank_loss_weights"],
    )
):
    phase51e_rank, phase51e_rank_weight = phase51e_values
    phase51e_candidates.append({
        "candidate_index": int(phase51e_candidate_index),
        "lora_rank": int(phase51e_rank),
        "rank_loss_weight": float(phase51e_rank_weight),
        "lora_alpha": float(
            PHASE51_ADAPTER_CONFIG_PRIVATE["lora_alpha"]
        ),
    })

assert phase51e_candidates == [
    {
        "candidate_index": 0,
        "lora_rank": 4,
        "rank_loss_weight": 0.0,
        "lora_alpha": 4.0,
    },
    {
        "candidate_index": 1,
        "lora_rank": 4,
        "rank_loss_weight": 0.05,
        "lora_alpha": 4.0,
    },
    {
        "candidate_index": 2,
        "lora_rank": 8,
        "rank_loss_weight": 0.0,
        "lora_alpha": 4.0,
    },
    {
        "candidate_index": 3,
        "lora_rank": 8,
        "rank_loss_weight": 0.05,
        "lora_alpha": 4.0,
    },
]

phase51e_selection_repeats = list(
    PHASE51_SELECTION_SCREEN_CONFIG["selection_repeats"]
)
phase51e_repeat_to_slot = {
    int(repeat): int(slot)
    for slot, repeat in enumerate(phase51e_selection_repeats)
}
phase51e_development_position = np.full(1362, -1, dtype=np.int64)
phase51e_development_position[phase51e_development_indices] = np.arange(
    phase51e_development_indices.size,
    dtype=np.int64,
)
assert np.all(
    phase51e_development_position[phase51e_development_indices] >= 0
)
assert np.all(
    phase51e_development_position[phase51e_future_audit_indices] == -1
)

phase51e_candidate_count = len(phase51e_candidates)
phase51e_epoch_count = int(
    PHASE51_SELECTION_SCREEN_CONFIG["maximum_epochs"]
)
phase51e_repeat_count = len(phase51e_selection_repeats)
phase51e_fold_count = int(PHASE51_SELECTION_SCREEN_CONFIG["fold_count"])
phase51e_planned_fit_count = int(
    phase51e_candidate_count
    * phase51e_repeat_count
    * phase51e_fold_count
)
assert phase51e_planned_fit_count == 40


# -------------------------------------------------------------------------
# Atomic, resumable private progress.  It contains development predictions
# only and is never a submission asset.
# -------------------------------------------------------------------------

phase51e_progress_directory = Path(
    PHASE51_SELECTION_SCREEN_CONFIG["progress_directory"]
)
phase51e_progress_directory.mkdir(parents=True, exist_ok=True)
phase51e_progress_path = (
    phase51e_progress_directory / "phase51_selection_progress.npz"
)
phase51e_records_path = (
    phase51e_progress_directory / "phase51_selection_records.json"
)
phase51e_contract_path = (
    phase51e_progress_directory / "phase51_selection_contract.json"
)

phase51e_contract_payload = {
    "schema_version": "phase51e_selection_screen_v1",
    "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
    "selection_repeats": phase51e_selection_repeats,
    "confirmation_repeats": list(
        PHASE51_SELECTION_SCREEN_CONFIG["confirmation_repeats"]
    ),
    "retired_pilot_repeat": int(
        PHASE51_SELECTION_SCREEN_CONFIG["retired_pilot_repeat"]
    ),
    "candidates": phase51e_candidates,
    "maximum_epochs": int(phase51e_epoch_count),
    "batch_size": int(
        PHASE51_SELECTION_SCREEN_CONFIG["batch_size"]
    ),
    "weight_decay": float(
        PHASE51_SELECTION_SCREEN_CONFIG["weight_decay"]
    ),
    "lora_learning_rate": float(
        PHASE51_ADAPTER_CONFIG_PRIVATE["lora_learning_rate"]
    ),
    "pooling_learning_rate": float(
        PHASE51_ADAPTER_CONFIG_PRIVATE["pooling_learning_rate"]
    ),
    "head_learning_rate": float(
        PHASE51_ADAPTER_CONFIG_PRIVATE["head_learning_rate"]
    ),
}
phase51e_contract_sha256 = hashlib.sha256(
    json.dumps(
        phase51e_contract_payload,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


def phase51e_atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
    os.replace(temporary, path)


def phase51e_atomic_progress(path, probability, completed):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(
            handle,
            contract_sha256=np.asarray(phase51e_contract_sha256),
            probability=np.asarray(probability, dtype=np.float32),
            completed=np.asarray(completed, dtype=np.uint8),
        )
    os.replace(temporary, path)


phase51e_expected_probability_shape = (
    phase51e_candidate_count,
    phase51e_epoch_count + 1,
    phase51e_repeat_count,
    phase51e_development_indices.size,
)
phase51e_expected_completed_shape = (
    phase51e_candidate_count,
    phase51e_repeat_count,
    phase51e_fold_count,
)

phase51e_progress_reused = False
if phase51e_progress_path.is_file():
    with np.load(phase51e_progress_path, allow_pickle=False) as archive:
        phase51e_loaded_hash = str(archive["contract_sha256"].item())
        phase51e_probability = np.asarray(
            archive["probability"],
            dtype=np.float32,
        ).copy()
        phase51e_completed = np.asarray(
            archive["completed"],
            dtype=np.uint8,
        ).astype(bool)
    assert phase51e_loaded_hash == phase51e_contract_sha256, {
        "message": "Existing Phase51 selection progress uses another contract.",
        "observed": phase51e_loaded_hash,
        "expected": phase51e_contract_sha256,
    }
    assert phase51e_probability.shape == phase51e_expected_probability_shape
    assert phase51e_completed.shape == phase51e_expected_completed_shape
    if phase51e_records_path.is_file():
        with phase51e_records_path.open("r", encoding="utf-8") as handle:
            phase51e_training_records = json.load(handle)
        assert isinstance(phase51e_training_records, list)
    else:
        phase51e_training_records = []
    phase51e_progress_reused = True
else:
    phase51e_probability = np.full(
        phase51e_expected_probability_shape,
        np.nan,
        dtype=np.float32,
    )
    phase51e_completed = np.zeros(
        phase51e_expected_completed_shape,
        dtype=bool,
    )
    phase51e_training_records = []

    phase51e_development_anchor = phase51e_anchor_probability[
        phase51e_development_indices
    ].astype(np.float32)
    phase51e_probability[:, 0, :, :] = (
        phase51e_development_anchor[None, None, :]
    )
    phase51e_atomic_progress(
        phase51e_progress_path,
        phase51e_probability,
        phase51e_completed,
    )
    phase51e_atomic_json(
        phase51e_contract_path,
        {
            **phase51e_contract_payload,
            "contract_sha256": phase51e_contract_sha256,
            "contains_future_audit_rows": False,
            "contains_case_identifiers": False,
        },
    )

# Epoch zero must always be fully initialized, including after resume.
phase51e_anchor_development = phase51e_anchor_probability[
    phase51e_development_indices
]
assert np.max(np.abs(
    phase51e_probability[:, 0, :, :].astype(np.float64)
    - phase51e_anchor_development[None, None, :]
)) <= 3.0e-8


def phase51e_make_optimizer(model):
    lora_parameters = []
    pooling_parameters = []
    head_parameters = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if ".lora_" in name:
            lora_parameters.append(parameter)
        elif name.startswith("view_projection."):
            pooling_parameters.append(parameter)
        elif name.startswith("head."):
            head_parameters.append(parameter)
        else:
            raise AssertionError({
                "message": "Unexpected Phase51 trainable parameter.",
                "name": name,
            })
    assert lora_parameters and pooling_parameters and head_parameters
    return torch.optim.AdamW([
        {
            "params": lora_parameters,
            "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
                "lora_learning_rate"
            ]),
        },
        {
            "params": pooling_parameters,
            "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
                "pooling_learning_rate"
            ]),
        },
        {
            "params": head_parameters,
            "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
                "head_learning_rate"
            ]),
        },
    ], betas=tuple(
        PHASE51_SELECTION_SCREEN_CONFIG["optimizer_betas"]
    ), weight_decay=PHASE51_SELECTION_SCREEN_CONFIG["weight_decay"])


def phase51e_lr_factor(step, total_steps, warmup_steps):
    step = int(step)
    total_steps = max(int(total_steps), 1)
    warmup_steps = max(int(warmup_steps), 1)
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(
        max(total_steps - warmup_steps, 1)
    )
    progress = min(max(progress, 0.0), 1.0)
    minimum = PHASE51_SELECTION_SCREEN_CONFIG[
        "minimum_learning_rate_fraction"
    ]
    return float(
        minimum
        + (1.0 - minimum)
        * 0.5
        * (1.0 + math.cos(math.pi * progress))
    )


# -------------------------------------------------------------------------
# Forty fits: four specifications x two untouched selection repeats x five
# folds.  Each model runs every epoch, enabling one global epoch choice.
# -------------------------------------------------------------------------

if phase51e_amp_enabled:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase51e_device)

phase51e_initial_completed_count = int(np.sum(phase51e_completed))
phase51e_fit_sequence_number = phase51e_initial_completed_count

for phase51e_specification in phase51e_candidates:
    phase51e_candidate_index = int(
        phase51e_specification["candidate_index"]
    )
    for phase51e_repeat in phase51e_selection_repeats:
        phase51e_repeat_slot = phase51e_repeat_to_slot[phase51e_repeat]
        for phase51e_fold in range(phase51e_fold_count):
            if phase51e_completed[
                phase51e_candidate_index,
                phase51e_repeat_slot,
                phase51e_fold,
            ]:
                continue

            phase51e_run_started = time.perf_counter()
            phase51e_run_seed = int(
                512000
                + 10000 * phase51e_candidate_index
                + 100 * phase51e_repeat
                + phase51e_fold
            )
            phase51e_set_seed(phase51e_run_seed)

            phase51e_assignment = phase51e_fold_assignment[
                phase51e_repeat
            ]
            phase51e_train_indices = phase51e_development_indices[
                phase51e_assignment[phase51e_development_indices]
                != phase51e_fold
            ]
            phase51e_valid_indices = phase51e_development_indices[
                phase51e_assignment[phase51e_development_indices]
                == phase51e_fold
            ]
            assert np.intersect1d(
                phase51e_train_indices,
                phase51e_valid_indices,
            ).size == 0
            assert np.union1d(
                phase51e_train_indices,
                phase51e_valid_indices,
            ).size == phase51e_development_indices.size
            assert np.intersect1d(
                phase51e_train_indices,
                phase51e_future_audit_indices,
            ).size == 0
            assert np.intersect1d(
                phase51e_valid_indices,
                phase51e_future_audit_indices,
            ).size == 0

            phase51e_train_weights = phase51e_training_weights(
                phase51e_groups[phase51e_train_indices]
            )
            phase51e_train_dataset = Phase51Dataset(
                phase51e_cache,
                phase51e_train_indices,
                phase51e_train_weights,
            )
            phase51e_valid_dataset = Phase51Dataset(
                phase51e_cache,
                phase51e_valid_indices,
            )
            phase51e_batch_sampler = Phase51BatchSampler(
                labels=phase51e_labels[phase51e_train_indices],
                groups=phase51e_groups[phase51e_train_indices],
                batch_size=PHASE51_SELECTION_SCREEN_CONFIG["batch_size"],
                seed=phase51e_run_seed + 1,
            )
            phase51e_train_loader = DataLoader(
                phase51e_train_dataset,
                batch_sampler=phase51e_batch_sampler,
                num_workers=PHASE51_SELECTION_SCREEN_CONFIG[
                    "worker_count"
                ],
                pin_memory=phase51e_amp_enabled,
            )
            phase51e_valid_loader = DataLoader(
                phase51e_valid_dataset,
                batch_size=PHASE51_SELECTION_SCREEN_CONFIG[
                    "evaluation_batch_size"
                ],
                shuffle=False,
                num_workers=PHASE51_SELECTION_SCREEN_CONFIG[
                    "worker_count"
                ],
                pin_memory=phase51e_amp_enabled,
                drop_last=False,
            )

            phase51e_model = phase51e_make_model(
                phase51e_specification["lora_rank"]
            )
            phase51e_optimizer = phase51e_make_optimizer(phase51e_model)
            phase51e_total_steps = int(
                phase51e_epoch_count * len(phase51e_train_loader)
            )
            phase51e_warmup_steps = max(
                1,
                int(round(
                    PHASE51_SELECTION_SCREEN_CONFIG["warmup_fraction"]
                    * phase51e_total_steps
                )),
            )
            phase51e_scheduler = torch.optim.lr_scheduler.LambdaLR(
                phase51e_optimizer,
                lr_lambda=lambda step, total=phase51e_total_steps,
                warmup=phase51e_warmup_steps: phase51e_lr_factor(
                    step,
                    total,
                    warmup,
                ),
            )

            phase51e_valid_positions = phase51e_development_position[
                phase51e_valid_indices
            ]
            assert np.all(phase51e_valid_positions >= 0)
            phase51e_epoch_records = []
            phase51e_total_rank_active_batches = 0

            for phase51e_epoch in range(1, phase51e_epoch_count + 1):
                phase51e_batch_sampler.set_epoch(phase51e_epoch)
                phase51e_set_training_mode(phase51e_model)
                phase51e_bce_sum = 0.0
                phase51e_rank_sum = 0.0
                phase51e_case_count = 0
                phase51e_rank_active_batches = 0
                phase51e_maximum_gradient_norm = 0.0

                for phase51e_batch in phase51e_train_loader:
                    phase51e_tokens = phase51e_batch["tokens"].to(
                        phase51e_device,
                        non_blocking=phase51e_amp_enabled,
                    ).float()
                    phase51e_batch_anchor = phase51e_batch[
                        "anchor_logit"
                    ].to(
                        phase51e_device,
                        non_blocking=phase51e_amp_enabled,
                    )
                    phase51e_batch_labels = phase51e_batch["label"].to(
                        phase51e_device,
                        non_blocking=phase51e_amp_enabled,
                    )
                    phase51e_batch_groups = phase51e_batch["group"].to(
                        phase51e_device,
                        non_blocking=phase51e_amp_enabled,
                    )
                    phase51e_batch_weights = phase51e_batch[
                        "case_weight"
                    ].to(
                        phase51e_device,
                        non_blocking=phase51e_amp_enabled,
                    )

                    phase51e_optimizer.zero_grad(set_to_none=True)
                    with torch.autocast(
                        device_type=phase51e_device.type,
                        dtype=phase51e_amp_dtype,
                        enabled=phase51e_amp_enabled,
                    ):
                        phase51e_output = phase51e_model(
                            phase51e_tokens,
                            phase51e_batch_anchor,
                            PHASE51_ROPE_SINCOS_PRIVATE,
                        )
                    phase51e_element_bce = (
                        F.binary_cross_entropy_with_logits(
                            phase51e_output["logit"].float(),
                            phase51e_batch_labels.float(),
                            reduction="none",
                        )
                    )
                    phase51e_bce_loss = torch.sum(
                        phase51e_element_bce
                        * phase51e_batch_weights.float()
                    ) / torch.sum(phase51e_batch_weights.float())
                    (
                        phase51e_batch_rank_loss,
                        phase51e_rank_group_count,
                    ) = phase51e_rank_loss(
                        phase51e_output["logit"].float(),
                        phase51e_batch_labels.float(),
                        phase51e_batch_groups,
                    )
                    phase51e_loss = (
                        phase51e_bce_loss
                        + phase51e_specification["rank_loss_weight"]
                        * phase51e_batch_rank_loss
                    )
                    assert torch.isfinite(phase51e_loss)
                    phase51e_loss.backward()
                    phase51e_gradient_norm = torch.nn.utils.clip_grad_norm_(
                        [
                            parameter
                            for parameter in phase51e_model.parameters()
                            if parameter.requires_grad
                        ],
                        PHASE51_SELECTION_SCREEN_CONFIG["gradient_clip"],
                    )
                    assert torch.isfinite(phase51e_gradient_norm)
                    phase51e_optimizer.step()
                    phase51e_scheduler.step()

                    phase51e_batch_n = int(
                        phase51e_batch_labels.numel()
                    )
                    phase51e_bce_sum += float(
                        phase51e_bce_loss.detach().item()
                    ) * phase51e_batch_n
                    phase51e_rank_sum += float(
                        phase51e_batch_rank_loss.detach().item()
                    ) * phase51e_batch_n
                    phase51e_case_count += phase51e_batch_n
                    phase51e_rank_active_batches += int(
                        phase51e_rank_group_count > 0
                    )
                    phase51e_maximum_gradient_norm = max(
                        phase51e_maximum_gradient_norm,
                        float(phase51e_gradient_norm.detach().item()),
                    )

                assert phase51e_case_count == phase51e_train_indices.size
                assert phase51e_rank_active_batches > 0
                phase51e_total_rank_active_batches += (
                    phase51e_rank_active_batches
                )

                phase51e_prediction = phase51e_predict(
                    phase51e_model,
                    phase51e_valid_loader,
                    phase51e_valid_indices,
                )
                phase51e_probability[
                    phase51e_candidate_index,
                    phase51e_epoch,
                    phase51e_repeat_slot,
                    phase51e_valid_positions,
                ] = phase51e_prediction["probability"].astype(np.float32)

                phase51e_epoch_records.append({
                    "epoch": int(phase51e_epoch),
                    "train_bce": float(
                        phase51e_bce_sum / phase51e_case_count
                    ),
                    "train_rank": float(
                        phase51e_rank_sum / phase51e_case_count
                    ),
                    "rank_active_batch_count": int(
                        phase51e_rank_active_batches
                    ),
                    "validation_log_loss": float(phase51e_log_loss(
                        phase51e_labels[phase51e_valid_indices],
                        phase51e_prediction["probability"],
                    )),
                    "validation_auroc": float(phase51e_auroc(
                        phase51e_labels[phase51e_valid_indices],
                        phase51e_prediction["probability"],
                    )),
                    "maximum_gradient_norm": float(
                        phase51e_maximum_gradient_norm
                    ),
                })

            assert phase51e_total_rank_active_batches > 0
            phase51e_completed[
                phase51e_candidate_index,
                phase51e_repeat_slot,
                phase51e_fold,
            ] = True
            phase51e_fit_sequence_number += 1
            phase51e_training_records.append({
                "candidate_index": int(phase51e_candidate_index),
                "repeat": int(phase51e_repeat),
                "fold": int(phase51e_fold),
                "seed": int(phase51e_run_seed),
                "train_n": int(phase51e_train_indices.size),
                "valid_n": int(phase51e_valid_indices.size),
                "epoch_count": int(phase51e_epoch_count),
                "rank_active_batch_count": int(
                    phase51e_total_rank_active_batches
                ),
                "records": phase51e_epoch_records,
                "elapsed_seconds": round(
                    time.perf_counter() - phase51e_run_started,
                    3,
                ),
            })

            phase51e_atomic_progress(
                phase51e_progress_path,
                phase51e_probability,
                phase51e_completed,
            )
            phase51e_atomic_json(
                phase51e_records_path,
                phase51e_training_records,
            )

            print(
                "Phase51 selection screen: "
                f"{phase51e_fit_sequence_number}/"
                f"{phase51e_planned_fit_count} fits complete; "
                f"candidate={phase51e_candidate_index}, "
                f"repeat={phase51e_repeat}, fold={phase51e_fold}"
            )

            del phase51e_optimizer
            del phase51e_scheduler
            del phase51e_model
            del phase51e_train_loader
            del phase51e_valid_loader
            del phase51e_train_dataset
            del phase51e_valid_dataset
            gc.collect()
            if phase51e_amp_enabled:
                torch.cuda.empty_cache()


assert np.all(phase51e_completed)
assert np.all(np.isfinite(phase51e_probability))
assert np.all(
    (phase51e_probability > 0.0)
    & (phase51e_probability < 1.0)
)


# -------------------------------------------------------------------------
# Selection metrics.  One epoch is shared across all ten trained folds.
# -------------------------------------------------------------------------

phase51e_y_development = phase51e_labels[
    phase51e_development_indices
]
phase51e_groups_development = phase51e_groups[
    phase51e_development_indices
]
phase51e_anchor_metrics = {
    "log_loss": float(phase51e_log_loss(
        phase51e_y_development,
        phase51e_anchor_development,
    )),
    "auroc": float(phase51e_auroc(
        phase51e_y_development,
        phase51e_anchor_development,
    )),
}

phase51e_major_groups = [
    int(group)
    for group in np.unique(phase51e_groups_development)
    if np.sum(phase51e_groups_development == group)
    >= PHASE51_SELECTION_SCREEN_CONFIG["major_group_minimum_n"]
]
phase51e_anchor_group_loss = {
    group: float(phase51e_log_loss(
        phase51e_y_development[
            phase51e_groups_development == group
        ],
        phase51e_anchor_development[
            phase51e_groups_development == group
        ],
    ))
    for group in phase51e_major_groups
}

phase51e_candidate_epoch_records = []
for phase51e_candidate_index in range(phase51e_candidate_count):
    for phase51e_epoch in range(phase51e_epoch_count + 1):
        if phase51e_epoch == 0:
            # Use the exact float64 anchor for the safe baseline; never allow
            # float32 cache roundoff to create a nominal gain or loss.
            phase51e_repeat_probability = np.repeat(
                phase51e_anchor_development[None, :],
                phase51e_repeat_count,
                axis=0,
            )
        else:
            phase51e_repeat_probability = phase51e_probability[
                phase51e_candidate_index,
                phase51e_epoch,
            ].astype(np.float64)
        phase51e_mean_probability = np.mean(
            phase51e_repeat_probability,
            axis=0,
        )
        phase51e_logloss = phase51e_log_loss(
            phase51e_y_development,
            phase51e_mean_probability,
        )
        phase51e_auc = phase51e_auroc(
            phase51e_y_development,
            phase51e_mean_probability,
        )
        phase51e_repeat_records = []
        for phase51e_repeat_slot, phase51e_repeat in enumerate(
            phase51e_selection_repeats
        ):
            phase51e_local_probability = phase51e_repeat_probability[
                phase51e_repeat_slot
            ]
            phase51e_repeat_logloss = phase51e_log_loss(
                phase51e_y_development,
                phase51e_local_probability,
            )
            phase51e_repeat_auc = phase51e_auroc(
                phase51e_y_development,
                phase51e_local_probability,
            )
            phase51e_repeat_records.append({
                "repeat": int(phase51e_repeat),
                "log_loss": float(phase51e_repeat_logloss),
                "auroc": float(phase51e_repeat_auc),
                "log_loss_gain": float(
                    phase51e_anchor_metrics["log_loss"]
                    - phase51e_repeat_logloss
                ),
                "auroc_gain": float(
                    phase51e_repeat_auc
                    - phase51e_anchor_metrics["auroc"]
                ),
            })

        phase51e_group_harms = []
        for phase51e_group in phase51e_major_groups:
            phase51e_mask = (
                phase51e_groups_development == phase51e_group
            )
            phase51e_group_candidate_loss = phase51e_log_loss(
                phase51e_y_development[phase51e_mask],
                phase51e_mean_probability[phase51e_mask],
            )
            phase51e_group_harms.append(
                phase51e_group_candidate_loss
                - phase51e_anchor_group_loss[phase51e_group]
            )

        phase51e_logloss_gain = (
            phase51e_anchor_metrics["log_loss"] - phase51e_logloss
        )
        phase51e_auroc_gain = (
            phase51e_auc - phase51e_anchor_metrics["auroc"]
        )
        phase51e_repeat_wins = int(sum(
            record["log_loss_gain"] > 0.0
            for record in phase51e_repeat_records
        ))
        phase51e_worst_repeat_regret = float(max(
            0.0,
            max(-record["log_loss_gain"] for record in phase51e_repeat_records),
        ))
        phase51e_maximum_group_harm = float(max(
            [0.0] + phase51e_group_harms
        ))
        phase51e_safe = bool(
            phase51e_epoch == 0
            or (
                phase51e_repeat_wins
                >= PHASE51_SELECTION_SCREEN_CONFIG[
                    "minimum_repeat_log_loss_wins"
                ]
                and phase51e_worst_repeat_regret
                <= PHASE51_SELECTION_SCREEN_CONFIG[
                    "maximum_repeat_log_loss_regret"
                ]
                and phase51e_maximum_group_harm
                <= PHASE51_SELECTION_SCREEN_CONFIG[
                    "maximum_major_group_harm"
                ]
                and phase51e_auroc_gain
                >= -PHASE51_SELECTION_SCREEN_CONFIG[
                    "maximum_auroc_deficit"
                ]
            )
        )
        phase51e_candidate_epoch_records.append({
            "candidate_index": int(phase51e_candidate_index),
            "epoch": int(phase51e_epoch),
            "specification": copy.deepcopy(
                phase51e_candidates[phase51e_candidate_index]
            ),
            "log_loss": float(phase51e_logloss),
            "auroc": float(phase51e_auc),
            "log_loss_gain": float(phase51e_logloss_gain),
            "auroc_gain": float(phase51e_auroc_gain),
            "repeat_log_loss_wins": int(phase51e_repeat_wins),
            "worst_repeat_log_loss_regret": float(
                phase51e_worst_repeat_regret
            ),
            "maximum_major_group_harm": float(
                phase51e_maximum_group_harm
            ),
            "safe": bool(phase51e_safe),
            "repeat_metrics": phase51e_repeat_records,
        })


def phase51e_public_record(record):
    return {
        key: copy.deepcopy(value)
        for key, value in record.items()
        if key != "repeat_metrics"
    } | {
        "repeat_metrics": copy.deepcopy(record["repeat_metrics"])
    }


phase51e_epoch_zero_record = next(
    record
    for record in phase51e_candidate_epoch_records
    if record["candidate_index"] == 0 and record["epoch"] == 0
)
phase51e_nonzero_safe_records = [
    record
    for record in phase51e_candidate_epoch_records
    if record["epoch"] > 0 and record["safe"]
]

if phase51e_nonzero_safe_records:
    phase51e_raw_log_loss_best = min(
        phase51e_nonzero_safe_records,
        key=lambda record: (
            record["log_loss"],
            -record["auroc"],
            record["candidate_index"],
            record["epoch"],
        ),
    )
    phase51e_log_loss_limit = (
        phase51e_raw_log_loss_best["log_loss"]
        + PHASE51_SELECTION_SCREEN_CONFIG["selection_log_loss_band"]
    )
    phase51e_band_records = [
        record
        for record in phase51e_nonzero_safe_records
        if record["log_loss"] <= phase51e_log_loss_limit
    ]
    assert phase51e_band_records
    phase51e_provisional = max(
        phase51e_band_records,
        key=lambda record: (
            record["auroc"],
            -record["log_loss"],
            -record["candidate_index"],
            -record["epoch"],
        ),
    )
else:
    phase51e_raw_log_loss_best = phase51e_epoch_zero_record
    phase51e_provisional = phase51e_epoch_zero_record

phase51e_advance_by_log_loss = bool(
    phase51e_provisional["epoch"] > 0
    and phase51e_provisional["log_loss_gain"]
    >= PHASE51_SELECTION_SCREEN_CONFIG[
        "minimum_screen_log_loss_gain"
    ]
    and phase51e_provisional["auroc_gain"]
    >= -PHASE51_SELECTION_SCREEN_CONFIG[
        "maximum_screen_auroc_deficit"
    ]
)
phase51e_advance_by_auroc = bool(
    phase51e_provisional["epoch"] > 0
    and phase51e_provisional["auroc_gain"]
    >= PHASE51_SELECTION_SCREEN_CONFIG[
        "minimum_screen_auroc_gain"
    ]
    and phase51e_provisional["log_loss_gain"]
    >= -PHASE51_SELECTION_SCREEN_CONFIG[
        "maximum_screen_log_loss_excess"
    ]
)
phase51e_screen_advanced = bool(
    phase51e_advance_by_log_loss or phase51e_advance_by_auroc
)
phase51e_selected = (
    phase51e_provisional
    if phase51e_screen_advanced else phase51e_epoch_zero_record
)

phase51e_selected_probability = (
    np.mean(
        phase51e_probability[
            phase51e_selected["candidate_index"],
            phase51e_selected["epoch"],
        ].astype(np.float64),
        axis=0,
    )
    if phase51e_selected["epoch"] > 0
    else phase51e_anchor_development.copy()
)

PHASE51_SELECTION_SCREEN_STATE_PRIVATE = {
    "contract_sha256": str(phase51e_contract_sha256),
    "screen_advanced": bool(phase51e_screen_advanced),
    "selected": copy.deepcopy(phase51e_selected),
    "selection_repeats": list(phase51e_selection_repeats),
    "confirmation_repeats": list(
        PHASE51_SELECTION_SCREEN_CONFIG["confirmation_repeats"]
    ),
    "selected_development_probability": (
        phase51e_selected_probability.copy()
    ),
}

phase51e_peak_vram_mb = (
    float(torch.cuda.max_memory_allocated(phase51e_device) / (1024 ** 2))
    if phase51e_amp_enabled else 0.0
)

phase51e_report = {
    "phase": "phase51_low_rank_dinov3_selection_screen",
    "status": (
        "candidate_frozen_for_independent_repeat_confirmation"
        if phase51e_screen_advanced
        else "screen_failed_no_update_stop"
    ),
    "validation_design": {
        "retired_pilot_repeat": int(
            PHASE51_SELECTION_SCREEN_CONFIG["retired_pilot_repeat"]
        ),
        "selection_repeats": list(phase51e_selection_repeats),
        "confirmation_repeats_not_used_by_phase51_selection": list(
            PHASE51_SELECTION_SCREEN_CONFIG["confirmation_repeats"]
        ),
        "selection_fold_count": int(
            phase51e_repeat_count * phase51e_fold_count
        ),
        "future_audit_case_count": int(
            phase51e_future_audit_indices.size
        ),
    },
    "search": {
        "candidate_count": int(phase51e_candidate_count),
        "epoch_zero_plus_trained_epoch_count": int(
            phase51e_epoch_count + 1
        ),
        "planned_model_fit_count": int(phase51e_planned_fit_count),
        "completed_model_fit_count": int(np.sum(phase51e_completed)),
        "progress_reused": bool(phase51e_progress_reused),
        "global_shared_epoch_across_all_selection_folds": True,
        "fold_specific_epoch_selection_used": False,
        "safe_nonzero_candidate_epoch_count": int(
            len(phase51e_nonzero_safe_records)
        ),
    },
    "anchor": copy.deepcopy(phase51e_anchor_metrics),
    "raw_log_loss_best": phase51e_public_record(
        phase51e_raw_log_loss_best
    ),
    "provisional": phase51e_public_record(phase51e_provisional),
    "selected": phase51e_public_record(phase51e_selected),
    "screen_gate": {
        "advance_by_log_loss": bool(phase51e_advance_by_log_loss),
        "advance_by_auroc": bool(phase51e_advance_by_auroc),
        "advanced": bool(phase51e_screen_advanced),
        "minimum_log_loss_gain": float(
            PHASE51_SELECTION_SCREEN_CONFIG[
                "minimum_screen_log_loss_gain"
            ]
        ),
        "minimum_auroc_gain": float(
            PHASE51_SELECTION_SCREEN_CONFIG[
                "minimum_screen_auroc_gain"
            ]
        ),
    },
    "selection_rule": (
        "among_safe_nonzero_candidate_epochs_choose_maximum_auroc_"
        "within_0p002_of_minimum_log_loss_then_require_screen_gate"
    ),
    "confirmation_plan": {
        "run_only_if_screen_advanced": True,
        "frozen_candidate_and_epoch": True,
        "confirmation_repeat_count": 2,
        "confirmation_model_fit_count": (
            10 if phase51e_screen_advanced else 0
        ),
        "future_audit_remains_locked_after_confirmation": True,
    },
    "validation_lock": {
        "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
        "selection_contract_sha256": str(phase51e_contract_sha256),
        "future_audit_features_used": False,
        "future_audit_labels_used": False,
        "future_audit_metrics_computed": False,
    },
    "execution": {
        "device": str(phase51e_device),
        "peak_vram_mb": float(phase51e_peak_vram_mb),
        "elapsed_seconds": round(
            time.perf_counter() - phase51e_started,
            3,
        ),
    },
    "fit_labels_used": True,
    "selection_labels_used": True,
    "confirmation_labels_used": False,
    "future_audit_labels_used": False,
    "public_leaderboard_used": False,
    "training_penultimate_cache_read": True,
    "persistent_private_development_predictions_written": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
}

PHASE51_SELECTION_SCREEN_REPORT_PRIVATE = copy.deepcopy(phase51e_report)
phase51e_atomic_json(
    phase51e_progress_directory / "phase51_selection_screen.json",
    {
        key: value
        for key, value in phase51e_report.items()
        if key != "selected_development_probability"
    },
)

print("BEGIN SANITIZED_PHASE51_SELECTION_SCREEN")
print(json.dumps(phase51e_report, indent=2))
print("END SANITIZED_PHASE51_SELECTION_SCREEN")

Phase51 selection screen: 1/40 fits complete; candidate=0, repeat=1, fold=0
Phase51 selection screen: 2/40 fits complete; candidate=0, repeat=1, fold=1
Phase51 selection screen: 3/40 fits complete; candidate=0, repeat=1, fold=2
Phase51 selection screen: 4/40 fits complete; candidate=0, repeat=1, fold=3
Phase51 selection screen: 5/40 fits complete; candidate=0, repeat=1, fold=4
Phase51 selection screen: 6/40 fits complete; candidate=0, repeat=2, fold=0
Phase51 selection screen: 7/40 fits complete; candidate=0, repeat=2, fold=1
Phase51 selection screen: 8/40 fits complete; candidate=0, repeat=2, fold=2
Phase51 selection screen: 9/40 fits complete; candidate=0, repeat=2, fold=3
Phase51 selection screen: 10/40 fits complete; candidate=0, repeat=2, fold=4
Phase51 selection screen: 11/40 fits complete; candidate=1, repeat=1, fold=0
Phase51 selection screen: 12/40 fits complete; candidate=1, repeat=1, fold=1
Phase51 selection screen: 13/40 fits complete; candidate=1, repeat=1, fold=2
Phase51 

In [72]:
# Phase51 Cell 151F 

import copy
import gc
import hashlib
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


phase51f_started = time.perf_counter()

PHASE51_CONFIRMATION_CONFIG = {
    "confirmation_repeats": [3, 4],
    "fold_count": 5,
    "batch_size": 16,
    "evaluation_batch_size": 24,
    "worker_count": 0,
    # The frozen epoch-8 checkpoint was produced inside the accepted
    # 12-epoch selection trajectory.  Confirmation stops after epoch 8 but
    # must preserve that original scheduler horizon exactly.
    "scheduler_horizon_epochs": 12,
    "gradient_clip": 1.0,
    "warmup_fraction": 0.10,
    "minimum_learning_rate_fraction": 0.10,
    "weight_decay": 0.02,
    "optimizer_betas": [0.9, 0.95],
    "major_group_minimum_n": 30,
    "minimum_log_loss_gain": 0.005,
    "minimum_auroc_gain": 0.002,
    "minimum_log_loss_repeat_wins": 2,
    "minimum_auroc_repeat_wins": 2,
    "maximum_worst_repeat_log_loss_regret": 0.003,
    "maximum_major_group_harm": 0.015,
    "log_loss_bootstrap_replicates": 5000,
    "auroc_bootstrap_replicates": 3000,
    "minimum_bootstrap_probability_positive": 0.95,
    "minimum_bootstrap_q05_log_loss_gain": 0.0,
    "minimum_bootstrap_q05_auroc_gain": 0.0,
    "bootstrap_seed": 513501,
    "progress_directory": "/kaggle/working/phase51_confirmation_private",
}


# -------------------------------------------------------------------------
# Frozen screen result and unchanged prospective lock.
# -------------------------------------------------------------------------

phase51f_required_names = [
    "PHASE51_ADAPTER_CONFIG_PRIVATE",
    "PHASE51_ROPE_SINCOS_PRIVATE",
    "PHASE51_SELECTION_SCREEN_REPORT_PRIVATE",
    "PHASE51_SELECTION_SCREEN_STATE_PRIVATE",
    "PHASE51_DATASET_CLASS_PRIVATE",
    "PHASE51_BATCH_SAMPLER_CLASS_PRIVATE",
    "PHASE51_TRAINING_FUNCTIONS_PRIVATE",
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase51d_cache",
    "phase51d_groups",
    "phase51d_labels",
    "phase51d_anchor_probability",
    "phase51d_anchor_logit",
    "phase51d_device",
    "phase51d_amp_enabled",
    "phase51d_amp_dtype",
]
phase51f_missing_names = [
    name for name in phase51f_required_names if name not in globals()
]
assert not phase51f_missing_names, {
    "message": "Run accepted Phase51 Cell 151E before confirmation.",
    "missing": phase51f_missing_names,
}

assert PHASE51_SELECTION_SCREEN_REPORT_PRIVATE[
    "status"
] == "candidate_frozen_for_independent_repeat_confirmation"
assert PHASE51_SELECTION_SCREEN_STATE_PRIVATE["screen_advanced"] is True
assert PHASE51_SELECTION_SCREEN_REPORT_PRIVATE[
    "validation_lock"
]["future_audit_labels_used"] is False

phase51f_frozen_record = copy.deepcopy(
    PHASE51_SELECTION_SCREEN_STATE_PRIVATE["selected"]
)
phase51f_frozen_specification = copy.deepcopy(
    phase51f_frozen_record["specification"]
)
phase51f_frozen_epoch = int(phase51f_frozen_record["epoch"])

assert phase51f_frozen_record["candidate_index"] == 3
assert phase51f_frozen_specification == {
    "candidate_index": 3,
    "lora_rank": 8,
    "rank_loss_weight": 0.05,
    "lora_alpha": 4.0,
}
assert phase51f_frozen_epoch == 8
assert PHASE51_CONFIRMATION_CONFIG["scheduler_horizon_epochs"] == int(
    PHASE51_SELECTION_SCREEN_REPORT_PRIVATE["search"][
        "epoch_zero_plus_trained_epoch_count"
    ] - 1
)
assert list(
    PHASE51_SELECTION_SCREEN_STATE_PRIVATE["confirmation_repeats"]
) == PHASE51_CONFIRMATION_CONFIG["confirmation_repeats"]

phase51f_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51f_future_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51f_fold_assignment = np.asarray(
    PHASE50_KNOWN_FOLD_ASSIGNMENT_PRIVATE,
    dtype=np.int8,
)
phase51f_groups = np.asarray(phase51d_groups, dtype=np.int64).reshape(-1)
phase51f_labels = np.asarray(phase51d_labels, dtype=np.float32).reshape(-1)
phase51f_anchor_probability = np.asarray(
    phase51d_anchor_probability,
    dtype=np.float64,
).reshape(-1)
phase51f_anchor_logit = np.asarray(
    phase51d_anchor_logit,
    dtype=np.float32,
).reshape(-1)
phase51f_cache = phase51d_cache
phase51f_device = phase51d_device
phase51f_amp_enabled = bool(phase51d_amp_enabled)
phase51f_amp_dtype = phase51d_amp_dtype

assert phase51f_development_indices.size == 1160
assert phase51f_future_audit_indices.size == 202
assert phase51f_fold_assignment.shape == (5, 1362)
assert phase51f_cache.shape == (1362, 6, 201, 384)
assert phase51f_cache.dtype == np.float16
assert np.intersect1d(
    phase51f_development_indices,
    phase51f_future_audit_indices,
).size == 0
assert np.all(
    phase51f_fold_assignment[:, phase51f_future_audit_indices] == -1
)

phase51f_functions = PHASE51_TRAINING_FUNCTIONS_PRIVATE
phase51f_set_seed = phase51f_functions["set_seed"]
phase51f_log_loss = phase51f_functions["log_loss"]
phase51f_auroc = phase51f_functions["auroc"]
phase51f_training_weights = phase51f_functions[
    "group_balanced_weights"
]
phase51f_rank_loss = phase51f_functions["within_group_rank_loss"]
phase51f_make_model = phase51f_functions["make_model"]
phase51f_set_training_mode = phase51f_functions["set_training_mode"]
phase51f_predict = phase51f_functions["predict"]
Phase51ConfirmationDataset = PHASE51_DATASET_CLASS_PRIVATE
Phase51ConfirmationBatchSampler = PHASE51_BATCH_SAMPLER_CLASS_PRIVATE

phase51f_confirmation_repeats = list(
    PHASE51_CONFIRMATION_CONFIG["confirmation_repeats"]
)
phase51f_repeat_to_slot = {
    int(repeat): int(slot)
    for slot, repeat in enumerate(phase51f_confirmation_repeats)
}
phase51f_fold_count = int(PHASE51_CONFIRMATION_CONFIG["fold_count"])
phase51f_planned_fit_count = int(
    len(phase51f_confirmation_repeats) * phase51f_fold_count
)
assert phase51f_planned_fit_count == 10

phase51f_development_position = np.full(1362, -1, dtype=np.int64)
phase51f_development_position[phase51f_development_indices] = np.arange(
    phase51f_development_indices.size,
    dtype=np.int64,
)
assert np.all(
    phase51f_development_position[phase51f_development_indices] >= 0
)
assert np.all(
    phase51f_development_position[phase51f_future_audit_indices] == -1
)


# -------------------------------------------------------------------------
# Atomic private progress and small trainable-only checkpoints.
# -------------------------------------------------------------------------

phase51f_progress_directory = Path(
    PHASE51_CONFIRMATION_CONFIG["progress_directory"]
)
phase51f_checkpoint_directory = (
    phase51f_progress_directory / "adapter_checkpoints"
)
phase51f_progress_directory.mkdir(parents=True, exist_ok=True)
phase51f_checkpoint_directory.mkdir(parents=True, exist_ok=True)
phase51f_progress_path = (
    phase51f_progress_directory / "phase51_confirmation_progress.npz"
)
phase51f_records_path = (
    phase51f_progress_directory / "phase51_confirmation_records.json"
)
phase51f_contract_path = (
    phase51f_progress_directory / "phase51_confirmation_contract.json"
)

phase51f_contract_payload = {
    "schema_version": "phase51f_confirmation_v2",
    "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
    "selection_contract_sha256": str(
        PHASE51_SELECTION_SCREEN_STATE_PRIVATE["contract_sha256"]
    ),
    "confirmation_repeats": phase51f_confirmation_repeats,
    "fold_count": int(phase51f_fold_count),
    "frozen_specification": phase51f_frozen_specification,
    "frozen_epoch": int(phase51f_frozen_epoch),
    "scheduler_horizon_epochs": int(
        PHASE51_CONFIRMATION_CONFIG["scheduler_horizon_epochs"]
    ),
    "batch_size": int(PHASE51_CONFIRMATION_CONFIG["batch_size"]),
    "weight_decay": float(PHASE51_CONFIRMATION_CONFIG["weight_decay"]),
    "lora_learning_rate": float(
        PHASE51_ADAPTER_CONFIG_PRIVATE["lora_learning_rate"]
    ),
    "pooling_learning_rate": float(
        PHASE51_ADAPTER_CONFIG_PRIVATE["pooling_learning_rate"]
    ),
    "head_learning_rate": float(
        PHASE51_ADAPTER_CONFIG_PRIVATE["head_learning_rate"]
    ),
}
phase51f_contract_sha256 = hashlib.sha256(
    json.dumps(
        phase51f_contract_payload,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


def phase51f_checkpoint_path(repeat, fold):
    return (
        phase51f_checkpoint_directory
        / f"phase51_repeat_{int(repeat)}_fold_{int(fold)}_adapter.pt"
    )


def phase51f_atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
    os.replace(temporary, path)


def phase51f_atomic_progress(path, probability, completed):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(
            handle,
            contract_sha256=np.asarray(phase51f_contract_sha256),
            probability=np.asarray(probability, dtype=np.float32),
            completed=np.asarray(completed, dtype=np.uint8),
        )
    os.replace(temporary, path)


def phase51f_trainable_state(model):
    output = {}
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            output[name] = parameter.detach().cpu().clone()
    assert output
    assert all(
        (
            ".lora_" in name
            or name.startswith("view_projection.")
            or name.startswith("head.")
        )
        for name in output
    )
    return output


def phase51f_atomic_checkpoint(path, model, repeat, fold, seed):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    payload = {
        "schema_version": "phase51_trainable_adapter_v1",
        "confirmation_contract_sha256": phase51f_contract_sha256,
        "repeat": int(repeat),
        "fold": int(fold),
        "seed": int(seed),
        "frozen_specification": copy.deepcopy(
            phase51f_frozen_specification
        ),
        "frozen_epoch": int(phase51f_frozen_epoch),
        "trainable_state_dict": phase51f_trainable_state(model),
    }
    torch.save(payload, temporary)
    os.replace(temporary, path)
    assert path.is_file()


phase51f_expected_probability_shape = (
    len(phase51f_confirmation_repeats),
    phase51f_development_indices.size,
)
phase51f_expected_completed_shape = (
    len(phase51f_confirmation_repeats),
    phase51f_fold_count,
)

phase51f_progress_reused = False
if phase51f_progress_path.is_file():
    with np.load(phase51f_progress_path, allow_pickle=False) as archive:
        phase51f_loaded_hash = str(archive["contract_sha256"].item())
        phase51f_probability = np.asarray(
            archive["probability"],
            dtype=np.float32,
        ).copy()
        phase51f_completed = np.asarray(
            archive["completed"],
            dtype=np.uint8,
        ).astype(bool)
    assert phase51f_loaded_hash == phase51f_contract_sha256, {
        "message": "Existing confirmation progress uses another contract.",
        "observed": phase51f_loaded_hash,
        "expected": phase51f_contract_sha256,
    }
    assert phase51f_probability.shape == phase51f_expected_probability_shape
    assert phase51f_completed.shape == phase51f_expected_completed_shape
    if phase51f_records_path.is_file():
        with phase51f_records_path.open("r", encoding="utf-8") as handle:
            phase51f_training_records = json.load(handle)
        assert isinstance(phase51f_training_records, list)
    else:
        phase51f_training_records = []
    for repeat in phase51f_confirmation_repeats:
        repeat_slot = phase51f_repeat_to_slot[repeat]
        for fold in range(phase51f_fold_count):
            if phase51f_completed[repeat_slot, fold]:
                assert phase51f_checkpoint_path(repeat, fold).is_file(), {
                    "message": "Completed confirmation fit lacks checkpoint.",
                    "repeat": int(repeat),
                    "fold": int(fold),
                }
    phase51f_progress_reused = True
else:
    phase51f_probability = np.full(
        phase51f_expected_probability_shape,
        np.nan,
        dtype=np.float32,
    )
    phase51f_completed = np.zeros(
        phase51f_expected_completed_shape,
        dtype=bool,
    )
    phase51f_training_records = []
    phase51f_atomic_progress(
        phase51f_progress_path,
        phase51f_probability,
        phase51f_completed,
    )
    phase51f_atomic_json(
        phase51f_contract_path,
        {
            **phase51f_contract_payload,
            "contract_sha256": phase51f_contract_sha256,
            "contains_labels": False,
            "contains_predictions": False,
            "contains_case_identifiers": False,
            "contains_voxel_data": False,
        },
    )


def phase51f_make_optimizer(model):
    lora_parameters = []
    pooling_parameters = []
    head_parameters = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if ".lora_" in name:
            lora_parameters.append(parameter)
        elif name.startswith("view_projection."):
            pooling_parameters.append(parameter)
        elif name.startswith("head."):
            head_parameters.append(parameter)
        else:
            raise AssertionError({
                "message": "Unexpected trainable parameter.",
                "name": name,
            })
    assert lora_parameters and pooling_parameters and head_parameters
    return torch.optim.AdamW([
        {
            "params": lora_parameters,
            "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
                "lora_learning_rate"
            ]),
        },
        {
            "params": pooling_parameters,
            "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
                "pooling_learning_rate"
            ]),
        },
        {
            "params": head_parameters,
            "lr": float(PHASE51_ADAPTER_CONFIG_PRIVATE[
                "head_learning_rate"
            ]),
        },
    ], betas=tuple(
        PHASE51_CONFIRMATION_CONFIG["optimizer_betas"]
    ), weight_decay=PHASE51_CONFIRMATION_CONFIG["weight_decay"])


def phase51f_lr_factor(step, total_steps, warmup_steps):
    step = int(step)
    total_steps = max(int(total_steps), 1)
    warmup_steps = max(int(warmup_steps), 1)
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(
        max(total_steps - warmup_steps, 1)
    )
    progress = min(max(progress, 0.0), 1.0)
    minimum = PHASE51_CONFIRMATION_CONFIG[
        "minimum_learning_rate_fraction"
    ]
    return float(
        minimum
        + (1.0 - minimum)
        * 0.5
        * (1.0 + math.cos(math.pi * progress))
    )


# -------------------------------------------------------------------------
# Ten fixed confirmation fits.  No candidate, epoch, or gate is searched.
# -------------------------------------------------------------------------

if phase51f_amp_enabled:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(phase51f_device)

phase51f_initial_completed_count = int(np.sum(phase51f_completed))
phase51f_fit_sequence_number = phase51f_initial_completed_count

for phase51f_repeat in phase51f_confirmation_repeats:
    phase51f_repeat_slot = phase51f_repeat_to_slot[phase51f_repeat]
    phase51f_assignment = phase51f_fold_assignment[phase51f_repeat]
    for phase51f_fold in range(phase51f_fold_count):
        if phase51f_completed[phase51f_repeat_slot, phase51f_fold]:
            continue

        phase51f_run_started = time.perf_counter()
        phase51f_run_seed = int(
            514000 + 100 * phase51f_repeat + phase51f_fold
        )
        phase51f_set_seed(phase51f_run_seed)

        phase51f_train_indices = phase51f_development_indices[
            phase51f_assignment[phase51f_development_indices]
            != phase51f_fold
        ]
        phase51f_valid_indices = phase51f_development_indices[
            phase51f_assignment[phase51f_development_indices]
            == phase51f_fold
        ]
        assert np.intersect1d(
            phase51f_train_indices,
            phase51f_valid_indices,
        ).size == 0
        assert np.union1d(
            phase51f_train_indices,
            phase51f_valid_indices,
        ).size == phase51f_development_indices.size
        assert np.intersect1d(
            phase51f_train_indices,
            phase51f_future_audit_indices,
        ).size == 0
        assert np.intersect1d(
            phase51f_valid_indices,
            phase51f_future_audit_indices,
        ).size == 0

        phase51f_train_weights = phase51f_training_weights(
            phase51f_groups[phase51f_train_indices]
        )
        phase51f_train_dataset = Phase51ConfirmationDataset(
            phase51f_cache,
            phase51f_train_indices,
            phase51f_train_weights,
        )
        phase51f_valid_dataset = Phase51ConfirmationDataset(
            phase51f_cache,
            phase51f_valid_indices,
        )
        phase51f_batch_sampler = Phase51ConfirmationBatchSampler(
            labels=phase51f_labels[phase51f_train_indices],
            groups=phase51f_groups[phase51f_train_indices],
            batch_size=PHASE51_CONFIRMATION_CONFIG["batch_size"],
            seed=phase51f_run_seed + 1,
        )
        phase51f_train_loader = DataLoader(
            phase51f_train_dataset,
            batch_sampler=phase51f_batch_sampler,
            num_workers=PHASE51_CONFIRMATION_CONFIG["worker_count"],
            pin_memory=phase51f_amp_enabled,
        )
        phase51f_valid_loader = DataLoader(
            phase51f_valid_dataset,
            batch_size=PHASE51_CONFIRMATION_CONFIG[
                "evaluation_batch_size"
            ],
            shuffle=False,
            num_workers=PHASE51_CONFIRMATION_CONFIG["worker_count"],
            pin_memory=phase51f_amp_enabled,
            drop_last=False,
        )

        phase51f_model = phase51f_make_model(
            phase51f_frozen_specification["lora_rank"]
        )
        phase51f_optimizer = phase51f_make_optimizer(phase51f_model)
        phase51f_total_steps = int(
            PHASE51_CONFIRMATION_CONFIG["scheduler_horizon_epochs"]
            * len(phase51f_train_loader)
        )
        phase51f_warmup_steps = max(
            1,
            int(round(
                PHASE51_CONFIRMATION_CONFIG["warmup_fraction"]
                * phase51f_total_steps
            )),
        )
        phase51f_scheduler = torch.optim.lr_scheduler.LambdaLR(
            phase51f_optimizer,
            lr_lambda=lambda step, total=phase51f_total_steps,
            warmup=phase51f_warmup_steps: phase51f_lr_factor(
                step,
                total,
                warmup,
            ),
        )

        phase51f_epoch_records = []
        phase51f_total_rank_active_batches = 0
        for phase51f_epoch in range(1, phase51f_frozen_epoch + 1):
            phase51f_batch_sampler.set_epoch(phase51f_epoch)
            phase51f_set_training_mode(phase51f_model)
            phase51f_bce_sum = 0.0
            phase51f_rank_sum = 0.0
            phase51f_case_count = 0
            phase51f_rank_active_batches = 0
            phase51f_maximum_gradient_norm = 0.0

            for phase51f_batch in phase51f_train_loader:
                phase51f_tokens = phase51f_batch["tokens"].to(
                    phase51f_device,
                    non_blocking=phase51f_amp_enabled,
                ).float()
                phase51f_batch_anchor = phase51f_batch[
                    "anchor_logit"
                ].to(
                    phase51f_device,
                    non_blocking=phase51f_amp_enabled,
                )
                phase51f_batch_labels = phase51f_batch["label"].to(
                    phase51f_device,
                    non_blocking=phase51f_amp_enabled,
                )
                phase51f_batch_groups = phase51f_batch["group"].to(
                    phase51f_device,
                    non_blocking=phase51f_amp_enabled,
                )
                phase51f_batch_weights = phase51f_batch[
                    "case_weight"
                ].to(
                    phase51f_device,
                    non_blocking=phase51f_amp_enabled,
                )

                phase51f_optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=phase51f_device.type,
                    dtype=phase51f_amp_dtype,
                    enabled=phase51f_amp_enabled,
                ):
                    phase51f_output = phase51f_model(
                        phase51f_tokens,
                        phase51f_batch_anchor,
                        PHASE51_ROPE_SINCOS_PRIVATE,
                    )
                phase51f_element_bce = F.binary_cross_entropy_with_logits(
                    phase51f_output["logit"].float(),
                    phase51f_batch_labels.float(),
                    reduction="none",
                )
                phase51f_bce_loss = torch.sum(
                    phase51f_element_bce
                    * phase51f_batch_weights.float()
                ) / torch.sum(phase51f_batch_weights.float())
                (
                    phase51f_batch_rank_loss,
                    phase51f_rank_group_count,
                ) = phase51f_rank_loss(
                    phase51f_output["logit"].float(),
                    phase51f_batch_labels.float(),
                    phase51f_batch_groups,
                )
                phase51f_loss = (
                    phase51f_bce_loss
                    + phase51f_frozen_specification["rank_loss_weight"]
                    * phase51f_batch_rank_loss
                )
                assert torch.isfinite(phase51f_loss)
                phase51f_loss.backward()
                phase51f_gradient_norm = torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for parameter in phase51f_model.parameters()
                        if parameter.requires_grad
                    ],
                    PHASE51_CONFIRMATION_CONFIG["gradient_clip"],
                )
                assert torch.isfinite(phase51f_gradient_norm)
                phase51f_optimizer.step()
                phase51f_scheduler.step()

                phase51f_batch_n = int(phase51f_batch_labels.numel())
                phase51f_bce_sum += float(
                    phase51f_bce_loss.detach().item()
                ) * phase51f_batch_n
                phase51f_rank_sum += float(
                    phase51f_batch_rank_loss.detach().item()
                ) * phase51f_batch_n
                phase51f_case_count += phase51f_batch_n
                phase51f_rank_active_batches += int(
                    phase51f_rank_group_count > 0
                )
                phase51f_maximum_gradient_norm = max(
                    phase51f_maximum_gradient_norm,
                    float(phase51f_gradient_norm.detach().item()),
                )

            assert phase51f_case_count == phase51f_train_indices.size
            assert phase51f_rank_active_batches > 0
            phase51f_total_rank_active_batches += (
                phase51f_rank_active_batches
            )
            phase51f_epoch_records.append({
                "epoch": int(phase51f_epoch),
                "train_bce": float(
                    phase51f_bce_sum / phase51f_case_count
                ),
                "train_rank": float(
                    phase51f_rank_sum / phase51f_case_count
                ),
                "rank_active_batch_count": int(
                    phase51f_rank_active_batches
                ),
                "maximum_gradient_norm": float(
                    phase51f_maximum_gradient_norm
                ),
            })

        phase51f_prediction = phase51f_predict(
            phase51f_model,
            phase51f_valid_loader,
            phase51f_valid_indices,
        )
        phase51f_valid_positions = phase51f_development_position[
            phase51f_valid_indices
        ]
        assert np.all(phase51f_valid_positions >= 0)
        phase51f_probability[
            phase51f_repeat_slot,
            phase51f_valid_positions,
        ] = phase51f_prediction["probability"].astype(np.float32)

        phase51f_checkpoint = phase51f_checkpoint_path(
            phase51f_repeat,
            phase51f_fold,
        )
        phase51f_atomic_checkpoint(
            phase51f_checkpoint,
            phase51f_model,
            phase51f_repeat,
            phase51f_fold,
            phase51f_run_seed,
        )

        phase51f_completed[
            phase51f_repeat_slot,
            phase51f_fold,
        ] = True
        phase51f_fit_sequence_number += 1
        phase51f_training_records.append({
            "repeat": int(phase51f_repeat),
            "fold": int(phase51f_fold),
            "seed": int(phase51f_run_seed),
            "train_n": int(phase51f_train_indices.size),
            "valid_n": int(phase51f_valid_indices.size),
            "epoch_count": int(phase51f_frozen_epoch),
            "scheduler_horizon_epochs": int(
                PHASE51_CONFIRMATION_CONFIG["scheduler_horizon_epochs"]
            ),
            "rank_active_batch_count": int(
                phase51f_total_rank_active_batches
            ),
            "validation_log_loss": float(phase51f_log_loss(
                phase51f_labels[phase51f_valid_indices],
                phase51f_prediction["probability"],
            )),
            "validation_auroc": float(phase51f_auroc(
                phase51f_labels[phase51f_valid_indices],
                phase51f_prediction["probability"],
            )),
            "records": phase51f_epoch_records,
            "checkpoint_file": phase51f_checkpoint.name,
            "elapsed_seconds": round(
                time.perf_counter() - phase51f_run_started,
                3,
            ),
        })

        phase51f_atomic_progress(
            phase51f_progress_path,
            phase51f_probability,
            phase51f_completed,
        )
        phase51f_atomic_json(
            phase51f_records_path,
            phase51f_training_records,
        )

        print(
            "Phase51 confirmation: "
            f"{phase51f_fit_sequence_number}/"
            f"{phase51f_planned_fit_count} fits complete; "
            f"repeat={phase51f_repeat}, fold={phase51f_fold}"
        )

        del phase51f_optimizer
        del phase51f_scheduler
        del phase51f_model
        del phase51f_train_loader
        del phase51f_valid_loader
        del phase51f_train_dataset
        del phase51f_valid_dataset
        gc.collect()
        if phase51f_amp_enabled:
            torch.cuda.empty_cache()


assert np.all(phase51f_completed)
assert np.all(np.isfinite(phase51f_probability))
assert np.all(
    (phase51f_probability > 0.0)
    & (phase51f_probability < 1.0)
)

phase51f_checkpoint_paths = [
    phase51f_checkpoint_path(repeat, fold)
    for repeat in phase51f_confirmation_repeats
    for fold in range(phase51f_fold_count)
]
assert len(phase51f_checkpoint_paths) == 10
assert all(path.is_file() for path in phase51f_checkpoint_paths)


# -------------------------------------------------------------------------
# Confirmation-only metrics and robustness.  Nothing is retuned here.
# -------------------------------------------------------------------------

phase51f_y_development = phase51f_labels[
    phase51f_development_indices
]
phase51f_groups_development = phase51f_groups[
    phase51f_development_indices
]
phase51f_anchor_development = phase51f_anchor_probability[
    phase51f_development_indices
]
phase51f_mean_probability = np.mean(
    phase51f_probability.astype(np.float64),
    axis=0,
)

phase51f_anchor_metrics = {
    "log_loss": float(phase51f_log_loss(
        phase51f_y_development,
        phase51f_anchor_development,
    )),
    "auroc": float(phase51f_auroc(
        phase51f_y_development,
        phase51f_anchor_development,
    )),
    "mean_probability": float(np.mean(phase51f_anchor_development)),
}
phase51f_candidate_metrics = {
    "log_loss": float(phase51f_log_loss(
        phase51f_y_development,
        phase51f_mean_probability,
    )),
    "auroc": float(phase51f_auroc(
        phase51f_y_development,
        phase51f_mean_probability,
    )),
    "mean_probability": float(np.mean(phase51f_mean_probability)),
}
phase51f_log_loss_gain = float(
    phase51f_anchor_metrics["log_loss"]
    - phase51f_candidate_metrics["log_loss"]
)
phase51f_auroc_gain = float(
    phase51f_candidate_metrics["auroc"]
    - phase51f_anchor_metrics["auroc"]
)

phase51f_repeat_metrics = []
for phase51f_repeat_slot, phase51f_repeat in enumerate(
    phase51f_confirmation_repeats
):
    phase51f_local_probability = phase51f_probability[
        phase51f_repeat_slot
    ].astype(np.float64)
    phase51f_local_log_loss = phase51f_log_loss(
        phase51f_y_development,
        phase51f_local_probability,
    )
    phase51f_local_auroc = phase51f_auroc(
        phase51f_y_development,
        phase51f_local_probability,
    )
    phase51f_repeat_metrics.append({
        "repeat": int(phase51f_repeat),
        "log_loss": float(phase51f_local_log_loss),
        "auroc": float(phase51f_local_auroc),
        "log_loss_gain": float(
            phase51f_anchor_metrics["log_loss"]
            - phase51f_local_log_loss
        ),
        "auroc_gain": float(
            phase51f_local_auroc
            - phase51f_anchor_metrics["auroc"]
        ),
    })

phase51f_log_loss_repeat_wins = int(sum(
    record["log_loss_gain"] > 0.0
    for record in phase51f_repeat_metrics
))
phase51f_auroc_repeat_wins = int(sum(
    record["auroc_gain"] > 0.0
    for record in phase51f_repeat_metrics
))
phase51f_worst_repeat_log_loss_regret = float(max(
    0.0,
    max(-record["log_loss_gain"] for record in phase51f_repeat_metrics),
))

phase51f_major_group_records = []
for phase51f_group in sorted(np.unique(
    phase51f_groups_development
).tolist()):
    phase51f_mask = phase51f_groups_development == phase51f_group
    phase51f_group_n = int(np.sum(phase51f_mask))
    if phase51f_group_n < PHASE51_CONFIRMATION_CONFIG[
        "major_group_minimum_n"
    ]:
        continue
    phase51f_group_anchor_loss = phase51f_log_loss(
        phase51f_y_development[phase51f_mask],
        phase51f_anchor_development[phase51f_mask],
    )
    phase51f_group_candidate_loss = phase51f_log_loss(
        phase51f_y_development[phase51f_mask],
        phase51f_mean_probability[phase51f_mask],
    )
    phase51f_major_group_records.append({
        "group": int(phase51f_group),
        "n": int(phase51f_group_n),
        "anchor_log_loss": float(phase51f_group_anchor_loss),
        "candidate_log_loss": float(phase51f_group_candidate_loss),
        "log_loss_gain": float(
            phase51f_group_anchor_loss - phase51f_group_candidate_loss
        ),
        "anchor_auroc": phase51f_auroc(
            phase51f_y_development[phase51f_mask],
            phase51f_anchor_development[phase51f_mask],
        ),
        "candidate_auroc": phase51f_auroc(
            phase51f_y_development[phase51f_mask],
            phase51f_mean_probability[phase51f_mask],
        ),
    })

phase51f_maximum_major_group_harm = float(max(
    [0.0]
    + [-record["log_loss_gain"] for record in phase51f_major_group_records]
))


def phase51f_stratified_bootstrap_indices(rng):
    parts = []
    for group in np.unique(phase51f_groups_development):
        for label in [0, 1]:
            stratum = np.flatnonzero(
                (phase51f_groups_development == group)
                & (phase51f_y_development == label)
            )
            if stratum.size:
                parts.append(rng.choice(
                    stratum,
                    size=stratum.size,
                    replace=True,
                ))
    output = np.concatenate(parts).astype(np.int64, copy=False)
    assert output.size == phase51f_development_indices.size
    return output


phase51f_bootstrap_rng = np.random.default_rng(
    PHASE51_CONFIRMATION_CONFIG["bootstrap_seed"]
)
phase51f_log_loss_bootstrap_gain = np.empty(
    PHASE51_CONFIRMATION_CONFIG["log_loss_bootstrap_replicates"],
    dtype=np.float64,
)
for phase51f_bootstrap_index in range(
    phase51f_log_loss_bootstrap_gain.size
):
    phase51f_sample = phase51f_stratified_bootstrap_indices(
        phase51f_bootstrap_rng
    )
    phase51f_log_loss_bootstrap_gain[phase51f_bootstrap_index] = (
        phase51f_log_loss(
            phase51f_y_development[phase51f_sample],
            phase51f_anchor_development[phase51f_sample],
        )
        - phase51f_log_loss(
            phase51f_y_development[phase51f_sample],
            phase51f_mean_probability[phase51f_sample],
        )
    )

phase51f_auroc_bootstrap_gain = np.empty(
    PHASE51_CONFIRMATION_CONFIG["auroc_bootstrap_replicates"],
    dtype=np.float64,
)
for phase51f_bootstrap_index in range(
    phase51f_auroc_bootstrap_gain.size
):
    phase51f_sample = phase51f_stratified_bootstrap_indices(
        phase51f_bootstrap_rng
    )
    phase51f_auroc_bootstrap_gain[phase51f_bootstrap_index] = (
        phase51f_auroc(
            phase51f_y_development[phase51f_sample],
            phase51f_mean_probability[phase51f_sample],
        )
        - phase51f_auroc(
            phase51f_y_development[phase51f_sample],
            phase51f_anchor_development[phase51f_sample],
        )
    )

assert np.all(np.isfinite(phase51f_log_loss_bootstrap_gain))
assert np.all(np.isfinite(phase51f_auroc_bootstrap_gain))

phase51f_bootstrap = {
    "log_loss": {
        "replicate_count": int(
            phase51f_log_loss_bootstrap_gain.size
        ),
        "mean_gain": float(np.mean(
            phase51f_log_loss_bootstrap_gain
        )),
        "q05_gain": float(np.quantile(
            phase51f_log_loss_bootstrap_gain,
            0.05,
        )),
        "probability_positive": float(np.mean(
            phase51f_log_loss_bootstrap_gain > 0.0
        )),
    },
    "auroc": {
        "replicate_count": int(
            phase51f_auroc_bootstrap_gain.size
        ),
        "mean_gain": float(np.mean(
            phase51f_auroc_bootstrap_gain
        )),
        "q05_gain": float(np.quantile(
            phase51f_auroc_bootstrap_gain,
            0.05,
        )),
        "probability_positive": float(np.mean(
            phase51f_auroc_bootstrap_gain > 0.0
        )),
    },
}

phase51f_confirmation_passed = bool(
    phase51f_log_loss_gain
    >= PHASE51_CONFIRMATION_CONFIG["minimum_log_loss_gain"]
    and phase51f_auroc_gain
    >= PHASE51_CONFIRMATION_CONFIG["minimum_auroc_gain"]
    and phase51f_log_loss_repeat_wins
    >= PHASE51_CONFIRMATION_CONFIG["minimum_log_loss_repeat_wins"]
    and phase51f_auroc_repeat_wins
    >= PHASE51_CONFIRMATION_CONFIG["minimum_auroc_repeat_wins"]
    and phase51f_worst_repeat_log_loss_regret
    <= PHASE51_CONFIRMATION_CONFIG[
        "maximum_worst_repeat_log_loss_regret"
    ]
    and phase51f_maximum_major_group_harm
    <= PHASE51_CONFIRMATION_CONFIG["maximum_major_group_harm"]
    and phase51f_bootstrap["log_loss"]["probability_positive"]
    >= PHASE51_CONFIRMATION_CONFIG[
        "minimum_bootstrap_probability_positive"
    ]
    and phase51f_bootstrap["auroc"]["probability_positive"]
    >= PHASE51_CONFIRMATION_CONFIG[
        "minimum_bootstrap_probability_positive"
    ]
    and phase51f_bootstrap["log_loss"]["q05_gain"]
    >= PHASE51_CONFIRMATION_CONFIG[
        "minimum_bootstrap_q05_log_loss_gain"
    ]
    and phase51f_bootstrap["auroc"]["q05_gain"]
    >= PHASE51_CONFIRMATION_CONFIG[
        "minimum_bootstrap_q05_auroc_gain"
    ]
)

phase51f_checkpoint_size_mb = float(sum(
    path.stat().st_size for path in phase51f_checkpoint_paths
) / (1024 ** 2))
phase51f_peak_vram_mb = (
    float(torch.cuda.max_memory_allocated(phase51f_device) / (1024 ** 2))
    if phase51f_amp_enabled else 0.0
)

PHASE51_CONFIRMATION_STATE_PRIVATE = {
    "contract_sha256": str(phase51f_contract_sha256),
    "confirmation_passed": bool(phase51f_confirmation_passed),
    "frozen_specification": copy.deepcopy(
        phase51f_frozen_specification
    ),
    "frozen_epoch": int(phase51f_frozen_epoch),
    "checkpoint_paths": tuple(phase51f_checkpoint_paths),
    "confirmation_probability": phase51f_probability.astype(
        np.float64
    ).copy(),
    "mean_confirmation_probability": (
        phase51f_mean_probability.copy()
    ),
}

phase51f_report = {
    "phase": "phase51_frozen_dinov3_adapter_repeat_confirmation",
    "status": (
        "confirmed_frozen_for_one_time_future_audit"
        if phase51f_confirmation_passed
        else "confirmation_failed_do_not_evaluate_future_audit"
    ),
    "frozen_candidate": {
        "specification": copy.deepcopy(phase51f_frozen_specification),
        "epoch": int(phase51f_frozen_epoch),
        "selected_before_confirmation": True,
        "modified_during_confirmation": False,
    },
    "confirmation_design": {
        "repeats": list(phase51f_confirmation_repeats),
        "fold_count_per_repeat": int(phase51f_fold_count),
        "planned_model_fit_count": int(phase51f_planned_fit_count),
        "completed_model_fit_count": int(np.sum(phase51f_completed)),
        "progress_reused": bool(phase51f_progress_reused),
        "selection_repeats_reused": False,
        "candidate_search_performed": False,
        "epoch_search_performed": False,
        "gate_search_performed": False,
        "scheduler_horizon_epochs": int(
            PHASE51_CONFIRMATION_CONFIG["scheduler_horizon_epochs"]
        ),
    },
    "anchor": copy.deepcopy(phase51f_anchor_metrics),
    "candidate": copy.deepcopy(phase51f_candidate_metrics),
    "improvements": {
        "log_loss_gain": float(phase51f_log_loss_gain),
        "auroc_gain": float(phase51f_auroc_gain),
        "log_loss_repeat_wins": int(
            phase51f_log_loss_repeat_wins
        ),
        "auroc_repeat_wins": int(phase51f_auroc_repeat_wins),
        "worst_repeat_log_loss_regret": float(
            phase51f_worst_repeat_log_loss_regret
        ),
        "maximum_major_group_harm": float(
            phase51f_maximum_major_group_harm
        ),
    },
    "repeat_metrics": phase51f_repeat_metrics,
    "major_group_metrics": phase51f_major_group_records,
    "bootstrap": phase51f_bootstrap,
    "thresholds": {
        "minimum_log_loss_gain": float(
            PHASE51_CONFIRMATION_CONFIG["minimum_log_loss_gain"]
        ),
        "minimum_auroc_gain": float(
            PHASE51_CONFIRMATION_CONFIG["minimum_auroc_gain"]
        ),
        "minimum_log_loss_repeat_wins": int(
            PHASE51_CONFIRMATION_CONFIG[
                "minimum_log_loss_repeat_wins"
            ]
        ),
        "minimum_auroc_repeat_wins": int(
            PHASE51_CONFIRMATION_CONFIG[
                "minimum_auroc_repeat_wins"
            ]
        ),
        "maximum_worst_repeat_log_loss_regret": float(
            PHASE51_CONFIRMATION_CONFIG[
                "maximum_worst_repeat_log_loss_regret"
            ]
        ),
        "maximum_major_group_harm": float(
            PHASE51_CONFIRMATION_CONFIG["maximum_major_group_harm"]
        ),
        "minimum_bootstrap_probability_positive": float(
            PHASE51_CONFIRMATION_CONFIG[
                "minimum_bootstrap_probability_positive"
            ]
        ),
        "minimum_bootstrap_q05_log_loss_gain": float(
            PHASE51_CONFIRMATION_CONFIG[
                "minimum_bootstrap_q05_log_loss_gain"
            ]
        ),
        "minimum_bootstrap_q05_auroc_gain": float(
            PHASE51_CONFIRMATION_CONFIG[
                "minimum_bootstrap_q05_auroc_gain"
            ]
        ),
    },
    "confirmation_passed": bool(phase51f_confirmation_passed),
    "checkpoint_contract": {
        "checkpoint_count": int(len(phase51f_checkpoint_paths)),
        "total_size_mb": float(phase51f_checkpoint_size_mb),
        "trainable_parameters_only": True,
        "contains_labels": False,
        "contains_predictions": False,
        "contains_case_rows": False,
        "contains_voxel_data": False,
    },
    "next_step": (
        "evaluate_locked_future_audit_exactly_once"
        if phase51f_confirmation_passed
        else "stop_phase51_without_audit_evaluation"
    ),
    "validation_lock": {
        "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
        "selection_contract_sha256": str(
            PHASE51_SELECTION_SCREEN_STATE_PRIVATE["contract_sha256"]
        ),
        "confirmation_contract_sha256": str(
            phase51f_contract_sha256
        ),
        "future_audit_case_count": int(
            phase51f_future_audit_indices.size
        ),
        "future_audit_features_used": False,
        "future_audit_labels_used": False,
        "future_audit_metrics_computed": False,
    },
    "execution": {
        "device": str(phase51f_device),
        "peak_vram_mb": float(phase51f_peak_vram_mb),
        "elapsed_seconds": round(
            time.perf_counter() - phase51f_started,
            3,
        ),
    },
    "fit_labels_used": True,
    "confirmation_labels_used_only_for_fixed_candidate_evaluation": True,
    "future_audit_labels_used": False,
    "public_leaderboard_used": False,
    "training_penultimate_cache_read": True,
    "persistent_private_development_predictions_written": True,
    "training_voxel_cache_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
}

PHASE51_CONFIRMATION_REPORT_PRIVATE = copy.deepcopy(phase51f_report)
phase51f_atomic_json(
    phase51f_progress_directory / "phase51_confirmation_report.json",
    phase51f_report,
)

print("BEGIN SANITIZED_PHASE51_CONFIRMATION")
print(json.dumps(phase51f_report, indent=2))
print("END SANITIZED_PHASE51_CONFIRMATION")

Phase51 confirmation: 1/10 fits complete; repeat=3, fold=0
Phase51 confirmation: 2/10 fits complete; repeat=3, fold=1
Phase51 confirmation: 3/10 fits complete; repeat=3, fold=2
Phase51 confirmation: 4/10 fits complete; repeat=3, fold=3
Phase51 confirmation: 5/10 fits complete; repeat=3, fold=4
Phase51 confirmation: 6/10 fits complete; repeat=4, fold=0
Phase51 confirmation: 7/10 fits complete; repeat=4, fold=1
Phase51 confirmation: 8/10 fits complete; repeat=4, fold=2
Phase51 confirmation: 9/10 fits complete; repeat=4, fold=3
Phase51 confirmation: 10/10 fits complete; repeat=4, fold=4
BEGIN SANITIZED_PHASE51_CONFIRMATION
{
  "phase": "phase51_frozen_dinov3_adapter_repeat_confirmation",
  "status": "confirmed_frozen_for_one_time_future_audit",
  "frozen_candidate": {
    "specification": {
      "candidate_index": 3,
      "lora_rank": 8,
      "rank_loss_weight": 0.05,
      "lora_alpha": 4.0
    },
    "epoch": 8,
    "selected_before_confirmation": true,
    "modified_during_confirmat

In [73]:
# Phase51 Cell 151G

import copy
import gc
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


phase51g_started = time.perf_counter()

PHASE51_FUTURE_AUDIT_CONFIG = {
    "evaluation_batch_size": 24,
    "worker_count": 0,
    "major_group_minimum_n": 20,
    "log_loss_bootstrap_replicates": 10000,
    "auroc_bootstrap_replicates": 5000,
    "bootstrap_seed": 515101,
    # These gates are frozen before the first audit-label access.
    "minimum_log_loss_gain": 0.003,
    "minimum_auroc_gain": 0.001,
    "minimum_bootstrap_probability_positive": 0.80,
    "minimum_bootstrap_q05_log_loss_gain": -0.005,
    "minimum_bootstrap_q05_auroc_gain": -0.002,
    "maximum_major_group_harm": 0.020,
    "stretch_target_log_loss": 0.22,
    "stretch_target_auroc": 0.96,
    "output_directory": "/kaggle/working/phase51_future_audit_private",
}


# -------------------------------------------------------------------------
# Resolve the already-frozen confirmation state without touching audit rows.
# -------------------------------------------------------------------------

phase51g_required_names = [
    "PHASE51_CONFIRMATION_REPORT_PRIVATE",
    "PHASE51_CONFIRMATION_STATE_PRIVATE",
    "PHASE51_SELECTION_SCREEN_STATE_PRIVATE",
    "PHASE51_TRAINING_FUNCTIONS_PRIVATE",
    "PHASE51_DATASET_CLASS_PRIVATE",
    "PHASE51_ADAPTER_CLASS_PRIVATE",
    "PHASE51_ADAPTER_CONFIG_PRIVATE",
    "PHASE51_BACKBONE_PRIVATE",
    "PHASE51_DINO_CONTRACT_PRIVATE",
    "PHASE51_ROPE_SINCOS_PRIVATE",
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase51d_cache",
    "phase51d_groups",
    "phase51d_labels",
    "phase51d_anchor_probability",
    "phase51d_anchor_logit",
    "phase51d_device",
    "phase51d_amp_enabled",
    "phase51d_amp_dtype",
]
phase51g_missing_names = [
    name for name in phase51g_required_names if name not in globals()
]
assert not phase51g_missing_names, {
    "message": "Run accepted Phase51 Cell 151F before the audit.",
    "missing": phase51g_missing_names,
}

assert PHASE51_CONFIRMATION_REPORT_PRIVATE[
    "status"
] == "confirmed_frozen_for_one_time_future_audit"
assert PHASE51_CONFIRMATION_STATE_PRIVATE[
    "confirmation_passed"
] is True
assert PHASE51_CONFIRMATION_REPORT_PRIVATE[
    "validation_lock"
]["future_audit_labels_used"] is False
assert PHASE51_CONFIRMATION_REPORT_PRIVATE[
    "validation_lock"
]["future_audit_metrics_computed"] is False

phase51g_frozen_specification = copy.deepcopy(
    PHASE51_CONFIRMATION_STATE_PRIVATE["frozen_specification"]
)
phase51g_frozen_epoch = int(
    PHASE51_CONFIRMATION_STATE_PRIVATE["frozen_epoch"]
)
assert phase51g_frozen_specification == {
    "candidate_index": 3,
    "lora_rank": 8,
    "rank_loss_weight": 0.05,
    "lora_alpha": 4.0,
}
assert phase51g_frozen_epoch == 8

phase51g_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51g_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase51g_cache = phase51d_cache
phase51g_device = phase51d_device
phase51g_amp_enabled = bool(phase51d_amp_enabled)
phase51g_amp_dtype = phase51d_amp_dtype

assert phase51g_development_indices.size == 1160
assert phase51g_audit_indices.size == 202
assert np.intersect1d(
    phase51g_development_indices,
    phase51g_audit_indices,
).size == 0
assert np.union1d(
    phase51g_development_indices,
    phase51g_audit_indices,
).size == 1362
assert phase51g_cache.shape == (1362, 6, 201, 384)
assert phase51g_cache.dtype == np.float16

phase51g_functions = PHASE51_TRAINING_FUNCTIONS_PRIVATE
phase51g_log_loss = phase51g_functions["log_loss"]
phase51g_auroc = phase51g_functions["auroc"]
phase51g_make_model = phase51g_functions["make_model"]

phase51g_checkpoint_paths = tuple(
    Path(path)
    for path in PHASE51_CONFIRMATION_STATE_PRIVATE["checkpoint_paths"]
)
assert len(phase51g_checkpoint_paths) == 10
assert len(set(phase51g_checkpoint_paths)) == 10
assert all(path.is_file() for path in phase51g_checkpoint_paths)


def phase51g_file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


# Hashes are retained only inside the private contract and are never printed.
phase51g_checkpoint_hashes_private = {
    path.name: phase51g_file_sha256(path)
    for path in phase51g_checkpoint_paths
}

phase51g_contract_payload_private = {
    "schema_version": "phase51g_one_time_future_audit_v1",
    "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
    "selection_contract_sha256": str(
        PHASE51_SELECTION_SCREEN_STATE_PRIVATE["contract_sha256"]
    ),
    "confirmation_contract_sha256": str(
        PHASE51_CONFIRMATION_STATE_PRIVATE["contract_sha256"]
    ),
    "frozen_specification": phase51g_frozen_specification,
    "frozen_epoch": int(phase51g_frozen_epoch),
    "ensemble_member_count": int(len(phase51g_checkpoint_paths)),
    "checkpoint_hashes": phase51g_checkpoint_hashes_private,
    "ensemble_rule": "arithmetic_mean_of_ten_member_probabilities",
    "audit_case_count": int(phase51g_audit_indices.size),
    "gates": {
        key: PHASE51_FUTURE_AUDIT_CONFIG[key]
        for key in [
            "minimum_log_loss_gain",
            "minimum_auroc_gain",
            "minimum_bootstrap_probability_positive",
            "minimum_bootstrap_q05_log_loss_gain",
            "minimum_bootstrap_q05_auroc_gain",
            "maximum_major_group_harm",
        ]
    },
}
phase51g_contract_sha256 = hashlib.sha256(
    json.dumps(
        phase51g_contract_payload_private,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

phase51g_output_directory = Path(
    PHASE51_FUTURE_AUDIT_CONFIG["output_directory"]
)
phase51g_output_directory.mkdir(parents=True, exist_ok=True)
phase51g_contract_path = (
    phase51g_output_directory / "phase51_future_audit_contract.json"
)
phase51g_report_path = (
    phase51g_output_directory / "phase51_future_audit_report.json"
)


def phase51g_atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
    os.replace(temporary, path)


# -------------------------------------------------------------------------
# An identical rerun only reloads the immutable report. It never scores the
# audit labels a second time. A changed contract is rejected.
# -------------------------------------------------------------------------

phase51g_report_reused = phase51g_report_path.is_file()
if phase51g_report_reused:
    with phase51g_report_path.open("r", encoding="utf-8") as handle:
        phase51g_report = json.load(handle)
    assert phase51g_report[
        "validation_lock"
    ]["audit_contract_sha256"] == phase51g_contract_sha256, {
        "message": "A completed audit exists under another contract.",
        "observed": phase51g_report["validation_lock"][
            "audit_contract_sha256"
        ],
        "expected": phase51g_contract_sha256,
    }
    assert phase51g_report["audit_evaluated_exactly_once"] is True
    PHASE51_FUTURE_AUDIT_REPORT_PRIVATE = copy.deepcopy(
        phase51g_report
    )
    PHASE51_FUTURE_AUDIT_STATE_PRIVATE = {
        "contract_sha256": str(phase51g_contract_sha256),
        "audit_gate_passed": bool(
            phase51g_report["audit_gate_passed"]
        ),
        "report_reused_without_rescoring": True,
        "checkpoint_paths": tuple(phase51g_checkpoint_paths),
    }

else:
    phase51g_atomic_json(
        phase51g_contract_path,
        {
            **phase51g_contract_payload_private,
            "contract_sha256": phase51g_contract_sha256,
            "checkpoint_hashes_stored_privately": True,
            "contains_labels": False,
            "contains_predictions": False,
            "contains_case_identifiers": False,
            "contains_voxel_data": False,
        },
    )

    class Phase51FutureAuditDataset(Dataset):
        """Label-free inference view over the prospectively locked rows."""

        def __init__(self, cache, indices, anchor_logit):
            self.cache = cache
            self.indices = np.asarray(indices, dtype=np.int64).reshape(-1)
            self.anchor_logit = np.asarray(
                anchor_logit,
                dtype=np.float32,
            ).reshape(-1)
            assert self.anchor_logit.shape == (1362,)

        def __len__(self):
            return int(self.indices.size)

        def __getitem__(self, position):
            case_index = int(self.indices[position])
            tokens = np.array(
                self.cache[case_index],
                dtype=np.float16,
                copy=True,
            )
            assert tokens.shape == (6, 201, 384)
            return {
                "tokens": torch.from_numpy(tokens),
                "anchor_logit": torch.tensor(
                    self.anchor_logit[case_index],
                    dtype=torch.float32,
                ),
                "case_index": torch.tensor(
                    case_index,
                    dtype=torch.int64,
                ),
            }


    phase51g_audit_dataset = Phase51FutureAuditDataset(
        phase51g_cache,
        phase51g_audit_indices,
        phase51d_anchor_logit,
    )
    phase51g_audit_loader = DataLoader(
        phase51g_audit_dataset,
        batch_size=PHASE51_FUTURE_AUDIT_CONFIG[
            "evaluation_batch_size"
        ],
        shuffle=False,
        num_workers=PHASE51_FUTURE_AUDIT_CONFIG["worker_count"],
        pin_memory=phase51g_amp_enabled,
        drop_last=False,
    )


    def phase51g_load_trainable_checkpoint(model, path):
        try:
            payload = torch.load(
                path,
                map_location="cpu",
                weights_only=True,
            )
        except TypeError:
            payload = torch.load(path, map_location="cpu")

        assert payload[
            "confirmation_contract_sha256"
        ] == PHASE51_CONFIRMATION_STATE_PRIVATE["contract_sha256"]
        assert payload["frozen_specification"] == phase51g_frozen_specification
        assert int(payload["frozen_epoch"]) == phase51g_frozen_epoch

        trainable_state = payload["trainable_state_dict"]
        expected_trainable_names = {
            name
            for name, parameter in model.named_parameters()
            if parameter.requires_grad
        }
        assert set(trainable_state) == expected_trainable_names, {
            "message": "Adapter checkpoint parameter contract changed.",
            "missing": sorted(
                expected_trainable_names - set(trainable_state)
            ),
            "unexpected": sorted(
                set(trainable_state) - expected_trainable_names
            ),
        }

        complete_state = model.state_dict()
        for name, value in trainable_state.items():
            assert isinstance(value, torch.Tensor)
            assert value.shape == complete_state[name].shape
            assert torch.all(torch.isfinite(value))
            complete_state[name] = value.to(
                dtype=complete_state[name].dtype
            )
        incompatible = model.load_state_dict(complete_state, strict=True)
        assert not incompatible.missing_keys
        assert not incompatible.unexpected_keys
        return payload


    @torch.no_grad()
    def phase51g_predict_member(model):
        model.eval()
        model.final_block.eval()
        model.final_norm.eval()
        probability_parts = []
        residual_parts = []
        index_parts = []

        for batch in phase51g_audit_loader:
            tokens = batch["tokens"].to(
                phase51g_device,
                non_blocking=phase51g_amp_enabled,
            ).float()
            anchor_logit = batch["anchor_logit"].to(
                phase51g_device,
                non_blocking=phase51g_amp_enabled,
            )
            with torch.autocast(
                device_type=phase51g_device.type,
                dtype=phase51g_amp_dtype,
                enabled=phase51g_amp_enabled,
            ):
                output = model(
                    tokens,
                    anchor_logit,
                    PHASE51_ROPE_SINCOS_PRIVATE,
                )
            probability_parts.append(
                torch.sigmoid(output["logit"].float()).cpu().numpy()
            )
            residual_parts.append(
                output["bounded_residual"].float().cpu().numpy()
            )
            index_parts.append(batch["case_index"].cpu().numpy())

        observed_indices = np.concatenate(index_parts).astype(
            np.int64,
            copy=False,
        )
        assert np.array_equal(observed_indices, phase51g_audit_indices)
        probability = np.concatenate(probability_parts).astype(
            np.float64,
            copy=False,
        )
        residual = np.concatenate(residual_parts).astype(
            np.float64,
            copy=False,
        )
        assert probability.shape == (202,)
        assert residual.shape == (202,)
        assert np.all(np.isfinite(probability))
        assert np.all(np.isfinite(residual))
        assert np.all((probability > 0.0) & (probability < 1.0))
        return probability, residual


    if phase51g_amp_enabled:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(phase51g_device)

    phase51g_member_probability = np.empty(
        (len(phase51g_checkpoint_paths), phase51g_audit_indices.size),
        dtype=np.float64,
    )
    phase51g_member_residual = np.empty_like(
        phase51g_member_probability
    )
    phase51g_member_records = []

    for member_index, checkpoint_path in enumerate(
        phase51g_checkpoint_paths
    ):
        model = phase51g_make_model(
            phase51g_frozen_specification["lora_rank"]
        )
        checkpoint_payload = phase51g_load_trainable_checkpoint(
            model,
            checkpoint_path,
        )
        probability, residual = phase51g_predict_member(model)
        phase51g_member_probability[member_index] = probability
        phase51g_member_residual[member_index] = residual
        phase51g_member_records.append({
            "member": int(member_index),
            "repeat": int(checkpoint_payload["repeat"]),
            "fold": int(checkpoint_payload["fold"]),
            "mean_probability": float(np.mean(probability)),
            "mean_absolute_residual": float(np.mean(np.abs(residual))),
        })
        print(
            "Phase51 future audit inference: "
            f"{member_index + 1}/{len(phase51g_checkpoint_paths)} "
            "frozen members complete"
        )
        del model
        del checkpoint_payload
        gc.collect()
        if phase51g_amp_enabled:
            torch.cuda.empty_cache()

    assert np.all(np.isfinite(phase51g_member_probability))
    assert np.all(np.isfinite(phase51g_member_residual))
    assert {
        (record["repeat"], record["fold"])
        for record in phase51g_member_records
    } == {
        (repeat, fold)
        for repeat in [3, 4]
        for fold in range(5)
    }, {
        "message": "The ten frozen confirmation members are incomplete."
    }
    phase51g_candidate_probability = np.mean(
        phase51g_member_probability,
        axis=0,
    )
    phase51g_anchor_probability = np.asarray(
        phase51d_anchor_probability,
        dtype=np.float64,
    )[phase51g_audit_indices]
    assert phase51g_candidate_probability.shape == (202,)
    assert phase51g_anchor_probability.shape == (202,)
    assert np.all((phase51g_candidate_probability > 0.0)
                  & (phase51g_candidate_probability < 1.0))
    assert np.all((phase51g_anchor_probability > 0.0)
                  & (phase51g_anchor_probability < 1.0))

    # All model decisions and audit predictions are now frozen. This is the
    # first and only audit-label access in this cell.
    phase51g_audit_labels = np.asarray(
        phase51d_labels,
        dtype=np.int64,
    )[phase51g_audit_indices]
    phase51g_audit_groups = np.asarray(
        phase51d_groups,
        dtype=np.int64,
    )[phase51g_audit_indices]
    assert phase51g_audit_labels.shape == (202,)
    assert set(np.unique(phase51g_audit_labels).tolist()) == {0, 1}

    phase51g_anchor_metrics = {
        "log_loss": float(phase51g_log_loss(
            phase51g_audit_labels,
            phase51g_anchor_probability,
        )),
        "auroc": float(phase51g_auroc(
            phase51g_audit_labels,
            phase51g_anchor_probability,
        )),
        "mean_probability": float(np.mean(
            phase51g_anchor_probability
        )),
    }
    phase51g_candidate_metrics = {
        "log_loss": float(phase51g_log_loss(
            phase51g_audit_labels,
            phase51g_candidate_probability,
        )),
        "auroc": float(phase51g_auroc(
            phase51g_audit_labels,
            phase51g_candidate_probability,
        )),
        "mean_probability": float(np.mean(
            phase51g_candidate_probability
        )),
    }
    phase51g_log_loss_gain = float(
        phase51g_anchor_metrics["log_loss"]
        - phase51g_candidate_metrics["log_loss"]
    )
    phase51g_auroc_gain = float(
        phase51g_candidate_metrics["auroc"]
        - phase51g_anchor_metrics["auroc"]
    )

    phase51g_major_group_records = []
    for group in sorted(np.unique(phase51g_audit_groups).tolist()):
        mask = phase51g_audit_groups == group
        group_n = int(np.sum(mask))
        if group_n < PHASE51_FUTURE_AUDIT_CONFIG[
            "major_group_minimum_n"
        ]:
            continue
        anchor_loss = phase51g_log_loss(
            phase51g_audit_labels[mask],
            phase51g_anchor_probability[mask],
        )
        candidate_loss = phase51g_log_loss(
            phase51g_audit_labels[mask],
            phase51g_candidate_probability[mask],
        )
        phase51g_major_group_records.append({
            "group": int(group),
            "n": int(group_n),
            "anchor_log_loss": float(anchor_loss),
            "candidate_log_loss": float(candidate_loss),
            "log_loss_gain": float(anchor_loss - candidate_loss),
            "anchor_auroc": phase51g_auroc(
                phase51g_audit_labels[mask],
                phase51g_anchor_probability[mask],
            ),
            "candidate_auroc": phase51g_auroc(
                phase51g_audit_labels[mask],
                phase51g_candidate_probability[mask],
            ),
        })

    phase51g_maximum_major_group_harm = float(max(
        [0.0]
        + [
            -record["log_loss_gain"]
            for record in phase51g_major_group_records
        ]
    ))


    def phase51g_stratified_bootstrap_indices(rng):
        sampled_parts = []
        for group in np.unique(phase51g_audit_groups):
            for label in [0, 1]:
                stratum = np.flatnonzero(
                    (phase51g_audit_groups == group)
                    & (phase51g_audit_labels == label)
                )
                if stratum.size:
                    sampled_parts.append(rng.choice(
                        stratum,
                        size=stratum.size,
                        replace=True,
                    ))
        sampled = np.concatenate(sampled_parts).astype(
            np.int64,
            copy=False,
        )
        assert sampled.size == phase51g_audit_indices.size
        return sampled


    bootstrap_rng = np.random.default_rng(
        PHASE51_FUTURE_AUDIT_CONFIG["bootstrap_seed"]
    )
    phase51g_log_loss_bootstrap_gain = np.empty(
        PHASE51_FUTURE_AUDIT_CONFIG[
            "log_loss_bootstrap_replicates"
        ],
        dtype=np.float64,
    )
    for bootstrap_index in range(
        phase51g_log_loss_bootstrap_gain.size
    ):
        sample = phase51g_stratified_bootstrap_indices(bootstrap_rng)
        phase51g_log_loss_bootstrap_gain[bootstrap_index] = (
            phase51g_log_loss(
                phase51g_audit_labels[sample],
                phase51g_anchor_probability[sample],
            )
            - phase51g_log_loss(
                phase51g_audit_labels[sample],
                phase51g_candidate_probability[sample],
            )
        )

    phase51g_auroc_bootstrap_gain = np.empty(
        PHASE51_FUTURE_AUDIT_CONFIG[
            "auroc_bootstrap_replicates"
        ],
        dtype=np.float64,
    )
    for bootstrap_index in range(
        phase51g_auroc_bootstrap_gain.size
    ):
        sample = phase51g_stratified_bootstrap_indices(bootstrap_rng)
        phase51g_auroc_bootstrap_gain[bootstrap_index] = (
            phase51g_auroc(
                phase51g_audit_labels[sample],
                phase51g_candidate_probability[sample],
            )
            - phase51g_auroc(
                phase51g_audit_labels[sample],
                phase51g_anchor_probability[sample],
            )
        )

    assert np.all(np.isfinite(phase51g_log_loss_bootstrap_gain))
    assert np.all(np.isfinite(phase51g_auroc_bootstrap_gain))
    phase51g_bootstrap = {
        "log_loss": {
            "replicate_count": int(
                phase51g_log_loss_bootstrap_gain.size
            ),
            "mean_gain": float(np.mean(
                phase51g_log_loss_bootstrap_gain
            )),
            "q05_gain": float(np.quantile(
                phase51g_log_loss_bootstrap_gain,
                0.05,
            )),
            "probability_positive": float(np.mean(
                phase51g_log_loss_bootstrap_gain > 0.0
            )),
        },
        "auroc": {
            "replicate_count": int(
                phase51g_auroc_bootstrap_gain.size
            ),
            "mean_gain": float(np.mean(
                phase51g_auroc_bootstrap_gain
            )),
            "q05_gain": float(np.quantile(
                phase51g_auroc_bootstrap_gain,
                0.05,
            )),
            "probability_positive": float(np.mean(
                phase51g_auroc_bootstrap_gain > 0.0
            )),
        },
    }

    phase51g_audit_gate_passed = bool(
        phase51g_log_loss_gain
        >= PHASE51_FUTURE_AUDIT_CONFIG["minimum_log_loss_gain"]
        and phase51g_auroc_gain
        >= PHASE51_FUTURE_AUDIT_CONFIG["minimum_auroc_gain"]
        and phase51g_bootstrap["log_loss"]["probability_positive"]
        >= PHASE51_FUTURE_AUDIT_CONFIG[
            "minimum_bootstrap_probability_positive"
        ]
        and phase51g_bootstrap["auroc"]["probability_positive"]
        >= PHASE51_FUTURE_AUDIT_CONFIG[
            "minimum_bootstrap_probability_positive"
        ]
        and phase51g_bootstrap["log_loss"]["q05_gain"]
        >= PHASE51_FUTURE_AUDIT_CONFIG[
            "minimum_bootstrap_q05_log_loss_gain"
        ]
        and phase51g_bootstrap["auroc"]["q05_gain"]
        >= PHASE51_FUTURE_AUDIT_CONFIG[
            "minimum_bootstrap_q05_auroc_gain"
        ]
        and phase51g_maximum_major_group_harm
        <= PHASE51_FUTURE_AUDIT_CONFIG["maximum_major_group_harm"]
    )

    phase51g_stretch_target_reached = bool(
        phase51g_candidate_metrics["log_loss"]
        <= PHASE51_FUTURE_AUDIT_CONFIG["stretch_target_log_loss"]
        and phase51g_candidate_metrics["auroc"]
        >= PHASE51_FUTURE_AUDIT_CONFIG["stretch_target_auroc"]
    )
    phase51g_peak_vram_mb = (
        float(torch.cuda.max_memory_allocated(
            phase51g_device
        ) / (1024 ** 2))
        if phase51g_amp_enabled else 0.0
    )

    phase51g_report = {
        "phase": "phase51_one_time_locked_future_audit",
        "status": (
            "audit_gate_passed_phase51_promoted"
            if phase51g_audit_gate_passed
            else "audit_gate_failed_phase51_not_promoted"
        ),
        "audit": {
            "case_count": int(phase51g_audit_indices.size),
            "evaluated_exactly_once": True,
            "previously_used_for_selection": False,
            "candidate_modified_after_confirmation": False,
            "ensemble_member_count": int(
                len(phase51g_checkpoint_paths)
            ),
            "ensemble_rule": (
                "arithmetic_mean_of_ten_member_probabilities"
            ),
        },
        "frozen_candidate": {
            "specification": copy.deepcopy(
                phase51g_frozen_specification
            ),
            "epoch": int(phase51g_frozen_epoch),
            "confirmation_log_loss": float(
                PHASE51_CONFIRMATION_REPORT_PRIVATE[
                    "candidate"
                ]["log_loss"]
            ),
            "confirmation_auroc": float(
                PHASE51_CONFIRMATION_REPORT_PRIVATE[
                    "candidate"
                ]["auroc"]
            ),
        },
        "anchor": phase51g_anchor_metrics,
        "candidate": phase51g_candidate_metrics,
        "improvements": {
            "log_loss_gain": float(phase51g_log_loss_gain),
            "auroc_gain": float(phase51g_auroc_gain),
            "maximum_major_group_harm": float(
                phase51g_maximum_major_group_harm
            ),
        },
        "ensemble_diagnostics": {
            "mean_member_probability_standard_deviation": float(
                np.mean(np.std(
                    phase51g_member_probability,
                    axis=0,
                    ddof=0,
                ))
            ),
            "q95_member_probability_standard_deviation": float(
                np.quantile(np.std(
                    phase51g_member_probability,
                    axis=0,
                    ddof=0,
                ), 0.95)
            ),
            "mean_absolute_probability_change_from_anchor": float(
                np.mean(np.abs(
                    phase51g_candidate_probability
                    - phase51g_anchor_probability
                ))
            ),
            "member_records": phase51g_member_records,
        },
        "major_group_metrics": phase51g_major_group_records,
        "bootstrap": phase51g_bootstrap,
        "audit_gate_thresholds": {
            key: PHASE51_FUTURE_AUDIT_CONFIG[key]
            for key in [
                "minimum_log_loss_gain",
                "minimum_auroc_gain",
                "minimum_bootstrap_probability_positive",
                "minimum_bootstrap_q05_log_loss_gain",
                "minimum_bootstrap_q05_auroc_gain",
                "maximum_major_group_harm",
            ]
        },
        "audit_gate_passed": bool(phase51g_audit_gate_passed),
        "stretch_target": {
            "maximum_log_loss": float(
                PHASE51_FUTURE_AUDIT_CONFIG[
                    "stretch_target_log_loss"
                ]
            ),
            "minimum_auroc": float(
                PHASE51_FUTURE_AUDIT_CONFIG[
                    "stretch_target_auroc"
                ]
            ),
            "reached_on_audit": bool(
                phase51g_stretch_target_reached
            ),
        },
        "decision": {
            "if_passed": (
                "refit_fixed_phase51_adapter_ensemble_on_all_1362_cases"
            ),
            "if_failed": (
                "retire_audit_and_stop_phase51_without_retuning_on_audit"
            ),
            "audit_labels_may_be_used_for_future_phase51_tuning": False,
        },
        "validation_lock": {
            "partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
            "selection_contract_sha256": str(
                PHASE51_SELECTION_SCREEN_STATE_PRIVATE[
                    "contract_sha256"
                ]
            ),
            "confirmation_contract_sha256": str(
                PHASE51_CONFIRMATION_STATE_PRIVATE[
                    "contract_sha256"
                ]
            ),
            "audit_contract_sha256": str(
                phase51g_contract_sha256
            ),
            "future_audit_labels_used_once": True,
            "future_audit_metrics_computed_once": True,
        },
        "audit_evaluated_exactly_once": True,
        "candidate_or_threshold_modified_after_audit": False,
        "public_leaderboard_used": False,
        "training_penultimate_cache_read": True,
        "training_voxel_cache_read": False,
        "smoke_data_read": False,
        "test_data_read": False,
        "case_level_indices_displayed": False,
        "case_level_predictions_displayed": False,
        "case_level_predictions_exported": False,
        "model_hashes_displayed": False,
        "execution": {
            "device": str(phase51g_device),
            "peak_vram_mb": float(phase51g_peak_vram_mb),
            "elapsed_seconds": round(
                time.perf_counter() - phase51g_started,
                3,
            ),
        },
    }

    # Persist only aggregate audit results. No rows, labels, or predictions.
    phase51g_atomic_json(phase51g_report_path, phase51g_report)
    PHASE51_FUTURE_AUDIT_REPORT_PRIVATE = copy.deepcopy(
        phase51g_report
    )
    PHASE51_FUTURE_AUDIT_STATE_PRIVATE = {
        "contract_sha256": str(phase51g_contract_sha256),
        "audit_gate_passed": bool(phase51g_audit_gate_passed),
        "report_reused_without_rescoring": False,
        "checkpoint_paths": tuple(phase51g_checkpoint_paths),
        "audit_member_probability": (
            phase51g_member_probability.copy()
        ),
        "audit_mean_probability": (
            phase51g_candidate_probability.copy()
        ),
    }


print("BEGIN SANITIZED_PHASE51_FUTURE_AUDIT")
print(json.dumps(phase51g_report, indent=2))
print("END SANITIZED_PHASE51_FUTURE_AUDIT")

Phase51 future audit inference: 1/10 frozen members complete
Phase51 future audit inference: 2/10 frozen members complete
Phase51 future audit inference: 3/10 frozen members complete
Phase51 future audit inference: 4/10 frozen members complete
Phase51 future audit inference: 5/10 frozen members complete
Phase51 future audit inference: 6/10 frozen members complete
Phase51 future audit inference: 7/10 frozen members complete
Phase51 future audit inference: 8/10 frozen members complete
Phase51 future audit inference: 9/10 frozen members complete
Phase51 future audit inference: 10/10 frozen members complete
BEGIN SANITIZED_PHASE51_FUTURE_AUDIT
{
  "phase": "phase51_one_time_locked_future_audit",
  "status": "audit_gate_failed_phase51_not_promoted",
  "audit": {
    "case_count": 202,
    "evaluated_exactly_once": true,
    "previously_used_for_selection": false,
    "candidate_modified_after_confirmation": false,
    "ensemble_member_count": 10,
    "ensemble_rule": "arithmetic_mean_of_ten

In [74]:
# Phase52 Cell 152A 

import copy
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np


phase52a_started = time.perf_counter()

PHASE52_CONTRACT_CONFIG = {
    "repeat_count": 5,
    "fold_count": 5,
    "partition_seeds": [520101, 520102, 520103, 520104, 520105],
    "major_group_minimum_n": 20,
    "scale_values": [0.0, 0.25, 0.5, 0.75, 1.0, 1.25],
    "support_shrinkage_values": [0.0, 5.0, 10.0, 20.0, 40.0, 80.0, 160.0],
    "residual_cap_values": [0.1, 0.2, 0.3, 0.4, 0.5],
    "uncertainty_exponent_values": [0.0, 0.5, 1.0],
    "partition_file": "/kaggle/working/phase52_repeated_partition.npz",
    "contract_file": "/kaggle/working/phase52_support_gate_contract.json",
}


# -------------------------------------------------------------------------
# Phase51 is retired. Its frozen, genuinely out-of-sample predictions become
# a read-only component for a new Phase52 nested-CV experiment.
# -------------------------------------------------------------------------

phase52a_required_names = [
    "PHASE51_FUTURE_AUDIT_REPORT_PRIVATE",
    "PHASE51_FUTURE_AUDIT_STATE_PRIVATE",
    "PHASE51_CONFIRMATION_STATE_PRIVATE",
    "PHASE50_DEVELOPMENT_INDICES_PRIVATE",
    "PHASE50_FUTURE_AUDIT_INDICES_PRIVATE",
    "PHASE50_PARTITION_SHA256_PRIVATE",
    "phase51d_labels",
    "phase51d_groups",
    "phase51d_anchor_probability",
]
phase52a_missing_names = [
    name for name in phase52a_required_names if name not in globals()
]
assert not phase52a_missing_names, {
    "message": "Run accepted Phase51 Cell 151G before Phase52.",
    "missing": phase52a_missing_names,
}

assert PHASE51_FUTURE_AUDIT_REPORT_PRIVATE[
    "status"
] == "audit_gate_failed_phase51_not_promoted"
assert PHASE51_FUTURE_AUDIT_REPORT_PRIVATE[
    "audit_gate_passed"
] is False
assert PHASE51_FUTURE_AUDIT_REPORT_PRIVATE[
    "audit_evaluated_exactly_once"
] is True
assert PHASE51_FUTURE_AUDIT_REPORT_PRIVATE[
    "candidate_or_threshold_modified_after_audit"
] is False

phase52a_labels = np.asarray(
    phase51d_labels,
    dtype=np.int64,
).reshape(-1)
phase52a_groups = np.asarray(
    phase51d_groups,
    dtype=np.int64,
).reshape(-1)
phase52a_anchor_probability = np.asarray(
    phase51d_anchor_probability,
    dtype=np.float64,
).reshape(-1)
phase52a_development_indices = np.asarray(
    PHASE50_DEVELOPMENT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)
phase52a_retired_audit_indices = np.asarray(
    PHASE50_FUTURE_AUDIT_INDICES_PRIVATE,
    dtype=np.int64,
).reshape(-1)

assert phase52a_labels.shape == (1362,)
assert phase52a_groups.shape == (1362,)
assert phase52a_anchor_probability.shape == (1362,)
assert phase52a_development_indices.size == 1160
assert phase52a_retired_audit_indices.size == 202
assert np.intersect1d(
    phase52a_development_indices,
    phase52a_retired_audit_indices,
).size == 0
assert np.union1d(
    phase52a_development_indices,
    phase52a_retired_audit_indices,
).size == 1362
assert set(np.unique(phase52a_labels).tolist()) == {0, 1}
assert np.unique(phase52a_groups).size == 15
assert np.all((phase52a_anchor_probability > 0.0)
              & (phase52a_anchor_probability < 1.0))

phase52a_development_probability = np.asarray(
    PHASE51_CONFIRMATION_STATE_PRIVATE[
        "mean_confirmation_probability"
    ],
    dtype=np.float64,
).reshape(-1)
assert phase52a_development_probability.shape == (1160,)

if "audit_mean_probability" in PHASE51_FUTURE_AUDIT_STATE_PRIVATE:
    phase52a_audit_probability = np.asarray(
        PHASE51_FUTURE_AUDIT_STATE_PRIVATE[
            "audit_mean_probability"
        ],
        dtype=np.float64,
    ).reshape(-1)
elif "phase51g_candidate_probability" in globals():
    phase52a_audit_probability = np.asarray(
        phase51g_candidate_probability,
        dtype=np.float64,
    ).reshape(-1)
else:
    raise AssertionError({
        "message": (
            "The retired-audit probability is unavailable in memory. "
            "Do not rescore labels; rerun Cell 151G only to restore its "
            "immutable report, then restore the current notebook state."
        )
    })
assert phase52a_audit_probability.shape == (202,)

phase52a_component_probability = np.full(
    1362,
    np.nan,
    dtype=np.float64,
)
phase52a_component_probability[
    phase52a_development_indices
] = phase52a_development_probability
phase52a_component_probability[
    phase52a_retired_audit_indices
] = phase52a_audit_probability
assert np.all(np.isfinite(phase52a_component_probability))
assert np.all((phase52a_component_probability > 0.0)
              & (phase52a_component_probability < 1.0))


def phase52a_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase52a_log_loss(labels, probability):
    labels = np.asarray(labels, dtype=np.float64).reshape(-1)
    probability = np.clip(
        np.asarray(probability, dtype=np.float64).reshape(-1),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    return float(np.mean(-(
        labels * np.log(probability)
        + (1.0 - labels) * np.log1p(-probability)
    )))


phase52a_anchor_logit = phase52a_logit(phase52a_anchor_probability)
phase52a_component_logit = phase52a_logit(
    phase52a_component_probability
)
phase52a_raw_residual = (
    phase52a_component_logit - phase52a_anchor_logit
)
phase52a_anchor_uncertainty = (
    4.0
    * phase52a_anchor_probability
    * (1.0 - phase52a_anchor_probability)
)
assert np.all(np.isfinite(phase52a_raw_residual))
assert np.all((phase52a_anchor_uncertainty >= 0.0)
              & (phase52a_anchor_uncertainty <= 1.0))


# -------------------------------------------------------------------------
# Diagnose the failed transport. This is Phase52 diagnosis after the audit
# was retired; it is not a second Phase51 selection or promotion attempt.
# -------------------------------------------------------------------------

phase52a_group_counts = {
    int(group): int(np.sum(phase52a_groups == group))
    for group in np.unique(phase52a_groups)
}
phase52a_audit_group_records = []
for group in sorted(np.unique(
    phase52a_groups[phase52a_retired_audit_indices]
).tolist()):
    local_indices = phase52a_retired_audit_indices[
        phase52a_groups[phase52a_retired_audit_indices] == group
    ]
    anchor_loss = phase52a_log_loss(
        phase52a_labels[local_indices],
        phase52a_anchor_probability[local_indices],
    )
    component_loss = phase52a_log_loss(
        phase52a_labels[local_indices],
        phase52a_component_probability[local_indices],
    )
    phase52a_audit_group_records.append({
        "group": int(group),
        "audit_n": int(local_indices.size),
        "training_group_n": int(phase52a_group_counts[int(group)]),
        "log_loss_gain": float(anchor_loss - component_loss),
        "mean_absolute_residual": float(np.mean(np.abs(
            phase52a_raw_residual[local_indices]
        ))),
    })

phase52a_major_audit_groups = {
    record["group"]
    for record in phase52a_audit_group_records
    if record["audit_n"]
    >= PHASE52_CONTRACT_CONFIG["major_group_minimum_n"]
}
phase52a_major_audit_mask = np.isin(
    phase52a_groups[phase52a_retired_audit_indices],
    sorted(phase52a_major_audit_groups),
)
phase52a_minor_audit_mask = ~phase52a_major_audit_mask
assert int(np.sum(phase52a_major_audit_mask)) == 160
assert int(np.sum(phase52a_minor_audit_mask)) == 42


def phase52a_partition_gain(mask):
    local_indices = phase52a_retired_audit_indices[mask]
    return float(
        phase52a_log_loss(
            phase52a_labels[local_indices],
            phase52a_anchor_probability[local_indices],
        )
        - phase52a_log_loss(
            phase52a_labels[local_indices],
            phase52a_component_probability[local_indices],
        )
    )


phase52a_major_audit_gain = phase52a_partition_gain(
    phase52a_major_audit_mask
)
phase52a_minor_audit_gain = phase52a_partition_gain(
    phase52a_minor_audit_mask
)
assert phase52a_major_audit_gain > 0.0
assert phase52a_minor_audit_gain < 0.0


# -------------------------------------------------------------------------
# A fresh, non-searched repeated partition for Phase52. All 1362 labels are
# now development labels; no pristine-holdout claim remains.
# -------------------------------------------------------------------------

phase52a_repeat_count = int(PHASE52_CONTRACT_CONFIG["repeat_count"])
phase52a_fold_count = int(PHASE52_CONTRACT_CONFIG["fold_count"])
phase52a_fold_assignment = np.full(
    (phase52a_repeat_count, phase52a_labels.size),
    -1,
    dtype=np.int8,
)

for repeat, seed in enumerate(
    PHASE52_CONTRACT_CONFIG["partition_seeds"]
):
    rng = np.random.default_rng(int(seed))
    for group in sorted(np.unique(phase52a_groups).tolist()):
        for label in [0, 1]:
            indices = np.flatnonzero(
                (phase52a_groups == group)
                & (phase52a_labels == label)
            )
            if indices.size == 0:
                continue
            indices = indices.copy()
            rng.shuffle(indices)
            fold_cycle = np.arange(indices.size, dtype=np.int64)
            fold_cycle = (
                fold_cycle + int(rng.integers(0, phase52a_fold_count))
            ) % phase52a_fold_count
            phase52a_fold_assignment[repeat, indices] = (
                fold_cycle.astype(np.int8)
            )

assert np.all(phase52a_fold_assignment >= 0)
assert np.all(phase52a_fold_assignment < phase52a_fold_count)

phase52a_fold_records = []
for repeat in range(phase52a_repeat_count):
    for fold in range(phase52a_fold_count):
        valid_mask = phase52a_fold_assignment[repeat] == fold
        train_mask = ~valid_mask
        assert np.sum(valid_mask) > 0
        assert np.sum(train_mask) + np.sum(valid_mask) == 1362
        phase52a_fold_records.append({
            "repeat": int(repeat),
            "fold": int(fold),
            "train_n": int(np.sum(train_mask)),
            "valid_n": int(np.sum(valid_mask)),
            "train_group_count": int(np.unique(
                phase52a_groups[train_mask]
            ).size),
            "valid_group_count": int(np.unique(
                phase52a_groups[valid_mask]
            ).size),
            "valid_prevalence": float(np.mean(
                phase52a_labels[valid_mask]
            )),
        })

phase52a_grid_count = int(
    len(PHASE52_CONTRACT_CONFIG["scale_values"])
    * len(PHASE52_CONTRACT_CONFIG["support_shrinkage_values"])
    * len(PHASE52_CONTRACT_CONFIG["residual_cap_values"])
    * len(PHASE52_CONTRACT_CONFIG["uncertainty_exponent_values"])
)
assert phase52a_grid_count == 630

phase52a_partition_digest = hashlib.sha256(
    np.ascontiguousarray(phase52a_fold_assignment).tobytes()
).hexdigest()
phase52a_contract_payload = {
    "schema_version": "phase52_support_aware_gate_v1",
    "source_partition_sha256": str(PHASE50_PARTITION_SHA256_PRIVATE),
    "phase51_audit_contract_sha256": str(
        PHASE51_FUTURE_AUDIT_REPORT_PRIVATE[
            "validation_lock"
        ]["audit_contract_sha256"]
    ),
    "phase52_partition_sha256": phase52a_partition_digest,
    "repeat_count": int(phase52a_repeat_count),
    "fold_count": int(phase52a_fold_count),
    "candidate_grid": {
        "scale_values": PHASE52_CONTRACT_CONFIG["scale_values"],
        "support_shrinkage_values": PHASE52_CONTRACT_CONFIG[
            "support_shrinkage_values"
        ],
        "residual_cap_values": PHASE52_CONTRACT_CONFIG[
            "residual_cap_values"
        ],
        "uncertainty_exponent_values": PHASE52_CONTRACT_CONFIG[
            "uncertainty_exponent_values"
        ],
    },
    "formula": (
        "z52 = z_anchor + scale * n_train_group/(n_train_group+kappa) "
        "* uncertainty(z_anchor)^gamma * clip(z51-z_anchor, -cap, cap)"
    ),
    "unknown_protocol_policy": "anchor_only",
}
phase52a_contract_sha256 = hashlib.sha256(
    json.dumps(
        phase52a_contract_payload,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


def phase52a_atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
    os.replace(temporary, path)


def phase52a_atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


phase52a_atomic_npz(
    PHASE52_CONTRACT_CONFIG["partition_file"],
    contract_sha256=np.asarray(phase52a_contract_sha256),
    fold_assignment=phase52a_fold_assignment,
)
phase52a_atomic_json(
    PHASE52_CONTRACT_CONFIG["contract_file"],
    {
        **phase52a_contract_payload,
        "contract_sha256": phase52a_contract_sha256,
        "contains_labels": False,
        "contains_probabilities": False,
        "contains_case_identifiers": False,
        "contains_voxel_data": False,
    },
)

PHASE52_SUPPORT_GATE_CONFIG_PRIVATE = copy.deepcopy(
    PHASE52_CONTRACT_CONFIG
)
PHASE52_SUPPORT_GATE_STATE_PRIVATE = {
    "contract_sha256": str(phase52a_contract_sha256),
    "fold_assignment": phase52a_fold_assignment.copy(),
    "anchor_probability": phase52a_anchor_probability.copy(),
    "component_probability": phase52a_component_probability.copy(),
    "anchor_logit": phase52a_anchor_logit.copy(),
    "component_raw_residual": phase52a_raw_residual.copy(),
    "anchor_uncertainty": phase52a_anchor_uncertainty.copy(),
    "labels": phase52a_labels.copy(),
    "groups": phase52a_groups.copy(),
}

phase52a_report = {
    "phase": "phase52_support_aware_adapter_gate_contract",
    "status": "accepted_phase51_retired_phase52_opened",
    "phase51_decision": {
        "promoted": False,
        "audit_log_loss_gain": float(
            PHASE51_FUTURE_AUDIT_REPORT_PRIVATE[
                "improvements"
            ]["log_loss_gain"]
        ),
        "audit_auroc_gain": float(
            PHASE51_FUTURE_AUDIT_REPORT_PRIVATE[
                "improvements"
            ]["auroc_gain"]
        ),
        "phase51_may_be_retuned_on_retired_audit": False,
    },
    "transport_diagnosis": {
        "major_audit_case_count": int(np.sum(
            phase52a_major_audit_mask
        )),
        "minor_audit_case_count": int(np.sum(
            phase52a_minor_audit_mask
        )),
        "major_group_log_loss_gain": float(
            phase52a_major_audit_gain
        ),
        "minor_group_log_loss_gain": float(
            phase52a_minor_audit_gain
        ),
        "failure_concentrated_in_sparse_protocols": bool(
            phase52a_major_audit_gain > 0.0
            and phase52a_minor_audit_gain < 0.0
        ),
        "audit_group_records": phase52a_audit_group_records,
    },
    "component_contract": {
        "case_count": 1362,
        "development_case_count": 1160,
        "retired_audit_case_count": 202,
        "development_prediction_semantics": (
            "mean_of_two_confirmation_repeat_oof_predictions"
        ),
        "retired_audit_prediction_semantics": (
            "mean_of_ten_frozen_out_of_sample_member_predictions"
        ),
        "mixed_ensemble_multiplicity_acknowledged": True,
        "component_used_only_as_read_only_input": True,
    },
    "validation_design": {
        "repeat_count": int(phase52a_repeat_count),
        "fold_count_per_repeat": int(phase52a_fold_count),
        "total_outer_fold_count": int(
            phase52a_repeat_count * phase52a_fold_count
        ),
        "stratification": "within_acquisition_group_and_label",
        "partition_search_performed": False,
        "exact_case_coverage_per_repeat": True,
        "fold_records": phase52a_fold_records,
        "new_pristine_holdout_claimed": False,
        "all_labels_now_development_labels": True,
    },
    "planned_gate": {
        "formula": phase52a_contract_payload["formula"],
        "candidate_count": int(phase52a_grid_count),
        "selection": "fully_nested_repeated_cross_validation",
        "group_support_count_source": "training_partition_only",
        "explicit_group_specific_free_parameter": False,
        "unknown_protocol_policy": "anchor_only",
    },
    "stretch_target": {
        "maximum_log_loss": 0.22,
        "minimum_auroc": 0.96,
        "target_used_as_selection_hyperparameter": False,
    },
    "contract_sha256": str(phase52a_contract_sha256),
    "partition_sha256": str(phase52a_partition_digest),
    "phase51_audit_labels_reused_for_phase51_selection": False,
    "retired_audit_labels_used_for_phase52_transport_diagnosis": True,
    "labels_used_for_phase52_fold_stratification": True,
    "model_parameters_selected": False,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(
        time.perf_counter() - phase52a_started,
        3,
    ),
}

PHASE52_SUPPORT_GATE_REPORT_PRIVATE = copy.deepcopy(phase52a_report)
print("BEGIN SANITIZED_PHASE52_SUPPORT_GATE_CONTRACT")
print(json.dumps(phase52a_report, indent=2))
print("END SANITIZED_PHASE52_SUPPORT_GATE_CONTRACT")

BEGIN SANITIZED_PHASE52_SUPPORT_GATE_CONTRACT
{
  "phase": "phase52_support_aware_adapter_gate_contract",
  "status": "accepted_phase51_retired_phase52_opened",
  "phase51_decision": {
    "promoted": false,
    "audit_log_loss_gain": -3.7043393609104136e-05,
    "audit_auroc_gain": 0.00306900306900304,
    "phase51_may_be_retuned_on_retired_audit": false
  },
  "transport_diagnosis": {
    "major_audit_case_count": 160,
    "minor_audit_case_count": 42,
    "major_group_log_loss_gain": 0.014681953926715952,
    "minor_group_log_loss_gain": -0.05610941413770443,
    "failure_concentrated_in_sparse_protocols": true,
    "audit_group_records": [
      {
        "group": 0,
        "audit_n": 6,
        "training_group_n": 39,
        "log_loss_gain": -0.04574960663223521,
        "mean_absolute_residual": 0.3524959521428837
      },
      {
        "group": 1,
        "audit_n": 69,
        "training_group_n": 456,
        "log_loss_gain": 0.02202805031631805,
        "mean_absolute_resi

In [75]:
# Phase52 Cell 152B

import copy
import hashlib
import itertools
import json
import os
import time
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.metrics import roc_auc_score


phase52b_started = time.perf_counter()


# -------------------------------------------------------------------------
# Cell 152B
# Fully nested selection of the Phase52 support-aware Phase51 residual gate.
#
# Important semantics:
#   * Phase51 remains retired and is never retuned here.
#   * Its immutable out-of-sample probabilities are a read-only component.
#   * Every outer validation row is excluded from its gate selection.
#   * The zero-update anchor is an explicit safe candidate in every search.
#   * Nested metrics are primary; shared-specification metrics are diagnostic.
# -------------------------------------------------------------------------

phase52b_required_names = [
    "PHASE52_SUPPORT_GATE_CONFIG_PRIVATE",
    "PHASE52_SUPPORT_GATE_STATE_PRIVATE",
    "PHASE52_SUPPORT_GATE_REPORT_PRIVATE",
]
phase52b_missing_names = [
    name for name in phase52b_required_names if name not in globals()
]
assert not phase52b_missing_names, {
    "message": "Run accepted Phase52 Cell 152A before Cell 152B.",
    "missing": phase52b_missing_names,
}

assert PHASE52_SUPPORT_GATE_REPORT_PRIVATE[
    "status"
] == "accepted_phase51_retired_phase52_opened"
assert PHASE52_SUPPORT_GATE_REPORT_PRIVATE[
    "model_parameters_selected"
] is False

phase52b_config = copy.deepcopy(
    PHASE52_SUPPORT_GATE_CONFIG_PRIVATE
)
phase52b_state = PHASE52_SUPPORT_GATE_STATE_PRIVATE

phase52b_labels = np.asarray(
    phase52b_state["labels"], dtype=np.int64
).reshape(-1)
phase52b_groups = np.asarray(
    phase52b_state["groups"], dtype=np.int64
).reshape(-1)
phase52b_anchor_probability = np.asarray(
    phase52b_state["anchor_probability"], dtype=np.float64
).reshape(-1)
phase52b_component_probability = np.asarray(
    phase52b_state["component_probability"], dtype=np.float64
).reshape(-1)
phase52b_anchor_logit = np.asarray(
    phase52b_state["anchor_logit"], dtype=np.float64
).reshape(-1)
phase52b_raw_residual = np.asarray(
    phase52b_state["component_raw_residual"], dtype=np.float64
).reshape(-1)
phase52b_anchor_uncertainty = np.asarray(
    phase52b_state["anchor_uncertainty"], dtype=np.float64
).reshape(-1)
phase52b_fold_assignment = np.asarray(
    phase52b_state["fold_assignment"], dtype=np.int64
)

phase52b_case_count = int(phase52b_labels.size)
phase52b_repeat_count = int(phase52b_config["repeat_count"])
phase52b_fold_count = int(phase52b_config["fold_count"])

assert phase52b_case_count == 1362
assert phase52b_labels.shape == (phase52b_case_count,)
assert phase52b_groups.shape == (phase52b_case_count,)
assert phase52b_anchor_probability.shape == (phase52b_case_count,)
assert phase52b_component_probability.shape == (phase52b_case_count,)
assert phase52b_anchor_logit.shape == (phase52b_case_count,)
assert phase52b_raw_residual.shape == (phase52b_case_count,)
assert phase52b_anchor_uncertainty.shape == (phase52b_case_count,)
assert phase52b_fold_assignment.shape == (
    phase52b_repeat_count,
    phase52b_case_count,
)
assert set(np.unique(phase52b_labels).tolist()) == {0, 1}
assert np.unique(phase52b_groups).size == 15
assert np.all(np.isfinite(phase52b_anchor_probability))
assert np.all(np.isfinite(phase52b_component_probability))
assert np.all(np.isfinite(phase52b_anchor_logit))
assert np.all(np.isfinite(phase52b_raw_residual))
assert np.all(np.isfinite(phase52b_anchor_uncertainty))
assert np.all((phase52b_anchor_probability > 0.0)
              & (phase52b_anchor_probability < 1.0))
assert np.all((phase52b_component_probability > 0.0)
              & (phase52b_component_probability < 1.0))
assert np.all((phase52b_anchor_uncertainty >= 0.0)
              & (phase52b_anchor_uncertainty <= 1.0))
assert np.all((phase52b_fold_assignment >= 0)
              & (phase52b_fold_assignment < phase52b_fold_count))


PHASE52_NESTED_CONFIG = {
    "selection_log_loss_band": 0.001,
    "inner_maximum_worst_fold_excess": 0.015,
    "inner_maximum_major_group_harm": 0.030,
    "shared_maximum_worst_repeat_excess": 0.010,
    "shared_maximum_major_group_harm": 0.020,
    "major_group_minimum_n": 20,
    "bootstrap_log_loss_replicates": 5000,
    "bootstrap_auroc_replicates": 3000,
    "bootstrap_seed": 520903,
    "balanced_minimum_log_loss_gain": 0.002,
    "balanced_minimum_auroc_gain": 0.0005,
    "rank_minimum_auroc_gain": 0.0015,
    "rank_maximum_log_loss_excess": 0.0005,
    "minimum_repeat_metric_wins": 3,
    "maximum_worst_repeat_log_loss_excess": 0.005,
    "maximum_major_group_harm": 0.015,
    "balanced_minimum_log_loss_bootstrap_probability": 0.80,
    "balanced_minimum_auroc_bootstrap_probability": 0.80,
    "rank_minimum_auroc_bootstrap_probability": 0.85,
    "rank_minimum_auroc_bootstrap_q05": -0.0015,
    "rank_minimum_log_loss_bootstrap_q05": -0.010,
    "private_directory": "/kaggle/working/phase52_private_checkpoint",
    "portable_state_file": (
        "/kaggle/working/phase52_private_checkpoint/"
        "phase52_support_gate.json"
    ),
    "nested_probability_file": (
        "/kaggle/working/phase52_private_checkpoint/"
        "phase52_nested_oof_float64.npy"
    ),
    "report_file": (
        "/kaggle/working/phase52_private_checkpoint/"
        "phase52_nested_report.json"
    ),
}


def phase52b_logit(probability):
    probability = np.clip(
        np.asarray(probability, dtype=np.float64),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    return np.log(probability) - np.log1p(-probability)


def phase52b_sigmoid(logit):
    logit = np.clip(
        np.asarray(logit, dtype=np.float64),
        -40.0,
        40.0,
    )
    return 1.0 / (1.0 + np.exp(-logit))


def phase52b_log_loss(labels, probability):
    labels = np.asarray(labels, dtype=np.float64).reshape(-1)
    probability = np.clip(
        np.asarray(probability, dtype=np.float64).reshape(-1),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    assert labels.shape == probability.shape
    return float(np.mean(-(
        labels * np.log(probability)
        + (1.0 - labels) * np.log1p(-probability)
    )))


def phase52b_log_loss_matrix(labels, probability_matrix):
    labels = np.asarray(labels, dtype=np.float64).reshape(1, -1)
    probability_matrix = np.clip(
        np.asarray(probability_matrix, dtype=np.float64),
        1.0e-7,
        1.0 - 1.0e-7,
    )
    assert probability_matrix.ndim == 2
    assert probability_matrix.shape[1] == labels.shape[1]
    return np.mean(-(
        labels * np.log(probability_matrix)
        + (1.0 - labels) * np.log1p(-probability_matrix)
    ), axis=1)


def phase52b_auroc(labels, probability):
    labels = np.asarray(labels, dtype=np.int64).reshape(-1)
    probability = np.asarray(
        probability, dtype=np.float64
    ).reshape(-1)
    assert labels.shape == probability.shape
    assert set(np.unique(labels).tolist()) == {0, 1}
    assert np.all(np.isfinite(probability))
    return float(roc_auc_score(labels, probability))


def phase52b_metrics(labels, probability):
    return {
        "n": int(np.asarray(labels).size),
        "log_loss": phase52b_log_loss(labels, probability),
        "auroc": phase52b_auroc(labels, probability),
        "mean_probability": float(np.mean(probability)),
    }


def phase52b_atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
    os.replace(temporary, path)


def phase52b_atomic_npy(path, array):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as handle:
        np.save(handle, np.asarray(array))
    os.replace(temporary, path)


# -------------------------------------------------------------------------
# Candidate grid. Candidate zero is deliberately the first candidate and is
# the exact anchor. The remaining scale-zero entries are harmless duplicates
# retained so Cell 152B exactly honors Cell 152A's 630-candidate contract.
# -------------------------------------------------------------------------

phase52b_candidate_records = []
for candidate_index, values in enumerate(itertools.product(
    phase52b_config["scale_values"],
    phase52b_config["support_shrinkage_values"],
    phase52b_config["residual_cap_values"],
    phase52b_config["uncertainty_exponent_values"],
)):
    scale, support_shrinkage, residual_cap, uncertainty_exponent = values
    phase52b_candidate_records.append({
        "candidate_index": int(candidate_index),
        "scale": float(scale),
        "support_shrinkage": float(support_shrinkage),
        "residual_cap": float(residual_cap),
        "uncertainty_exponent": float(uncertainty_exponent),
    })

phase52b_candidate_count = len(phase52b_candidate_records)
assert phase52b_candidate_count == 630
assert phase52b_candidate_records[0] == {
    "candidate_index": 0,
    "scale": 0.0,
    "support_shrinkage": 0.0,
    "residual_cap": 0.1,
    "uncertainty_exponent": 0.0,
}

phase52b_candidate_scale = np.asarray([
    record["scale"] for record in phase52b_candidate_records
], dtype=np.float64)
phase52b_candidate_shrinkage = np.asarray([
    record["support_shrinkage"]
    for record in phase52b_candidate_records
], dtype=np.float64)
phase52b_candidate_cap = np.asarray([
    record["residual_cap"]
    for record in phase52b_candidate_records
], dtype=np.float64)
phase52b_candidate_gamma = np.asarray([
    record["uncertainty_exponent"]
    for record in phase52b_candidate_records
], dtype=np.float64)


def phase52b_group_support_counts(training_indices):
    training_indices = np.asarray(
        training_indices, dtype=np.int64
    ).reshape(-1)
    counts = {
        int(group): int(np.sum(
            phase52b_groups[training_indices] == group
        ))
        for group in np.unique(phase52b_groups[training_indices])
    }
    return counts


def phase52b_candidate_probability_matrix(
    evaluation_indices,
    training_group_counts,
):
    evaluation_indices = np.asarray(
        evaluation_indices, dtype=np.int64
    ).reshape(-1)
    support_count = np.asarray([
        float(training_group_counts.get(
            int(group), 0
        ))
        for group in phase52b_groups[evaluation_indices]
    ], dtype=np.float64)

    support_weight = np.divide(
        support_count[None, :],
        support_count[None, :]
        + phase52b_candidate_shrinkage[:, None],
        out=np.zeros(
            (phase52b_candidate_count, evaluation_indices.size),
            dtype=np.float64,
        ),
        where=(
            support_count[None, :]
            + phase52b_candidate_shrinkage[:, None]
        ) > 0.0,
    )
    uncertainty_weight = np.power(
        np.clip(
            phase52b_anchor_uncertainty[evaluation_indices],
            0.0,
            1.0,
        )[None, :],
        phase52b_candidate_gamma[:, None],
    )
    bounded_residual = np.clip(
        phase52b_raw_residual[evaluation_indices][None, :],
        -phase52b_candidate_cap[:, None],
        phase52b_candidate_cap[:, None],
    )
    update = (
        phase52b_candidate_scale[:, None]
        * support_weight
        * uncertainty_weight
        * bounded_residual
    )
    probability = phase52b_sigmoid(
        phase52b_anchor_logit[evaluation_indices][None, :]
        + update
    )

    assert probability.shape == (
        phase52b_candidate_count,
        evaluation_indices.size,
    )
    assert np.all(np.isfinite(probability))
    assert np.all((probability > 0.0) & (probability < 1.0))
    assert np.max(np.abs(
        probability[0]
        - phase52b_anchor_probability[evaluation_indices]
    )) <= 1.0e-12
    return probability


def phase52b_select_candidate(
    labels,
    groups,
    anchor_probability,
    candidate_probability,
    partition_ids,
    maximum_worst_partition_excess,
    maximum_major_group_harm,
):
    labels = np.asarray(labels, dtype=np.int64).reshape(-1)
    groups = np.asarray(groups, dtype=np.int64).reshape(-1)
    anchor_probability = np.asarray(
        anchor_probability, dtype=np.float64
    ).reshape(-1)
    candidate_probability = np.asarray(
        candidate_probability, dtype=np.float64
    )
    partition_ids = np.asarray(
        partition_ids, dtype=np.int64
    ).reshape(-1)

    assert labels.shape == groups.shape == anchor_probability.shape
    assert labels.shape == partition_ids.shape
    assert candidate_probability.shape == (
        phase52b_candidate_count,
        labels.size,
    )

    anchor_loss = phase52b_log_loss(labels, anchor_probability)
    candidate_loss = phase52b_log_loss_matrix(
        labels,
        candidate_probability,
    )

    partition_excesses = []
    for partition_id in sorted(np.unique(partition_ids).tolist()):
        mask = partition_ids == partition_id
        partition_anchor_loss = phase52b_log_loss(
            labels[mask],
            anchor_probability[mask],
        )
        partition_candidate_loss = phase52b_log_loss_matrix(
            labels[mask],
            candidate_probability[:, mask],
        )
        partition_excesses.append(
            partition_candidate_loss - partition_anchor_loss
        )
    partition_excesses = np.stack(partition_excesses, axis=1)
    worst_partition_excess = np.maximum(
        0.0,
        np.max(partition_excesses, axis=1),
    )
    partition_wins = np.sum(partition_excesses < 0.0, axis=1)

    major_group_harms = []
    major_group_count = 0
    for group in sorted(np.unique(groups).tolist()):
        mask = groups == group
        if int(np.sum(mask)) < int(
            PHASE52_NESTED_CONFIG["major_group_minimum_n"]
        ):
            continue
        major_group_count += 1
        group_anchor_loss = phase52b_log_loss(
            labels[mask],
            anchor_probability[mask],
        )
        group_candidate_loss = phase52b_log_loss_matrix(
            labels[mask],
            candidate_probability[:, mask],
        )
        major_group_harms.append(
            group_candidate_loss - group_anchor_loss
        )

    if major_group_harms:
        maximum_group_harm = np.maximum(
            0.0,
            np.max(
                np.stack(major_group_harms, axis=1),
                axis=1,
            ),
        )
    else:
        maximum_group_harm = np.zeros(
            phase52b_candidate_count,
            dtype=np.float64,
        )

    safe = (
        np.isfinite(candidate_loss)
        & (worst_partition_excess
           <= float(maximum_worst_partition_excess) + 1.0e-15)
        & (maximum_group_harm
           <= float(maximum_major_group_harm) + 1.0e-15)
    )
    assert bool(safe[0]), {
        "message": "The exact zero-update candidate was not safe.",
        "candidate_zero_loss": float(candidate_loss[0]),
        "anchor_loss": float(anchor_loss),
        "candidate_zero_worst_partition_excess": float(
            worst_partition_excess[0]
        ),
        "candidate_zero_maximum_group_harm": float(
            maximum_group_harm[0]
        ),
    }

    safe_indices = np.flatnonzero(safe)
    safe_minimum_loss = float(np.min(candidate_loss[safe_indices]))
    band_mask = (
        safe
        & (candidate_loss <= (
            safe_minimum_loss
            + float(PHASE52_NESTED_CONFIG[
                "selection_log_loss_band"
            ])
            + 1.0e-15
        ))
    )
    band_indices = np.flatnonzero(band_mask)
    assert band_indices.size > 0

    band_auroc = {
        int(index): phase52b_auroc(
            labels,
            candidate_probability[index],
        )
        for index in band_indices.tolist()
    }
    selected_index = min(
        band_indices.tolist(),
        key=lambda index: (
            -band_auroc[int(index)],
            float(candidate_loss[index]),
            int(index),
        ),
    )

    raw_best_index = int(np.argmin(candidate_loss))
    selected_auroc = float(band_auroc[int(selected_index)])
    anchor_auroc = phase52b_auroc(labels, anchor_probability)

    return {
        "selected_index": int(selected_index),
        "raw_best_index": int(raw_best_index),
        "safe_candidate_count": int(np.sum(safe)),
        "band_candidate_count": int(band_indices.size),
        "major_group_count": int(major_group_count),
        "anchor_log_loss": float(anchor_loss),
        "anchor_auroc": float(anchor_auroc),
        "selected_log_loss": float(candidate_loss[selected_index]),
        "selected_auroc": float(selected_auroc),
        "selected_log_loss_gain": float(
            anchor_loss - candidate_loss[selected_index]
        ),
        "selected_auroc_gain": float(
            selected_auroc - anchor_auroc
        ),
        "selected_partition_wins": int(
            partition_wins[selected_index]
        ),
        "selected_worst_partition_excess": float(
            worst_partition_excess[selected_index]
        ),
        "selected_maximum_major_group_harm": float(
            maximum_group_harm[selected_index]
        ),
        "raw_best_log_loss": float(candidate_loss[raw_best_index]),
        "raw_best_safe": bool(safe[raw_best_index]),
    }


# -------------------------------------------------------------------------
# Fully nested repeated cross-validation.
# -------------------------------------------------------------------------

phase52b_nested_probability_by_repeat = np.full(
    (phase52b_repeat_count, phase52b_case_count),
    np.nan,
    dtype=np.float64,
)
phase52b_outer_records = []

for repeat in range(phase52b_repeat_count):
    repeat_assignment = phase52b_fold_assignment[repeat]

    for outer_fold in range(phase52b_fold_count):
        outer_valid_indices = np.flatnonzero(
            repeat_assignment == outer_fold
        )
        outer_train_indices = np.flatnonzero(
            repeat_assignment != outer_fold
        )
        assert outer_valid_indices.size > 0
        assert outer_train_indices.size + outer_valid_indices.size \
            == phase52b_case_count

        inner_probability_parts = []
        inner_index_parts = []
        inner_partition_parts = []

        for inner_fold in range(phase52b_fold_count):
            if inner_fold == outer_fold:
                continue
            inner_valid_indices = np.flatnonzero(
                repeat_assignment == inner_fold
            )
            inner_train_indices = np.flatnonzero(
                (repeat_assignment != outer_fold)
                & (repeat_assignment != inner_fold)
            )
            assert inner_valid_indices.size > 0
            assert np.intersect1d(
                inner_train_indices,
                inner_valid_indices,
            ).size == 0
            assert np.intersect1d(
                inner_train_indices,
                outer_valid_indices,
            ).size == 0

            inner_counts = phase52b_group_support_counts(
                inner_train_indices
            )
            inner_probability_parts.append(
                phase52b_candidate_probability_matrix(
                    inner_valid_indices,
                    inner_counts,
                )
            )
            inner_index_parts.append(inner_valid_indices)
            inner_partition_parts.append(np.full(
                inner_valid_indices.size,
                inner_fold,
                dtype=np.int64,
            ))

        inner_indices = np.concatenate(inner_index_parts)
        inner_partition_ids = np.concatenate(inner_partition_parts)
        inner_probability = np.concatenate(
            inner_probability_parts,
            axis=1,
        )
        assert np.array_equal(
            np.sort(inner_indices),
            np.sort(outer_train_indices),
        )
        assert inner_probability.shape == (
            phase52b_candidate_count,
            outer_train_indices.size,
        )

        inner_selection = phase52b_select_candidate(
            labels=phase52b_labels[inner_indices],
            groups=phase52b_groups[inner_indices],
            anchor_probability=phase52b_anchor_probability[inner_indices],
            candidate_probability=inner_probability,
            partition_ids=inner_partition_ids,
            maximum_worst_partition_excess=PHASE52_NESTED_CONFIG[
                "inner_maximum_worst_fold_excess"
            ],
            maximum_major_group_harm=PHASE52_NESTED_CONFIG[
                "inner_maximum_major_group_harm"
            ],
        )
        selected_index = int(inner_selection["selected_index"])

        outer_counts = phase52b_group_support_counts(
            outer_train_indices
        )
        outer_probability_matrix = (
            phase52b_candidate_probability_matrix(
                outer_valid_indices,
                outer_counts,
            )
        )
        outer_probability = outer_probability_matrix[selected_index]
        phase52b_nested_probability_by_repeat[
            repeat,
            outer_valid_indices,
        ] = outer_probability

        outer_anchor_metrics = phase52b_metrics(
            phase52b_labels[outer_valid_indices],
            phase52b_anchor_probability[outer_valid_indices],
        )
        outer_candidate_metrics = phase52b_metrics(
            phase52b_labels[outer_valid_indices],
            outer_probability,
        )
        phase52b_outer_records.append({
            "repeat": int(repeat),
            "outer_fold": int(outer_fold),
            "outer_train_n": int(outer_train_indices.size),
            "outer_valid_n": int(outer_valid_indices.size),
            "selected_specification": copy.deepcopy(
                phase52b_candidate_records[selected_index]
            ),
            "inner": inner_selection,
            "outer": {
                "anchor_log_loss": float(
                    outer_anchor_metrics["log_loss"]
                ),
                "candidate_log_loss": float(
                    outer_candidate_metrics["log_loss"]
                ),
                "log_loss_gain": float(
                    outer_anchor_metrics["log_loss"]
                    - outer_candidate_metrics["log_loss"]
                ),
                "anchor_auroc": float(
                    outer_anchor_metrics["auroc"]
                ),
                "candidate_auroc": float(
                    outer_candidate_metrics["auroc"]
                ),
                "auroc_gain": float(
                    outer_candidate_metrics["auroc"]
                    - outer_anchor_metrics["auroc"]
                ),
            },
        })

        print(
            "Phase52 nested gate: "
            f"repeat={repeat + 1}/{phase52b_repeat_count}, "
            f"fold={outer_fold + 1}/{phase52b_fold_count}, "
            f"candidate={selected_index}, "
            f"outer_gain={outer_anchor_metrics['log_loss'] - outer_candidate_metrics['log_loss']:.6f}"
        )

assert np.all(np.isfinite(phase52b_nested_probability_by_repeat))
assert np.all((phase52b_nested_probability_by_repeat > 0.0)
              & (phase52b_nested_probability_by_repeat < 1.0))

phase52b_nested_probability = np.mean(
    phase52b_nested_probability_by_repeat,
    axis=0,
)
assert phase52b_nested_probability.shape == (phase52b_case_count,)
assert np.all(np.isfinite(phase52b_nested_probability))

phase52b_anchor_metrics = phase52b_metrics(
    phase52b_labels,
    phase52b_anchor_probability,
)
phase52b_nested_metrics = phase52b_metrics(
    phase52b_labels,
    phase52b_nested_probability,
)
phase52b_nested_log_loss_gain = float(
    phase52b_anchor_metrics["log_loss"]
    - phase52b_nested_metrics["log_loss"]
)
phase52b_nested_auroc_gain = float(
    phase52b_nested_metrics["auroc"]
    - phase52b_anchor_metrics["auroc"]
)

phase52b_repeat_records = []
for repeat in range(phase52b_repeat_count):
    anchor_metrics = phase52b_anchor_metrics
    candidate_metrics = phase52b_metrics(
        phase52b_labels,
        phase52b_nested_probability_by_repeat[repeat],
    )
    phase52b_repeat_records.append({
        "repeat": int(repeat),
        "anchor_log_loss": float(anchor_metrics["log_loss"]),
        "candidate_log_loss": float(candidate_metrics["log_loss"]),
        "log_loss_gain": float(
            anchor_metrics["log_loss"]
            - candidate_metrics["log_loss"]
        ),
        "anchor_auroc": float(anchor_metrics["auroc"]),
        "candidate_auroc": float(candidate_metrics["auroc"]),
        "auroc_gain": float(
            candidate_metrics["auroc"]
            - anchor_metrics["auroc"]
        ),
    })

phase52b_log_loss_repeat_wins = int(sum(
    record["log_loss_gain"] > 0.0
    for record in phase52b_repeat_records
))
phase52b_auroc_repeat_wins = int(sum(
    record["auroc_gain"] > 0.0
    for record in phase52b_repeat_records
))
phase52b_worst_repeat_log_loss_excess = float(max(
    0.0,
    max(
        -record["log_loss_gain"]
        for record in phase52b_repeat_records
    ),
))


# -------------------------------------------------------------------------
# Group robustness for the primary nested estimate.
# -------------------------------------------------------------------------

phase52b_major_group_records = []
for group in sorted(np.unique(phase52b_groups).tolist()):
    mask = phase52b_groups == group
    if int(np.sum(mask)) < int(
        PHASE52_NESTED_CONFIG["major_group_minimum_n"]
    ):
        continue
    anchor_metrics = phase52b_metrics(
        phase52b_labels[mask],
        phase52b_anchor_probability[mask],
    )
    candidate_metrics = phase52b_metrics(
        phase52b_labels[mask],
        phase52b_nested_probability[mask],
    )
    phase52b_major_group_records.append({
        "group": int(group),
        "n": int(np.sum(mask)),
        "anchor_log_loss": float(anchor_metrics["log_loss"]),
        "candidate_log_loss": float(candidate_metrics["log_loss"]),
        "log_loss_gain": float(
            anchor_metrics["log_loss"]
            - candidate_metrics["log_loss"]
        ),
        "anchor_auroc": float(anchor_metrics["auroc"]),
        "candidate_auroc": float(candidate_metrics["auroc"]),
        "auroc_gain": float(
            candidate_metrics["auroc"]
            - anchor_metrics["auroc"]
        ),
    })

phase52b_maximum_major_group_harm = float(max(
    0.0,
    max(
        -record["log_loss_gain"]
        for record in phase52b_major_group_records
    ),
))
phase52b_major_group_wins = int(sum(
    record["log_loss_gain"] > 0.0
    for record in phase52b_major_group_records
))


# -------------------------------------------------------------------------
# Group-and-label-stratified paired bootstrap. Each bootstrap retains every
# observed acquisition-group/label stratum and therefore does not turn the
# estimate into an IID-only bootstrap.
# -------------------------------------------------------------------------

phase52b_bootstrap_rng = np.random.default_rng(
    int(PHASE52_NESTED_CONFIG["bootstrap_seed"])
)
phase52b_bootstrap_strata = []
for group in sorted(np.unique(phase52b_groups).tolist()):
    for label in [0, 1]:
        indices = np.flatnonzero(
            (phase52b_groups == group)
            & (phase52b_labels == label)
        )
        if indices.size:
            phase52b_bootstrap_strata.append(indices)

phase52b_log_loss_bootstrap = np.empty(
    int(PHASE52_NESTED_CONFIG[
        "bootstrap_log_loss_replicates"
    ]),
    dtype=np.float64,
)
phase52b_auroc_bootstrap = np.empty(
    int(PHASE52_NESTED_CONFIG[
        "bootstrap_auroc_replicates"
    ]),
    dtype=np.float64,
)

for replicate in range(phase52b_log_loss_bootstrap.size):
    sampled_indices = np.concatenate([
        phase52b_bootstrap_rng.choice(
            indices,
            size=indices.size,
            replace=True,
        )
        for indices in phase52b_bootstrap_strata
    ])
    phase52b_log_loss_bootstrap[replicate] = (
        phase52b_log_loss(
            phase52b_labels[sampled_indices],
            phase52b_anchor_probability[sampled_indices],
        )
        - phase52b_log_loss(
            phase52b_labels[sampled_indices],
            phase52b_nested_probability[sampled_indices],
        )
    )
    if replicate < phase52b_auroc_bootstrap.size:
        phase52b_auroc_bootstrap[replicate] = (
            phase52b_auroc(
                phase52b_labels[sampled_indices],
                phase52b_nested_probability[sampled_indices],
            )
            - phase52b_auroc(
                phase52b_labels[sampled_indices],
                phase52b_anchor_probability[sampled_indices],
            )
        )

phase52b_bootstrap_report = {
    "log_loss": {
        "replicate_count": int(phase52b_log_loss_bootstrap.size),
        "mean_gain": float(np.mean(phase52b_log_loss_bootstrap)),
        "q05_gain": float(np.quantile(
            phase52b_log_loss_bootstrap, 0.05
        )),
        "probability_positive": float(np.mean(
            phase52b_log_loss_bootstrap > 0.0
        )),
    },
    "auroc": {
        "replicate_count": int(phase52b_auroc_bootstrap.size),
        "mean_gain": float(np.mean(phase52b_auroc_bootstrap)),
        "q05_gain": float(np.quantile(
            phase52b_auroc_bootstrap, 0.05
        )),
        "probability_positive": float(np.mean(
            phase52b_auroc_bootstrap > 0.0
        )),
    },
}


# -------------------------------------------------------------------------
# Honest nested gate. Two predeclared tracks are allowed:
#   1. balanced improvement in both log loss and AUROC;
#   2. material rank improvement with essentially neutral log loss.
# -------------------------------------------------------------------------

phase52b_common_robustness_gate = bool(
    phase52b_log_loss_repeat_wins
    >= int(PHASE52_NESTED_CONFIG["minimum_repeat_metric_wins"])
    and phase52b_auroc_repeat_wins
    >= int(PHASE52_NESTED_CONFIG["minimum_repeat_metric_wins"])
    and phase52b_worst_repeat_log_loss_excess
    <= float(PHASE52_NESTED_CONFIG[
        "maximum_worst_repeat_log_loss_excess"
    ])
    and phase52b_maximum_major_group_harm
    <= float(PHASE52_NESTED_CONFIG[
        "maximum_major_group_harm"
    ])
)

phase52b_balanced_track_passed = bool(
    phase52b_common_robustness_gate
    and phase52b_nested_log_loss_gain
    >= float(PHASE52_NESTED_CONFIG[
        "balanced_minimum_log_loss_gain"
    ])
    and phase52b_nested_auroc_gain
    >= float(PHASE52_NESTED_CONFIG[
        "balanced_minimum_auroc_gain"
    ])
    and phase52b_bootstrap_report["log_loss"][
        "probability_positive"
    ] >= float(PHASE52_NESTED_CONFIG[
        "balanced_minimum_log_loss_bootstrap_probability"
    ])
    and phase52b_bootstrap_report["auroc"][
        "probability_positive"
    ] >= float(PHASE52_NESTED_CONFIG[
        "balanced_minimum_auroc_bootstrap_probability"
    ])
)

phase52b_rank_track_passed = bool(
    phase52b_common_robustness_gate
    and phase52b_nested_auroc_gain
    >= float(PHASE52_NESTED_CONFIG[
        "rank_minimum_auroc_gain"
    ])
    and -phase52b_nested_log_loss_gain
    <= float(PHASE52_NESTED_CONFIG[
        "rank_maximum_log_loss_excess"
    ])
    and phase52b_bootstrap_report["auroc"][
        "probability_positive"
    ] >= float(PHASE52_NESTED_CONFIG[
        "rank_minimum_auroc_bootstrap_probability"
    ])
    and phase52b_bootstrap_report["auroc"]["q05_gain"]
    >= float(PHASE52_NESTED_CONFIG[
        "rank_minimum_auroc_bootstrap_q05"
    ])
    and phase52b_bootstrap_report["log_loss"]["q05_gain"]
    >= float(PHASE52_NESTED_CONFIG[
        "rank_minimum_log_loss_bootstrap_q05"
    ])
)

phase52b_nested_gate_passed = bool(
    phase52b_balanced_track_passed
    or phase52b_rank_track_passed
)


# -------------------------------------------------------------------------
# Select one shared portable specification after nested evaluation. This is
# the normal final-fit step. Its metrics are explicitly marked optimistic and
# never replace the primary nested estimate above.
# -------------------------------------------------------------------------

phase52b_shared_probability_sum = np.zeros(
    (phase52b_candidate_count, phase52b_case_count),
    dtype=np.float64,
)
phase52b_shared_repeat_loss = np.empty(
    (phase52b_candidate_count, phase52b_repeat_count),
    dtype=np.float64,
)

for repeat in range(phase52b_repeat_count):
    repeat_assignment = phase52b_fold_assignment[repeat]
    repeat_probability = np.full(
        (phase52b_candidate_count, phase52b_case_count),
        np.nan,
        dtype=np.float64,
    )
    for fold in range(phase52b_fold_count):
        valid_indices = np.flatnonzero(repeat_assignment == fold)
        train_indices = np.flatnonzero(repeat_assignment != fold)
        training_counts = phase52b_group_support_counts(train_indices)
        repeat_probability[:, valid_indices] = (
            phase52b_candidate_probability_matrix(
                valid_indices,
                training_counts,
            )
        )
    assert np.all(np.isfinite(repeat_probability))
    phase52b_shared_probability_sum += repeat_probability
    phase52b_shared_repeat_loss[:, repeat] = (
        phase52b_log_loss_matrix(
            phase52b_labels,
            repeat_probability,
        )
    )

phase52b_shared_probability = (
    phase52b_shared_probability_sum
    / float(phase52b_repeat_count)
)
phase52b_shared_candidate_loss = phase52b_log_loss_matrix(
    phase52b_labels,
    phase52b_shared_probability,
)
phase52b_shared_anchor_loss = float(
    phase52b_anchor_metrics["log_loss"]
)
phase52b_shared_repeat_excess = (
    phase52b_shared_repeat_loss - phase52b_shared_anchor_loss
)
phase52b_shared_worst_repeat_excess = np.maximum(
    0.0,
    np.max(
        phase52b_shared_repeat_excess,
        axis=1,
    ),
)

phase52b_shared_group_harms = []
for group in sorted(np.unique(phase52b_groups).tolist()):
    mask = phase52b_groups == group
    if int(np.sum(mask)) < int(
        PHASE52_NESTED_CONFIG["major_group_minimum_n"]
    ):
        continue
    group_anchor_loss = phase52b_log_loss(
        phase52b_labels[mask],
        phase52b_anchor_probability[mask],
    )
    group_candidate_loss = phase52b_log_loss_matrix(
        phase52b_labels[mask],
        phase52b_shared_probability[:, mask],
    )
    phase52b_shared_group_harms.append(
        group_candidate_loss - group_anchor_loss
    )
phase52b_shared_maximum_group_harm = np.maximum(
    0.0,
    np.max(
        np.stack(phase52b_shared_group_harms, axis=1),
        axis=1,
    ),
)

phase52b_shared_safe = (
    np.isfinite(phase52b_shared_candidate_loss)
    & (phase52b_shared_worst_repeat_excess
       <= float(PHASE52_NESTED_CONFIG[
           "shared_maximum_worst_repeat_excess"
       ]) + 1.0e-15)
    & (phase52b_shared_maximum_group_harm
       <= float(PHASE52_NESTED_CONFIG[
           "shared_maximum_major_group_harm"
       ]) + 1.0e-15)
)
assert bool(phase52b_shared_safe[0])

phase52b_shared_safe_indices = np.flatnonzero(phase52b_shared_safe)
phase52b_shared_minimum_loss = float(np.min(
    phase52b_shared_candidate_loss[phase52b_shared_safe_indices]
))
phase52b_shared_band_indices = np.flatnonzero(
    phase52b_shared_safe
    & (phase52b_shared_candidate_loss <= (
        phase52b_shared_minimum_loss
        + float(PHASE52_NESTED_CONFIG[
            "selection_log_loss_band"
        ])
        + 1.0e-15
    ))
)
assert phase52b_shared_band_indices.size > 0

phase52b_shared_band_auroc = {
    int(index): phase52b_auroc(
        phase52b_labels,
        phase52b_shared_probability[index],
    )
    for index in phase52b_shared_band_indices.tolist()
}
phase52b_shared_selected_index = min(
    phase52b_shared_band_indices.tolist(),
    key=lambda index: (
        -phase52b_shared_band_auroc[int(index)],
        float(phase52b_shared_candidate_loss[index]),
        int(index),
    ),
)

if not phase52b_nested_gate_passed:
    phase52b_portable_selected_index = 0
else:
    phase52b_portable_selected_index = int(
        phase52b_shared_selected_index
    )

phase52b_shared_selected_probability = (
    phase52b_shared_probability[phase52b_shared_selected_index]
)
phase52b_portable_probability_diagnostic = (
    phase52b_shared_probability[phase52b_portable_selected_index]
)


# Selection stability is descriptive only.
phase52b_selected_specification_counter = Counter(
    json.dumps(
        record["selected_specification"],
        sort_keys=True,
        separators=(",", ":"),
    )
    for record in phase52b_outer_records
)
phase52b_selection_stability = []
for serialized, count in phase52b_selected_specification_counter.most_common():
    phase52b_selection_stability.append({
        "specification": json.loads(serialized),
        "outer_fold_count": int(count),
    })


# -------------------------------------------------------------------------
# Persist private state. Only the small portable JSON is eligible for a later
# staged package; nested probabilities remain private notebook diagnostics.
# -------------------------------------------------------------------------

phase52b_full_group_counts = {
    str(int(group)): int(np.sum(phase52b_groups == group))
    for group in sorted(np.unique(phase52b_groups).tolist())
}
phase52b_portable_specification = copy.deepcopy(
    phase52b_candidate_records[phase52b_portable_selected_index]
)
phase52b_portable_payload = {
    "schema_version": "phase52_support_gate_portable_v1",
    "status": (
        "promoted"
        if phase52b_nested_gate_passed
        else "not_promoted_anchor_only"
    ),
    "source_contract_sha256": str(
        phase52b_state["contract_sha256"]
    ),
    "specification": phase52b_portable_specification,
    "formula": (
        "z52 = z_anchor + scale * n_group/(n_group+kappa) * "
        "uncertainty(z_anchor)^gamma * clip(z51-z_anchor,-cap,cap)"
    ),
    "known_group_training_counts": phase52b_full_group_counts,
    "unknown_protocol_policy": "anchor_only",
    "contains_labels": False,
    "contains_probabilities": False,
    "contains_case_rows": False,
    "contains_voxel_data": False,
}
phase52b_portable_sha256 = hashlib.sha256(
    json.dumps(
        phase52b_portable_payload,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()
phase52b_portable_payload["state_sha256"] = phase52b_portable_sha256

Path(PHASE52_NESTED_CONFIG["private_directory"]).mkdir(
    parents=True,
    exist_ok=True,
)
phase52b_atomic_json(
    PHASE52_NESTED_CONFIG["portable_state_file"],
    phase52b_portable_payload,
)
phase52b_atomic_npy(
    PHASE52_NESTED_CONFIG["nested_probability_file"],
    phase52b_nested_probability.astype(np.float64),
)


phase52b_shared_selected_metrics = phase52b_metrics(
    phase52b_labels,
    phase52b_shared_selected_probability,
)
phase52b_portable_diagnostic_metrics = phase52b_metrics(
    phase52b_labels,
    phase52b_portable_probability_diagnostic,
)

phase52b_report = {
    "phase": "phase52_fully_nested_support_aware_adapter_gate",
    "status": (
        "nested_gate_passed_portable_state_frozen"
        if phase52b_nested_gate_passed
        else "nested_gate_failed_anchor_only_frozen"
    ),
    "candidate_grid": {
        "candidate_count": int(phase52b_candidate_count),
        "zero_update_candidate_index": 0,
        "zero_update_candidate_always_safe": True,
        "selection_log_loss_band": float(
            PHASE52_NESTED_CONFIG["selection_log_loss_band"]
        ),
    },
    "validation": {
        "repeat_count": int(phase52b_repeat_count),
        "fold_count_per_repeat": int(phase52b_fold_count),
        "outer_fold_count": int(
            phase52b_repeat_count * phase52b_fold_count
        ),
        "inner_fold_count_per_outer_fold": int(
            phase52b_fold_count - 1
        ),
        "outer_validation_row_used_for_its_gate_selection": False,
        "group_support_recomputed_from_training_partition_only": True,
        "new_pristine_holdout_claimed": False,
    },
    "anchor": phase52b_anchor_metrics,
    "nested": {
        **phase52b_nested_metrics,
        "log_loss_gain": float(phase52b_nested_log_loss_gain),
        "auroc_gain": float(phase52b_nested_auroc_gain),
        "log_loss_repeat_wins": int(
            phase52b_log_loss_repeat_wins
        ),
        "auroc_repeat_wins": int(phase52b_auroc_repeat_wins),
        "worst_repeat_log_loss_excess": float(
            phase52b_worst_repeat_log_loss_excess
        ),
        "major_group_wins": int(phase52b_major_group_wins),
        "major_group_count": int(len(
            phase52b_major_group_records
        )),
        "maximum_major_group_harm": float(
            phase52b_maximum_major_group_harm
        ),
        "primary_estimate": True,
    },
    "repeat_metrics": phase52b_repeat_records,
    "major_group_metrics": phase52b_major_group_records,
    "bootstrap": phase52b_bootstrap_report,
    "gate": {
        "common_robustness_gate": bool(
            phase52b_common_robustness_gate
        ),
        "balanced_track_passed": bool(
            phase52b_balanced_track_passed
        ),
        "rank_track_passed": bool(phase52b_rank_track_passed),
        "nested_gate_passed": bool(phase52b_nested_gate_passed),
        "thresholds": {
            key: value
            for key, value in PHASE52_NESTED_CONFIG.items()
            if key not in {
                "private_directory",
                "portable_state_file",
                "nested_probability_file",
                "report_file",
                "bootstrap_seed",
            }
        },
    },
    "outer_selection_stability": phase52b_selection_stability,
    "shared_final_fit": {
        "selected_before_nested_gate_application": copy.deepcopy(
            phase52b_candidate_records[
                phase52b_shared_selected_index
            ]
        ),
        "safe_candidate_count": int(np.sum(
            phase52b_shared_safe
        )),
        "band_candidate_count": int(
            phase52b_shared_band_indices.size
        ),
        "metrics": phase52b_shared_selected_metrics,
        "selection_metrics_optimistic": True,
        "nested_metrics_remain_primary": True,
    },
    "portable_state": {
        "promoted": bool(phase52b_nested_gate_passed),
        "specification": phase52b_portable_specification,
        "diagnostic_metrics": phase52b_portable_diagnostic_metrics,
        "unknown_protocol_policy": "anchor_only",
        "file": Path(
            PHASE52_NESTED_CONFIG["portable_state_file"]
        ).name,
        "state_sha256": str(phase52b_portable_sha256),
    },
    "stretch_target": {
        "maximum_log_loss": 0.22,
        "minimum_auroc": 0.96,
        "reached_on_nested_estimate": bool(
            phase52b_nested_metrics["log_loss"] <= 0.22
            and phase52b_nested_metrics["auroc"] >= 0.96
        ),
        "used_for_candidate_selection": False,
    },
    "phase51_retired_component_modified": False,
    "phase51_retired_audit_used_as_a_new_holdout": False,
    "all_1362_labels_are_development_labels": True,
    "public_leaderboard_used": False,
    "training_voxel_cache_read": False,
    "training_nifti_files_read": False,
    "smoke_data_read": False,
    "test_data_read": False,
    "case_level_predictions_exported": False,
    "persistent_private_prediction_written": True,
    "elapsed_seconds": round(
        time.perf_counter() - phase52b_started,
        3,
    ),
}

phase52b_atomic_json(
    PHASE52_NESTED_CONFIG["report_file"],
    phase52b_report,
)

PHASE52_NESTED_GATE_REPORT_PRIVATE = copy.deepcopy(
    phase52b_report
)
PHASE52_NESTED_GATE_STATE_PRIVATE = {
    "nested_probability": phase52b_nested_probability.copy(),
    "nested_probability_by_repeat": (
        phase52b_nested_probability_by_repeat.copy()
    ),
    "shared_selected_probability": (
        phase52b_shared_selected_probability.copy()
    ),
    "portable_probability_diagnostic": (
        phase52b_portable_probability_diagnostic.copy()
    ),
    "portable_specification": copy.deepcopy(
        phase52b_portable_specification
    ),
    "nested_gate_passed": bool(phase52b_nested_gate_passed),
    "portable_state_sha256": str(phase52b_portable_sha256),
}

print("BEGIN SANITIZED_PHASE52_NESTED_GATE")
print(json.dumps(phase52b_report, indent=2))
print("END SANITIZED_PHASE52_NESTED_GATE")
print(PHASE52_NESTED_CONFIG["portable_state_file"])

Phase52 nested gate: repeat=1/5, fold=1/5, candidate=567, outer_gain=0.008040
Phase52 nested gate: repeat=1/5, fold=2/5, candidate=582, outer_gain=0.010598
Phase52 nested gate: repeat=1/5, fold=3/5, candidate=582, outer_gain=0.013269
Phase52 nested gate: repeat=1/5, fold=4/5, candidate=567, outer_gain=0.009893
Phase52 nested gate: repeat=1/5, fold=5/5, candidate=582, outer_gain=0.015491
Phase52 nested gate: repeat=2/5, fold=1/5, candidate=582, outer_gain=0.016559
Phase52 nested gate: repeat=2/5, fold=2/5, candidate=582, outer_gain=0.013026
Phase52 nested gate: repeat=2/5, fold=3/5, candidate=582, outer_gain=0.019112
Phase52 nested gate: repeat=2/5, fold=4/5, candidate=582, outer_gain=0.012145
Phase52 nested gate: repeat=2/5, fold=5/5, candidate=567, outer_gain=-0.003575
Phase52 nested gate: repeat=3/5, fold=1/5, candidate=582, outer_gain=0.012245
Phase52 nested gate: repeat=3/5, fold=2/5, candidate=582, outer_gain=0.006255
Phase52 nested gate: repeat=3/5, fold=3/5, candidate=567, outer

> Phase53

In [2]:
# Cell 153A — Phase53 

from __future__ import annotations

import copy
import json
import math
import os
import random
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


phase53a_started = time.perf_counter()

PHASE53_CONTRACT_CONFIG = {
    "seed": 530101,
    "cache_file": Path(
        "/kaggle/working/phase51_dinov3_penultimate_float16.npy"
    ),
    "case_count": 1362,
    "view_count": 6,
    "token_count": 201,
    "prefix_token_count": 5,
    "patch_grid": (14, 14),
    "embedding_dimension": 384,
    "freqfit_block_count": 1,
    "dora_rank": 4,
    "dora_alpha": 4.0,
    "view_projection_dimension": 96,
    "head_hidden_dimension": 128,
    "residual_cap": 0.5,
    "contract_batch_size": 8,
    "contract_optimizer_steps": 3,
    "learning_rates": {
        "freqfit": 1.0e-4,
        "dora": 1.0e-5,
        "pooling": 1.0e-4,
        "head": 2.0e-4,
    },
}


def phase53_seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


phase53_seed_everything(PHASE53_CONTRACT_CONFIG["seed"])
phase53_device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)
phase53_cuda = phase53_device.type == "cuda"


def phase53_as_backbone(value: Any, depth: int = 0) -> Optional[nn.Module]:
    """Resolve a DINOv3 ViT without scanning or mutating globals()."""
    if value is None or depth > 3:
        return None
    if (
        isinstance(value, nn.Module)
        and hasattr(value, "blocks")
        and hasattr(value, "norm")
        and hasattr(value, "rope_embed")
    ):
        try:
            if len(value.blocks) == 12:
                return value
        except TypeError:
            pass

    child_names = (
        "backbone",
        "dinov3_backbone",
        "model",
        "encoder",
        "network",
    )
    if isinstance(value, dict):
        for child_name in child_names:
            if child_name in value:
                result = phase53_as_backbone(
                    value[child_name], depth=depth + 1
                )
                if result is not None:
                    return result
    else:
        for child_name in child_names:
            if hasattr(value, child_name):
                result = phase53_as_backbone(
                    getattr(value, child_name), depth=depth + 1
                )
                if result is not None:
                    return result
    return None


phase53_backbone_candidates = (
    "PHASE51_OFFLINE_RELOAD",
    "PHASE51_DINOV3_BACKBONE_PRIVATE",
    "PHASE39_BACKBONE_PRIVATE",
    "PHASE32_DINOV3_BACKBONE",
    "PHASE30_DINOV3_BACKBONE",
)
phase53_backbone = None
phase53_backbone_source = None
for phase53_candidate_name in phase53_backbone_candidates:
    phase53_candidate_value = globals().get(phase53_candidate_name)
    phase53_candidate_backbone = phase53_as_backbone(
        phase53_candidate_value
    )
    if phase53_candidate_backbone is not None:
        phase53_backbone = phase53_candidate_backbone
        phase53_backbone_source = phase53_candidate_name
        break

assert phase53_backbone is not None, {
    "message": (
        "No initialized accepted DINOv3 backbone was found. Re-run the "
        "accepted Phase51 DINO offline-reload contract cell, then rerun "
        "this cell."
    ),
    "checked_names": list(phase53_backbone_candidates),
}
assert len(phase53_backbone.blocks) == 12
assert int(getattr(phase53_backbone, "embed_dim", 384)) == 384
assert int(getattr(phase53_backbone, "n_storage_tokens", 4)) == 4


phase53_cache_path = PHASE53_CONTRACT_CONFIG["cache_file"]
assert phase53_cache_path.is_file(), {
    "message": "The accepted Phase51 penultimate-token cache is missing.",
    "expected": str(phase53_cache_path),
}
phase53_penultimate_cache = np.load(
    phase53_cache_path, mmap_mode="r"
)
assert phase53_penultimate_cache.shape == (
    PHASE53_CONTRACT_CONFIG["case_count"],
    PHASE53_CONTRACT_CONFIG["view_count"],
    PHASE53_CONTRACT_CONFIG["token_count"],
    PHASE53_CONTRACT_CONFIG["embedding_dimension"],
)
assert phase53_penultimate_cache.dtype == np.float16


assert "PHASE52_SUPPORT_GATE_STATE_PRIVATE" in globals(), {
    "message": (
        "Phase52 state is missing. Re-run accepted Cell 152A before "
        "Phase53."
    )
}
phase53_validation_state = PHASE52_SUPPORT_GATE_STATE_PRIVATE
phase53_labels = np.asarray(
    phase53_validation_state["labels"], dtype=np.int64
).reshape(-1)
phase53_groups = np.asarray(
    phase53_validation_state["groups"], dtype=np.int64
).reshape(-1)
phase53_anchor_logit = np.asarray(
    phase53_validation_state["anchor_logit"], dtype=np.float64
).reshape(-1)
phase53_fold_assignment = np.asarray(
    phase53_validation_state["fold_assignment"], dtype=np.int64
)

assert phase53_labels.shape == (1362,)
assert phase53_groups.shape == (1362,)
assert phase53_anchor_logit.shape == (1362,)
assert phase53_fold_assignment.shape == (5, 1362)
assert set(np.unique(phase53_labels).tolist()) == {0, 1}
assert np.unique(phase53_groups).size == 15
assert np.all(np.isfinite(phase53_anchor_logit))


def phase53_balanced_contract_indices(
    labels: np.ndarray,
    groups: np.ndarray,
    eligible: np.ndarray,
    count: int,
) -> np.ndarray:
    """Deterministic fit-only sample with both labels and several groups."""
    eligible = np.asarray(eligible, dtype=np.int64)
    chosen: List[int] = []
    for group in np.unique(groups[eligible]):
        local = eligible[groups[eligible] == group]
        for label in (0, 1):
            hits = local[labels[local] == label]
            if hits.size:
                chosen.append(int(hits[0]))
                if len(chosen) == count:
                    return np.asarray(chosen, dtype=np.int64)
    for index in eligible:
        if int(index) not in chosen:
            chosen.append(int(index))
            if len(chosen) == count:
                break
    result = np.asarray(chosen, dtype=np.int64)
    assert result.size == count
    assert np.unique(labels[result]).size == 2
    return result


# Contract data come only from repeat-0/fold-0's fitting partition.
phase53_contract_fit_indices = np.flatnonzero(
    phase53_fold_assignment[0] != 0
)
phase53_contract_indices = phase53_balanced_contract_indices(
    labels=phase53_labels,
    groups=phase53_groups,
    eligible=phase53_contract_fit_indices,
    count=PHASE53_CONTRACT_CONFIG["contract_batch_size"],
)
assert np.all(phase53_fold_assignment[0, phase53_contract_indices] != 0)


class Phase53FreqFiT2D(nn.Module):
    """Published FreqFiT GlobalFilter2D specialized to one 14x14 layer.

    The implementation follows the official MICCAI 2025 code path:
    rFFT2 -> learned complex elementwise filter -> irFFT2 -> SSF -> skip.
    FFT and complex multiplication are always float32/complex64 even when the
    surrounding DINO block runs under BF16 autocast.
    """

    def __init__(
        self,
        dim: int = 384,
        height: int = 14,
        width_rfft: int = 8,
    ) -> None:
        super().__init__()
        self.dim = int(dim)
        self.height = int(height)
        self.width_rfft = int(width_rfft)
        self.complex_weight = nn.Parameter(
            torch.randn(
                self.height,
                self.width_rfft,
                self.dim,
                2,
                dtype=torch.float32,
            )
            * 0.02
        )
        self.ssf_scale = nn.Parameter(
            torch.ones(self.dim, dtype=torch.float32)
        )
        self.ssf_shift = nn.Parameter(
            torch.zeros(self.dim, dtype=torch.float32)
        )
        nn.init.normal_(self.ssf_scale, mean=1.0, std=0.02)
        nn.init.normal_(self.ssf_shift, mean=0.0, std=0.02)

    def forward(
        self,
        patch_tokens: torch.Tensor,
        spatial_size: Tuple[int, int] = (14, 14),
    ) -> torch.Tensor:
        if patch_tokens.ndim != 3:
            raise ValueError(
                f"Expected [B,N,C], received {tuple(patch_tokens.shape)}"
            )
        batch, token_count, channels = patch_tokens.shape
        height, width = map(int, spatial_size)
        if token_count != height * width or channels != self.dim:
            raise ValueError(
                "FreqFiT token contract mismatch: "
                f"shape={tuple(patch_tokens.shape)}, spatial={spatial_size}"
            )
        if height != self.height or width // 2 + 1 != self.width_rfft:
            raise ValueError("FreqFiT frequency-grid contract mismatch.")

        input_dtype = patch_tokens.dtype
        input_device_type = patch_tokens.device.type
        # Explicitly disable autocast: CUDA FFT does not accept BF16 and the
        # official FreqFiT implementation performs the transform in FP32.
        with torch.autocast(
            device_type=input_device_type,
            enabled=False,
        ):
            spatial = patch_tokens.float().reshape(
                batch, height, width, channels
            )
            residual = spatial
            spectrum = torch.fft.rfft2(
                spatial,
                dim=(1, 2),
                norm="ortho",
            )
            complex_filter = torch.view_as_complex(
                self.complex_weight.contiguous()
            )
            if spectrum.shape[1:] != complex_filter.shape:
                raise RuntimeError(
                    "FreqFiT spectrum/filter mismatch: "
                    f"{tuple(spectrum.shape[1:])} vs "
                    f"{tuple(complex_filter.shape)}"
                )
            filtered = torch.fft.irfft2(
                spectrum * complex_filter,
                s=(height, width),
                dim=(1, 2),
                norm="ortho",
            )
            filtered = (
                filtered * self.ssf_scale.view(1, 1, 1, channels)
                + self.ssf_shift.view(1, 1, 1, channels)
            )
            output = (filtered + residual).reshape(
                batch, token_count, channels
            )
        return output.to(dtype=input_dtype)


class Phase53DoRALinear(nn.Module):
    """Shape-compatible DoRA replacement for torch.nn.Linear.

    Weight magnitude is learned per output channel. Direction is the frozen
    pretrained weight plus a low-rank update. B starts at zero, so the exact
    pretrained linear map is recovered at initialization (up to FP rounding).
    """

    def __init__(
        self,
        base_linear: nn.Linear,
        rank: int,
        alpha: float,
    ) -> None:
        super().__init__()
        if not isinstance(base_linear, nn.Linear):
            raise TypeError(type(base_linear))
        if rank <= 0:
            raise ValueError("DoRA rank must be positive.")

        self.in_features = int(base_linear.in_features)
        self.out_features = int(base_linear.out_features)
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / float(self.rank)

        self.weight = nn.Parameter(
            base_linear.weight.detach().clone(), requires_grad=False
        )
        if base_linear.bias is None:
            self.register_parameter("bias", None)
        else:
            self.bias = nn.Parameter(
                base_linear.bias.detach().clone(), requires_grad=False
            )

        initial_magnitude = torch.linalg.vector_norm(
            self.weight.detach().float(), dim=1
        )
        self.magnitude = nn.Parameter(initial_magnitude)
        self.lora_a = nn.Parameter(
            torch.empty(self.rank, self.in_features)
        )
        self.lora_b = nn.Parameter(
            torch.zeros(self.out_features, self.rank)
        )
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5.0))

    def effective_weight(self) -> torch.Tensor:
        update = torch.matmul(self.lora_b, self.lora_a) * self.scaling
        direction = self.weight.float() + update.float()
        direction_norm = torch.linalg.vector_norm(
            direction, dim=1, keepdim=True
        ).clamp_min(1.0e-8)
        return direction * (
            self.magnitude.float().unsqueeze(1) / direction_norm
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        effective_weight = self.effective_weight().to(
            dtype=inputs.dtype, device=inputs.device
        )
        bias = self.bias
        if bias is not None:
            bias = bias.to(dtype=inputs.dtype, device=inputs.device)
        return F.linear(inputs, effective_weight, bias)


def phase53_get_child(module: nn.Module, path: str) -> nn.Module:
    value: nn.Module = module
    for part in path.split("."):
        value = getattr(value, part)
    return value


def phase53_set_child(
    module: nn.Module,
    path: str,
    replacement: nn.Module,
) -> None:
    parts = path.split(".")
    parent = module
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], replacement)


PHASE53_DORA_TARGETS = (
    "attn.qkv",
    "attn.proj",
    "mlp.fc1",
    "mlp.fc2",
)


class Phase53FreqDoRAResidualModel(nn.Module):
    def __init__(
        self,
        final_block: nn.Module,
        final_norm: nn.Module,
        dora_rank: int = 4,
        dora_alpha: float = 4.0,
        view_projection_dimension: int = 96,
        head_hidden_dimension: int = 128,
        residual_cap: float = 0.5,
    ) -> None:
        super().__init__()
        self.embedding_dimension = 384
        self.prefix_token_count = 5
        self.patch_grid = (14, 14)
        self.residual_cap = float(residual_cap)

        self.freqfit = Phase53FreqFiT2D(
            dim=self.embedding_dimension,
            height=self.patch_grid[0],
            width_rfft=self.patch_grid[1] // 2 + 1,
        )
        self.final_block = copy.deepcopy(final_block)
        self.final_norm = copy.deepcopy(final_norm)
        self.final_block.requires_grad_(False)
        self.final_norm.requires_grad_(False)

        self.dora_module_names: List[str] = []
        for path in PHASE53_DORA_TARGETS:
            original = phase53_get_child(self.final_block, path)
            if not isinstance(original, nn.Linear):
                raise TypeError(
                    f"Expected nn.Linear at {path}; received {type(original)}"
                )
            replacement = Phase53DoRALinear(
                original,
                rank=dora_rank,
                alpha=dora_alpha,
            )
            phase53_set_child(self.final_block, path, replacement)
            self.dora_module_names.append(path)

        # 3 pools x 384 channels for each of six views.
        per_view_dimension = 3 * self.embedding_dimension
        self.view_projection = nn.Sequential(
            nn.LayerNorm(per_view_dimension),
            nn.Linear(
                per_view_dimension,
                view_projection_dimension,
            ),
            nn.GELU(),
        )
        # Ordered six-view projections plus their mean and standard deviation.
        pooled_dimension = view_projection_dimension * (6 + 2)
        self.head = nn.Sequential(
            nn.LayerNorm(pooled_dimension),
            nn.Linear(pooled_dimension, head_hidden_dimension),
            nn.GELU(),
            nn.Dropout(p=0.10),
            nn.Linear(head_hidden_dimension, 1),
        )
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

        central = torch.zeros(14, 14, dtype=torch.bool)
        central[3:11, 3:11] = True
        self.register_buffer(
            "central_prompt_mask",
            central.reshape(-1),
            persistent=True,
        )

    def train(self, mode: bool = True):
        super().train(mode)
        # The pretrained DINO block has no trainable base parameters and must
        # retain inference behavior. Gradients still flow through DoRA tensors.
        self.final_block.eval()
        self.final_norm.eval()
        return self

    def forward(
        self,
        penultimate_tokens: torch.Tensor,
        anchor_logit: torch.Tensor,
        rope_sincos: Any,
    ) -> Dict[str, torch.Tensor]:
        expected = (
            penultimate_tokens.shape[0],
            6,
            201,
            384,
        )
        if tuple(penultimate_tokens.shape) != expected:
            raise ValueError(
                f"Token shape mismatch: {tuple(penultimate_tokens.shape)}"
            )
        if tuple(anchor_logit.shape) != (penultimate_tokens.shape[0],):
            raise ValueError(
                f"Anchor shape mismatch: {tuple(anchor_logit.shape)}"
            )

        batch = penultimate_tokens.shape[0]
        flat = penultimate_tokens.reshape(-1, 201, 384)
        prefix = flat[:, : self.prefix_token_count]
        patches = flat[:, self.prefix_token_count :]
        patches = self.freqfit(patches, self.patch_grid)
        adapted_input = torch.cat((prefix, patches), dim=1)

        final_tokens = self.final_block(adapted_input, rope_sincos)
        final_tokens = self.final_norm(final_tokens)
        patch_tokens = final_tokens[:, self.prefix_token_count :]
        patch_tokens = patch_tokens.reshape(batch, 6, 196, 384)

        global_pool = patch_tokens.mean(dim=2)
        prompt_pool = patch_tokens[
            :, :, self.central_prompt_mask, :
        ].mean(dim=2)
        contrast_pool = prompt_pool - global_pool
        per_view = torch.cat(
            (prompt_pool, global_pool, contrast_pool), dim=-1
        )
        view_projection = self.view_projection(per_view)
        view_mean = view_projection.mean(dim=1)
        view_std = view_projection.std(dim=1, unbiased=False)
        representation = torch.cat(
            (
                view_projection.reshape(batch, -1),
                view_mean,
                view_std,
            ),
            dim=-1,
        )

        raw_residual = self.head(representation).squeeze(-1)
        bounded_residual = self.residual_cap * torch.tanh(
            raw_residual / self.residual_cap
        )
        logit = anchor_logit + bounded_residual
        probability = torch.sigmoid(logit)
        return {
            "logit": logit,
            "probability": probability,
            "raw_residual": raw_residual,
            "bounded_residual": bounded_residual,
            "representation": representation,
            "view_projection": view_projection,
            "frequency_patch_tokens": patches,
        }


phase53_base_final_block = phase53_backbone.blocks[-1]
phase53_base_final_norm = phase53_backbone.norm
phase53_model = Phase53FreqDoRAResidualModel(
    final_block=phase53_base_final_block,
    final_norm=phase53_base_final_norm,
    dora_rank=PHASE53_CONTRACT_CONFIG["dora_rank"],
    dora_alpha=PHASE53_CONTRACT_CONFIG["dora_alpha"],
    view_projection_dimension=(
        PHASE53_CONTRACT_CONFIG["view_projection_dimension"]
    ),
    head_hidden_dimension=(
        PHASE53_CONTRACT_CONFIG["head_hidden_dimension"]
    ),
    residual_cap=PHASE53_CONTRACT_CONFIG["residual_cap"],
).to(phase53_device)

assert phase53_model.dora_module_names == list(PHASE53_DORA_TARGETS)
for phase53_path in PHASE53_DORA_TARGETS:
    phase53_module = phase53_get_child(
        phase53_model.final_block, phase53_path
    )
    assert isinstance(phase53_module, Phase53DoRALinear)
    assert hasattr(phase53_module, "in_features")
    assert hasattr(phase53_module, "out_features")
    assert hasattr(phase53_module, "weight")
    assert hasattr(phase53_module, "bias")


def phase53_to_device_nested(value: Any, device: torch.device) -> Any:
    """Move DINOv3 RoPE tensors without altering container semantics."""
    if value is None:
        return None
    if torch.is_tensor(value):
        return value.to(device)
    if isinstance(value, tuple):
        return tuple(
            phase53_to_device_nested(item, device) for item in value
        )
    if isinstance(value, list):
        return [
            phase53_to_device_nested(item, device) for item in value
        ]
    if isinstance(value, dict):
        return {
            key: phase53_to_device_nested(item, device)
            for key, item in value.items()
        }
    raise TypeError(
        "Unsupported DINOv3 RoPE container: "
        f"{type(value).__name__}"
    )


if phase53_backbone.rope_embed is None:
    phase53_rope_sincos = None
else:
    with torch.no_grad():
        phase53_rope_sincos = phase53_to_device_nested(
            phase53_backbone.rope_embed(H=14, W=14),
            phase53_device,
        )


phase53_contract_tokens = torch.from_numpy(
    np.array(
        phase53_penultimate_cache[phase53_contract_indices],
        dtype=np.float32,
        copy=True,
    )
).to(phase53_device)
phase53_contract_anchor = torch.from_numpy(
    phase53_anchor_logit[phase53_contract_indices].astype(
        np.float32, copy=True
    )
).to(phase53_device)
phase53_contract_labels = torch.from_numpy(
    phase53_labels[phase53_contract_indices].astype(
        np.float32, copy=True
    )
).to(phase53_device)


phase53_model.eval()
with torch.no_grad(), torch.autocast(
    device_type=phase53_device.type,
    dtype=torch.bfloat16 if phase53_cuda else torch.float32,
    enabled=phase53_cuda,
):
    phase53_initial_output = phase53_model(
        phase53_contract_tokens,
        phase53_contract_anchor,
        phase53_rope_sincos,
    )

phase53_initial_logit_error = float(
    torch.max(
        torch.abs(
            phase53_initial_output["logit"].float()
            - phase53_contract_anchor.float()
        )
    ).item()
)
phase53_initial_probability_error = float(
    torch.max(
        torch.abs(
            phase53_initial_output["probability"].float()
            - torch.sigmoid(phase53_contract_anchor.float())
        )
    ).item()
)
phase53_initial_residual_maximum = float(
    torch.max(
        torch.abs(
            phase53_initial_output["bounded_residual"].float()
        )
    ).item()
)
assert phase53_initial_logit_error <= 1.0e-7
assert phase53_initial_probability_error <= 1.0e-7
assert phase53_initial_residual_maximum == 0.0
assert torch.all(
    torch.isfinite(phase53_initial_output["probability"])
)


phase53_dora_initial_weight_errors: Dict[str, float] = {}
for phase53_path in PHASE53_DORA_TARGETS:
    phase53_module = phase53_get_child(
        phase53_model.final_block, phase53_path
    )
    phase53_error = float(
        torch.max(
            torch.abs(
                phase53_module.effective_weight().detach().float()
                - phase53_module.weight.detach().float()
            )
        ).item()
    )
    phase53_dora_initial_weight_errors[phase53_path] = phase53_error
assert max(phase53_dora_initial_weight_errors.values()) <= 1.0e-6


def phase53_gradient_summary(
    named_parameters: Iterable[Tuple[str, nn.Parameter]],
    predicate,
) -> Dict[str, Any]:
    values = []
    for name, parameter in named_parameters:
        if predicate(name, parameter) and parameter.grad is not None:
            values.append(parameter.grad.detach().float())
    if not values:
        return {"tensor_count": 0, "norm": 0.0, "all_finite": True}
    squared_norm = sum(
        float(torch.sum(value * value).item()) for value in values
    )
    return {
        "tensor_count": len(values),
        "norm": float(math.sqrt(max(squared_norm, 0.0))),
        "all_finite": bool(
            all(torch.all(torch.isfinite(value)).item() for value in values)
        ),
    }


phase53_freqfit_parameters = list(phase53_model.freqfit.parameters())
phase53_dora_parameters = []
for phase53_path in PHASE53_DORA_TARGETS:
    phase53_dora_parameters.extend(
        list(
            phase53_get_child(
                phase53_model.final_block, phase53_path
            ).parameters()
        )
    )
phase53_dora_parameters = [
    parameter
    for parameter in phase53_dora_parameters
    if parameter.requires_grad
]
phase53_pooling_parameters = list(
    phase53_model.view_projection.parameters()
)
phase53_head_parameters = list(phase53_model.head.parameters())

phase53_optimizer = torch.optim.AdamW(
    [
        {
            "params": phase53_freqfit_parameters,
            "lr": PHASE53_CONTRACT_CONFIG["learning_rates"]["freqfit"],
        },
        {
            "params": phase53_dora_parameters,
            "lr": PHASE53_CONTRACT_CONFIG["learning_rates"]["dora"],
        },
        {
            "params": phase53_pooling_parameters,
            "lr": PHASE53_CONTRACT_CONFIG["learning_rates"]["pooling"],
        },
        {
            "params": phase53_head_parameters,
            "lr": PHASE53_CONTRACT_CONFIG["learning_rates"]["head"],
        },
    ],
    betas=(0.9, 0.95),
    weight_decay=0.02,
)


phase53_gradient_records = []
phase53_model.train()
for phase53_step in range(1, 4):
    phase53_optimizer.zero_grad(set_to_none=True)
    with torch.autocast(
        device_type=phase53_device.type,
        dtype=torch.bfloat16 if phase53_cuda else torch.float32,
        enabled=phase53_cuda,
    ):
        phase53_output = phase53_model(
            phase53_contract_tokens,
            phase53_contract_anchor,
            phase53_rope_sincos,
        )
        phase53_loss = F.binary_cross_entropy_with_logits(
            phase53_output["logit"].float(),
            phase53_contract_labels.float(),
        )
    phase53_loss.backward()

    phase53_named_parameters = tuple(
        phase53_model.named_parameters()
    )
    phase53_record = {
        "step": phase53_step,
        "loss": float(phase53_loss.detach().item()),
        "freqfit_gradient": phase53_gradient_summary(
            phase53_named_parameters,
            lambda name, parameter: name.startswith("freqfit."),
        ),
        "dora_a_gradient": phase53_gradient_summary(
            phase53_named_parameters,
            lambda name, parameter: name.endswith("lora_a"),
        ),
        "dora_b_gradient": phase53_gradient_summary(
            phase53_named_parameters,
            lambda name, parameter: name.endswith("lora_b"),
        ),
        "dora_magnitude_gradient": phase53_gradient_summary(
            phase53_named_parameters,
            lambda name, parameter: name.endswith("magnitude"),
        ),
        "pooling_gradient": phase53_gradient_summary(
            phase53_named_parameters,
            lambda name, parameter: name.startswith(
                "view_projection."
            ),
        ),
        "head_gradient": phase53_gradient_summary(
            phase53_named_parameters,
            lambda name, parameter: name.startswith("head."),
        ),
        "maximum_absolute_residual": float(
            torch.max(
                torch.abs(phase53_output["bounded_residual"].float())
            ).detach().item()
        ),
    }
    phase53_gradient_records.append(phase53_record)

    all_trainable_gradients = [
        parameter.grad
        for parameter in phase53_model.parameters()
        if parameter.requires_grad and parameter.grad is not None
    ]
    assert all_trainable_gradients
    assert all(
        torch.all(torch.isfinite(gradient)).item()
        for gradient in all_trainable_gradients
    )
    torch.nn.utils.clip_grad_norm_(
        [
            parameter
            for parameter in phase53_model.parameters()
            if parameter.requires_grad
        ],
        max_norm=1.0,
    )
    phase53_optimizer.step()


# Zero-initialized residual head intentionally delays upstream gradients.
assert phase53_gradient_records[0]["head_gradient"]["norm"] > 0.0
assert phase53_gradient_records[1]["freqfit_gradient"]["norm"] > 0.0
assert phase53_gradient_records[1]["dora_b_gradient"]["norm"] > 0.0
assert phase53_gradient_records[1]["dora_magnitude_gradient"]["norm"] > 0.0
assert phase53_gradient_records[1]["pooling_gradient"]["norm"] > 0.0
assert phase53_gradient_records[2]["dora_a_gradient"]["norm"] > 0.0


phase53_total_parameter_count = sum(
    parameter.numel() for parameter in phase53_model.parameters()
)
phase53_trainable_parameter_count = sum(
    parameter.numel()
    for parameter in phase53_model.parameters()
    if parameter.requires_grad
)
phase53_freqfit_parameter_count = sum(
    parameter.numel() for parameter in phase53_model.freqfit.parameters()
)
phase53_dora_parameter_count = sum(
    parameter.numel() for parameter in phase53_dora_parameters
)

phase53_report = {
    "phase": "phase53_cached_token_freqfit_dora_contract",
    "status": "accepted_ready_for_frequency_adapter_screen",
    "phase52_decision": {
        "nested_log_loss_gain": 0.011452189652529088,
        "nested_auroc_gain": 0.004196732730379571,
        "repeat_log_loss_wins": 5,
        "repeat_auroc_wins": 5,
        "maximum_major_group_harm": 0.027626981310280574,
        "predeclared_harm_limit": 0.015,
        "promoted": False,
        "interpretation": (
            "signal_is_real_but_not_protocol_safe; do_not_relax_the_gate"
        ),
    },
    "research_basis": {
        "freqfit": (
            "MICCAI_2025_official_GlobalFilter2D_equations_and_code"
        ),
        "dora": "ICML_2024_weight_magnitude_direction_decomposition",
        "loft": (
            "ICLR_2026_optimizer_arm_deferred_until_standard_DoRA_"
            "architecture_screen_establishes_nonzero_signal"
        ),
        "qlora": (
            "excluded_because_21p6M_backbone_and_1p3GB_peak_do_not_"
            "justify_quantization_error"
        ),
    },
    "backbone": {
        "resolved_global": phase53_backbone_source,
        "block_count": len(phase53_backbone.blocks),
        "adapted_block": 11,
        "embedding_dimension": 384,
        "prefix_token_count": 5,
        "patch_token_count": 196,
        "rope_position_encoding_used": phase53_rope_sincos is not None,
    },
    "cache": {
        "file": phase53_cache_path.name,
        "shape": list(phase53_penultimate_cache.shape),
        "dtype": str(phase53_penultimate_cache.dtype),
        "semantics": "output_after_blocks_0_through_10_before_block_11",
        "new_cache_created": False,
    },
    "freqfit": {
        "insertion": "between_dinov3_blocks_10_and_11",
        "patch_grid": [14, 14],
        "rfft_grid": [14, 8],
        "operation_order": [
            "reshape_patch_tokens_to_14_by_14",
            "float32_rfft2_ortho",
            "learned_complex_elementwise_filter",
            "float32_irfft2_ortho",
            "scale_shift",
            "residual_add",
        ],
        "parameter_count": phase53_freqfit_parameter_count,
        "fft_float32_under_amp": True,
    },
    "dora": {
        "rank": PHASE53_CONTRACT_CONFIG["dora_rank"],
        "alpha": PHASE53_CONTRACT_CONFIG["dora_alpha"],
        "target_module_count": len(PHASE53_DORA_TARGETS),
        "target_modules": list(PHASE53_DORA_TARGETS),
        "trainable_parameter_count": phase53_dora_parameter_count,
        "linear_interface_preserved": True,
        "initial_effective_weight_maximum_errors": (
            phase53_dora_initial_weight_errors
        ),
    },
    "prediction_head": {
        "pooling": (
            "central_prompt_global_contrast_per_view_plus_ordered_"
            "six_view_moments"
        ),
        "representation_dimension": int(
            phase53_initial_output["representation"].shape[1]
        ),
        "residual_cap": PHASE53_CONTRACT_CONFIG["residual_cap"],
        "anchor": "phase43_deployment_matched_logit",
    },
    "parameters": {
        "total": phase53_total_parameter_count,
        "trainable": phase53_trainable_parameter_count,
        "frozen": (
            phase53_total_parameter_count
            - phase53_trainable_parameter_count
        ),
    },
    "initial_identity": {
        "maximum_logit_error": phase53_initial_logit_error,
        "maximum_probability_error": phase53_initial_probability_error,
        "maximum_absolute_residual": phase53_initial_residual_maximum,
        "exact": phase53_initial_residual_maximum == 0.0,
    },
    "three_step_backward_contract": {
        "records": phase53_gradient_records,
        "head_receives_step_1_gradient": True,
        "freqfit_receives_step_2_gradient": True,
        "dora_b_receives_step_2_gradient": True,
        "dora_magnitude_receives_step_2_gradient": True,
        "dora_a_receives_step_3_gradient": True,
        "all_gradients_finite": True,
    },
    "screen_plan": {
        "architecture_arms": [
            "freqfit_plus_frozen_final_block",
            "freqfit_plus_dora_rank_4",
            "freqfit_plus_dora_rank_8",
            "freqfit_plus_full_final_block_control",
        ],
        "rank_loss_weights": [0.0, 0.05],
        "optimizer_arms_after_architecture_screen": [
            "adamw",
            "loft_for_low_rank_winner_only",
        ],
        "selection_rule": (
            "pooled_log_loss_then_auroc_within_band_subject_to_"
            "per_protocol_regret_constraints"
        ),
        "phase52_failure_groups_to_track": [8, 10, 12],
        "anchor_only_epoch_zero_allowed": True,
        "public_leaderboard_used": False,
    },
    "validation": {
        "fold_assignment_source": (
            "PHASE52_SUPPORT_GATE_STATE_PRIVATE"
        ),
        "repeat_count": int(phase53_fold_assignment.shape[0]),
        "fold_count_per_repeat": 5,
        "all_1362_labels_are_development_labels": True,
        "new_pristine_holdout_claimed": False,
        "contract_batch_from_repeat_0_fold_0_fit_only": True,
    },
    "execution": {
        "device": str(phase53_device),
        "mixed_precision": (
            "bfloat16_except_float32_fft"
            if phase53_cuda
            else "float32"
        ),
        "peak_vram_mb": (
            float(torch.cuda.max_memory_allocated() / (1024.0 ** 2))
            if phase53_cuda
            else 0.0
        ),
    },
    "fit_labels_used_for_backward_contract": True,
    "validation_labels_used_for_selection": False,
    "test_data_read": False,
    "smoke_data_read": False,
    "training_penultimate_cache_read": True,
    "training_voxel_cache_read": False,
    "case_level_indices_displayed": False,
    "case_level_predictions_exported": False,
    "elapsed_seconds": round(time.perf_counter() - phase53a_started, 3),
}

PHASE53_CONTRACT_REPORT_PRIVATE = copy.deepcopy(phase53_report)
PHASE53_CONTRACT_STATE_PRIVATE = {
    "model_class": Phase53FreqDoRAResidualModel,
    "freqfit_class": Phase53FreqFiT2D,
    "dora_linear_class": Phase53DoRALinear,
    "dora_targets": tuple(PHASE53_DORA_TARGETS),
    "rope_sincos": phase53_to_device_nested(
        phase53_rope_sincos, torch.device("cpu")
    ),
    "backbone_source": phase53_backbone_source,
}

# Contract model was intentionally modified by three optimizer steps. Do not
# reuse it for selection; Phase53B will create every model from a fresh copy.
del phase53_optimizer
phase53_model.to("cpu")
if phase53_cuda:
    torch.cuda.empty_cache()

print("BEGIN SANITIZED_PHASE53_FREQFIT_DORA_CONTRACT")
print(json.dumps(phase53_report, indent=2))
print("END SANITIZED_PHASE53_FREQFIT_DORA_CONTRACT")

AssertionError: {'message': 'No initialized accepted DINOv3 backbone was found. Re-run the accepted Phase51 DINO offline-reload contract cell, then rerun this cell.', 'checked_names': ['PHASE51_OFFLINE_RELOAD', 'PHASE51_DINOV3_BACKBONE_PRIVATE', 'PHASE39_BACKBONE_PRIVATE', 'PHASE32_DINOV3_BACKBONE', 'PHASE30_DINOV3_BACKBONE']}

In [78]:
# Cell 153B — Phase53

from __future__ import annotations

import copy
import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader, Dataset


phase53b_started = time.perf_counter()

PHASE53_PILOT_CONFIG = {
    "seed": 530201,
    "repeat": 0,
    "fold": 0,
    "training_case_count": 256,
    "validation_case_count": 96,
    "batch_size": 8,
    "worker_count": 2,
    "epoch_count": 2,
    "warmup_fraction": 0.10,
    "weight_decay": 0.02,
    "gradient_clip": 1.0,
    "rank_loss_weight": 0.05,
    "adaptation_mode": "dora",
    "dora_rank": 8,
    "dora_alpha": 4.0,
    "residual_cap": 0.5,
    "learning_rates": {
        "freqfit": 1.0e-4,
        "dora": 1.0e-5,
        "full_final_block": 2.0e-6,
        "pooling": 1.0e-4,
        "head": 2.0e-4,
    },
}


phase53b_required_globals = (
    "PHASE53_CONTRACT_REPORT_PRIVATE",
    "PHASE53_CONTRACT_STATE_PRIVATE",
    "PHASE52_SUPPORT_GATE_STATE_PRIVATE",
    "Phase53FreqDoRAResidualModel",
    "Phase53DoRALinear",
    "phase53_get_child",
    "phase53_set_child",
    "phase53_base_final_block",
    "phase53_base_final_norm",
    "phase53_penultimate_cache",
    "phase53_rope_sincos",
    "phase53_device",
    "phase53_cuda",
)
phase53b_missing_globals = [
    name for name in phase53b_required_globals if name not in globals()
]
assert not phase53b_missing_globals, {
    "message": "Run the accepted corrected Cell 153A first.",
    "missing": phase53b_missing_globals,
}
assert (
    PHASE53_CONTRACT_REPORT_PRIVATE["status"]
    == "accepted_ready_for_frequency_adapter_screen"
)


def phase53b_seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


phase53b_seed_everything(PHASE53_PILOT_CONFIG["seed"])
if phase53_cuda:
    torch.cuda.reset_peak_memory_stats()
torch.set_float32_matmul_precision("high")


phase53b_state = PHASE52_SUPPORT_GATE_STATE_PRIVATE
phase53b_labels = np.asarray(
    phase53b_state["labels"], dtype=np.int64
).reshape(-1)
phase53b_groups = np.asarray(
    phase53b_state["groups"], dtype=np.int64
).reshape(-1)
phase53b_anchor_logit = np.asarray(
    phase53b_state["anchor_logit"], dtype=np.float64
).reshape(-1)
phase53b_fold_assignment = np.asarray(
    phase53b_state["fold_assignment"], dtype=np.int64
)

assert phase53b_labels.shape == (1362,)
assert phase53b_groups.shape == (1362,)
assert phase53b_anchor_logit.shape == (1362,)
assert phase53b_fold_assignment.shape == (5, 1362)
assert phase53_penultimate_cache.shape == (1362, 6, 201, 384)
assert phase53_penultimate_cache.dtype == np.float16
assert np.all(np.isfinite(phase53b_anchor_logit))


def phase53b_stratified_subsample(
    indices: np.ndarray,
    labels: np.ndarray,
    groups: np.ndarray,
    requested_count: int,
    seed: int,
) -> np.ndarray:
    """Deterministic group×label proportional subsample with exact size."""
    indices = np.asarray(indices, dtype=np.int64)
    if requested_count > indices.size:
        raise ValueError(
            f"Requested {requested_count} from only {indices.size} cases."
        )
    if requested_count == indices.size:
        return np.sort(indices.copy())

    rng = np.random.default_rng(seed)
    strata: List[np.ndarray] = []
    stratum_keys: List[Tuple[int, int]] = []
    for group in np.unique(groups[indices]):
        for label in (0, 1):
            local = indices[
                (groups[indices] == group)
                & (labels[indices] == label)
            ].copy()
            if local.size:
                rng.shuffle(local)
                strata.append(local)
                stratum_keys.append((int(group), int(label)))

    sizes = np.asarray([len(stratum) for stratum in strata], dtype=float)
    exact = requested_count * sizes / sizes.sum()
    allocation = np.floor(exact).astype(int)

    # Preserve each available stratum when the requested sample is large
    # enough, then correct back to the exact requested total.
    if requested_count >= len(strata):
        allocation = np.maximum(allocation, 1)
    allocation = np.minimum(allocation, sizes.astype(int))

    while int(allocation.sum()) > requested_count:
        removable = np.flatnonzero(allocation > 1)
        if removable.size == 0:
            removable = np.flatnonzero(allocation > 0)
        priorities = allocation[removable] - exact[removable]
        target = removable[int(np.argmax(priorities))]
        allocation[target] -= 1

    while int(allocation.sum()) < requested_count:
        available = np.flatnonzero(allocation < sizes)
        assert available.size > 0
        priorities = exact[available] - allocation[available]
        target = available[int(np.argmax(priorities))]
        allocation[target] += 1

    selected = np.concatenate(
        [
            stratum[:count]
            for stratum, count in zip(strata, allocation)
            if count > 0
        ]
    )
    rng.shuffle(selected)
    selected = selected.astype(np.int64, copy=False)
    assert selected.size == requested_count
    assert np.unique(selected).size == requested_count
    assert np.all(np.isin(selected, indices))
    assert np.unique(labels[selected]).size == 2
    return selected


phase53b_assignment = phase53b_fold_assignment[
    PHASE53_PILOT_CONFIG["repeat"]
]
phase53b_complete_train = np.flatnonzero(
    phase53b_assignment != PHASE53_PILOT_CONFIG["fold"]
)
phase53b_complete_valid = np.flatnonzero(
    phase53b_assignment == PHASE53_PILOT_CONFIG["fold"]
)
assert np.intersect1d(
    phase53b_complete_train, phase53b_complete_valid
).size == 0
assert np.union1d(
    phase53b_complete_train, phase53b_complete_valid
).size == 1362

phase53b_train_indices = phase53b_stratified_subsample(
    phase53b_complete_train,
    phase53b_labels,
    phase53b_groups,
    PHASE53_PILOT_CONFIG["training_case_count"],
    PHASE53_PILOT_CONFIG["seed"] + 1,
)
phase53b_valid_indices = phase53b_stratified_subsample(
    phase53b_complete_valid,
    phase53b_labels,
    phase53b_groups,
    PHASE53_PILOT_CONFIG["validation_case_count"],
    PHASE53_PILOT_CONFIG["seed"] + 2,
)
assert np.intersect1d(
    phase53b_train_indices, phase53b_valid_indices
).size == 0


def phase53b_group_balanced_weights(
    indices: np.ndarray,
    groups: np.ndarray,
) -> np.ndarray:
    local_groups = groups[indices]
    unique_groups, counts = np.unique(local_groups, return_counts=True)
    count_map = {
        int(group): int(count)
        for group, count in zip(unique_groups, counts)
    }
    weights = np.asarray(
        [1.0 / count_map[int(group)] for group in local_groups],
        dtype=np.float64,
    )
    weights /= weights.mean()
    assert np.all(np.isfinite(weights))
    assert np.all(weights > 0.0)
    return weights.astype(np.float32)


phase53b_train_case_weights = phase53b_group_balanced_weights(
    phase53b_train_indices, phase53b_groups
)


class Phase53TokenDataset(Dataset):
    def __init__(
        self,
        cache: np.ndarray,
        indices: np.ndarray,
        labels: np.ndarray,
        groups: np.ndarray,
        anchor_logit: np.ndarray,
        case_weights: Optional[np.ndarray] = None,
    ) -> None:
        self.cache = cache
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.groups = np.asarray(groups, dtype=np.int64)
        self.anchor_logit = np.asarray(anchor_logit, dtype=np.float64)
        if case_weights is None:
            self.case_weights = np.ones(len(self.indices), dtype=np.float32)
        else:
            self.case_weights = np.asarray(
                case_weights, dtype=np.float32
            ).reshape(-1)
        assert self.case_weights.shape == (len(self.indices),)

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, position: int) -> Dict[str, torch.Tensor]:
        case_index = int(self.indices[position])
        # Explicit copy prevents read-only memmap tensors and keeps DataLoader
        # workers independent.
        tokens = np.array(
            self.cache[case_index],
            dtype=np.float32,
            copy=True,
        )
        assert tokens.shape == (6, 201, 384)
        return {
            "tokens": torch.from_numpy(tokens),
            "anchor_logit": torch.tensor(
                self.anchor_logit[case_index], dtype=torch.float32
            ),
            "label": torch.tensor(
                self.labels[case_index], dtype=torch.float32
            ),
            "group": torch.tensor(
                self.groups[case_index], dtype=torch.int64
            ),
            "case_weight": torch.tensor(
                self.case_weights[position], dtype=torch.float32
            ),
        }


phase53b_train_dataset = Phase53TokenDataset(
    cache=phase53_penultimate_cache,
    indices=phase53b_train_indices,
    labels=phase53b_labels,
    groups=phase53b_groups,
    anchor_logit=phase53b_anchor_logit,
    case_weights=phase53b_train_case_weights,
)
phase53b_valid_dataset = Phase53TokenDataset(
    cache=phase53_penultimate_cache,
    indices=phase53b_valid_indices,
    labels=phase53b_labels,
    groups=phase53b_groups,
    anchor_logit=phase53b_anchor_logit,
)


def phase53b_make_loader(
    dataset: Dataset,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    workers = int(PHASE53_PILOT_CONFIG["worker_count"])
    return DataLoader(
        dataset,
        batch_size=int(PHASE53_PILOT_CONFIG["batch_size"]),
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=phase53_cuda,
        persistent_workers=workers > 0,
        drop_last=False,
        generator=generator,
    )


phase53b_train_loader = phase53b_make_loader(
    phase53b_train_dataset,
    shuffle=True,
    seed=PHASE53_PILOT_CONFIG["seed"] + 3,
)
phase53b_valid_loader = phase53b_make_loader(
    phase53b_valid_dataset,
    shuffle=False,
    seed=PHASE53_PILOT_CONFIG["seed"] + 4,
)


def phase53b_linear_from_dora(
    module: Phase53DoRALinear,
) -> nn.Linear:
    if not isinstance(module, Phase53DoRALinear):
        raise TypeError(type(module))
    replacement = nn.Linear(
        module.in_features,
        module.out_features,
        bias=module.bias is not None,
        device=module.weight.device,
        dtype=module.weight.dtype,
    )
    with torch.no_grad():
        replacement.weight.copy_(module.weight.detach())
        if replacement.bias is not None:
            replacement.bias.copy_(module.bias.detach())
    return replacement


class Phase53ScreenModel(Phase53FreqDoRAResidualModel):
    """One fresh model supporting all four predeclared architecture arms."""

    VALID_MODES = ("frozen", "dora", "full")

    def __init__(
        self,
        final_block: nn.Module,
        final_norm: nn.Module,
        adaptation_mode: str,
        dora_rank: int,
        dora_alpha: float,
        residual_cap: float,
    ) -> None:
        if adaptation_mode not in self.VALID_MODES:
            raise ValueError(adaptation_mode)
        self.adaptation_mode = adaptation_mode
        super().__init__(
            final_block=final_block,
            final_norm=final_norm,
            dora_rank=max(1, int(dora_rank)),
            dora_alpha=float(dora_alpha),
            view_projection_dimension=96,
            head_hidden_dimension=128,
            residual_cap=float(residual_cap),
        )

        if adaptation_mode in ("frozen", "full"):
            for path in tuple(self.dora_module_names):
                dora_module = phase53_get_child(self.final_block, path)
                replacement = phase53b_linear_from_dora(dora_module)
                phase53_set_child(self.final_block, path, replacement)

        if adaptation_mode == "frozen":
            self.final_block.requires_grad_(False)
            self.final_norm.requires_grad_(False)
        elif adaptation_mode == "dora":
            # Base weights are already frozen inside each DoRA module.
            self.final_norm.requires_grad_(False)
        elif adaptation_mode == "full":
            self.final_block.requires_grad_(True)
            self.final_norm.requires_grad_(True)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.adaptation_mode == "full":
            self.final_block.train(mode)
            self.final_norm.train(mode)
        else:
            self.final_block.eval()
            self.final_norm.eval()
        return self


def phase53b_fresh_model(
    adaptation_mode: str,
    dora_rank: int,
    seed: int,
) -> Phase53ScreenModel:
    phase53b_seed_everything(seed)
    model = Phase53ScreenModel(
        final_block=phase53_base_final_block,
        final_norm=phase53_base_final_norm,
        adaptation_mode=adaptation_mode,
        dora_rank=dora_rank,
        dora_alpha=PHASE53_PILOT_CONFIG["dora_alpha"],
        residual_cap=PHASE53_PILOT_CONFIG["residual_cap"],
    )
    return model.to(phase53_device)


def phase53b_trainable_parameter_groups(
    model: Phase53ScreenModel,
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    used_ids = set()
    groups: List[Dict[str, Any]] = []
    counts: Dict[str, int] = {}

    def add_group(name: str, parameters: Iterable[nn.Parameter], lr: float):
        selected = []
        for parameter in parameters:
            if parameter.requires_grad and id(parameter) not in used_ids:
                selected.append(parameter)
                used_ids.add(id(parameter))
        if selected:
            groups.append({"params": selected, "lr": float(lr), "name": name})
            counts[name] = int(sum(p.numel() for p in selected))

    add_group(
        "freqfit",
        model.freqfit.parameters(),
        PHASE53_PILOT_CONFIG["learning_rates"]["freqfit"],
    )
    if model.adaptation_mode == "dora":
        adapter_parameters = []
        for path in model.dora_module_names:
            adapter_parameters.extend(
                list(phase53_get_child(model.final_block, path).parameters())
            )
        add_group(
            "dora",
            adapter_parameters,
            PHASE53_PILOT_CONFIG["learning_rates"]["dora"],
        )
    elif model.adaptation_mode == "full":
        add_group(
            "full_final_block",
            list(model.final_block.parameters())
            + list(model.final_norm.parameters()),
            PHASE53_PILOT_CONFIG["learning_rates"]["full_final_block"],
        )
    add_group(
        "pooling",
        model.view_projection.parameters(),
        PHASE53_PILOT_CONFIG["learning_rates"]["pooling"],
    )
    add_group(
        "head",
        model.head.parameters(),
        PHASE53_PILOT_CONFIG["learning_rates"]["head"],
    )

    expected_ids = {
        id(parameter)
        for parameter in model.parameters()
        if parameter.requires_grad
    }
    assert used_ids == expected_ids, {
        "message": "Optimizer parameter partition is incomplete.",
        "missing_parameter_count": len(expected_ids - used_ids),
        "extra_parameter_count": len(used_ids - expected_ids),
    }
    return groups, counts


def phase53b_group_pairwise_rank_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    groups: torch.Tensor,
) -> Tuple[torch.Tensor, int]:
    losses = []
    active_group_count = 0
    for group in torch.unique(groups):
        mask = groups == group
        positive = logits[mask & (labels > 0.5)]
        negative = logits[mask & (labels <= 0.5)]
        if positive.numel() == 0 or negative.numel() == 0:
            continue
        differences = positive[:, None] - negative[None, :]
        losses.append(F.softplus(-differences).mean())
        active_group_count += 1
    if not losses:
        return logits.sum() * 0.0, 0
    return torch.stack(losses).mean(), active_group_count


def phase53b_snapshot_trainable(
    model: nn.Module,
) -> Dict[str, torch.Tensor]:
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }


def phase53b_restore_trainable(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    current = dict(model.named_parameters())
    assert set(state) == {
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }
    with torch.no_grad():
        for name, tensor in state.items():
            current[name].copy_(
                tensor.to(
                    device=current[name].device,
                    dtype=current[name].dtype,
                )
            )


def phase53b_binary_metrics(
    labels: np.ndarray,
    probabilities: np.ndarray,
) -> Dict[str, float]:
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    probabilities = np.clip(probabilities, 1.0e-7, 1.0 - 1.0e-7)
    return {
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
        "auroc": float(roc_auc_score(labels, probabilities)),
        "mean_probability": float(np.mean(probabilities)),
    }


@torch.no_grad()
def phase53b_evaluate(
    model: Phase53ScreenModel,
    loader: DataLoader,
) -> Tuple[np.ndarray, Dict[str, float]]:
    model.eval()
    probabilities = []
    labels = []
    for batch in loader:
        tokens = batch["tokens"].to(
            phase53_device, non_blocking=phase53_cuda
        )
        anchor = batch["anchor_logit"].to(
            phase53_device, non_blocking=phase53_cuda
        )
        with torch.autocast(
            device_type=phase53_device.type,
            dtype=torch.bfloat16 if phase53_cuda else torch.float32,
            enabled=phase53_cuda,
        ):
            output = model(tokens, anchor, phase53_rope_sincos)
        probability = output["probability"].detach().float().cpu().numpy()
        probabilities.append(probability)
        labels.append(batch["label"].numpy())
    probabilities_array = np.concatenate(probabilities).astype(np.float64)
    labels_array = np.concatenate(labels).astype(np.int64)
    assert probabilities_array.shape == labels_array.shape
    assert np.all(np.isfinite(probabilities_array))
    assert np.all(
        (probabilities_array > 0.0) & (probabilities_array < 1.0)
    )
    return probabilities_array, phase53b_binary_metrics(
        labels_array, probabilities_array
    )


def phase53b_learning_rate_multiplier(
    step_index: int,
    total_steps: int,
    warmup_fraction: float,
) -> float:
    if total_steps <= 1:
        return 1.0
    warmup_steps = max(1, int(round(total_steps * warmup_fraction)))
    if step_index < warmup_steps:
        return float(step_index + 1) / float(warmup_steps)
    progress = (step_index - warmup_steps) / max(
        1, total_steps - warmup_steps - 1
    )
    progress = min(max(float(progress), 0.0), 1.0)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))


# Validate construction, identity, and optimizer partition for all modes before
# spending time on the superset pilot.
phase53b_mode_contracts = []
for phase53b_mode, phase53b_rank in (
    ("frozen", 0),
    ("dora", 4),
    ("dora", 8),
    ("full", 0),
):
    phase53b_contract_model = phase53b_fresh_model(
        adaptation_mode=phase53b_mode,
        dora_rank=phase53b_rank,
        seed=(
            PHASE53_PILOT_CONFIG["seed"]
            + 10
            + len(phase53b_mode_contracts)
        ),
    )
    parameter_groups, parameter_counts = (
        phase53b_trainable_parameter_groups(phase53b_contract_model)
    )
    contract_batch = next(iter(phase53b_valid_loader))
    contract_tokens = contract_batch["tokens"].to(phase53_device)
    contract_anchor = contract_batch["anchor_logit"].to(phase53_device)
    phase53b_contract_model.eval()
    with torch.no_grad(), torch.autocast(
        device_type=phase53_device.type,
        dtype=torch.bfloat16 if phase53_cuda else torch.float32,
        enabled=phase53_cuda,
    ):
        contract_output = phase53b_contract_model(
            contract_tokens,
            contract_anchor,
            phase53_rope_sincos,
        )
    identity_error = float(
        torch.max(
            torch.abs(
                contract_output["logit"].float()
                - contract_anchor.float()
            )
        ).item()
    )
    assert identity_error <= 1.0e-7
    phase53b_mode_contracts.append(
        {
            "mode": phase53b_mode,
            "dora_rank": phase53b_rank,
            "trainable_parameter_count": int(
                sum(
                    parameter.numel()
                    for parameter in phase53b_contract_model.parameters()
                    if parameter.requires_grad
                )
            ),
            "optimizer_parameter_counts": parameter_counts,
            "initial_identity_logit_error": identity_error,
        }
    )
    phase53b_contract_model.to("cpu")
    del phase53b_contract_model, parameter_groups, contract_output
    if phase53_cuda:
        torch.cuda.empty_cache()


phase53b_model = phase53b_fresh_model(
    adaptation_mode=PHASE53_PILOT_CONFIG["adaptation_mode"],
    dora_rank=PHASE53_PILOT_CONFIG["dora_rank"],
    seed=PHASE53_PILOT_CONFIG["seed"] + 100,
)
phase53b_optimizer_groups, phase53b_parameter_counts = (
    phase53b_trainable_parameter_groups(phase53b_model)
)
phase53b_optimizer = torch.optim.AdamW(
    phase53b_optimizer_groups,
    betas=(0.9, 0.95),
    weight_decay=PHASE53_PILOT_CONFIG["weight_decay"],
)
phase53b_base_lrs = [
    float(group["lr"]) for group in phase53b_optimizer.param_groups
]


phase53b_epoch_zero_probability, phase53b_epoch_zero_metrics = (
    phase53b_evaluate(phase53b_model, phase53b_valid_loader)
)
phase53b_valid_anchor_probability = 1.0 / (
    1.0 + np.exp(-phase53b_anchor_logit[phase53b_valid_indices])
)
phase53b_epoch_zero_probability_error = float(
    np.max(
        np.abs(
            phase53b_epoch_zero_probability
            - phase53b_valid_anchor_probability
        )
    )
)
assert phase53b_epoch_zero_probability_error <= 1.0e-7

phase53b_best_epoch = 0
phase53b_best_metrics = copy.deepcopy(phase53b_epoch_zero_metrics)
phase53b_best_probability = phase53b_epoch_zero_probability.copy()
phase53b_best_trainable_state = phase53b_snapshot_trainable(
    phase53b_model
)

phase53b_steps_per_epoch = len(phase53b_train_loader)
phase53b_total_steps = (
    PHASE53_PILOT_CONFIG["epoch_count"] * phase53b_steps_per_epoch
)
phase53b_global_step = 0
phase53b_history = []
phase53b_all_gradient_norms_finite = True

for phase53b_epoch in range(1, PHASE53_PILOT_CONFIG["epoch_count"] + 1):
    epoch_started = time.perf_counter()
    phase53b_model.train()
    epoch_bce_sum = 0.0
    epoch_rank_sum = 0.0
    epoch_case_count = 0
    epoch_rank_active_batch_count = 0
    epoch_maximum_gradient_norm = 0.0

    for batch in phase53b_train_loader:
        multiplier = phase53b_learning_rate_multiplier(
            phase53b_global_step,
            phase53b_total_steps,
            PHASE53_PILOT_CONFIG["warmup_fraction"],
        )
        for optimizer_group, base_lr in zip(
            phase53b_optimizer.param_groups, phase53b_base_lrs
        ):
            optimizer_group["lr"] = base_lr * multiplier

        tokens = batch["tokens"].to(
            phase53_device, non_blocking=phase53_cuda
        )
        anchor = batch["anchor_logit"].to(
            phase53_device, non_blocking=phase53_cuda
        )
        labels = batch["label"].to(
            phase53_device, non_blocking=phase53_cuda
        )
        groups = batch["group"].to(
            phase53_device, non_blocking=phase53_cuda
        )
        case_weights = batch["case_weight"].to(
            phase53_device, non_blocking=phase53_cuda
        )

        phase53b_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=phase53_device.type,
            dtype=torch.bfloat16 if phase53_cuda else torch.float32,
            enabled=phase53_cuda,
        ):
            output = phase53b_model(
                tokens, anchor, phase53_rope_sincos
            )
            per_case_bce = F.binary_cross_entropy_with_logits(
                output["logit"].float(),
                labels.float(),
                reduction="none",
            )
            bce_loss = torch.sum(per_case_bce * case_weights) / torch.sum(
                case_weights
            ).clamp_min(1.0e-8)
            rank_loss, active_groups = phase53b_group_pairwise_rank_loss(
                output["logit"].float(),
                labels.float(),
                groups,
            )
            total_loss = (
                bce_loss
                + PHASE53_PILOT_CONFIG["rank_loss_weight"] * rank_loss
            )
        total_loss.backward()

        gradients = [
            parameter.grad
            for parameter in phase53b_model.parameters()
            if parameter.requires_grad and parameter.grad is not None
        ]
        gradients_finite = bool(
            gradients
            and all(
                torch.all(torch.isfinite(gradient)).item()
                for gradient in gradients
            )
        )
        phase53b_all_gradient_norms_finite &= gradients_finite
        assert gradients_finite
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter in phase53b_model.parameters()
                if parameter.requires_grad
            ],
            max_norm=PHASE53_PILOT_CONFIG["gradient_clip"],
        )
        gradient_norm_float = float(gradient_norm.detach().item())
        assert math.isfinite(gradient_norm_float)
        epoch_maximum_gradient_norm = max(
            epoch_maximum_gradient_norm, gradient_norm_float
        )
        phase53b_optimizer.step()

        batch_size = int(labels.numel())
        epoch_bce_sum += float(bce_loss.detach().item()) * batch_size
        epoch_rank_sum += float(rank_loss.detach().item()) * batch_size
        epoch_case_count += batch_size
        epoch_rank_active_batch_count += int(active_groups > 0)
        phase53b_global_step += 1

    valid_probability, valid_metrics = phase53b_evaluate(
        phase53b_model, phase53b_valid_loader
    )
    improved = bool(
        valid_metrics["log_loss"]
        < phase53b_best_metrics["log_loss"] - 1.0e-12
    )
    if improved:
        phase53b_best_epoch = int(phase53b_epoch)
        phase53b_best_metrics = copy.deepcopy(valid_metrics)
        phase53b_best_probability = valid_probability.copy()
        phase53b_best_trainable_state = phase53b_snapshot_trainable(
            phase53b_model
        )

    history_record = {
        "epoch": int(phase53b_epoch),
        "train_bce": float(epoch_bce_sum / epoch_case_count),
        "train_rank": float(epoch_rank_sum / epoch_case_count),
        "rank_active_batch_count": int(epoch_rank_active_batch_count),
        "validation": valid_metrics,
        "maximum_gradient_norm": epoch_maximum_gradient_norm,
        "improved_over_previous_best": improved,
        "learning_rate_multiplier_end": float(
            phase53b_learning_rate_multiplier(
                phase53b_global_step - 1,
                phase53b_total_steps,
                PHASE53_PILOT_CONFIG["warmup_fraction"],
            )
        ),
        "elapsed_seconds": round(
            time.perf_counter() - epoch_started, 3
        ),
    }
    phase53b_history.append(history_record)
    print(
        "Phase53 training-engine pilot "
        f"epoch {phase53b_epoch}/{PHASE53_PILOT_CONFIG['epoch_count']}: "
        f"valid_log_loss={valid_metrics['log_loss']:.6f}, "
        f"valid_auroc={valid_metrics['auroc']:.6f}"
    )


phase53b_restore_trainable(
    phase53b_model, phase53b_best_trainable_state
)
phase53b_restored_probability, phase53b_restored_metrics = (
    phase53b_evaluate(phase53b_model, phase53b_valid_loader)
)
phase53b_restored_probability_error = float(
    np.max(
        np.abs(
            phase53b_restored_probability - phase53b_best_probability
        )
    )
)
assert phase53b_restored_probability_error <= 1.0e-7
assert abs(
    phase53b_restored_metrics["log_loss"]
    - phase53b_best_metrics["log_loss"]
) <= 1.0e-10
assert abs(
    phase53b_restored_metrics["auroc"]
    - phase53b_best_metrics["auroc"]
) <= 1.0e-12


phase53b_report = {
    "phase": "phase53_freqfit_dora_training_engine_pilot",
    "status": "accepted_ready_for_repeated_architecture_screen",
    "purpose": (
        "validate_complete_multi_mode_training_engine_before_expensive_screen"
    ),
    "partition": {
        "repeat": PHASE53_PILOT_CONFIG["repeat"],
        "fold": PHASE53_PILOT_CONFIG["fold"],
        "complete_train_n": int(phase53b_complete_train.size),
        "complete_valid_n": int(phase53b_complete_valid.size),
        "pilot_train_n": int(phase53b_train_indices.size),
        "pilot_valid_n": int(phase53b_valid_indices.size),
        "pilot_train_group_count": int(
            np.unique(phase53b_groups[phase53b_train_indices]).size
        ),
        "pilot_valid_group_count": int(
            np.unique(phase53b_groups[phase53b_valid_indices]).size
        ),
        "train_valid_disjoint": True,
    },
    "mode_construction_contracts": phase53b_mode_contracts,
    "pilot_arm": {
        "adaptation_mode": PHASE53_PILOT_CONFIG["adaptation_mode"],
        "dora_rank": PHASE53_PILOT_CONFIG["dora_rank"],
        "dora_alpha": PHASE53_PILOT_CONFIG["dora_alpha"],
        "rank_loss_weight": PHASE53_PILOT_CONFIG["rank_loss_weight"],
        "optimizer_parameter_counts": phase53b_parameter_counts,
    },
    "engine_contract": {
        "group_balanced_bce_weights": True,
        "within_batch_group_pairwise_rank_loss": True,
        "each_training_case_used_once_per_epoch": True,
        "mixed_precision": (
            "bfloat16_except_float32_fft"
            if phase53_cuda
            else "float32"
        ),
        "epoch_zero_anchor_fallback": True,
        "trainable_only_checkpoint": True,
        "checkpoint_restoration_verified": True,
        "frozen_final_block_eval_behavior": True,
    },
    "epoch_zero": {
        "probability_maximum_error": phase53b_epoch_zero_probability_error,
        "validation": phase53b_epoch_zero_metrics,
    },
    "training": {
        "epoch_count": PHASE53_PILOT_CONFIG["epoch_count"],
        "history": phase53b_history,
        "all_gradient_norms_finite": phase53b_all_gradient_norms_finite,
    },
    "checkpoint_selection": {
        "selected_epoch": phase53b_best_epoch,
        "selected_validation": phase53b_best_metrics,
        "restored_probability_maximum_error": (
            phase53b_restored_probability_error
        ),
        "epoch_zero_remained_selected": phase53b_best_epoch == 0,
    },
    "next_screen": {
        "architecture_arms": [
            {
                "adaptation_mode": "frozen",
                "dora_rank": 0,
                "label": "freqfit_only",
            },
            {
                "adaptation_mode": "dora",
                "dora_rank": 4,
                "label": "freqfit_plus_dora_rank4",
            },
            {
                "adaptation_mode": "dora",
                "dora_rank": 8,
                "label": "freqfit_plus_dora_rank8",
            },
            {
                "adaptation_mode": "full",
                "dora_rank": 0,
                "label": "freqfit_plus_full_final_block_control",
            },
        ],
        "rank_loss_weights": [0.0, 0.05],
        "candidate_count": 8,
        "screen_repeats": [0, 1],
        "confirmation_repeats": [2, 3, 4],
        "screen_fold_count": 10,
        "planned_screen_fit_count": 80,
        "shared_epoch_per_candidate": True,
        "epoch_zero_identity_allowed": True,
        "protocol_regret_groups": [8, 10, 12],
        "loft_optimizer_used_during_architecture_screen": False,
        "loft_tested_only_for_frozen_low_rank_winner": True,
    },
    "validation_semantics": {
        "all_1362_labels_are_development_labels": True,
        "new_pristine_holdout_claimed": False,
        "pilot_validation_used_only_for_engine_testing": True,
        "public_leaderboard_used": False,
    },
    "execution": {
        "device": str(phase53_device),
        "peak_vram_mb": (
            float(torch.cuda.max_memory_allocated() / (1024.0 ** 2))
            if phase53_cuda
            else 0.0
        ),
        "elapsed_seconds": round(
            time.perf_counter() - phase53b_started, 3
        ),
    },
    "fit_labels_used": True,
    "pilot_validation_labels_used_for_engine_test": True,
    "test_data_read": False,
    "smoke_data_read": False,
    "training_penultimate_cache_read": True,
    "training_voxel_cache_read": False,
    "case_level_indices_displayed": False,
    "case_level_predictions_exported": False,
}

PHASE53_TRAINING_ENGINE_REPORT_PRIVATE = copy.deepcopy(
    phase53b_report
)
PHASE53_TRAINING_ENGINE_STATE_PRIVATE = {
    "screen_model_class": Phase53ScreenModel,
    "dataset_class": Phase53TokenDataset,
    "fresh_model_function": phase53b_fresh_model,
    "parameter_group_function": phase53b_trainable_parameter_groups,
    "rank_loss_function": phase53b_group_pairwise_rank_loss,
    "evaluate_function": phase53b_evaluate,
    "snapshot_function": phase53b_snapshot_trainable,
    "restore_function": phase53b_restore_trainable,
    "subsample_function": phase53b_stratified_subsample,
    "group_weight_function": phase53b_group_balanced_weights,
}

phase53b_model.to("cpu")
del phase53b_optimizer, phase53b_model
if phase53_cuda:
    torch.cuda.empty_cache()

print("BEGIN SANITIZED_PHASE53_TRAINING_ENGINE")
print(json.dumps(phase53b_report, indent=2))
print("END SANITIZED_PHASE53_TRAINING_ENGINE")


Phase53 training-engine pilot epoch 1/2: valid_log_loss=0.242327, valid_auroc=0.960758
Phase53 training-engine pilot epoch 2/2: valid_log_loss=0.242194, valid_auroc=0.960758
BEGIN SANITIZED_PHASE53_TRAINING_ENGINE
{
  "phase": "phase53_freqfit_dora_training_engine_pilot",
  "status": "accepted_ready_for_repeated_architecture_screen",
  "purpose": "validate_complete_multi_mode_training_engine_before_expensive_screen",
  "partition": {
    "repeat": 0,
    "fold": 0,
    "complete_train_n": 1089,
    "complete_valid_n": 273,
    "pilot_train_n": 256,
    "pilot_valid_n": 96,
    "pilot_train_group_count": 15,
    "pilot_valid_group_count": 15,
    "train_valid_disjoint": true
  },
  "mode_construction_contracts": [
    {
      "mode": "frozen",
      "dora_rank": 0,
      "trainable_parameter_count": 299873,
      "optimizer_parameter_counts": {
        "freqfit": 86784,
        "pooling": 112992,
        "head": 100097
      },
      "initial_identity_logit_error": 0.0
    },
    {
    

In [1]:
# Cell 153C — Phase53 

from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader


phase53c_started = time.perf_counter()

PHASE53_SCREEN_CONFIG = {
    "seed": 530301,
    "screen_repeats": (0, 1),
    "fold_count": 5,
    "maximum_epochs": 8,
    "batch_size": 8,
    "worker_count": 2,
    "warmup_fraction": 0.10,
    "weight_decay": 0.02,
    "gradient_clip": 1.0,
    "selection_log_loss_band": 0.002,
    "major_group_minimum_n": 30,
    "maximum_major_group_harm": 0.015,
    "maximum_worst_repeat_log_loss_regret": 0.003,
    "minimum_major_group_wins": 7,
    "minimum_repeat_log_loss_wins": 2,
    "minimum_repeat_auroc_wins": 2,
    "minimum_screen_log_loss_gain": 0.003,
    "minimum_screen_auroc_gain": 0.001,
    "checkpoint_directory": Path(
        "/kaggle/working/phase53_private_checkpoint"
    ),
}


phase53c_required_globals = (
    "PHASE53_TRAINING_ENGINE_REPORT_PRIVATE",
    "PHASE53_TRAINING_ENGINE_STATE_PRIVATE",
    "PHASE52_SUPPORT_GATE_STATE_PRIVATE",
    "Phase53ScreenModel",
    "Phase53TokenDataset",
    "phase53b_fresh_model",
    "phase53b_trainable_parameter_groups",
    "phase53b_group_pairwise_rank_loss",
    "phase53b_group_balanced_weights",
    "phase53b_evaluate",
    "phase53b_learning_rate_multiplier",
    "phase53_penultimate_cache",
    "phase53_rope_sincos",
    "phase53_device",
    "phase53_cuda",
    "phase53b_labels",
    "phase53b_groups",
    "phase53b_anchor_logit",
    "phase53b_fold_assignment",
)
phase53c_missing_globals = [
    name for name in phase53c_required_globals if name not in globals()
]
assert not phase53c_missing_globals, {
    "message": "Run accepted Cells 153A and 153B first.",
    "missing": phase53c_missing_globals,
}
assert (
    PHASE53_TRAINING_ENGINE_REPORT_PRIVATE["status"]
    == "accepted_ready_for_repeated_architecture_screen"
)


def phase53c_seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


phase53c_seed_everything(PHASE53_SCREEN_CONFIG["seed"])
torch.set_float32_matmul_precision("high")
if phase53_cuda:
    torch.cuda.reset_peak_memory_stats()


phase53c_labels = np.asarray(phase53b_labels, dtype=np.int64).reshape(-1)
phase53c_groups = np.asarray(phase53b_groups, dtype=np.int64).reshape(-1)
phase53c_anchor_logit = np.asarray(
    phase53b_anchor_logit, dtype=np.float64
).reshape(-1)
phase53c_anchor_probability = 1.0 / (
    1.0 + np.exp(-phase53c_anchor_logit)
)
phase53c_fold_assignment = np.asarray(
    phase53b_fold_assignment, dtype=np.int64
)
phase53c_case_count = int(phase53c_labels.size)

assert phase53c_case_count == 1362
assert phase53c_groups.shape == (phase53c_case_count,)
assert phase53c_anchor_logit.shape == (phase53c_case_count,)
assert phase53c_anchor_probability.shape == (phase53c_case_count,)
assert phase53c_fold_assignment.shape == (5, phase53c_case_count)
assert set(np.unique(phase53c_labels).tolist()) == {0, 1}
assert np.unique(phase53c_groups).size == 15
assert np.all(np.isfinite(phase53c_anchor_probability))
assert np.all(
    (phase53c_anchor_probability > 0.0)
    & (phase53c_anchor_probability < 1.0)
)


PHASE53_ARCHITECTURE_CANDIDATES = (
    {
        "candidate_index": 0,
        "label": "freqfit_only_bce",
        "adaptation_mode": "frozen",
        "dora_rank": 0,
        "rank_loss_weight": 0.0,
    },
    {
        "candidate_index": 1,
        "label": "freqfit_only_rank",
        "adaptation_mode": "frozen",
        "dora_rank": 0,
        "rank_loss_weight": 0.05,
    },
    {
        "candidate_index": 2,
        "label": "freqfit_dora4_bce",
        "adaptation_mode": "dora",
        "dora_rank": 4,
        "rank_loss_weight": 0.0,
    },
    {
        "candidate_index": 3,
        "label": "freqfit_dora4_rank",
        "adaptation_mode": "dora",
        "dora_rank": 4,
        "rank_loss_weight": 0.05,
    },
    {
        "candidate_index": 4,
        "label": "freqfit_dora8_bce",
        "adaptation_mode": "dora",
        "dora_rank": 8,
        "rank_loss_weight": 0.0,
    },
    {
        "candidate_index": 5,
        "label": "freqfit_dora8_rank",
        "adaptation_mode": "dora",
        "dora_rank": 8,
        "rank_loss_weight": 0.05,
    },
    {
        "candidate_index": 6,
        "label": "freqfit_full_block_bce",
        "adaptation_mode": "full",
        "dora_rank": 0,
        "rank_loss_weight": 0.0,
    },
    {
        "candidate_index": 7,
        "label": "freqfit_full_block_rank",
        "adaptation_mode": "full",
        "dora_rank": 0,
        "rank_loss_weight": 0.05,
    },
)

assert [
    candidate["candidate_index"]
    for candidate in PHASE53_ARCHITECTURE_CANDIDATES
] == list(range(8))
assert len(
    {
        (
            candidate["adaptation_mode"],
            candidate["dora_rank"],
            candidate["rank_loss_weight"],
        )
        for candidate in PHASE53_ARCHITECTURE_CANDIDATES
    }
) == 8


def phase53c_make_loader(
    dataset: Phase53TokenDataset,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    workers = int(PHASE53_SCREEN_CONFIG["worker_count"])
    return DataLoader(
        dataset,
        batch_size=int(PHASE53_SCREEN_CONFIG["batch_size"]),
        shuffle=bool(shuffle),
        num_workers=workers,
        pin_memory=phase53_cuda,
        persistent_workers=False,
        drop_last=False,
        generator=generator,
    )


def phase53c_binary_metrics(
    labels: np.ndarray,
    probabilities: np.ndarray,
) -> Dict[str, float]:
    labels = np.asarray(labels, dtype=np.int64).reshape(-1)
    probabilities = np.asarray(
        probabilities, dtype=np.float64
    ).reshape(-1)
    assert labels.shape == probabilities.shape
    probabilities = np.clip(probabilities, 1.0e-7, 1.0 - 1.0e-7)
    return {
        "n": int(labels.size),
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
        "auroc": float(roc_auc_score(labels, probabilities)),
        "mean_probability": float(np.mean(probabilities)),
    }


def phase53c_safe_group_auroc(
    labels: np.ndarray,
    probabilities: np.ndarray,
) -> Optional[float]:
    labels = np.asarray(labels, dtype=np.int64)
    if np.unique(labels).size < 2:
        return None
    return float(roc_auc_score(labels, probabilities))


phase53c_anchor_metrics = phase53c_binary_metrics(
    phase53c_labels, phase53c_anchor_probability
)
phase53c_major_groups = [
    int(group)
    for group, count in zip(
        *np.unique(phase53c_groups, return_counts=True)
    )
    if int(count) >= PHASE53_SCREEN_CONFIG["major_group_minimum_n"]
]
assert len(phase53c_major_groups) == 10


def phase53c_train_one_fold(
    candidate: Dict[str, Any],
    repeat: int,
    fold: int,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    fit_started = time.perf_counter()
    assignment = phase53c_fold_assignment[repeat]
    train_indices = np.flatnonzero(assignment != fold)
    valid_indices = np.flatnonzero(assignment == fold)
    assert train_indices.size + valid_indices.size == phase53c_case_count
    assert np.intersect1d(train_indices, valid_indices).size == 0

    train_weights = phase53b_group_balanced_weights(
        train_indices, phase53c_groups
    )
    train_dataset = Phase53TokenDataset(
        cache=phase53_penultimate_cache,
        indices=train_indices,
        labels=phase53c_labels,
        groups=phase53c_groups,
        anchor_logit=phase53c_anchor_logit,
        case_weights=train_weights,
    )
    valid_dataset = Phase53TokenDataset(
        cache=phase53_penultimate_cache,
        indices=valid_indices,
        labels=phase53c_labels,
        groups=phase53c_groups,
        anchor_logit=phase53c_anchor_logit,
    )
    fit_seed = int(
        PHASE53_SCREEN_CONFIG["seed"]
        + candidate["candidate_index"] * 10000
        + repeat * 100
        + fold
    )
    train_loader = phase53c_make_loader(
        train_dataset, shuffle=True, seed=fit_seed + 1
    )
    valid_loader = phase53c_make_loader(
        valid_dataset, shuffle=False, seed=fit_seed + 2
    )

    model = phase53b_fresh_model(
        adaptation_mode=candidate["adaptation_mode"],
        dora_rank=candidate["dora_rank"],
        seed=fit_seed + 3,
    )
    optimizer_groups, parameter_counts = (
        phase53b_trainable_parameter_groups(model)
    )
    optimizer = torch.optim.AdamW(
        optimizer_groups,
        betas=(0.9, 0.95),
        weight_decay=PHASE53_SCREEN_CONFIG["weight_decay"],
    )
    base_lrs = [float(group["lr"]) for group in optimizer.param_groups]

    maximum_epochs = int(PHASE53_SCREEN_CONFIG["maximum_epochs"])
    predictions_by_epoch = np.full(
        (maximum_epochs + 1, valid_indices.size),
        np.nan,
        dtype=np.float32,
    )
    predictions_by_epoch[0] = phase53c_anchor_probability[
        valid_indices
    ].astype(np.float32)

    steps_per_epoch = len(train_loader)
    total_steps = maximum_epochs * steps_per_epoch
    global_step = 0
    maximum_gradient_norm = 0.0
    rank_active_batch_count = 0
    final_train_bce = None
    final_train_rank = None

    for epoch in range(1, maximum_epochs + 1):
        model.train()
        epoch_bce_sum = 0.0
        epoch_rank_sum = 0.0
        epoch_case_count = 0
        for batch in train_loader:
            lr_multiplier = phase53b_learning_rate_multiplier(
                global_step,
                total_steps,
                PHASE53_SCREEN_CONFIG["warmup_fraction"],
            )
            for optimizer_group, base_lr in zip(
                optimizer.param_groups, base_lrs
            ):
                optimizer_group["lr"] = base_lr * lr_multiplier

            tokens = batch["tokens"].to(
                phase53_device, non_blocking=phase53_cuda
            )
            anchor = batch["anchor_logit"].to(
                phase53_device, non_blocking=phase53_cuda
            )
            labels = batch["label"].to(
                phase53_device, non_blocking=phase53_cuda
            )
            groups = batch["group"].to(
                phase53_device, non_blocking=phase53_cuda
            )
            case_weights = batch["case_weight"].to(
                phase53_device, non_blocking=phase53_cuda
            )

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=phase53_device.type,
                dtype=torch.bfloat16 if phase53_cuda else torch.float32,
                enabled=phase53_cuda,
            ):
                output = model(tokens, anchor, phase53_rope_sincos)
                per_case_bce = F.binary_cross_entropy_with_logits(
                    output["logit"].float(),
                    labels.float(),
                    reduction="none",
                )
                bce_loss = torch.sum(
                    per_case_bce * case_weights
                ) / torch.sum(case_weights).clamp_min(1.0e-8)
                rank_loss, active_group_count = (
                    phase53b_group_pairwise_rank_loss(
                        output["logit"].float(),
                        labels.float(),
                        groups,
                    )
                )
                total_loss = (
                    bce_loss
                    + float(candidate["rank_loss_weight"]) * rank_loss
                )
            total_loss.backward()
            gradients = [
                parameter.grad
                for parameter in model.parameters()
                if parameter.requires_grad and parameter.grad is not None
            ]
            assert gradients
            assert all(
                torch.all(torch.isfinite(gradient)).item()
                for gradient in gradients
            )
            gradient_norm = torch.nn.utils.clip_grad_norm_(
                [
                    parameter
                    for parameter in model.parameters()
                    if parameter.requires_grad
                ],
                max_norm=PHASE53_SCREEN_CONFIG["gradient_clip"],
            )
            gradient_norm_float = float(gradient_norm.detach().item())
            assert math.isfinite(gradient_norm_float)
            maximum_gradient_norm = max(
                maximum_gradient_norm, gradient_norm_float
            )
            optimizer.step()

            batch_n = int(labels.numel())
            epoch_bce_sum += float(bce_loss.detach().item()) * batch_n
            epoch_rank_sum += float(rank_loss.detach().item()) * batch_n
            epoch_case_count += batch_n
            rank_active_batch_count += int(active_group_count > 0)
            global_step += 1

        valid_probability, _ = phase53b_evaluate(model, valid_loader)
        predictions_by_epoch[epoch] = valid_probability.astype(np.float32)
        final_train_bce = float(epoch_bce_sum / epoch_case_count)
        final_train_rank = float(epoch_rank_sum / epoch_case_count)

    assert np.all(np.isfinite(predictions_by_epoch))
    assert np.all(
        (predictions_by_epoch > 0.0)
        & (predictions_by_epoch < 1.0)
    )
    identity_error = float(
        np.max(
            np.abs(
                predictions_by_epoch[0].astype(np.float64)
                - phase53c_anchor_probability[valid_indices]
            )
        )
    )
    assert identity_error <= 1.0e-7

    fit_record = {
        "candidate_index": int(candidate["candidate_index"]),
        "repeat": int(repeat),
        "fold": int(fold),
        "train_n": int(train_indices.size),
        "valid_n": int(valid_indices.size),
        "optimizer_parameter_counts": parameter_counts,
        "rank_active_batch_count": int(rank_active_batch_count),
        "maximum_gradient_norm": maximum_gradient_norm,
        "final_train_bce": final_train_bce,
        "final_train_rank": final_train_rank,
        "epoch_zero_identity_error": identity_error,
        "elapsed_seconds": round(time.perf_counter() - fit_started, 3),
    }

    model.to("cpu")
    del model, optimizer, train_loader, valid_loader
    if phase53_cuda:
        torch.cuda.empty_cache()
    return valid_indices, predictions_by_epoch, fit_record


phase53c_candidate_count = len(PHASE53_ARCHITECTURE_CANDIDATES)
phase53c_epoch_count = int(PHASE53_SCREEN_CONFIG["maximum_epochs"])
phase53c_screen_repeat_count = len(PHASE53_SCREEN_CONFIG["screen_repeats"])
phase53c_predictions = np.full(
    (
        phase53c_candidate_count,
        phase53c_epoch_count + 1,
        phase53c_screen_repeat_count,
        phase53c_case_count,
    ),
    np.nan,
    dtype=np.float32,
)
phase53c_coverage = np.zeros(
    (
        phase53c_candidate_count,
        phase53c_screen_repeat_count,
        phase53c_case_count,
    ),
    dtype=np.int8,
)
phase53c_fit_records = []
phase53c_fit_counter = 0
phase53c_planned_fit_count = (
    phase53c_candidate_count
    * phase53c_screen_repeat_count
    * PHASE53_SCREEN_CONFIG["fold_count"]
)

for candidate in PHASE53_ARCHITECTURE_CANDIDATES:
    candidate_index = int(candidate["candidate_index"])
    for repeat_position, repeat in enumerate(
        PHASE53_SCREEN_CONFIG["screen_repeats"]
    ):
        for fold in range(PHASE53_SCREEN_CONFIG["fold_count"]):
            valid_indices, fold_predictions, fit_record = (
                phase53c_train_one_fold(candidate, int(repeat), fold)
            )
            # Take basic-index views first. Combining a full slice with an
            # integer-array index in one NumPy expression moves the advanced
            # index to the front and produces (valid_n, epoch_n), whereas the
            # fold output is deliberately (epoch_n, valid_n).
            prediction_view = phase53c_predictions[
                candidate_index, :, repeat_position, :
            ]
            coverage_view = phase53c_coverage[
                candidate_index, repeat_position, :
            ]
            assert prediction_view.shape == (
                phase53c_epoch_count + 1,
                phase53c_case_count,
            )
            assert coverage_view.shape == (phase53c_case_count,)
            assert fold_predictions.shape == (
                phase53c_epoch_count + 1,
                valid_indices.size,
            )
            assert np.all(coverage_view[valid_indices] == 0)
            prediction_view[:, valid_indices] = fold_predictions
            coverage_view[valid_indices] += 1
            assert np.array_equal(
                prediction_view[:, valid_indices], fold_predictions
            )
            phase53c_fit_records.append(fit_record)
            phase53c_fit_counter += 1
            print(
                "Phase53 architecture screen: "
                f"{phase53c_fit_counter}/{phase53c_planned_fit_count}; "
                f"candidate={candidate_index}, repeat={repeat}, fold={fold}"
            )

assert phase53c_fit_counter == phase53c_planned_fit_count
assert np.all(phase53c_coverage == 1)
assert np.all(np.isfinite(phase53c_predictions))
assert np.all(
    (phase53c_predictions > 0.0) & (phase53c_predictions < 1.0)
)
phase53c_epoch_zero_error = float(
    np.max(
        np.abs(
            phase53c_predictions[:, 0].astype(np.float64)
            - phase53c_anchor_probability[None, None, :]
        )
    )
)
assert phase53c_epoch_zero_error <= 1.0e-7


def phase53c_candidate_epoch_record(
    candidate_index: int,
    epoch: int,
) -> Dict[str, Any]:
    repeat_predictions = phase53c_predictions[
        candidate_index, epoch
    ].astype(np.float64)
    mean_probability = np.mean(repeat_predictions, axis=0)
    metrics = phase53c_binary_metrics(
        phase53c_labels, mean_probability
    )
    repeat_metrics = []
    repeat_log_loss_wins = 0
    repeat_auroc_wins = 0
    worst_repeat_log_loss_regret = -float("inf")
    for repeat_position, repeat in enumerate(
        PHASE53_SCREEN_CONFIG["screen_repeats"]
    ):
        local_metrics = phase53c_binary_metrics(
            phase53c_labels, repeat_predictions[repeat_position]
        )
        log_loss_gain = (
            phase53c_anchor_metrics["log_loss"]
            - local_metrics["log_loss"]
        )
        auroc_gain = (
            local_metrics["auroc"]
            - phase53c_anchor_metrics["auroc"]
        )
        repeat_log_loss_wins += int(log_loss_gain > 1.0e-12)
        repeat_auroc_wins += int(auroc_gain > 1.0e-12)
        worst_repeat_log_loss_regret = max(
            worst_repeat_log_loss_regret,
            local_metrics["log_loss"]
            - phase53c_anchor_metrics["log_loss"],
        )
        repeat_metrics.append(
            {
                "repeat": int(repeat),
                "log_loss": local_metrics["log_loss"],
                "auroc": local_metrics["auroc"],
                "log_loss_gain": log_loss_gain,
                "auroc_gain": auroc_gain,
            }
        )

    group_records = []
    major_group_wins = 0
    maximum_major_group_harm = 0.0
    maximum_repeat_group_harm = 0.0
    for group in phase53c_major_groups:
        mask = phase53c_groups == group
        anchor_group_log_loss = float(
            log_loss(
                phase53c_labels[mask],
                phase53c_anchor_probability[mask],
                labels=[0, 1],
            )
        )
        candidate_group_log_loss = float(
            log_loss(
                phase53c_labels[mask],
                mean_probability[mask],
                labels=[0, 1],
            )
        )
        gain = anchor_group_log_loss - candidate_group_log_loss
        harm = max(0.0, -gain)
        major_group_wins += int(gain > 1.0e-12)
        maximum_major_group_harm = max(
            maximum_major_group_harm, harm
        )
        repeat_group_metrics = []
        for repeat_position, repeat in enumerate(
            PHASE53_SCREEN_CONFIG["screen_repeats"]
        ):
            repeat_group_log_loss = float(
                log_loss(
                    phase53c_labels[mask],
                    repeat_predictions[repeat_position, mask],
                    labels=[0, 1],
                )
            )
            repeat_group_gain = (
                anchor_group_log_loss - repeat_group_log_loss
            )
            maximum_repeat_group_harm = max(
                maximum_repeat_group_harm,
                max(0.0, -repeat_group_gain),
            )
            repeat_group_metrics.append(
                {
                    "repeat": int(repeat),
                    "candidate_log_loss": repeat_group_log_loss,
                    "log_loss_gain": repeat_group_gain,
                }
            )
        group_records.append(
            {
                "group": int(group),
                "n": int(np.sum(mask)),
                "anchor_log_loss": anchor_group_log_loss,
                "candidate_log_loss": candidate_group_log_loss,
                "log_loss_gain": gain,
                "anchor_auroc": phase53c_safe_group_auroc(
                    phase53c_labels[mask],
                    phase53c_anchor_probability[mask],
                ),
                "candidate_auroc": phase53c_safe_group_auroc(
                    phase53c_labels[mask], mean_probability[mask]
                ),
                "repeat_metrics": repeat_group_metrics,
            }
        )

    maximum_protocol_harm = max(
        maximum_major_group_harm,
        maximum_repeat_group_harm,
    )
    safe = bool(
        maximum_protocol_harm
        <= PHASE53_SCREEN_CONFIG["maximum_major_group_harm"]
        and worst_repeat_log_loss_regret
        <= PHASE53_SCREEN_CONFIG[
            "maximum_worst_repeat_log_loss_regret"
        ]
    )
    candidate = PHASE53_ARCHITECTURE_CANDIDATES[candidate_index]
    return {
        "candidate_index": int(candidate_index),
        "epoch": int(epoch),
        "specification": {
            "label": candidate["label"],
            "adaptation_mode": candidate["adaptation_mode"],
            "dora_rank": int(candidate["dora_rank"]),
            "rank_loss_weight": float(candidate["rank_loss_weight"]),
        },
        "log_loss": metrics["log_loss"],
        "auroc": metrics["auroc"],
        "mean_probability": metrics["mean_probability"],
        "log_loss_gain": (
            phase53c_anchor_metrics["log_loss"] - metrics["log_loss"]
        ),
        "auroc_gain": metrics["auroc"] - phase53c_anchor_metrics["auroc"],
        "repeat_log_loss_wins": int(repeat_log_loss_wins),
        "repeat_auroc_wins": int(repeat_auroc_wins),
        "worst_repeat_log_loss_regret": float(
            worst_repeat_log_loss_regret
        ),
        "major_group_wins": int(major_group_wins),
        "major_group_count": int(len(phase53c_major_groups)),
        "maximum_major_group_harm": float(maximum_major_group_harm),
        "maximum_repeat_group_harm": float(
            maximum_repeat_group_harm
        ),
        "maximum_protocol_harm": float(maximum_protocol_harm),
        "safe": safe,
        "repeat_metrics": repeat_metrics,
        "major_group_metrics": group_records,
    }


phase53c_records = []
for candidate_index in range(phase53c_candidate_count):
    for epoch in range(phase53c_epoch_count + 1):
        phase53c_records.append(
            phase53c_candidate_epoch_record(candidate_index, epoch)
        )

phase53c_safe_records = [
    record for record in phase53c_records if record["safe"]
]
assert phase53c_safe_records, {
    "message": "No protocol-safe screen candidate, including epoch zero."
}
phase53c_raw_log_loss_best = min(
    phase53c_safe_records,
    key=lambda record: (
        record["log_loss"],
        -record["auroc"],
        record["candidate_index"],
        record["epoch"],
    ),
)
phase53c_log_loss_limit = (
    phase53c_raw_log_loss_best["log_loss"]
    + PHASE53_SCREEN_CONFIG["selection_log_loss_band"]
)
phase53c_band_records = [
    record
    for record in phase53c_safe_records
    if record["log_loss"] <= phase53c_log_loss_limit + 1.0e-15
]
assert phase53c_band_records
phase53c_provisional = max(
    phase53c_band_records,
    key=lambda record: (
        record["auroc"],
        -record["log_loss"],
        record["major_group_wins"],
        -record["maximum_major_group_harm"],
        -record["candidate_index"],
        -record["epoch"],
    ),
)

phase53c_screen_advanced = bool(
    phase53c_provisional["epoch"] > 0
    and phase53c_provisional["log_loss_gain"]
    >= PHASE53_SCREEN_CONFIG["minimum_screen_log_loss_gain"]
    and phase53c_provisional["auroc_gain"]
    >= PHASE53_SCREEN_CONFIG["minimum_screen_auroc_gain"]
    and phase53c_provisional["repeat_log_loss_wins"]
    >= PHASE53_SCREEN_CONFIG["minimum_repeat_log_loss_wins"]
    and phase53c_provisional["repeat_auroc_wins"]
    >= PHASE53_SCREEN_CONFIG["minimum_repeat_auroc_wins"]
    and phase53c_provisional["major_group_wins"]
    >= PHASE53_SCREEN_CONFIG["minimum_major_group_wins"]
)

phase53c_epoch_zero_record = phase53c_candidate_epoch_record(0, 0)
phase53c_selected = (
    phase53c_provisional
    if phase53c_screen_advanced
    else phase53c_epoch_zero_record
)


phase53c_candidate_summaries = []
for candidate_index in range(phase53c_candidate_count):
    candidate_records = [
        record
        for record in phase53c_records
        if record["candidate_index"] == candidate_index
    ]
    safe_candidate_records = [
        record for record in candidate_records if record["safe"]
    ]
    best_record = min(
        safe_candidate_records or candidate_records,
        key=lambda record: (
            record["log_loss"],
            -record["auroc"],
            record["epoch"],
        ),
    )
    candidate = PHASE53_ARCHITECTURE_CANDIDATES[candidate_index]
    candidate_fit_records = [
        record
        for record in phase53c_fit_records
        if record["candidate_index"] == candidate_index
    ]
    phase53c_candidate_summaries.append(
        {
            "candidate_index": int(candidate_index),
            "specification": {
                "label": candidate["label"],
                "adaptation_mode": candidate["adaptation_mode"],
                "dora_rank": int(candidate["dora_rank"]),
                "rank_loss_weight": float(
                    candidate["rank_loss_weight"]
                ),
            },
            "best_safe_or_raw_epoch": int(best_record["epoch"]),
            "log_loss": best_record["log_loss"],
            "auroc": best_record["auroc"],
            "log_loss_gain": best_record["log_loss_gain"],
            "auroc_gain": best_record["auroc_gain"],
            "repeat_log_loss_wins": best_record[
                "repeat_log_loss_wins"
            ],
            "repeat_auroc_wins": best_record["repeat_auroc_wins"],
            "major_group_wins": best_record["major_group_wins"],
            "maximum_major_group_harm": best_record[
                "maximum_major_group_harm"
            ],
            "maximum_repeat_group_harm": best_record[
                "maximum_repeat_group_harm"
            ],
            "maximum_protocol_harm": best_record[
                "maximum_protocol_harm"
            ],
            "safe": best_record["safe"],
            "mean_fit_seconds": float(
                np.mean(
                    [
                        record["elapsed_seconds"]
                        for record in candidate_fit_records
                    ]
                )
            ),
            "maximum_gradient_norm": float(
                max(
                    record["maximum_gradient_norm"]
                    for record in candidate_fit_records
                )
            ),
        }
    )


phase53c_checkpoint_directory = PHASE53_SCREEN_CONFIG[
    "checkpoint_directory"
]
phase53c_checkpoint_directory.mkdir(parents=True, exist_ok=True)
phase53c_prediction_file = (
    phase53c_checkpoint_directory
    / "phase53_screen_predictions_float32.npy"
)
np.save(phase53c_prediction_file, phase53c_predictions)
assert phase53c_prediction_file.is_file()

phase53c_selection_contract = {
    "phase": "phase53_architecture_screen_selection_state",
    "status": (
        "candidate_frozen_for_confirmation"
        if phase53c_screen_advanced
        else "anchor_only_stop_before_confirmation"
    ),
    "screen_advanced": phase53c_screen_advanced,
    "selected_candidate_index": int(
        phase53c_selected["candidate_index"]
    ),
    "selected_epoch": int(phase53c_selected["epoch"]),
    "selected_specification": copy.deepcopy(
        phase53c_selected["specification"]
    ),
    "screen_repeats": list(PHASE53_SCREEN_CONFIG["screen_repeats"]),
    "confirmation_repeats": [2, 3, 4],
}
phase53c_selection_payload = json.dumps(
    phase53c_selection_contract,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")
phase53c_selection_sha256 = hashlib.sha256(
    phase53c_selection_payload
).hexdigest()
phase53c_selection_contract["selection_sha256"] = (
    phase53c_selection_sha256
)
phase53c_selection_file = (
    phase53c_checkpoint_directory
    / "phase53_architecture_selection.json"
)
phase53c_temporary_selection_file = phase53c_selection_file.with_suffix(
    ".json.tmp"
)
with open(phase53c_temporary_selection_file, "w", encoding="utf-8") as file:
    json.dump(phase53c_selection_contract, file, indent=2)
    file.flush()
    os.fsync(file.fileno())
os.replace(phase53c_temporary_selection_file, phase53c_selection_file)
assert phase53c_selection_file.is_file()


def phase53c_public_record(record: Dict[str, Any]) -> Dict[str, Any]:
    return copy.deepcopy(record)


phase53c_total_fit_seconds = float(
    sum(record["elapsed_seconds"] for record in phase53c_fit_records)
)
phase53c_report = {
    "phase": "phase53_repeated_freqfit_dora_architecture_screen",
    "status": (
        "candidate_frozen_for_confirmation"
        if phase53c_screen_advanced
        else "screen_gate_failed_anchor_only_frozen"
    ),
    "screen": {
        "candidate_count": phase53c_candidate_count,
        "epoch_zero_plus_trained_epoch_count": phase53c_epoch_count + 1,
        "repeat_count": phase53c_screen_repeat_count,
        "fold_count_per_repeat": PHASE53_SCREEN_CONFIG["fold_count"],
        "planned_model_fit_count": phase53c_planned_fit_count,
        "completed_model_fit_count": phase53c_fit_counter,
        "complete_case_coverage_per_repeat": True,
        "shared_candidate_and_epoch_across_all_screen_folds": True,
        "fold_specific_epoch_selection_used": False,
        "loft_optimizer_used": False,
    },
    "anchor": phase53c_anchor_metrics,
    "raw_log_loss_best": phase53c_public_record(
        phase53c_raw_log_loss_best
    ),
    "provisional": phase53c_public_record(phase53c_provisional),
    "selected": phase53c_public_record(phase53c_selected),
    "candidate_summaries": phase53c_candidate_summaries,
    "selection": {
        "safe_candidate_epoch_count": len(phase53c_safe_records),
        "log_loss_band": PHASE53_SCREEN_CONFIG[
            "selection_log_loss_band"
        ],
        "screen_advanced": phase53c_screen_advanced,
        "selection_rule": (
            "among_protocol_safe_candidate_epochs_choose_maximum_AUROC_"
            "within_0p002_of_minimum_log_loss_then_require_all_screen_gates"
        ),
        "selection_sha256": phase53c_selection_sha256,
    },
    "screen_gate_thresholds": {
        "minimum_log_loss_gain": PHASE53_SCREEN_CONFIG[
            "minimum_screen_log_loss_gain"
        ],
        "minimum_auroc_gain": PHASE53_SCREEN_CONFIG[
            "minimum_screen_auroc_gain"
        ],
        "minimum_repeat_log_loss_wins": PHASE53_SCREEN_CONFIG[
            "minimum_repeat_log_loss_wins"
        ],
        "minimum_repeat_auroc_wins": PHASE53_SCREEN_CONFIG[
            "minimum_repeat_auroc_wins"
        ],
        "minimum_major_group_wins": PHASE53_SCREEN_CONFIG[
            "minimum_major_group_wins"
        ],
        "maximum_major_group_harm": PHASE53_SCREEN_CONFIG[
            "maximum_major_group_harm"
        ],
        "maximum_protocol_harm_semantics": (
            "maximum_of_repeat_ensemble_group_harm_and_each_repeat_group_harm"
        ),
        "maximum_worst_repeat_log_loss_regret": (
            PHASE53_SCREEN_CONFIG[
                "maximum_worst_repeat_log_loss_regret"
            ]
        ),
    },
    "phase52_failure_group_audit": {
        "groups": [8, 10, 12],
        "selected_group_records": [
            record
            for record in phase53c_selected["major_group_metrics"]
            if record["group"] in (8, 10, 12)
        ],
    },
    "next_step": (
        "train_frozen_candidate_on_confirmation_repeats_2_3_4"
        if phase53c_screen_advanced
        else "stop_phase53_before_confirmation_and_open_new_hypothesis"
    ),
    "private_state": {
        "prediction_file": phase53c_prediction_file.name,
        "prediction_shape": list(phase53c_predictions.shape),
        "selection_file": phase53c_selection_file.name,
        "contains_development_predictions": True,
        "include_in_submission": False,
    },
    "validation_semantics": {
        "all_1362_labels_are_development_labels": True,
        "new_pristine_holdout_claimed": False,
        "screen_repeats_used_for_selection": [0, 1],
        "confirmation_repeats_not_used": [2, 3, 4],
        "confirmation_repeats_are_stability_checks_not_pristine_holdouts": True,
        "public_leaderboard_used": False,
    },
    "execution": {
        "device": str(phase53_device),
        "summed_model_fit_seconds": phase53c_total_fit_seconds,
        "peak_vram_mb": (
            float(torch.cuda.max_memory_allocated() / (1024.0 ** 2))
            if phase53_cuda
            else 0.0
        ),
        "elapsed_seconds": round(
            time.perf_counter() - phase53c_started, 3
        ),
    },
    "fit_labels_used": True,
    "screen_validation_labels_used_for_selection": True,
    "confirmation_labels_used": False,
    "test_data_read": False,
    "smoke_data_read": False,
    "training_penultimate_cache_read": True,
    "training_voxel_cache_read": False,
    "case_level_indices_displayed": False,
    "case_level_predictions_exported": False,
}

PHASE53_ARCHITECTURE_SCREEN_REPORT_PRIVATE = copy.deepcopy(
    phase53c_report
)
PHASE53_ARCHITECTURE_SCREEN_STATE_PRIVATE = {
    "candidates": copy.deepcopy(PHASE53_ARCHITECTURE_CANDIDATES),
    "selected_record": copy.deepcopy(phase53c_selected),
    "provisional_record": copy.deepcopy(phase53c_provisional),
    "screen_advanced": bool(phase53c_screen_advanced),
    "selection_sha256": phase53c_selection_sha256,
    "screen_predictions": phase53c_predictions,
    "fit_records": copy.deepcopy(phase53c_fit_records),
}

if phase53_cuda:
    torch.cuda.empty_cache()

print("BEGIN SANITIZED_PHASE53_ARCHITECTURE_SCREEN")
print(json.dumps(phase53c_report, indent=2))
print("END SANITIZED_PHASE53_ARCHITECTURE_SCREEN")
print(phase53c_selection_file)

AssertionError: {'message': 'Run accepted Cells 153A and 153B first.', 'missing': ['PHASE53_TRAINING_ENGINE_REPORT_PRIVATE', 'PHASE53_TRAINING_ENGINE_STATE_PRIVATE', 'PHASE52_SUPPORT_GATE_STATE_PRIVATE', 'Phase53ScreenModel', 'Phase53TokenDataset', 'phase53b_fresh_model', 'phase53b_trainable_parameter_groups', 'phase53b_group_pairwise_rank_loss', 'phase53b_group_balanced_weights', 'phase53b_evaluate', 'phase53b_learning_rate_multiplier', 'phase53_penultimate_cache', 'phase53_rope_sincos', 'phase53_device', 'phase53_cuda', 'phase53b_labels', 'phase53b_groups', 'phase53b_anchor_logit', 'phase53b_fold_assignment']}